# FlashAttention forward kernel in Triton

Standard attention writes the $T \times T$ score matrix to memory, softmaxes it and reads it back. FlashAttention keeps a tile of queries in registers, streams key/value tiles past it, and folds each tile into a running softmax, so the scores never leave the chip.

Write `flash_attn_fwd(q, k, v, causal)`.

## Inputs
- `q`, `k`, `v`: contiguous `float32` CUDA tensors of shape `(B, H, T, D)`. $T \ge 1$ is arbitrary (not a multiple of the tile); $D \in \{16, 32, 64, 128\}$.
- `causal`: Python bool. If true, query $i$ attends only to keys $j \le i$.

## Output
A tuple `(o, lse)`:
- `o`: `float32` `(B, H, T, D)`, $o_i = \sum_j \mathrm{softmax}_j(s\, q_i \cdot k_j)\, v_j$ with $s = 1/\sqrt{D}$.
- `lse`: `float32` `(B, H, T)`, the natural-log log-sum-exp of each query row's scaled (and masked) scores, $\log\sum_j e^{s\, q_i\cdot k_j}$. The backward pass needs it.

## Requirements
- Grid `(triton.cdiv(T, BLOCK_M), B * H)`: program `(pid_m, bh)` owns query rows `pid_m*BLOCK_M + [0, BLOCK_M)` of head `bh`. Use `BLOCK_M = BLOCK_N = 64`, all `tl.constexpr`.
- Keep per row a running max `m`, running normaliser `l` and an unnormalised `(BLOCK_M, D)` accumulator, all in `float32`. For every key tile: scores, new max, rescale factor `alpha = exp(m - m_new)`, then update `l` and `acc`.
- Mask keys at positions $\ge T$ (score $-\infty$), and with `causal`, keys $j > i$. With `causal`, stop the loop after the tile that contains the diagonal: later tiles are fully masked.
- Use `tl.dot(..., input_precision="ieee")` so float32 inputs are not rounded to TF32.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import math
import torch
import triton
import triton.language as tl

def flash_attn_fwd(q, k, v, causal):
    """q, k, v: (B, H, T, D) float32 CUDA. Return (o (B, H, T, D), lse (B, H, T))."""
    raise NotImplementedError


@triton.jit
def _attn_fwd_kernel(q_ptr, k_ptr, v_ptr, o_ptr, lse_ptr, T, sm_scale,
                     D: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, CAUSAL: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, false], "e": {"__tup__": [{"__nd__": [[[[1.0530052185058594, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.9469947814941406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[4.035976409912109, 4.035976409912109, 4.035976409912109]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[[4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, true], "e": {"__tup__": [{"__nd__": [[[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.9820138216018677, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.9469947814941406, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[4.0, 4.0181498527526855, 4.035976409912109]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[[0.04872092232108116, 0.5181635022163391, -0.11009939759969711, -0.08903376013040543, 0.5025036334991455, 0.5105738639831543, 0.34507566690444946, -1.4208372831344604, -0.5697727203369141, -0.3065755069255829, 0.7395539879798889, 0.04545522481203079, -0.3938700556755066, 0.3775947391986847, 0.32589447498321533, -1.1415613889694214]]]], "dtype": "float32"}, {"__nd__": [[[[0.5414602160453796, -0.42470860481262207, 0.21265631914138794, 0.46147429943084717, -0.9584651589393616, 1.2063082456588745, -1.009986400604248, 0.8593730926513672, 0.6420454382896423, -0.17440564930438995, 1.2201077938079834, -0.617811918258667, -1.3640137910842896, 0.02170197106897831, -0.9355947375297546, -0.07422322034835815]]]], "dtype": "float32"}, {"__nd__": [[[[0.2685922384262085, 1.52106511592865, -0.2289394736289978, 0.10297729074954987, 0.6583266258239746, 0.7939301133155823, -0.11635640263557434, 0.9060273170471191, -1.0007978677749634, -0.5254555940628052, -0.6878044605255127, -0.3192591667175293, 0.5993638038635254, 0.01907079480588436, 0.9518969655036926, -1.3533518314361572]]]], "dtype": "float32"}, true], "e": {"__tup__": [{"__nd__": [[[[0.2685922384262085, 1.52106511592865, -0.2289394736289978, 0.10297729074954987, 0.6583266258239746, 0.7939301133155823, -0.11635640263557434, 0.9060273170471191, -1.0007978677749634, -0.5254555940628052, -0.6878044605255127, -0.3192591667175293, 0.5993638038635254, 0.01907079480588436, 0.9518969655036926, -1.3533518314361572]]]], "dtype": "float32"}, {"__nd__": [[[-0.2015741765499115]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[[-0.5711071491241455, 0.7486174702644348, -0.13961124420166016, -0.3699210286140442, -1.9281774759292603, 0.6268024444580078, -1.087469458580017, 0.4138473570346832, -1.0303705930709839, -0.5407230257987976, 1.4969027042388916, -1.0131771564483643, -0.9171861410140991, -0.21792398393154144, -1.8159656524658203, -0.012320639565587044, 0.02520827017724514, -0.29935529828071594, -0.8431467413902283, -0.10252751410007477, -0.6760656237602234, -0.3564867675304413, -0.8273131251335144, 0.559451699256897, -0.41618162393569946, 0.9196897149085999, -0.8695139288902283, -0.30105826258659363, -0.7113582491874695, 0.4203914999961853, 1.0969128608703613, 0.7305814027786255], [1.565186619758606, 0.5025708079338074, -1.5536410808563232, 0.17941762506961823, 1.0023841857910156, 0.6680580377578735, 0.07020885497331619, 0.1353430151939392, 0.1122274249792099, -1.7653273344039917, -0.8340529799461365, 0.5179054737091064, 1.1437108516693115, -0.7984316945075989, -0.5777889490127563, 0.3661237061023712, 0.38133734464645386, -0.22430668771266937, -0.8162416815757751, 0.7306479215621948, 0.8725808262825012, -1.4093871116638184, -1.4838693141937256, 0.5013472437858582, 0.1330358386039734, -0.9248915314674377, -0.2991701364517212, 0.09971760958433151, 1.0078803300857544, 0.5280818343162537, 0.5757765769958496, 0.6408995985984802], [0.11672814935445786, 0.013027729466557503, -0.11740097403526306, 0.6544362306594849, -0.5241007208824158, -1.3520896434783936, 0.9510064721107483, 0.6253566741943359, 2.3171353340148926, 0.8227531909942627, -0.3098030984401703, -0.5023126602172852, 1.8532140254974365, 0.8065117001533508, -1.4745843410491943, -0.5987508893013, -0.08372674882411957, 1.0662610530853271, -0.5926604270935059, -2.217514753341675, 0.6017329692840576, -1.4539823532104492, -0.33014407753944397, -1.7416822910308838, -0.5377973318099976, -0.05332942679524422, -1.8670737743377686, -1.0546900033950806, 0.29624414443969727, -0.5211326479911804, 0.7090000510215759, -0.6403698921203613], [-1.1723206043243408, -1.1586509943008423, 0.5074805617332458, 0.9464078545570374, 0.1683867871761322, 0.773796558380127, 0.7898746132850647, -1.3561590909957886, -0.9127295017242432, -0.11946743726730347, 0.16370923817157745, 0.14613518118858337, -0.11622894555330276, -0.5972924828529358, 0.09008230268955231, 0.09158270806074142, -0.2846396565437317, 0.5838499665260315, -0.280200332403183, -0.10511665046215057, 0.2540622353553772, -0.6119998097419739, 1.491460919380188, -0.9476284384727478, 0.4597335755825043, -1.399294137954712, -0.17507301270961761, -0.92941814661026, -0.5520604252815247, 1.6470988988876343, -0.3974962532520294, 0.21804076433181763], [-0.5786077976226807, 1.5830315351486206, 2.7190213203430176, -1.0582464933395386, -1.5740925073623657, -0.8747435212135315, 1.1525790691375732, -0.12010890990495682, -2.1175410747528076, -1.2815890312194824, 0.2002095878124237, -0.1656995564699173, 1.3764373064041138, 0.14471736550331116, -0.382088840007782, -0.5983319878578186, -0.7518711090087891, -1.1034449338912964, -0.24955207109451294, 0.798419713973999, -0.740263819694519, -0.3609309494495392, 0.17356324195861816, 1.2326958179473877, 0.14134320616722107, 0.14468811452388763, 0.6607032418251038, 0.13428185880184174, 0.15333819389343262, -0.37028494477272034, 1.2579143047332764, 0.21053867042064667], [-0.6709585189819336, 0.5300102233886719, 0.11835848540067673, -1.534484624862671, 0.2305147796869278, -0.682871401309967, -0.3075435161590576, 0.7066054344177246, 0.8869092464447021, 0.6384521722793579, 0.21334180235862732, 1.2677327394485474, -1.0554381608963013, 1.4953525066375732, 0.8444609045982361, -0.6229085326194763, -1.5650079250335693, 0.09182847291231155, -0.06305495649576187, -0.873446524143219, -0.7171552181243896, 0.525734543800354, 0.5065853595733643, -0.14360375702381134, 0.4552403390407562, -0.032062824815511703, -1.391355037689209, -1.021620273590088, 0.2065853625535965, -0.2783859968185425, 0.20223742723464966, 0.7025311589241028], [-1.0341068506240845, -1.328961968421936, 0.36825263500213623, -0.03743480145931244, 0.2073766142129898, 0.29479652643203735, 3.3055365085601807, 0.07820013910531998, 0.18960045278072357, -0.2740197479724884, -0.5896667838096619, -0.7085010409355164, 1.1300667524337769, 0.8650707602500916, -1.137020230293274, -1.9154417514801025, 0.9046825766563416, 0.9885274767875671, -0.29039114713668823, 0.0956614762544632, 0.10587695986032486, -0.7083949446678162, -1.0790334939956665, 0.36161622405052185, 0.9433946013450623, 0.1537339985370636, 1.1917288303375244, 0.1981733739376068, -0.7762905359268188, 0.5287501215934753, -0.2786925137042999, -0.769592821598053], [0.7297295928001404, 0.6113874912261963, -0.8105672597885132, -0.49236834049224854, -0.07367169111967087, -1.6112115383148193, -0.3081238567829132, -1.07228684425354, -0.17589561641216278, -0.896223783493042, 0.48738718032836914, -2.0109806060791016, 1.9530797004699707, -2.0485806465148926, 1.622982382774353, -0.011455648578703403, -0.6902891993522644, 2.679032802581787, -0.2773389518260956, 0.1629936844110489, -1.2882914543151855, -0.5591825246810913, -0.5284154415130615, 0.5479545593261719, 0.4601057469844818, 1.2897783517837524, -0.1650518774986267, -0.19122613966464996, -1.8905354738235474, 0.9259980320930481, -0.9113741517066956, -1.2010164260864258], [1.7167037725448608, -0.714726984500885, -1.3629883527755737, -0.7906105518341064, -0.6244349479675293, -0.7196376919746399, 0.5746213793754578, -0.09684613347053528, 1.3605562448501587, 0.3818510174751282, 0.6931704878807068, -1.8878551721572876, -0.8598743677139282, 0.2131253480911255, -0.4610397219657898, -0.23067954182624817, -0.12658239901065826, 0.5790060758590698, -0.1609402298927307, -0.410539835691452, -0.8210746049880981, 0.41476932168006897, 0.6748908162117004, 0.13311433792114258, -0.5722525715827942, -0.47288885712623596, 1.2942167520523071, 0.3898385167121887, -0.14289875328540802, 0.09327957034111023, 0.22390514612197876, 0.5630545020103455], [0.5783628821372986, -2.1265358924865723, -0.8023886680603027, 0.2419191598892212, 0.17282533645629883, -0.6372288465499878, 0.5345518589019775, -0.08348527550697327, 0.014838709495961666, -0.9665727615356445, 0.9550256133079529, 1.3619513511657715, -0.027091192081570625, 0.7083673477172852, -0.6768053770065308, -0.3164782226085663, -1.4375628232955933, -0.8856755495071411, 0.13599148392677307, -1.2996143102645874, 1.0721712112426758, 1.007942795753479, -0.08692941814661026, 0.9393733739852905, -0.6975956559181213, -0.87925785779953, -1.539903998374939, 0.3445165455341339, -0.32902511954307556, 1.9396557807922363, 1.3117687702178955, 0.20469464361667633], [0.0018182671628892422, 0.4411156475543976, -1.3234487771987915, 0.2750537693500519, 1.7730560302734375, -0.1895321160554886, -0.7430213093757629, 0.6139585971832275, -0.38850530982017517, 0.19314755499362946, 0.015471919439733028, -0.3171238303184509, 0.1426655352115631, -0.22640421986579895, 1.1619393825531006, -1.1575199365615845, 1.7017755508422852, -0.11577785015106201, 2.323349952697754, 0.2487034797668457, 0.10579796880483627, 1.412100911140442, -0.5964362621307373, -1.45677649974823, -1.2116987705230713, 0.4674382209777832, -0.10490261018276215, 1.3763384819030762, -0.13277116417884827, -0.44577157497406006, 2.6382293701171875, 0.8833984136581421], [-1.1237375736236572, 0.8747566938400269, -0.3410646319389343, 0.7846139669418335, 1.300171971321106, -0.11957236379384995, -2.0344913005828857, 0.518682062625885, 0.637843668460846, -0.4815058410167694, -1.1624295711517334, -0.9451951384544373, -1.4374983310699463, -0.9504721760749817, -0.46235114336013794, 0.4067630469799042, 0.0782713070511818, -0.24991059303283691, -0.2186633050441742, -0.022331563755869865, 0.7402938604354858, 0.7313165068626404, -0.18825773894786835, -0.1981009691953659, -0.06272578984498978, -0.8477129936218262, -2.6591312885284424, 1.703487515449524, 0.9950373768806458, 0.4757612645626068, 0.7838479280471802, 0.27309438586235046], [0.8005062341690063, 0.11353623121976852, -1.6323164701461792, -0.8401644229888916, 0.5754337906837463, 1.460955262184143, -0.8200792670249939, 1.335993766784668, 0.030976444482803345, -1.4005769491195679, 0.15958112478256226, -0.37992432713508606, 1.959632158279419, 0.324777215719223, 0.3309266269207001, 1.4892796277999878, 1.3781367540359497, -0.1828872710466385, -0.549013614654541, 0.055609289556741714, 0.7644767761230469, -0.04971843212842941, 0.07117439061403275, 1.1748807430267334, 1.4775290489196777, -0.12872333824634552, -0.7381842136383057, -0.5295759439468384, -0.45212623476982117, 0.6848577857017517, -0.0710008516907692, -0.5345975160598755], [0.7441983819007874, 0.9004515409469604, -1.045381784439087, 1.2047919034957886, 0.630961537361145, -0.43132057785987854, -0.3505767583847046, 0.058493442833423615, 0.41246461868286133, 0.42003777623176575, -0.2767229974269867, 0.5076912641525269, -0.9956649541854858, -1.5453784465789795, 1.7773479223251343, -0.39412209391593933, -0.6149293184280396, 0.5990170836448669, -1.2360858917236328, 0.7947745323181152, -0.32458993792533875, -0.6715646982192993, -1.045676350593567, -0.7969610095024109, -1.8249751329421997, 0.6165146231651306, 0.3650895953178406, 0.48712384700775146, 0.3599419593811035, 0.331895649433136, 0.8960137963294983, -0.68539959192276], [1.3656023740768433, 0.14084291458129883, 0.06743975728750229, -0.5003002882003784, -1.4237240552902222, 0.31442564725875854, 0.06565529108047485, -0.11595097929239273, -0.6086143255233765, -2.034482002258301, -0.4454667866230011, -0.23728047311306, 0.3907792866230011, 1.0539271831512451, -1.001478910446167, -1.722405195236206, 0.9878845810890198, 0.7412670850753784, -0.3913945257663727, 0.014208143576979637, -1.0526514053344727, 0.2170972377061844, 1.4739183187484741, 0.017085302621126175, -1.4593089818954468, 0.14609242975711823, 0.5475282073020935, 2.409393310546875, 1.0776804685592651, 0.1872231811285019, 0.5456570982933044, -1.3808116912841797], [0.17571218311786652, -0.7456919550895691, -0.12414606660604477, 0.15924742817878723, 1.214184284210205, -2.4542412757873535, 0.6598262786865234, 0.9132221937179565, -0.9444805383682251, 0.16608043015003204, -1.1312916278839111, 2.0737547874450684, 0.047390080988407135, 1.5896843671798706, 1.5619487762451172, -1.5796794891357422, -0.5198557376861572, -1.0461878776550293, -0.35194748640060425, 0.25441986322402954, 0.8211721181869507, -0.08500123769044876, 0.27033668756484985, -0.3915262818336487, 0.005323802120983601, -1.0855447053909302, 0.6937570571899414, 1.442460060119629, -0.7213464379310608, -0.6982472538948059, -1.2320847511291504, 0.13813577592372894], [-2.292654275894165, -0.5914709568023682, -0.6675620079040527, 1.1902759075164795, 2.2131540775299072, 0.6863650679588318, -0.028401341289281845, 1.0199650526046753, 0.620107114315033, -0.04734829440712929, 0.437673956155777, 0.8492860794067383, -0.8929786682128906, 0.1813795119524002, 0.38348278403282166, 1.299355387687683, -0.6885471343994141, 0.012074104510247707, -0.2032511979341507, 0.2126316875219345, 0.292612224817276, 2.2919762134552, 1.070661187171936, -0.512802243232727, 0.4909736216068268, 0.233438178896904, 0.1305806040763855, 2.252750873565674, 0.10080242156982422, 1.5772762298583984, -1.6372888088226318, -0.029350969940423965], [-1.8435782194137573, 1.560070276260376, 1.821323037147522, -1.1192705631256104, 1.254629135131836, 1.1824698448181152, -1.268728494644165, 1.121245265007019, 0.06777805835008621, -0.4914257824420929, -0.5869178771972656, -0.4497150778770447, -1.4650732278823853, 0.22763986885547638, -1.0989888906478882, 1.1580613851547241, 0.3466879725456238, -2.203975200653076, -1.2219312191009521, -0.1392490267753601, 0.5429985523223877, 0.18215930461883545, -0.07430271804332733, 1.471907615661621, 0.7102028727531433, 0.39612823724746704, 0.4667230248451233, 1.2311738729476929, 1.0360147953033447, 0.31074047088623047, -0.38139647245407104, 1.3493952751159668], [2.0014560222625732, -0.35597673058509827, 0.2500087022781372, 1.0978931188583374, 0.26316192746162415, 0.8769741654396057, 1.4497336149215698, -0.9710275530815125, -0.7194429039955139, -1.5479289293289185, 0.7993379831314087, 0.07277250289916992, -1.410115122795105, 1.8015131950378418, 0.8635219931602478, -2.1047096252441406, -0.3742450475692749, 0.4425109028816223, 1.278433084487915, 0.11364532262086868, -0.4767404794692993, 1.3615201711654663, -1.3413435220718384, -0.6320266723632812, -1.0771362781524658, -1.1871997117996216, -0.9256467223167419, -0.3262292146682739, 0.3331514298915863, -1.1662805080413818, 0.1659986674785614, 1.214821696281433], [-0.045358117669820786, 0.8160781860351562, 1.1441335678100586, -0.06059788540005684, -0.843548595905304, 0.44179767370224, -0.07805727422237396, -0.9365600943565369, 0.9796317219734192, -0.6458291411399841, 0.298237681388855, 1.2777401208877563, -0.19278115034103394, -0.2403300404548645, 0.17191094160079956, 0.1979835331439972, -0.4485890567302704, -0.8819940686225891, -1.7202330827713013, 0.8288927674293518, 0.3771398365497589, -0.5675224661827087, -1.5846760272979736, 0.7266027331352234, -0.05550965294241905, -0.37690362334251404, 1.2863692045211792, -0.32440370321273804, 0.8037386536598206, -0.4283314645290375, 0.7559956312179565, -0.10578499734401703], [-0.8559205532073975, 0.7356048822402954, 0.35673806071281433, 0.443580687046051, 0.5540941953659058, 0.5987257361412048, -0.2970691919326782, -0.32482752203941345, 0.28271663188934326, -0.7410796880722046, -0.15600022673606873, 0.5949143767356873, -1.708673357963562, -0.5343353152275085, -0.918925940990448, -1.9145435094833374, 0.09923519194126129, -0.0003157206519972533, 0.910858690738678, -1.9800505638122559, -0.744765043258667, -0.10602384060621262, 0.36271142959594727, 1.2127586603164673, -1.6764940023422241, 0.3231693208217621, 0.764237105846405, -0.4286108613014221, -0.37378865480422974, -0.36003732681274414, -0.962248682975769, -0.9237236380577087], [-0.027173271402716637, 1.4779548645019531, -0.7142900228500366, 0.39668208360671997, 0.3413379192352295, -0.8177231550216675, -0.040237899869680405, -0.5896549224853516, 1.1227275133132935, 0.5115037560462952, -1.1053475141525269, 1.485097885131836, -1.110391616821289, -1.123386263847351, 0.21364198625087738, -1.7751914262771606, -2.8052010536193848, 1.0663710832595825, 0.03266904503107071, -0.8895775675773621, 0.8932214379310608, -1.5777091979980469, 0.5753169059753418, 0.6651140451431274, -0.44646966457366943, -0.013410014100372791, -1.5039974451065063, 0.23791103065013885, -0.8139247298240662, -0.9902523159980774, -0.18104860186576843, 0.6237853765487671], [-0.06053720414638519, 0.976330578327179, 0.3031362295150757, 1.097317099571228, 1.3514033555984497, 1.1546472311019897, -1.1498537063598633, 1.5031464099884033, -0.6703864932060242, -1.3068718910217285, -1.3302550315856934, 1.4916423559188843, -0.43923088908195496, -1.8832288980484009, 0.7276493906974792, 1.0827765464782715, -0.5902062058448792, -0.6791067123413086, 0.2676251530647278, -0.6053687334060669, 0.4073125123977661, -0.3103947043418884, 1.4296715259552002, 0.8960241079330444, 0.03256695345044136, -1.1513488292694092, -0.016172975301742554, 0.43362829089164734, -0.02917659282684326, -0.3021993339061737, 0.6563273668289185, -0.8748019933700562], [-0.9790818095207214, -0.06300534307956696, -1.7231292724609375, 0.3778780996799469, -1.2099599838256836, -0.29818981885910034, -0.3367534875869751, -2.14214825630188, -0.7572486996650696, 0.67326819896698, -1.121488094329834, 0.8603390455245972, -0.6268475651741028, 0.2660391330718994, 0.9087768197059631, -0.8268638849258423, -2.703312397003174, 0.2927767038345337, 2.3047351837158203, 1.1135329008102417, -0.519212543964386, 0.27337294816970825, 0.9041796922683716, 0.024122800678014755, -0.06010270118713379, 0.6420649886131287, -0.736758828163147, -0.30742621421813965, -0.8097161054611206, -0.5138042569160461, -0.33882400393486023, 0.5088647603988647], [2.319040536880493, -0.9380783438682556, -0.34076452255249023, 1.7544023990631104, 0.9902322292327881, -2.5816073417663574, -0.5342567563056946, 0.34000855684280396, 0.8766614198684692, -0.8100249171257019, -0.6418392658233643, 0.03178934007883072, 0.9414798021316528, 1.9823832511901855, -0.7900949716567993, -0.9092683792114258, 1.5606797933578491, -0.16616030037403107, 2.096970796585083, -0.865391731262207, -1.6896971464157104, 1.9425374269485474, -1.472352147102356, 0.5374644994735718, 0.3786996603012085, 0.7053480744361877, -1.3114063739776611, -0.7232299447059631, -0.7339903712272644, 1.591999888420105, 0.12781621515750885, -0.8053142428398132], [-1.0171958208084106, -0.06061486527323723, -2.4313409328460693, -0.7124186158180237, 1.1789580583572388, -0.05553477630019188, -0.37557676434516907, -0.34302985668182373, -0.49495649337768555, 0.45631006360054016, -1.4631601572036743, -0.6238868832588196, -1.0017980337142944, -1.3296778202056885, 0.11327420175075531, 0.24932752549648285, -0.11939658224582672, 0.4841581881046295, 1.5435434579849243, -0.4455634653568268, -0.26935938000679016, 0.18500567972660065, -0.6949172019958496, 1.0978351831436157, -1.1317530870437622, 0.5423033237457275, -0.4973224997520447, -0.005225715227425098, -0.9933284521102905, 1.3199889659881592, -0.5533369779586792, 0.0523274801671505], [0.3193652927875519, -0.06488509476184845, 1.001107096672058, -0.24840523302555084, 1.4464389085769653, 1.402843952178955, 0.5024484992027283, -2.5142698287963867, -0.3677133619785309, -0.49103230237960815, 0.8023913502693176, -1.3163272142410278, -0.5819990634918213, 0.45150354504585266, -0.5270552635192871, -0.09253152459859848, 2.6460723876953125, -0.678256094455719, -0.6915042400360107, 0.3875012993812561, -0.2853985130786896, -1.1176772117614746, 1.7240397930145264, 0.33451545238494873, -0.0128516536206007, 0.25432288646698, 0.19153842329978943, 1.5335636138916016, 1.5007635354995728, 0.3053017258644104, -0.09168710559606552, 0.036612894386053085], [1.7089581489562988, 0.1686815619468689, 0.9398368000984192, 0.2369885891675949, -1.2591989040374756, -0.30328860878944397, -0.15129685401916504, 1.6975785493850708, 1.774092197418213, -0.24930916726589203, 1.0925201177597046, -0.4753134250640869, 0.8895339965820312, 0.03303278237581253, 2.8933112621307373, -1.3808772563934326, -0.447554349899292, 0.28942403197288513, -0.07045570015907288, -1.8562887907028198, -0.06058727204799652, -0.389212965965271, 0.7714863419532776, -0.6987364292144775, 0.10815681517124176, -0.9581615328788757, 0.7505576610565186, -0.22583328187465668, 0.24222829937934875, 0.32995322346687317, 0.6837705373764038, -0.619624137878418], [1.1975719928741455, -1.2214884757995605, 0.06510958820581436, 1.214811086654663, 0.4285524785518646, -0.7543465495109558, -0.6364130973815918, 0.1087566539645195, -0.5066852569580078, 0.857844889163971, -1.425465703010559, 0.8730531930923462, 0.6456966400146484, -0.7174435257911682, 1.0984797477722168, 0.9292356967926025, -0.9502996206283569, 0.2888489365577698, 0.7884560227394104, -0.8736156225204468, 0.4287305176258087, -0.37043285369873047, 0.03656260296702385, -0.4663321077823639, -0.5198636651039124, -0.21681752800941467, -0.45863109827041626, -0.4616168141365051, 0.14952793717384338, -0.32500338554382324, 0.28456905484199524, 0.35893386602401733], [-0.6353835463523865, 0.5332883596420288, -1.083778977394104, -1.3216443061828613, -2.831800699234009, 1.7442128658294678, -0.015162062831223011, -2.9686667919158936, -1.718284249305725, 1.9785007238388062, 1.902456521987915, -1.2998857498168945, 0.6163374185562134, 1.0777649879455566, 0.7064472436904907, 0.47876086831092834, -0.7389126420021057, -0.5565173029899597, -0.8434063196182251, -1.5721935033798218, -0.1859365701675415, 1.2831112146377563, -0.4360371232032776, 0.7814118266105652, 0.558421790599823, 1.4628875255584717, 0.024454792961478233, -0.3672807216644287, 0.730937659740448, 2.0055651664733887, -0.9452900290489197, 0.7814456224441528], [1.047976016998291, -0.3659614026546478, 1.5360691547393799, -0.19207243621349335, 0.9483858942985535, -0.7088074684143066, -0.10835633426904678, 0.5655299425125122, 0.8022135496139526, -0.8905640840530396, -1.1386606693267822, 1.0999647378921509, -0.6010248064994812, 0.19067713618278503, -0.32137420773506165, -0.6418540477752686, 0.31656259298324585, -0.018463464453816414, 1.3937705755233765, -0.45742207765579224, -0.19492915272712708, -1.0890055894851685, -0.5899211764335632, 0.21239882707595825, 0.37029770016670227, 1.8163797855377197, -0.2504335939884186, 1.5657923221588135, -0.4585914611816406, 0.4577764570713043, -0.3853009045124054, -0.7525272369384766], [0.0343184769153595, -0.04390588402748108, -0.5728034377098083, -0.4255119264125824, -0.07438542693853378, 0.46109217405319214, -1.6810296773910522, -0.631695568561554, 1.4768279790878296, -1.0186702013015747, -1.5734500885009766, -0.18194638192653656, 0.4192865192890167, -0.8293042182922363, -0.7781350612640381, -0.1157522052526474, 1.123807430267334, 1.5034828186035156, -1.3256407976150513, -0.5939935445785522, 0.009812690317630768, 0.6191354990005493, 0.271956205368042, 2.2107391357421875, 0.8552555441856384, 0.44606003165245056, 0.18112680315971375, -1.1936315298080444, -2.155146837234497, -1.7903400659561157, 1.5051556825637817, -0.32033565640449524], [-0.49698352813720703, 0.5194408297538757, -1.1631797552108765, -0.5384976267814636, 0.059560175985097885, -0.5529543161392212, -0.07472044229507446, -1.111872911453247, -0.6518339514732361, 0.8320042490959167, -0.12039368599653244, -0.874093234539032, -1.2841262817382812, -0.8609664440155029, -1.1380597352981567, -0.8952639698982239, -0.6830893754959106, 0.7073987722396851, 0.06637139618396759, -0.6258844137191772, -2.3293490409851074, -0.080672986805439, 2.223214864730835, -0.4160442650318146, 2.443943738937378, -0.6177162528038025, 0.9467898011207581, -0.9403091073036194, -0.4456101953983307, 0.9191681742668152, 0.2243596613407135, -0.5204241275787354], [0.37791571021080017, -1.2113181352615356, -1.945594072341919, -0.6231790781021118, -1.318184494972229, 0.3367491364479065, -0.5382822751998901, -0.4565867781639099, 0.6647740006446838, -0.6125893592834473, -0.10735540837049484, 0.24329185485839844, -1.7362242937088013, -1.0227210521697998, 0.4117751717567444, 0.02841504104435444, 1.5447684526443481, 1.760480523109436, 0.2605721652507782, -1.5846214294433594, -0.0780075415968895, -0.6772035360336304, -1.111640214920044, 1.5660828351974487, 0.12378997355699539, -1.1442711353302002, 0.29720425605773926, 1.8626923561096191, 1.1346189975738525, -0.2178102731704712, 0.7866746783256531, -0.37935587763786316], [1.1129937171936035, -1.1915611028671265, 0.18098421394824982, 1.8271198272705078, 0.14259658753871918, -0.350055068731308, 0.6682699918746948, -2.0632734298706055, -0.7296035289764404, -0.4043238162994385, 0.6802783012390137, -0.2736852467060089, 0.5344865322113037, -0.6277139782905579, -0.6335799694061279, -0.7972806096076965, 0.8224325776100159, -2.000775098800659, 0.2905333340167999, 0.3419753909111023, -1.0241773128509521, 1.299225091934204, 0.439092218875885, -0.8595224618911743, 2.223482608795166, 1.4639273881912231, -0.3763924241065979, -1.8707029819488525, -0.8487588763237, -0.018315063789486885, -1.5154292583465576, -0.2680908143520355], [2.526948928833008, -0.9447445869445801, -0.5937231183052063, 0.7892521619796753, 0.1672019362449646, -0.887852668762207, -1.6644177436828613, 1.6250605583190918, 0.6890074014663696, 2.0060513019561768, 0.4954502284526825, 0.1760166734457016, -0.025983460247516632, 1.662792444229126, 0.7310762405395508, -1.2198638916015625, 0.42958736419677734, -0.3975721001625061, -0.8068553805351257, 0.22027043998241425, -0.03136096149682999, 1.463888168334961, -0.4747212827205658, 0.12635573744773865, -0.3410622179508209, 0.6024972796440125, -1.0683839321136475, -0.3014465272426605, 0.5877568125724792, -0.021432287991046906, 1.018109917640686, 0.7606783509254456], [1.2530276775360107, -1.7454519271850586, 0.8884132504463196, -1.3592220544815063, 1.0894156694412231, -0.8697578310966492, -1.6230264902114868, -0.270749568939209, -0.44928085803985596, 0.663528323173523, 0.5143634080886841, -1.9904242753982544, 0.6666988730430603, 0.451007604598999, -0.007686280645430088, 0.06197432056069374, 0.026005089282989502, -0.14158616960048676, -0.5473532676696777, 1.4656599760055542, 0.085380919277668, 1.384940266609192, 0.5448814034461975, -0.688001811504364, -1.1782017946243286, -0.2895303964614868, 0.21902567148208618, -1.219030499458313, -0.19935482740402222, -1.2942547798156738, -1.2447806596755981, 0.23400774598121643], [-1.6309555768966675, -0.08832040429115295, 0.27753356099128723, 0.8081737756729126, 2.071747064590454, 0.13040946424007416, 0.738076388835907, 0.0434018075466156, -0.3992083966732025, -0.7608205080032349, 1.8682060241699219, 0.13712309300899506, 0.1335013210773468, 0.06388906389474869, 0.2791336178779602, 1.1715176105499268, -2.5089051723480225, -0.2403590828180313, -0.8249417543411255, -0.4214577376842499, -0.5750836133956909, -0.8502267599105835, -1.4776252508163452, -0.8513633608818054, 0.865890383720398, -0.3457089066505432, -0.5346845388412476, -1.0177202224731445, -0.1059052050113678, -0.4721577763557434, 0.06536421924829483, 0.8155560493469238], [0.3265758752822876, 0.25320181250572205, 0.7995244860649109, 0.0055257114581763744, -1.1739517450332642, -0.637694239616394, -0.2820774018764496, -0.6757495999336243, 0.6845538020133972, -0.4203838109970093, -0.6529328227043152, -0.014331387355923653, -0.7890904545783997, -0.7668626308441162, -1.3738104104995728, -0.2626018226146698, -0.45553719997406006, 1.0087846517562866, 0.3055973947048187, -1.0659090280532837, -0.331407755613327, 0.04968487471342087, -1.051263451576233, -0.08730234950780869, -0.5716754794120789, -0.25088152289390564, 0.1669253259897232, 1.0592280626296997, -0.8856536149978638, -0.18722040951251984, -0.4987136721611023, 0.04421575739979744], [0.3368934094905853, -0.7453097701072693, -1.2803668975830078, 0.8327067494392395, 0.01794395036995411, 0.38148316740989685, -1.2880052328109741, -0.7842360138893127, 1.0501744747161865, 0.0400753952562809, -0.7025089263916016, -0.7476526498794556, -0.2649180293083191, -0.9114682674407959, 0.20243065059185028, 2.3904330730438232, 0.7828192114830017, -0.16382230818271637, -1.6295243501663208, -1.3338236808776855, 1.098690390586853, 0.06534387171268463, -0.6226271390914917, 1.413629174232483, 0.1043602004647255, 1.0381343364715576, 0.6478719115257263, -0.4077005088329315, -1.2708338499069214, -0.7020826935768127, -1.3039780855178833, -0.5461114048957825], [2.1240413188934326, 0.2077215164899826, 0.7268062233924866, -1.2007381916046143, 0.20011119544506073, -0.32129475474357605, -0.5592502951622009, -0.6300821900367737, -1.1149005889892578, -0.6388518214225769, 2.239220380783081, 0.5266312956809998, -0.23038724064826965, -4.530286241788417e-05, -0.050346337258815765, -0.33105772733688354, -0.49246102571487427, 1.0431417226791382, 0.7538149356842041, -0.23584972321987152, 0.1411580592393875, -1.8872261047363281, -1.6098533868789673, 0.6844892501831055, -0.440327525138855, 0.8740701675415039, -2.350593090057373, 0.5340750813484192, -1.0376285314559937, 0.9419577717781067, 0.20876988768577576, -0.35813137888908386], [1.471565842628479, 0.36497294902801514, 0.6453447937965393, 1.8565783500671387, -0.48154154419898987, 0.37809062004089355, -0.45485949516296387, -0.08078327775001526, -0.7242640256881714, 0.5414860248565674, -0.9011055827140808, 0.22890602052211761, -0.25575754046440125, -0.2994254231452942, 0.20953091979026794, -0.2868611514568329, -0.47570887207984924, 0.13774988055229187, 0.3074672222137451, -0.553034782409668, 0.4754883944988251, 1.9324984550476074, -0.8758987784385681, -0.4976099133491516, 0.1889951229095459, 1.9336340427398682, 0.20773930847644806, 0.09334348887205124, -0.5895768404006958, 1.2634929418563843, -0.9447687268257141, -0.25417861342430115], [0.9408750534057617, 0.7934116721153259, 0.9579066634178162, 0.7649646401405334, -1.053736686706543, 0.26368942856788635, -0.28086039423942566, -0.39862683415412903, -0.6161508560180664, 1.3810935020446777, -0.09235157817602158, 0.6274193525314331, 0.1747148483991623, -1.1569466590881348, 1.6409913301467896, 0.3994278013706207, -0.09672906994819641, 0.40496397018432617, -1.5792206525802612, -1.0852439403533936, 0.1556655764579773, -0.4154050648212433, 0.8917772173881531, 0.8591153621673584, -0.28046733140945435, -0.47996944189071655, -2.0853071212768555, 0.36724692583084106, 1.2155972719192505, -0.12890107929706573, -0.15195004642009735, -0.8485981822013855], [0.12717574834823608, 0.2789997160434723, -1.55374014377594, 1.0114184617996216, -1.1825181245803833, 0.7415741682052612, -0.6348181366920471, 0.0018064449541270733, 1.571774959564209, -0.07006840407848358, 0.3269606828689575, -1.164521336555481, 0.9049395322799683, -2.9641976356506348, 0.14392021298408508, -0.11298723518848419, -0.5346825122833252, 0.33260637521743774, -1.7073335647583008, 0.5944084525108337, 2.022709846496582, 0.6407431364059448, -0.8812025785446167, -0.9723910093307495, -0.5239782333374023, 0.5431154370307922, -0.6771889925003052, -0.010708222165703773, 1.2041844129562378, 0.43173646926879883, 1.5822337865829468, -1.3720645904541016], [0.3512631058692932, 1.5894962549209595, -0.6044730544090271, 0.552078127861023, 0.45853206515312195, -1.1420369148254395, -0.6002029776573181, 1.723434329032898, -0.07707226276397705, 0.708266019821167, 0.5200387835502625, -0.7581027746200562, 1.7374074459075928, 0.20377811789512634, 1.2559841871261597, 0.23746950924396515, 0.01361570693552494, 0.3956787884235382, -0.35677504539489746, -0.4070720374584198, 0.13595838844776154, 0.1754731386899948, 0.5726758241653442, 0.344054639339447, -1.2820615768432617, 2.0134711265563965, -0.08303839713335037, -0.6376135349273682, -1.0571702718734741, -1.3587172031402588, 1.4978052377700806, -0.6046946048736572], [-0.2401008903980255, -0.5845263600349426, 0.4036729335784912, -0.46713918447494507, 0.9264323115348816, 0.10366183519363403, -0.3907225430011749, 0.08783149719238281, 1.2705925703048706, -0.49877747893333435, -0.3998459577560425, -0.4702531099319458, 0.16579388082027435, -0.4706784784793854, 0.26876771450042725, 0.29488131403923035, 0.8493618965148926, -0.9589297771453857, 0.30788490176200867, -0.23885908722877502, -0.6821021437644958, 0.3183493912220001, -0.997602105140686, -0.649141788482666, 2.041903018951416, -1.214779257774353, 0.23215724527835846, 1.869574785232544, -0.2466447949409485, -0.08465415239334106, -1.0884981155395508, 0.6835134029388428], [0.04915649816393852, 0.1670406311750412, 0.7887337803840637, 1.5396678447723389, 0.8324601650238037, -1.3921313285827637, -0.4554942846298218, -0.053858235478401184, 1.5433622598648071, -0.4824502468109131, -1.773803472518921, 0.8754212260246277, 0.2595598101615906, -0.3522268235683441, 0.04853006824851036, -0.7237195372581482, -0.508891224861145, -0.20599862933158875, 1.5583617687225342, 1.2572835683822632, 0.9554580450057983, -0.9964706897735596, -1.5375231504440308, 1.1380704641342163, -1.5463790893554688, -0.0057617914862930775, -1.1118199825286865, -1.0150142908096313, -1.3023396730422974, 0.6463121771812439, -0.7043594121932983, 0.18931391835212708], [0.2693203389644623, -0.5605635643005371, -0.11498241126537323, -0.40769192576408386, 0.28010791540145874, 2.3173229694366455, 1.1948906183242798, -1.5004178285598755, 0.050050679594278336, -0.34811121225357056, -2.10530424118042, -0.10322800278663635, -0.5048542022705078, 1.5761648416519165, -1.7135390043258667, 1.067238450050354, -0.24515195190906525, 0.604157567024231, -1.0971953868865967, 0.21689169108867645, 0.19923387467861176, 0.23250815272331238, -1.5155316591262817, -0.3483315706253052, -1.8011723756790161, 0.5396257638931274, 1.4703625440597534, 0.42176946997642517, 0.7163140177726746, 0.7068132758140564, 0.824352502822876, 0.01599009707570076], [-0.549567461013794, 0.871793270111084, -0.15649749338626862, -0.7479071617126465, -0.34458595514297485, -0.24624790251255035, 0.4906599819660187, 0.9056861400604248, -1.0520682334899902, -0.14419886469841003, -1.5235234498977661, -1.755375623703003, -0.7849141955375671, 0.6222015619277954, -1.7630003690719604, -1.492795705795288, -0.7000830769538879, 1.1242820024490356, 0.22826044261455536, -2.5794808864593506, -0.1869298666715622, -1.548098087310791, -1.6992300748825073, 0.016152244061231613, -0.000565417343750596, -1.367581844329834, 1.866479754447937, 1.0197120904922485, 1.2627570629119873, -0.8932747840881348, 0.3008306324481964, 1.3278839588165283], [-1.2119921445846558, 1.539123773574829, 0.7530760169029236, -2.0934178829193115, -1.6745256185531616, 0.760580837726593, 0.4186745882034302, 1.6406211853027344, -0.4948405623435974, 0.7352362871170044, -0.530296802520752, -1.35343599319458, 0.4636293649673462, 1.8884657621383667, -1.9488773345947266, -0.5781363248825073, 2.3318936824798584, 1.0582278966903687, 2.3162662982940674, -0.80328768491745, -0.2315383404493332, -1.6903481483459473, 0.1369585394859314, 0.3354032039642334, 0.6908445954322815, -1.4685344696044922, 0.5733089447021484, 0.1807049959897995, -0.5514854788780212, 0.18357326090335846, 0.813408374786377, 0.10031294077634811], [-0.18235401809215546, -0.3756595849990845, -1.1656057834625244, -0.5930789113044739, -0.2637903392314911, 0.8852347135543823, -1.5247783660888672, 1.3868308067321777, -0.9450622200965881, -2.843135118484497, 2.3169848918914795, -2.13641095161438, 0.7714181542396545, 0.2766216993331909, 0.8289578557014465, -0.5258476138114929, 1.534627914428711, 0.4314627945423126, -0.20207634568214417, -0.3217719793319702, 0.28778794407844543, -1.3751273155212402, 0.45329341292381287, -0.004731177818030119, 1.5560576915740967, 0.3201525807380676, 0.5193175077438354, 2.124467611312866, 0.5393806099891663, 0.32553035020828247, -0.8540568351745605, -1.2466434240341187], [-0.9947860836982727, -0.8324660658836365, 1.4571075439453125, -0.4956040382385254, -0.16143262386322021, -1.8051884174346924, -0.13030315935611725, 2.058502674102783, 3.0869181156158447, 1.1422467231750488, 1.7491189241409302, -1.1082762479782104, 0.28928396105766296, 0.38090062141418457, -1.4830371141433716, -0.6874107718467712, -1.7631860971450806, -0.4687187671661377, -0.11043455451726913, -2.039931297302246, -0.31740036606788635, -0.7465214133262634, 0.47104984521865845, -1.0525343418121338, -0.25713250041007996, 0.05486020818352699, 0.2158956378698349, -0.08384730666875839, 0.8460491299629211, -1.023758053779602, 0.3836386203765869, 0.5088040232658386], [-0.45197075605392456, -0.3233831226825714, -0.7248632907867432, 0.45599567890167236, 0.3359530568122864, -1.2520318031311035, -0.712676465511322, -0.44355273246765137, 0.4832492470741272, 1.8053693771362305, -0.2288283258676529, -0.0187374334782362, -0.7771744728088379, -0.8767930865287781, -1.4017337560653687, -0.3227483630180359, -0.22455590963363647, -0.0801214948296547, 0.17695322632789612, 0.32450106739997864, -0.14659292995929718, 0.5076157450675964, 2.000554084777832, 1.0358996391296387, -0.3939155638217926, 0.9284428954124451, 0.3935225009918213, -0.6797478199005127, 1.3108160495758057, 1.385948657989502, -0.712882399559021, 0.8082733154296875], [-0.4932190477848053, -0.2808852791786194, -0.3843676745891571, 1.6570243835449219, -1.2767345905303955, 0.24824662506580353, -0.6501756906509399, -1.283963918685913, -2.546171188354492, -0.534694492816925, -0.11811991035938263, 0.5311814546585083, -0.823302686214447, -0.38879454135894775, 0.0982590988278389, -0.47334587574005127, 0.9207426309585571, 1.0938345193862915, 1.2080750465393066, 0.3449821472167969, 0.9581735134124756, 0.2258557677268982, 1.3788690567016602, 0.0470859594643116, -2.5659890174865723, 1.094581961631775, -0.4348050355911255, -0.0847194716334343, -0.18547356128692627, -0.6214925646781921, -0.5050517320632935, 0.16479718685150146], [-0.5123116970062256, 1.7011215686798096, 0.7819312810897827, 0.030680490657687187, -0.07323924452066422, -0.7610680460929871, 0.9703941345214844, 1.0864063501358032, -1.3281506299972534, -0.7382791042327881, 0.0355612188577652, 0.23882746696472168, -0.021278372034430504, 0.20969949662685394, -0.6092653870582581, 0.16617296636104584, -0.0778898373246193, 0.4912220239639282, -0.10506007075309753, -0.04962875694036484, -0.05875704437494278, -1.2003178596496582, -0.5529096126556396, -0.1655372977256775, 0.9207357168197632, -1.587710976600647, -1.0238852500915527, -0.36934494972229004, 0.028618058189749718, -0.055214304476976395, -0.009671924635767937, -1.5147596597671509], [-2.262936592102051, -0.9983112812042236, 1.2034529447555542, 0.9378233551979065, 1.261868953704834, -0.7009766697883606, -1.1555055379867554, 0.24544395506381989, -0.14091938734054565, -0.7429848313331604, -2.3044486045837402, -1.6633614301681519, -0.9754585027694702, 1.309618353843689, -1.5779955387115479, 0.28211089968681335, 0.01585138402879238, -1.3375555276870728, 1.2816022634506226, 0.17315325140953064, 2.064183235168457, 0.5013779401779175, 0.6254610419273376, 1.4491761922836304, 0.7299931645393372, -0.4264858663082123, 2.01669979095459, -0.0057058185338974, -0.4993649423122406, -0.7011497020721436, -0.5575145483016968, -0.8191906213760376], [-0.047948356717824936, -0.7691776752471924, 1.4556570053100586, 0.8778539896011353, -1.1352949142456055, -0.47898784279823303, -0.6467173099517822, 1.483057975769043, 1.557570457458496, -0.7713934183120728, -0.11601728200912476, 0.38936328887939453, -0.0904710590839386, -0.7553232908248901, 0.10115877538919449, 2.0728919506073, -1.8053559064865112, -0.31797850131988525, -1.8175175189971924, 1.746942400932312, 0.6529265642166138, -0.052349548786878586, 0.500397264957428, -1.3287949562072754, 0.7763972878456116, -0.8749026656150818, 1.0590225458145142, -0.42207083106040955, -0.744674563407898, 0.23771165311336517, 0.4509567618370056, 0.76930832862854], [-0.43839263916015625, 1.7174873352050781, -1.0143877267837524, 0.2189456969499588, 0.8300456404685974, 0.4596760869026184, 0.6746571063995361, 1.3557469844818115, 0.1612565815448761, -0.8910676836967468, -0.2810567021369934, 0.06928519159555435, 0.08775327354669571, 0.4941251575946808, 0.8712652325630188, -0.5677798390388489, 0.17413000762462616, -0.9976649880409241, -0.6198732852935791, -0.9300905466079712, 0.30588799715042114, -0.023274563252925873, -0.22197070717811584, 0.14622676372528076, -0.5131596326828003, -1.9718619585037231, -0.45633208751678467, 1.5241836309432983, -0.6294962167739868, 0.26288920640945435, 0.21960900723934174, -1.4143840074539185], [-2.0204784870147705, 0.7943611741065979, 0.35847601294517517, -0.7505093216896057, 1.6228519678115845, 1.0701863765716553, 2.6812028884887695, 1.7376750707626343, 0.5428348183631897, -1.5022798776626587, -0.2688724398612976, -0.2911193370819092, 0.8670584559440613, -0.19927693903446198, -0.499999463558197, 1.3885719776153564, 0.23667412996292114, 0.17965635657310486, 0.30399322509765625, -0.0340639166533947, 0.8527908325195312, 0.3167192041873932, 1.246262788772583, -0.1202038899064064, 0.601703405380249, -0.44970571994781494, -0.8484731316566467, 1.2813475131988525, 0.8054792284965515, 0.6345633268356323, 1.0589501857757568, -0.21216921508312225], [0.67251056432724, -1.6512575149536133, -0.22577285766601562, 0.9200028777122498, 0.6248562335968018, 0.6346920728683472, -0.07235009968280792, 0.1294703185558319, 1.1438844203948975, -2.184676170349121, 1.2498893737792969, 0.5581673383712769, -0.5299237370491028, 0.7971905469894409, 1.3468255996704102, -1.4731279611587524, -0.8555464744567871, -0.9815869927406311, -0.15688295662403107, 0.17614693939685822, -0.42367786169052124, -0.3761601150035858, 1.6842707395553589, 0.4752444326877594, -0.6166600584983826, -0.6494665145874023, -0.7004124522209167, 1.0626434087753296, 1.0004396438598633, -0.350593626499176, -0.24199455976486206, 1.9444270133972168], [0.3781266510486603, 0.8196561336517334, -1.1758548021316528, 0.11320750415325165, 0.849242627620697, -0.1192048192024231, -0.28566595911979675, -1.368755578994751, 0.9718296527862549, 0.4762253761291504, -0.635583221912384, -0.2871948778629303, -0.4435177445411682, 1.7351757287979126, -0.45626598596572876, -0.0061797816306352615, 0.6464920043945312, 0.27599865198135376, 1.0160751342773438, -0.2499682754278183, 0.1392829269170761, -0.5647467970848083, 0.9615628719329834, 0.325023353099823, -0.4205816686153412, 1.5408542156219482, 1.5226786136627197, -0.0162968747317791, 0.982323408126831, -0.2903147041797638, -2.244149684906006, -0.5716681480407715], [0.5392432808876038, -0.1780741661787033, 0.5655303001403809, -0.9069817662239075, 0.6553664803504944, 0.9671200513839722, 2.0793867111206055, 0.3870700001716614, 0.2033098042011261, -0.34640151262283325, 0.29827556014060974, 1.096638560295105, -0.292109876871109, -0.8966102004051208, -1.0938729047775269, -0.8926969766616821, -0.6495668888092041, 0.6852874159812927, -1.454823613166809, 0.06953178346157074, 1.9200607538223267, 1.5130125284194946, 0.34107285737991333, -1.0400059223175049, 0.780947208404541, 0.32878822088241577, -0.8839582800865173, -0.14280463755130768, -0.08692572265863419, -0.49263688921928406, 0.4346473813056946, 0.9260168671607971], [0.31573888659477234, 1.0792864561080933, -0.7444131374359131, -0.19309884309768677, 1.973323106765747, -0.6774653196334839, -0.34074077010154724, -0.25844821333885193, -0.7190929055213928, -0.3539658188819885, -1.7342816591262817, -0.6250594854354858, 1.1493804454803467, -1.4296764135360718, 2.346926212310791, 0.07028224319219589, -0.2780722379684448, 0.2709212005138397, -0.3013729155063629, -0.045693445950746536, -1.8880057334899902, -0.3350140154361725, -1.0145632028579712, 0.31069618463516235, 1.4276677370071411, -1.342238187789917, -0.10304339230060577, -0.3527536690235138, 1.6721361875534058, -0.17646118998527527, 1.048997402191162, 0.448932021856308], [-0.9480135440826416, 0.8775873184204102, 0.060154713690280914, 1.0798428058624268, 0.012698347680270672, -1.0832409858703613, -1.3398925065994263, -1.420586109161377, -0.5879766941070557, 0.542784571647644, -1.4330315589904785, 0.7808912992477417, -0.8185814023017883, -0.5776499509811401, 0.35737118124961853, -1.0404938459396362, 0.13946117460727692, 0.45652711391448975, -1.3677737712860107, -1.567115068435669, -0.6079737544059753, 0.4215453565120697, 1.6371866464614868, -1.8793920278549194, -1.1201142072677612, 0.14692911505699158, 1.3532296419143677, 0.9837518930435181, 1.349026083946228, -1.2963895797729492, 0.36417636275291443, -1.1155840158462524]], [[-0.9493884444236755, 0.1730397492647171, 0.8695212006568909, -1.132668375968933, 0.39190343022346497, -1.9200646877288818, -0.8995542526245117, 0.03661082312464714, -1.1438771486282349, 1.2386558055877686, -0.6263585090637207, 0.8815894722938538, 0.1947745680809021, -0.5698726773262024, -1.2192054986953735, -1.038014531135559, -0.49688881635665894, 0.25453445315361023, -2.2590079307556152, 1.696484923362732, -0.7375369668006897, 0.05459548532962799, 1.0588059425354004, -0.22522585093975067, 1.6410285234451294, -0.02537977695465088, 0.2620353698730469, 1.4107496738433838, 1.37062668800354, -1.1150963306427002, -2.2965142726898193, -0.42791590094566345], [0.4735459089279175, -0.3143670856952667, -0.08757232129573822, -0.16869334876537323, -2.803006410598755, 0.3565008342266083, 0.5843415856361389, -0.6719173192977905, 0.7864843606948853, 0.13672135770320892, -0.029532786458730698, 0.8134918808937073, 0.4599771201610565, 2.323763132095337, 1.697171926498413, -2.004227638244629, 0.8620202541351318, 0.3725229799747467, 1.1257997751235962, 0.19577181339263916, 0.2877010405063629, 2.268728017807007, -1.9043906927108765, 0.2105107307434082, -2.331575393676758, -0.7389007806777954, 0.25143030285835266, -1.3826040029525757, -0.6098973155021667, 0.3812611997127533, -0.5482743382453918, 0.5325624942779541], [-1.454511046409607, 0.855514645576477, 1.6616638898849487, -1.5287842750549316, -0.7100025415420532, -1.3500028848648071, -0.9600785970687866, 0.6040751338005066, 0.4920158088207245, 0.09101240336894989, -0.7426339387893677, 0.18630129098892212, -0.934120237827301, -0.34186404943466187, 1.2729296684265137, 1.334719181060791, 0.39097198843955994, 0.6968147158622742, -0.642700731754303, 1.1215555667877197, -1.4148350954055786, -1.2276495695114136, 0.2471771538257599, -0.14063189923763275, -1.1697801351547241, 0.41338324546813965, 1.6057974100112915, 0.5498571395874023, -1.1710615158081055, -0.8250589966773987, 0.12497033178806305, 0.4296378195285797], [-0.6741430163383484, 0.5696811676025391, 1.1828041076660156, -0.6456137895584106, 0.2933771014213562, -0.6564556956291199, -1.3240875005722046, 1.398324966430664, 0.35729074478149414, -0.4913039207458496, 0.018540041521191597, 1.0166815519332886, 1.2698620557785034, -0.5370134711265564, 2.045614719390869, 2.00443172454834, 0.7099073529243469, -0.06497611105442047, 0.717626690864563, 1.5024404525756836, 0.9399152398109436, 0.837429940700531, -0.8107317686080933, -0.2701147198677063, 0.8279613852500916, -1.0400546789169312, 2.0850653648376465, -0.8956151604652405, -0.7799778580665588, -1.7004034519195557, 0.43620720505714417, -1.210984468460083], [-0.06203426420688629, 0.36189064383506775, 0.5206429958343506, 1.4648196697235107, -0.3992008864879608, -0.9151882529258728, -0.5152549743652344, 0.5625333189964294, -0.7793399095535278, 0.0021276972256600857, -1.4129692316055298, 1.0776244401931763, 1.0591837167739868, 1.6376389265060425, -0.012877373956143856, -0.32339611649513245, 0.7344698309898376, -0.1617477387189865, -2.1213622093200684, -1.1138108968734741, 0.24257652461528778, -1.5220708847045898, -0.5778576731681824, -0.739679217338562, -0.7265855073928833, -0.1788424402475357, 0.7594922780990601, 0.878642737865448, 1.2319167852401733, -2.013575315475464, 0.3453732132911682, -0.9919412732124329], [-0.6408587694168091, 0.5937864184379578, -1.4971489906311035, -1.3612600564956665, -0.2974109947681427, 1.2142184972763062, -1.8988752365112305, -0.049398500472307205, 2.0935232639312744, 0.031878236681222916, -0.7244561314582825, 0.060237474739551544, -1.3878556489944458, -0.9687252640724182, 1.7092514038085938, 0.4214366674423218, -0.3555627763271332, 0.5707060694694519, 1.2227790355682373, 2.018740653991699, -0.4824448525905609, 1.886568307876587, 1.0645456314086914, -1.364768147468567, 1.450379490852356, 1.3772330284118652, 0.8914166688919067, -0.20622435212135315, -0.4716581702232361, -0.13653698563575745, -2.2466704845428467, -0.8095410466194153], [-0.107632577419281, 0.9682790040969849, -0.8289300203323364, 0.957891047000885, 0.7321668267250061, 0.34848976135253906, -0.014515810646116734, 0.1541294902563095, -0.7656482458114624, 0.06458377093076706, 0.6610586643218994, 0.5510320663452148, -0.5291332006454468, 0.2589453160762787, -0.33366110920906067, -0.5847913026809692, -0.8268478512763977, 0.46414685249328613, 0.029468916356563568, 2.6887452602386475, -0.4162750542163849, 1.4253137111663818, 0.5470304489135742, 0.372873455286026, 1.2370368242263794, -0.34037289023399353, 0.7280240654945374, -0.01623542234301567, -1.569854736328125, -0.10631423443555832, -0.6796159744262695, -2.139225959777832], [1.7299848794937134, 0.783105194568634, 0.2420063018798828, -0.8158010244369507, 0.2595873177051544, -0.6529978513717651, -0.6107621788978577, -0.07815589755773544, 0.5795190334320068, -1.9373526573181152, -0.4317098557949066, 0.11826259642839432, 0.046562448143959045, -0.8178259134292603, -0.26098185777664185, 1.4346028566360474, 0.11707066744565964, -0.6300582885742188, 1.4036996364593506, -1.1412348747253418, 0.1229264959692955, -1.7830184698104858, -0.33990752696990967, -1.06621253490448, -0.04559137672185898, 0.4590088129043579, 0.9111959338188171, -1.053218960762024, 0.10705403238534927, 0.7189962863922119, 1.1679787635803223, -1.1361042261123657], [-0.6450481414794922, -0.5893622040748596, 0.027112068608403206, 0.7447285652160645, -0.23444780707359314, -0.2898879051208496, 0.9539908170700073, 0.8885358572006226, 0.2486903965473175, 0.612762451171875, 0.14088162779808044, 0.255428671836853, -0.33502617478370667, 1.442853569984436, 0.49068424105644226, -1.9677367210388184, 0.2574741840362549, 0.5836874842643738, 0.8047139048576355, -0.40133142471313477, -1.4124919176101685, 0.5280855894088745, -0.17802900075912476, 0.23033730685710907, 0.8475977182388306, -0.4544120728969574, -0.8957938551902771, -0.36264798045158386, -0.41805216670036316, -0.07680502533912659, -0.63633793592453, -3.076401948928833], [-0.7901473641395569, -2.0732967853546143, -0.13995645940303802, 0.011836310848593712, 0.31306910514831543, 0.6326484084129333, -0.6254123449325562, 1.544181227684021, -0.5857232809066772, -1.2348041534423828, -0.7658287286758423, 1.170054316520691, 0.5884557962417603, 1.6232248544692993, -1.217043161392212, -0.51375812292099, -1.1649914979934692, 1.0746641159057617, 0.5282885432243347, 0.8223034739494324, 0.9733681678771973, 0.1855328530073166, -0.5043895840644836, 0.496277779340744, 0.9490037560462952, -0.8571286201477051, 0.36167845129966736, 1.0631012916564941, 0.18671688437461853, 0.2656647861003876, 0.5130903124809265, 1.55988347530365], [-0.25282561779022217, 1.1382821798324585, -0.5470945239067078, 0.0003650039725471288, -0.31840580701828003, -0.9868404269218445, -0.5881132483482361, 1.4424241781234741, -0.11739141494035721, 2.497344970703125, -1.2775208950042725, 0.9573671221733093, -0.5455979704856873, 0.5013530254364014, 0.08798572421073914, -0.9492742419242859, -0.27329063415527344, 1.2515922784805298, -0.35589319467544556, -0.8265780806541443, -0.030072292312979698, -0.8819093108177185, 0.6868001222610474, -0.30529358983039856, -1.0622284412384033, 0.11116084456443787, 0.5194235444068909, -0.9524093270301819, -0.3440937399864197, 1.617125391960144, 0.8976169228553772, 0.30218568444252014], [0.13295471668243408, 0.8550606966018677, 0.29547247290611267, 0.6191001534461975, -0.40759411454200745, -1.0255537033081055, -0.45932668447494507, 0.5644075870513916, -1.0525435209274292, 2.246699333190918, -1.3234546184539795, 0.1785895824432373, 3.176680326461792, -1.010903239250183, 1.4885684251785278, 2.6881520748138428, -0.6959863901138306, 0.26495787501335144, -0.20786535739898682, -0.17473821341991425, 0.45464056730270386, -1.2495044469833374, 0.763652503490448, -0.16596965491771698, -1.4772019386291504, 0.20584717392921448, -0.33262109756469727, 0.1503710001707077, -0.5577635765075684, -0.20109672844409943, 1.2249592542648315, -1.706153392791748], [-0.1830536127090454, -0.05903812125325203, -1.6812407970428467, 0.6466783881187439, -0.7817293405532837, 0.6185795664787292, 0.38577139377593994, -0.11108551919460297, 0.21707740426063538, -0.5562609434127808, 0.10646314918994904, -0.5477095246315002, -0.2627837359905243, -1.1927249431610107, 0.0689256563782692, 0.23252511024475098, -0.3954048454761505, 0.2786998152732849, -1.4825562238693237, 0.7185631990432739, -0.06907541304826736, -2.0086934566497803, -1.1276220083236694, -1.5583089590072632, -0.2157522588968277, 0.38797828555107117, -0.7418487071990967, 0.4836966395378113, -0.4416430592536926, -0.41367635130882263, 0.3063286542892456, 0.5295501947402954], [0.9719704985618591, 0.6714870929718018, -0.041891515254974365, -0.49879544973373413, 1.2129713296890259, 0.16064083576202393, 0.776533842086792, 0.8082419037818909, 1.217560887336731, -0.8504176735877991, -0.18945661187171936, 0.30767351388931274, 0.06415033340454102, -1.0898298025131226, 0.2304409295320511, -0.9474180936813354, 2.577761173248291, 0.5912365317344666, -0.9221423864364624, -1.6797484159469604, -0.8462814092636108, 0.8597274422645569, -0.6310579180717468, -0.688142716884613, 0.6046081781387329, 0.08672945201396942, -0.042852215468883514, -0.0901670828461647, 1.0649771690368652, 0.05625796690583229, 0.43866121768951416, -1.126692533493042], [0.26605767011642456, -0.07991892844438553, -1.104347586631775, 0.5746904015541077, 2.144401788711548, 1.0460634231567383, -0.533495306968689, 1.044413447380066, 0.6988413333892822, -0.44233307242393494, -1.4164650440216064, -0.056694939732551575, -0.7889028191566467, -0.5541242361068726, 1.686920404434204, 0.03269856050610542, 0.5822827219963074, -2.0625927448272705, 0.0816841870546341, -0.9903674125671387, 0.2198440432548523, -1.5559022426605225, 0.04758228361606598, -0.2822871208190918, -0.5454151630401611, 0.9281792640686035, -0.9274110794067383, 0.010719353333115578, -0.8221731185913086, 0.16055499017238617, 0.1916113942861557, 0.6798489689826965], [0.10814204812049866, -1.9582892656326294, 0.040296513587236404, -0.020712750032544136, 1.2885448932647705, -0.20868319272994995, 0.34853604435920715, 1.3473610877990723, -0.7053698301315308, -0.8635324239730835, 1.27444326877594, -0.7969356179237366, 0.46374067664146423, 2.060324192047119, -0.34770986437797546, -0.11487710475921631, 1.7811782360076904, 0.019164983183145523, -0.5119941830635071, 0.34974193572998047, -0.32710614800453186, -0.810418963432312, -1.8020209074020386, 0.7485795617103577, -0.9188054203987122, -1.1975553035736084, 2.8105316162109375, 0.04822574555873871, 0.347800076007843, -0.4113589823246002, 0.5346338152885437, -0.3949110507965088], [-1.279829740524292, 2.0621132850646973, 0.3530649244785309, -1.0342891216278076, -0.7134289145469666, 0.16002264618873596, 0.1194867268204689, -1.1003148555755615, 0.77567458152771, 0.6558230519294739, -0.026482978835701942, -0.015177462249994278, 0.11496592313051224, 1.0643068552017212, 0.32989978790283203, -1.1171327829360962, 0.2661726772785187, -0.3068391978740692, -0.21582995355129242, -1.0804057121276855, -0.29368868470191956, -0.3723287284374237, 1.4313730001449585, 0.1416633129119873, 0.09653864055871964, -1.045432448387146, -0.5068370699882507, 1.9406871795654297, -1.0545657873153687, -1.5764254331588745, 0.9894017577171326, 0.4786556363105774], [-0.08403083682060242, -0.2581324279308319, 0.2883390784263611, -1.0353950262069702, -0.1938818246126175, 0.2692219316959381, -0.6693233251571655, 1.1760340929031372, 0.38291314244270325, -0.50710129737854, 0.41004452109336853, 0.2112744301557541, 2.0829057693481445, 0.1919592171907425, 0.9966380596160889, 0.4156109690666199, -2.1587395668029785, 0.5380740165710449, 0.05948038399219513, 1.1980432271957397, -0.43254753947257996, 2.0946009159088135, 0.14495746791362762, -0.4008631408214569, -0.648857593536377, 0.7203611135482788, 0.5030878782272339, 0.9019452333450317, 0.6708611249923706, -0.20354124903678894, -0.04314344376325607, 0.9352304935455322], [-0.8980045318603516, -0.29389429092407227, -0.9458957314491272, -0.2812750041484833, -0.05380098521709442, -0.1257360577583313, 0.29841306805610657, -1.0762734413146973, -0.21722178161144257, -0.6979267597198486, 2.0047249794006348, -0.8024545311927795, -0.033889006823301315, -1.6080074310302734, -0.5759910941123962, 0.695206880569458, 0.8344599604606628, -0.31824004650115967, 0.9719584584236145, -2.2591636180877686, -0.5982708930969238, -1.1668713092803955, -0.37163692712783813, 0.3970494270324707, -0.047689322382211685, 1.4501417875289917, 0.6343445777893066, -0.518130898475647, -1.0646640062332153, 1.0170670747756958, -1.503151774406433, -2.4995975494384766], [-0.1795744001865387, 0.07860907912254333, -1.2717617750167847, 1.9175318479537964, -0.817370593547821, 0.6050367951393127, 0.20956794917583466, 0.6704418659210205, 1.0388849973678589, 0.5419399738311768, -0.9571018815040588, -1.173107624053955, 0.6976274847984314, -1.3818951845169067, 1.0722256898880005, -1.138304591178894, 0.41701173782348633, 0.3296623229980469, -0.4159771800041199, 0.045006874948740005, -0.7278556823730469, -0.05757439136505127, -0.9903121590614319, 2.2225375175476074, 1.0954424142837524, -1.0434606075286865, -0.25654131174087524, -1.3020853996276855, -1.6044905185699463, -0.735165536403656, 1.6679683923721313, 0.8724588751792908], [-1.4040091037750244, -0.6270011067390442, 0.6903915405273438, -1.1522666215896606, 0.8660097122192383, -0.16014395654201508, 0.8029969930648804, 0.5711198449134827, -0.9341187477111816, -1.134278655052185, 1.2551956176757812, -0.9841153025627136, -0.26298457384109497, -1.912994146347046, 1.000095248222351, 0.8470329642295837, -2.7458527088165283, -0.4503967761993408, 0.6144101619720459, 0.7459110617637634, 0.8604170083999634, -1.2345811128616333, 1.3461297750473022, 1.0115894079208374, -0.6379734873771667, -0.540215253829956, -0.5164032578468323, 0.723366379737854, -1.273237705230713, 1.3661682605743408, 1.5489782094955444, 0.4992368817329407], [1.6010422706604004, -0.010862435214221478, -0.39673569798469543, -0.7106658220291138, 1.1145052909851074, -1.3481693267822266, -0.11867334693670273, 0.6248895525932312, 0.6450303792953491, 0.24649125337600708, -1.1014111042022705, 0.09009543061256409, -0.2520733177661896, -0.8466487526893616, -0.9288331866264343, -0.254209041595459, -1.131545901298523, -1.8134719133377075, 0.9203183054924011, 0.2850635349750519, 1.1484792232513428, 0.7077824473381042, 0.28260910511016846, -0.07851089537143707, -0.3138943314552307, -0.14688196778297424, 0.029797418043017387, 0.1377888172864914, -1.0229625701904297, -0.3843095898628235, -0.791394054889679, 0.1957089751958847], [-1.0674008131027222, -0.9642996788024902, -0.7779941558837891, -2.2828502655029297, -0.7201079726219177, -0.17181892693042755, 0.8239410519599915, -1.1251428127288818, 0.8587992191314697, -0.5011835098266602, -0.6356008052825928, -0.7582680583000183, -0.25717589259147644, -0.11377930641174316, 0.865496039390564, -0.4963753819465637, -0.17386309802532196, 1.3536608219146729, -0.37358301877975464, -0.9640522003173828, 0.2206713855266571, 2.220691680908203, 0.5030059218406677, -0.298908531665802, -0.879009485244751, -0.7178882360458374, 1.8077445030212402, -0.4068184196949005, -1.0901459455490112, -0.42128700017929077, -0.6744721531867981, -0.5810827016830444], [-1.490179419517517, -0.3419434130191803, -1.2490570545196533, 2.753492593765259, -1.212124228477478, -0.7279037833213806, 0.7122620344161987, -0.9549559354782104, 0.899345338344574, 0.08534315973520279, -1.6246274709701538, -0.07903871685266495, 0.24569444358348846, 1.4327203035354614, 1.0432102680206299, 0.5269429087638855, 0.01571616902947426, 1.3966255187988281, 1.202673316001892, 0.37523719668388367, -0.8410703539848328, -1.3066426515579224, 0.5293347835540771, 0.9655592441558838, -1.742174506187439, -0.6485329270362854, 1.2892916202545166, 0.7353497743606567, 0.12695059180259705, -0.19618842005729675, 0.9856601357460022, -0.8000612854957581], [-0.3593064546585083, -0.2868511974811554, 0.4852082431316376, 0.9336809515953064, 0.16562078893184662, -0.6018903255462646, 1.4895210266113281, 1.5810192823410034, 0.8139674067497253, 0.49637946486473083, -0.9458824992179871, 0.31620222330093384, 0.5683102607727051, -0.8790397644042969, -0.1669236421585083, -0.7764904499053955, -1.4151406288146973, 0.7001476287841797, 0.17331251502037048, 0.844265341758728, 1.2392748594284058, -0.6376322507858276, 1.2669507265090942, -0.6426641345024109, 0.28503379225730896, -0.6531175374984741, 0.580272912979126, -0.050463296473026276, 0.6449626088142395, 0.03192347288131714, -0.7214434146881104, -1.342484712600708], [-0.4654734432697296, -0.45623642206192017, 0.5662972927093506, -1.2507237195968628, 0.6351491212844849, -1.8826063871383667, 0.512907862663269, 0.32022392749786377, 0.749903678894043, -0.9849002361297607, -0.11552232503890991, -1.050703525543213, 3.0312604904174805, 1.6878353357315063, 0.2852044105529785, 0.7484886050224304, 0.07228241115808487, 0.4901992976665497, -1.643246054649353, -0.247522234916687, 1.1880605220794678, 1.3015475273132324, -0.7520740628242493, 1.2234565019607544, -0.5112934112548828, 0.007013573311269283, -0.4785305857658386, -0.8126289248466492, 0.5772273540496826, 0.6933116316795349, 0.131571963429451, 0.5707135200500488], [0.2763778865337372, 0.47648516297340393, -0.8145994544029236, -0.12607109546661377, 0.4421922564506531, -1.2637853622436523, -0.8739619851112366, -0.4200928509235382, 0.8139781355857849, -0.6939455270767212, -0.9317151308059692, 1.2062208652496338, 0.8973332047462463, 1.6717393398284912, -0.2403610795736313, -0.3464220464229584, 0.5392727255821228, -1.1264420747756958, -1.3403555154800415, 0.20877066254615784, -0.7628005743026733, 0.5154978632926941, -1.240698218345642, 0.5254268050193787, 0.7096357941627502, -0.20331314206123352, -1.1030992269515991, 0.23180238902568817, 0.9527621269226074, -0.6762179136276245, 0.2717170715332031, 0.2976946532726288], [0.20146657526493073, -0.7825055718421936, -0.8669832348823547, -0.7272058725357056, 0.28527456521987915, 0.5765158534049988, 1.3880650997161865, 0.4039364159107208, -0.6479132771492004, -0.1925765872001648, 0.7475546002388, 1.118756890296936, -0.9211335778236389, 0.07281918823719025, -0.36903318762779236, 0.10922086983919144, -0.19600088894367218, 0.6507042646408081, 0.6480743288993835, -0.32858890295028687, -1.1588414907455444, -1.2615054845809937, 0.22564804553985596, 2.915250778198242, 0.5730214715003967, 0.9385123252868652, -0.28525370359420776, -2.6099274158477783, -0.6198316812515259, 1.2967480421066284, -0.5645855069160461, 0.46844345331192017], [-1.4871826171875, -0.22444142401218414, 1.4787033796310425, -0.22216564416885376, -0.22409601509571075, -1.6073840856552124, 0.4737640619277954, 1.2640775442123413, 0.9573232531547546, 0.9098758101463318, -0.21527154743671417, 0.5048248767852783, 0.617653489112854, -0.7092148661613464, -0.21497602760791779, -0.3223700523376465, -1.5537617206573486, -0.6001750230789185, 0.7850821614265442, -0.48582929372787476, -1.7514761686325073, -0.4386896789073944, 0.12768298387527466, 0.4747585952281952, 1.9732136726379395, -0.6768838167190552, -0.7721245288848877, -0.7212429046630859, 0.05485258996486664, -0.30245551466941833, 1.9093005657196045, -0.8246363997459412], [-1.0492957830429077, -0.3685469329357147, -1.524105191230774, -0.029438775032758713, 0.8633372783660889, 0.14353980123996735, -0.3450111746788025, 0.32715487480163574, -0.059324827045202255, 1.1492609977722168, -0.9691851735115051, -1.1040034294128418, -0.5944600701332092, 2.3317949771881104, -0.32044869661331177, -0.8980411291122437, 0.8663799166679382, 0.957679808139801, 0.6176072955131531, -2.0848283767700195, 1.1522016525268555, -0.953914225101471, -0.058172814548015594, -0.25286442041397095, 0.8740918636322021, -0.8926450610160828, 0.03685237467288971, 0.4463835656642914, -0.8600043058395386, -1.3284777402877808, 0.39798083901405334, 0.20151418447494507], [0.6912832856178284, 0.40328314900398254, 0.5746885538101196, -1.7212494611740112, -0.6966434121131897, 0.9621672630310059, -0.3622716963291168, 0.670626163482666, 1.1334245204925537, -0.11420352011919022, 0.4700857996940613, -1.6022095680236816, 0.2536766529083252, 0.5987067222595215, -0.5471681952476501, 0.9915952682495117, -0.7590373754501343, -0.19322073459625244, -1.370970368385315, 0.3500419557094574, -1.219754695892334, -0.3227057456970215, -2.5838825702667236, -1.1290363073349, -0.31003275513648987, 0.0583784393966198, 0.7894942164421082, -0.5881481170654297, 0.8459188938140869, -0.1879119873046875, 0.3590144217014313, -0.4448716342449188], [-0.22351495921611786, 1.1319599151611328, 1.1790950298309326, -0.15999288856983185, -1.5741335153579712, -0.38610294461250305, 0.7976711392402649, -0.19207774102687836, 0.7985972166061401, -2.256348133087158, 0.025551768019795418, 0.008612864650785923, -1.0817043781280518, 0.47657454013824463, -0.5923132300376892, -0.358996719121933, -0.8256494998931885, -0.42167165875434875, -1.2030973434448242, 1.0442466735839844, 1.6781058311462402, -1.2542046308517456, -1.081550121307373, 0.47955334186553955, -0.6837116479873657, -0.9709417819976807, 0.3859111964702606, 0.8390386700630188, -1.1343352794647217, -1.7900786399841309, 0.10452103614807129, -0.9669626355171204], [1.5410971641540527, 0.3411135971546173, -0.5948267579078674, -0.4857005476951599, 1.4779367446899414, -0.9728205800056458, 1.7021856307983398, 2.3694591522216797, -0.7810680866241455, -0.10639521479606628, -1.533439040184021, 0.7411418557167053, -0.5683579444885254, -0.7915495038032532, 1.0508085489273071, -0.44343066215515137, -0.1812218278646469, 1.4428646564483643, 0.36906683444976807, -0.5618343353271484, -0.6306573748588562, -0.5770754218101501, 0.20472554862499237, -0.25013843178749084, -0.750159740447998, 1.000744104385376, -0.588080644607544, -1.1468698978424072, -0.29490265250205994, -0.4561995565891266, -0.6728546023368835, 1.0080803632736206], [0.2663434147834778, -0.23109106719493866, 0.5910833477973938, 1.7673113346099854, 1.1637376546859741, 1.0569865703582764, -0.45205673575401306, -0.24861732125282288, 0.8001043796539307, 0.12444724142551422, 0.5961864590644836, -0.2659587264060974, -0.6059004664421082, -0.16142620146274567, -1.9298405647277832, -0.6134223341941833, 0.72561115026474, 0.5045760869979858, 0.7978392243385315, 9.293800394516438e-05, -0.4906332194805145, 0.35336506366729736, 0.3770976960659027, 0.4996224343776703, -0.7093527317047119, -1.4648336172103882, -1.1173723936080933, 0.051251430064439774, -0.5544875860214233, -0.8604406118392944, -0.5917556285858154, -1.0337345600128174], [1.3641245365142822, -0.2446046620607376, 1.0834214687347412, 0.681164562702179, 0.29422685503959656, 0.06440695375204086, -0.5212385058403015, 0.8382812142372131, -0.4207347631454468, -0.8793670535087585, 0.9132486581802368, -0.38928455114364624, 0.06867148727178574, 0.5896914005279541, -1.147239327430725, 1.9610929489135742, 0.8011451959609985, 0.9297353029251099, 0.49160459637641907, 0.006310367025434971, -0.9481208324432373, 2.3272340297698975, -0.8222191333770752, -0.4841051697731018, -0.8623604774475098, -0.7295206189155579, 1.6756432056427002, 0.6207063794136047, 0.9409874677658081, -0.36367180943489075, -0.032623305916786194, -1.3855375051498413], [0.02883872017264366, -0.256577730178833, -0.2844809591770172, 1.3387982845306396, -0.6546439528465271, 0.9956566691398621, 0.6754859685897827, -0.6253687143325806, -1.3346774578094482, 1.5154451131820679, -0.36557385325431824, -1.9269394874572754, -1.1336127519607544, -1.0374056100845337, 0.6874322295188904, 0.573168158531189, 0.10255115479230881, 2.190526008605957, 1.0098930597305298, -1.598989725112915, 1.6373517513275146, 1.049710988998413, 0.15761882066726685, 0.6996732354164124, 0.8072047233581543, 0.45745110511779785, 0.9048800468444824, -0.35416850447654724, -0.10140207409858704, 0.10792975127696991, -0.44777411222457886, -0.609394371509552], [0.9167945384979248, -0.20095403492450714, 1.0207161903381348, -0.5081570744514465, -0.6787961721420288, -0.19864100217819214, 1.5580661296844482, 0.2404050976037979, 0.9166786670684814, 0.9914230108261108, -1.420636773109436, -1.2725896835327148, 0.2641424834728241, 0.07486496865749359, -0.49300628900527954, -0.8544469475746155, 0.026140229776501656, -0.4442876875400543, 0.650627851486206, -1.344538927078247, 1.0675103664398193, -0.8406602144241333, 0.15307898819446564, 0.5464704036712646, 1.1677358150482178, 0.7685307860374451, -0.5429056286811829, 1.4035935401916504, -0.8631851673126221, -0.21149112284183502, 1.308237910270691, 1.2068856954574585], [0.13617244362831116, 0.22443746030330658, -0.6805202960968018, 1.1628334522247314, 1.7291330099105835, -0.33667248487472534, 1.35645592212677, -0.0819099023938179, 0.7385042309761047, -1.782629132270813, -0.9152383804321289, -0.793471097946167, 1.1623916625976562, -0.0071566784754395485, -1.0425875186920166, -0.8822154998779297, -1.3332679271697998, -1.3934049606323242, 1.0673189163208008, -0.6344740390777588, -0.432819664478302, 0.8238062858581543, 0.9796702861785889, -0.46641480922698975, 0.6990382075309753, -1.2603892087936401, 0.126618891954422, -0.10547157377004623, -0.7952365875244141, 0.16021254658699036, 0.6388126015663147, 2.1036434173583984], [-0.884308397769928, 0.47300079464912415, -0.4101853370666504, -0.596863865852356, -0.34715330600738525, 0.4962288439273834, -1.522605538368225, -1.4309128522872925, 1.5111559629440308, -1.011454701423645, 1.7941250801086426, -0.03024238348007202, -0.2860790193080902, -0.37102577090263367, 0.6681654453277588, 0.8394709825515747, 0.023692557588219643, -1.0092307329177856, -1.5101016759872437, -1.4153392314910889, -2.1139638423919678, 0.4892623722553253, -0.28867289423942566, 0.03697631508111954, 0.2762930691242218, 1.0292165279388428, -0.4324290156364441, 1.630713939666748, -0.09603251516819, -0.016150690615177155, 1.5482597351074219, 1.0660548210144043], [-0.21290980279445648, -1.188646674156189, 2.1479756832122803, -0.22970366477966309, -0.8605321049690247, -1.5405638217926025, 0.2695530652999878, 2.809497356414795, 0.42097967863082886, -0.07544160634279251, -0.008756409399211407, -1.3121830224990845, -0.4197392761707306, 1.0761590003967285, -1.3963110446929932, 0.35511448979377747, -1.3031165599822998, 1.2299935817718506, 0.9570500254631042, 2.112044334411621, -0.12178036570549011, -0.9583468437194824, 1.1159842014312744, 1.361931562423706, -0.9444960355758667, -1.6641154289245605, -1.536458134651184, -1.0149718523025513, 0.5352301597595215, -0.9320443272590637, 1.1273192167282104, 0.2949797213077545], [0.8949549198150635, -1.023146390914917, -1.1343542337417603, 1.3241217136383057, 0.2999702990055084, 1.0332368612289429, -0.11309071630239487, -1.3173747062683105, -1.5492489337921143, -0.908893346786499, -1.5212106704711914, -0.25498640537261963, -0.4743823707103729, -1.0314198732376099, 0.28045859932899475, 0.7430112957954407, -0.9553440809249878, -0.4109712243080139, 0.8346555829048157, -0.24677059054374695, -0.14036402106285095, -1.0767754316329956, 0.00479145348072052, 0.35193291306495667, -0.6547667980194092, 0.2852243185043335, 0.5683643817901611, -0.7568632364273071, -0.2235645204782486, 0.18355843424797058, -1.2923802137374878, -1.8220499753952026], [0.324347585439682, -0.01949435845017433, 0.5862147808074951, -1.2418168783187866, 0.6739475727081299, -0.8112627267837524, 0.2769595682621002, 0.5777695775032043, -0.13647030293941498, -0.8752682209014893, -0.47232377529144287, 0.09209912270307541, 0.37762972712516785, 0.3841021955013275, 0.966923177242279, 0.2738698720932007, -1.1455037593841553, -0.22289738059043884, -1.2974156141281128, 0.14839987456798553, -1.2090849876403809, 1.461300253868103, -0.5814957022666931, -0.034355513751506805, 1.234276533126831, -0.43307262659072876, 0.21170152723789215, 0.8331531286239624, 0.6236559748649597, -0.37137460708618164, -1.0075229406356812, -0.8920676112174988], [-0.9828388690948486, 0.47937044501304626, 0.3565853536128998, 1.842231035232544, -1.7233695983886719, -0.958246648311615, 1.1996653079986572, -0.7259106040000916, 1.5046406984329224, -0.4328362047672272, 0.9027497172355652, -0.4534187614917755, -0.039246365427970886, 0.8823350667953491, 1.4171721935272217, -1.1421834230422974, -1.752030611038208, 1.4272964000701904, -0.6416510939598083, -0.9289944171905518, 0.8132224678993225, 0.26777002215385437, -0.7150846719741821, 0.4145452082157135, -0.4909479320049286, 0.2891392707824707, 0.33117613196372986, -0.05697491392493248, -0.21433904767036438, -0.27054542303085327, 0.7299823760986328, 0.1666933298110962], [0.7780404090881348, -1.5332684516906738, -0.49822333455085754, -0.593406081199646, 0.18612854182720184, 0.3061097264289856, 2.099066734313965, -0.9214239716529846, -0.2665259838104248, -1.751041054725647, -0.24860891699790955, -0.9809888601303101, 0.38577795028686523, 0.5323613286018372, -1.2871263027191162, -0.7112923264503479, 0.7049189805984497, -0.4555721580982208, -0.15030302107334137, -0.1659613996744156, 1.7817678451538086, 1.3768389225006104, 0.4701693654060364, -1.2384108304977417, 0.46072056889533997, -2.385465383529663, 0.9623525738716125, 0.03285250812768936, -0.23742587864398956, 2.01654314994812, 1.354824423789978, 2.362351179122925], [0.2240050733089447, -0.7464287877082825, 1.297757625579834, -0.028985152021050453, 1.0210212469100952, -0.27710631489753723, 1.066360354423523, -1.4694899320602417, 0.8028745055198669, -0.747296929359436, 0.17960794270038605, -0.8159031867980957, -1.8782368898391724, -0.6178669929504395, 0.3833332657814026, 0.5580363869667053, 0.032811783254146576, -1.0884960889816284, -0.5168884992599487, 0.5424613356590271, -1.228007435798645, 0.5747820734977722, -1.745268702507019, -0.4459129273891449, -0.1830907166004181, 0.20733360946178436, -0.4491215944290161, -0.026262372732162476, -0.6313536763191223, 1.5072094202041626, 0.36676356196403503, 0.5168550610542297], [0.09665980935096741, -0.19483055174350739, 1.228266716003418, -0.12297872453927994, 0.21342681348323822, -1.5665732622146606, 0.5495107173919678, -0.40103664994239807, -0.04832916706800461, -0.666620135307312, 0.002094320021569729, -0.25780725479125977, 0.28519752621650696, 0.10970339924097061, 0.014762693084776402, -0.8163790702819824, 0.6742121577262878, -0.5961988568305969, 1.3113313913345337, 0.9256239533424377, 0.5368191599845886, -0.24248845875263214, 1.3636244535446167, -0.7958228588104248, 2.1769888401031494, -0.08004260808229446, 0.6785144209861755, 0.8378666043281555, -1.1255204677581787, -1.5333622694015503, 0.2053665816783905, -0.3954477608203888], [0.06658173352479935, -0.05203486233949661, -1.2092944383621216, -0.08176986128091812, -0.35210198163986206, -0.38422587513923645, -0.13279485702514648, 0.6990593075752258, 0.6089919805526733, -1.369713306427002, 1.2111951112747192, -0.5059067010879517, -1.8296265602111816, 1.5894787311553955, -0.3033526837825775, -0.43262314796447754, -2.459296226501465, -2.2045795917510986, 1.1414692401885986, 0.2864077091217041, -0.1032891646027565, 0.28907209634780884, 0.381730318069458, -2.2309422492980957, -1.6651623249053955, -0.27918970584869385, 0.04344974085688591, -0.3576955795288086, -0.6585538387298584, -0.9245619177818298, 0.6865019798278809, -0.49744877219200134], [1.7833772897720337, 1.5843783617019653, 1.370727300643921, -0.3204886317253113, 0.13043531775474548, -0.554458498954773, 2.3148772716522217, -0.6268656253814697, -0.112180195748806, -0.0010826295474544168, -0.4536949396133423, -2.031590700149536, -0.6723988056182861, -0.48265352845191956, 0.2686949670314789, -0.7122518420219421, -0.31354495882987976, 2.3606786727905273, 0.3564547598361969, -0.4029349982738495, -0.22263340651988983, 0.7918210029602051, 2.021296262741089, 0.3390846848487854, 1.490789532661438, 0.035159382969141006, 0.7126163244247437, -0.9539300799369812, -0.8673622012138367, -1.4167101383209229, 1.2914044857025146, 1.3565444946289062], [1.9609745740890503, -1.1839015483856201, -2.2639739513397217, -1.685530662536621, -0.7223189473152161, -0.07754618674516678, 0.02284599281847477, 0.5225764513015747, -1.338257074356079, 0.9038136005401611, 0.13711665570735931, -1.3470914363861084, -0.5404021143913269, -2.053981304168701, 0.6981663703918457, 1.0787150859832764, -0.5615264773368835, 0.2607494294643402, -0.3469821810722351, -0.7474547624588013, 0.17358924448490143, -1.327933430671692, 1.2187446355819702, 0.9371638298034668, -0.8452500700950623, 0.6984584331512451, -0.947898268699646, -1.348107099533081, 0.4068317413330078, 0.33876392245292664, 1.013540267944336, -0.9522358179092407], [0.20135372877120972, 0.06018942967057228, -1.0844883918762207, -0.5431888103485107, 0.4323507249355316, -0.6162508726119995, 0.4883609116077423, -0.40268954634666443, 1.492126703262329, -0.8979966044425964, -0.03112085536122322, 0.4480423927307129, -1.0272401571273804, 0.7172968983650208, -0.14058493077754974, 0.6576988101005554, 0.805107831954956, 2.4212920665740967, -2.2049031257629395, 0.9449397921562195, 1.4631630182266235, -0.37019920349121094, 0.03090929612517357, -0.44459518790245056, 1.9037500619888306, -0.04488013684749603, -0.7838672995567322, 1.4232261180877686, 1.757358193397522, -0.7037136554718018, 0.9140541553497314, 0.7577577829360962], [-0.5294200778007507, -0.835299015045166, -0.6192008852958679, 1.5117088556289673, -0.7708255648612976, 0.2947627305984497, 0.9449784755706787, 0.2728728950023651, -0.5350672006607056, 0.21854643523693085, -1.8752524852752686, 0.9995776414871216, 0.13900110125541687, 2.1829147338867188, 0.7991070747375488, 0.9525001645088196, 1.3902686834335327, 0.34171316027641296, 0.5461615920066833, -1.418595790863037, 0.21179084479808807, 0.008023660629987717, -1.5073986053466797, 0.36447250843048096, 0.2629748582839966, 1.534297227859497, 0.19004441797733307, 1.3094899654388428, -0.38566669821739197, 0.5659535527229309, -1.0587615966796875, -1.3517895936965942], [-1.2316241264343262, -0.026703698560595512, -1.1414577960968018, -2.4785242080688477, -0.3430410325527191, 0.24905192852020264, 0.5146161913871765, -0.0012040795991197228, -1.1333855390548706, 0.20458540320396423, -0.8910597562789917, 0.5565512180328369, 0.16459594666957855, -0.63034987449646, 0.8776552677154541, -1.7935203313827515, 0.27659332752227783, 0.6978636980056763, -0.11607157438993454, 0.06551019102334976, 0.14278855919837952, 1.855000376701355, 0.40998584032058716, -0.6940328478813171, -1.0045915842056274, -1.66425621509552, 0.27158892154693604, 0.6916665434837341, -0.6289660930633545, 0.46936777234077454, 1.2015953063964844, 1.5203214883804321], [0.3549479842185974, 1.714645266532898, 2.0730984210968018, -0.6755119562149048, -0.3921760022640228, -0.940186619758606, 1.5360163450241089, 0.044436611235141754, -0.34658992290496826, 1.3070194721221924, 0.027168622240424156, -1.1114321947097778, 0.5773744583129883, 0.28175491094589233, 0.7557360529899597, 0.4305751919746399, -0.26727885007858276, 1.0153510570526123, -0.4535088837146759, -0.5397780537605286, -1.0283796787261963, 0.18659672141075134, 1.3634772300720215, -0.3023066520690918, 0.9547934532165527, -1.715395450592041, -0.6417754292488098, -0.10062567889690399, 0.0899827629327774, -0.4806351661682129, -1.4147202968597412, -0.6689436435699463], [1.9682120084762573, 0.00811779499053955, -1.9489688873291016, -0.302231103181839, -2.0787699222564697, -0.49584442377090454, 0.22994953393936157, -0.462350457906723, 0.7192221879959106, -0.7624685764312744, 1.6051993370056152, -0.8685618042945862, -0.6046966314315796, 1.455214262008667, -0.3221820890903473, -2.2022857666015625, 1.8591829538345337, 0.6737621426582336, 0.49287301301956177, 0.7705868482589722, 0.9147725701332092, 0.33092668652534485, 0.09731941670179367, 0.8152309656143188, 1.5115320682525635, -1.1769201755523682, 0.1431063413619995, -0.7128568291664124, 1.748317837715149, 0.4850308299064636, 0.34703630208969116, 1.102548599243164], [0.6935747861862183, -1.2970516681671143, 1.434989333152771, 1.5468255281448364, -1.164493441581726, -0.6836651563644409, -2.000488519668579, 0.305496484041214, 0.07904116064310074, 0.2760688364505768, -0.8426881432533264, 0.35821446776390076, -0.8605337142944336, 0.029327144846320152, 0.5086856484413147, 0.48014718294143677, -0.9404879212379456, 0.1540491133928299, -0.4966766834259033, 1.5696173906326294, -0.4451989233493805, 1.707063913345337, 0.518682599067688, -1.1757065057754517, 0.3204360902309418, 0.13217899203300476, 1.048948049545288, 0.39709100127220154, -0.7807794213294983, 1.5694525241851807, -1.0383280515670776, -0.9537650942802429], [-1.1255664825439453, -0.8812884092330933, -1.109798550605774, 1.2189487218856812, -0.14961868524551392, -0.8618932366371155, 1.0064468383789062, -0.17372040450572968, -0.6109999418258667, 0.37214919924736023, -0.8680477738380432, 1.1411210298538208, 0.5004844665527344, 1.4050414562225342, 0.6831692457199097, 1.8240597248077393, 0.3984667658805847, 1.595839500427246, -0.19925855100154877, -0.24886977672576904, -0.23316705226898193, 2.490621566772461, 0.8918154239654541, 1.3083528280258179, -0.36113643646240234, 1.3520903587341309, 1.8010483980178833, -0.6128950715065002, -0.3309764862060547, -0.3321523070335388, 0.1999180167913437, 0.8618344664573669], [-0.10351499170064926, 0.8804091215133667, -0.6041839122772217, 0.6853331923484802, 0.3291451632976532, 0.17799249291419983, 0.1830315887928009, -1.908869981765747, -1.12925386428833, -0.9372844099998474, -1.1732244491577148, 1.045149564743042, 0.747805118560791, 1.0962401628494263, 0.07973771542310715, -0.337520033121109, 0.553648829460144, 0.8931864500045776, 1.8970520496368408, -0.10716614127159119, -0.08142846822738647, 0.3864307701587677, -1.1186931133270264, 0.6490329504013062, 1.439792513847351, -0.07603033632040024, 1.0739179849624634, -1.853981614112854, 0.8303126692771912, -0.15340052545070648, -0.48949334025382996, 0.6038614511489868], [1.3776806592941284, -1.635585069656372, -0.6947888731956482, 0.4133410155773163, -0.5213840007781982, -0.2600272595882416, -0.4687231481075287, 0.6445262432098389, -1.7656968832015991, -1.0475493669509888, -1.2907183170318604, 0.017998265102505684, 0.37916499376296997, 0.9488127827644348, 0.3082205057144165, -1.998746395111084, -0.44905292987823486, -1.6861989498138428, -0.3398386538028717, 1.5753003358840942, -1.8065216541290283, -0.44242626428604126, -1.6252729892730713, 0.6283732056617737, -0.2829960584640503, 0.453652560710907, 0.018184317275881767, -0.18234296143054962, -2.353837490081787, 0.43503084778785706, 0.7713801264762878, -2.1786224842071533], [1.2278509140014648, 0.8172297477722168, 0.060940902680158615, -1.2433531284332275, -1.3274067640304565, 1.4992904663085938, -0.11983314901590347, -0.14320145547389984, -0.308480441570282, -1.167520523071289, -0.6816169619560242, -0.4550848603248596, 0.745040237903595, -1.0308235883712769, 0.39744070172309875, -0.04131884500384331, -0.4776129126548767, 0.04561518132686615, -0.7311521768569946, 1.7752102613449097, 0.6189435124397278, 1.1339306831359863, 0.735576331615448, 1.493210792541504, -0.09621024131774902, -0.1451079547405243, -0.5066493153572083, -0.37036675214767456, 0.021598514169454575, 1.5290148258209229, 0.5224801898002625, 1.166848063468933], [-0.11005708575248718, -0.7291586995124817, 0.9312685132026672, 0.6198006868362427, -1.55009925365448, 0.12303364276885986, -2.03410267829895, -0.33181044459342957, -1.0073095560073853, -0.01850583590567112, -0.13509400188922882, -1.4662717580795288, 1.9874823093414307, 0.8223257064819336, 0.18816666305065155, -0.6405514478683472, -0.12689396739006042, 0.762818455696106, -0.040995821356773376, -0.7822871804237366, -0.9232364296913147, -0.7943727374076843, 1.5545388460159302, -2.059718132019043, -0.8364178538322449, 0.6990721225738525, -0.19337384402751923, 1.1171865463256836, -0.9847024083137512, -1.9913543462753296, -0.29372233152389526, -0.9657847881317139], [0.16181977093219757, -0.16659511625766754, -2.124290704727173, 2.1233901977539062, -0.22295868396759033, 0.09560268372297287, -0.6987744569778442, -1.8176944255828857, -0.29680001735687256, 0.40991777181625366, 1.9484225511550903, 1.0974748134613037, -0.5430763363838196, -2.031395196914673, -1.0604726076126099, 0.0016200583195313811, -0.6796971559524536, 0.472468763589859, -0.7972771525382996, -1.2304291725158691, 0.7937923669815063, 2.3442025184631348, -1.2066118717193604, 0.33363044261932373, 0.6212607622146606, 1.3874297142028809, 0.13997310400009155, 1.3777093887329102, 0.9437268376350403, 0.6901546716690063, -0.7143368721008301, 0.19010132551193237], [-0.6820833086967468, -0.7026262879371643, -0.11808812618255615, -0.49919694662094116, -2.267169952392578, 1.2096326351165771, -0.22598609328269958, 0.2804016172885895, -1.15907621383667, -0.5316203236579895, 1.06304931640625, 0.7641220688819885, 0.16212935745716095, -1.506083607673645, -0.5733864307403564, -0.5814061760902405, 1.3999894857406616, 0.9876452684402466, 1.450034260749817, -2.615072011947632, -0.7167386412620544, 0.6711179614067078, -1.1946516036987305, -1.0813934803009033, -0.7284694910049438, -1.4884358644485474, 0.7896466255187988, 0.43767690658569336, 1.3675870895385742, -1.2517614364624023, -1.3328453302383423, 0.37637993693351746], [-0.5300191640853882, -0.7994304895401001, -0.3067184090614319, 0.10559827089309692, 0.8578422665596008, 0.24775166809558868, 0.552024781703949, 0.5634179711341858, -0.06608456373214722, -0.17512203752994537, -0.06735177338123322, -0.665745198726654, -0.0034168586134910583, 1.4884593486785889, 1.4086843729019165, -0.2209593951702118, 0.2612302899360657, 0.7126425504684448, 0.26157036423683167, -1.5927455425262451, -0.3917039930820465, 1.2343653440475464, 0.011718468740582466, 0.7764731645584106, -0.5882905721664429, 0.020995033904910088, -0.32935020327568054, -1.5363601446151733, 0.027576206251978874, 0.32561439275741577, 0.3608386516571045, 0.283825546503067], [-1.2547539472579956, 0.6630526185035706, -0.6697019338607788, 0.021844519302248955, -0.932742714881897, -0.8552324175834656, 0.6053575873374939, -0.763717532157898, 0.6749657988548279, -0.7217453122138977, 0.3098680377006531, 1.2640700340270996, -0.5707029104232788, -0.5290201902389526, -1.419033169746399, 0.6675782203674316, -2.544792652130127, -0.8200016021728516, -0.13375434279441833, -1.0457789897918701, 0.5248938798904419, -0.3938320279121399, -1.6835347414016724, 1.402531385421753, -0.9913906455039978, -1.9280600547790527, 0.03860955685377121, -0.6328365802764893, -0.6447938680648804, 1.6680816411972046, -1.3165206909179688, -0.19306300580501556]]]], "dtype": "float32"}, {"__nd__": [[[[0.33206531405448914, -0.029363056644797325, 0.7379007935523987, 0.6801329851150513, 2.2498202323913574, 1.348322868347168, 0.9426519870758057, -0.8133345246315002, -0.2586393654346466, 1.0607547760009766, 0.21825096011161804, 0.4194328188896179, -0.4682190418243408, -0.6326202154159546, 2.3383984565734863, 0.020821837708353996, -0.06479653716087341, 0.4021001160144806, 0.431131511926651, 0.26690807938575745, -0.007465502712875605, -0.24185924232006073, 0.47825586795806885, -2.6451690196990967, -0.11811346560716629, 0.3020249009132385, 1.4319117069244385, 0.9354065656661987, 1.2702044248580933, -0.324429452419281, -0.6305550336837769, -2.0904808044433594], [0.13454687595367432, 1.4686638116836548, 0.9206038117408752, -1.0354160070419312, -0.8352996706962585, 0.0233696810901165, 0.9325871467590332, -0.20775124430656433, -0.6371008157730103, 0.4412039518356323, -1.1002205610275269, -1.0603350400924683, 0.9563703536987305, 0.11142680048942566, 0.3901078701019287, 0.10529644042253494, -0.5275490880012512, 1.5336592197418213, -0.2623010277748108, -0.6953835487365723, -0.7860399484634399, -0.3564513623714447, -0.28144943714141846, 0.39828169345855713, 2.0160884857177734, 1.5942460298538208, -0.20594826340675354, -0.1347983479499817, 1.429686427116394, 1.4695155620574951, -0.44783687591552734, -0.10191787779331207], [-1.8840972185134888, 0.7952232360839844, -0.41373369097709656, -0.3541550040245056, 0.744273841381073, -2.3232004642486572, 0.6809470653533936, 1.7690703868865967, 1.5704396963119507, -0.7442638278007507, 0.4123642146587372, -0.1324830949306488, 0.472378671169281, -0.3761278986930847, 0.6881970763206482, -1.4863789081573486, -1.2526638507843018, 0.25754350423812866, 0.37468624114990234, 0.5846417546272278, 1.1374108791351318, -0.06743424385786057, -1.0932161808013916, -0.45170220732688904, -1.7738046646118164, -1.0868253707885742, -1.6633219718933105, 0.09336700290441513, -0.8252085447311401, -0.432757169008255, 0.141261488199234, 0.14872783422470093], [0.12371183186769485, 0.8908788561820984, 1.1430398225784302, -0.8728911876678467, -0.43956711888313293, -1.058292031288147, -0.17454348504543304, -0.09492579102516174, 0.13173553347587585, -0.12480478733778, 0.3405725359916687, -1.2419968843460083, -0.2745552659034729, 0.743459165096283, 1.0877529382705688, 0.2088557779788971, -2.393815279006958, -0.6713849902153015, 1.1223963499069214, 1.1570589542388916, -0.0020573902875185013, -1.0427659749984741, -0.35628750920295715, 0.2472277134656906, -0.5676382780075073, 0.14615438878536224, 0.38826948404312134, -0.8749523758888245, -0.9055384993553162, 1.2436728477478027, 0.38192981481552124, -1.6365935802459717], [-1.3105723857879639, 1.0585471391677856, 0.385326087474823, -0.923699140548706, -1.101395606994629, -0.5243852734565735, 0.8682103753089905, -0.45309942960739136, -0.030792882665991783, 0.8092794418334961, 0.1860310137271881, -0.5372605919837952, -0.020527273416519165, -1.2512872219085693, -0.19139641523361206, 0.03222616761922836, 0.6207157373428345, 0.46260902285575867, -0.3850666880607605, 0.5829167366027832, -0.9803309440612793, 0.3329104483127594, -1.195227861404419, -1.1532933712005615, 0.568302571773529, -0.7965371012687683, -0.4228149652481079, 0.23315884172916412, 0.5933312773704529, 0.7705323696136475, 1.3466837406158447, -0.14438539743423462], [0.27306175231933594, -1.2246404886245728, 0.21134577691555023, -0.5807629227638245, 0.25050804018974304, -0.419834703207016, 1.354883074760437, 0.5312620401382446, 0.09738563746213913, -0.4655199348926544, 1.8116934299468994, -1.391918659210205, 0.6157365441322327, -1.141325831413269, 1.244512915611267, -1.377383828163147, 0.01883343607187271, -0.24668431282043457, 1.0314923524856567, -0.4555903375148773, -0.8322356939315796, -0.38461360335350037, 1.1535236835479736, 0.581504225730896, -1.7662856578826904, -0.22039265930652618, 1.5053917169570923, 1.6700044870376587, -0.052488017827272415, 0.2929793894290924, 0.5001517534255981, 1.020308494567871], [-0.7025745511054993, 1.3012219667434692, -1.1418877840042114, -0.24138256907463074, -0.14889444410800934, 0.0724279135465622, 0.9497162699699402, -0.30926015973091125, 0.6963284015655518, -0.2599910795688629, 1.8385016918182373, -1.6799055337905884, 0.03270348161458969, 0.42534175515174866, -0.9299044609069824, 1.4377479553222656, 0.5988395810127258, 0.5694878101348877, -0.9544000625610352, -0.04694917052984238, 1.1818130016326904, 0.9524471759796143, -0.5975223183631897, -0.003466219874098897, -0.7588592767715454, 1.6285841464996338, -0.3160152733325958, 0.8883460760116577, 1.3937770128250122, 0.9540063738822937, 1.1333869695663452, 0.5072653889656067], [-1.5947142839431763, 0.16265226900577545, -0.32301944494247437, 0.30159854888916016, 1.5384505987167358, -2.4450340270996094, 0.06263083964586258, 0.7842135429382324, 2.6794469356536865, -0.9261296987533569, -0.7421364784240723, -1.4289182424545288, 0.35813772678375244, 0.16914859414100647, 1.0207629203796387, 1.322153091430664, -1.3651139736175537, 0.283016175031662, 0.39350613951683044, -0.562491238117218, 0.7333999276161194, -0.18164829909801483, 0.6056464314460754, -0.3797589838504791, 0.2657853960990906, 0.6688196659088135, 0.6682741045951843, 0.6467475295066833, 1.048432469367981, -0.6406170725822449, -0.6943012475967407, -1.230055332183838], [1.1224607229232788, -0.17247360944747925, -1.2766772508621216, 1.4594861268997192, 0.17924517393112183, -0.6856234669685364, 0.0015283888205885887, 0.7683480381965637, 0.8613447546958923, -1.7404699325561523, 0.3872450292110443, 0.991158127784729, 0.07382620126008987, 0.5269758701324463, -0.6852434873580933, 0.2855496108531952, 0.5607441067695618, -0.8826482892036438, 3.191877841949463, 1.4626009464263916, -0.10298921167850494, 0.2458093762397766, -1.1456429958343506, -0.623891294002533, 0.8383635878562927, -1.5046088695526123, 0.5292803049087524, -1.1224815845489502, 0.933580219745636, -1.721881628036499, 1.3943233489990234, 0.7285540103912354], [-0.9734846353530884, -0.30468037724494934, 0.628153383731842, 0.2606079578399658, -0.5256046056747437, 0.10891137272119522, 0.7541759014129639, 1.7194766998291016, -1.387944221496582, 0.1936330944299698, -0.6951242089271545, -0.9563776850700378, 1.6099457740783691, 0.21407507359981537, 0.5548672080039978, -0.5659571886062622, -0.9848856329917908, 0.8477463126182556, 1.044313669204712, 0.3675835132598877, -0.6321842074394226, -0.0002971649810206145, 0.029366595670580864, -0.1829010546207428, 1.2005860805511475, -1.583103895187378, -0.5684680342674255, 1.0779411792755127, -1.4530822038650513, 2.4181830883026123, -1.852609634399414, 0.3149690628051758], [1.191868543624878, -0.0636003166437149, 0.6664915084838867, -1.5370625257492065, 1.8242002725601196, 0.7856680750846863, 1.9610514640808105, -0.6626397371292114, -0.7408915162086487, -2.063927412033081, -0.5706186890602112, 0.11311684548854828, 0.7188474535942078, -2.1199722290039062, -1.2789846658706665, -1.2339526414871216, 1.1471227407455444, -0.05505707114934921, -0.578348696231842, 0.019942643120884895, 0.5761673450469971, -0.07098357379436493, -0.6822953224182129, 0.9171721935272217, 0.4184150695800781, -1.029405951499939, 1.7092770338058472, 1.4102330207824707, 0.2500711977481842, 0.23162738978862762, -0.7802513241767883, 0.9561675786972046], [-1.1007531881332397, -1.0876705646514893, -1.0728167295455933, -0.5528934597969055, 0.7537311911582947, -0.30762404203414917, 1.952374815940857, -0.6404436826705933, -2.181913137435913, 1.4490717649459839, 1.5072206258773804, 0.151338592171669, 1.0089830160140991, -1.0237482786178589, -1.361106514930725, -0.8033122420310974, 0.6142545342445374, 0.6324228644371033, -1.4159092903137207, 1.5808708667755127, -1.894952416419983, 1.0967655181884766, -0.9199466109275818, -1.3140097856521606, -0.6991804242134094, -0.38505443930625916, -0.7794825434684753, -0.9482223987579346, 0.14081469178199768, 0.1699954867362976, -0.6556538939476013, 0.2867887318134308], [0.6123151183128357, -0.13673117756843567, 2.121756076812744, -0.4567168653011322, 0.5221173763275146, -1.415848970413208, 0.31087562441825867, -1.0740381479263306, -1.139024019241333, 0.4917326867580414, -1.5144466161727905, 2.1241650581359863, 1.6619622707366943, 0.980745255947113, -0.6545658111572266, 0.9622148275375366, 1.0776747465133667, -0.48336684703826904, -0.6334977149963379, 0.386453777551651, -2.797759771347046, 0.2931479215621948, 1.392952799797058, 1.2762126922607422, 0.911469578742981, 0.125993013381958, -0.03889903798699379, 0.29919788241386414, 0.07255006581544876, -0.9494040012359619, -1.4845086336135864, -0.08193511515855789], [-1.3374049663543701, 1.6969256401062012, 0.2671516537666321, 0.11344391107559204, 0.2132396697998047, 0.4893345832824707, 1.1076350212097168, 0.36074861884117126, -0.21599820256233215, 0.8051584362983704, -0.12179138511419296, -2.167189359664917, 1.1819311380386353, 1.3436715602874756, 0.3709785044193268, 1.1797480583190918, 1.6464332342147827, 1.6889561414718628, -0.12654748558998108, 1.1204638481140137, -1.0862363576889038, -0.1100863441824913, -0.0905018225312233, 0.9334242939949036, -0.9869511723518372, 0.016659043729305267, 2.473660707473755, 0.30049440264701843, 0.6993603706359863, 1.2249412536621094, -0.7524265646934509, 0.11112958937883377], [0.09029524773359299, -1.3275301456451416, -0.4836782217025757, -1.7851890325546265, 1.0827616453170776, 1.1574050188064575, 1.5568368434906006, -1.7534490823745728, 0.9005609154701233, -0.23093511164188385, 0.3994050920009613, -1.2234400510787964, -0.6001613140106201, 1.348628044128418, 1.32101309299469, 1.7585691213607788, -0.9428030848503113, -0.12487536668777466, 1.4753236770629883, -1.829158067703247, -0.78741055727005, 0.5086134672164917, -0.8039859533309937, -0.8756436109542847, -0.8334494233131409, 0.4624062180519104, 0.9478142261505127, 1.2860465049743652, 0.9565622210502625, -0.814210057258606, -0.3115657567977905, 2.5803091526031494], [-0.7738001942634583, 0.26887473464012146, -0.02554170787334442, -1.2268685102462769, 1.704293131828308, 1.0229482650756836, -1.019699215888977, 0.8023404479026794, -0.9396281838417053, -1.4875025749206543, -2.640638589859009, 0.8686591982841492, -1.4743807315826416, 0.9840986132621765, 0.03775275498628616, -1.3049860000610352, -0.3390880227088928, -0.36563464999198914, 0.44297853112220764, 0.8347322940826416, -0.14080147445201874, 0.7228685617446899, 2.0889062881469727, 0.42758938670158386, 0.20336900651454926, 0.02129739336669445, -0.4920625686645508, -0.541742205619812, -1.6401246786117554, 0.5462440252304077, 0.16228795051574707, -0.6266461610794067], [-1.1810017824172974, 0.6738204956054688, 1.646897315979004, -0.17555934190750122, 0.4713399410247803, 0.7304035425186157, -0.4765366017818451, 0.17959488928318024, -0.2406788468360901, 0.21811281144618988, -0.7325571179389954, 0.05887714400887489, -0.7151795029640198, -0.025702007114887238, -3.285273551940918, -1.8689384460449219, 0.9632506966590881, -0.20783179998397827, -1.5610769987106323, 0.7378050088882446, -1.6994850635528564, -0.9625471234321594, -0.7078526020050049, 0.4938967227935791, 1.6164696216583252, -0.1768822818994522, -0.0924084261059761, 1.8409539461135864, -0.7007566094398499, -0.43635064363479614, -0.16847316920757294, -2.7441389560699463], [0.6019973754882812, -0.4455298185348511, -1.3498060703277588, 1.0215117931365967, 1.390354037284851, -1.166311502456665, -1.242322564125061, -1.1423088312149048, -0.4896085858345032, 0.19052903354167938, 0.9255644083023071, -1.2490402460098267, 0.05809291824698448, 1.166715145111084, -0.24483703076839447, 1.4521849155426025, 0.8170358538627625, 0.749504029750824, 0.816787600517273, -0.6682109832763672, -0.1686801165342331, 0.7485556602478027, -1.0622427463531494, -0.26642873883247375, -1.4943363666534424, -0.4284482002258301, 0.9816815257072449, 0.0895281508564949, 0.021797696128487587, -0.38318249583244324, -0.1637377142906189, -1.362815499305725], [0.612187922000885, -2.361253023147583, 0.9999837279319763, 0.13788951933383942, 0.09293796122074127, 0.7470930218696594, 1.503839373588562, -1.7250515222549438, -0.8958454728126526, -1.1422851085662842, -1.8335731029510498, -0.026204120367765427, 0.11253929883241653, -1.0554558038711548, -0.7401418685913086, 0.20771944522857666, 1.8830792903900146, 0.33277666568756104, 0.8929184079170227, -0.552057683467865, -0.9649027585983276, -0.9177322387695312, 0.7561297416687012, -0.29711034893989563, 0.20011328160762787, -0.28713253140449524, -0.9699872732162476, 1.3157438039779663, 0.9887353181838989, 0.13153648376464844, 0.9731553196907043, -2.5629687309265137], [0.14198461174964905, 1.2263413667678833, 0.4189833998680115, 0.15812167525291443, -0.6549466848373413, 0.35051605105400085, -1.0231624841690063, 1.0270798206329346, -0.07445646822452545, 1.2587100267410278, 0.25384318828582764, 0.5805128216743469, 0.057891033589839935, 0.25634267926216125, 0.1725226789712906, 1.3580738306045532, 1.8773229122161865, -1.0212478637695312, -0.22952906787395477, -0.13902021944522858, 1.0689281225204468, -0.3148275911808014, -1.1633356809616089, 1.0791391134262085, 0.35582733154296875, -0.056941740214824677, 0.4979712665081024, -0.4466783106327057, 0.11595344543457031, -1.4275423288345337, -0.24306300282478333, -0.8734750747680664], [-0.3327368199825287, 1.2809542417526245, 3.3792428970336914, -0.709570586681366, -1.5998300313949585, -0.94021075963974, -0.3498584032058716, -0.3865060806274414, -0.36641034483909607, 0.34710824489593506, -0.7549129128456116, -0.24062775075435638, 0.2166863977909088, 0.5419386029243469, -0.6939422488212585, -0.3126958906650543, 0.3449689745903015, -1.038267970085144, -0.19238047301769257, -0.5864167809486389, 0.8093959093093872, -1.542535662651062, -0.9842550754547119, -0.347382515668869, -0.8150114417076111, 0.4795016348361969, -1.287204623222351, 0.19309179484844208, -0.11753150075674057, 0.04337873309850693, 2.1733579635620117, -0.25465700030326843], [0.23053014278411865, 1.693055272102356, 1.5959464311599731, -0.5897409915924072, 0.9522718191146851, -0.3999265134334564, 0.16992895305156708, -0.15611353516578674, -0.4827582836151123, 0.8460887670516968, -0.8792711496353149, 0.08103634417057037, -0.17032013833522797, -1.0369787216186523, -0.2635870575904846, 0.05636705085635185, -0.397183358669281, 0.9694145917892456, 0.08144861459732056, -0.2549644708633423, -0.11464685201644897, -1.7503448724746704, -0.8322192430496216, -0.11543121933937073, 0.3087868392467499, -0.2517983913421631, -0.09930283576250076, -0.37067508697509766, 1.1816418170928955, 0.22903306782245636, -0.4309960603713989, 1.85765540599823], [0.775506317615509, 0.41262513399124146, -0.04737478122115135, -0.11756067723035812, 1.393876552581787, -0.9612725973129272, 0.7348047494888306, 0.5928824543952942, -1.6215527057647705, 1.3010042905807495, 0.6820552945137024, -1.3478450775146484, 0.4607110321521759, 1.121767520904541, -0.22172687947750092, 0.41331231594085693, -0.48018553853034973, -1.983364224433899, -0.058214422315359116, 1.4496105909347534, -0.6272851228713989, 2.0753273963928223, -1.3321483135223389, -0.31259143352508545, 0.7670371532440186, -0.2692345082759857, 0.8784577250480652, -0.3162672817707062, -1.0922644138336182, 0.47817936539649963, 0.9982187151908875, -1.3984696865081787], [-0.8356988430023193, 1.4043911695480347, 1.4714652299880981, -0.7726997137069702, 1.4523935317993164, 1.2297852039337158, -0.8876304626464844, 0.8390160202980042, 1.461177945137024, 0.46337705850601196, -0.3385971784591675, 1.4363257884979248, -0.009213813580572605, 0.28694358468055725, -0.3095996379852295, 0.7510741949081421, 0.09380904585123062, -1.7048120498657227, 0.25320613384246826, 0.5282295346260071, -0.20358341932296753, -1.3598413467407227, -0.6463770866394043, 0.09717241674661636, -0.8163115978240967, 0.6440659761428833, 1.4165096282958984, -0.009163395501673222, 2.0564255714416504, 0.8232832551002502, 1.213994026184082, 0.7359878420829773], [0.4081263542175293, 1.1551448106765747, -0.8930507302284241, 1.3741552829742432, -0.8008794188499451, -0.18972302973270416, 0.18112465739250183, 0.12437333166599274, 0.6384878754615784, -0.3350384831428528, -0.360551118850708, -1.6017307043075562, 0.41211575269699097, -1.0278222560882568, -0.9881596565246582, 0.5021986365318298, 0.3911513090133667, 0.6826462745666504, -0.30791643261909485, 0.3563554584980011, -0.12527301907539368, -0.626569926738739, 1.8966808319091797, 0.4716722071170807, -0.07435748726129532, -0.5699169635772705, 1.241528034210205, 0.00659734383225441, -0.47246354818344116, 0.11169638484716415, 0.27749186754226685, -1.475113868713379], [0.8546268939971924, 0.8634775876998901, 0.41231048107147217, -0.7469010949134827, -0.335002601146698, 1.4954187870025635, 1.017112135887146, -0.5022333860397339, 1.0348525047302246, 0.4347032904624939, -0.7114963531494141, -0.8045520782470703, 0.2697116732597351, -0.49896112084388733, 0.0002663473424036056, 0.436725378036499, -1.3574296236038208, 1.18207848072052, -1.5645416975021362, 1.3129220008850098, -0.16372358798980713, 1.7112159729003906, 1.1931607723236084, 0.09204113483428955, 0.08617927879095078, -0.44876012206077576, -0.37079891562461853, 0.5839703679084778, -0.6441820859909058, 0.40915367007255554, 0.7638806700706482, -0.9962109327316284], [0.18602974712848663, 0.6620935797691345, 0.28272631764411926, 0.7525647878646851, -0.9209079146385193, 1.556519865989685, 0.3021092116832733, 0.7489725351333618, -0.08945336192846298, -0.9853000044822693, 0.43198880553245544, -0.605566680431366, -1.0419107675552368, -1.0707541704177856, -0.6833502054214478, 0.8369333744049072, -1.85599684715271, -0.16169697046279907, 0.9017726182937622, -0.3798210918903351, -1.6894444227218628, 0.8161041140556335, -1.020942211151123, -1.5691062211990356, -0.5395594835281372, -1.8775427341461182, 0.6693203449249268, -0.29623356461524963, -0.8566837310791016, -1.1638792753219604, 0.18676407635211945, -0.747611939907074], [0.08861461281776428, -0.6140969395637512, -1.2978527545928955, -0.9831274151802063, 0.12381573021411896, 1.6394140720367432, -0.4988197088241577, -0.9321851134300232, 0.39017245173454285, 0.44159042835235596, 1.1377770900726318, -0.9112101197242737, 1.1067311763763428, 0.1719893366098404, -0.5239139199256897, -1.003545880317688, 0.27543359994888306, -0.7119362354278564, -0.4205082952976227, 0.8541073203086853, -0.7822416424751282, -0.6382720470428467, -0.43750089406967163, 0.3542960584163666, -0.4370625615119934, -1.113794207572937, -0.7725145816802979, 0.5449771285057068, 0.5393977761268616, 2.829261541366577, -1.5538653135299683, 0.4721035659313202], [1.7487573623657227, -0.8967691659927368, 1.854438304901123, -0.676198422908783, 0.1521887630224228, -0.9372746348381042, 1.1031782627105713, -0.18846723437309265, -0.8217625021934509, 0.4718661904335022, -0.8865642547607422, 0.12611712515354156, -1.0658961534500122, 0.2951154410839081, 0.2606123089790344, 0.2286699116230011, 1.6562440395355225, 0.8323714137077332, -0.8804595470428467, 0.9318182468414307, -0.47511547803878784, 0.8099711537361145, 0.426098495721817, 2.029963254928589, 1.3361140489578247, 0.800320029258728, 0.08401335030794144, 0.8414065837860107, -0.18631823360919952, 1.2272214889526367, 0.28521719574928284, -1.8496359586715698], [0.7295281887054443, -0.7586296200752258, -0.03171038255095482, 0.8563434481620789, 1.2738314867019653, -0.08517603576183319, 0.16314484179019928, -0.7288689017295837, -0.27696529030799866, 0.6900295615196228, 1.694321632385254, 0.5774341821670532, 1.5880846977233887, 1.8734290599822998, -0.9868658781051636, 0.5481885075569153, 0.46835899353027344, 0.5053855776786804, 0.28665536642074585, -0.6941864490509033, 0.1017719954252243, -1.297547459602356, -0.07307847589254379, -1.0557525157928467, -0.2972846031188965, -2.2162227630615234, 1.2804975509643555, 1.6079765558242798, 1.2734730243682861, 0.3939914405345917, -0.1593269258737564, 0.41835924983024597], [0.23195993900299072, 0.42970043420791626, 0.5337948203086853, 1.8757442235946655, -0.2384202629327774, 2.895167350769043, 0.5843674540519714, 0.9090331196784973, 0.7451987862586975, 0.4890654683113098, 0.6243856549263, 1.4381431341171265, -0.044594656676054, -0.5375602841377258, -0.5971137285232544, 0.10410653054714203, -0.36540186405181885, 0.49841707944869995, 0.07900360971689224, 0.22383207082748413, -0.04905133321881294, 0.0024992520920932293, -0.5032021999359131, 0.10078302770853043, 0.7028322815895081, 1.0287243127822876, 1.9481234550476074, 0.07319539785385132, -0.35732510685920715, 0.052174557000398636, 1.2018429040908813, 1.4683958292007446], [0.6609378457069397, -0.5701746940612793, -1.7546085119247437, 1.5867136716842651, -1.7828166484832764, -0.6159214377403259, -0.42851021885871887, -0.5855112075805664, -0.13132762908935547, -1.288130760192871, -1.8106542825698853, -0.2910291254520416, -0.6453278064727783, -0.09801306575536728, 1.7506464719772339, 0.6059069633483887, -0.7824568152427673, 0.03970194235444069, -1.1993135213851929, 0.6260332465171814, 0.17938882112503052, -0.5957688093185425, -0.05817048251628876, -1.2021063566207886, 0.9381446242332458, -1.4054957628250122, -1.3477407693862915, 0.1820286065340042, 0.20114503800868988, -0.06424953043460846, -0.8934426307678223, -1.2287840843200684], [0.4036865532398224, -0.25663623213768005, 0.510267972946167, -0.04380980506539345, 0.43453407287597656, 0.0654800683259964, -0.4366506040096283, 0.6301074624061584, 0.8955625891685486, -0.1574501246213913, -1.2181284427642822, 1.6452748775482178, -0.09235773980617523, 1.3177635669708252, -0.14980369806289673, 1.4643847942352295, -2.471494197845459, -1.7256819009780884, 0.05656133219599724, -0.8305230140686035, -0.7268106341362, 1.8626779317855835, 0.381268173456192, 1.0058839321136475, -0.7578893303871155, 0.41793984174728394, -0.7355098128318787, 1.8911597728729248, 0.6960708498954773, 1.790907621383667, -0.7888814806938171, -0.09751787036657333], [-1.7361689805984497, 1.3071439266204834, -0.9880759716033936, 1.5831387042999268, -0.023877399042248726, -0.9532069563865662, 0.03358728811144829, -2.445803642272949, 1.258679986000061, 0.11765100061893463, 1.4500046968460083, -0.032314740121364594, 0.6835272312164307, -0.525345504283905, 0.470601886510849, 0.40313881635665894, 1.3797706365585327, -1.0388286113739014, -0.7496911883354187, 0.5561345219612122, 0.7702604532241821, -1.0307542085647583, 0.7987223863601685, 1.0689126253128052, 1.9283547401428223, -0.43176743388175964, 0.3626141846179962, 0.6432821154594421, -0.34327346086502075, 0.7220786809921265, -1.0263406038284302, -0.26546066999435425], [-0.937268853187561, 0.4543648362159729, -1.2131539583206177, -1.4255592823028564, -2.046536922454834, -0.5356037616729736, -0.019679201766848564, 0.05481547489762306, 0.09751689434051514, 2.3420488834381104, 1.0457913875579834, 0.8741746544837952, -0.6140568256378174, 0.8885337710380554, -2.296478748321533, -0.18663476407527924, -1.0483818054199219, -0.24884498119354248, -1.5012614727020264, 1.3942627906799316, -0.7317970991134644, -0.5763565301895142, 0.8524308800697327, -0.4064207673072815, -2.356447458267212, -2.0410797595977783, -0.21101437509059906, 0.15038979053497314, 0.7885061502456665, -1.1002888679504395, 2.054041862487793, -0.4719623625278473], [0.9553107619285583, -1.9811971187591553, 0.2699265778064728, -0.3536475598812103, -0.8821202516555786, 0.45541492104530334, 0.3745134174823761, -0.8489851355552673, -1.335033893585205, -1.0173763036727905, -0.2550414800643921, 0.03917640075087547, -0.8414429426193237, -0.934402346611023, 1.6477981805801392, 0.23158106207847595, -1.0991442203521729, -0.5096994042396545, -0.8218351602554321, 0.4664078950881958, 1.5130817890167236, -1.4393963813781738, -0.004176096059381962, -1.024981141090393, -0.7911962270736694, 0.1985553801059723, 0.49774256348609924, -1.8159573078155518, -1.5067211389541626, 0.9395725131034851, -0.7591069340705872, 0.07710602879524231], [-0.6755719184875488, 1.2363438606262207, -1.0571904182434082, 1.2244094610214233, 0.2949393689632416, 0.7868191599845886, -0.27639657258987427, 0.6346108913421631, 1.5814992189407349, -0.2334904670715332, -0.6146318912506104, -0.7413211464881897, -0.5142958760261536, 0.2940201163291931, 1.6171866655349731, -1.2253895998001099, -0.053886380046606064, -1.8246212005615234, -0.44869548082351685, -0.2860300838947296, -1.237709879875183, 1.7173436880111694, 0.10147874802350998, -0.9822590947151184, -0.34031832218170166, -0.4848845899105072, -0.4626166522502899, -0.8531250953674316, 0.6345599889755249, 2.4997875690460205, -0.6359425187110901, 0.7028487324714661], [0.46151140332221985, -0.22521446645259857, -0.33411940932273865, -0.6380692720413208, 0.41412362456321716, -1.9161491394042969, -0.1437038779258728, -0.5821640491485596, 0.561724066734314, 0.8600612878799438, 0.015472259372472763, 0.8647553324699402, -1.0072544813156128, -0.2548607587814331, -0.6964929103851318, 0.5386083126068115, -1.1482903957366943, 0.670667290687561, 1.1477241516113281, 0.7608764171600342, 1.070649266242981, 1.3586965799331665, -1.33966064453125, 0.49823564291000366, -0.13675937056541443, 0.950039803981781, -0.655194878578186, 0.33652159571647644, -0.15327322483062744, 0.007481453940272331, 0.956310510635376, -0.0412144772708416], [-0.8149768710136414, -0.041710466146469116, -0.3040640652179718, 0.3836258053779602, -2.25197172164917, -0.5738746523857117, -0.6595815420150757, -1.3953630924224854, -0.28461822867393494, 0.9484779834747314, -0.3599538207054138, -0.40403375029563904, 0.9089184403419495, 0.7830702662467957, 0.8933433890342712, -0.9849299192428589, -1.2347900867462158, -0.016082921996712685, 0.9878366589546204, -2.0100600719451904, -0.9832687377929688, -0.2956390976905823, 0.2988712787628174, -0.6196295619010925, 0.9212871193885803, 0.08388494700193405, 0.7228894233703613, 0.3925513029098511, 0.044000349938869476, -0.9766561388969421, -1.3785226345062256, -0.04728874936699867], [-0.9286741018295288, 1.0346790552139282, 0.023983526974916458, -2.0095155239105225, -0.5914156436920166, 0.2983779311180115, 0.3042842149734497, 0.8394564390182495, -0.18380016088485718, -0.46900051832199097, -0.40708449482917786, 0.042819589376449585, 0.9427974224090576, -0.13048668205738068, -0.03726683184504509, -0.38298049569129944, -0.556380033493042, -0.1130686029791832, -0.6136434674263, 1.0010186433792114, 1.9589346647262573, 0.43998444080352783, -0.24976509809494019, -0.4956243932247162, -0.32313716411590576, 1.2866488695144653, 0.41244253516197205, -0.05013434588909149, -1.2758265733718872, -0.5207396149635315, -1.5271413326263428, 1.2584748268127441], [-0.9239803552627563, -1.3442426919937134, 0.2651465833187103, -0.939944326877594, 1.169732928276062, -0.1616254597902298, 0.7957188487052917, 0.98848557472229, 0.11692279577255249, -0.7607405185699463, 0.6166096925735474, 1.8762108087539673, 0.15223993360996246, -0.6457886099815369, 1.5417792797088623, 1.8712939023971558, -0.22228196263313293, 0.518379807472229, -1.1199339628219604, -1.0318710803985596, 0.25811466574668884, 0.4471026062965393, 0.7588208317756653, 0.9112160205841064, -1.643730640411377, 1.197108507156372, 0.7956867814064026, 0.936131477355957, 1.4388450384140015, 1.224816918373108, -0.37822937965393066, 0.21175876259803772], [-0.28476330637931824, 0.5700922012329102, -0.6034106016159058, 0.2507672905921936, -1.5892196893692017, -0.41153842210769653, -1.8874260187149048, 0.36086001992225647, 0.31094905734062195, 2.2341277599334717, -0.7315167784690857, 0.5937100052833557, -0.7896606922149658, 1.5654844045639038, -1.2240437269210815, 0.01733057014644146, -0.4234756529331207, -0.404972106218338, -1.4088027477264404, 1.4621663093566895, -1.787310242652893, 1.2855685949325562, -0.718218982219696, 1.499387264251709, -1.9885815382003784, 1.0069047212600708, 0.5766610503196716, -1.0894688367843628, -0.9687299132347107, 1.3720531463623047, 0.9634714126586914, 0.5854231119155884], [0.19149820506572723, 0.08641628921031952, 1.9503191709518433, -0.18710774183273315, -0.6697835922241211, 2.3652536869049072, -1.5745611190795898, -0.2372247278690338, -0.6126907467842102, -0.2184343934059143, 1.4189475774765015, -1.2173981666564941, -0.4343428611755371, 2.6499953269958496, 0.9760462045669556, 0.7106214761734009, -0.12233085185289383, 0.508150041103363, -0.13842979073524475, -1.342971920967102, -0.8899589776992798, -0.2963964343070984, 0.6202533841133118, -0.13221536576747894, -0.03672356903553009, -0.6225810647010803, 1.1559884548187256, -0.31855982542037964, -1.5976283550262451, -2.349490165710449, -1.2682206630706787, -0.26181700825691223], [-0.8128211498260498, -1.3078680038452148, -0.21214450895786285, 0.21003983914852142, 0.5312204957008362, 1.3332761526107788, -0.8776886463165283, 0.583382248878479, -0.37702423334121704, 1.0943779945373535, -0.9435672163963318, 0.6218777298927307, 0.6673628687858582, 1.3790676593780518, 0.08196287602186203, -0.3552868068218231, -1.0835716724395752, -1.143021821975708, 0.5969586968421936, 0.8508197069168091, 0.5457800030708313, 0.628925621509552, 1.3997488021850586, -0.6655504107475281, -0.2530701458454132, 0.7138920426368713, -2.4880974292755127, 1.1861636638641357, -0.48337700963020325, 0.38645946979522705, -0.9168773293495178, -2.019463062286377], [0.3485230505466461, -0.5543140769004822, 0.17316009104251862, -0.4210553765296936, -0.15105509757995605, 0.52956223487854, -1.5203299522399902, 0.8973915576934814, -0.7432641983032227, -1.1453416347503662, -0.2837521731853485, 0.6000621318817139, 1.1253893375396729, -0.2970740497112274, 0.06998327374458313, -0.07402960956096649, -0.174841970205307, 1.2910504341125488, -0.7722058892250061, 0.4616665244102478, -0.7192683219909668, -0.4807763993740082, -0.08026102930307388, 0.12999416887760162, -1.5399469137191772, -0.3411584496498108, 0.9350863695144653, -0.31735095381736755, 0.6574865579605103, -0.20238439738750458, 1.1756781339645386, 1.2461196184158325], [-0.35233110189437866, 0.6488024592399597, -0.0008310899138450623, -1.7989376783370972, 0.07043780386447906, -0.6724681854248047, -0.5461757183074951, 1.2197670936584473, 0.3147369921207428, 1.2801337242126465, -0.8876705765724182, -1.217402458190918, 0.058833178132772446, 2.123936891555786, 1.5942802429199219, -0.04712861776351929, 0.8593623042106628, 1.513717770576477, -2.0772695541381836, 0.8152574896812439, 0.9070959091186523, -1.6546260118484497, 0.2316310703754425, 0.17626571655273438, 0.8495712876319885, 1.3267426490783691, 1.585015058517456, -0.11649735271930695, 0.7174882292747498, 0.20103687047958374, -0.5288408398628235, 0.8319919109344482], [-0.8754391074180603, -0.7172108292579651, 0.10202527791261673, 0.59889155626297, -0.3693738579750061, -0.08905910700559616, 0.5705711245536804, -0.5410933494567871, -1.485573410987854, -0.8674760460853577, -2.1579315662384033, -1.1450388431549072, -1.965462565422058, 0.7731881737709045, -1.1743829250335693, -1.0535295009613037, 1.552879810333252, -1.95858633518219, -1.062494158744812, 0.8939170837402344, -1.8307288885116577, -0.20729102194309235, -0.5319276452064514, 1.513346552848816, 2.1289591789245605, 1.2881683111190796, 0.08459951728582382, -0.5572412014007568, 0.9623297452926636, -0.8189293146133423, -0.6822688579559326, -1.243863582611084], [-0.772844672203064, 1.533509373664856, 1.2152526378631592, -1.399641513824463, -0.5535670518875122, -0.3698071539402008, 1.0065559148788452, -1.2408994436264038, -0.6043217182159424, 0.9200901985168457, 1.2366633415222168, 0.1727413833141327, 0.34260210394859314, 0.6410065293312073, 0.8580202460289001, 2.239790678024292, 1.117957353591919, 0.062426481395959854, 2.240922212600708, 0.11997698247432709, -1.7662742137908936, -1.2775774002075195, 0.13913853466510773, -0.17884057760238647, 2.82399320602417, 0.06246807426214218, -0.42762625217437744, -1.7282286882400513, -0.47080615162849426, 0.05610215291380882, -0.9439184665679932, -0.16804543137550354], [-0.23086929321289062, -0.2533514201641083, 0.005205619614571333, -1.0660133361816406, -0.1439133733510971, 0.6021014451980591, 2.189452886581421, 0.0436934232711792, 1.5282244682312012, 1.0493810176849365, 1.0250287055969238, 0.4804343283176422, -0.05055118724703789, -0.25872179865837097, 1.112896203994751, -0.22785696387290955, 0.6034923791885376, -0.058579280972480774, -1.751927137374878, -0.026921335607767105, 0.37558525800704956, 1.0288201570510864, -1.071643590927124, -0.3105715215206146, -1.8274810314178467, -1.8917200565338135, 1.6945805549621582, 0.7690633535385132, -0.7803962230682373, -0.2846607267856598, -0.37601372599601746, 0.5228899717330933], [0.9146456718444824, 1.7731084823608398, 1.8676137924194336, -0.35709553956985474, -0.8311439156532288, 0.36072978377342224, 1.95006263256073, -1.2776211500167847, -0.20747579634189606, 0.22301523387432098, -0.5353811383247375, 0.22581125795841217, 0.8822901844978333, 1.557834267616272, 0.1680113524198532, -2.132288694381714, 0.2753520607948303, -0.4382103383541107, -0.6504737734794617, 0.30354925990104675, -1.208984136581421, -1.4682387113571167, 0.2956066131591797, 0.26770204305648804, 0.6137347221374512, 0.21664974093437195, -0.18314862251281738, 0.9229517579078674, 1.642324686050415, -0.7552332878112793, 0.425357460975647, -0.7006987929344177], [-1.9202568531036377, 0.7217435836791992, -0.35260358452796936, 1.1507281064987183, -1.9759002923965454, -0.614988386631012, -0.0923851802945137, -0.13373269140720367, -0.5030393004417419, -0.5875332355499268, -1.3029954433441162, -0.04491918534040451, -1.0348225831985474, 0.5451272130012512, 1.111086130142212, -2.1742334365844727, -0.15429463982582092, -0.7456822991371155, -1.5779579877853394, -0.19631008803844452, 1.3911837339401245, -1.4614182710647583, 0.6303560137748718, -0.8745503425598145, -2.1543359756469727, -0.426315575838089, 0.14172019064426422, -2.223435640335083, 0.5817977786064148, 1.6819463968276978, -2.6281797885894775, -0.7898172736167908], [0.9863523244857788, -0.08916675299406052, 1.7458819150924683, 0.7688002586364746, -0.7328208684921265, -0.23779533803462982, -0.6337134838104248, -2.553199052810669, -1.427283525466919, -0.5733054876327515, 0.765002429485321, -0.45506206154823303, -1.6438592672348022, -1.8905986547470093, -0.5465583205223083, -1.016656756401062, 1.4407271146774292, 0.5868988633155823, 1.2047027349472046, 0.031647488474845886, -2.170409679412842, -0.04891487956047058, 0.30443599820137024, 0.6968680024147034, 0.444659948348999, -0.2063339501619339, 0.5722381472587585, 0.31780558824539185, 0.8333075046539307, -0.3862931430339813, 0.10399358719587326, 1.371614933013916], [0.354341983795166, 0.8152997493743896, 1.0801750421524048, -1.3548367023468018, 1.4693821668624878, -0.030543949455022812, -1.2621731758117676, 0.33361926674842834, -0.2556512951850891, 1.4279752969741821, -0.815289318561554, 0.3643212616443634, 0.9609394669532776, -0.2633854150772095, 2.014709949493408, -0.997742235660553, -0.17634370923042297, 0.39594587683677673, 0.40966784954071045, 0.45022979378700256, -0.763973593711853, -0.3274574875831604, 0.1821473240852356, 0.8781635165214539, -1.153153896331787, -2.079359531402588, 1.5312514305114746, -1.0032193660736084, -0.8889221549034119, -0.5681928992271423, 0.8940308094024658, 1.1155301332473755], [-0.2000960409641266, -1.0921846628189087, 1.6721737384796143, -0.8855717778205872, 1.9966813325881958, 1.1883045434951782, -0.28833261132240295, -0.21807709336280823, 0.4045838713645935, 1.051902174949646, 0.965280294418335, -1.1882368326187134, 1.3713293075561523, -0.4160170257091522, 0.49960094690322876, -1.5390136241912842, -0.19578219950199127, -1.1003674268722534, -1.2715450525283813, -0.3512006103992462, 0.4584784507751465, 1.4580787420272827, 0.14916212856769562, -1.3119802474975586, -0.5594852566719055, -1.5203831195831299, 0.016622627153992653, -1.0066341161727905, -0.9336729049682617, 1.3569843769073486, -0.05151909962296486, 1.3353848457336426], [-0.5643390417098999, -0.6548381447792053, -0.7925096750259399, -0.7723740935325623, 0.9071651697158813, 0.23701366782188416, -1.363284707069397, 0.9798943400382996, 0.7126366496086121, 2.8918118476867676, 0.4805188775062561, -0.15042221546173096, -0.5100787281990051, 0.17673969268798828, 1.3374205827713013, -1.261386752128601, 0.2241850346326828, 0.1923605352640152, -0.5561561584472656, 0.9176196455955505, -0.32998329401016235, -1.620416522026062, 0.30577394366264343, -0.39877644181251526, 0.8394991159439087, 1.9663666486740112, -0.9603293538093567, 1.2573697566986084, 0.10595119744539261, -1.3698551654815674, 0.8473494052886963, 0.49132856726646423], [1.2798765897750854, 0.7048367261886597, 1.6812000274658203, -0.8712319135665894, -0.38323479890823364, -0.49961304664611816, -0.5096026659011841, -0.2671641707420349, 0.6802166104316711, 0.7860121726989746, 1.0997508764266968, -0.4381064772605896, -0.3364882469177246, 0.4230354428291321, 1.1744420528411865, -0.9797546863555908, -0.08432364463806152, 1.2185992002487183, 0.39593902230262756, 1.4275318384170532, -0.191771000623703, 0.3488873243331909, 0.44657978415489197, -1.7827777862548828, -0.07859122008085251, 1.0628994703292847, 1.5432062149047852, -0.3856947422027588, 0.06706441938877106, 0.47389012575149536, 1.78885817527771, 0.8541538715362549], [0.15505504608154297, -1.7814542055130005, 1.0784311294555664, -1.008223056793213, -0.523453414440155, -3.0883309841156006, -0.42937853932380676, 1.7229042053222656, 0.9398290514945984, 0.6845747232437134, -0.6508971452713013, 0.48105984926223755, -0.4859360158443451, 0.0850590243935585, 0.1495489776134491, 0.29906561970710754, -0.6141637563705444, 0.14838014543056488, 1.2133013010025024, 1.4523156881332397, 2.3369665145874023, 0.6439335942268372, -0.3702560067176819, -0.22900983691215515, 0.45838701725006104, 0.6960745453834534, 0.8953744173049927, 0.6410335302352905, -0.16441716253757477, -0.5481765866279602, 0.8548956513404846, -0.5328088998794556], [-1.1919913291931152, -0.5219529867172241, 0.32121366262435913, 0.2957614064216614, -2.123215675354004, -0.7594923377037048, -0.7036260366439819, 0.9268012642860413, 0.6970070600509644, -0.02329401671886444, 0.06753730773925781, -0.26309239864349365, 0.6735981106758118, 0.28235161304473877, -0.32012468576431274, -0.3131276071071625, -0.8496441841125488, 0.4020501375198364, -1.0025713443756104, -0.7969765663146973, 0.02698875032365322, -0.6739020943641663, 1.5634303092956543, -0.37173324823379517, 1.0847671031951904, -1.5911842584609985, 0.7245530486106873, -1.9764394760131836, -1.0252078771591187, -0.29269078373908997, -0.14785610139369965, -0.11848564445972443], [-1.4438526630401611, 0.25542646646499634, 0.06494773179292679, -0.6946434378623962, -1.2460848093032837, 1.2532395124435425, 0.6445086002349854, 0.25064191222190857, 0.6583598852157593, 0.327169269323349, 1.1332937479019165, 0.5015547871589661, -0.6052513122558594, 1.0301928520202637, -0.18188625574111938, 0.5785789489746094, -1.451686143875122, -0.17456044256687164, 0.6293436288833618, -0.3575252592563629, -1.0526360273361206, -0.06229513883590698, 1.050573706626892, 0.658832311630249, 0.34611374139785767, 0.9013683199882507, 0.42316165566444397, -1.058194875717163, -1.0775461196899414, 0.08589573949575424, 0.8142381310462952, -0.35640212893486023], [0.7988730669021606, 1.2059718370437622, -0.27776622772216797, 0.8688762784004211, -0.03384203091263771, -0.17841967940330505, 0.9599360823631287, 0.6104335784912109, -0.776115894317627, -0.299386590719223, -1.0531915426254272, 0.8432106971740723, -0.6717587113380432, -1.4059584140777588, 0.40294599533081055, 2.174905776977539, -1.12911057472229, -1.6213929653167725, 1.969385027885437, -0.08309432864189148, 0.70396488904953, 2.009767532348633, 0.31695038080215454, 0.17923994362354279, -0.41033056378364563, 0.34443607926368713, 0.6049257516860962, 3.1867949962615967, 3.06333065032959, -0.6729020476341248, 0.3434625566005707, -0.7694199085235596], [1.3235174417495728, -0.30797892808914185, 1.3505374193191528, 1.5693572759628296, 0.5220796465873718, -0.1460181474685669, -1.967646598815918, 0.23574219644069672, -0.1893642693758011, -0.674567461013794, 1.3473106622695923, 0.944231390953064, -0.34971651434898376, 0.24067403376102448, 0.38573116064071655, -0.1047479435801506, 0.48765647411346436, 0.2109718769788742, 0.6384807825088501, -1.6166658401489258, -0.017561430111527443, -0.14384794235229492, -0.1944371461868286, -0.6006945371627808, 0.10462163388729095, 1.568996787071228, -0.05364123731851578, -0.6283523440361023, -0.08514651656150818, 0.5119274854660034, -1.652130126953125, 0.10436312854290009], [2.4367549419403076, -0.14430484175682068, 0.977149248123169, -0.3297746181488037, 0.4292721152305603, 1.0051101446151733, 0.27274665236473083, 0.5254254937171936, -0.4652837812900543, -0.2594554126262665, 1.0819154977798462, 1.0887309312820435, 0.8412970304489136, -0.04383692890405655, 0.49053341150283813, 2.9996588230133057, 0.8084745407104492, -0.15849053859710693, 1.424438238143921, -1.730762004852295, 1.1615729331970215, -0.6774224042892456, 0.6263283491134644, -2.3696305751800537, 1.9863591194152832, 0.91887366771698, 0.5759230256080627, 0.13080911338329315, -0.5600143074989319, 1.4671751260757446, 0.23968009650707245, 0.6698094010353088], [-0.38024812936782837, 1.6848037242889404, -1.5590829849243164, -2.8952019214630127, 1.20409095287323, -0.1811273843050003, -1.4724987745285034, 1.0347553491592407, 1.3501700162887573, -0.3728969991207123, 0.9894309043884277, -0.6845602989196777, 0.17973022162914276, -0.9596982002258301, 0.11936084181070328, 0.13129031658172607, -0.7830011248588562, 0.2103123962879181, -0.5574268102645874, 1.3338289260864258, 1.947067141532898, -0.8988144993782043, -0.48459917306900024, -0.03967518359422684, 0.6045507788658142, 1.2906008958816528, 0.2292536497116089, -0.24422012269496918, -0.6701723337173462, -1.527219533920288, -1.0025781393051147, -0.8063309788703918], [0.6478857398033142, -0.8346234560012817, 0.6489139199256897, 0.047923993319272995, -0.18132171034812927, -0.8626083135604858, 1.9289804697036743, -1.2443665266036987, 0.6733003258705139, -1.8506680727005005, 0.051676712930202484, 0.8802493810653687, -0.11091974377632141, -0.8811653256416321, -0.1572417914867401, -0.7380533218383789, -0.5000995993614197, 0.7297184467315674, -2.2218165397644043, 0.40213531255722046, 0.2326807677745819, 1.4341204166412354, 0.4015296399593353, 0.06978282332420349, -0.4467288851737976, 0.2803480923175812, -0.24153991043567657, -8.46573748276569e-05, 0.9836826920509338, 0.3266742527484894, 0.6636833548545837, -0.8282070755958557]], [[-1.160972237586975, 0.21627163887023926, -0.9106174111366272, -2.5161049365997314, 0.5398393869400024, -0.36020350456237793, -0.007181776687502861, -0.030748195946216583, 0.6108949184417725, 1.2264699935913086, -1.3305528163909912, 1.0362143516540527, 0.23294328153133392, 0.4087333083152771, -0.7560223937034607, 0.3504479229450226, -0.03530067577958107, 0.6550058126449585, 0.40996378660202026, -0.04618516191840172, -0.5352154970169067, 0.9112318158149719, 1.0865191221237183, -0.1948971003293991, 0.24298006296157837, -0.08069195598363876, 0.22998222708702087, 0.7302045226097107, -2.3566861152648926, -0.25758063793182373, 0.8578508496284485, 0.17317648231983185], [1.4860659837722778, 0.26722076535224915, 1.8948392868041992, 0.00507913064211607, 0.9995149970054626, 0.4924987852573395, -0.08684276044368744, 0.6435702443122864, 1.211272954940796, -0.6560399532318115, 1.34591805934906, 0.028186433017253876, -1.5242692232131958, 2.9368674755096436, -1.0259215831756592, 0.06013292074203491, 2.225801944732666, 0.11462262272834778, 0.6469066143035889, 2.863999128341675, -1.283839464187622, 1.7635692358016968, -0.2781800329685211, -1.227590560913086, -1.7545782327651978, 2.9523088932037354, 0.1172037273645401, 0.6229009032249451, 1.057234287261963, 0.04019925370812416, -0.318083256483078, -1.2691950798034668], [-0.8779951333999634, 0.7249729037284851, -1.3902769088745117, -1.1644091606140137, -1.2362397909164429, 0.9193994402885437, 0.08983777463436127, 0.1491224765777588, -1.108493447303772, 0.1257920116186142, 1.2696189880371094, -0.20543010532855988, 0.20391301810741425, -0.5571600198745728, 0.511997640132904, 0.04578807204961777, 2.4752213954925537, -1.0134333372116089, 0.1616765856742859, -0.750640869140625, -0.061712004244327545, -0.6431711316108704, -1.1524447202682495, -0.13732072710990906, 0.49803152680397034, -0.5385473370552063, -0.6769018173217773, -0.2497028112411499, -0.4589681923389435, 1.838341236114502, -1.4097988605499268, -0.5296518206596375], [1.3043025732040405, 0.5458295941352844, 0.4529445171356201, 0.3791705071926117, 0.12409378588199615, -1.0786657333374023, -0.34543031454086304, 2.065190076828003, 0.7623035311698914, -0.1299009919166565, -0.1550869643688202, -0.1772296279668808, -0.09871280938386917, 0.13235200941562653, 1.1701672077178955, 0.02683110721409321, 0.48961561918258667, -0.9292558431625366, -1.2110590934753418, 1.0885915756225586, -0.4165976345539093, 0.01919308677315712, 0.6675660014152527, -0.7370855212211609, -0.838973343372345, -0.4344198703765869, 0.6993136405944824, -0.6524580121040344, 2.0415403842926025, 0.6420931816101074, 0.020960023626685143, 0.4094410240650177], [-0.10909872502088547, -1.409937858581543, 0.5832210779190063, -0.06111770123243332, -1.0216883420944214, -0.5013129115104675, -0.18175330758094788, -0.40844956040382385, -0.06520532071590424, 0.16794715821743011, -0.20510324835777283, 1.315711498260498, 0.7446864247322083, -0.3386310040950775, -0.9307876229286194, -1.7070415019989014, -1.0471620559692383, 0.27028796076774597, 2.2793850898742676, 0.18540935218334198, -0.08024664223194122, 0.3184097707271576, 0.6594198942184448, -0.03109881281852722, 1.13486647605896, -0.8308833241462708, 1.1550532579421997, -2.202639579772949, -1.036415696144104, -1.2174080610275269, -2.0430386066436768, 0.9622345566749573], [-1.5232951641082764, 0.39420086145401, 0.30792853236198425, 0.2091905027627945, -0.7553526759147644, 0.741025984287262, 1.4297270774841309, 0.1958833634853363, 1.0548709630966187, -0.9160282611846924, -0.8305845856666565, 0.34210869669914246, 0.8956710696220398, -0.8883693814277649, 1.4822838306427002, -1.513780117034912, 0.0577949695289135, -0.7977644801139832, -0.711233377456665, 0.24004900455474854, -1.3411892652511597, 0.31915542483329773, 1.8682470321655273, 0.6827621459960938, -0.3076542913913727, 3.170396089553833, -0.1147570013999939, -1.564997911453247, -0.4420783519744873, 0.2414655089378357, 0.5799600481987, 0.16074801981449127], [-0.7377521991729736, -0.1729888916015625, -1.1312443017959595, -0.11943813413381577, -1.5209025144577026, 0.7384672164916992, 1.7686840295791626, -0.6111705899238586, 0.7154211401939392, 0.1517621874809265, -0.4733530580997467, 0.46972960233688354, -0.9557561278343201, -0.0582764632999897, 0.17743916809558868, -0.5506517291069031, 0.9006099104881287, 0.7569592595100403, 0.7210062742233276, -0.18894904851913452, 0.7449531555175781, -0.06632246822118759, -1.0199123620986938, -0.20973987877368927, -2.003995180130005, 1.5210708379745483, 0.20958766341209412, -0.811739444732666, 0.7594450116157532, 0.5994517803192139, -1.9019074440002441, -1.9771385192871094], [-0.05645088106393814, -0.49612507224082947, 0.2522565424442291, 1.3120965957641602, -0.8272566199302673, -0.5452042818069458, 1.578913688659668, 0.04507927969098091, 0.2802060842514038, -1.2517890930175781, -0.5730937719345093, 0.7993866801261902, 0.3524940609931946, -0.842199981212616, -0.04131258651614189, 0.16964367032051086, -0.9586501717567444, 1.3787767887115479, 0.06757695972919464, 0.09803559631109238, -1.336308479309082, 0.09691502153873444, 1.4759676456451416, 1.1023849248886108, -1.2793093919754028, 2.264312505722046, 0.21541208028793335, 0.025687597692012787, -0.37514254450798035, 1.3769937753677368, -0.739250659942627, 1.462148904800415], [-0.8218879103660583, 2.2259182929992676, -0.2148926556110382, 1.7507333755493164, 0.945284366607666, 0.3828147351741791, -1.4488592147827148, 0.7784854173660278, -0.8671013116836548, 0.07020776718854904, -0.6537513732910156, 2.209465265274048, 1.2226930856704712, 0.6735546588897705, -3.1218955516815186, -0.6073977947235107, 0.8967666029930115, -1.5116102695465088, -0.35104525089263916, -1.3069278001785278, -1.586488962173462, 0.8809124827384949, -1.8142619132995605, 0.7502723336219788, 1.8262355327606201, 2.1801564693450928, -1.176477074623108, 0.23604151606559753, 0.15592671930789948, 0.07389751821756363, -1.761360764503479, -0.8829479217529297], [1.462884783744812, 0.9101659655570984, -1.0099449157714844, -0.8561080098152161, 0.9657373428344727, 0.18672311305999756, -1.0957225561141968, 0.692212700843811, 1.8061208724975586, -0.7962629199028015, 0.8833699822425842, -0.6503257751464844, 0.36033642292022705, -1.3332489728927612, 0.2099231481552124, -0.9473462700843811, 0.989397406578064, -0.13971930742263794, 0.08488153666257858, -0.33198681473731995, 0.8731229305267334, 1.6216397285461426, -0.08858907967805862, -0.5497695207595825, -0.9469489455223083, -3.0388216972351074, -0.5744065046310425, -1.1508070230484009, -1.3138893842697144, 0.6836387515068054, -1.9132148027420044, -0.9883093237876892], [1.2454676628112793, 0.032678067684173584, 0.523765504360199, -1.492793083190918, -0.6829902529716492, 1.094365119934082, 0.10145539045333862, 0.293148010969162, -1.1199895143508911, -0.10332784056663513, 0.23945046961307526, -0.2304029017686844, -1.019775629043579, -0.026732508093118668, -0.15773048996925354, -1.6220557689666748, -1.5492867231369019, 0.41626766324043274, -1.5188610553741455, 0.283782422542572, -1.5168757438659668, 0.03930637612938881, 1.3758524656295776, -1.1631717681884766, -1.4843077659606934, -0.4114041030406952, 0.8556129336357117, -0.28399208188056946, -1.370302438735962, -0.5847355127334595, 0.09153293073177338, -1.7521687746047974], [-1.027090311050415, -1.50482177734375, 0.819728434085846, 0.5167133212089539, -2.1238059997558594, 0.5982266068458557, 0.49521714448928833, -0.4143354594707489, 0.802270770072937, 0.9621462225914001, 0.5677551031112671, 0.6097889542579651, -0.18590804934501648, -1.8872665166854858, 0.8643592000007629, -0.2231486290693283, -0.86788409948349, 0.4567060172557831, 0.0995321273803711, 0.3301868140697479, -0.12590636312961578, 0.6128317713737488, 0.06909165531396866, 0.6328863501548767, 0.9729226231575012, -0.41991329193115234, 0.7300727963447571, 0.29875728487968445, 1.0407556295394897, 0.2298724353313446, -0.7810695767402649, -0.813336193561554], [0.09972061961889267, 1.0693672895431519, -0.18410536646842957, -1.360409140586853, -1.473671793937683, 0.4605140686035156, -1.080873727798462, 1.3808127641677856, 0.44849228858947754, -0.6371365785598755, 0.9559239745140076, 0.2550911605358124, -2.1128804683685303, -0.08080766350030899, 0.3549371063709259, 0.31333303451538086, -0.9308983683586121, 0.16342325508594513, -1.275734305381775, -0.47967588901519775, -0.37902238965034485, 0.13909263908863068, -0.7624998092651367, 1.5593888759613037, -1.3847311735153198, 1.3343549966812134, 1.2069135904312134, 0.24235700070858002, 0.3516850769519806, 0.3195590078830719, 0.06029798090457916, 0.3718816936016083], [-2.3873608112335205, 1.3762545585632324, 0.32236573100090027, -0.7052441239356995, -1.5132410526275635, 2.4658775329589844, 1.2661737203598022, -0.06316807866096497, -0.776592493057251, 1.3035074472427368, -1.0634808540344238, -1.061649203300476, -0.17430172860622406, -1.2447099685668945, -0.26164743304252625, 2.105032205581665, -0.7597718834877014, 2.7656028270721436, -1.7155441045761108, -1.6015876531600952, 0.5062073469161987, -0.23662276566028595, -1.659076452255249, 0.42511457204818726, 1.728858232498169, 1.2157187461853027, 0.7070371508598328, 1.6867187023162842, -0.9657818675041199, -1.0474679470062256, -0.4028013348579407, -1.8534895181655884], [1.8442425727844238, -0.07419226318597794, 0.6463632583618164, -0.3747304379940033, 0.3105252683162689, -0.22939752042293549, -0.7696878910064697, 0.4780620336532593, 0.6244131922721863, -0.6685665249824524, 0.07977039366960526, 1.8054287433624268, -0.20580366253852844, 0.0434294268488884, 2.503842830657959, 1.352175235748291, 0.4744685888290405, -0.17074811458587646, -0.9455626010894775, 0.40373653173446655, 1.9050673246383667, 1.1733959913253784, 1.2720377445220947, 0.7994369268417358, 1.698679804801941, -1.1106339693069458, 0.037504326552152634, -1.609646201133728, 0.04324258863925934, -1.1880815029144287, -0.6337353587150574, 0.25522562861442566], [0.9258811473846436, 0.07721902430057526, -0.25227341055870056, 0.3271002769470215, -0.5090212225914001, -0.209943488240242, 0.4551093280315399, -0.21698832511901855, 1.268980622291565, 0.04141960293054581, 1.2385003566741943, -1.5306522846221924, -0.9748092293739319, 0.4545139968395233, -2.156651735305786, -0.4623170793056488, 0.6872140169143677, 1.3001192808151245, 1.364195704460144, 0.7381541132926941, -1.0018945932388306, 0.8675530552864075, 1.0582244396209717, 0.011972151696681976, 0.49959346652030945, 0.18991999328136444, 0.4922938644886017, 1.294205904006958, -0.8837810754776001, -1.2347140312194824, 0.960552453994751, 0.5781009793281555], [-0.44153085350990295, 0.0678737610578537, -1.6276737451553345, 0.5223215818405151, -0.8129712343215942, 0.4286859631538391, 0.21299760043621063, 0.9595838189125061, -0.01588977873325348, 1.1607743501663208, 0.17497380077838898, -1.0086009502410889, 1.0408350229263306, -1.2260799407958984, -0.022985730320215225, 0.8831597566604614, -0.8303647637367249, 0.9224768877029419, 0.8817298412322998, -0.6925384402275085, -1.9004745483398438, 0.698133111000061, 0.5564383268356323, -0.48624783754348755, 0.6757519841194153, 0.012360876426100731, 1.2901358604431152, -1.1074669361114502, -0.7176918387413025, -0.9443975687026978, 1.4766193628311157, -0.6236874461174011], [0.4394720494747162, 0.639683187007904, -1.808194637298584, 0.6607192754745483, 0.9140229821205139, -0.5511003732681274, -1.7388665676116943, -0.0918794646859169, 1.516405701637268, 1.371987223625183, 1.8491021394729614, -0.127865269780159, 0.2714875042438507, 0.494707852602005, -0.34066423773765564, 0.22568698227405548, -0.5642655491828918, -0.05165346339344978, 2.133312940597534, -0.26524752378463745, 0.15809722244739532, -0.20224140584468842, 0.813812792301178, 0.04851551726460457, -0.4028855264186859, -1.2139604091644287, -0.7083487510681152, -0.3196919858455658, 1.6002804040908813, -1.3812458515167236, 1.116766333580017, -1.9735522270202637], [0.6687839031219482, 0.1869923621416092, 1.4967914819717407, 1.2676783800125122, -0.7104217410087585, -0.6557543873786926, -0.5079914331436157, -0.07847519963979721, -0.12713588774204254, 1.1076204776763916, 1.6068792343139648, 0.46339336037635803, 0.45767277479171753, 1.6780405044555664, 0.6438630223274231, 0.7185418009757996, -2.3438854217529297, -1.8764305114746094, -0.22529663145542145, -0.618963897228241, 1.3098567724227905, 0.2756754159927368, 0.4854365885257721, 0.8228014707565308, -0.2361525446176529, -0.6992337107658386, -1.9448567628860474, 1.2716559171676636, -1.4396997690200806, 1.9150792360305786, 1.6115233898162842, -0.2873857915401459], [0.2039080560207367, -0.7186628580093384, 0.11426995694637299, 1.2566018104553223, -1.597221851348877, 1.4631946086883545, 0.027002427726984024, -1.266187071800232, 0.003574547590687871, -1.2695449590682983, -1.6244481801986694, -0.4369075298309326, -0.7345224022865295, -0.02060067467391491, 0.16205234825611115, -1.8529874086380005, -1.4548289775848389, -0.0005960540729574859, -0.962429404258728, 2.1410162448883057, 0.41038212180137634, 0.7030314207077026, -1.0148200988769531, 2.0925889015197754, 0.17732593417167664, -1.2724483013153076, 0.26347532868385315, -1.2637848854064941, -1.1853129863739014, 0.6038423776626587, -1.303541898727417, -0.10677041113376617], [-2.2127833366394043, 0.7252811789512634, 0.4307262599468231, 0.4006360173225403, -0.2614075243473053, 0.04177151247859001, 1.0949242115020752, 0.06721005588769913, 2.2304084300994873, -0.420184463262558, -1.0469540357589722, -0.9209458231925964, -0.3869688808917999, 2.534012794494629, -1.124855875968933, -1.200812578201294, 1.929918646812439, 1.5379374027252197, -1.9356606006622314, -1.387266993522644, 0.5542568564414978, -0.7283985018730164, -0.6911291480064392, -1.1948308944702148, -1.185038685798645, 0.07369314879179001, 1.3679509162902832, 0.4935159981250763, 1.1854294538497925, -0.36793872714042664, 0.9544819593429565, 0.46334558725357056], [0.47471311688423157, -0.055051036179065704, 0.21805717051029205, 1.8258016109466553, 0.13065294921398163, 0.6108136177062988, -0.8952620029449463, 1.7842607498168945, 0.8621631264686584, -1.5252231359481812, -0.34690192341804504, 0.3856084644794464, -1.5880591869354248, -0.15772663056850433, -0.0031056071165949106, 2.168593168258667, -0.1830163300037384, -0.8821423649787903, 1.3635681867599487, -1.172318458557129, 0.03838932514190674, -0.22832977771759033, 0.6794785857200623, 0.08293797075748444, 0.417285293340683, 0.07445606589317322, -0.9827186465263367, 2.2770683765411377, 0.5044338703155518, -0.48617517948150635, -2.6482040882110596, 1.094690203666687], [-0.33657652139663696, -1.9812132120132446, 0.2812120318412781, 2.0034396648406982, 0.4128955602645874, -1.245408058166504, -1.1520377397537231, -0.5798438191413879, 0.8010992407798767, -1.3039830923080444, 1.3731788396835327, -0.8447923064231873, 2.1533985137939453, -0.46076685190200806, -1.1643835306167603, -1.3390196561813354, 0.1716478019952774, 1.0362398624420166, 0.09615465998649597, -0.11438190937042236, 2.315084218978882, 0.21308697760105133, 0.2593190371990204, 0.11353127658367157, -1.9703532457351685, -1.1373955011367798, 0.6065428256988525, 2.7170259952545166, 1.2021980285644531, -2.3738718032836914, -0.5034024119377136, 0.3546159565448761], [-0.3793729543685913, 0.6748352646827698, 0.4502761662006378, -0.9329554438591003, 0.46883371472358704, -1.4708032608032227, 0.23108865320682526, 0.9735708832740784, 0.48742741346359253, -0.8421452641487122, -0.681759238243103, 0.7841552495956421, -0.25987377762794495, -0.8080251216888428, -1.09831702709198, 0.7262656688690186, -0.35583218932151794, 0.8050122261047363, -1.409009337425232, 1.2853562831878662, 0.010546054691076279, 0.5067903399467468, 1.7303977012634277, -2.4075396060943604, 0.908206582069397, -0.149329274892807, -0.12722226977348328, 0.49658098816871643, 0.34338322281837463, -1.2551487684249878, 0.30764949321746826, 0.22837531566619873], [-0.6674877405166626, 0.5793856978416443, -0.4509735405445099, -0.32295751571655273, -1.1000255346298218, -0.32777097821235657, -0.5359549522399902, -0.9056559205055237, -0.9990221261978149, 0.6372968554496765, -1.8087077140808105, 0.18086358904838562, 0.8587813377380371, 1.2051711082458496, -0.13390667736530304, 0.21146732568740845, 0.8249237537384033, -0.7285251021385193, -1.7191264629364014, -0.7940554022789001, 0.26519110798835754, 0.2747499644756317, 0.3782491385936737, -0.49296265840530396, -1.6709015369415283, 0.3480106294155121, -0.8744136095046997, 0.3197958767414093, -0.5055860280990601, -0.2197054624557495, -1.1446831226348877, -0.1638588011264801], [0.12695878744125366, -0.12899595499038696, -0.5278947949409485, 0.46925854682922363, 0.07369601726531982, 1.5503164529800415, -0.23107996582984924, 1.09092116355896, 1.035366415977478, -0.3511843681335449, -1.908132553100586, 0.9677557349205017, 0.0684335008263588, 0.3964579403400421, 0.3460429310798645, -1.3316586017608643, 0.903107225894928, -0.7145670652389526, -0.6268275380134583, 1.8983032703399658, 0.17386667430400848, -0.02152487076818943, 0.4631805419921875, 0.21467533707618713, -0.8794466853141785, 1.0095099210739136, -0.07500457763671875, -1.1925609111785889, 0.27744346857070923, 0.9282866716384888, -0.750649094581604, -1.2617026567459106], [0.21660925447940826, -0.23116721212863922, -0.24211357533931732, -2.066525459289551, 1.520714282989502, 2.128586530685425, -0.6178485751152039, 0.9234476685523987, 0.9034806489944458, -1.0814225673675537, -1.0957344770431519, -1.1173145771026611, -1.932569146156311, 0.750872015953064, -1.0263779163360596, 0.3184364140033722, -0.3825145363807678, -0.024766990914940834, -0.7323412895202637, -0.3746274411678314, 0.8421679735183716, -0.5723891854286194, 1.7780719995498657, 1.1019487380981445, 1.9541770219802856, 1.239719033241272, 0.49923864006996155, -0.683811604976654, 1.023815631866455, -0.6466386318206787, -0.8024542331695557, -0.5551307797431946], [0.46389511227607727, -0.7007938027381897, 2.2001729011535645, 0.41286346316337585, 0.8122349977493286, 1.5047364234924316, -0.7754657864570618, -0.4201190769672394, -0.6911383867263794, 1.7087770700454712, -0.47297579050064087, 0.4190773367881775, -0.5553425550460815, 0.5027146339416504, -1.0155001878738403, 0.8312976360321045, -1.0765800476074219, -1.722928524017334, 0.03542468324303627, 1.8223470449447632, 2.068087100982666, -1.5398014783859253, 0.8906332850456238, -1.058065414428711, -1.3406645059585571, 2.1035282611846924, 0.0663699135184288, 2.4145331382751465, 0.7829437255859375, -0.3632832169532776, 0.5434998273849487, -0.409580796957016], [-1.5385326147079468, 0.35406753420829773, -0.4872795343399048, -0.5158228874206543, -1.4261810779571533, 0.495648592710495, -0.1189393699169159, 0.39107444882392883, 1.0994349718093872, -0.6964942812919617, -0.3713575005531311, 1.3095089197158813, 1.1205053329467773, -0.07724293321371078, -0.2601107656955719, 0.506851077079773, -1.2683740854263306, 0.662855863571167, 0.352099746465683, -0.05105764791369438, -0.7295470237731934, 1.3703763484954834, 1.2697633504867554, 0.3959130346775055, 0.7709853053092957, 0.22141502797603607, 1.5227618217468262, 1.0099982023239136, 0.464171826839447, 0.8899874687194824, -0.9801260232925415, 0.05556826293468475], [-0.6171790957450867, -0.8967196345329285, 0.055324915796518326, 1.0161595344543457, 0.15186789631843567, 0.09747564792633057, -0.49389952421188354, 0.8383477926254272, 1.5512666702270508, -1.100848913192749, -0.15475721657276154, -0.0452452078461647, -2.154688835144043, 1.7516101598739624, 0.16506235301494598, 1.1545734405517578, -0.5199921131134033, -0.917401909828186, 0.35041555762290955, 0.46354007720947266, 1.483558177947998, 2.429048538208008, -0.32295283675193787, -0.15575824677944183, 0.4192110300064087, -0.007037133444100618, -0.22669750452041626, 0.9069482088088989, -1.2653818130493164, -1.5313687324523926, 0.5009651780128479, -0.42642638087272644], [-1.1821006536483765, 2.234144926071167, 0.1360936164855957, 1.020979404449463, -0.7544276118278503, -1.5328956842422485, -0.31556040048599243, 1.3216198682785034, 1.0959272384643555, 0.433580219745636, 0.04350811243057251, -1.6001743078231812, -0.4135427474975586, 1.411145567893982, 0.516325831413269, -2.098597526550293, 0.9899717569351196, -0.5905952453613281, 0.2705238461494446, -0.5177340507507324, -0.13369016349315643, 1.2654716968536377, 0.9515307545661926, 0.41704756021499634, -0.020140958949923515, -1.2813235521316528, -0.9425931572914124, -0.12222856283187866, 0.2907414138317108, -1.066990613937378, -0.7778140306472778, -0.3363957703113556], [-0.5390997529029846, -0.4141084849834442, 1.1849994659423828, 0.16675172746181488, 0.27896809577941895, -0.9899389743804932, 0.25544241070747375, 1.4302279949188232, 0.09206252545118332, -0.9852389693260193, -0.08675282448530197, 0.7945486903190613, 1.0810599327087402, -2.0345542430877686, -0.3129176199436188, -0.8533888459205627, -1.162939190864563, -0.06490594893693924, -0.10232176631689072, 0.7198451161384583, 0.7437376976013184, -1.1440602540969849, -0.20269332826137543, 1.2334009408950806, -0.8549003601074219, 0.5357237458229065, 0.6854962110519409, -0.8518987894058228, 0.44219622015953064, 0.6914545893669128, 1.7848237752914429, 1.5504730939865112], [0.952042818069458, -0.7920490503311157, -2.4885852336883545, -0.7929884195327759, 2.358130693435669, -0.0775451809167862, -0.7020515203475952, -0.7743642330169678, 0.9487954378128052, 0.6243767738342285, -0.8803575038909912, -0.9040034413337708, -1.4996219873428345, 1.8767194747924805, 0.498056024312973, 0.033555060625076294, 0.6810861825942993, 0.6058643460273743, -1.2830729484558105, -0.8118281364440918, 0.007108672056347132, -0.8389776349067688, -0.37295687198638916, 0.4590243697166443, -0.6092609763145447, -0.04485000669956207, 0.15786442160606384, -0.6828593611717224, 1.2336963415145874, -0.5351903438568115, -1.2092461585998535, -0.35278141498565674], [1.08747136592865, -0.9792037010192871, 1.172592282295227, -0.48736271262168884, 1.023165225982666, -0.553049623966217, -0.9040979743003845, -0.11873392015695572, -0.12688395380973816, -1.3722853660583496, -0.04086841270327568, -0.19764360785484314, -0.41889768838882446, 0.4289247989654541, -0.8408551812171936, 1.7610112428665161, -0.0479191429913044, 1.0595617294311523, 0.5186015963554382, 0.5757139325141907, -0.25000107288360596, -1.2968711853027344, 0.562286913394928, -0.6328004598617554, -0.3490341901779175, -1.7626755237579346, 1.0787619352340698, 0.4761882424354553, 0.2673538029193878, -0.24255718290805817, -1.5538774728775024, 0.679872453212738], [-0.48406776785850525, -0.17649070918560028, -0.3792050778865814, 0.757969319820404, 0.9733757376670837, 1.1006717681884766, 0.4438783526420593, -0.3172665536403656, 1.569265604019165, 0.3725915253162384, -0.02195565588772297, -0.5623072385787964, -0.9116818308830261, -1.1818722486495972, 0.6509290337562561, 0.2318018078804016, 0.21499323844909668, 0.20849037170410156, -0.618190348148346, 0.4330258369445801, -0.7249557971954346, 2.0298874378204346, 0.6254863739013672, -0.009052014909684658, -0.12518839538097382, -1.5218747854232788, 1.2028788328170776, 0.6627123951911926, -0.1905641257762909, -0.8219707012176514, -0.6880034804344177, -1.0294547080993652], [-0.27976110577583313, -0.5983132123947144, -0.8622349500656128, 0.10342580825090408, -0.0807257890701294, 2.32489013671875, 2.008206367492676, 0.9221370220184326, 2.4954802989959717, -1.387641191482544, -1.0419163703918457, -1.3843035697937012, 0.6353391408920288, -1.2222257852554321, -0.16204072535037994, 0.44920143485069275, -0.07107382267713547, -0.6143817901611328, 1.3192412853240967, -0.09912170469760895, 0.39665332436561584, 0.6989766955375671, -0.42346617579460144, 1.0661896467208862, -1.7464474439620972, -0.10224995017051697, 0.47503572702407837, -0.6950427889823914, 0.7180564999580383, -0.05208471789956093, 0.11567029356956482, 0.39593127369880676], [-0.810566782951355, 1.7690824270248413, -0.6777644753456116, -1.0397409200668335, -0.8975039124488831, 0.0371539406478405, 1.5579220056533813, -0.4233733117580414, -1.766568899154663, -0.10796213895082474, -0.2950795292854309, -0.8111677765846252, 0.2874963581562042, 2.0101897716522217, -0.9285228252410889, 1.4772638082504272, 0.21066972613334656, -0.3845251500606537, -0.7130046486854553, 0.14837202429771423, -1.4396905899047852, 0.29890644550323486, -0.2989567518234253, 0.3440183401107788, -2.0190823078155518, -0.36210373044013977, -1.230857253074646, -1.9865409135818481, 0.21141456067562103, 0.9368591904640198, 0.5520452260971069, 0.8310173749923706], [-0.30522212386131287, -0.7387053966522217, 1.6603034734725952, -1.3796567916870117, -1.9634264707565308, -2.0862252712249756, -0.2951396405696869, -0.59079909324646, 0.6454905867576599, -0.8543146848678589, -0.8542230129241943, 0.43254923820495605, 1.5732029676437378, 1.0115861892700195, -1.0138649940490723, -0.9758750200271606, -1.4701573848724365, 0.16018617153167725, -0.040573179721832275, -0.376144677400589, 0.04082430526614189, -1.2197130918502808, 1.1441056728363037, -0.6282225251197815, -2.0330331325531006, -0.3380727469921112, -0.38998091220855713, -0.6346848011016846, 0.9565277099609375, 1.0029889345169067, -0.8786844611167908, 0.8262885212898254], [-0.08188726752996445, -0.41795074939727783, -0.7932062745094299, -1.0205668210983276, 1.0094951391220093, -0.5869789123535156, 0.678070604801178, -0.11347319930791855, 0.7594063878059387, 0.985849916934967, -0.7170910239219666, 0.7654742002487183, -0.24843044579029083, 1.7155731916427612, -0.6422243714332581, 0.28734636306762695, 0.5072784423828125, 0.4461323916912079, 0.013628906570374966, -0.47591692209243774, -1.1694388389587402, 0.481180340051651, 0.15025345981121063, 1.4180241823196411, 0.5058949589729309, -0.5153346657752991, -0.6047289967536926, -0.005247096996754408, 1.1924692392349243, 0.614588737487793, -0.6513325572013855, 0.3329048454761505], [-1.0265376567840576, -0.5220299363136292, 0.3281688392162323, 1.2674509286880493, 0.2651129961013794, -0.2414221465587616, -0.39584165811538696, -0.3746558725833893, 0.5943777561187744, 0.15057465434074402, 1.2694449424743652, 0.16463367640972137, 0.7263674736022949, -1.1706280708312988, -0.7386935353279114, -0.19932284951210022, 1.5629212856292725, -0.29860231280326843, 1.4126969575881958, 0.6310936212539673, 1.975806474685669, -0.27591636776924133, 1.018352746963501, -0.015942169353365898, 1.1270356178283691, -0.44489333033561707, -0.9938929677009583, 0.35497552156448364, 2.690582513809204, 0.2600962221622467, -0.11922581493854523, 0.9131779074668884], [1.4066376686096191, 0.6938364505767822, 1.6343903541564941, -0.46801289916038513, 0.14660492539405823, -1.0250656604766846, -0.34579893946647644, -0.7405830025672913, 0.8856383562088013, -1.0179044008255005, 0.9144312143325806, -0.6120162010192871, 1.4221075773239136, -0.8696024417877197, -0.3795270323753357, -0.31104207038879395, 0.9249821901321411, 1.3403663635253906, 0.0056580728851258755, 0.16993595659732819, 0.3063691258430481, -0.34505486488342285, 0.29783156514167786, -1.9188169240951538, 0.3017134964466095, 1.1187715530395508, -0.0969625785946846, -2.1790473461151123, 0.203815758228302, -0.7477149963378906, 0.4749161899089813, -1.3083217144012451], [-0.07449202984571457, 1.2779455184936523, -0.4746231138706207, -1.6834899187088013, -1.8674410581588745, -1.315998911857605, -0.6853309273719788, -0.32188576459884644, -0.27802446484565735, 0.9886625409126282, -2.325385093688965, -0.7878074049949646, -1.9986752271652222, -0.6623749732971191, 1.0905661582946777, -0.027236269786953926, -0.1451551765203476, 0.194471076130867, 1.2369122505187988, 0.41288095712661743, 1.6313176155090332, -0.5076408386230469, 3.3915398120880127, 1.7729955911636353, -1.0976262092590332, -0.013292133808135986, 0.37889862060546875, 0.5842434763908386, -1.4191029071807861, -0.4818800985813141, -0.8923570513725281, 0.6037595272064209], [0.6229597330093384, 1.7714955806732178, 0.8276659846305847, 0.9749020338058472, 0.5439630150794983, -0.46513596177101135, 0.39656779170036316, -0.4092487692832947, -0.33498311042785645, -0.5223236680030823, -0.306550532579422, -0.9295787811279297, -0.8274264931678772, 0.0588713139295578, 0.25688809156417847, -1.0370508432388306, -1.941820740699768, -0.4541237950325012, -0.19668373465538025, -1.6041854619979858, 0.40739676356315613, 0.27804407477378845, 0.39322224259376526, -0.5021377205848694, 0.04372219368815422, -0.7458622455596924, 0.7868167757987976, -0.27747729420661926, 0.09738459438085556, -0.6151397824287415, 0.7197486758232117, 0.08210621029138565], [-2.5988030433654785, 1.1178637742996216, -1.6554815769195557, -0.8535523414611816, 1.5510448217391968, 2.248095750808716, -0.19950665533542633, 1.7770582437515259, -0.15241937339305878, 1.3083851337432861, 0.5806679725646973, -0.29971182346343994, 0.3073023557662964, -0.5477265119552612, -0.7430710792541504, -0.39168018102645874, 0.11039406806230545, 0.11164405196905136, 0.11560355126857758, 0.7290616631507874, -0.937907874584198, -0.5662071108818054, -0.20142488181591034, 0.3109149634838104, 0.46103814244270325, -1.111672282218933, -0.06497303396463394, -0.6858597993850708, 0.23899324238300323, 0.9135720133781433, -1.8275396823883057, -1.8902307748794556], [-0.412526935338974, -0.07464375346899033, -1.2439574003219604, -0.6081591844558716, 1.5556234121322632, -0.2045201063156128, -0.6196132898330688, 0.6898490786552429, 0.14411616325378418, -0.39797234535217285, 0.07720726728439331, 1.7307754755020142, -0.38480260968208313, 0.8778324723243713, 0.19560173153877258, 2.22188401222229, -0.3293342590332031, 0.2630901634693146, 0.6771577000617981, -0.7124027013778687, 0.8024610280990601, 0.40559184551239014, -1.059984564781189, 0.6902820467948914, -1.2735763788223267, 0.2764832079410553, -1.0333055257797241, 0.43073198199272156, -0.4493750035762787, 0.4603687822818756, -0.5023434162139893, -1.1299830675125122], [1.2587093114852905, 0.6879462599754333, -0.3951820135116577, -0.08871660381555557, -0.5628088712692261, 0.15543977916240692, -0.049800336360931396, 0.6213781237602234, -1.3310003280639648, -1.6319897174835205, -0.6097842454910278, -0.21924568712711334, -0.17093481123447418, -0.08197304606437683, -0.6516767144203186, -0.5191516280174255, -0.9946691989898682, 0.6173650622367859, 1.5018384456634521, 2.055607557296753, -0.13802658021450043, 1.2184637784957886, 0.962927520275116, 0.3592472970485687, 0.8224541544914246, 0.6504451036453247, 1.835792899131775, -0.25296956300735474, 0.23870305716991425, -0.8941618204116821, 0.6452321410179138, 0.12266133725643158], [0.3300304412841797, -0.6471491456031799, 1.6605385541915894, 0.1835137903690338, -0.0891185849905014, 0.9579882025718689, 1.4095860719680786, -0.22247296571731567, -0.16519387066364288, 0.20594336092472076, 0.7103984951972961, 1.2550910711288452, -2.0690817832946777, -1.5013707876205444, -1.980495572090149, -0.24503132700920105, -1.3154088258743286, 0.5142807364463806, -0.661880373954773, 0.5231135487556458, -0.5044698715209961, -1.750352382659912, -0.8951860070228577, -0.08956584334373474, 0.5053697824478149, -0.8366103768348694, -0.45271632075309753, 2.4464848041534424, 0.45340147614479065, -0.8695275187492371, 0.8757634162902832, -1.5569230318069458], [1.1865845918655396, 0.6344082951545715, -0.1617000550031662, 0.47042012214660645, -1.9262694120407104, -0.7846440672874451, 0.23308677971363068, -0.6464636325836182, 0.8610295057296753, 0.5683885216712952, 0.6170763373374939, 0.8156485557556152, 0.2954612076282501, 0.40326154232025146, 1.119286060333252, 0.47628167271614075, 0.5150174498558044, 0.4898911118507385, -0.13128067553043365, 0.9001479148864746, 0.5715101957321167, -0.7952935099601746, 0.8331786394119263, 0.5125935673713684, -1.8223614692687988, 1.5234637260437012, 0.7425864934921265, 1.3288533687591553, 0.0022316290996968746, -0.34899407625198364, -1.1057966947555542, -1.2403498888015747], [1.0029700994491577, -0.6679309010505676, -0.12037177383899689, 0.1433548629283905, 1.1856210231781006, 0.9267095923423767, 0.4124205410480499, -1.6525940895080566, 0.9823572635650635, -1.839978814125061, 0.9566845297813416, 2.50765061378479, 2.482956647872925, 0.4193361699581146, -0.013586525805294514, 0.2531755566596985, -0.962336003780365, -0.946651816368103, 1.5811874866485596, -0.0988679751753807, 0.05559087544679642, -1.243053674697876, -1.23032546043396, -0.49925121665000916, 0.2991774380207062, 0.32233795523643494, 0.45582717657089233, -0.044486332684755325, -0.5145625472068787, -0.7371697425842285, 0.427013635635376, 0.7145277261734009], [2.2891132831573486, 0.26087287068367004, 0.5261791944503784, 0.9296374917030334, -0.2448090761899948, -0.7571375370025635, -0.5715674161911011, 0.6241964101791382, -0.3172259032726288, -0.1559751182794571, -0.2991018295288086, 2.0447096824645996, -1.0285433530807495, -1.2822777032852173, 0.9455842971801758, -1.0761512517929077, 2.386784791946411, -0.533780038356781, -1.0993014574050903, -0.7922462821006775, -0.5921096205711365, -0.2852739691734314, -2.277801990509033, -0.6537380814552307, -0.3928283751010895, 0.052607886493206024, -0.17376559972763062, 0.46747317910194397, -0.08344584703445435, -0.0032996779773384333, 0.014797983691096306, -2.3768887519836426], [-0.064417265355587, 1.1982113122940063, -0.8168403506278992, -0.1260286122560501, 0.23446336388587952, -1.0455870628356934, -1.1423556804656982, 1.6720794439315796, 1.3076030015945435, -0.4235510230064392, -0.4544419050216675, -1.892180323600769, -1.856103539466858, 1.1540004014968872, 0.3493160307407379, 0.22566437721252441, 1.0123941898345947, 0.1720803678035736, -1.447510838508606, -0.3626377284526825, -0.3858499825000763, -2.020413875579834, 0.07962074875831604, -0.11339586973190308, 2.3916237354278564, 1.0018548965454102, -0.34760478138923645, 0.5247368216514587, -2.000389575958252, -1.3554880619049072, -0.7432737946510315, 0.9192044138908386], [0.40804338455200195, -0.07932484149932861, 1.0708942413330078, 0.5791569948196411, -1.1745606660842896, -2.0004594326019287, -0.4553985297679901, 0.5159437656402588, 0.12835904955863953, -1.5832043886184692, -0.26613253355026245, -1.7084643840789795, -1.3317254781723022, 1.2208682298660278, -0.2676984667778015, -1.1080405712127686, 0.4265044629573822, 1.6943997144699097, 1.4260131120681763, 0.6164180636405945, 0.14927439391613007, -1.9364337921142578, -0.6566801071166992, -0.34927278757095337, -0.479512095451355, -0.6387869715690613, -2.044602155685425, -1.4714020490646362, 0.0013547433773055673, -0.7036876082420349, -0.2750527858734131, -0.5839908123016357], [-0.8536421060562134, -1.0514522790908813, 1.7334579229354858, -0.7534340620040894, -0.6112151741981506, 0.9540044069290161, 0.45936447381973267, 0.7160902619361877, 0.2986631989479065, -0.39352503418922424, -1.3587054014205933, -0.2086343914270401, -1.4265098571777344, -0.7556148171424866, -2.0052671432495117, 0.6078655123710632, -1.5205482244491577, -1.2492221593856812, -0.7373173236846924, -0.04919282719492912, 0.17529785633087158, -1.0987274646759033, 0.8658286929130554, 0.06325583159923553, 0.8766634464263916, -0.5959438681602478, 0.08836261183023453, -1.9203627109527588, 0.591874897480011, -1.9804357290267944, -0.09140338003635406, 1.071889877319336], [-0.822795033454895, -1.0795115232467651, -0.9454615116119385, 1.0792474746704102, -0.11614218354225159, 2.2045023441314697, 0.1724858582019806, 0.10225389152765274, -0.38137346506118774, 0.031926486641168594, 0.6283490061759949, 0.8478274941444397, 1.2634835243225098, 0.0728975310921669, 1.0471994876861572, 0.02505951188504696, -1.2077281475067139, 1.6606636047363281, 1.1099920272827148, 0.07268135994672775, 0.6528176665306091, 0.5192611813545227, -0.7789990305900574, 0.004209316801279783, 0.5107542276382446, -1.4512403011322021, 1.6662213802337646, 0.32440468668937683, -0.5492811799049377, -1.1504194736480713, 0.0841141864657402, 1.9752936363220215], [-0.06626523286104202, 2.276042938232422, 0.9529306888580322, 0.6263500452041626, 0.7290124893188477, -1.0935074090957642, 0.45822659134864807, 0.4994790554046631, -0.22715556621551514, 2.753129243850708, -1.2481231689453125, -0.18227171897888184, -0.6784999370574951, -0.6347852349281311, 0.3774901330471039, 1.1165128946304321, -0.11574982106685638, -0.19066432118415833, 0.37442630529403687, 1.347497582435608, -1.1054316759109497, 0.006042588502168655, 0.6709321737289429, -0.38621410727500916, -1.0565438270568848, -0.49183332920074463, -2.0640387535095215, 0.18487264215946198, 0.61216801404953, -1.4353017807006836, -0.9619702696800232, 1.6305011510849], [0.6042764782905579, 0.7464104890823364, -0.040168073028326035, 0.5759369134902954, 0.17942994832992554, 0.9813398718833923, 0.03928504139184952, 0.6010880470275879, -0.03278756141662598, 1.2583705186843872, -0.6879632472991943, -0.2964279055595398, 0.2082747370004654, 1.3749289512634277, -0.5323280692100525, -1.3286755084991455, 0.11604851484298706, 1.0925827026367188, -0.20532368123531342, 0.6477003693580627, -0.9170101881027222, -0.6108093857765198, -0.5217986106872559, -1.2099226713180542, -2.32267165184021, 0.3610471785068512, 1.3823069334030151, 0.0021160796750336885, 0.26198989152908325, -0.6861627101898193, 0.06844125688076019, -1.396129846572876], [-0.11219656467437744, -1.4412407875061035, -0.3682823181152344, 0.9482772350311279, 0.001881228992715478, 0.3328437805175781, 2.6890788078308105, 1.7569429874420166, 0.25665852427482605, -0.011037302203476429, -1.6372499465942383, 0.752037525177002, 1.119753122329712, -0.9348852634429932, 0.7267623543739319, 1.3075878620147705, -1.6676561832427979, 0.17201180756092072, 0.2782815396785736, -0.7056634426116943, 1.9468990564346313, 1.1908345222473145, 0.5433870553970337, 2.4615986347198486, -1.5366203784942627, -1.712013602256775, -0.5605461001396179, 1.377021074295044, -1.5458853244781494, -1.538312315940857, 2.3588948249816895, -0.4002431333065033], [-0.047634657472372055, -0.16781075298786163, -3.1895833015441895, 0.18850275874137878, 0.3045883774757385, -0.18789063394069672, -0.07093043625354767, 0.06492701172828674, -0.8050820827484131, -0.19223523139953613, 0.34196048974990845, 0.09042982757091522, -1.0788884162902832, -0.4963521659374237, -0.3292309641838074, -1.848686933517456, 0.21924206614494324, 1.6525717973709106, -0.5289410948753357, -1.3592784404754639, 0.047351714223623276, -0.37047243118286133, -0.32230979204177856, -0.43842747807502747, 0.5245561599731445, -0.7440506815910339, 0.9513848423957825, -0.7369256019592285, -2.3426358699798584, -0.48406630754470825, 0.5386787056922913, 2.2006311416625977], [-0.44836053252220154, 0.0776110514998436, 0.40109312534332275, -0.4577501714229584, 0.5532911419868469, 0.6902996897697449, -2.390963077545166, -1.7447882890701294, -0.3084917962551117, 1.0499433279037476, 1.049473762512207, -0.36689522862434387, 0.39537331461906433, 1.4340596199035645, -1.0988727807998657, -1.5109610557556152, -0.7635661363601685, 0.4883502721786499, -0.3561071455478668, 0.5150727033615112, -0.7286236882209778, 0.5477308034896851, 0.7050683498382568, -0.6095656752586365, -0.8597927093505859, 0.5970889329910278, 1.4053031206130981, 0.07245037704706192, -0.6642079949378967, 0.33539071679115295, -1.779198169708252, -1.9499424695968628], [0.33678296208381653, -0.517588198184967, 2.362795829772949, 0.9721435308456421, 1.1334798336029053, 0.19966082274913788, 0.41751450300216675, -0.16177938878536224, -1.5179139375686646, -0.08531462401151657, -0.5459592938423157, -0.47779780626296997, 0.6636409759521484, 1.3561829328536987, 1.8416684865951538, -0.3096485733985901, -0.7837244272232056, 0.2898804843425751, 0.3280928432941437, -0.9966838359832764, 0.6666019558906555, -0.7340785264968872, 1.1101182699203491, 0.15263229608535767, -0.09457436203956604, -0.360870361328125, -1.0258185863494873, 0.7464780807495117, -0.2394116222858429, 1.2656266689300537, -1.2143915891647339, 0.4779680371284485], [-1.5256708860397339, -1.016038179397583, -0.09835053980350494, -0.578618586063385, 0.38116148114204407, -1.133648157119751, 1.9047402143478394, -0.054266516119241714, -1.5165404081344604, 1.276197910308838, -2.1215603351593018, -0.41237854957580566, 0.36625954508781433, 0.4837467074394226, -0.26348790526390076, -1.9034101963043213, 0.3558109700679779, 1.0026605129241943, -1.4540822505950928, 1.973761796951294, -0.1664426475763321, 0.5702120661735535, -1.6101112365722656, -0.8146844506263733, -0.42828476428985596, -0.3841952383518219, -1.2474408149719238, 0.29411232471466064, -0.02711864747107029, 0.6832104921340942, -0.5908443927764893, 0.452376127243042], [-0.5897694826126099, -0.9675794243812561, 0.1453838348388672, 0.36374586820602417, -1.6467503309249878, -0.362539678812027, 0.47750476002693176, 0.2395753562450409, 0.3671944737434387, -0.9820613265037537, -0.6062897443771362, -0.559822678565979, 0.14891116321086884, 0.020218325778841972, 0.08543797582387924, -1.0226420164108276, 0.7634792327880859, 0.7005181312561035, 1.909034013748169, 0.3826126158237457, 1.4639132022857666, 0.7637600302696228, 0.17052499949932098, 0.4300468862056732, -0.9588789939880371, -1.1713387966156006, -1.378757357597351, -0.16184966266155243, 0.2535192370414734, 1.2500851154327393, -0.7588641047477722, 0.34408581256866455], [-0.41392284631729126, -0.9767245054244995, 1.3175400495529175, 0.8128976225852966, -0.2401016801595688, -0.9191856384277344, -0.2890149652957916, 1.9231795072555542, -0.6463940143585205, 0.9180777072906494, -0.7210178971290588, -0.8670187592506409, -0.027440575882792473, 1.4243764877319336, -0.6439211964607239, 0.27526387572288513, 1.7702348232269287, 0.917087972164154, -0.3956356346607208, 0.39172327518463135, -0.4629686176776886, -0.5666924715042114, -1.025118112564087, 1.150830626487732, -0.049476537853479385, -0.22792403399944305, 0.6635265946388245, -0.6155432462692261, 1.094171166419983, 1.369266390800476, -0.07739878445863724, -1.7382128238677979], [-0.8104763031005859, -0.3507961928844452, 0.2290015071630478, -0.01650255359709263, -0.5375254154205322, 0.2564249336719513, -1.5114015340805054, -0.3413161337375641, 0.02892497554421425, -0.49491462111473083, -0.7819815278053284, -0.15190592408180237, 0.30635911226272583, 0.03882373869419098, -0.2322380244731903, 0.22246412932872772, -2.4279236793518066, -0.7104170322418213, -1.1286206245422363, -0.9583737254142761, 1.2866582870483398, -0.6733745336532593, 0.5203955769538879, -0.952364981174469, 0.848937451839447, 0.7810806632041931, 0.6240447163581848, -0.5961446166038513, -1.136501669883728, 0.019911788403987885, -1.8636174201965332, 0.6677452921867371]]]], "dtype": "float32"}, {"__nd__": [[[[-1.5371047258377075, -1.2686132192611694, 0.6744449138641357, 0.2740436792373657, -0.8831652998924255, 0.16563408076763153, -0.8642957210540771, 0.0028528275433927774, -1.1291755437850952, 1.0266000032424927, 1.2561639547348022, 1.3951659202575684, -1.3821978569030762, -1.730888843536377, 0.40075454115867615, -0.6904687881469727, 0.3751736581325531, 1.021026611328125, 0.22254766523838043, 0.40999510884284973, -0.42729803919792175, 1.5265982151031494, -0.8996940851211548, -2.4971888065338135, 2.0652098655700684, 1.323256492614746, -0.8233359456062317, -2.4962241649627686, 0.5853623151779175, 0.22599157691001892, 0.9143296480178833, 0.8638685941696167], [-0.7137227058410645, 0.49229487776756287, 0.2983599305152893, -0.6836933493614197, -0.9061499238014221, -0.5817194581031799, -1.292782187461853, -0.7523001432418823, -0.32724472880363464, 0.9377314448356628, -0.1268705576658249, 0.10175289213657379, 0.4621322751045227, -0.18781325221061707, 0.2463075965642929, 0.7381154298782349, -1.1271955966949463, -0.6620123386383057, -0.9672502279281616, -0.09263307601213455, 0.20734700560569763, 0.7496247887611389, 0.40193209052085876, -0.5882615447044373, 1.8305236101150513, 1.5626541376113892, 0.5666230320930481, -1.2065190076828003, 1.4211643934249878, 1.6124821901321411, -0.9577391743659973, -0.011698917485773563], [0.6676076650619507, -0.29161548614501953, 1.6215978860855103, -0.06747172027826309, 0.517005443572998, -0.1822848618030548, 1.0512086153030396, 0.08041847497224808, -0.04430919885635376, 1.253711223602295, -0.914222776889801, 0.050279341638088226, 0.00903649628162384, 1.5776394605636597, -1.595238208770752, -0.21785388886928558, -1.4149659872055054, -1.0865001678466797, 0.4497518837451935, -1.0753200054168701, -1.289512276649475, -0.1009301170706749, 0.1728096306324005, -0.31427982449531555, 0.7061266899108887, 0.26577985286712646, 0.5178019404411316, -0.6690976619720459, 0.3306649327278137, -0.9968169927597046, -0.6834527850151062, 0.039773307740688324], [-0.9613604545593262, -0.061060234904289246, 1.1134955883026123, -0.009988493286073208, -0.07330066710710526, 0.7589565515518188, 1.2395896911621094, -0.177157461643219, 0.8611419796943665, 0.890587329864502, 0.24770519137382507, -1.2718812227249146, -0.546651303768158, 0.5997049808502197, 1.7869575023651123, 2.461854934692383, 0.456670880317688, -0.3439691364765167, -0.6744892597198486, -1.6815077066421509, 2.2388250827789307, -0.3012012243270874, 0.11034476011991501, 1.5112495422363281, -0.6022894978523254, -1.558754324913025, 2.4730887413024902, 0.7513017058372498, -1.202053189277649, 1.3315553665161133, -1.969736099243164, 1.2507296800613403], [0.8827179670333862, 0.04716521129012108, -0.46206721663475037, -1.6397730112075806, -0.24683265388011932, -1.4410569667816162, -0.5892373323440552, 0.7783672213554382, -0.21298840641975403, 0.0037118184845894575, -0.3455551266670227, 1.942862868309021, -0.8686574697494507, -1.0515854358673096, 0.05859224498271942, 1.6301268339157104, -0.8068117499351501, -0.7053261399269104, -1.3072696924209595, 0.830328106880188, 0.9464598894119263, -1.3815233707427979, 1.210801362991333, -0.7780870199203491, 0.6610997915267944, 0.4290289282798767, -1.0237284898757935, -0.3858586847782135, 0.49018529057502747, -0.6834075450897217, -0.9199208617210388, 0.20143137872219086], [0.5657734274864197, -0.8123696446418762, 0.7578520774841309, 0.6391189694404602, 0.8868009448051453, -1.109415054321289, -0.7553274631500244, 0.003356381319463253, -1.245443344116211, 0.6023238301277161, -1.0972530841827393, -0.09297909587621689, -0.5848579406738281, 0.5306897759437561, -0.37729746103286743, 1.1334335803985596, 0.22076693177223206, 0.8061825037002563, 0.3988996744155884, 0.614457368850708, 1.1987422704696655, 0.4742083251476288, -0.6942992210388184, -0.32416221499443054, -1.036298394203186, -0.9543995261192322, -0.014483069069683552, 0.10935091227293015, 1.5097671747207642, -1.0497965812683105, -0.1148611530661583, -0.06222948804497719], [1.0364211797714233, -0.13957317173480988, 0.5622105002403259, -0.6918379068374634, 2.2550253868103027, 1.2803068161010742, -1.2391983270645142, 1.3402303457260132, -0.4958529770374298, -0.3638768792152405, -0.3949153423309326, -0.15590175986289978, 2.1416983604431152, -0.5410980582237244, -0.4450601637363434, 0.5594677329063416, -1.1459405422210693, 1.49190354347229, -0.46381792426109314, -0.3531874716281891, 0.3782525360584259, 1.9610906839370728, -0.33607736229896545, -1.9187893867492676, 2.440826177597046, -0.005667413584887981, -0.20585514605045319, -0.2253180593252182, 0.030663898214697838, 0.13937777280807495, 2.6393096446990967, -0.4331413805484772], [0.5035655498504639, 1.0652424097061157, 0.3434810936450958, -0.10458438098430634, 0.7891378402709961, 1.21791672706604, 0.7779955267906189, 0.7588061094284058, 2.5859477519989014, 0.04380461946129799, -0.1561615765094757, 0.32179221510887146, 0.6310672163963318, 0.7831384539604187, 1.879805326461792, -0.24031785130500793, 0.6137083172798157, 0.27592533826828003, 0.6311396956443787, -0.8371238708496094, 0.3908490836620331, 0.4729698896408081, 1.1957309246063232, 3.1141746044158936, -0.41290223598480225, 0.9060449004173279, -1.3876875638961792, 0.8150113224983215, -0.16034996509552002, 1.4195308685302734, 0.8180647492408752, -1.1450421810150146], [0.5683033466339111, 1.5977946519851685, -0.9828183054924011, -2.085909843444824, 0.8966062664985657, 1.3866124153137207, -0.9736902713775635, -0.8433971405029297, 0.4940680265426636, 0.054210491478443146, 1.0984148979187012, 1.544407844543457, 0.09004556387662888, 1.1388232707977295, 0.024528229609131813, -1.3872874975204468, 1.326600193977356, -1.5801163911819458, 0.4091382324695587, 1.3427188396453857, -1.248978853225708, 0.6938190460205078, 0.8842808604240417, 0.7195508480072021, 1.5695197582244873, 0.8285602331161499, 0.1666259765625, -0.466598242521286, 1.3105368614196777, 0.75065678358078, -0.9260618686676025, -0.7840384244918823], [0.5647850632667542, -0.8247392773628235, 0.6053531765937805, 0.7172132134437561, 0.3636873662471771, 0.3323765993118286, -0.6884569525718689, -0.7068842649459839, -0.7106373310089111, 0.6739404797554016, 0.10060817748308182, -1.6548196077346802, 1.8340562582015991, 0.3495897948741913, 0.3042777478694916, -0.04621736332774162, 0.3575393259525299, -1.8991749286651611, -0.5878263115882874, 0.7429977059364319, 0.17384903132915497, 0.5646764636039734, -0.32077574729919434, 1.2197120189666748, 2.5105228424072266, -0.08421102911233902, -1.0422371625900269, -0.48585665225982666, 0.7381713390350342, -0.9575991034507751, -0.10342307388782501, -0.9220962524414062], [-0.7771132588386536, 0.28669804334640503, -1.3818113803863525, -0.19782574474811554, 0.8256277441978455, 0.6499590277671814, -0.6613459587097168, -0.004838938359171152, -1.7296620607376099, 0.47470787167549133, 1.536498785018921, 0.38087064027786255, 0.10014841705560684, 0.7890043258666992, 1.0991363525390625, 0.5320740938186646, -0.3315863609313965, 0.0880696177482605, -0.7376686930656433, -1.0633424520492554, -2.7115108966827393, 0.8249402046203613, -0.2406357079744339, 0.6907387971878052, 0.8520296216011047, -0.6687374711036682, 1.8526452779769897, 0.7445104718208313, 0.7394213676452637, 2.485574245452881, 1.8914971351623535, -1.2803717851638794], [-0.7521361708641052, -0.08133067935705185, 0.6726334691047668, 1.1267321109771729, 0.2817297577857971, 0.2035725712776184, 0.9330042600631714, -0.8068007826805115, 1.908027172088623, 0.650693416595459, -0.9865424036979675, -0.20502479374408722, 0.6412825584411621, -1.6285579204559326, 0.9274517297744751, -0.202969491481781, 1.6473628282546997, 1.1578686237335205, -0.31163737177848816, -0.0602823942899704, -1.22657310962677, 0.9369065165519714, -0.14623428881168365, 0.1706540286540985, -1.4935941696166992, -0.17369158565998077, 0.0750146135687828, -1.059097170829773, -0.7966455817222595, 0.2325131893157959, -0.07148101180791855, 0.019597917795181274], [1.2440978288650513, 0.23277679085731506, 1.043030858039856, 0.5097717046737671, -1.4746686220169067, -1.0502711534500122, -0.15937842428684235, 0.20519928634166718, 0.07719624787569046, 0.6894049048423767, 0.920204758644104, 0.044211722910404205, 1.4364110231399536, 1.2663732767105103, 0.6747130751609802, 0.10200344026088715, -0.026439931243658066, 0.39873579144477844, -0.6239338517189026, -0.10667169094085693, -0.5104123950004578, -0.8415445685386658, 0.42298412322998047, 1.2515124082565308, 0.22323954105377197, 0.3808019757270813, 0.974236011505127, -1.0130826234817505, 0.2169754058122635, 0.5688327550888062, -0.5865612030029297, 0.4137505888938904], [0.5521157383918762, 0.022028207778930664, 1.3810439109802246, -0.634341299533844, -0.5984578728675842, -0.2604577839374542, -0.3799308240413666, 0.5911881327629089, -1.4858460426330566, 0.6005412936210632, -0.3692900240421295, 0.005242793820798397, -0.15740765631198883, 0.9034736752510071, 0.47239309549331665, 0.23741528391838074, -0.7658208608627319, -0.1477590948343277, 0.8762475252151489, 1.0365580320358276, 0.11023106426000595, -1.666229248046875, 2.4160144329071045, -1.6952863931655884, -0.49242162704467773, 0.6564777493476868, 0.6332066655158997, 2.8434197902679443, 0.2633230686187744, -0.41413721442222595, -1.3905057907104492, 1.0541952848434448], [-0.650934636592865, -0.014952105470001698, 1.4455260038375854, -2.5065693855285645, 0.9562122821807861, 0.9525745511054993, 0.8282140493392944, -0.9156089425086975, -1.001037836074829, -0.07369076460599899, -0.5936138033866882, -0.2972041368484497, 1.1971228122711182, 2.355597496032715, -0.05115947499871254, -0.3088722825050354, 1.053573489189148, 0.7099035978317261, -0.8116152286529541, 0.2460508942604065, 0.6582638621330261, 0.3209684193134308, -0.1662525236606598, -0.4661373198032379, 1.3005911111831665, 1.2418553829193115, 1.3924378156661987, -1.6587352752685547, -1.9291107654571533, 0.44744765758514404, 0.5673607587814331, 0.46389004588127136], [0.6123347282409668, 0.11984780430793762, 0.9608395099639893, -2.8894460201263428, -0.9102967381477356, -0.27996981143951416, 0.4096474349498749, 1.2612820863723755, -1.0721092224121094, -0.5797557234764099, 0.34520086646080017, -0.18083815276622772, 0.9012671709060669, -0.3581151068210602, -0.737433671951294, 0.20444011688232422, -1.0536956787109375, -1.3221032619476318, 0.5387212038040161, 0.015412822365760803, 1.0171715021133423, 0.5319854617118835, 0.08358236402273178, 0.5733174085617065, 0.5067651271820068, 0.7560315132141113, 0.7815488576889038, -1.2605174779891968, -0.7310253977775574, 1.0397727489471436, -0.3035494089126587, 0.960425853729248], [-1.2288954257965088, 0.791692316532135, 0.6963607668876648, 0.006560005247592926, 0.2757372558116913, -0.9415580630302429, -0.719009280204773, -0.7858418226242065, -0.06075335666537285, -1.268641710281372, 0.0542861707508564, 0.1297425627708435, -0.3520829975605011, -0.023620378226041794, -0.9658955931663513, 1.6765038967132568, 0.4550338089466095, 0.014335989020764828, -1.1751359701156616, -0.45922335982322693, 0.42283177375793457, -2.1247525215148926, -0.7428426146507263, 0.8935660719871521, -0.028706220909953117, -0.7503459453582764, -1.124221682548523, 0.0805366039276123, -0.12679652869701385, 0.3589813709259033, 0.3130023181438446, -0.7658705711364746], [-0.4963231384754181, 0.3833414614200592, 0.9685549736022949, -1.0884586572647095, -0.34232303500175476, 0.4088503420352936, 0.33533814549446106, 0.4980519413948059, -0.29120001196861267, 0.9576464891433716, 1.0988291501998901, 0.19472646713256836, -1.122589111328125, -0.5927927494049072, -0.8169450759887695, 0.17896422743797302, -0.6513125896453857, 0.09159012883901596, 1.1642968654632568, 0.017560649663209915, 1.2758219242095947, -1.9915966987609863, 0.7062646150588989, 1.0211260318756104, 0.44599732756614685, -0.19697216153144836, 0.35231783986091614, -0.2570880949497223, -1.4773982763290405, 0.5301862955093384, -1.0687148571014404, -0.6548067331314087], [1.222881555557251, 0.19634470343589783, -0.056707460433244705, -2.142432928085327, -1.435664415359497, 0.8778481483459473, -0.4170464277267456, -1.7238997220993042, -0.3207252025604248, 2.423089027404785, -0.21946211159229279, 0.0651693120598793, -0.0762268453836441, 1.77336847782135, 0.3341582417488098, -0.6344299912452698, 0.2802640199661255, -1.0526304244995117, 1.0523111820220947, 0.24008549749851227, 0.1971307396888733, 0.0031402220483869314, -0.02571266144514084, -0.18827036023139954, -0.060977015644311905, 2.1004233360290527, 0.49394044280052185, 1.0193716287612915, 0.9793210029602051, -1.733046054840088, -0.9275054931640625, -0.3054145872592926], [0.9139457941055298, 0.6306187510490417, 0.3045366108417511, 0.4752112925052643, -0.20082910358905792, 1.578283429145813, 0.36748453974723816, -0.2418018877506256, 1.7781537771224976, -0.4411819279193878, -1.393965721130371, 1.178625226020813, 0.5394062995910645, -1.5612238645553589, -0.24735207855701447, -1.9847062826156616, -1.7327412366867065, 0.5989399552345276, -0.5980170965194702, 0.41294315457344055, 0.5708563923835754, 0.33025306463241577, -0.8951755166053772, 1.3948827981948853, 0.5653268098831177, -0.5935975313186646, 1.090366244316101, 1.117887258529663, -0.8830333352088928, -0.3992867171764374, 1.0668585300445557, 1.0991806983947754], [-0.19045290350914001, -2.312107801437378, 0.08115655183792114, 0.02657557651400566, 0.2621338665485382, 1.6020318269729614, 1.3893510103225708, 0.43617090582847595, -0.6423661112785339, 0.5185650587081909, -0.388810396194458, -0.4079720973968506, 1.1342805624008179, 0.13060010969638824, -0.2086668759584427, 0.6073548197746277, -0.18610802292823792, -0.48170363903045654, -0.6135643720626831, 1.09401273727417, -1.0847277641296387, -0.01217898353934288, 0.03946036845445633, -1.6644712686538696, 0.6778581738471985, 0.48008400201797485, 1.1973233222961426, 1.4613934755325317, 1.020844578742981, 2.2609095573425293, 0.38088253140449524, -0.7371269464492798], [0.3129691779613495, -1.1977882385253906, -0.6927613019943237, 1.03848397731781, -1.425064206123352, -0.34291911125183105, -0.4014533460140228, 1.7305190563201904, 0.39628735184669495, 0.4019426703453064, 0.03652624413371086, 0.24532319605350494, 0.18276990950107574, -1.6113355159759521, 1.837225317955017, -0.5889036655426025, -1.0963106155395508, 1.482467770576477, 0.8601175546646118, 1.0120820999145508, 2.4908461570739746, 0.12991690635681152, 0.05495542287826538, -0.441753089427948, 1.1730449199676514, 0.19566984474658966, -0.528730571269989, -0.9999684691429138, 0.7254394292831421, -0.6147432923316956, -0.2184351086616516, -0.6842812299728394], [-0.12334590405225754, -0.5913785099983215, 1.4635504484176636, 1.7639846801757812, -0.43818968534469604, 1.5893285274505615, 0.4667212665081024, 0.5769220590591431, 0.48539474606513977, 1.9815157651901245, -1.0655766725540161, 0.8423792123794556, -1.0957093238830566, 1.3273087739944458, 0.9310389757156372, -0.5064404606819153, -1.3901787996292114, -0.44435635209083557, -0.19783368706703186, 0.30621904134750366, 0.47724828124046326, 0.7808665037155151, -1.1703319549560547, -0.3632904291152954, 1.0367685556411743, 1.7853963375091553, 0.47874271869659424, -1.691108226776123, -1.0626938343048096, 0.1780795007944107, -1.1805446147918701, -0.5773800015449524], [1.1698713302612305, -0.14780041575431824, 0.4826955199241638, 0.20091703534126282, 1.0345860719680786, -0.01028104592114687, 0.48326632380485535, 0.4976951777935028, -0.8438259959220886, 0.9869264960289001, -0.9044489860534668, -0.30594339966773987, 1.8864942789077759, 0.8163778781890869, -0.5448090434074402, -0.1701328158378601, -0.4551391005516052, -0.2785777449607849, -0.11002377420663834, -0.4995957911014557, -0.020717371255159378, 2.0667245388031006, 0.5008004903793335, 1.973541021347046, -0.24301263689994812, -1.270232915878296, -1.214126706123352, 0.5282597541809082, 1.3473834991455078, 0.5092896223068237, 0.16500763595104218, 0.8103050589561462], [-0.2049487829208374, -1.0905464887619019, -2.434324026107788, -0.7332132458686829, 1.8058816194534302, -1.0458006858825684, -0.7269986867904663, -2.2701632976531982, -0.05576986446976662, 1.4672555923461914, -1.507595419883728, -1.2858774662017822, -1.0115866661071777, -0.1015390157699585, 2.1922717094421387, -0.20385879278182983, 0.965692400932312, -0.6133961081504822, 0.25183215737342834, 0.1761038452386856, -1.6156262159347534, 1.1208221912384033, 0.14688144624233246, -0.5018061399459839, -0.1696206033229828, 0.5223748087882996, -0.13690361380577087, 0.17401078343391418, 0.8770060539245605, -0.5416051745414734, -0.0004760995216201991, 1.0808686017990112], [-1.2724006175994873, 0.7022383213043213, -0.37461718916893005, 1.3146640062332153, 1.1954764127731323, -0.17361167073249817, 0.2780396044254303, 0.40248554944992065, -0.05485355108976364, -0.0827883929014206, -0.27071741223335266, 0.06914563477039337, 0.29093870520591736, -0.40270373225212097, 0.43223679065704346, -1.9034730195999146, 1.1484999656677246, 0.41724255681037903, -0.07798878848552704, -0.6216181516647339, 0.8458336591720581, -0.31640443205833435, -1.1137079000473022, -0.5199587941169739, -1.7128819227218628, -2.533353328704834, 0.6784856915473938, -0.27383968234062195, -0.06185787916183472, 1.7677944898605347, -1.5664498805999756, -1.7713656425476074], [0.7719005942344666, -0.9367444515228271, 0.31653204560279846, 0.11691861599683762, -0.14314982295036316, -0.03573324531316757, -0.9675952196121216, 1.257813811302185, 0.3465489149093628, -0.6327127814292908, 2.9993205070495605, 0.559385359287262, -2.0095341205596924, -0.24340830743312836, 1.1494433879852295, -0.7690458297729492, 0.1909606009721756, -1.2390086650848389, -0.9918531775474548, 1.2043503522872925, 1.337100863456726, -0.14776282012462616, -0.40433624386787415, -1.3620047569274902, 2.0590202808380127, -0.6821703910827637, -1.5692189931869507, -1.9408044815063477, 0.3173283040523529, 1.2732672691345215, -1.911629319190979, -1.4246896505355835], [1.6735285520553589, 1.9180562496185303, -0.7539540529251099, 1.900077223777771, 1.0030382871627808, -0.9212742447853088, -0.4445406496524811, -0.39925312995910645, -1.2869367599487305, -0.5536053776741028, 0.47086435556411743, 0.4575190544128418, 1.1766787767410278, 0.2881011962890625, -0.7904955148696899, -1.170994758605957, 1.802319884300232, -1.169379472732544, -0.8035115003585815, -0.3289327025413513, -0.012236388400197029, -0.5505511164665222, 0.3536580204963684, 0.17485105991363525, 0.4707653224468231, -1.4450606107711792, 1.439072847366333, -1.5786703824996948, 0.452050119638443, -1.1974724531173706, -2.059251546859741, -0.7150587439537048], [-0.09165330231189728, 0.1946379691362381, -1.2282435894012451, 0.6228917241096497, -0.046914126724004745, 0.5139539241790771, -0.56248939037323, 0.26865410804748535, 0.8680742979049683, -0.14608004689216614, -0.15862774848937988, 0.04271645098924637, 0.04467090591788292, 0.8513710498809814, -0.6274441480636597, 0.09540573507547379, 0.3960355520248413, 0.6940964460372925, -0.5102137923240662, -0.513498067855835, 0.5773934721946716, -0.47164207696914673, -0.5702822208404541, -1.0547542572021484, -0.11868228763341904, 0.5702711343765259, -0.8495053648948669, -0.23375746607780457, 0.5836036205291748, -0.8378204107284546, 0.6421242952346802, 0.5934938192367554], [-0.3040716350078583, -0.4154284596443176, -1.4185352325439453, 0.9925209283828735, -0.056714657694101334, 1.2160009145736694, 0.9300781488418579, -0.8584867119789124, -0.2643864154815674, -0.20232535898685455, 0.7269624471664429, -0.41570010781288147, 0.5181956887245178, -0.2246297299861908, 0.27589964866638184, 0.4712475836277008, -0.7028790712356567, -0.4244382381439209, -0.811558723449707, -0.47332385182380676, -0.3847756087779999, -0.8792982697486877, -0.6312567591667175, -2.158384323120117, 0.02230772003531456, 0.5469509959220886, -0.2106604278087616, -0.5551937222480774, 0.14747916162014008, -0.3495766222476959, 0.9559143781661987, -0.8011085987091064], [0.23703673481941223, -0.782512366771698, -1.8228936195373535, 2.0711681842803955, 0.8864408135414124, 1.129032850265503, -0.5473644733428955, -0.8506479263305664, 0.009096641093492508, 1.6098438501358032, -0.4901028871536255, 0.3951363265514374, 0.34432750940322876, 1.2425938844680786, -0.23216582834720612, -1.8809373378753662, -0.1294652670621872, 0.8980827331542969, -0.909912645816803, -2.221663475036621, 1.6631543636322021, 0.8369135856628418, 0.7645788788795471, 0.10830482095479965, 1.112461805343628, 0.7443076372146606, -1.3474431037902832, 0.11417089402675629, 0.44897231459617615, -0.04162980988621712, 0.49128827452659607, 0.6597849726676941], [-1.239187240600586, 0.3932695984840393, 0.6533666849136353, 0.054832182824611664, -0.6338876485824585, -0.12845423817634583, -0.637219250202179, 1.0414479970932007, 0.23368613421916962, -0.08958345651626587, 0.22458532452583313, 1.185257911682129, -0.46323922276496887, -0.09302514046430588, 1.512878179550171, -0.13192139565944672, -2.044715166091919, 0.39712288975715637, -1.3177952766418457, -0.5312904715538025, -0.9728684425354004, -0.8754982948303223, 1.0890415906906128, 0.4386983811855316, -1.5624066591262817, -1.0868887901306152, 0.7737745046615601, 1.134743094444275, 0.5139946341514587, 1.4423366785049438, 0.039210718125104904, 0.3921022117137909], [0.27159157395362854, -0.5162986516952515, 1.5236397981643677, -2.176300048828125, -0.7510383129119873, -1.884451985359192, -2.0845208168029785, -0.6746828556060791, 0.3470552861690521, 0.08237937092781067, 0.06249433010816574, -0.4901778995990753, 0.4615647792816162, 0.4711483120918274, -0.08875443786382675, -1.5001981258392334, 0.9570583701133728, 1.1318331956863403, 0.2405698150396347, 1.0167126655578613, -1.7283167839050293, 0.32614028453826904, -0.7487331032752991, -1.086474895477295, 1.653175711631775, -1.1575430631637573, 0.6111364364624023, 0.10417549312114716, -0.7193174362182617, -0.08331697434186935, 1.6916218996047974, -0.6343467831611633], [0.04754766449332237, 0.436528742313385, -2.260127067565918, -2.474764347076416, 0.16337808966636658, 1.573624849319458, -1.0006622076034546, -0.09854994714260101, -0.3452942669391632, -0.5321128964424133, 0.044734805822372437, 0.7901162505149841, -1.1444505453109741, -0.2696881592273712, -1.0613270998001099, 0.18603451550006866, 1.7539793252944946, -0.5104018449783325, 2.9692940711975098, 0.8327443599700928, -1.179528832435608, -1.466806173324585, 0.5325479507446289, -0.015551190823316574, -0.719335675239563, 0.06996545195579529, 0.07064401358366013, 0.5412425994873047, -0.2423933446407318, -1.8944295644760132, 0.15925496816635132, 0.5150870680809021], [0.5333948731422424, -0.8640487194061279, 0.4363374412059784, 0.7184373736381531, -1.892063856124878, -0.6615135073661804, -0.3961988389492035, -0.6577550172805786, 0.4002875089645386, -0.1590946763753891, -0.523145854473114, -0.007977605797350407, 1.878424882888794, -1.6958481073379517, 0.5647805333137512, 0.7409221529960632, 0.1021326333284378, -1.0741785764694214, 0.7839024662971497, 0.6281579732894897, -0.8421282172203064, 0.1985035538673401, -0.8408389687538147, 1.4359678030014038, -0.9433152079582214, -1.4804325103759766, -0.08563844114542007, 1.4908252954483032, 1.2925755977630615, 0.46359843015670776, 0.7982007265090942, 0.036195412278175354], [1.436947226524353, -0.2317737340927124, -0.9044883251190186, 0.7105255722999573, 1.004158616065979, -1.7603580951690674, 0.14441855251789093, 0.33357053995132446, 1.5236830711364746, -0.6038382649421692, 1.085330843925476, -1.1957614421844482, 0.5682549476623535, 0.7928569316864014, -0.3484862446784973, 0.6314862370491028, -1.4585233926773071, -0.7175700068473816, -1.0499011278152466, -0.311884343624115, 0.8361397981643677, -0.5733191967010498, -0.3073769807815552, 0.905665934085846, -1.181837558746338, -0.31304216384887695, -1.8592009544372559, 0.4970642030239105, -0.11066200584173203, -1.5373709201812744, 0.6304357647895813, 0.4263574481010437], [2.4198219776153564, 0.18720269203186035, -0.5330878496170044, -0.7008323073387146, 0.7934848070144653, -0.575115978717804, 3.034003734588623, 0.13498753309249878, 2.270329236984253, -0.7732309699058533, 0.5667624473571777, 1.7074456214904785, -0.9335732460021973, 0.9588139057159424, 0.5222452878952026, 1.069989800453186, 0.24713103473186493, 0.6359302997589111, -0.2844242751598358, -0.6790604591369629, -1.4855282306671143, 0.19274240732192993, -0.25916048884391785, -1.774867057800293, -0.05701060965657234, 0.2097172886133194, -0.6710215210914612, -1.735865831375122, -0.7560160160064697, 0.19492721557617188, -0.43785572052001953, -0.8296710252761841], [0.20171907544136047, -1.2950819730758667, 1.087573528289795, -0.13560740649700165, 1.3716645240783691, -1.998917818069458, 0.24749882519245148, 0.7440361380577087, 0.18149377405643463, 0.3764452338218689, -0.8240392208099365, 2.1141810417175293, -1.7453503608703613, 1.6363722085952759, 0.9966806173324585, 0.5808207392692566, 0.7689083814620972, -1.2211709022521973, 0.11850633472204208, -0.4426117241382599, -0.44856345653533936, -1.4932326078414917, 0.20933863520622253, -0.09965530782938004, -1.0835117101669312, 0.099761001765728, -0.4556630253791809, -0.2710697650909424, -0.648287296295166, 0.3104213774204254, -0.3739107549190521, -0.03798019140958786], [0.2023320198059082, -0.9007354974746704, 0.12119431793689728, -0.8852491974830627, 1.2632508277893066, -0.9819695949554443, -3.018160581588745, -2.188722610473633, -0.8640403151512146, -0.44102150201797485, -0.7612175941467285, 2.497828483581543, 0.327752023935318, -0.5052052736282349, -1.8081105947494507, -0.5715276598930359, 1.424715280532837, 1.0719060897827148, 0.4875878691673279, 1.0205447673797607, 1.1029149293899536, -1.1281344890594482, -0.474919855594635, 1.6328792572021484, -0.021510425955057144, 0.9132413268089294, -1.0652445554733276, 1.6344385147094727, 0.06016438081860542, 0.06027717515826225, -0.999758780002594, 0.5936492085456848], [-0.1717909425497055, 0.2972865104675293, 0.060797370970249176, -1.023697853088379, -1.637750506401062, -0.04664934426546097, -0.2754853069782257, -0.5960527062416077, -0.3592573404312134, -0.0731818825006485, 0.5346106290817261, -0.5229978561401367, 1.245843529701233, -0.6678914427757263, -0.8256770372390747, 0.7943860292434692, 0.21599169075489044, 0.4820164442062378, 0.42650166153907776, -0.20561745762825012, 0.7473987340927124, -0.1537594348192215, -0.8872213363647461, 0.6811449527740479, -0.33429819345474243, -0.10740060359239578, 0.4160528779029846, 0.6890560388565063, -0.6799925565719604, 1.8880196809768677, -2.508338689804077, 0.5151010751724243], [-0.7445590496063232, -0.6216796636581421, 2.258622169494629, -0.3275270164012909, 0.7030179500579834, 2.1205759048461914, -1.3487175703048706, 3.0161187648773193, 2.5266807079315186, -0.2245740294456482, 0.02087116613984108, -0.0759849026799202, 0.3834114670753479, 2.2821125984191895, -0.27948784828186035, -1.0613998174667358, -0.9754331707954407, 0.5452587008476257, -1.2611521482467651, 1.7123243808746338, 1.309572696685791, -1.9070053100585938, 1.1679567098617554, 1.005726933479309, -0.14294123649597168, -0.5810289978981018, 1.3736895322799683, -0.9858267307281494, 0.5963941216468811, -0.6878533363342285, 0.8646059632301331, 0.02110247127711773], [-1.1097958087921143, 0.9466045498847961, -0.02614651620388031, -1.3591228723526, -1.288903832435608, 1.4634559154510498, -0.2919454276561737, 0.2834591567516327, -1.2699800729751587, -1.0818963050842285, 0.04873612895607948, -0.9642577767372131, 0.43932947516441345, 0.5801178812980652, 0.4638366401195526, -0.8180992007255554, -0.06877413392066956, 1.4194234609603882, 0.24778948724269867, 1.2683113813400269, 2.4396889209747314, -0.057217296212911606, -2.044508218765259, 0.08506373316049576, 0.6997916102409363, 0.5949050188064575, -0.38319045305252075, -0.08691456913948059, -0.8314367532730103, 1.7655246257781982, 0.09147509187459946, 1.9110429286956787], [-1.7294636964797974, 0.524876594543457, -0.5824087858200073, 0.42217615246772766, 0.48801517486572266, 0.5995274782180786, -0.25325098633766174, 1.81000816822052, -0.4672029912471771, -1.3632045984268188, 0.26085227727890015, 1.787980079650879, 1.3987962007522583, -1.9179202318191528, -1.3429402112960815, -0.658210277557373, -0.5614153742790222, -1.2913044691085815, -0.29954901337623596, 0.7657477259635925, 0.7443166971206665, -1.0922752618789673, 0.515681803226471, -0.35557928681373596, -0.30343005061149597, 0.04141278564929962, 1.2132147550582886, 0.31793180108070374, 0.5480003952980042, -0.336137592792511, 0.3307654559612274, 1.709784984588623], [-0.6233059167861938, -1.3538521528244019, -0.6764007210731506, 0.420716792345047, -0.6960150599479675, 0.22566308081150055, -0.07642609626054764, -1.602219581604004, -0.2881383001804352, -0.1762770414352417, -1.35881507396698, -1.7189171314239502, -0.36100253462791443, 0.33634984493255615, -1.133467435836792, -0.07659462094306946, 1.5986740589141846, 0.677804708480835, 0.6053532958030701, 0.6800292730331421, -0.2591671645641327, -1.734329104423523, 0.09256801754236221, 0.056236978620290756, 0.4423927068710327, -0.35170209407806396, 0.41817376017570496, -1.3816027641296387, 1.0460522174835205, -1.7752625942230225, 1.8770618438720703, -2.1380362510681152], [1.5857961177825928, -0.5391849279403687, -0.1870071142911911, -0.2500256299972534, -0.37529608607292175, 0.03889064863324165, -0.47170400619506836, 1.0108777284622192, 0.9599971175193787, 1.652562141418457, 0.6740255355834961, -1.1391667127609253, -0.6838980317115784, 2.160281181335449, -0.465437114238739, -0.9563416242599487, 2.203688621520996, 1.8813761472702026, -1.0611677169799805, -0.6260448694229126, 1.720738410949707, -0.40236738324165344, 1.5554640293121338, 0.12009599059820175, 0.17894165217876434, 0.049141354858875275, -0.6172294616699219, 1.6351804733276367, -1.507979393005371, -0.5032192468643188, 1.0081605911254883, 0.4520266354084015], [0.6456876397132874, -0.8865782618522644, 0.8768588304519653, -0.3148179054260254, 1.7338941097259521, -1.3095703125, 0.7781305313110352, -2.224090576171875, 0.08601748198270798, 0.10414578020572662, -2.771008014678955, -1.1778141260147095, 0.8221161961555481, -0.5820257067680359, -0.7749415040016174, 0.009886413812637329, -0.1338157206773758, -1.1346780061721802, -1.1926943063735962, 0.12299881130456924, 0.8938530683517456, -0.03809671476483345, -0.4544147849082947, -0.33814117312431335, 0.56763756275177, 0.9629026651382446, -1.9251892566680908, -0.6863254904747009, 0.6274230480194092, -0.2133779227733612, 0.21757890284061432, 1.8600757122039795], [-1.7100614309310913, -1.5606790781021118, 0.6829759478569031, 0.4236416518688202, -0.8182729482650757, 0.31247350573539734, -0.4415266215801239, -0.6758260130882263, -1.7111194133758545, 0.6709026098251343, 1.077760934829712, -1.274170994758606, -0.17967194318771362, -0.7313642501831055, -1.0854672193527222, 1.96280038356781, 0.17697970569133759, -1.4771260023117065, 1.4594358205795288, -0.7653865814208984, 1.1672836542129517, -0.9012619853019714, -0.12174303829669952, -0.2773086726665497, 2.1544723510742188, 1.9451996088027954, 0.9198073744773865, -0.014254007488489151, 0.636335015296936, -0.5059208273887634, -0.5836862325668335, 0.45832592248916626], [0.5478407144546509, 0.5196899175643921, -0.8018433451652527, -0.3980696499347687, 0.8946511745452881, -1.0838541984558105, 1.6106864213943481, 1.7099863290786743, -0.23145394027233124, -0.5431131720542908, 0.9705742001533508, -0.39399999380111694, 2.0388684272766113, -1.2187007665634155, 0.9391580820083618, -0.4740859270095825, 1.6293020248413086, 1.285548210144043, 0.9184884428977966, -0.09834431856870651, 0.5523307919502258, 0.968160092830658, -0.20620428025722504, -0.08915212005376816, 0.3786841034889221, -0.4142232835292816, 0.8068442940711975, -0.6303219795227051, -0.858284056186676, -1.4464876651763916, 0.9815159440040588, 0.6929910182952881], [-0.6301521062850952, 0.29707062244415283, -1.1775634288787842, -1.5305001735687256, -0.4216819405555725, 0.17984703183174133, -0.29284629225730896, -1.7723265886306763, 0.5562141537666321, 0.1416395604610443, -0.6227132081985474, -0.7144955992698669, -0.8156819343566895, 0.9843832850456238, 0.5635186433792114, -2.2253546714782715, 1.2433487176895142, -0.1878967434167862, -2.704758644104004, -0.538310170173645, 0.04424523562192917, -0.09931807219982147, 0.0896226242184639, 1.2143319845199585, 0.12936073541641235, -3.2294938564300537, -0.06043016538023949, 0.6745021343231201, -2.102557897567749, 0.46729975938796997, -0.08610807359218597, 1.3303518295288086], [1.722339153289795, 1.6144769191741943, 0.7127851247787476, 1.508126139640808, 0.6131758093833923, 0.6973530650138855, -0.5084711909294128, -1.2320828437805176, 0.4531657099723816, 0.8037003874778748, 1.3293429613113403, -1.1958916187286377, 0.918513298034668, 0.6051903963088989, -0.6516320705413818, -0.09080115705728531, 0.01598583534359932, 0.22069303691387177, 0.0028970022685825825, -0.6437483429908752, 0.8567633628845215, 0.4266390800476074, -1.7858936786651611, 1.7568387985229492, 0.586014986038208, -1.1314061880111694, -0.032857444137334824, 0.6831444501876831, -0.3933037221431732, -0.5299332141876221, 0.4808300733566284, 1.0026026964187622], [-1.9295196533203125, -0.4551435112953186, 0.9856116771697998, -1.1793910264968872, 0.296280175447464, 1.204714059829712, 2.3222522735595703, 0.86667400598526, 0.07674271613359451, -1.8152201175689697, 0.6531978845596313, -0.12703020870685577, -0.0738949328660965, -0.9048470258712769, 0.6842615008354187, -0.7639097571372986, -1.1388695240020752, -0.864130973815918, 0.6205222010612488, 1.6819827556610107, 1.1900781393051147, 2.0345940589904785, -1.2357460260391235, -0.9857560992240906, 0.3251984417438507, -0.19508206844329834, 0.8499928116798401, 0.596618115901947, -1.3855435848236084, 0.03232376649975777, 1.1326442956924438, 0.9783244729042053], [-1.5410953760147095, -2.2555747032165527, 1.195214867591858, -0.2969805896282196, -0.9950957298278809, -0.1892087310552597, -0.004003932699561119, 0.44328516721725464, 0.5896213054656982, -0.32969921827316284, -0.2415716052055359, 0.6421565413475037, -0.20792458951473236, 1.325811505317688, 0.5563498735427856, -0.0821758359670639, 1.66639244556427, 0.19574886560440063, 0.360725462436676, -0.6127736568450928, 0.509838342666626, -0.8413416743278503, 1.4977833032608032, 0.40187084674835205, 1.05677330493927, -0.08385080844163895, -1.9200857877731323, -0.6301896572113037, 0.43351709842681885, 0.7911681532859802, 1.3971061706542969, -0.6357635259628296], [0.44252994656562805, 1.110075831413269, -0.5678883194923401, 0.063962921500206, -0.8735268712043762, 0.27343258261680603, -0.379702627658844, -1.1101157665252686, 0.008395503275096416, -0.2366194874048233, 0.5041543245315552, -0.6728379130363464, 0.9492623805999756, 0.7992695569992065, -1.4625779390335083, -1.3693385124206543, 0.6307162642478943, 1.6696436405181885, 0.31137019395828247, -0.11477991938591003, 0.18500222265720367, -0.4987969994544983, -0.04003840312361717, -0.1405150294303894, -1.4425231218338013, -1.0179311037063599, -0.22456276416778564, 0.9939222931861877, 0.1851319521665573, -0.40769845247268677, 0.0961211547255516, 1.0026952028274536], [-1.4170489311218262, -0.13900937139987946, 2.1745009422302246, -0.025700852274894714, 0.10396726429462433, -0.025953788310289383, 0.6097310781478882, -1.4676731824874878, -1.0426985025405884, 0.8496533036231995, 0.17633526027202606, 0.07819817960262299, -2.979910373687744, -0.7181060314178467, -0.09554868191480637, 1.9991122484207153, -1.6638011932373047, -0.242176815867424, 1.672633171081543, -0.40360215306282043, -0.515817403793335, -0.6169214248657227, -0.8332198858261108, -1.0426263809204102, 0.07700742781162262, 0.07515133172273636, -0.222013458609581, 0.7623410820960999, 1.4989526271820068, 2.0006930828094482, 0.636357307434082, -0.6473392248153687], [2.5410549640655518, 0.7265422940254211, -0.5414595603942871, -0.5238775014877319, -0.9964321255683899, 1.2771013975143433, -0.391812264919281, -0.09246913343667984, -2.038822650909424, -2.3188092708587646, 0.0477665513753891, -1.7693167924880981, -0.6581265330314636, 1.516599416732788, 0.4222804009914398, 0.10287196934223175, -1.4637601375579834, 0.7131072878837585, -0.48496174812316895, -0.03737304359674454, 0.39966508746147156, 2.650918483734131, -0.613532543182373, 1.1054763793945312, -1.1795661449432373, 0.18920540809631348, -0.831818163394928, -0.5168529748916626, 1.7058607339859009, -0.1783895641565323, -1.2828693389892578, -0.7063300013542175], [1.420326590538025, 0.8230463266372681, -0.20526725053787231, -1.4513323307037354, -0.8864818811416626, 0.1321137249469757, 1.062112808227539, 0.19447313249111176, 3.1944830417633057, 0.4187638759613037, 1.2077147960662842, -0.2127794325351715, -0.17375986278057098, 2.4826996326446533, 1.61055588722229, 1.097901463508606, 0.043958645313978195, 1.414810061454773, 0.08238079398870468, -0.3585774004459381, -1.9771419763565063, 0.1527099907398224, 0.07402807474136353, -0.8741352558135986, -0.20535048842430115, -2.0014989376068115, -0.9596236944198608, 0.3987375795841217, 0.43007025122642517, 0.9091708064079285, 0.660933792591095, 0.4377595782279968], [0.21076959371566772, -0.42574214935302734, 0.0772266760468483, -0.4719395935535431, -0.8738275170326233, 0.5240929126739502, 0.9376402497291565, -1.423429012298584, 0.9067891240119934, -1.4349901676177979, -1.0123099088668823, 0.9324703216552734, -0.01862923614680767, -0.32653677463531494, -1.3946069478988647, 0.45646119117736816, -0.4958342909812927, 0.5537104606628418, 0.13620604574680328, -1.3173085451126099, -0.9542975425720215, -0.0014645224437117577, 0.27319982647895813, 2.1965222358703613, 1.323033332824707, 1.8802934885025024, 1.0050653219223022, 0.8788009285926819, -0.4739610254764557, 1.1830928325653076, 0.6795551776885986, 0.5512412786483765], [-0.2552371621131897, -1.7942191362380981, -0.24479107558727264, -2.553924083709717, 1.234679102897644, 0.3117857277393341, 0.005994895473122597, -1.3406051397323608, 0.8077842593193054, -0.20704670250415802, -1.280701994895935, -0.35147804021835327, -0.3685416877269745, -1.536968469619751, -0.33663755655288696, 1.6229180097579956, 0.164280965924263, -1.124544382095337, -1.051580786705017, -0.7516524195671082, -1.693996787071228, 0.8802785277366638, -0.8524115085601807, -0.38155508041381836, 0.5056065320968628, -0.5774620175361633, 0.8514648675918579, -1.3258126974105835, -0.43647661805152893, 1.1256134510040283, -1.9724177122116089, 0.04329010844230652], [0.026068849489092827, 0.9077119827270508, -1.3949341773986816, 0.12145339697599411, 0.4381537139415741, 0.5465057492256165, -0.5729128122329712, -1.2283217906951904, -0.570643961429596, -0.8583333492279053, -1.3750752210617065, 0.32996803522109985, 1.3934216499328613, 1.6698904037475586, -0.7918121814727783, -1.3226730823516846, -0.9926436543464661, 0.665709912776947, 1.8763363361358643, 1.2804793119430542, 0.842620313167572, -0.7933393120765686, -1.2356972694396973, -0.8080751299858093, 0.23386508226394653, -0.6456813216209412, 0.30373191833496094, -0.49634304642677307, 0.35426560044288635, -0.14123255014419556, 0.38700994849205017, -0.5351178646087646], [-0.22298845648765564, -0.45073992013931274, -0.9097872376441956, 0.32089316844940186, -1.3508414030075073, 0.2578585147857666, -1.6275032758712769, 1.19749116897583, 0.8272178769111633, 0.4969272017478943, 0.9196450710296631, 0.018076764419674873, -1.23314368724823, -1.0529887676239014, -0.3403060734272003, -0.6853731274604797, 0.4797126054763794, 1.091357707977295, 0.8238212466239929, 1.5343855619430542, -0.3039216995239258, 0.792402982711792, -1.7764551639556885, 0.4182622730731964, -1.4259740114212036, -0.7826449871063232, 1.8090852499008179, -0.8260422348976135, 0.10647905617952347, -1.1521344184875488, -0.26310059428215027, -0.3186311423778534], [-0.70155930519104, 0.10224881768226624, 0.16043175756931305, -1.3184887170791626, -0.8840784430503845, 1.193363070487976, 0.30645960569381714, 1.1428859233856201, -0.6552409529685974, 0.19141028821468353, -0.46526482701301575, 0.29725027084350586, 0.505170464515686, -0.5045437812805176, -1.285656452178955, 0.7156260013580322, 0.4300738275051117, -0.06356547772884369, -0.5293523073196411, -1.740524172782898, 0.6454852223396301, 1.067173957824707, 0.6527473330497742, 1.083356261253357, -0.22937387228012085, 0.5718953609466553, -0.7970023155212402, 0.1633378118276596, 1.4210416078567505, -0.8551786541938782, -1.6973713636398315, 0.31458836793899536], [-0.598595380783081, -0.9241909384727478, 0.7959237694740295, -0.6163753271102905, 0.8428397178649902, -1.9479671716690063, -0.716604471206665, 2.9786481857299805, -0.05384567752480507, 0.6314568519592285, 0.9582423567771912, -1.084304928779602, 0.9311014413833618, 0.5900158882141113, 0.2702368199825287, -1.9237110614776611, -0.5950533747673035, -1.790537714958191, 1.9493021965026855, -1.974530577659607, -0.3543647527694702, -1.2270441055297852, -0.7543738484382629, -0.6583878397941589, -1.1495869159698486, -0.7155190110206604, 0.786983847618103, -0.01204209215939045, -0.8401438593864441, 1.3095251321792603, -1.2588934898376465, -0.5927640795707703], [0.4353331923484802, 0.577577531337738, -0.6303601264953613, -0.7514076828956604, -0.6430760025978088, 1.951438307762146, -1.3756554126739502, 0.34257835149765015, 0.11291562020778656, -1.2548528909683228, 0.4579280912876129, -1.6074289083480835, -0.8421337604522705, 0.3948214650154114, 0.6505945920944214, 2.5579464435577393, -1.139938473701477, 0.3297579884529114, -0.806313157081604, -0.6821925044059753, 1.2998088598251343, 0.11148593574762344, 0.6271746158599854, -1.0682650804519653, -0.44934314489364624, 0.19871209561824799, -1.1454229354858398, -0.2436496913433075, 0.8918471932411194, -0.9335352182388306, -0.7036242485046387, -0.39368313550949097], [-0.7822917699813843, 0.14992360770702362, 0.7542290091514587, 2.0379478931427, 0.8841001391410828, 1.4919544458389282, -2.353830575942993, 0.03001260571181774, -0.5827341079711914, 0.9298702478408813, -1.4207485914230347, 1.643958330154419, 0.1503695398569107, -1.0048161745071411, 0.19791866838932037, -0.011850926093757153, -0.042123936116695404, 1.8744009733200073, -0.835037887096405, -0.4901537299156189, 0.08221415430307388, -0.45678263902664185, -0.2222636193037033, -1.5915337800979614, -1.3818691968917847, -0.19607704877853394, -1.227537751197815, 0.8434197902679443, -2.1570043563842773, 0.08143878728151321, 1.105726718902588, 0.514478862285614]], [[0.11670180410146713, -0.5994892716407776, 0.10090378671884537, 0.1680304855108261, 1.2160691022872925, -0.04152819141745567, -0.1639225035905838, 0.9988359808921814, -0.333453893661499, -1.1009504795074463, 1.3500174283981323, 0.8616025447845459, 1.5462548732757568, -1.771088719367981, 0.0561467707157135, 0.44886213541030884, -0.8089423179626465, 0.8565836548805237, -1.0074061155319214, 0.4364472031593323, -0.8240872621536255, 0.715165913105011, -0.7487936615943909, -0.6471454501152039, 2.387286424636841, 0.7065833806991577, 0.9432373046875, -0.9039589166641235, 0.7794665694236755, -0.7123070955276489, -0.5906912684440613, -1.2456094026565552], [-1.0061731338500977, -0.8625510931015015, -0.3984196186065674, 0.6058627963066101, 0.9102181792259216, -0.47522783279418945, 0.5153470635414124, 1.22166109085083, 0.14315414428710938, -0.20619720220565796, -0.07055380195379257, -0.8924959897994995, -0.15181903541088104, -0.8831653594970703, -2.0995097160339355, -0.16526345908641815, -2.03888201713562, -1.6341047286987305, 0.3003574013710022, -0.9809024333953857, -1.700474500656128, -2.3275699615478516, 0.35140493512153625, -0.529573380947113, 0.414720743894577, 0.3558327853679657, 0.21817000210285187, 1.484236240386963, -0.687981128692627, -0.8802768588066101, 0.4911685287952423, 1.2022216320037842], [-0.6912594437599182, 0.6322237849235535, -1.129215121269226, 1.2268285751342773, -0.9937476515769958, -0.9145369529724121, -2.2706708908081055, -1.8567421436309814, 0.6255279779434204, -0.0655939131975174, 2.489985704421997, -0.2807995676994324, 0.4205505847930908, 0.5348236560821533, 0.09650631994009018, 0.8966635465621948, 1.81642746925354, -0.8340564966201782, -1.1061415672302246, -2.135702610015869, 1.0726613998413086, 0.047036122530698776, -0.7530352473258972, 0.3206043541431427, -0.2911219894886017, -0.5718106031417847, -1.1882576942443848, -1.1012042760849, -1.2985219955444336, 0.4809625446796417, -1.6848238706588745, 1.113998532295227], [1.1683111190795898, -0.882309079170227, -0.5029122829437256, -0.9482777118682861, 0.043711304664611816, 2.1187760829925537, -0.054968252778053284, -1.1499103307724, -1.9831045866012573, 1.0950753688812256, 0.1264829784631729, 0.08505306392908096, -0.9953927993774414, 0.09157643467187881, 0.48311322927474976, 1.2084729671478271, -1.371049404144287, -1.8250597715377808, -1.508828043937683, -1.877160668373108, -1.0259414911270142, 1.3164501190185547, 0.2958364188671112, 0.49752408266067505, 0.4190804362297058, -1.9678723812103271, 0.7032366394996643, 0.09179786592721939, -1.6381845474243164, -3.0120036602020264, -2.6923019886016846, -0.1346150040626526], [0.6062726378440857, 0.08136965334415436, -2.7007505893707275, -2.852325916290283, -0.5689045786857605, -0.1572316288948059, 1.6614283323287964, -0.42970824241638184, 0.49483630061149597, 0.18804799020290375, -2.168494462966919, -1.1470073461532593, -0.962487518787384, 1.15616774559021, -1.3017675876617432, 1.109165072441101, -2.6020348072052, -0.5512663722038269, 0.4157479405403137, 0.4845049977302551, 1.8059515953063965, -0.4315641522407532, 0.6028234958648682, -1.3651819229125977, 0.3901357054710388, 0.7942166328430176, -0.7529104948043823, -0.31046777963638306, 0.029530631378293037, -0.25561127066612244, 0.40051326155662537, 1.6638820171356201], [-0.15691304206848145, 0.6117231845855713, 1.3156269788742065, -0.42381376028060913, -0.6564837098121643, 1.0696804523468018, 0.5172799825668335, 0.21261285245418549, -2.074697494506836, 0.647071361541748, -1.9089876413345337, -1.5008511543273926, -0.28947511315345764, -0.6862955093383789, 0.8345276713371277, -0.17283357679843903, -0.20136311650276184, 0.23046281933784485, 0.5046076774597168, -0.30379581451416016, 0.04026488587260246, -0.7224891781806946, 1.1698181629180908, -1.4208602905273438, -1.8106679916381836, 1.5559594631195068, 1.2073582410812378, -0.22011803090572357, -0.032534800469875336, 0.25974467396736145, 0.9317438006401062, 1.1619778871536255], [-1.057942509651184, -0.7342724800109863, 1.5908749103546143, 0.5555914044380188, -1.7281099557876587, -0.3859001696109772, -0.616180419921875, 1.746071457862854, 1.0202605724334717, 1.1929810047149658, 2.4183120727539062, -1.0493696928024292, -1.2136141061782837, 0.7256993651390076, 0.5375776290893555, -0.11106055229902267, -0.07399053871631622, -0.25450021028518677, 0.0012420633574947715, -0.7374963760375977, 1.8187769651412964, 0.19280381500720978, 0.5945557355880737, 0.3117828965187073, -1.0720266103744507, 0.18418656289577484, -0.5714613795280457, -0.334223210811615, -1.7149935960769653, -1.0010019540786743, -0.10537967830896378, 2.691619634628296], [-0.07251060009002686, -1.4626661539077759, -0.8312073349952698, -0.542750358581543, -0.6641325354576111, 0.8713651299476624, 2.3514959812164307, -0.3517868220806122, -0.05423249676823616, 0.7360050678253174, -1.5094865560531616, 0.23816347122192383, -0.47331753373146057, -0.7751996517181396, -0.11950496584177017, -0.35601967573165894, -0.08660287410020828, -0.3986740708351135, -0.8101473450660706, 0.6653396487236023, 1.0706332921981812, -1.1782902479171753, -2.162515163421631, -1.6101963520050049, 0.9433848261833191, 0.36995965242385864, 1.5324056148529053, 1.0526280403137207, -1.337480068206787, -1.0407015085220337, -0.032531168311834335, -1.4292292594909668], [0.09851981699466705, -0.8946466445922852, -0.3671334385871887, 0.9546323418617249, -0.27300894260406494, 0.8903084993362427, -1.5595099925994873, -1.9544531106948853, 1.4582674503326416, 1.3766238689422607, -0.36224132776260376, -0.6586462259292603, 1.3875229358673096, 0.7452458143234253, -1.3559490442276, 1.023330807685852, 0.09193915128707886, -0.8960728049278259, -1.5769208669662476, 1.2479689121246338, 1.368945598602295, 0.12550882995128632, -1.61472749710083, 0.05259423702955246, -1.8730629682540894, 0.808134913444519, -0.6666529774665833, 0.7977012991905212, -2.0939695835113525, 0.6849948763847351, 0.14988549053668976, 0.8064521551132202], [0.4038464426994324, -1.3883469104766846, 1.1434214115142822, 0.33837491273880005, 0.3350549340248108, 0.20383185148239136, 1.761212944984436, -1.3732640743255615, 0.013639915734529495, 0.39468857645988464, 1.2808197736740112, -0.5894752144813538, 0.8894374966621399, 0.43763577938079834, 0.6452189087867737, 0.2089080959558487, -1.0023891925811768, 1.1316545009613037, 1.397515892982483, 0.02239901013672352, 1.4633026123046875, 0.2039823830127716, -0.13950863480567932, -0.21573852002620697, -0.402702659368515, 0.4853701591491699, 0.7406814694404602, 1.9315263032913208, 1.34385347366333, 0.4311359226703644, -0.49694767594337463, 0.5990041494369507], [1.0101083517074585, -0.9228969812393188, 0.06388209015130997, 0.6464184522628784, -1.798072099685669, 0.8509229421615601, 2.4449846744537354, 0.10322177410125732, -0.9405099153518677, -1.6179590225219727, -0.33484122157096863, 1.170966386795044, -0.7641103863716125, -1.8685460090637207, -0.46474945545196533, 0.48921242356300354, 1.5678534507751465, 1.6757867336273193, 0.20089341700077057, -1.4711118936538696, -0.19873756170272827, 0.20189540088176727, -0.21947447955608368, 0.5732294321060181, 1.1659480333328247, 1.0189868211746216, -0.3738718330860138, -1.2634525299072266, 0.44093164801597595, -0.9080179929733276, -0.9684621095657349, -0.9088929891586304], [1.238221287727356, 0.1862477958202362, -0.2718212604522705, -0.747035801410675, -0.4392608404159546, 0.21726016700267792, 1.160392165184021, 0.010842588730156422, -0.9831286668777466, 0.8201795220375061, -1.3937755823135376, -0.11387071013450623, 1.230100154876709, -0.0436444915831089, 1.157275915145874, 0.2828788757324219, -0.9271085262298584, -2.3502984046936035, 0.5594682097434998, -0.2558121085166931, 1.2746814489364624, -0.30813145637512207, -1.4919732809066772, -0.322958379983902, -0.5782631039619446, 0.15154346823692322, 1.0431981086730957, 2.059988498687744, -1.0322257280349731, -1.6416993141174316, -0.2009432166814804, 0.3168086111545563], [-0.20488110184669495, 0.20614950358867645, 0.817589521408081, 0.32385095953941345, -0.7572526335716248, 0.29742032289505005, 0.5436155796051025, 0.6337766647338867, 0.4932345449924469, 1.2341392040252686, -0.2826001048088074, 1.2647966146469116, 1.3503698110580444, 0.30422893166542053, 0.291166216135025, 1.0951170921325684, -1.891972541809082, 1.0720360279083252, 0.731789767742157, -0.5979081392288208, 1.3996747732162476, -0.030128559097647667, -0.285610169172287, 0.21707069873809814, 0.3848535418510437, -1.2201834917068481, 0.2615213394165039, -0.9052452445030212, -0.04593940079212189, 1.1870614290237427, 0.7446017265319824, 0.09091828763484955], [0.3636167645454407, 1.1722501516342163, -0.7152145504951477, -0.6603994965553284, 0.7831908464431763, 1.0518710613250732, 0.05876569822430611, 1.4046400785446167, 0.8175981044769287, -0.47788205742836, 2.315736770629883, -0.7388178706169128, 0.9276295900344849, -1.0473664999008179, 0.24751128256320953, 0.34026357531547546, -1.3917837142944336, -0.6180936098098755, 0.1699179857969284, 0.42868557572364807, 2.4229390621185303, -2.3110828399658203, 0.6918259263038635, 0.7398463487625122, -0.40309926867485046, -1.5253747701644897, 0.41534939408302307, -0.7282882332801819, 0.2145548015832901, -0.6600383520126343, -0.0011732480488717556, -0.189572274684906], [-0.7111399173736572, 0.5850725769996643, -1.0210567712783813, -1.2523244619369507, 0.035029150545597076, 0.18216952681541443, -1.149127721786499, 1.7739702463150024, -0.7077486515045166, -0.4405488669872284, -1.297729730606079, -0.12871238589286804, -2.0723605155944824, -0.7651450634002686, 0.14947059750556946, 1.1876739263534546, -0.013579349964857101, -0.5404225587844849, 1.3357638120651245, -0.24016216397285461, -2.298921585083008, -0.005801540799438953, -0.7187244892120361, 0.21953728795051575, 0.5624750256538391, -0.20079165697097778, 0.22383522987365723, -1.0674694776535034, 0.2291872501373291, 0.46609920263290405, 0.2947630286216736, 0.8930454254150391], [-0.8790578246116638, 0.16896507143974304, -0.7249350547790527, 1.6101832389831543, 0.2660936117172241, 0.6091605424880981, 0.24354521930217743, -0.4885053038597107, -0.26578882336616516, -0.0755661278963089, -0.08183153718709946, 0.3631387948989868, 0.7623686194419861, 0.11886485666036606, 0.9372273087501526, -0.7395417094230652, -0.5408776998519897, 0.010808194987475872, -0.25512638688087463, -2.232999086380005, 0.2163049280643463, 0.010461930185556412, -2.4037375450134277, 0.5138205885887146, 1.3146424293518066, 0.21878361701965332, -0.5006200075149536, 0.14863649010658264, 0.5580341815948486, -0.5808536410331726, -0.9318731427192688, 1.3846921920776367], [0.8976743221282959, 0.3452535569667816, 0.9531956315040588, -0.41968294978141785, 2.115279197692871, -1.255556344985962, -0.45148032903671265, -0.4140927195549011, -0.30038559436798096, 1.2491940259933472, 0.3867740035057068, 2.6819236278533936, 1.0081287622451782, 0.557030975818634, -2.6664254665374756, -2.0167489051818848, -0.8786680102348328, -0.31497514247894287, -0.6942921876907349, -0.394427090883255, 0.5462972521781921, -1.5057346820831299, 0.38857996463775635, 1.1180462837219238, -0.8054011464118958, -2.7566308975219727, -0.2825911343097687, 0.5080764889717102, 0.22203943133354187, -0.18922950327396393, 0.06857480853796005, 1.0351290702819824], [1.6214301586151123, 0.3595527410507202, -1.403423547744751, 0.1895717978477478, -0.44256502389907837, -2.5160725116729736, 0.07089520990848541, 0.8245778679847717, 0.3474844694137573, -0.48390117287635803, -1.3103777170181274, -0.23534785211086273, -0.011741469614207745, 0.03748044744133949, 0.44351381063461304, 1.1557685136795044, 0.42590799927711487, -0.8043362498283386, 1.3475289344787598, 1.8731189966201782, -0.651116669178009, -1.1471706628799438, 1.574945092201233, -1.2428690195083618, 0.5286309719085693, -0.4351092576980591, -0.3912210166454315, -1.235284686088562, -0.06364347785711288, 0.13686878979206085, 0.02872462011873722, 1.4372413158416748], [0.6026671528816223, 0.04128875955939293, 0.45277515053749084, -0.8091033697128296, 0.08573456853628159, -0.006760299671441317, -2.553988456726074, -1.0413641929626465, 0.49142831563949585, -1.5802451372146606, -0.22964216768741608, -0.8868263959884644, 0.264653742313385, -0.29913121461868286, 0.010196489281952381, 2.3694581985473633, -1.5540485382080078, 1.1567494869232178, 0.04683586582541466, 0.6547541618347168, 0.2525302767753601, -1.3245614767074585, -0.6239014863967896, 0.8968602418899536, -0.5836341381072998, 0.21717481315135956, -0.07540806382894516, -1.4841612577438354, 1.1758276224136353, -0.5514189004898071, -0.1070120707154274, -0.21411651372909546], [0.19655127823352814, 0.0798405334353447, -0.10372234880924225, 1.1161730289459229, 1.4072185754776, -0.4073134958744049, 0.3822420835494995, -0.20335043966770172, -1.0037919282913208, -0.9691647291183472, 0.07040825486183167, 0.2125043123960495, 1.0727359056472778, -0.31489744782447815, -1.0623842477798462, 0.5987516045570374, 1.3599504232406616, 0.35810860991477966, -0.6264346837997437, 0.9627121686935425, -0.252286434173584, 0.30071112513542175, -0.8320974707603455, 0.47283026576042175, 0.3597702383995056, -1.5890735387802124, 0.645871102809906, 0.15216244757175446, -1.1941115856170654, 0.7154822945594788, -0.19955995678901672, -0.022285999730229378], [1.743888020515442, -1.1958407163619995, 0.4548361897468567, 0.13900035619735718, 0.1310247927904129, 1.138386845588684, 0.29884105920791626, -0.2826770544052124, 0.8136121034622192, 0.8084145784378052, 0.6427261233329773, 0.8548035621643066, 1.3212108612060547, 0.9022470712661743, 1.233899474143982, 0.8611017465591431, -0.01756878010928631, -0.9701790809631348, 0.7132189273834229, 1.1130619049072266, 1.3258476257324219, -0.7961660623550415, 1.0568265914916992, 1.3377934694290161, -1.6447498798370361, -0.7458640336990356, 0.43071863055229187, -0.024283170700073242, 0.2630200982093811, 1.1885669231414795, 0.21458789706230164, 0.4512426555156708], [0.4749736487865448, -0.5031089186668396, -1.4258414506912231, 1.5126763582229614, 1.2966511249542236, 0.7405425906181335, 0.45485958456993103, 2.067976951599121, -1.2007479667663574, -0.4200926423072815, -0.3636986017227173, 0.5035310387611389, 1.0788848400115967, 0.8827081918716431, 0.2877501845359802, -0.15032047033309937, -0.0339309498667717, 0.1115858256816864, 0.5603495836257935, -1.1094179153442383, -0.7422361373901367, 2.0865318775177, 0.3193822205066681, -0.3318030536174774, 0.31130173802375793, 1.7997229099273682, -0.7174240946769714, 0.1963859349489212, -0.08987127989530563, -0.38254791498184204, 0.0120002506300807, -0.3961738348007202], [0.7133396863937378, 1.4014354944229126, 0.5598180890083313, 0.37421929836273193, 1.4463568925857544, -0.5195868015289307, 1.6229119300842285, 0.7568725347518921, 0.2811916470527649, -0.07564404606819153, -0.25974830985069275, 0.16246141493320465, -1.1478323936462402, -0.6794478297233582, -2.230973243713379, 0.1335543543100357, -1.1105196475982666, 0.7670174837112427, 0.8006375432014465, 2.269883632659912, -0.2941688895225525, -1.3410214185714722, 0.6085354089736938, 0.1098848506808281, 0.03807435929775238, -1.611620545387268, -1.0605525970458984, 0.3905503749847412, 0.26151901483535767, 0.0858636423945427, -0.37074100971221924, 0.7197871804237366], [-1.4748636484146118, 1.2386934757232666, -0.3988512456417084, -0.14470334351062775, 1.2299091815948486, -0.729326069355011, 0.15360789000988007, 0.10948352515697479, -0.9129863977432251, 1.3437180519104004, 0.16532976925373077, 0.2181047648191452, 0.27441883087158203, 0.2632857859134674, -0.9197088479995728, -0.7712104916572571, -0.6901306509971619, 0.7984473705291748, 0.8519716858863831, -1.5705760717391968, 0.4187368154525757, 1.265354871749878, -0.6253599524497986, 1.8975868225097656, -2.1513662338256836, 0.6377567052841187, -0.5540189146995544, 0.6430092453956604, -0.029605885967612267, 2.1085519790649414, -0.485213041305542, -1.0881006717681885], [0.6008411049842834, 0.9674563407897949, -0.6070663928985596, -1.3551397323608398, 0.720630943775177, 0.5562260150909424, 0.4007544219493866, -0.37957990169525146, 1.0681953430175781, -0.20647940039634705, -0.7922423481941223, 1.2849127054214478, -1.350956916809082, -0.73710036277771, 0.2917100489139557, 0.13133779168128967, 0.22324880957603455, -0.9224900603294373, -0.5985824465751648, -0.7375795841217041, 1.430311918258667, -0.5708227753639221, -1.2132105827331543, -0.6081411838531494, 0.9839770793914795, 1.3309543132781982, -0.7375630736351013, 0.32197874784469604, 0.09063203632831573, 1.0619598627090454, 0.42297428846359253, 1.0065138339996338], [-1.1649142503738403, -1.0158902406692505, -2.3025712966918945, 0.8432151675224304, -1.6514912843704224, -1.3596808910369873, -0.3300933539867401, 0.5026426315307617, 0.48179954290390015, -0.17000006139278412, -0.6674867272377014, 0.6761403679847717, -0.7940565347671509, 0.8525412678718567, -0.4736805856227875, 0.40402883291244507, 0.8166978359222412, -1.5603476762771606, -0.3982624411582947, 0.07029487192630768, 0.01478483621031046, -0.3777097761631012, -0.6699038147926331, -0.8809071183204651, -0.19204291701316833, 2.7805838584899902, 0.06006268411874771, -0.5110993981361389, 0.19180762767791748, -2.3357222080230713, 0.465575248003006, 0.3501620590686798], [0.36108657717704773, -1.5439866781234741, -0.5243462324142456, 0.015305567532777786, 1.775984525680542, 1.4965473413467407, -0.868240237236023, -0.9916673302650452, 1.1474279165267944, -1.4299180507659912, -0.9807472229003906, 0.274262011051178, 0.9572847485542297, -0.6803134083747864, 0.06463723629713058, -0.20456768572330475, 0.3788049519062042, 1.5425286293029785, 0.2814287841320038, -0.45390358567237854, -1.1222206354141235, 0.1572461873292923, -0.015755146741867065, -0.7042990922927856, 0.3458525538444519, 0.7876753807067871, 1.7677669525146484, 0.7458828091621399, 0.7512679100036621, 1.2002067565917969, -0.5053747296333313, -0.31822097301483154], [-0.4410586357116699, 0.16121269762516022, 1.1928014755249023, -0.7538695931434631, 1.273054599761963, 0.03893401101231575, -1.9581886529922485, 1.3871252536773682, -0.6114303469657898, -0.9709621071815491, -0.836286723613739, 1.7424005270004272, 2.196847677230835, -0.1567208617925644, 0.31943151354789734, 0.8285279273986816, 0.14535903930664062, -0.10172183066606522, -0.4322330355644226, -1.4594874382019043, 0.3765794038772583, -0.14329028129577637, -0.8398863077163696, -0.9891769886016846, 0.30135202407836914, -0.13878273963928223, -1.770637035369873, 0.12589623034000397, -0.221719890832901, 1.0483170747756958, -0.9663041234016418, 0.3811625838279724], [-0.25307729840278625, -1.7503361701965332, -0.2558869421482086, -0.3527265191078186, -2.258923053741455, 0.5150963068008423, 1.9508312940597534, -0.4489188492298126, 0.4220092296600342, 0.5457028746604919, -0.8447533249855042, -0.14269378781318665, 0.802657425403595, -0.5169216394424438, -0.13159199059009552, 0.13956761360168457, 0.22232332825660706, 2.1235108375549316, 0.7741706967353821, -0.5898328423500061, 1.9202988147735596, 0.8177593350410461, -1.1034725904464722, 0.9597660303115845, -1.766526222229004, 0.44328367710113525, -0.22073176503181458, 1.8686593770980835, 1.6259620189666748, 1.1083570718765259, -0.7181217074394226, -0.399241179227829], [1.83888840675354, -1.5541294813156128, 0.5100775361061096, -0.41651540994644165, 1.1170308589935303, 0.2384900450706482, 2.2099547386169434, 1.5121939182281494, -0.5105745196342468, -0.13512708246707916, -0.6876676678657532, -1.9140653610229492, -0.7724896669387817, 0.5079299807548523, 0.4867548942565918, -0.09939396381378174, -0.8522717952728271, -0.15783466398715973, 1.9715533256530762, -0.32317784428596497, 0.004959234967827797, 0.5762633681297302, 0.20305345952510834, -0.8105824589729309, 0.1806710660457611, -0.15744027495384216, -2.1465792655944824, -0.9094575047492981, -0.5765203237533569, -0.6390260457992554, -1.5329023599624634, -0.8721157908439636], [-0.8622320890426636, -0.5107072591781616, 0.27305376529693604, 1.019966959953308, -0.30353671312332153, 0.009940287098288536, -1.1842213869094849, 0.7593536972999573, 1.9133098125457764, -0.15929622948169708, 0.05548468977212906, 0.7053302526473999, 0.2826721668243408, -2.1300957202911377, -1.2398172616958618, 0.24280983209609985, 0.1568174660205841, -1.914651870727539, -0.19787320494651794, 0.8454154133796692, 0.9171520471572876, -1.077368140220642, -0.6331356763839722, 0.6699633598327637, -1.1730537414550781, 0.15454231202602386, 0.3152340352535248, 0.8861495852470398, 0.2893174886703491, 0.7539419531822205, -0.303363174200058, -0.7122785449028015], [-0.1908593326807022, 0.9449055194854736, 1.6573385000228882, 0.3697237968444824, -0.0967305526137352, -0.16166050732135773, 0.20090708136558533, -0.6270011067390442, -0.14212536811828613, 0.5563234686851501, -1.2792142629623413, -1.9067779779434204, -0.7399492263793945, -0.7626221179962158, -0.03905203938484192, 0.195418581366539, 0.21924619376659393, 0.5003939867019653, 0.8616652488708496, 0.7949460744857788, 1.1345546245574951, -1.8809195756912231, 0.6776459813117981, 0.5230711102485657, 0.46529361605644226, -1.0347787141799927, -0.4184075593948364, -0.0231882743537426, -0.2713395357131958, 0.970593273639679, 0.8140037655830383, 1.0225520133972168], [0.1876494139432907, -0.09595227241516113, -0.34591925144195557, -0.7985976338386536, 0.8154292702674866, 0.9781477451324463, -0.3501882255077362, 0.4563015103340149, -0.9628342390060425, -0.8812041282653809, 0.4810534119606018, -0.9420708417892456, -1.6728885173797607, -0.6588429808616638, 0.015534840524196625, 0.7253905534744263, -0.14467938244342804, -0.08316252380609512, 0.7026206254959106, 0.2278217375278473, -0.5641582012176514, -1.4166090488433838, -0.548856258392334, 0.13944029808044434, -0.861166775226593, 0.6050404906272888, 0.8064576387405396, -0.06979626417160034, 1.0236341953277588, -0.2795502543449402, -0.09012290835380554, 0.9713252782821655], [0.0751929059624672, 0.34118276834487915, -0.7833627462387085, -0.3493199646472931, 1.555732011795044, -0.6937333345413208, 2.3130667209625244, -1.5976598262786865, 0.16948702931404114, -0.08049073070287704, -1.788372278213501, 0.3061445355415344, -1.2429313659667969, 0.2073083072900772, -0.6631945371627808, -0.9710310697555542, -0.38474026322364807, -0.23976342380046844, 0.665529191493988, -0.9761400818824768, -0.1675347089767456, 0.16403242945671082, -1.558376431465149, -1.1920865774154663, -1.8443913459777832, -0.43680739402770996, -0.30077311396598816, -1.413968801498413, -0.9433932900428772, -2.5185577869415283, -0.15636372566223145, 0.6937729120254517], [-0.9564032554626465, 1.441569447517395, -0.5537344217300415, 0.6387203335762024, -1.3361897468566895, 0.7893281579017639, -0.9245087504386902, -0.3307451605796814, 0.006736517883837223, 1.5507783889770508, 0.4110122621059418, -0.02389788068830967, -1.4566998481750488, 0.4128197133541107, -0.7614941000938416, -0.14151929318904877, 0.28676581382751465, -1.0670665502548218, -1.1719330549240112, 2.221566915512085, -1.8412034511566162, 0.3128265142440796, -0.5009629726409912, 0.13145944476127625, 0.2748382091522217, 1.234745740890503, -2.3050734996795654, 0.524129331111908, -0.32579106092453003, 0.1452007293701172, -0.15315163135528564, 0.6030914783477783], [0.6657822132110596, 0.1806323081254959, -1.14404296875, -0.665409505367279, -0.2658118009567261, -0.833327054977417, 0.12505434453487396, 0.5880513191223145, -0.5242913365364075, -0.036703500896692276, 0.11456893384456635, 1.5898783206939697, 0.410454660654068, -0.6718758940696716, 0.26719924807548523, 1.433190107345581, 0.9530278444290161, -0.9860666394233704, -0.2402103692293167, -0.7593211531639099, 0.35996749997138977, -1.021291971206665, -0.6131392121315002, -1.3623820543289185, -1.6094026565551758, 0.39734697341918945, -0.551444411277771, 0.9981251358985901, 1.2505931854248047, 0.07795967161655426, 0.3898647427558899, 1.1191405057907104], [-2.0463438034057617, -1.2807221412658691, 0.5362564325332642, -1.0136123895645142, -0.15079374611377716, 2.789057731628418, -1.2800568342208862, -0.28291055560112, 0.2837042510509491, -0.15357337892055511, 1.1969965696334839, -0.440277099609375, -0.7086054086685181, -1.7831153869628906, 0.22357602417469025, -0.15335455536842346, 2.092714786529541, -0.33468061685562134, -1.187612771987915, 0.5750643610954285, -1.8312997817993164, -0.5671773552894592, 0.5360913872718811, 0.225680410861969, 1.2322862148284912, 0.4877065122127533, -0.09244129806756973, -1.6490063667297363, 1.9570790529251099, 0.2723363935947418, -0.7597178816795349, 0.648038923740387], [-1.4139255285263062, -1.8074361085891724, 1.3683953285217285, -0.7372972369194031, -0.18499431014060974, 1.303147315979004, 0.44135695695877075, 0.9286194443702698, 0.07032351940870285, 0.1535320281982422, 0.059997934848070145, 0.4299863278865814, 0.6332555413246155, -0.8501521944999695, -0.3916700482368469, 1.4805890321731567, 0.058261144906282425, 0.6013527512550354, -1.4582598209381104, -1.0371724367141724, 0.9919149279594421, 1.045662760734558, -0.16229033470153809, 0.5797756314277649, -0.13662289083003998, -0.4940266013145447, 0.5235093832015991, 0.516801655292511, 1.297874927520752, -0.02768096700310707, -0.38643312454223633, 1.345384120941162], [-0.5606477856636047, 0.32427775859832764, 0.29364311695098877, 0.4685075879096985, -0.05505862459540367, 0.5287291407585144, -0.605392336845398, 1.091181993484497, -0.9662637710571289, 0.9069308638572693, -0.5169322490692139, 0.8763180375099182, -0.9262341856956482, -1.781969428062439, 1.7000588178634644, -0.5836461782455444, -0.9237158298492432, -1.6248083114624023, 0.6686537265777588, -0.19697672128677368, -0.9488831758499146, -1.6135624647140503, -1.3661532402038574, -2.000882387161255, 1.5997554063796997, 0.8061766028404236, -1.8203407526016235, 0.5395367741584778, 0.7505868077278137, -0.960774838924408, 1.5985655784606934, -2.245034694671631], [1.3128985166549683, 1.4948251247406006, 0.023583853617310524, 0.1598360240459442, -1.2220145463943481, -0.7702428698539734, 1.2467397451400757, 0.5033196806907654, -0.5971153974533081, -1.5428669452667236, 1.419324278831482, 0.19880415499210358, 0.12927675247192383, -0.46009358763694763, -1.6403287649154663, 0.32062289118766785, 0.9229691624641418, 0.19871214032173157, -0.7643494606018066, -0.9005821347236633, -0.10113801807165146, -0.3651140332221985, 1.8952674865722656, 1.4593505859375, 0.3953048884868622, 0.16850236058235168, 1.4592275619506836, -1.867455244064331, -0.5734171867370605, -1.277215838432312, -0.9691019058227539, 0.7203426361083984], [-0.9187887907028198, 1.5323930978775024, -0.1487221121788025, -0.7082316279411316, -0.9061970114707947, -1.2866560220718384, 0.10798376798629761, 0.6175021529197693, -1.4400030374526978, 0.10125607997179031, 0.9510864019393921, 0.7323089241981506, 0.3215796649456024, -0.023497821763157845, -0.2239730805158615, 0.4299947917461395, 0.7663543820381165, -1.2689940929412842, 0.675331711769104, -0.630513608455658, 0.5423393249511719, 0.365190327167511, -0.2828259766101837, -0.3610628545284271, 0.012336942367255688, 0.15827445685863495, 0.8805466294288635, 0.25505331158638, 0.3907768428325653, -0.043431539088487625, 0.2657829821109772, -0.7275839447975159], [1.1445788145065308, 0.1989409327507019, -0.496846079826355, 0.8106191158294678, -0.16765449941158295, 0.8649848699569702, -0.030867766588926315, 0.6517370939254761, 0.4368166923522949, -0.20130614936351776, 0.00654766708612442, 0.7993689179420471, 0.5177685618400574, -1.4385364055633545, 0.3406665325164795, 0.5303531289100647, -0.16828125715255737, 1.1712456941604614, 0.22217710316181183, -2.202500343322754, -0.4681161940097809, 0.6501945853233337, 0.043039437383413315, 0.5022656917572021, 0.2833259701728821, -1.7841500043869019, 1.5363471508026123, -0.5938145518302917, 0.7598727345466614, -0.6765064597129822, 0.5757693648338318, 0.36459341645240784], [-0.23083704710006714, 0.6860819458961487, -0.3056527078151703, -1.7703577280044556, 1.3997409343719482, 1.368628978729248, 0.13865119218826294, -0.17037193477153778, -0.5065910816192627, 1.0164226293563843, 0.6953203678131104, -1.0530060529708862, 1.2087271213531494, -0.3036107122898102, 0.4166851341724396, 1.6880204677581787, -1.0112179517745972, -0.09727593511343002, -0.902253270149231, 0.7613644003868103, 2.7673144340515137, -0.4842943847179413, 0.786240816116333, -0.40164828300476074, -1.4340474605560303, -1.0311583280563354, -1.0668177604675293, -0.6160726547241211, 0.23883123695850372, 1.1410267353057861, 1.1877273321151733, 0.8201329708099365], [0.7896323800086975, 1.8501040935516357, -0.8624826073646545, 0.43165871500968933, -1.1067652702331543, -1.5397402048110962, -0.7307540774345398, 0.24784107506275177, -0.16221608221530914, -0.9824053049087524, -1.7946885824203491, 0.4854113459587097, 1.0970851182937622, 0.40561723709106445, -1.1986465454101562, 1.8710272312164307, 1.243577241897583, -2.031196117401123, -0.08620373904705048, -0.050423961132764816, -0.053041379898786545, 1.812962293624878, -0.08075030148029327, -1.6820814609527588, 0.7420558333396912, -1.2146049737930298, 0.014863484539091587, 1.6583364009857178, 1.2389527559280396, 1.0988551378250122, -1.1028798818588257, -0.17279323935508728], [-1.5830647945404053, -0.5820047855377197, -0.5120915174484253, -0.5020730495452881, 0.9726109504699707, -0.9745843410491943, 2.6450510025024414, 1.2479151487350464, -0.189921572804451, -1.5607556104660034, -0.10693787783384323, 0.12703880667686462, 0.730261504650116, 1.1968127489089966, 0.054390985518693924, -1.630421757698059, -0.12357445061206818, -0.6906689405441284, 0.6861582398414612, -0.09196819365024567, 1.636783480644226, 0.5966131091117859, -0.24238848686218262, 2.233682632446289, -0.923194408416748, 1.3807315826416016, 0.4454313814640045, 2.2896008491516113, -0.8845624327659607, -0.22357644140720367, -0.4699329733848572, 1.8151406049728394], [-0.6868827939033508, -0.556235134601593, -1.7063243389129639, 0.3156282901763916, -0.7226455211639404, 1.7532140016555786, -1.0280518531799316, -0.03972677141427994, -1.0474693775177002, 0.7318886518478394, 0.5715288519859314, 0.9902028441429138, -0.1309160441160202, 0.4810701608657837, 0.1303200125694275, 2.246324062347412, 1.1495556831359863, 0.5820081233978271, -2.7457997798919678, 0.5270251631736755, -0.716155469417572, -1.1048964262008667, -0.16488152742385864, 0.3529406785964966, 0.23227891325950623, 0.7547061443328857, -1.7539595365524292, 0.3700345754623413, -0.5548334717750549, 0.898605465888977, 0.43576034903526306, 0.592780590057373], [-0.013210644945502281, -0.8053457736968994, 0.8911877870559692, 0.9329994320869446, -1.0131045579910278, 1.5884640216827393, -0.680094301700592, -0.3184449076652527, 0.24297340214252472, -1.3565926551818848, -0.6245326995849609, 1.634719967842102, -0.6399205923080444, -0.6720808148384094, 0.019373809918761253, 0.36447903513908386, -0.28627264499664307, 0.46324485540390015, 0.9251227974891663, -0.4937773644924164, -0.8281036019325256, -0.12924663722515106, 0.7828741073608398, -1.0937248468399048, 0.5176827907562256, 0.5094234347343445, -0.1904406100511551, 2.4854536056518555, -0.3381505310535431, 0.24849160015583038, -0.27159056067466736, -1.1217081546783447], [-0.958389163017273, 0.31977614760398865, 0.028966160491108894, 0.4928528666496277, -1.0384423732757568, -0.11855322867631912, -2.781384229660034, 0.6371845006942749, 0.1482221782207489, -0.4502577483654022, 0.3567178547382355, 0.3465360999107361, 0.6150719523429871, 2.0486772060394287, 1.7154344320297241, -1.4129676818847656, 0.043044526129961014, 0.20101772248744965, -0.18071416020393372, 2.2819581031799316, 0.7211291790008545, 1.0864429473876953, 0.035310931503772736, -3.249271685490385e-05, -0.8548387885093689, -0.7826257944107056, 0.9928581118583679, 2.2632133960723877, 2.94671368598938, -0.230141744017601, 0.1992502510547638, 0.6455627083778381], [0.678812563419342, -0.8150573372840881, -1.0640101432800293, 0.1178106963634491, -0.020282499492168427, -0.015554417856037617, -0.5202929973602295, 1.448245644569397, 0.5365598797798157, -0.3772469758987427, 2.3804688453674316, -1.1688114404678345, 0.6213590502738953, 2.257164239883423, 0.8415716886520386, -0.6886373162269592, -0.3771471381187439, -0.2399650663137436, 0.11432837694883347, 0.3019096553325653, -1.1650598049163818, -1.1188905239105225, -0.00011865363194374368, -0.26895079016685486, 0.0748281255364418, 0.4495811462402344, 0.17280060052871704, 0.8344967365264893, -0.9061663150787354, 0.6313430666923523, 0.11731022596359253, -0.16717524826526642], [1.6846842765808105, 0.36704757809638977, -0.24387404322624207, 0.005281543824821711, -0.6299365758895874, -1.5162771940231323, -1.3167190551757812, -0.23939304053783417, 1.168479323387146, -1.3361496925354004, 1.456875205039978, 1.3860806226730347, -0.3946438729763031, -0.7897989153862, -1.0203133821487427, -1.033982753753662, 0.3936573266983032, -1.1449201107025146, -1.2323764562606812, 0.6336621642112732, -2.072911024093628, -1.0430405139923096, 0.2821662425994873, -1.4138576984405518, -1.4197536706924438, -0.10945772379636765, -0.35519328713417053, -0.7585960030555725, -0.6765955686569214, 0.3739003837108612, -0.8495618104934692, 0.7642538547515869], [0.14260077476501465, 0.052790507674217224, -0.07893484830856323, 1.0616270303726196, 0.06301931291818619, -0.5016964077949524, -1.1560018062591553, 0.6629073023796082, 0.26548895239830017, -0.8005719780921936, 0.1927931010723114, 2.203277587890625, -0.4020061790943146, -0.5121482610702515, 1.9402457475662231, -0.7830322980880737, -1.1935389041900635, -0.31108319759368896, 0.27684733271598816, -0.6999903321266174, 0.504706859588623, -0.27404916286468506, 1.7580828666687012, 1.0724060535430908, 0.9314202666282654, 0.45917248725891113, -0.5945932269096375, 0.3486199378967285, 0.38157111406326294, -0.8840613961219788, -0.1264266073703766, 1.0480623245239258], [0.401268869638443, 2.5962979793548584, 0.0074392035603523254, -1.4337103366851807, 0.8008620738983154, 0.36578208208084106, -1.6750471591949463, 0.631294310092926, 1.1254584789276123, -0.8780521750450134, 0.3618755042552948, 0.5070745348930359, -1.058454990386963, -0.07402323186397552, 1.2476304769515991, 0.4325467050075531, -0.16980071365833282, -1.8754397630691528, 1.001563310623169, -2.027799129486084, -2.128126621246338, -0.19234547019004822, -0.9351896047592163, -0.5307635068893433, 0.3026488125324249, -1.016229510307312, -0.4448508322238922, -1.1278470754623413, 2.4195973873138428, -0.10747075080871582, -0.2536918520927429, -0.020441709086298943], [-0.16971847414970398, 0.3749472200870514, -1.4743540287017822, -0.3422280251979828, 0.4210119843482971, 1.2748849391937256, 0.4013982117176056, 1.3536871671676636, 0.5932406187057495, -0.2805182635784149, -0.1216621994972229, -1.3343788385391235, 1.2471795082092285, 1.9941505193710327, 0.7914679646492004, -0.39748334884643555, -0.18230599164962769, 0.6381914615631104, -2.6797022819519043, 0.9591124057769775, 1.600490689277649, -1.0126076936721802, 1.3515994548797607, 0.4135955274105072, -0.455215185880661, -0.30872678756713867, -0.5477104187011719, -0.48578938841819763, 1.036169171333313, -0.7835978865623474, -0.30162742733955383, -0.4661358892917633], [-0.31315186619758606, 2.160984992980957, 0.48326608538627625, -0.17669574916362762, -1.003569483757019, 0.22282882034778595, -1.4553108215332031, -0.398811012506485, -0.05458148568868637, 0.10352707654237747, 2.2528433799743652, 0.8495749831199646, -0.7686076760292053, 0.4526778757572174, -2.0683200359344482, 0.4985591769218445, 3.0519726276397705, 0.4825485348701477, -1.6926568746566772, -0.8522558808326721, -1.0136734247207642, -2.1481430530548096, 0.58461993932724, 0.8582673072814941, -0.13932256400585175, -1.20894455909729, 0.8043388724327087, 2.6387722492218018, -2.7692999839782715, 1.0877982378005981, 0.2080053836107254, -0.3861183226108551], [0.9307947754859924, 0.17873138189315796, -1.3294188976287842, -0.7790887355804443, -0.9606648683547974, -1.5619797706604004, -0.25694629549980164, 1.1763399839401245, -0.20182496309280396, 0.4169243574142456, -1.0218876600265503, -0.20860010385513306, -1.0310579538345337, 1.646209478378296, -0.0009892407106235623, -1.1636453866958618, 1.17281174659729, 0.2809160053730011, 1.0547254085540771, -1.5439085960388184, 0.8885408639907837, -1.2474128007888794, -0.29453518986701965, 1.203719973564148, -0.13369601964950562, -0.11278633028268814, -0.15461936593055725, -1.298414707183838, 0.17028623819351196, -0.5640543699264526, -0.6543647646903992, -1.1711225509643555], [-0.22608841955661774, -0.5916053056716919, 0.1887812614440918, -0.1741274744272232, -0.37430834770202637, -0.11367085576057434, -0.35777825117111206, -0.4116499423980713, 1.0253885984420776, -2.1677660942077637, 0.7199142575263977, -0.8021821975708008, -2.6465401649475098, -0.05751035362482071, 1.3646390438079834, 0.2933403253555298, 0.3257642984390259, -2.750460624694824, -1.0211306810379028, -0.1240791529417038, -0.05664750933647156, 0.07109735906124115, -1.7775626182556152, 0.16736038029193878, 1.2071622610092163, 0.27325156331062317, 0.6781629920005798, 0.4856957495212555, -2.297837495803833, -0.5546560287475586, 2.1191275119781494, -1.2797589302062988], [-0.9279884099960327, 1.2211014032363892, 0.7751157283782959, -2.1035537719726562, -1.4872945547103882, 1.1185694932937622, -0.6652461290359497, 0.28051090240478516, -1.0119378566741943, -1.0300923585891724, -0.5788179039955139, -1.5625786781311035, 0.9936971068382263, 0.12105771899223328, -0.1770615130662918, 1.211055874824524, -0.3541201949119568, 0.29291388392448425, -1.3031328916549683, 0.5010689496994019, -0.4374904930591583, -0.0549699105322361, -0.41597941517829895, 0.5664204359054565, -0.48041191697120667, -0.4861515164375305, 0.5564119815826416, -0.7193665504455566, -0.8008739948272705, 0.06969808787107468, -0.7075362801551819, 0.9677611589431763], [0.09566772729158401, 0.871388852596283, 0.5205419659614563, 0.09350422769784927, 0.8498883843421936, -0.5166255831718445, -0.6685552597045898, 0.7498165965080261, 0.5234130620956421, -0.7832129001617432, -0.7542692422866821, 0.6303520202636719, -1.0978318452835083, -0.1114141047000885, -0.7431883811950684, 1.3844314813613892, -0.29163458943367004, 1.2539501190185547, 0.4510977864265442, 3.055795669555664, -1.33938467502594, 1.258989691734314, -0.513885498046875, -1.311523675918579, 0.24761202931404114, -2.1480722427368164, 0.1304992437362671, -2.0302951335906982, -0.647850513458252, 1.3083347082138062, 1.3582402467727661, 1.1788169145584106], [1.234850525856018, 0.16219137609004974, -0.5597047209739685, -0.7045463919639587, 0.5360165238380432, 0.4848795533180237, -0.612532913684845, -1.4185383319854736, 0.14171738922595978, -1.3854851722717285, 3.7061572074890137, -0.925551176071167, 1.0095851421356201, 0.9955487847328186, -1.3968671560287476, -1.1072485446929932, 0.016728734597563744, 0.6137100458145142, -0.0844544917345047, -2.346731662750244, 0.9602729678153992, -1.132742166519165, -1.0084739923477173, -1.3729206323623657, -0.33960238099098206, -0.8682615756988525, -0.2513824999332428, 0.4233294129371643, 1.1802186965942383, -0.144254669547081, -2.164485216140747, -0.2556951940059662], [2.1088669300079346, 1.1782773733139038, -2.8247275352478027, 1.233175277709961, 2.4887211322784424, 1.136043906211853, 1.100469946861267, 0.9308897852897644, 0.10094422847032547, -0.02653774805366993, 0.9083611369132996, -1.4736990928649902, 0.6796696782112122, -0.6314210891723633, -0.010595526546239853, 0.8560630679130554, -1.1077591180801392, -0.01569744385778904, 0.6999229788780212, 0.3911619186401367, 1.2095739841461182, 1.2643746137619019, 0.54673832654953, -1.2754826545715332, -1.068519115447998, -0.7121805548667908, 0.29766589403152466, 0.6624121069908142, 1.4813776016235352, -0.6385601758956909, 0.6482464075088501, 0.23367789387702942], [1.0202219486236572, -0.8506918549537659, -0.41852110624313354, 0.7603725790977478, -1.727078914642334, 0.6588780283927917, 0.8987589478492737, 1.4726579189300537, -0.07291381061077118, 0.40824007987976074, 0.20538853108882904, 0.08574937283992767, -0.7287930846214294, 0.40139931440353394, -1.5319850444793701, -0.6548436284065247, -0.39126190543174744, 0.29933637380599976, -0.01909610815346241, 1.2097818851470947, 1.2272343635559082, -0.08951369673013687, -2.0259039402008057, 0.8429383635520935, 1.1198480129241943, -0.32105302810668945, -1.9695557355880737, 0.13444161415100098, -2.083293914794922, -0.9162159562110901, 1.6923270225524902, 1.173518419265747], [-0.2565756142139435, -1.5481524467468262, 0.2995416522026062, -0.14473026990890503, -0.6592455506324768, -0.8272826671600342, 0.31345564126968384, 0.1856921911239624, -0.6533248424530029, -0.8023183941841125, -1.360005259513855, -0.030399009585380554, -2.0872182846069336, 0.5516906380653381, 1.9192038774490356, 0.6315039992332458, -0.713774561882019, -0.7134467959403992, 1.9701464176177979, -2.6597728729248047, 0.5462008118629456, 0.6729850172996521, -0.8047307729721069, -1.2315735816955566, 0.5331286191940308, -0.3263229429721832, -0.20482629537582397, 1.0199296474456787, -0.7012523412704468, -0.22811326384544373, -0.2715452015399933, -1.266308069229126], [1.9028667211532593, 1.8549489974975586, 1.392328143119812, 0.2569591701030731, -0.07527906447649002, -0.6915037631988525, 0.5344788432121277, 1.1558066606521606, 1.415128231048584, -0.3183157742023468, -1.5634766817092896, -2.0504019260406494, 2.6421360969543457, 0.26889899373054504, -1.322201132774353, -0.4677058160305023, -0.7757973670959473, -0.3004724383354187, 0.0997546836733818, -1.3185908794403076, 0.8605034947395325, 0.5373196601867676, -0.9317695498466492, 1.9497839212417603, 2.2139852046966553, 0.9597598314285278, -1.8470348119735718, 1.004101276397705, 1.1329559087753296, 0.4232461452484131, 0.6748434901237488, 0.4469982385635376], [0.6694958806037903, -0.6896273493766785, -1.0030373334884644, 0.5167791843414307, 1.8278342485427856, -1.5059473514556885, 0.20913740992546082, -0.245247021317482, -0.8456830978393555, 0.831230640411377, 1.0166196823120117, 1.2527797222137451, 0.498924195766449, 0.06517095863819122, 2.1636204719543457, -1.0382025241851807, -0.11202438175678253, -0.8889426589012146, 0.8267565369606018, -0.009532555006444454, 0.1073668822646141, 1.1489579677581787, 0.148797869682312, -0.35211899876594543, -2.5679800510406494, -0.680475652217865, -1.6353449821472168, 0.49662044644355774, 0.036187201738357544, -0.3637681305408478, 0.4129868745803833, -1.7021970748901367]]]], "dtype": "float32"}, false], "e": {"__tup__": [{"__nd__": [[[[-0.04866810142993927, -0.18590721487998962, 0.09048973768949509, -0.231441929936409, -0.0849895104765892, 0.3085799515247345, -0.18017306923866272, 0.046749297529459, -0.15122734010219574, -0.12927879393100739, -0.08619777858257294, -0.07843766361474991, 0.3807068169116974, 0.033317506313323975, 0.02169855497777462, 0.09579405188560486, -0.014554718509316444, 0.09695538878440857, -0.06793463975191116, 0.17082971334457397, 0.44691672921180725, -0.03831245005130768, -0.30382195115089417, -0.1057002916932106, 0.37560170888900757, -0.09200800210237503, -0.08152555674314499, 0.044091273099184036, 0.11426369100809097, 0.40419697761535645, 0.0678352490067482, 0.1608540266752243], [0.11031799018383026, 0.08718695491552353, -0.263827383518219, -0.18542803823947906, 0.22638319432735443, 0.3191049098968506, -0.3760462999343872, 0.04871581122279167, -0.1752297431230545, 0.2766224145889282, 0.30833956599235535, 0.10633248835802078, 0.1302362084388733, 0.3698364794254303, 0.2074664682149887, -0.12459393590688705, 0.008156538009643555, -0.003786297282204032, -0.20957107841968536, -0.19246643781661987, -0.2614968419075012, 0.11646124720573425, 0.05464060604572296, 0.0965438261628151, 0.2737911343574524, -0.1620771586894989, 0.2461342215538025, 0.07410745322704315, 0.22786028683185577, 0.35930538177490234, 0.1722688525915146, -0.19580596685409546], [0.13969731330871582, -0.3261532187461853, 0.22086457908153534, -0.3046506941318512, 0.21674908697605133, 0.29289162158966064, 0.10145895928144455, -0.17369301617145538, 0.15058282017707825, 0.17429664731025696, -0.2792828381061554, 0.005836335476487875, 0.24275217950344086, 0.06267562508583069, -0.017193783074617386, 0.14016851782798767, -0.1719161719083786, -0.19969891011714935, -0.057323433458805084, -0.09883157163858414, -0.17009158432483673, 0.08665731549263, -0.1199130117893219, 0.011007540859282017, 0.2233012169599533, 0.013891308568418026, 0.05145210400223732, 0.07951776683330536, 0.15718720853328705, 0.2917427718639374, -0.07965872436761856, -0.059326235204935074], [-0.08185387402772903, -0.23612448573112488, 0.08266813308000565, -0.23270738124847412, 0.1621020883321762, 0.05283080041408539, -0.013512026518583298, -0.08452348411083221, 0.03854001313447952, 0.0865815132856369, 0.09951808303594589, -0.06024801731109619, 0.06576839834451675, 0.011691001243889332, 0.10008309781551361, 0.003356018103659153, -0.027543501928448677, -0.20022711157798767, 0.03084813430905342, 0.025662286207079887, 0.08731723576784134, -0.026234565302729607, -0.1735876500606537, -0.12762968242168427, 0.11949291825294495, -0.08295529335737228, -0.05628034099936485, -0.10998757928609848, 0.04815823584794998, -0.044379085302352905, 0.01618141494691372, 0.02215507999062538], [0.1951943188905716, -0.22305597364902496, 0.25270023941993713, 0.14596687257289886, -0.022858180105686188, 0.3085925877094269, -0.015954138711094856, -0.10438115149736404, -0.0466279499232769, 0.3569793105125427, 0.16689884662628174, -0.22365354001522064, 0.4933415651321411, 0.2532181441783905, -0.006205175071954727, 0.24059708416461945, 0.024286409839987755, 0.029078036546707153, -0.1551734209060669, 0.018922319635748863, 0.15024098753929138, -0.05325203388929367, -0.2652930021286011, 0.09179205447435379, 0.36981892585754395, -0.11961483955383301, 0.28379443287849426, 0.26897358894348145, 0.16803696751594543, 0.43137991428375244, 0.028171364217996597, 0.13894398510456085], [0.22712190449237823, -0.06798770278692245, 0.23356972634792328, -0.4630832374095917, -0.04094812273979187, 0.08596249669790268, -0.02476823888719082, -0.09661654382944107, 0.03693360090255737, -0.12139733135700226, -0.16800960898399353, -0.05954829230904579, 0.29510897397994995, 0.29836151003837585, -0.06486773490905762, -0.03154873475432396, -0.07624030858278275, 0.10491636395454407, 0.03920675069093704, 0.04878701642155647, 0.14610649645328522, 0.05849157273769379, -0.22823482751846313, 0.0612419992685318, 0.18185165524482727, -0.12736466526985168, 0.053685013204813004, -0.17207589745521545, 0.05377275496721268, 0.18599112331867218, -0.03409620374441147, 0.11019470542669296], [-0.11444270610809326, -0.0036204741336405277, 0.08853194117546082, -0.04839435592293739, 0.10477065294981003, 0.1670243740081787, -0.31018364429473877, -0.41958853602409363, -0.27742379903793335, 0.3102364242076874, 0.05459331348538399, -0.11089568585157394, 0.14994220435619354, 0.22510622441768646, 0.02124018780887127, 0.21848846971988678, 0.08511772751808167, -0.121104396879673, -0.18916836380958557, -0.16782471537590027, -0.023914460092782974, -0.15747714042663574, -0.15581320226192474, 0.07557275146245956, 0.3520275056362152, 0.020257987082004547, 0.08819826692342758, 0.1408730149269104, 0.13142861425876617, 0.1343858242034912, 0.04949892684817314, -0.009941425174474716], [0.014942215755581856, -0.01988726109266281, 0.05330784246325493, -0.24533575773239136, 0.10873325169086456, 0.08291658014059067, -0.2847102880477905, -0.06560149788856506, 0.05817022919654846, 0.2113134115934372, -0.06488099694252014, -0.14753898978233337, 0.08828441798686981, 0.16235123574733734, 0.2886154353618622, 0.29506465792655945, 0.008654437027871609, -0.01967710629105568, -0.13205918669700623, -0.13772156834602356, 0.2839697599411011, -0.045036591589450836, 0.13751743733882904, 0.027551593258976936, 0.12213801592588425, 0.001214734511449933, -0.045811284333467484, -0.025289740413427353, 0.1835489720106125, 0.08422654122114182, -0.30880826711654663, 0.09601637721061707], [0.05497909337282181, -0.1194538101553917, 0.13239364326000214, -0.3387240767478943, 0.20333652198314667, 0.09999377280473709, -0.16306376457214355, -0.21531833708286285, -0.03822503238916397, 0.13040220737457275, -0.1882375180721283, -0.007780172396451235, 0.0726974755525589, 0.3622627854347229, 0.14309769868850708, -0.009629824198782444, 0.17208828032016754, 0.13820892572402954, -0.14321306347846985, 0.012303957715630531, 0.18995951116085052, -0.019488239660859108, -0.12129897624254227, -0.06923460215330124, 0.11918845027685165, -0.23640218377113342, 0.01701134257018566, -0.06539978832006454, -0.01889857091009617, 0.183583065867424, -0.01942555606365204, 0.11893905699253082], [0.05020473152399063, -0.24142441153526306, 0.4695773124694824, -0.3642750680446625, 0.06794584542512894, -0.0691390186548233, -0.46065619587898254, -0.0567123144865036, 0.11904232203960419, 0.1729523241519928, -0.14168305695056915, 0.05460231378674507, 0.07944931089878082, 0.4032081663608551, -0.06735439598560333, -0.25472593307495117, 0.12312838435173035, 0.1556885838508606, -0.03771898150444031, 0.11201463639736176, -0.19851750135421753, -0.1905139684677124, -0.20101574063301086, -0.1958341896533966, 0.26231279969215393, -0.256597638130188, 0.1903773993253708, -0.1291784644126892, -0.12835028767585754, 0.012260723859071732, 0.309821754693985, -0.18884699046611786], [0.3351685106754303, 0.030594918876886368, 0.12788033485412598, -0.47570469975471497, -0.0055463253520429134, 0.3184603750705719, -0.10985326766967773, -0.0028622790705412626, 0.06906992942094803, 0.19262239336967468, 0.13073235750198364, 0.17525409162044525, -0.16851213574409485, 0.5482764840126038, -0.02708413451910019, -0.129253089427948, 0.04401031881570816, 0.12065388262271881, 0.0981416255235672, 0.08976832777261734, -0.06644992530345917, 0.21027818322181702, -0.05303472280502319, -0.004068159498274326, 0.21425224840641022, 0.030344411730766296, 0.011292274110019207, -0.28324195742607117, 0.2121068388223648, 0.1256672888994217, -0.148764505982399, -0.09280107915401459], [0.221864715218544, -0.14685358107089996, 0.17532029747962952, -0.3744830787181854, -0.0446527935564518, 0.10643155872821808, -0.11691958457231522, 0.17025482654571533, 0.16315202414989471, 0.030926240608096123, -0.02172181010246277, 0.09203237295150757, -0.069956474006176, 0.1969921886920929, -0.009254730306565762, 0.026876356452703476, -0.047490037977695465, 0.10747122764587402, 0.047750141471624374, 0.04325609654188156, -0.07602615654468536, 0.013508937321603298, -0.10596254467964172, 0.007227099966257811, 0.14688584208488464, -0.08365543186664581, 0.08437348902225494, -0.24767988920211792, 0.12069886922836304, 0.036818403750658035, 0.06374306976795197, -0.24288077652454376], [0.1381218284368515, 0.01394325029104948, 0.003725968534126878, -0.3030600845813751, 0.174864262342453, 0.1604994833469391, -0.3060414493083954, 0.23058359324932098, -0.017067084088921547, 0.04268015921115875, 0.02474910393357277, -0.20723624527454376, 0.23967379331588745, 0.17750094830989838, 0.020765667781233788, -0.11685382574796677, -0.11446899175643921, -0.16190946102142334, 0.020350856706500053, -0.22020356357097626, 0.16481970250606537, -0.13290676474571228, -0.023675352334976196, -0.08383160829544067, 0.11679647117853165, -0.0877828299999237, 0.14155972003936768, -0.10797017067670822, 0.08167609572410583, 0.04327698051929474, -0.19379182159900665, -0.016994895413517952], [0.04386840760707855, -0.14087986946105957, 0.16276882588863373, -0.18388336896896362, -0.07165803015232086, 0.24637414515018463, -0.115447998046875, 0.1334395557641983, 0.14296302199363708, 0.10575638711452484, 0.07831446081399918, 0.08623084425926208, -0.15829993784427643, 0.2025458961725235, 0.20444723963737488, -0.11869511008262634, -0.11205276846885681, 0.27804476022720337, -0.16891002655029297, 0.0033107949420809746, 0.17569613456726074, 0.17816372215747833, -0.12377317994832993, -0.13815465569496155, 0.07130984961986542, -0.15919777750968933, -0.16180899739265442, -0.15888555347919464, 0.037423450499773026, 0.13141851127147675, 0.04299837723374367, 0.200331449508667], [0.17916850745677948, 0.022559070959687233, 0.13216206431388855, -0.0998779833316803, -0.08103200048208237, 0.19935186207294464, -0.44245436787605286, -0.3540540337562561, -0.08562164008617401, 0.4603150188922882, 0.0848885178565979, -0.07918927073478699, 0.15436716377735138, 0.3733326196670532, -0.01596934348344803, 0.025313524529337883, 0.19371336698532104, 0.04088982567191124, -0.022734828293323517, -0.052421435713768005, 0.1744498759508133, -0.13394901156425476, -0.2740342915058136, 0.07628688216209412, 0.11137862503528595, -0.02627004124224186, 0.046207085251808167, 0.23982523381710052, 0.1577380895614624, -0.11295268684625626, 0.11028803139925003, 0.059096336364746094], [0.0805947482585907, -0.15931494534015656, 0.3129277527332306, -0.2683250308036804, -0.17211554944515228, 0.15670917928218842, -0.015022552572190762, -0.20868591964244843, 0.1034819632768631, 0.0063821906223893166, -0.06944813579320908, -0.020551791414618492, 0.15817959606647491, 0.25747308135032654, -0.17924104630947113, -0.09658477455377579, -0.12577135860919952, 0.029173921793699265, -0.08192808926105499, -0.07024910300970078, -0.05258440598845482, -0.02028076723217964, -0.14311937987804413, 0.2981807291507721, 0.2863945960998535, 0.08608593791723251, 0.17551575601100922, -0.10531222820281982, -0.02527620829641819, 0.12177565693855286, 0.09620924293994904, 0.14115603268146515], [0.009791004471480846, -0.28600674867630005, 0.45837220549583435, -0.4979988634586334, -0.057699546217918396, 0.14832594990730286, -0.4978254437446594, 0.23791998624801636, 0.4358132481575012, 0.06733715534210205, 0.004976174328476191, -0.0734051987528801, 0.032609112560749054, 0.4035740792751312, -0.04380600526928902, -0.41629642248153687, 0.2268550992012024, 0.36997440457344055, 0.03766975551843643, 0.3816498816013336, -0.1380837857723236, -0.13412955403327942, -0.11517897993326187, -0.040544118732213974, 0.3630841374397278, -0.2160448282957077, 0.2625255286693573, -0.4644124209880829, 0.005808534100651741, -0.15238939225673676, 0.4651452302932739, -0.326617032289505], [0.20584595203399658, -0.15476106107234955, 0.3071909248828888, -0.23079901933670044, 0.14146047830581665, 0.10874661058187485, -0.15631486475467682, 0.14108870923519135, -0.24276383221149445, 0.1731845736503601, -0.1794779747724533, -0.16523326933383942, 0.5679821968078613, 0.21679286658763885, -0.2403923124074936, 0.05304918438196182, -0.1726483702659607, 0.01885848678648472, -0.05710962787270546, -0.04985632002353668, 0.18457604944705963, 0.3287590742111206, -0.05072369426488876, 0.41211575269699097, 0.2885269522666931, -0.27416732907295227, -0.1269819140434265, 0.07322372496128082, 0.3580525815486908, 0.24199728667736053, 0.1488429456949234, 0.20596344769001007], [-0.038851916790008545, 0.017779065296053886, 0.17112122476100922, -0.4826182723045349, 0.17796464264392853, 0.3916700482368469, -0.16272731125354767, -0.11499660462141037, 0.06411843001842499, 0.12772181630134583, 0.3080098330974579, 0.36436018347740173, 0.0262914951890707, 0.5126911997795105, 0.059531208127737045, -0.27442997694015503, 0.3490900695323944, -0.10445064306259155, -0.13773036003112793, 0.10681415349245071, 0.059769559651613235, 0.014854219742119312, 0.0423920638859272, -0.05892438068985939, 0.5124170780181885, 0.03729163855314255, -0.033800479024648666, -0.3491591513156891, 0.011811398901045322, 0.28830084204673767, -0.03814542293548584, -0.049837805330753326], [0.11089795082807541, -0.09435153752565384, 0.014328950084745884, -0.07439210265874863, 0.08704479783773422, 0.3353882133960724, -0.1799689084291458, -0.05028112232685089, 0.04613833874464035, 0.1730733960866928, -0.07897830754518509, 0.009393424727022648, 0.2721535265445709, 0.19426332414150238, 0.0037881985772401094, -0.12112120538949966, -0.02504872716963291, 0.11424770951271057, -0.21710984408855438, -0.07463526725769043, 0.21626101434230804, 0.07891380786895752, -0.11519721895456314, 0.17826592922210693, 0.15310341119766235, -0.33976316452026367, -0.09755361825227737, 0.21279920637607574, 0.031721875071525574, 0.24635665118694305, 0.12530995905399323, 0.26445403695106506], [-0.2598389685153961, -0.20825165510177612, 0.24391156435012817, -0.43408337235450745, -0.03254448622465134, 0.2034226506948471, -0.05368730425834656, 0.07682343572378159, -0.12367353588342667, -0.13123944401741028, 0.10193122923374176, 0.10270173847675323, 0.08473580330610275, 0.10689841210842133, -0.10843542963266373, 0.005954011343419552, 0.019673967733979225, -0.08713444322347641, 0.03520115464925766, 0.06825203448534012, 0.3975895047187805, -0.021511347964406013, -0.134242445230484, 0.04329126700758934, 0.3031746745109558, -0.036547742784023285, -0.028379913419485092, -0.07141169905662537, 0.026055973023176193, 0.14880506694316864, 0.051467347890138626, 0.1395951211452484], [0.17162445187568665, -0.18784238398075104, 0.2736438810825348, -0.28141242265701294, -0.01685306243598461, 0.14288856089115143, -0.02890063263475895, 0.04098972678184509, 0.030866293236613274, -0.005185012239962816, -0.15107989311218262, -0.006499526556581259, 0.047966811805963516, 0.28188297152519226, -0.03388943895697594, 0.016587067395448685, -0.29735496640205383, -0.03151024132966995, 0.059737879782915115, -0.08505763858556747, 0.14942598342895508, 0.07899768650531769, -0.10869511216878891, 0.08677344024181366, 0.06804307550191879, -0.08212647587060928, -0.018099691718816757, -0.058251142501831055, 0.11559877544641495, 0.02631177380681038, -0.1688171923160553, 0.043637778609991074], [0.08002021908760071, -0.21072258055210114, 0.031110752373933792, -0.42846086621284485, -0.26469728350639343, 0.13847412168979645, -0.4726846516132355, 0.39297032356262207, 0.14787626266479492, 0.19349434971809387, 0.254077672958374, -0.024244187399744987, -0.03763578087091446, 0.07313227653503418, -0.06492989510297775, -0.36544838547706604, 0.005442231427878141, 0.10872721672058105, 0.1095677837729454, 0.22274114191532135, 0.11887919902801514, 0.15902431309223175, -0.2452225238084793, 0.15637317299842834, 0.015698520466685295, -0.1650432050228119, 0.3301437497138977, -0.3048447370529175, 0.09266543388366699, 0.0038726695347577333, -0.006224632263183594, 0.014756063930690289], [-0.012750805355608463, -0.24106867611408234, 0.14157430827617645, -0.46040409803390503, 0.03911210224032402, -0.058047182857990265, -0.38234877586364746, -0.2906772792339325, -0.10330472886562347, -0.16236673295497894, -0.11824061721563339, 0.23924937844276428, 0.14729799330234528, 0.18319620192050934, -0.12787781655788422, -0.09209200739860535, 0.21368242800235748, 0.13404782116413116, 0.14405250549316406, 0.22956281900405884, 0.3397449851036072, -0.2291555553674698, -0.21156667172908783, 0.30474773049354553, 0.09019795805215836, -0.025245873257517815, 0.02206982672214508, -0.0009373310022056103, -0.00954020582139492, 0.12607835233211517, -0.11666443198919296, 0.1044255718588829], [0.06584407389163971, 0.09967630356550217, 0.21889939904212952, -0.5567030310630798, -0.09955435246229172, 0.34963276982307434, -0.09569332003593445, 0.022890234366059303, 0.103143610060215, 0.3184628486633301, 0.15468545258045197, 0.353092223405838, -0.05645192041993141, 0.4502071440219879, -0.07882370799779892, -0.11694350093603134, 0.18222256004810333, -0.17734050750732422, 0.13359998166561127, -0.021328547969460487, 0.030593929812312126, -0.14076721668243408, 0.04709745943546295, 0.19776831567287445, 0.49208924174308777, 0.37813326716423035, 0.05591563507914543, -0.30430614948272705, 0.05248457193374634, 0.1553083062171936, -0.3912925720214844, -0.14442436397075653], [0.1281791627407074, -0.06548238545656204, 0.2663135826587677, -0.4584232568740845, 0.00631612166762352, 0.11003271490335464, -0.07157395035028458, 0.09072789549827576, -0.06309521943330765, 0.024038800969719887, 0.0009931691456586123, -0.03288906067609787, 0.06619145721197128, 0.33333179354667664, 0.02292017452418804, 0.012382781133055687, 0.0073802596889436245, -0.010141422972083092, 0.02362249605357647, 0.05663719028234482, 0.29642266035079956, -0.09574069827795029, -0.039998602122068405, 0.043114472180604935, 0.21150268614292145, 0.019419124349951744, 0.11673854291439056, -0.2770000100135803, -0.07178346067667007, 0.06598033756017685, -0.1458429992198944, 0.02667638100683689], [-0.006790694780647755, -0.11785844713449478, 0.06999846547842026, -0.2644759714603424, -0.09216400235891342, 0.28250524401664734, -0.2943856716156006, -0.24493134021759033, -0.3129299283027649, 0.37954819202423096, 0.1824404001235962, 0.0248725488781929, 0.13342921435832977, 0.42542096972465515, -0.002771854167804122, 0.05752737447619438, 0.26315775513648987, -0.08034758269786835, 0.17280374467372894, -0.14073842763900757, 0.05085787549614906, -0.1821831464767456, -0.021882586181163788, -0.08372239023447037, 0.3605630099773407, 0.1979401409626007, 0.06220420077443123, 0.02759011648595333, 0.28333181142807007, -0.12282542884349823, 0.1869414746761322, -0.0048708226531744], [0.1119219958782196, -0.16074660420417786, 0.13134220242500305, -0.34612107276916504, 0.21429134905338287, 0.06955830007791519, -0.03344633802771568, -0.09447988867759705, 0.25811687111854553, 0.11351238191127777, -0.040364332497119904, 0.037500716745853424, -0.0060852267779409885, 0.23891983926296234, 0.011072641238570213, 0.06686939299106598, 0.0035505257546901703, 0.07330209016799927, -0.10482067614793777, -0.16233401000499725, 0.037753310054540634, 0.04599744826555252, -0.16755186021327972, -0.06659485399723053, 0.009162443690001965, -0.318357914686203, -0.05815242975950241, 0.008017621003091335, 0.1286604106426239, 0.2077876180410385, -0.16685529053211212, 0.196705162525177], [0.04110279306769371, -0.22946560382843018, 0.16312605142593384, -0.30528709292411804, -0.08806651085615158, 0.05436541885137558, -0.25218290090560913, 0.14859461784362793, 0.1852659434080124, 0.1624365597963333, 0.10504253953695297, 0.04852987825870514, 0.04989663511514664, 0.2692537307739258, -0.0032590157352387905, -0.24872954189777374, -0.01930839940905571, 0.10363870859146118, 0.0077940309420228004, -0.06816170364618301, 0.10656078159809113, -0.12247881293296814, -0.05687669664621353, 0.1467442661523819, 0.0517878532409668, -0.0021475232206285, 0.062199536710977554, -0.052867233753204346, 0.061049290001392365, 0.09269583225250244, -0.0452692024409771, 0.008028870448470116], [0.09515862166881561, 0.1566178798675537, 0.08077128231525421, -0.33285319805145264, 0.3897043466567993, -0.01486489549279213, -0.41414862871170044, -0.20145292580127716, -0.3563637435436249, -0.14009599387645721, -0.12847508490085602, 0.2623380124568939, 0.6118744015693665, 0.05508143827319145, -0.2244587391614914, -0.17709879577159882, 0.2674197852611542, 0.1671876460313797, -0.03252362832427025, 0.1911705732345581, 0.4012558162212372, -0.03894083574414253, -0.1553986519575119, -0.1375836879014969, 0.4922032654285431, 0.06548762321472168, 0.2182365506887436, -0.23696120083332062, -0.021148372441530228, 0.059400297701358795, -0.14703840017318726, 0.18144819140434265], [0.05009014159440994, -0.15835264325141907, 0.19804970920085907, -0.3501496911048889, -0.1661951094865799, 0.13014748692512512, -0.2228514701128006, 0.018001945689320564, 0.011696231551468372, 0.16722092032432556, 0.007723175920546055, -0.03207165375351906, 0.20234832167625427, 0.3662211298942566, -0.12052850425243378, 0.04568904638290405, 0.058002572506666183, 0.017712583765387535, -0.014327667653560638, -0.21866820752620697, 0.09186487644910812, -0.07439438998699188, -0.10495853424072266, 0.22692739963531494, 0.25682881474494934, 0.16138595342636108, 0.06711927801370621, -0.01717078685760498, 0.2152867317199707, 0.10930611193180084, 0.017178455367684364, -0.013577164150774479], [-0.06135374680161476, -0.04034678265452385, -0.11437772214412689, -0.43261608481407166, 0.025069979950785637, 0.25784391164779663, -0.3041607439517975, -0.14805254340171814, -0.07306042313575745, -0.043437883257865906, -0.11764684319496155, -0.12159545719623566, 0.07048502564430237, 0.11275569349527359, -0.04861171916127205, 0.09707140177488327, 0.025143245235085487, 0.028983261436223984, -0.11580802500247955, -0.1448812335729599, 0.30264604091644287, -0.13976499438285828, -0.10576330870389938, 0.01856445148587227, 0.08820336312055588, 0.004944988526403904, -0.04308047890663147, 0.004538023844361305, 0.038419466465711594, 0.23827357590198517, -0.13014502823352814, 0.22110390663146973], [-0.12001965194940567, -0.26524609327316284, -9.444067836739123e-05, -0.37389230728149414, 0.06260305643081665, -0.10996262729167938, -0.20333699882030487, -0.20682542026042938, -0.053836967796087265, 0.019707605242729187, -0.049725960940122604, 0.05016342177987099, 0.15081916749477386, -0.04595666751265526, 0.10029379278421402, 0.1883261501789093, 0.3011441230773926, -0.06012886017560959, 0.10258953273296356, -0.018974460661411285, 0.16715604066848755, -0.12304225564002991, -0.1009499728679657, 0.0065622893162071705, 0.31473007798194885, 0.0409681610763073, -0.07639680057764053, -0.25242388248443604, 0.11315830796957016, 0.09772636741399765, -0.14854493737220764, 0.07861243188381195], [0.025180291384458542, -0.17303800582885742, 0.06758904457092285, -0.4645122289657593, -0.07355567812919617, 0.26802849769592285, -0.3905164301395416, -0.10666254162788391, -0.022374184802174568, 0.2797344923019409, -0.03247169777750969, 0.17129766941070557, -0.01304576639086008, 0.4573509097099304, 0.08563046902418137, -0.15123796463012695, 0.14525994658470154, 0.03329817205667496, -0.09040374308824539, 0.011367960833013058, 0.16330701112747192, -0.16700118780136108, 0.07578735798597336, 0.0725032165646553, 0.15960188210010529, 0.06526091694831848, 0.059168409556150436, 0.04842459782958031, 0.1397167593240738, -0.038141556084156036, 0.06547197699546814, -0.010284236632287502], [-0.3183402419090271, -0.3752490282058716, 0.21941575407981873, -0.11110646277666092, -0.16964133083820343, 0.13059373199939728, -0.11299549043178558, -0.030090151354670525, -0.13732832670211792, 0.2873257100582123, 0.19972148537635803, 0.012844339944422245, 0.0800091028213501, 0.030815061181783676, -0.06448612362146378, 0.25131309032440186, 0.3306485712528229, -0.15923883020877838, 0.3536091446876526, -0.2543189823627472, 0.1576438546180725, -0.22192099690437317, -0.00820389948785305, 0.04289750009775162, 0.4008511006832123, 0.40769273042678833, 0.0536530576646328, -0.2469983994960785, 0.11768189817667007, -0.10173213481903076, -0.15203894674777985, 0.06684053689241409], [0.26737305521965027, 0.11243102699518204, 0.12459579110145569, -0.4148850440979004, -0.31288859248161316, 0.44735443592071533, 0.019977081567049026, -0.10196790099143982, -0.10078445822000504, -0.26757797598838806, -0.13510854542255402, -0.18367084860801697, -0.03656548261642456, 0.4853305220603943, 0.04716137424111366, -0.09356490522623062, -0.15458354353904724, 0.36460041999816895, -0.05775905400514603, 0.04427700489759445, 0.2994726002216339, 0.20794087648391724, -0.43099841475486755, 0.12836165726184845, 0.1417548656463623, 0.08674925565719604, -0.18162353336811066, -0.13192711770534515, 0.09413561224937439, 0.36117109656333923, -0.08532985299825668, 0.2246721386909485], [-0.15818583965301514, -0.02575564570724964, 0.4160805344581604, -0.11753518879413605, -0.09196405112743378, 0.2261543869972229, 0.09205660223960876, -0.19556757807731628, 0.04178149253129959, 0.14829660952091217, -0.013573403470218182, -0.009142798371613026, -0.2345215529203415, 0.06981422752141953, -0.06030000373721123, 0.24466145038604736, -0.19503256678581238, 0.09275612980127335, 0.08031988888978958, -0.11529707163572311, 0.15299047529697418, -0.20651131868362427, -0.1334071308374405, -0.03185791149735451, 0.029776405543088913, -0.013349559158086777, 0.015724843367934227, -0.012136328965425491, 0.09755947440862656, 0.35650473833084106, -0.060070231556892395, 0.10325144976377487], [0.013823906891047955, -0.13674314320087433, 0.23299938440322876, -0.17579582333564758, 0.19480422139167786, 0.2640131711959839, -0.016937633976340294, 0.13212525844573975, 0.1286516934633255, 0.14873459935188293, 0.06459777802228928, 0.056578706949949265, 0.09665686637163162, 0.19844628870487213, 0.13594819605350494, -0.08269839733839035, -0.02451896481215954, 0.03578782081604004, -0.050656020641326904, -0.04683644697070122, 0.22842811048030853, 0.029922233894467354, -0.061128392815589905, -0.09040651470422745, 0.2946426570415497, -0.011740759946405888, -0.02733396738767624, -0.35226649045944214, -0.010380778461694717, 0.13512100279331207, -0.08543743938207626, -0.054164908826351166], [-0.08448692411184311, -0.29783621430397034, 0.14657992124557495, -0.2767665684223175, 0.0005082209827378392, 0.08128535747528076, -0.27758732438087463, -0.028293145820498466, 0.007960372604429722, 0.07039123773574829, 0.03813724219799042, 0.183407261967659, 0.02537657879292965, 0.19246603548526764, 0.06618773937225342, 0.01181319821625948, 0.1393083930015564, 0.003541473997756839, -0.1559937745332718, 0.001746131805703044, 0.20527800917625427, -0.20498016476631165, -0.06422896683216095, 0.040539100766181946, 0.2983574867248535, -0.08336319029331207, -0.10607918351888657, -0.004126168321818113, 0.06831079721450806, 0.30395111441612244, -0.06137607619166374, -0.040835000574588776], [-0.05644125118851662, 0.005597200244665146, -0.03094291500747204, -0.45890066027641296, 0.11862599849700928, 0.41898205876350403, -0.2636878192424774, 0.12979137897491455, 0.17302240431308746, -0.05319589003920555, -0.07738853991031647, 0.05063372850418091, 0.019103532657027245, 0.010885423980653286, -0.02225486934185028, -0.08770519495010376, -0.16450205445289612, 0.021791838109493256, 0.029001658782362938, -0.04123672470450401, 0.28436025977134705, -0.23504921793937683, 0.050347551703453064, 0.19767002761363983, 0.04622160643339157, -0.014774026349186897, 0.03788736090064049, 0.10643161088228226, -0.06268572062253952, -0.024001259356737137, -0.01688753068447113, 0.21820025146007538], [0.05460190027952194, -0.22653433680534363, 0.13941437005996704, -0.14553707838058472, 0.04591100662946701, 0.1160023957490921, -0.01116473600268364, 0.312396764755249, 0.013111717998981476, 0.14181236922740936, 0.05868608504533768, -0.11155523359775543, 0.24611221253871918, 0.30428025126457214, 0.06420435756444931, 0.1518055647611618, 0.04037034511566162, -0.05673931539058685, -0.0977967381477356, -0.2895381450653076, 0.2807440757751465, -0.11136607080698013, -0.01842537708580494, -0.009353521279990673, 0.08272746205329895, -0.11957293748855591, 0.06801849603652954, -0.019061196595430374, 0.1562439203262329, 0.1104547530412674, -0.18685831129550934, -0.010988511145114899], [-0.1112554669380188, -0.23508810997009277, 0.1500825732946396, -0.278086394071579, -0.1264701932668686, 0.11013800650835037, -0.22589386999607086, 0.1861119419336319, 0.07116808742284775, 0.1749444305896759, 0.038311950862407684, 0.09255632013082504, 0.037887707352638245, 0.1739383041858673, -0.04561876505613327, -0.14678855240345, 0.04514498636126518, 0.12098672240972519, -0.019796712324023247, -0.16079719364643097, 0.2038901001214981, 0.03251393511891365, -0.2002732902765274, -0.052474360913038254, 0.24119767546653748, 0.046758998185396194, -0.05137237533926964, -0.19114255905151367, 0.07715094089508057, 0.11901972442865372, -0.13804012537002563, 0.03343495354056358], [-0.01761479303240776, -0.14596913754940033, 0.11444167047739029, -0.09464612603187561, -0.11028599739074707, 0.20259946584701538, -0.2945843040943146, 0.04666456580162048, 0.10080873966217041, 0.12428978085517883, -0.0022749530617147684, 0.06395336240530014, 0.15061533451080322, 0.043286457657814026, 0.06643844395875931, -0.17963729798793793, -0.05952346324920654, 0.1644669771194458, -0.11351007968187332, 0.09903650730848312, 0.16262386739253998, -0.041493531316518784, -0.1842728555202484, -0.0021278669591993093, 0.05704081431031227, -0.15022702515125275, 0.11830639839172363, 0.011282453313469887, 0.16390898823738098, 0.06603188067674637, 0.052617255598306656, 0.09627053886651993], [0.1014212816953659, -0.049855802208185196, -0.012699165381491184, -0.15620474517345428, 0.3378257751464844, 0.3331136107444763, -0.3711632788181305, 0.15358580648899078, 0.10900511592626572, 0.05600085109472275, -0.15008139610290527, 0.12254359573125839, 0.0127718485891819, -0.01190092507749796, 0.23555763065814972, 0.08282620459794998, -0.1968824416399002, 0.261130690574646, -0.23379993438720703, -0.1434352695941925, 0.0683971419930458, 0.16615159809589386, -0.05294457823038101, -0.326532781124115, 0.03640558570623398, -0.24305370450019836, -0.11369749903678894, -0.004829905461519957, 0.03671073913574219, 0.1609492152929306, 0.12494542449712753, -0.06304483115673065], [0.2818022072315216, -0.003918229136615992, 0.1517743319272995, -0.29656097292900085, 0.004848684184253216, 0.4195898175239563, -0.06899164617061615, -0.017860183492302895, 0.11632473021745682, 0.013842171058058739, -0.22421003878116608, -0.21624164283275604, 0.0797463208436966, 0.3493010997772217, 0.042314209043979645, 0.05939669907093048, -0.23702718317508698, 0.18449734151363373, -0.06331053376197815, -0.08565448224544525, 0.30022910237312317, 0.11638471484184265, -0.06003548949956894, 0.0834253802895546, 0.06879410892724991, 0.021370496600866318, -0.08865836262702942, 0.039050180464982986, 0.12933723628520966, 0.13729901611804962, -0.15422332286834717, 0.17811764776706696], [0.002220824593678117, -0.03691114857792854, 0.06436192244291306, -0.34795787930488586, 0.09931861609220505, 0.15064038336277008, -0.20125748217105865, -0.07004556059837341, -0.138885036110878, 0.09875567257404327, 0.1592065989971161, 0.07531283050775528, 0.09264402836561203, 0.28385722637176514, 0.061023835092782974, -0.017781933769583702, 0.13581068813800812, -0.05426722392439842, -0.036778226494789124, -0.10017972439527512, -0.05076010897755623, -0.09086112678050995, -0.07337034493684769, 0.08951272815465927, 0.34188172221183777, -0.011436499655246735, 0.10501421988010406, -0.17055891454219818, 0.0901864767074585, 0.18514247238636017, 0.018991461023688316, -0.12906941771507263], [0.10175766795873642, -0.14701098203659058, 0.39878082275390625, -0.3855595290660858, 0.06599543243646622, 0.21161025762557983, 0.30908459424972534, 0.033359356224536896, 0.13099807500839233, 0.13495667278766632, -0.12311398983001709, 0.047085169702768326, 0.06584211438894272, 0.4622904658317566, -0.1520957350730896, -0.012627669610083103, -0.21432313323020935, -0.22047534584999084, 0.09030476212501526, -0.1628004014492035, -0.004757269751280546, 0.03591503947973251, -0.01240443903952837, 0.17439597845077515, 0.2741970419883728, 0.043354332447052, 0.1896173506975174, -0.012919728644192219, -0.00014019043010193855, 0.10652587562799454, -0.14966604113578796, 0.12974517047405243], [-0.03370726481080055, -0.06936398148536682, 0.2344793826341629, -0.3661421835422516, 0.11536698043346405, 0.30423402786254883, -0.22702689468860626, -0.10303371399641037, -0.255056768655777, 0.2678793668746948, -0.050229594111442566, -0.07054752856492996, 0.3276617228984833, 0.466224730014801, 0.0511033833026886, -0.23311349749565125, 0.09082150459289551, 0.19551298022270203, -0.20357535779476166, 0.011332360096275806, 0.29514408111572266, 0.0020272901747375727, -0.1078343316912651, -0.16354982554912567, 0.3495716452598572, -0.022547397762537003, 0.15406818687915802, -0.009231822565197945, -0.1607547253370285, 0.2488846331834793, 0.14886125922203064, 0.1522841602563858], [-0.0018250250723212957, -0.2865830361843109, 0.09208441525697708, -0.18773305416107178, 0.05977921560406685, 0.09135153144598007, -0.27582842111587524, -0.16167469322681427, -0.21755450963974, 0.14130961894989014, 0.046170640736818314, 0.03663644939661026, 0.23141902685165405, 0.08159520477056503, 0.07372642308473587, 0.06804057210683823, -0.010617424733936787, 0.03432513028383255, -0.17049472033977509, 0.09986741840839386, 0.32064616680145264, -0.08069901168346405, -0.0778532326221466, -0.005420264787971973, 0.49715444445610046, 0.06433579325675964, -0.015591810457408428, 0.0190216563642025, 0.18276947736740112, 0.26629507541656494, 0.024658434092998505, 0.009592299349606037], [0.16041645407676697, -0.11494342237710953, 0.11660511046648026, -0.2074965536594391, 0.11635316908359528, -0.04537814110517502, 0.0015302200336009264, -0.00552852638065815, -0.2614578604698181, 0.03426108509302139, -0.0793871060013771, -0.1166137233376503, 0.5929384827613831, -0.07774163782596588, 0.007579274009913206, 0.10870151221752167, 0.03627335652709007, -0.12902608513832092, -0.01146257296204567, 0.20065169036388397, 0.20824174582958221, -0.19950975477695465, 0.040840402245521545, -0.13002073764801025, 0.3222653865814209, -0.022741535678505898, 0.1184106320142746, 0.30014532804489136, 0.19249244034290314, 0.003733307123184204, -0.07358747720718384, 0.2558836340904236], [0.047463491559028625, -0.004598825704306364, -0.040591850876808167, -0.24147240817546844, 0.22627727687358856, 0.2354162633419037, -0.37715381383895874, 0.0623420774936676, -0.2392587959766388, -0.0482647605240345, -0.022267570719122887, -0.11697909235954285, 0.165304034948349, 0.08165356516838074, -0.08506161719560623, 0.19972851872444153, -0.02750450000166893, -0.17891961336135864, -0.08205127716064453, -0.08944670855998993, 0.25354892015457153, -0.13903401792049408, 0.06976693123579025, -0.09098087251186371, 0.13621068000793457, 0.008271203376352787, -0.007793325465172529, -0.032449379563331604, 0.327163428068161, -0.17187826335430145, -0.12934660911560059, 0.062301572412252426], [0.25401589274406433, -0.1830761432647705, 0.28310349583625793, -0.3645987808704376, 0.18430471420288086, 0.312222421169281, 0.12005311250686646, -0.19883979856967926, 0.3640266954898834, -0.010982577688992023, -0.3657805621623993, 0.031747423112392426, 0.21210308372974396, 0.22065691649913788, 0.04654718190431595, 0.2158006727695465, -0.1091100424528122, -0.058621570467948914, 0.03622070327401161, -0.22985297441482544, -0.10299874097108841, 0.18413661420345306, -0.04001429304480553, 0.5009346604347229, 0.13247312605381012, 0.0029441560618579388, -0.10504285991191864, 0.10014940053224564, 0.20184586942195892, 0.41920629143714905, -0.030869774520397186, -0.10249386727809906], [-0.0013193641789257526, -0.1208074614405632, 0.13317973911762238, -0.38203778862953186, -0.09699463099241257, 0.1982903778553009, -0.1353210061788559, -0.011308860965073109, -0.006240566726773977, -0.09020201861858368, -0.13125386834144592, -0.03483614698052406, 0.13139277696609497, 0.20655439794063568, 0.1078748106956482, -0.048144593834877014, 0.11613256484270096, 0.18965663015842438, 0.11867582052946091, 0.15768475830554962, 0.25994572043418884, -0.0008806458208709955, -0.21567218005657196, 0.04827115684747696, 0.17596283555030823, -0.004354170989245176, -0.04438105970621109, -0.10438526421785355, 0.03056081384420395, 0.17917095124721527, 0.05647400766611099, 0.2045820951461792], [-0.33538150787353516, -0.4141322076320648, 0.2618407607078552, -0.42897501587867737, -0.13290296494960785, 0.24632979929447174, 0.07724650949239731, 0.17322780191898346, 0.08418038487434387, -0.09465670585632324, 0.083821140229702, 0.03773123025894165, 0.04722301661968231, 0.11122629791498184, 0.030338238924741745, -0.1402561068534851, 0.044144269078969955, -0.031083285808563232, 0.11947820335626602, 0.20020373165607452, 0.42781007289886475, 0.05167606845498085, -0.05469156801700592, 0.00948463473469019, 0.1397205889225006, 0.015395155176520348, -0.06426796317100525, 0.0521601065993309, -0.05892433971166611, -0.01713804341852665, 0.21369978785514832, 0.17727619409561157], [0.0072792088612914085, -0.13423749804496765, 0.14868368208408356, -0.13985757529735565, 0.03458136320114136, 0.09514030069112778, -0.07098612189292908, 0.00018195986922364682, 0.01712643913924694, 0.11603645980358124, 0.08048713952302933, -0.03941933438181877, 0.23427897691726685, -0.030688656494021416, 0.0992840826511383, 0.1544715166091919, -0.019577395170927048, -0.1511632353067398, -0.12214170396327972, 0.01761222444474697, 0.1855795979499817, -0.0939689427614212, -0.16290885210037231, 0.05240759998559952, 0.3648669421672821, -0.05483483895659447, 0.07379037886857986, -0.0674714744091034, 0.09365460276603699, 0.09145058691501617, -0.09984277188777924, 0.011661341413855553], [-0.21444955468177795, -0.2269039899110794, 0.2957412302494049, -0.48937293887138367, -0.17258940637111664, 0.294424444437027, 0.01468066405504942, -0.11743227392435074, -0.05284266173839569, -0.03894480690360069, 0.03197547048330307, -0.08499228954315186, 0.11247444897890091, -0.040619008243083954, -0.22874009609222412, 0.3590877950191498, -0.04320069029927254, -0.3396087884902954, 0.3299679458141327, -0.20514114201068878, 0.2952653467655182, -0.16937661170959473, 0.02795194648206234, 0.5231010913848877, 0.5492981672286987, 0.5665783286094666, 0.24467769265174866, 0.010672877542674541, -0.009663852863013744, 0.28617438673973083, -0.03363088518381119, 0.12997545301914215], [0.05510896444320679, -0.2829757630825043, 0.023493343964219093, -0.3569348454475403, 0.16218170523643494, 0.1397484689950943, -0.0857597142457962, -0.08919543772935867, 0.415582537651062, -0.002980724209919572, -0.10622689872980118, -0.004691999405622482, 0.022541411221027374, 0.10805459320545197, 0.08269096165895462, 0.006529365200549364, -0.11119190603494644, -0.0477634035050869, -0.24292878806591034, -0.28790482878685, -0.025233618915081024, 0.012533222325146198, -0.09888029843568802, 0.1852431297302246, 0.12910158932209015, -0.19518372416496277, -0.011368454433977604, 0.013884791173040867, -0.06015018746256828, 0.46239981055259705, -0.19595500826835632, 0.08913813531398773], [0.11322653293609619, -0.07294023036956787, 0.15883482992649078, -0.2788068950176239, 0.05494222044944763, 0.16083469986915588, -0.03192299231886864, -0.04962052032351494, 0.10261929035186768, 0.11073404550552368, 0.02133042737841606, 0.03968482092022896, -0.05032040923833847, 0.2267201542854309, -0.002100494224578142, -0.10067806392908096, -0.0668572410941124, -0.09064566344022751, -0.10556533932685852, -0.0005684045609086752, -0.05608221888542175, -0.016450636088848114, -0.16437961161136627, -0.033237457275390625, 0.2252802550792694, -0.23787063360214233, 0.10645195096731186, -0.18103182315826416, -0.016806351020932198, 0.050936128944158554, -0.04842162877321243, -0.04536956548690796], [0.0911947637796402, -0.09674078226089478, 0.21158598363399506, -0.2727065086364746, 0.1989804059267044, 0.31418362259864807, -0.3913959562778473, 0.30596721172332764, 0.0936896950006485, 0.2839849889278412, -0.006877597887068987, -0.041554294526576996, 0.2556368410587311, 0.3382972478866577, 0.07980582863092422, -0.1795884221792221, -0.07200540602207184, 0.11358009278774261, 0.008868228644132614, 0.05661838501691818, 0.0554129034280777, 0.07318232208490372, -0.05681648105382919, 0.051746174693107605, 0.21490325033664703, -0.09336336702108383, 0.22778789699077606, -0.11440553516149521, 0.1474459022283554, 0.09672283381223679, 0.1587933450937271, -0.18031226098537445], [0.14626537263393402, -0.10974562168121338, 0.2729383707046509, -0.3070599436759949, 0.22677144408226013, 0.1431208699941635, -0.19546985626220703, -0.02078969217836857, -0.052067361772060394, 0.09934048354625702, -0.04905611649155617, 0.08866352587938309, 0.05691846087574959, 0.5195119976997375, -0.10108348727226257, -0.0641188770532608, 0.16125304996967316, 0.19798745214939117, -0.11463256180286407, 0.04669390246272087, 0.11756835877895355, 0.019194334745407104, -0.07468947768211365, -0.09103473275899887, 0.11385798454284668, -0.18137405812740326, 0.087517149746418, -0.2116299271583557, 0.11472941190004349, -0.046545885503292084, 0.12861816585063934, -0.04084295034408569], [-0.006669558119028807, -0.05184273049235344, 0.20136964321136475, -0.4543004631996155, 0.08784934133291245, 0.2337728589773178, -0.1146634966135025, -0.05916440859436989, -0.11580774933099747, 0.052701082080602646, -0.041047338396310806, 0.008156347088515759, 0.17074479162693024, 0.10438618808984756, -0.020342033356428146, -0.008353032171726227, 0.05133438482880592, 0.03540287911891937, 0.1432400792837143, 0.07352745532989502, 0.31846651434898376, -0.10376126319169998, 0.05910052731633186, 0.037612732499837875, 0.2537844479084015, 0.2169536054134369, -0.0013815262354910374, -0.014321456663310528, 0.007603402249515057, 0.008718092925846577, -0.051176805049180984, 0.25329843163490295], [-0.2014882117509842, -0.05809502303600311, 0.09850519150495529, 0.08284970372915268, 0.19713947176933289, 0.28182774782180786, -0.3891442120075226, -0.06326831877231598, -0.07572215050458908, 0.2796841859817505, -0.06704649329185486, 0.13515493273735046, 0.04305875301361084, 0.08345036953687668, 0.12489835172891617, -0.04657553508877754, -0.044493187218904495, 0.2876330316066742, -0.22364503145217896, -0.32976919412612915, -0.05043449252843857, -0.007312336936593056, -0.23274150490760803, -0.15701813995838165, -0.04079492390155792, -0.31299784779548645, 0.004484486300498247, 0.04789309576153755, -0.14181864261627197, 0.5320698022842407, 0.11733141541481018, -0.09288346022367477], [0.17526160180568695, 0.022405436262488365, 0.043423086404800415, -0.15335819125175476, -0.19183461368083954, 0.086180679500103, -0.21176950633525848, 0.008246644400060177, 0.058546651154756546, 0.25119224190711975, 0.17629949748516083, 0.074219711124897, 0.11603076756000519, 0.21866796910762787, 0.12922333180904388, -0.13979537785053253, 0.01639905944466591, 0.3017328083515167, -0.035937946289777756, 0.003375394269824028, 0.1872478872537613, -0.0005435705534182489, 0.05785174295306206, -0.0218814667314291, 0.16445007920265198, 0.011194463819265366, -0.023724839091300964, -0.1201314926147461, 0.20049801468849182, 0.10666411370038986, -0.06628992408514023, 0.08422524482011795], [-0.23429973423480988, -0.33086565136909485, 0.24397000670433044, -0.2809543311595917, -0.15945562720298767, 0.19162997603416443, -0.28434962034225464, -0.07200512290000916, -0.004797124303877354, 0.06821238249540329, 0.13124054670333862, 0.2896690368652344, -0.0677613765001297, -0.1647937148809433, 0.0625872015953064, -0.12392529845237732, 0.09416211396455765, 0.1860247403383255, 0.08105253428220749, 0.23722194135189056, 0.18171502649784088, 0.12143457680940628, -0.2586616277694702, -0.05156008154153824, 0.19534392654895782, 0.06762530654668808, -0.11261380463838577, -0.0645236149430275, 0.054174575954675674, 0.11644454300403595, 0.147894948720932, 0.22366517782211304]], [[0.11357373744249344, 0.07391698658466339, -0.3438231348991394, -0.0007920771022327244, 0.0003948694793507457, 0.12025783956050873, 0.022658642381429672, 0.16700248420238495, -0.030277088284492493, 0.08051586896181107, 0.025333382189273834, 0.13600584864616394, 0.14052629470825195, 0.06431427597999573, -0.4085036516189575, -0.031068747863173485, -0.18496845662593842, -0.0028105906676501036, 0.017513127997517586, -0.3669128119945526, 0.47041603922843933, -0.001994745573028922, -0.5003429651260376, 0.21294774115085602, -0.30313384532928467, 0.03491267189383507, -0.40321701765060425, 0.3090706169605255, -0.09963985532522202, 0.12086330354213715, -0.15573018789291382, -0.019560515880584717], [-0.11623413115739822, -0.31736671924591064, 0.05689135938882828, -0.0012309369631111622, -0.2676754295825958, 0.004622915759682655, -0.03288776054978371, 0.314159631729126, 0.09772272408008575, -0.14162732660770416, 0.24243533611297607, -0.07125038653612137, -0.3254074156284332, 0.02611006610095501, 0.032487791031599045, 0.31621623039245605, -0.008371283300220966, -0.369046688079834, 0.055046141147613525, -0.29629600048065186, 0.3426910936832428, -0.1540250927209854, -0.3038097321987152, -0.11405505239963531, -0.05809307098388672, -0.1296369880437851, -0.05269691348075867, 0.2894591987133026, -0.2576149106025696, -0.09531772136688232, -0.022360051050782204, 0.4453957676887512], [0.16418848931789398, 0.05948920175433159, -0.15142717957496643, 0.047020673751831055, -0.0031750607304275036, 0.20534563064575195, 0.020249413326382637, 0.3817518949508667, 0.040275689214468, -0.06583055853843689, -0.002311245072633028, 0.35075604915618896, 0.15378805994987488, -0.2207058221101761, 0.16790355741977692, 0.07395806163549423, -0.2972015142440796, 0.05646168813109398, 0.06685136258602142, -0.5371841788291931, 0.28258198499679565, -0.08276242017745972, -0.12053290754556656, 0.21980683505535126, 0.0047305976040661335, -0.2876785397529602, 0.10817502439022064, -0.03240833804011345, 0.24051283299922943, -0.18517856299877167, -0.031160738319158554, 0.12762551009655], [-0.06117270886898041, 0.28840646147727966, -0.40242868661880493, -0.45333734154701233, 0.001198859652504325, 0.007433360442519188, -0.16363193094730377, 0.5876502990722656, -0.28917041420936584, -0.12837043404579163, -0.21249128878116608, -0.034898292273283005, -0.4290170669555664, -0.07285048067569733, -0.13367070257663727, 0.3931470215320587, -0.09679292142391205, -0.2947850227355957, 0.3048877716064453, -0.17214906215667725, -0.45937561988830566, -0.15477116405963898, -0.25497546792030334, 0.1152850016951561, -0.006646531634032726, -0.13481923937797546, -0.09001142531633377, -0.029324693605303764, -0.049931541085243225, 0.07904127985239029, -0.11015979200601578, 0.37126466631889343], [0.5357959866523743, -0.008385549299418926, 0.012520245276391506, -0.06247294694185257, 0.11368682235479355, 0.21818412840366364, -0.11165475845336914, 0.1136794164776802, 0.40913060307502747, -0.1401098668575287, 0.1844734251499176, 0.2518397271633148, 0.07778075337409973, 0.007660095114260912, 0.0013131432933732867, 0.23587602376937866, -0.13867540657520294, -0.5656994581222534, -0.09641288220882416, 0.221775621175766, 0.34137383103370667, -0.37020954489707947, -0.10254744440317154, 0.1748587191104889, -0.35281074047088623, -0.15278573334217072, -0.2017989605665207, 0.15989524126052856, -0.0779518410563469, 0.21145576238632202, 0.026127712801098824, 0.3258506655693054], [0.09666399657726288, -0.1578928381204605, -0.4062647819519043, 0.07674180716276169, -0.09958584606647491, 0.12956511974334717, 0.07157039642333984, 0.18507875502109528, -0.03813987225294113, -0.056843630969524384, -0.008005790412425995, 0.12262149900197983, 0.13174906373023987, -0.07386299967765808, -0.09467583894729614, 0.17391155660152435, -0.1063193753361702, -0.013232432305812836, 0.11592989414930344, -0.31848275661468506, 0.06483825296163559, 0.02987961657345295, -0.2281428724527359, -0.12246052920818329, -0.13161903619766235, 0.10837168991565704, -0.04705045372247696, 0.3829849362373352, 0.24196280539035797, -0.023660808801651, -0.22345216572284698, 0.1983737349510193], [0.14154626429080963, 0.03542999178171158, -0.40161454677581787, 0.12948065996170044, -0.11515440046787262, 0.08678215742111206, -0.06460097432136536, 0.04067248851060867, -0.09752046316862106, -0.22933799028396606, 0.1286325305700302, 0.08250695466995239, 0.1407100260257721, 0.04009294882416725, -0.41906020045280457, 0.35383936762809753, 0.10907914489507675, -0.217291459441185, -0.21758612990379333, -0.10682755708694458, 0.047525178641080856, -0.15967610478401184, -0.372202605009079, -0.1676492840051651, 0.0523751862347126, -0.05609197914600372, -0.29467588663101196, 0.34867310523986816, -0.03373686596751213, 0.061941057443618774, -0.264303594827652, 0.1855115294456482], [0.05297986790537834, 0.1462920755147934, -0.27158844470977783, -0.1789952963590622, 0.03645467013120651, -0.13103538751602173, -0.03280717134475708, 0.28006112575531006, -0.10228067636489868, -0.17049875855445862, 0.29032307863235474, 0.1526634842157364, 0.08911508321762085, 0.10882166773080826, 0.05544570833444595, 0.1423063725233078, -0.04195643961429596, -0.3590773940086365, 0.06237269565463066, -0.26502853631973267, 0.004889796022325754, -0.19524940848350525, -0.1256525218486786, -0.16772863268852234, -0.21461009979248047, -0.031710777431726456, -0.004860455635935068, 0.10614445805549622, 0.058106958866119385, -0.04200289770960808, -0.07376465201377869, 0.24483296275138855], [0.250416100025177, 0.12674091756343842, -0.1273002326488495, 0.10316973179578781, -0.22639347612857819, -0.0836532786488533, -0.1903492510318756, 0.26986923813819885, 0.32829755544662476, -0.23177821934223175, 0.015524355694651604, -0.030657609924674034, 0.06892365217208862, -0.14446625113487244, -0.280907541513443, 0.21863466501235962, -0.1024884283542633, -0.6633540987968445, 0.0026220930740237236, -0.14998376369476318, 0.2121822088956833, -0.22423461079597473, -0.3708426058292389, -0.12535858154296875, -0.02289218083024025, 0.03236746788024902, -0.26841044425964355, 0.2960796058177948, 0.027646470814943314, -0.05577456206083298, -0.07941862940788269, 0.25942593812942505], [0.11954648047685623, 0.24923810362815857, -0.1280505359172821, -0.13183386623859406, -0.01688583381474018, 0.1906360387802124, 0.11195647716522217, 0.23589836061000824, -0.08059635013341904, -0.15912848711013794, 0.060671981424093246, -0.016055354848504066, -0.053309060633182526, 0.09209214150905609, -0.40132278203964233, 0.330974280834198, 0.1296718418598175, 0.1012234017252922, -0.16247738897800446, -0.03296034783124924, 0.039419371634721756, -0.3919648826122284, -0.14684270322322845, 0.05580217391252518, -0.06103534251451492, -0.26205191016197205, -0.19276317954063416, 0.4342518746852875, -0.41580286622047424, 0.11742503196001053, -0.03259586915373802, 0.15853716433048248], [0.2433449625968933, -0.015926890075206757, -0.0474683903157711, -0.044741734862327576, -0.07733748853206635, 0.1588660627603531, -0.08540786057710648, 0.2803923785686493, 0.14774321019649506, -0.18373145163059235, -0.0015121034812182188, 0.14388932287693024, -0.011838664300739765, -0.2216213494539261, 0.003943637479096651, 0.2791522443294525, -0.10914476215839386, -0.20478153228759766, -0.07425599545240402, -0.17619545757770538, 0.2407466322183609, -0.0988151803612709, -0.3063682019710541, 0.07829957455396652, 0.1289612203836441, -0.2347562462091446, 0.01865358091890812, -0.05982094630599022, 0.08089383691549301, -0.09252601861953735, 0.11570011079311371, 0.19143733382225037], [0.0443141870200634, 0.42305588722229004, 0.009614318609237671, -0.6055184602737427, -0.12431073933839798, 0.12917648255825043, -0.42887476086616516, 0.34696266055107117, -0.12257552891969681, -0.38174325227737427, -0.06096729636192322, -0.13157135248184204, 0.3296627998352051, 0.02171112410724163, -0.049197111278772354, 0.28154027462005615, -0.20024257898330688, -0.07152290642261505, -0.23250193893909454, -0.0335756354033947, 0.3326764404773712, -0.3586028218269348, -0.21960756182670593, 0.27152830362319946, -0.17070086300373077, -0.348694771528244, 0.05657707527279854, -0.11443916708230972, 0.1253584325313568, -0.09188660234212875, -0.1783798485994339, 0.34349068999290466], [0.1389176994562149, 0.15620361268520355, -0.10895460844039917, 0.04488326981663704, 0.013996479101479053, 0.1002383604645729, -0.17599305510520935, 0.3181261420249939, 0.05863886699080467, -0.22402992844581604, 0.22231002151966095, 0.32313451170921326, -0.0509297139942646, -0.05416019633412361, 0.05643830820918083, 0.16869103908538818, -0.09995485842227936, -0.17800229787826538, 0.01742519997060299, -0.012523571960628033, 0.21597152948379517, -0.24520494043827057, -0.06226975470781326, -0.0046140761114656925, -0.10654691606760025, -0.26565298438072205, -0.15951986610889435, 0.08446376770734787, -0.098589226603508, -0.004343106877058744, -0.0047626253217458725, 0.25823935866355896], [0.3862355947494507, 0.028951626271009445, -0.11807963252067566, 0.06040897220373154, -0.18913540244102478, -0.1946502923965454, -0.2662564814090729, 0.008142730221152306, 0.2188287377357483, -0.18767638504505157, 0.44439253211021423, 0.3085983395576477, 0.010864900425076485, -0.16215826570987701, -0.22935248911380768, 0.05506141856312752, -0.04698794707655907, -0.5923610329627991, -0.14574134349822998, 0.03668072074651718, -0.18200798332691193, -0.37071478366851807, -0.09460808336734772, -0.25294721126556396, -0.46487659215927124, 0.05979949235916138, -0.08084440231323242, 0.12767964601516724, -0.12338006496429443, 0.08565473556518555, -0.21176940202713013, 0.41191360354423523], [0.1882087141275406, -0.05940403789281845, -0.4034573435783386, 0.09135698527097702, 0.19829188287258148, 0.06904816627502441, -0.14495228230953217, 0.43212881684303284, -0.13992221653461456, -0.3159102201461792, -0.04445033520460129, 0.2087709605693817, 0.03439153730869293, -0.0011869751615449786, 0.20020751655101776, 0.1725979447364807, -0.09615810215473175, -0.23871728777885437, 0.0847242921590805, -0.22180181741714478, 0.014325842261314392, -0.054153572767972946, -0.005063260439783335, -0.1533477008342743, -0.19847756624221802, 0.28958481550216675, -0.08917073160409927, 0.11000529676675797, 0.05965059623122215, -0.15078388154506683, -0.08563049882650375, 0.3709656000137329], [0.5023155808448792, 0.29568350315093994, 0.13236595690250397, 0.0452970452606678, 0.1718481034040451, 0.0009816825622692704, 0.2227020263671875, 0.3078111410140991, 0.3036413788795471, -0.22190815210342407, -0.09769831597805023, -0.26313063502311707, 0.1605004519224167, -0.0007212756318040192, -0.4214729964733124, 0.1223519891500473, -0.29850730299949646, -0.4383154809474945, 0.08741597831249237, -0.14635537564754486, 0.042193759232759476, -0.3959513008594513, -0.2591056525707245, 0.2765056788921356, 0.19245074689388275, -0.09526938945055008, -0.3736383318901062, 0.33152300119400024, 0.016437547281384468, -0.04606582224369049, 0.09484773129224777, 0.4004132151603699], [0.2067514955997467, -0.021575048565864563, -0.12220370769500732, 0.08365324139595032, 0.07131446152925491, 0.28225642442703247, -0.19302187860012054, 0.3245605528354645, 0.24392884969711304, -0.2035038322210312, 0.1850389987230301, 0.3109546899795532, 0.2508246600627899, -0.44629818201065063, 0.07208681106567383, 0.2996397018432617, -0.19305509328842163, -0.11644981056451797, -0.014734385535120964, -0.11818467080593109, 0.3403960168361664, -0.26405423879623413, -0.14267612993717194, 0.17464958131313324, -0.06002078205347061, -0.1904800981283188, 0.07033541798591614, -0.028309330344200134, 0.2896103262901306, 0.03912169858813286, -0.08565355092287064, 0.08510717004537582], [-0.01144952792674303, -0.08076491951942444, -0.16950345039367676, -0.18215353786945343, -0.18850533664226532, 0.2254088968038559, 0.23608872294425964, 0.18239586055278778, -0.185881108045578, 0.056150250136852264, -0.009095217101275921, 0.057818058878183365, 0.07413747161626816, -0.06067422404885292, -0.23044200241565704, 0.2846446931362152, -0.04842062294483185, 0.09465708583593369, -0.1006038635969162, -0.21067236363887787, 0.24251829087734222, -0.28204047679901123, -0.2850528955459595, 0.08472553640604019, -0.24544501304626465, -0.10347473621368408, -0.10721790790557861, 0.4784526824951172, -0.036500826478004456, 0.07021307945251465, -0.14299382269382477, 0.14615750312805176], [0.006699965801090002, 0.19281890988349915, -0.23165227472782135, 0.18537461757659912, -0.2907794415950775, -0.20229461789131165, -0.41939881443977356, 0.0010209680767729878, 0.26142945885658264, -0.07151638716459274, 0.8005883693695068, -0.06399958580732346, 0.23695984482765198, 0.16919590532779694, -0.06248993054032326, 0.22761258482933044, 0.150313138961792, -0.4466009736061096, -0.13777336478233337, -0.4339684247970581, 0.6857879757881165, -0.1644611805677414, -0.19357582926750183, -0.08190259337425232, -0.3256019949913025, -0.23252075910568237, -0.2219105064868927, 0.08671600371599197, -0.22160464525222778, -0.019042719155550003, -0.4057118892669678, 0.6119692325592041], [0.07444711029529572, 0.4248736500740051, 0.09074763208627701, -0.17064203321933746, 0.04804157838225365, -0.0024612336419522762, -0.21666821837425232, 0.18134228885173798, -0.21325096487998962, -0.18928685784339905, -0.08697543293237686, 0.17942209541797638, 0.14353282749652863, -0.10213800519704819, -0.39058101177215576, 0.3632097840309143, 0.05457892641425133, -0.07390594482421875, -0.26437777280807495, 0.3012333810329437, -0.00012660108041018248, -0.32655569911003113, -0.18468214571475983, 0.02973051741719246, -0.1677478700876236, -0.6045925617218018, 0.054419178515672684, 0.012776640243828297, -0.2791455090045929, 0.18355022370815277, 0.06033334508538246, 0.5003678798675537], [0.013779683038592339, 0.35516995191574097, 0.20726515352725983, -0.3905019462108612, -0.16286106407642365, 0.3717709183692932, -0.2039407342672348, 0.07250960171222687, -0.22932137548923492, -0.3542415499687195, -0.3307570517063141, -0.4597770869731903, 0.24704433977603912, -0.22984251379966736, -0.008321681059896946, 0.6494817137718201, -0.23733851313591003, 0.2704252600669861, -0.12199926376342773, 0.0379105843603611, 0.2540148198604584, -0.41273564100265503, -0.11908119916915894, 0.13447825610637665, -0.16271749138832092, -0.33760011196136475, 0.03985271975398064, -0.11682448536157608, 0.007734432816505432, 0.027798864990472794, -0.0630539208650589, 0.31889215111732483], [0.16873952746391296, -0.0811554491519928, -0.2894054353237152, -0.1436050832271576, 0.14004865288734436, 0.13782666623592377, 0.13633425533771515, 0.2703465223312378, -0.11609157919883728, -0.15678763389587402, -0.0627453625202179, 0.016596298664808273, 0.1171116754412651, -0.009845206514000893, 0.04368551820516586, 0.2963695526123047, -0.254917174577713, 0.032876256853342056, 0.07939741760492325, -0.22793208062648773, 0.17609860002994537, -0.05989082530140877, -0.17899632453918457, -0.06407490372657776, -0.11032706499099731, -0.0376833900809288, -0.12517701089382172, 0.07496695220470428, 0.121897853910923, -0.07691626995801926, -0.13547173142433167, 0.23796015977859497], [0.14732623100280762, -0.0607951395213604, -0.085936039686203, -0.07468785345554352, -0.11728111654520035, 0.263824999332428, 0.20962969958782196, 0.22950296103954315, -0.022774776443839073, -0.05412152409553528, 0.2840674817562103, 0.1303376406431198, 0.050772927701473236, -0.2209898829460144, -0.06564410030841827, 0.2819294035434723, -0.13280804455280304, 0.06176028028130531, -0.0029531484469771385, -0.08124423027038574, 0.29398462176322937, -0.11720629781484604, -0.27709612250328064, -0.022551260888576508, -0.06590411067008972, -0.1407758742570877, -0.0434018038213253, 0.12331854552030563, 0.10605257004499435, -0.04650802165269852, -0.09217188507318497, 0.27737948298454285], [0.1459779143333435, 0.09589561820030212, 0.04715614765882492, -0.17351844906806946, -0.2181226760149002, 0.19842469692230225, -0.07507705688476562, 0.2992649972438812, 0.08385276794433594, -0.132543683052063, 0.028534863144159317, 0.013542369939386845, 0.041961923241615295, 0.05607297271490097, 0.07580816000699997, 0.23187071084976196, -0.049715716391801834, -0.3097713887691498, -0.08664315193891525, 0.10261478275060654, 0.26190486550331116, -0.29024139046669006, -0.2321552336215973, 0.11061383038759232, -0.1694396585226059, -0.29378753900527954, 0.06863116472959518, 0.2773763835430145, 0.013150110840797424, -0.049481287598609924, 0.034664593636989594, 0.3482673764228821], [0.09714796394109726, 0.1666816771030426, -0.18350748717784882, -0.2053980529308319, -0.1726934313774109, 0.14062409102916718, 0.11188112944364548, 0.2719121277332306, -0.1411030888557434, -0.09666499495506287, -0.1887943595647812, -0.037003807723522186, 0.06374763697385788, -0.010346739552915096, -0.12482435256242752, 0.3238396644592285, -0.15173128247261047, -0.20423203706741333, -0.05574193596839905, -0.13784052431583405, 0.30144375562667847, -0.12487685680389404, -0.1908101737499237, 0.011550228111445904, -0.16905102133750916, -0.10838283598423004, -0.11117763072252274, 0.25923824310302734, -0.031521640717983246, -0.0981171503663063, -0.1063028946518898, 0.22074741125106812], [-0.011011342518031597, -0.1097927913069725, 0.10406051576137543, -0.24422544240951538, -0.0033352074678987265, 0.40249574184417725, 0.12425491958856583, 0.33691954612731934, -0.024633020162582397, -0.05219274014234543, -0.002487932564690709, -0.0746913030743599, -0.004694756120443344, -0.2708160877227783, -0.07260560244321823, 0.3471388518810272, -0.12127923965454102, -0.19616740942001343, -0.01638772524893284, -0.041469424962997437, 0.24437668919563293, -0.21629899740219116, -0.2418876737356186, 0.16422533988952637, -0.0026320088654756546, -0.09794946014881134, -0.1362847238779068, 0.1386425793170929, 0.25589847564697266, -0.01600435934960842, -0.021624239161610603, 0.3814048767089844], [0.12397253513336182, -0.21839043498039246, -0.23943644762039185, 0.10356519371271133, 0.001103269518353045, 0.2569946348667145, -0.3359571099281311, -2.298423623869894e-06, 0.25985848903656006, 0.09127436578273773, 0.009066118858754635, -0.013999504037201405, 0.08203595876693726, -0.026833320036530495, -0.2009209245443344, 0.2679421305656433, -0.13132621347904205, -0.5016269683837891, -0.20222356915473938, 0.14963896572589874, 0.20444779098033905, -0.2929692566394806, -0.4716775715351105, -0.020634226500988007, -0.3184013366699219, 0.24063821136951447, -0.292702317237854, 0.22191716730594635, -0.29151761531829834, 0.045568518340587616, -0.008259753696620464, 0.31608471274375916], [-0.037035681307315826, -0.047653928399086, -0.2293938547372818, -0.11627840995788574, -0.09356250613927841, 0.27078473567962646, 0.05659623071551323, 0.1670081466436386, -0.01359116192907095, -0.06571957468986511, -0.2736235558986664, -0.037851482629776, -0.006204930599778891, -0.1606171429157257, -0.02130352146923542, 0.24754060804843903, -0.012936646118760109, -0.17904680967330933, -0.09424290060997009, -0.010128997266292572, 0.16653186082839966, -0.2778801918029785, -0.3039705157279968, -0.3591160774230957, 0.1026477962732315, 0.0297827310860157, 0.055241651833057404, 0.12162415683269501, 0.012505579739809036, -0.008029747754335403, 0.05009196698665619, 0.24318334460258484], [0.1715121567249298, 0.2045941948890686, -0.0657048150897026, -0.2500249445438385, -0.027986569330096245, 0.12077587842941284, -0.05779280886054039, 0.20296433568000793, -0.04674671217799187, -0.0409441702067852, -0.2849595546722412, -0.07303572446107864, 0.2646239101886749, -0.028803065419197083, -0.15695340931415558, 0.22405415773391724, -0.3010365962982178, -0.14471197128295898, -0.12104343622922897, -0.024728069081902504, 0.40200290083885193, -0.4194444417953491, -0.03344093635678291, 0.07686789333820343, -0.14224624633789062, -0.17230771481990814, -0.1404637098312378, 0.12754221260547638, 0.13105347752571106, -0.043246399611234665, -0.0567273311316967, 0.19260278344154358], [0.3827051520347595, 0.0695151537656784, -0.07661265879869461, 0.051311880350112915, 0.3090110719203949, 0.1679975688457489, -0.22415383160114288, 0.3564451336860657, 0.22810959815979004, -0.32087281346321106, 0.1463954746723175, 0.33267420530319214, -0.06920971721410751, -0.1810930222272873, 0.18051159381866455, 0.34767675399780273, -0.2294309139251709, -0.21138571202754974, 0.21765287220478058, 0.29801860451698303, 0.11423804610967636, -0.2796042263507843, 0.03305288776755333, 0.07968660444021225, -0.0724465548992157, -0.31084227561950684, -0.02319648675620556, -0.10393892228603363, 0.13478624820709229, 0.14555910229682922, 0.09075216948986053, 0.393370658159256], [0.09452349692583084, -0.05557168647646904, -0.10545399785041809, -0.05538113787770271, 0.02556207776069641, 0.25641223788261414, 0.017052408307790756, 0.241225928068161, 0.09161125123500824, -0.1624418944120407, 0.2666885554790497, 0.13452787697315216, 0.13247181475162506, -0.12951985001564026, -0.02508997544646263, 0.2285582721233368, -0.20610260963439941, -0.3807918131351471, -0.05352948606014252, -0.3031970262527466, 0.21992357075214386, -0.3931160867214203, -0.111481674015522, 0.05770876631140709, -0.009346658363938332, -0.16451427340507507, -0.06298907846212387, 0.12227728962898254, 0.07656893134117126, -0.0930110365152359, -0.0656166598200798, 0.19177164137363434], [0.16761070489883423, 0.06132030114531517, 0.02041023224592209, -0.012459343299269676, 0.11187992244958878, 0.34161022305488586, -0.027995580807328224, 0.32617491483688354, 0.042880963534116745, -0.3300842344760895, 0.03799392655491829, 0.1898781657218933, 0.08130308985710144, -0.07616662234067917, 0.10664017498493195, 0.29873719811439514, -0.1799931526184082, -0.07467503100633621, 0.09810944646596909, -0.08411765098571777, 0.12134459614753723, -0.20076674222946167, -0.027305692434310913, 0.008073079399764538, -0.11004984378814697, -0.30837738513946533, -0.18621951341629028, 0.1510777324438095, 0.11502895504236221, 0.04113594815135002, -0.15663233399391174, 0.08785267919301987], [0.11637531965970993, 0.05259757861495018, -0.27882760763168335, -0.17472286522388458, -0.0923079252243042, 0.15060684084892273, 0.06257729232311249, 0.3479912281036377, -0.12061252444982529, -0.07866371423006058, -0.18788649141788483, 0.0663314089179039, -0.2592184245586395, -0.0871928483247757, -0.01751064322888851, 0.15074294805526733, -0.08462728559970856, -0.1485995352268219, -0.004894856363534927, -0.13475635647773743, 0.0956149771809578, -0.21115660667419434, -0.3070591390132904, -0.08005084097385406, -0.016579803079366684, -0.0883958637714386, 0.01689084805548191, -0.06463650614023209, -0.08661441504955292, -0.17096994817256927, 0.05012265592813492, 0.2062588632106781], [0.23476020991802216, 0.11978252232074738, -0.2205648273229599, 0.1425742208957672, 0.05853522941470146, -0.045974623411893845, 0.11449196934700012, 0.15075810253620148, 0.09384400397539139, -0.24776440858840942, -0.0017154432134702802, 0.01802048459649086, -0.01514813955873251, 0.00011862779501825571, -0.25611940026283264, 0.22091718018054962, -0.1730390191078186, -0.30732688307762146, 0.16100725531578064, -0.046899594366550446, 0.017596792429685593, -0.27417758107185364, -0.2089732140302658, -0.21031787991523743, 0.006935927085578442, -0.023089615628123283, -0.2659081220626831, 0.29786187410354614, 0.05961662903428078, -0.06680002063512802, -0.2184690237045288, 0.30734214186668396], [-0.09149471670389175, -0.11838320642709732, -0.2253946214914322, 0.17133888602256775, 0.32360219955444336, -0.09369277954101562, 0.32258474826812744, 0.4123879671096802, 0.049653537571430206, -0.14767643809318542, -0.027936294674873352, -0.22243383526802063, -0.037375058978796005, -0.18304890394210815, -0.7322334051132202, -0.004636638332158327, -0.6595781445503235, -0.6377237439155579, 0.07944102585315704, -0.41403064131736755, -0.4173867404460907, -0.7237013578414917, -0.18513227999210358, -0.13293391466140747, 0.07848615944385529, 0.08363249897956848, -0.2533093988895416, 0.5429649353027344, -0.22754646837711334, -0.3632350564002991, -0.010725561529397964, 0.5257048010826111], [0.27873167395591736, 0.5809803009033203, -0.3317915201187134, -0.26432979106903076, 0.26334601640701294, 0.43323850631713867, -0.032141804695129395, 0.6542877554893494, 0.2593175172805786, -0.17837052047252655, 0.8904028534889221, -0.163380429148674, 0.37945377826690674, -0.3794826567173004, -0.022354960441589355, 0.2781865894794464, -0.5093463659286499, -0.31943371891975403, -0.005659205839037895, 0.007445295341312885, 0.979453444480896, -0.915741503238678, 0.1072797030210495, 0.2677483856678009, -0.23053377866744995, -0.7873721718788147, 0.12266335636377335, -0.09330306947231293, -0.025873681530356407, -0.27834632992744446, 0.0028153667226433754, 0.20535270869731903], [0.1557978242635727, 0.14914323389530182, -0.11295148730278015, -0.1348222941160202, 0.05837639793753624, 0.36281341314315796, -0.17791055142879486, 0.3535856604576111, -0.038472894579172134, -0.28760215640068054, -0.018847012892365456, 0.06353388726711273, 0.29281479120254517, -0.18805722892284393, 0.19633246958255768, 0.36586764454841614, -0.30669066309928894, -0.01290665939450264, -0.09617992490530014, -0.20607706904411316, 0.29249459505081177, -0.2587336301803589, -0.11363425105810165, 0.0594695508480072, -0.062100887298583984, -0.2088453769683838, -0.03844750300049782, -0.023574089631438255, 0.14785166084766388, -0.08747327327728271, -0.05787342041730881, 0.2644502818584442], [0.0999770313501358, 0.09864868223667145, -0.3116430342197418, -0.2554306983947754, 0.10869397968053818, 0.22455324232578278, 0.09913433343172073, 0.2387877106666565, -0.14512141048908234, -0.012045351788401604, 0.054518233984708786, -0.09625957906246185, 0.11678489297628403, 0.06470022350549698, -0.08017220348119736, 0.42377549409866333, -0.154572531580925, -0.0283164344727993, -0.16595377027988434, 0.13361716270446777, 0.22088105976581573, -0.3136295974254608, -0.1427920162677765, -0.16384528577327728, -0.2596856653690338, -0.14956171810626984, -0.1966320425271988, 0.12685827910900116, -0.04908253997564316, 0.1447337120771408, 0.054641179740428925, 0.3218093514442444], [0.12269126623868942, -0.043361734598875046, -0.0705975741147995, 0.07386435568332672, -0.0035303663462400436, 0.1952236294746399, -0.05835441127419472, 0.2060452401638031, 0.02737666480243206, 0.0031963528599590063, 0.10320638120174408, 0.20527318120002747, 0.270376980304718, -0.03889567777514458, 0.10923095792531967, 0.21279145777225494, -0.3704400956630707, -0.026651276275515556, 0.09595347195863724, -0.12185703963041306, 0.3136601150035858, -0.2726503312587738, -0.11937525123357773, 0.017404930666089058, -0.164411723613739, -0.04885685816407204, -0.11548284441232681, 0.06376096606254578, 0.03688378259539604, 0.09528548270463943, -0.07434991002082825, 0.23532027006149292], [0.11029983311891556, 0.7226914763450623, -0.024840887635946274, -0.5293817520141602, 0.16526086628437042, 0.31188860535621643, -0.31434425711631775, 0.459994375705719, 0.30007579922676086, -0.35654935240745544, -0.20666812360286713, -0.0942026898264885, -0.13071225583553314, 0.0036390202585607767, 0.26170673966407776, 0.31688961386680603, -0.19248652458190918, -0.4842500388622284, 0.057458627969026566, -0.7574959397315979, -0.2830890417098999, -0.22075015306472778, -0.34146296977996826, 0.042814649641513824, 0.13581609725952148, -0.395405650138855, -0.2742930054664612, -0.27887773513793945, 0.774372398853302, -0.169952854514122, -0.1794070452451706, 0.16663265228271484], [0.144317165017128, -0.0021461332216858864, -0.3389282822608948, 0.011217436753213406, 0.08708122372627258, 0.05827390030026436, 0.02266700193285942, 0.25520431995391846, -0.056266944855451584, -0.2407071888446808, 0.18852084875106812, 0.08186053484678268, 0.05764183774590492, 0.04327380657196045, -0.06264520436525345, 0.2072848379611969, 0.08102614432573318, -0.2097640037536621, -0.11364926397800446, -0.12027795612812042, 0.2052621990442276, -0.13394275307655334, -0.2583574950695038, -0.17916233837604523, -0.2228059321641922, -0.07012411206960678, -0.1041586771607399, 0.19239948689937592, -0.12903723120689392, -0.12191354483366013, -0.07547271251678467, 0.33111676573753357], [0.10514437407255173, -0.026385797187685966, -0.30828657746315, -0.05451924353837967, 0.037236906588077545, 0.2515604794025421, 0.0933685302734375, 0.23828135430812836, -0.10167907178401947, -0.040507230907678604, -0.0030217289458960295, 0.05717480927705765, 0.012591809965670109, -0.13596388697624207, -0.11362860351800919, 0.17787665128707886, -0.19876568019390106, -0.16416305303573608, 0.03669470548629761, -0.17720447480678558, 0.13646668195724487, -0.10092540085315704, -0.31839507818222046, 0.02873879484832287, -0.16057585179805756, 0.041715290397405624, -0.2411365956068039, 0.22679777443408966, 0.004081284161657095, -0.0465347096323967, -0.13173529505729675, 0.10844235867261887], [0.21463672816753387, 0.07356709986925125, 0.08589348196983337, -0.18578596413135529, 0.006431751884520054, 0.3032555878162384, -0.006090409122407436, 0.22905611991882324, 0.05274415761232376, -0.017166903242468834, 0.08022825419902802, -0.010561123490333557, 0.14421118795871735, -0.08175208419561386, 0.07152070850133896, 0.4385412633419037, -0.21558186411857605, -0.22858723998069763, 0.05118123069405556, 0.21249417960643768, 0.47368761897087097, -0.40601322054862976, -0.09482802450656891, 0.09920930117368698, -0.37607693672180176, -0.39314404129981995, 0.023973319679498672, 0.1524631530046463, 0.037058789283037186, 0.09314548969268799, 0.024390818551182747, 0.2630400061607361], [0.12145943939685822, 0.005363408010452986, -0.08530489355325699, -0.10539454221725464, 0.01926582306623459, 0.17972779273986816, 0.15605491399765015, 0.17626318335533142, -0.11912759393453598, -0.14392077922821045, 0.06549578160047531, 0.06177796423435211, -0.13998624682426453, -0.007615435402840376, -0.09330818057060242, 0.4771744906902313, -0.025945162400603294, 0.032183825969696045, 0.010200852528214455, 0.040907375514507294, 0.11317865550518036, -0.16538383066654205, -0.26285308599472046, -0.09796550124883652, -0.08591648191213608, -0.33177265524864197, -0.1441221535205841, 0.06787226349115372, -0.1363549530506134, 0.09168849885463715, 0.021343814209103584, 0.34093034267425537], [0.10398127883672714, -0.1092272698879242, -0.16904550790786743, 0.04988221824169159, 0.04018380120396614, 0.1855747401714325, 0.05960121005773544, 0.3010994791984558, -0.06296996772289276, -0.15759584307670593, 0.0491635762155056, -0.004995051305741072, -0.04398919641971588, -0.12171628326177597, -0.13863183557987213, 0.14579430222511292, -0.22739410400390625, -0.3120059669017792, 0.03315171226859093, -0.18021556735038757, -0.02579367719590664, -0.3589942157268524, -0.23051191866397858, -0.1683356612920761, -0.030736345797777176, 0.024914247915148735, -0.24085725843906403, 0.20783956348896027, -0.11231600493192673, -0.18732808530330658, -0.07439568638801575, 0.26153236627578735], [0.0761234387755394, 0.20782381296157837, -0.423069566488266, -0.100143201649189, 0.12952397763729095, 0.09266114979982376, 0.06841591000556946, 0.3059278428554535, -0.10000903159379959, -0.11673954129219055, -0.03904338181018829, 0.13492517173290253, -0.007570894435048103, -0.03416145592927933, -0.09466855973005295, 0.2228548675775528, -0.3238798677921295, -0.11885225772857666, 0.0585813894867897, -0.31131619215011597, 0.11047162115573883, -0.15627944469451904, -0.14984340965747833, 0.0018242623191326857, -0.03228600695729256, -0.026543226093053818, -0.23273099958896637, 0.06406667828559875, 0.11263071745634079, -0.041206784546375275, -0.12769633531570435, 0.25351428985595703], [0.31283074617385864, -0.2600156366825104, -0.14935046434402466, -0.11685970425605774, 0.19331423938274384, 0.12679284811019897, 0.2951144576072693, 0.4143054485321045, -0.09034532308578491, -0.27603188157081604, -0.039568282663822174, -0.12328856438398361, -0.05047985911369324, -0.0786769911646843, 0.05504308640956879, 0.3431164622306824, -0.23579804599285126, -0.13289576768875122, 0.2364470213651657, -0.22927968204021454, 0.05344086512923241, -0.206288680434227, -0.055582061409950256, -0.15192218124866486, -0.009659403003752232, -0.05654112249612808, -0.4505116641521454, -0.15134699642658234, 0.08315176516771317, -0.13886593282222748, -0.28307655453681946, 0.13048022985458374], [-0.06259439140558243, 0.2464316189289093, -0.39694568514823914, -0.04500027000904083, 0.1230112761259079, 0.4025287628173828, -0.02423880062997341, 0.19778645038604736, -0.20373429358005524, 0.03575410693883896, 0.028426911681890488, 0.24281607568264008, 0.14979857206344604, -0.1858302652835846, 0.07369201630353928, 0.3122289180755615, -0.15199558436870575, 0.023349227383732796, -0.21525810658931732, -0.3742070496082306, 0.23925825953483582, -0.23517568409442902, -0.3112430274486542, 0.08965475857257843, 0.024581152945756912, -0.25596684217453003, -0.09633831679821014, -0.08880563825368881, 0.18947544693946838, 0.027298426255583763, 0.04687320813536644, 0.32855767011642456], [0.1974027156829834, 0.08571624755859375, -0.10977306962013245, -0.08610326051712036, -0.023814380168914795, 0.1956433653831482, -0.1043495386838913, 0.24095748364925385, -0.04922755807638168, -0.21965885162353516, 0.009179129265248775, 0.2725389003753662, 0.07665664702653885, -0.27774056792259216, -0.0275113582611084, 0.25926488637924194, -0.022307289764285088, 0.06505372375249863, -0.020532505586743355, -0.33826911449432373, 0.04179281368851662, -0.19928136467933655, -0.11033366620540619, 0.04351947829127312, -0.016042836010456085, -0.3817116916179657, 0.17810718715190887, -0.13193151354789734, 0.2243465632200241, -0.10545189678668976, -0.06578640639781952, 0.4055098295211792], [0.15764299035072327, 0.0483618900179863, -0.1432335376739502, 0.026710916310548782, 0.2796325385570526, 0.2401411533355713, -0.026027197018265724, 0.32514598965644836, -0.03798636794090271, -0.08112428337335587, 0.08717945963144302, 0.3096156418323517, -0.012656820006668568, -0.086890310049057, 0.11303101480007172, 0.10992997139692307, -0.2993185818195343, -0.12299402803182602, 0.3537077009677887, -0.1583733856678009, 0.09275201708078384, -0.2217523604631424, 0.03159399330615997, 0.36928990483283997, -0.28973013162612915, -0.026844294741749763, -0.05056940019130707, 0.13247357308864594, 0.1057075634598732, 0.1865452378988266, -0.11258786171674728, 0.1626698076725006], [0.16943739354610443, -0.012262375093996525, -0.12556537985801697, 0.08710486441850662, -0.06568902730941772, 0.15687645971775055, -0.13776057958602905, 0.4144488275051117, 0.39440351724624634, -0.06046821549534798, 0.3624635636806488, -0.20854288339614868, 0.24360309541225433, 0.09051135927438736, -0.052336111664772034, 0.09546405076980591, -0.2133326381444931, -0.4849134683609009, -0.10585623979568481, -0.030442966148257256, 0.6652735471725464, -0.24431797862052917, -0.27609825134277344, 0.13433203101158142, -0.3018694818019867, 0.13271120190620422, -0.27020397782325745, 0.2721061408519745, -0.294192910194397, -0.1171841025352478, 0.01011411752551794, 0.5168924927711487], [0.13418687880039215, -0.01889750175178051, -0.028789715841412544, 0.0017314240103587508, -0.19173593819141388, 0.24417221546173096, 0.05920916050672531, 0.3665699362754822, -0.06423749774694443, -0.20059461891651154, 0.16159701347351074, 0.23174837231636047, -0.04742259904742241, -0.30822309851646423, -0.28802692890167236, 0.37371253967285156, -0.02903374284505844, 0.2529747784137726, -0.15485543012619019, 0.1580876111984253, 0.06277526170015335, 0.03152431547641754, -0.48450419306755066, -0.019537942484021187, 0.3291373550891876, -0.2915169596672058, -0.08422824740409851, -0.11376571655273438, -0.20316104590892792, 0.020659668371081352, 0.14532223343849182, 0.24978847801685333], [0.21343488991260529, 0.18882811069488525, -0.4318082630634308, -0.12449304759502411, 0.004074815195053816, 0.12567366659641266, -0.042677562683820724, 0.40656179189682007, 0.07344383746385574, -0.04531366750597954, 0.0370388962328434, 0.025045374408364296, -0.01852911151945591, -0.1767456978559494, -0.07529277354478836, 0.11965455114841461, -0.07073456048965454, -0.2585585117340088, 0.1324426233768463, -0.34516841173171997, 0.4454064667224884, -0.27056729793548584, -0.22717894613742828, 0.17476177215576172, -0.1522735357284546, -0.16637590527534485, -0.08231233805418015, -0.036464232951402664, 0.3166593909263611, -0.15351344645023346, -0.12747713923454285, -0.046096112579107285], [0.07148470729589462, 0.14064456522464752, -0.17576053738594055, 0.09400429576635361, -0.03485393896698952, 0.024810856208205223, -0.1291796714067459, 0.20341439545154572, 0.03554254397749901, -0.25480517745018005, 0.0739656314253807, 0.1776224672794342, -0.20013874769210815, -0.1351723074913025, 0.04732133820652962, 0.39877450466156006, -0.0921543762087822, -0.3419150114059448, 0.12527267634868622, -0.3288564682006836, -0.07363500446081161, -0.21390525996685028, -0.29951053857803345, -0.03873629495501518, 0.12188979983329773, -0.23283661901950836, -0.0574985072016716, -0.03899901732802391, 0.10464925318956375, -0.05107059329748154, -0.15745529532432556, 0.3893406391143799], [0.26829251646995544, -0.11186348646879196, -0.3000468909740448, -0.04237440600991249, 0.13689599931240082, 0.08194877952337265, 0.15611912310123444, 0.17912280559539795, -0.1516684740781784, -0.18546754121780396, -0.007218432612717152, -0.017777230590581894, 0.13379690051078796, -0.012787451036274433, -0.18808995187282562, 0.15731044113636017, -0.24018201231956482, -0.22248069941997528, 0.03718141093850136, -0.3500215411186218, 0.11734677851200104, -0.1031888872385025, -0.4050394296646118, -0.1290374994277954, -0.1148940697312355, -0.04597873613238335, -0.2570166289806366, 0.2635112404823303, -0.02103910781443119, -0.25250425934791565, -0.25563785433769226, 0.10657438635826111], [-0.05638686940073967, -0.02468149922788143, -0.08319541066884995, -0.3348029553890228, -0.22746872901916504, 0.37830570340156555, 0.21532343327999115, 0.2651137709617615, -0.1471538096666336, 0.034381840378046036, -0.12423396855592728, -0.02075783722102642, 0.023142438381910324, -0.1624511331319809, -0.15263541042804718, 0.1448785662651062, -0.058530453592538834, -0.06658975034952164, -0.21832606196403503, 0.04305773228406906, 0.21254928410053253, -0.3852666914463043, -0.42259106040000916, -0.030246172100305557, -0.03172217682003975, -0.06608089059591293, 0.067087821662426, 0.3098345696926117, -0.1795438826084137, -0.05489131435751915, -0.025173839181661606, 0.15868337452411652], [0.09842794388532639, 0.02602008730173111, -0.5290498733520508, -0.224226713180542, -0.11993079632520676, 0.18381831049919128, -0.10682738572359085, 0.09668545424938202, 0.09387193620204926, 0.01014670915901661, 0.15185853838920593, -0.11687499284744263, -0.06116899102926254, 0.2480064183473587, -0.2939780652523041, 0.410091757774353, -0.009401433169841766, -0.2665918171405792, -0.2505125105381012, 0.07581187039613724, 0.1673247218132019, -0.39204835891723633, -0.23963910341262817, -0.16767919063568115, -0.10764669626951218, 0.014944424852728844, -0.19886083900928497, 0.23898674547672272, -0.3457018733024597, 0.13235381245613098, 0.10067809373140335, 0.39275413751602173], [0.34566670656204224, -0.07597880810499191, -0.2369421124458313, 0.23202507197856903, -0.1716957837343216, -0.069289930164814, -0.017548831179738045, 0.0855400338768959, 0.0077080805785954, -0.5554392337799072, 0.1546430140733719, 0.20683541893959045, -0.002290950855240226, -0.1930643767118454, -0.4367334842681885, 0.20145061612129211, 0.2975526452064514, -0.21096597611904144, -0.34793534874916077, 0.017914796248078346, -0.15059791505336761, -0.22732169926166534, -0.45679599046707153, -0.09626180678606033, 0.06647901982069016, 0.007297117728739977, -0.2009778767824173, -0.07010775804519653, -0.32078292965888977, -0.09249463677406311, -0.07466912269592285, 0.23874711990356445], [-0.017029011622071266, -0.10294078290462494, -0.22354045510292053, 0.022406749427318573, -0.06902199238538742, 0.28204429149627686, 0.06653571873903275, 0.12432282418012619, -0.30218860507011414, -0.12153232842683792, -0.12378139793872833, 0.102401502430439, 0.09472078830003738, -0.2321498692035675, -0.03823772445321083, 0.49529045820236206, 0.20580758154392242, 0.06240770220756531, -0.18036332726478577, -0.15944430232048035, 0.08345168083906174, -0.17901727557182312, -0.3251996338367462, -0.028912534937262535, -0.02083352394402027, -0.186011403799057, 0.10621800273656845, 0.10448027402162552, -0.030629321932792664, 0.04704362899065018, -0.030090903863310814, 0.23781011998653412], [0.31053611636161804, 0.2368336170911789, -0.12658953666687012, -0.12955887615680695, 0.29469627141952515, 0.08496394753456116, 0.16536250710487366, 0.1622692495584488, 0.06455150991678238, -0.36596810817718506, 0.375912606716156, 0.22756104171276093, -0.06876635551452637, -0.1668013334274292, -0.2991364300251007, 0.07807552069425583, -0.15066586434841156, -0.10217337310314178, 0.031498197466135025, -0.2666851282119751, 0.31624624133110046, -0.3063410222530365, -0.2606535255908966, -0.12789417803287506, -0.06062943488359451, -0.3188454508781433, -0.25142645835876465, 0.17653490602970123, 0.30489832162857056, -0.08059579133987427, -0.22635948657989502, 0.1538945734500885], [0.24501091241836548, 0.09621678292751312, -0.15270406007766724, 0.15355785191059113, -0.058112502098083496, 0.10902216285467148, -0.07504591345787048, -0.029916202649474144, 0.21821914613246918, 0.07510949671268463, 0.16579023003578186, 0.008549174293875694, 0.22505836188793182, 0.08703390508890152, -0.4413432776927948, 0.29949501156806946, -0.13129271566867828, -0.21776342391967773, -0.1684812307357788, 0.35155966877937317, 0.3982027769088745, -0.280237078666687, -0.32525599002838135, -0.034190088510513306, -0.4295347332954407, -0.13327501714229584, -0.19575026631355286, 0.32491111755371094, -0.48330119252204895, 0.05046780779957771, -0.12234652042388916, 0.4025035798549652], [0.19056618213653564, 0.2754313051700592, -0.1973477452993393, -0.05418064072728157, -0.19817765057086945, -0.0689924880862236, 0.01602395810186863, 0.0774272233247757, 0.15873083472251892, -0.05062582343816757, 0.35625553131103516, 0.11868394911289215, -0.07927273213863373, 0.08859873563051224, -0.33440086245536804, 0.31461814045906067, 0.007754262536764145, -0.25359320640563965, -0.076240174472332, -0.16910381615161896, 0.38860493898391724, -0.3562571406364441, -0.09599103778600693, -0.0641888976097107, -0.21759337186813354, -0.3108900487422943, -0.18171019852161407, 0.2882768213748932, -0.346527099609375, 0.009131056256592274, -0.18686458468437195, 0.44769471883773804], [0.09849075973033905, 0.03861629217863083, -0.12478663772344589, -0.15777844190597534, 0.04659232124686241, 0.263857901096344, -0.0123446648940444, 0.31121334433555603, -0.04379965737462044, -0.12922996282577515, 0.02243671752512455, -0.06668628752231598, -0.07193218916654587, -0.20660479366779327, -0.018530327826738358, 0.2959209978580475, -0.06899344176054001, -0.3189965784549713, 0.04238409548997879, -0.05908942595124245, 0.11208918690681458, -0.29284951090812683, -0.2210102081298828, -0.09693700820207596, -0.09873301535844803, -0.02719877101480961, -0.07841768860816956, 0.1302885115146637, 0.12441369891166687, -0.02614641934633255, 0.021873658522963524, 0.3173949718475342], [-0.052881523966789246, -0.172337144613266, -0.05718238651752472, -0.12337962538003922, -0.00391408009454608, 0.15878033638000488, 0.10104932636022568, 0.13058212399482727, -0.015865596011281013, -0.17811253666877747, 0.060681167989969254, -0.03700766712427139, 0.25286588072776794, -0.019453909248113632, 0.0166354738175869, 0.39918777346611023, -0.07251755148172379, -0.03815699741244316, -0.061090100556612015, -0.02518760785460472, 0.4494858682155609, -0.06706247478723526, -0.25747618079185486, 0.04654408618807793, -0.22732730209827423, -0.16999280452728271, -0.07514441013336182, 0.2431950718164444, 0.02677714079618454, 0.04784814268350601, -0.1250096708536148, 0.29738953709602356]]]], "dtype": "float32"}, {"__nd__": [[[4.466150760650635, 4.391090393066406, 4.455203056335449, 4.5211501121521, 5.04282808303833, 4.487778663635254, 4.807657241821289, 4.698809623718262, 4.472257614135742, 4.417579650878906, 4.588783264160156, 4.400468349456787, 4.354161262512207, 4.5076446533203125, 4.809414386749268, 4.672336101531982, 4.994312286376953, 4.985067844390869, 4.594513416290283, 4.553776741027832, 4.377108573913574, 4.425689697265625, 4.568533420562744, 4.545281887054443, 4.726986885070801, 4.2636871337890625, 5.05588960647583, 4.702642440795898, 4.280727863311768, 5.112557888031006, 4.584402561187744, 4.335085391998291, 4.6867289543151855, 4.333950519561768, 4.749272346496582, 4.5815887451171875, 4.662835597991943, 4.446969509124756, 4.279399871826172, 4.334447860717773, 4.419992923736572, 4.42582893371582, 4.4038825035095215, 4.582939147949219, 4.481939315795898, 4.475002288818359, 4.519717693328857, 4.742208957672119, 4.682601451873779, 5.023257255554199, 4.638187885284424, 4.896790504455566, 4.380489826202393, 4.333329200744629, 4.5122246742248535, 4.826665878295898, 4.699033737182617, 4.465027809143066, 4.749913692474365, 4.487576484680176, 4.567475318908691, 4.522494792938232, 4.628721237182617, 4.654586315155029], [4.936459064483643, 5.06341552734375, 4.751301288604736, 4.734513282775879, 4.878973484039307, 5.1442365646362305, 4.73399543762207, 4.404800891876221, 4.727814197540283, 4.745046615600586, 4.597319602966309, 4.808379650115967, 4.3292765617370605, 4.676347255706787, 4.510926723480225, 4.7222442626953125, 4.649495601654053, 4.572422981262207, 4.6190290451049805, 4.782656669616699, 4.782093524932861, 4.510197162628174, 4.850003719329834, 4.865315914154053, 4.703007221221924, 4.588194370269775, 4.508722305297852, 4.56740140914917, 4.5017194747924805, 4.888218402862549, 4.523769855499268, 4.872211456298828, 4.603984355926514, 4.703784465789795, 4.875594615936279, 4.851465225219727, 4.4340996742248535, 4.611323833465576, 4.2380690574646, 5.397207260131836, 4.472955226898193, 4.352097511291504, 4.730923175811768, 4.52269172668457, 4.169842720031738, 4.486912250518799, 4.767268180847168, 4.896250247955322, 4.425382137298584, 4.77356481552124, 4.859588146209717, 4.684165954589844, 4.654336452484131, 4.713303089141846, 4.720395565032959, 4.743037223815918, 4.478006362915039, 4.790146827697754, 4.578256130218506, 4.849863529205322, 4.580252170562744, 4.707918167114258, 4.3735785484313965, 4.702763080596924]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[[0.13330504298210144, 1.4910486936569214, 0.6109818816184998, 2.05320143699646, -0.5285097360610962, -1.2784080505371094, -2.006896734237671, -0.7847475409507751, -0.8696005940437317, -1.0819967985153198, -1.8414982557296753, -0.17432953417301178, 0.7824640870094299, -0.9883134961128235, -1.6637723445892334, 0.48134633898735046], [-0.5891247987747192, 0.8860313892364502, 0.2537461519241333, 0.09706807881593704, -2.2234714031219482, 0.6290082335472107, -1.1879736185073853, -0.8603944182395935, -1.487733244895935, 0.7892815470695496, 0.8420666456222534, -2.5292420387268066, -0.4565160572528839, 0.10173431783914566, -0.37358564138412476, -2.6811575889587402], [-0.24291996657848358, -0.7138994336128235, 0.031109819188714027, 1.5089995861053467, -0.9503809809684753, -0.23292651772499084, 0.9563882946968079, -0.1643671989440918, -1.209294319152832, 0.31976374983787537, 1.0478044748306274, -1.5640419721603394, 0.7875579595565796, 0.8434394001960754, 0.4706260561943054, 1.5187855958938599], [-1.2428090572357178, -0.9634780287742615, 0.17448696494102478, -0.9190019965171814, 1.1567189693450928, 0.06778954714536667, 1.3031431436538696, -0.010222278535366058, -1.5243215560913086, -0.4687434136867523, -0.7530341148376465, -0.5920475125312805, -1.4341745376586914, 0.6934674978256226, -0.5638181567192078, -0.9522987604141235], [0.7722970843315125, 1.3166016340255737, -1.444740891456604, -0.287548303604126, 0.7651690244674683, -0.012905120849609375, -0.7323826551437378, 0.9631933569908142, 0.2918793857097626, 0.7197622656822205, -0.13453160226345062, 0.5637255311012268, -1.7484186887741089, -0.11264066398143768, -0.595210075378418, -0.8224455714225769], [-0.6945875883102417, 2.009725570678711, -0.527673065662384, -1.6383211612701416, -1.665627360343933, 0.2578955590724945, -0.6960684061050415, 0.011806189082562923, -0.3067002296447754, -0.40850090980529785, -1.1596368551254272, 0.5123271942138672, 1.2318089008331299, 0.5413280725479126, 0.3235149383544922, 0.442084401845932], [-0.41011476516723633, 1.2553482055664062, 0.7150257229804993, 0.6581845283508301, 1.963604211807251, -0.16338592767715454, 0.24791669845581055, 0.8270450830459595, -0.448852002620697, 1.139016032218933, -0.4270252287387848, -0.8658986687660217, 0.5703872442245483, -0.17366215586662292, -0.33565858006477356, 0.6499215364456177], [-0.26080816984176636, -0.6118655800819397, 0.335814505815506, -0.8492388129234314, -0.25480079650878906, -0.5321511030197144, -0.5200638771057129, -0.5759143233299255, -0.44666334986686707, 1.9966058731079102, -0.3996218144893646, 2.5307483673095703, 1.864804744720459, -1.5674973726272583, -1.3336691856384277, -0.18156571686267853], [0.29215237498283386, 0.8335446119308472, 1.0658986568450928, -1.2309379577636719, -1.6301913261413574, 0.9146075248718262, -0.45491549372673035, 0.03940404951572418, -1.1424295902252197, -0.6438719034194946, 1.625343918800354, 1.097844123840332, 0.6114834547042847, 1.863270878791809, 0.8278267979621887, 1.2621121406555176], [0.12677323818206787, -0.7632479071617126, -0.07862455397844315, 0.47196105122566223, -1.1012742519378662, 0.3669239580631256, 0.26802366971969604, 0.047024790197610855, -0.32042163610458374, 2.2097277641296387, -1.1781493425369263, -0.569364070892334, 1.0438297986984253, -0.8226983547210693, -0.20617152750492096, 0.7928037047386169], [0.9819760322570801, -1.1666127443313599, 0.1878056824207306, -1.0589382648468018, 0.9072839021682739, -0.7360327243804932, -0.033581580966711044, -0.47195306420326233, -0.04968930408358574, 0.7388177514076233, -0.8991991877555847, 1.5807443857192993, -0.026517195627093315, -0.003676332999020815, -0.1482372283935547, 0.3531649112701416], [0.9491258859634399, -0.6980014443397522, 0.3266681134700775, -1.371989369392395, -0.8767952919006348, -0.18678449094295502, 1.4975154399871826, -1.750929355621338, -0.9216020107269287, 0.6709652543067932, 0.36177071928977966, 1.624995470046997, -1.2171024084091187, 0.7984981536865234, -0.32490894198417664, -0.2630734443664551], [0.9090593457221985, 0.54717618227005, 1.111419916152954, -0.3098312318325043, 0.48444288969039917, 0.8729915618896484, 0.4617348611354828, 0.6653139591217041, -0.46338406205177307, 1.4913897514343262, 1.9564827680587769, -0.745421826839447, 0.006484166253358126, 0.4757482409477234, 1.7111941576004028, 0.48808449506759644], [-0.12970608472824097, 0.546627938747406, -1.2104697227478027, -1.8721779584884644, -1.129196047782898, 2.032019853591919, 0.8337365388870239, -0.03669983893632889, 0.9818455576896667, -0.5263484120368958, 1.5181435346603394, -1.5296050310134888, -0.13047409057617188, 1.0461692810058594, 0.4774642586708069, 0.046382274478673935], [1.219864010810852, 0.5317400693893433, 1.0630546808242798, -0.8441208004951477, -0.4960431754589081, 0.768880307674408, -1.1016836166381836, 1.2844469547271729, 0.6782668828964233, 0.15873175859451294, 0.22863240540027618, 1.0894379615783691, 1.4059325456619263, 1.666567087173462, 0.29343390464782715, 1.1125589609146118], [-0.5869429707527161, 0.4072088897228241, 0.3947410583496094, -0.5951215028762817, 1.6801705360412598, -0.5130396485328674, -1.2513397932052612, 0.22462402284145355, -0.49932846426963806, 0.890886127948761, 0.24648024141788483, -0.04560310021042824, -0.030127882957458496, -0.2620977759361267, 1.3246328830718994, -0.4078001081943512], [0.2325987070798874, 2.5492889881134033, -1.0899912118911743, 0.7432142496109009, -0.8854728937149048, -1.913645625114441, 1.1067630052566528, -1.1211076974868774, -0.7260372638702393, 0.7328561544418335, 0.04228243976831436, 0.5326107144355774, 0.5101863741874695, 0.4194883406162262, -0.08187247812747955, -0.017454585060477257], [0.756252110004425, -0.5350097417831421, 0.8203164935112, 2.3157923221588135, 1.7875607013702393, 0.36989906430244446, -0.06995578855276108, -0.9234216213226318, 0.19992972910404205, -0.6401841640472412, 0.6130253672599792, 0.8203253149986267, 1.7080912590026855, -0.042957745492458344, -2.5643162727355957, 0.8743153810501099], [1.5401475429534912, 0.8211886286735535, -0.5654230713844299, -1.2403522729873657, -0.6531610488891602, -1.04459547996521, -0.7010635137557983, 2.5036556720733643, -0.22549067437648773, 1.7460154294967651, -0.9264006018638611, -1.8291395902633667, 1.8149795532226562, 1.4599509239196777, 0.008215554989874363, -0.4089318513870239], [-0.8725183010101318, 0.012403854168951511, 1.3473598957061768, -1.0302196741104126, -0.5605232119560242, -0.2688085734844208, 0.5357114672660828, 0.579345166683197, -1.8806512355804443, 1.0942795276641846, 0.06648542732000351, 0.6184306740760803, 0.9332147240638733, -0.9137335419654846, 1.098842978477478, -0.2165941298007965], [0.8321337699890137, -0.5649413466453552, 1.5111159086227417, -0.21804477274417877, 1.7293750047683716, 0.953348696231842, -1.0521843433380127, 0.14992013573646545, 0.24637296795845032, -0.3286976218223572, 0.9637889862060547, -0.7373505234718323, -1.1298284530639648, 0.31821924448013306, 0.11269102990627289, -1.4176541566848755], [0.4464697241783142, 1.7188756465911865, 1.090140461921692, -0.8849309682846069, -1.2669135332107544, 2.8658580780029297, -1.8936810493469238, -0.32947078347206116, 0.16847296059131622, -2.6555800437927246, -0.314036101102829, 0.409774512052536, 0.6273510456085205, 1.9459583759307861, 1.660569429397583, -1.126280665397644], [-0.6858968734741211, 0.18222661316394806, 0.5154560208320618, -0.5824453830718994, -0.4537375569343567, 0.30349087715148926, 0.039907749742269516, -2.2709953784942627, 0.13095712661743164, 0.6454851627349854, -1.379940152168274, 0.08181355148553848, 0.6666683554649353, -0.6958046555519104, 1.3683077096939087, -2.2938883304595947], [1.0107781887054443, 0.46303704380989075, 0.9058675765991211, -0.00790945254266262, 1.2447253465652466, -1.0521897077560425, 0.5425131916999817, -0.9033523797988892, 1.3506410121917725, -1.117161750793457, 0.6287633776664734, -0.620672881603241, -1.645143747329712, -0.6513966917991638, 0.5990286469459534, 0.643665611743927], [1.021228551864624, -0.7975413203239441, -0.7157789468765259, -0.4100338816642761, 0.5384141206741333, 1.875374436378479, 0.6644631624221802, 0.7563202381134033, 1.1153696775436401, -0.993977963924408, -1.9333864450454712, 1.099883794784546, -0.5290693044662476, 0.10923438519239426, -0.6838627457618713, -0.106840580701828], [0.8814848065376282, 0.01293101068586111, -1.178944706916809, -0.48832765221595764, 0.19286145269870758, 0.46689942479133606, -0.8352930545806885, 0.424905925989151, -1.1957855224609375, 0.03210877999663353, 0.024158146232366562, 0.5983030796051025, 0.2801969647407532, -1.6268466711044312, 1.7137726545333862, 1.6160556077957153], [0.7643815279006958, -0.6974748969078064, 0.7018036246299744, -0.7100887894630432, 0.7950292229652405, 0.6265374422073364, -0.8460241556167603, 0.08189325034618378, 1.2652431726455688, -0.43244534730911255, 0.8777797818183899, -0.5439783930778503, -0.8067765235900879, 1.5571929216384888, -0.22451449930667877, -1.5409163236618042], [-2.3700900077819824, -0.23615865409374237, -0.2166934758424759, -0.9522140026092529, 0.21271055936813354, -0.121744804084301, -0.34846755862236023, 0.002023806795477867, -0.43861114978790283, -0.027483360841870308, -0.8926962018013, 1.1593964099884033, 0.5511358976364136, -0.6829182505607605, 0.12638550996780396, -0.09678760915994644], [-0.654645562171936, 0.02411019802093506, -1.8883249759674072, 0.29923108220100403, -0.876293957233429, 3.254112958908081, 0.19683264195919037, 0.17575006186962128, -1.6476002931594849, -0.7259235382080078, 1.146077036857605, 1.033575177192688, 1.7131773233413696, 0.5048545598983765, -1.474063754081726, 1.2665190696716309], [-0.9063137173652649, -2.465951681137085, -0.7265600562095642, 0.3533046245574951, -0.23306351900100708, -0.5393447279930115, 0.3012640178203583, -0.4920230805873871, -0.16115912795066833, 0.28476521372795105, -1.6568771600723267, 0.6906084418296814, -0.33099761605262756, 0.10274911671876907, 0.3484383225440979, -1.349795937538147], [0.7529594302177429, -2.0359294414520264, 3.1462607383728027, -0.15352299809455872, -1.0838390588760376, -2.1279468536376953, 0.5966499447822571, 0.7334855198860168, -0.293288916349411, -0.6481013894081116, 0.2755115032196045, -0.08800890296697617, 1.1962958574295044, 1.432478427886963, 0.9415808320045471, 0.7786052227020264], [0.35764726996421814, 0.5062377452850342, -0.7556841969490051, -2.2651865482330322, -1.3067020177841187, 0.6327863931655884, -1.981598973274231, -0.19681695103645325, 1.298410177230835, 0.8945063948631287, 0.39315328001976013, -0.7052547931671143, 1.125108003616333, 0.21693314611911774, -0.6809102892875671, 1.3997300863265991], [-0.5899627208709717, -0.3630416691303253, 0.4789142608642578, 1.359444260597229, -2.523872137069702, -0.3761809766292572, 0.2188040316104889, 0.37968963384628296, 1.3221979141235352, -0.3484684228897095, 0.6687780022621155, 0.9320436120033264, -1.7781885862350464, -0.8698291778564453, -0.6168245673179626, 0.7355572581291199], [1.5413336753845215, -0.22848907113075256, -0.31168514490127563, 0.07576718926429749, -1.5683989524841309, -1.7511142492294312, 0.009749851189553738, -0.30284416675567627, -0.5829787850379944, -0.6812323927879333, -0.9133625626564026, -0.6450837850570679, -2.574441909790039, 3.342195749282837, 1.2568572759628296, 0.6063619256019592], [-0.7239543795585632, 0.3542020916938782, 0.01741831935942173, 1.5959917306900024, 0.40134942531585693, -0.30333060026168823, 2.4333908557891846, -0.862005889415741, -0.11792450398206711, 2.040858268737793, 1.4520814418792725, -0.5081736445426941, 1.0746814012527466, -0.16492633521556854, -0.4882376492023468, 0.6866196990013123], [1.3179950714111328, 0.766258180141449, -1.2620469331741333, 1.254563331604004, -0.46226102113723755, 1.4616284370422363, 0.5776212215423584, -0.34524455666542053, -1.2109121084213257, -1.4691436290740967, -0.12293248623609543, 1.7902956008911133, 1.216412901878357, 0.635233461856842, 1.1321728229522705, -0.03255199268460274], [-1.8723284006118774, 0.987971305847168, -0.19313688576221466, 0.7345898151397705, 1.1112366914749146, 0.20206047594547272, 0.4208405613899231, -0.5728827118873596, 0.15244895219802856, 1.5169025659561157, 0.9324826002120972, -1.7871220111846924, -0.5893006324768066, -0.8506209850311279, -0.7449735999107361, 0.01908065378665924], [0.17414990067481995, -0.7029238343238831, 1.5613605976104736, 0.5704569816589355, 0.2853071093559265, 0.9991650581359863, -0.8981286883354187, -1.0965067148208618, -0.10236136615276337, -1.1615225076675415, -1.4132505655288696, -0.3693709075450897, -0.9860135912895203, -0.16148516535758972, -1.1715729236602783, -0.17012980580329895], [-1.333693027496338, -0.30956050753593445, 0.6105716228485107, 0.4646625816822052, 0.05145539343357086, -0.34889861941337585, -1.2698769569396973, -0.8734114766120911, 0.05576789379119873, 0.4480924904346466, 0.32957762479782104, 0.6024667024612427, -1.913825511932373, 1.5952966213226318, -0.5167425274848938, -0.6573928594589233], [-1.1971714496612549, -0.34087568521499634, 2.888235330581665, -1.6635854244232178, -0.9223362803459167, 1.0915801525115967, 0.09964055567979813, 0.9449408054351807, -0.6696105599403381, 0.6613572835922241, -0.9541202783584595, 0.5479474067687988, 1.3811137676239014, 1.1598204374313354, -0.12096627801656723, 0.4181166887283325], [-0.10236667096614838, 0.9303434491157532, 0.6122298240661621, -0.42102500796318054, 0.5469202995300293, 0.2757231593132019, -1.1901296377182007, -0.006916177924722433, -0.1398654729127884, -0.029241347685456276, -0.9357616901397705, -0.6170617341995239, -0.5532816052436829, -1.7624419927597046, 0.8731303811073303, 0.012095966376364231], [-1.2587883472442627, -0.05341573804616928, -1.6910266876220703, 1.2469371557235718, -0.9014124274253845, 0.4162355065345764, 0.13447824120521545, 0.5777370929718018, 0.9627954363822937, -0.6041966676712036, -1.37594473361969, -0.4374791085720062, -0.3032079041004181, -1.3415924310684204, -0.1867535412311554, -1.561356544494629], [-0.7898269891738892, -0.6789525747299194, -1.2017475366592407, 0.1932152658700943, -0.5633156299591064, 0.5766313076019287, -0.20556952059268951, 0.7173729538917542, 0.3862297236919403, 0.8963649868965149, -1.0223139524459839, 1.638648509979248, -1.4133129119873047, -0.026975316926836967, -0.08821175992488861, -0.710710346698761], [0.033462729305028915, 0.36370617151260376, 1.0273255109786987, 0.6525850296020508, 0.33940479159355164, 0.19944022595882416, -0.14327678084373474, -1.0812301635742188, 0.1266186386346817, -1.3237131834030151, 0.013529770076274872, 0.41377881169319153, 0.7182692289352417, -1.2725539207458496, -0.8338112235069275, -0.15457001328468323], [-1.4233137369155884, 2.520958423614502, -1.2878257036209106, 0.7955147624015808, -1.3892759084701538, -2.2422637939453125, -1.4993678331375122, 0.18059879541397095, 0.1862216740846634, 0.641930103302002, 0.2538403272628784, -0.014040797017514706, 0.06821870058774948, -2.0870985984802246, -1.7561814785003662, -1.3295971155166626], [0.47352272272109985, 0.5500563383102417, 0.09553581476211548, 1.0200830698013306, -0.7506929039955139, -1.8506659269332886, 0.37131091952323914, -0.9817770719528198, 0.1619456708431244, -1.3533852100372314, -1.042495846748352, 0.5125029683113098, 0.5275166630744934, -0.12871485948562622, 0.5327001810073853, -0.7592951655387878], [0.10730943083763123, -0.06619923561811447, -1.7730932235717773, 0.06093130260705948, 0.03091057576239109, 0.8152212500572205, -1.5650259256362915, 1.360496163368225, 0.058405883610248566, 1.31344473361969, -1.7907689809799194, 2.08673357963562, -1.0194637775421143, -1.4447537660598755, 0.016042238101363182, -0.31589990854263306], [-0.010163964703679085, 0.4698469936847687, 0.7611091136932373, -0.9770491123199463, -1.5055477619171143, 1.3321372270584106, 1.3507622480392456, 0.35924407839775085, -0.4880352318286896, 0.45966365933418274, 0.2595900297164917, 0.14649052917957306, 0.736362099647522, 2.688585042953491, -2.3230457305908203, -0.03707259148359299], [-0.3852834105491638, 0.9287059307098389, 0.5488197207450867, -0.5911343097686768, 1.1028084754943848, 1.351158857345581, -1.4952248334884644, -0.08613375574350357, 0.8853294849395752, 0.13845953345298767, 0.07812800258398056, 1.602125883102417, -0.3565477430820465, -0.09376271069049835, 0.5831667184829712, -1.285130262374878], [0.2993335723876953, -0.6722191572189331, 1.2027968168258667, 0.12643472850322723, -1.391068935394287, 2.0921390056610107, -0.41667118668556213, -0.9278083443641663, 0.9133402109146118, -0.4443890154361725, -0.9306362867355347, -0.182315394282341, -0.15010754764080048, 1.082855224609375, 0.2914249897003174, -0.5817404389381409], [0.6289581060409546, -0.8460964560508728, -0.43841779232025146, 0.20752263069152832, -0.7463093400001526, -1.1208933591842651, 0.31331053376197815, -0.19857510924339294, 0.48744842410087585, -0.9619563221931458, -0.21453392505645752, 0.4991854429244995, 1.0898830890655518, -0.7329736351966858, -1.1379672288894653, -1.416303277015686], [-1.2861847877502441, 0.058052342385053635, 0.8189430236816406, -0.45438551902770996, 1.2306979894638062, 1.0245211124420166, 0.9737606048583984, -0.2917996048927307, 1.7819414138793945, 1.5076987743377686, 1.049490213394165, 1.4656157493591309, -1.0655046701431274, 0.05751826986670494, -0.28106969594955444, -1.4951717853546143], [0.06982829421758652, 0.8436436653137207, -2.6007418632507324, 0.7121527791023254, -0.6613038182258606, 0.767387330532074, -0.887654185295105, 0.38605624437332153, -0.9981585741043091, -0.057273007929325104, 0.36383721232414246, -0.33454629778862, 1.2086849212646484, -0.8514679074287415, 1.672524094581604, -0.7132201194763184], [0.8721745014190674, 0.3315659165382385, 0.8330997228622437, -0.2040053904056549, -0.2294747680425644, -0.6339548230171204, -0.692834734916687, -0.39142245054244995, -0.4053466320037842, -0.22048072516918182, 0.14194242656230927, 0.7313022017478943, 0.18484023213386536, -2.1158056259155273, -1.526183843612671, 0.975866973400116], [-0.7375048398971558, 0.34129297733306885, 0.7373197078704834, -0.4859161376953125, 0.7471673488616943, -1.2918944358825684, 2.3812758922576904, 0.1888199746608734, -1.321340560913086, 0.3481503427028656, -0.06746464967727661, -2.2913167476654053, 2.3497650623321533, 0.9302200078964233, 0.4500560462474823, 1.1944767236709595], [0.1481478214263916, -0.31167376041412354, 1.0628845691680908, -0.663857102394104, -0.29188504815101624, -0.1546429544687271, 2.452476978302002, -2.3893260955810547, 0.8433363437652588, -1.7813305854797363, 0.86468905210495, 1.5767487287521362, 0.08078993856906891, -0.8230350613594055, -0.8719322681427002, -0.3618725836277008], [-0.9203747510910034, -1.2973334789276123, 0.3243095278739929, -0.4813374876976013, -0.23996974527835846, -1.4687293767929077, 0.8774304986000061, 0.6109337210655212, 1.3258512020111084, 1.391445279121399, 0.871029794216156, 0.24276475608348846, -0.11374689638614655, -1.3908939361572266, 0.38024991750717163, -0.36212533712387085], [0.9891305565834045, -0.9353022575378418, 0.40497538447380066, -0.8414228558540344, 0.6060794591903687, 0.8088871836662292, -1.19901442527771, 0.6606763005256653, 1.500872254371643, 0.1048358827829361, 0.5622133016586304, 1.251497745513916, 0.18219031393527985, -0.06762398779392242, -0.48294365406036377, -0.12828561663627625], [-0.4537413716316223, 0.12624190747737885, -1.228974461555481, 0.7638502717018127, -0.5807831287384033, -1.4857134819030762, 0.807003378868103, 0.7264586091041565, 0.1377248913049698, 0.03733489662408829, -0.14055560529232025, -1.2760696411132812, -0.1614595502614975, 1.4904539585113525, 0.8784578442573547, -1.4151949471852276e-05], [-0.7902208566665649, 1.7249658107757568, 1.460909366607666, 1.6186552047729492, 0.1936643421649933, 0.3456355929374695, 0.06706848740577698, 0.5695479512214661, -0.913330614566803, -0.8472815155982971, -0.6367875337600708, 0.2969701886177063, 1.1531381607055664, -0.8265749216079712, 0.5472242832183838, 0.6347817182540894], [1.7404998540878296, 1.0455454587936401, 0.2263709455728531, 1.029384970664978, -1.185464859008789, -0.03531605750322342, -0.6919371485710144, 1.6593196392059326, -1.3283530473709106, -0.9076527953147888, -0.6850077509880066, 0.2212328463792801, -0.9123932123184204, 0.013077789917588234, 0.2148662656545639, -0.9314834475517273], [0.6413435935974121, -1.3364017009735107, 0.913642168045044, -0.6887804865837097, 1.33746337890625, 0.01422648411244154, -0.019656339660286903, -0.3799389600753784, 0.23899757862091064, -0.9291253089904785, 0.40765202045440674, -0.34077975153923035, -0.5139511823654175, -0.47190433740615845, -0.4702962338924408, 1.142762303352356], [0.04388539493083954, -2.1977341175079346, -0.945619523525238, 1.695318341255188, -0.040673159062862396, 0.5642637014389038, -1.2674734592437744, -0.09247364103794098, 0.6176145076751709, 1.1457566022872925, 1.7352632284164429, 0.39342883229255676, 0.14689448475837708, 1.6097986698150635, 0.24557454884052277, -0.17430736124515533], [1.2877188920974731, -0.5281745195388794, -1.0142724514007568, -0.3788793087005615, -1.1445289850234985, -2.2257630825042725, -0.5835485458374023, -1.691839575767517, 1.2947185039520264, 1.5482014417648315, -0.1349859982728958, -0.8312556147575378, 0.5856690406799316, -0.444909006357193, 0.3001546859741211, -1.371085286140442], [-0.2505711317062378, -0.5676702857017517, 0.8331801295280457, -1.7473658323287964, 0.09959659725427628, 0.9331953525543213, 1.2643617391586304, -0.20138372480869293, -0.255024254322052, 0.4314090311527252, 0.4911743402481079, 0.2959538698196411, 0.329332172870636, -0.9425513744354248, 0.10086400806903839, -0.09137794375419617], [1.4663012027740479, 0.09816104173660278, -2.346007823944092, -0.962441623210907, -1.5909571647644043, -0.7304274439811707, 0.15909874439239502, -0.6179895401000977, 1.1159696578979492, -0.787135660648346, -0.43185967206954956, 0.5577873587608337, 1.3768563270568848, -1.255922794342041, 1.117679238319397, 0.8683500289916992], [-0.2661478817462921, -0.3630671501159668, -0.6345202326774597, 0.16510675847530365, -2.514920473098755, 0.21987579762935638, 0.8887497782707214, -0.5585604906082153, -0.01235290803015232, 0.04060812667012215, -1.0128470659255981, 0.55159991979599, 0.11519274115562439, -0.2732103168964386, 0.3600856363773346, 0.37314629554748535], [0.23116818070411682, 0.2936532199382782, -0.8193170428276062, -0.3738860487937927, -0.8623349666595459, -0.3817955255508423, -0.8486524224281311, -0.18125921487808228, 2.2688064575195312, -1.7708441019058228, -0.7543948888778687, 0.3773951828479767, 1.8661953210830688, 1.0493706464767456, -0.5173119306564331, 0.43237656354904175], [0.14534515142440796, -0.3761618733406067, -1.722910761833191, 0.343957781791687, -0.265287846326828, -0.5570301413536072, 0.19931212067604065, 0.7495797276496887, 0.38157856464385986, 0.151878222823143, -0.43151652812957764, -2.2451248168945312, -0.5290864706039429, 0.9784330129623413, -0.42881131172180176, -1.2697029113769531], [-1.201733112335205, -1.003463864326477, 0.14807003736495972, 1.9289557933807373, 0.9576544165611267, -0.9914398193359375, 0.618766188621521, 0.54118412733078, -0.24738004803657532, 0.13821405172348022, 0.8631260991096497, 0.25710606575012207, -0.4309600293636322, 0.4709619879722595, 0.10490682721138, 0.12296861410140991], [-0.3798135817050934, 0.456539511680603, -0.4134225845336914, 2.1279478073120117, 0.10693920403718948, 0.9554699063301086, 0.4967629611492157, 0.7512245178222656, -0.1540941447019577, -1.376795768737793, -0.20131653547286987, -0.5155193209648132, -0.9446777701377869, 0.4976307451725006, 1.4261149168014526, -1.5998737812042236], [-0.4962160289287567, -1.4663225412368774, 1.414919137954712, 0.04283658787608147, 1.6589887142181396, 0.652970552444458, -3.6358957290649414, -1.0789546966552734, 0.16666185855865479, 0.5212730765342712, -0.05240286886692047, -2.461914539337158, -0.4048074185848236, 0.7602876424789429, 1.433642029762268, -0.9438589811325073], [-0.11654401570558548, 1.0373225212097168, 0.9087467193603516, 0.7937292456626892, -0.6470456719398499, -0.9974110722541809, -1.802066445350647, -0.6087375283241272, 0.9117648005485535, -0.10052460432052612, 1.8963905572891235, 0.10110954195261002, -1.030332326889038, -1.5442798137664795, 0.270016610622406, 1.0683315992355347], [0.26427215337753296, 0.08037666231393814, -0.1494807004928589, 2.008554697036743, 0.851879358291626, -0.403490275144577, 0.37486082315444946, -0.8680236339569092, -0.14846304059028625, 1.2255282402038574, 0.4692016839981079, -0.16499298810958862, 0.2296454906463623, 0.054515913128852844, 0.4630379378795624, 0.1400698721408844], [-1.4674148559570312, -0.18575824797153473, 0.855180025100708, -1.1271812915802002, 0.43068838119506836, -0.26076745986938477, -1.281353235244751, -0.6409416198730469, 0.7009341716766357, -0.5737764239311218, 0.84290611743927, 1.8361321687698364, -2.127254009246826, -0.8351868391036987, -0.8876603245735168, -0.2662203311920166], [0.07409767061471939, 0.20518119633197784, -0.5806970596313477, -1.449303150177002, -0.5092164874076843, -0.24777357280254364, 0.7351910471916199, -0.9930914640426636, -0.16309775412082672, -0.14846423268318176, 0.6471735239028931, 2.2846992015838623, 1.11884605884552, 0.8238731622695923, -0.8973081111907959, 0.535220742225647], [-0.06740739196538925, 0.9823708534240723, -0.012435182929039001, 0.2654053270816803, 0.4179065525531769, 2.1112513542175293, -2.0221123695373535, -3.783360242843628, -0.9546812176704407, -0.7694025635719299, -2.3538668155670166, 0.1892833560705185, -1.4646331071853638, -0.99830561876297, -1.1807894706726074, -0.3338397145271301], [-1.2145649194717407, 0.02486223541200161, -0.3177279829978943, 2.0978844165802, 0.02681373804807663, 0.9442622661590576, 1.0415332317352295, 0.29028013348579407, -0.38306719064712524, -0.5165719389915466, -0.5720486640930176, -0.6995270252227783, -0.12037502974271774, -0.9883437752723694, -1.0976544618606567, 0.9208232164382935], [-1.5085326433181763, -0.4901864230632782, -0.5065883994102478, 2.5371768474578857, -1.0403289794921875, -0.7936264872550964, 0.5547252893447876, -0.7445487976074219, 1.199683666229248, 0.32853421568870544, -0.06332114338874817, 1.1095035076141357, -0.9882816672325134, -0.08427248150110245, -0.3178746700286865, -1.8106690645217896], [-0.06400050967931747, 0.6254827380180359, -1.9879207611083984, 0.5401389002799988, 0.7269148230552673, -0.420106440782547, 1.0978070497512817, -1.269649624824524, -0.5052011609077454, 1.5472502708435059, 1.5873216390609741, -0.22054706513881683, -0.9102280735969543, 1.6042338609695435, -1.0546634197235107, 0.47783979773521423], [-0.09675050526857376, -0.8290745615959167, 0.5341485738754272, 0.24615716934204102, -1.6207666397094727, -0.6756663918495178, -1.9001246690750122, -1.3048018217086792, -0.892692506313324, -0.1927059292793274, -1.403090476989746, 2.4051506519317627, -0.6916829347610474, 0.004924094770103693, -2.031506061553955, -0.3740135133266449], [-0.5203049778938293, -0.16872157156467438, -0.5646893382072449, 0.5479320883750916, 0.3893997073173523, -0.1054583266377449, -0.5640608072280884, -0.053430259227752686, -0.8897965550422668, -1.3116915225982666, 1.0394150018692017, 0.6003221273422241, 0.6035557389259338, 1.5578216314315796, -0.1448875069618225, -1.0704034566879272], [-1.1759999990463257, -0.042308710515499115, 1.5723985433578491, -0.8476079106330872, -0.42344799637794495, -1.0575531721115112, 0.08600135892629623, 1.4087772369384766, 2.0421555042266846, 0.04918854311108589, -0.8728472590446472, -0.856312096118927, 0.6103052496910095, -0.45368680357933044, -0.981312096118927, -2.4102485179901123], [0.4786953330039978, -1.043670415878296, 0.6632682085037231, -1.5035648345947266, -1.0186243057250977, 0.03878231719136238, 0.1388954073190689, -0.4783124029636383, -0.3781839907169342, -0.08895948529243469, -0.5238608121871948, 0.5283395051956177, 0.35279908776283264, 0.5401337146759033, 1.240864872932434, 1.8982598781585693], [-0.3597062826156616, -1.2586435079574585, -1.207769751548767, -1.0679529905319214, -2.014836549758911, -1.4762085676193237, -0.9089048504829407, 0.643683135509491, 0.8251939415931702, -1.0418589115142822, -1.1829180717468262, -0.049843400716781616, -1.1668130159378052, 1.0469402074813843, -0.06346901506185532, -0.16681720316410065], [-1.7544513940811157, -0.9645135998725891, -0.3145250380039215, 0.08336454629898071, 0.05307392030954361, -1.1799474954605103, 0.0017470759339630604, -0.7722172141075134, -0.056232303380966187, -0.5395185351371765, -0.4562157690525055, -0.3833140432834625, 0.501549243927002, -1.3864688873291016, 0.17384152114391327, 1.3240108489990234], [0.0393223874270916, 0.22656399011611938, 0.9610204100608826, 1.1328206062316895, -0.19925029575824738, -0.9206358194351196, -0.1610315889120102, -0.6725370287895203, -0.6405514478683472, -0.025021590292453766, 0.22047816216945648, -0.01447926927357912, -1.2365881204605103, 0.27166077494621277, 0.18295036256313324, -0.6324132084846497], [0.01922248676419258, 0.5173690319061279, -0.608652651309967, 0.42210182547569275, 0.1404048651456833, 0.05212593823671341, -2.3383593559265137, -0.4837743043899536, -0.7217063903808594, 0.8032975196838379, 0.18026110529899597, -1.6387746334075928, -0.39897337555885315, -1.0003098249435425, -0.7008534669876099, -0.11609212309122086], [0.8643497228622437, 0.17067982256412506, -0.5036020278930664, 1.0681190490722656, -1.8795948028564453, 0.6888510584831238, 0.35301241278648376, 0.8328481316566467, 1.5763883590698242, -0.9399192333221436, -1.0169726610183716, 0.06187625229358673, -0.5733895897865295, 0.6333543062210083, -1.1576741933822632, -0.3510206937789917], [0.018833229318261147, 0.2014303207397461, 1.5333802700042725, 0.5030067563056946, -0.1077006384730339, 0.7000550627708435, -0.10909951478242874, -0.05587676167488098, 1.125518560409546, 0.14840534329414368, 0.18147793412208557, -3.7417678833007812, -0.4272332787513733, 0.3111020028591156, -0.3112797439098358, 0.5650931000709534], [-1.4248132705688477, 1.8954135179519653, -0.11752098053693771, -0.8312615752220154, -0.2609535753726959, 0.8366956114768982, 0.3299366235733032, -0.35574355721473694, -0.2369079440832138, 2.0660672187805176, -0.9431426525115967, -0.1016567125916481, 1.2887530326843262, -1.9776779413223267, 0.9519980549812317, 0.03171107918024063], [-1.4975634813308716, 0.611338198184967, 0.669234573841095, -0.23595792055130005, -0.8514759540557861, 0.006355503108352423, 0.17280960083007812, 1.5669386386871338, 1.0875933170318604, -1.3903919458389282, 0.9380669593811035, -1.443196415901184, -0.41750043630599976, -0.3362485468387604, -0.9436092376708984, 0.7655092477798462], [-1.409433126449585, -0.2572942078113556, -0.013812166638672352, 0.4463295042514801, 0.4221092760562897, 0.2627871036529541, 1.6269252300262451, -0.17944909632205963, -0.698309063911438, 1.7098026275634766, 1.88456392288208, -0.37550467252731323, -0.46121537685394287, 1.3347556591033936, 1.1611427068710327, -0.8927966356277466], [0.6926891207695007, -0.9342475533485413, 1.6358646154403687, 0.11186949908733368, -0.1927986890077591, -0.44637495279312134, 0.39440006017684937, -0.716552734375, -0.7167235612869263, -2.192201614379883, 1.4460833072662354, -0.4122113585472107, 1.2524281740188599, 1.0866211652755737, 1.6107945442199707, 0.8159913420677185], [2.3271262645721436, -1.8945865631103516, -1.4920239448547363, 0.07201123237609863, 1.8343836069107056, -0.8390268683433533, 0.08941318839788437, -1.1012938022613525, 0.417933851480484, 0.5218793153762817, -2.0444533824920654, 0.5099270939826965, -0.24582771956920624, 0.7278149127960205, -0.5533500909805298, -0.12130436301231384], [0.7299577593803406, 0.2925228178501129, -0.7033482193946838, -0.16799835860729218, -0.34038403630256653, 0.5542559027671814, 0.38827189803123474, -1.5459028482437134, 1.0153003931045532, 1.9417357444763184, 1.3504585027694702, -1.9500850439071655, 0.9224194288253784, -0.4879498779773712, -1.0036312341690063, 0.3539402186870575], [-1.2700637578964233, 1.3115965127944946, 2.007880926132202, -1.0328665971755981, 2.623511791229248, -0.23132140934467316, -1.3859320878982544, 0.8164256811141968, -1.4622459411621094, 0.9612318277359009, 0.747159481048584, 2.468445301055908, -1.1136119365692139, -0.9773337244987488, -1.9708278179168701, 0.171357661485672], [0.7349387407302856, 1.0714268684387207, 1.3010374307632446, 1.1449865102767944, -0.3994594216346741, 0.13511928915977478, -1.2295535802841187, -0.317673921585083, 0.7937770485877991, 0.48854750394821167, 0.43126651644706726, 1.8071194887161255, 0.8628434538841248, -0.11560829728841782, -1.5174882411956787, -0.28802773356437683], [-0.09628576040267944, 0.20512275397777557, -0.8541321754455566, -0.093813456594944, -2.135745048522949, 0.9358698129653931, -1.021539330482483, 1.1794062852859497, 0.12603150308132172, 1.4916836023330688, 0.14377136528491974, 1.3187415599822998, -0.8062314987182617, -0.0918993279337883, 0.42147397994995117, -0.9281116127967834], [-1.332000970840454, 0.4799162745475769, -0.5313593745231628, -0.021150637418031693, 0.27546581625938416, -0.03451228886842728, 1.3006927967071533, 1.0427924394607544, 0.8127074837684631, -0.4332570433616638, -0.8812570571899414, 2.127558946609497, -1.6386946439743042, -0.09395550191402435, 0.6060341000556946, -0.5102307796478271]]], [[[0.6190990209579468, 1.9920412302017212, -0.37778717279434204, 0.29216378927230835, 0.1668761670589447, 0.12762761116027832, -0.43654167652130127, 2.3042173385620117, 1.9732733964920044, -2.1559526920318604, 0.8913894295692444, 1.1733285188674927, -0.2599791884422302, -0.22008730471134186, 0.8822119235992432, 0.9168374538421631], [-1.0393019914627075, -1.2391433715820312, -1.30681312084198, -0.45719242095947266, 1.6220813989639282, -0.14954409003257751, 0.9064602255821228, -0.755266547203064, 1.8307418823242188, -1.2636380195617676, 0.3938800096511841, -0.3330819606781006, -0.21666304767131805, -0.8854506611824036, -1.5455647706985474, -0.4208565056324005], [-1.3543275594711304, -1.5248138904571533, 0.7258330583572388, 0.03757975250482559, 0.234918475151062, 1.2533714771270752, 0.7628413438796997, -1.1855264902114868, 0.33528491854667664, -1.203574538230896, -0.5009971857070923, 1.2733029127120972, 0.8811219334602356, -0.41430050134658813, 0.3053549826145172, 1.9754892587661743], [1.1182392835617065, -1.4271160364151, 2.398348808288574, 1.0702273845672607, -0.6077843308448792, -0.360038697719574, -1.0550214052200317, -0.840415358543396, 0.5946411490440369, 1.3052881956100464, -1.1964054107666016, -0.04702048748731613, -1.2906005382537842, 0.27616938948631287, -0.8945838212966919, 0.03550108149647713], [1.1817600727081299, -0.04878893867135048, -0.5574979186058044, -0.6495499610900879, 1.7768982648849487, 0.19753524661064148, 0.5110436677932739, 1.954078197479248, 0.5181072950363159, 0.34144607186317444, 0.27778276801109314, -0.23712056875228882, 0.7801878452301025, 0.563642680644989, -0.7541232705116272, -0.8762988448143005], [0.3222193717956543, 0.29640835523605347, 0.6723923683166504, 0.6187875866889954, 1.6058064699172974, -0.06358521431684494, -0.3814387917518616, 0.7930206060409546, -2.791080951690674, -1.1520153284072876, -0.6183295249938965, -0.2742953896522522, -1.6867831945419312, -0.2627868354320526, -0.11624985933303833, 1.5509016513824463], [0.3596038222312927, 0.04488638415932655, -0.6376012563705444, -0.2427293211221695, 0.5575271248817444, -0.420965313911438, -0.04426543042063713, -0.23970657587051392, -0.0009214035817421973, 1.1257250308990479, -0.5312110781669617, 0.22064127027988434, 1.5371721982955933, 0.11985556781291962, -0.12118688225746155, 0.2420751005411148], [0.5811243057250977, 1.4326506853103638, -0.6184731125831604, -1.8668583631515503, -0.1758638322353363, -0.9460121393203735, -1.2281060218811035, -1.2610852718353271, 1.2786805629730225, 1.574512004852295, -1.334851861000061, -0.8642780780792236, 1.7569819688796997, 0.9997614026069641, 0.4610117971897125, -0.19447851181030273], [1.2682805061340332, 0.580406665802002, 0.32927703857421875, -1.136993169784546, 1.2305686473846436, 0.22047647833824158, 0.43273985385894775, 0.405296266078949, 0.19132587313652039, -0.28530868887901306, 1.1924484968185425, 0.2677907943725586, -1.0924572944641113, 0.4791263937950134, -1.02870774269104, 0.6416240930557251], [-1.1279329061508179, -1.4726307392120361, -0.7194027304649353, 0.41902443766593933, -1.8606268167495728, 0.17724791169166565, -0.10033434629440308, -0.12081541866064072, -1.355832576751709, -0.1466999053955078, -0.7987380027770996, 0.2906322777271271, 0.7221776247024536, -1.7386833429336548, 0.16765452921390533, 1.3707438707351685], [0.6170694828033447, 1.0971006155014038, 0.8491906523704529, -0.9853155016899109, 1.2893571853637695, -0.682174563407898, 1.7493807077407837, -1.1528480052947998, -0.4689994752407074, -0.736100971698761, -0.20310576260089874, 0.3784581422805786, 0.5283339619636536, 2.0072274208068848, 0.4743400812149048, 0.7211856842041016], [-0.27042993903160095, 2.0462920665740967, 0.15434034168720245, -1.6306037902832031, -2.0459511280059814, -1.7855379581451416, 0.7323380708694458, 0.8031576871871948, 0.0484405979514122, 0.8415586948394775, 1.0969682931900024, 1.8937147855758667, -0.7557432055473328, 1.2315709590911865, -0.9356893301010132, -0.6557797789573669], [0.46284475922584534, -0.1310533881187439, -0.5140691995620728, -0.8028002977371216, -0.13013789057731628, -0.573064923286438, -1.0146362781524658, -0.21327263116836548, 1.5499719381332397, -0.4818371534347534, -0.6019563674926758, -0.7511541247367859, 0.8105642795562744, 0.5819318294525146, 0.5246874690055847, 0.5217199325561523], [1.7804425954818726, -0.02085530199110508, 0.9959266781806946, -0.684906542301178, 0.20600374042987823, 0.944225013256073, -2.3985507488250732, -0.9250736832618713, 2.324781894683838, -1.3635491132736206, 0.15096601843833923, -1.136606216430664, 1.3968263864517212, 0.02029619738459587, 0.0984959602355957, -1.6153901815414429], [0.6312296390533447, -0.4967539310455322, -2.1106314659118652, 0.25592896342277527, -0.9931660294532776, 1.4771950244903564, -0.41338109970092773, -0.4009163975715637, -0.04219254478812218, -2.353734016418457, -0.1677645742893219, 2.25486159324646, -1.779897689819336, 1.834997296333313, 0.4160751402378082, 1.3346819877624512], [-0.3425859808921814, 0.4468996524810791, -1.498034119606018, -0.003911035601049662, -1.1479507684707642, -0.6188424825668335, 0.757793128490448, -2.3290839195251465, -1.4684648513793945, 0.4621680676937103, -0.6469239592552185, 0.7917042374610901, -0.9190731644630432, 0.7323545217514038, 0.09851642698049545, -0.2265062928199768], [0.2096564620733261, -0.017859065905213356, -1.6983574628829956, -1.2822346687316895, -0.1456238180398941, -1.623079776763916, -1.7206530570983887, 0.595361053943634, -0.883781909942627, 0.5665685534477234, -0.036862682551145554, -1.109933853149414, -1.323495864868164, -0.2769964635372162, -0.1948099136352539, 0.6580731272697449], [-1.668236494064331, -1.4783587455749512, -0.5540177226066589, 0.508586585521698, -0.8218719959259033, 1.102102518081665, -0.10864275693893433, -0.05351797863841057, 0.024871300905942917, -1.4191595315933228, 0.5678501129150391, -1.8492214679718018, 0.061928506940603256, -1.7830913066864014, 0.2951427102088928, 0.9884946942329407], [-2.2468631267547607, 1.3038020133972168, -1.1479032039642334, -0.3298123776912689, -0.050153154879808426, -0.952431321144104, -0.9490115642547607, -1.5304913520812988, 1.156173825263977, 0.9593681693077087, -1.6783602237701416, -0.26332882046699524, 0.9872110486030579, -1.1627814769744873, -0.06367728859186172, 1.8555020093917847], [-0.10325895994901657, 1.736761212348938, 0.9991523027420044, -0.20621493458747864, -2.190840244293213, -0.6235980987548828, -0.8177523612976074, 1.4682247638702393, -0.9775723218917847, 0.3435048460960388, -0.21510328352451324, -0.042823802679777145, -0.17382070422172546, -1.0077989101409912, -2.4788618087768555, 1.2959545850753784], [1.2242227792739868, 0.35536307096481323, 1.1146994829177856, 0.694932222366333, 0.41869762539863586, -0.00783401820808649, 0.7036008238792419, -0.24245880544185638, -0.498292475938797, -0.8448150753974915, 0.06654734164476395, -0.7793020606040955, 1.5040152072906494, -0.9588861465454102, -0.48018598556518555, -1.2459338903427124], [-1.3005813360214233, -0.27008679509162903, -2.2913224697113037, -0.04158184677362442, -0.5291565656661987, 0.9857859015464783, -0.28076833486557007, 0.7984365820884705, -0.1494196504354477, -1.3867658376693726, 3.1333656311035156, 0.9365652203559875, 0.10846102982759476, -0.6724570989608765, -0.7022879719734192, 0.8595026135444641], [0.10297907888889313, -0.4145515561103821, -0.10008341819047928, -1.344032645225525, -0.19952751696109772, -0.6276336908340454, 1.229594111442566, -0.3767021894454956, -0.06868677586317062, 0.9065285921096802, 2.4437851905822754, 0.2696952819824219, 0.2043990045785904, 0.35340577363967896, 0.1544719636440277, 0.07185612618923187], [-0.9734488129615784, 0.6235488057136536, -0.3262675702571869, 2.056960344314575, -0.1526864916086197, -1.4809678792953491, -0.5056832432746887, -2.630661725997925, 1.428733229637146, -0.20467273890972137, 0.23164433240890503, 0.1368168294429779, 1.4862927198410034, -0.5610829591751099, -0.6824162006378174, 0.6926668882369995], [0.9755182862281799, 0.8171613216400146, 0.5965325236320496, -0.493087500333786, 0.5290699601173401, 0.5861620306968689, 0.8369585275650024, 0.8000129461288452, -0.8112790584564209, 1.247997760772705, -0.6078190207481384, 0.1711086928844452, -0.5348014235496521, 0.02465684525668621, 0.2959015667438507, 1.3986767530441284], [0.1485242396593094, -0.5366931557655334, -0.20669017732143402, 0.40935230255126953, -1.1000360250473022, 0.4646027684211731, -0.3912951648235321, 0.25604158639907837, 0.3003855347633362, -0.6398887634277344, -1.1619658470153809, 0.9060654044151306, 0.33254435658454895, -0.5429566502571106, 1.5065900087356567, 0.3881593942642212], [-1.4187684059143066, 0.6997607350349426, -0.31149765849113464, -2.1760640144348145, -0.1598881036043167, 0.35908347368240356, 0.018238823860883713, 0.5604909658432007, -0.22346606850624084, -0.09480618685483932, 1.1031162738800049, 0.02270880714058876, -1.6317853927612305, 0.6067107319831848, -1.280401587486267, -1.72317373752594], [-0.9457321166992188, 0.3530672788619995, -1.1116605997085571, 0.06576615571975708, -1.0675615072250366, -1.16609525680542, 0.617554247379303, 0.23671621084213257, 0.5745284557342529, 0.7931155562400818, -1.3159595727920532, -0.6526999473571777, -0.641884446144104, -1.8295419216156006, 0.11670979857444763, -1.3810274600982666], [0.8010039925575256, 1.2161074876785278, 1.0878959894180298, 0.3057481646537781, -0.5521078705787659, -0.3365093767642975, 1.2819960117340088, -0.9704239368438721, -0.48718127608299255, -1.1912627220153809, 1.6906346082687378, -1.1214148998260498, -1.1709693670272827, 1.0790095329284668, -0.04316128045320511, 0.21686196327209473], [0.3676638901233673, -0.3230561316013336, 0.9363071918487549, -0.942459225654602, 1.3942409753799438, -1.6197595596313477, -1.3011627197265625, -0.5354216694831848, -0.13482338190078735, -2.010803699493408, -0.6542514562606812, -0.4546181559562683, 0.8634675145149231, -1.64421546459198, 0.347688227891922, -0.0936896800994873], [-1.1317874193191528, 1.5694421529769897, -1.2446867227554321, -0.17590400576591492, -0.6331354379653931, 1.8582978248596191, 0.5814776420593262, 0.7623143792152405, 2.591601610183716, -1.806391954421997, -2.0607433319091797, 1.2392570972442627, 2.258324384689331, 1.383805274963379, 1.191310167312622, 0.16544584929943085], [0.46316081285476685, -0.41251635551452637, -0.7427520155906677, 1.5281623601913452, 0.13228431344032288, 0.19541247189044952, 0.8333896994590759, -1.4539649486541748, 0.018065195530653, 1.9945605993270874, -0.4726548194885254, 1.7714356184005737, 0.13817933201789856, 0.4678402245044708, 1.326958417892456, -0.02988392859697342], [0.9565868973731995, -0.6317659020423889, -0.08623799681663513, -0.09788951277732849, -0.33876875042915344, -0.5704273581504822, 1.1939949989318848, 0.4282805621623993, -1.2518445253372192, 1.409295678138733, 0.3197687268257141, 1.3599576950073242, 0.11582421511411667, 0.40971270203590393, -0.5276893377304077, 1.762243628501892], [0.6635997295379639, 0.3944971561431885, -0.4959293007850647, 1.018532156944275, -1.5893678665161133, -0.8080751895904541, 0.8122575879096985, 0.6017649173736572, 0.5690456628799438, -0.4919089376926422, -0.15587849915027618, -0.5817727446556091, -0.6073050498962402, -0.9157541394233704, -0.12364457547664642, -0.31710678339004517], [-0.577498197555542, -0.8375629782676697, 0.0928448960185051, -2.1805946826934814, 1.13409423828125, -1.3422503471374512, -1.2053312063217163, 1.3013293743133545, 0.12121222913265228, -1.5108973979949951, 1.3954923152923584, -0.7928781509399414, -0.179207906126976, -0.022518932819366455, 1.1219286918640137, -1.3592618703842163], [-1.936202049255371, -0.49008920788764954, 0.7838054299354553, 0.032797921448946, 1.3573253154754639, 1.9004584550857544, -0.48433467745780945, 0.4754544794559479, -0.06436040252447128, 1.8546298742294312, -1.3253298997879028, 0.6763583421707153, 0.24174249172210693, 0.3883466422557831, 0.8168814182281494, 0.6068406105041504], [-0.4792885184288025, -0.15785770118236542, -0.29363924264907837, 0.20962780714035034, -1.2846879959106445, 0.4590904414653778, 0.4353247284889221, -1.731354832649231, -0.7109532952308655, -1.3907665014266968, 0.27036723494529724, -1.2812436819076538, -0.8563088774681091, -0.42769432067871094, 0.5072962641716003, 0.6358882784843445], [0.8191593885421753, -0.09527871012687683, -3.1755666732788086, 1.063952088356018, 1.1162463426589966, 0.9863528609275818, -0.14318040013313293, 0.8866115212440491, -0.3499194085597992, 0.7710961699485779, 0.8507605195045471, 1.3710625171661377, 0.5840084552764893, 1.0121029615402222, -0.4222142994403839, 0.8248040676116943], [0.3404015302658081, 1.1019376516342163, -2.7120039463043213, 1.934295654296875, 1.6585906744003296, -0.8896370530128479, 0.7843953967094421, -1.0955404043197632, -0.7283797264099121, 2.0126841068267822, 0.6926205158233643, 1.0040442943572998, 0.9336320161819458, -0.09111388027667999, -1.302432656288147, -0.1390564739704132], [-1.594935655593872, 0.6003852486610413, -1.1969237327575684, -1.2468178272247314, -0.05752493813633919, 0.6017515063285828, -0.7637234330177307, -0.9639463424682617, -1.5180038213729858, -0.33325135707855225, 1.1549489498138428, -0.8693196177482605, 0.34118470549583435, -1.2923407554626465, 0.8695271611213684, 0.39695167541503906], [-0.47497430443763733, -0.3794851005077362, 0.26640892028808594, -0.5731312036514282, 0.5078534483909607, -0.1576865017414093, -1.0305067300796509, -0.6617370843887329, 1.125755786895752, -0.847674548625946, -0.511663556098938, 1.005067229270935, 0.8051034808158875, 0.31442081928253174, 1.0845727920532227, 0.9304544925689697], [0.8922170400619507, 0.8162976503372192, -0.05520941689610481, 0.3567754626274109, 1.8281307220458984, 0.3465166389942169, -0.7474405169487, -0.4105507433414459, -0.32741233706474304, -2.3207027912139893, 0.3500398099422455, 0.5872087478637695, 0.24204115569591522, 0.059030111879110336, -1.515554666519165, -0.028762567788362503], [0.27310314774513245, -2.3345229625701904, 0.6645947098731995, 1.8338247537612915, -0.9912219047546387, -0.8726678490638733, 0.15803828835487366, -1.0107167959213257, -0.9366841912269592, 0.0881868302822113, 1.2322351932525635, -1.271202802658081, 0.841129720211029, 1.648465871810913, 1.9073903560638428, 2.024725914001465], [1.4316792488098145, -0.5615046620368958, 0.02830411307513714, 0.04277822747826576, -0.9363431334495544, 1.2874023914337158, 0.4156029224395752, -0.41340190172195435, 1.6102265119552612, 0.5352993607521057, -1.9822535514831543, 0.12376821786165237, -1.2885702848434448, 2.966073751449585, -1.3656455278396606, -1.3209927082061768], [-0.6379877328872681, -0.9488415718078613, -0.6267486810684204, -1.0637463331222534, -1.6080554723739624, 1.0678776502609253, -2.452925205230713, 1.8306864500045776, 0.3404850959777832, -1.6727197170257568, -0.2603091895580292, 1.423911690711975, -0.9395245909690857, -0.07221334427595139, -0.015668068081140518, 0.6086104512214661], [0.46443846821784973, 1.296255111694336, 1.010135531425476, 0.29904085397720337, 1.9308547973632812, -1.0743223428726196, 1.7187081575393677, -1.2464739084243774, -0.5356380343437195, 0.6483258008956909, 0.7022829651832581, -1.3870387077331543, 0.6440395712852478, 0.6510958671569824, 1.544079065322876, -0.31440526247024536], [-1.7117695808410645, 0.5991445779800415, -1.0684458017349243, 0.15920017659664154, 0.15569819509983063, 0.8363324999809265, -2.715374708175659, 0.03235631808638573, 1.1044107675552368, 0.8501959443092346, 0.32467320561408997, -1.3586714267730713, -0.7928950190544128, 0.9379492998123169, 2.342841625213623, -0.1374935805797577], [0.8612241148948669, 1.582661747932434, 0.25711992383003235, 1.9591755867004395, -1.1806553602218628, -0.4380571246147156, 0.922879159450531, 1.7776122093200684, 0.7377429604530334, -1.124213457107544, 0.5540854930877686, -0.2796376943588257, 0.9066040515899658, -1.0936014652252197, -1.177649736404419, -1.0822557210922241], [-0.8645921945571899, 0.4242359399795532, -0.059462644159793854, 0.6549109220504761, 0.25220394134521484, 1.0418763160705566, 1.7828553915023804, 1.5426896810531616, 1.5038374662399292, -0.20335078239440918, -0.021959273144602776, -0.41987356543540955, 0.4940100908279419, 1.1189119815826416, 0.3953976631164551, 0.2987998425960541], [-0.6693188548088074, -1.9388067722320557, 0.9347158074378967, -1.061036229133606, 1.0915230512619019, -0.40498021245002747, 0.06494230777025223, 0.3794907033443451, 0.2503241300582886, 1.068556785583496, -0.14948110282421112, 1.2096141576766968, 0.1791123002767563, -0.9451075792312622, 0.2896168828010559, 1.4000569581985474], [-0.5736224055290222, -1.5792092084884644, 1.0966660976409912, 0.2805531919002533, -0.73978590965271, -0.21742427349090576, 1.3393243551254272, 0.35415101051330566, 1.156150221824646, -1.4079930782318115, 0.7958744764328003, 1.012331485748291, 0.29702165722846985, 0.6079814434051514, -2.1447930335998535, -0.7603641748428345], [0.9077106714248657, -0.2892133295536041, 1.2231398820877075, 1.1322805881500244, 0.44949665665626526, 0.4025796353816986, -2.7556984424591064, -0.24634620547294617, 0.0015755571657791734, 1.77730131149292, 2.0939338207244873, 2.1180694103240967, -0.7179867625236511, -0.15873564779758453, -0.49245980381965637, 1.7219996452331543], [0.7951585650444031, -1.239044189453125, -0.015535556711256504, -0.36715275049209595, 2.1910436153411865, -0.21776391565799713, -0.8909707069396973, 1.101399302482605, 1.0317611694335938, 1.2829787731170654, -0.6068064570426941, 1.1290310621261597, -0.1258113831281662, -0.4623081386089325, -0.8362492322921753, 2.1033451557159424], [0.8097897171974182, -0.85097336769104, 1.1210435628890991, -0.978115975856781, -0.22932285070419312, 0.09498608857393265, -2.1089961528778076, -0.20590992271900177, 0.17825376987457275, 1.7161858081817627, 0.21739032864570618, 0.45705321431159973, -0.38246795535087585, -0.7960654497146606, -0.6367489695549011, -1.0408488512039185], [0.04639506712555885, -1.84507417678833, 1.8774672746658325, 0.8060832619667053, -0.8935285806655884, 1.3817299604415894, 0.3147563636302948, 0.07314928621053696, -0.878368079662323, 0.44678276777267456, -0.15109729766845703, -1.6626060009002686, 2.5204174518585205, -0.6611849069595337, -0.6117845177650452, -0.7128961682319641], [-0.9354702830314636, 0.642721951007843, 0.7309648990631104, -0.3796626627445221, 0.7228525280952454, -1.2344787120819092, -1.0534640550613403, -0.7246179580688477, 1.06553316116333, -0.48961883783340454, 1.9643189907073975, 0.8020994663238525, -0.18923254311084747, -1.0230571031570435, 1.103614091873169, 1.1704505681991577], [0.4219421446323395, -0.8964350819587708, 1.7650705575942993, 0.09564962983131409, 0.6644282341003418, 0.271793931722641, 0.6370952129364014, -1.9446027278900146, 1.2256786823272705, -2.2075846195220947, -0.6017493009567261, -2.0265958309173584, 0.085381880402565, -1.2574217319488525, -1.6470404863357544, 0.047184523195028305], [0.37008610367774963, 1.3115712404251099, 1.3411386013031006, 1.018365502357483, -0.27305057644844055, 1.0186183452606201, -0.25619664788246155, -1.2257205247879028, 0.2074006050825119, -0.20966081321239471, 0.7752047777175903, 0.7641274929046631, -0.9106703400611877, -2.0761947631835938, -0.8150126934051514, -0.22513633966445923], [2.334848642349243, -0.1672413945198059, 0.008955596014857292, 0.41724640130996704, 1.2092701196670532, 0.47806844115257263, 1.3161097764968872, 1.2922098636627197, -1.6415592432022095, -2.128138780593872, -0.48454225063323975, 1.031287670135498, -1.8656624555587769, 1.1108144521713257, 1.2377972602844238, 0.2973823547363281], [-0.3407678008079529, -0.8554630279541016, 0.10048649460077286, 1.143139123916626, 1.01083505153656, 0.3868542015552521, 0.20192426443099976, -1.4491302967071533, -2.3125345706939697, -0.482179194688797, 1.0033204555511475, 0.6841670870780945, 0.17750237882137299, 0.03961643949151039, 0.2134745568037033, 0.029422270134091377], [0.17981112003326416, -0.40437090396881104, 0.06544474512338638, -0.356050580739975, -0.043835949152708054, 0.11131154000759125, 2.053926706314087, -0.7171054482460022, -1.874923825263977, 1.3483662605285645, 0.45887017250061035, 0.3632042109966278, -1.333936333656311, 1.2064754962921143, -0.2635047137737274, -0.8091459274291992], [0.2820219099521637, 1.4502507448196411, 0.3206147253513336, 0.25758183002471924, 0.6729072332382202, 1.094065546989441, 1.476183295249939, -0.6554365754127502, -0.37909919023513794, 0.335053950548172, -0.6300948262214661, 0.5908803939819336, 0.1695491522550583, -1.2201058864593506, 0.38761192560195923, 1.5610064268112183], [-0.45439866185188293, -0.0697987750172615, -0.4518868625164032, -0.3065829277038574, 0.36917391419410706, -1.2691640853881836, 1.053580403327942, 1.258355975151062, 0.32645976543426514, 1.0928901433944702, 0.4091610014438629, -0.5685818195343018, 1.8490631580352783, 0.7560557126998901, -0.13411633670330048, -0.7350700497627258], [1.0927790403366089, -1.033492088317871, -0.7091169357299805, 0.4204461872577667, -0.5173595547676086, -1.1870640516281128, 0.579258382320404, 0.1512063592672348, 1.7513011693954468, 1.1084781885147095, -0.43840155005455017, -0.25404980778694153, 1.4260183572769165, 0.11546052247285843, -0.3263479173183441, 0.14104889333248138], [1.0035767555236816, 0.9558037519454956, 1.0362765789031982, 1.369507908821106, 0.26647046208381653, -0.046417418867349625, 3.0503313541412354, -1.8001669645309448, 0.38424015045166016, -0.7417163848876953, 1.3405903577804565, -1.7941993474960327, 1.2576549053192139, -0.3531121611595154, 1.1156597137451172, 1.0778324604034424], [0.5307190418243408, -0.5460835099220276, 0.0802682489156723, -0.02225106954574585, 0.5354295969009399, -0.8636881113052368, -1.4766939878463745, -0.34913402795791626, 1.0564944744110107, -0.11332465708255768, 0.5485967993736267, -0.2614850103855133, 0.7399646639823914, 0.22280825674533844, 0.6175712943077087, 0.31755557656288147], [-0.421925812959671, 0.2092219442129135, -0.5730165839195251, -0.2855241894721985, 0.1946217268705368, 1.0316996574401855, -0.920680820941925, -1.0910570621490479, 0.14191001653671265, -0.9297279119491577, -0.06737162917852402, -2.2394981384277344, -0.5956242680549622, 1.3188722133636475, -1.3326138257980347, 0.8234695792198181], [1.4679276943206787, 0.22307969629764557, -0.38901227712631226, 0.20793890953063965, -0.03105008602142334, -2.5757038593292236, 0.2572988271713257, -1.037252426147461, 1.26801335811615, 0.91374671459198, -0.4163517951965332, 1.8263046741485596, 0.5058879256248474, -2.153956651687622, -0.4395507872104645, 0.017398511990904808], [1.4489659070968628, -0.055246125906705856, -0.6966588497161865, -0.30402788519859314, -0.17750674486160278, 0.0661674216389656, 1.1975977420806885, 0.06727015227079391, 0.018530113622546196, 1.5054771900177002, 0.08738942444324493, 1.0397300720214844, 0.7810977697372437, 0.2104569375514984, -0.06465242803096771, 0.06661156564950943], [-1.7728866338729858, 1.3693238496780396, -1.930554986000061, -0.7738750576972961, 0.5257774591445923, 1.239470362663269, 3.3782918453216553, -0.5399970412254333, 0.6594095826148987, 0.3930071294307709, -0.4002932906150818, 0.6311060786247253, -0.5345761775970459, -0.6941938996315002, 1.3252910375595093, 0.13194827735424042], [-0.6327157020568848, -1.0345205068588257, 1.2646311521530151, 0.811555027961731, -0.543825089931488, -1.9759172201156616, 0.4691888689994812, -0.8092067241668701, 0.37973856925964355, -0.00971910823136568, 0.3023473620414734, -0.032170895487070084, 0.4711425304412842, -1.3352357149124146, -0.8878031373023987, 0.09098093956708908], [-0.3128436207771301, 0.859779953956604, 1.077890157699585, 2.303255081176758, -0.03700416535139084, 0.10205733776092529, 0.8343076705932617, 1.1568495035171509, 0.7238041162490845, -0.6313753724098206, 0.6061670184135437, -0.7886480689048767, 0.5166542530059814, -0.5086916089057922, 0.253968209028244, -0.05418112128973007], [-0.5441164970397949, 0.5619909167289734, -2.5976290702819824, 1.2598928213119507, -0.02092720754444599, 1.6258140802383423, -1.0032718181610107, -0.9697862863540649, -1.5572788715362549, -1.1597669124603271, -0.4286015033721924, 0.14198395609855652, -0.3968472480773926, -0.21015413105487823, -0.7751386165618896, -0.9391518831253052], [1.3561077117919922, 0.40791869163513184, 0.6719008684158325, -1.5045794248580933, -0.10549821704626083, -0.9960025548934937, -0.314315527677536, 1.1102620363235474, -0.798187792301178, -2.231152057647705, -0.03382488712668419, 1.7909045219421387, -0.9498957395553589, 0.6594157814979553, -0.21881571412086487, -1.4124644994735718], [0.18177786469459534, 0.02409978397190571, 0.4868983030319214, 0.8082239627838135, -0.19556665420532227, -1.054168462753296, 0.3876035809516907, 0.7278494238853455, 0.6577284932136536, 0.5308020710945129, -0.19075623154640198, -0.598858118057251, 0.20668673515319824, 0.6245039701461792, -0.31262004375457764, 0.15585148334503174], [0.39122217893600464, -0.5289657115936279, 0.773133397102356, -0.12788540124893188, 1.6476582288742065, 1.0158915519714355, -1.2030020952224731, -1.3892343044281006, -2.2886102199554443, 2.050433397293091, -2.9079477787017822, 1.3811832666397095, 0.533862292766571, -0.1699131429195404, -1.861748456954956, -1.049098014831543], [-1.0261224508285522, -1.0086846351623535, 0.8009606003761292, 0.7426342368125916, -0.20048663020133972, -0.9944292306900024, -0.1273893266916275, 1.0885125398635864, -1.6010886430740356, 0.18949376046657562, 0.902491569519043, -0.7021741271018982, -0.03557080402970314, 0.16020332276821136, -1.466019630432129, -0.9432874917984009], [-2.5285696983337402, -2.156376600265503, -0.6029864549636841, -1.2067633867263794, -0.27908194065093994, 0.6561122536659241, -0.8351583480834961, -0.8254553079605103, 1.0007175207138062, -0.9460660815238953, -1.5859463214874268, 0.00533743342384696, 2.106971025466919, -0.14735329151153564, -0.3680342733860016, 1.0709434747695923], [-0.07387929409742355, 1.354354739189148, 0.5717483758926392, 1.0247644186019897, -0.29062366485595703, -1.1133140325546265, 0.9501490592956543, 0.7519890069961548, -2.3211684226989746, -0.39733269810676575, 0.20327922701835632, -0.6038413643836975, -0.8361278176307678, -0.44672536849975586, -1.8284603357315063, -0.6289345026016235], [-1.3565196990966797, -0.017398275434970856, 2.6452229022979736, -0.04330207034945488, 0.08896497637033463, 0.8834692239761353, 0.008378011174499989, -1.1841366291046143, -1.0392993688583374, 0.217946395277977, -0.06805925816297531, -0.3310982584953308, -0.8232784271240234, 0.8218733072280884, 0.3855641186237335, -1.0193467140197754], [0.37762436270713806, -0.33788853883743286, -1.0423001050949097, -0.48467496037483215, 0.28007757663726807, 0.11381019651889801, -0.9078229069709778, -0.27047422528266907, -0.1790010780096054, 0.005345586221665144, 1.1623485088348389, 0.6746706366539001, 1.3115094900131226, -2.528292417526245, -1.4015614986419678, -1.0656640529632568], [-1.9499691724777222, -2.394526243209839, -1.098890781402588, 1.1912058591842651, -0.854220449924469, -1.2624375820159912, -0.10068316012620926, 1.5894497632980347, 0.17761647701263428, 2.2956676483154297, 0.9157560467720032, 0.940658450126648, 0.6351718902587891, 0.3944130539894104, -1.9428237676620483, 0.14072652161121368], [1.267972469329834, -0.022532420232892036, 0.4567057490348816, -0.3475337028503418, -0.3024948239326477, -0.8753632307052612, -0.1570294201374054, 0.3343643844127655, 0.15392674505710602, -0.6593806743621826, 0.9933474659919739, 2.0145275592803955, 0.47383394837379456, -0.5125718712806702, -0.28727713227272034, 0.3501225411891937], [-2.1770458221435547, 0.43640923500061035, 0.19167077541351318, -0.7201350331306458, -0.1281791627407074, -0.8913299441337585, -1.158388376235962, 0.8175568580627441, -0.0758599042892456, -0.6879959106445312, 2.33896803855896, -0.4864920377731323, 0.04157990589737892, 0.006008394528180361, 0.6561594605445862, 0.7553781270980835], [-0.09797409921884537, -0.976187527179718, 0.9920290112495422, -0.45663249492645264, 0.20797449350357056, -0.528285562992096, 0.6408441066741943, -1.0108442306518555, -0.2189008891582489, 0.35439571738243103, -0.35985562205314636, 0.11204779148101807, -0.7586107850074768, 2.1398487091064453, -0.1658264696598053, -1.4704737663269043], [2.5682597160339355, 0.21582655608654022, -1.0950579643249512, 0.3444427251815796, 1.4453983306884766, 0.9598819613456726, 1.4108283519744873, 1.351441502571106, -1.5175645351409912, -0.107341468334198, -1.6915782690048218, -0.7364931702613831, -1.746345043182373, -0.3882129490375519, -1.2663017511367798, 0.6820553541183472], [-0.5497916340827942, -1.183701515197754, 0.5028421878814697, -0.34138333797454834, 0.43848663568496704, 0.20153747498989105, 1.0272444486618042, 0.8600150942802429, 0.9130594730377197, 0.8158469200134277, 0.24068886041641235, 1.0059317350387573, -0.11219225823879242, -0.6506595611572266, 0.4838504493236542, -0.3306254744529724], [-0.5677542686462402, -0.8417717814445496, 0.09483613818883896, -1.3650102615356445, 0.41477975249290466, -0.6529437899589539, 1.5521093606948853, -0.6993933916091919, -0.08995936810970306, 1.0353676080703735, -0.7886035442352295, -0.44220393896102905, 2.2832915782928467, -0.28666362166404724, 0.9352362751960754, 0.4093557596206665], [-0.15823481976985931, 1.043776512145996, -0.044130563735961914, 1.0000401735305786, 1.132124662399292, 1.6731748580932617, -1.8036996126174927, -2.1625125408172607, 0.7674569487571716, -1.4610614776611328, -0.9584659934043884, -0.2796846330165863, -0.19172190129756927, 0.5054864883422852, -0.0024964662734419107, 1.7038636207580566], [-0.459216445684433, 0.13512404263019562, 0.5582292675971985, -0.3388207256793976, -0.8270124197006226, 0.18332433700561523, -0.6278234720230103, 0.25106900930404663, 0.6251027584075928, -0.42218369245529175, 1.4111616611480713, 0.6371419429779053, -0.7829948663711548, -0.47989076375961304, -1.299065113067627, -1.4727160930633545], [0.2503730356693268, -0.2569248378276825, 0.9953745603561401, 0.6929482221603394, -1.5619882345199585, 0.544857382774353, 0.7847468852996826, -0.14488786458969116, 0.43476250767707825, -0.11859302222728729, 1.7392959594726562, 0.5974975228309631, 0.21674302220344543, 0.4333752691745758, -0.0327850878238678, 2.122497081756592], [0.09750895947217941, -0.0059356773272156715, 0.46159878373146057, -0.5792231559753418, 0.11142601817846298, 0.8125987648963928, 2.315829038619995, 0.4126667380332947, -1.0378599166870117, 0.0689752995967865, -0.390613853931427, 1.2378937005996704, 1.6917251348495483, 0.16666245460510254, 0.7360092401504517, -0.4935731887817383], [0.6075791716575623, 0.5511153936386108, -1.8518890142440796, -0.9014989733695984, -0.8822454810142517, -0.4570472538471222, -0.7099087238311768, 0.6398376822471619, 0.42589449882507324, -0.7789954543113708, -1.3354724645614624, 1.2749558687210083, 1.4872686862945557, -0.28775280714035034, -1.9799833297729492, 0.3163333535194397], [2.0437471866607666, 0.2069777399301529, 0.46257779002189636, 0.33013221621513367, 0.3517319858074188, -0.27617114782333374, -0.49301034212112427, 0.736109733581543, -1.4705495834350586, 0.044212665408849716, -2.22244930267334, 0.8359979391098022, 0.562656044960022, 0.7445355653762817, 0.6229268908500671, 0.08632680028676987], [-0.9919139742851257, -0.2959744930267334, -0.9434859156608582, 0.20393216609954834, 0.5836942195892334, -1.6696795225143433, 0.5331446528434753, -0.2940788269042969, 0.24355758726596832, 1.5047187805175781, 1.035139799118042, -0.24845649302005768, 1.0345885753631592, -0.8129381537437439, -0.5314380526542664, 2.412975788116455], [0.2707187831401825, -0.29392704367637634, 0.9246504902839661, -0.6399813294410706, 0.9661384224891663, -0.8020764589309692, -1.2830899953842163, -1.5742464065551758, -1.3949850797653198, -0.645080029964447, -0.7694464921951294, -1.8229515552520752, 1.7735859155654907, 0.8497406840324402, -1.6460621356964111, 1.1603484153747559], [-0.01126557495445013, 0.6110743880271912, -0.2129514366388321, 1.3568871021270752, 0.3375442624092102, 0.25361204147338867, -1.8187552690505981, -0.8205825090408325, 0.49097979068756104, 2.871605396270752, 0.7643994092941284, 1.0114943981170654, 0.4588148295879364, -1.136073112487793, -0.5847184062004089, -0.17186152935028076], [1.783551812171936, -0.9164493083953857, 0.657941997051239, -0.0030303276143968105, 0.5394150018692017, -0.09601441025733948, 1.2888293266296387, -0.2795168161392212, -0.8179894685745239, -0.8509021401405334, -2.264096736907959, -1.6561872959136963, -0.5251046419143677, 0.7601138353347778, -1.014844298362732, -0.7208995819091797], [-0.4254010021686554, 1.0170563459396362, 0.4552135467529297, -0.6780408620834351, -1.8936189413070679, -1.3866727352142334, 0.7690764665603638, 0.4189239740371704, 0.4760596454143524, 0.31210145354270935, -0.9860285520553589, 0.48474326729774475, 0.49003544449806213, -1.4808427095413208, -0.2953121066093445, -0.6996137499809265], [-0.5183232426643372, 0.4947586953639984, -0.48637068271636963, 0.17400318384170532, -0.0631481185555458, 0.2110118418931961, -0.9503359794616699, -0.09426266700029373, -0.09207937866449356, -0.9379962086677551, 0.3319803774356842, 1.2018368244171143, -1.485551118850708, -1.5511717796325684, -1.8208163976669312, 0.6335818767547607]]]], "dtype": "float32"}, {"__nd__": [[[[0.09885680675506592, -0.7686902284622192, 0.4758129417896271, -1.0095746517181396, 0.3734457790851593, 2.3798532485961914, 0.13885720074176788, -0.40838727355003357, 1.5927034616470337, -0.02053685113787651, -0.5977640151977539, 0.696997344493866, 2.8663315773010254, -0.09443413466215134, -1.4652739763259888, 0.6656098365783691], [-0.7776222229003906, -0.08734942227602005, 0.8843730688095093, -0.5012063384056091, -0.5631266832351685, -0.33739298582077026, 0.6056699752807617, -0.004128985106945038, -0.004705640021711588, 0.5305670499801636, -1.0823873281478882, 1.555364727973938, -0.7598653435707092, -1.9994616508483887, -0.9024202823638916, 0.18755964934825897], [0.611352264881134, -0.3315112292766571, 0.5797607898712158, -0.46828892827033997, -0.018500149250030518, 0.20742274820804596, 0.526335597038269, -1.4125889539718628, 1.4607042074203491, -0.5723496079444885, -0.818322479724884, 0.6128790378570557, -0.45679473876953125, 2.847849130630493, -1.1310101747512817, -0.9177968502044678], [-0.8459677696228027, -0.41573694348335266, -1.3585658073425293, 0.4681015908718109, -0.730241596698761, -0.20596952736377716, -0.5246561765670776, 0.4289975166320801, 0.24183562397956848, 0.3734804391860962, -2.6836462020874023, -1.9089635610580444, -0.36338967084884644, -0.9832794666290283, -0.5083566308021545, 0.09190548211336136], [0.8494448661804199, 0.1616073101758957, 0.5077755451202393, 0.9536358118057251, -0.7448828220367432, 1.140958309173584, -0.10547193139791489, -0.1518762707710266, 0.7418233156204224, -1.5935622453689575, 0.5230794548988342, -0.4676448702812195, -0.002399526769295335, -1.6923584938049316, 1.220990777015686, -3.0719430446624756], [1.0126582384109497, -0.24600067734718323, -0.10402257740497589, 0.5374569892883301, 0.1866045892238617, -0.03755415976047516, 0.08231504261493683, -0.16187232732772827, 1.9316091537475586, 1.2485769987106323, -0.7695960402488708, 0.19857336580753326, 0.5710021257400513, -0.3447815179824829, -1.0381920337677002, -0.3255959749221802], [0.15021945536136627, -1.6143200397491455, -1.5344462394714355, -0.2770954370498657, 1.1614811420440674, -0.9533610343933105, 2.0241172313690186, 0.17074541747570038, -0.5914846658706665, -0.030977917835116386, -0.08083784580230713, -2.0116524696350098, 0.5295067429542542, 0.4342467784881592, 0.911736786365509, -0.714364767074585], [-0.6869297027587891, -0.3360867500305176, 1.0024579763412476, 0.9467070698738098, 0.1127261221408844, -0.7335115075111389, 0.8185750246047974, -1.720227837562561, 1.001632571220398, 1.8106330633163452, 1.4783042669296265, -1.0734912157058716, -1.1003895998001099, 0.14058151841163635, -1.1850600242614746, -0.1874736249446869], [-0.8219991326332092, 0.3383534848690033, 0.7814988493919373, 0.3349807858467102, 0.662874698638916, 1.477424144744873, 0.1993488222360611, 0.741206705570221, 2.1021888256073, -0.4749213457107544, 2.820380449295044, 1.7938241958618164, -0.5292896628379822, 0.9045339822769165, -0.4594288468360901, -0.18706966936588287], [-0.7874674201011658, -0.13712559640407562, 1.2260658740997314, -0.3658718168735504, -0.34522512555122375, 0.20352701842784882, 0.695491373538971, 1.482752799987793, 0.5145388841629028, 0.7173761129379272, -0.1998426616191864, -0.13701534271240234, -0.23479025065898895, -0.8697887659072876, 0.5994142293930054, -1.8425031900405884], [0.15618368983268738, -0.6737575531005859, 0.8418619632720947, -1.116335391998291, 0.3898922801017761, 0.712091326713562, 0.9581887722015381, -1.71213698387146, -1.7933824062347412, -1.6203114986419678, 0.412649542093277, 0.974317729473114, -0.3530532419681549, 0.7313933372497559, -0.13361743092536926, -0.7602655291557312], [-0.5846898555755615, 0.12428167462348938, -1.4402369260787964, 0.23855462670326233, -0.7373809218406677, -0.9337352514266968, 1.887449860572815, 0.5844630002975464, 0.8983526229858398, 1.103685975074768, 0.23775948584079742, -0.4872620403766632, -0.2907276749610901, -1.9227266311645508, -1.7201446294784546, 1.1260267496109009], [-0.6477773189544678, 0.6615862846374512, 0.2000366449356079, -2.36794114112854, 0.4361206293106079, -3.805748224258423, -0.9839896559715271, 0.0006658948259428144, -1.024733543395996, -0.2250736504793167, -1.1899610757827759, -0.8336256742477417, 0.9616028070449829, 1.2824006080627441, -0.36864110827445984, 0.3236874043941498], [0.6188890337944031, -0.16909901797771454, 0.6323449611663818, 0.4908674657344818, -0.8930754065513611, 0.9136263728141785, 0.7838831543922424, 0.7387869954109192, -1.6019210815429688, -0.2450345754623413, 0.7537053823471069, 1.76094651222229, -0.13865026831626892, -1.7943311929702759, 0.7569079995155334, 0.7344298362731934], [1.5611591339111328, -0.5016456842422485, 0.27526992559432983, -1.0891698598861694, -0.3033437430858612, -1.7440004348754883, 0.09721415489912033, -0.21871350705623627, 0.1950458288192749, -0.8590006828308105, 0.3392140567302704, 1.23296320438385, -0.9244217276573181, 1.2061448097229004, 0.04232201725244522, -1.6045202016830444], [1.2330424785614014, 1.2361260652542114, 0.9435772895812988, 0.543995201587677, 1.5345920324325562, -1.4370778799057007, 0.4902651011943817, -0.5792527794837952, -1.7454228401184082, -1.0077544450759888, 1.7689101696014404, 1.1988396644592285, 1.046321988105774, -0.19393523037433624, -0.4979415535926819, 0.48324060440063477], [-0.9144372344017029, 0.15113277733325958, 0.4509970247745514, 1.294094204902649, 1.3985973596572876, -1.0031967163085938, 0.2474355697631836, 0.7652804851531982, -1.0302178859710693, -1.6661616563796997, -1.207511067390442, 0.4798823595046997, 0.23286442458629608, 0.6373563408851624, -0.3674663007259369, -0.9847732186317444], [0.12096215784549713, -0.829867959022522, 0.14114271104335785, 1.8147740364074707, -0.42281612753868103, -0.33129119873046875, -0.9626626968383789, -0.9792646765708923, 0.3882999122142792, 0.3645055890083313, -0.89810711145401, 1.04362154006958, 1.1916041374206543, 0.26003575325012207, 1.5668020248413086, 1.3652006387710571], [1.0131559371948242, -1.0910600423812866, -0.717644214630127, -0.5636574029922485, -0.5111389756202698, -0.6977792978286743, -0.9379425048828125, -0.09100328385829926, -2.287781238555908, -0.9821315407752991, -2.189847707748413, -1.1338390111923218, -0.4379008114337921, -0.10826915502548218, -0.7575275897979736, -0.6179754734039307], [-2.659959554672241, 0.5506571531295776, 0.4145202040672302, 0.10834812372922897, -0.9846239686012268, 1.1515337228775024, 0.8772263526916504, 0.403001606464386, 1.4178580045700073, 1.473713755607605, 0.06502291560173035, 1.1008366346359253, -1.0911071300506592, 0.3950606882572174, -0.6074989438056946, 0.8934504389762878], [-1.1941685676574707, -0.13752679526805878, -0.44444701075553894, -0.8968021869659424, -1.5049041509628296, -0.4553627669811249, 1.885197401046753, -0.6232954263687134, -0.6713100671768188, -1.0701096057891846, -0.2610006034374237, 0.1941961646080017, -1.9460216760635376, -2.150174140930176, -0.7774286270141602, -1.0488051176071167], [-1.050093650817871, -0.7694194316864014, -0.9340271949768066, 0.8366853594779968, -0.6801614165306091, -0.5067230463027954, -0.7724338173866272, -0.21904298663139343, 2.4649202823638916, -0.4555332362651825, 2.398221969604492, 1.2235703468322754, -0.07138136029243469, 1.2782891988754272, 1.2254282236099243, -0.5633304715156555], [-1.256202220916748, 0.658961296081543, 0.60451340675354, -0.6351873278617859, -0.6183362007141113, -0.1692628562450409, 1.703313946723938, -1.1392662525177002, 0.766520082950592, 1.8491339683532715, 0.18419836461544037, 0.23991188406944275, -0.6472548842430115, -2.613492727279663, 0.7201079726219177, -0.37433192133903503], [0.4880152940750122, -0.6292263269424438, 1.1030129194259644, 0.12354858964681625, 0.3063582479953766, 0.30943530797958374, -1.2789819240570068, 2.0521724224090576, -0.9141941070556641, 2.5889594554901123, -0.646661639213562, 0.858284056186676, 0.05828462913632393, 0.08596096932888031, -0.771839439868927, -0.7717419862747192], [0.8789775371551514, 0.2883567810058594, -1.4478574991226196, 0.34672513604164124, 0.757697343826294, -0.5395013689994812, -1.1726548671722412, 0.5718544125556946, -1.513888955116272, -1.2727545499801636, 1.0850179195404053, -0.4974837303161621, -0.8654763698577881, 1.5293447971343994, -1.8727613687515259, -0.15301746129989624], [0.043964967131614685, 0.3761623501777649, 0.4299320578575134, 0.769507884979248, -1.4452192783355713, 1.3366280794143677, -0.6536841988563538, -1.1239745616912842, 0.05865439772605896, 0.24599704146385193, 0.5182931423187256, 0.3616841733455658, 1.3404455184936523, 1.2646640539169312, 0.3906812071800232, 1.6853560209274292], [1.044704556465149, -0.3958463966846466, 0.4680217206478119, 1.3864105939865112, -0.3353554904460907, 0.9285228848457336, 0.6103321313858032, -1.3571925163269043, 1.152139663696289, -0.25703954696655273, -0.5913340449333191, 0.7671235799789429, -0.5258536338806152, 0.831392228603363, 1.277337670326233, -0.33162426948547363], [-0.6928640007972717, -1.2758768796920776, 1.290425181388855, -1.1499768495559692, -2.8034956455230713, -0.5254656076431274, -0.13736668229103088, -0.6160829067230225, -0.9828174710273743, -0.24888892471790314, -0.32850027084350586, 0.6015385389328003, -1.04081392288208, 0.5504366159439087, 0.3390769362449646, 0.4294681251049042], [0.8715279698371887, 0.3482343852519989, 0.20362380146980286, -0.41796842217445374, -1.1225435733795166, -1.1327897310256958, -0.3648746609687805, -0.00037318633985705674, -1.1591789722442627, 0.8543827533721924, 0.4709363579750061, 0.8261079788208008, -0.07584641873836517, 0.17297984659671783, 0.15571442246437073, -1.0470798015594482], [-0.04599079117178917, -0.44092071056365967, -1.0700410604476929, 0.4609338939189911, 0.48196354508399963, 0.18042950332164764, -0.48551836609840393, 0.1972060650587082, 0.16439282894134521, -0.18900416791439056, -0.8326107859611511, 0.9286265969276428, 1.1238389015197754, -1.7972229719161987, 0.3102210760116577, 0.436508446931839], [-0.4121722877025604, 0.10547387599945068, 0.41184940934181213, 2.0624208450317383, -0.6780178546905518, -1.2708649635314941, -1.6063177585601807, -1.7865921258926392, 0.7038947939872742, 0.8409071564674377, 1.2360975742340088, -0.510388195514679, -1.228675365447998, 0.4189049005508423, -0.5754023790359497, -0.44106894731521606], [-0.5779284834861755, -1.4136157035827637, -1.621277093887329, -1.0016326904296875, -0.2828172743320465, 0.8180058002471924, -0.5933224558830261, -1.2931047677993774, 0.823310911655426, 0.04815695062279701, 0.6754617691040039, 0.53653883934021, -1.1743748188018799, 1.568210244178772, -0.4985780715942383, 0.25546297430992126], [-0.4872198700904846, -0.1916533261537552, -0.7074363827705383, -0.3493274748325348, -0.9170193076133728, -0.443484902381897, 0.8046969771385193, -0.9767975807189941, 1.633241057395935, 0.970053493976593, 1.067176103591919, 0.3629600405693054, 0.7316402196884155, -0.5061887502670288, 0.16453050076961517, 0.6467412114143372], [-1.6765938997268677, -0.07246732711791992, -1.982215166091919, -0.3356369435787201, 0.22291937470436096, 1.6674537658691406, -0.6122564673423767, 1.8419808149337769, 0.46133118867874146, 0.2900073826313019, -0.2571370601654053, -1.3201990127563477, -0.9347559809684753, -0.041046638041734695, -0.06408008933067322, 0.512457013130188], [0.147862508893013, 0.44709184765815735, -0.5080260634422302, -1.1091440916061401, 1.0511482954025269, 0.019093820825219154, -0.787807047367096, 0.6789709329605103, 1.5722880363464355, 0.27971550822257996, -0.24424080550670624, 0.5223124027252197, -0.31699901819229126, 0.7648951411247253, -1.8086390495300293, 1.7710230350494385], [-0.5042325258255005, -0.13697603344917297, 0.8271538019180298, 0.3818703591823578, -0.6114845871925354, -0.3753276467323303, 1.838754415512085, -0.019537804648280144, -0.4165535867214203, 0.37300702929496765, 0.2087068259716034, -0.4119375944137573, -0.972166121006012, -0.573839545249939, 2.055053472518921, 1.5167925357818604], [0.3989494740962982, 0.04201444610953331, -0.691928505897522, 0.6374893188476562, 2.6450822353363037, -1.0465726852416992, 1.3874303102493286, -0.029301734641194344, 0.3045133948326111, -1.8045148849487305, 1.1175587177276611, 0.6894822120666504, 0.2859693169593811, -1.087786316871643, -1.2184664011001587, 0.004801476374268532], [-1.0908674001693726, -0.8216953277587891, 0.08242716640233994, -0.004804638680070639, 0.6612783074378967, 0.026215525344014168, -0.6647602915763855, 0.2627134323120117, -0.7037596702575684, 1.1941943168640137, -0.9838835597038269, -0.0017405200051143765, 0.06000614911317825, -0.28924697637557983, 0.30621445178985596, 1.0005844831466675], [1.757850170135498, 0.45935702323913574, -0.1048620194196701, -0.4805353879928589, 0.1296265721321106, 1.2149467468261719, 2.2921628952026367, -0.6774945855140686, -0.6633237600326538, 0.7375312447547913, -0.42864611744880676, 1.2000025510787964, 1.1894340515136719, -0.5855157971382141, 0.12166003882884979, -1.3067868947982788], [0.4953060448169708, -1.4501612186431885, 1.064618468284607, -0.720705509185791, 0.24756354093551636, -0.6812767386436462, -1.429207682609558, -0.8662673234939575, -1.329271674156189, -1.3922028541564941, 0.40637916326522827, 2.157151460647583, -0.7341169118881226, -1.4574464559555054, -0.012919901870191097, -0.37254542112350464], [-0.05365024134516716, 0.7185850739479065, -0.421877920627594, -0.5060009360313416, 0.1878419667482376, -2.4476747512817383, 1.0815142393112183, -0.469023197889328, -1.1597014665603638, 1.2197072505950928, -0.5719502568244934, -0.6438673138618469, 0.054198626428842545, 0.7936153411865234, 1.2379378080368042, -0.5187895894050598], [1.3433732986450195, 0.7025353908538818, -0.5111188888549805, 1.5891729593276978, -0.6886721849441528, -0.9289019703865051, -0.35838040709495544, -0.365324467420578, -1.1856861114501953, -0.8256915807723999, -0.8725287318229675, 1.1579214334487915, -0.9380178451538086, 0.11956379562616348, -0.8092793822288513, 0.5047311186790466], [-0.46740517020225525, 0.4278809428215027, -1.1948310136795044, 1.7367154359817505, 0.48865893483161926, -0.8513844609260559, 0.745778501033783, 0.8863475918769836, 1.3382411003112793, 0.8344062566757202, -0.24437746405601501, 0.7765719890594482, 0.10107246041297913, 0.44318827986717224, -0.920102059841156, 2.5344274044036865], [-0.8536836504936218, 0.9667221903800964, -0.46568140387535095, 0.6064530611038208, -1.0410398244857788, 1.0401270389556885, -0.27334293723106384, -1.1785484552383423, -0.06226261332631111, -0.8430889248847961, -0.9349086880683899, 0.9462332129478455, 0.16824990510940552, 1.4523389339447021, -0.24582161009311676, 0.43269217014312744], [-0.347879558801651, 0.3100513815879822, 0.2714212238788605, 1.5092180967330933, -0.7166181206703186, -0.6782811880111694, 0.7360895276069641, 0.26103129982948303, 0.6307172179222107, 0.2776421308517456, 1.0766161680221558, -2.1337571144104004, -0.3637026846408844, -0.13383880257606506, -0.5283076763153076, -1.0137137174606323], [0.6541566848754883, -0.8995127081871033, -0.47901350259780884, 0.42467641830444336, -0.4452195465564728, 1.1475886106491089, -0.6319096088409424, -0.9647387266159058, 1.3296970129013062, -0.015581639483571053, 0.4385074973106384, -0.3283191919326782, -0.006108012516051531, 1.6447871923446655, 0.8400260210037231, 0.8691381812095642], [2.1412999629974365, -0.255347341299057, 0.49290230870246887, 1.1872234344482422, 0.12388955056667328, 0.613832414150238, -0.9138479232788086, 1.1725425720214844, -0.9671351909637451, 0.014943096786737442, 0.2785399258136749, 0.6116301417350769, -1.6455984115600586, -1.5444414615631104, -1.137154459953308, 0.2607991695404053], [-0.11857222765684128, 1.3812227249145508, -0.05544469505548477, 0.7330273389816284, -0.8289392590522766, 1.5254732370376587, -1.8913509845733643, -1.69515061378479, -0.25867682695388794, -1.206388235092163, -0.030016597360372543, -0.5887773633003235, -0.396809458732605, -0.6453407406806946, -0.4229198694229126, 0.3207465708255768], [1.8743394613265991, 1.3812589645385742, 0.5168241858482361, 0.6010940074920654, -0.6945822834968567, -0.30982133746147156, -0.802506685256958, -1.7619879245758057, 0.7295357584953308, -0.6723343133926392, 0.40093305706977844, 0.6955093741416931, -1.8771995306015015, 0.21163462102413177, 0.134893998503685, -0.22874800860881805], [1.0318944454193115, 0.36006197333335876, -0.36669784784317017, -0.15679046511650085, -0.9063026905059814, 0.6687875390052795, -0.9668130874633789, 2.046243667602539, -0.48564866185188293, -0.7417737245559692, 0.03482076898217201, 0.09624278545379639, 0.6034490466117859, 0.4168494939804077, -0.10226717591285706, -0.020153112709522247], [-0.36003363132476807, -0.4453093409538269, -1.7488864660263062, 0.7023872137069702, -1.1345752477645874, -2.1912245750427246, 2.8359625339508057, -0.6470020413398743, 0.3596061170101166, -0.5910618901252747, 0.814555823802948, 0.36564773321151733, -0.6242470145225525, 1.257874846458435, -0.6715710759162903, -0.14903484284877777], [-1.5260530710220337, 1.9535592794418335, -1.0399686098098755, 1.2509080171585083, -0.20365536212921143, -0.08726608753204346, -1.54017174243927, 1.1774847507476807, 0.42215996980667114, 0.7660241723060608, 0.30297762155532837, 1.0316659212112427, -0.4745054543018341, 0.20227502286434174, -0.6504113674163818, -2.420184373855591], [-0.014646667055785656, 1.1880892515182495, -0.2921546995639801, 1.0419437885284424, 3.1424288749694824, 0.7086310386657715, 0.08942701667547226, -1.6595702171325684, 0.6932240724563599, -1.110129475593567, 0.31678110361099243, 0.752113401889801, -0.6873395442962646, -0.8470608592033386, 0.8008924126625061, 0.22718322277069092], [-0.15024526417255402, -1.3633687496185303, 0.22619274258613586, 0.6975641846656799, -0.9221338629722595, 0.4760168194770813, -0.290233314037323, -0.7102282047271729, -1.307232141494751, 0.5646468997001648, 1.4875274896621704, -0.14995305240154266, -1.1663758754730225, -0.08244787901639938, 0.6246080994606018, -0.8394992351531982], [-0.5690097212791443, -0.8419121503829956, 1.8399968147277832, 0.09277347475290298, 0.6463606953620911, -0.568054735660553, 1.9004456996917725, 0.5788648128509521, 0.5176041126251221, -0.4499644339084625, -1.4829586744308472, -0.8119346499443054, -0.2998819947242737, 0.6342311501502991, -0.4054219424724579, -0.8216274976730347], [0.04844432324171066, 0.41670748591423035, -0.28918442130088806, 1.1564915180206299, 0.1925300657749176, 1.1267164945602417, -0.37565165758132935, -0.3618699610233307, 0.3032432794570923, -0.32687631249427795, -0.02676285430788994, 0.4818649888038635, 1.515411138534546, -1.9109954833984375, -0.3080219030380249, 2.7128005027770996], [0.3016677498817444, 2.164116382598877, -0.6361040472984314, -1.2658491134643555, -0.47960448265075684, -0.11812839657068253, -1.902369737625122, 0.36294281482696533, 0.48528558015823364, -1.3795201778411865, -1.9541069269180298, 0.020252661779522896, -1.263730764389038, 1.2206116914749146, 1.3197441101074219, -0.8511323928833008], [-0.44609662890434265, -0.3810940980911255, 1.4657539129257202, 1.8367832899093628, 0.8879315853118896, -0.032405875623226166, -0.8104501366615295, -1.748113751411438, -2.2857913970947266, 0.092405766248703, -1.3170318603515625, -0.593900203704834, -0.39479002356529236, 0.03777409717440605, -1.0646569728851318, 1.0333513021469116], [-0.0314137302339077, 1.3596004247665405, -0.19764912128448486, -0.9876863956451416, -1.7020567655563354, -0.9841808080673218, 0.35196277499198914, -0.005112703889608383, -0.6012963652610779, 0.5174590349197388, 0.40739554166793823, 0.08536101132631302, -0.4036283493041992, 2.0064609050750732, 0.6383399963378906, 0.743848979473114], [0.13211897015571594, 0.22611218690872192, 1.5098822116851807, -1.0614837408065796, 0.6379413604736328, -0.6416321992874146, 2.133539915084839, 0.5724356770515442, 0.6802279949188232, 0.23877929151058197, 1.0957061052322388, -0.6268523335456848, 1.4789888858795166, 1.1756833791732788, 0.5206583142280579, -0.2737501263618469], [-0.8861825466156006, 0.9201706647872925, -0.4833987057209015, 1.3506935834884644, -1.3923118114471436, 0.7566882371902466, 0.4324662983417511, 0.01573403738439083, -0.7849215269088745, 0.026978133246302605, -0.3674437701702118, 0.1076216772198677, 0.36146628856658936, -0.024177316576242447, 0.6157096028327942, 0.4589855372905731], [0.4937101900577545, -0.06533406674861908, -0.31957489252090454, -0.9726748466491699, -0.5107295513153076, -2.167632579803467, 0.08647070825099945, -0.8729773163795471, 0.24143871665000916, 1.3990494012832642, -1.0994161367416382, -0.03814287483692169, 1.0607109069824219, 0.8776023387908936, -0.7510790228843689, 1.2003920078277588], [-0.6577054858207703, -0.7606831789016724, -0.9191552400588989, -0.16457247734069824, 0.44186264276504517, 0.7745039463043213, 0.5764167308807373, -0.28474992513656616, -1.6285181045532227, 0.02800861932337284, -0.09939824789762497, 0.39060479402542114, -2.603148937225342, -0.9971799850463867, -1.0967767238616943, 1.118194818496704], [0.5925921201705933, -0.3654228448867798, -0.7267407774925232, -1.5196280479431152, 1.0056759119033813, -0.7153856158256531, -0.3212081789970398, -0.1446264684200287, -0.9732428193092346, 0.10803725570440292, 0.7519980072975159, -0.786567211151123, -1.226917028427124, -0.2555898129940033, -0.8374283909797668, -0.48636603355407715], [0.09702737629413605, 0.4803459942340851, -0.4592895209789276, -0.061911288648843765, 0.05879676342010498, -1.2086542844772339, 1.2181036472320557, 1.2507866621017456, -1.0333360433578491, -1.040095567703247, 0.7490714192390442, -0.8984323740005493, -1.2521979808807373, 1.746229648590088, -1.4167512655258179, -1.04058039188385], [-0.17183226346969604, -0.20480409264564514, 0.36984091997146606, 0.4842729866504669, -0.47504329681396484, -0.14574752748012543, -1.4824104309082031, -0.6049264073371887, -0.4373786151409149, 0.9787808060646057, -1.7215888500213623, 1.8271318674087524, 0.49298936128616333, 1.6216005086898804, -0.2374671846628189, 0.46778109669685364], [0.5347065925598145, -0.3689659833908081, -0.2455170601606369, -2.0378336906433105, 1.6338173151016235, 1.034333348274231, 0.5174242258071899, -0.7896615266799927, 0.07393209636211395, 0.5428175330162048, -0.81914222240448, 0.0002913412172347307, -0.6416047215461731, 0.07913761585950851, 0.18850967288017273, 2.575911283493042], [-2.6990978717803955, 1.1014834642410278, -1.7098329067230225, -0.7707108855247498, 0.27340370416641235, 1.1683504581451416, 0.7199340462684631, -1.9914456605911255, 0.8355981111526489, 0.8542432188987732, -1.9256573915481567, -1.0077062845230103, 0.028203334659337997, 1.0632297992706299, 2.9950387477874756, -0.35796189308166504], [-0.7368902564048767, -1.281954050064087, 0.22822442650794983, 3.3584320545196533, -0.6215561032295227, 1.8767577409744263, -1.1694551706314087, -0.5216382145881653, -0.20117133855819702, -0.37276017665863037, 0.3540520966053009, -0.4774392545223236, -1.9872878789901733, 0.5583416819572449, -0.9860069751739502, 0.3383164405822754], [1.6470396518707275, -0.6636109352111816, 0.5397699475288391, 0.41660961508750916, 2.0534801483154297, -0.4780093729496002, -1.8283933401107788, -0.2978978753089905, 0.5777950882911682, 1.122323989868164, -0.5477479696273804, -0.9616227746009827, -0.9270205497741699, -0.992055356502533, -0.2287166565656662, -0.06051856279373169], [0.2465287148952484, -0.07013614475727081, -1.5306638479232788, 1.8995535373687744, 0.48126959800720215, -1.5702400207519531, -1.1959551572799683, 0.616649329662323, 0.5032526254653931, -1.0125118494033813, 0.9086098670959473, -1.0886213779449463, -1.9927256107330322, 1.0310932397842407, 1.3689643144607544, 1.1908044815063477], [0.9922071099281311, 0.3597605228424072, 0.846054196357727, 0.48765504360198975, 0.6911603808403015, -1.1999024152755737, 0.5436349511146545, 0.18068547546863556, -0.7182241082191467, -0.017936760559678078, -0.9839882254600525, -0.34533584117889404, -0.4357621371746063, 0.08085955679416656, 0.5043322443962097, 1.2358704805374146], [1.3331633806228638, -0.15169638395309448, 2.716921091079712, 1.965357780456543, 1.197701096534729, 0.14841140806674957, 0.4293399453163147, 0.4176660180091858, 0.8249506950378418, -0.6302512288093567, 1.3433666229248047, 2.0711276531219482, 0.9071744680404663, -0.8813989758491516, 0.9904624223709106, -1.8504923582077026], [0.18704195320606232, -0.39885765314102173, 0.9532428979873657, -0.44144582748413086, -0.46101832389831543, -0.16245205700397491, -0.456824392080307, 1.4789767265319824, -0.40954363346099854, -0.5216248631477356, -0.23427905142307281, 0.08043691515922546, -1.0147647857666016, 1.5002435445785522, -0.06447536498308182, 0.616500973701477], [1.5500166416168213, -1.3136627674102783, 0.0008470954489894211, 0.7214308381080627, 0.26701822876930237, 0.2735704779624939, 0.7925113439559937, -0.28246763348579407, 2.3367106914520264, -1.5510576963424683, 1.442130208015442, -0.5559502840042114, 0.0340360589325428, 0.43256160616874695, 1.0187112092971802, 0.8028316497802734], [-0.10584212094545364, -0.08043961226940155, -1.3499048948287964, 0.0006396890385076404, -0.9000036120414734, 1.1431957483291626, 0.13791371881961823, -1.0682932138442993, 1.15154230594635, 1.958417296409607, 0.04986323043704033, -1.0333359241485596, 1.2980003356933594, 0.0929575264453888, -0.0325537733733654, -1.9924956560134888], [-1.0416932106018066, -2.7177557945251465, 0.16725464165210724, -1.4414300918579102, -0.007753652986139059, -1.3934696912765503, 1.0761359930038452, 0.5268332958221436, 1.007047414779663, -2.3527472019195557, -2.5949313640594482, 0.33271992206573486, -0.7542711496353149, -0.49173134565353394, -0.6362085342407227, 0.15069609880447388], [1.4567136764526367, 0.4227370023727417, -0.35662826895713806, 1.0855352878570557, 0.39024877548217773, 1.2517943382263184, -0.4391721189022064, 0.6796350479125977, 0.7790923118591309, -1.4313268661499023, 1.53237783908844, 0.2397051751613617, 0.7129706740379333, 0.0038785303477197886, 0.4689757227897644, -0.18192581832408905], [0.7777020931243896, -0.45651155710220337, -1.100649118423462, -0.6269960403442383, 2.072770833969116, 1.3149460554122925, -0.7973103523254395, -0.48356667160987854, -0.3393123149871826, 1.5650948286056519, 1.1052141189575195, -1.1415585279464722, 0.5778348445892334, 2.47426176071167, -0.3806203305721283, -0.2507130205631256], [0.5831398367881775, 0.45274028182029724, -0.6580448746681213, 0.3985879421234131, 2.2634429931640625, 0.43629512190818787, -0.46099522709846497, -1.078992247581482, -1.3690606355667114, -0.05700920149683952, 0.46939998865127563, -0.22084298729896545, 1.1700646877288818, 0.37870529294013977, -0.3037880063056946, -1.5500999689102173], [-0.6718465685844421, -0.5986917614936829, -0.4038598835468292, 0.6821426749229431, 1.3572108745574951, 0.3782214820384979, 0.07611661404371262, -0.04515423625707626, -0.6036547422409058, 0.5104264616966248, -0.84846431016922, 0.5811565518379211, -0.21055656671524048, 0.3409121334552765, -0.12485724687576294, -0.3897944390773773], [-0.21527999639511108, 1.4137840270996094, -0.43626049160957336, 0.8299379944801331, -1.136176347732544, 1.544063687324524, 0.34904852509498596, -0.008495381101965904, 0.10719211399555206, 1.6109567880630493, 0.6740801334381104, -0.3562217950820923, -0.1901181936264038, -1.007667899131775, -0.5074703097343445, 0.14829537272453308], [0.047632113099098206, -0.5217315554618835, 0.13466627895832062, -1.749703049659729, 1.816796064376831, -0.5928231477737427, 1.593872308731079, -0.4019779860973358, 1.48257577419281, 0.8234943747520447, 0.9747697710990906, -0.6552295684814453, -0.22728231549263, -0.3494906425476074, -0.518221378326416, 1.3724318742752075], [-0.21890926361083984, -0.44108468294143677, 2.2874879837036133, 0.4634236693382263, -0.7867411375045776, 0.2791910469532013, -0.9373906254768372, 0.8930471539497375, 0.1608438491821289, 0.07786964625120163, 2.281357765197754, 0.41170525550842285, 2.489729404449463, 0.8342171311378479, -0.26206231117248535, -0.8050717711448669], [-0.19122876226902008, -0.8286558985710144, -0.12390947341918945, 0.09116093069314957, -0.4766756296157837, 0.4588474631309509, -0.2744723856449127, 0.018364204093813896, -1.080188512802124, 0.6889798045158386, 0.71729975938797, 0.5000729560852051, -0.8008824586868286, -0.14174342155456543, 0.06131018325686455, -1.01629638671875], [1.5387992858886719, 1.815841555595398, 0.15802110731601715, -0.38473695516586304, -1.2174742221832275, 0.13710449635982513, -1.24466073513031, -0.9788503050804138, 1.3426036834716797, 0.3464301526546478, 1.2002476453781128, 0.5179231762886047, 0.8408336043357849, 1.7203829288482666, 1.4214645624160767, -1.8424986600875854], [0.4832940399646759, 0.4134579002857208, 2.1261801719665527, -0.9870172739028931, 1.0842649936676025, -1.3019170761108398, 0.37744683027267456, 0.6545843482017517, -0.3384557068347931, 1.0581135749816895, 1.5769106149673462, 0.5805351734161377, 1.846693992614746, -2.1841588020324707, -1.2972102165222168, -0.007666024379432201], [0.18712301552295685, 0.8424714207649231, 0.5546365976333618, 0.7975795865058899, 0.24670462310314178, 0.0996314138174057, 0.37317126989364624, -0.7698569893836975, 0.37664100527763367, 0.4607703387737274, -1.6341712474822998, -0.5305081009864807, -0.7242163419723511, -1.3355358839035034, -0.4697873890399933, 0.7806558012962341], [-2.0616025924682617, 0.6184630393981934, 0.836666464805603, 0.1353795826435089, -0.31781619787216187, 0.813459575176239, 1.5154088735580444, 0.7835072875022888, 0.24698273837566376, -0.7407435774803162, -0.34998661279678345, 1.310579776763916, 1.6092909574508667, -0.4403804540634155, -1.078505277633667, -0.5709575414657593], [-1.3889960050582886, 0.8237109780311584, -0.6387034058570862, -0.7490994334220886, 0.2215491533279419, 0.3620551824569702, -0.1310500055551529, 0.04716769978404045, 1.2653554677963257, 0.7260428667068481, -0.4810034930706024, -0.002804900985211134, 0.8368595242500305, 0.7783224582672119, 0.761837899684906, -1.7090178728103638], [0.39914026856422424, -0.175459086894989, 0.23217523097991943, -0.35423552989959717, 1.035705804824829, 0.8095960021018982, -0.6858214735984802, -0.15441104769706726, 0.7334017157554626, -0.03630441054701805, 0.6544730067253113, 1.3145194053649902, -1.0349206924438477, 0.1514449566602707, -0.7768169641494751, 0.9416435956954956], [-0.5041820406913757, 1.498416543006897, -0.5661313533782959, -1.2587835788726807, 1.4438462257385254, -0.14139023423194885, -1.3285541534423828, -0.39203375577926636, 1.6369179487228394, -1.6767746210098267, -0.6294627785682678, -1.0607703924179077, 0.029695658013224602, -0.388917475938797, 0.042920466512441635, 0.4184053838253021], [0.3336964547634125, -1.4936153888702393, 0.40288978815078735, -1.5600775480270386, 0.9067257642745972, 0.09339595586061478, -0.7086426615715027, -0.7708362340927124, 1.6255147457122803, 0.9197036623954773, -0.04708471521735191, 1.3466298580169678, 1.3342382907867432, -0.8844771385192871, -0.6342881321907043, 2.672682523727417], [-0.23344388604164124, 2.6886942386627197, 0.053235676139593124, 0.7255024909973145, -0.3357662558555603, 0.41280198097229004, 1.4498634338378906, 0.8920101523399353, -0.21215584874153137, -1.0659947395324707, -2.0649664402008057, 0.7622078657150269, -1.0031989812850952, -0.6132933497428894, -0.3253801167011261, 0.9192216396331787], [-0.5383666157722473, 0.3835059404373169, 0.13609854876995087, -0.9776757955551147, 0.9732441306114197, -0.37702685594558716, -0.4284857511520386, 1.1243853569030762, -0.03768278658390045, 0.3367345631122589, 0.3676789104938507, 0.7060843110084534, -0.7044142484664917, 1.7144837379455566, 0.8913341760635376, -3.399348258972168], [-0.08469615131616592, 0.3067413568496704, 1.745510220527649, -0.028582938015460968, 0.2019350379705429, -1.6712677478790283, 0.8645943999290466, 1.6921988725662231, -0.2959223687648773, -0.06038188189268112, -0.40581417083740234, 0.057581137865781784, -0.674599289894104, -0.060387976467609406, -0.1162605807185173, -0.006218023598194122], [0.23664548993110657, -2.0641331672668457, -0.3497953712940216, 1.1164929866790771, 0.7809985280036926, -0.12806765735149384, 0.2807786166667938, -0.24432209134101868, 0.4155727028846741, -1.1303284168243408, 0.35321855545043945, 0.2582172453403473, 0.8101481795310974, -0.3800361752510071, 0.6420699954032898, -0.13664062321186066], [0.27423524856567383, -0.5892035365104675, 1.072157621383667, 0.05306382477283478, -1.2516134977340698, -0.06455869972705841, -1.1149219274520874, 1.046005129814148, 0.2798939049243927, -0.5284796357154846, 1.0120787620544434, -0.7382946014404297, 0.7758311033248901, -0.6451094150543213, 0.4572233557701111, 0.5570217370986938], [-1.0277749300003052, -0.552404522895813, 0.4395810067653656, -0.9354389905929565, -0.17027314007282257, -0.39393702149391174, 1.0512124300003052, -0.19370274245738983, 0.3262133002281189, -1.986130714416504, 1.242697834968567, -0.12042085826396942, 1.0749669075012207, -0.8975563049316406, -1.8933008909225464, -1.4626086950302124], [-0.3945704400539398, -0.6454266905784607, -0.19679217040538788, -0.7409617900848389, -0.07140713185071945, 0.20903679728507996, 0.8192503452301025, -0.007737375795841217, 2.3564090728759766, -0.6954357624053955, 1.120995283126831, -0.8479461669921875, 0.8411503434181213, 2.254000186920166, -0.5417918562889099, 0.5424348711967468]]], [[[-0.9825416803359985, 0.9571484327316284, -0.02972027286887169, -0.9954846501350403, -1.167845368385315, -1.020090103149414, -0.3800366520881653, 0.6706047654151917, -1.6487703323364258, -1.6424450874328613, -1.096841812133789, 0.35933759808540344, 0.2978525161743164, -0.8170545101165771, -0.22177889943122864, -1.6267404556274414], [-0.13074932992458344, 2.057568073272705, 0.5422895550727844, 0.3132480978965759, 1.3799264430999756, -0.30897775292396545, -0.6911891102790833, -0.5258079171180725, 1.3823533058166504, -0.048974789679050446, 0.24943600594997406, 1.7275331020355225, -0.7096526026725769, 1.6352156400680542, 1.1603188514709473, -0.14631935954093933], [0.15297465026378632, -0.16947117447853088, -0.07046138495206833, -0.3477366268634796, 1.8993043899536133, 0.020538166165351868, 1.6183034181594849, 0.88957279920578, 0.1433495432138443, -1.8480517864227295, -0.6605022549629211, 0.17234574258327484, 0.465820848941803, 2.1865408420562744, -0.7559401392936707, -0.8734214901924133], [-0.022906146943569183, -0.20674656331539154, -0.9219496846199036, -0.6816068291664124, -1.1466280221939087, -1.1757181882858276, 0.3151431083679199, 0.7852557897567749, 0.305858850479126, 0.7148933410644531, -0.6947716474533081, -1.2879512310028076, -0.07360780984163284, -0.42556604743003845, -0.34197184443473816, 0.6090074181556702], [-0.5111677050590515, -0.40028220415115356, 3.096889019012451, -0.10156265646219254, -1.2491047382354736, -1.4424570798873901, 0.20217104256153107, 0.16835589706897736, 0.20943889021873474, 0.4587244689464569, 1.606871485710144, 0.20864467322826385, 0.4469117224216461, -1.576442837715149, 0.2489892989397049, -0.6589238047599792], [0.49077510833740234, -0.12544770538806915, 0.5418175458908081, -0.953230082988739, -0.15208454430103302, 1.2095725536346436, -0.993708610534668, -0.7559686303138733, 0.28650540113449097, -1.4391900300979614, 0.5769286155700684, 0.37440037727355957, 0.4441834092140198, 1.145462155342102, -0.07974815368652344, 0.2351633906364441], [0.6876345276832581, -0.6186298131942749, 2.2897117137908936, -0.6985188126564026, 0.0637882873415947, -0.7300485968589783, 0.7665796279907227, 1.7496525049209595, -2.2622315883636475, 0.889197051525116, 0.6658636331558228, 1.0261956453323364, -1.3755505084991455, -1.3406546115875244, 0.44651544094085693, 0.4155914783477783], [1.3026833534240723, 0.6636824011802673, -1.9116841554641724, 1.0161336660385132, 1.939200758934021, -1.039707064628601, 1.0992010831832886, 2.124195098876953, -0.3685442805290222, -0.5606180429458618, 1.4942599534988403, 0.8047378063201904, -0.6551411151885986, 0.5630702376365662, -0.06630675494670868, -1.1140176057815552], [1.4964274168014526, -0.007604589685797691, -1.5772706270217896, 0.4804884195327759, 0.9851157665252686, -1.4114834070205688, 0.5753647685050964, -1.0168205499649048, -0.23967468738555908, 0.8615180850028992, 0.16681475937366486, -0.6204359531402588, 0.004885764792561531, -0.6069108843803406, -0.07767102867364883, -1.6601176261901855], [0.4623652398586273, -0.9450894594192505, -1.0212512016296387, -0.0984978973865509, -0.6395210027694702, 0.3316286504268646, 0.30435073375701904, 1.3054789304733276, -0.41467607021331787, -0.7789342999458313, -0.2132108509540558, 0.07236943393945694, -0.5447788238525391, -0.7372717261314392, 2.106837034225464, -0.4456098973751068], [1.0169651508331299, -1.02493155002594, 0.7181110978126526, 1.3998284339904785, 0.07033012807369232, 0.276333749294281, -0.25756174325942993, 0.7106848359107971, -1.5346076488494873, 0.05692071467638016, -1.1215832233428955, -0.6822109818458557, -0.23391693830490112, -0.5444096922874451, 0.5271977782249451, 2.4258759021759033], [-0.1557840257883072, 0.5399811267852783, 2.6849939823150635, -1.9026036262512207, -0.41279736161231995, -2.0030276775360107, 1.102205753326416, 0.09626191854476929, 2.0251288414001465, -0.5381283760070801, -1.8629276752471924, 1.59763503074646, 0.17106309533119202, -0.6169751286506653, 1.8317890167236328, -0.9840375781059265], [-1.6354426145553589, 1.0769506692886353, -0.5049888491630554, 0.443803071975708, 0.5765183568000793, 1.334038496017456, 0.668093204498291, 0.8811350464820862, -1.5443800687789917, -0.8559057712554932, 2.441896677017212, -1.2125046253204346, -0.14526565372943878, -0.011087113991379738, 0.6018567681312561, 1.1122279167175293], [1.122193694114685, 0.1230953261256218, 0.6876458525657654, -1.2881320714950562, 1.0663747787475586, 1.053732991218567, -0.21379826962947845, 0.1348937749862671, -0.6429203748703003, 0.9369596838951111, -1.4786933660507202, 0.03356228023767471, 0.997391939163208, -0.3995034992694855, -0.17333321273326874, 1.9713491201400757], [-0.8403412699699402, 0.8618853688240051, 0.4048175513744354, 0.24759286642074585, -1.5760297775268555, -0.6648134589195251, 1.245395302772522, -0.7466459274291992, 0.9223512411117554, -0.5932885408401489, -1.190743327140808, 0.8708281517028809, -0.3683726489543915, 0.26701977849006653, 0.799588143825531, -0.7273731231689453], [0.21888886392116547, 0.40616893768310547, 1.1020317077636719, -0.20988745987415314, -0.24122437834739685, -0.7103859782218933, -1.0496454238891602, 1.9253596067428589, 0.37529850006103516, -1.006083607673645, 0.11023468524217606, 0.21958835422992706, 0.3637153208255768, 1.1217985153198242, 1.745962142944336, -1.2804049253463745], [-0.5950741767883301, 0.9449004530906677, 1.0188928842544556, 0.36129429936408997, -0.38408002257347107, -1.672633409500122, 0.36676618456840515, -0.5409958362579346, 0.38807594776153564, -0.25446420907974243, -0.9611981511116028, -1.0167869329452515, -2.0885469913482666, -1.0301802158355713, -2.040510892868042, -1.1782702207565308], [-0.22452303767204285, 0.30762749910354614, -0.1390456259250641, -0.5430358648300171, 0.07783050090074539, -1.6201355457305908, 0.9385866522789001, 0.7472105622291565, 0.38676023483276367, 0.4707011282444, -1.3184486627578735, 0.011973871849477291, -1.3514204025268555, 0.21074151992797852, 0.6152306199073792, -0.23212862014770508], [0.32519370317459106, -0.8308274149894714, 0.32182827591896057, -0.7550464868545532, -1.594201683998108, 0.6107977032661438, -0.2286376804113388, 0.8761788010597229, 0.7546525001525879, 0.6245750188827515, 0.09964174032211304, -1.0233063697814941, 0.6608484983444214, -1.5945096015930176, 1.4013670682907104, -0.14740586280822754], [0.8048528432846069, -0.44116365909576416, 0.18237809836864471, -1.4405440092086792, 1.6602495908737183, 0.1923598349094391, -1.4021989107131958, -0.1262846291065216, -0.35404327511787415, 0.6040729880332947, -0.04761854559183121, -0.790665864944458, -0.6219707131385803, 0.3754352629184723, 0.3539440631866455, -1.413301944732666], [0.38093477487564087, 2.5433874130249023, 0.6776208877563477, 0.09543848782777786, 0.8625667095184326, -0.7955343723297119, 0.7217567563056946, -0.20219558477401733, -0.20897303521633148, -0.8067664504051208, -0.5683435797691345, -0.1624031364917755, -0.38105088472366333, 0.009134531952440739, 2.195554733276367, -0.09611158072948456], [-0.6257342100143433, 0.15462426841259003, 0.15987473726272583, 1.3987460136413574, -0.3583102524280548, -2.108074903488159, -0.14150327444076538, 0.4413449466228485, 0.7876975536346436, 0.24066497385501862, 0.37267154455184937, -1.517197608947754, 1.4611048698425293, 0.5592553615570068, 0.19857478141784668, 0.5371400117874146], [-0.4730188250541687, 0.01560616958886385, 0.8476620316505432, -0.303482323884964, -1.7471638917922974, -0.7143383026123047, -0.2953355610370636, 0.0699300467967987, -0.1153109222650528, 1.9024953842163086, 0.16888639330863953, 0.5516371726989746, -0.43587109446525574, 0.13762623071670532, -0.41994184255599976, 0.7294026017189026], [0.8000739812850952, 0.029893502593040466, -0.10721847414970398, -0.9731065630912781, 0.6174349188804626, -0.42245185375213623, 0.5542132258415222, 0.387848436832428, -1.3368529081344604, 0.3702608644962311, 0.9055028557777405, -0.799676775932312, -1.0509730577468872, -0.6518732905387878, 0.7674434781074524, -1.3880940675735474], [0.3578869700431824, 0.5120199918746948, 0.5079120993614197, -0.27299052476882935, 0.2370709776878357, 0.2329297959804535, 0.7779850959777832, 0.28165486454963684, -1.3538378477096558, 0.8283401727676392, 2.537100315093994, -0.8516349792480469, 0.09444519132375717, 0.247809037566185, 0.3888317942619324, -0.3268028795719147], [-1.143384575843811, 1.4427748918533325, 0.637914776802063, 2.0717904567718506, 1.6907273530960083, 0.8060397505760193, 0.8118057250976562, -0.7299172282218933, -1.533899188041687, 1.3008052110671997, 1.0520600080490112, 0.6239809393882751, 1.731349229812622, 0.7812447547912598, -0.492149293422699, -0.3180674612522125], [0.9472149610519409, -0.16113430261611938, -0.7222912311553955, 0.08966754376888275, 0.7398009300231934, -0.6243999600410461, -0.9433414340019226, -0.03840490058064461, -1.263325572013855, -1.7140512466430664, 0.5022323727607727, -0.9798910617828369, 0.31793683767318726, -0.23948663473129272, 0.49447429180145264, 1.446796178817749], [0.6121588349342346, 0.079474538564682, -0.4848487973213196, 0.22166450321674347, -0.5756861567497253, -0.723919689655304, -0.7601882815361023, 0.4691024124622345, 1.2814743518829346, -2.350882053375244, -0.7511014938354492, 1.5136873722076416, -0.06858645379543304, 1.246330976486206, 1.1176241636276245, -0.24591882526874542], [1.4743685722351074, 1.6112737655639648, -1.4769985675811768, 0.45811066031455994, 0.10405274480581284, 0.1282484382390976, 1.112203598022461, 1.0324265956878662, -1.18374502658844, 0.6821691989898682, -1.0923240184783936, 0.4651864469051361, -0.6838061809539795, -0.20860914885997772, 1.0619992017745972, -0.6771587133407593], [-1.2162925004959106, -0.16175787150859833, -1.3468458652496338, 0.26596397161483765, -1.1252942085266113, 0.33083996176719666, -0.0611158162355423, -0.2850145697593689, -0.6007517576217651, 1.7739300727844238, -2.348263740539551, 0.5987293124198914, -2.303704261779785, -0.6314071416854858, 0.303305059671402, 1.3613098859786987], [-1.623512625694275, 0.4432721436023712, 2.4628520011901855, -0.8286944627761841, 0.8635622262954712, 1.959152102470398, 0.8179883360862732, -1.8553980588912964, -0.8237919211387634, -0.062443528324365616, 0.6678635478019714, -0.9286313652992249, 1.1711679697036743, 0.6025844216346741, -0.12437628209590912, -0.8269157409667969], [0.3553113341331482, -0.8898493647575378, 0.9737478494644165, 0.21007820963859558, -0.8047425746917725, 0.2832544445991516, 0.24542804062366486, 0.07226568460464478, 0.6089552044868469, -0.13482427597045898, 0.6332535743713379, 0.384051114320755, 1.730852723121643, 1.2337472438812256, -0.505588173866272, 0.700427234172821], [0.7458099722862244, -1.2065273523330688, 0.0445052869617939, 0.08554524928331375, -1.5364645719528198, -1.6226022243499756, 1.459999442100525, 0.5219301581382751, 0.9133703112602234, -0.09154719114303589, -0.37878596782684326, 0.49337127804756165, -0.589156448841095, -1.6238703727722168, 0.03814125061035156, -0.9142774939537048], [-0.3571180999279022, -0.3677898943424225, 0.5540623664855957, -0.1901831179857254, -1.7139012813568115, -0.8694384098052979, 0.24960778653621674, -0.9106621742248535, 0.5913808345794678, 0.6045854091644287, 0.012833667919039726, 0.038936540484428406, -0.20408280193805695, 0.12514910101890564, 1.157328486442566, -0.06600132584571838], [-0.5906880497932434, -0.36124494671821594, -1.6051356792449951, 1.206181526184082, -0.8782116770744324, -0.16797283291816711, 0.855855405330658, -0.9855815172195435, 1.3455032110214233, -0.36184918880462646, -0.2858107089996338, -0.3912706971168518, -1.3303886651992798, 1.587420105934143, -1.8851820230484009, -1.4126497507095337], [-0.39898568391799927, 0.8088194131851196, -0.4974314868450165, 0.5173529386520386, 0.7596689462661743, -1.3216636180877686, -1.750625729560852, -0.5810393691062927, 0.14995552599430084, -0.08263503760099411, 0.31230926513671875, 1.8884122371673584, -1.653865098953247, -1.1529474258422852, 0.2247118502855301, 1.1663827896118164], [-1.1210185289382935, 0.43433237075805664, -0.5639803409576416, 0.4955408573150635, 0.6896073222160339, 0.3181232810020447, -0.8314278721809387, 0.31048333644866943, -0.12195759266614914, 0.41679736971855164, 0.29787591099739075, -0.9027953743934631, -0.32216715812683105, -0.9342168569564819, -2.146284580230713, 1.2607214450836182], [-0.5260366797447205, -1.7516018152236938, -0.5487911701202393, -0.05551630258560181, 2.474555492401123, -0.6504966020584106, -0.2695743143558502, 1.381514072418213, 0.5766853094100952, 0.8015269041061401, 1.2216891050338745, 1.143510341644287, 2.8180198669433594, 0.19487875699996948, 1.0305994749069214, 0.9115147590637207], [-0.11983975768089294, -0.5064128041267395, -0.21215902268886566, -0.557870626449585, -1.7568647861480713, 0.5654172897338867, 1.5761475563049316, 0.4489724040031433, -0.9436934590339661, -0.19275841116905212, -0.9608004093170166, -0.16766560077667236, 1.0105899572372437, 0.40604346990585327, -0.8285069465637207, -0.669436514377594], [1.909517765045166, 1.295875072479248, -0.2411266416311264, 0.3316739797592163, 0.4761074483394623, 0.21898964047431946, 1.3026396036148071, 0.17410165071487427, -0.053705718368291855, 0.3338947296142578, 1.2063345909118652, -1.4961568117141724, -0.14078764617443085, 2.594244956970215, -1.3083800077438354, 0.7281542420387268], [0.8375934362411499, -2.5638787746429443, -1.123193621635437, 1.6963675022125244, -0.32855111360549927, 0.2756492793560028, -0.28295785188674927, -0.7287777066230774, -0.0024471303913742304, 0.4153279662132263, 1.5216991901397705, 1.0804661512374878, 0.5094670057296753, 1.8553171157836914, 1.2953475713729858, 1.1898874044418335], [-0.9209265112876892, 1.235470175743103, 3.207768440246582, -0.6971904635429382, 0.43763479590415955, -1.2986376285552979, -1.0374504327774048, 0.5615904927253723, 0.9442012310028076, 0.47039785981178284, -0.3933814465999603, -1.1642284393310547, -0.710581362247467, 2.3886008262634277, 1.232702612876892, 0.3495796322822571], [-0.03450818732380867, -0.40354010462760925, 1.712019443511963, -0.7349240183830261, 0.5788107514381409, 0.351849228143692, 0.7866792678833008, -0.8016834259033203, 0.6496961116790771, 1.7804067134857178, 1.7468633651733398, 1.0107897520065308, -0.7485429644584656, 1.7886357307434082, 0.1970990002155304, 1.6307191848754883], [0.4908006191253662, -0.7603963613510132, 0.261810839176178, 0.29864558577537537, -0.3022785782814026, 1.426012635231018, 0.42156338691711426, 1.7589651346206665, -0.4025271236896515, -0.005037412513047457, 0.5908640027046204, -1.2816925048828125, 0.23829054832458496, 0.3507852852344513, 1.218461513519287, 1.5258013010025024], [-0.2041344791650772, 1.029226541519165, -0.8557174205780029, -1.0368057489395142, 0.26543790102005005, 0.923089861869812, 0.4020415246486664, 0.9651408195495605, 0.049316275864839554, 1.2931228876113892, -0.3880177140235901, 0.05314183235168457, -0.05227183923125267, -0.6287742853164673, -0.3286454975605011, 1.0272163152694702], [-0.8230711817741394, 0.04997743293642998, -0.8914188146591187, -0.36486783623695374, -1.0305901765823364, 1.2407667636871338, -0.4717288911342621, 1.0798511505126953, -0.5141529440879822, 0.3925749957561493, -2.8983771800994873, -2.191009283065796, -1.2826213836669922, -0.4731138348579407, -1.0653618574142456, -0.25273948907852173], [0.028870869427919388, 1.9720544815063477, 1.0336592197418213, -0.5989109873771667, -0.24758876860141754, 0.4333595633506775, 0.7568955421447754, 0.08809185028076172, -0.44936490058898926, 0.709529459476471, 0.6966766119003296, 0.7217057347297668, 0.587960422039032, 0.7135232090950012, 0.9376266002655029, -0.1283196359872818], [-0.4256981313228607, -0.15242134034633636, 0.5098118185997009, -1.9790174961090088, -0.7312183976173401, -0.42685720324516296, 0.973493754863739, -0.8830862641334534, -1.1711245775222778, 0.7356760501861572, -0.1862744241952896, -0.49187028408050537, 0.6826720237731934, -0.29408708214759827, 0.6471720337867737, 0.31352096796035767], [2.031637191772461, 0.40375494956970215, 0.8372580409049988, 0.49305543303489685, -0.33345356583595276, 0.2762196362018585, 1.2305997610092163, 0.26513075828552246, -0.9092104434967041, 1.0129892826080322, -0.9393949508666992, 0.4657868444919586, -0.12011545151472092, -0.4278329014778137, -0.27939966320991516, -0.752512514591217], [0.8239907622337341, -0.5148369073867798, -0.6921831965446472, -1.4182238578796387, 0.10810986906290054, -0.061439916491508484, -1.9192097187042236, 0.07411126792430878, -1.0136555433273315, -0.8431915044784546, 0.7298697233200073, 1.149908185005188, -0.23994328081607819, 0.5645113587379456, -0.665972888469696, 0.8972254395484924], [-1.0280870199203491, 0.7689438462257385, -0.289098858833313, -0.48360884189605713, 0.6266566514968872, -0.49224188923835754, 0.7507272362709045, 0.049809377640485764, 0.1575257033109665, 0.7578725814819336, -0.4038442373275757, 1.5682274103164673, 0.05091476812958717, 0.5275746583938599, 0.5129252076148987, 0.10240194946527481], [-0.6468260288238525, -1.2009180784225464, -1.2949943542480469, -1.470047950744629, 0.12326665222644806, -0.6121310591697693, -1.028091311454773, 0.5876369476318359, -1.8766722679138184, 0.2654010057449341, 0.4415273070335388, -1.1281814575195312, 1.27961266040802, 0.2323908656835556, 0.6739837527275085, -0.07240605354309082], [-0.81536465883255, 0.4897598326206207, -0.9543529748916626, -0.5952321290969849, 1.4295296669006348, -1.1490827798843384, 1.16571044921875, 1.4412405490875244, -0.9298697710037231, -1.7738415002822876, -0.3222591280937195, 0.15686769783496857, -2.8211658000946045, -0.5292407870292664, -0.5122314691543579, 0.6244828104972839], [-2.7685773372650146, -2.9449610710144043, -0.11744147539138794, -1.3780444860458374, 1.1079528331756592, 0.5451540946960449, -1.2380589246749878, 1.114702820777893, 0.2940637767314911, 0.5120353698730469, -0.7440498471260071, 0.22320790588855743, 0.3316728174686432, -1.1715614795684814, -0.31585726141929626, -0.01593875139951706], [0.6487693786621094, -0.20469945669174194, 0.6063652634620667, -0.2296798676252365, -1.1498304605484009, -0.3650336265563965, -0.9907780289649963, 0.7302831411361694, 1.6721148490905762, 0.5652827024459839, -0.8054800629615784, -0.1556592881679535, 1.2126615047454834, -0.837408185005188, 1.1404166221618652, 0.14538614451885223], [0.6380048990249634, 0.7428986430168152, -0.8834285140037537, 1.2754490375518799, 2.1622369289398193, 1.1328856945037842, -0.17644479870796204, 0.18848812580108643, 0.36248886585235596, -0.08193604648113251, 0.9645118713378906, 0.9923766255378723, 1.3807272911071777, 1.5221693515777588, -1.7931619882583618, 0.3612094521522522], [1.102589726448059, 0.17100881040096283, 0.765873372554779, -1.6373416185379028, -0.31704387068748474, 0.8593335151672363, -1.2501672506332397, 0.6611674427986145, 0.9001930952072144, 0.6408220529556274, 2.5775771141052246, -0.415104478597641, 0.040146604180336, 1.9705020189285278, -0.3662191331386566, -0.8996152281761169], [0.552257239818573, 0.9013472199440002, -1.3268376588821411, -1.6277693510055542, -0.5965613126754761, -0.08477179706096649, 1.0183202028274536, -0.6770983934402466, 0.13285909593105316, -0.35717305541038513, 0.6449111104011536, 0.3314442038536072, -0.562113344669342, -0.6661465167999268, -0.19050775468349457, 1.197211742401123], [0.13392607867717743, -0.6227474808692932, 1.8381640911102295, 1.3442425727844238, 0.7431174516677856, 0.30118656158447266, 1.169684648513794, 1.2294706106185913, 0.9077869653701782, 0.33179762959480286, -0.9872658848762512, -1.8473093509674072, 1.2026562690734863, 0.35926011204719543, 0.2927379310131073, -0.36964672803878784], [-0.5569959878921509, -0.6908987164497375, 1.3727978467941284, -0.04271000996232033, 1.8341922760009766, 0.030164193361997604, -0.5874032378196716, 1.2760062217712402, -2.670773983001709, -0.4398660957813263, -2.3256189823150635, 1.5662717819213867, -1.1510353088378906, -0.3329871892929077, 1.430551528930664, 0.5824296474456787], [-1.4928866624832153, -0.5347640514373779, -0.4290558695793152, -0.6669399738311768, 0.6010293960571289, 0.5354210734367371, 0.8671211004257202, 0.8256282806396484, -3.0240235328674316, -0.7060385346412659, 1.812984585762024, 1.4268327951431274, -0.1415739804506302, -0.8383100032806396, -1.0957152843475342, -0.6750138401985168], [-1.3690274953842163, 0.26021841168403625, -0.5841938853263855, 0.8092044591903687, -0.7368687987327576, 0.3430982530117035, -0.7181091904640198, -0.164811909198761, -1.3122224807739258, 0.6402062177658081, -1.016463279724121, 0.18184009194374084, 0.251849889755249, -0.5282145142555237, -0.9132541418075562, -1.4594473838806152], [-2.0098776817321777, 0.002395747462287545, 0.9097121357917786, -1.240106225013733, 0.31030192971229553, -0.09386692941188812, 0.3860597014427185, -1.127238154411316, -2.841876745223999, -0.19186799228191376, -0.5487304329872131, -0.6558883190155029, -0.49684616923332214, -0.0647527277469635, 0.8094566464424133, 0.851815402507782], [0.4343053996562958, -0.17519532144069672, 1.0505093336105347, -0.5719826221466064, -0.18717585504055023, 0.5712134838104248, 1.4405980110168457, -0.539320707321167, -0.3331337571144104, 1.4392534494400024, -1.1750115156173706, -1.2563186883926392, -1.9950190782546997, -2.176459312438965, 0.12792743742465973, -1.3055310249328613], [-2.0308730602264404, 0.4299858510494232, 0.06676168739795685, 1.4179396629333496, 1.140687346458435, 1.2028428316116333, 1.0433757305145264, -1.0084527730941772, 0.5765801668167114, 2.920201301574707, 1.3612103462219238, 1.6886239051818848, -0.9993689656257629, -0.4692036509513855, -2.004188060760498, 0.7174939513206482], [1.0056848526000977, -1.2479057312011719, 0.3385479748249054, 0.20314739644527435, 0.24957729876041412, -3.0016088485717773, 0.3688869774341583, 1.0704586505889893, 0.566134512424469, 0.9552826285362244, -0.17661115527153015, 0.008798615075647831, -0.44420936703681946, 0.019366255030035973, 0.05592731386423111, -0.11116683483123779], [0.28387126326560974, -2.07356858253479, 0.05654597654938698, -0.0908002182841301, -0.5888429284095764, 1.115605354309082, 0.20502012968063354, 1.3257755041122437, 0.7050222158432007, 1.8101767301559448, -1.7976651191711426, 0.8035069704055786, 0.4751439392566681, 0.6576083898544312, -0.7496406435966492, -0.8920360207557678], [2.3186252117156982, 1.2079871892929077, 1.8662818670272827, -0.07857055217027664, -1.181667685508728, -1.825497031211853, -0.8837424516677856, -0.06989780813455582, -1.0912703275680542, -0.974974513053894, 0.4767288565635681, 0.3606366515159607, 1.172699213027954, -1.3003528118133545, -0.058543771505355835, -1.1444406509399414], [-1.8955786228179932, -0.6460073590278625, -1.2209721803665161, 0.6075244545936584, -1.623741626739502, -0.5876876711845398, 0.5686070322990417, 0.8465045094490051, 0.48280060291290283, -0.7888425588607788, -1.1743370294570923, -0.1097596138715744, 0.7028003334999084, 0.11385300010442734, -2.6073038578033447, 1.1996222734451294], [-1.6391030550003052, -0.6545800566673279, -0.5514859557151794, 1.161937952041626, -0.7786533236503601, -0.6758076548576355, -0.8038479089736938, -0.4354337453842163, -0.7602939009666443, -1.5384328365325928, 0.2342565357685089, 0.4102895259857178, -1.2949676513671875, 0.7786357998847961, -1.899080514907837, 0.48790013790130615], [-0.3707312345504761, 0.9090854525566101, 1.4148727655410767, 1.9443835020065308, -1.1925086975097656, 0.32790032029151917, -1.0859756469726562, 0.56233149766922, -1.4060136079788208, 0.11103799194097519, -0.05661608278751373, -0.02153133414685726, -0.03400145843625069, -0.1491968333721161, 1.1764447689056396, -0.5198283195495605], [-0.17157083749771118, -0.5390210747718811, -0.08680792897939682, -0.4299475848674774, 1.5276589393615723, -0.3055117726325989, 1.946109652519226, -0.9023308157920837, 0.5418969392776489, 1.4808865785598755, -0.8643398284912109, -0.15595309436321259, -1.011666178703308, -1.3947081565856934, -0.4402186870574951, 0.8344694972038269], [1.7856382131576538, 0.16892582178115845, -1.9191676378250122, -0.8891273140907288, 1.0483437776565552, -0.9207745790481567, 0.5154447555541992, -1.8647189140319824, -1.7902591228485107, 0.3320835530757904, 2.3417725563049316, -0.1700151562690735, 0.8439788818359375, -0.38844919204711914, -1.6928784847259521, 0.851533055305481], [1.0604636669158936, -1.1140457391738892, -0.03644149377942085, 0.6671929955482483, 0.00667925039306283, -1.7387717962265015, -2.255557060241699, 0.46874481439590454, -1.5557337999343872, -1.0100914239883423, 0.36904507875442505, -0.7817965149879456, -0.30527904629707336, -0.8035172820091248, 2.6338913440704346, -0.13773232698440552], [-0.7932729721069336, -0.6056867837905884, -0.36297133564949036, 0.4837215840816498, 1.1730457544326782, 1.5446438789367676, -0.727233350276947, -0.1939220130443573, 0.46858853101730347, 0.22806213796138763, 0.775960385799408, -1.8136191368103027, -0.5986073613166809, 1.5009218454360962, -0.4686800539493561, -0.17626060545444489], [-0.8271334171295166, -0.13576455414295197, -1.0232129096984863, 0.31881651282310486, -0.3047572374343872, -1.1607376337051392, -0.8631731867790222, -1.121235966682434, 0.2988651990890503, 0.3268880248069763, 1.5631839036941528, 1.1390275955200195, 0.4228099286556244, 0.26112377643585205, 0.18880638480186462, 0.8802370429039001], [-0.768972635269165, -0.41268831491470337, -1.4329913854599, -0.6882075667381287, -1.6082061529159546, -0.009118848480284214, -0.12915386259555817, -0.9815880656242371, 0.8386761546134949, -1.21007239818573, -0.35439759492874146, -0.6883910894393921, 0.7972153425216675, 1.0152872800827026, -1.3939783573150635, 0.21886694431304932], [0.968620240688324, -1.4181110858917236, -2.210023880004883, -0.9006934762001038, -0.27159246802330017, 0.8494245409965515, -0.15677517652511597, -0.9072117805480957, 1.4567596912384033, -1.8456101417541504, -0.31918346881866455, 1.1466858386993408, 1.178571343421936, -1.0696078538894653, 0.05675569549202919, -0.13188284635543823], [2.822575092315674, -1.6066190004348755, -0.6754428744316101, 0.7475777268409729, 0.08979478478431702, 0.1731126755475998, -0.2660279870033264, 0.35692858695983887, 0.35108301043510437, 1.049464225769043, -2.250662326812744, -0.6280660629272461, -0.22824670374393463, 0.5877313613891602, 0.011272555217146873, -1.0417569875717163], [0.46067526936531067, -0.31377458572387695, -1.103942632675171, 1.2950807809829712, 0.3445977568626404, -0.37487244606018066, -1.0951108932495117, 0.8193233013153076, -0.26768264174461365, -0.8545438647270203, -1.2117667198181152, 0.8571308851242065, -0.9225881695747375, -1.9992293119430542, 0.1038493812084198, 1.1874074935913086], [-1.013804316520691, -2.548860788345337, -0.6408869028091431, 0.24746614694595337, -0.5700685977935791, -0.7166486382484436, 0.6554064750671387, -2.0740323066711426, -0.664991557598114, -1.0168936252593994, 1.0701420307159424, 1.6351650953292847, 0.3219885230064392, 0.35663336515426636, 0.045165903866291046, 1.9588265419006348], [0.7795172333717346, 0.8640371561050415, -2.8977060317993164, 0.7350252866744995, 0.3352293372154236, 1.13872230052948, -0.20965783298015594, -0.4063490331172943, -0.8865591883659363, 0.9292722940444946, -0.2648739516735077, 0.3787913918495178, -1.2288849353790283, 1.0332789421081543, -0.2626516819000244, 0.4675217866897583], [-0.49893277883529663, -0.5583586692810059, 1.4448198080062866, -1.860766887664795, 0.13131916522979736, -1.3618570566177368, 0.009197428822517395, 0.49186187982559204, -1.7983288764953613, -0.7877161502838135, 0.5849089622497559, 0.7324184775352478, 0.14956845343112946, 0.13522639870643616, -1.0802801847457886, -0.018277030438184738], [0.595488429069519, -0.3800067603588104, -0.41082513332366943, -1.4590269327163696, 1.9818886518478394, -0.4546453654766083, -0.5666159987449646, -0.659488320350647, 1.5625133514404297, 0.0616840235888958, -0.8971825242042542, -0.8861263394355774, 0.5291706919670105, 0.067752905189991, -0.5671545267105103, 0.5089972019195557], [-0.7212976813316345, 1.6184775829315186, 0.27157893776893616, -1.292707085609436, -0.20315255224704742, -0.831217348575592, -0.26301586627960205, -0.8576390147209167, -0.40656614303588867, -0.9433164000511169, -0.049807045608758926, -1.6030484437942505, -1.2163784503936768, 0.2022220343351364, -0.8972168564796448, 1.626707673072815], [-1.1686290502548218, 1.375287413597107, 0.28899767994880676, -0.6053110361099243, 1.0619802474975586, -0.4067865014076233, 0.633304238319397, 0.34550267457962036, -0.27689290046691895, -0.14813560247421265, -0.7453833818435669, -0.6883114576339722, -0.341398686170578, -1.0328011512756348, 0.33923032879829407, 0.8802425265312195], [0.4916771650314331, 0.360636442899704, 0.17654211819171906, 0.17824439704418182, -1.2280131578445435, 0.4294224977493286, -2.4097113609313965, -2.2012596130371094, 0.4912489354610443, 0.8857320547103882, 0.07889816164970398, -1.227343201637268, 0.3575074076652527, 0.11228309571743011, -0.4651709198951721, -1.2278761863708496], [0.7044920921325684, 0.5792807340621948, 0.7888727784156799, -0.6557751297950745, -1.2467693090438843, -0.7543272972106934, -0.4226575195789337, -0.651898205280304, -0.020845122635364532, -0.5976188778877258, -0.37464895844459534, 0.030848369002342224, -1.5509508848190308, 0.9182730913162231, -0.011420526541769505, -0.01780831813812256], [0.1761033833026886, -0.279874712228775, 1.5314708948135376, 0.07410688698291779, 0.20822811126708984, -0.6344364285469055, -0.3040001392364502, -0.34580400586128235, -0.5118165016174316, 1.1642875671386719, 0.5660794377326965, 1.2728166580200195, -0.49933379888534546, 0.7951135039329529, -1.1154943704605103, 1.7146580219268799], [2.4732160568237305, -2.4289982318878174, 0.2932099997997284, 0.6654711961746216, -0.2704603672027588, 0.39168283343315125, -1.311996579170227, 1.1225181818008423, 1.0329948663711548, 1.5159131288528442, -0.32251793146133423, 0.10609927773475647, -0.40551987290382385, -1.6623696088790894, -0.7342094779014587, -0.2599128186702728], [0.7286458611488342, -0.7356008291244507, -0.33973249793052673, 1.4052269458770752, 1.3670274019241333, -0.5329135060310364, -1.1279083490371704, -0.7542614936828613, -0.7580870389938354, -0.2619117498397827, 1.2235097885131836, -0.5802081227302551, 0.5709948539733887, -1.0501381158828735, -1.459639549255371, -0.022008780390024185], [0.16347308456897736, 1.4137123823165894, -1.1241570711135864, 0.10062158852815628, -2.173320770263672, 2.7340445518493652, 0.46902868151664734, 1.4933772087097168, -0.6446773409843445, 1.1861989498138428, 0.22772161662578583, 0.024921981617808342, -0.3739107847213745, -0.028686154633760452, 0.04098673537373543, -0.04788615554571152], [-0.07044564187526703, -1.329604148864746, -0.7462408542633057, 1.5271776914596558, 0.12296763062477112, -0.6925225257873535, -0.8354539275169373, 1.0876853466033936, 0.7945120334625244, 0.08022620528936386, 0.894790768623352, -0.02069757878780365, -0.09702856838703156, 0.9426939487457275, -0.18745172023773193, -1.001271367073059], [-0.5856838822364807, 0.23361293971538544, -0.7978371381759644, 0.06593585014343262, -0.965420663356781, 0.784080445766449, -0.1682661771774292, -0.345840722322464, 0.19885675609111786, -0.568150520324707, -0.1607162207365036, 0.3307247459888458, 1.1444298028945923, 0.6073564291000366, -0.02783219888806343, -1.1860783100128174], [-1.8073824644088745, -0.509067952632904, -0.7741835713386536, -0.8914722204208374, -0.45913180708885193, 1.1973741054534912, 1.7202000617980957, 0.682829737663269, -1.2347468137741089, -0.20797990262508392, 0.857926070690155, -0.30929985642433167, -1.0130946636199951, -0.5522801280021667, 1.5374714136123657, -0.505279541015625], [2.0166168212890625, 0.049712810665369034, 2.3521974086761475, -0.3938780725002289, 0.4066573977470398, -1.3019592761993408, -0.22777076065540314, -1.3798627853393555, -0.3346044421195984, 0.007557789795100689, 0.973744809627533, -0.5613633394241333, -1.0572973489761353, -0.290587455034256, -1.697609305381775, 0.4104686975479126], [1.644145131111145, 2.60604190826416, 1.7714380025863647, -0.431692510843277, -1.00387704372406, -0.19538599252700806, 0.26847293972969055, -0.4987276494503021, -0.4433605372905731, 1.7127019166946411, -0.5796970725059509, 0.5729553699493408, 0.6030345559120178, -0.2614459991455078, -0.10862047225236893, 0.8500170707702637], [-0.7932007908821106, -0.22763855755329132, -2.473186492919922, -0.06735479831695557, -1.3387956619262695, -0.31689706444740295, -0.6570969820022583, -0.8091338276863098, -1.2352839708328247, -1.3036738634109497, 1.1060322523117065, -1.217160701751709, 2.2678284645080566, -1.2261450290679932, -0.17274722456932068, 0.35021960735321045], [0.17507950961589813, -0.07592587918043137, 0.1365250051021576, 1.0423613786697388, 0.8181489706039429, -0.8522696495056152, -0.3355240821838379, 0.14036113023757935, 0.46023640036582947, -0.0019908640533685684, -0.11356420069932938, 0.762401282787323, 0.09096229821443558, -1.1665419340133667, 0.14233838021755219, -0.5943528413772583], [0.21046310663223267, 0.902368426322937, -0.2092207372188568, -0.21266743540763855, 0.4608015716075897, 1.040185809135437, -0.5131694078445435, 0.619136393070221, 1.6590076684951782, -2.237865686416626, -1.352421760559082, 1.572009563446045, 0.07385954260826111, -0.7046142220497131, 0.23026207089424133, 0.1917262226343155]]]], "dtype": "float32"}, {"__nd__": [[[[-0.3477587103843689, 1.1351977586746216, -1.1724717617034912, -0.060395993292331696, -1.1256744861602783, 0.5584741830825806, -1.5332393646240234, 1.3305784463882446, 1.6059633493423462, -0.047179874032735825, -1.6343008279800415, 0.7779059410095215, 0.15686142444610596, 0.7698484659194946, -1.255402684211731, 1.4547362327575684], [0.15690910816192627, -1.7221163511276245, 0.772402286529541, -1.199449062347412, -0.7027536630630493, -0.8989793062210083, 0.9854040145874023, 1.3195979595184326, -0.17078466713428497, 0.4468439519405365, -1.9224309921264648, -0.3320707082748413, -1.1704237461090088, -1.290549874305725, -1.1436084508895874, 2.6592116355895996], [0.05426940321922302, -0.8287556767463684, -0.46688327193260193, 0.33961376547813416, 0.8432217240333557, 0.15840201079845428, -1.4913837909698486, -0.43891316652297974, 0.9146648645401001, 0.9833434224128723, 0.9349918365478516, 1.4681135416030884, -0.44552525877952576, 0.01961376518011093, -0.5773783922195435, -0.252360999584198], [-1.455292820930481, -0.594575822353363, 1.3874870538711548, -1.6532604694366455, 0.7387562394142151, 0.06741468608379364, 1.348235011100769, -0.5381307601928711, 0.6667723655700684, 1.2500805854797363, 0.24140037596225739, 0.1625189185142517, -0.3079259395599365, 0.5631109476089478, 1.2470148801803589, -0.7817544937133789], [0.6460810303688049, -0.2700987458229065, 0.5712381601333618, -1.8441946506500244, -0.2578870952129364, 0.5228337049484253, -0.17737272381782532, 0.5325812101364136, 0.248552605509758, -2.9457948207855225, -0.701629102230072, -0.4293884038925171, -0.390524297952652, -1.8122745752334595, -0.39230117201805115, -0.6428139209747314], [-0.1022765040397644, -0.32053419947624207, 0.9651353359222412, -0.7489327192306519, -0.04251696169376373, 0.26570504903793335, -0.1721617579460144, 0.18866342306137085, -0.11916330456733704, 1.6684279441833496, -0.0745033249258995, 1.2019463777542114, -0.25906386971473694, 0.6492882966995239, -0.18751917779445648, -0.33978697657585144], [-0.6182916164398193, 0.18831473588943481, -1.562533974647522, -1.2244434356689453, -0.24259231984615326, -1.8969917297363281, 0.24529694020748138, 1.5803310871124268, 0.6534358859062195, 1.5176862478256226, 2.1435253620147705, -0.6290318965911865, -0.5722147226333618, -0.15019862353801727, -1.4056854248046875, -0.5065937042236328], [-0.4146438539028168, 1.1884222030639648, 0.1979389190673828, -0.9524566531181335, -0.6774844527244568, 0.04285847395658493, 0.7337731122970581, 0.17348597943782806, -1.1084741353988647, 0.6364173889160156, 0.3394891619682312, -1.493098497390747, 0.9406834840774536, 0.9948481321334839, -0.03423541411757469, 0.10599752515554428], [0.08729305863380432, 0.3884156346321106, 0.7054958343505859, 0.22942903637886047, -1.4899049997329712, -0.14465828239917755, 0.38899314403533936, 1.3296465873718262, 1.0895965099334717, -0.36202114820480347, 0.929327130317688, -0.3758343458175659, 0.4114268720149994, 1.7811588048934937, -2.65633225440979, 0.7055037021636963], [-0.3604351878166199, 1.0935096740722656, 0.2968296706676483, 0.24910280108451843, 0.45193248987197876, -0.19276946783065796, -0.18252502381801605, 0.426152765750885, 0.537871241569519, -1.0306264162063599, -1.3052045106887817, 0.7867745161056519, -0.31896162033081055, 1.0364655256271362, -0.3452414274215698, 2.9131336212158203], [-0.3649686872959137, -0.9748628735542297, -0.39287030696868896, 0.6811378002166748, 1.1275060176849365, 0.043438032269477844, 1.0258519649505615, -0.5681594014167786, 0.38146334886550903, -0.2953322231769562, -0.3672391176223755, 0.9260938167572021, -1.1685864925384521, -0.44830191135406494, 0.9049286842346191, -1.2659071683883667], [0.3309350311756134, 0.9533856511116028, 2.4345340728759766, -1.0493875741958618, -0.3119541108608246, -0.8392150402069092, -1.4353595972061157, 0.6902679204940796, -0.18168684840202332, 0.18255944550037384, -1.0414987802505493, -0.003399185836315155, 0.015119800344109535, -0.9072312116622925, 0.16129960119724274, -0.2554164528846741], [-0.7227466702461243, 1.4681380987167358, -0.5790901780128479, 0.301141619682312, -0.6985600590705872, -2.5196123123168945, 0.1839386522769928, -0.8228990435600281, -0.2284715473651886, -1.0225309133529663, 0.18341204524040222, -0.61121666431427, -0.2616509199142456, -0.0329912044107914, 0.4525078535079956, -0.10488928109407425], [-0.4698367118835449, 1.5306599140167236, 0.9409412741661072, -0.2416314333677292, -0.33616161346435547, 0.6366376876831055, -0.6100145578384399, 1.8604989051818848, 0.8799805045127869, 0.26755350828170776, -2.1538712978363037, -0.6609747409820557, 1.0825735330581665, -0.0694306492805481, -1.3701709508895874, 0.3135940730571747], [-0.466566801071167, 0.2888714075088501, 1.1459109783172607, 1.0576200485229492, 1.7012248039245605, -0.21526794135570526, -0.6580014824867249, -0.43031251430511475, -1.6108351945877075, -2.512300491333008, -0.0728091150522232, -1.826614499092102, 0.8610690236091614, -0.8277296423912048, -0.684046745300293, -0.8860406279563904], [0.7275274991989136, -0.6474321484565735, 0.41991564631462097, 0.5582130551338196, 0.3246384859085083, 0.7298229932785034, -0.7990005612373352, -0.008779003284871578, 0.8275399804115295, 0.5141094326972961, -0.6913857460021973, 0.5180770754814148, 1.0656810998916626, -0.5763044953346252, -0.21287965774536133, 0.7352984547615051], [0.6914079189300537, -0.07206077128648758, 1.1165027618408203, -0.23601706326007843, -0.26706117391586304, -0.7256082892417908, 1.2699856758117676, -0.1027381494641304, -1.4974627494812012, 0.6656116843223572, 1.1915498971939087, -0.9429987668991089, -0.28569307923316956, 0.7730205059051514, 1.016247034072876, 0.09020490199327469], [-0.5480382442474365, 0.9947953820228577, -1.758216142654419, -0.9724995493888855, -0.974300742149353, 0.47281700372695923, -0.29514941573143005, 1.7530858516693115, 1.5084799528121948, -1.6106271743774414, -1.4708995819091797, 1.8044512271881104, -0.8963301181793213, -1.0510680675506592, 0.9219262599945068, 0.799591064453125], [-0.09708791226148605, -0.006330667994916439, 0.1406809687614441, 0.20524606108665466, -0.02997373603284359, -1.0923012495040894, 0.8955515623092651, 1.3856873512268066, 1.3249380588531494, 2.2298583984375, 1.903289794921875, -1.5460574626922607, -0.49416235089302063, 0.02315661497414112, -0.357412189245224, -0.3673875331878662], [-1.6142287254333496, -0.185331791639328, -0.4580133855342865, -0.7410127520561218, 1.0044009685516357, 1.3842854499816895, -1.6220974922180176, -0.023645147681236267, 1.1380833387374878, 1.5394326448440552, 0.2799999415874481, 0.5501680970191956, 0.4330785274505615, 2.323460817337036, -0.46900928020477295, 0.48415833711624146], [-0.5207771062850952, -0.7515568137168884, -0.860710859298706, -0.055347587913274765, 1.0396575927734375, 0.016153568401932716, 0.0158709529787302, 0.2618084251880646, 0.39447158575057983, -0.9332196116447449, 1.9223872423171997, -1.2472152709960938, 1.8814142942428589, 0.226599782705307, 0.04311119765043259, -0.48986077308654785], [-0.6952133178710938, 0.4893336594104767, -1.9642966985702515, 2.364396333694458, 0.5589250922203064, -0.3218187093734741, 1.221937656402588, -1.0544135570526123, 0.923781156539917, -0.05938960239291191, 0.3377217948436737, 1.6907051801681519, 0.21406932175159454, 2.711611747741699, 0.04211066663265228, 1.1032252311706543], [1.2492378950119019, 0.9030095338821411, 1.0446943044662476, 0.22172677516937256, 0.2748428285121918, 0.03902077302336693, -0.9686636924743652, -0.5292930006980896, -0.1616913229227066, 0.056643787771463394, 0.009237321093678474, -1.0989644527435303, -0.6529669761657715, -0.8547690510749817, 0.7472813725471497, 0.21898159384727478], [0.7302605509757996, -0.3088238835334778, 0.09863302856683731, -0.01178685948252678, 0.22955812513828278, -0.4735342860221863, 0.011674962937831879, 0.36948466300964355, 1.2340461015701294, 0.9631996750831604, -1.3873815536499023, -0.2569928467273712, 1.1983273029327393, 1.1095932722091675, 1.0977920293807983, -1.1494570970535278], [1.971897006034851, 1.6525958776474, -0.07262612134218216, 0.542197585105896, -0.7067819237709045, 1.481126070022583, -1.686967372894287, 0.7053986191749573, -0.03130030632019043, 0.8378163576126099, -1.9287817478179932, 0.3093283474445343, -1.490824818611145, -0.79487144947052, 0.014765392057597637, -0.8847799897193909], [-0.29182809591293335, 0.8083825707435608, 0.5883147120475769, 0.9060648679733276, 0.5035146474838257, -1.3158299922943115, 0.6183932423591614, 0.3182920515537262, 1.1042290925979614, -0.9210225939750671, -0.015156948007643223, -0.5733786821365356, -0.41532811522483826, -0.2716131806373596, -1.2718616724014282, -0.8075479865074158], [-1.2289490699768066, 0.38111889362335205, -0.18556731939315796, 0.5524625778198242, -0.1908658742904663, -1.2024308443069458, -0.35680705308914185, -1.4002189636230469, -0.47151222825050354, 0.4879191219806671, 1.3009719848632812, 0.32614457607269287, 0.3897004723548889, 0.4031980037689209, 1.3211922645568848, -1.9921824932098389], [-0.5084649324417114, -0.3825419247150421, 0.14872080087661743, 0.017896199598908424, 0.3030189573764801, -0.05995940789580345, 0.25014618039131165, 1.018110752105713, -0.8735786080360413, -1.4301378726959229, 0.38522905111312866, -0.9340893626213074, -1.653818964958191, 0.520408570766449, 1.5815703868865967, -0.3313627243041992], [-0.8283421397209167, -0.12015708535909653, 0.21140599250793457, 0.27222445607185364, -0.7583075165748596, 0.7856866121292114, -0.8121389150619507, 0.20756635069847107, -1.5598139762878418, -0.7130118608474731, -0.8657357692718506, -0.2773689031600952, -0.37184059619903564, -0.5854317545890808, 1.1045581102371216, -0.12730015814304352], [0.9137268662452698, -0.5396634936332703, 0.8940178155899048, -1.0232819318771362, 1.7326109409332275, 0.6778402924537659, 0.05745861306786537, -0.5148126482963562, 0.09728136658668518, -0.6289576888084412, -1.100363850593567, -0.7432474493980408, 0.9058812856674194, 0.16912128031253815, 0.5718235373497009, 1.5102429389953613], [0.21359407901763916, -1.4084033966064453, 1.500174880027771, -0.9462065696716309, -0.5477582216262817, -0.47860872745513916, 0.6305756568908691, -0.4896514415740967, 0.5901010632514954, -1.3294557332992554, -0.1391977220773697, -0.27836042642593384, 1.8235130310058594, -0.030074160546064377, -0.8908602595329285, -0.7882776260375977], [1.9803798198699951, -0.4864231050014496, -2.139580488204956, 1.2818719148635864, 1.3925093412399292, 0.8964081406593323, 1.4264329671859741, -1.5753990411758423, 0.3750864267349243, -0.04741571843624115, -0.0649745985865593, 0.26328906416893005, -1.2117568254470825, 0.5637369155883789, 0.6261695027351379, 0.35618436336517334], [-1.6334260702133179, 0.876518726348877, 0.30210524797439575, -0.28308069705963135, -0.8876720070838928, -0.5067449808120728, -1.4145210981369019, 1.407650351524353, 0.04562654346227646, 1.3658424615859985, 0.07802879810333252, -2.2533318996429443, 1.9350472688674927, -0.6030352115631104, -0.7093380689620972, -0.39141619205474854], [0.29572203755378723, -0.5647414326667786, 0.5833916068077087, 1.2665594816207886, -1.9338083267211914, 0.5478413105010986, -1.761735439300537, -1.0031672716140747, 2.238358497619629, -1.9143189191818237, 1.7804509401321411, 1.9497489929199219, -0.4357236623764038, -1.4409130811691284, -0.7831922769546509, 0.3577955961227417], [-0.3080213963985443, -2.8692996501922607, 1.624629259109497, 0.8287776708602905, -1.239370584487915, -0.4017912447452545, -0.5579759478569031, 0.6887428760528564, 0.8695886135101318, 0.384227454662323, -0.1000032126903534, 0.6665736436843872, 0.12771070003509521, -1.3438555002212524, -0.33941224217414856, -1.3776884078979492], [-0.1080726683139801, -2.3560006618499756, 0.13922467827796936, 1.0969464778900146, -0.4435489773750305, 0.15415224432945251, 1.459912896156311, -0.27028870582580566, 0.21845974028110504, 0.4595547020435333, 0.4289160966873169, 0.28772416710853577, -0.16397175192832947, -0.8356380462646484, -2.033189535140991, 0.42134326696395874], [-1.064451813697815, 1.0496978759765625, 0.17485815286636353, 0.9127991795539856, -0.7329766154289246, 0.07769065350294113, 1.3949612379074097, 3.149986982345581, 0.3967069983482361, -0.03723502531647682, 0.27113255858421326, -2.0753016471862793, 0.8782730102539062, 1.1707136631011963, -0.7441587448120117, 1.225152611732483], [0.2505965232849121, 0.24751771986484528, 1.5014339685440063, -1.1348187923431396, 0.8545037508010864, 0.15807847678661346, -1.1207337379455566, 0.5660768151283264, 0.34911274909973145, 0.8564654588699341, -0.9194267988204956, 1.9166597127914429, 0.27384036779403687, 0.7952496409416199, 1.0167745351791382, 1.1445934772491455], [2.4101648330688477, -0.8433612585067749, -0.31817957758903503, 0.16078093647956848, -1.037778377532959, 0.7400898337364197, -1.3605812788009644, 0.7451105117797852, -0.10735847055912018, -0.2156190276145935, -0.3343380093574524, 0.12663552165031433, 0.8690540194511414, -1.3787908554077148, -0.23522678017616272, -0.5006142854690552], [-1.5481479167938232, 2.153710126876831, -1.1341311931610107, 1.212110996246338, -0.24631337821483612, -0.7142740488052368, 0.21861590445041656, 0.08980884402990341, -1.4053610563278198, -0.13350412249565125, 1.6419507265090942, 0.7703648805618286, -2.415045976638794, 0.09427598863840103, -1.2797571420669556, -0.7635713815689087], [-0.09589830040931702, 0.0457458421587944, -0.10222332179546356, -0.34499043226242065, -1.4548296928405762, -0.18947473168373108, 2.0665783882141113, -1.025032639503479, -0.5740215182304382, 0.7834168076515198, -0.5612953305244446, -1.2587167024612427, -0.04323992505669594, -0.9498697519302368, -0.38818928599357605, -0.03415890410542488], [-0.5585857033729553, 0.2763228416442871, 0.3159295618534088, 0.027233591303229332, -0.2102176994085312, 1.122933030128479, 0.27233362197875977, -0.8552871942520142, -1.275505781173706, -1.8489716053009033, -0.9908117055892944, -1.079156517982483, -1.3611758947372437, -0.4189022183418274, 0.38080647587776184, -1.0939239263534546], [-0.7858083844184875, -0.8608941435813904, -0.8762958645820618, -0.4981195330619812, -0.3938189446926117, -1.9355212450027466, -0.5788702964782715, 0.060793567448854446, 0.858100950717926, -1.9517817497253418, 1.082429051399231, -0.5274598002433777, 1.0054088830947876, 0.04018252715468407, 0.5374941229820251, 1.042754054069519], [-0.6234230995178223, 0.3134690523147583, 0.06831123679876328, 0.576364278793335, -0.11711341887712479, 0.8348449468612671, -1.6854618787765503, 0.567596435546875, 0.1262027621269226, -2.212965250015259, 0.8068020343780518, 0.7567835450172424, 1.3802845478057861, 1.4807442426681519, 1.0991278886795044, -1.590349555015564], [0.3380230665206909, 0.12103112041950226, 1.2035881280899048, 1.4022297859191895, 0.4904872477054596, -1.1329345703125, 0.6542523503303528, 1.9150831699371338, 0.550848662853241, 0.085227832198143, -0.5953713059425354, -0.7548704743385315, -0.9624438285827637, 0.7625935077667236, -1.4525738954544067, 1.880617618560791], [0.26170042157173157, 0.01966818980872631, 0.2771787941455841, -0.2358001470565796, -0.33186954259872437, 1.0245697498321533, -0.53908371925354, -0.5647140741348267, -1.1787115335464478, 0.39033272862434387, 0.3586561977863312, -0.07304488867521286, -0.9676889777183533, -0.5690983533859253, 1.0871310234069824, -0.5893135666847229], [-0.7090213298797607, 0.7979806065559387, -0.8241353631019592, 0.4127095937728882, 0.43364521861076355, -0.4863661527633667, 1.2678532600402832, 0.48128369450569153, -0.8836742043495178, 0.2300463765859604, -0.928648829460144, -0.5855201482772827, -0.009943371638655663, 0.1654966175556183, 0.36224865913391113, 0.8199384808540344], [-0.09639249742031097, 0.7394700646400452, -1.5051301717758179, -1.3553611040115356, 0.8166968822479248, 0.046034738421440125, 1.6349395513534546, -1.165461778640747, -0.7971428036689758, -0.5273438692092896, -0.6233314871788025, -1.387020230293274, 1.5570210218429565, -0.3853681981563568, 0.15194837749004364, 0.018247423693537712], [-0.9988908171653748, -1.3781441450119019, 1.4166655540466309, 0.24581049382686615, 0.7883925437927246, 2.320817470550537, -1.0037728548049927, 1.8766828775405884, -0.7258834838867188, -1.1916965246200562, -0.7372320294380188, 0.5485026240348816, -0.013309129513800144, -0.19628211855888367, -0.4012449383735657, -0.6105980277061462], [-0.0755959078669548, -0.2507482171058655, -0.20775890350341797, 2.097942590713501, -0.007488423027098179, -0.06825236976146698, -0.6108099818229675, 1.551953673362732, 0.3368583023548126, -1.6164768934249878, 0.4713418185710907, -0.8950070738792419, -1.0921850204467773, 1.7869831323623657, 1.3144084215164185, 1.3317784070968628], [1.1081165075302124, 0.26153364777565, 1.9230973720550537, 0.1607513278722763, 0.6233657002449036, 1.3863085508346558, 2.2722713947296143, 1.270713210105896, 0.5744290947914124, 0.49121662974357605, -0.20212498307228088, -3.1819732189178467, -1.0863099098205566, -0.7177525758743286, -1.3085296154022217, -1.151231288909912], [1.202452540397644, -0.7997497320175171, -0.962986409664154, -0.3966887295246124, -0.7904275059700012, 1.9598021507263184, -1.2405503988265991, 0.41779735684394836, 1.1155648231506348, -2.350162982940674, -1.039287805557251, 0.5103009343147278, 0.3360672891139984, 0.8096204996109009, 0.8089005351066589, -0.9668203592300415], [0.9527938961982727, -1.472630500793457, 0.20386624336242676, -0.6435145735740662, -0.4137987792491913, -0.20579403638839722, 0.02047187089920044, 1.2694517374038696, 0.026590250432491302, 0.00974179245531559, 0.23521079123020172, -0.11375115066766739, -1.2321999073028564, -0.23965033888816833, -0.579997718334198, 2.427190065383911], [0.8552350401878357, 0.2573482394218445, -0.005421833600848913, -0.24594275653362274, -1.586836814880371, -2.104556083679199, 0.837232768535614, -1.1308239698410034, -1.3208622932434082, 0.7389135360717773, 0.7610483765602112, 0.12658049166202545, -1.4190328121185303, -0.40190041065216064, -0.8492687940597534, 0.35672318935394287], [0.25742271542549133, 0.4724472761154175, -0.4168269634246826, -0.3327730596065521, 0.04239515960216522, -0.22179445624351501, 0.9593522548675537, 2.1284351348876953, -1.2014352083206177, -1.1237237453460693, -0.0387876033782959, -0.002644515596330166, -1.3262503147125244, 0.11240068078041077, -0.2107681781053543, 0.09956420212984085], [1.3598976135253906, 1.7707545757293701, 0.954349935054779, 0.7056225538253784, -1.401297688484192, 1.0538488626480103, 0.17189358174800873, 1.2720216512680054, -0.4030788242816925, -1.834903597831726, 0.24185675382614136, 0.9425528645515442, 0.8239012360572815, 1.4077208042144775, 0.18792609870433807, -1.6223976612091064], [-1.832700252532959, -1.9885658025741577, 0.7345412969589233, -0.22440610826015472, 0.8160857558250427, 1.393304467201233, 0.600391149520874, 0.18234139680862427, 0.7510932683944702, 0.05136661231517792, 0.004974811803549528, -1.4190033674240112, 0.9282814860343933, 2.016709327697754, 0.6077548265457153, 0.4402194917201996], [0.8439367413520813, 1.042060136795044, 1.6108685731887817, 0.10855454206466675, -0.17300182580947876, -1.1383094787597656, -0.18045486509799957, -1.729485034942627, 0.8548465967178345, -1.401679277420044, -1.0025757551193237, -0.11098900437355042, -0.09135542064905167, 1.7401933670043945, -2.0893168449401855, 0.609814465045929], [-0.38642552495002747, -0.49097776412963867, 0.7692749500274658, -0.47183650732040405, 0.14364495873451233, 0.17011676728725433, 1.8615247011184692, -0.2730069160461426, -0.34305423498153687, -0.16733773052692413, -0.34674614667892456, 0.2909007966518402, 0.46714380383491516, 0.21551650762557983, 1.0100892782211304, -1.0384631156921387], [0.9534372687339783, -0.23834049701690674, -1.3542444705963135, -1.5276618003845215, -0.16496653854846954, -0.20788051187992096, 1.7459267377853394, 0.05658453330397606, 0.4764968156814575, -0.8796457052230835, -0.08113458752632141, -0.23026467859745026, 0.004923769272863865, 0.608974039554596, 0.8116799592971802, 0.8639718890190125], [-0.23005928099155426, -1.2143474817276, -1.2708815336227417, -0.5993252992630005, -0.039670225232839584, 0.0765828862786293, 0.24189580976963043, 2.6165173053741455, -0.5282339453697205, -0.5828468799591064, -0.1872488111257553, -1.951988697052002, 0.6559839844703674, -0.18734680116176605, -0.6756044626235962, -1.165992259979248], [-1.1441211700439453, 0.10716089606285095, -1.508034110069275, -0.642055869102478, 0.6790066957473755, 0.1548781543970108, 0.029792776331305504, 2.05049991607666, -1.4985768795013428, -1.3488495349884033, -0.9688904881477356, 0.2985934019088745, 1.1645246744155884, 0.4771946966648102, -0.3472268581390381, 1.3243036270141602], [1.3227235078811646, -0.8119819760322571, 0.909332811832428, 0.11103823781013489, 0.5720774531364441, 2.0624067783355713, -0.28436121344566345, -1.227704644203186, -1.0999952554702759, -0.7025666236877441, 1.1532384157180786, 1.8935143947601318, -1.278144359588623, 0.3502819240093231, 0.7524001002311707, -1.1517369747161865], [1.2893444299697876, 0.8488454222679138, -0.8818499445915222, 1.2116246223449707, 0.32130318880081177, 1.2760286331176758, -0.42848101258277893, -0.12589995563030243, 0.05522192642092705, 0.0727376937866211, -0.44247573614120483, 1.0057505369186401, -0.7307270169258118, 0.4120118021965027, -1.230809211730957, 0.2694200575351715], [0.6379985809326172, 2.335709571838379, -0.6427366137504578, -0.23298680782318115, 0.6136143207550049, -0.5094049572944641, 0.21840700507164001, -0.2707185447216034, 1.208245873451233, -1.2688955068588257, 0.6710903644561768, 0.48191091418266296, 0.9903212189674377, 1.0675557851791382, 0.6295899152755737, -0.8167622685432434], [-2.4437384605407715, 0.5277374982833862, 2.435410737991333, 0.1908501833677292, 2.6944055557250977, 0.5979834794998169, -0.6385928988456726, -0.7942132949829102, -0.8484609127044678, 0.11828863620758057, 0.6821801662445068, 0.63691246509552, 2.198148250579834, 0.3609955906867981, 0.2545713782310486, -0.734350323677063], [-2.3002562522888184, 0.1870056688785553, -0.9470012784004211, 0.19537419080734253, -0.6979648470878601, 0.33360937237739563, 1.192291021347046, 0.7676479816436768, 0.9320163726806641, 0.28786322474479675, -1.3219468593597412, 0.8103855848312378, 0.567213237285614, -1.1722724437713623, -0.3743709921836853, -0.35014328360557556], [0.9902762770652771, 1.8037110567092896, 1.7194383144378662, 0.708605170249939, -0.20672374963760376, 0.09859532117843628, 1.3937525749206543, 1.9805773496627808, -0.10036037862300873, 0.26150915026664734, 1.9284878969192505, 0.16037048399448395, -3.1035563945770264, 1.3682198524475098, 1.0228160619735718, -1.1907007694244385], [-0.5846240520477295, 0.19591611623764038, -0.9207226634025574, -0.3348202407360077, 0.5425940752029419, 0.34991881251335144, 0.11171062290668488, 0.6039888858795166, 1.77835214138031, 0.017925094813108444, -1.2213830947875977, 0.04843810945749283, 2.4420111179351807, -1.8196873664855957, -1.3112062215805054, -0.8939399123191833], [-0.7210665345191956, -0.014630413614213467, -0.4211235046386719, -0.16885906457901, 1.2171376943588257, -1.0454256534576416, 1.3234726190567017, -2.0554776191711426, -0.45610037446022034, -1.0599114894866943, 1.3948793411254883, 0.14061060547828674, 0.8269839286804199, 0.58502596616745, -2.1994130611419678, -1.1854692697525024], [-0.1960531771183014, -1.5177510976791382, 0.6628561019897461, 1.6732189655303955, 0.9548859000205994, 0.06547079980373383, -1.7145124673843384, -1.8588335514068604, 2.644854784011841, 1.3242006301879883, -1.3459399938583374, -0.17394760251045227, -0.39522501826286316, -0.7591055035591125, -0.1975974589586258, -0.49377313256263733], [-0.2913736402988434, -0.014158559031784534, -0.09410315752029419, -0.5913339257240295, 1.4527101516723633, 1.3930789232254028, 2.227961301803589, 1.4652371406555176, 0.11658530682325363, -0.1620742678642273, 0.2049936056137085, -0.14998140931129456, -0.45769524574279785, 1.3246121406555176, 0.045694343745708466, -1.9270579814910889], [0.1223849430680275, 1.0281264781951904, 0.40333104133605957, -1.2573349475860596, -0.32171472907066345, -1.9737647771835327, -1.1394208669662476, -0.8176489472389221, 0.07945820689201355, 0.27453070878982544, -0.9705356955528259, -0.15183113515377045, -0.6420989036560059, -0.6391345858573914, 0.47033146023750305, 0.30161017179489136], [0.5990909337997437, 1.178521990776062, 0.8042289614677429, -1.8013947010040283, -0.04365234822034836, 1.258833646774292, 0.2998146712779999, -0.5547888875007629, -1.4831526279449463, -1.2586064338684082, 1.4420645236968994, 1.871807336807251, 1.0022673606872559, 0.4033178389072418, 0.745892345905304, -0.30222299695014954], [0.783536434173584, 0.47191694378852844, 0.45942747592926025, 0.3180059492588043, 0.5192133188247681, 1.102481484413147, -0.1507333219051361, 0.7270886898040771, -0.7594103813171387, -0.8430196642875671, 0.25863879919052124, -1.3278192281723022, 0.621943473815918, 0.6124492883682251, -0.9736514091491699, 1.1304038763046265], [-0.48614901304244995, -0.21746578812599182, 0.2502044141292572, -1.38554847240448, 0.12326112389564514, 0.10406568646430969, 0.16735075414180756, 0.5886519551277161, 0.9455108642578125, -0.5621095895767212, 0.06910412758588791, -0.5900202989578247, 0.05709679797291756, -0.8287695646286011, 0.2683790624141693, 0.7705428600311279], [-0.23114441335201263, 0.6192699074745178, 1.2722649574279785, 1.3659703731536865, -0.30906930565834045, -0.7101008892059326, -0.4013066589832306, 0.652332067489624, -0.7395345568656921, 1.4909695386886597, -0.833560585975647, -0.0035710863303393126, 1.0093083381652832, 0.03453606367111206, -2.1617491245269775, 1.21544349193573], [-0.8265560269355774, 0.11738889664411545, -0.49974575638771057, 0.8325714468955994, -0.42968717217445374, -0.9022040963172913, -1.0962121486663818, -0.6033182740211487, -0.753516435623169, 1.3665127754211426, -0.8454551696777344, 1.2738709449768066, -0.5561404228210449, -0.6652547121047974, 0.5511170625686646, -0.5589005947113037], [-1.0659199953079224, 0.6706284284591675, 0.05216236785054207, 1.6565347909927368, -1.695181131362915, 1.0662577152252197, -0.4787434935569763, 0.7542374730110168, -0.22613365948200226, 0.09910225868225098, 1.6268192529678345, -1.4859092235565186, 0.623177707195282, -1.7186015844345093, -2.586549758911133, -0.6347616910934448], [0.22705918550491333, 0.5971773266792297, 0.567414402961731, 0.8378515243530273, 1.1914311647415161, -2.6264491081237793, 0.3973698914051056, -1.424145221710205, -0.7942142486572266, -0.7682948112487793, -0.7523486614227295, -0.4447856843471527, 1.3151373863220215, -0.10220567882061005, 0.8934640288352966, 0.5551102757453918], [0.8403207659721375, -0.7923531532287598, -1.376552939414978, 1.1711854934692383, -0.17458990216255188, 1.1649107933044434, 0.3134436309337616, 0.47435683012008667, -0.9354580640792847, -0.7244316935539246, 0.81349778175354, 0.20651444792747498, -0.9012369513511658, 0.009322081692516804, -0.9715471267700195, -0.9954483509063721], [-1.0278034210205078, 0.6339502334594727, -1.6168075799942017, 1.1485451459884644, 0.46025270223617554, 0.3973049819469452, 0.4835110604763031, -0.022593369707465172, 1.3817344903945923, 1.3079911470413208, -0.4702642560005188, 0.03370482474565506, -0.8637194037437439, -1.9397153854370117, -0.9066193103790283, 0.041312579065561295], [-1.1207000017166138, -1.394960641860962, -0.24246320128440857, 2.065896987915039, -1.1138702630996704, -0.5415138602256775, -1.742395043373108, -0.5998680591583252, 0.4165767431259155, 0.04808976873755455, -0.1552819013595581, -1.7809540033340454, -0.9198591709136963, -0.9143444895744324, -0.15774337947368622, -0.20740462839603424], [-0.917203962802887, -0.6097437739372253, 0.08620303124189377, -0.7245517373085022, -0.3211163282394409, 0.5684423446655273, -0.6617532968521118, -0.7913180589675903, -0.45032215118408203, -0.16747282445430756, -0.7028459310531616, -0.9141055345535278, 0.7301812767982483, -0.24696983397006989, 0.13713254034519196, -0.2920856773853302], [0.6994678378105164, 1.1421540975570679, -1.9118701219558716, -0.6913868188858032, 1.5111407041549683, 0.7986689805984497, 1.394637107849121, -1.1417491436004639, 0.016593510285019875, -1.1613328456878662, -0.9995819330215454, 0.9603275656700134, -0.9643399715423584, 1.5364710092544556, -0.3808053433895111, 0.42236804962158203], [-0.16332697868347168, -0.376370370388031, 0.37610965967178345, -0.6031579375267029, 0.04318538308143616, 0.7522983551025391, 0.83127361536026, 0.3710745573043823, -0.044126421213150024, 0.7981945872306824, -1.2395862340927124, 0.07301177084445953, 0.4433697760105133, -1.4831165075302124, -1.024827480316162, -1.0616915225982666], [-1.6327672004699707, 1.250441312789917, 0.6878125667572021, 0.2362106293439865, 0.8295528292655945, -1.781854271888733, -0.5176188349723816, 1.4015719890594482, -1.2686007022857666, -0.7230980396270752, 1.859622836112976, 3.300570011138916, 0.233402818441391, -0.7208954095840454, -1.7573192119598389, 0.3839287757873535], [0.3970673382282257, -0.35143589973449707, -2.0724036693573, 1.6801165342330933, -0.6082354187965393, -1.0853389501571655, -0.5053861737251282, -0.16007396578788757, -0.6805469393730164, 0.05200587585568428, -1.3785109519958496, -1.1252844333648682, 0.7177082300186157, 0.30931392312049866, 0.42103245854377747, -0.1391705423593521], [-0.1646130084991455, 0.26222869753837585, -0.4579293429851532, -3.121213912963867, 0.6488162279129028, -1.2527989149093628, -0.10441876947879791, 0.0873277336359024, 1.5901762247085571, -0.7329944968223572, -0.12245912849903107, 2.3122408390045166, -0.028292929753661156, -1.264373779296875, 2.070075035095215, 0.11155956983566284], [-0.1727716326713562, -0.25573089718818665, 1.2573245763778687, -0.30472853779792786, -2.237388849258423, 0.8471542000770569, -0.10891149938106537, -0.4255199730396271, 0.40957111120224, 1.2816121578216553, 0.012641366571187973, 1.514439344406128, 1.5225896835327148, -0.475921094417572, -0.6924541592597961, -2.7229020595550537], [1.6439332962036133, -0.42164528369903564, 0.08389365673065186, 0.20212726294994354, 1.1539376974105835, -0.9409369230270386, 0.5460794568061829, 0.3643396198749542, -0.9743422865867615, 0.20130516588687897, 1.346191167831421, 1.5253442525863647, -0.17377720773220062, 0.1436721384525299, -0.017128052189946175, 0.26835691928863525], [0.5032781362533569, -0.5878957509994507, 0.5265870094299316, 0.08954823762178421, 0.5985444784164429, -1.8742140531539917, -0.49762260913848877, 0.7701318860054016, 1.1864383220672607, 1.4156605005264282, 0.7108560800552368, 1.3082478046417236, -1.6368951797485352, 0.09316540509462357, -0.480031818151474, -0.439325213432312], [2.55250883102417, 0.3354549705982208, -0.5959732532501221, -0.3149561285972595, 1.3806567192077637, -0.5175570845603943, 0.321956992149353, 0.9365115165710449, 0.002044579479843378, 1.4042693376541138, -0.11171498894691467, 0.781584620475769, 1.0474059581756592, 0.6811201572418213, 0.33757904171943665, -0.7213723063468933], [-0.16339917480945587, 0.21631981432437897, -2.1062638759613037, -0.595344603061676, -2.2000038623809814, -1.5383946895599365, -0.24303248524665833, -0.7551096081733704, 0.16034035384655, -0.4345971345901489, 0.41207155585289, 0.4992656409740448, 0.6341780424118042, 0.043179940432310104, -1.1104116439819336, -1.6808768510818481], [-0.3714864253997803, 0.5017632246017456, 1.615840196609497, 1.3813291788101196, -1.3044253587722778, -0.051738493144512177, -0.7589229941368103, -0.0736568346619606, -1.8297992944717407, 0.8800026774406433, 0.724010169506073, 0.02460821345448494, -0.6072109341621399, 0.803777277469635, 2.228438138961792, -0.7183882594108582], [0.3247944712638855, 0.21807287633419037, -0.06254370510578156, 0.6620221138000488, -0.5878599882125854, 0.5341727137565613, 0.29365184903144836, -0.860508143901825, 0.395809531211853, -1.5414563417434692, -1.4883285760879517, 0.05486614629626274, -2.743302822113037, -0.3136350214481354, 0.4821333885192871, 0.3230965733528137], [-0.8575579524040222, -0.019876642152667046, -0.32351163029670715, 0.6133106350898743, 0.4900319278240204, 0.10342208296060562, 1.5247480869293213, 1.3482431173324585, -0.7144473791122437, 1.0420300960540771, -0.8140591979026794, 0.4769391119480133, -1.1203135251998901, 0.4664725065231323, -0.577728807926178, -0.37328100204467773], [1.6443674564361572, 0.5776501893997192, 1.051943302154541, 0.37583306431770325, 0.007880441844463348, -0.32457926869392395, 0.6806588768959045, -1.1949201822280884, 1.2249306440353394, -0.5385608077049255, 1.7427842617034912, 0.5459097623825073, -0.5813286900520325, -2.015552043914795, -0.029717840254306793, 0.2940318286418915], [1.2400099039077759, -1.263075828552246, -1.1141437292099, -0.17070215940475464, -1.1124821901321411, 0.05453193560242653, 0.20823536813259125, 0.5690673589706421, -0.5198549628257751, -1.2783234119415283, 0.11506107449531555, -0.22658313810825348, 0.11829224973917007, -0.3318104147911072, 1.2940943241119385, 0.5261975526809692], [1.7903558015823364, -0.9530279040336609, 1.0837514400482178, 0.5468810200691223, 1.6602463722229004, -0.2523908317089081, 0.6908981204032898, 1.3799834251403809, -0.12182144820690155, -0.1644720435142517, 0.06586939096450806, -1.3905013799667358, 2.1539664268493652, -0.48558393120765686, 1.120884895324707, 0.20008786022663116]]], [[[0.5264384746551514, 0.9609838724136353, -0.7655600905418396, 1.5669718980789185, 0.776878833770752, -0.633483350276947, 0.08277176320552826, 0.592110812664032, -0.4000115990638733, -0.7810429334640503, 2.3143436908721924, -0.17136234045028687, 0.9301018118858337, 0.16298554837703705, -1.8059523105621338, -1.5600194931030273], [-0.9332621097564697, -0.18559567630290985, -0.02049669437110424, 0.8364947438240051, 0.037016794085502625, 1.2992711067199707, -0.8456512689590454, -0.7603766918182373, -1.1429555416107178, -1.701564073562622, 0.2013109177350998, -1.1492962837219238, 0.04871667921543121, 0.04462609067559242, -0.38625913858413696, 0.40188321471214294], [-1.5051789283752441, -1.1747738122940063, -0.7739086747169495, 1.1878260374069214, -0.9930825233459473, -0.23697152733802795, -1.0585503578186035, -0.8100903034210205, -0.6999544501304626, -0.2771223187446594, 1.3094456195831299, 1.466464638710022, -0.9157254695892334, 1.5205364227294922, 2.564903974533081, -0.5345767736434937], [1.162922978401184, 0.3402850031852722, -0.2591560184955597, 0.5040013790130615, 0.03700702637434006, -0.0068250480107963085, 1.0372861623764038, -0.35190069675445557, 1.8504767417907715, 0.47998398542404175, 1.915083646774292, 0.8644211292266846, 0.9167661666870117, -0.8715342283248901, -0.3522034287452698, -0.41786763072013855], [0.7281426787376404, -0.0702284723520279, 0.9161118865013123, -0.058645330369472504, 0.549397885799408, -0.4142574667930603, -0.024580249562859535, -0.4921755790710449, 0.8077589869499207, -0.016201937571167946, 0.5118562579154968, 1.9674837589263916, 1.2232930660247803, 0.7023523449897766, -0.5359473824501038, 0.1236821860074997], [-0.5432080626487732, -0.9800943732261658, 0.43778738379478455, -1.057437539100647, 0.6590498089790344, 0.1550317257642746, -0.806207001209259, -0.29181134700775146, -0.13234548270702362, -0.7291472554206848, 1.8402318954467773, 2.4212253093719482, -0.8331514000892639, 1.5380825996398926, -1.1210206747055054, 1.1971724033355713], [-1.0084253549575806, -0.7205125689506531, -0.5269010663032532, 0.9306541085243225, 0.005456522572785616, -0.2829076945781708, 2.5506033897399902, 1.156150221824646, 0.189229354262352, -0.8573461771011353, -0.4899710714817047, 1.3291689157485962, 1.6774035692214966, -1.0940864086151123, -0.4744095802307129, 0.2370973378419876], [1.6165874004364014, 1.4777847528457642, -1.3730580806732178, 0.3923833668231964, 0.7701041102409363, -0.2606174945831299, -0.679887056350708, -0.0347474180161953, -1.462183952331543, 0.8282602429389954, 0.20299556851387024, -1.3625463247299194, -0.18770813941955566, 1.3528680801391602, 2.441737174987793, -0.09186351299285889], [2.470327615737915, 0.4314471483230591, 0.5945879817008972, 1.258143424987793, -1.1175037622451782, 0.8771997690200806, -1.0061753988265991, -0.17247897386550903, -0.5965355038642883, -0.7802546620368958, 0.44346746802330017, -0.7196664214134216, -0.02540411986410618, 1.727120280265808, 0.45474451780319214, 1.978295922279358], [1.9109833240509033, 1.649457335472107, -0.850298285484314, -0.8232324719429016, -2.081197738647461, 0.8665252923965454, 0.5108618140220642, -0.5994790196418762, -1.4944403171539307, -1.3683944940567017, 2.3046631813049316, 1.0901100635528564, -2.678893566131592, 1.2359349727630615, -0.9910565614700317, 0.2834133505821228], [-0.5517691969871521, 0.35097020864486694, -0.4369031488895416, -0.1847754269838333, 0.7575712203979492, -0.07732414454221725, 0.9501577615737915, -1.1113197803497314, -0.4565918743610382, -0.32021158933639526, -0.268460214138031, 0.9645935297012329, -1.3953430652618408, -1.1195580959320068, -1.0149880647659302, 1.176673173904419], [1.0109444856643677, 1.0577752590179443, -0.3647436201572418, -0.6765384674072266, -0.8478482961654663, 0.43117979168891907, -0.573117196559906, 0.3840326964855194, -0.31103014945983887, -0.22841835021972656, 0.005230997689068317, -0.7263473272323608, 1.0356868505477905, 1.2812774181365967, 0.3804127871990204, 1.9240431785583496], [1.5108909606933594, 0.12160076200962067, -0.05959084630012512, 0.23265545070171356, 0.7386595010757446, -1.3879201412200928, 0.7336409091949463, -0.2513499855995178, -0.8460879325866699, -1.2088979482650757, 0.5957406759262085, -0.5357304215431213, -0.7454754114151001, 1.8509602546691895, -2.551015853881836, -0.23649945855140686], [-0.6092138290405273, 0.8948749303817749, -0.8914248943328857, 0.9193309545516968, 1.7823176383972168, -0.0775694027543068, 0.4888315796852112, 2.2985777854919434, -0.46302586793899536, -0.07801159471273422, -0.7517125606536865, 0.692947506904602, -0.6838816404342651, -0.6343634128570557, 0.3725639581680298, 0.7903409004211426], [-1.0757062435150146, -0.08708643168210983, -1.3949424028396606, 0.1287040114402771, -0.520962655544281, 1.060674786567688, 1.2879546880722046, 2.4564120769500732, -0.08455067127943039, -1.0160762071609497, -1.7627685070037842, 0.5192176103591919, -0.9858614206314087, -0.1086997464299202, 0.00860520452260971, 0.026473484933376312], [0.8508628606796265, -1.1454626321792603, 1.4454807043075562, 1.6714264154434204, -0.37284988164901733, -0.593384861946106, 1.062847375869751, 0.3975217938423157, -0.17446216940879822, 0.42714449763298035, -0.04445512965321541, -0.5733084678649902, -0.280821830034256, 0.6076517701148987, -0.5719078183174133, -1.1000018119812012], [0.8338187336921692, -1.3749938011169434, -0.6137100458145142, 1.3625848293304443, 0.028409171849489212, -0.21532315015792847, 1.3896211385726929, 1.169416069984436, -1.2303872108459473, -0.24854139983654022, 0.3761460781097412, -0.7946042418479919, 0.46608516573905945, 0.999119758605957, -0.14657500386238098, 0.5700910091400146], [-1.3826017379760742, 0.26887989044189453, -0.9936061501502991, 1.4307348728179932, -0.010408037342131138, 0.1767861247062683, -0.8909774422645569, -0.6274263858795166, -0.05659172683954239, 0.42549625039100647, -1.129066824913025, 1.0748568773269653, -1.024145483970642, -2.3319571018218994, -0.06372833997011185, -0.25823286175727844], [0.6072583794593811, 0.7420929670333862, -0.8864916563034058, -0.23520860075950623, 0.3852234184741974, 0.22983838617801666, 0.62920081615448, -1.5804362297058105, 0.43195873498916626, -0.2998371124267578, -0.4938754439353943, -0.7239154577255249, -0.5802389979362488, 1.193515658378601, 0.03148288279771805, -1.830080270767212], [-1.12785005569458, 1.1850597858428955, 0.7885907292366028, 0.17361286282539368, -0.40884310007095337, 0.32804757356643677, -0.008208733052015305, 0.2862590551376343, 1.5836859941482544, 1.0112844705581665, -0.21809501945972443, -1.3098536729812622, -2.1054115295410156, -0.3223443329334259, 1.2090691328048706, -0.8592037558555603], [0.10308673232793808, -0.2409840077161789, -0.9691089391708374, -0.14010559022426605, 0.625331461429596, -0.5791015028953552, 0.48109009861946106, 0.5146860480308533, -0.05909222364425659, -3.1452882289886475, 1.3389959335327148, 0.39693790674209595, 0.9186674952507019, -0.028251999989151955, -0.36589089035987854, 1.9353829622268677], [1.1993472576141357, 1.3557130098342896, -0.33642497658729553, -0.3098869323730469, -1.103466510772705, 2.293623447418213, 1.7149754762649536, 0.47663944959640503, 2.8704702854156494, -1.139563798904419, 0.8043788075447083, -0.45121675729751587, -0.34063372015953064, 0.4379189908504486, 0.11591833084821701, 0.6265192627906799], [-1.0357023477554321, 0.6802583336830139, -1.0414894819259644, 0.22610744833946228, -1.707287311553955, 0.3779226839542389, -0.2832169830799103, 2.4652838706970215, -0.23260806500911713, 0.08543195575475693, 0.36425328254699707, -0.1281307190656662, 0.41081494092941284, 2.5370380878448486, -1.6630032062530518, 0.5216479897499084], [-0.6253702640533447, -0.5353548526763916, -0.37740933895111084, 1.369101881980896, 1.11231529712677, 0.8683149218559265, 0.9923553466796875, -0.6209985017776489, -0.7067280411720276, -1.7593389749526978, -1.3432648181915283, 1.0814746618270874, 0.6621056199073792, 1.369702935218811, 1.0527650117874146, -0.7775014638900757], [-0.716755747795105, 0.020074307918548584, -1.2880676984786987, 1.20003342628479, -0.07963822036981583, 1.3642592430114746, 1.4030107259750366, -2.0749869346618652, 0.4771929681301117, 0.20231513679027557, 0.9943550825119019, -1.2966349124908447, -0.5565795302391052, 1.4745255708694458, 0.09194126725196838, -0.27571848034858704], [0.08825443685054779, -0.09978258609771729, 0.8487510085105896, -1.3423653841018677, -1.269973874092102, -0.12022507190704346, 1.1895742416381836, 0.5483492016792297, -0.5425247550010681, 0.4403094947338104, -2.4038286209106445, 1.1085610389709473, -1.555653691291809, -0.11572908610105515, -1.4252408742904663, -2.808727264404297], [0.8558642268180847, -0.9761813282966614, -2.0386791229248047, -0.02043500915169716, 1.5071873664855957, -0.17237098515033722, 0.43267929553985596, -0.8674675822257996, 0.12543122470378876, 1.9740394353866577, -1.8980786800384521, 0.4098052382469177, 0.4616852104663849, -1.2114551067352295, -0.2426798939704895, -0.6362662315368652], [1.4504945278167725, 1.1582220792770386, 0.9180559515953064, -0.32494133710861206, -1.8074175119400024, -0.9207123517990112, -0.782538115978241, -0.6532730460166931, -0.6193444132804871, -0.2774246633052826, -0.2984604835510254, 0.5464093685150146, 0.4044395387172699, -1.7642375230789185, 0.6709093451499939, 1.9513684511184692], [-2.1063003540039062, 0.100550077855587, 0.15301816165447235, -1.8363261222839355, -0.13365209102630615, 0.8845604062080383, -0.607670783996582, -0.3581845164299011, 0.7188615798950195, -0.1088125929236412, 0.5464630722999573, -0.38084787130355835, 0.1991017758846283, -1.4066987037658691, -2.8192272186279297, 0.5914245843887329], [1.5306408405303955, 0.816420316696167, 1.2232238054275513, 0.3359125852584839, 0.9548201560974121, 1.4769903421401978, -1.0546629428863525, -1.2113873958587646, 0.6656379699707031, -0.1691572368144989, 0.06966138631105423, -1.0724564790725708, -0.6181827783584595, -2.2174081802368164, -0.5556608438491821, 0.7043513655662537], [-0.30429983139038086, 0.3975425362586975, 1.364164113998413, -1.6148245334625244, -0.6613015532493591, 2.0025291442871094, 0.08098562806844711, 0.2709972858428955, 1.0384433269500732, -0.37405925989151, -0.3314291536808014, 0.5229375958442688, -2.4037959575653076, -0.00718934740871191, 2.2717654705047607, 0.007489808835089207], [1.3575115203857422, -1.3468445539474487, 1.8158036470413208, 0.438455194234848, -1.5811046361923218, 0.3330339193344116, 1.149359107017517, 0.5807358622550964, -0.0685008242726326, -0.3334771394729614, 0.27762749791145325, 1.0062087774276733, 0.013663571327924728, -0.9614542126655579, -0.279019296169281, 2.7956249713897705], [0.2199985682964325, 0.055512722581624985, -0.5505170226097107, 2.721632957458496, 1.1059770584106445, 0.35500696301460266, 0.9925709962844849, -0.8934872150421143, -1.708821415901184, -1.7508915662765503, -1.6699086427688599, -0.2277049869298935, -0.6159771680831909, -0.023607823997735977, 1.4339134693145752, -2.1902685165405273], [0.9806491136550903, 0.2163444310426712, 1.313197374343872, -0.5609197020530701, -0.7484129071235657, 1.2368074655532837, 0.4190306067466736, 1.0826975107192993, -0.5731909275054932, 0.9023897647857666, 0.6718497276306152, -1.0235321521759033, -0.8077040314674377, -0.5649228692054749, 1.6722373962402344, -0.2655738890171051], [0.2752916216850281, 0.9377992153167725, -0.4137205183506012, 0.03199975937604904, 0.4936264753341675, -1.7323399782180786, -0.9189168214797974, 1.444222331047058, -0.0036568434443324804, -0.2319655418395996, 0.6853755712509155, 0.8475558757781982, 1.3495017290115356, 0.22881612181663513, 0.08859024196863174, -0.498874306678772], [-0.919441819190979, 1.6172491312026978, -1.5550086498260498, -0.573357105255127, -0.49457138776779175, 0.2210320085287094, 0.5090346932411194, 2.217695951461792, 2.0939271450042725, 0.3615487217903137, 0.1906592845916748, 0.2889350652694702, -1.1115148067474365, -1.1358789205551147, -0.7588648200035095, 1.0517494678497314], [-1.0400619506835938, -0.9408407211303711, 0.511428713798523, -1.2721842527389526, 1.903763771057129, 1.4306814670562744, -0.14556202292442322, 1.9029737710952759, 1.3375619649887085, -0.01637260615825653, 0.34217575192451477, -0.7164466977119446, 0.5663966536521912, 0.40856847167015076, 1.0937455892562866, 1.1565005779266357], [0.3826950788497925, -2.4812755584716797, -0.1974283754825592, -0.2448081225156784, -1.2304083108901978, -0.4117436707019806, -0.22625581920146942, 2.0548202991485596, 0.6494737863540649, -1.5061875581741333, -0.11186841130256653, -0.7244070768356323, 1.154941439628601, 0.6824806928634644, -0.38708069920539856, -0.6384623050689697], [-1.2881531715393066, 0.7138086557388306, -0.7628492116928101, -0.04491828754544258, 0.08431825041770935, -1.338687777519226, -1.3246363401412964, -1.2507997751235962, -0.36281657218933105, 1.017749309539795, 0.7256361246109009, -1.1873152256011963, -0.6674928665161133, -2.686267852783203, -1.0406098365783691, -0.9954547882080078], [-1.06076979637146, -0.8843858242034912, -0.4676821529865265, 0.6064593195915222, 1.0174497365951538, 0.5918678641319275, 0.6384279727935791, -1.1506210565567017, 0.45523351430892944, -2.180558204650879, 1.307045340538025, 0.17746002972126007, 0.47459232807159424, 0.8291500210762024, 1.5890120267868042, -0.4363866150379181], [-2.516227960586548, -1.3895375728607178, -0.5208635926246643, 0.41294485330581665, 0.12970830500125885, -0.3412758409976959, -0.7515174746513367, -0.5964050889015198, 2.3772270679473877, -0.347841739654541, -0.5801699161529541, -0.06219187378883362, -0.38646772503852844, 0.4565788805484772, -0.9964280128479004, 0.9272125363349915], [0.02997240424156189, -0.03601272404193878, -1.069854497909546, -1.0450775623321533, -1.1143354177474976, 0.2362736612558365, 1.3650224208831787, -1.4623703956604004, 2.9098637104034424, 1.2383086681365967, -0.23166190087795258, -0.3544456958770752, 1.2241445779800415, -0.5005992650985718, -0.6111705303192139, 0.18048225343227386], [-0.3457854092121124, 1.3207755088806152, 0.7614116072654724, -0.9791204333305359, -0.32881611585617065, 0.17203445732593536, -1.4001715183258057, -0.3146475553512573, 1.0417712926864624, -0.9414072632789612, -0.6687132716178894, -1.3090338706970215, 0.2596997022628784, 0.5219808220863342, 1.0488942861557007, -0.5730659365653992], [-1.703861951828003, 1.171122670173645, -0.97015780210495, -0.20868517458438873, -0.08398246765136719, 0.9506077766418457, -1.5280166864395142, 0.4256907105445862, 0.1715741753578186, -0.52644282579422, -0.5918903350830078, -0.48340338468551636, -0.5304456353187561, -1.0635759830474854, 1.1075446605682373, -0.4223855435848236], [0.3016729950904846, 0.8598081469535828, -1.4313716888427734, -0.2184583842754364, 0.5099251866340637, 0.3545297086238861, 0.9056428670883179, 1.1486974954605103, -0.31455475091934204, -1.1111100912094116, 2.1139299869537354, 1.0345027446746826, -1.5810420513153076, -0.5945751070976257, -0.5999667048454285, 0.7812716960906982], [-1.9325450658798218, -1.2421455383300781, -0.9473626017570496, 1.4860142469406128, 1.3380800485610962, 0.303329199552536, 0.10009480267763138, 1.2459739446640015, -1.3944581747055054, -0.11376786231994629, -0.10458359867334366, 0.9886298775672913, -0.8532251119613647, 0.24979755282402039, -0.4279870390892029, 0.04709278419613838], [2.010704517364502, 0.9287466406822205, 0.2749757766723633, -0.049373652786016464, -1.437587857246399, -2.033656120300293, 0.5109540820121765, 1.3008663654327393, -0.17668400704860687, -0.37663212418556213, 1.4321327209472656, -0.5454530715942383, -0.48925134539604187, -1.1609134674072266, 1.335476040840149, -0.0724199190735817], [0.36023014783859253, -1.05311918258667, 1.3621065616607666, 0.2205246090888977, -0.6357882022857666, -0.14384298026561737, 0.1671711802482605, 0.9006714820861816, 0.2411148101091385, -0.8109420537948608, 0.2884383797645569, 0.543056070804596, 0.43062981963157654, 0.7718148827552795, -1.245275855064392, -1.5329101085662842], [2.1524789333343506, -0.45986154675483704, -0.5144847631454468, 0.3723395764827728, -0.5851119160652161, 0.6714791655540466, 1.0789402723312378, 1.3409674167633057, -0.1437022089958191, -0.38716578483581543, 0.5591685175895691, -1.561306118965149, -1.0474441051483154, -0.9126269817352295, 0.6524645686149597, 0.8538622856140137], [-1.7260385751724243, 1.3097169399261475, 0.2093905806541443, -0.0036205870565027, -0.16143830120563507, 0.527775764465332, -0.5238910913467407, -1.3319528102874756, -0.4756247401237488, 0.9918462038040161, -0.8721315264701843, 0.3747420310974121, -0.2275540828704834, 1.0250744819641113, -1.0843509435653687, 1.1541328430175781], [1.7432520389556885, -0.6182175874710083, 2.119863986968994, 0.4291909337043762, -0.3005661368370056, -0.7921057939529419, -0.19531521201133728, 1.3996169567108154, -0.6669910550117493, 0.19997094571590424, 1.1715095043182373, 1.1959165334701538, -0.5387915372848511, -0.21525561809539795, 0.48953017592430115, 0.6368771195411682], [0.49136966466903687, 0.1134120374917984, 0.3227732479572296, 1.7346420288085938, 0.11116267740726471, -0.18259185552597046, -0.3944973647594452, 0.10677865147590637, -0.8124512434005737, -1.0149322748184204, 0.05172664299607277, -0.4085488021373749, -0.3190467655658722, 0.43228647112846375, 0.6525877118110657, -0.009478656575083733], [1.7032470703125, 0.34444576501846313, 0.7690147757530212, -1.9742650985717773, 0.16728495061397552, -1.4611310958862305, 0.7787615656852722, 2.0755200386047363, -0.06255770474672318, 0.9641323685646057, -1.3811405897140503, -0.7493636012077332, 0.8718419671058655, 0.011263802647590637, 0.48093006014823914, -1.3733712434768677], [-1.1487023830413818, 0.7257533669471741, 0.10973835736513138, -1.4511834383010864, -0.3994908630847931, -0.9354255795478821, 0.5206809043884277, 0.5707343220710754, -1.015859603881836, 0.4595145285129547, 0.3065977692604065, -0.9905972480773926, 0.3094387650489807, -0.9499620199203491, 0.02606436051428318, 1.0465631484985352], [0.7091071605682373, 1.7800164222717285, -0.9746436476707458, -0.4602552652359009, 0.13321973383426666, -2.617366313934326, -0.9701530337333679, -1.1328685283660889, -1.029345154762268, -0.44200846552848816, -0.5322073101997375, -0.7493310570716858, -0.8088230490684509, 2.244434118270874, 1.614456057548523, -0.20651830732822418], [-0.07821101695299149, 0.7037709951400757, -0.7138645648956299, 0.39454302191734314, -0.558525800704956, -0.6027506589889526, 1.4319876432418823, 1.3652114868164062, -1.2650164365768433, -2.1552422046661377, 0.08742634952068329, -0.6523454189300537, -0.17176830768585205, -0.6799665093421936, 1.0057907104492188, -0.4611428678035736], [-1.565809726715088, 0.2091025710105896, 1.7454946041107178, 2.542567253112793, -0.42701372504234314, -0.8917062282562256, 0.5774135589599609, -1.307674527168274, -1.3383921384811401, -0.5581215023994446, 1.2152692079544067, 0.06749202311038971, -1.6699265241622925, -0.20461632311344147, -0.5287172794342041, 0.7756745219230652], [-0.9095529317855835, 0.19181552529335022, 0.5843536853790283, -0.0994207113981247, 1.2601629495620728, -0.2063981592655182, -0.23030854761600494, -1.78731369972229, -0.6163747906684875, 1.732021689414978, -1.3996806144714355, -0.36188456416130066, 0.3176821172237396, -0.3898233473300934, 1.0875767469406128, 0.3016543686389923], [-2.1037116050720215, -1.287824034690857, -1.0927386283874512, -0.704863965511322, -0.7170324921607971, 1.0186489820480347, -0.6515965461730957, 2.6528773307800293, -0.2601467967033386, -0.1112576276063919, -0.5052596926689148, -1.8222718238830566, -0.5089131593704224, 0.6521395444869995, -0.9623372554779053, 0.44353407621383667], [0.17342635989189148, -0.9014526009559631, -0.42319855093955994, -0.23458608984947205, 0.028760863468050957, -0.6333576440811157, -1.1032373905181885, -0.26528283953666687, -1.2531371116638184, -1.024594783782959, 1.1001081466674805, -0.5670959949493408, -0.7748911380767822, 1.204341173171997, -0.019921626895666122, -0.20262981951236725], [0.5892127752304077, -1.4961421489715576, -1.0731159448623657, -0.20824186503887177, -0.1992361843585968, 0.913164496421814, -1.160573959350586, -1.1290067434310913, 0.35473763942718506, -0.6777421832084656, 0.4899607300758362, 0.09457451105117798, -1.7926056385040283, -0.728691816329956, 0.5476433634757996, 2.644564390182495], [-0.172187939286232, -0.9957706928253174, -0.8466358184814453, -1.0879950523376465, -0.5668206214904785, 0.7152926325798035, -0.07373645901679993, 0.17871156334877014, -0.24302485585212708, -1.9333393573760986, -1.0382789373397827, 0.21987028419971466, 1.7602630853652954, 0.7836616039276123, -1.2288081645965576, -0.4679684638977051], [0.5813926458358765, -0.35318800806999207, 0.20337098836898804, -0.463280588388443, -0.19820362329483032, -1.4706089496612549, -1.2438551187515259, 0.9706090092658997, 0.44684481620788574, -0.5597801804542542, 0.35173872113227844, 0.4019453823566437, -0.7085951566696167, 0.5622180104255676, 0.36424174904823303, -0.7101930379867554], [0.19643579423427582, 1.3903647661209106, 0.6762120723724365, -1.2731763124465942, 0.016522543504834175, 1.8569473028182983, 0.5843791365623474, -0.8120075464248657, 2.2974915504455566, 0.0628495141863823, 0.009383348748087883, -0.3659873604774475, -0.011885538697242737, -0.23736584186553955, 1.2400261163711548, 0.3159486651420593], [-0.9760790467262268, 0.040021926164627075, -1.1771161556243896, 0.374187707901001, -0.756062388420105, 0.8613985180854797, -0.48993968963623047, -0.07100959122180939, -0.4921203851699829, 0.03160442039370537, -0.07129501551389694, -0.27829357981681824, -0.8883888125419617, 1.6438450813293457, 2.5547876358032227, -0.015060937963426113], [0.9722691774368286, 0.017707549035549164, 0.2920825481414795, 0.23202461004257202, -0.5816867351531982, -0.8954391479492188, -0.06758102029561996, -0.6981649994850159, 0.6802229285240173, -0.30897220969200134, 0.5626740455627441, 1.4432483911514282, 0.22093361616134644, 0.10280072689056396, -0.046169213950634, 0.9920073747634888], [-0.41742557287216187, -0.14957089722156525, -0.7024603486061096, -2.1159980297088623, 1.0444172620773315, -2.315772294998169, -0.8477979898452759, -1.7705352306365967, -0.8562341332435608, -0.21096695959568024, 1.3259819746017456, -0.47402527928352356, 0.591714084148407, -2.183431386947632, -1.5164786577224731, 0.02756720595061779], [0.9943919777870178, 0.3680124282836914, 1.5517102479934692, 1.0741063356399536, -0.8160308003425598, 0.41534945368766785, -0.6094395518302917, 1.1641887426376343, -1.1108434200286865, -0.0974753201007843, 1.2971409559249878, 0.09095802903175354, -0.11649634689092636, 2.562053680419922, -0.023216841742396355, 1.938615322113037], [-0.5861467123031616, 0.7870228886604309, 1.1421704292297363, -0.40553122758865356, -1.465325951576233, -1.570222020149231, 0.5509101748466492, 0.12746889889240265, -0.19145344197750092, 0.323144793510437, -0.8568448424339294, -0.8814761638641357, 0.7043116092681885, -1.3758000135421753, 1.592191457748413, -2.017106771469116], [-1.5112391710281372, -1.3570014238357544, -1.3007062673568726, 1.8307949304580688, 1.0780329704284668, 0.012918014079332352, 0.2170259952545166, 0.8936240673065186, 0.1518542617559433, -1.7847800254821777, -2.249746799468994, -0.9027803540229797, -1.0491185188293457, 0.08793336898088455, -0.21745789051055908, 0.6162961721420288], [0.12187057733535767, -0.19660907983779907, -0.5466854572296143, 0.010387440212070942, -0.651541531085968, 0.7703033089637756, 1.050948143005371, -0.004369207192212343, -0.16883592307567596, -0.6172234416007996, 1.4017881155014038, 1.0311386585235596, 0.30032724142074585, 0.07811994850635529, -0.10582844913005829, -0.3537484407424927], [-0.16572369635105133, 1.8227381706237793, -0.2748263478279114, 0.18422400951385498, -0.11334024369716644, -2.7614476680755615, -0.04185499623417854, 0.034395020455121994, 1.1922657489776611, 0.018985461443662643, -0.7634583711624146, 0.13821633160114288, 2.215820789337158, 1.2730700969696045, 2.843247890472412, -2.126596689224243], [-0.8271237015724182, -1.5382708311080933, 1.1596686840057373, -0.034491367638111115, -0.5249233841896057, 1.3006608486175537, -1.069863200187683, 1.5579564571380615, 0.4899541735649109, 1.0833899974822998, -0.4264831244945526, 0.5758235454559326, -1.0388880968093872, 0.19227392971515656, 1.4042420387268066, -0.36259719729423523], [-0.29832783341407776, -0.040723707526922226, -0.1330081820487976, 0.29280149936676025, -0.963488757610321, -0.2749170660972595, 1.0110864639282227, 2.1151976585388184, -1.0030027627944946, 1.295817255973816, -1.25552237033844, 0.9723857641220093, 0.23380957543849945, -1.4885709285736084, -0.7103999853134155, 0.8853698372840881], [-0.5043923258781433, -0.07143361121416092, -0.6820681691169739, -0.2540862560272217, -1.1099164485931396, -0.48277735710144043, 0.736189067363739, 0.22350917756557465, 0.57033371925354, -0.9165511131286621, 1.1200037002563477, -0.3909510672092438, -0.1887872815132141, -0.3661378026008606, 0.34560659527778625, 1.005145788192749], [2.2104499340057373, -0.38828712701797485, -0.6041688919067383, 0.10589362680912018, -0.09227962791919708, 2.057896137237549, -0.41531646251678467, 0.4761650264263153, 0.38443198800086975, 0.016092574223876, -0.8015086650848389, 1.9288716316223145, 0.7101160883903503, -0.1076471135020256, -0.2543080747127533, -0.36270278692245483], [-0.24896971881389618, 1.0657572746276855, 0.3992241322994232, 1.8318724632263184, 0.8773829340934753, -0.7938995957374573, -0.6885619759559631, -1.3685479164123535, 0.3718249797821045, -1.2842576503753662, 1.1023772954940796, -1.2662372589111328, -0.329496294260025, 0.38772842288017273, 0.7212056517601013, 0.4205266237258911], [-0.3024976849555969, 0.30787408351898193, -0.12998226284980774, -1.7004601955413818, -0.27922868728637695, 0.014473838731646538, 1.170369029045105, -0.32123783230781555, 0.7719763517379761, 0.22833915054798126, -1.1721408367156982, -1.6746834516525269, -0.2800550162792206, 0.06274274736642838, 0.04011820629239082, 0.08097849041223526], [-0.08477720618247986, -2.168093681335449, -0.5929596424102783, 0.29284805059432983, -1.411669373512268, 1.2845429182052612, 0.09002327919006348, 1.0750882625579834, -0.4119122624397278, -0.28674814105033875, 2.06977915763855, -0.6366896033287048, 0.13911877572536469, -1.159828782081604, -1.9906866550445557, -0.6120983362197876], [1.5068525075912476, -0.5177246332168579, 0.10176145285367966, 0.5107371807098389, 0.7260156869888306, 0.23277045786380768, 0.6857165694236755, 1.0413522720336914, -2.701826572418213, 0.2304147481918335, -0.9321871995925903, -0.23638516664505005, -0.5873189568519592, 0.7206330895423889, -0.36926940083503723, 0.17941220104694366], [-1.2389127016067505, -0.3896816074848175, -1.5214351415634155, -0.9338034391403198, -0.8798294067382812, 0.7381418943405151, 1.4365073442459106, -0.36017751693725586, -1.2542413473129272, 1.3306297063827515, 0.05439479276537895, -0.39979034662246704, -0.25841066241264343, -1.0247465372085571, 0.3219502866268158, 0.3568544387817383], [0.24342690408229828, 2.212027072906494, -2.4422245025634766, -0.2626314163208008, 1.1145187616348267, -1.5673718452453613, -0.5364006161689758, 0.9440572261810303, 0.106373131275177, 0.05570533499121666, -0.5002425909042358, 0.04687565937638283, -0.0927833840250969, 1.5026617050170898, 0.6155038475990295, 0.8205632567405701], [-1.4803639650344849, 0.4466645121574402, -0.16544590890407562, -0.98050457239151, 0.28541433811187744, -0.3962402939796448, 0.649391233921051, -1.4964574575424194, -1.6239155530929565, -1.3840315341949463, 1.035897135734558, 0.6307447552680969, 0.9178016781806946, 1.938774585723877, 0.42043206095695496, 1.6518696546554565], [0.30104851722717285, 1.4124352931976318, 0.8974745869636536, 1.384085774421692, 1.0829943418502808, 1.0504980087280273, 0.5382764935493469, -0.39637491106987, -1.7104713916778564, 0.5305439829826355, -0.003594379173591733, -0.9492050409317017, 2.63454532623291, -0.1876293569803238, -0.7587955594062805, 0.4939553141593933], [1.5822495222091675, 0.2604658007621765, 1.7021033763885498, -0.4144122302532196, 0.7008878588676453, 0.5302091836929321, -1.929867148399353, -0.6762552857398987, 0.8239123225212097, -0.5105460286140442, -0.6826401352882385, -0.6734173893928528, -0.33772560954093933, -0.16324414312839508, -0.853597104549408, 0.4640547037124634], [1.365966796875, 1.0452218055725098, -0.6826182007789612, 0.2427673488855362, 3.273361921310425, 0.6352096199989319, -0.7442842125892639, -1.225232481956482, -1.2437622547149658, -1.1037318706512451, 1.3658193349838257, 0.6436141133308411, -1.842842936515808, -1.030773639678955, 2.5741121768951416, 1.2967196702957153], [-0.11964226514101028, -0.0893610417842865, -0.34868189692497253, 1.359423279762268, -1.3934553861618042, 1.6086771488189697, 2.251755475997925, -0.149402916431427, -1.4152268171310425, 0.4464538097381592, 0.9387784004211426, -0.7921950221061707, 0.4249611794948578, 1.116014838218689, 0.6599693298339844, 0.22445131838321686], [-1.6480510234832764, -1.0051147937774658, 1.6791154146194458, 0.8319731950759888, 1.609593391418457, -0.3916480541229248, 1.0707398653030396, -0.5721765160560608, 0.6501364707946777, -0.4144589900970459, -1.5691555738449097, -0.16377639770507812, 0.15286514163017273, 0.8221145272254944, -0.8913756012916565, 0.3239841163158417], [0.18784616887569427, -0.003968612756580114, -1.0939466953277588, -0.1835758537054062, 0.3717833459377289, 0.4677911102771759, -1.4002556800842285, 1.7045629024505615, -0.19166280329227448, -0.08274742215871811, -0.8341444134712219, 0.9566844701766968, 0.9659146666526794, 0.8110052943229675, -0.2494327425956726, 1.0605133771896362], [0.7739273905754089, -1.0871299505233765, 0.6539737582206726, -1.5630406141281128, -0.4209822118282318, -0.27796506881713867, -1.1029027700424194, -0.2992328703403473, 1.1587294340133667, -0.4379335939884186, 0.7340940833091736, 0.10646335035562515, 0.060874368995428085, -0.06705011427402496, 0.6543713808059692, -1.4813637733459473], [0.3602750599384308, 0.1321955919265747, -0.15808315575122833, -0.5800241827964783, 0.06701779365539551, -0.20876003801822662, -1.1686867475509644, 0.37300169467926025, 0.05749255418777466, 1.248905897140503, 0.10924340039491653, 0.689346432685852, 0.07618541270494461, 1.3004182577133179, 0.6093422770500183, -1.5874048471450806], [-0.010495363734662533, 0.2668464779853821, -1.01278817653656, 1.0047075748443604, 0.860832691192627, 0.46808692812919617, -1.146575927734375, 0.12240312993526459, -1.1156129837036133, -0.5958549976348877, -0.07563508301973343, -0.5527976155281067, -1.5440757274627686, 1.8538068532943726, 0.434209406375885, 1.793303370475769], [0.7427906394004822, -1.0681023597717285, 0.6255298256874084, -0.8312895894050598, -0.09142932295799255, -0.4154985547065735, -0.6057747602462769, -0.6589491367340088, -0.02259867824614048, 1.8549686670303345, -1.3362027406692505, 0.11377165466547012, 1.1227705478668213, -1.7676581144332886, -0.3103307783603668, 0.520386278629303], [0.3898279368877411, -0.8536648154258728, -1.4206316471099854, -0.05315454676747322, 0.31108251214027405, -2.2044026851654053, -1.1345609426498413, 1.3728896379470825, -0.09045042097568512, -1.1584502458572388, 0.6215466856956482, -0.3262251615524292, 0.4477897882461548, 0.8934149146080017, -1.4670286178588867, -0.4326236844062805], [0.5222887396812439, -0.8202633261680603, 0.10291755199432373, 1.7489433288574219, -0.04669131338596344, -0.35205724835395813, -0.5642502903938293, -1.2394521236419678, 1.5913119316101074, 0.13297608494758606, 0.8229002356529236, 0.8666621446609497, 0.33186811208724976, 1.1796422004699707, -0.8763779997825623, -0.736201286315918], [-0.4695740342140198, 0.17591820657253265, 0.2993777394294739, -0.27474793791770935, -0.8039671182632446, -0.1907990425825119, 0.8828825950622559, -0.8320678472518921, -0.5831761956214905, 1.2390003204345703, 0.331073135137558, -0.9436893463134766, 0.1671074777841568, -0.4900413751602173, 0.06614633649587631, -1.0151771306991577], [0.1072780191898346, -0.4021446108818054, 2.454434871673584, 0.771117627620697, 0.1403215378522873, 1.61838960647583, 0.5591112971305847, -1.6657522916793823, 0.9509473443031311, 1.3467155694961548, -0.17326955497264862, -0.3531319797039032, 0.4044094681739807, -0.25697290897369385, -0.813917875289917, 1.0422548055648804], [-1.3022336959838867, 0.16443206369876862, 0.8727725744247437, 0.9601840376853943, 0.7505213022232056, -1.328152060508728, 1.7680836915969849, -0.7225486636161804, -0.2687915563583374, 0.3897753655910492, 1.708075761795044, -0.7417208552360535, -1.7540498971939087, -0.4942249059677124, 1.9024066925048828, 2.1914024353027344], [0.8183793425559998, -0.08757726103067398, 1.2672383785247803, -0.8514835238456726, 1.1140143871307373, 0.1958276927471161, 1.9635741710662842, -1.9044421911239624, -0.21382999420166016, -1.4649629592895508, -0.3231692314147949, 1.2468626499176025, 0.2624722421169281, -0.3983705937862396, 1.3720271587371826, 0.8822223544120789], [-0.7820969820022583, 1.5022618770599365, 0.45482051372528076, -1.7494980096817017, -0.31695884466171265, 0.3917123079299927, 1.2181909084320068, -0.1490868479013443, -0.4295344650745392, -0.453856885433197, -0.4728310704231262, 0.11251972615718842, 0.48877036571502686, -0.024946555495262146, -0.25559237599372864, 1.2542327642440796]]]], "dtype": "float32"}, true], "e": {"__tup__": [{"__nd__": [[[[-0.3477587103843689, 1.1351977586746216, -1.1724717617034912, -0.060395993292331696, -1.1256744861602783, 0.5584741830825806, -1.5332393646240234, 1.3305784463882446, 1.6059633493423462, -0.047179874032735825, -1.6343008279800415, 0.7779059410095215, 0.15686142444610596, 0.7698484659194946, -1.255402684211731, 1.4547362327575684], [0.008674582466483116, -0.8828462958335876, 0.2011404186487198, -0.8648785352706909, -0.8269768953323364, -0.4708859622478485, 0.24561063945293427, 1.322823166847229, 0.3510940670967102, 0.30173584818840027, -1.8377994298934937, -0.00604068161919713, -0.7805643677711487, -0.6853554248809814, -1.176445484161377, 2.305424690246582], [-0.08844643086194992, -0.22416441142559052, -0.46715620160102844, -0.1900859922170639, -0.33967074751853943, 0.07476532459259033, -0.9236707091331482, 0.7113741040229797, 0.945344090461731, 0.428628146648407, -0.8070511817932129, 0.7562092542648315, -0.36659759283065796, 0.02168464846909046, -0.9927238821983337, 1.1444034576416016], [-0.3735981285572052, -0.8612735271453857, 0.32832032442092896, -0.6440460085868835, 0.28956544399261475, -0.11093882471323013, -0.05758248642086983, 0.09817627817392349, 0.61286461353302, 0.8542013764381409, -0.15975730121135712, 0.5934138894081116, -0.5519476532936096, -0.11148151755332947, -0.24967750906944275, 0.4530629515647888], [-0.2722410559654236, -0.7535464763641357, 0.5563098788261414, -1.0901130437850952, 0.10872206836938858, -0.027455344796180725, 0.2154875248670578, 0.24889624118804932, 0.48177391290664673, 0.07894612848758698, -0.43232664465904236, 0.22255125641822815, -0.5350914001464844, -0.4653909206390381, -0.169748917222023, 0.3048337399959564], [-0.37081196904182434, -0.2048070728778839, 0.09334895014762878, -0.7140407562255859, -0.21410103142261505, 0.12895530462265015, -0.28127655386924744, 0.5146730542182922, 0.7888979911804199, 0.40513983368873596, -0.709242582321167, 0.5099930167198181, -0.3183746039867401, 0.09663648903369904, -0.4415515065193176, 0.6289238929748535], [-0.45102038979530334, -0.26364564895629883, 0.08302699774503708, -0.975310742855072, -0.14505447447299957, -0.3227057158946991, 0.10288398712873459, 0.6373827457427979, 0.5381830930709839, 0.8752874135971069, -0.03644512966275215, 0.2797597348690033, -0.408690482378006, 0.08062012493610382, -0.46637654304504395, 0.2264617532491684], [-0.12533695995807648, -0.3158617913722992, 0.04731160029768944, -0.7103906869888306, -0.6850225329399109, -0.1294299066066742, -0.14251969754695892, 1.034364104270935, 0.4906010329723358, 0.44033557176589966, -1.3151851892471313, 0.270039439201355, -0.4383010268211365, -0.12577030062675476, -0.9309175610542297, 1.5071697235107422], [-0.011155795305967331, 0.17329764366149902, 0.11751159280538559, -0.06459184736013412, -0.8160787224769592, 0.009914424270391464, -0.27398785948753357, 0.9016534686088562, 0.9380338191986084, -0.023015445098280907, 0.2613506019115448, 0.14680558443069458, 0.08460579812526703, 0.887829065322876, -1.67279052734375, 0.6121423244476318], [-0.571895182132721, -0.07905445992946625, 0.40477511286735535, -0.9353131651878357, -0.08964948356151581, -0.08708295226097107, 0.28417882323265076, 0.40377673506736755, 0.4354935884475708, 0.7130287885665894, -0.32975468039512634, 0.22167368233203888, -0.24330681562423706, 0.3585682511329651, -0.1580166518688202, 0.42835983633995056], [-0.2020961344242096, -0.33489441871643066, -0.00822475180029869, -0.3954567313194275, -0.06881173700094223, -0.1492317020893097, -0.03541668504476547, 0.479562371969223, 0.45306330919265747, 0.4847109317779541, -0.3480755090713501, 0.4617689549922943, -0.4563460350036621, 0.05742326006293297, -0.5389752984046936, 0.5078893899917603], [-0.18833424150943756, -0.540019690990448, -0.1030193343758583, 0.03978503495454788, 0.4324706196784973, -0.09393925964832306, 0.19938477873802185, -0.020949676632881165, 0.34698134660720825, 0.2103116810321808, -0.10248781740665436, 0.559586763381958, -0.6189689636230469, -0.13865740597248077, -0.03356565162539482, -0.24677008390426636], [-0.17235919833183289, 0.39263972640037537, 0.1598322093486786, -0.38977789878845215, -0.432487428188324, -0.2973451018333435, 0.13079603016376495, 0.6664445400238037, 0.37598177790641785, -0.11647932976484299, 0.12459342926740646, -0.13415300846099854, -0.03570815175771713, 0.5330122709274292, -0.8927050828933716, 0.4808749258518219], [-0.14805270731449127, 0.25567442178726196, 0.0596592016518116, -0.3635307252407074, -0.382681667804718, -0.17707480490207672, -0.2140626162290573, 0.7073668837547302, 0.6697934865951538, -0.038652461022138596, 0.04459838569164276, 0.11368723958730698, -0.0999850183725357, 0.3358975648880005, -0.9383047223091125, 0.2464352250099182], [-0.23077087104320526, 0.5186927914619446, -0.10362010449171066, 0.021570956334471703, -0.47729113698005676, 0.008610772900283337, -0.6146785616874695, 0.7111099362373352, 0.767392635345459, -0.14679016172885895, -0.4706820547580719, 0.2344077080488205, 0.09305869042873383, 0.5266860723495483, -1.075928807258606, 0.5928223133087158], [-0.27761775255203247, 0.4116241931915283, 0.13271601498126984, -0.15701718628406525, -0.11550947278738022, -0.6422629356384277, -0.0041955262422561646, 0.1950548142194748, 0.1875699907541275, -0.29599469900131226, -0.2237071990966797, -0.1702241152524948, -0.016225945204496384, 0.0065920716151595116, -0.3273915648460388, 0.3460937738418579], [-0.04046959802508354, 0.2805335521697998, 0.4904167950153351, -0.13232848048210144, -0.04526562988758087, -0.5216416716575623, -0.23375028371810913, 0.08090775460004807, 0.016294755041599274, 0.010537310503423214, -0.24310994148254395, -0.17837613821029663, 0.1888444572687149, -0.20737408101558685, -0.09780538082122803, 0.13650162518024445], [0.08979439735412598, 0.18065448105335236, 0.007671177387237549, -0.04951101168990135, -0.34323424100875854, 0.2896946668624878, -0.5242217779159546, 0.5738862752914429, 0.6743881106376648, 0.21256154775619507, -0.5666406154632568, 0.3848404884338379, 0.2782043218612671, 0.17034438252449036, -0.5121446847915649, 0.6198846101760864], [-0.571363091468811, 0.8114984035491943, -0.18540136516094208, -0.0931243821978569, -0.30522486567497253, -1.5514196157455444, 0.18612328171730042, -0.13912640511989594, 0.1211559996008873, -0.1654636114835739, 0.31514135003089905, -0.41538888216018677, -0.2534773647785187, 0.045197322964668274, 0.1200266033411026, -0.06410416960716248], [-0.280039519071579, 0.3697914779186249, 0.28734493255615234, -0.2500416338443756, -0.07631812989711761, -0.2141333669424057, -0.07301169633865356, 0.6712751984596252, 0.41378703713417053, -0.029267510399222374, -0.7737711071968079, -0.0831221267580986, -0.01735624484717846, -0.008707438595592976, -0.4407000243663788, 0.6992545127868652], [-0.024392230436205864, 0.06344463676214218, 0.29272422194480896, -0.2728739082813263, -0.030419526621699333, -0.02546384185552597, -0.014693843200802803, 0.41329067945480347, 0.267922967672348, -0.5375474095344543, 0.018992407247424126, -0.17607370018959045, -0.014629275538027287, 0.025375373661518097, -0.5870700478553772, 0.08796834200620651], [0.048296354711055756, -0.08606524020433426, -0.01040661707520485, -0.3215828835964203, 0.02678595297038555, 0.18153080344200134, -0.17164045572280884, 0.3131944239139557, 0.5448976755142212, -0.9478739500045776, -0.2163187712430954, 0.21898242831230164, -0.24059516191482544, -0.2667015790939331, -0.4521542489528656, -0.09966679662466049], [0.11335909366607666, 0.14682675898075104, 0.23773278295993805, -0.3818381428718567, 0.14192745089530945, -0.02863500826060772, -0.1758175641298294, 0.20393183827400208, 0.2594788372516632, -0.481941282749176, -0.12804998457431793, -0.23911097645759583, -0.23655638098716736, -0.35248255729675293, -0.0042022159323096275, 0.15356503427028656], [-0.04822094738483429, 0.16034655272960663, 0.22260542213916779, -0.09698013961315155, 0.09817852824926376, -0.1189318522810936, -0.08472517877817154, 0.20839747786521912, 0.12135453522205353, -0.156938374042511, 0.050677042454481125, -0.2605874240398407, 0.18870556354522705, 0.06380348652601242, -0.27470412850379944, 0.12121280282735825], [-0.16307638585567474, 0.11298111826181412, -0.029353618621826172, -0.21431714296340942, -0.06948284059762955, 0.06080297380685806, -0.4122632145881653, 0.529805064201355, 0.6536082625389099, 0.26821354031562805, -0.28318628668785095, 0.24319398403167725, -0.07386860996484756, 0.27131739258766174, -0.451707661151886, 0.2830016613006592], [-0.14667469263076782, 0.5456622242927551, 0.2439657747745514, -0.22541964054107666, -0.12655870616436005, -0.01132362149655819, -0.18573296070098877, 0.8831295967102051, 0.6952646970748901, 0.07282672822475433, -0.7620381116867065, -0.13757523894309998, 0.1169145405292511, -0.16207735240459442, -0.3673000931739807, 0.17279066145420074], [-0.055288493633270264, 0.15615057945251465, -0.015863675624132156, 0.2619565725326538, 0.11684568226337433, -0.06664248555898666, -0.2534179091453552, 0.10712787508964539, 0.3390030860900879, -0.1330638825893402, 0.14927445352077484, 0.15387441217899323, -0.019716855138540268, 0.3744061589241028, -0.500968873500824, -0.07732153683900833], [-0.2540484666824341, 0.09386654943227768, 0.10827414691448212, -0.21631428599357605, 0.05655263736844063, -0.2669704258441925, -0.036717332899570465, 0.33342933654785156, 0.30405184626579285, 0.047759734094142914, -0.14653347432613373, -0.18020272254943848, -0.13283979892730713, 0.23279275000095367, -0.00810472946614027, 0.33317509293556213], [-0.16706347465515137, 0.6875729560852051, -0.1000051274895668, 0.15989400446414948, -0.28496670722961426, 0.06827635318040848, -0.49480438232421875, 0.7734083533287048, 0.9378997683525085, -0.05703138932585716, -0.7353004813194275, 0.1299266517162323, -0.056079793721437454, 0.30634811520576477, -0.8128857612609863, 0.2333684116601944], [-0.2907061278820038, -0.021695462986826897, 0.05220627412199974, -0.2583836019039154, 0.208875834941864, -0.3167102038860321, 0.13758055865764618, 0.33054888248443604, 0.24053488671779633, 0.14096400141716003, 0.2507021427154541, -0.24150393903255463, -0.14679871499538422, 0.13986752927303314, 0.14863932132720947, -0.04572686180472374], [-0.33666300773620605, 0.24073149263858795, -0.012477047741413116, 0.13757628202438354, 0.07954596728086472, -0.47809192538261414, -0.012157637625932693, 0.259428471326828, -0.09297633916139603, -0.6461291909217834, -0.027254320681095123, -0.32155466079711914, -0.3219260573387146, 0.055715467780828476, 0.31810304522514343, -0.10891924053430557], [-0.07405401766300201, 0.3962227404117584, -0.24540704488754272, 0.10324850678443909, -0.032645806670188904, -0.11230743676424026, -0.1721111238002777, 0.2195388376712799, 0.609989583492279, -0.09009701758623123, -0.42214250564575195, 0.11583326756954193, -0.17287670075893402, 0.37469494342803955, -0.2540439963340759, 0.24136964976787567], [-0.3261692523956299, 0.02069351077079773, 0.21954230964183807, -0.18601103127002716, 0.09005601704120636, -0.01194505300372839, -0.15161018073558807, 0.2879912257194519, 0.2844427525997162, -0.09121294319629669, -0.024979867041110992, -0.22393034398555756, 0.25692182779312134, 0.4481154680252075, -0.18342243134975433, 0.1580960750579834], [-0.14402391016483307, 0.1052236333489418, 0.00866719987243414, 0.32799509167671204, 0.31798991560935974, -0.31798094511032104, -0.15722332894802094, 0.016722876578569412, -0.11078055948019028, -0.36434072256088257, 0.26667094230651855, -0.36989304423332214, -0.29867884516716003, -0.043946824967861176, 0.13257160782814026, -0.4376815855503082], [-0.15505114197731018, 0.5034885406494141, 0.4991213083267212, -0.4196174740791321, -0.22544793784618378, -0.16214141249656677, -0.3226526975631714, 0.3394002318382263, 0.02380317449569702, 0.34476011991500854, -0.07226085662841797, -0.5175506472587585, 0.3703022599220276, 0.12380418181419373, -0.17304885387420654, 0.047483205795288086], [-0.1054772213101387, 0.21241271495819092, 0.09042634814977646, 0.0729110836982727, 0.029404116794466972, -0.00024462619330734015, -0.1170281395316124, 0.31957754492759705, 0.449602335691452, -0.23984995484352112, -0.3754882216453552, 0.0432768277823925, 0.023759735748171806, -0.14969611167907715, -0.14995311200618744, -0.12105686962604523], [-0.11591967940330505, 0.22268100082874298, 0.5183825492858887, -0.3084762394428253, -0.2606606185436249, -0.03608926385641098, -0.1729130744934082, 0.15800999104976654, 0.16600348055362701, 0.14077895879745483, 0.11848036199808121, -0.37657156586647034, 0.2944687604904175, 0.161165252327919, -0.19317954778671265, 0.09218748658895493], [-0.09400463104248047, -0.17243149876594543, 0.19935870170593262, -0.09806215763092041, 0.06559392064809799, -0.13007330894470215, 0.053629226982593536, 0.26361212134361267, 0.36098790168762207, 0.10871746391057968, 0.08053002506494522, -0.06229076534509659, -0.11582077294588089, 0.039400290697813034, 0.0022543275263160467, -0.12284739315509796], [-0.005178744439035654, -0.1977367252111435, 0.10074771195650101, 0.07712750136852264, 0.04532179608941078, -0.019980538636446, 0.08223862200975418, -0.08309593796730042, 0.2890329957008362, -0.15248411893844604, 0.0595792680978775, 0.01307138241827488, 0.19169771671295166, 0.4126076102256775, -0.16335366666316986, -0.09172174334526062], [-0.2193826586008072, 0.11011571437120438, -0.02039189636707306, 0.008138950914144516, -0.040091123431921005, -0.07038843631744385, -0.31954169273376465, 0.4907481074333191, 0.48015284538269043, -0.06256956607103348, -0.4169899821281433, 0.08493949472904205, -0.1509418934583664, 0.39245033264160156, -0.053139232099056244, 0.24104991555213928], [-0.0742705762386322, 0.006850219331681728, 0.36571937799453735, -0.2006576955318451, -0.052525997161865234, -0.10295791178941727, -0.009756366722285748, 0.24055922031402588, 0.2626599371433258, -0.08708801865577698, -0.11140330135822296, -0.0932045578956604, -0.10567004233598709, -0.1484050303697586, -0.07218923419713974, 0.15800261497497559], [-0.23012399673461914, -0.16154560446739197, 0.48862722516059875, -0.49135372042655945, 0.06295637786388397, 0.032439783215522766, 0.026489688083529472, 0.06774071604013443, 0.4406796991825104, -0.19721539318561554, 0.15973302721977234, -0.05533601716160774, 0.01938091777265072, -0.08585233986377716, 0.08415161818265915, -0.03418900445103645], [-0.14910976588726044, -0.1382947713136673, 0.10404054820537567, -0.028734875842928886, 0.046102140098810196, 0.041764408349990845, -0.1800142079591751, 0.08792123943567276, 0.4085018038749695, -0.031077388674020767, -0.013180575333535671, 0.0713166818022728, -0.0032631326466798782, 0.22674119472503662, -0.043088868260383606, 0.16065892577171326], [-0.1323007047176361, 0.14634640514850616, 0.14475366473197937, 0.00984199158847332, -0.13176999986171722, 0.022549262270331383, -0.06296650320291519, 0.47112369537353516, 0.18015284836292267, -0.27131667733192444, -0.1250981092453003, -0.16533343493938446, 0.02675059624016285, -0.0130467489361763, -0.2436748594045639, 0.08656282722949982], [-0.09340349584817886, 0.010700816288590431, 0.6764806509017944, -0.2795301079750061, -0.10810408741235733, -0.30852222442626953, -0.07727154344320297, 0.19476363062858582, 0.11258825659751892, -0.23752541840076447, -0.18147794902324677, -0.3683946132659912, 0.18121901154518127, -0.08991546928882599, -0.05085969343781471, 0.035986319184303284], [-0.19001968204975128, 0.08865530788898468, 0.14082947373390198, 0.018662462010979652, -0.07708882540464401, -0.1457916498184204, 0.077382393181324, 0.1484091430902481, -0.07593447715044022, -0.4050588309764862, 0.012462512589991093, -0.31821006536483765, -0.06266933679580688, -0.11641466617584229, 0.014933623373508453, -0.14823098480701447], [-0.06087081879377365, -0.11302804201841354, 0.2885976731777191, -0.14803935587406158, 0.018359575420618057, -0.037565167993307114, -0.06668198108673096, 0.21387897431850433, 0.36920619010925293, 0.008344968780875206, -0.3604617118835449, 0.020652156323194504, 0.056919071823358536, 0.06666474044322968, 0.14109371602535248, 0.16446122527122498], [-0.07406215369701385, -0.05164843425154686, -0.08366316556930542, 0.1884877234697342, 0.014506455510854721, 0.1162140741944313, -0.527559220790863, 0.24675625562667847, 0.45610448718070984, -0.0003603766381274909, -0.021143607795238495, 0.2201920449733734, -0.05171492323279381, 0.25307223200798035, -0.23223768174648285, -0.10138960927724838], [-0.11471667885780334, 0.07719481736421585, 0.08099599927663803, 0.0982273519039154, -0.11240138113498688, 0.10554396361112595, -0.09566008299589157, 0.3024788498878479, 0.26506102085113525, -0.31128957867622375, -0.07324767857789993, 0.05923614650964737, 0.05386168509721756, 0.29242563247680664, -0.3699605166912079, 0.09092879295349121], [-0.19957450032234192, 0.014391753822565079, -0.07791125029325485, 0.07657238841056824, 0.08087790012359619, 0.10047049820423126, -0.24511504173278809, 0.10146772116422653, 0.18536905944347382, -0.3085542917251587, 0.056989554315805435, 0.12187130749225616, -0.09596367925405502, 0.13233739137649536, 0.04764242470264435, -0.24321503937244415], [-0.07414271682500839, 0.11342541128396988, 0.2513803243637085, 0.038417916744947433, 0.04288419336080551, 0.0024607330560684204, 0.08197640627622604, 0.4773733913898468, 0.054689861834049225, -0.23489834368228912, -0.03918639197945595, -0.488503098487854, -0.06236359104514122, -0.06228361651301384, -0.20240242779254913, -0.03848840668797493], [-0.08912766724824905, 0.12628521025180817, 0.15651679039001465, 0.040203213691711426, -0.24939902126789093, 0.19151316583156586, -0.2050819844007492, 0.39826029539108276, 0.49624529480934143, -0.01726473681628704, 0.13213086128234863, -0.10908469557762146, 0.194906085729599, 0.7870246171951294, -0.6554057002067566, 0.3304443955421448], [0.11547033488750458, -0.023033645004034042, 0.039476025849580765, -0.09363285452127457, -0.12958021461963654, 0.22804096341133118, -0.15147371590137482, 0.3000504970550537, 0.35903221368789673, -0.6084847450256348, -0.17021048069000244, -0.12628240883350372, -0.01841564103960991, -0.062295202165842056, 0.037220507860183716, 0.051237553358078], [-0.20038866996765137, 0.14766019582748413, 0.15854164958000183, 0.026668565347790718, -0.056529849767684937, -0.05426163598895073, 0.04150037467479706, 0.47341057658195496, -0.06814927607774734, -0.1288856565952301, -0.30939772725105286, -0.2602153420448303, -0.10787519067525864, -0.10588188469409943, -0.19946710765361786, 0.21294808387756348], [-0.19178274273872375, 0.24705956876277924, -0.20750446617603302, -0.15370456874370575, -0.37613582611083984, -0.8309995532035828, 0.44000786542892456, 0.3575843870639801, 0.06608546525239944, 0.1715344935655594, 0.3078024983406067, -0.528178870677948, -0.23093779385089874, -0.18860404193401337, -0.382564514875412, -0.00264682131819427], [-0.03987354040145874, -0.047571174800395966, 0.1182936280965805, 0.18417198956012726, 0.11031311750411987, 0.14315462112426758, 0.12125600874423981, 0.5286379456520081, 0.10518714040517807, -0.2237359583377838, 0.10615325719118118, -0.4204299747943878, -0.008542307652533054, -0.05826614797115326, -0.2839038670063019, -0.09835559129714966], [-0.019824352115392685, 0.20960219204425812, 0.38446685671806335, -0.06520495563745499, -0.07659882307052612, -0.14814305305480957, -0.0863126739859581, 0.3564395010471344, 0.09305068850517273, 0.008299687877297401, -0.11153343319892883, -0.40056946873664856, 0.00021127320360392332, 0.04561397060751915, -0.18710438907146454, 0.2852074205875397], [-0.08009776473045349, 0.04122133180499077, 0.030318675562739372, 0.23265555500984192, -0.23007814586162567, 0.07035775482654572, -0.15354157984256744, 0.4477023482322693, 0.32579505443573, -0.18920864164829254, -0.15147006511688232, 0.07762936502695084, -0.025331255048513412, 0.36305561661720276, -0.33338966965675354, 0.15241459012031555], [-0.05261113867163658, -0.0030013644136488438, 0.25548678636550903, -0.008347070775926113, -0.103922538459301, -0.12180230766534805, 0.5055045485496521, 0.3179263472557068, 0.1885949969291687, -0.017331814393401146, 0.11761979013681412, -0.5658465623855591, -0.20438221096992493, 0.014834682457149029, -0.23252949118614197, -0.1376376748085022], [0.15217085182666779, 0.1659199595451355, 0.30198851227760315, -0.03171418607234955, -0.18102356791496277, 0.06726571172475815, 0.07053723186254501, 0.37352269887924194, 0.14846333861351013, -0.5012944340705872, -0.26505523920059204, -0.09606817364692688, 0.09873590618371964, 0.2710042893886566, -0.14027507603168488, 0.15615782141685486], [-0.17681629955768585, -0.04986229166388512, 0.1445472687482834, 0.10758531838655472, 0.03215405344963074, 0.2348266988992691, 0.1112079992890358, 0.434081494808197, -0.07005749642848969, -0.4591044485569, -0.2939366102218628, -0.4939855933189392, -0.14061659574508667, 0.17586347460746765, 0.1348259449005127, -0.06934034824371338], [-0.07227589190006256, 0.11209291964769363, 0.07086540758609772, 0.11556887626647949, -0.132229745388031, -0.08247076719999313, 0.1427968293428421, 0.49555307626724243, 0.0485665388405323, -0.1853056102991104, -0.028414297848939896, -0.1418166607618332, -0.16343314945697784, 0.1476549655199051, -0.2751907706260681, 0.140359565615654], [0.012972433120012283, 0.09315905719995499, -0.3594958186149597, 0.4461834728717804, -0.07435042411088943, -0.04321446642279625, 0.2873294949531555, -0.11825749278068542, 0.29176050424575806, -0.20959818363189697, 0.06464685499668121, 0.2807389199733734, -0.053241100162267685, 0.6339088678359985, -0.10025987774133682, 0.13646434247493744], [-0.21170441806316376, 0.05466046929359436, 0.1287839710712433, -0.15219943225383759, -0.042714256793260574, -0.07739951461553574, 0.15975624322891235, 0.3339959383010864, -0.17545375227928162, -0.23057898879051208, -0.1457468420267105, -0.37541720271110535, 0.16148978471755981, 0.005888909567147493, -0.15320833027362823, -0.024219853803515434], [0.14681768417358398, 0.06070219725370407, 0.041308850049972534, -0.03128049150109291, -0.08075841516256332, 0.06289711594581604, -0.06453477591276169, 0.3994731307029724, 0.12297245860099792, -0.19232574105262756, -0.14204077422618866, -0.12026453763246536, -0.06142861023545265, 0.026281755417585373, -0.18335255980491638, 0.17615874111652374], [-0.1328524947166443, 0.10167788714170456, 0.15795622766017914, -0.0676443874835968, -0.019579315558075905, 0.1427694857120514, -0.031790923327207565, 0.5160669088363647, 0.050253789871931076, -0.3328399658203125, -0.16225959360599518, -0.30408892035484314, 0.19191588461399078, 0.04996871203184128, -0.03666885197162628, -0.031037991866469383], [-0.19653993844985962, -0.049628615379333496, 0.17365604639053345, -0.07673534005880356, 0.0918273776769638, 0.11524024605751038, 0.06103627756237984, 0.4156892001628876, 0.036347828805446625, -0.3188367486000061, -0.0180316474288702, -0.31020671129226685, 0.02592235989868641, 0.07610167562961578, 0.010794988833367825, -0.16624568402767181], [-0.2895187437534332, 0.030163150280714035, -0.047087181359529495, 0.16327506303787231, -0.017336174845695496, 0.13246001303195953, -0.1365002691745758, 0.37539881467819214, 0.3078846335411072, -0.33152833580970764, -0.06753937155008316, 0.07305823266506195, 0.10752151161432266, 0.3912259638309479, -0.04420167952775955, 0.01987789012491703], [0.03971954807639122, 0.18299028277397156, 0.12792779505252838, -0.07088357955217361, -0.005020837299525738, -0.09890421479940414, 0.22285768389701843, 0.39458009600639343, 0.3680170178413391, -0.03204430267214775, 0.24389150738716125, -0.2052920162677765, -0.19396860897541046, 0.13348418474197388, -0.14068983495235443, -0.17401957511901855], [0.003248616587370634, 0.11458008736371994, 0.11795172840356827, -0.004718785174190998, -0.06338007748126984, -0.07691600173711777, 0.27762994170188904, 0.3577844202518463, 0.22113627195358276, -0.21848931908607483, 0.009224411100149155, -0.22449804842472076, 0.12506607174873352, 0.21150872111320496, -0.31400758028030396, 0.06141865253448486], [0.05902158096432686, -0.08646396547555923, 0.13093969225883484, -0.07212921231985092, 0.013916299678385258, 0.13318955898284912, 0.05837578698992729, 0.30577757954597473, 0.3660728633403778, -0.5091997981071472, -0.08005249500274658, -0.17965249717235565, -0.019483299925923347, -0.05835361406207085, -0.1511765569448471, -0.19886146485805511], [-0.14202232658863068, 0.04008520022034645, 0.11181598156690598, -0.0727250948548317, 0.2721458077430725, -0.26197734475135803, 0.3955410122871399, -0.4358920753002167, 0.19075575470924377, -0.36840325593948364, 0.18797293305397034, 0.022372109815478325, 0.18728435039520264, 0.2224850058555603, -0.6132774949073792, -0.35217833518981934], [-0.1302972435951233, -0.10756082087755203, 0.26539066433906555, 0.005644684657454491, -0.00020087357552256435, 0.06058207526803017, 0.07536855340003967, 0.12198423594236374, 0.1782878190279007, -0.3955692648887634, -0.21713554859161377, -0.12380754202604294, 0.26896336674690247, 0.10100564360618591, -0.34099632501602173, -0.1034490242600441], [-0.019890300929546356, 0.057133328169584274, 0.19496241211891174, -0.11229006201028824, -0.057966649532318115, -0.0900234580039978, 0.12120101600885391, 0.2171444296836853, 0.18144607543945312, -0.22057808935642242, -0.12553170323371887, -0.14888472855091095, 0.1058381199836731, 0.08490907400846481, -0.23768830299377441, -0.03592100739479065], [-0.18957699835300446, 0.07868823409080505, -0.002480375347658992, 0.15708523988723755, 0.036782052367925644, 0.10370048135519028, 0.07315880805253983, 0.18773314356803894, -0.019613180309534073, -0.29041779041290283, 0.17650555074214935, 0.04996615648269653, -0.20429925620555878, 0.26674607396125793, -0.3774624764919281, -0.0023301183246076107], [-0.11507569998502731, 0.06602257490158081, 0.05061374977231026, 0.12870734930038452, 0.06548348069190979, 0.16257598996162415, 0.0006547110970132053, 0.3717339038848877, 0.10768570005893707, -0.27818727493286133, -0.12469762563705444, -0.09474484622478485, 0.0787014290690422, 0.08462513983249664, -0.14234457910060883, -0.09083112329244614], [-0.044537752866744995, 0.21921779215335846, -0.0018386016599833965, -0.3132134974002838, 0.3114545941352844, 0.05930045619606972, 0.42251187562942505, -0.3090948462486267, -0.01024243701249361, -0.5059075355529785, -0.20836584270000458, -0.3055174648761749, 0.3406403064727783, 0.19408181309700012, -0.3113439381122589, -0.09559066593647003], [-0.010686444118618965, 0.11204522103071213, 0.2664046585559845, -0.11834385246038437, -0.02048821747303009, 0.048342615365982056, 0.012114233337342739, 0.35482412576675415, 0.29773545265197754, -0.30138441920280457, -0.12419228255748749, -0.07884859293699265, 0.267505943775177, 0.05163774639368057, -0.27118122577667236, -0.09504355490207672], [-0.013862079940736294, -0.026607707142829895, 0.14943791925907135, -0.05933210253715515, 0.04201724752783775, 0.15140418708324432, 0.12399741262197495, 0.2281317114830017, 0.34395530819892883, -0.3404354453086853, -0.1176799088716507, -0.23988428711891174, 0.24501779675483704, 0.1824071705341339, -0.23739831149578094, -0.14304588735103607], [0.08475612103939056, 0.17958171665668488, 0.22550003230571747, 0.17156240344047546, -0.10271355509757996, 0.22575487196445465, 0.1982569545507431, 0.26123034954071045, 0.1464368849992752, -0.06871006637811661, 0.07327008247375488, -0.4789588451385498, 0.032400332391262054, -0.10504698008298874, -0.3993109166622162, -0.23536480963230133], [-0.4735935926437378, 0.2155674546957016, 0.2646777629852295, 0.13427461683750153, 0.31350845098495483, 0.04369095340371132, 0.023154575377702713, 0.08678033202886581, -0.2655683159828186, -0.36747777462005615, 0.04510769993066788, -0.037633199244737625, -0.07342397421598434, 0.1261563003063202, -0.1520693600177765, -0.26400089263916016], [0.08091539144515991, 0.20957957208156586, 0.017480330541729927, 0.2817612588405609, 0.03784980624914169, 0.03677697107195854, 0.013774780556559563, 0.11376792192459106, 0.09431428462266922, -0.2547253668308258, -0.06303002685308456, -0.08508262038230896, -0.01181548647582531, 0.11084990948438644, -0.10377378761768341, -0.14830279350280762], [-0.046956032514572144, 0.20446889102458954, 0.18717536330223083, -0.06650851666927338, 0.012044179253280163, -0.1579534411430359, 0.039894040673971176, 0.45374998450279236, 0.07211040705442429, -0.3298696279525757, -0.2574463486671448, -0.07501211762428284, -0.06628762930631638, 0.2348719984292984, -0.2278509885072708, 0.44057387113571167], [-0.34924283623695374, 0.0763435810804367, 0.04659094661474228, 0.1182645708322525, 0.01405185554176569, 0.06107407808303833, 0.1713521033525467, 0.365149587392807, -0.024177327752113342, -0.2753065228462219, -0.09311249852180481, -0.022300442680716515, -0.08818069845438004, 0.02737336978316307, -0.05512595549225807, -0.11131896823644638], [-0.24800750613212585, 0.02251995913684368, 0.3179495632648468, 0.32220155000686646, 0.18771196901798248, -0.02724635601043701, 0.11283200234174728, 0.18011638522148132, -0.01183121744543314, 0.006688747555017471, 0.003634887281805277, -0.2517940104007721, 0.06775211542844772, 0.2236507683992386, -0.22007931768894196, 0.02874741330742836], [-0.1020975336432457, 0.15533451735973358, 0.28604021668434143, 0.04976106807589531, -0.0396585687994957, -0.09848009049892426, 0.08482861518859863, 0.332645058631897, 0.05914877727627754, -0.10444813966751099, -0.06301920861005783, -0.1393660306930542, 0.015603341162204742, 0.09061215817928314, -0.2560754716396332, 0.0754372701048851], [-0.07480854541063309, 0.03928350657224655, 0.19759295880794525, -0.007622242905199528, 0.09135812520980835, 0.019174065440893173, 0.1535378098487854, 0.07875827699899673, 0.005875886417925358, -0.32573097944259644, -0.1430360972881317, -0.15375520288944244, 0.05082803964614868, 0.03436461463570595, -0.251491516828537, -0.16548795998096466], [-0.061243679374456406, 0.07819851487874985, 0.06085232272744179, -0.001317957416176796, 0.08809758722782135, -0.016753042116761208, 0.16570429503917694, -0.08046554774045944, 0.15795035660266876, -0.25729483366012573, -0.12418432533740997, -0.10506388545036316, 0.17416943609714508, 0.006622533779591322, -0.30609530210494995, -0.1807822734117508], [-0.1671440601348877, -0.022027218714356422, 0.10448698699474335, 0.013433084823191166, 0.1162990927696228, 0.1615653783082962, -0.06429820507764816, 0.26339754462242126, 0.1944606900215149, -0.24807219207286835, -0.10300736874341965, -0.07374735176563263, 0.12030111998319626, 0.0516803115606308, -0.10188565403223038, -0.15608561038970947], [-0.1152423545718193, 0.08041561394929886, 0.13168205320835114, 0.05601666122674942, -0.013427028432488441, -0.08673544228076935, 0.17819702625274658, 0.24011285603046417, 0.17038489878177643, -0.14261524379253387, -0.049917008727788925, -0.21816273033618927, 0.08006449788808823, 0.041796281933784485, -0.40601420402526855, 0.016986187547445297], [0.18468676507472992, 0.398222953081131, 0.41550755500793457, -0.012176371179521084, -0.06027578189969063, 0.05116374045610428, 0.012494154274463654, 0.46434083580970764, 0.15483194589614868, -0.021238192915916443, 0.10656043142080307, -0.026608118787407875, -0.396023690700531, -0.03781440109014511, 0.13111737370491028, -0.12095804512500763], [-0.07554620504379272, 0.04508977010846138, 0.17566734552383423, 0.03389253839850426, -0.039867132902145386, -0.07882663607597351, -0.09347273409366608, 0.3030613660812378, 0.34336480498313904, -0.1984737366437912, 0.04086886718869209, 0.0857718363404274, 0.048563241958618164, 0.11264821887016296, -0.24321873486042023, 0.07151338458061218], [0.02823992446064949, 0.17027467489242554, 0.04473832994699478, 0.03506232425570488, -0.15633100271224976, -0.015162687748670578, 0.30893075466156006, 0.21967342495918274, 0.12156007438898087, -0.00022637873189523816, 0.17351900041103363, -0.19876894354820251, -0.12237539142370224, 0.09649107605218887, -0.312016099691391, -0.08106609433889389], [-0.04498903080821037, 0.08708962798118591, 0.020773813128471375, 0.04404172673821449, 0.06450112909078598, -0.03286273777484894, 0.07939967513084412, 0.1304745227098465, -0.010083483532071114, -0.22977618873119354, -0.11622875183820724, -0.09272979944944382, -0.022530274465680122, -0.026193588972091675, -0.1573813557624817, -0.039881542325019836], [-0.141069233417511, 0.04327850788831711, -0.0011910408502444625, 0.17021837830543518, 0.14799055457115173, -0.1180979311466217, 0.10578765720129013, 0.1952633261680603, -0.044456083327531815, 0.04591326415538788, 0.14095360040664673, -0.06769844889640808, 0.09099727123975754, 0.005327992141246796, -0.4059356451034546, -0.1891079992055893], [-0.17547912895679474, 0.18241146206855774, 0.10967716574668884, -0.04376405104994774, -0.1501048356294632, -0.04423336312174797, -0.04002566263079643, 0.34802883863449097, 0.12305716425180435, 0.024781284853816032, -0.01810598559677601, -0.2915734350681305, 0.15017946064472198, -0.2030012458562851, -0.3348507285118103, -0.007848445326089859], [-0.16230754554271698, 0.24107259511947632, 0.18726347386837006, 0.1278550624847412, 0.10822346806526184, -0.37293577194213867, -0.08596548438072205, 0.3741452097892761, -0.1660880446434021, -0.1777779757976532, 0.15396811068058014, 0.6860486268997192, -0.041923049837350845, 0.03332928195595741, -0.4012325406074524, 0.05716176703572273], [-0.14110016822814941, 0.1138269305229187, 0.13259074091911316, -0.1252892166376114, 0.0005175715778023005, -0.0843236893415451, -0.16436555981636047, 0.18096613883972168, 0.06416013091802597, -0.23502568900585175, -0.2439514845609665, 0.15851958096027374, 0.18918290734291077, -0.015240970067679882, -0.1797230839729309, -0.1138271689414978], [-0.0686315968632698, -0.03333674743771553, 0.0008124582818709314, 0.014265671372413635, -0.013682272285223007, 0.27597007155418396, -0.06823161244392395, 0.16658122837543488, 0.227897047996521, -0.2817424535751343, -0.19454388320446014, 0.07266002893447876, 0.06071368604898453, 0.15325452387332916, 0.054449401795864105, -0.16353321075439453], [-0.04801961034536362, -0.02983849309384823, 0.07477805763483047, 0.005300011020153761, -0.12071393430233002, 0.013176096603274345, -0.02890554815530777, 0.23986497521400452, 0.1286441534757614, -0.13839079439640045, 0.07172498852014542, 0.037814509123563766, -0.003667988581582904, 0.21863390505313873, -0.13485586643218994, -0.06872418522834778]]], [[[0.5264384746551514, 0.9609838724136353, -0.7655600905418396, 1.5669718980789185, 0.776878833770752, -0.633483350276947, 0.08277176320552826, 0.592110812664032, -0.4000115990638733, -0.7810429334640503, 2.3143436908721924, -0.17136234045028687, 0.9301018118858337, 0.16298554837703705, -1.8059523105621338, -1.5600194931030273], [-0.19895599782466888, 0.39119410514831543, -0.39530274271965027, 1.2039631605148315, 0.4092063009738922, 0.32699403166770935, -0.3786056935787201, -0.08000438660383224, -0.7692157030105591, -1.23849356174469, 1.264277458190918, -0.6573441028594971, 0.49209973216056824, 0.10416711866855621, -1.100439429283142, -0.5850569605827332], [-0.9570525288581848, -0.4824395477771759, -0.5468652248382568, 1.1530213356018066, -0.3563717007637024, 0.14925216138362885, -0.7830654382705688, -0.5350421071052551, -0.7768912315368652, -0.7969534993171692, 1.1642355918884277, 0.3796856999397278, -0.2845892608165741, 0.826915442943573, 0.8706445693969727, -0.44456055760383606], [-0.03578954562544823, 0.023143868893384933, -0.2937476933002472, 0.8390648365020752, -0.007484149187803268, 0.3811890780925751, -0.04587492719292641, -0.45343711972236633, 0.1422821432352066, -0.5812039971351624, 1.2417255640029907, 0.06466249376535416, 0.3603608310222626, -0.10427597165107727, -0.16436240077018738, -0.2579216957092285], [-1.0271865129470825, -0.798481822013855, -0.6002910733222961, 1.0695558786392212, -0.6621213555335999, -0.10100191086530685, -0.7516472339630127, -0.6814516186714172, -0.43732449412345886, -0.35351768136024475, 1.288732886314392, 1.1052314043045044, -0.4922585189342499, 1.0668329000473022, 1.6991761922836304, -0.46791040897369385], [-0.30722758173942566, -0.17102175951004028, -0.36282259225845337, 0.825359582901001, 0.07348527759313583, -0.09439700096845627, -0.39313241839408875, -0.25696465373039246, -0.2564406096935272, -0.5806469321250916, 1.5383256673812866, 0.7342801094055176, 0.08067157864570618, 0.6459875106811523, -0.06266094744205475, -0.4448840916156769], [-0.16894350945949554, -0.23985856771469116, -0.19251291453838348, 0.5895304679870605, 0.06481078267097473, 0.04907291755080223, 0.060191452503204346, -0.26029014587402344, 0.16225919127464294, -0.47049981355667114, 1.2132612466812134, 0.8669252991676331, 0.33697301149368286, 0.24643614888191223, -0.13646474480628967, -0.12323123216629028], [-0.008180193603038788, -0.061118364334106445, -0.13291822373867035, 0.4971640408039093, 0.17899039387702942, 0.31992119550704956, -0.1226668506860733, -0.38983282446861267, 0.10734912008047104, -0.5839080214500427, 1.2563916444778442, 0.4189068078994751, 0.2683412432670593, 0.14456747472286224, -0.4007933735847473, -0.004901262000203133], [0.05710207670927048, 0.00011039348464692011, -0.4955722987651825, 0.573094367980957, 0.08989410847425461, 0.0500650629401207, -0.17981036007404327, -0.1280200481414795, -0.5867096781730652, -0.3098612427711487, 0.6089550256729126, 0.28892385959625244, 0.07189158350229263, 0.7291896343231201, 0.7981986403465271, 0.17449381947517395], [0.7590750455856323, 0.505630373954773, -0.36086317896842957, 0.40090811252593994, -0.2283446341753006, -0.0002170844527427107, 0.6058163642883301, 0.003226333297789097, 0.01141462940722704, -0.5661019086837769, 1.4994001388549805, 0.8729715943336487, 0.12216757982969284, 0.2178644835948944, -0.8007602095603943, -0.20345735549926758], [-0.6660486459732056, -0.4745798110961914, -0.414651483297348, 0.7515258193016052, -0.3265663683414459, 0.20021803677082062, -0.6450217962265015, -0.548535943031311, -0.6796184778213501, -0.6238813996315002, 0.91713947057724, 0.606132447719574, -0.41181832551956177, 0.9019656181335449, 0.9536612033843994, 0.04432231932878494], [0.43094316124916077, 0.4215935468673706, -0.28890952467918396, 0.3393438160419464, -0.026999477297067642, 0.10342226177453995, -0.024845940992236137, 0.08444511145353317, -0.22848236560821533, -0.44747546315193176, 0.630545973777771, 0.08135329186916351, 0.6995205283164978, 0.5253192186355591, -0.1136670634150505, 0.3921414315700531], [0.4484606087207794, 0.2346254587173462, -0.20532619953155518, 0.10390041768550873, -0.2090310901403427, 0.24338945746421814, -0.20097672939300537, -0.26822319626808167, -0.2208714783191681, -0.5029875636100769, 1.002686619758606, 0.4468727111816406, -0.0631357729434967, 0.7107264995574951, -0.1725439578294754, 0.5918093919754028], [0.01666906103491783, -0.17448456585407257, 0.05901062861084938, -0.22665952146053314, 0.1594037264585495, 0.24250739812850952, -0.5335009694099426, -0.1323399841785431, -0.2990742027759552, -0.6284676194190979, 1.0730172395706177, 1.0507911443710327, -0.2544584274291992, 1.0585689544677734, -0.4045398235321045, 0.8909590244293213], [0.03476106747984886, 0.07603219151496887, -0.40878424048423767, -0.0046799201518297195, -0.0792272761464119, 0.2778240740299225, -0.1776362806558609, -0.16747482120990753, -0.6891071200370789, -0.7471127510070801, 0.9289259910583496, 0.705330491065979, -0.8388323783874512, 0.8649845719337463, -0.21515288949012756, 0.3811543583869934], [0.38772255182266235, 0.2767947316169739, -0.46184390783309937, 0.5508899688720703, -0.21632051467895508, 0.37321990728378296, 0.2159694880247116, 0.5223307013511658, -0.3277747333049774, -0.7009035348892212, 0.3190280795097351, 0.17550452053546906, -0.244469553232193, 0.49868735671043396, -0.24034236371517181, 0.30584344267845154], [0.7993646264076233, 0.25488319993019104, -0.34018540382385254, 0.6380441188812256, 0.034231092780828476, -0.021166302263736725, 0.5415797829627991, 0.12966173887252808, -0.09171446412801743, -0.2799988389015198, 0.8772231340408325, 0.24405862390995026, 0.06780100613832474, 0.2563336193561554, -0.3202045261859894, 0.15728707611560822], [0.8234869241714478, 0.2599869668483734, -0.2565467357635498, 0.1949014663696289, 0.20243078470230103, -0.42913374304771423, 0.5972903370857239, -0.1741209775209427, -0.47037410736083984, -0.7250100374221802, 0.7588462233543396, 0.35192620754241943, -0.6338469386100769, 0.7526379823684692, -1.3029969930648804, 0.10813307017087936], [0.17989149689674377, 0.2448616474866867, -0.5304473042488098, 0.5406490564346313, 0.1633097380399704, 0.19164688885211945, 0.3633485436439514, 0.4159879684448242, -0.07422374933958054, -0.33699533343315125, 0.19479437172412872, 0.20742785930633545, -0.05825970694422722, 0.023107068613171577, -0.2363886535167694, 0.2257726788520813], [0.2673383355140686, -0.09621623158454895, -0.37765246629714966, 0.7548100352287292, 0.3107602298259735, -0.21451105177402496, 0.8534778952598572, 0.41812437772750854, -0.11709258705377579, -0.34636297821998596, 0.502930760383606, 0.4015399217605591, 0.4242185950279236, 0.08018359541893005, -0.5586726069450378, -0.01897936500608921], [0.4028421640396118, 0.10710405558347702, -0.23315906524658203, 0.5094740986824036, 0.032247770577669144, -0.045102596282958984, 0.16443279385566711, 0.10515344142913818, -0.22733989357948303, -0.4877379834651947, 0.49878427386283875, 0.2580309212207794, -0.011172681115567684, 0.5947150588035583, -0.021624205633997917, 0.27855801582336426], [1.191563367843628, 0.32093772292137146, -0.2794927656650543, 0.22199447453022003, 0.46696439385414124, -0.7698664665222168, 0.42560505867004395, -0.19616325199604034, -0.7019123435020447, -0.7961184978485107, 0.6661252975463867, -0.28536441922187805, -0.6184071898460388, 1.4315780401229858, -1.3309170007705688, -0.12873293459415436], [0.3440178334712982, 0.2531343996524811, -0.2490791231393814, 0.3459837734699249, -0.08363376557826996, 0.04073777422308922, 0.294788658618927, 0.20052605867385864, -0.009694363921880722, -0.39326393604278564, 0.36955001950263977, 0.27817174792289734, 0.04023275896906853, 0.6677023768424988, -0.26150253415107727, 0.20299509167671204], [0.5148258209228516, 0.38562342524528503, -0.3353112041950226, 0.25496500730514526, -0.4237425923347473, 0.8955979347229004, 0.6456382870674133, 0.4469701945781708, 0.6466545462608337, -0.812283456325531, 0.3978869616985321, -0.07646744698286057, -0.08342665433883667, 0.5707870125770569, -0.15156090259552002, 0.5425488948822021], [-0.2730228006839752, 0.2063099890947342, -0.5740388631820679, 0.5561710596084595, 0.33253899216651917, 0.06836341321468353, 0.703123927116394, 0.42760857939720154, -0.16988712549209595, -0.5182058215141296, 0.001386673073284328, 0.43309900164604187, -0.10068901628255844, -0.005094070453196764, -0.17219200730323792, 0.3786497116088867], [0.3012358248233795, 0.4160049259662628, -0.4625859558582306, 0.1527266949415207, -0.23949849605560303, 0.2620273530483246, 0.41173553466796875, 0.09568428248167038, -0.21801820397377014, -0.5863215923309326, 0.3568119406700134, 0.2749756872653961, -0.4469679892063141, 0.4776779115200043, -0.362021267414093, 0.1603907346725464], [0.04973027855157852, 0.025367533788084984, -0.4055646061897278, 0.7326223850250244, 0.07082381844520569, 0.07717721909284592, 0.3888724446296692, 0.08578966557979584, -0.2600570321083069, -0.3883311450481415, 0.3766065835952759, 0.0038938026409596205, -0.11519216001033783, 0.6692190766334534, -0.11890096217393875, -0.2185428887605667], [0.45540404319763184, 0.16513094305992126, -0.5178296566009521, 0.6213222742080688, -0.1807437539100647, 0.20853972434997559, 0.4319126009941101, 0.30351701378822327, -0.1359506994485855, -0.39953184127807617, 0.3155477046966553, 0.04506813362240791, 0.05035974085330963, 0.3706567585468292, -0.20122429728507996, 0.14036454260349274], [0.14122098684310913, -0.07114291936159134, -0.45743659138679504, 0.39606887102127075, 0.08477950096130371, 0.11684201657772064, 0.5044931769371033, -0.14306800067424774, -0.17815451323986053, -0.5695062279701233, 0.2739485800266266, 0.07653108984231949, -0.09834172576665878, 0.6049771308898926, -0.323554664850235, 0.13966621458530426], [0.40871918201446533, 0.202418252825737, -0.40133562684059143, 0.2732691466808319, 0.06938023865222931, 0.05002947524189949, 0.19502869248390198, 0.048425644636154175, -0.05122632160782814, -0.19749069213867188, 0.13439668715000153, 0.13122788071632385, 0.15664087235927582, 0.20635655522346497, -0.12157481908798218, 0.2906455099582672], [0.26105427742004395, 0.3264748156070709, -0.016480159014463425, 0.05278931185603142, -0.7361654043197632, 0.024356437847018242, -0.22366245090961456, 0.01102759875357151, -0.3737996518611908, -0.5639952421188354, 0.061535533517599106, 0.29299965500831604, -0.1103036180138588, -0.14067518711090088, 0.250520259141922, 0.7603973150253296], [0.15958048403263092, 0.2668215036392212, 0.07200049608945847, -0.019751595333218575, -0.27361956238746643, 0.5494869947433472, 0.05669013410806656, 0.00792337954044342, -0.10192316025495529, -0.3798384964466095, -0.18113581836223602, 0.013075011782348156, -0.43712177872657776, -0.1623518019914627, -0.4880605638027191, 0.18587923049926758], [-0.10243584215641022, 0.11208322644233704, -0.29635998606681824, 0.39258214831352234, 0.02521318942308426, 0.22956706583499908, 0.6538186073303223, 0.2343284785747528, -0.06878729909658432, -0.32039502263069153, -0.09855148941278458, 0.3760632276535034, -0.0433214008808136, -0.1636458933353424, -0.35908043384552, 0.25739485025405884], [0.3904385566711426, 0.1987019181251526, -0.31926026940345764, 0.5894094109535217, -0.02148492820560932, 0.3271951377391815, 0.450018972158432, -0.01811237633228302, -0.26130127906799316, -0.5210073590278625, 0.042888425290584564, 0.005159294698387384, -0.17490310966968536, 0.10879332572221756, 0.0003717412182595581, -0.07878881692886353], [0.2986930310726166, 0.12586268782615662, -0.029826423153281212, 0.550999641418457, -0.07845043390989304, 0.07667846977710724, 0.3262688219547272, -0.07912872731685638, -0.0811426043510437, -0.18855559825897217, 0.17615295946598053, -0.009326954372227192, -0.2558092176914215, 0.41971418261528015, 0.1229301318526268, -0.21212062239646912], [0.044307976961135864, 0.3425339162349701, 0.18522991240024567, -0.06415724754333496, -0.0005867736181244254, 0.5198726654052734, 0.22564482688903809, 0.23388886451721191, 0.07930272072553635, -0.23720648884773254, -0.26143068075180054, 0.17666670680046082, -0.7755530476570129, -0.2858644723892212, -0.12254523485898972, 0.017313510179519653], [0.3405412435531616, 0.15536914765834808, -0.20535868406295776, 0.10447967797517776, 0.11934597790241241, 0.24129939079284668, 0.34176042675971985, 0.11725273728370667, -0.024294110015034676, -0.36022186279296875, 0.18009065091609955, 0.1486145257949829, -0.34128162264823914, 0.13353173434734344, -0.21013087034225464, 0.09749504178762436], [0.5221157670021057, 0.11010625958442688, -0.4591527581214905, -0.010129678063094616, -0.03482433035969734, -0.00487423874437809, -0.17036178708076477, 0.3625982701778412, -0.3006388545036316, -0.12215971946716309, 0.06279323250055313, -0.4076296389102936, -0.09230513125658035, 0.48147550225257874, 0.3383931517601013, -0.06657695025205612], [0.5735611915588379, 0.23907390236854553, 0.04583703726530075, -0.19712339341640472, -0.4514217674732208, 0.18309152126312256, 0.09514646232128143, 0.3370833992958069, -0.31231704354286194, -0.027318010106682777, -0.5279065370559692, -0.048114094883203506, -0.4741433262825012, 0.4242731034755707, -0.14593110978603363, -0.466876745223999], [0.36810603737831116, 0.1292775422334671, -0.21454279124736786, 0.069135881960392, 0.2722163200378418, 0.05866057798266411, 0.4107133150100708, 0.005594464018940926, 0.01267801970243454, -0.43863800168037415, 0.24711726605892181, -0.056923601776361465, -0.4367798864841461, 0.4683002233505249, -0.5738544464111328, -0.12809456884860992], [0.11810632050037384, -0.018484506756067276, -0.10357869416475296, 0.018035050481557846, -0.350145161151886, 0.1675998866558075, -0.027574613690376282, 0.3180709779262543, 0.11096318066120148, -0.45775148272514343, 0.09131092578172684, 0.06389708071947098, -0.08054085820913315, 0.057569440454244614, -0.07192101329565048, 0.5204393863677979], [-0.038259029388427734, -0.09512817859649658, -0.35802122950553894, 0.15186595916748047, 0.06567837297916412, 0.07534004747867584, -0.020434943959116936, 0.0295691080391407, -0.08601722866296768, -0.23206916451454163, 0.22277332842350006, 0.2275848537683487, -0.259561687707901, 0.28588560223579407, 0.312611848115921, 0.10128726810216904], [-1.1867260932922363, -0.696110188961029, -0.32132476568222046, 0.17139452695846558, -0.030995694920420647, 0.001676888787187636, -0.12760618329048157, -0.3610497713088989, 1.4514425992965698, -0.3455870449542999, -0.2777976393699646, 0.011049182154238224, -0.3257937729358673, 0.2244396209716797, -0.6000751256942749, 0.6144766211509705], [-0.14273720979690552, 0.25943800806999207, -0.2525608539581299, 0.19080741703510284, 0.048789821565151215, -0.438885360956192, -0.3709173798561096, 0.3356269896030426, 0.06636615842580795, -0.38202783465385437, 0.43138357996940613, 0.3362887501716614, 0.3154892325401306, 0.08201904594898224, 0.2784607410430908, -0.07726027071475983], [0.30712980031967163, 0.3800128698348999, -0.10547693073749542, 0.08275017142295837, -0.2748119533061981, 0.03095792979001999, 0.006747037172317505, 0.03965475410223007, 0.013204104267060757, -0.29781582951545715, 0.35525479912757874, 0.21511223912239075, -0.2998727858066559, -0.22385366261005402, -0.28087031841278076, 0.31212317943573], [-0.026825113222002983, 0.0183021891862154, -0.1887122541666031, -0.08910195529460907, -0.17867787182331085, 0.3730849325656891, 0.3806406259536743, -0.09961910545825958, 0.27131685614585876, -0.6829099059104919, 0.04609743878245354, 0.043631505221128464, -0.19318129122257233, 0.2904496192932129, 0.11113385856151581, 0.04086840897798538], [-0.031125899404287338, 0.13604675233364105, -0.23732410371303558, 0.015994751825928688, -0.17351669073104858, 0.30013298988342285, 0.23925602436065674, -0.05959498509764671, 0.5364992618560791, -0.2065754383802414, 0.14712972939014435, -0.20147806406021118, -0.2360013723373413, -0.034622982144355774, -0.24185214936733246, 0.09881988167762756], [0.2940060794353485, 0.1903766542673111, -0.3304431438446045, 0.4031164348125458, 0.045119453221559525, 0.08314080536365509, 0.2850452661514282, 0.10514400154352188, -0.15443947911262512, -0.4068147838115692, 0.25381577014923096, -0.04968731477856636, -0.05797475576400757, 0.22020113468170166, 0.12257841229438782, -0.09545785933732986], [-0.06613197177648544, 0.02957221120595932, -0.2794930040836334, 0.12091035395860672, -0.2100624144077301, 0.11200962215662003, 0.05675899237394333, 0.0965365394949913, 0.06117634475231171, -0.48532047867774963, 0.27145856618881226, 0.047164835035800934, -0.28158268332481384, 0.13613910973072052, 0.19157084822654724, -0.023454222828149796], [-0.05740359425544739, -0.19806279242038727, -0.16987816989421844, 0.05477643013000488, -0.23010964691638947, 0.10257835686206818, 0.1848127394914627, 0.6109292507171631, 0.3273654580116272, -0.5285337567329407, -0.0775182917714119, 0.004532638471573591, 0.16508513689041138, 0.07250025123357773, -0.12500523030757904, 0.08428721874952316], [0.061553485691547394, -0.037437476217746735, -0.04593278467655182, 0.3215276896953583, -0.19515085220336914, -0.0909881442785263, 0.004366606939584017, 0.18889781832695007, -0.0732734277844429, -0.34139254689216614, 0.23585304617881775, 0.31417232751846313, -0.015746667981147766, 0.1556289941072464, 0.2560941278934479, 0.07992904633283615], [-0.6045740246772766, 0.31945452094078064, -0.22114641964435577, -0.1546543389558792, -0.201169952750206, 0.18035969138145447, -0.04026225954294205, 0.41249850392341614, 0.7495445609092712, -0.2556743323802948, -0.1581873744726181, -0.05712486803531647, -0.19053253531455994, 0.06615018844604492, -0.27148059010505676, 0.3381871283054352], [-0.1244443729519844, -0.19594867527484894, -0.2635429799556732, 0.028581516817212105, -0.021498098969459534, 0.07016056030988693, 0.07995939999818802, 0.8414902091026306, 0.3047715127468109, -0.4329524636268616, -0.07783493399620056, -0.050033025443553925, -0.044923752546310425, -0.03287035599350929, -0.05995120108127594, 0.19442714750766754], [-0.1370946615934372, 0.23893262445926666, -0.03346268832683563, 0.1324448138475418, -0.12009170651435852, 0.1540459543466568, 0.2736268639564514, 0.29927176237106323, 0.1788126826286316, -0.12400558590888977, 0.0731998011469841, -0.033747944980859756, -0.1498347371816635, 0.1289258748292923, -0.024092860519886017, 0.13015295565128326], [-0.03689225763082504, 0.1290213018655777, 0.11250665783882141, -0.15043321251869202, -0.16105008125305176, 0.2408185452222824, 0.21079115569591522, 0.05201558396220207, 0.1611100435256958, -0.23994649946689606, 0.01391011942178011, 0.127089723944664, -0.556172251701355, -0.05245134234428406, 0.1885644644498825, -0.04862971976399422], [0.038159798830747604, 0.2760486900806427, -0.2343369871377945, -0.12003488093614578, -0.2247464954853058, 0.07620365917682648, 0.17347808182239532, 0.40521660447120667, 0.4230964481830597, -0.3159758150577545, 0.09709783643484116, -0.022392040118575096, -0.0369085893034935, 0.08559830486774445, -0.1546936184167862, 0.30251315236091614], [0.09483632445335388, -0.12696918845176697, -0.06041351705789566, 0.10761868208646774, 0.0020839243661612272, 0.22043849527835846, 0.32115378975868225, 0.34742212295532227, -0.0648132786154747, -0.2920842468738556, 0.13187119364738464, 0.11050903797149658, -0.2876870334148407, 0.26564276218414307, 0.3690585792064667, 0.18591634929180145], [0.08949508517980576, 0.11299552023410797, -0.11286308616399765, 0.08355185389518738, -0.012389121577143669, 0.11518364399671555, 0.38656800985336304, 0.41412633657455444, 0.023618178442120552, -0.32836976647377014, 0.08844754844903946, 0.06419339776039124, -0.19790704548358917, 0.08836883306503296, -0.019561531022191048, 0.16778850555419922], [0.16818661987781525, 0.22178785502910614, -0.3432735800743103, -0.07129736244678497, -0.002632635645568371, -0.14290527999401093, 0.10528479516506195, 0.14985467493534088, -0.3143742084503174, -0.1575089395046234, 0.05927842855453491, -0.06251812726259232, -0.10109266638755798, 0.0840606614947319, 0.3450082838535309, 0.02807736210525036], [-0.09893253445625305, -0.07722266763448715, -0.033333927392959595, -0.09819067269563675, -0.1647799164056778, 0.017561277374625206, 0.23906365036964417, 0.13845400512218475, -0.006444826256483793, -0.31539902091026306, -0.2019512951374054, 0.0769922137260437, -0.5047855377197266, 0.19570671021938324, -0.19461335241794586, -0.22266997396945953], [-0.03695058450102806, -0.035274893045425415, -0.11848229169845581, 0.16366057097911835, -0.0881265178322792, 0.1709011346101761, 0.1216425746679306, 0.03226234391331673, 0.005009111016988754, -0.4116309881210327, 0.11365145444869995, -0.06962820142507553, -0.2909395694732666, 0.02506687305867672, 0.14722193777561188, 0.056142259389162064], [0.018025314435362816, 0.12758445739746094, -0.1799432933330536, -0.06927142292261124, 0.07419808954000473, 0.07898704707622528, 0.21159547567367554, 0.37891721725463867, -0.09887581318616867, -0.4434569776058197, 0.07350371032953262, 0.01933651976287365, -0.375230073928833, -0.16452699899673462, -0.08751246333122253, 0.19197653234004974], [0.11206799000501633, -0.1429031640291214, -0.1656322032213211, 0.17366287112236023, -0.340092271566391, 0.0857037678360939, 0.14116252958774567, 0.35216572880744934, 0.08452165871858597, -0.4767633378505707, 0.18958348035812378, -0.18802407383918762, -0.12440118938684464, 0.24494722485542297, 0.10166048258543015, -0.03149433061480522], [0.13621091842651367, 0.08786794543266296, -0.20830540359020233, 0.22020335495471954, -0.2266882359981537, 0.08378861099481583, 0.10996818542480469, 0.21342016756534576, 0.10507237166166306, -0.4584876298904419, 0.07545588910579681, -0.18208467960357666, -0.1534394770860672, 0.21590964496135712, 0.15677225589752197, 0.11672455072402954], [-0.072334885597229, -0.04313293471932411, -0.14184632897377014, -0.09835878759622574, -0.12031637132167816, 0.32049399614334106, 0.29694443941116333, 0.1298874020576477, 0.1781160682439804, -0.6346808671951294, 0.1126866266131401, -0.0839373767375946, -0.3479951322078705, 0.26926475763320923, 0.1718265265226364, 0.030157215893268585], [0.0033639532048255205, 0.03292667120695114, -0.09698524326086044, 0.11091402173042297, -0.2661527097225189, -0.013792880810797215, 0.09231062978506088, 0.0926593691110611, 0.20312897861003876, -0.33060309290885925, 0.07722727954387665, -0.07501349598169327, -0.1637112945318222, 0.1893698126077652, 0.02835049480199814, 0.2383272647857666], [-0.3185718059539795, -0.09448466449975967, -0.15590819716453552, 0.09612691402435303, 0.25355643033981323, 0.07345429062843323, 0.08350912481546402, 0.19762066006660461, 0.021551767364144325, -0.3478447496891022, 0.23515932261943817, 0.12594887614250183, -0.23650696873664856, 0.18691661953926086, 0.15608327090740204, 0.09524441510438919], [0.44770705699920654, 0.32725998759269714, -0.09963952004909515, 0.3717910647392273, -0.22181175649166107, 0.11335839331150055, 0.038001637905836105, 0.23080193996429443, -0.1484665870666504, -0.3804571330547333, 0.07090570777654648, -0.025078166276216507, -0.056967027485370636, 0.3960118591785431, 0.21677270531654358, 0.4670269787311554], [0.003784692147746682, 0.08062130212783813, -0.14271025359630585, 0.08339804410934448, -0.13145208358764648, -0.0034071514382958412, 0.03742718696594238, 0.08916176855564117, -0.07176942378282547, -0.3841785788536072, 0.20358410477638245, -0.12831509113311768, -0.22276374697685242, -0.012356655672192574, 0.09451374411582947, 0.15772676467895508], [0.09959407895803452, 0.17704801261425018, -0.07558684796094894, -0.10858316719532013, -0.0359964445233345, 0.10541783273220062, -0.01293774414807558, 0.19434310495853424, -0.10568408668041229, -0.2922531068325043, 0.11402668803930283, -0.0470520555973053, -0.35621386766433716, -0.04719766974449158, 0.17182718217372894, 0.103626549243927], [0.22419416904449463, 0.041346948593854904, 0.0005680997273884714, 0.21560223400592804, -0.14295709133148193, 0.05060100555419922, 0.2440241277217865, 0.2262350618839264, 0.08226300030946732, -0.337456613779068, 0.06936901062726974, 0.14407220482826233, 0.08789637684822083, 0.2943785488605499, 0.08854926377534866, 0.1463743895292282], [-0.014627759344875813, -0.002786914585158229, -0.2502300441265106, -0.009334699250757694, -0.24532783031463623, 0.16454924643039703, 0.26653680205345154, 0.3556954562664032, 0.024939844384789467, -0.42912647128105164, 0.09997262805700302, -0.13293348252773285, -0.20019541680812836, 0.2589126229286194, -0.03166620433330536, 0.03105088695883751], [-0.14516888558864594, -0.13570216298103333, -0.313372939825058, 0.07338472455739975, 0.08838347345590591, 0.052161168307065964, -0.07239779829978943, 0.2943735718727112, -0.134481281042099, -0.47458720207214355, -0.0848245844244957, -0.018048731610178947, -0.187728613615036, 0.05277138203382492, -0.1166406124830246, 0.06655779480934143], [0.25918519496917725, 0.1603047400712967, 0.022966917604207993, 0.3281593322753906, -0.2513967752456665, -0.12897829711437225, -0.015575352124869823, 0.18988002836704254, -0.36296650767326355, -0.21846294403076172, 0.27318233251571655, 0.02831258811056614, -0.02753608115017414, 0.3065165877342224, 0.06207437813282013, 0.3421511948108673], [1.7727883459883742e-05, 0.06431681662797928, -0.1914563924074173, 0.09956285357475281, -0.24723446369171143, 0.03421757370233536, 0.20059435069561005, 0.2019629180431366, 0.15066449344158173, -0.32518139481544495, 0.13807392120361328, -0.06356781721115112, -0.06341972947120667, 0.09073161333799362, 0.11708292365074158, 0.12198028713464737], [-0.11381915211677551, -0.07102234661579132, -0.21491141617298126, -0.2208629697561264, 0.03249213472008705, -0.08896982669830322, -0.12138697504997253, 0.23933134973049164, -0.2663504183292389, -0.4044013023376465, 0.11697813868522644, -0.11026374995708466, -0.35122644901275635, 6.891259545227513e-05, 0.02894187904894352, -0.01687351055443287], [-0.03157886862754822, -0.17921335995197296, -0.21109406650066376, 0.18341977894306183, -0.10914689302444458, 0.03872179612517357, 0.17305432260036469, 0.18678998947143555, 0.0076788607984781265, -0.37645402550697327, 0.06411564350128174, 0.012302590534090996, -0.1483677327632904, 0.09866424649953842, 0.03839050978422165, 0.16405928134918213], [-0.4892500638961792, 0.33371758460998535, 0.09630100429058075, -0.521123468875885, -0.2570679783821106, -0.5522619485855103, 0.20829634368419647, 0.22561101615428925, -0.27704325318336487, -0.06955336779356003, 0.05950979143381119, -0.6023930311203003, 0.05590491741895676, -0.3994874358177185, 0.1960008144378662, 0.2508065700531006], [0.15997785329818726, -0.1997203379869461, -0.2320510596036911, 0.22596506774425507, -0.005480067804455757, 0.03021647036075592, 0.2762814462184906, 0.375252902507782, -0.14119738340377808, -0.34083038568496704, 0.11788474023342133, -0.07045179605484009, -0.05725271254777908, 0.2016938328742981, 0.04172227904200554, 0.0844205915927887], [-0.054989010095596313, 0.05095796659588814, 0.17848654091358185, -0.43169522285461426, -0.3808808922767639, 0.41072070598602295, 0.1554172933101654, 0.14623156189918518, 0.45898380875587463, -0.28820234537124634, 0.04102101922035217, 0.10998661071062088, -0.5605202913284302, 0.13018456101417542, 0.47257283329963684, -0.00634143827483058], [0.013886191882193089, -0.13008740544319153, -0.025542395189404488, 0.007249325048178434, -0.15417276322841644, 0.2304767221212387, 0.010477426461875439, 0.27068325877189636, -0.05273067578673363, -0.12448236346244812, -0.037632159888744354, -0.11761236935853958, -0.2683301568031311, 0.21501456201076508, 0.24171404540538788, 0.214896559715271], [-0.3259129226207733, -0.04242263734340668, -0.21140587329864502, -0.16795828938484192, -0.37269631028175354, -0.2428579032421112, 0.029536841437220573, 0.21347716450691223, -0.055569007992744446, -0.2104002684354782, -0.02575429156422615, -0.2704344689846039, 0.05842914059758186, -0.1072206124663353, 0.33127203583717346, -0.03976357728242874], [0.07386753708124161, 0.0737985372543335, -0.030065933242440224, 0.1434708684682846, -0.22685061395168304, 0.043895892798900604, 0.11057312041521072, 0.19227281212806702, -0.2099866271018982, -0.2197393923997879, 0.09258317947387695, 0.07090764492750168, -0.07754363119602203, 0.26927968859672546, 0.0907970666885376, 0.41183945536613464], [0.14738526940345764, 0.0018410858465358615, -0.13248403370380402, 0.08129093050956726, -0.10230346769094467, -0.0485917292535305, 0.26779305934906006, 0.09576216340065002, 0.02239423431456089, -0.3076780140399933, 0.12693239748477936, -0.01822565123438835, -0.13279420137405396, 0.27750352025032043, -0.21363234519958496, 0.16242097318172455], [-0.13991039991378784, 0.028331227600574493, -0.08655212074518204, 0.05126933753490448, -0.2502901554107666, 0.022532165050506592, 0.024283673614263535, 0.04267733544111252, 0.06389068812131882, -0.29670193791389465, 0.20046551525592804, -0.03995692729949951, -0.10644704103469849, 0.11030977219343185, 0.19894494116306305, 0.11591119319200516], [-0.02690451219677925, 0.036056648939847946, -0.3750496804714203, -0.0726042091846466, 0.2108774483203888, 0.055353179574012756, 0.0792500227689743, 0.34399178624153137, -0.203569233417511, -0.17583294212818146, 0.15197831392288208, -0.14384064078330994, -0.1405242383480072, -0.07634292542934418, 0.02989806979894638, 0.05278472974896431], [-0.010406728833913803, 0.10694769769906998, -0.18246936798095703, -0.07056224346160889, -0.167534738779068, -0.08088099211454391, 0.0826050341129303, 0.13122498989105225, -0.10910457372665405, -0.3149142861366272, 0.09636081755161285, -0.11839865148067474, -0.014824355021119118, 0.08177830278873444, 0.21053250133991241, 0.11098480224609375], [0.0764380395412445, 0.010671108029782772, -0.004770983941853046, -0.02405429258942604, -0.19577892124652863, -0.02963874861598015, 0.04243038594722748, 0.3085092306137085, -0.001834679627791047, -0.33076319098472595, 0.09736752510070801, -0.09484739601612091, -0.0075321742333471775, 0.14246362447738647, 0.16069984436035156, -0.014343040063977242], [-0.11398818343877792, 0.1260567158460617, -0.19722923636436462, 0.041511569172143936, 0.09754754602909088, 0.16789226233959198, 0.1269424557685852, 0.18925799429416656, -0.10963890701532364, -0.34401369094848633, 0.0005464793066494167, -0.06466276943683624, -0.1939028650522232, 0.04735632613301277, 0.058147531002759933, 0.3390159606933594], [-0.051215242594480515, 0.018144696950912476, -0.035758789628744125, 0.26047879457473755, -0.06154404208064079, 0.03401245176792145, 0.15881088376045227, 0.07208024710416794, -0.15730774402618408, -0.32771217823028564, 0.16789042949676514, -0.002799769164994359, -0.11110775917768478, 0.27378901839256287, 0.10012660920619965, 0.22753973305225372], [-0.22275999188423157, 0.008697516284883022, -0.11564429849386215, -0.01607816480100155, -0.1687117964029312, 0.1364569216966629, 0.04134161025285721, 0.06612905859947205, 0.10819754749536514, -0.258541077375412, -0.026732301339507103, -0.008987427689135075, -0.13593854010105133, 0.136610209941864, 0.10966110974550247, 0.22056782245635986], [0.01964685507118702, -0.02727917768061161, -0.14611604809761047, -0.05822474882006645, -0.21018318831920624, -0.015759482979774475, 0.004572473932057619, 0.17446951568126678, -0.1594398468732834, -0.3629792332649231, 0.19961397349834442, -0.028489334508776665, -0.32239285111427307, 0.021616434678435326, 0.12265647947788239, 0.10404971987009048], [-0.0768512487411499, 0.25256630778312683, -0.017938755452632904, 0.036305930465459824, -0.05821582302451134, -0.21755920350551605, 0.0696180909872055, 0.05896802991628647, -0.18859459459781647, -0.14679455757141113, -0.014300167560577393, -0.2786426246166229, -0.014987397007644176, -0.09319064021110535, 0.17011192440986633, -0.0073705208487808704], [0.029090695083141327, -0.17789991199970245, -0.2342529594898224, 0.05090419948101044, -0.11573784798383713, 0.01602795161306858, -0.020781226456165314, 0.25431060791015625, -0.3325590193271637, -0.3301060199737549, 0.3579555153846741, -0.07095922529697418, -0.20326915383338928, -0.013835190795361996, -0.2544127106666565, 0.2308729737997055], [0.021138833835721016, -0.0978233590722084, -0.1516699194908142, -0.03996476158499718, -0.1657775491476059, 0.18715831637382507, -0.009720674715936184, 0.4057796001434326, 0.16977624595165253, -0.13757848739624023, -0.06110569089651108, -0.00013116114132571965, 0.000580279272980988, 0.19656942784786224, 0.35620880126953125, -0.019145717844367027], [-0.09314844757318497, -0.026877041906118393, 0.06873004138469696, 0.059288956224918365, 0.011286129243671894, 0.19692517817020416, 0.03362852707505226, 0.15339863300323486, -0.050043392926454544, -0.03773202374577522, 0.022751830518245697, -0.0035522242542356253, -0.12554536759853363, 0.1584460437297821, 0.18357029557228088, 0.075368233025074], [-0.1312110871076584, 0.028700076043605804, -0.30207860469818115, 0.022065628319978714, -0.33556807041168213, 0.3613378405570984, -0.012950900010764599, 0.23943644762039185, -0.07718779891729355, -0.08799207210540771, -0.004013418685644865, -0.06078094244003296, -0.2382047325372696, 0.49533116817474365, 0.5173271894454956, -0.010754482820630074], [-0.18314848840236664, -0.3094843029975891, -0.23191052675247192, 0.18890368938446045, -0.12591683864593506, 0.17461413145065308, 0.05159877613186836, 0.21039777994155884, -0.16154131293296814, -0.21489685773849487, 0.3898450434207916, -0.14235366880893707, -0.10526669025421143, -0.05001378059387207, -0.024142589420080185, -0.027402598410844803], [0.2635831832885742, 0.2369726300239563, -0.04844808578491211, 0.23719432950019836, -0.06459593027830124, 0.0690053179860115, 0.13842710852622986, 0.12187762558460236, -0.15908099710941315, -0.29086023569107056, 0.1373499631881714, -0.08590451627969742, 0.09989089518785477, 0.2721710801124573, 0.06101163476705551, 0.2202126532793045], [-0.05037842318415642, 0.07293612509965897, -0.20076297223567963, -0.03336518630385399, 0.15758588910102844, 0.05681220442056656, 0.12586615979671478, 0.35988107323646545, -0.08080671727657318, -0.13040490448474884, -0.15854202210903168, -0.1187310516834259, -0.17130626738071442, 0.0883677750825882, 0.14735326170921326, 0.2754538357257843]]]], "dtype": "float32"}, {"__nd__": [[[-0.3505564033985138, -0.5426102876663208, 0.16794636845588684, 1.732257604598999, 1.577834129333496, 2.0046865940093994, 1.8924530744552612, 3.2397100925445557, 2.2144978046417236, 2.9025161266326904, 2.6862924098968506, 3.372039556503296, 2.543606996536255, 2.9742512702941895, 3.104109048843384, 2.7631099224090576, 3.2900257110595703, 3.9938530921936035, 3.824112892150879, 3.3874831199645996, 3.3786518573760986, 3.898688793182373, 3.8379857540130615, 3.4301517009735107, 3.6857752799987793, 3.3325235843658447, 3.8791120052337646, 3.675304651260376, 4.11530876159668, 3.977996826171875, 4.199965476989746, 3.8086705207824707, 4.442379951477051, 4.537628650665283, 4.311715602874756, 3.9998655319213867, 4.203598976135254, 3.9118969440460205, 4.301759719848633, 4.281805992126465, 3.759591817855835, 4.358607292175293, 4.214366436004639, 4.072595119476318, 4.59873104095459, 4.262528419494629, 4.475486755371094, 4.426951885223389, 4.178484916687012, 4.450808048248291, 4.305112361907959, 4.8764214515686035, 4.184743881225586, 4.427650451660156, 4.669093132019043, 5.05511999130249, 4.58315896987915, 4.374515533447266, 4.510813236236572, 4.407557010650635, 4.696520805358887, 4.261454105377197, 4.8348612785339355, 4.735744476318359, 4.383384704589844, 4.5349249839782715, 4.616202354431152, 4.679405689239502, 4.684486389160156, 4.619951248168945, 4.786736965179443, 4.95272970199585, 4.868539810180664, 4.630632400512695, 4.9367499351501465, 4.738496780395508, 5.74763298034668, 4.832208633422852, 5.229931831359863, 5.139452934265137, 5.322988510131836, 4.757528781890869, 4.904352188110352, 4.743802070617676, 5.160854816436768, 4.7289347648620605, 4.797844886779785, 4.810469627380371, 4.865416049957275, 4.822384357452393, 5.15106201171875, 4.760766983032227, 5.031163215637207, 5.010831356048584, 5.2157087326049805, 5.09002161026001, 5.603096008300781, 5.159097194671631, 4.990807056427002, 5.111788749694824]], [[0.13788951933383942, 0.16903837025165558, 0.9157365560531616, 0.9051486253738403, 2.449758291244507, 1.834592342376709, 1.7678847312927246, 2.250934600830078, 2.8609070777893066, 2.5347342491149902, 3.246790885925293, 3.514981985092163, 2.6356260776519775, 3.0747509002685547, 3.254633903503418, 2.8668274879455566, 3.153059720993042, 3.1794426441192627, 3.0627615451812744, 3.6754727363586426, 3.3483505249023438, 3.7668588161468506, 3.3472015857696533, 3.3591737747192383, 3.7640671730041504, 3.491215229034424, 3.716031789779663, 3.9934191703796387, 3.8822505474090576, 3.9994943141937256, 4.452080249786377, 3.7631778717041016, 3.9113693237304688, 3.984675884246826, 4.327300071716309, 3.8905134201049805, 3.7704970836639404, 4.343207359313965, 4.7630205154418945, 4.0352253913879395, 4.017789840698242, 4.042886734008789, 5.2113800048828125, 4.5853424072265625, 4.157127380371094, 4.741497039794922, 4.282037734985352, 4.277505397796631, 4.27747106552124, 4.448457717895508, 4.282925128936768, 4.797131538391113, 4.554254531860352, 4.234549522399902, 4.564208507537842, 4.525844573974609, 4.213825225830078, 4.143630027770996, 5.151317596435547, 4.434988975524902, 4.665405750274658, 4.447225093841553, 4.5902581214904785, 4.406547546386719, 4.796529293060303, 4.329684734344482, 4.413061618804932, 4.8686041831970215, 4.514459609985352, 4.906661033630371, 4.689298152923584, 4.585915565490723, 4.6660003662109375, 5.134589672088623, 4.524377822875977, 5.2274346351623535, 4.8247199058532715, 5.368717193603516, 5.05872917175293, 5.079237461090088, 4.734795093536377, 5.489293098449707, 4.7525529861450195, 4.932735443115234, 4.769742012023926, 5.496558666229248, 4.734997272491455, 4.926766395568848, 4.793905735015869, 4.6517863273620605, 4.895308494567871, 4.893279075622559, 4.993531703948975, 5.117990493774414, 5.159120082855225, 5.17233943939209, 5.163593769073486, 5.141720771789551, 5.168652057647705, 5.036965370178223]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[[-1.370058536529541, -0.5879568457603455, -1.502514362335205, 0.16637291014194489, 1.5311920642852783, 0.21448959410190582, -0.6697167754173279, 0.2330651730298996, 0.564822793006897, -0.6629310846328735, -0.012343832291662693, 0.4038533866405487, 0.10934454947710037, 1.6557015180587769, -0.5961175560951233, -0.6913192272186279, -0.1616685688495636, 0.19830670952796936, 0.06357932090759277, -0.9075645208358765, 1.6197878122329712, -0.17237626016139984, -0.02010730467736721, -1.8721071481704712, 1.218940019607544, 0.5464960336685181, -0.4539892077445984, -0.009606677107512951, -0.04999721422791481, 0.5726111531257629, -0.15838412940502167, 1.5644482374191284, 1.548439621925354, 0.42789024114608765, -2.2609879970550537, -2.0572988986968994, -1.3547372817993164, 1.3961637020111084, -0.05140925198793411, -0.6651734113693237, 0.16898010671138763, 0.205722376704216, -0.6100446581840515, 0.0301965419203043, -1.081830382347107, -0.4688011109828949, 0.08105766028165817, 0.9169610142707825, 1.1148993968963623, -0.11925458163022995, 0.3887290954589844, 0.7464841604232788, -0.2481316328048706, -0.0020668930374085903, 0.3321312367916107, 1.5171948671340942, 0.11047922819852829, 1.1072137355804443, 1.2408032417297363, 1.1477065086364746, -0.5216225981712341, -0.4703063666820526, 0.7283271551132202, 0.9473475813865662], [1.378595232963562, 0.4758325219154358, 0.03260981664061546, -0.035367853939533234, 0.21037334203720093, 0.20165233314037323, -0.7618257403373718, 1.0073219537734985, -0.6368894577026367, -2.045698881149292, 1.4721063375473022, -1.1123861074447632, -0.9582911729812622, 0.3010748326778412, -0.36427488923072815, 0.5366742610931396, -0.7195695638656616, 0.055637430399656296, -0.3062596321105957, -1.0289424657821655, -0.5536255836486816, 0.4536896347999573, -0.017046503722667694, -0.5525435209274292, 1.485633134841919, 0.8239853978157043, 0.40640369057655334, 2.027897357940674, -2.0190112590789795, 1.0018668174743652, -0.5258347988128662, 1.2971874475479126, -0.4117576777935028, 0.5331856608390808, -0.029625670984387398, 0.8434082865715027, 0.25159865617752075, -0.11905710399150848, 0.4264087975025177, -1.6937472820281982, 0.6102964282035828, -0.6783223152160645, 0.5853421092033386, -0.5597332119941711, -0.9078384041786194, 0.5197678208351135, 0.9960188269615173, -1.260785698890686, 0.1666446328163147, -1.167965054512024, -0.669610321521759, 0.9570968151092529, 0.9904149174690247, -0.8772154450416565, -1.4519869089126587, 0.8057480454444885, 0.6399925947189331, -0.02120456099510193, 1.0890711545944214, 0.6530075669288635, 0.9184770584106445, -0.4445347189903259, -0.6216778755187988, 1.3001600503921509], [-0.7521504759788513, 3.4455833435058594, -0.5245345830917358, 0.8836036920547485, 0.09111487120389938, -1.3778231143951416, 0.15210723876953125, -0.005030334927141666, -0.4772002398967743, -0.5360614061355591, -0.5958536863327026, -1.172239065170288, 0.8793954849243164, 1.4977216720581055, 1.0700634717941284, -1.710484266281128, 0.3690609931945801, -0.29425132274627686, 1.5843043327331543, -0.7538123726844788, 0.5922998189926147, -0.08816581219434738, 0.6640022397041321, 2.051687002182007, 1.9802641868591309, 0.42197278141975403, -0.8398612141609192, 2.168281078338623, -2.204319715499878, -0.02816067636013031, 1.4578187465667725, -0.2602345645427704, -0.43538275361061096, -0.19510185718536377, 1.4065481424331665, -0.7095956206321716, 2.196002960205078, 1.1531378030776978, 0.08667822182178497, 0.801421582698822, -0.3846682906150818, -0.34361934661865234, 0.2008230835199356, -1.0863752365112305, 0.5603545308113098, -0.655731201171875, -0.324146032333374, 0.2757314145565033, -0.8418743014335632, -0.3255428373813629, -0.023197956383228302, 1.3033570051193237, 0.10620037466287613, 0.8147671222686768, 1.4852709770202637, 0.6925541758537292, 1.4935728311538696, 0.6963328123092651, -0.8325705528259277, 1.364992618560791, -1.3134167194366455, 0.9713249802589417, 0.046062909066677094, 0.5493705868721008], [-0.6364585757255554, 0.7315975427627563, 0.2853121757507324, 0.16446812450885773, -0.2536971867084503, -2.3689849376678467, -0.2827795445919037, -1.3095779418945312, -0.9376201629638672, -0.33682432770729065, -1.160157561302185, 0.5682568550109863, 1.2421387434005737, 0.6688330769538879, -1.0059404373168945, -0.225439190864563, -0.7643522024154663, -0.8721856474876404, -1.8235132694244385, 0.03029632195830345, 0.02923281118273735, -1.8123444318771362, -1.4058946371078491, -0.11554373055696487, -0.08230702579021454, 2.3042311668395996, -0.27918216586112976, -1.7919883728027344, -0.0988953486084938, -0.9787543416023254, -0.8929929733276367, 0.023162931203842163, 0.5112975239753723, -0.4294256865978241, 1.6269152164459229, -1.429545521736145, -0.4256143867969513, -0.7389616966247559, -0.03430217504501343, -0.4895874261856079, -0.8145567178726196, -0.6055353879928589, -1.7099432945251465, 0.3514788746833801, 1.3677637577056885, 1.9815129041671753, -0.7106155753135681, 1.4071699380874634, 0.6253969669342041, 0.8246644139289856, -0.23332472145557404, -0.5225895047187805, 0.23816923797130585, -1.056643009185791, -0.6872238516807556, -0.6411012411117554, 0.5779575109481812, -1.0686477422714233, -0.7677693367004395, 0.5683121085166931, -0.6636968851089478, 0.0772920623421669, -0.3195269703865051, 0.2806141972541809], [-0.05207225680351257, 0.8577948212623596, -0.8540801405906677, -0.41082099080085754, 0.48030081391334534, -0.6304622888565063, 0.4030364155769348, -0.5572888851165771, -0.39862126111984253, 1.503382921218872, -0.9109518527984619, -0.6880726218223572, -0.308710515499115, 1.371031641960144, 0.16632765531539917, -0.4193688631057739, 1.3011066913604736, 0.11782350391149521, 1.321511149406433, -0.6301447153091431, -1.8996859788894653, -1.5973306894302368, 0.27060145139694214, 0.23638784885406494, -0.9862188100814819, -0.01428561843931675, 1.9345948696136475, -0.19857758283615112, 0.7311981916427612, 0.38871562480926514, 2.0322256088256836, -0.5754297375679016, -1.1676383018493652, 0.14638841152191162, -0.162979856133461, -0.1133371964097023, 0.3580965995788574, -0.7058637142181396, 0.21873442828655243, 0.6119241118431091, 1.0091135501861572, -0.07800193876028061, 0.8008320331573486, 0.7672320008277893, -0.9455108642578125, 0.5848893523216248, 0.809206485748291, -0.9828126430511475, 0.5778599381446838, -0.1277671605348587, -0.1603727638721466, -1.5894109010696411, -0.4358034133911133, -1.7085152864456177, 0.7189631462097168, -0.24123553931713104, -0.6718295216560364, -0.2412252426147461, -0.123560830950737, 0.2476208209991455, 0.12668566405773163, 0.4924798309803009, 1.0561693906784058, -0.18185845017433167], [1.9397945404052734, 1.0853432416915894, 1.3051286935806274, 0.09848209470510483, -0.6068784594535828, -0.292449027299881, 0.3354359269142151, -0.3623450994491577, 1.6857662200927734, -2.527860403060913, -0.302324116230011, -0.315466970205307, 0.3376176357269287, -2.7094154357910156, 0.17793430387973785, -1.2120152711868286, -0.028755448758602142, -0.7865321636199951, 0.727974534034729, 0.5270065069198608, 0.701690673828125, 0.7014594078063965, -0.2670707702636719, -0.5462040305137634, -0.6041191816329956, -0.14513613283634186, -0.8653408885002136, 0.8967368602752686, -1.0926737785339355, -0.02082670107483864, -2.012624979019165, -0.45371752977371216, 0.20514358580112457, 0.042044784873723984, -1.4645211696624756, -0.4457690715789795, 0.7969598174095154, -0.8184024095535278, -0.8229978680610657, -2.883734703063965, 0.5747581124305725, 0.325568825006485, -0.11944134533405304, -0.9096273183822632, 1.1034449338912964, -1.3184380531311035, 1.7460743188858032, 0.8038305640220642, 1.3429409265518188, 0.9579473733901978, -1.00519859790802, 0.1190534308552742, -0.15652965009212494, -0.23073188960552216, 0.879822313785553, -0.12980082631111145, -1.1203781366348267, 0.14022748172283173, 1.2023510932922363, -1.3546338081359863, 1.2359437942504883, 0.8758013248443604, -2.0142340660095215, 1.1032507419586182], [-0.698693037033081, 0.4625265598297119, -0.8906804919242859, 0.31577858328819275, 0.04123103246092796, -0.3425983190536499, -0.7880545854568481, -0.02341447025537491, -0.3156677782535553, -0.5371732711791992, 0.13201606273651123, 0.0561467669904232, 0.3102770447731018, -1.0153473615646362, -1.4319288730621338, -1.400438904762268, 0.14660803973674774, 1.1578625440597534, -1.2403967380523682, -0.9355631470680237, -0.7193537354469299, 0.3872933089733124, 0.8581750988960266, 2.483583688735962, -0.4114287793636322, -0.38676849007606506, 1.3259531259536743, -0.19998222589492798, 0.6879836916923523, 0.25556591153144836, -0.16781945526599884, -0.1404673457145691, -1.2579102516174316, -2.7182817459106445, -1.2067747116088867, -0.4324388802051544, 0.46686962246894836, -1.0290181636810303, 1.0621894598007202, 0.7870644927024841, 1.270957112312317, -0.03319524601101875, -0.4238884747028351, -0.6357313394546509, -0.8516936302185059, -0.01984364539384842, -2.0367062091827393, -2.055737018585205, -0.004399778321385384, -0.5485220551490784, -0.7561668753623962, 0.9661426544189453, 0.02672274224460125, -0.8029084205627441, 0.6838753819465637, 0.5796257853507996, 0.4011611342430115, 0.5896127820014954, 1.0299696922302246, -0.4757952094078064, -1.6265653371810913, -0.002029198221862316, -0.5318458676338196, -0.5179759860038757], [-0.3826941251754761, 1.139696478843689, 1.709552526473999, 0.4767869710922241, 0.8532151579856873, -1.0790435075759888, 1.578329086303711, 0.024236371740698814, 0.39373934268951416, 1.382655382156372, 1.3220787048339844, 1.1635831594467163, -0.22239114344120026, 0.160675510764122, -0.16252177953720093, -0.1161544919013977, -1.206377387046814, 0.7160005569458008, 1.1622506380081177, 0.01007179357111454, -0.6772481799125671, -1.5911449193954468, 0.44375845789909363, -1.0365089178085327, 0.16648603975772858, 0.05526704341173172, 1.2767829895019531, 1.9907854795455933, 0.7655178904533386, -0.9426007270812988, -0.8809289336204529, -0.4919731020927429, -0.6670589447021484, 0.34688809514045715, 1.82762610912323, 0.3181239068508148, 0.2884209454059601, -0.7206515073776245, 0.4923332929611206, 1.0344873666763306, 1.2578214406967163, 1.3785896301269531, 0.44521939754486084, -2.0580129623413086, 1.4291088581085205, 0.13380634784698486, -0.6607062816619873, 1.0887775421142578, 1.32974112033844, -1.5379235744476318, 1.2469724416732788, 1.6049513816833496, -1.6878741979599, 0.2818850576877594, 2.1040396690368652, -1.4081753492355347, -0.2578747272491455, 0.03955027088522911, -1.642783284187317, -0.1069408431649208, -2.184011220932007, 0.299899160861969, -1.774222493171692, -0.8921471834182739], [-0.41796135902404785, -1.0746909379959106, -0.17945146560668945, 1.0005220174789429, -1.4904813766479492, -1.2700132131576538, -1.107548713684082, -0.5970859527587891, -0.6980898976325989, -0.31128814816474915, -1.7782419919967651, 0.7935945391654968, -1.3347615003585815, -0.5122609734535217, 1.6134878396987915, -0.3300796151161194, 0.1675792634487152, 0.895770788192749, 0.4020364582538605, 0.6638374328613281, -0.2864813804626465, 0.6038784384727478, -0.21674367785453796, -0.45899322628974915, -0.9352098107337952, 1.3540242910385132, 1.2492575645446777, -0.7968605160713196, -0.4555066227912903, 1.251168131828308, 1.381386399269104, -0.01975698582828045, -0.13236884772777557, 0.5332053899765015, -1.3911253213882446, -1.1918525695800781, 1.3329523801803589, 0.057537563145160675, -2.6445703506469727, -0.14616884291172028, 0.9315137267112732, 0.028189312666654587, -2.1835367679595947, 0.7807531952857971, 0.7106959819793701, -1.4705458879470825, -0.7696529030799866, -0.8626565337181091, 0.6185007691383362, -0.8538914322853088, -1.6926305294036865, 0.36256030201911926, -1.3781732320785522, -0.44987520575523376, 0.48452892899513245, -1.6899853944778442, -0.26481691002845764, -1.4533724784851074, -0.2158975899219513, 0.37276268005371094, -0.5681220293045044, 0.5169311761856079, 0.8390988707542419, 0.2737114131450653], [0.7369569540023804, -0.6964616179466248, 1.3309296369552612, 0.5194447636604309, 1.88864004611969, -0.7353415489196777, -0.562619149684906, 0.7078731656074524, -0.6970364451408386, 1.1973522901535034, 0.9800843596458435, 0.26095643639564514, 0.22082599997520447, -1.5170648097991943, 0.5313991904258728, -0.21746599674224854, 0.7274262309074402, 0.6178433895111084, 1.9072898626327515, -0.9229642748832703, 1.294385313987732, 0.010417300276458263, -0.3492317497730255, -0.07279374450445175, -0.19512073695659637, 0.7562322020530701, -0.36303651332855225, 1.1945117712020874, 1.828324794769287, 0.6151494383811951, -0.9118992686271667, 1.0508196353912354, -0.7648095488548279, -0.07050985097885132, -2.7376627922058105, 0.7499747276306152, 1.0193971395492554, 0.658387303352356, 1.778863549232483, 1.459733486175537, 0.11281689256429672, 0.536561906337738, -0.7352763414382935, 0.31535646319389343, 0.8880329728126526, 0.9534261226654053, 3.1727192401885986, 3.353429079055786, -1.5484710931777954, 1.419543981552124, 0.36017417907714844, 1.295756220817566, -1.289762258529663, 0.1803424209356308, 0.9287723302841187, -1.3767950534820557, 2.604949951171875, -0.7857267260551453, 0.8226247429847717, 0.43711623549461365, 0.2765576243400574, -0.9528444409370422, -0.21980486810207367, 0.19284938275814056], [-0.7575167417526245, 1.0276639461517334, 1.7352455854415894, 1.7660858631134033, -0.5155335664749146, 0.180616095662117, -0.7123876810073853, 1.0412989854812622, 0.6317008137702942, 0.036748211830854416, -0.07087084650993347, -0.20155549049377441, -0.2713949382305145, -1.4349339008331299, 0.9555802345275879, -0.02428891323506832, -1.3004127740859985, 0.5867255330085754, -0.5365208387374878, 0.858332097530365, -0.20785345137119293, -1.7715550661087036, -0.34320735931396484, -0.6483829021453857, -1.5600136518478394, 0.9510555267333984, 0.3832393288612366, -0.43465083837509155, -0.7229474186897278, 0.4660070836544037, -0.1836390346288681, 0.6731738448143005, -0.43203818798065186, 1.2330269813537598, -1.4673027992248535, -0.9090335965156555, -1.3504385948181152, -0.02458307519555092, 1.0050159692764282, 0.1282263547182083, -1.4133301973342896, 0.14664888381958008, 0.3946624994277954, 0.7267865538597107, -0.7981359362602234, -0.888361930847168, -1.368424654006958, -1.3977941274642944, -0.4060153067111969, 0.4088038504123688, -1.793464183807373, -0.8434560298919678, 0.5910720229148865, -1.6169852018356323, 0.712252676486969, -1.6197153329849243, 0.24928610026836395, -0.617096483707428, -1.4860919713974, 1.4714690446853638, 0.052621982991695404, -1.0023576021194458, -0.6249405741691589, 1.1901609897613525], [0.16050393879413605, 0.4074215292930603, 0.5146529078483582, 0.008159194141626358, 0.4602412283420563, -1.0200518369674683, 0.4827055037021637, -0.5958649516105652, 1.4069629907608032, 0.1408660113811493, -0.5458800196647644, 0.5346534848213196, 1.1701751947402954, 0.2201026976108551, 0.17737308144569397, 0.4003283977508545, 1.4755545854568481, -1.8364920616149902, -0.7878170013427734, -0.5589385628700256, 2.062896728515625, 0.7293491959571838, -0.6410693526268005, 0.21180738508701324, 1.0810887813568115, -0.10101982206106186, 2.2614634037017822, -1.1208211183547974, 2.2009761333465576, 0.4240923821926117, -1.1467475891113281, -1.2872852087020874, -1.0266815423965454, 0.48369133472442627, 1.3396995067596436, -1.524255394935608, 0.49406421184539795, 0.4548969566822052, 1.4133645296096802, 0.40853896737098694, -0.41617387533187866, -1.2158256769180298, 1.039961814880371, 0.3695409595966339, -2.0165882110595703, 1.0549219846725464, -0.3196476101875305, -0.23448339104652405, 0.24275930225849152, -0.5188391208648682, 0.11454799771308899, 0.29546403884887695, 0.47582802176475525, -1.2200586795806885, -1.231872320175171, 0.1942722648382187, 0.9139917492866516, -0.2740544080734253, -0.3875791132450104, 0.02036600187420845, -0.17559117078781128, 0.6905619502067566, 1.6120470762252808, -1.0741697549819946], [-2.1438777446746826, 1.5970649719238281, -2.2466022968292236, -0.6524283289909363, -1.2321784496307373, 1.3986157178878784, 0.42409682273864746, -0.3341137170791626, 1.1547467708587646, 0.929082989692688, 1.070886492729187, 1.1620887517929077, 1.8747555017471313, -0.4517441689968109, 1.4013192653656006, -0.9938770532608032, -0.9339568018913269, -1.082625389099121, 1.017885446548462, 0.05108106508851051, -1.6395286321640015, -0.30288824439048767, 1.2930914163589478, -0.03918146342039108, -0.4095781743526459, -0.8123607635498047, 1.2558842897415161, 0.4557417333126068, -1.0998305082321167, 0.6080530285835266, 1.1691877841949463, -1.5028901100158691, 0.731826901435852, 1.530198574066162, 0.3957033157348633, -0.05022197589278221, -0.3557826280593872, -1.2566052675247192, 0.6412049531936646, 1.5442354679107666, 1.6174836158752441, 0.156499445438385, -0.2671438455581665, -1.0933352708816528, 0.37124595046043396, 0.30139464139938354, -0.975073516368866, 0.07677400857210159, -0.3829438090324402, 0.6582315564155579, -1.0031588077545166, 1.0822557210922241, -1.1932966709136963, -0.37299343943595886, 0.5966276526451111, 0.32045042514801025, 1.1559383869171143, -1.5847043991088867, 0.10962536931037903, 0.6998414397239685, 0.20023837685585022, -0.49726641178131104, -1.963618278503418, 0.8917320370674133], [0.626569390296936, 1.0912336111068726, -0.3850831389427185, 0.19499124586582184, -0.022422200068831444, -0.861530065536499, -1.0592044591903687, -0.10085360705852509, -1.7919297218322754, 1.480322003364563, 0.10661926120519638, -0.11146547645330429, -1.6700057983398438, 1.1937533617019653, 0.6078963875770569, -1.6329232454299927, -0.559544026851654, -1.5192782878875732, -1.074129343032837, 0.7846699357032776, 0.8149898052215576, -0.9039285778999329, 0.09704666584730148, -1.1395295858383179, 0.23683808743953705, 0.11060584336519241, 1.4918149709701538, -0.8500811457633972, -0.5201418995857239, -0.9670723080635071, 1.093671202659607, -1.2793924808502197, 0.930683434009552, -1.5720561742782593, 0.006572572980076075, 0.1603013575077057, 0.3028605580329895, -0.5833982825279236, 0.5176328420639038, 1.0696673393249512, 1.901015043258667, -0.2351706624031067, 0.2483578771352768, -0.8915964365005493, 0.03729262575507164, -0.5851566195487976, -0.02762470208108425, -0.6436660885810852, 0.5890169143676758, 0.02692888118326664, 1.1588435173034668, 0.8063227534294128, -0.48528268933296204, 0.5086537003517151, -2.3421671390533447, -0.6522974371910095, -0.8623486757278442, -0.07238930463790894, -1.4036366939544678, -0.586205005645752, 0.8837509751319885, 0.5965719223022461, 1.2988637685775757, -0.1336718201637268], [-0.8134807348251343, -1.3644551038742065, 0.3727726340293884, 0.5061157941818237, 0.15475451946258545, -0.14447297155857086, -0.6273181438446045, -3.498380422592163, -0.29644185304641724, 0.5712807774543762, -1.1419142484664917, 0.43171361088752747, -0.8591311573982239, 1.2771319150924683, -0.6915903091430664, 0.981475293636322, -1.3356858491897583, 0.2897227108478546, 0.2610912621021271, -2.0451273918151855, -2.2221388816833496, -1.033972144126892, -0.503118097782135, -0.04174518212676048, 0.6931251883506775, -0.43907979130744934, -0.28711166977882385, -1.9952868223190308, 1.5903652906417847, -0.8058093786239624, 0.283210813999176, 1.8396704196929932, -0.255243182182312, 0.737546443939209, 1.9635032415390015, 1.96855890750885, -0.9637091159820557, 1.2918763160705566, -0.7691627144813538, -0.2324080765247345, 0.8327487707138062, 0.16315436363220215, -1.3376712799072266, -0.19497330486774445, 0.31301769614219666, 1.0287102460861206, -0.9369550943374634, -0.12137481570243835, -0.5673749446868896, 1.0764660835266113, 0.29853934049606323, 0.24185310304164886, -0.5865044593811035, 0.9186873435974121, -0.6016058921813965, -0.33589065074920654, -0.5757680535316467, -0.5732900500297546, 0.4691968560218811, 2.373883008956909, 0.35224175453186035, -0.21509060263633728, 0.12553417682647705, 1.0592557191848755], [0.8861691951751709, 1.3119733333587646, 0.14145570993423462, 1.6261733770370483, -0.20345845818519592, -0.07394026219844818, -2.3789117336273193, -0.3052769899368286, 0.19578252732753754, -1.8133693933486938, -0.11060594022274017, -0.5011626482009888, 0.8475383520126343, -0.28856921195983887, -0.5390604734420776, -3.049657106399536, -0.025350814685225487, -0.24669264256954193, 2.1764094829559326, -0.3998563289642334, -3.0820517539978027, -0.4656826853752136, -1.000588297843933, -0.1832301765680313, 0.3761623501777649, -0.7231963276863098, 0.6358408331871033, 1.7903761863708496, -0.015866803005337715, -1.335779070854187, 0.2623099088668823, 0.37459874153137207, 0.3382357954978943, 0.0354667566716671, 0.14211134612560272, -0.4981233775615692, 0.9406378865242004, 0.32740554213523865, 2.600865125656128, -1.6277787685394287, 0.3055301606655121, 0.32799282670021057, -0.23195770382881165, -0.012647824361920357, 1.8411736488342285, -0.39229655265808105, -0.6361029744148254, 0.03309793397784233, 1.531242847442627, 0.8088700771331787, -0.9355842471122742, 2.3930065631866455, -0.23191091418266296, -0.37379610538482666, 0.8822566270828247, 0.4006710946559906, 0.3616648316383362, 0.633357584476471, -0.1428258866071701, -0.05968543514609337, -0.48886731266975403, -2.434258460998535, 0.22443170845508575, -1.2292861938476562], [-1.3588590621948242, 0.7862247228622437, 1.657854676246643, 1.7862210273742676, -0.6583777070045471, 0.6994473934173584, 0.719597578048706, -0.30842405557632446, 1.0703767538070679, 0.8765100240707397, 0.9912036061286926, 0.9171043634414673, -0.1698867678642273, -0.9766726493835449, -0.7571990489959717, 0.3168984651565552, -0.4241204261779785, -1.4331661462783813, -1.5500257015228271, 0.5291951298713684, -0.059452228248119354, -0.4250136613845825, 0.5743505358695984, -0.3126089870929718, -0.6535664796829224, 0.1586182862520218, -0.558926522731781, 1.6383068561553955, -0.1748291701078415, 0.02252482809126377, -1.6812955141067505, -0.20846045017242432, 0.9163282513618469, 0.8056768774986267, -1.9258712530136108, 0.4763026237487793, 0.8343164324760437, 1.0105299949645996, -1.1939934492111206, 0.4204978346824646, 0.2720978260040283, -1.2437820434570312, 0.6547191143035889, -0.16144151985645294, 1.1404694318771362, -1.2161405086517334, -0.48788583278656006, 2.2815396785736084, -1.0520524978637695, -0.7241337299346924, 1.303520679473877, -1.8761307001113892, 0.8340445160865784, 0.9065616130828857, -0.8973287343978882, 0.7238656878471375, 0.7086912393569946, 0.9223244190216064, -1.1139662265777588, 0.8242486119270325, 0.6154112219810486, 0.1586785763502121, 0.2517513036727905, -1.148383617401123], [0.24542981386184692, -0.24507474899291992, 0.07421305030584335, -0.7640449404716492, 1.3601422309875488, 0.36017054319381714, -0.2055749148130417, 1.1987839937210083, 0.04129061475396156, 0.4339326024055481, 1.04399836063385, -1.1982020139694214, -0.7912289500236511, -0.37294408679008484, -0.2686121165752411, -0.20628879964351654, 0.6039243936538696, 0.030940763652324677, 0.043587975203990936, -0.18178167939186096, -0.8088198900222778, 0.679798424243927, 1.2498623132705688, 1.0903544425964355, 0.12717847526073456, -0.428140789270401, 2.068857431411743, -0.03604806959629059, -1.8713154792785645, -0.2894080579280853, 0.16210582852363586, 0.5266361832618713, 3.6496009826660156, 1.5820753574371338, -0.8984948992729187, -0.27054277062416077, 0.7595963478088379, 0.20734252035617828, 0.2506002187728882, 0.4687146842479706, -0.5121101140975952, 0.6932440996170044, 0.06178925186395645, -0.6265631318092346, -0.8361020088195801, -0.7841773629188538, 1.5030405521392822, 1.1707382202148438, -0.3112821578979492, -0.5442565679550171, 0.3667006194591522, 0.6766461730003357, 0.3426288664340973, 1.2664653062820435, 0.2594621777534485, -0.942671000957489, 0.5534142255783081, -1.2478408813476562, 0.94892418384552, 0.6720461845397949, -1.0012357234954834, 0.6261153221130371, -0.37400662899017334, 2.145676612854004], [-1.1439152956008911, 1.5673495531082153, -0.14542797207832336, 0.9580941796302795, 0.21687345206737518, -0.41982170939445496, -2.255753755569458, -0.46500760316848755, 0.024590134620666504, -1.0673539638519287, 1.4776811599731445, -0.14949612319469452, 2.7369089126586914, -1.1397311687469482, -0.8765159845352173, -0.7522352933883667, -0.8498299717903137, -1.4086675643920898, -0.7051848769187927, 0.7909004092216492, -2.0464301109313965, 0.6475104689598083, -1.6815556287765503, 1.0866843461990356, -1.2233847379684448, 0.5427005290985107, -0.9262780547142029, -1.2054965496063232, -0.43079981207847595, 0.455021470785141, 0.15533143281936646, -1.6293761730194092, 1.8153427839279175, -2.1836953163146973, -0.13595154881477356, -0.5966192483901978, 1.6074886322021484, -1.074737548828125, -0.10082230716943741, 0.6535438895225525, 0.5907131433486938, 0.8099566102027893, 0.0791742205619812, 0.5843701362609863, -0.9884233474731445, -0.31356415152549744, 2.1097493171691895, -0.6657274961471558, 0.9926702976226807, -1.607795238494873, -0.7129000425338745, -0.8479222655296326, 1.1785757541656494, 0.5643452405929565, -0.29522696137428284, 0.09858155995607376, 0.021458491683006287, -0.2660920321941376, 0.543066143989563, 0.5696267485618591, -0.22161389887332916, -0.6211438775062561, -0.8470491766929626, -0.23933488130569458], [-1.0354543924331665, -0.2499842494726181, -1.6408092975616455, -1.83061683177948, -0.21350520849227905, 1.0737768411636353, 0.9743120670318604, -0.41015633940696716, 0.6277496218681335, -1.1751407384872437, 0.9736154079437256, 1.228519320487976, 0.587346076965332, 0.12208220362663269, 1.0813066959381104, 0.04941767454147339, -0.2188442349433899, -0.22226117551326752, 0.3965526223182678, -1.3710459470748901, -1.3242820501327515, 0.18082773685455322, 0.30339452624320984, -0.7700283527374268, -0.8085914254188538, 0.7593917846679688, -1.8012306690216064, 0.1286039799451828, 1.146917462348938, 0.013188780285418034, -0.810198962688446, 0.5322813391685486, -0.22900819778442383, -0.8049245476722717, 1.339440941810608, 0.24442708492279053, -0.2048204392194748, 0.28609615564346313, -0.1775292456150055, -0.2424493134021759, -0.16018521785736084, 0.2306564897298813, 0.22175298631191254, 2.2284743785858154, 0.024738363921642303, -0.2567543387413025, 1.9778579473495483, -1.286065936088562, -0.4842907190322876, 0.6648333072662354, -0.527638852596283, 1.0608227252960205, 0.4228222668170929, 0.7707402110099792, 0.02118726633489132, 0.07659449428319931, 0.4428750276565552, 1.240232229232788, 0.787797749042511, 0.23828814923763275, -0.6945180892944336, -1.9524012804031372, 0.5224801301956177, 1.5398422479629517], [-0.1577143669128418, 0.03176630660891533, -0.9071998000144958, -0.03228187561035156, -0.7722839713096619, 0.9457516670227051, -1.4070696830749512, -0.27511340379714966, -0.21333356201648712, 0.8660402894020081, -0.8367920517921448, -0.12390179932117462, 0.9779165983200073, -0.0844644159078598, -0.7669700384140015, 0.42783379554748535, 0.6291110515594482, -0.06279671937227249, 1.4590908288955688, 0.2637295722961426, -0.3659268617630005, -0.9760780334472656, -0.446988582611084, -0.3522586226463318, 0.9952237606048584, -1.767543911933899, -0.28778669238090515, 0.6452839374542236, -0.5494129061698914, 0.09809572249650955, -0.35442280769348145, 0.4743262231349945, 0.004603440407663584, 0.9230574369430542, 0.22640569508075714, 0.4328945279121399, 0.9197311401367188, -0.6668230295181274, 1.8244787454605103, -1.242084264755249, -1.915220022201538, 1.2931628227233887, 0.26400327682495117, 1.1193149089813232, 1.1105800867080688, 1.0976569652557373, -0.9516882300376892, -0.10397256910800934, -1.181911826133728, 1.2435932159423828, -0.2540632486343384, -1.932763934135437, -0.09078437089920044, 0.27339982986450195, 0.12239380925893784, -0.3298998475074768, -0.996236264705658, 1.289107322692871, 0.48930612206459045, -1.3282760381698608, 0.6853948831558228, 0.18117298185825348, 0.6613230109214783, 0.2690422534942627], [0.9700022339820862, -0.6342743039131165, 0.9284206628799438, -1.5871014595031738, -1.4501763582229614, 1.8926658630371094, 0.8168705105781555, 0.7679343819618225, 1.0013346672058105, 2.107703447341919, -0.3044452965259552, 1.0037461519241333, -0.11378313601016998, -0.05267215892672539, -0.855857253074646, 0.6819711327552795, 1.241935133934021, 0.652011513710022, 0.2479722797870636, 1.0040770769119263, -0.7008741497993469, 0.2644624710083008, -0.1417078822851181, 1.2684040069580078, 0.7181600332260132, -0.47508904337882996, 0.6423245072364807, 0.35453900694847107, 0.09297492355108261, -0.7739052772521973, -0.27726396918296814, 0.27685800194740295, 0.3117157220840454, -1.838849663734436, -1.0152705907821655, 1.1575595140457153, 0.2359023243188858, 0.619104266166687, -0.8610339760780334, 0.01721654273569584, 1.304512619972229, 0.21535201370716095, -1.3183519840240479, 0.3999727666378021, 0.8212399482727051, -0.04935977980494499, 1.380987286567688, 0.09598205983638763, 1.0349818468093872, 0.18385139107704163, -1.5412640571594238, 0.0969097912311554, -1.3230124711990356, 0.5174423456192017, 0.06514298170804977, 0.7519978880882263, 0.37033358216285706, -0.866863489151001, 0.4201166331768036, 0.025464465841650963, 0.19493752717971802, 1.094846487045288, 0.17994093894958496, 0.16427931189537048], [-0.810597836971283, -0.28932297229766846, -0.7232614159584045, 0.12019292265176773, 0.7797241806983948, 0.52152419090271, -0.5601122379302979, 0.9645031094551086, -0.18223844468593597, -0.7083640098571777, -0.23670710623264313, -0.6515010595321655, 0.5147454142570496, 0.019487082958221436, 0.30299970507621765, 0.21756628155708313, 0.9780404567718506, -0.530754804611206, -1.0897469520568848, -0.7167195081710815, 0.3397562801837921, 0.7592676281929016, -0.5643410682678223, 1.2671476602554321, -0.01649247482419014, -0.19040773808956146, 1.0571439266204834, 0.1769295632839203, -0.2240385264158249, -0.5958148241043091, 0.006892552133649588, -0.2976606786251068, -0.44908419251441956, 0.030113236978650093, 0.5161604285240173, 0.8802324533462524, 0.013071144931018353, -0.6617294549942017, 0.3107862174510956, 1.8882473707199097, 1.031421422958374, -1.4458540678024292, 0.6672574281692505, -0.6177909970283508, 0.02563013881444931, 0.2838965952396393, 0.2759929299354553, -0.43290194869041443, -0.7629843354225159, -0.845373272895813, -0.6572499871253967, 0.65927654504776, -1.182199239730835, -0.030867362394928932, 0.25156328082084656, -1.4720091819763184, -1.0153465270996094, -1.3902840614318848, 0.6303681135177612, 1.0997556447982788, -1.4543075561523438, 0.08546146005392075, -0.7248790860176086, -1.232298493385315], [-1.156814455986023, -0.998054563999176, 0.7752406001091003, -0.8062528371810913, 1.3131628036499023, 0.5977626442909241, -0.21051745116710663, 0.9688917398452759, 1.1637808084487915, 1.1578714847564697, 1.4965553283691406, -0.05389420688152313, -0.59040367603302, -1.3108947277069092, 0.6069908738136292, -0.06878532469272614, 0.953862726688385, 0.8392143845558167, 0.8054677248001099, 2.08955454826355, 1.1699401140213013, -1.2123757600784302, -0.4913559854030609, 0.7139042019844055, -0.11833345144987106, -0.03710715472698212, -0.6874256134033203, -0.026468228548765182, 0.11423841118812561, -0.48653262853622437, 0.33085551857948303, 0.09659361839294434, -1.330105185508728, 1.1054375171661377, -1.661048173904419, 0.7151630520820618, -1.5476984977722168, 0.3534354865550995, -0.4186035990715027, -0.9981258511543274, 1.9577627182006836, -2.0417566299438477, 2.325108528137207, -0.20098939538002014, 1.778794288635254, 1.9576842784881592, -0.20230978727340698, -0.8808099031448364, -0.4143657684326172, 0.7084246277809143, 1.8331555128097534, 1.4059727191925049, 1.563894271850586, -0.13074389100074768, 0.11781869828701019, -1.0341893434524536, -0.09090258181095123, 1.0683594942092896, 1.5826390981674194, -0.35882729291915894, -3.1058826446533203, 0.009764219634234905, 0.04563016816973686, -0.7134729027748108], [-1.1562256813049316, 0.5846444368362427, 0.2927483320236206, -1.073530912399292, 0.9234433770179749, 0.8834920525550842, 0.3715744912624359, -0.9538534879684448, 1.127779483795166, 0.5874561667442322, -0.45907026529312134, 0.24738425016403198, -1.0681134462356567, 1.6551026105880737, -1.2584404945373535, -0.08932362496852875, 0.6627534031867981, 0.900198757648468, 0.5833079814910889, 0.026512276381254196, -0.07638955116271973, 0.8084039092063904, 0.5035133957862854, 0.08420263975858688, 1.9677470922470093, 0.012821518816053867, -0.8478789925575256, -1.2959306240081787, -1.5536106824874878, 0.8446151614189148, -1.0496129989624023, 1.4527432918548584, 0.29843205213546753, -1.0567892789840698, -0.6815244555473328, 0.9828180074691772, -0.6506343483924866, 0.8424051403999329, -0.1928187608718872, 0.5711467266082764, -1.1787073612213135, -0.8361167311668396, 0.9238064289093018, -0.4947162866592407, 0.6407309174537659, 0.990196943283081, -0.251485139131546, -1.5447887182235718, 0.3722963333129883, -1.8104536533355713, 0.3859040141105652, 0.18275263905525208, -0.3948136270046234, 0.4900268316268921, -0.2280346304178238, 0.39910688996315, -0.4440512955188751, 1.6045106649398804, -0.2495046854019165, -0.5612384676933289, 1.1244392395019531, -0.18794533610343933, -1.6121684312820435, -2.0100808143615723], [0.7130427956581116, -0.48086485266685486, 0.6185590624809265, -0.8659870624542236, -1.5088533163070679, -0.9994429349899292, 0.9408797025680542, -0.3146432340145111, -2.3979811668395996, -0.1763341873884201, 0.8457604050636292, -1.6112861633300781, 0.7511681914329529, -1.807702660560608, 1.7124251127243042, -1.1065356731414795, 0.2967627942562103, -1.3402812480926514, -0.17943184077739716, 0.6012787222862244, 0.5942726135253906, 2.081956624984741, -0.5540602803230286, -0.7782888412475586, -1.2752901315689087, -0.1436101198196411, -1.0177695751190186, -1.9649559259414673, -0.6829189658164978, -1.0504107475280762, 0.6597286462783813, -1.2066141366958618, 0.30763906240463257, 2.7858047485351562, 1.2643934488296509, -0.507064938545227, -1.5281106233596802, 0.6121849417686462, -0.396533340215683, -0.5006678700447083, -1.7401747703552246, 0.48713093996047974, 0.536201536655426, 0.6381709575653076, -0.3867376744747162, 1.3041725158691406, -0.03626531362533569, -0.587567150592804, -0.7434924840927124, -1.050506830215454, 1.7596088647842407, 0.4687788188457489, 1.473166823387146, -0.6160359382629395, -0.37263792753219604, -0.9396113157272339, -1.248050332069397, -1.7868393659591675, -0.6767472624778748, 1.7162481546401978, 0.36424311995506287, -0.44099873304367065, 0.10156092047691345, -2.1084892749786377], [1.3674722909927368, -1.4459013938903809, -0.35969293117523193, -2.0906524658203125, 0.44178277254104614, -0.6824944615364075, 0.1404382288455963, -0.47853684425354004, 1.4247852563858032, 0.5030811429023743, -0.5257282853126526, -0.766184389591217, 0.144788458943367, -0.902101993560791, 1.973298192024231, -0.7855646014213562, -0.3494400084018707, -0.6204627156257629, 0.20210622251033783, -0.20405659079551697, -0.5602586269378662, 1.5486962795257568, 1.5878080129623413, -0.3698563873767853, 0.39992237091064453, 0.5029011964797974, -0.4546653628349304, -0.36373308300971985, -0.9413654804229736, -0.464753121137619, 0.5047202110290527, 1.119615912437439, 1.750722885131836, 0.28979673981666565, -1.610149621963501, 0.1576610803604126, -0.2647417187690735, 0.02098066359758377, 0.3124426305294037, 0.08771485835313797, 0.3446238338947296, 0.6790370345115662, 0.6262951493263245, -0.39836543798446655, -0.5311524271965027, 0.33025386929512024, -0.6436128616333008, -1.2126778364181519, 1.9397170543670654, 1.7058885097503662, -0.17047646641731262, -0.5387756824493408, 0.6381821036338806, -0.4619523286819458, -0.120987169444561, 0.25776180624961853, -0.4074927866458893, 0.08818890154361725, -1.5809400081634521, 0.4546254873275757, 0.397461473941803, 1.2411483526229858, 0.08910010755062103, 0.6108302474021912], [0.11155811697244644, 0.25096744298934937, -0.14697128534317017, 0.6640573740005493, 0.30182018876075745, -0.24227216839790344, -0.25222504138946533, -0.21048428118228912, 0.7300900816917419, -1.7991729974746704, -0.4265497326850891, -1.3960717916488647, -0.0309333223849535, -0.03190779685974121, 0.8305768370628357, -0.04984340816736221, -0.5904037952423096, -2.044802665710449, -0.043270476162433624, -0.2224479764699936, 0.7550517320632935, 1.1959295272827148, -1.1976385116577148, 0.2561972439289093, 1.4456119537353516, -0.892168402671814, 0.9106014370918274, 1.0893296003341675, -1.1899540424346924, 0.6217271685600281, -0.3726658225059509, 2.1631381511688232, 0.3885526657104492, 0.012098702602088451, -1.2573940753936768, -0.34232789278030396, -1.5575922727584839, 1.8030524253845215, -0.2732740640640259, 0.5077587962150574, -0.985971987247467, 2.2131636142730713, 0.01013198308646679, -1.4845895767211914, 0.7770795226097107, -2.228073835372925, -0.12021636217832565, -0.5858745574951172, -1.38167405128479, 0.0964808464050293, 1.4184513092041016, 0.5849534869194031, -0.6747003197669983, -1.3935540914535522, -0.23350317776203156, 1.4251172542572021, -0.6542011499404907, 0.13495714962482452, 0.9081244468688965, 1.3423898220062256, -0.6274181008338928, 2.1953070163726807, -0.8352974057197571, 0.5630743503570557], [1.2519505023956299, 0.6506537795066833, 0.6042162775993347, 1.025229811668396, -0.8024682402610779, -1.5579537153244019, 0.931900680065155, 0.34198591113090515, -1.5140032768249512, 0.8702594637870789, 0.7878649830818176, 0.6814136505126953, 1.3008002042770386, -1.0329967737197876, 0.2766261398792267, 0.9213026762008667, -0.85658860206604, -1.2146433591842651, -1.12770414352417, 0.041858136653900146, 1.2127881050109863, 0.25946006178855896, -0.7319473028182983, 0.31089287996292114, 0.6760329008102417, -1.8387781381607056, 1.2528389692306519, 0.2979404032230377, -0.4147162437438965, -1.1608421802520752, -0.6203206777572632, -1.0818098783493042, 0.3147079050540924, -1.5885910987854004, 1.1579627990722656, 1.773574709892273, 1.0840251445770264, -0.5807884335517883, -1.943899393081665, 1.3892419338226318, -2.1044554710388184, -0.807280957698822, 2.166046619415283, 2.301882743835449, -1.3784234523773193, -0.44426488876342773, 2.0521199703216553, 0.6227158904075623, 0.2984006106853485, 2.8667502403259277, -1.7630889415740967, -1.2501827478408813, -1.4813886880874634, 0.3565344214439392, 0.8416608572006226, 0.3369571566581726, -1.2429883480072021, -0.11893133074045181, 2.713374137878418, -0.5976125597953796, -0.06033208593726158, -0.46174678206443787, -0.8055493831634521, 1.3706772327423096], [0.7897357940673828, -0.8524225950241089, -0.21138469874858856, 0.8134710192680359, -1.9891670942306519, -0.7903385162353516, -0.005413539707660675, 0.5820096731185913, 0.32746726274490356, -1.4980484247207642, -0.6222487092018127, -0.6205058693885803, -0.2967110872268677, -0.8749358057975769, 0.7710193991661072, 0.4909253716468811, -0.9230025410652161, 0.4535239636898041, -0.3066296875476837, 1.080910325050354, -1.6117868423461914, -0.8443894386291504, 0.21447136998176575, -0.5523637533187866, 0.698519229888916, -0.07143916189670563, -0.5639549493789673, 0.8981062173843384, -0.7040199637413025, 0.18892207741737366, 0.040299851447343826, -0.6353099942207336, 0.7681683897972107, -0.686098039150238, 1.298952579498291, -0.28006523847579956, 0.736097514629364, -0.007162835448980331, 0.7059094905853271, -0.5407759547233582, -0.05784897133708, 0.6405425667762756, 0.9858159422874451, -0.8292602896690369, 0.4523176848888397, 1.1226197481155396, 0.2573952078819275, -0.37123700976371765, -0.9783905744552612, 1.353901743888855, 0.7098444700241089, 0.553730845451355, -0.5668650269508362, 2.9388620853424072, -0.861598014831543, -1.9662864208221436, 0.023236602544784546, 3.031529426574707, -0.0693511813879013, -0.659217357635498, -0.8183273077011108, -0.16033713519573212, -2.816594362258911, -1.0299347639083862], [1.8906654119491577, 0.04343893378973007, 0.03344836086034775, 1.6649291515350342, 1.9142382144927979, 0.40591293573379517, -0.7546760439872742, 0.45833757519721985, -1.0860626697540283, 0.27941426634788513, 0.0120922876521945, -0.9933831691741943, 1.0841038227081299, -1.775622844696045, -0.449921578168869, 0.6638209819793701, -1.0097393989562988, -2.217731475830078, -0.20741784572601318, 0.05562601238489151, -1.8346519470214844, 0.14110921323299408, -0.08379928767681122, -0.6526626944541931, -0.6433238983154297, -0.8841322660446167, 0.16862939298152924, -0.44782546162605286, 1.228267788887024, 0.03416477143764496, 0.224683940410614, 1.5733047723770142, -0.6733923554420471, 2.048954486846924, 1.0706416368484497, 1.8834041357040405, 0.534023642539978, 0.4173855781555176, 0.4492197036743164, -0.4603825807571411, -1.2847710847854614, -0.821363091468811, -0.002636565128341317, 0.5746076703071594, -1.3386250734329224, 1.522741675376892, 0.30588170886039734, 0.7071648240089417, -0.24360986053943634, -0.30921798944473267, 2.1252501010894775, 1.2488062381744385, -0.6791510581970215, -0.8850370049476624, -0.10842347890138626, 1.4249403476715088, -0.7352674007415771, -0.5004236102104187, 2.14717698097229, -0.20082123577594757, -1.1480799913406372, 0.302799254655838, 0.7473616003990173, 2.2098748683929443], [-0.7898775935173035, 1.4083212614059448, 0.9448147416114807, 0.739574670791626, 0.6686469316482544, -1.8402774333953857, -0.4389905333518982, 0.3686705529689789, 1.3018462657928467, 1.319408655166626, -0.7726836800575256, 0.9032322764396667, -1.2322044372558594, -1.0450713634490967, 0.7545745968818665, -0.831036388874054, -1.5050437450408936, 0.7878034114837646, 0.9876546263694763, 0.504808783531189, 0.518804132938385, 0.24085362255573273, 0.48643386363983154, 0.7046155333518982, 0.5068907737731934, 0.918666422367096, -0.6738479733467102, 0.2062888741493225, -2.343909978866577, -0.378755122423172, 1.3865972757339478, -2.1667959690093994, -0.7462921142578125, 0.7968075275421143, -0.3406185805797577, -0.8553224802017212, 0.08114635944366455, -0.5290870070457458, -1.0466598272323608, -0.8913193941116333, 0.5152505040168762, 0.3661174774169922, 0.2866145372390747, -0.6746951937675476, 1.3235116004943848, -0.9880008101463318, 0.27647221088409424, -0.2184886336326599, -1.5331575870513916, 0.376455694437027, 0.3650074005126953, -0.5448473691940308, 0.7924646139144897, 0.6702035665512085, -0.23062178492546082, -1.7207647562026978, -0.0798947662115097, -0.9868742823600769, 0.7164625525474548, 2.1720309257507324, 1.5652724504470825, 1.0722521543502808, -0.10477451980113983, 1.2334833145141602], [1.0842792987823486, 1.1741799116134644, 0.4300832152366638, -0.7848132252693176, -0.8213453888893127, -0.6937457323074341, 1.3305997848510742, 0.7530589699745178, 0.07171474397182465, -0.049594514071941376, 0.8094422221183777, -2.2960991859436035, 0.1596805453300476, 0.4985559284687042, -0.36690086126327515, -0.44913172721862793, -1.3141063451766968, 0.43093445897102356, -0.17178387939929962, 0.40752995014190674, 0.3149336874485016, 0.16338218748569489, 0.5207220911979675, 0.3335595428943634, -2.1411983966827393, 0.5242326259613037, -0.8861052393913269, 0.5709084272384644, -2.4022440910339355, -0.4096478521823883, 2.0339653491973877, -0.2888743579387665, -0.5833646655082703, 1.2544641494750977, 0.6256068348884583, 0.07554753124713898, -0.5215460062026978, 0.3700200021266937, 0.659587025642395, -1.1705142259597778, 1.2059147357940674, 0.2398584932088852, 0.6146986484527588, -0.9959743022918701, -1.3575385808944702, -0.035259027034044266, 0.9740976691246033, -0.08578372746706009, 0.45685529708862305, 1.8949977159500122, -0.374470591545105, 0.1280096471309662, 0.6799623370170593, 0.06611447781324387, -0.2554279863834381, -1.2577459812164307, -1.585311770439148, -0.3304458260536194, 1.4106107950210571, -0.990656852722168, -0.45987021923065186, 1.8484141826629639, 0.20252849161624908, -0.8152874112129211], [0.3581524193286896, -0.48447850346565247, -0.7287949919700623, 0.034155167639255524, -1.970075249671936, -0.3987930417060852, -2.91471266746521, -0.28684717416763306, -1.2615054845809937, 0.4482787251472473, -0.35535433888435364, 0.8273187279701233, -1.096884846687317, 0.544392466545105, -0.5124727487564087, 2.147007703781128, -0.3543727397918701, 0.43943163752555847, 0.17579109966754913, 1.661444067955017, 0.1774396300315857, 0.44255533814430237, -0.6151127219200134, -0.5770606994628906, 0.7439755797386169, -0.21486972272396088, -0.8642297387123108, 0.46594172716140747, 0.14219726622104645, 1.0215563774108887, -3.202633857727051, -0.4033809006214142, -0.47904643416404724, -0.06719028949737549, 1.6447947025299072, 0.86597740650177, -0.21333502233028412, 0.6702309846878052, -2.987018346786499, -0.814685046672821, -0.4355347752571106, 0.26466041803359985, -0.18442431092262268, -0.04540453478693962, 0.8471983671188354, 0.34784024953842163, 0.08292906731367111, -0.4876363277435303, 1.234953761100769, -0.11591558903455734, -1.0604428052902222, -0.5616526007652283, 0.059217777103185654, -0.2564494013786316, -0.18453428149223328, -0.46789485216140747, 0.6349290013313293, -0.46972188353538513, 0.1304795742034912, 0.1876271814107895, -0.9105769395828247, 1.9159890413284302, 3.3518941402435303, 0.8049249053001404], [0.08255228400230408, -0.3279873728752136, -1.3591995239257812, 1.8023113012313843, -0.8261181712150574, 1.252134084701538, -1.0996246337890625, -0.6462066173553467, 0.24488604068756104, -1.0712331533432007, 0.21875326335430145, -1.2890934944152832, -0.06590961664915085, 0.8820580840110779, -0.9285538792610168, 0.8925803303718567, 0.7700725793838501, 1.02340567111969, 2.003674030303955, 0.711093544960022, -0.008552859537303448, -0.23452123999595642, -0.30121874809265137, 0.5348849296569824, 0.23127086460590363, 0.07156997919082642, -0.27917349338531494, -0.23703379929065704, 0.45993930101394653, -0.7182422280311584, 1.020400881767273, 0.275607705116272, 1.910322666168213, -1.106696367263794, -0.2913576066493988, 1.3905091285705566, -0.7765195965766907, 1.3605833053588867, 1.4661885499954224, -1.0197744369506836, 0.5932469964027405, -0.07205510139465332, -0.6199255585670471, 1.8745187520980835, 0.013792484067380428, 0.5882729887962341, -2.056042432785034, 0.34135496616363525, -0.37689757347106934, -0.7597454786300659, -0.8467106223106384, 0.6094604730606079, -0.7791066765785217, 1.3477327823638916, 0.7671114802360535, -1.053251028060913, -1.0413424968719482, -0.4502337574958801, 0.09295801818370819, 0.35070690512657166, -0.23149710893630981, 1.3408071994781494, -0.5657564401626587, 0.9030424356460571], [-0.27546998858451843, 1.6757184267044067, -0.9918628931045532, -1.0462374687194824, -2.3610708713531494, -0.7030544877052307, 2.66674542427063, 0.2745921015739441, 0.47172656655311584, 1.1679542064666748, 0.7072480916976929, 0.6555783152580261, -0.9241074919700623, -0.10700840502977371, 0.6231171488761902, 0.022303326055407524, -0.5957457423210144, 0.7921419739723206, 0.011857567355036736, -0.667670726776123, 1.7345730066299438, -0.4032299816608429, -1.4266725778579712, -2.0404787063598633, 1.1887600421905518, -0.7400059103965759, 0.11830984055995941, 0.4322997033596039, -1.4051861763000488, 1.139145016670227, 1.000016450881958, -1.0040879249572754, 0.5677390694618225, -0.2378711849451065, -0.9351164102554321, -0.8954054713249207, 0.9912950396537781, -0.6672540307044983, 0.5238309502601624, -1.1373939514160156, -2.1411685943603516, -0.5082383155822754, -0.705184280872345, 0.5693297386169434, 0.29638931155204773, 0.0262051522731781, 0.4871445298194885, 0.4507594704627991, 0.16693682968616486, 0.6178614497184753, -0.2244478166103363, 0.0906497910618782, -1.1894264221191406, 0.4337853193283081, -0.5747566819190979, -1.4971199035644531, 0.21883633732795715, 0.5344771146774292, 0.19800816476345062, -0.45974797010421753, 1.3721853494644165, -1.4607267379760742, -1.44599187374115, -1.1272759437561035], [0.21997471153736115, 1.5233800411224365, 0.4621466100215912, 0.05856594815850258, -0.6550129055976868, 0.6906240582466125, 0.5684999227523804, -1.7847895622253418, 0.4610942006111145, -0.10567247867584229, -1.0882441997528076, 0.6875296831130981, 0.5048717260360718, 0.03163804113864899, -1.2668403387069702, -1.7781373262405396, 1.574355959892273, 0.3789377212524414, -0.6853289008140564, -0.7325708270072937, 1.8266947269439697, -0.41355153918266296, -0.18339936435222626, -0.2678527534008026, -0.791793942451477, 1.2648143768310547, 0.8309531211853027, 0.28411680459976196, -0.8657774329185486, -1.362080693244934, 0.4395797550678253, 1.991121530532837, -0.2927916347980499, -0.9571981430053711, -0.4612278640270233, -1.40610933303833, 1.351844072341919, -1.7900044918060303, 0.6853929162025452, 0.41909170150756836, 0.31169241666793823, 1.4751709699630737, 0.19800955057144165, 0.06974846124649048, -0.5104092359542847, 1.9354242086410522, -0.11290896683931351, 0.3735348582267761, 0.7678398489952087, -0.12208624184131622, 1.0716261863708496, -0.4999878704547882, 2.3005731105804443, -0.7197540402412415, -1.2147537469863892, -0.37872952222824097, 0.6638018488883972, 0.6279715895652771, 1.4758387804031372, 0.28661084175109863, -0.19554172456264496, 2.105828285217285, 0.4089486300945282, 0.4866982102394104], [-0.3002510964870453, -0.31215402483940125, -1.0705251693725586, -1.3936632871627808, 0.1007176861166954, -0.34190627932548523, -1.2935079336166382, 1.4386687278747559, -1.1066510677337646, 2.7876217365264893, -1.0503227710723877, -0.2674752473831177, 0.7357232570648193, -0.2229100912809372, -0.21231670677661896, 0.1769527792930603, -1.2167527675628662, -1.3189579248428345, -0.3722987473011017, 0.09068609029054642, 0.323002427816391, 0.20461653172969818, 0.8567017912864685, 0.14372530579566956, 0.34717777371406555, -0.3471033275127411, 0.002291366457939148, -0.3025895357131958, 0.2912989556789398, 0.033696457743644714, -0.5674015879631042, -0.05823179706931114, 0.12519167363643646, -0.2861722409725189, 1.2204654216766357, -0.1115274727344513, -0.8763622045516968, -0.7364735007286072, 1.1579638719558716, -0.8101323246955872, -0.1751243621110916, -0.857399046421051, -1.3380146026611328, -1.1130995750427246, -0.3022352159023285, -0.34624722599983215, -0.8299233317375183, -0.392682820558548, -2.3465726375579834, 0.26941636204719543, 0.5014147758483887, -1.256265640258789, -0.3819405436515808, -0.35716089606285095, 1.8305742740631104, -0.13165195286273956, -0.35087376832962036, -0.9732813835144043, 0.5340390801429749, -1.4041911363601685, 1.2759150266647339, -0.9654729962348938, -0.7508963346481323, 0.3434414267539978], [1.4804577827453613, -0.12812325358390808, 0.26622098684310913, -0.9407832026481628, 0.6964617967605591, -0.1764349341392517, 1.5159833431243896, 0.7693036198616028, 0.10611019283533096, -0.5774424076080322, 0.024818608537316322, 0.6361810564994812, -0.9054800868034363, 0.25573843717575073, 1.0167396068572998, -1.3127912282943726, 0.8119356632232666, 0.03468507528305054, -0.7582525610923767, 0.14976850152015686, -2.6792614459991455, -0.794577419757843, -1.4714640378952026, 0.16576135158538818, 1.3308813571929932, 0.5865533947944641, 0.41704055666923523, -0.3080677092075348, 0.6990355849266052, 1.1184378862380981, -0.43553078174591064, -0.10132312029600143, -0.3650766909122467, 0.7297959327697754, 1.375154972076416, -1.1878955364227295, -0.8701900839805603, 0.06991085410118103, -0.6288850903511047, -0.5145487785339355, 0.6983023881912231, 1.0582985877990723, -0.23965194821357727, 0.4182981252670288, 0.5637228488922119, -0.4880954623222351, -0.8502990007400513, -0.8165566325187683, 0.5537161827087402, 0.6857990622520447, -0.18393108248710632, 0.07660608738660812, 0.7691475749015808, -0.2744334042072296, 0.286515474319458, 0.3276098370552063, -0.3532680571079254, 1.366056203842163, 0.2232472449541092, 0.7758660316467285, -1.3057669401168823, 0.6176061630249023, 2.323787212371826, 0.8370241522789001], [-1.6676721572875977, 1.2794110774993896, -1.377217411994934, -0.9119807481765747, -0.08250000327825546, -0.29149457812309265, 1.273723840713501, 2.0964624881744385, -0.5285541415214539, -1.060508370399475, 1.1206622123718262, 0.6686956286430359, 0.1784747838973999, -1.0354832410812378, 0.20887380838394165, 1.561950445175171, -0.8094844222068787, -1.3235447406768799, -0.17538413405418396, -1.4586130380630493, -0.8428499698638916, -1.1024919748306274, -0.2750561833381653, 1.4170256853103638, 0.29834166169166565, 1.0359258651733398, 0.26163479685783386, -0.6231769323348999, -0.6597053408622742, 1.8935176134109497, 0.004471947439014912, -2.279578447341919, 0.009321548976004124, -0.5816386342048645, -2.009406328201294, 1.2423633337020874, -0.06897231936454773, -0.4777091145515442, -0.8224136233329773, 0.07736261188983917, -0.010453863069415092, -0.0060107712633907795, 0.750282347202301, 1.16428542137146, 0.3109838664531708, -0.10632997751235962, 0.6014696955680847, -0.791573703289032, -2.9650514125823975, -0.8236482739448547, -0.050833314657211304, 0.30354270339012146, 0.2821538746356964, 0.04265213757753372, 0.5746898651123047, 0.768845796585083, 0.5478605031967163, -0.33280473947525024, 1.9487539529800415, 0.2006695419549942, -1.3434460163116455, 0.3003665804862976, 0.6545971632003784, 0.274975448846817], [1.6494070291519165, 0.07440648227930069, 0.9814927577972412, -1.5479997396469116, 1.472383737564087, 1.9322623014450073, -1.8822849988937378, -0.703228235244751, -1.7987929582595825, 0.9835909605026245, -0.9749536514282227, 0.3870633840560913, 0.6253506541252136, -0.09020791947841644, -0.4096508026123047, 0.6140135526657104, -1.3664544820785522, -0.07236375659704208, -0.16609083116054535, -0.14389188587665558, 1.3512691259384155, -0.16065126657485962, 0.3044346868991852, 1.3163424730300903, 0.922045111656189, 0.7514543533325195, -1.4087988138198853, -0.6804100275039673, 0.13177524507045746, -1.734136700630188, 1.2460353374481201, -1.4669623374938965, -0.28266212344169617, 0.7264702916145325, -0.37736931443214417, -0.6370421648025513, 0.4839462339878082, -1.3658796548843384, 0.047299113124608994, 0.4829648733139038, -1.2876126766204834, -0.5843847393989563, -1.3577990531921387, -0.7371117472648621, 0.7755967974662781, 1.4314770698547363, 1.0442558526992798, 0.4499724507331848, -0.35987764596939087, 0.11883505433797836, -0.026799343526363373, 0.16192515194416046, -1.2077556848526, 0.41680774092674255, -0.7986276745796204, -0.4874255955219269, 0.5108028650283813, 1.1700586080551147, 0.796707808971405, 1.7627546787261963, 1.385214924812317, -0.41257503628730774, 1.5621141195297241, -0.573419988155365], [-1.2499303817749023, 0.17733874917030334, -0.31534695625305176, 0.6695414781570435, -0.03966911882162094, -0.6215096116065979, -0.5085048675537109, -0.2628789246082306, -1.6075811386108398, 0.02631441131234169, 0.048338234424591064, 0.8280558586120605, -0.22266755998134613, 0.5807528495788574, 1.9669674634933472, 0.2276790887117386, -0.32875314354896545, -0.45915666222572327, -2.2887840270996094, 0.4686340391635895, -0.8855507373809814, -0.5100123286247253, 0.14759862422943115, 1.2937133312225342, -0.9794505834579468, -1.546223521232605, 0.3080621659755707, 1.3649522066116333, -0.5198624134063721, 1.1285134553909302, -0.698198139667511, 1.1178231239318848, -1.7522681951522827, 2.423471212387085, 1.4893654584884644, 0.612666130065918, -0.1797514259815216, -0.01608888804912567, 0.8422127366065979, 0.1628750115633011, -0.2955399453639984, -1.0249460935592651, -1.6018610000610352, -1.2554153203964233, -1.3284664154052734, 0.23624511063098907, -0.4844973087310791, 0.7270676493644714, -0.1730959564447403, 1.0368008613586426, 0.3618207275867462, 0.13861827552318573, -0.32732775807380676, 0.6111679077148438, 1.5155621767044067, 0.3121718466281891, 1.0152889490127563, 0.15007491409778595, -1.5264813899993896, -1.172403335571289, 1.6790180206298828, -0.18860071897506714, 0.9320958256721497, -0.9659008979797363], [-0.542643666267395, -0.6786354184150696, -0.2728540897369385, 0.200741708278656, 0.26198694109916687, -1.3850271701812744, -1.3193210363388062, 1.8224692344665527, -1.4240838289260864, 0.08520174771547318, 0.3726454973220825, 0.1416368931531906, 0.6727687120437622, -0.4623330235481262, 0.6687974333763123, -0.28425517678260803, -1.5225168466567993, -0.9478655457496643, 0.056639160960912704, 2.413412094116211, 0.6902576684951782, -0.5724012851715088, 1.2825363874435425, -1.170798897743225, -0.8936175107955933, 0.7242244482040405, -1.757115364074707, -3.1895182132720947, 1.5115668773651123, -0.03639883175492287, -0.15025171637535095, -0.18147344887256622, 1.9897918701171875, 0.4212706983089447, 0.023366957902908325, 0.3529609441757202, -0.906822681427002, 0.6011107563972473, -3.4025380611419678, 0.5461993217468262, 1.640116572380066, 0.8338161706924438, -2.1762449741363525, 0.35321664810180664, 0.2683051824569702, 0.6952967643737793, -1.0910017490386963, 1.8581442832946777, 0.7754867672920227, -1.073133945465088, -1.507806420326233, 0.6089255809783936, 0.888035774230957, -0.4300117790699005, -0.06376773118972778, 0.7915838360786438, -0.3195665180683136, 1.2489842176437378, 1.1122828722000122, 0.16934049129486084, 0.012870516628026962, 1.4424033164978027, 0.42385566234588623, -0.20460504293441772], [0.19597013294696808, -0.31605058908462524, -0.7620691657066345, -0.46364644169807434, 0.2978861331939697, 1.5193347930908203, 0.020318353548645973, -0.11140608787536621, 1.0941485166549683, 0.2892172634601593, 0.2650691270828247, 0.908879816532135, 0.20565973222255707, -1.1630574464797974, 1.3073244094848633, -0.8679158091545105, 0.6476472616195679, -0.12848441302776337, -1.9053914546966553, 0.8556350469589233, -0.020883677527308464, -0.26675933599472046, 0.009736968204379082, 0.9297618269920349, -0.9153320789337158, 2.06699800491333, 0.13481144607067108, -0.04203607514500618, 0.02512546256184578, -1.1275479793548584, 0.0991659015417099, -1.2093631029129028, 1.1079087257385254, 1.4795385599136353, 0.13578587770462036, -0.576762318611145, 0.5648165345191956, 1.3286341428756714, 1.5744856595993042, 0.1469516009092331, -0.041721493005752563, -2.292156934738159, -1.341020941734314, -0.6532006859779358, -0.2479476034641266, 0.4080048203468323, 1.0500869750976562, -1.3460133075714111, -0.6047925353050232, -0.48392727971076965, 1.3078101873397827, 0.912821888923645, -0.6806646585464478, -0.6745569705963135, -0.4117558002471924, 0.31144919991493225, 1.1573256254196167, -1.3908804655075073, 0.6050080060958862, 0.9885020852088928, -1.2138283252716064, -1.3065829277038574, -0.2515089511871338, -0.688801646232605], [0.37111368775367737, -0.5091226696968079, -0.9430646896362305, -0.4728451073169708, -0.3834805190563202, -0.6004031300544739, -0.614177942276001, -2.194512367248535, -1.6674284934997559, 0.0013962811790406704, 1.0575345754623413, 0.7363579869270325, 1.4057978391647339, 0.2917519807815552, 0.1364031881093979, 0.8997516632080078, -0.49916353821754456, -1.195223093032837, 0.14251400530338287, -0.09205610305070877, 1.6085892915725708, -0.5294390320777893, 1.6020013093948364, 0.4898552894592285, -0.7992507815361023, -0.6491929292678833, -0.0250374898314476, -1.4369019269943237, 0.7718725800514221, 1.1709578037261963, 2.47981595993042, 0.05951296538114548, -0.7926342487335205, -2.313455104827881, -0.15093053877353668, -1.262117624282837, 1.1612136363983154, -0.07851915806531906, 0.8908761739730835, -0.8105869889259338, 0.1685052514076233, -2.620643138885498, -0.24867720901966095, -0.759867250919342, -1.6424518823623657, 1.210113286972046, 0.9198210835456848, 0.29184141755104065, -1.1148265600204468, 0.957658052444458, 0.6763683557510376, -1.973252773284912, 0.2275509387254715, -0.6972339749336243, -0.8470055460929871, 0.5004566311836243, -1.3996455669403076, 0.2912934720516205, 0.3685026466846466, 0.7072097063064575, -0.30499279499053955, 0.935185432434082, -0.3055114150047302, 0.07174402475357056], [-1.4429963827133179, 0.8249539732933044, -0.8305770754814148, 1.1924210786819458, -0.5008295178413391, -0.542363166809082, -0.13008874654769897, 0.2665325105190277, -1.9285259246826172, -0.6498976349830627, 0.15416650474071503, 0.023284953087568283, -0.5709749460220337, -0.984592080116272, -1.0595848560333252, 0.5552182197570801, 0.3382115662097931, -0.002774803899228573, 0.3517444431781769, 1.9403228759765625, -1.504878044128418, -0.6318541169166565, 0.2999894917011261, -0.27728521823883057, -0.6095118522644043, -0.6519935131072998, 1.8258397579193115, -0.9064083695411682, -0.29224637150764465, -0.552226722240448, -0.680479109287262, -0.8427226543426514, -0.8684852719306946, 0.9022378921508789, 0.4674132168292999, -0.3829305171966553, -1.1357152462005615, -0.30971184372901917, -0.6390941739082336, -0.995001494884491, 0.9166516661643982, -0.47105076909065247, 1.3509032726287842, -1.767787218093872, 0.029222864657640457, -2.2682037353515625, -1.1544325351715088, 0.7464046478271484, -0.775010883808136, -3.110139846801758, 0.2320818305015564, 0.10298560559749603, -1.3866113424301147, -0.7671753168106079, -0.6421209573745728, 0.6972465515136719, 0.7390784025192261, -0.8880450129508972, -1.0625855922698975, -0.0676218792796135, -0.052332401275634766, -1.1217823028564453, 0.7335661053657532, 1.1117421388626099], [-1.344582200050354, -1.0426201820373535, 0.8486204743385315, 1.1321405172348022, 0.3748423159122467, 0.28330370783805847, -0.9591496586799622, -2.203054189682007, -0.565949022769928, 0.5226148962974548, -0.7608564496040344, 0.8136206269264221, -0.2465047389268875, 1.8378201723098755, 0.15924087166786194, 1.3119457960128784, -0.6732763648033142, -0.08111457526683807, 0.846933901309967, -1.28067946434021, -2.5788755416870117, -2.4206085205078125, -0.6221650838851929, 1.7948373556137085, 0.076429583132267, -1.3112690448760986, -0.6346091628074646, -0.9910813570022583, 0.4957197904586792, 0.6315639615058899, 1.1681255102157593, -1.3080735206604004, 0.10349363088607788, -0.09550625830888748, 0.1813064068555832, 0.23120880126953125, -0.6644036769866943, -2.4322619438171387, -1.4229750633239746, -0.8291974663734436, 0.57760089635849, -0.36114099621772766, -0.5474813580513, 0.831351637840271, -0.8646992444992065, 0.5220491290092468, -0.48853471875190735, 0.12278765439987183, -1.0490009784698486, 2.2919681072235107, -0.6442570686340332, 0.6478750705718994, -1.0002442598342896, -1.7453786134719849, -1.2594776153564453, -0.7245028018951416, -1.366668701171875, 0.24579870700836182, 0.5362985730171204, -1.8465749025344849, 0.7768747806549072, -0.32450196146965027, -1.5210762023925781, 2.4269521236419678], [-1.678469181060791, -2.4376869201660156, 0.8686484098434448, -1.6817740201950073, -1.5032232999801636, -1.75698983669281, -0.017806001007556915, -0.607864499092102, 0.33192944526672363, 0.4949135482311249, 0.3124132454395294, 0.9225172996520996, -0.21342575550079346, -1.2313838005065918, 0.4324195086956024, -0.10846598446369171, -0.0802115872502327, -2.048614263534546, -1.2879506349563599, -1.1511179208755493, 0.07117129117250443, -0.256267786026001, -0.5757402777671814, -0.9625957608222961, 0.705502986907959, 0.531093180179596, 0.4403436481952667, -1.6105611324310303, 1.5235904455184937, -0.004274515900760889, 1.956302285194397, 0.660556435585022, 0.19071677327156067, 0.9050233960151672, 1.2968600988388062, 0.8936457633972168, 1.3380322456359863, -0.21720346808433533, -1.3043326139450073, -1.89686119556427, 1.4625966548919678, 0.036710210144519806, 2.2606492042541504, 1.4362183809280396, 0.06853407621383667, -1.0596946477890015, 1.1426551342010498, -0.9321224689483643, -1.2952070236206055, 0.3474411070346832, 0.09075670689344406, 0.5715646147727966, -1.0087097883224487, -0.12455211579799652, 1.2529168128967285, -0.11075075715780258, 1.741662859916687, 1.3263583183288574, 0.3078233003616333, 0.2991127073764801, -0.31261497735977173, 0.9075859785079956, -1.0094183683395386, -1.0191832780838013], [0.6732720732688904, -1.3185398578643799, 0.030156394466757774, -0.19482995569705963, 0.661543607711792, 0.25511425733566284, 1.6944141387939453, -0.4903692901134491, -0.30650827288627625, 1.745406985282898, 0.6760940551757812, -2.050123453140259, 1.0497480630874634, -0.06651335954666138, -0.530422031879425, -2.1823649406433105, 1.1184276342391968, -0.7812203764915466, -1.1877309083938599, -0.5080978274345398, -1.052250623703003, 0.5077224969863892, 0.3738723695278168, -0.6674859523773193, 0.6059492230415344, -0.5548107028007507, 0.41451671719551086, 0.022691577672958374, -1.2730756998062134, -0.23221226036548615, 0.4791702330112457, -1.7107442617416382, -1.0578007698059082, -0.7215058207511902, 0.7890362739562988, -0.07822216302156448, -0.8684036135673523, -0.5702608823776245, -0.8739522099494934, 1.381319284439087, 0.8621722459793091, 2.245490789413452, 0.26881223917007446, 0.9026712775230408, -0.4291498064994812, 0.11142770946025848, -0.08797796815633774, -0.2874239385128021, 0.048695869743824005, 1.113366723060608, 0.3406815528869629, 0.10930569469928741, 1.284774899482727, -0.005657211411744356, 0.30380621552467346, 1.83391535282135, -1.7315831184387207, 1.4455267190933228, -0.523321807384491, 0.8465303182601929, -0.9905098676681519, -1.4255681037902832, 0.266061931848526, -1.0548475980758667], [0.1408819854259491, -1.0635402202606201, 1.0795679092407227, -0.5284725427627563, -0.7455523610115051, 0.2375108301639557, 0.7832934856414795, -2.133039712905884, 1.2381577491760254, -2.183763027191162, 1.2698825597763062, -0.17810116708278656, 1.879820704460144, -1.3570736646652222, 0.6940580010414124, -0.09378355741500854, -0.40424615144729614, 1.606370449066162, 0.7562985420227051, -1.6312676668167114, -1.0399677753448486, -0.3018217384815216, -1.0953723192214966, -0.45708373188972473, 0.371202290058136, 0.717266857624054, 0.2892422080039978, -0.4547746777534485, -0.617138683795929, -1.4956257343292236, 0.8241078853607178, 2.038084030151367, -0.23356400430202484, -1.0388140678405762, -0.2636132836341858, 0.6981489062309265, -0.13163970410823822, -1.1163288354873657, 0.14699625968933105, -0.9880703091621399, -0.05837765708565712, 0.22098194062709808, 0.35085591673851013, -0.358756959438324, -0.27546799182891846, -0.4423374831676483, -0.9509467482566833, -0.8534757494926453, -1.2491123676300049, 0.8041781783103943, 0.12376823276281357, -0.016486721113324165, -0.7900736927986145, 0.6095991134643555, 0.0036265458911657333, -0.5175578594207764, -0.5229445695877075, -0.013909959234297276, -0.5023171901702881, 0.4783375561237335, 1.2719573974609375, -0.28795483708381653, -0.9511564373970032, 0.31434139609336853], [1.5245333909988403, 1.1386690139770508, -0.8804996013641357, 0.18882310390472412, -1.3920705318450928, -0.47002455592155457, 0.8320056796073914, 0.4295872151851654, 0.6461541056632996, 0.3202352523803711, 0.5643700957298279, 1.3140190839767456, 0.1931745857000351, 0.6344009041786194, -0.9025281667709351, 1.5446879863739014, -0.6745656728744507, 0.6035366654396057, 0.34802067279815674, 1.013264536857605, -0.15525782108306885, 0.961756706237793, -0.4448696970939636, -0.3398366868495941, 1.106404185295105, 0.5365340709686279, -0.6868749260902405, -0.8679141998291016, -1.6279335021972656, 0.5670866370201111, -0.6120213270187378, 0.9156913161277771, 0.4199088513851166, -0.17728962004184723, -1.9561831951141357, -1.3048062324523926, 2.0349864959716797, -1.728818416595459, 0.8328403830528259, 0.43136072158813477, 0.8981956839561462, -0.2003355622291565, -1.770339846611023, 0.7347666621208191, 1.0318732261657715, -1.058814287185669, 0.4139862060546875, -0.3280009627342224, -1.4110448360443115, -0.21183328330516815, -0.7743691802024841, 0.5207183957099915, -0.36996081471443176, 0.2767610251903534, -0.38212695717811584, -0.676422119140625, -1.2117478847503662, 0.05543798580765724, 1.2779995203018188, -0.6445444822311401, -0.11182943731546402, 1.7499843835830688, -0.41436269879341125, 1.3139684200286865], [0.8972454071044922, 0.49077752232551575, -0.4942087233066559, -1.4488052129745483, 0.371332049369812, 0.22580085694789886, -0.7288393974304199, -1.1226547956466675, 0.925942599773407, -1.248664379119873, 0.0009272750467061996, -0.19708219170570374, -1.1379374265670776, -1.09846031665802, 0.621705949306488, 0.5101522207260132, -0.8404812216758728, -0.15134863555431366, -0.16337567567825317, 1.9223427772521973, -0.21658185124397278, -1.5287679433822632, 0.5775974988937378, -0.9416013360023499, 0.6720682382583618, 0.18421021103858948, 1.2081396579742432, -0.80799400806427, -1.1258758306503296, 1.1203604936599731, 0.20474693179130554, 2.587185859680176, -0.5418621897697449, -0.06726596504449844, -0.8231900930404663, -1.1003966331481934, -0.15805210173130035, 1.683738350868225, 0.0518806166946888, -0.4332282245159149, -0.771531879901886, 0.007761744316667318, 0.39662379026412964, 0.173410102725029, 1.2577170133590698, -0.22105365991592407, 0.2551203966140747, -0.653042733669281, -0.6090558171272278, -0.34237709641456604, -0.4992464780807495, -1.5851657390594482, 0.5732770562171936, 1.3878421783447266, 0.37089675664901733, -0.5123499035835266, -0.7220097780227661, 0.6940488815307617, 0.30926501750946045, -0.1043170839548111, -1.8948962688446045, -0.28442034125328064, -0.26369112730026245, -1.527432918548584], [-0.612921416759491, -0.9343467354774475, 1.5182414054870605, -1.5134730339050293, 0.03217799961566925, -0.3365403115749359, 0.9562633037567139, 0.8846501708030701, 0.2382798045873642, 1.09426748752594, 0.039735425263643265, 1.0916777849197388, -0.7524212598800659, -2.149700403213501, -0.3819229006767273, 0.7781103253364563, 1.3746309280395508, -1.3052098751068115, -1.0797624588012695, -1.0950849056243896, 0.13113750517368317, -1.133247971534729, 0.911911129951477, -0.049467142671346664, 0.2994841933250427, -0.9236466884613037, 0.5203896760940552, 0.8255240321159363, -0.6177451014518738, -1.7050036191940308, 1.1110801696777344, -0.04862121120095253, -0.6536596417427063, -0.05965770035982132, -0.35799774527549744, 2.1695058345794678, -0.7911375164985657, -1.1182646751403809, 0.21191678941249847, -0.8838262557983398, 0.8231825232505798, 0.4927811622619629, 0.21913275122642517, 0.3033693730831146, 1.4193068742752075, -0.39278894662857056, 0.9622581005096436, 0.9673852324485779, -1.1491971015930176, 0.2620250880718231, 0.19849392771720886, -0.09766259789466858, 2.0013539791107178, -1.1447961330413818, -0.46215009689331055, 0.024364303797483444, 0.4649135172367096, -1.4437825679779053, -0.5054280161857605, -2.4948439598083496, 0.2216818779706955, -0.9322713613510132, -0.040199484676122665, 0.10036689788103104], [-0.3031817078590393, -0.18040040135383606, -0.29803967475891113, -0.31809157133102417, 0.9747849106788635, 0.9538121819496155, -2.0196261405944824, 3.606773853302002, 1.2091081142425537, -0.5670635104179382, -0.6446287631988525, -0.6657311320304871, -0.5998803973197937, -0.9187802672386169, -0.5928632020950317, 0.20844392478466034, 0.5161709189414978, -0.10136359930038452, -1.7201892137527466, 0.055597178637981415, 0.19149689376354218, 0.10997063666582108, -0.5225638747215271, 1.3927134275436401, 1.6552140712738037, -0.0127754220739007, -0.34561750292778015, -0.3364291191101074, -0.3148611783981323, 0.8542741537094116, 1.3500231504440308, -0.45971906185150146, 1.1864229440689087, -1.1256396770477295, -0.972350001335144, 1.2780905961990356, 0.28002774715423584, -0.7180047631263733, 0.5414977669715881, 0.6189765930175781, -0.523786723613739, -2.1415090560913086, 1.0243785381317139, -0.2317368984222412, 0.5590478181838989, 1.13717520236969, -0.8092815279960632, -1.974135160446167, 0.6264738440513611, -0.7630562782287598, 0.6708058714866638, 0.037329886108636856, 0.4229238033294678, 0.03773196041584015, -1.5731192827224731, -0.5000095963478088, 0.7663285732269287, 0.5595079064369202, 0.03319229930639267, -0.6122186779975891, -0.613598644733429, -0.19376689195632935, -0.6189589500427246, 3.240450620651245], [1.4889355897903442, 0.7250878810882568, 0.7768054604530334, -0.9557408690452576, -0.4468795359134674, -0.9725900292396545, 0.36369913816452026, 0.49911215901374817, 0.173963725566864, 0.3330006003379822, 1.2519853115081787, -0.12849166989326477, -0.9147776365280151, -2.1823952198028564, -0.8072630167007446, -0.600562334060669, 0.26866474747657776, -0.591976523399353, -0.19955529272556305, -0.18402224779129028, 0.6366990208625793, 0.550564169883728, -0.1707289069890976, -0.7451319694519043, 0.4000174105167389, 0.174327552318573, 1.649276852607727, 0.8258261680603027, -0.06786453723907471, 1.2712249755859375, 0.03252405673265457, -1.3220763206481934, -1.374058723449707, 1.2061965465545654, 1.7227822542190552, -0.27264735102653503, -1.3793996572494507, 1.1367897987365723, -1.1123703718185425, -0.25284022092819214, -0.274356871843338, -0.16192755103111267, -0.8432000875473022, -0.22168733179569244, -0.79179847240448, 0.07343655079603195, 0.7882714867591858, -0.5183101296424866, 1.851941704750061, -0.2605377733707428, -0.2633956968784332, 1.1535700559616089, -1.2567280530929565, -0.1577426940202713, 0.211202934384346, -0.7778464555740356, -0.6150678396224976, -0.21615572273731232, 1.3165982961654663, -0.03963823616504669, -1.8759567737579346, 1.1834588050842285, 0.21324723958969116, -2.66141414642334], [0.6145491003990173, 0.9714862108230591, 0.11349016427993774, 0.22655057907104492, -1.1022132635116577, 0.7254752516746521, -0.11794495582580566, 1.28687584400177, 0.3233983814716339, 1.3624738454818726, 1.4804972410202026, -1.318894386291504, 1.1699424982070923, 0.31723102927207947, 2.1697263717651367, -0.8294320702552795, -1.1078203916549683, -1.1734931468963623, -0.04743977263569832, -1.4900250434875488, 0.5335181355476379, 0.27544066309928894, 0.8201345205307007, 0.6058348417282104, 1.3911620378494263, 0.0463140532374382, 1.570196270942688, 0.43086570501327515, -0.1568136215209961, -0.3268173038959503, 0.5147791504859924, 0.6309952139854431, 0.030624549835920334, -0.5045161247253418, -0.10019849240779877, 1.3371844291687012, -0.12047053128480911, -1.051581621170044, 1.178086280822754, 0.6990505456924438, 0.19116871058940887, -0.304591566324234, 1.299440860748291, 1.9244399070739746, -0.5455582141876221, 1.5026448965072632, 1.331381916999817, -1.576686143875122, -0.4804416298866272, 1.5184541940689087, -1.0347143411636353, 2.1312646865844727, -1.8308480978012085, 0.35957178473472595, -0.4625092148780823, 1.509301781654358, 0.25994518399238586, 0.8618618249893188, 1.5971559286117554, -0.5843750834465027, -0.783933162689209, -0.7847636938095093, 0.25589004158973694, 0.4313831329345703], [0.9284312725067139, 1.6781623363494873, -0.23036451637744904, -0.22714771330356598, 1.5012710094451904, -0.725656270980835, -0.6161056160926819, 1.2258108854293823, -0.24635721743106842, -0.9810172915458679, 0.6328300833702087, -0.16860049962997437, -0.5849798321723938, 1.2852678298950195, 0.6706856489181519, 1.8094135522842407, 0.9023816585540771, -3.168391704559326, -0.8565383553504944, 0.30602577328681946, -0.5488079786300659, 1.01030695438385, 0.30211031436920166, 0.6493859887123108, -0.10834769904613495, -0.17869004607200623, -0.991365373134613, 1.3498455286026, -1.397067904472351, 0.20563530921936035, 2.1910135746002197, -1.6785566806793213, -0.7857769131660461, 0.48283520340919495, -1.6602829694747925, 1.1181983947753906, -1.6333543062210083, -1.8232004642486572, -0.6422561407089233, 0.7256489992141724, -0.6705120801925659, -0.08438762277364731, 2.1859617233276367, 0.16824527084827423, 0.8895905017852783, -0.7712540626525879, -0.13397255539894104, -0.9976439476013184, 1.2848082780838013, 0.022241882979869843, 0.5820675492286682, -0.46409982442855835, 0.15486879646778107, 0.9478862881660461, -0.562372624874115, -2.746652364730835, 0.04487770050764084, -0.2815229296684265, -0.9565258622169495, -1.6897410154342651, 0.7195357084274292, -0.07823187112808228, -0.4103128910064697, 0.5527893304824829], [-0.26781538128852844, 2.262416362762451, 0.7543505430221558, -1.9170104265213013, 0.512150764465332, -0.7458502054214478, 0.7055249810218811, -0.02963508665561676, -0.7397029399871826, -0.6086012721061707, -0.8622791767120361, -1.0192599296569824, 1.5286849737167358, -0.1267523318529129, 0.07088588923215866, -1.0618376731872559, -0.954338788986206, 1.1890618801116943, 0.8404805660247803, 1.8221479654312134, 1.235884428024292, 0.2507012188434601, -0.3898761570453644, -1.4418200254440308, -0.7061004042625427, 0.44270312786102295, -0.00028230471070855856, 0.36604440212249756, -0.2573912441730499, -0.6733684539794922, 1.3097949028015137, 1.0472698211669922, 0.42730286717414856, -0.9129706621170044, -0.529334545135498, 0.1023968830704689, -0.12109260261058807, 0.7050039768218994, 0.2169870138168335, 0.28972485661506653, 0.9333177804946899, -0.610379159450531, 0.2739284336566925, 0.7116110920906067, 0.35580161213874817, 0.34402143955230713, -0.6142535209655762, -0.28688132762908936, -1.368216633796692, 1.3628093004226685, -0.7290237545967102, -0.20223551988601685, 2.1749839782714844, 0.03989781439304352, -1.1884937286376953, -0.3157738447189331, -0.7900847792625427, -0.15038831532001495, -0.08022335171699524, 1.04945707321167, 1.437001347541809, 1.5193796157836914, -0.0356147401034832, -0.04436377435922623], [-0.5318407416343689, -1.0308083295822144, 0.18604521453380585, -0.4090668857097626, 0.12361665070056915, -1.4988601207733154, -1.6898138523101807, 1.3038140535354614, 1.0535352230072021, 0.06968729197978973, -0.9753168821334839, -1.0446901321411133, -0.7261367440223694, 1.227878451347351, -0.843534529209137, 0.4076768755912781, -0.3485696017742157, 0.7329705953598022, -1.5515087842941284, -0.4070887863636017, -0.9436920285224915, 0.7276445031166077, 1.6323293447494507, -0.07576267421245575, -0.38381776213645935, 0.36001625657081604, -0.016702424734830856, -0.03157014399766922, 0.8568024039268494, -1.4415053129196167, -0.04613038897514343, -0.5498122572898865, 1.19435453414917, 1.112053394317627, -0.19949205219745636, 0.7425518035888672, 0.5283064842224121, 0.3624688684940338, -0.6353136301040649, -0.0678313747048378, -0.3485100567340851, -0.0919375941157341, -0.1287676841020584, 0.07611063122749329, -2.087474822998047, 1.6763107776641846, -0.7971993088722229, -2.2887215614318848, -1.7434470653533936, 0.6295711994171143, 0.9371106624603271, -2.123628616333008, 0.04948681220412254, 0.9745271801948547, 1.0523114204406738, 0.9650108814239502, 0.6883586049079895, -1.3162546157836914, -0.3297172486782074, -0.7628571391105652, -3.3800711631774902, -1.7268680334091187, 0.39350229501724243, -2.983171224594116], [1.3371191024780273, -0.6888765096664429, 0.260057657957077, -1.0142182111740112, 0.4792485535144806, 0.9535477757453918, -1.5522310733795166, -2.046013116836548, -0.6252785325050354, -1.824174404144287, -1.385124683380127, -1.1242246627807617, 1.4224154949188232, -0.4931156039237976, 0.15584860742092133, 0.038221292197704315, 0.43514832854270935, -0.8171049952507019, -1.335140585899353, -0.07002127915620804, -1.0566688776016235, -1.1334329843521118, 0.3508351445198059, 0.5139689445495605, -0.46441468596458435, -1.060476541519165, -2.710531234741211, 0.02105429768562317, 0.5327560305595398, 1.1930428743362427, -0.4913159906864166, -0.06193014979362488, 0.07315424084663391, 0.1534537374973297, -1.389194369316101, -0.5864909291267395, 0.3278043568134308, 0.2105950266122818, -1.2519168853759766, 0.5632777810096741, 0.8568647503852844, -0.9620345830917358, -0.44109463691711426, 1.7947176694869995, 1.1846729516983032, -1.4686592817306519, 1.3538057804107666, -0.39358171820640564, -1.8213951587677002, 0.11286895722150803, -1.683964729309082, -1.8644828796386719, -0.39013174176216125, -1.533381700515747, 0.3046494424343109, 1.6631666421890259, -0.5118927359580994, 0.14213424921035767, -0.833198070526123, -0.9286824464797974, -0.3075356185436249, 0.5384818911552429, -1.5996954441070557, -1.9246244430541992], [-1.6627603769302368, -0.1264459639787674, -0.01799039915204048, 0.39468470215797424, 0.09934420138597488, -1.2024829387664795, 0.6917973160743713, -0.9406099319458008, 0.36069029569625854, 0.5453439950942993, 0.7145026922225952, -0.31008392572402954, 0.18893346190452576, 1.436067819595337, -1.3276915550231934, 0.8255829811096191, -0.6870090961456299, -0.9624531865119934, 0.5581130385398865, 0.6065146923065186, 0.3243437707424164, 0.6033799648284912, 1.7241997718811035, -1.8221389055252075, 0.8428553938865662, 0.8825592994689941, 0.8479925990104675, -0.7589349746704102, 0.6573185324668884, 0.09863793849945068, -0.5690882205963135, -1.1046022176742554, -0.7012397646903992, -2.434871196746826, 0.5170193314552307, 1.9440276622772217, -0.3278115689754486, -1.182378888130188, -0.6949798464775085, -0.8931626677513123, 0.025358401238918304, -0.6863003969192505, -0.2787637710571289, 0.33458471298217773, -1.0979108810424805, -1.2398381233215332, 0.9027131795883179, 0.330314576625824, 0.43915754556655884, 0.351678729057312, 0.6653703451156616, -0.033297061920166016, -0.5307284593582153, -1.0929137468338013, -0.9925536513328552, 0.4011025130748749, -0.6937310099601746, 0.06675262004137039, -0.276981920003891, -0.8271044492721558, -2.390742301940918, 1.5232456922531128, 0.14038115739822388, -0.46364617347717285], [0.7395400404930115, -1.0677733421325684, 1.1352399587631226, -1.2245216369628906, -1.0035083293914795, -1.0549564361572266, 0.5615648031234741, -0.8438355922698975, 1.67436945438385, 1.7792956829071045, -0.8441148400306702, 0.8735030889511108, -0.767698347568512, -1.627066969871521, 0.7680050134658813, -0.2639577090740204, 1.0603448152542114, 0.6349446177482605, 1.4562567472457886, 0.5598831176757812, 0.3124467730522156, 0.673805832862854, -1.1233611106872559, -1.1345964670181274, 0.3910319209098816, -0.6581020355224609, -1.2104350328445435, -2.0063796043395996, -1.3779743909835815, -0.7479833960533142, -0.08923783153295517, 0.08381390571594238, 0.018983639776706696, 0.2740919888019562, -0.8965399861335754, -1.099198341369629, 0.18670497834682465, -0.22501543164253235, -0.5072634220123291, 1.1934798955917358, -0.5533477663993835, -0.705671489238739, 1.0242353677749634, -0.5662878155708313, 0.43938642740249634, 1.0687780380249023, -0.6641544103622437, -0.3908945918083191, 0.9630682468414307, 1.3296836614608765, -0.27081388235092163, -0.7722589373588562, 1.7939965724945068, -0.8465247750282288, 0.6209031939506531, -0.7795130610466003, -0.09823136031627655, -1.0619211196899414, -0.21128126978874207, 0.3866526186466217, 0.3378506600856781, -1.09524405002594, 0.6774613261222839, 1.1651718616485596], [0.017460105940699577, 0.8726158738136292, 0.9694608449935913, 1.5684268474578857, 0.8025372624397278, 0.04640430212020874, -2.1717700958251953, 0.8939377665519714, -0.5964484214782715, 0.68784099817276, -0.9123677015304565, -1.284479022026062, 0.3260655403137207, 1.0411287546157837, 1.1030632257461548, 0.10140250623226166, -1.580132007598877, -1.4338406324386597, -0.3707619607448578, -2.3020036220550537, 0.09570403397083282, -0.89023756980896, 0.29851269721984863, -0.4591028094291687, 2.3681511878967285, 0.9446857571601868, 1.5407772064208984, 0.741303026676178, 0.10400069504976273, -0.10085484385490417, -0.28556129336357117, -1.0938059091567993, 0.16621027886867523, 0.8392727375030518, 0.9294852614402771, -0.9103633165359497, -1.0474522113800049, -1.6432820558547974, 1.1868445873260498, 1.9213545322418213, 0.9476873278617859, -1.3156421184539795, 0.01908409409224987, -3.2088568210601807, -0.8472592830657959, 0.1423376500606537, -0.12706787884235382, 0.3179478049278259, 1.125385046005249, 0.3933437168598175, 0.03288353234529495, 1.9998117685317993, -0.4810865521430969, -1.191573977470398, 0.8325822949409485, 1.443312406539917, 0.9662548303604126, -0.5597343444824219, 0.9503501057624817, -0.23903733491897583, -0.04551908001303673, 0.36256837844848633, -0.5485302805900574, -1.9373654127120972], [-0.45191171765327454, -0.3033958673477173, -1.3720510005950928, 0.1987667679786682, -0.1755356341600418, 0.6710509657859802, -1.8514372110366821, 0.859446108341217, -0.52110755443573, 0.21056318283081055, -1.7022794485092163, -1.2041771411895752, -0.4722578823566437, 0.8428348898887634, 0.25502216815948486, 0.9229522347450256, 0.5169466137886047, 1.0056567192077637, -0.7634777426719666, 1.4156827926635742, -0.9325491189956665, -0.4288921058177948, 0.20344890654087067, -0.4073197841644287, 0.07613780349493027, -1.3046669960021973, 0.9027582406997681, 0.480183482170105, -0.27389609813690186, 1.7669684886932373, 0.13111285865306854, -0.1517430543899536, -0.9080892205238342, 0.7863037586212158, 2.359161376953125, -0.14610198140144348, 0.11665864288806915, 0.3126622140407562, 1.6527553796768188, 1.6483367681503296, 0.6134049296379089, -0.8917091488838196, 1.2000137567520142, -1.4832552671432495, 0.9345014095306396, 0.6990865468978882, -0.47207561135292053, 1.4145557880401611, -0.1207280158996582, 0.4985785186290741, 0.008469084277749062, -0.48049700260162354, 0.6325719952583313, -1.6359479427337646, -0.09514163434505463, -1.1644576787948608, -1.6403166055679321, -0.10324937105178833, 0.520878791809082, 0.9185346961021423, 0.7838477492332458, 1.099713683128357, 0.48890790343284607, 0.5947924852371216], [1.2230857610702515, -1.5581328868865967, -1.1375181674957275, 0.054715003818273544, 0.37558236718177795, -0.15393351018428802, -0.8185817003250122, 0.7759615778923035, -1.4067586660385132, -0.05090753734111786, -0.024725066497921944, 0.4484575688838959, 0.03554503247141838, 0.8893797397613525, 1.617403507232666, -0.2523389458656311, -0.23189258575439453, -1.2878963947296143, 0.2162574827671051, -0.7925839424133301, 0.36643463373184204, -0.7823907136917114, -0.049362506717443466, 1.202842354774475, 0.05203426629304886, -0.042623940855264664, -0.16314846277236938, -0.38123267889022827, 0.09237530827522278, -0.18990087509155273, 1.7942808866500854, -0.2425466626882553, -0.45882171392440796, 1.9065395593643188, -0.15929631888866425, -1.007063865661621, 0.8218938112258911, -0.6154588460922241, 0.5831270217895508, -0.4839642643928528, 1.124823808670044, -2.629392147064209, 0.525066077709198, -1.3659933805465698, -0.4918559789657593, 0.6195708513259888, -1.0471678972244263, -0.8992210626602173, -0.6973399519920349, 0.29324933886528015, -0.4728180170059204, -1.4326263666152954, 0.44265010952949524, 1.0113080739974976, 2.0907506942749023, 3.429283380508423, 1.9889734983444214, 0.49934402108192444, 0.6765379309654236, 1.5204405784606934, -0.24606828391551971, 1.212144374847412, -2.4434115886688232, -0.7430596351623535], [0.9899663329124451, -0.3880414664745331, 0.16745302081108093, 0.1839524209499359, 0.864068329334259, 1.3914300203323364, 0.8050693869590759, 0.13225415349006653, -0.9218882322311401, 0.8518266677856445, -0.6767262816429138, 2.3604893684387207, 0.6203365325927734, 0.6225194931030273, 2.524045705795288, 0.7984907627105713, 1.3055304288864136, 1.1875981092453003, -0.8907848000526428, -0.932223916053772, -1.2787911891937256, -0.2946324646472931, -0.8931676149368286, -0.4503096640110016, 1.0315207242965698, 0.8995550870895386, 0.47442516684532166, 1.128689169883728, 0.8698562383651733, -1.6220512390136719, 0.16197620332241058, -1.4695448875427246, -1.531313419342041, 0.23823551833629608, 1.0852268934249878, -0.0006216297624632716, -0.29984426498413086, -0.8526277542114258, -0.07099675387144089, 0.9579885005950928, 1.1463744640350342, -0.9774044156074524, -2.1970605850219727, -0.5037580132484436, 0.06476910412311554, -1.0652788877487183, -0.5337664484977722, -2.074587821960449, -0.8369809985160828, -0.5904276371002197, -2.2263121604919434, 0.5449471473693848, -1.435081958770752, 1.0313023328781128, 0.9977725148200989, -1.2907543182373047, -1.2418036460876465, -0.42857080698013306, 1.1734719276428223, -0.2573554515838623, 2.4701499938964844, -1.2131634950637817, 0.91484534740448, 0.41911786794662476], [-2.1153013706207275, 1.551020860671997, -0.8731377720832825, 0.5653997659683228, 0.11800000816583633, -0.9330654144287109, -0.4489969313144684, 0.43088188767433167, -0.6975055932998657, -2.398209810256958, -1.1500182151794434, 1.0170482397079468, 0.9483156204223633, -0.716765820980072, -0.042057104408741, 1.3296254873275757, -0.24989336729049683, -2.7115628719329834, 0.3368813097476959, 0.788033664226532, 0.5882259607315063, -1.3156172037124634, -0.7347075343132019, 0.356896311044693, 0.303176611661911, 0.6476500034332275, -0.4059271216392517, 1.7999149560928345, -0.10445341467857361, 0.08864124864339828, -1.6187376976013184, 0.13320261240005493, -1.7676339149475098, -1.241794466972351, -1.3682563304901123, -0.06270372867584229, -0.5676303505897522, 0.06402876228094101, -0.49450597167015076, 1.808126449584961, -0.48456010222435, -0.43451303243637085, 1.0619808435440063, 0.5151874423027039, -1.2588813304901123, -1.0977604389190674, -0.38639575242996216, -1.7503858804702759, 1.1784493923187256, -0.8465350270271301, -0.5208759307861328, -2.0448904037475586, -0.3962348699569702, -1.5465798377990723, -0.5484538674354553, -1.9672073125839233, 0.559334933757782, 0.05254412069916725, -0.2591710090637207, -0.202398881316185, 0.5002695322036743, -0.2675478160381317, -0.9175098538398743, -0.0820874497294426], [0.7925017476081848, -0.10987485945224762, -0.23465344309806824, 1.728452205657959, -0.26727917790412903, 0.2381819635629654, -0.749117374420166, -0.20641003549098969, -0.026249106973409653, 1.2703750133514404, -1.1299039125442505, 1.04645836353302, 0.2682651877403259, 0.6243360042572021, 0.5892789363861084, 0.7718554735183716, 0.27861684560775757, -0.7420773506164551, -1.0606576204299927, 1.1275124549865723, -1.3923532962799072, 0.8112056851387024, -0.16390688717365265, 0.30426669120788574, 0.9454904198646545, -0.19583666324615479, -0.6825816631317139, -0.4581086337566376, -0.10673101246356964, 1.526114821434021, -0.6156101226806641, -0.43659207224845886, 0.13921977579593658, 1.1186631917953491, 0.8220068216323853, -0.7666946053504944, 1.542115569114685, 2.3639891147613525, 0.015581810846924782, 1.8347305059432983, -1.0804423093795776, 0.5432010889053345, -0.14097540080547333, 0.3724060356616974, 0.4279254376888275, 0.9298095703125, 1.2597525119781494, -0.4187953770160675, 0.05332094430923462, 0.923879861831665, 0.9865953326225281, -0.27373993396759033, 0.7638239860534668, 2.5170154571533203, 1.5368775129318237, -0.5317464470863342, -1.171002745628357, 0.8127171993255615, 0.8455464243888855, -1.2300306558609009, -0.1949877142906189, 0.439017653465271, -0.38014736771583557, 0.14691753685474396], [-0.3958134651184082, -1.2471892833709717, 1.1939390897750854, 0.06500360369682312, 1.1944564580917358, -0.5252939462661743, 0.23142804205417633, -0.6238486766815186, -0.5674618482589722, -1.1215344667434692, -1.1357760429382324, 0.6923083066940308, 2.0026917457580566, -0.7525389194488525, 2.3999087810516357, 1.59409761428833, 1.7499322891235352, -1.324864387512207, -2.851247549057007, -0.3136928081512451, 0.36553382873535156, -0.4816214144229889, -0.8076115250587463, 0.20706366002559662, -1.2535508871078491, 1.9165757894515991, -0.03174793720245361, 1.485564112663269, 0.2879997491836548, 1.0273813009262085, -0.31029489636421204, 1.2637919187545776, -0.7335999011993408, 0.8899279236793518, -0.5759331583976746, 0.7927131056785583, -0.2580908536911011, 1.190025806427002, -0.08351222425699234, 0.6386273503303528, 0.5619540810585022, -1.4337502717971802, 1.185645580291748, 0.13065531849861145, 0.9836543798446655, 0.18833930790424347, 0.1766366809606552, -0.6023330092430115, -0.15128815174102783, -1.0058127641677856, -0.23565369844436646, -0.8558872938156128, 0.8922633528709412, 0.8304644227027893, -1.023537516593933, 0.6997227668762207, 0.1080515906214714, -0.35748958587646484, -0.8127859234809875, -1.220493197441101, 0.29608669877052307, 1.5500810146331787, 0.34370705485343933, 1.3762255907058716], [-2.213430404663086, 0.009858930483460426, 0.32447242736816406, 1.2594306468963623, -0.11493634432554245, 1.5427260398864746, -0.3446556031703949, 1.877913236618042, 0.29445889592170715, -0.5665608644485474, -1.570910096168518, -0.8639976978302002, 0.7994933128356934, -1.0073556900024414, 0.178764209151268, 0.5672069787979126, 1.2330610752105713, 0.20033042132854462, 1.0851716995239258, 0.9466822743415833, 0.9608513116836548, 1.0504343509674072, 1.1488186120986938, 1.4893583059310913, -0.2596766948699951, 0.8005566000938416, 0.5505247712135315, 0.31268376111984253, 0.45737770199775696, 0.3798011541366577, 0.813360333442688, -0.4540660083293915, 1.1676979064941406, -1.0028047561645508, -1.8096120357513428, -1.2894880771636963, -1.6038305759429932, -0.15913242101669312, 0.581794798374176, -0.07030008733272552, 0.3669234812259674, -0.25067460536956787, -0.48587244749069214, 0.6242137551307678, 0.7117756605148315, -0.18185773491859436, -0.8286377191543579, 1.2235465049743652, -0.062062203884124756, -2.080040454864502, 0.771648108959198, -0.11032293736934662, -0.5898216366767883, -0.9554017782211304, 0.07070820033550262, 0.08237487077713013, 1.383917212486267, -0.45154455304145813, -1.499282956123352, 0.31257227063179016, 2.0629324913024902, -0.2833802402019501, -2.266469717025757, 1.3751211166381836], [-0.5910635590553284, 0.42726805806159973, -1.0401448011398315, 1.3057845830917358, -0.5684289336204529, 0.6469315886497498, 1.6193851232528687, 0.6450068354606628, -2.1101443767547607, 1.5702131986618042, -0.30236202478408813, 0.22249320149421692, 1.3302862644195557, 0.2361973375082016, 1.8752424716949463, 0.08216048777103424, -0.6833911538124084, 0.30964359641075134, 0.28086400032043457, 1.3618029356002808, -0.7677399516105652, -0.9781878590583801, 0.45268842577934265, 0.8461610674858093, 0.31052878499031067, 0.22828051447868347, 0.9573119282722473, -0.012220721691846848, 0.979955792427063, -0.7660605907440186, -1.199249029159546, 1.9742732048034668, 0.5793815851211548, 0.4342479407787323, 0.5831930041313171, -1.1204421520233154, -0.6096171140670776, 2.0429158210754395, 0.1759076863527298, -0.11987604200839996, 0.3384944498538971, -0.8381562829017639, -0.09281794726848602, 0.27332359552383423, -0.8275724053382874, -0.6928375363349915, 0.8436453342437744, -0.07767642289400101, 1.3555402755737305, -1.5284475088119507, -0.3006189465522766, 0.3827100098133087, 2.117135524749756, -0.13340416550636292, -0.404214084148407, -0.027927260845899582, -1.039806604385376, -0.4205609858036041, -0.7000495195388794, -0.015703311190009117, -2.1210086345672607, 0.7925882935523987, -1.5716948509216309, -0.27529478073120117], [0.974457323551178, -1.4470114707946777, 0.4822166860103607, 2.578822612762451, 0.33958107233047485, 0.7924783825874329, -0.6213264465332031, -0.145827978849411, 1.3557136058807373, 0.5561226010322571, 0.34842783212661743, -1.837429404258728, 0.1650378704071045, 0.28735679388046265, 1.80636727809906, 1.163360834121704, 1.3070307970046997, -0.1929834932088852, -2.9857101440429688, -0.36316853761672974, 1.0628207921981812, -1.0557910203933716, -0.668346643447876, 1.458585262298584, 1.4089189767837524, 0.3116670250892639, -0.32403242588043213, 0.7894427180290222, -1.0650187730789185, 1.4649187326431274, -0.23731252551078796, 1.6930787563323975, 0.2969248592853546, -0.3570350706577301, 0.49503153562545776, -1.8594175577163696, -0.6328810453414917, 1.1147948503494263, -0.4970841407775879, 1.0287822484970093, 0.5614879727363586, 0.19386273622512817, 0.18377287685871124, -1.3748401403427124, 0.28720784187316895, 2.308558940887451, 1.0929008722305298, -0.10617576539516449, -1.3928488492965698, 1.1825745105743408, 0.4569340646266937, 0.2844412326812744, 0.017374804243445396, -0.2664766311645508, -0.629545271396637, -0.4863317310810089, 0.8884119391441345, -0.2537989318370819, -0.7049800157546997, -0.5332602262496948, 1.108719825744629, -0.5053837299346924, -0.322162002325058, -0.25605717301368713], [1.0786793231964111, -0.6980627179145813, -1.1607056856155396, 0.3099730312824249, -0.28552553057670593, 0.27017608284950256, 0.3551872968673706, 0.7308268547058105, -1.6196216344833374, 1.6473634243011475, 0.11179277300834656, 1.469820499420166, 0.14063385128974915, 1.7376887798309326, 0.720094621181488, -1.3954600095748901, -0.4499521553516388, 1.3620200157165527, 0.31258425116539, -0.7330281138420105, -0.6099537014961243, 0.5177463889122009, -0.968212366104126, 2.0705769062042236, -1.3426412343978882, -0.10842304676771164, 0.28105613589286804, 0.564384400844574, -1.2767974138259888, 0.20563876628875732, 0.16954585909843445, 0.005988285876810551, -0.13531392812728882, 0.52669358253479, -1.583571434020996, -0.24405533075332642, 1.2076505422592163, 1.6294761896133423, -1.2135645151138306, 0.99261474609375, 1.3496609926223755, 2.125741958618164, 0.3855486512184143, -0.2094583809375763, 0.5879612565040588, -1.7471699714660645, 1.417601466178894, -1.615058183670044, 1.3360021114349365, 1.180450677871704, 0.35820627212524414, 0.14442701637744904, 0.5289159417152405, -0.06405269354581833, 0.5475842952728271, -2.472003698348999, 2.133234977722168, 1.16788911819458, 0.0064665270037949085, 1.1090303659439087, 0.6257075071334839, 0.6529183983802795, 0.6495307683944702, -0.5386527180671692], [-0.6699104309082031, -0.7718663811683655, -0.2904960513114929, -0.36835724115371704, -1.272375464439392, -0.8139930367469788, -0.050219107419252396, 0.8486243486404419, -0.40976089239120483, 2.7328548431396484, -0.9438807964324951, 1.3410993814468384, -0.2545459568500519, 0.1728859543800354, 0.5164057016372681, 1.2460800409317017, 1.4880956411361694, 0.54805588722229, -0.5005587935447693, -0.9896432161331177, 3.020280122756958, 0.37917211651802063, 0.849311888217926, -0.8670551180839539, -0.3054036498069763, -0.5739697813987732, -0.10113096982240677, 1.3177731037139893, 0.9610666632652283, -0.848637044429779, 0.6283299326896667, 0.15070657432079315, 0.603360116481781, -1.042235016822815, 0.1832760125398636, -1.1662485599517822, -0.35395529866218567, -0.1479792445898056, 0.5290138721466064, 1.1760175228118896, -0.7002864480018616, 1.7709678411483765, -1.0902827978134155, 0.6079094409942627, -0.48696964979171753, 0.9218037128448486, 0.786931037902832, -0.4394652843475342, 0.6976035237312317, 0.35849496722221375, -0.08618520945310593, -0.15746772289276123, -0.26700296998023987, 2.2910268306732178, -0.3635200262069702, 0.4013996422290802, 1.201366901397705, 0.03176633268594742, -0.1944257915019989, -0.3013245761394501, -0.9136337041854858, 0.9978044033050537, 1.0701851844787598, -0.6280463337898254], [-0.5209980607032776, -0.827894389629364, -1.6950814723968506, 2.407541275024414, 1.2096798419952393, 1.858564853668213, 0.916634202003479, -0.7140551209449768, -0.6582242846488953, -0.4750080406665802, 1.2542015314102173, -0.9329515695571899, -0.6939086318016052, -0.9738593697547913, 1.770459532737732, 0.07277187705039978, -0.18406403064727783, -1.0264538526535034, 1.0481470823287964, 0.6508021950721741, 1.4804819822311401, -1.062119483947754, -1.017874836921692, 0.07120605558156967, -0.45273953676223755, -1.7171322107315063, -2.0064713954925537, -1.7065582275390625, -0.24844172596931458, 2.4602766036987305, -0.8633603453636169, -0.12759895622730255, -0.19351400434970856, -0.5060099363327026, 0.3096359968185425, 0.5721799731254578, 0.5033611059188843, -0.2570194602012634, 1.2023309469223022, -1.037640929222107, -1.130301833152771, 0.05918828397989273, 1.7446414232254028, 0.6228752732276917, 0.6118288040161133, 0.24821323156356812, 0.49355342984199524, -0.4370381236076355, -0.6215299367904663, -1.0114126205444336, -1.3614295721054077, -0.47005581855773926, -0.3995784819126129, -0.7597801685333252, 0.5274028778076172, -0.47403058409690857, -0.9979562163352966, -1.466934084892273, 0.22409766912460327, -0.24091587960720062, -0.7234287261962891, -0.5538893938064575, -0.7983282804489136, -0.210174560546875], [-0.44316843152046204, -0.5320510268211365, -0.2966032028198242, 1.6218053102493286, 0.8516132831573486, 0.34207701683044434, 0.9590677618980408, -2.1381378173828125, 7.439550245180726e-05, -1.4096332788467407, -0.5731884837150574, 1.8251315355300903, -1.016179084777832, -1.2665553092956543, -0.4901729226112366, 1.1077237129211426, 0.7165125608444214, -0.3037932515144348, -0.7962767481803894, 2.136410713195801, 0.8287441730499268, -0.7923381328582764, 2.697787046432495, -0.857160210609436, 0.0066116428934037685, -0.15381073951721191, 1.6052104234695435, -0.6659634113311768, 0.8273650407791138, -0.19109490513801575, 0.1194031685590744, 1.7797276973724365, -0.4163118004798889, 1.123924732208252, -2.025529146194458, 0.2203223556280136, -0.9516527652740479, 1.1705189943313599, 0.18249580264091492, 1.6397569179534912, -2.0501482486724854, 1.684748888015747, 0.8689939975738525, -0.14195607602596283, 0.7377640008926392, 0.47716793417930603, 0.9503431916236877, 0.48252829909324646, 1.1313495635986328, -0.8074721097946167, 2.517518997192383, 0.23487016558647156, 0.6397970914840698, 0.25720494985580444, -0.38424062728881836, 0.41597482562065125, 1.7549078464508057, -0.5316081047058105, -0.608216404914856, -0.00583001459017396, -2.3942346572875977, -1.3332678079605103, 1.6653956174850464, -1.6493438482284546], [-0.5604453682899475, 1.2866811752319336, 0.8262913227081299, 0.41509658098220825, -0.2868524193763733, -0.6324208378791809, -2.674298048019409, 0.4589037597179413, -0.45507338643074036, -0.14039337635040283, 0.7498410940170288, -0.8241403698921204, -2.3974921703338623, -1.883087158203125, -1.420201301574707, -0.13155339658260345, 1.3345746994018555, 0.3224433660507202, 1.5314208269119263, 1.6673935651779175, 1.6332727670669556, -0.31826695799827576, 1.9375925064086914, 0.4955444931983948, 0.339063823223114, 0.9991092085838318, -1.1473357677459717, 0.026834076270461082, -0.5641118288040161, -0.3907342553138733, 0.7005397081375122, 1.5706384181976318, 0.08125706762075424, -0.4438358545303345, -0.09026779979467392, 0.29688379168510437, 1.3333193063735962, -0.5441356301307678, 1.253462314605713, -1.201436161994934, -0.4059564769268036, 0.4936526417732239, 0.755512535572052, -1.0713274478912354, 0.509506344795227, 2.6537671089172363, 0.9448918700218201, 0.7807536125183105, 1.3794424533843994, -1.136985421180725, -0.845525324344635, 1.4170427322387695, -0.6894973516464233, 1.8407961130142212, -0.4262813329696655, 1.6411203145980835, 1.6503609418869019, 0.8331395983695984, -0.6208426356315613, 0.26595616340637207, -0.47512197494506836, -0.4229852855205536, -0.4757136404514313, 0.2560778856277466], [0.05139768496155739, -1.3866742849349976, 2.6109161376953125, 1.448848009109497, -0.8830339908599854, 1.3609223365783691, 0.04756724834442139, 0.21362172067165375, 0.5180573463439941, -0.5230199694633484, -2.626034736633301, -1.309561014175415, -0.031540222465991974, 0.7987470030784607, 0.1354551762342453, -0.3886392116546631, 0.4166327118873596, 0.5108798742294312, 1.176409125328064, -1.2029184103012085, -0.9208698272705078, -0.53072589635849, 0.6615850329399109, 0.5503515601158142, 1.5452333688735962, 0.21005818247795105, 0.09206119924783707, -0.7145877480506897, -1.4124490022659302, 0.8067770004272461, 1.0179389715194702, -1.4244575500488281, -0.9499384760856628, 1.1712863445281982, 0.11096273362636566, -0.07419700175523758, 0.9982513785362244, 1.9537227153778076, -1.414463996887207, -2.203387498855591, 0.004121455829590559, 0.7268692255020142, -1.0468138456344604, -0.36173391342163086, 1.0598677396774292, -0.11686508357524872, 0.9311862587928772, 1.1864596605300903, -0.4164997935295105, 0.21950466930866241, -1.2685463428497314, -1.3721065521240234, -1.457836627960205, -0.49101507663726807, -0.5259053111076355, -0.27623647451400757, -0.8287172913551331, -0.05033254995942116, -0.21123787760734558, -1.5976990461349487, -0.7402005195617676, 0.9694517254829407, 0.8929420113563538, 1.9551609754562378], [-1.0281566381454468, 0.23506569862365723, 0.341279536485672, -0.8952533006668091, 0.47843897342681885, 0.7679904103279114, 0.4825972318649292, 0.3240835964679718, 0.04133491590619087, 0.6976810693740845, 0.21268068253993988, -1.008727788925171, 0.5324535965919495, 0.9146618247032166, 1.0943589210510254, -1.2591791152954102, 0.940496563911438, -1.0132557153701782, 0.5050221681594849, 0.36403870582580566, 1.385946273803711, 1.452582836151123, 1.6590648889541626, 0.8497804999351501, 0.4601995050907135, -0.000675244489684701, -1.2320857048034668, 0.04108645021915436, 0.2876136600971222, -0.9716917872428894, 0.8205673694610596, -0.044302601367235184, -0.745572030544281, -1.6461501121520996, -1.0073846578598022, 0.9524077773094177, -0.11938834935426712, -0.4007343053817749, 0.07596663385629654, 0.09679854661226273, -0.6928013563156128, 0.8371564149856567, 0.5450209975242615, -0.10190384089946747, 1.8865878582000732, 0.4800194799900055, 0.5807450413703918, -0.8138107061386108, 0.2463589608669281, 0.06897369027137756, 0.03258003666996956, 0.8112074136734009, -0.6587793827056885, 0.8410204648971558, 0.43173596262931824, 0.26974862813949585, -1.0478014945983887, 1.4486379623413086, 0.6196801066398621, 1.6133073568344116, -0.7709608674049377, -0.20501762628555298, 0.08747302740812302, -0.3870032727718353], [0.584471583366394, 0.9411905407905579, 0.018411843106150627, -1.7040975093841553, -2.231027841567993, 0.6371644735336304, 0.31599533557891846, -0.2591552436351776, 1.010114073753357, -0.4916573166847229, 0.4902377128601074, -0.121552973985672, 1.670520544052124, -0.7297491431236267, 0.14062035083770752, 0.6734126210212708, 0.65443354845047, 0.6423238515853882, -1.407448649406433, -0.9254648685455322, -3.3298630714416504, 0.9103561639785767, 0.8212472796440125, 0.5797500610351562, 0.002758258953690529, -0.08134858310222626, -0.09217502176761627, 1.3677643537521362, -0.43883442878723145, 0.18989703059196472, -2.6600160598754883, -0.7410624623298645, -0.3010081946849823, -0.27058494091033936, -0.16541413962841034, 0.22997619211673737, -0.8827086687088013, 0.2809467613697052, -1.2360864877700806, -0.6892266273498535, -0.04642628878355026, -1.4910928010940552, 1.6971120834350586, 0.18953004479408264, 0.576263427734375, 0.4312776029109955, 0.8368116021156311, -0.09461693465709686, -2.207723379135132, 0.6028951406478882, -1.4670794010162354, -1.1061774492263794, 0.21124543249607086, -0.6554004549980164, -1.6429250240325928, -0.36050334572792053, -1.1894210577011108, -0.33162781596183777, -0.5528295040130615, 0.48649972677230835, -0.45689713954925537, -0.2618378698825836, -0.995133101940155, -0.022418327629566193], [1.2620481252670288, -0.8588239550590515, -0.5192165970802307, -0.818302571773529, 0.00017521799600217491, -0.04957151040434837, -0.003397643333300948, 1.216606616973877, -0.03711473196744919, 0.6314241886138916, -0.1277623176574707, 0.31239697337150574, 1.1295090913772583, -1.7375487089157104, 0.9203624129295349, -1.9838398694992065, 0.8582209348678589, 0.020532038062810898, 1.6141011714935303, -0.818535566329956, -1.1057134866714478, -1.3713220357894897, 0.10322444140911102, -0.48366251587867737, 1.4638501405715942, 0.7803297638893127, 1.5282716751098633, 0.4655739963054657, 0.15400969982147217, -1.847746729850769, 1.5110257863998413, -0.12044388800859451, 1.0535212755203247, 1.1593796014785767, 2.257467746734619, -1.6293857097625732, 0.2276453822851181, -0.6837578415870667, -0.05145673453807831, 0.3904118537902832, 1.0137765407562256, -1.145182490348816, -0.35043853521347046, 0.19514970481395721, 0.7461117506027222, 0.7269078493118286, 0.2423524111509323, -0.14064079523086548, 0.3439670205116272, -0.6420183777809143, -1.296216607093811, 0.11294713616371155, -0.5744484663009644, 0.20477017760276794, 0.1514073610305786, -1.742340087890625, 1.750274896621704, 2.130345582962036, -0.9547264575958252, 0.5863714814186096, -1.697736382484436, -0.5551427602767944, -0.12596933543682098, -0.685084879398346], [-0.4898660182952881, 0.4005800485610962, 0.32444334030151367, 1.6015931367874146, -0.2696305513381958, -0.4492812752723694, 2.0945398807525635, 0.6974930763244629, 1.4188088178634644, 1.9197769165039062, 0.23467113077640533, 1.3817707300186157, -0.0035963517148047686, 0.9369656443595886, 1.7603611946105957, 0.5366321802139282, 0.9975389242172241, -0.3069803714752197, -0.33801692724227905, -0.6679145097732544, 0.5411327481269836, -0.8420705795288086, 0.9242180585861206, -0.38936811685562134, -2.022026777267456, 1.0749083757400513, -1.088991403579712, -2.289930582046509, 0.08965827524662018, -0.6193745136260986, 0.6989818215370178, 0.8388091325759888, -0.2339617908000946, 0.2277597188949585, -0.2883789837360382, -0.08941546827554703, 0.13177122175693512, -1.136120319366455, 1.1552481651306152, -0.014935826882719994, -0.36321350932121277, 0.836323618888855, 1.308618426322937, 0.7341143488883972, 0.14111526310443878, -1.3540984392166138, 1.160394549369812, -0.7100550532341003, -0.32102879881858826, 0.5715274214744568, -0.2779254913330078, 0.30517908930778503, 0.3482355773448944, 0.19360807538032532, 0.6644327640533447, -0.5388613939285278, -0.47329869866371155, -0.5684868693351746, 1.351341724395752, 1.5166027545928955, -1.8374183177947998, -1.0499818325042725, 0.566402018070221, 0.2126842439174652], [0.6136142611503601, -0.14115649461746216, 0.39286723732948303, -1.7594572305679321, -0.41822946071624756, 1.4031096696853638, -0.18317873775959015, -0.566367506980896, 1.8090258836746216, 0.703283965587616, -0.0338420644402504, -0.5240882039070129, 0.5379869937896729, -2.245728015899658, 0.07156437635421753, 0.9043058753013611, -0.6456037163734436, -0.9447115063667297, -1.1830885410308838, -1.3814316987991333, 0.13060295581817627, -0.19147318601608276, 0.01366309355944395, -0.4398440420627594, -0.6935551762580872, 1.7120404243469238, -0.9338700175285339, 1.02274489402771, -1.685591459274292, 0.8409233093261719, -0.00269373063929379, -0.6962577104568481, -0.2410144805908203, 0.16932986676692963, -0.5318463444709778, 0.4913245141506195, -0.1669521927833557, -0.5254574418067932, -1.9358274936676025, -0.266937792301178, 0.7382233738899231, -0.21976469457149506, -1.37295663356781, 0.19482047855854034, 0.4156433939933777, 0.24140644073486328, 0.510970413684845, 0.32230451703071594, 0.7011350393295288, 0.038525085896253586, 0.11120082437992096, -1.4566102027893066, 1.3471424579620361, 0.42277875542640686, -0.19064141809940338, 0.24329635500907898, -0.9615210294723511, 0.3968866467475891, -1.0318207740783691, 0.5666616559028625, 1.838922142982483, -0.82896888256073, 0.4766479432582855, -1.0485050678253174], [-1.007384181022644, -0.11891869455575943, 1.0050859451293945, -2.061241865158081, 0.2996334433555603, -3.3554608821868896, -0.42946019768714905, -1.6919728517532349, 0.29414013028144836, 0.3291693925857544, 0.39787182211875916, -0.3462564945220947, 0.2848310172557831, -0.07701962441205978, 0.7841612100601196, 0.2663775682449341, -2.3636810779571533, -0.14990867674350739, 0.009686818346381187, 1.0806243419647217, 0.9031047224998474, 2.5851547718048096, -0.8141987323760986, 0.7804303765296936, -1.6029224395751953, 1.1510217189788818, -0.3249679505825043, -1.0070003271102905, -0.6853255033493042, -0.16464217007160187, -0.5257835388183594, 1.6822354793548584, 0.5225366353988647, -0.1785062998533249, -0.49493056535720825, -1.4818822145462036, -1.6037765741348267, -0.2879225015640259, -1.1342196464538574, -0.6032208204269409, -0.07085894048213959, 0.3491722345352173, 1.9306682348251343, -0.8194807767868042, -1.0062154531478882, -0.5910767316818237, -0.6837294697761536, -1.5445526838302612, 0.5957945585250854, -0.8419394493103027, 1.1850847005844116, -0.011459250934422016, -0.41685396432876587, 0.841694176197052, -0.2369796484708786, -0.2153209000825882, 0.7863758206367493, 0.8771159648895264, 1.1187199354171753, 0.6020265817642212, -0.9967291355133057, 1.77666437625885, 1.1280642747879028, -0.24719345569610596], [0.32439202070236206, 0.9341447949409485, 0.35273972153663635, -1.1921409368515015, -0.6684991121292114, 1.462459921836853, 0.9307796955108643, 0.17751182615756989, -0.06408405303955078, -2.1366658210754395, -1.9282236099243164, -0.2806905508041382, 1.9299217462539673, 0.06774505972862244, -2.263371229171753, 1.3924589157104492, -1.3739173412322998, -1.3233152627944946, -1.64798104763031, -0.22386322915554047, -0.3294772207736969, 0.22699867188930511, 0.27721694111824036, 1.2531081438064575, -1.0413718223571777, -1.3637346029281616, -0.6289311647415161, -0.7216386198997498, -0.19525375962257385, 0.6394496560096741, -1.2503983974456787, 1.4046636819839478, 0.03970284387469292, 1.0756481885910034, -1.1688436269760132, -1.0287519693374634, 0.9684745073318481, -0.49326807260513306, 0.3569548428058624, 1.4768208265304565, 1.3472672700881958, 0.9208601117134094, -0.15321524441242218, -0.2998853027820587, -1.3550755977630615, 0.02033144235610962, 0.13634075224399567, 0.3780171275138855, -0.20835751295089722, 0.7647559642791748, -0.053247470408678055, 1.2501059770584106, -1.6574103832244873, -0.45125389099121094, 0.25461944937705994, -0.9010862708091736, 0.5631440281867981, -0.09390044212341309, -0.12880989909172058, -0.4514530301094055, -0.29108574986457825, 0.3165614902973175, -0.8681737780570984, -0.37777942419052124], [0.00491437129676342, -0.9421246647834778, 1.1738969087600708, -0.4932613670825958, -0.988715410232544, -0.8534536957740784, -0.10663385689258575, 1.5499275922775269, -1.9960713386535645, 0.2988967299461365, -1.4376604557037354, 2.1497256755828857, -2.498182535171509, 0.7132775187492371, 0.15153498947620392, -0.24930797517299652, 0.6433717012405396, 1.2175861597061157, 0.2704122066497803, 1.3708454370498657, -0.6449921131134033, 0.20406979322433472, 1.927442193031311, 0.8300220370292664, 1.2158775329589844, -0.6158233880996704, 0.14317472279071808, -0.6276161074638367, 0.6241928935050964, 1.9390835762023926, 2.309291362762451, 0.016536349430680275, 1.0381009578704834, 0.27261778712272644, 0.0012748524313792586, 1.2269247770309448, 1.1331480741500854, 0.27391916513442993, -1.1088554859161377, -0.9472588896751404, -0.8263952732086182, -1.4100747108459473, 0.3400859832763672, 0.16343878209590912, 0.11799958348274231, 0.44528844952583313, 0.5813244581222534, -1.3529328107833862, 1.2077741622924805, 1.257047176361084, 0.39602383971214294, 0.7172206044197083, 0.16815997660160065, 1.3882158994674683, -0.09350044280290604, -0.09877514094114304, 0.8520665764808655, 2.225822687149048, 1.5208052396774292, 0.9170939922332764, -1.0396201610565186, -0.637787938117981, -0.9709586501121521, 0.12461361289024353], [-1.4376389980316162, -1.4829763174057007, -0.10951802879571915, 1.1119807958602905, -0.7905001640319824, -0.607886016368866, -0.2796427309513092, -1.1791502237319946, 0.36575907468795776, 1.7235227823257446, 0.6712727546691895, 0.1003745049238205, -0.4722751975059509, -2.3814339637756348, -0.4343257546424866, 0.3043373227119446, -0.015575306490063667, 0.32803475856781006, 1.0106070041656494, -1.2978219985961914, 1.0571218729019165, -0.702513575553894, -1.4407167434692383, 1.7181851863861084, 1.1732432842254639, -1.9755969047546387, 0.43214964866638184, 0.6736150979995728, -0.18031540513038635, 0.14688004553318024, 0.8760855197906494, -0.9140881299972534, 1.0106728076934814, -0.8187149167060852, -0.2292342483997345, -0.5163086652755737, 0.12355555593967438, -0.1918404996395111, 0.06309711188077927, 0.41722407937049866, 0.3842577040195465, -0.6328533887863159, -1.7890695333480835, 1.3770241737365723, 0.23518773913383484, 0.3270842134952545, -0.893760621547699, -1.9702006578445435, 0.11259210854768753, 0.5023215413093567, 0.31975847482681274, -0.9916917085647583, 0.7557430267333984, -0.9360809326171875, -1.5335636138916016, -0.8628156781196594, -0.4362741708755493, 0.1245141327381134, 2.071808338165283, -0.40921106934547424, -0.03643075376749039, 1.7851284742355347, 0.6226438879966736, -0.06572344154119492], [0.7105295062065125, -1.3597110509872437, -0.42524731159210205, -0.47510483860969543, -0.4013701379299164, -0.41505131125450134, -1.1445910930633545, 0.2605213522911072, 0.5682749152183533, -0.06880523264408112, 0.9227980971336365, 0.6143525838851929, -0.8975446820259094, 0.016881057992577553, -0.9778105616569519, -0.5665702819824219, -2.4703240394592285, -1.601715326309204, -0.1775902658700943, 0.922694742679596, -0.8287487030029297, 0.5352493524551392, 0.8444617986679077, 0.7512888312339783, 0.3430290222167969, -1.8548738956451416, -0.1042434498667717, 0.5357267260551453, -0.7453447580337524, 1.1228020191192627, -0.4122619926929474, -0.9139136672019958, 0.10240744054317474, 1.583945393562317, 0.4069347381591797, 2.8149664402008057, 0.5838784575462341, 1.811931848526001, -1.3698731660842896, -0.26144829392433167, -0.4865235984325409, 0.16797563433647156, 0.36199259757995605, -1.720903992652893, 1.0187252759933472, 0.44484943151474, -0.19664329290390015, -1.884286642074585, -0.7213107943534851, -1.6375854015350342, -0.2132633477449417, -1.823191523551941, 1.396788239479065, 0.8517214059829712, 0.6127004027366638, -1.0442556142807007, -0.42690137028694153, -0.06178848445415497, -0.8271058797836304, 0.28283825516700745, 0.7343356013298035, -0.37376391887664795, 0.102317214012146, -1.0870283842086792], [1.039384126663208, 1.3059732913970947, -0.589041531085968, 0.8543475866317749, 0.5879030823707581, -0.9733262658119202, 1.2747752666473389, 1.4969265460968018, 0.8851943016052246, 0.9518744945526123, 3.2618515491485596, -0.48401206731796265, 2.3897035121917725, -0.13286392390727997, 1.3430328369140625, 1.310485601425171, 0.4208831787109375, 0.24757172167301178, 1.9619858264923096, 0.7831097841262817, 1.3236675262451172, -1.257091760635376, 1.2739061117172241, 0.14276207983493805, 0.8416503071784973, -0.7053878307342529, -0.00159061246085912, 0.0776711106300354, 1.6676491498947144, 0.17697809636592865, -2.199077606201172, 1.0846092700958252, 1.0262582302093506, -0.5785278081893921, -2.405445098876953, 1.0245078802108765, 0.3533377945423126, 0.09881827980279922, 2.1094889640808105, -0.6729413866996765, 0.7014160752296448, -0.6509676575660706, 0.8087548613548279, -1.1203519105911255, -1.379274606704712, -0.38765719532966614, 1.575383186340332, 0.831339955329895, 1.4964232444763184, -0.7533538937568665, -1.3623749017715454, -0.8872895836830139, -0.011185253970324993, -2.071958065032959, -0.26315003633499146, -0.45449891686439514, -1.5058280229568481, 0.2208717316389084, 0.42605116963386536, -1.7441350221633911, 1.5187159776687622, 1.2335577011108398, 0.5972962379455566, -0.20051637291908264], [-1.1547492742538452, -1.102242350578308, -0.48227110505104065, -0.6708102822303772, -1.1414345502853394, -0.28033629059791565, 1.336620807647705, 1.0809593200683594, -2.4966366291046143, 0.3838953375816345, 0.029874373227357864, -0.16850358247756958, -2.795508861541748, 0.24220675230026245, 1.2098264694213867, -0.03731445595622063, -0.02932908572256565, 0.1050170511007309, -0.11968006193637848, -1.8336681127548218, 0.05304345488548279, 0.24125072360038757, 2.4784154891967773, 0.23156586289405823, -0.1522630751132965, -1.5826119184494019, 1.929344654083252, -1.083845615386963, -0.61060631275177, -0.20445087552070618, -0.23987120389938354, -0.6074584722518921, -0.1407080441713333, 1.7580713033676147, 0.32063010334968567, 1.1164090633392334, 0.8539443612098694, 0.3670237064361572, -0.824783205986023, 0.8807315826416016, -1.9208076000213623, -0.17985738813877106, 1.5599132776260376, -1.5358539819717407, -1.0088245868682861, 0.8579064011573792, -0.7988616228103638, -0.3873772919178009, -0.6552340388298035, 0.7435322403907776, -0.6558968424797058, 1.3235117197036743, 0.0712614580988884, -0.1899564266204834, 1.8612141609191895, 1.8307609558105469, 0.8199194073677063, -0.22314153611660004, 1.1780391931533813, -0.27787551283836365, -0.7639196515083313, 0.42704012989997864, -0.9295715689659119, -0.3810712695121765], [0.4256444573402405, -0.06496347486972809, -1.2563143968582153, -0.8752142190933228, 1.634446620941162, -0.7678566575050354, 0.5888761281967163, -1.0907478332519531, -0.0552913136780262, 1.090469479560852, -1.1410067081451416, -0.8520093560218811, -1.6665006875991821, 0.24432773888111115, -1.2682238817214966, -1.3011865615844727, -0.0763113796710968, -0.4006243944168091, 0.14791002869606018, 0.28717926144599915, 0.10087964683771133, -1.1790964603424072, 0.9180822968482971, -0.8346135020256042, 0.39600375294685364, -0.3299403786659241, -0.8115624189376831, 0.17445649206638336, 0.4715411961078644, 1.1564013957977295, 0.8717420101165771, -0.06979281455278397, 0.3803374171257019, 0.646368145942688, 2.001469135284424, -0.02289307303726673, -1.2472935914993286, 0.6756022572517395, -0.9809154868125916, 0.6222427487373352, 0.9289849400520325, -1.2311674356460571, -1.120609164237976, -0.17123909294605255, 0.4959341585636139, 1.921183705329895, 0.2295730710029602, -1.6187477111816406, 0.05710042268037796, 0.6133657097816467, 0.9197294116020203, 3.184302806854248, -1.283799171447754, 1.0165804624557495, 0.18108466267585754, 0.05143900215625763, 1.1651077270507812, 0.0350525788962841, -0.5812356472015381, -1.5833203792572021, -0.1425812840461731, 0.0944424569606781, 0.5910161733627319, -0.7078841924667358], [0.4935412108898163, -0.20243585109710693, 1.1916782855987549, -0.042434412986040115, -1.5891008377075195, -0.9813868403434753, -0.8828484416007996, -1.413314938545227, 0.49098244309425354, 1.1398375034332275, -1.0874673128128052, 0.3086461126804352, -1.5581929683685303, -0.5540727376937866, -1.3861219882965088, 0.8301064372062683, 0.4306789040565491, 1.089337944984436, 0.04324045032262802, -1.486467957496643, 1.485026478767395, -0.0660180076956749, -0.32745346426963806, -0.4129795730113983, 0.7901298999786377, -1.15121328830719, -1.096892237663269, -0.17804312705993652, 1.4642503261566162, 1.414843201637268, 1.0281776189804077, 0.8617487549781799, -0.23495282232761383, -0.7802327871322632, -0.6286378502845764, -1.2596348524093628, 1.161236047744751, -0.7220059633255005, -1.0008726119995117, 1.5633673667907715, 0.6707150936126709, 0.12122996151447296, 0.5546948313713074, -0.8178644776344299, -0.33724260330200195, 0.4429849088191986, 0.7440886497497559, -0.7401099801063538, -0.9607430100440979, 0.11866533756256104, -0.40166279673576355, 0.41038018465042114, -0.15396949648857117, -0.5134865641593933, 1.1137275695800781, 0.2541065216064453, -1.212911605834961, 0.649797797203064, -1.4879329204559326, 0.8923478126525879, 0.5210708975791931, -0.41283902525901794, 0.26257070899009705, 0.18946638703346252], [0.1320621371269226, 0.9884281158447266, 1.581254005432129, 1.2770370244979858, 0.9527239799499512, 1.4490766525268555, -0.14975252747535706, -1.2907699346542358, -0.6750937104225159, -1.0177183151245117, 1.237560749053955, 1.531861662864685, 0.9485154151916504, 1.0546889305114746, 1.9864813089370728, -0.7441931962966919, 0.8815070986747742, -0.868727445602417, -0.5482585430145264, 0.7827881574630737, 0.9079533219337463, -1.523458480834961, 1.6943129301071167, -1.2044858932495117, -0.35432684421539307, 0.5284489393234253, -0.009065900929272175, 0.9873880743980408, -1.365257740020752, 0.1354416161775589, -0.5962800979614258, 0.6925839781761169, -0.0072641693986952305, -0.18128027021884918, 0.38030481338500977, -0.7839900851249695, -1.2340288162231445, 0.5627651214599609, -0.24650195240974426, 1.1712689399719238, -0.8676676154136658, -0.03524259105324745, -0.13768115639686584, 0.5114052295684814, 0.28824079036712646, -0.21493792533874512, 1.0190033912658691, -1.6051058769226074, 0.22122938930988312, -0.24307766556739807, -0.3745833933353424, 0.01434276346117258, 1.0672976970672607, -1.033194661140442, -0.7111543416976929, -0.25176435708999634, 0.6195745468139648, 1.035686731338501, 1.230503797531128, -0.32627227902412415, -0.08145566284656525, -0.18725840747356415, -0.8452860713005066, -1.1128276586532593], [0.18486222624778748, -0.2413358837366104, 0.2717718183994293, 0.18764974176883698, -0.47100090980529785, -0.1756463497877121, 0.457465261220932, 0.8979355692863464, -0.0503215454518795, 0.2780540883541107, 0.2024897038936615, -2.3917758464813232, 1.5203075408935547, 0.661893904209137, -1.8282414674758911, 0.7357209324836731, -0.8906047344207764, -1.6314051151275635, 1.1795192956924438, -0.3733780086040497, 0.19739821553230286, -0.8371931910514832, -0.6306250691413879, 0.35472843050956726, -0.298572838306427, -0.19602632522583008, 0.5561214685440063, -0.19061224162578583, 1.4503979682922363, -2.005715847015381, 0.6178602576255798, -0.8857406973838806, -0.5485758781433105, 0.5057477355003357, -0.6471157073974609, 1.1616575717926025, 0.763710081577301, 0.6335052847862244, -0.5051604509353638, 0.33692872524261475, -0.6201534271240234, 0.2047109305858612, 1.2964266538619995, -0.655938982963562, 2.3679144382476807, -0.5151242017745972, 1.156063199043274, 2.1938843727111816, -1.285788655281067, -0.21409793198108673, 0.14282377064228058, -1.6186531782150269, -0.8152089715003967, -2.430116891860962, -0.2692042291164398, -0.22070413827896118, -1.6364208459854126, 0.6499605774879456, -1.9411888122558594, 1.246337890625, -0.5560115575790405, -1.3444254398345947, -0.3198360204696655, 1.2874493598937988], [-1.3318690061569214, -0.009235775098204613, 0.38597720861434937, -1.1656098365783691, -1.7130711078643799, -1.195415735244751, -0.7927744388580322, 0.2667798399925232, -1.328012228012085, 0.027054419741034508, 0.5071826577186584, -2.1351308822631836, 2.595750570297241, -0.5099091529846191, 1.3491631746292114, -0.26760074496269226, 1.3080294132232666, 0.36502182483673096, -0.5203104019165039, 0.6360083222389221, -0.02129198983311653, -0.32620564103126526, -0.9942580461502075, -1.1114872694015503, -2.772462844848633, -1.248860478401184, -1.620246410369873, -1.2575767040252686, -0.6889851689338684, -0.913069486618042, 0.638433575630188, 0.3356121778488159, 0.8523957133293152, 0.11729656904935837, 1.3457738161087036, -1.18367600440979, -2.7778303623199463, 0.5190230011940002, 0.03064539097249508, -0.17799347639083862, -0.12535710632801056, 0.2047082781791687, 0.34189489483833313, -0.002084876876324415, -0.7275368571281433, 0.1593073606491089, -1.1529680490493774, -0.5845142602920532, 1.3229199647903442, -0.9355796575546265, -2.931401252746582, 1.1060091257095337, -1.8944002389907837, -1.153552532196045, 0.26493003964424133, -1.3131709098815918, 1.574540376663208, -0.174132838845253, -1.501133680343628, -0.35263651609420776, 1.5558702945709229, -1.038903832435608, -0.43117615580558777, -2.032142400741577], [0.41423889994621277, 1.422201156616211, -1.621533751487732, 0.9744565486907959, 0.15082202851772308, -0.8867470026016235, 0.4358574151992798, 1.0319106578826904, -1.781943678855896, -0.8175095319747925, -0.31903600692749023, 0.1769563853740692, -1.0137327909469604, 0.508013129234314, 0.40752851963043213, 0.5608786940574646, 0.7718796730041504, 2.3869504928588867, -0.3997654914855957, -1.9919564723968506, -0.13384534418582916, 0.9339451193809509, -1.1192104816436768, 0.792072057723999, 1.3628736734390259, -1.40275239944458, 0.11394268274307251, 0.06513238698244095, 0.21466104686260223, -0.4370221793651581, 0.6881771087646484, 2.078305959701538, -0.4671006500720978, 0.5072116851806641, 1.154160737991333, -0.15854689478874207, 1.2203483581542969, -0.46167609095573425, 0.14067965745925903, 0.0705149695277214, 1.4894760847091675, 0.25317704677581787, -1.1208839416503906, -0.8335617780685425, -0.24184830486774445, -0.04541803151369095, -0.5221810340881348, 0.8196588754653931, 1.0919127464294434, 0.8527442812919617, 0.26817771792411804, 0.5005730390548706, -0.7263628244400024, -0.09395355731248856, -0.3896711468696594, -1.9073703289031982, 0.10985228419303894, -0.37040355801582336, -0.25375697016716003, 1.344024896621704, 0.3003704249858856, -0.9805795550346375, -0.13250014185905457, -0.5251226425170898], [2.0079636573791504, -2.360926389694214, -0.5129676461219788, -0.4514305591583252, 0.8669536709785461, 0.35601353645324707, -0.2264687567949295, 0.5522439479827881, -2.2595081329345703, -1.0629103183746338, 1.36947762966156, -2.5515389442443848, 0.3085438013076782, -0.03661017864942551, -1.2728264331817627, -0.27032744884490967, 0.13598065078258514, -1.183221459388733, -0.6431754231452942, 0.04778745025396347, -0.07638822495937347, -0.7541792988777161, -0.23618756234645844, -0.19930650293827057, 0.4007328748703003, 0.40741923451423645, 1.1809557676315308, -1.3179936408996582, 0.8163197040557861, 0.537882924079895, 0.015752946957945824, -0.43817853927612305, 1.9211971759796143, 1.7309768199920654, 0.519289493560791, 0.7426850199699402, 0.66419517993927, 0.564979076385498, -0.24253371357917786, -1.354622721672058, -1.5300720930099487, 0.06737634539604187, -0.5926605463027954, 0.664942741394043, 0.7200598120689392, 0.4608656167984009, -0.722378134727478, 0.03960935398936272, 0.7557641863822937, -0.6393021941184998, -0.02192538045346737, 0.8535053133964539, 0.8222246766090393, 0.5774011611938477, 0.594097375869751, -0.2294984757900238, -0.28157344460487366, -0.5844199657440186, 0.7571092247962952, -1.542078971862793, -0.7849512696266174, -0.5688645243644714, 0.14895892143249512, -0.3861408531665802], [0.14556190371513367, -1.5522459745407104, 0.6272284388542175, 0.3038316071033478, -0.5040123462677002, -1.3266621828079224, -1.1936461925506592, 0.43507564067840576, 0.08707421272993088, -0.201783686876297, 0.9751681685447693, -1.239030361175537, 0.3342580497264862, -0.6864611506462097, 0.14174196124076843, 0.7206293940544128, 1.671839714050293, -0.2400919497013092, 1.1437867879867554, -0.565397322177887, 0.8479434847831726, 0.8260723352432251, -0.18246664106845856, -2.550621747970581, 0.7572113871574402, 0.5634809136390686, 0.40344706177711487, -0.5977178812026978, -1.2526264190673828, -0.06196848303079605, -0.7001023292541504, -0.01885915733873844, 0.7088626623153687, 0.16311688721179962, 0.4619099199771881, 0.059421565383672714, 0.8280066251754761, -0.3699873089790344, 0.22331039607524872, -1.8087046146392822, 0.20272967219352722, 0.7950544953346252, -1.7945666313171387, 0.25388291478157043, -1.2261197566986084, -1.4130357503890991, 0.48635029792785645, 1.0221387147903442, -1.7113463878631592, -1.1977686882019043, 0.12723928689956665, -0.09179403632879257, -2.1619772911071777, -1.3922452926635742, 1.1779334545135498, 1.796209692955017, 1.3962833881378174, 0.33463889360427856, 0.09124499559402466, -0.1875132918357849, 0.6915057301521301, -1.8205630779266357, -0.8953593373298645, -0.9870830774307251], [-1.7912204265594482, 0.7943393588066101, 2.5793585777282715, -1.1859923601150513, -0.38419488072395325, 1.190262794494629, 0.030531831085681915, -0.35515138506889343, -0.013088890351355076, 0.709787905216217, 0.3736669719219208, -1.1976351737976074, -1.3785773515701294, 1.4497971534729004, 1.47836172580719, -0.4959498643875122, -0.9633329510688782, -1.3352266550064087, 0.095799520611763, -0.24402070045471191, 1.0518242120742798, 0.9715229272842407, 1.2340278625488281, 1.8268970251083374, 1.0496573448181152, 1.0988192558288574, 2.5477280616760254, -0.032417356967926025, -0.445191353559494, -0.4203985929489136, 0.7641783356666565, -0.546904444694519, 0.12169928103685379, 0.15669377148151398, -0.5694229006767273, -0.1993798315525055, 0.08505305647850037, -0.49946895241737366, 1.3609099388122559, -0.08141729980707169, 1.2704437971115112, -0.4455755054950714, 0.34645816683769226, -1.4196261167526245, 0.09404974430799484, 0.14323946833610535, -1.9432377815246582, 0.5384747385978699, 0.5474854707717896, 1.1707067489624023, -1.8134369850158691, 0.9835683703422546, 1.0104777812957764, 0.20602324604988098, 0.8420546054840088, 0.19048458337783813, 0.10738827288150787, 0.20522040128707886, -1.2629671096801758, 1.759642243385315, -0.5552811026573181, -0.5381370186805725, -0.26408761739730835, 0.4070073366165161], [1.6632206439971924, -0.4050378203392029, 0.06795474886894226, 0.24718011915683746, -0.6223222613334656, -0.06489727646112442, -1.4179291725158691, -0.8969506621360779, -1.0313780307769775, 1.599514126777649, -1.1018779277801514, 0.2538260519504547, -1.2074443101882935, -2.17783260345459, 0.3908974230289459, 1.217604398727417, 0.28005850315093994, -1.216810941696167, -1.071230173110962, -0.7764846086502075, 0.23357081413269043, -1.2252577543258667, 0.7485267519950867, -1.579498291015625, 1.7964791059494019, 2.248345375061035, 0.32041212916374207, 0.17435432970523834, 0.3482007384300232, 0.7408660054206848, 0.10941273719072342, -1.00390625, -0.27022865414619446, 0.03695029392838478, -0.6622953414916992, -1.556100606918335, 0.021572353318333626, 0.15541741251945496, 0.015276236459612846, -0.9252709746360779, 0.2338879257440567, -0.2652001976966858, -1.0464515686035156, 0.7716378569602966, -0.3777323067188263, 1.2762911319732666, -0.002630619565024972, -1.9564958810806274, -0.39097779989242554, 0.5486555099487305, 0.1030166894197464, 1.8016763925552368, 0.779064416885376, -0.33622440695762634, -1.4102281332015991, 0.7495335936546326, -1.2421231269836426, -0.9071198105812073, -1.901584267616272, 1.053515076637268, 0.8057953119277954, 0.1742265224456787, 0.7026642560958862, 0.7088843584060669], [0.45809367299079895, 0.03202611580491066, -0.06322035193443298, -0.09228590875864029, 0.19219782948493958, -1.9914169311523438, 1.2717549800872803, -1.3493129014968872, -0.8854674696922302, 0.978327751159668, 1.1186772584915161, 0.9461169242858887, 1.739111065864563, -0.3437644839286804, 0.6224526762962341, -0.9686833620071411, 1.8637988567352295, -1.3866655826568604, 0.448534220457077, -0.12215514481067657, -0.5393951535224915, 0.7303124666213989, 0.3664223849773407, -0.00554649205878377, 0.7361724972724915, 0.41205883026123047, -0.3327869474887848, -1.2877223491668701, 0.6781179308891296, -1.00048828125, 0.8409332036972046, 0.31804171204566956, 0.27486085891723633, 0.28364434838294983, -1.0755106210708618, -0.3396068513393402, -1.7935055494308472, -0.6183075904846191, -1.8564515113830566, -0.5264477133750916, -0.19957533478736877, -1.0066275596618652, 0.9038496017456055, 0.6574748158454895, -0.6533893942832947, 1.482880711555481, 0.31639987230300903, 0.5412465333938599, 0.8644823431968689, -1.2716206312179565, 0.4665200412273407, -1.2326769828796387, 0.040980421006679535, 0.2047700434923172, -0.255990207195282, 0.3283088505268097, 0.6840417981147766, 0.5780918002128601, -0.04609767720103264, 1.5922315120697021, 0.14106407761573792, -0.2605704069137573, -1.3500274419784546, -0.7242953181266785], [-1.137762188911438, -0.5520994067192078, -1.2693462371826172, 2.196775197982788, -0.5429372191429138, 0.823442280292511, -0.40430203080177307, -0.4943242073059082, -1.5185716152191162, 0.39800825715065, -0.7773585915565491, -0.13057829439640045, 0.3311862349510193, -0.7489627003669739, -0.9885604381561279, -0.1347620189189911, -0.800007700920105, 0.28703296184539795, -0.42410603165626526, 0.6718047857284546, -0.6570162773132324, 0.13511152565479279, -0.34964439272880554, 0.1592528223991394, -0.13129067420959473, 1.0065122842788696, -0.010442156344652176, 0.762054443359375, -0.4647219479084015, 0.6233704090118408, 1.4074854850769043, -0.19409051537513733, 0.1955920159816742, 0.019237784668803215, 0.6535439491271973, -0.621764063835144, 0.0695929005742073, 0.24139831960201263, 0.8299991488456726, -1.1083152294158936, -0.025622760877013206, -0.8143265843391418, 0.39041051268577576, -0.11312668770551682, 0.16382862627506256, -0.36360737681388855, -0.14131256937980652, -0.22055059671401978, 0.21898522973060608, -0.01881878823041916, 2.355931282043457, -1.28329336643219, -0.25485968589782715, 1.3410292863845825, 1.0941941738128662, 0.6620941162109375, 0.20568813383579254, 0.2298150211572647, 1.3075790405273438, 0.8278151154518127, 0.05207006633281708, -0.7198677659034729, 0.2152625471353531, -1.235579013824463], [1.208512544631958, 0.17107966542243958, -0.8582411408424377, -1.969350814819336, 1.699218988418579, 1.0127061605453491, -0.6663439869880676, 1.5052107572555542, 0.34679079055786133, 0.26305896043777466, -0.9256718158721924, 0.3241278827190399, 0.08531693369150162, 0.03627463057637215, 0.42873239517211914, 1.4757627248764038, 0.8784202337265015, -0.9432007670402527, -0.06253333389759064, -0.7181290984153748, -0.4423221945762634, -0.23809084296226501, -0.7520397901535034, 1.2019942998886108, -0.10302076488733292, -0.10768790543079376, -0.39386123418807983, 1.782878041267395, 0.3553878366947174, -0.19687607884407043, -0.3208939731121063, -0.9018557667732239, -0.7209249138832092, -1.2156676054000854, -1.7741771936416626, 0.4248694181442261, -0.31375429034233093, -1.808650255203247, -1.2419289350509644, 0.041756510734558105, -1.9032173156738281, 0.6565813422203064, 2.060171604156494, -1.3617446422576904, -0.8227568864822388, 1.4660123586654663, -1.6972994804382324, 1.4742960929870605, -1.0200881958007812, 0.2573724389076233, -0.41682615876197815, -0.6349557638168335, -0.49762558937072754, -0.5281882286071777, -0.4642338752746582, 0.36358174681663513, -0.9681117534637451, 0.1356075406074524, 0.21259480714797974, 1.0114697217941284, -0.8509888648986816, -0.5029863715171814, 0.490337073802948, -1.600264310836792], [-0.6785191893577576, -0.484483003616333, -0.18071894347667694, 1.6006830930709839, 1.4656461477279663, 1.1690902709960938, -0.9850112795829773, -0.010504527948796749, 0.7825491428375244, 2.188007354736328, 0.4472951889038086, 1.380244255065918, -1.4809775352478027, -0.7891050577163696, 0.5787960886955261, -1.6758382320404053, 0.7613996863365173, 1.660033941268921, 0.737707793712616, -0.14033865928649902, 0.12058885395526886, -0.09800148010253906, 0.9158180356025696, 0.5578685402870178, 0.8039187788963318, 0.6712297797203064, -1.0336401462554932, 0.6569346189498901, -0.22436103224754333, -0.024432217702269554, 0.7627165913581848, 2.401944875717163, -1.6608362197875977, -0.2088821530342102, -1.1446443796157837, -1.3481298685073853, 0.4578217566013336, -2.0157041549682617, -0.42557060718536377, 0.8728009462356567, 1.9171746969223022, -0.5815407037734985, -2.100919723510742, 2.146254062652588, 0.7659295797348022, 0.5409351587295532, -0.86581951379776, -0.627339243888855, 0.17725326120853424, 0.04970165714621544, 0.6092255115509033, 1.7741745710372925, 2.055722236633301, 1.5637335777282715, 0.13711510598659515, 0.47514498233795166, 0.07795016467571259, 0.17238695919513702, -0.3157239258289337, -0.49053600430488586, -1.0167427062988281, 1.1973344087600708, -0.8907076120376587, -0.23187732696533203], [1.78849458694458, 0.5286170244216919, 2.153831958770752, -0.2028522938489914, 0.371969997882843, 1.399657964706421, -0.3951058089733124, -0.8143128156661987, 1.3167496919631958, 1.4740772247314453, -0.9467524886131287, -0.3719835877418518, -0.2966557443141937, -0.4676750600337982, 0.08633686602115631, -2.3338913917541504, -0.10187871009111404, 0.31261441111564636, 1.1334898471832275, 0.2338753342628479, 0.7218273282051086, 0.18718822300434113, -2.1274871826171875, 0.9273198843002319, -1.0254080295562744, -3.1021697521209717, 1.3271945714950562, 0.8435083031654358, 0.774303138256073, 0.2836938500404358, -0.32890817523002625, 0.4888821840286255, -0.7421330809593201, 0.4527372717857361, -0.2964652478694916, 1.3584483861923218, 1.11406672000885, 0.851573646068573, -0.973395824432373, 0.7494581341743469, -1.0347799062728882, -1.0067613124847412, 1.2021607160568237, -0.5505269765853882, 1.0169776678085327, -0.5439204573631287, -0.9481701850891113, -2.2052791118621826, -0.9462576508522034, -1.2788746356964111, 0.8233209252357483, 0.3450676202774048, 0.27525851130485535, 0.9859636425971985, 0.4534914195537567, 1.271579623222351, -1.500072717666626, 0.21937541663646698, -0.394432932138443, 0.3172489106655121, 1.9738998413085938, 1.147812008857727, 1.033703088760376, 2.109755277633667], [-0.6910362243652344, 0.9914242029190063, 0.3943922519683838, 0.30374568700790405, -0.24869586527347565, -0.030741745606064796, 0.319191038608551, -0.5361175537109375, -1.3241008520126343, -0.554552435874939, 0.4499579071998596, 0.006319904699921608, 0.23196102678775787, 1.360737681388855, -0.5839667916297913, 0.3121429681777954, 0.6987394690513611, 0.7968841195106506, -0.5564084649085999, -1.2559653520584106, 1.2976363897323608, 1.1712192296981812, -0.36455291509628296, 1.1159777641296387, 0.2034992128610611, -0.579880952835083, -1.6188632249832153, 0.16465486586093903, 0.14084629714488983, 1.3726915121078491, -1.3752824068069458, -1.4359644651412964, -0.39408138394355774, 0.3811839520931244, -0.822391152381897, 0.06335463374853134, 0.48778796195983887, -0.2924354076385498, 1.8910666704177856, -0.28461936116218567, 0.06741806119680405, 1.1641782522201538, -1.0933929681777954, 1.53180992603302, -0.3462086021900177, 0.12908093631267548, 0.0342324860394001, -1.6586027145385742, -0.12950919568538666, -1.5707803964614868, 0.39749667048454285, -1.0703386068344116, -0.1095590591430664, -0.08599105477333069, 0.07256986200809479, 0.6771249175071716, 1.1501152515411377, -2.5606260299682617, 1.0350242853164673, -0.4049842953681946, 1.0026960372924805, -1.0690069198608398, 0.6689854860305786, 0.20568574965000153], [1.4693902730941772, 0.09312082082033157, -0.023308299481868744, -2.099022388458252, 1.0181487798690796, 1.1857329607009888, -0.22949883341789246, -0.956222653388977, 0.7885907888412476, -0.9914489984512329, 0.4627212882041931, -0.858491837978363, 0.5718389749526978, 0.4396360218524933, -0.8859826326370239, -1.9484593868255615, -1.5776549577713013, -0.9115079045295715, -0.057698581367731094, 0.2651621699333191, 1.1131532192230225, 0.12900526821613312, 0.4861844480037689, -1.1807681322097778, -0.9003213047981262, 0.5576627254486084, -2.1914045810699463, -0.15308938920497894, -1.4976489543914795, 0.008519606664776802, -1.3756622076034546, -1.346401572227478, 0.4428660571575165, -1.8529953956604004, 0.7374233603477478, -0.178688183426857, 1.0874196290969849, 0.4538377523422241, -2.2012453079223633, -0.20856179296970367, -0.2488904595375061, 0.576332688331604, 1.4334352016448975, 0.07795777171850204, -1.5351741313934326, 0.7821755409240723, -0.4067467451095581, 0.4131160378456116, -1.3084031343460083, 0.5976443886756897, 1.6395620107650757, 0.803244411945343, -0.32777202129364014, -0.4193083941936493, 0.9554656147956848, -1.6676628589630127, 0.6488930583000183, 0.5149512887001038, 0.9166991710662842, 0.221248060464859, -1.5044562816619873, -1.2258672714233398, 0.20802846550941467, -1.1731088161468506], [-0.11360839754343033, -0.6060351729393005, 0.2588857412338257, 2.282278537750244, 0.5079193711280823, -0.5968238115310669, -1.3061695098876953, -0.0004046294779982418, 1.0646597146987915, 0.31991687417030334, 1.1926255226135254, 0.7536251544952393, 1.8941216468811035, 0.021562743932008743, -1.2205640077590942, -0.1591833531856537, 1.137821078300476, -1.3611656427383423, 0.4189090132713318, -0.47346511483192444, 0.10744268447160721, 0.6138221621513367, 0.7136813402175903, -0.5881592631340027, -1.4813209772109985, 1.1058429479599, -0.43416959047317505, -0.24075090885162354, -1.802589774131775, -2.3660154342651367, -0.8395485281944275, -0.9598350524902344, 0.5595794916152954, 1.894800066947937, -1.0422722101211548, -1.9909614324569702, -1.9758785963058472, 1.4982911348342896, 2.6657004356384277, 2.715092658996582, 0.15235215425491333, 0.19139251112937927, 0.6929807662963867, 1.9601635932922363, -0.21801680326461792, 0.662395715713501, -1.6324101686477661, 0.29174792766571045, 0.48259955644607544, -0.7493109107017517, 1.3520402908325195, -0.36728426814079285, 0.23086272180080414, 1.4028748273849487, -0.09887168556451797, 0.5900971293449402, 0.2815956771373749, 0.5013480186462402, 0.46905577182769775, 0.25994226336479187, 1.0788623094558716, -0.44610002636909485, -0.38328713178634644, 1.7293479442596436], [1.21406888961792, 0.9996771812438965, 1.082338809967041, -1.3851573467254639, -0.47808122634887695, -0.05356648191809654, 0.4580647945404053, 0.7588040232658386, 2.024083375930786, -1.9232112169265747, 1.4459391832351685, -1.1015043258666992, -0.6137833595275879, 1.2167083024978638, -0.2970190644264221, -0.6556850075721741, -0.2711471617221832, -0.2837112247943878, 1.0231214761734009, -0.903847873210907, -1.4185200929641724, -0.2537962794303894, -1.1610406637191772, 1.2481287717819214, -2.0526092052459717, -0.24278727173805237, 1.3737653493881226, 0.5444096922874451, 0.40204936265945435, -0.7700538635253906, -0.5070391297340393, 2.0626866817474365, -0.684307873249054, 1.403822422027588, -0.6269587874412537, -0.34829390048980713, 0.36677342653274536, -0.6431832313537598, 1.149099349975586, 0.9996809959411621, 1.9808728694915771, -0.26865649223327637, 0.9718517065048218, 1.802628755569458, 0.8368584513664246, 0.26077234745025635, -0.8218713998794556, 0.45764607191085815, 1.070443868637085, -2.1837522983551025, 0.4515138864517212, -0.09453622996807098, 0.42017680406570435, 0.9069626927375793, 0.4558734595775604, 0.4114575982093811, 1.4315624237060547, 2.1072285175323486, -0.013048524968326092, -0.3056618571281433, -0.7067250609397888, 0.1488867700099945, -1.1525851488113403, -2.630587339401245], [0.08845161646604538, -1.3119702339172363, -1.2637336254119873, 0.9064241647720337, -0.32526451349258423, 0.9956566095352173, 1.1849331855773926, -1.5465167760849, 1.6943309307098389, 0.5342502593994141, 0.29942944645881653, -0.18067380785942078, 0.552635669708252, -0.1705581694841385, 0.8606722950935364, -0.6338952779769897, -1.359519362449646, -0.40449729561805725, 0.8524457812309265, 0.03766373544931412, -0.7002695202827454, -0.2087348997592926, 1.1003508567810059, -0.8211223483085632, 0.5193884968757629, 2.2505991458892822, 0.2842423617839813, 0.4800310730934143, 0.9453125596046448, 1.0583456754684448, 0.18029700219631195, -0.5521008372306824, 0.3961915373802185, 1.3244177103042603, 1.8319929838180542, -1.4770095348358154, 0.31977295875549316, 0.800865113735199, 0.21931831538677216, -0.43405428528785706, -0.5923031568527222, -0.5676142573356628, 2.1315181255340576, 0.6623985171318054, 0.1922559291124344, 0.7227588295936584, 0.6992391347885132, 0.0368785560131073, 0.7627658247947693, -0.3553502559661865, -0.9509709477424622, 0.2593074440956116, 1.6709058284759521, 1.3587877750396729, 1.1971986293792725, -0.8956770896911621, -1.271821141242981, 0.00047905536484904587, 0.546886146068573, -0.539254903793335, 0.8021630048751831, 0.6823890209197998, 0.10102176666259766, -0.9469990134239197], [0.21094007790088654, -0.6745872497558594, 0.2953798174858093, -0.5516225099563599, 1.6101202964782715, -0.7540399432182312, 1.3343119621276855, -0.3738377094268799, 0.597844123840332, -0.9456911087036133, 0.45280951261520386, 0.08659198135137558, 0.1825137436389923, 1.7381138801574707, -1.070216417312622, 0.8466307520866394, -0.7297548055648804, -0.2918497920036316, -0.4663641154766083, -2.680997133255005, -2.1078689098358154, 1.86572265625, 0.3894398808479309, 1.225171446800232, 0.16160473227500916, 1.192512035369873, 2.4144601821899414, -0.493592232465744, -1.3770496845245361, 0.4165869951248169, -0.2300490289926529, 0.485688716173172, -1.4923477172851562, -1.7011927366256714, 0.4348120391368866, -0.8349406123161316, -0.36536845564842224, -0.6820513606071472, -0.13278241455554962, -1.7175474166870117, -0.1797846257686615, -1.7530384063720703, -1.2295886278152466, 0.03282623738050461, -0.8292782306671143, -1.0179001092910767, -0.745778501033783, -0.26048484444618225, 0.2377266138792038, -0.9341781139373779, 0.7764912843704224, -1.4190860986709595, -0.10933225601911545, -0.2809271812438965, 0.40310198068618774, -1.1980364322662354, 0.6493783593177795, -0.08930187672376633, -1.1864248514175415, 0.5370813012123108, 0.6787540316581726, 0.043724898248910904, -1.4182606935501099, -0.1782434582710266], [0.1533864438533783, 1.3670501708984375, -0.45198744535446167, 1.1843101978302002, 0.1895812749862671, 0.13224323093891144, 0.06408876180648804, -0.22557030618190765, -1.084242582321167, -1.2138980627059937, 1.671865463256836, -0.6809478402137756, -0.7983133792877197, 1.2590394020080566, -2.205360174179077, 0.88636314868927, 0.24927249550819397, 0.8490355014801025, -0.4301389157772064, -1.4849857091903687, 0.03659965842962265, -2.2174623012542725, -0.7116193175315857, -0.8900171518325806, 0.22432686388492584, -0.8923438787460327, -1.3225480318069458, 1.5005782842636108, 1.8877309560775757, -0.9463830590248108, 0.8300773501396179, 1.2957894802093506, 0.3164558410644531, -0.1483244150876999, 0.13767507672309875, 2.313549518585205, 1.0457669496536255, 1.3008793592453003, -1.2994734048843384, 1.2404454946517944, 0.2679113745689392, -0.7412367463111877, 0.7502023577690125, 0.3614591062068939, 0.0718671977519989, 0.40060824155807495, -0.8950742483139038, -1.5145162343978882, 0.962486982345581, 0.21966171264648438, 1.8367401361465454, -0.7705519795417786, -1.6800075769424438, 0.7084203362464905, 0.43896421790122986, 1.3822706937789917, -1.9685149192810059, -0.5169795751571655, -0.4661734104156494, -0.6078326106071472, 0.8064115643501282, -0.05040554702281952, -0.47029176354408264, 1.6346702575683594], [-0.7894842028617859, -2.5513548851013184, -0.29009005427360535, -0.13530614972114563, -0.47441959381103516, 0.18597055971622467, 0.873529851436615, 0.6870267987251282, 1.7149548530578613, -0.9399374723434448, 0.3116501271724701, 2.0308728218078613, -0.015433840453624725, -1.7996978759765625, 0.6526739597320557, -0.6618031859397888, 1.8716830015182495, 0.2730744779109955, -0.1265881359577179, -2.6319942474365234, -0.007875146344304085, -0.933347225189209, 0.6785823106765747, -1.2264679670333862, 1.3177597522735596, 0.8625108599662781, -1.373356580734253, -0.539392352104187, -2.0358822345733643, 0.9725825786590576, -0.13325512409210205, 1.2929332256317139, -0.28862908482551575, -0.0041677989065647125, 0.48618024587631226, 0.5056095719337463, -0.6123315095901489, 1.4598720073699951, -0.09849850833415985, -2.872709035873413, 0.3050166964530945, 0.6055032014846802, 0.8815659284591675, 0.42794540524482727, -1.7618809938430786, 0.15878859162330627, -0.5599163174629211, -1.514197587966919, 0.38017094135284424, -0.49366244673728943, 1.7410986423492432, 0.13779990375041962, -0.22509628534317017, -1.5719555616378784, 0.30920836329460144, -0.5541496872901917, -0.139642596244812, 2.009932041168213, -1.6805955171585083, 1.243468165397644, 0.7034885883331299, -0.2137477695941925, 0.9105415344238281, -0.40244969725608826], [0.07872552424669266, -0.13777019083499908, 1.8073393106460571, -1.3531122207641602, 2.0788583755493164, -0.011924861930310726, 0.8260843753814697, -0.776103138923645, -0.22746196389198303, 0.45683011412620544, -0.90675950050354, 0.8940785527229309, 0.9364124536514282, 1.2894166707992554, -0.5454146265983582, 0.11797907948493958, 0.4421754479408264, 0.23956435918807983, -0.38378942012786865, -0.991650402545929, -0.41894078254699707, 0.5069056153297424, 1.5568312406539917, 1.8757010698318481, -1.1820100545883179, 1.0754008293151855, -0.046065784990787506, 1.0022796392440796, -0.5882329940795898, -1.6775492429733276, 1.0313513278961182, 2.242784023284912, 0.6074286103248596, -0.5278621315956116, -0.43135568499565125, 0.5807077884674072, -0.5100691914558411, 0.31817013025283813, -0.2350825071334839, 0.3870628774166107, -0.5950922966003418, -0.6511870622634888, -1.354528784751892, -1.3818281888961792, 0.4646327793598175, -1.0505985021591187, 0.1348564773797989, 1.1323446035385132, 0.7998069524765015, -1.4661582708358765, -1.3047065734863281, -0.9271105527877808, -1.8703850507736206, 2.891512632369995, -1.4244041442871094, 0.37287744879722595, 0.13046003878116608, 1.2478646039962769, 0.438775897026062, -1.0463510751724243, 2.6608357429504395, -1.527247428894043, 1.349392294883728, 0.3798043429851532], [0.5244458317756653, 1.8318017721176147, 0.37595030665397644, -0.5563600063323975, 0.030114764347672462, -1.92524254322052, -2.110109567642212, 1.0763535499572754, -0.042867906391620636, 0.9816819429397583, -0.35401904582977295, 1.6117278337478638, -1.1611424684524536, 0.00030580878956243396, -1.6485148668289185, -0.39043495059013367, 1.1320672035217285, -1.1042670011520386, 0.6288154721260071, -2.252746105194092, 0.6968732476234436, 0.9403111934661865, -0.0442747138440609, -0.7613812685012817, -1.3608845472335815, -0.20545032620429993, 0.6762239336967468, -0.7275702953338623, -0.1842406690120697, 0.0037633152678608894, -0.2021908015012741, -0.5362264513969421, -0.04056738317012787, 1.7293328046798706, -0.5721859335899353, -1.7161328792572021, 0.49211448431015015, -0.10746250301599503, 2.683748245239258, 0.7262082695960999, -0.9406684041023254, -0.657860517501831, -0.12512412667274475, -0.38901832699775696, -1.598462462425232, 0.4341246485710144, -1.0891118049621582, 0.9291698932647705, 0.3341943621635437, 0.22799615561962128, -0.6065936088562012, -0.561941921710968, 0.3167010545730591, -1.2343460321426392, 0.8667526841163635, 0.9275113344192505, 0.25725334882736206, -0.6824886202812195, -0.8793138861656189, 0.1131804957985878, 0.6298519372940063, -2.0393996238708496, 1.4407331943511963, -0.6530267596244812], [0.3314548134803772, 2.373796224594116, 1.6491706371307373, 0.6197987198829651, 1.0548739433288574, -0.2402344048023224, 0.8312934041023254, -0.46816566586494446, -0.9590691328048706, -1.3516976833343506, 0.6403603553771973, -1.9812381267547607, 1.640859603881836, -0.40488505363464355, -0.5686401128768921, 0.42138153314590454, -1.0440847873687744, 1.5898311138153076, -0.9417543411254883, 0.03468494862318039, -1.2980314493179321, -0.13361209630966187, -1.6842079162597656, 1.3028501272201538, 0.5842384099960327, -0.3058842420578003, -0.6498005390167236, -0.14800260961055756, 0.43189889192581177, 1.5190949440002441, 0.7447288036346436, 1.8947991132736206, 0.6005774736404419, -1.173545479774475, 0.37430328130722046, -0.9122443795204163, 1.1598807573318481, 0.004156582523137331, 2.9106199741363525, 0.28512412309646606, 1.222949504852295, -0.10202310979366302, 0.35335320234298706, 0.02058839052915573, 0.6460615992546082, 0.9768327474594116, 0.39678052067756653, -0.37594982981681824, -0.6310237050056458, 1.6108936071395874, -0.39219799637794495, -0.13477221131324768, 1.3850852251052856, 0.6357476711273193, 1.7617144584655762, -1.4174065589904785, -1.7820303440093994, -2.1372642517089844, -0.7750781178474426, -1.4287195205688477, 0.05669095739722252, -2.4398272037506104, 1.9266468286514282, 0.17954182624816895], [1.8348820209503174, -0.5272670984268188, 1.167331576347351, 0.4698180854320526, -0.9329659342765808, 0.22684723138809204, 3.227673053741455, 0.22608590126037598, -0.7662935256958008, 1.2105833292007446, 1.1721177101135254, 0.1723986566066742, 0.8710218071937561, -2.534088134765625, -0.8672894239425659, 0.5202491879463196, -0.6819640398025513, 0.15272186696529388, -1.043548822402954, 0.11788321286439896, -0.6764343976974487, -1.8356119394302368, 0.15005232393741608, -0.4111861288547516, -1.0014740228652954, 2.1474034786224365, 2.3534233570098877, 0.9633916616439819, 0.001973716774955392, -0.05503139644861221, 0.4143402576446533, -2.9998703002929688, -0.2656787633895874, 0.5129998326301575, -1.651969313621521, -0.2383514791727066, 1.1527349948883057, -0.5805224776268005, -0.522077739238739, -0.6798012852668762, -0.11154262721538544, 0.5566501617431641, -0.6270144581794739, -0.2573646903038025, -1.2067480087280273, -0.6935625076293945, -1.3644756078720093, -0.6248882412910461, 0.844649076461792, -0.40676960349082947, 0.21287117898464203, 0.7435085773468018, 0.9967423677444458, 1.8002640008926392, 0.06615482270717621, -0.575680136680603, 0.5962603688240051, 0.548090934753418, -0.31415843963623047, 0.35417985916137695, -2.3883063793182373, 0.24992412328720093, -0.3306182324886322, -0.8794111609458923], [0.900567352771759, -0.7452062368392944, -2.5727295875549316, -0.9455052018165588, 0.06385538727045059, 0.20871545374393463, 2.0829672813415527, -1.22978675365448, 1.4340020418167114, 0.7537766098976135, 0.7432114481925964, 0.94569331407547, -0.44882985949516296, -0.05056945979595184, 0.364165335893631, 0.6925076246261597, 0.1678038239479065, -1.0920374393463135, 0.6615155935287476, -1.3702954053878784, 0.4207763075828552, 0.4674346446990967, 0.4437185823917389, 0.21628119051456451, -0.03975265100598335, -2.0818212032318115, -0.16686029732227325, -1.5827701091766357, 0.4977903664112091, -0.6268079876899719, 0.16611741483211517, 0.13655908405780792, -1.3027886152267456, 0.5285088419914246, 0.13889959454536438, 1.1024090051651, -0.6114252209663391, -0.31500357389450073, 0.22463704645633698, -1.3201515674591064, 0.550633430480957, -2.5030713081359863, 0.6842885613441467, -1.3866477012634277, 0.03956351801753044, 0.3196249306201935, -0.029494265094399452, 0.5251003503799438, 0.35121357440948486, 0.3819564878940582, 2.3808066844940186, 0.9603602886199951, 1.2440942525863647, -0.43058255314826965, -0.21690180897712708, 2.4868626594543457, -1.1772443056106567, 0.12907211482524872, -0.9084253311157227, -0.40475326776504517, 0.35558876395225525, 1.871131181716919, -0.35970523953437805, 1.70461106300354], [-0.6972724199295044, 1.328453779220581, -2.004608154296875, 0.20968562364578247, 0.31008851528167725, -1.4945958852767944, 0.299256294965744, -1.1442333459854126, 1.415838599205017, 0.4539320170879364, -0.1888885498046875, -0.3053142726421356, -0.3727565109729767, -1.5090115070343018, 0.08292923122644424, -0.8881947994232178, 0.36503905057907104, 1.9745278358459473, -0.030468307435512543, 0.930772602558136, 0.017344431951642036, -0.909421443939209, -0.7933349609375, -1.903933048248291, 2.7972445487976074, 0.8628860712051392, 0.7816857099533081, 1.8919754028320312, 0.9330288171768188, 0.07804851233959198, -1.2369835376739502, -0.30936968326568604, 0.5709415674209595, -0.03067030943930149, 0.3202615976333618, -0.03556863218545914, -0.6831537485122681, 1.3131548166275024, -0.6794283390045166, -1.8648895025253296, -0.12964244186878204, -1.3018531799316406, -0.7702664732933044, 0.2768080532550812, -0.022456111386418343, -1.3400453329086304, 0.6752222180366516, -1.84583580493927, 0.21043172478675842, -0.34310558438301086, 0.21180006861686707, -0.7345698475837708, 0.6073403358459473, 0.615456759929657, -0.6054530143737793, -0.7368713021278381, 0.10717548429965973, 0.04342811927199364, 0.643201470375061, -0.34336185455322266, 0.010373996570706367, 0.15871959924697876, -1.0801559686660767, -0.7653635144233704], [-0.7412828207015991, 0.7353923320770264, 1.5849952697753906, -0.22320938110351562, 1.0462819337844849, 1.1940712928771973, 0.495274156332016, -0.6631202697753906, 1.39368736743927, -0.2729538083076477, -0.06188704073429108, -0.5550811290740967, 0.5133947730064392, 1.0935910940170288, -0.2779385447502136, 0.7042689919471741, -0.4146726131439209, -1.349276065826416, 0.007460531312972307, -0.47879788279533386, 1.6716035604476929, -1.0227285623550415, -0.51251220703125, -0.15818727016448975, 2.142302989959717, -1.4514235258102417, 0.46403202414512634, 1.6059232950210571, -1.5809986591339111, 1.0909727811813354, -1.3801394701004028, 0.13834983110427856, -0.09776432067155838, -1.1682567596435547, 0.7008893489837646, 0.05991914123296738, -0.8617501258850098, -1.824467420578003, 1.0005762577056885, -0.4404209554195404, 2.140254259109497, -0.3573527932167053, -0.12137285619974136, 0.30321478843688965, 1.0037325620651245, 0.5421083569526672, 0.4971645772457123, -0.38905099034309387, -1.2784572839736938, -0.11108149588108063, 0.9061456322669983, 0.5265390872955322, -0.40352240204811096, -0.9889832139015198, -1.345011591911316, -0.32373157143592834, -0.8658010363578796, 0.14699149131774902, -0.06069043278694153, 0.7519211173057556, -1.0894076824188232, 2.075739622116089, 0.5198187828063965, -0.6062445640563965], [-0.6820645332336426, -0.4851994216442108, -2.0586142539978027, -0.7303698062896729, -1.541262149810791, 0.29276591539382935, 0.2647480368614197, -0.5050867795944214, -0.2750564515590668, -2.51092529296875, -0.7294883131980896, -0.6729379892349243, -0.5904134511947632, 1.726048469543457, -0.7385344505310059, -1.0107042789459229, -1.403633952140808, 0.1920205056667328, -0.4799098074436188, -1.3889951705932617, 1.0620745420455933, -0.018879549577832222, -0.34399011731147766, -0.46552643179893494, 0.8017386198043823, 1.1039011478424072, -1.5565696954727173, -0.41429680585861206, 0.8500890135765076, -0.33600661158561707, -0.18972817063331604, 0.08430882543325424, 2.060150623321533, 0.5045018792152405, -0.8154760599136353, -0.31289395689964294, -1.3557504415512085, 1.5420371294021606, -1.229150414466858, -2.141963481903076, 1.5160030126571655, -0.11891079694032669, 0.7006597518920898, -0.7995149493217468, 1.1402331590652466, -2.3826675415039062, -1.1081582307815552, -0.9341670274734497, 0.7806623578071594, 1.0711917877197266, -2.772674083709717, -0.28581124544143677, 1.3459290266036987, -0.7826802730560303, 0.6384267210960388, -1.1330897808074951, -2.216343879699707, 0.828267514705658, 0.04740840941667557, 1.0922292470932007, -0.8362880349159241, -0.6376180648803711, -0.18966960906982422, -0.4268895387649536], [0.40777722001075745, -1.3747129440307617, 0.13140985369682312, -0.5037581324577332, 0.6042238473892212, 1.631529688835144, 0.6775302886962891, 0.08152070641517639, -1.3369485139846802, 2.324960470199585, -1.1139146089553833, -0.21975715458393097, 0.43744775652885437, -1.3117196559906006, 1.0666615962982178, 0.4281163215637207, 0.10674309730529785, -1.340316653251648, 0.900012195110321, 0.3483213186264038, 0.1023164838552475, 0.09718959033489227, -0.8254855275154114, -1.6531294584274292, -0.7733998894691467, 1.296478033065796, -0.27956992387771606, 1.1769278049468994, -0.6552348136901855, -1.1047059297561646, -0.3861304819583893, -0.032570283859968185, 0.5062909722328186, -0.8406297564506531, -0.6829375624656677, 1.1807951927185059, 0.43593043088912964, 1.6776117086410522, -0.19897852838039398, 0.1329267919063568, -0.7538896799087524, -0.7652217149734497, 1.2024400234222412, 0.9054014682769775, -0.8856411576271057, -1.2112644910812378, -1.54714035987854, -0.6632801294326782, 0.6297401189804077, 0.3300905227661133, 0.7286788821220398, -2.39072322845459, 1.1671322584152222, -1.8446604013442993, -1.0423598289489746, -0.3248705267906189, 1.109157919883728, -1.8632382154464722, 1.0471646785736084, 0.5633671283721924, -0.007540245074778795, 0.8590568900108337, -0.8070321679115295, -0.1377810537815094], [-0.15140752494335175, -1.132165551185608, -0.4599606692790985, 0.603018045425415, 0.5847792029380798, 0.24340607225894928, 0.25117602944374084, 0.8745695948600769, -0.9697771668434143, 1.888676404953003, -0.5531130433082581, 0.8685763478279114, 0.4973510205745697, -1.3788431882858276, -0.5265825390815735, -0.3269556164741516, -2.090344190597534, -0.038248684257268906, -0.3769308924674988, -0.13655103743076324, -1.25080406665802, -0.11944444477558136, -1.1391990184783936, 0.8565539121627808, 0.8105648756027222, 0.6076843738555908, -0.21173804998397827, 1.198228120803833, -0.5109068751335144, -0.6304873824119568, 0.892949104309082, -0.21226182579994202, -0.3858024775981903, 2.0836973190307617, -0.6635866761207581, 0.37767454981803894, -1.8345626592636108, -0.8723955750465393, -0.2703433334827423, -1.2638353109359741, -0.7970742583274841, -0.8030523657798767, 0.20687039196491241, -0.12349788099527359, 0.2747874855995178, -0.13474707305431366, 0.4293581545352936, 0.4527294933795929, -0.8965150117874146, 0.15385909378528595, 1.5636022090911865, 0.9412620067596436, -0.01874176599085331, 0.15235887467861176, 1.133984923362732, 1.1936334371566772, -1.5423426628112793, -0.7190322875976562, 0.535083532333374, -0.6231642961502075, 0.7269845008850098, -0.17933796346187592, 0.14111272990703583, 0.9289544224739075], [0.23223231732845306, -1.1474106311798096, -0.3456176519393921, 1.5322771072387695, 0.4726394712924957, -0.7133371829986572, 1.0505456924438477, -0.10468132793903351, 0.11876041442155838, -0.3817889392375946, 0.15754710137844086, 0.95174640417099, 0.2646694481372833, -0.9775293469429016, 0.3159225583076477, -1.5988436937332153, 0.2818056643009186, 0.6352058053016663, 0.34029245376586914, 0.1934162974357605, 0.6086205244064331, -0.8999294638633728, -0.765591561794281, 0.512759268283844, 0.8439558744430542, -0.7149333357810974, 0.3250599205493927, -0.9860570430755615, -0.6044340133666992, -0.04673227295279503, -0.4033932387828827, 0.205492302775383, -0.7370491027832031, -1.8205549716949463, -0.7029755711555481, 0.7253630757331848, -0.01740054041147232, -1.0884040594100952, 0.6336804032325745, 1.4190064668655396, 0.1698879450559616, 0.9609441757202148, 0.6706419587135315, -1.2704603672027588, 0.7086185216903687, -0.18843723833560944, 0.10055533796548843, -0.7229260802268982, -0.03326961770653725, -0.15087519586086273, -0.22819820046424866, -0.7768595218658447, 2.4683077335357666, 0.8763196468353271, 0.6937940120697021, 1.66983962059021, 0.6477465033531189, 1.2618197202682495, -0.3542095422744751, -1.2614963054656982, 2.1969170570373535, -0.36977657675743103, -0.7881441116333008, -0.0045058331452310085], [1.0167194604873657, 0.7897876501083374, -0.27731290459632874, -0.5307894945144653, 1.1620885133743286, 0.8974130153656006, 0.42549073696136475, 0.2831207513809204, 1.112316608428955, -1.4652498960494995, -0.06870873272418976, -0.34956997632980347, 0.30426865816116333, 0.4166966676712036, 0.22602620720863342, 0.5967501401901245, 0.4310896098613739, -1.131056308746338, -0.6756595969200134, 0.13941365480422974, 1.5190218687057495, 0.6121548414230347, -0.6777894496917725, 0.3849066495895386, 1.1259701251983643, 0.5394887328147888, 0.34939107298851013, 0.055355437099933624, 1.7937467098236084, -0.8919561505317688, 0.33847907185554504, -0.04024901241064072, -2.0477843284606934, 1.1311815977096558, 0.47187307476997375, -0.3408844470977783, 0.9782889485359192, -1.0927317142486572, -0.3003920316696167, 0.1546320915222168, -1.673296332359314, -0.5966504812240601, 1.3435245752334595, 0.4647256135940552, 1.1975436210632324, -1.3024282455444336, 0.21178267896175385, -0.7533735632896423, 1.106971263885498, -0.1319262981414795, -1.1141695976257324, -1.5008466243743896, 0.8438469171524048, 0.016496604308485985, 0.517204225063324, 1.2330793142318726, -0.9342157244682312, 1.3816825151443481, -0.8709374070167542, -0.28200411796569824, 0.17278671264648438, 0.870045006275177, -0.2463669329881668, 1.1353601217269897], [-0.1475326269865036, 0.8935924172401428, -0.16839417815208435, 1.615190863609314, -0.47753778100013733, 0.25281697511672974, 1.2258721590042114, 0.9019585847854614, 0.6934176087379456, -0.3445132374763489, 0.4153152406215668, -1.551486849784851, 0.48257017135620117, -0.28665387630462646, 0.19944517314434052, 0.8013145923614502, -0.6012262105941772, -0.4553661346435547, 0.17767184972763062, -1.8833461999893188, 0.948172926902771, 0.325114369392395, 1.1792182922363281, -1.9943519830703735, 0.7359018921852112, 0.5775824785232544, 2.2490901947021484, 0.25907474756240845, -0.3998930752277374, 1.0094753503799438, 0.8238102197647095, 1.0103962421417236, 0.6814818978309631, -0.08000122010707855, 0.5311240553855896, 0.7497687935829163, 1.3488898277282715, 0.08062981814146042, 0.11199954152107239, -0.12122496962547302, -0.14714562892913818, 1.374537467956543, 0.07672888040542603, -0.22704628109931946, 1.0558643341064453, -0.22059781849384308, 1.2060859203338623, -0.4261820912361145, 1.533342719078064, 0.007323481608182192, 0.36578384041786194, 0.057508811354637146, -0.061661042273044586, 0.8501505255699158, -0.9145338535308838, -1.479325532913208, 0.24310791492462158, 0.7563761472702026, 1.28165864944458, -1.040590524673462, -0.03835388273000717, 1.128753900527954, 1.5541586875915527, -1.7179292440414429], [-2.2152183055877686, 0.5560832023620605, -0.3996051549911499, 1.0891557931900024, -0.44190704822540283, 0.27210286259651184, -0.35614103078842163, 0.24348731338977814, -0.6203408241271973, 0.5685017108917236, -0.44602257013320923, -1.0914592742919922, 0.7240308523178101, 0.52913898229599, 0.14725394546985626, 1.1190316677093506, 1.7349854707717896, -0.24255430698394775, 0.5233017206192017, -0.9142062067985535, 0.45859232544898987, 0.5207625031471252, 1.8131909370422363, -0.5827947854995728, 0.02908191829919815, -0.8514741063117981, -0.38353174924850464, 0.3352326452732086, -0.6463900804519653, 1.5684949159622192, 1.4574942588806152, -1.9627037048339844, -0.9093319773674011, -1.3159843683242798, -1.276706337928772, -1.1613742113113403, -0.6741882562637329, 1.0950498580932617, -0.17437413334846497, -1.0491137504577637, 1.5347187519073486, 0.9980862140655518, 0.3653809428215027, -0.4105541706085205, 0.8827275037765503, -0.41072073578834534, 0.8030648231506348, -0.37156352400779724, -0.3152242600917816, 0.2516016960144043, 1.7630095481872559, -0.008219256065785885, -1.5050874948501587, -0.3092882037162781, 0.49056005477905273, 0.698375403881073, 0.7975420951843262, 0.043744031339883804, 0.1719626486301422, -1.260272741317749, -1.015129804611206, -1.3497111797332764, -0.029316304251551628, 0.6612155437469482], [0.5748038291931152, 0.596823513507843, -0.5578446388244629, 0.02107478305697441, -1.9574775695800781, -0.08937430381774902, -1.108526587486267, 0.19560402631759644, -0.4714047610759735, 0.4697507619857788, -0.10644084215164185, -0.3936157524585724, -0.3666359782218933, 0.01931084506213665, 0.8881840109825134, -0.6319223046302795, 0.7526683807373047, 2.1509110927581787, 0.8464654088020325, -0.7889106869697571, 1.2985916137695312, 1.5286986827850342, 0.3207158148288727, 1.7394752502441406, 1.4308806657791138, 0.041875023394823074, 0.38643619418144226, 2.586458921432495, -1.5239958763122559, -1.0395665168762207, 1.1466552019119263, 0.09768473356962204, -1.1487314701080322, -0.26066896319389343, -1.8132147789001465, -1.0588537454605103, -0.6245258450508118, 1.1206930875778198, 1.6315995454788208, 1.7401484251022339, -0.10586702823638916, -0.6342111229896545, -1.126312255859375, 0.36294496059417725, -0.13433080911636353, -0.7873978614807129, -1.1901785135269165, 0.8075748682022095, -0.3988381326198578, -0.20963558554649353, 0.9445657134056091, 0.4653874635696411, -1.229933261871338, -2.1906487941741943, 0.6417255401611328, 0.9443941712379456, -0.05284415930509567, 0.5232030749320984, 0.3633210062980652, 0.6192106008529663, -2.0008928775787354, 0.4085187017917633, -0.8253006339073181, -1.2777832746505737], [0.4807579815387726, -0.8288476467132568, 2.1056246757507324, 0.5269430875778198, 0.89896559715271, 0.5073306560516357, 0.41456136107444763, -1.5658756494522095, -0.4868578612804413, -1.2675031423568726, 1.0046836137771606, 1.0418726205825806, 0.4585515558719635, 0.0023944193962961435, 0.9734357595443726, -0.2924944758415222, -0.35763657093048096, -1.0539593696594238, -1.0335662364959717, -0.5941059589385986, 0.6991885900497437, -0.17540431022644043, -1.8428069353103638, 0.04566135257482529, -1.6785635948181152, 1.0573400259017944, 0.21125271916389465, 0.37356236577033997, 0.820889949798584, 0.21927964687347412, 0.45414841175079346, -1.9803531169891357, 1.6448073387145996, 0.7075308561325073, 0.2105606198310852, 0.40443283319473267, 0.020698046311736107, 0.1999650001525879, 0.24269412457942963, -1.6510690450668335, 0.15111210942268372, -0.5269127488136292, -0.20395894348621368, -0.40767747163772583, 0.3367076516151428, -1.447131633758545, 1.3176010847091675, 1.1807690858840942, 0.3736834228038788, 1.4258756637573242, 0.30364516377449036, 0.6470872163772583, -0.5101912617683411, 1.8809003829956055, 0.05982036888599396, -0.8937757015228271, 0.1965920478105545, 0.20919808745384216, 0.044491980224847794, 1.1374539136886597, 0.658147931098938, 1.4453320503234863, 2.381047010421753, 0.2771495282649994], [-1.2595587968826294, 1.6966464519500732, 0.35285815596580505, 0.6182202696800232, -0.9807801842689514, 2.0051963329315186, -0.4724925756454468, -0.798613429069519, 0.03615392744541168, 0.21322797238826752, 0.18602101504802704, -0.32567137479782104, 0.07212546467781067, 1.0652167797088623, -0.9818449020385742, -0.24051667749881744, 0.555370032787323, 0.27080038189888, -0.5618364214897156, 1.0384392738342285, 0.7626233100891113, 1.1851826906204224, 1.8112013339996338, 1.5726529359817505, 0.009796016849577427, 0.5085613131523132, -0.9100888967514038, -0.738132119178772, -1.4852244853973389, -0.41744330525398254, 0.5596503615379333, 0.6665238738059998, 0.3004603087902069, 1.494968295097351, 0.4626661241054535, 1.8794434070587158, -0.6105385422706604, 1.544455885887146, 0.7709699273109436, -0.9621633887290955, 0.5719617605209351, 1.124305248260498, -0.0637257993221283, 1.4594025611877441, 1.3577390909194946, 0.3244532346725464, 0.004207148682326078, 0.7164549231529236, -0.8903455138206482, 0.013590196147561073, -1.8001457452774048, 0.8323220610618591, 0.97849041223526, -0.3556593060493469, -0.419902503490448, 0.06959595531225204, 1.2438145875930786, 0.97166508436203, -1.1404943466186523, 0.15253981947898865, 1.2889724969863892, 0.17479579150676727, 0.33615952730178833, 0.5831484198570251]]]], "dtype": "float32"}, {"__nd__": [[[[-1.135108470916748, -0.11047404259443283, 0.039571069180965424, -1.0444952249526978, 0.7928351163864136, -0.29697129130363464, -0.4657374918460846, 0.5214921832084656, 0.3087640404701233, 1.3334218263626099, 1.00160551071167, 0.39738887548446655, -0.7666030526161194, -0.633163332939148, 1.2968380451202393, 0.3171042203903198, 0.4386861324310303, -0.15877199172973633, 0.4301871955394745, 1.9636415243148804, 0.740665853023529, 0.4985436201095581, -0.2837604582309723, -0.23915943503379822, 1.9973543882369995, -0.09204424917697906, -1.364961862564087, 1.07706880569458, -0.4259079098701477, -0.08986043930053711, 1.378446102142334, -0.057759299874305725, -1.0564285516738892, -0.904555082321167, 0.5352736711502075, -1.0455783605575562, -2.192622661590576, 1.1032752990722656, -0.2546484172344208, 0.34414759278297424, -0.664306104183197, -1.6341677904129028, -1.0398391485214233, -1.8383361101150513, 0.3782438039779663, -0.2164543867111206, 1.1953362226486206, 0.02482319436967373, 0.4860151708126068, -0.03355832397937775, 1.3376137018203735, 1.2175692319869995, -1.2967315912246704, 0.14000511169433594, 1.6496903896331787, -0.649057924747467, -0.7065867185592651, 0.029335740953683853, 1.1203980445861816, -1.2028101682662964, 0.11524870246648788, 0.25540924072265625, -0.8500273823738098, -2.336402177810669], [0.1378673017024994, -1.1316694021224976, -1.3475492000579834, -0.17031462490558624, -1.6797820329666138, -0.3067282736301422, 0.5611713528633118, -1.496181845664978, 0.9094937443733215, 0.14129605889320374, 1.0237030982971191, -1.175337791442871, 1.641861915588379, 1.9430879354476929, 0.9495202898979187, -1.1219416856765747, 0.7985203862190247, 2.6856040954589844, -1.3171614408493042, 0.01329591404646635, 0.16668055951595306, 2.452214002609253, -0.11675703525543213, 0.5238156914710999, 0.3029778003692627, -1.4337692260742188, 1.108322262763977, 1.3965444564819336, 0.2268497347831726, 1.2432101964950562, -0.23280012607574463, -0.5222916603088379, -0.5626459121704102, 1.3422988653182983, 0.47446250915527344, -0.07779815793037415, 0.3606436252593994, 2.080918073654175, 0.1629985272884369, -0.4533369839191437, -0.7496532797813416, -1.3697806596755981, -1.662973165512085, 1.6354883909225464, -1.78280770778656, 0.4374949336051941, 0.5562154054641724, -0.5106225609779358, 0.9942342638969421, 0.2436365932226181, 1.1604384183883667, -0.3760474920272827, -1.9740071296691895, -0.528336763381958, 0.057022955268621445, -1.1123096942901611, 1.9695396423339844, 0.6025668978691101, 0.08649658411741257, -1.6670171022415161, 1.7995468378067017, -1.0541939735412598, -0.3599366247653961, 0.299031525850296], [1.7608333826065063, 0.6115601062774658, -1.3690578937530518, -0.5782014727592468, 1.0100325345993042, -0.991909384727478, 0.165072962641716, 0.3341846168041229, 0.4652223289012909, -0.7460474371910095, -0.34645646810531616, -1.0574264526367188, -0.453937828540802, -1.0069735050201416, -0.6743786334991455, -0.6490890383720398, -2.280613899230957, 0.45185554027557373, -2.0569875240325928, -1.22243332862854, 1.3147573471069336, 1.339841604232788, -1.8728737831115723, -0.6780422329902649, -0.17854858934879303, 0.3690129816532135, 0.7897945046424866, 0.3263530433177948, 1.3043797016143799, 0.4177861511707306, -2.6102242469787598, -1.336350679397583, -1.2969627380371094, 0.271885484457016, 1.0626009702682495, -0.2778577506542206, 0.7093196511268616, 1.9293040037155151, 0.32394686341285706, -0.6148298382759094, 1.7164143323898315, 0.22517725825309753, 0.4680662155151367, 0.34058743715286255, -1.011408805847168, 0.7337945699691772, -0.04530332610011101, 1.8696010112762451, -0.8056390881538391, -0.9674270153045654, -2.632310390472412, 0.7195470929145813, -0.17885051667690277, -0.9882292747497559, 1.5416280031204224, 0.0794842541217804, 1.1114202737808228, -0.8228325247764587, -0.01561897061765194, -0.6318750977516174, -0.5713396668434143, 1.6839444637298584, 0.8646970391273499, 1.0291497707366943], [1.2139431238174438, -0.7802361845970154, 1.0891320705413818, 0.38560035824775696, 0.8455661535263062, -0.18644414842128754, -1.3296297788619995, 0.6592510342597961, -0.9965850114822388, 1.0269825458526611, 1.0410287380218506, -0.9359402656555176, 1.904813289642334, -0.28926923871040344, 0.781404435634613, -1.3070043325424194, -2.6571249961853027, 2.183077812194824, 1.0030826330184937, 1.5106096267700195, -0.5398225784301758, -1.8221757411956787, -0.597629189491272, 1.4262526035308838, 1.9935599565505981, 0.9893631935119629, 2.5994508266448975, 0.9858952760696411, 1.7986118793487549, -0.650874674320221, 0.03414728865027428, -1.2597742080688477, -0.12249883264303207, -1.4008679389953613, -0.30079028010368347, 1.0722582340240479, -0.31301724910736084, -0.5083231329917908, -1.3687057495117188, -0.00806144904345274, 1.2259273529052734, -0.3980320990085602, -0.1990031599998474, -0.7251408100128174, 1.0631717443466187, 2.1188933849334717, 0.3395763635635376, 0.9965440034866333, -0.7389307618141174, -1.2802462577819824, 0.7500675320625305, 0.17214956879615784, -1.4713479280471802, -0.25860798358917236, -0.30801254510879517, 2.1218068599700928, -0.0691334679722786, 0.7506815791130066, 0.28004175424575806, 0.3180614411830902, 0.24151939153671265, -1.3395249843597412, 0.6287113428115845, -0.33836841583251953], [-0.022150179371237755, -0.3179228901863098, -0.512337863445282, -1.9702996015548706, -0.1708083599805832, -0.41509726643562317, 0.11634758859872818, -1.0977637767791748, 1.8720853328704834, 2.4604382514953613, 0.4160498380661011, 0.8664469718933105, 0.9696992039680481, 1.5034219026565552, -1.8813220262527466, -0.12845559418201447, -0.21055130660533905, -1.2016563415527344, -0.3037687838077545, 1.7161134481430054, -2.0605900287628174, -0.19850774109363556, 0.37049436569213867, 0.189339280128479, -0.06101877987384796, 0.5318050384521484, 0.0031399636063724756, 0.08240624517202377, -0.0016708496259525418, -1.0469378232955933, 0.13076849281787872, 1.0596842765808105, -0.3136212229728699, 0.08307530730962753, -0.053091861307621, -0.5996268391609192, 0.8769480586051941, -0.4880303740501404, -0.10700205713510513, -1.006426215171814, -0.9872376918792725, 0.3459181785583496, -0.2708631455898285, -0.8123078942298889, -0.010230234824120998, 0.5152823328971863, 1.095937728881836, 0.03928758203983307, -0.3549956679344177, 0.43595772981643677, -0.49322494864463806, 1.1300268173217773, 0.9113551378250122, 1.2289512157440186, -0.969053328037262, -1.1450861692428589, -1.0167292356491089, -0.7981637120246887, -0.4893835484981537, -1.3618371486663818, 0.13357210159301758, -0.2755504846572876, 1.613049864768982, 1.5344293117523193], [0.9248735904693604, -0.4150315523147583, -0.9747617840766907, 1.3541604280471802, -0.8017617464065552, 0.21298359334468842, 0.007458247244358063, 0.6012161374092102, -0.6983135938644409, 0.4304172694683075, 0.2659541070461273, 0.09944675862789154, -0.07285761833190918, 2.4503417015075684, 1.6429044008255005, -0.47417446970939636, -0.2254735678434372, -0.5928166508674622, -2.49855899810791, -1.4528647661209106, -0.26313361525535583, 0.5291370749473572, 0.3361812233924866, -0.5041777491569519, 0.16314594447612762, 0.5742023587226868, 0.47141584753990173, -0.7450984120368958, 0.42079824209213257, -0.23615135252475739, 0.03000788204371929, -1.1175020933151245, -1.465756893157959, -0.023171141743659973, -0.4495251178741455, 1.7312517166137695, -1.5796687602996826, 1.2219059467315674, -1.8034427165985107, -1.5647320747375488, 0.7061650156974792, -0.22138391435146332, -0.43252748250961304, 1.2401546239852905, 0.15337909758090973, 0.24300409853458405, 0.39325228333473206, 0.2867826223373413, 0.21181537210941315, -1.2738474607467651, 0.9833677411079407, 0.28053462505340576, 1.0161505937576294, 1.8918776512145996, -0.45837512612342834, 0.44471636414527893, 1.4342447519302368, -1.133759617805481, 0.40079548954963684, -1.2610411643981934, 1.5695064067840576, -1.2216707468032837, -0.21959882974624634, -1.2659157514572144], [-1.7907744646072388, -0.3370714783668518, -2.8098902702331543, 1.0042247772216797, 0.6381349563598633, -0.1903683990240097, -2.1251838207244873, -0.12147741764783859, 0.3238663673400879, -0.7587234973907471, -0.6655521988868713, 0.6654614806175232, -0.08472654968500137, 0.8204739689826965, 1.219614863395691, 0.8089708685874939, 1.1895687580108643, -1.8251157999038696, -0.19191336631774902, -1.3110586404800415, -1.0983704328536987, -1.4908981323242188, -0.5800374746322632, -0.8728469014167786, -1.6652947664260864, -0.01987089402973652, -0.18634605407714844, 0.39560776948928833, -0.6845167279243469, 0.19646258652210236, 0.47029125690460205, 0.8660976886749268, -0.9784597754478455, 0.4238426387310028, -1.0779180526733398, 0.17188963294029236, -1.731509804725647, -1.4724912643432617, 0.4039022624492645, -1.159197211265564, -0.8387466073036194, -0.14835648238658905, -0.2650825083255768, -3.090144634246826, 1.0057777166366577, 0.11528513580560684, 0.5180410146713257, -1.1434226036071777, 0.8606857061386108, -0.747353732585907, 0.020797498524188995, 0.5659464597702026, -0.46471577882766724, -1.0234392881393433, 0.25374656915664673, -0.4877869486808777, -1.1825810670852661, 0.5283614993095398, -1.3606486320495605, -1.2129703760147095, 0.8948197960853577, -1.5304161310195923, -0.6711468696594238, -0.3410978615283966], [1.1641349792480469, 1.6049511432647705, -0.6324971914291382, -0.7083069682121277, 0.7752008438110352, -0.7010605931282043, 0.2371218204498291, -0.5749958157539368, 0.6860553622245789, 0.20375369489192963, 0.33551108837127686, -0.5609605312347412, 1.4587643146514893, 1.7434474229812622, 0.11291921138763428, 0.8249786496162415, 0.39707571268081665, 2.7552649974823, -0.05260667949914932, 1.2863198518753052, 0.06515662372112274, -1.4983609914779663, 1.3567959070205688, -0.22192656993865967, -1.8854408264160156, 1.2676588296890259, 0.44528868794441223, -1.8972952365875244, 0.3255477547645569, -0.1792290061712265, -0.05197206512093544, -0.8365287780761719, -0.570915937423706, 2.3206820487976074, -0.08347858488559723, -0.15117664635181427, -0.0984978899359703, -0.819530725479126, -0.4211907684803009, 0.48771947622299194, -0.45502325892448425, 0.9301509261131287, 1.501726746559143, 2.5029044151306152, 0.4027017056941986, 0.003690336598083377, -1.225795030593872, 0.37463292479515076, -1.2390204668045044, -1.7634105682373047, 0.14392058551311493, -0.31516870856285095, 2.9566080570220947, -1.8749233484268188, -0.6191011667251587, -1.2049860954284668, -2.775374174118042, 1.402803897857666, 0.1217551901936531, 0.27760791778564453, -0.19295650720596313, -0.08829935640096664, -0.001689664670266211, -1.2534676790237427], [1.1301689147949219, -0.6851998567581177, 2.612438678741455, 0.7327632904052734, 0.8159846663475037, 0.3106406033039093, 1.2158536911010742, -0.13868050277233124, 1.1089979410171509, 0.5127747058868408, -0.6777017712593079, 0.16501352190971375, -0.05206068605184555, 0.014286994934082031, -0.5237331986427307, 0.8801894783973694, -1.1801347732543945, -0.6498528122901917, -1.05307936668396, -3.295336961746216, -1.2979413270950317, 0.5692477226257324, 0.3482034504413605, 0.04279598966240883, 0.16952289640903473, 0.04034418985247612, -0.7285064458847046, -0.8465868234634399, 0.09520000964403152, -0.4121014475822449, 0.7418424487113953, 0.13802538812160492, 0.6597496271133423, 0.4838263988494873, -0.08662227541208267, 0.33904922008514404, 1.1271079778671265, -0.6863554120063782, -0.6726003289222717, 0.70479816198349, 1.0873111486434937, -1.2769113779067993, -0.22347819805145264, 0.17842695116996765, 0.5794515609741211, -0.38033491373062134, 0.3191472887992859, 0.819499671459198, 0.7651779651641846, 0.46096867322921753, 0.6893439292907715, -1.2521774768829346, -2.0488736629486084, -0.25232449173927307, 0.8834445476531982, -0.5471819043159485, -1.0203157663345337, 0.3803165555000305, 1.4224215745925903, 1.7369139194488525, -0.7002508044242859, -0.5431696772575378, -1.0826152563095093, 0.4522821605205536], [1.4186105728149414, -0.3558448851108551, 1.6882268190383911, -0.3881339132785797, -1.332743525505066, -0.44993680715560913, -1.442803978919983, -1.643171787261963, -1.6552008390426636, -1.2281111478805542, -0.4428138732910156, 0.11905372142791748, -0.6884656548500061, -0.0012654821621254086, -1.159072995185852, -0.32127493619918823, 0.11449100822210312, 1.3332033157348633, 0.15684285759925842, 2.8960540294647217, 0.9740899801254272, 0.8944441676139832, 1.9639513492584229, -0.02728094346821308, -1.292415976524353, 0.3900657594203949, 1.676313877105713, -0.9420041441917419, -0.41441911458969116, -0.6126832962036133, -1.0684720277786255, 1.040839672088623, 0.4075724184513092, -0.9693492650985718, 0.0981922596693039, -0.404341459274292, 1.2962791919708252, -0.026426928117871284, -2.2291688919067383, -0.8273556232452393, -1.4964137077331543, -0.40157532691955566, -0.5910871028900146, -0.7035123705863953, -0.7067965865135193, -0.6699645519256592, 0.9431537985801697, -0.5113483667373657, 0.33310163021087646, 1.2720924615859985, -0.6842653751373291, -0.5403780341148376, -0.5830827355384827, -1.0996863842010498, -0.24442064762115479, -1.2654612064361572, 0.20886541903018951, 0.8306083083152771, 1.4787843227386475, -1.65831458568573, -0.8084852695465088, -0.8416950702667236, -0.32253703474998474, -1.53937566280365], [-1.3910797834396362, 1.8294223546981812, -0.4898239076137543, 0.004958987236022949, 0.14181487262248993, 1.3728679418563843, -0.9269619584083557, 1.4490200281143188, -0.5261853337287903, 0.4205920696258545, -0.3960227370262146, -0.20769497752189636, -0.9825403690338135, -2.3755576610565186, 0.8961331844329834, -1.2293410301208496, -2.4938395023345947, 0.7565094232559204, -0.7990286946296692, -1.1139556169509888, 1.1371827125549316, -1.2134056091308594, 1.962209701538086, -0.8449558615684509, 0.04744498431682587, -1.7728242874145508, -0.1570642739534378, 0.1369510442018509, 0.7276898622512817, 0.45838430523872375, -0.7577946782112122, 0.23685170710086823, -0.5804347991943359, 0.735762894153595, -0.47231218218803406, 0.5589150786399841, 1.2348096370697021, 0.33922863006591797, -0.4758371114730835, 1.358981966972351, -1.0878084897994995, 0.09641960263252258, -0.056946128606796265, -0.1295127123594284, -0.0455072820186615, -0.39871230721473694, 2.4918618202209473, 0.7780474424362183, -0.16932323575019836, -1.2231578826904297, 0.9702207446098328, -0.10190378874540329, 0.15425395965576172, -0.12176048755645752, 0.08761356770992279, -0.7048045992851257, -0.5128798484802246, -0.2794143855571747, -1.6203465461730957, -1.4919553995132446, -0.10205534845590591, 1.3258179426193237, -0.5872412323951721, 1.2997794151306152], [0.5197045207023621, 0.007386294659227133, -0.5846737623214722, 0.27312934398651123, -0.7009100317955017, -0.03251347318291664, 0.4008093476295471, -0.2139117419719696, 1.8134477138519287, 0.17256999015808105, 0.12298855930566788, 0.2716396152973175, 1.2189948558807373, -1.1564511060714722, 0.5121822953224182, -0.5454586148262024, 1.169250249862671, -1.2223283052444458, 0.07635018974542618, 1.1382498741149902, 1.600077748298645, 0.35657787322998047, -1.3983079195022583, -0.04216466471552849, -0.5450995564460754, -0.48887550830841064, 2.205444574356079, -0.9047159552574158, -0.23906324803829193, 0.9263485074043274, -0.024527758359909058, -1.6534242630004883, 0.03599124774336815, 0.09873795509338379, 0.12088897079229355, -1.1719741821289062, -0.5525228977203369, 1.1095893383026123, -2.105844020843506, -0.05334848910570145, 1.3468286991119385, 0.03872210904955864, -0.5133243799209595, -1.6121304035186768, 0.9075465798377991, 0.757885754108429, 1.8990516662597656, -0.3455783426761627, -0.7037559151649475, 0.38323646783828735, 1.6142351627349854, -0.30120694637298584, 1.621456265449524, 0.3469710946083069, 0.11548655480146408, -0.5937813520431519, -0.8600530028343201, 0.5313781499862671, -0.7252610325813293, 1.153302550315857, -0.985905110836029, 1.3101650476455688, 1.2879436016082764, -1.269059419631958], [0.8685950040817261, 0.9801178574562073, -1.2376261949539185, -0.5298362970352173, -0.14742764830589294, -0.8152076601982117, 0.5563361644744873, -0.6707075834274292, 1.2949224710464478, 0.6948302388191223, -1.9056493043899536, -1.289103388786316, -0.7233707904815674, -0.9563685059547424, 0.8627899289131165, 1.1867369413375854, -0.0969928503036499, 0.364595890045166, 0.6587028503417969, -0.6345229744911194, -0.3222782611846924, -0.4513768255710602, 1.6721946001052856, 0.668144702911377, 0.5909765362739563, 0.07480055093765259, 0.5480362176895142, -1.728044033050537, 0.31628456711769104, 0.08274634182453156, -0.5047204494476318, -0.6805579662322998, -0.5185620784759521, -0.021196529269218445, 0.10465746372938156, -2.7824363708496094, 0.043309085071086884, 0.43823856115341187, -0.1301209032535553, -0.8854029774665833, 0.22273299098014832, -1.0911970138549805, -0.5554147362709045, 0.4491503834724426, 2.0316002368927, -0.25753799080848694, -2.7870428562164307, 0.8949379920959473, -0.2587753236293793, 0.3927975296974182, -2.6791555881500244, 0.04330924525856972, 0.38659703731536865, 0.8269792199134827, 0.47319263219833374, -0.8444198369979858, 0.20923477411270142, -0.008778640069067478, 0.6516918540000916, 0.2427535504102707, 0.8566558361053467, 0.1245635375380516, 0.4715731739997864, 1.2540278434753418], [1.2707347869873047, -1.9156146049499512, -1.7342805862426758, -0.3807692229747772, 0.7992662191390991, 1.204280972480774, -0.36900290846824646, -0.9800496101379395, -0.7585935592651367, -0.5973328351974487, 0.05595283582806587, 0.5227291584014893, -1.2507187128067017, -0.2557852268218994, 0.3612577021121979, 1.514320969581604, 0.7359942197799683, -0.689818263053894, -0.759166419506073, -0.19222083687782288, -0.4673536717891693, 0.542549729347229, -0.3964923918247223, -0.16124635934829712, -1.2227798700332642, -0.499532550573349, -0.7832133769989014, -0.5813369154930115, 0.06095052510499954, -0.8761451244354248, -0.24248017370700836, -1.9739476442337036, -0.7372568845748901, 1.5178124904632568, -1.3860160112380981, -0.8904646635055542, 0.48071789741516113, -0.2300642430782318, 0.7230523228645325, -1.2557296752929688, 0.1113639697432518, 0.6587108969688416, 0.6111974716186523, -0.4890232980251312, -0.18933545053005219, -0.5262401700019836, 1.1133856773376465, -0.6739315986633301, 0.2529149353504181, 0.020404662936925888, 0.5518001317977905, -0.6649273633956909, 0.6269404292106628, -0.8691341280937195, -0.23521533608436584, -0.13602514564990997, -1.7917531728744507, 0.20911575853824615, -0.5873545408248901, 1.1600627899169922, -1.9954630136489868, -1.6422443389892578, 0.778801441192627, -0.9579588770866394], [1.2266370058059692, -0.2669682204723358, -1.4285142421722412, 0.19224070012569427, -0.5619749426841736, -1.2340470552444458, 0.8768487572669983, 0.9573484063148499, -1.9966109991073608, -0.287881463766098, 0.31857991218566895, 1.067814826965332, -2.1256890296936035, 0.39337286353111267, 0.0648997575044632, 1.9056049585342407, 1.0529369115829468, 0.275023490190506, -0.510624885559082, -1.1138006448745728, 1.2240654230117798, 0.003800705773755908, -1.3960460424423218, -0.5338696837425232, 1.4474204778671265, -0.5813987255096436, -0.5980837941169739, -2.1768410205841064, -1.6472402811050415, -1.2045791149139404, 1.4436802864074707, 1.5066105127334595, -0.4495711922645569, -1.4647870063781738, -0.7596388459205627, 0.63636314868927, 0.2952834963798523, -0.9096544981002808, -1.4018369913101196, -0.6308139562606812, -1.0376232862472534, -1.3732327222824097, 1.74843430519104, 2.159655809402466, 0.6900794506072998, -0.08314181119203568, 0.14065814018249512, 0.5681691765785217, 0.26195526123046875, -0.9109018445014954, 1.3004639148712158, 0.0009913332760334015, 0.36304301023483276, 0.42494380474090576, 1.0771286487579346, -0.4437148869037628, -0.6397693157196045, -1.482022762298584, 1.7242807149887085, -0.5107744336128235, -0.7373552918434143, -0.07142295688390732, 0.5623354315757751, 0.2973279058933258], [-2.028646945953369, -0.833681583404541, -1.2148455381393433, 1.1324180364608765, 1.13666832447052, 0.8695427179336548, 0.6624603867530823, -0.7380701303482056, -0.38627520203590393, 0.8673523664474487, 0.48488911986351013, -0.13510538637638092, -1.318799614906311, 1.8209317922592163, -2.1069631576538086, 0.8870230317115784, -1.0440274477005005, -1.8484410047531128, 0.024599295109510422, -0.3581032156944275, 0.9934787154197693, 0.7738174200057983, 3.189474105834961, -0.6208766102790833, -0.6374984383583069, -0.2652210295200348, -0.27589818835258484, -0.21447768807411194, -0.6560237407684326, 0.2836954891681671, -0.7925909757614136, 1.10389244556427, 0.39634448289871216, 0.788860559463501, -0.1264336258172989, 1.2946966886520386, -2.374396800994873, 1.524254560470581, 0.6865529417991638, 1.1250942945480347, -0.5168825387954712, -1.3108402490615845, -0.2360190451145172, -1.3712677955627441, -1.069280743598938, 0.3087543845176697, 0.16439054906368256, -0.9231338500976562, 1.4210481643676758, 0.22908318042755127, 0.46457093954086304, -1.2141848802566528, 0.5119299292564392, 0.3915761411190033, 0.6336179971694946, 0.9755792021751404, -1.8282520771026611, 0.21028614044189453, -0.07324492931365967, 0.6380603909492493, 0.5998871922492981, 0.02254713699221611, -0.4222809970378876, -1.7528218030929565], [1.2077678442001343, 0.19005858898162842, 0.07477055490016937, -0.4263308644294739, 0.7548055052757263, -0.8929784893989563, 1.7775603532791138, -0.4735243618488312, -1.9165465831756592, 1.3735765218734741, 0.2416308969259262, -0.124057337641716, -0.45113080739974976, 0.7387990355491638, 1.670287847518921, -2.4276654720306396, -0.2721618115901947, 0.6435298919677734, -1.3291478157043457, 0.17247922718524933, 0.10526879876852036, -0.9534848928451538, 1.636772632598877, 0.7027982473373413, -0.7498270869255066, 0.5812432169914246, -0.8866482973098755, 0.17790789902210236, 1.1407604217529297, 0.023598389700055122, -1.8323838710784912, -1.4287939071655273, 0.14568254351615906, 0.15729670226573944, 0.13538481295108795, 0.14606782793998718, -0.9559935331344604, -0.3888033330440521, 0.3606625199317932, -0.07737306505441666, 0.5709099173545837, -0.43670040369033813, 0.8906234502792358, 1.250489354133606, 0.27389925718307495, -1.3085743188858032, -0.13624458014965057, 0.0349794365465641, -0.20265555381774902, 0.88287752866745, 1.290399193763733, -0.5470571517944336, -0.20887519419193268, -1.2019637823104858, 1.2622408866882324, -1.0364511013031006, 1.2360057830810547, -0.35334205627441406, 0.3353642523288727, 0.10240212082862854, -0.12556904554367065, 1.7794126272201538, 1.250102162361145, -2.725695848464966], [-0.3996847867965698, 0.3001324534416199, 0.5686107873916626, 1.1278791427612305, -0.3281280994415283, -0.30109548568725586, 1.4149503707885742, 0.7713242769241333, -0.09749409556388855, -0.05716539919376373, 0.0731249451637268, -0.2488044798374176, -1.016524076461792, 0.08507271111011505, -1.0722739696502686, 0.8077808022499084, 2.039308547973633, 0.2895497977733612, -0.850462019443512, -0.24961991608142853, 2.1846883296966553, 0.6175784468650818, 0.174112930893898, -0.06387368589639664, -0.2861570119857788, 0.3066883385181427, -0.5312633514404297, -0.2988103926181793, -0.5162323713302612, -0.18123900890350342, 0.1606818437576294, 0.04774084687232971, 0.035426996648311615, 0.695694625377655, 0.7664135694503784, 0.740229606628418, 0.9136157035827637, -0.7702763080596924, -0.9674980044364929, 0.5600305795669556, 2.2837584018707275, 0.21544863283634186, 0.6369513273239136, 0.6742216944694519, -0.3735455572605133, -0.24749380350112915, 0.6159095168113708, 1.6119354963302612, -0.3195524215698242, 0.034383874386548996, -1.0908374786376953, -0.302899032831192, 1.1120223999023438, 0.32425040006637573, 0.9278742074966431, -0.2790534198284149, 1.0057506561279297, 0.5625754594802856, 0.2298857420682907, -0.24284669756889343, -0.9477835297584534, -0.7957781553268433, -0.19790253043174744, 0.5066902041435242], [0.3208078145980835, -0.5024933815002441, 0.9001086354255676, 0.4283169209957123, -0.9011004567146301, 0.5025418400764465, -0.8036245107650757, 0.6349579095840454, 0.4242096245288849, 1.3124499320983887, -0.12143013626337051, -1.1445964574813843, -0.28866466879844666, 0.15825989842414856, -1.8720898628234863, 1.097009539604187, 0.3920111358165741, 0.6556127071380615, -0.12324654310941696, 0.5676215291023254, 0.03821022808551788, 0.022274751216173172, -0.03686321899294853, -2.1756014823913574, -0.008943598717451096, -1.6579527854919434, -0.6880204677581787, 0.9590941667556763, 0.1542908400297165, -0.62297523021698, -1.2869070768356323, 0.8781416416168213, 0.5232490301132202, 1.2044847011566162, -0.28334948420524597, 0.0900569036602974, 0.4447924494743347, -0.10585793852806091, -0.6453775763511658, -0.2757314443588257, -1.6475591659545898, 0.18460261821746826, 0.47298815846443176, 0.4995168447494507, -1.5706536769866943, 0.40525001287460327, -0.35835129022598267, -0.12850502133369446, 0.2655632793903351, -0.441981703042984, -1.5666987895965576, 0.7304347157478333, -0.537166178226471, 0.05672701448202133, 0.6265983581542969, 0.7473217844963074, 0.5450230240821838, -0.5914388298988342, 1.2611745595932007, -0.4934815466403961, -1.6090975999832153, 0.3963642418384552, -1.9298168420791626, -1.0501234531402588], [0.11304204910993576, -0.4808540642261505, 1.3385766744613647, -0.7494778633117676, -1.1415473222732544, 0.10428667813539505, 0.11073673516511917, -1.5343694686889648, -0.8373206853866577, -0.3878406882286072, -0.8736957311630249, 0.9260431528091431, 0.06895104050636292, -0.7300960421562195, 0.7126016020774841, -0.733627200126648, -0.2030290812253952, -0.8214071989059448, 1.0201979875564575, 0.7004051804542542, 0.11242018640041351, 0.5100566148757935, -0.31561002135276794, -1.2561230659484863, 1.9478415250778198, -0.024952750653028488, -0.5661991238594055, -0.5406724810600281, -1.1600195169448853, 0.2584954798221588, -1.0353518724441528, -0.27979350090026855, 0.09098535776138306, 0.5194320678710938, 1.2159736156463623, -0.21604590117931366, 1.121305227279663, -1.5610779523849487, -0.5835365653038025, 1.3699595928192139, -0.38450706005096436, -0.5792680978775024, 1.5718083381652832, 0.24708572030067444, 0.3441758453845978, -0.8483476638793945, 0.24659156799316406, -0.10799767822027206, 0.49264878034591675, 0.9670135974884033, -1.550511121749878, -0.9155608415603638, 0.1477295309305191, -0.7509998679161072, -0.4091736078262329, -1.127034306526184, 0.6113424897193909, 0.9121615886688232, 0.9965831637382507, -0.5059157609939575, 0.22170202434062958, -1.5435625314712524, 1.7383695840835571, 0.8268258571624756], [1.1742360591888428, 1.011903166770935, 1.5120820999145508, -0.12035980820655823, 1.3677356243133545, 0.5215775966644287, -1.0649017095565796, -0.5722687244415283, 0.828309178352356, 0.6187275052070618, 0.8110771775245667, 1.1270196437835693, -0.8562880754470825, 0.4002537429332733, 0.5897560715675354, 1.8578157424926758, 0.5194470286369324, 0.208784818649292, -0.3140476942062378, -0.18804578483104706, 0.6238621473312378, -1.4908645153045654, 1.5496063232421875, 0.23981070518493652, 0.05978154391050339, 0.49307766556739807, 0.017851000651717186, 0.29201820492744446, -0.05120812729001045, -0.7743216753005981, -0.7214130163192749, -0.10016091167926788, 2.069014072418213, 0.5729444622993469, 0.2720331847667694, 0.40355154871940613, 0.9151215553283691, -0.34668663144111633, -0.46602606773376465, -0.6410204172134399, 0.25342050194740295, 0.6618168950080872, 0.15735702216625214, -0.6127186417579651, -1.019579291343689, 0.20981040596961975, 0.9564438462257385, -0.2527633011341095, -1.053127408027649, 1.408774971961975, -0.8018870949745178, 0.03771417215466499, 0.46343910694122314, -0.2681215703487396, -1.4602842330932617, 0.6256026029586792, -0.30744919180870056, 2.1937246322631836, -0.4692284166812897, -2.135607957839966, 2.0207552909851074, 0.2872418165206909, -0.8303964138031006, 1.4363725185394287], [-0.4781113266944885, 0.6753811240196228, 0.3312321901321411, -0.08561291545629501, -0.4933528006076813, 0.1346759796142578, -1.112403154373169, -0.543935239315033, 0.07528460770845413, -2.454179525375366, -0.3638800084590912, -1.2889493703842163, -0.6092457175254822, -1.7724040746688843, 1.7717690467834473, 0.18998906016349792, 0.4208453893661499, -1.1972938776016235, -0.1952594816684723, 0.27849799394607544, 0.7721586227416992, -0.039743952453136444, -0.42331695556640625, 0.43058595061302185, -0.4491371512413025, -1.4428647756576538, -0.20590424537658691, -0.36293354630470276, -0.008046602830290794, -0.12303750962018967, -0.5736309885978699, 1.6519020795822144, 1.5862704515457153, -0.6022480726242065, -0.2867550551891327, -1.841234564781189, -1.2716014385223389, -1.276992917060852, 0.10251853615045547, -0.33886170387268066, -0.47215020656585693, -1.5086477994918823, -2.0177817344665527, -0.3806009590625763, -1.6142557859420776, 0.9440017938613892, 1.7431391477584839, -0.8969202041625977, 1.0086524486541748, -0.826051652431488, -0.5426989793777466, -0.13193614780902863, -1.2439031600952148, 0.1500692069530487, 0.9776619672775269, 3.608112096786499, -1.0546783208847046, -0.8163489699363708, -0.7892816066741943, 2.43154239654541, 0.08411607146263123, -0.6177152991294861, -0.0588388666510582, 0.34941643476486206], [0.6045693159103394, 1.0902509689331055, 0.17448358237743378, -2.1280784606933594, -0.6881906390190125, 0.11950536072254181, -0.6695483922958374, 3.186439037322998, 0.6405315399169922, 2.467219591140747, 0.331094354391098, -0.9883133769035339, -0.7037931084632874, -1.5769811868667603, 0.5206218957901001, -1.0169081687927246, -1.0878114700317383, -1.8588931560516357, 0.07916593551635742, -2.6487157344818115, -1.4043715000152588, -1.9504621028900146, 1.3218358755111694, 0.09939484298229218, -2.8823699951171875, -0.7799131274223328, -1.555931806564331, -2.0329036712646484, -0.5752116441726685, -0.6482564806938171, 0.2525798976421356, -1.1270564794540405, -1.7283936738967896, -0.54010009765625, 2.1270217895507812, -1.046461820602417, -1.0940301418304443, 0.030032750219106674, 0.5143487453460693, -2.110032320022583, 0.6353487372398376, 0.239878848195076, 2.562034845352173, -0.57120680809021, 0.4656663239002228, -1.6899806261062622, -0.3267142176628113, 0.09854667633771896, 0.453710675239563, -0.5214612483978271, -0.2704346477985382, 0.5509083867073059, 1.02109956741333, -0.6637100577354431, -0.3798581063747406, -0.48048004508018494, 1.2923808097839355, 0.9549016952514648, 1.5487927198410034, 0.20445963740348816, -0.768656313419342, 1.4380710124969482, -1.7848632335662842, 0.10747434198856354], [-0.18803898990154266, -1.6609163284301758, 0.9053961038589478, 1.1980459690093994, -1.0084154605865479, 0.42164379358291626, 0.5600516200065613, -1.4498625993728638, -0.6402572989463806, 1.0194661617279053, -0.5456814169883728, -1.4385889768600464, -0.8904415369033813, 0.18337559700012207, -0.5904192328453064, -0.3529956042766571, -0.2713625133037567, 0.7358868718147278, 0.6759295463562012, -1.7403943538665771, 1.019850492477417, -0.2581899166107178, 0.8053086996078491, 0.6312663555145264, -0.5936530828475952, 0.12251105159521103, 1.6769003868103027, 0.40273168683052063, -1.3332593441009521, -0.7522734999656677, 1.6266554594039917, -1.7975445985794067, 0.002717644674703479, -2.5909535884857178, -0.8334385752677917, 0.0637800544500351, 0.017410805448889732, 0.03094891458749771, -1.1090666055679321, 1.377461314201355, 1.4785616397857666, -0.08771705627441406, -0.69390869140625, 1.8301033973693848, 0.5032135248184204, 0.2549997866153717, 0.7599242925643921, -2.0735931396484375, -0.461681067943573, 1.5061852931976318, 1.9855903387069702, -0.8928959369659424, -1.284136414527893, 0.35577014088630676, 0.24729101359844208, -1.287906289100647, 0.21482014656066895, -0.24880066514015198, -0.39166876673698425, -1.810294270515442, -2.0739176273345947, 0.9891675710678101, 0.9888156652450562, -2.0866036415100098], [0.7587960362434387, -1.0810810327529907, -0.3486478328704834, -0.5984100103378296, 0.4191209077835083, 1.0561662912368774, -1.21078622341156, 1.1776665449142456, -0.241582453250885, -0.32928547263145447, 0.7661278247833252, -0.4083934724330902, -1.424813985824585, 0.3071019649505615, -3.026423931121826, 0.13422706723213196, 0.5835436582565308, 0.9656878709793091, 0.5952029228210449, 0.01967117190361023, 0.8296189904212952, -0.7900910377502441, -0.013095895759761333, 0.8948361277580261, -0.8336079120635986, 0.9193822145462036, 0.06191408634185791, 0.18630841374397278, 0.4116126596927643, 0.7245984673500061, -0.35660356283187866, -0.039871569722890854, -0.6881523728370667, 1.1601414680480957, 0.044157594442367554, 0.7535353302955627, -1.5370153188705444, 0.3867250680923462, 0.7547463774681091, 0.06615373492240906, -1.1261132955551147, 0.7642435431480408, 0.06881723552942276, -0.3720301687717438, 0.4194449484348297, 0.858776330947876, 1.7534233331680298, -1.2640248537063599, -0.8599380850791931, -0.9870112538337708, 1.1968156099319458, -0.3076072037220001, -0.8814102411270142, 0.15581166744232178, -0.8690599799156189, -1.9420443773269653, -1.8327475786209106, -1.2766997814178467, -0.6300952434539795, -0.3966478705406189, -1.3933541774749756, 0.06838047504425049, -0.10166534781455994, -1.2314772605895996], [-1.1840487718582153, 0.49498966336250305, -1.7080249786376953, -0.5512537360191345, -0.002150567015632987, 1.6336146593093872, 0.8855589628219604, -0.896862268447876, -2.021477222442627, -0.3789559304714203, 2.0003979206085205, 1.2181756496429443, 0.5241624116897583, 0.7277690172195435, 1.4085763692855835, -0.5941400527954102, 0.6646552681922913, 0.18044108152389526, 3.4416465759277344, 0.8225274085998535, 0.12031849473714828, 2.4603500366210938, 1.5236718654632568, -0.8998675346374512, -1.5292130708694458, -0.6911231875419617, -1.8071843385696411, -2.6743619441986084, 0.16164523363113403, -1.9586833715438843, -0.49758055806159973, -0.9834335446357727, 0.9513663053512573, 0.02181856706738472, 1.7791094779968262, -0.4874967634677887, -2.9959895610809326, -1.309061884880066, 0.6220923662185669, 0.42119136452674866, 2.107975959777832, -2.0755748748779297, -1.5919733047485352, 0.37351658940315247, -0.22910642623901367, 1.5862513780593872, 0.05542214214801788, -0.10637844353914261, -1.082387924194336, 0.9858747124671936, -0.3033171594142914, 1.55168616771698, 0.5111459493637085, -0.5801016092300415, -0.8411566615104675, 0.4328341782093048, -0.08407564461231232, -0.5738629698753357, -0.9183543920516968, -0.43667006492614746, 1.4161086082458496, 1.0786104202270508, 0.44466710090637207, -0.4457337558269501], [-0.2359512448310852, 1.658942699432373, 1.7735724449157715, 2.235456705093384, 0.09646262973546982, -0.2533229887485504, 2.172133445739746, 0.671096682548523, -0.09164277464151382, 0.28004443645477295, 0.28578242659568787, 0.6685911417007446, 1.3288044929504395, -0.12002461403608322, -1.106744647026062, -0.4236499071121216, 2.4508254528045654, 1.8837283849716187, -1.7596347332000732, -0.06525112688541412, -2.067495346069336, -0.6970440745353699, -0.5550974607467651, -0.1420687437057495, 1.17800772190094, -0.772990882396698, -0.08300801366567612, -0.5834707617759705, -1.6037306785583496, -1.4874523878097534, -0.8532248139381409, 0.6504089832305908, 0.3795722424983978, 0.1281503438949585, 1.099335789680481, 0.9840819835662842, -2.21486234664917, -0.6027041077613831, 0.17753279209136963, 1.0017896890640259, -0.49000102281570435, 0.6425092816352844, -1.0440571308135986, -0.37825968861579895, -0.6364213228225708, 0.2007492184638977, -1.9845309257507324, -1.1589264869689941, 0.6738049387931824, -1.090253233909607, 0.3637325167655945, -0.9839088916778564, 0.8215797543525696, 0.1848200410604477, -0.3775635361671448, 1.0645544528961182, -0.7173910140991211, -0.2137797325849533, -0.6152011752128601, 1.5546650886535645, -0.33147063851356506, 0.6172029972076416, -1.4268500804901123, 0.1350029557943344], [1.8047176599502563, -0.17711052298545837, 0.39211124181747437, -1.0350111722946167, -0.5353764891624451, 0.502710223197937, 0.5743263959884644, -1.0915933847427368, -0.019931672140955925, 0.9595281481742859, -0.540859580039978, 0.6438908576965332, -0.6737558245658875, 0.42608118057250977, -1.6174050569534302, 0.24377378821372986, 0.9661926031112671, -1.1159309148788452, 0.13482998311519623, -0.6454088091850281, 0.3851884603500366, -0.7541911005973816, -1.0081888437271118, 0.682339608669281, 0.2745607793331146, 0.3430974781513214, 0.21820640563964844, -0.029528021812438965, 0.8790622353553772, 0.9718518257141113, 0.06496322154998779, -0.15412937104701996, 1.5058917999267578, 0.01507706567645073, -0.20329657196998596, 0.5283178687095642, -0.6063066720962524, 1.0263235569000244, -0.12081210315227509, -1.6528843641281128, 0.16022911667823792, 0.09440895915031433, 1.1937650442123413, -0.5800959467887878, 0.1692095696926117, -0.467393696308136, 0.2067280262708664, 0.5064075589179993, -0.9545890688896179, -1.352809190750122, 0.9077404141426086, -1.2060041427612305, -1.2536917924880981, -0.013601556420326233, 0.28173401951789856, 1.141743779182434, -1.0803242921829224, 0.19321511685848236, 0.3734106421470642, 0.08662717789411545, -1.0197230577468872, -0.5830064415931702, 0.4955563545227051, -0.8399351835250854], [-2.414201021194458, -0.4162304997444153, 0.8231728076934814, -0.037233542650938034, -1.5714167356491089, -0.6850781440734863, -1.1892168521881104, 0.7339327931404114, 0.5176345109939575, -0.5698782205581665, -0.1383369117975235, 1.7881935834884644, 0.45937973260879517, -0.7387499809265137, 1.3927953243255615, -0.5581511855125427, -0.35152849555015564, -1.021562933921814, 0.4553822875022888, 0.03421567752957344, 1.2697737216949463, -0.7147654294967651, -0.09575971961021423, -1.5692236423492432, 0.19710011780261993, -0.9433912634849548, 0.2660267651081085, -0.06543177366256714, -0.21764016151428223, 0.9470422863960266, -0.35250088572502136, 0.4296758472919464, 0.6018635034561157, -0.6014538407325745, -0.7941186428070068, 0.9118567109107971, 0.38008153438568115, 1.3842989206314087, -0.06896266341209412, -0.09481354802846909, 0.796940803527832, -0.08469673991203308, 0.9566867351531982, -0.22579772770404816, 0.022022482007741928, -0.17507629096508026, 0.9520134925842285, 0.30074337124824524, 0.06818541139364243, -2.1434574127197266, 0.44588401913642883, 0.2583569586277008, -0.11345241963863373, -0.14986072480678558, -1.2095409631729126, 0.6043477058410645, -0.6428991556167603, 1.9897161722183228, -1.005101203918457, -1.6960701942443848, 1.382262110710144, -0.5484349131584167, 1.4600703716278076, -0.9266229271888733], [-0.2494393289089203, -1.1811540126800537, 0.20165683329105377, -2.6158976554870605, -1.7300913333892822, 0.3732682168483734, 0.5004523992538452, -0.43526843190193176, 0.7126774191856384, 0.4521808922290802, 1.755676031112671, 0.49065855145454407, 0.43370601534843445, 0.9591177701950073, -1.4129133224487305, 0.14851920306682587, 1.3368312120437622, 0.4881107211112976, 1.4369425773620605, -0.44486019015312195, 1.3609237670898438, -1.5146199464797974, -0.13665585219860077, 0.8130127787590027, -2.065399408340454, -0.8702005743980408, -0.7820149064064026, 0.5919377207756042, 1.162031888961792, 0.5792801380157471, 1.2433602809906006, 1.1046037673950195, 2.2855358123779297, -0.3005525469779968, 1.0729279518127441, -0.7304513454437256, -0.20376300811767578, 1.7606570720672607, 0.23297031223773956, 0.3629674017429352, -1.2194995880126953, -3.0824365615844727, -1.8363292217254639, -0.8262953758239746, 2.6946957111358643, -1.3719507455825806, -0.3110751211643219, 0.4156305491924286, 1.1037791967391968, 0.029646309092640877, -0.28386712074279785, -0.5334158539772034, 1.6666409969329834, 0.21721863746643066, -0.4910149872303009, -0.19439996778964996, 0.849022626876831, -0.410066157579422, 1.2837530374526978, 0.5110153555870056, -0.5676084756851196, 2.1167314052581787, 2.159782648086548, -0.03198961913585663], [0.03171420842409134, 0.060498110949993134, -0.458260715007782, -0.6922035813331604, -0.35577264428138733, 1.0804359912872314, 0.5541538596153259, 1.534228801727295, -0.4675692319869995, 0.07230208814144135, -0.48013392090797424, 0.7122424840927124, -0.591221809387207, -0.9507652521133423, 0.19150292873382568, -1.558411955833435, 0.3825124502182007, 0.6607407927513123, 0.4376586377620697, 0.022261442616581917, -1.2712037563323975, 1.2866275310516357, -0.8134554624557495, 0.9806614518165588, -0.20433051884174347, 0.8243391513824463, 1.4256949424743652, -0.8389217257499695, 0.3281194865703583, 1.1438816785812378, -0.6122774481773376, -0.25299254059791565, -0.35337650775909424, 0.4484315514564514, 0.13938109576702118, -0.46668821573257446, -0.583728551864624, 0.10730938613414764, -1.1928281784057617, -0.6995508670806885, -0.06118040159344673, -0.001988760894164443, -0.5488621592521667, 0.9133828282356262, 0.5860869288444519, 1.1550084352493286, -0.3937329053878784, 0.73021399974823, 1.6695178747177124, 1.1943598985671997, -2.4741501808166504, 0.3042535185813904, -0.33069711923599243, 1.2756556272506714, -0.4842621386051178, 0.4996131956577301, 0.1962939202785492, -0.3538557291030884, -0.27398666739463806, -0.7094243764877319, 0.29404959082603455, 0.45910459756851196, -0.2413322478532791, -0.1684267520904541], [-0.17877982556819916, -0.03381552919745445, 0.623657763004303, 1.443647861480713, -2.263970136642456, 0.10413012653589249, -0.872773289680481, 1.2620368003845215, 1.343653678894043, -0.7021306157112122, 0.29854512214660645, 0.9951519966125488, -0.3683170676231384, -0.08757807314395905, -0.6503846645355225, -0.3566935658454895, 0.2734875977039337, 1.4308499097824097, -0.4078332185745239, -0.1500556617975235, 0.07007665187120438, 0.08032163232564926, -0.8602839708328247, 0.2500397562980652, 0.9168115854263306, -1.1939884424209595, -1.1472415924072266, 0.5706568956375122, -0.6738406419754028, 0.7871795296669006, -1.1349149942398071, 1.7612513303756714, -0.8405147790908813, 1.192734956741333, -0.2200413942337036, 0.8499865531921387, 0.23513071238994598, 0.5002472996711731, -0.09413604438304901, -1.0720843076705933, -1.5576168298721313, 1.3012561798095703, 1.089348316192627, -0.38102129101753235, -0.11875767260789871, -1.1012811660766602, 0.012240057811141014, -0.2878827452659607, 0.2244315892457962, -0.6778616905212402, 1.0352412462234497, 1.7730724811553955, -0.08203043043613434, -0.12504775822162628, 0.025522669777274132, 0.9745765924453735, 0.8383489847183228, 1.3532330989837646, -0.09485559910535812, 0.5131216645240784, -0.08062180876731873, -1.6728367805480957, 1.0627143383026123, -1.6982614994049072], [0.05559840053319931, 0.12825201451778412, -0.5723047256469727, 0.0748603567481041, 0.8233498334884644, 0.19419033825397491, -0.19111500680446625, -1.469433307647705, -1.6855332851409912, -0.6342630386352539, 0.5303094387054443, 2.4306602478027344, -1.2861828804016113, -1.265717625617981, -1.8435572385787964, -0.13093648850917816, -0.2757549285888672, -2.021986722946167, 0.032022736966609955, -0.3164618909358978, 1.765711784362793, 0.7779393792152405, 1.4217638969421387, 0.9899656176567078, -0.4775729477405548, -0.23628373444080353, 0.8158105611801147, -0.21901404857635498, 0.37769776582717896, 0.9288881421089172, -1.0089595317840576, -1.507690668106079, 0.254810631275177, -0.5903598666191101, -0.1695580929517746, -0.11162249743938446, 0.20492395758628845, -1.3489621877670288, 0.22756339609622955, 0.530252993106842, 0.41599348187446594, 0.1527576893568039, -1.3357737064361572, -0.9401493668556213, -1.4334255456924438, 1.2988760471343994, 0.23455658555030823, 0.8635023832321167, -1.0836116075515747, -0.19152511656284332, 1.0824881792068481, 1.5859249830245972, -0.48878464102745056, -0.9621860980987549, -0.583221971988678, 0.7486255168914795, 1.0344481468200684, 0.7657085061073303, 1.4953714609146118, 0.7990680932998657, -0.04232989251613617, 3.3132784366607666, -0.12521770596504211, 0.5930148959159851], [-1.7223601341247559, -0.5031747817993164, -0.5803993940353394, -1.0094763040542603, 0.013163093477487564, -0.04391390457749367, -0.9560339450836182, 0.7037829160690308, -0.7546831965446472, -0.022453906014561653, -0.6051906943321228, 0.7505728006362915, 0.2891283631324768, -0.009354891255497932, 0.0882500410079956, -2.0303022861480713, -0.5364880561828613, -0.4492802619934082, 0.7915300130844116, -2.2985212802886963, 0.9984705448150635, -1.4867119789123535, 0.4594893455505371, 1.1167948246002197, 2.9675865173339844, 0.05863172560930252, 0.7567073106765747, -0.10388308018445969, -1.4052530527114868, 0.3679124116897583, -0.008850454352796078, 0.7111177444458008, 1.6941752433776855, -0.4290352463722229, -0.07255735248327255, 0.3513339161872864, -1.3010386228561401, -1.1337496042251587, 0.07467379420995712, 2.00447940826416, -1.7259312868118286, -0.9518582224845886, -0.3439141809940338, 0.3816032409667969, -0.1631346344947815, 2.8636655807495117, -2.1080169677734375, -1.0871162414550781, 0.6939853429794312, -0.6902876496315002, 1.5351258516311646, 0.0943254828453064, -0.6238143444061279, 0.868109405040741, 0.1654123067855835, -0.48651358485221863, -0.08789695054292679, 0.6506611704826355, -0.032626282423734665, 1.0109082460403442, -0.30726009607315063, 0.6326033473014832, -2.046058416366577, -0.9966054558753967], [1.0063070058822632, 0.27258485555648804, 0.21597938239574432, 1.656875729560852, -0.499607115983963, -0.28785672783851624, -2.1343023777008057, 1.1043620109558105, -1.0313550233840942, -0.4682544767856598, 0.5138817429542542, -1.1596869230270386, 0.19850681722164154, -1.7888829708099365, -1.4463316202163696, -0.32949748635292053, 0.9350069761276245, 0.2106195092201233, -0.3220500349998474, -0.8121495842933655, -0.359546035528183, 1.1369318962097168, 0.8278579115867615, 0.6424853801727295, -1.90658438205719, 1.3144850730895996, -0.6854119300842285, 0.9484584331512451, -0.09413065016269684, -1.1726981401443481, -0.9737204313278198, 0.6704725027084351, 1.0641193389892578, 1.7290756702423096, 1.1236811876296997, 1.811347484588623, -0.8565034866333008, 0.7674700617790222, -0.5866438150405884, 0.16432888805866241, -1.0038081407546997, 1.157533884048462, 0.8629323840141296, -1.7903809547424316, -0.2859257161617279, 0.654351532459259, 1.3864531517028809, -0.9071546792984009, 0.4210561215877533, -1.4886417388916016, -1.4507488012313843, 0.6039697527885437, 0.3293103873729706, -0.886870265007019, -0.6130504608154297, 0.09714945405721664, -0.47837480902671814, 1.3618704080581665, 1.6447780132293701, 0.890765905380249, 2.097660541534424, 0.609196662902832, -1.4558466672897339, -0.14527074992656708], [0.8167614936828613, 0.20535103976726532, 2.262575387954712, 0.009256191551685333, 0.780271589756012, 0.8735963702201843, -1.9792847633361816, 0.43232202529907227, 0.2495361566543579, 1.3253759145736694, 1.0586645603179932, 0.27026012539863586, -0.49610745906829834, 1.538447380065918, -0.19439101219177246, -0.10288670659065247, -1.9441035985946655, 1.0400896072387695, -0.0528269037604332, -0.10949066281318665, 2.2251198291778564, -0.8820524215698242, 1.2591642141342163, -2.628936529159546, 0.2649940550327301, -1.3725306987762451, 0.1807704120874405, -0.9011470675468445, -0.2628706395626068, 0.3329748809337616, -1.9104206562042236, -0.415352463722229, 0.663483202457428, -1.8863598108291626, -1.5182603597640991, -0.9338945150375366, -0.6705508232116699, -0.5435245633125305, 0.283500999212265, -0.27055487036705017, 2.115175247192383, 0.4130491316318512, 0.9622112512588501, -0.04225996881723404, -0.2918233573436737, 0.06962589174509048, 0.2173037827014923, -0.3993419408798218, -0.7048538327217102, 0.9521868824958801, -0.8131299614906311, -0.03315902128815651, 1.1348387002944946, -0.37750381231307983, 0.5510818362236023, -0.5560930371284485, 0.6174235939979553, -1.1695513725280762, -2.0104739665985107, 0.20340295135974884, 1.4639848470687866, 0.2086428850889206, -0.5098547339439392, 0.6957942843437195], [-1.8110594749450684, -0.3844953179359436, 0.2608039677143097, 0.694404661655426, -0.9240220189094543, 0.7200371026992798, -0.5381103157997131, 0.7446979880332947, 0.7942177057266235, 0.03445877507328987, 0.3883775472640991, 0.9600305557250977, 0.13818402588367462, 0.5431811809539795, 1.3322045803070068, -0.9250195026397705, -1.668387770652771, -0.2964913845062256, 0.6193633079528809, 0.2656770348548889, -0.12272771447896957, -0.11313513666391373, 0.07098420709371567, 1.6608827114105225, 2.717288017272949, -0.7411156296730042, -0.27191218733787537, 0.3887670040130615, 0.024866798892617226, -0.26792198419570923, -0.3059895932674408, -0.8550208806991577, -2.420741558074951, 1.200505018234253, 1.3787541389465332, 0.8698338270187378, -0.39775827527046204, -1.0912182331085205, -0.8495139479637146, -0.2691839039325714, 0.7037180662155151, 1.6106078624725342, 0.259714275598526, -0.8586860299110413, 0.9927133321762085, -0.29323264956474304, 0.7664375901222229, 0.45728740096092224, 0.010335354134440422, -0.057230472564697266, 0.5161139965057373, -0.07064122706651688, -1.7418079376220703, 0.2743851840496063, 0.6130658388137817, -1.8666263818740845, 0.5723899006843567, -0.1336316168308258, 1.5557996034622192, -0.4560360908508301, 1.5570626258850098, 1.3036649227142334, -1.6050077676773071, -1.4941383600234985], [0.7009825706481934, -1.4255892038345337, 0.5527242422103882, 0.5367396473884583, 0.558997631072998, 0.9654163122177124, 0.8986754417419434, 0.821252703666687, -0.6673207879066467, -0.6722199320793152, 1.7260288000106812, -1.2467572689056396, -1.632449984550476, 1.9952538013458252, 0.866539716720581, -1.705132007598877, -0.35298067331314087, 1.6346237659454346, 1.1231822967529297, 0.7722664475440979, -0.46881726384162903, -1.331738829612732, 0.12073638290166855, -1.4951711893081665, 0.7419618964195251, -0.7597180008888245, -0.18330587446689606, -0.0030643781647086143, -0.4242681860923767, 0.46046891808509827, -0.3506245017051697, -0.36138227581977844, 0.8167281746864319, -0.36307618021965027, 0.3985593318939209, 0.9301934242248535, -0.5671082735061646, 0.6740325689315796, -0.018150819465517998, -3.578538417816162, 2.071721315383911, 0.1253764033317566, -1.9063321352005005, 1.3095903396606445, -0.6750683188438416, -0.862774133682251, 0.2843412458896637, 0.28653475642204285, -0.6406452655792236, -1.0358822345733643, -0.7834124565124512, -1.3515543937683105, -1.2806581258773804, 0.5979626178741455, 0.3277570605278015, -0.5611581206321716, 1.4178717136383057, -1.617385983467102, 2.3762145042419434, 1.218021273612976, 0.6109021306037903, -0.7953308820724487, -1.0829274654388428, 2.4832143783569336], [-1.116782546043396, -0.004145825281739235, 0.021675508469343185, 0.9804598093032837, -0.2825935184955597, 1.3134373426437378, -0.4664057195186615, 0.15951907634735107, -0.6123756170272827, -0.062220003455877304, -0.39349880814552307, -0.7992380261421204, 1.2579315900802612, 0.20574072003364563, -0.9100826382637024, 0.2416083961725235, 1.043850302696228, -1.790716528892517, 1.4641823768615723, -0.725362241268158, 0.27171146869659424, 0.4196278750896454, 3.6121020317077637, 1.0418940782546997, 2.182516098022461, -0.7553260326385498, -0.2124742567539215, -0.5240064263343811, -0.01236699614673853, -0.49684938788414, 1.4801822900772095, -1.314717411994934, -0.45604661107063293, 1.215172529220581, -1.3622825145721436, 0.7480221390724182, -0.4595721364021301, -0.3432157635688782, -0.841585099697113, -0.5461096167564392, -0.4953151047229767, 1.1613672971725464, 0.5971266031265259, -0.043806686997413635, 1.0455313920974731, -0.576517641544342, -0.2626380920410156, -1.9125559329986572, 0.14743919670581818, -0.06073123961687088, 0.5406295657157898, 0.05159793794155121, 0.11078456789255142, -1.3248363733291626, -1.1046435832977295, 1.7622703313827515, 0.12301792204380035, -0.018995633348822594, 0.7124584317207336, 0.8243498206138611, -0.8333467245101929, -0.40717893838882446, 0.7492013573646545, -0.5523141622543335], [-1.6242506504058838, 1.382866621017456, 0.00017865961126517504, 0.8903847932815552, 1.1589645147323608, -1.5748896598815918, 0.35622328519821167, -0.7891693711280823, -1.3080917596817017, 0.12556320428848267, 0.4131486415863037, 0.16182741522789001, 0.33171138167381287, -0.1026584580540657, -0.06163109838962555, 0.5180566310882568, 1.201967477798462, -0.8932591676712036, 0.8329664468765259, -0.0920523852109909, -0.6668786406517029, 1.2535656690597534, -0.12175852060317993, 1.5241519212722778, -0.11176690459251404, -1.5061147212982178, -1.692399501800537, -1.0738673210144043, 0.4388517439365387, -0.6642979979515076, -0.22544065117835999, -1.5099263191223145, 0.09244327247142792, -1.6589289903640747, 0.16871294379234314, -0.16815967857837677, 1.6570850610733032, -0.4791458547115326, -0.19978947937488556, -1.377953052520752, 1.3966882228851318, 0.0633741095662117, -0.284333735704422, -0.018542805686593056, 2.6112797260284424, -1.2813568115234375, 1.205411434173584, 0.11628367006778717, -1.6687091588974, 0.4448682963848114, -1.1396058797836304, -1.1231578588485718, -0.38945069909095764, 1.1669750213623047, 1.633346676826477, -0.08818965405225754, -0.43013671040534973, -0.40883561968803406, 0.5581605434417725, -0.5920177102088928, 1.953122615814209, 0.7298818826675415, -0.45232653617858887, -0.6004767417907715], [-1.2160766124725342, 0.1519167125225067, -0.9605162739753723, -0.7187443971633911, 0.0004377833683975041, -0.8129843473434448, 1.0487533807754517, -1.5303677320480347, -1.3838847875595093, 2.238853931427002, -0.1122174859046936, -0.05265510827302933, -1.278124451637268, -1.844539761543274, 1.3844568729400635, -0.0993112325668335, -0.4225369393825531, -0.44009020924568176, -0.004848124925047159, -0.9701233506202698, -0.26219797134399414, -2.570791721343994, -0.06697256118059158, -1.4262267351150513, 2.0743567943573, 0.12412871420383453, -2.4657366275787354, 1.481393575668335, 1.223517894744873, 0.30159878730773926, 0.6070063710212708, -0.17903192341327667, -0.27364349365234375, 0.272781640291214, -2.00683856010437, -1.6979668140411377, -0.4774267077445984, -1.0401332378387451, 1.527461051940918, -0.3778087794780731, -0.26816076040267944, -1.4255080223083496, 1.4347946643829346, -0.046004291623830795, -1.2751778364181519, -0.8315077424049377, 1.9370388984680176, -1.0009069442749023, -0.4057418406009674, 1.03457772731781, 1.1001125574111938, 0.7358756065368652, -0.23871548473834991, 0.8331454992294312, -0.9489060640335083, 2.020658016204834, -1.4803630113601685, -1.6917855739593506, 0.6720098257064819, 0.2225147783756256, -1.4933419227600098, -0.39746391773223877, 0.29495969414711, 0.8472939729690552], [-0.180669903755188, 1.1304341554641724, 0.2790699601173401, -0.1267457902431488, -0.5157018303871155, -1.5716267824172974, -0.5621532797813416, 0.07008981704711914, 1.3736222982406616, 0.8714932799339294, -0.20516522228717804, 0.6247917413711548, -0.16814225912094116, 1.351466178894043, 0.5581647753715515, -0.5953531265258789, 0.752093493938446, -1.6339510679244995, -1.3504557609558105, -0.16001920402050018, 0.32147911190986633, -0.5716273188591003, -0.9685190916061401, 1.3547310829162598, 0.05486682057380676, -1.1249520778656006, -0.983914852142334, -0.11701902002096176, -1.0675550699234009, 0.22434492409229279, -1.3205549716949463, -0.2033056765794754, -1.0507159233093262, -0.3316345810890198, -0.2608169615268707, -0.6196290254592896, 0.9910196661949158, 0.5403067469596863, -0.2741653323173523, 0.11557567864656448, 0.248396098613739, 1.7664086818695068, 0.5504074096679688, -1.6442936658859253, 0.12398983538150787, 0.6296337842941284, -0.47526514530181885, 0.24522551894187927, 0.2875575125217438, 0.9979867935180664, 1.3138352632522583, -0.5329737663269043, -0.2387419492006302, -0.678167998790741, -0.5069507956504822, 0.40068140625953674, 0.5350112318992615, 1.0566188097000122, 0.256828248500824, 1.7036352157592773, -0.8109594583511353, 0.8895132541656494, 0.9426076412200928, 0.5802962183952332], [0.376382052898407, -0.37070804834365845, 1.264387607574463, 0.36658716201782227, 0.44152647256851196, 0.21340742707252502, 0.5204954743385315, 1.9088059663772583, 0.6729703545570374, -0.7726131081581116, 0.15642264485359192, -1.374497890472412, 0.6376261115074158, -0.702275276184082, 1.4586790800094604, -0.5303177237510681, 1.7114657163619995, 2.0441670417785645, -0.42791256308555603, 0.5056142807006836, -0.2523577809333801, -0.04209807142615318, -1.7291364669799805, 0.6200231313705444, 0.037006836384534836, -0.37410876154899597, -0.523120105266571, -0.0675918385386467, 0.19352521002292633, 0.7224642038345337, 1.3683278560638428, -0.0411284863948822, -0.3025943636894226, -0.16077569127082825, -1.343590259552002, 0.42366352677345276, -1.3793554306030273, -1.6286671161651611, -0.01819906197488308, -1.032021164894104, -1.3410098552703857, 0.6262019872665405, -0.29274940490722656, 0.05642697587609291, 0.5363662242889404, 0.06450243294239044, -0.08815402537584305, 0.1504921019077301, 0.885530412197113, 1.8890259265899658, 0.24518245458602905, -0.40656834840774536, -1.0598357915878296, -1.2883131504058838, -0.12166790664196014, -0.8315436244010925, -2.5595738887786865, -1.3414325714111328, 0.05966208502650261, 0.022228986024856567, -0.6905244588851929, -0.745281994342804, -0.1615501046180725, 1.0190623998641968], [-1.316206693649292, 0.39660826325416565, 1.4813755750656128, -0.2590322196483612, -1.5454217195510864, 0.6499019861221313, 0.24316006898880005, 1.0513100624084473, 0.10564742237329483, 1.7424274682998657, -0.7250163555145264, 0.02389303222298622, -1.3337297439575195, 0.7242003679275513, 1.6076351404190063, -0.5502045154571533, 0.40945881605148315, -0.9106729030609131, 1.3738747835159302, -0.9400762915611267, -1.9319075345993042, -0.2822864055633545, -0.13345353305339813, -0.374052494764328, -0.44729748368263245, 0.09993273764848709, 1.2519980669021606, 1.5725681781768799, 2.0716676712036133, 1.2718535661697388, -0.3716912865638733, -0.4154978394508362, -2.1854896545410156, 0.19989293813705444, 0.5349374413490295, -1.5381546020507812, -0.19864614307880402, -0.2766759991645813, 1.4652734994888306, 1.1084144115447998, 1.1686559915542603, -0.3481329083442688, -1.202548623085022, -0.7161210179328918, 0.021711988374590874, 0.3246942162513733, 0.7439550161361694, 0.7275136709213257, 0.0581948421895504, 1.5452451705932617, 1.2397466897964478, -0.6053862571716309, -0.28591227531433105, -2.328799247741699, 1.931389570236206, 0.1941937655210495, 0.8538382053375244, 1.1271522045135498, -0.42372024059295654, -0.7508448362350464, 1.0744644403457642, -0.051389843225479126, 1.8447911739349365, -0.9348735213279724], [-0.12686941027641296, 0.8171846270561218, -0.061002884060144424, -1.683608055114746, 0.27433326840400696, 0.49848422408103943, -0.6505028605461121, 0.7857693433761597, -2.188751220703125, 1.1951125860214233, -0.15670540928840637, 0.1000545546412468, -0.6080535650253296, -0.37697333097457886, -1.5657509565353394, 0.5287325382232666, -0.5492590665817261, 0.5557799339294434, -1.9183357954025269, -1.6659555435180664, -0.9729858636856079, 0.2771921753883362, 1.1299587488174438, -0.04559694230556488, 0.38765403628349304, -1.2553443908691406, 0.11286138743162155, -0.3383869230747223, 0.78177410364151, 0.5306888818740845, 0.5217439532279968, -0.3724989593029022, -0.7944161295890808, -0.4119683504104614, -0.09867286682128906, -0.057393815368413925, 0.015592805109918118, -0.5754621624946594, 0.2499944269657135, 2.383918523788452, -0.09761891514062881, -0.7780584096908569, -0.7123186588287354, -0.4350275993347168, -1.4229209423065186, -0.5927702784538269, 0.17556869983673096, -0.01832491345703602, -0.9455552101135254, -0.07381179183721542, -0.03806678205728531, -0.12866753339767456, 0.6517291069030762, 1.0176297426223755, -0.7514894604682922, 0.12113116681575775, -0.48856887221336365, -0.1233607679605484, 1.5506608486175537, -0.4271366596221924, 0.9203137755393982, 0.6290761828422546, -0.8137832880020142, -1.0776557922363281], [0.5306140780448914, 0.8528011441230774, -0.4932660162448883, -0.8228225111961365, -0.1500519961118698, -0.5510014891624451, -0.6754040718078613, -0.38694265484809875, -1.0340816974639893, -0.07166839390993118, 0.23825933039188385, -0.07230353355407715, -1.3658677339553833, 0.510934591293335, 0.8969221711158752, -0.3274156451225281, -0.875303328037262, 1.392020344734192, 0.23061352968215942, 1.777156114578247, 1.3137214183807373, 0.9806295037269592, 1.1948785781860352, -0.5264860391616821, -0.43652787804603577, 0.8069480061531067, -0.6447222828865051, -0.12816768884658813, 1.1847119331359863, -0.03717793896794319, 1.521984577178955, -0.5281385779380798, 0.7258338928222656, -0.13513363897800446, -1.0508556365966797, -1.878255009651184, -1.2600455284118652, 0.7482354044914246, 2.2534639835357666, 0.8818984031677246, 0.5691956281661987, 0.29675331711769104, 1.7499308586120605, -0.7871026396751404, 1.930808424949646, 0.3709467947483063, 0.8554016351699829, 1.2953373193740845, 0.21041642129421234, -1.6169309616088867, 0.12515230476856232, 1.58021879196167, 1.9967225790023804, 0.8606085777282715, 0.9651504158973694, 0.001607935526408255, 0.334300696849823, 0.2851538062095642, -0.05354837700724602, 0.9811029434204102, -0.20380310714244843, 0.517751932144165, 1.2727160453796387, -0.9970711469650269], [-0.1089702844619751, -0.14475969970226288, -0.9502519369125366, 0.8325520157814026, 2.6037087440490723, -1.2755553722381592, 0.495319664478302, 1.6177091598510742, 0.7690663933753967, -0.4338361918926239, -1.0232001543045044, 1.0366206169128418, -1.546350359916687, -1.0903208255767822, 0.5356873273849487, 0.7011776566505432, 1.418174386024475, 1.3883551359176636, 0.6766787767410278, 0.5909234285354614, 0.2778880298137665, 0.26750102639198303, 0.45883169770240784, 1.9273978471755981, 0.5225945115089417, 0.03972603380680084, 1.500191569328308, 0.10127401351928711, -1.2595806121826172, -0.6787168979644775, 0.5738875269889832, 2.0533926486968994, -3.1253459453582764, -0.524459719657898, -1.1637612581253052, 0.3788450360298157, -2.025672674179077, 0.141459122300148, 0.290597528219223, 0.897213876247406, 0.025416916236281395, 0.46870291233062744, 0.7679995894432068, 0.21355994045734406, 1.017068862915039, 0.7948376536369324, -0.7011215686798096, -1.0520859956741333, 0.3632918894290924, -1.0591005086898804, 0.12934993207454681, 0.3471432626247406, 0.6754175424575806, -0.05649728700518608, -1.763315200805664, -0.040478795766830444, 0.2941313683986664, -0.26916828751564026, 0.3095000684261322, -2.1331660747528076, -0.8796126842498779, -0.7714475989341736, -0.5152551531791687, 2.1887314319610596], [1.1898584365844727, 0.5875613689422607, 0.03316458314657211, -1.6460509300231934, 1.346002459526062, 0.45958802103996277, -1.0694907903671265, 0.24794040620326996, -1.431986927986145, 0.04304051399230957, -1.0987263917922974, -0.7483142614364624, 0.8258535861968994, 0.5154452323913574, -0.4766288995742798, -1.8704774379730225, -1.1735187768936157, 1.4693554639816284, -1.220611333847046, 1.27276611328125, -0.9278754591941833, 0.7051791548728943, -0.0120859919115901, 1.3006664514541626, 0.6975619792938232, -1.573889136314392, 0.03843531757593155, 0.6025159955024719, 0.260717511177063, 0.345267653465271, 1.2410295009613037, 0.9071099758148193, -1.5864485502243042, 0.4525175988674164, -0.8760986328125, 0.16260991990566254, -0.49893516302108765, 1.2590867280960083, 0.6158940196037292, -0.2507002353668213, -1.364490270614624, 0.4196408689022064, 0.21680574119091034, 1.3373757600784302, -0.22344611585140228, -1.0889856815338135, 0.7535061836242676, 1.2746541500091553, -2.0048224925994873, -0.8827170133590698, -0.39422357082366943, 2.81221604347229, 1.6034204959869385, 0.3628654479980469, -0.22818692028522491, 0.25412997603416443, 1.785833477973938, -0.960623562335968, -0.04982142895460129, -0.019278690218925476, -0.5797070264816284, -1.0646852254867554, 0.4889144003391266, -0.7018632888793945], [1.659522294998169, -0.6208308339118958, 1.1476261615753174, -0.12808531522750854, 0.697124719619751, 1.2965047359466553, -0.757965087890625, -0.2654297649860382, -1.7638846635818481, -0.4943591356277466, 0.3281879127025604, -0.03541719913482666, 1.7755885124206543, -0.6905247569084167, 0.17616865038871765, 0.11177461594343185, -0.48240676522254944, -2.114673376083374, 0.8278213143348694, 1.20589017868042, 1.4614427089691162, 1.091056227684021, 2.267643928527832, 0.09466011822223663, 1.2168265581130981, 1.1288745403289795, 0.7710297703742981, 1.6318129301071167, -0.27045923471450806, -0.8707520961761475, 0.8365505337715149, -1.0608752965927124, 0.7370741963386536, -0.9255519509315491, -1.6129037141799927, -0.10564953833818436, -0.8876588344573975, -0.4816252291202545, -1.7653676271438599, -1.1830024719238281, -0.9140381813049316, -0.40995433926582336, -0.1411120891571045, 1.7169370651245117, -1.7546296119689941, -0.17238306999206543, 0.7337172627449036, 1.2044601440429688, 0.2480388730764389, 0.17547164857387543, 0.8189848065376282, -0.4062912166118622, -1.8492836952209473, 0.7298008799552917, -0.9105274081230164, -0.025957707315683365, 0.6748930811882019, 0.26853442192077637, -0.7109785079956055, -1.1476444005966187, -2.6211493015289307, -1.6247398853302002, 0.5163533091545105, -0.27213114500045776], [-0.29543372988700867, -0.49637147784233093, 0.37893402576446533, 0.12639085948467255, -0.24510319530963898, 1.1982934474945068, -0.7672038078308105, 0.03484029695391655, -0.12552767992019653, -0.46368440985679626, 1.288834810256958, 0.11285372078418732, 0.218169167637825, -0.437032550573349, 1.6201328039169312, -0.303846150636673, -0.6826529502868652, -0.05613217130303383, -0.09194165468215942, 1.0356265306472778, -0.6631011366844177, -0.38533782958984375, 0.7157530188560486, 0.7471185922622681, 1.364866852760315, -0.11613790690898895, -1.6168150901794434, -1.275955080986023, -0.08311017602682114, 1.5391641855239868, 0.5792650580406189, 0.3054652214050293, 1.3285797834396362, -0.11020225286483765, -1.3626528978347778, -0.962354302406311, -0.0936543419957161, 0.18862469494342804, -0.3014761507511139, 0.002825219416990876, -0.4450239837169647, -1.206899642944336, 0.6686697602272034, 0.8937199711799622, -1.4718014001846313, 0.406749963760376, -0.4916255474090576, -1.0144778490066528, -0.3483031988143921, 0.6352742314338684, 0.20853856205940247, 1.4417380094528198, 1.5487295389175415, 0.4637244641780853, -0.9022368788719177, 2.558706283569336, -0.5291906595230103, 0.2768755257129669, -1.1515800952911377, 0.08340995013713837, 0.06985340267419815, -1.8528788089752197, -1.9440397024154663, -1.4832576513290405], [-0.15886852145195007, 1.6444001197814941, 0.3352434039115906, 0.3121379315853119, 1.307498812675476, 0.993013322353363, -0.645068347454071, 0.15470899641513824, 0.4848255515098572, -1.0114336013793945, 0.44919630885124207, 0.3658059239387512, -1.0267637968063354, -0.2986607253551483, -0.751686692237854, 0.21898065507411957, 0.054405052214860916, 0.7784983515739441, 0.6633500456809998, 0.42151689529418945, -0.7967904210090637, -0.5983067750930786, 0.0178510881960392, 0.1112775132060051, -1.095794916152954, -1.43740975856781, -0.5564369559288025, -1.0105459690093994, 0.16612477600574493, -0.8257120847702026, -0.6367100477218628, 0.2630935311317444, 0.3780513107776642, -0.793901801109314, 0.5236195921897888, -1.5258530378341675, 0.6323060989379883, 1.1694517135620117, -0.6000551581382751, 0.41401395201683044, -0.08148820698261261, -0.27785632014274597, -0.3428393006324768, 0.22011631727218628, -1.018320918083191, 0.997223436832428, -0.3199978768825531, 0.4302966296672821, -0.08185936510562897, 0.1580575406551361, -0.6840452551841736, 0.685825765132904, 0.4859766662120819, 0.6693289279937744, -0.02676832117140293, 0.7644466757774353, -0.4691363573074341, 0.3580988347530365, 0.3502470850944519, 0.12321487814188004, 0.32417312264442444, 0.03783999755978584, -0.5891729593276978, -1.2252603769302368], [0.08703584969043732, 1.4475033283233643, -0.3566853106021881, -1.491233468055725, -1.2308183908462524, 0.5215165019035339, 1.4040433168411255, -0.5435448288917542, 0.9926483631134033, 2.1791257858276367, -1.4681050777435303, 1.860750675201416, -0.20309479534626007, -0.11131119728088379, -0.01513144001364708, 0.03382159397006035, -1.0360944271087646, -0.42258504033088684, -1.2984983921051025, -1.8806917667388916, -0.026147587224841118, 0.7958362698554993, 1.5827242136001587, 1.3404492139816284, -0.302015483379364, -0.8785681128501892, -1.4642753601074219, -1.4499520063400269, -0.5273982286453247, -2.5442962646484375, -0.3560064435005188, -1.6123627424240112, -0.31229469180107117, -0.9157318472862244, 1.8672935962677002, -0.4610469937324524, 1.1031850576400757, -0.5633415579795837, 1.221321702003479, -1.6578221321105957, -0.3283691704273224, 1.2228869199752808, -0.44929617643356323, -0.583022952079773, 0.8120725750923157, -0.8931776881217957, -0.8285102248191833, 0.9892627000808716, 0.23227839171886444, -2.2975826263427734, 0.18682916462421417, 0.12231576442718506, -0.4831800162792206, 0.7288390398025513, -1.0599391460418701, 0.4336627125740051, 0.6994613409042358, 1.7796326875686646, 1.0733721256256104, 0.5007445812225342, -0.6424044370651245, 1.6287646293640137, -0.01965472660958767, 0.2944558560848236], [-0.47105875611305237, -0.12670700252056122, -1.9790090322494507, 0.24442754685878754, -0.7050334215164185, -1.0706764459609985, -2.1788220405578613, -1.2650070190429688, 0.13743607699871063, -1.0560953617095947, 0.3126198947429657, -0.023257989436388016, 0.3641963601112366, 0.15953071415424347, 1.5656704902648926, -0.22810976207256317, -1.2893242835998535, 1.1123192310333252, -0.36830374598503113, 1.265548825263977, 1.2041585445404053, -0.47408556938171387, -0.45414644479751587, -0.442268967628479, 1.3196239471435547, -2.5191993713378906, -1.187715768814087, 0.9290003180503845, 1.866899847984314, -1.127629280090332, -0.866860568523407, 0.5288610458374023, 0.5991625785827637, 0.29315242171287537, 0.2485472857952118, -0.1356758028268814, -0.19301187992095947, 1.7726157903671265, -1.3852297067642212, 0.13414373993873596, 0.36320459842681885, 2.9040367603302, 0.9611846804618835, -0.8678766489028931, 0.7683405876159668, 0.13608898222446442, 0.24088154733181, 1.1698788404464722, 0.8504167199134827, -0.4608970582485199, 1.0582200288772583, 0.7960649728775024, 0.18020623922348022, 0.6271065473556519, -0.8593497276306152, 0.19817866384983063, 0.07329612970352173, 2.4476382732391357, -1.2870157957077026, -0.734670877456665, -1.2097141742706299, -0.520672619342804, 0.11985455453395844, 0.5199826955795288], [1.4167205095291138, 1.4465179443359375, 0.47395405173301697, -0.42547300457954407, 1.1087851524353027, -1.8229527473449707, -0.5653355121612549, 0.5972387790679932, -1.246448040008545, -0.15778960287570953, 0.7544678449630737, -0.09542248398065567, 0.8191931247711182, 0.3057866096496582, 2.418735980987549, 0.9451150298118591, -0.5026654005050659, -2.152716636657715, -0.32919955253601074, 0.44591227173805237, 1.8788251876831055, -1.045698881149292, 1.409192442893982, -0.01820564642548561, 3.161015748977661, -0.5460104942321777, -1.3869541883468628, -2.092458724975586, -1.2800908088684082, 0.2690081298351288, 1.349499225616455, -1.045859694480896, -0.06960811465978622, 0.0872846320271492, -0.5803240537643433, 0.8645344972610474, -0.2611994445323944, 1.4963467121124268, 1.0036704540252686, 0.41943466663360596, -0.6937271356582642, -0.0497659333050251, 0.7751172184944153, 1.8174128532409668, -1.053138017654419, -0.16879254579544067, 0.1954742670059204, 0.028805088251829147, -0.42010465264320374, 1.2703497409820557, 1.615350365638733, -1.386476993560791, -0.9699214696884155, -0.5433776378631592, 0.808934211730957, -0.0091925747692585, -0.09556461870670319, 0.11148272454738617, 0.1593262404203415, 1.1563071012496948, -1.079030990600586, -0.6906258463859558, -0.12674064934253693, 1.078310251235962], [-1.1340564489364624, 0.3292965590953827, -1.131043791770935, -0.3726922273635864, 0.21597722172737122, 0.3442634046077728, -0.009428979828953743, 0.2481534779071808, 0.47481802105903625, -0.8745116591453552, 0.5307603478431702, 0.07981732487678528, -0.4638160765171051, 0.6778635382652283, -0.6947554349899292, 0.5648581385612488, 1.1609190702438354, -0.12197840213775635, -0.525982141494751, -1.2242873907089233, 0.41735172271728516, 1.7308430671691895, -0.3566964268684387, 0.4896860420703888, 0.8721275925636292, -0.6142645478248596, 0.9786731600761414, -0.10024520754814148, -1.6443225145339966, -0.1300872266292572, -0.677199125289917, 0.5701320767402649, -0.8129664063453674, 1.7250349521636963, -1.4689745903015137, 1.1567822694778442, 0.5307046175003052, 0.1083923876285553, -1.3189148902893066, 1.3719310760498047, -0.32381001114845276, 0.5844414830207825, 1.1469714641571045, 1.4151841402053833, 0.5582089424133301, 0.9142632484436035, -0.2299787998199463, -1.0323703289031982, 1.0272414684295654, -0.7221837639808655, 0.40336504578590393, -0.7993202805519104, -0.6666535139083862, -0.4897668957710266, 0.15297916531562805, 0.8387472033500671, -1.7454897165298462, -0.08551180362701416, -0.6991578936576843, 0.8208429217338562, 0.34182843565940857, 0.9088786244392395, 0.3733065724372864, 1.5227675437927246], [0.22516599297523499, 0.942008376121521, -0.27515581250190735, 1.5599840879440308, 0.2650827169418335, -0.21929584443569183, -0.769902229309082, 1.0965197086334229, 0.44262170791625977, -0.7696371078491211, -0.07885249704122543, 0.6451517343521118, 1.2737621068954468, 0.006490248721092939, 0.330783486366272, 2.025838613510132, 1.996126413345337, -0.23611848056316376, 1.0448527336120605, -1.2846651077270508, 0.6690673232078552, -0.2830185294151306, -0.3102579414844513, -1.371675968170166, 0.3553500175476074, 0.23461046814918518, 0.9936142563819885, -0.8750125169754028, 0.2944287955760956, 0.0821811705827713, 0.3293604552745819, -0.8985633254051208, -1.2305680513381958, -0.6353505849838257, -0.16647377610206604, -1.198387861251831, 1.156104564666748, -0.898331344127655, -0.26665541529655457, -0.7408407330513, -0.17750456929206848, 1.0017049312591553, 0.9284355640411377, -0.05189727619290352, 2.3289504051208496, 0.487602561712265, 0.3260692358016968, 0.5048747658729553, -1.9129008054733276, -0.5085877776145935, 1.1364636421203613, 0.2718232572078705, 0.2577120363712311, -0.32576122879981995, 1.998079776763916, 1.6971368789672852, -1.353918194770813, 0.9297356009483337, 3.2893638610839844, -0.521137535572052, -1.8651869297027588, -0.2979702651500702, 1.0694704055786133, -1.6489943265914917], [1.1422483921051025, 1.2317856550216675, -1.3044599294662476, -0.4813094735145569, -1.4178985357284546, -1.068044900894165, 1.7164075374603271, 0.3161380887031555, 1.9090766906738281, 0.2025240808725357, 0.9688254594802856, -0.06792160868644714, 0.493291974067688, 0.11746235191822052, 0.7210642099380493, -0.22949430346488953, 0.5872448086738586, -0.8894498348236084, 0.8406074047088623, 0.06418278068304062, 1.130671501159668, -0.18300199508666992, 0.4711747169494629, 2.6602892875671387, 0.02611701749265194, 1.0394662618637085, 0.6731948256492615, 0.4536007344722748, 0.6480392217636108, 0.8355993628501892, -0.6024964451789856, 0.5338634252548218, 1.2135978937149048, 0.46726539731025696, -0.28004977107048035, -0.09410158544778824, 0.7049005031585693, 0.38668012619018555, 1.8445663452148438, 0.14837634563446045, 1.3602527379989624, 0.7331759333610535, 0.3153976798057556, 0.9978415369987488, 0.012822798453271389, -0.6851705312728882, -0.3551826775074005, 0.9068630337715149, -0.9931431412696838, -0.5399719476699829, 0.41077402234077454, -0.8049159646034241, 2.0890133380889893, -0.6403691172599792, -0.913920521736145, -0.05390303581953049, -0.43478667736053467, -2.186455726623535, -0.8314594626426697, 0.45351162552833557, -0.8678922653198242, 2.54195237159729, 0.3587881326675415, 0.9170958399772644], [-0.931098461151123, 1.4165087938308716, -0.9296305179595947, 0.21235276758670807, -0.14574921131134033, 0.6750812530517578, 0.9545176029205322, -0.4962766468524933, 0.6217936277389526, -1.892702579498291, 1.108761191368103, -1.0439262390136719, -0.8939159512519836, -0.7369320392608643, -0.03152631223201752, 0.6494885683059692, -1.129970908164978, 1.3090076446533203, 1.1503303050994873, -0.6926812529563904, 0.2905735969543457, -0.7312619686126709, 0.998211681842804, 0.1060178279876709, 0.8423979878425598, 0.480053186416626, 1.0654984712600708, -0.8269691467285156, 0.5524786710739136, -0.07221024483442307, 0.1865060180425644, -0.31586694717407227, 0.40037524700164795, 0.8615016341209412, -0.9066298007965088, -0.36470115184783936, -0.3436054587364197, -2.1059415340423584, 1.2767486572265625, 1.1058158874511719, -0.22157242894172668, 0.1830233782529831, 2.005695343017578, 0.404898464679718, -0.012444782070815563, 0.013182047754526138, -0.0384378582239151, 0.8080772757530212, 0.5642075538635254, -2.772770643234253, 1.456860899925232, -1.0344034433364868, -0.0647515207529068, 2.177189588546753, 1.4045987129211426, -0.12578651309013367, -1.3517227172851562, 0.32748860120773315, -0.7668874263763428, -0.7925007343292236, -0.023041432723402977, 0.7898514866828918, -0.6353142857551575, -0.47937965393066406], [-0.4554106593132019, 0.6902866959571838, -0.9897903800010681, 1.5362675189971924, -0.23183709383010864, -1.082053542137146, 0.20024065673351288, 0.3495613634586334, -1.0322728157043457, -1.224075198173523, 0.9993968605995178, -0.3811177611351013, -0.09383796900510788, -1.4944396018981934, -0.6270008087158203, -0.984621524810791, -1.4896527528762817, -0.2561165392398834, -2.5919270515441895, 0.3277903199195862, -1.1840602159500122, -0.5314381122589111, -2.017768621444702, -0.37551185488700867, 0.516718864440918, -0.22066515684127808, -0.279864102602005, -1.4474908113479614, 0.6502143144607544, 0.9112321138381958, 2.1073710918426514, -1.6685391664505005, 0.03600592538714409, -0.31144633889198303, 0.296332448720932, -1.0363866090774536, 1.279871940612793, -1.3895505666732788, 0.8154040575027466, 0.46969082951545715, 2.063075065612793, 0.2542760670185089, 0.8380488157272339, -0.5387879610061646, 0.05255218967795372, 0.9535031914710999, 0.623788595199585, -0.8562994599342346, 0.969992458820343, -2.3021676540374756, 1.53249990940094, 1.080693244934082, 1.406212568283081, -0.15583699941635132, 0.4751082956790924, 1.8199599981307983, -1.074908971786499, -0.24535220861434937, 1.2460330724716187, 0.11054597049951553, -0.47238507866859436, 1.2286492586135864, 2.56493878364563, 3.1215627193450928], [1.7998847961425781, 0.5993401408195496, 0.3217133581638336, -1.1265368461608887, 0.38743311166763306, -0.9765521287918091, -0.39770805835723877, 0.3883295953273773, -0.05941407009959221, 0.4912641942501068, 0.5388498902320862, 0.5086820125579834, -0.6657435297966003, -0.09464489668607712, 1.2047021389007568, 0.8114016056060791, -1.0804822444915771, 0.31585758924484253, -0.03450494632124901, 0.5570052266120911, 0.20140956342220306, 0.3028394281864166, -0.13539251685142517, -0.6996780037879944, 1.2398601770401, 1.2097722291946411, -0.38629651069641113, -0.39741745591163635, -1.3540645837783813, -0.5174691677093506, 0.26792630553245544, 0.03182320296764374, -0.12092499434947968, -1.4588721990585327, -0.26521483063697815, -1.2723016738891602, -0.8175806999206543, -1.781835675239563, -1.0718578100204468, -0.39979782700538635, 1.6704455614089966, 0.2723231911659241, -1.4136855602264404, -0.4057525098323822, 0.8004436492919922, 2.3769819736480713, -0.13699723780155182, 0.008246848359704018, -0.008106710389256477, -0.34063097834587097, 0.6785732507705688, 1.036941409111023, -0.6120016574859619, -0.8451163172721863, 0.3324708938598633, -1.2350430488586426, -1.1992614269256592, -0.9642140865325928, -0.20686550438404083, 0.320147305727005, -0.6171246767044067, 0.6530420780181885, 0.0690891444683075, -0.6527832746505737], [-0.7351260185241699, 0.6073142290115356, 0.48394709825515747, -0.5673051476478577, -0.43368813395500183, -0.49856898188591003, 0.6539179086685181, -1.35438072681427, -0.8909378051757812, -0.8702235817909241, -0.3697502911090851, -0.7946397662162781, -0.7221526503562927, 0.3893837034702301, -1.2215036153793335, 0.07288524508476257, -1.6274592876434326, 0.14454035460948944, -1.0537006855010986, 1.7601253986358643, -0.3434828221797943, -1.152045488357544, -1.03482985496521, -1.1274337768554688, -1.8980438709259033, 0.7616297006607056, 1.313451886177063, 0.7437876462936401, -0.11695107817649841, -1.302335500717163, 0.548719584941864, 1.1568615436553955, -0.7528568506240845, -0.02762093022465706, -0.5070526599884033, -1.888450264930725, -0.6413280963897705, 0.9023112654685974, 2.4339358806610107, 0.09366872161626816, -0.9391969442367554, -1.3555227518081665, -0.435849130153656, 1.6417956352233887, 0.21709921956062317, -1.4404628276824951, 1.5091185569763184, -0.19427496194839478, -1.9608402252197266, 0.7898635864257812, 1.8570479154586792, 2.762425184249878, -0.7136033773422241, 0.45771461725234985, 0.09384968876838684, 0.5690767765045166, -0.5945001244544983, 0.06320656836032867, -0.6376058459281921, 0.4667486548423767, -1.5720051527023315, 0.3342224955558777, -0.08156643807888031, -0.46557408571243286], [-0.3862003982067108, -1.526508092880249, -2.2734575271606445, -0.7748158574104309, -2.862034797668457, 0.07453101128339767, -0.051598623394966125, -0.5698555111885071, 1.3447694778442383, -0.7579887509346008, -2.685518503189087, -0.8114815950393677, -1.9452342987060547, 1.9396653175354004, -0.07519257068634033, 0.11013449728488922, 0.4665481746196747, 0.38936758041381836, -0.5755061507225037, -0.8129744529724121, 0.26038938760757446, -1.09927237033844, -1.378383994102478, 0.7125315070152283, 1.2746753692626953, 0.7426901459693909, 1.1969157457351685, -1.5316368341445923, 0.7387553453445435, 0.19709548354148865, -0.10457069426774979, 0.05810913071036339, -0.45571818947792053, 0.7903388738632202, 0.5978400707244873, -0.33013197779655457, -0.9492345452308655, 1.9844385385513306, -1.2338038682937622, -0.4762941896915436, -0.8003338575363159, -0.3953305780887604, -0.8139934539794922, -0.5849859714508057, 0.43677452206611633, -0.8053088784217834, 1.3496841192245483, -0.48892244696617126, 0.1612401008605957, 0.2743000388145447, 2.1971592903137207, 0.5135030150413513, 0.2720988094806671, -1.4166711568832397, 1.1368147134780884, -1.69332754611969, -0.35425642132759094, 2.1821460723876953, 0.1894918531179428, -0.10303059965372086, 0.7780839204788208, 0.8545011878013611, -0.2218315750360489, -0.27454522252082825], [-0.18076464533805847, -0.5475796461105347, 1.1907799243927002, 0.7140780091285706, -1.4620250463485718, 1.2935479879379272, -1.7740696668624878, -1.6358259916305542, 0.5166925191879272, -0.7576017379760742, 0.442818820476532, 0.346671998500824, -0.39742016792297363, -1.2816756963729858, -0.9956610798835754, -0.4353134334087372, -3.0997700691223145, 1.3632237911224365, -0.6414472460746765, -1.6111811399459839, 1.4755762815475464, -0.055670179426670074, 0.9478222131729126, -0.5900915861129761, 2.258418321609497, 0.3075261116027832, 1.2728039026260376, 1.6718894243240356, -1.375755786895752, -0.263884961605072, -0.7708011865615845, -0.07589054107666016, -0.38507458567619324, 2.147373676300049, 0.6011794209480286, -0.2411448210477829, 0.016911078244447708, 2.07326340675354, -1.1584630012512207, 2.0225110054016113, -0.47796234488487244, -1.0843108892440796, -1.8048640489578247, 0.2690103054046631, -0.4380338788032532, -0.4040572941303253, 1.0053582191467285, 1.2327858209609985, -1.2099798917770386, 0.04399311542510986, 0.36624985933303833, 0.5700044631958008, 0.33828097581863403, -0.7982574105262756, -0.605542778968811, 2.138014554977417, 0.460879385471344, -0.13507165014743805, -0.9498510360717773, 0.04083835333585739, -1.08134925365448, -0.49775364995002747, -0.5178782343864441, 0.3377907872200012], [-0.7936002612113953, 0.9700609445571899, 0.7212954759597778, 0.4375018775463104, -0.1547987461090088, 0.05345218628644943, -0.3523164987564087, -0.001952665508724749, -0.22094856202602386, -1.53574538230896, -0.02433801256120205, 0.11684528738260269, -0.9071410298347473, -0.19337642192840576, -0.0990835577249527, 0.7089835405349731, 1.0315570831298828, -1.263353705406189, 1.5182615518569946, 1.1118433475494385, -3.051398277282715, -0.4955918788909912, -1.7502610683441162, -0.0650726854801178, -0.24705831706523895, 0.32370370626449585, 1.2434163093566895, 1.5928484201431274, -0.45448121428489685, 0.17430242896080017, 1.631282925605774, -0.9993020296096802, -0.2832820415496826, 1.040924310684204, -0.28936469554901123, -1.1454038619995117, 1.552059292793274, 1.93120276927948, -0.012542366981506348, -0.5521873831748962, 1.761301875114441, -1.3552836179733276, 0.7025455236434937, 1.6720495223999023, -2.035490036010742, -0.8457406759262085, -1.9075734615325928, 0.4145563244819641, 0.2701038718223572, -0.254524290561676, -1.692765712738037, -0.06264427304267883, -0.8066194653511047, -0.41161656379699707, -0.3048337697982788, -0.7882770895957947, 2.1652963161468506, -0.2126079797744751, 1.1792348623275757, -1.128043293952942, 0.6836549043655396, -0.21956846117973328, 0.3755223751068115, -1.2559653520584106], [-0.43006470799446106, 0.5328512787818909, 0.15388745069503784, -1.608408808708191, 0.5215497016906738, -1.2332974672317505, -2.030951976776123, -1.1397180557250977, -3.1047019958496094, 1.2827476263046265, 0.004372694995254278, 0.5883945226669312, 0.2910561263561249, -0.4939821660518646, -0.5875959396362305, 0.8486403226852417, 0.5874081254005432, -0.9534430503845215, 0.11315930634737015, 0.41486114263534546, 1.6272348165512085, 1.1637053489685059, 1.1301716566085815, 0.1698063313961029, -2.2628793716430664, -0.7122140526771545, -1.0701470375061035, -0.5311328172683716, -0.7252611517906189, 0.6657282114028931, 1.2821059226989746, -0.15042255818843842, -0.9109840989112854, -0.5829911231994629, -0.012286808341741562, 1.3293709754943848, 0.22722750902175903, 0.34078434109687805, -0.13824445009231567, 0.40242376923561096, 1.075387954711914, -0.21161024272441864, 2.4686217308044434, 1.13177490234375, -0.3872264325618744, -1.024841070175171, -0.40698957443237305, 0.1914939433336258, 0.39512723684310913, -0.5420352816581726, 1.54328191280365, 0.6199641823768616, -0.19939033687114716, 1.6464024782180786, 0.7935338616371155, -0.48002099990844727, -0.18171332776546478, -0.5847986340522766, -0.26636433601379395, 0.331978440284729, -0.6563993096351624, -0.14217640459537506, 1.836613416671753, 0.31881552934646606], [-1.5226448774337769, -0.7829865217208862, 2.148607015609741, 0.059427861124277115, -1.198526382446289, -0.41700053215026855, -0.405638724565506, -0.5179935693740845, -0.7255373001098633, 0.9874647855758667, 0.21696828305721283, 0.8975062370300293, 0.10580768436193466, 1.0393768548965454, -1.296716570854187, -1.7608675956726074, 2.2264747619628906, 0.8203107714653015, 0.4859063923358917, 1.1897939443588257, -0.6353862881660461, 0.7197460532188416, -1.280232310295105, -0.6725485324859619, -0.5532522797584534, 0.06345728784799576, -0.6617586016654968, -1.8290917873382568, 0.07172787189483643, -1.277036190032959, -0.18123410642147064, 1.377664566040039, 0.7693678140640259, 0.5494769215583801, 2.3974392414093018, -0.1622190773487091, 1.8920350074768066, 0.3037773072719574, 1.6113051176071167, 1.7251908779144287, -0.41995710134506226, 0.42930302023887634, 0.1117943525314331, 0.2990012764930725, -0.9078993201255798, -0.47231876850128174, -1.3989959955215454, 0.2778232991695404, 0.6136963367462158, 0.3366437554359436, 0.4272511303424835, -0.40452301502227783, -0.4005221724510193, 0.3967536985874176, 0.6085641980171204, 0.6313375234603882, -0.4440607726573944, -1.7692443132400513, 0.19667412340641022, -1.3978685140609741, 0.014601979404687881, -1.3294801712036133, 0.1923094242811203, -0.2734583914279938], [1.1462509632110596, -0.00736742839217186, 0.6833741664886475, -1.1345124244689941, 2.1383914947509766, -0.4125474989414215, 1.603870153427124, 1.15232253074646, 0.4822346568107605, 0.9940911531448364, -1.4929975271224976, 1.022731065750122, 1.2239303588867188, 0.8287338614463806, -1.189675211906433, 0.5609671473503113, -0.7886167764663696, -0.4218914806842804, -0.830044150352478, 1.1644957065582275, -0.0470307432115078, 1.5073161125183105, -0.47410881519317627, 2.1565957069396973, 0.4816755950450897, 0.8116413950920105, -0.5904093384742737, -0.7057715058326721, -0.9104637503623962, 0.4887925684452057, -0.14674033224582672, -1.3827879428863525, 1.0733879804611206, 0.5862494707107544, 0.288408100605011, -0.409000039100647, 0.8256693482398987, -0.28670966625213623, -0.5077855587005615, -0.5080394148826599, 1.2768102884292603, -0.7919881343841553, -0.6107755303382874, -0.01525973342359066, 1.393640160560608, -0.7232269048690796, -0.22298189997673035, -0.41287925839424133, 0.568816065788269, -0.3413984477519989, 0.0581374354660511, 0.8966179490089417, -1.6749178171157837, -0.04113500565290451, 0.11276350170373917, 1.0580658912658691, 0.03811303526163101, -0.027673514559864998, -0.5399137735366821, 1.657621145248413, 0.7354762554168701, 0.45229941606521606, 0.6132844686508179, -1.063881754875183], [-1.5150338411331177, 1.3013513088226318, -0.2483479380607605, 0.3877870738506317, -1.1563901901245117, -1.1515140533447266, 1.4343247413635254, -1.0123450756072998, 0.788352370262146, -1.7799863815307617, -1.9982715845108032, -0.09193811565637589, 0.0035353577695786953, -0.6079611778259277, 0.38628897070884705, 1.3980658054351807, 2.127619743347168, 0.2630639672279358, 1.4001951217651367, 0.9348925352096558, 0.22958500683307648, -1.4400795698165894, 0.7223917841911316, 0.5737022161483765, -0.20692797005176544, -0.18702936172485352, 0.4912365674972534, -0.8828220963478088, -0.7228869795799255, 0.6666908264160156, -0.5980530381202698, 0.44694697856903076, -1.367311716079712, -1.693274974822998, 0.8137199282646179, -2.2291982173919678, -1.0264796018600464, -0.13090397417545319, 0.5851041078567505, -0.07102157920598984, -1.383193850517273, -0.9524353742599487, 0.7780696153640747, -1.2728005647659302, -2.489886999130249, 1.9274952411651611, -1.6096746921539307, -1.0474635362625122, -0.20995739102363586, -0.26010069251060486, -0.8270041346549988, 0.2004777491092682, -0.8106060028076172, -0.7679147124290466, -1.3553199768066406, 0.47603321075439453, 0.2236538827419281, 0.20000217854976654, -0.2996169328689575, -0.15851475298404694, 2.9510624408721924, -0.50586998462677, -0.6846444606781006, -0.3242436945438385], [-0.46035337448120117, -0.4259258508682251, -1.0507333278656006, 0.40110403299331665, 1.3897507190704346, 0.194111630320549, 0.11932792514562607, -0.18155212700366974, -0.7512534260749817, 0.5679784417152405, -0.6780257821083069, 0.21168524026870728, 0.0662708654999733, 0.4637475907802582, -0.3494051396846771, -1.0001347064971924, 0.42815589904785156, -1.611535906791687, 3.0768229961395264, 2.249814748764038, -0.3539750576019287, -1.4151289463043213, -0.7287226915359497, 0.6177274584770203, 0.5110260844230652, 0.8126523494720459, -0.05385490506887436, -0.8214991092681885, 1.275128960609436, -0.06966769695281982, -0.3108674883842468, 0.9509978890419006, 0.126988485455513, 0.20708419382572174, -0.11573212593793869, 1.0500984191894531, -0.6615749597549438, 1.2476487159729004, 0.018373161554336548, -0.17764578759670258, -1.3478866815567017, 0.13640184700489044, 0.5265435576438904, 0.17652444541454315, -0.5911948680877686, 1.9724065065383911, 0.9172742366790771, -0.6497326493263245, 1.2879118919372559, -0.3549294173717499, -1.169561505317688, -1.29100501537323, -1.4062135219573975, -0.07050735503435135, -1.3580728769302368, 0.6172462701797485, -1.9446102380752563, -2.9146571159362793, -0.021254966035485268, 1.1543447971343994, 1.9206753969192505, -1.3582744598388672, -0.3587075471878052, -0.47073274850845337], [0.030465789139270782, 1.0219616889953613, 0.14364071190357208, -0.4990203082561493, 1.3071714639663696, 0.8755145072937012, 0.4830053448677063, -0.018726807087659836, -0.609714150428772, 0.7982919216156006, -0.060538023710250854, 1.2450714111328125, 0.25306203961372375, 1.227463722229004, -0.7059615254402161, -0.23346459865570068, 0.013656487688422203, -1.1205838918685913, 1.8207910060882568, -0.6277734041213989, 0.5898225903511047, -0.6994605660438538, -0.5172751545906067, -0.20740242302417755, -0.32103100419044495, -0.9845790863037109, -0.1620226800441742, -0.8285167217254639, 0.2742578983306885, 0.711946427822113, 1.3114591836929321, -0.3904390037059784, 1.6583101749420166, 0.25290602445602417, -1.4533615112304688, 0.7318073511123657, -0.6640012860298157, -2.3702967166900635, 0.16452041268348694, 1.1866530179977417, 0.756666362285614, -1.5884679555892944, -0.5458483695983887, 0.1791340410709381, -0.8472086191177368, -0.7990190386772156, -0.13268165290355682, -1.5439718961715698, -1.1658860445022583, 1.3128118515014648, 0.9405869245529175, 0.8004096746444702, -0.5304769277572632, -0.07665816694498062, 0.022925151512026787, -1.0060170888900757, -0.3199046552181244, -0.5916834473609924, 0.47557878494262695, -1.0393495559692383, 0.9316326379776001, -1.5509904623031616, 0.09114640951156616, 1.0289061069488525], [1.789557933807373, -1.2489013671875, -0.7341890335083008, -0.9558185338973999, -0.07263939827680588, -0.07407405227422714, -0.948327898979187, -0.4223930239677429, -0.45855772495269775, -0.10751841217279434, 0.04992448911070824, 0.5397143363952637, 0.636836051940918, -1.0234094858169556, -0.1662895530462265, -0.41118136048316956, 0.33195093274116516, -3.0383920669555664, -0.14354513585567474, 0.3074641525745392, -0.054232027381658554, 0.7861201763153076, -2.235363721847534, 2.0822901725769043, -3.1646766662597656, 0.796019434928894, -0.2176303118467331, -1.3749324083328247, -0.003292985027655959, -1.1221368312835693, 0.17683729529380798, 0.5823179483413696, -0.5290491580963135, 2.61983060836792, -0.8938204646110535, 1.488439917564392, -1.4524697065353394, 0.17364974319934845, 1.594300627708435, -0.08669934421777725, 0.8159651756286621, 0.15183497965335846, 0.003501380095258355, -0.9400987029075623, 1.917129397392273, -0.8364514708518982, -1.6836371421813965, -0.7901896834373474, 1.9313368797302246, -1.6084530353546143, -1.0461986064910889, -1.0896371603012085, -0.20942656695842743, -0.6759598255157471, 0.019372140988707542, 1.2814841270446777, -0.37121933698654175, -1.7867995500564575, 1.8627357482910156, -0.5104987621307373, -2.0025582313537598, 1.6023470163345337, 1.6929384469985962, 1.014025092124939], [-1.1334164142608643, 0.8089805245399475, -0.8484431505203247, 0.35108813643455505, 1.0965824127197266, -1.4097412824630737, -0.9151525497436523, 1.290488600730896, -1.2611654996871948, 1.3592538833618164, 0.17564277350902557, -1.2378562688827515, 0.18448051810264587, -0.15297918021678925, -0.08684848248958588, -0.21680282056331635, -0.47567203640937805, 0.027085302397608757, -0.6695860624313354, -1.2949802875518799, 1.107040286064148, -0.05386916548013687, 0.818182110786438, 0.6293862462043762, 0.41608864068984985, -0.6027560234069824, -0.5050682425498962, -2.599727153778076, -0.6616582870483398, 0.20404048264026642, -0.07338574528694153, 0.08171282708644867, 0.020599665120244026, -1.9401195049285889, 0.717018723487854, 0.634141743183136, -0.34218546748161316, -0.5352765917778015, 0.2689327597618103, 0.8514208197593689, 0.44717830419540405, 1.3422749042510986, 1.1958128213882446, 1.1167511940002441, 0.6976336240768433, 0.08061306923627853, 0.8739036917686462, -1.719984531402588, -2.1185250282287598, -1.5100902318954468, -1.493499994277954, 0.2231425642967224, 0.8539701104164124, 1.6905417442321777, -0.18102335929870605, -0.22879146039485931, 1.5146231651306152, 1.1455602645874023, -0.10379428416490555, 1.2329901456832886, -0.514827311038971, 0.09125892072916031, -1.0811015367507935, 0.2977510392665863], [1.6630586385726929, -0.5868094563484192, -0.5931934118270874, -0.7696167826652527, 0.8099244236946106, 0.09763292223215103, -1.5089012384414673, 0.5509088635444641, -0.18406616151332855, -0.9044962525367737, -1.4899333715438843, 1.5728918313980103, -2.3985347747802734, -0.8647645711898804, 0.3839077949523926, 0.021739769726991653, 0.8259000778198242, 0.6701468229293823, -0.7464443445205688, 0.5663891434669495, -0.7416811585426331, -0.7390331029891968, -0.3854324221611023, 1.7855159044265747, 0.0889308750629425, 0.5856038928031921, -0.2933639585971832, 1.3561619520187378, 0.33247920870780945, -1.2535454034805298, -0.7088133692741394, 1.7224937677383423, -0.8565213084220886, 0.1369735598564148, -2.3011109828948975, 0.2747677266597748, 0.874959409236908, 1.3110979795455933, 1.992292881011963, 0.2696962058544159, -1.0967105627059937, 0.3462710678577423, -0.07882925868034363, 0.8714900016784668, -0.30497753620147705, 0.7668668031692505, 0.9499703645706177, 0.014645000919699669, 0.46827200055122375, -1.3268933296203613, -0.5609549283981323, 1.225458025932312, -1.7432284355163574, -1.133392095565796, 1.792670726776123, 0.24410410225391388, 1.2228114604949951, -0.0476212240755558, 1.4405263662338257, -1.8401726484298706, 0.7624494433403015, -0.8208494782447815, 0.4690020978450775, 0.5541293621063232], [-1.717820405960083, 0.3836047649383545, 1.4118150472640991, -2.0927515029907227, -0.536522388458252, -1.0002483129501343, -0.7167136073112488, 0.846950113773346, -0.6399950385093689, 0.3067869544029236, 0.6131193041801453, -2.6865944862365723, 1.1636070013046265, 1.3651996850967407, -1.2096490859985352, 1.2446357011795044, 0.4677749276161194, -1.94553804397583, -0.03265393525362015, 0.18672911822795868, 0.40453460812568665, 0.3207371234893799, 0.25043338537216187, 0.03494058921933174, -1.6054836511611938, -0.8223243355751038, 0.019470809027552605, -0.07517415285110474, 0.954384982585907, -0.34747472405433655, 0.8124284148216248, 1.802346110343933, 1.776957631111145, -1.2476577758789062, -0.48413559794425964, -0.6069950461387634, -0.7200875878334045, -1.1597691774368286, 1.0968358516693115, -0.4022885262966156, 0.17642155289649963, -0.8639234304428101, -0.8797585368156433, -0.08114220201969147, -0.4730871915817261, -0.19631771743297577, -1.0846304893493652, -2.013653516769409, -0.130310520529747, -0.28187108039855957, -0.09450222551822662, 0.2721420228481293, -0.3130910098552704, -0.9564591646194458, 0.7667965292930603, 0.4330894947052002, -0.1885223239660263, -2.0156030654907227, 0.23925060033798218, -0.7638310194015503, 0.06396868824958801, -0.9477446675300598, 1.1576257944107056, -0.8832991123199463], [-0.1611076444387436, -0.7851499915122986, -0.5828990340232849, -0.3187158405780792, 0.1683708131313324, -1.761684536933899, 0.15240180492401123, -0.9440807700157166, -0.3597063720226288, -0.13593529164791107, 0.37772300839424133, -0.42603760957717896, -0.6390514373779297, -0.7294350862503052, 0.719074547290802, 0.12445759028196335, 1.4148131608963013, 0.9479615688323975, 1.4583864212036133, 1.0663105249404907, 1.020149827003479, 0.07118171453475952, -0.28314414620399475, -0.09687574207782745, 0.6539757251739502, -0.3841288089752197, 0.225795716047287, -1.5695387125015259, -1.2017029523849487, 1.5260059833526611, -0.3090376853942871, 0.8720490336418152, 1.1573368310928345, 0.24168361723423004, 2.65863037109375, -0.01666974276304245, 0.4656532406806946, -0.6024198532104492, -0.6040464639663696, -0.11157439649105072, -0.6248482465744019, 0.4143812954425812, 0.1224968209862709, -0.11176349967718124, -0.06066722422838211, -0.2820552587509155, -0.16109415888786316, 0.01809222251176834, 1.2983425855636597, 0.6269485354423523, -0.9306443929672241, 1.0640462636947632, 0.7230753898620605, 0.3913631737232208, -0.12513728439807892, -0.4903137981891632, -1.36784029006958, 0.11494256556034088, 1.4599336385726929, -0.008541555143892765, -0.3238731622695923, 0.6244100332260132, -1.3711811304092407, 1.9910951852798462], [-0.6851039528846741, -0.17648103833198547, 0.008392127230763435, -1.0170663595199585, 0.6850698590278625, 0.3341246545314789, 2.111384868621826, -0.5477772951126099, 0.4964890480041504, 1.2402905225753784, -0.8679700493812561, 1.4306421279907227, -1.1408904790878296, -0.6629409193992615, -0.6501443982124329, 0.7849056720733643, 0.5438088774681091, -0.23580272495746613, 0.5827665328979492, 0.6012390851974487, -0.45759567618370056, 2.6350507736206055, 1.6021685600280762, -0.4500705599784851, -1.4096635580062866, -0.19512933492660522, -0.7044264078140259, 1.4099537134170532, -0.06270872056484222, -0.8112069368362427, 1.4623143672943115, 0.8148695230484009, 0.5668959617614746, 0.36093869805336, -1.2369500398635864, -1.6633864641189575, -0.9054927229881287, 0.11717850714921951, 1.544777750968933, -0.5691786408424377, -0.24692638218402863, 0.32010236382484436, -0.25255540013313293, 0.4212173521518707, 0.15045487880706787, -1.0125060081481934, 0.3479442596435547, -1.3390389680862427, 0.6149886250495911, 1.0103607177734375, 0.09309785068035126, -2.042387008666992, -0.34413042664527893, -0.44712746143341064, 1.7838170528411865, 0.9847069382667542, -0.18489590287208557, 0.48453688621520996, -0.9339094758033752, -1.5146186351776123, 0.3780733644962311, -0.10610385984182358, -0.3478691875934601, 1.6914077997207642], [0.4332787096500397, 0.7516282200813293, 0.2827572524547577, -0.14768914878368378, -0.4127093255519867, -0.05049088969826698, 0.6723689436912537, 0.24892295897006989, 1.3809434175491333, -1.1219708919525146, 0.48799988627433777, 0.08674488961696625, -1.8120297193527222, -1.0200328826904297, 1.042031168937683, 1.0941898822784424, 2.104443311691284, -1.6872081756591797, -2.0398807525634766, 1.747401237487793, 0.721107542514801, 0.5631135106086731, -1.6002557277679443, 2.0772697925567627, -0.49400249123573303, 1.0090819597244263, 2.0439839363098145, 0.6475533246994019, -0.2034873515367508, 0.13383740186691284, -0.7558810710906982, 0.057970255613327026, 1.477629542350769, 0.07440503686666489, -0.9440878629684448, 1.4572699069976807, 0.909915030002594, -0.1700599044561386, -0.20131173729896545, 0.3348054885864258, 1.3016021251678467, 0.879525899887085, -0.14621809124946594, 1.2394018173217773, -0.7036475539207458, 1.8280938863754272, -1.420236587524414, -0.4974415898323059, 1.121212363243103, -0.8960101008415222, -0.6442577242851257, -1.000253438949585, -0.18026040494441986, 1.152283787727356, 0.2759239077568054, -0.1421181559562683, -0.8664575219154358, 0.5012230277061462, 0.15389101207256317, 0.5379239916801453, 0.006851463112980127, -0.10557805746793747, 1.4015405178070068, 0.8952721357345581], [0.36743292212486267, -0.5446260571479797, 1.004654884338379, 1.0349797010421753, -1.251540184020996, 0.4667423665523529, -2.1701886653900146, -0.3367306888103485, 0.5090106129646301, 0.7252306938171387, 1.795637607574463, -0.03391861170530319, 0.2978453040122986, -0.8933081030845642, -0.9922756552696228, -0.6893356442451477, 1.7791938781738281, -0.8313881158828735, 1.2061285972595215, -1.8786102533340454, 1.077406644821167, 0.15087252855300903, -0.40209540724754333, -2.346947193145752, 2.676605701446533, -0.4800475835800171, 1.2188395261764526, -0.6828793287277222, -1.7552834749221802, 0.20006483793258667, 1.0350265502929688, -0.0276477113366127, -0.9780735969543457, 0.18121632933616638, -0.38032230734825134, -1.4197434186935425, -0.5846529006958008, 0.6039826273918152, -0.6755964756011963, -0.27992093563079834, 0.4913701117038727, -0.553838849067688, -0.7458122968673706, 0.8203516006469727, 0.12694326043128967, 0.8127093315124512, 0.5826709866523743, -0.45873358845710754, 0.107764832675457, -1.7747293710708618, -0.8936468958854675, -0.09139237552881241, -0.3355085551738739, 1.6732805967330933, -0.2831422686576843, 0.8507428765296936, 0.24084806442260742, 0.7496804594993591, 0.7972444295883179, -0.39007821679115295, 0.2688775658607483, 0.11065146327018738, -1.3478049039840698, 1.5260387659072876], [-1.5171257257461548, -0.15889985859394073, 0.7807919383049011, -0.3083256483078003, 0.8144230842590332, 0.16125044226646423, -1.6370861530303955, -2.1711299419403076, -0.9914065599441528, -0.18756520748138428, -0.48498624563217163, -0.3231406509876251, -0.09451314061880112, 0.6278771758079529, 0.882931113243103, -0.6078876852989197, -0.6708800792694092, 1.6949630975723267, 0.2989003360271454, -1.5584076642990112, -1.0942496061325073, 1.4333701133728027, -0.6089532971382141, 1.443569302558899, -2.789865016937256, -0.6726236343383789, 0.26844242215156555, -0.4367867112159729, 0.05337309464812279, 1.0841522216796875, 0.8703248500823975, -1.0439562797546387, -1.08737051486969, 1.213587999343872, -0.32351797819137573, -0.47783103585243225, -1.7844643592834473, -0.6735294461250305, -0.12078165262937546, 0.9347646236419678, -0.5200390219688416, 0.20510976016521454, -0.4766468405723572, 0.18365135788917542, 0.14789879322052002, -0.6427104473114014, 0.2795724868774414, -0.7357584238052368, -0.39066067337989807, -1.901155710220337, 1.5686390399932861, 1.2682874202728271, 1.4014452695846558, -0.2843412756919861, -0.2637288272380829, -0.24585820734500885, 0.7997114062309265, -0.2870865762233734, 0.4963715374469757, 1.6809240579605103, -1.5585436820983887, 2.7387306690216064, -1.3527222871780396, 0.6070809364318848], [-0.5790762305259705, 0.14722713828086853, 0.07194758206605911, 1.0532615184783936, -0.19283156096935272, -1.3133589029312134, -0.17645233869552612, 1.0942964553833008, 0.40647879242897034, 0.22389335930347443, -0.8816636204719543, 1.0962913036346436, 0.4270390570163727, -0.5724919438362122, 0.5920963287353516, 1.5168726444244385, -0.6179252862930298, 0.6875125765800476, 0.8699319958686829, -0.37650057673454285, 0.5477635264396667, -0.028182880952954292, 1.1347720623016357, -0.3777177929878235, -0.21662233769893646, -0.8347930908203125, -1.2781282663345337, 0.2346721738576889, -0.6023474931716919, -0.3255864083766937, -0.512688398361206, 0.9556446075439453, 0.5662487745285034, 0.5960721373558044, 0.08651414513587952, 0.22822557389736176, -0.918745756149292, -0.8322111368179321, -1.280311107635498, -0.03663313016295433, -0.20548169314861298, 0.13631772994995117, -0.04895289987325668, -0.6650592088699341, -0.9810507893562317, 0.30036577582359314, -1.4885581731796265, -1.1782091856002808, -1.9379042387008667, 0.6864845156669617, -0.13328005373477936, 0.1798080950975418, -0.3720821142196655, 0.3088386058807373, -1.122318148612976, 0.16112945973873138, -0.3782385289669037, 1.4092692136764526, 0.04993772506713867, -1.7760026454925537, -0.9450134634971619, -0.5264045000076294, -1.9942234754562378, 0.5663476586341858], [0.17447271943092346, -0.5084210634231567, -1.2184181213378906, 0.4031650125980377, 0.3017577826976776, -0.25997161865234375, -0.2402854710817337, -1.0647979974746704, -0.1185578778386116, -1.3358511924743652, 0.7790660858154297, 0.4601113796234131, 0.8290278315544128, -0.9445602893829346, -1.1397764682769775, -0.4116221070289612, 1.0976037979125977, 0.015266569331288338, -0.2696293592453003, -1.907200813293457, 1.23338782787323, 0.8836260437965393, 1.2805668115615845, -0.7017561197280884, 0.7633634209632874, 0.20125184953212738, 0.6264791488647461, -1.0000395774841309, -0.9952325224876404, 0.122390978038311, -1.996324896812439, -0.30647900700569153, 0.39039990305900574, -0.6969050168991089, 0.7336923480033875, 1.5075669288635254, 0.8961986303329468, -0.7509834170341492, 0.14669321477413177, -0.17773187160491943, -1.0701335668563843, 0.2956971526145935, 0.36503955721855164, 0.7031440138816833, 0.5211101770401001, -0.8876417279243469, 0.4276345670223236, -2.1577258110046387, -0.22856836020946503, 0.7638190388679504, 0.8456647992134094, -2.0293493270874023, 0.6498652696609497, 1.0671032667160034, -0.5972644090652466, 1.053758978843689, 0.023462943732738495, 0.6549549698829651, -0.7335615158081055, -1.7513543367385864, -1.6388883590698242, -0.25939181447029114, 0.24720081686973572, -0.23366984724998474], [1.6015976667404175, -1.1865637302398682, -1.256151795387268, -1.0183371305465698, 0.4329274892807007, -0.4690394699573517, 0.03454028442502022, -0.6546134352684021, 1.1203804016113281, 0.769792914390564, 0.603367030620575, -0.8746485114097595, 1.8856103420257568, -0.47155705094337463, 1.9116852283477783, 1.7158023118972778, -1.6046544313430786, -0.33976224064826965, 0.07110800594091415, -1.9536176919937134, -0.4032447338104248, -0.22751319408416748, 1.1424535512924194, -0.7018927335739136, -0.4430082142353058, 0.011785241775214672, 0.2796027660369873, -2.809666633605957, 1.318957805633545, -0.6921160817146301, 0.3809331953525543, 0.6799331307411194, 0.8017556667327881, -0.12256433069705963, 0.7037513852119446, 1.373419165611267, -0.13196803629398346, 0.6470955014228821, -0.04162716493010521, 0.7875072360038757, 0.2878216505050659, -0.7259761691093445, 0.1102939322590828, 0.7164950370788574, -0.84154212474823, -1.2520753145217896, 1.5870710611343384, -0.7325608730316162, -0.6420661807060242, 0.6924123167991638, 0.23780466616153717, 0.7852115631103516, 1.090793490409851, -1.5932685136795044, -1.405440092086792, 0.3115255534648895, 2.218769073486328, -1.0418983697891235, 1.612517237663269, 1.1024436950683594, -1.351078987121582, 0.19705578684806824, 1.1416257619857788, 0.07283742725849152], [0.1272730976343155, 0.35934683680534363, -1.2187182903289795, 1.6387231349945068, 0.6246684789657593, 1.9564036130905151, 1.1053581237792969, 0.009960265830159187, 0.1137268990278244, -1.0903685092926025, 0.5620765686035156, -0.5747072696685791, 0.33856070041656494, -0.2396955043077469, 0.5914702415466309, 0.27502816915512085, 1.0382013320922852, -1.7993520498275757, -0.6054056286811829, 0.3023160994052887, -1.1502033472061157, -1.5837193727493286, -0.5768147110939026, 0.22980286180973053, -0.408864825963974, -2.1482067108154297, 1.2002198696136475, 1.3272103071212769, -0.4153887629508972, -1.640342116355896, 0.11088390648365021, -0.8440245985984802, 1.4573944807052612, 0.944061279296875, -0.5178300738334656, 1.460754632949829, -0.7907434105873108, 0.3204367160797119, -0.7172985076904297, -0.7308175563812256, 0.08820372074842453, 0.578527569770813, -0.1448909491300583, 1.6147193908691406, -1.586495041847229, -0.9424169063568115, 1.7331433296203613, -0.23048605024814606, 0.15193238854408264, -0.1173684373497963, 1.4934674501419067, 0.3585793375968933, 0.8393155336380005, 0.011187891475856304, 0.6429154872894287, 0.3877928555011749, -0.2230713814496994, -1.5572209358215332, -0.7144017219543457, -1.0188015699386597, 0.5580360293388367, -0.8469417691230774, -0.21691258251667023, 0.441244900226593], [0.23121362924575806, -1.339845061302185, -0.31964367628097534, -2.067519187927246, 0.9012292623519897, -0.9702116250991821, 0.8826924562454224, 1.1385118961334229, -1.1407040357589722, 1.3563729524612427, -0.136552631855011, 1.66554594039917, 1.5065094232559204, -0.09099231660366058, -2.478327512741089, 0.5522303581237793, -1.0767313241958618, 0.28920137882232666, -1.1188971996307373, 0.29690423607826233, 1.5508720874786377, -0.21649326384067535, -0.09759305417537689, 1.3270577192306519, 0.6493985056877136, -0.8638606071472168, 0.8457738161087036, 0.8218417763710022, -0.9562163352966309, 1.5143821239471436, -1.2984821796417236, -0.8838005065917969, -2.016171932220459, -0.8548790216445923, 1.016353964805603, 0.26329511404037476, 1.5087015628814697, -0.3868681788444519, -0.035988226532936096, 0.7609995603561401, 0.911855936050415, 0.7998630404472351, -0.2899188697338104, -0.15222151577472687, 0.22475115954875946, 0.9890807271003723, 0.8908804059028625, -0.9859058260917664, 0.024102626368403435, -1.643564224243164, 0.3424539566040039, 0.869716465473175, -0.11599476635456085, 0.8298074007034302, -0.9655212163925171, -0.03009260632097721, 0.17096033692359924, 0.6292437314987183, 0.12938061356544495, -0.0248248390853405, 1.01322340965271, -1.2170860767364502, 0.005376511253416538, 0.35263681411743164], [-1.1388295888900757, 0.02619202621281147, 1.0518543720245361, 1.544128179550171, -0.28913453221321106, -0.13141220808029175, 1.2038536071777344, 0.7770922780036926, -0.42593052983283997, 1.1655241250991821, 2.664158582687378, 0.4826499819755554, -0.16560059785842896, 0.1462746113538742, -1.068778157234192, -0.14970797300338745, 1.1727426052093506, -0.760774552822113, 0.03185606002807617, 0.22554773092269897, 1.1619153022766113, 0.45918554067611694, -1.9202159643173218, 1.50385582447052, -0.12330560386180878, -0.16199181973934174, -2.2211453914642334, -0.04239490628242493, -1.7500618696212769, -2.2122888565063477, 0.2349942922592163, -0.21015988290309906, 0.4396454989910126, 0.028295187279582024, 1.8163822889328003, 0.9039364457130432, -1.5010167360305786, -1.2339240312576294, -0.32457271218299866, -0.19491438567638397, -0.6695268154144287, -0.1763339638710022, 1.974859356880188, -0.47824326157569885, 0.440118670463562, -0.5426881313323975, -0.7886008620262146, 1.3849561214447021, -1.597244143486023, -0.7359582185745239, -1.450863242149353, -1.1355586051940918, -0.8632428646087646, -0.4691249430179596, -0.37930914759635925, 0.25921300053596497, 0.836148738861084, -0.08851300925016403, -1.2110655307769775, -0.6994471549987793, -1.4185954332351685, 1.8973840475082397, 0.5411103367805481, 0.7815205454826355], [-0.09265660494565964, -0.9020425081253052, 0.026608632877469063, 0.6419464349746704, 1.3631192445755005, -0.7367568016052246, 0.7294765114784241, 0.14340850710868835, 2.0539116859436035, 1.7472515106201172, 0.721989631652832, -0.1905159205198288, -1.0269776582717896, 0.5301536917686462, -0.5009282231330872, 0.28325849771499634, 0.34413668513298035, -1.1560876369476318, 0.8244045376777649, -0.10504333674907684, 2.2306525707244873, -0.4175207018852234, 0.07168962061405182, 0.22654609382152557, -0.28071656823158264, -0.5882483124732971, 0.6229001879692078, -0.818397581577301, -0.7576954364776611, 0.1942872554063797, -0.3480719029903412, -0.5144311785697937, 1.0324522256851196, -0.21749000251293182, -1.2010481357574463, 0.16193713247776031, -0.6337306499481201, -0.12909021973609924, 0.9400618672370911, 0.2921515703201294, 0.1475585699081421, 0.28036388754844666, 0.7046062350273132, 0.2545856535434723, -0.9075810313224792, 0.17695459723472595, -0.7098475694656372, -1.4451813697814941, -0.3710542619228363, 0.0331600122153759, -1.0151318311691284, -0.20173801481723785, 0.42508944869041443, -0.529426634311676, -1.8348225355148315, -1.074839472770691, 0.5500211119651794, -0.4866955280303955, 0.8445979356765747, 0.7296433448791504, -0.40750014781951904, -0.5271884799003601, -0.7919033169746399, -1.4835340976715088], [-1.6603764295578003, -0.04056717827916145, -0.8945791125297546, 0.8209601640701294, -0.5880798101425171, 0.1884436160326004, 2.0534186363220215, -0.5692256093025208, -0.22419030964374542, -1.3530478477478027, 0.12155916541814804, -1.3151475191116333, -0.07507050782442093, -0.961437463760376, -0.45974960923194885, -0.4198169410228729, 1.6679186820983887, -1.470062255859375, -0.7362477779388428, -0.6467820405960083, -0.8237699866294861, 1.5326299667358398, 0.007609986700117588, -1.1553529500961304, 1.983059048652649, -0.5174973011016846, 0.2270803153514862, 0.429457426071167, 1.224751591682434, -0.6711525917053223, -1.1924543380737305, 0.3246755599975586, 0.2895093560218811, 0.40795275568962097, 0.6068442463874817, -2.285219430923462, 0.007252061739563942, 1.9295411109924316, -2.7303011417388916, -0.8563970327377319, -0.11777089536190033, -0.966744601726532, 0.8009498715400696, 1.3181461095809937, -0.6565883159637451, 1.4481643438339233, -0.6183760762214661, 1.2144522666931152, 1.1274168491363525, -0.02804552949965, -0.31910064816474915, -0.36502015590667725, -0.46852588653564453, 0.042883943766355515, 0.9126715064048767, 0.02555614709854126, 0.27387818694114685, 0.9757286310195923, -0.06349614262580872, -0.9642175436019897, 0.8672663569450378, -1.272500991821289, 0.5429897904396057, 0.48051318526268005], [-0.007636290974915028, -0.04130217805504799, -0.312907338142395, 0.15058398246765137, 2.1197597980499268, 0.13419146835803986, -1.4202953577041626, -1.8206963539123535, -1.7810633182525635, 1.79010009765625, 1.3391762971878052, -0.38527944684028625, 0.3968692123889923, -0.37748339772224426, -1.6374560594558716, -0.5560529828071594, -0.5764027237892151, -0.28088608384132385, 0.7295649647712708, -2.209523916244507, 1.1078922748565674, 0.06798191368579865, -1.3555124998092651, 0.8149493336677551, 0.1757325828075409, -0.13281144201755524, -0.7232757806777954, -0.3650723099708557, 0.13637615740299225, 0.9196789860725403, 0.8365421891212463, 1.4561517238616943, 0.6211549639701843, 1.1922085285186768, -0.4415263533592224, 0.9821566343307495, -0.48444414138793945, -0.6640602946281433, -0.22220593690872192, 0.1626916378736496, -1.1034619808197021, -2.4773125648498535, 0.9382684230804443, -0.726193368434906, -0.909774661064148, -0.23296009004116058, -0.6119842529296875, -0.10335525125265121, -0.8079317808151245, -1.1297799348831177, 1.1324762105941772, 0.7699466943740845, -0.5868514776229858, -0.7486451268196106, -0.9849841594696045, 1.4673891067504883, -0.6616708040237427, -0.004838263615965843, -0.8536432385444641, -0.23227232694625854, 1.1946862936019897, 1.3757169246673584, -0.13724298775196075, 0.9073039293289185], [-0.018843185156583786, 0.699968695640564, -0.3685576617717743, 0.07060108333826065, 1.8257248401641846, -0.3577859103679657, -1.213245153427124, -1.0241354703903198, -0.46502721309661865, -1.2975202798843384, -1.103411078453064, 1.4494187831878662, 1.603857398033142, 0.5570977330207825, 1.320765495300293, 0.5765150189399719, 1.6330331563949585, 1.6711972951889038, 1.2469638586044312, 0.216975599527359, 1.7857398986816406, 0.47552016377449036, 0.656724750995636, -0.0410081222653389, -0.37484079599380493, 0.28461340069770813, -0.23876383900642395, 0.7191036343574524, -0.18551647663116455, -0.012367880903184414, 2.748142957687378, 0.6068114042282104, -0.24406661093235016, 0.2019009292125702, 0.6295701861381531, -0.38405731320381165, -0.03970067948102951, -1.5699293613433838, -1.2243385314941406, -0.5051313042640686, -0.946223795413971, 0.30283889174461365, 0.5157076716423035, 0.3916763663291931, 0.1018194779753685, -0.8334493637084961, 1.6092309951782227, -0.9698077440261841, 1.0318269729614258, -1.5353363752365112, -2.160146713256836, 0.4082919955253601, -0.8718723654747009, 2.02746319770813, -0.6186785697937012, -0.7681962251663208, -0.3657895028591156, 0.20392842590808868, 0.4852495491504669, -0.43745487928390503, -1.3483026027679443, -2.3270184993743896, -0.13516908884048462, -1.6376280784606934], [-0.5032854080200195, -1.6966025829315186, 0.529895544052124, 1.4112658500671387, -0.4191773235797882, -0.5875741243362427, 0.6256098747253418, 0.4333069622516632, 0.6181830763816833, -1.6425546407699585, 0.20246131718158722, 0.2656771242618561, -1.5384234189987183, -0.22200536727905273, 1.728127121925354, 0.7890565395355225, -0.73626708984375, 1.6302330493927002, 0.42271727323532104, -0.24293379485607147, -1.4243215322494507, 1.258744716644287, -0.6127883195877075, -0.6340454816818237, -0.05186166241765022, -0.7277556657791138, -0.78473299741745, -0.28107383847236633, -0.022476695477962494, 0.4006653428077698, -1.4346338510513306, 1.0288662910461426, 0.5376490950584412, -0.2234567403793335, -0.8300214409828186, 0.5791252851486206, 0.9027196168899536, -0.6320667266845703, 1.19210946559906, 1.5002602338790894, -2.0604498386383057, -1.2069448232650757, -0.41588708758354187, 0.24397113919258118, -0.9733898639678955, 0.9805946350097656, 0.8217757940292358, -1.0006206035614014, -0.6118453145027161, -1.6703630685806274, 2.7542693614959717, -0.5676654577255249, -0.5397909879684448, -0.7402714490890503, -1.1354249715805054, 1.1001684665679932, 0.5095803737640381, 0.794951856136322, -0.2744750380516052, 0.7470582127571106, 2.6946518421173096, 0.5527571439743042, 0.20611467957496643, 0.031562622636556625], [0.5790489315986633, -0.3985579013824463, 1.1605374813079834, 0.28123509883880615, 0.9274603128433228, -0.7890510559082031, -0.043302640318870544, -0.16923940181732178, 0.7910668849945068, 0.8626334071159363, 1.1690679788589478, 1.3211599588394165, -0.8820646405220032, 1.326606035232544, -1.010541558265686, 1.5874524116516113, -0.47535908222198486, 0.6363741159439087, -0.18530386686325073, -1.0306819677352905, 1.0207622051239014, -1.5043898820877075, -0.6168449521064758, 1.0551553964614868, 1.287269115447998, 0.9415072798728943, -1.2303693294525146, 0.17302635312080383, 0.28028467297554016, 0.32455122470855713, 2.262094497680664, 0.13087579607963562, -1.0490930080413818, -0.38063815236091614, 0.6958110332489014, -1.7944482564926147, -2.4600327014923096, 0.7636134028434753, 0.8534963726997375, 1.3669860363006592, -0.6566296815872192, -0.9277305006980896, -1.8573615550994873, 0.7211889028549194, 0.5239778161048889, -0.4235396683216095, 0.33044183254241943, 0.48975393176078796, 0.38391196727752686, 0.13754220306873322, -1.1734087467193604, 0.19383041560649872, 0.3842020034790039, -0.875413715839386, -1.5462766885757446, 0.41124358773231506, 0.17290586233139038, -1.2686810493469238, -1.7338097095489502, -0.2160705029964447, -2.6027257442474365, -0.6895663142204285, -0.02457256428897381, -0.40888532996177673], [-1.1054608821868896, 0.19650378823280334, 0.7553699016571045, 0.21920309960842133, -0.008605238981544971, -0.29964062571525574, -0.5442020893096924, 2.0366764068603516, -0.0781354233622551, -1.1084661483764648, 0.196584552526474, -1.728695034980774, -0.3756767511367798, 0.5931087136268616, -0.8139728307723999, -0.7764638066291809, 1.5033466815948486, -0.18612925708293915, -1.157755970954895, -0.023882081732153893, -0.4241791069507599, 1.0775967836380005, 0.6000701189041138, -0.38157594203948975, 0.05571777746081352, -1.0634877681732178, -1.028105616569519, 0.022402023896574974, -0.12599772214889526, -0.6104589104652405, 0.3701989948749542, -0.23378540575504303, 0.6698512434959412, 0.3662891983985901, -0.8145106434822083, -0.6598049998283386, -0.10939312726259232, -1.4310349225997925, 1.840181827545166, 0.47759780287742615, -0.18639852106571198, -1.3114047050476074, 0.3348931670188904, -0.29594850540161133, -0.4978635013103485, -0.4315205216407776, -1.1395790576934814, -0.21790190041065216, 0.22277213633060455, 0.7286479473114014, -2.061777353286743, 0.5769126415252686, 0.3061271607875824, 0.8344311714172363, -0.5497039556503296, -0.37418073415756226, 1.0604182481765747, 1.8999099731445312, -1.149173378944397, 0.344172865152359, 0.5917354822158813, 0.9759715795516968, 0.3632109463214874, -1.567230224609375], [0.47192737460136414, 1.698229193687439, 0.36187273263931274, -0.6679702997207642, 1.3990364074707031, -0.284529447555542, 0.3198699653148651, -0.3851016163825989, -1.0455478429794312, 0.6074849963188171, -0.6831612586975098, -1.6150386333465576, -1.3147226572036743, 1.2771587371826172, -1.191691517829895, 0.2973981201648712, -1.2427996397018433, -0.3086124062538147, -0.8395440578460693, -0.1857680231332779, 0.8816675543785095, 0.40073132514953613, 0.1272670179605484, -0.6405582427978516, -1.7178994417190552, -0.13677550852298737, -0.7968301177024841, -0.34134507179260254, -0.19878996908664703, -0.01476146187633276, 1.2115073204040527, -0.488267183303833, 0.4382670521736145, -0.9981235861778259, 0.060307011008262634, 0.17044001817703247, 0.48093339800834656, -0.34918156266212463, 2.5731005668640137, 0.6088142395019531, -0.7727739810943604, -0.3215208649635315, -1.2347562313079834, 1.1619871854782104, -0.3791145086288452, 0.7892847061157227, 0.6254483461380005, -1.3484234809875488, -2.1883318424224854, 0.18076784908771515, -0.5035088062286377, -0.5780588984489441, -1.2498561143875122, 0.5490082502365112, 1.6506175994873047, -1.7385351657867432, 1.824356198310852, 0.1681939959526062, 0.1670272797346115, 0.5638228058815002, 1.2609423398971558, -0.3515840172767639, -0.8211994767189026, 1.071613073348999], [0.5597342848777771, -1.3265630006790161, -0.05315525457262993, -0.011942208744585514, -0.8433552980422974, 0.5768100619316101, -0.6910897493362427, 0.8684545755386353, 2.471156597137451, 0.9997179508209229, 0.4177349805831909, -0.5360986590385437, 0.19984665513038635, 1.7034711837768555, -0.4877747893333435, -0.5498042702674866, 0.1046212688088417, -0.056186314672231674, 1.5213377475738525, 0.9511516690254211, 0.7163248658180237, -1.9823368787765503, 0.5111053586006165, -0.3367191553115845, -0.0786370113492012, 0.44920533895492554, -1.9302637577056885, 0.2167135328054428, 0.12764020264148712, -1.2821459770202637, 0.4492380619049072, -0.09962022304534912, 1.4712358713150024, -0.6138001084327698, 0.18685376644134521, 0.22854578495025635, -0.8938831090927124, 0.2457195222377777, 0.6971560120582581, -0.3131520450115204, -1.1252532005310059, -0.8064461350440979, -0.26164770126342773, -0.8491968512535095, 0.6016659736633301, -0.4449860751628876, -0.39385080337524414, 0.0831945464015007, -0.05802734196186066, -0.10376185923814774, -0.9970167279243469, -3.519037961959839, 0.697441041469574, -0.7447414994239807, 0.25825145840644836, 0.3117174804210663, 2.7844457626342773, 0.15544284880161285, 0.1170993223786354, 1.1423180103302002, 0.22063371539115906, 1.686710238456726, -0.1162528246641159, -0.6597535014152527], [1.2470014095306396, -0.45533010363578796, 0.9948353171348572, -0.193320170044899, 0.4451645314693451, -0.5929231643676758, 0.593448281288147, -0.9928163290023804, -0.19401097297668457, 1.4172827005386353, -0.001723331632092595, 0.4750366806983948, 1.701918125152588, 0.6573238968849182, -0.3982953131198883, -0.1852939873933792, 0.5566871166229248, 0.6255121231079102, -0.708107054233551, -0.692308247089386, 0.18069541454315186, 0.1011430099606514, 0.12580032646656036, 1.0851707458496094, -0.6048701405525208, 1.341857671737671, -1.965826153755188, -0.3770564794540405, 1.5177054405212402, -0.6326304078102112, 0.9387301802635193, 0.18854334950447083, -1.2142282724380493, 1.2541829347610474, -0.1320340782403946, -0.470587819814682, 0.5583321452140808, 0.6567152142524719, -0.48516976833343506, 1.6126658916473389, 0.6958855390548706, -2.1352810859680176, -0.7076768279075623, 0.043970346450805664, 0.9098062515258789, 0.4138204753398895, 0.879744291305542, -1.2666528224945068, 1.4123483896255493, 0.2804529666900635, -0.3660866618156433, -0.8536554574966431, -1.2782063484191895, 1.9606549739837646, -0.27597618103027344, -1.9070022106170654, -0.7436890006065369, -0.1414993405342102, -1.0618302822113037, -0.5008677840232849, -0.7679827213287354, -0.3183373808860779, -1.606070876121521, -0.16607944667339325], [1.6161596775054932, 0.10623455792665482, -0.29481932520866394, 1.4255238771438599, 1.2340902090072632, 1.2438393831253052, 0.658820629119873, -0.539238691329956, 0.7047791481018066, 0.336173415184021, 0.4937431216239929, 0.09006102383136749, -1.5827317237854004, 0.17636661231517792, -2.37664794921875, 0.8284662961959839, 0.8696601390838623, 0.006658606696873903, 0.6698755025863647, 0.545725405216217, -0.4234897494316101, -1.3349885940551758, -1.695894718170166, 0.36794009804725647, -0.11241723597049713, 1.0379928350448608, -1.2375222444534302, -1.0384329557418823, -0.4717453420162201, 1.513131022453308, 0.4255641996860504, -0.12123078107833862, 1.0397213697433472, 0.09947286546230316, 1.0078238248825073, 0.17168790102005005, -0.5917279124259949, 1.646310806274414, -1.5617221593856812, -1.7402865886688232, -1.4940539598464966, 1.0353214740753174, 0.46745845675468445, -0.9211674332618713, 1.6610931158065796, 1.7987048625946045, 0.5565701127052307, 1.2550026178359985, -1.9426732063293457, 0.09491375833749771, 0.022732751443982124, 0.5413084626197815, -0.235181525349617, 0.1424044966697693, -0.4451049268245697, 1.1014374494552612, 0.4131963551044464, -0.2289525270462036, -0.9621386528015137, 0.2531071603298187, 0.8711991310119629, -3.0193593502044678, 0.40166690945625305, -0.04245702549815178], [0.3139086663722992, -1.0625643730163574, 0.5944715738296509, -0.2825283408164978, -0.6838991641998291, -0.9479557871818542, -0.6239889860153198, 1.1541978120803833, -0.28154778480529785, -0.2056703120470047, 1.6158233880996704, 0.3055834472179413, 0.5983703136444092, 1.4174513816833496, -0.3103695213794708, 1.4983974695205688, -0.28366008400917053, -2.579108476638794, -0.3279998302459717, 0.07639358192682266, -0.35016584396362305, 0.1850307434797287, -0.48446938395500183, 0.12499051541090012, 0.944707989692688, 0.6234481334686279, 1.0626130104064941, -1.0845162868499756, 0.21177147328853607, 0.007635818794369698, -1.1683155298233032, 2.0469460487365723, 0.10519575327634811, -0.8472601175308228, -0.3431548476219177, 1.3914902210235596, -1.7416110038757324, 0.37855640053749084, -1.216692328453064, 0.7694059014320374, -2.2228827476501465, -0.15987615287303925, -1.3960627317428589, -0.9433920979499817, -0.19346269965171814, -0.16955123841762543, 0.4029017686843872, 1.059281587600708, -0.19557763636112213, 1.9623963832855225, -1.2289564609527588, -0.48289015889167786, -0.7094244360923767, 0.13898274302482605, -0.24645891785621643, 0.7772912979125977, -1.1309071779251099, 0.2536921501159668, 1.2717760801315308, 0.8391913771629333, 1.8084932565689087, -0.47861653566360474, -0.6163828372955322, 1.3411235809326172], [-0.25322261452674866, 1.0593229532241821, -0.0217090155929327, 0.8202691674232483, -0.46643832325935364, 0.4346601068973541, -0.4080042541027069, 2.3637709617614746, -1.4668424129486084, -0.07717437297105789, 0.8270216584205627, 0.926393449306488, 0.5690649747848511, 0.5556004047393799, 1.3104685544967651, 0.5331488847732544, -1.2303738594055176, -0.8404882550239563, 2.23830509185791, 1.1772645711898804, -1.438154935836792, -1.3445714712142944, -1.0544508695602417, 1.5215497016906738, -0.010263503529131413, -1.3520541191101074, 0.11195605993270874, -0.1075926274061203, -0.5210272073745728, 2.0037405490875244, -0.04536018893122673, 0.08706258237361908, 0.33085963129997253, 0.6151914596557617, 0.5869424939155579, 1.3237574100494385, -1.1240618228912354, 0.3756496012210846, 0.9840155839920044, -0.7282865643501282, -0.3441116511821747, 0.03998536616563797, -1.4292079210281372, 0.6919079422950745, -0.2611108422279358, 0.5381265878677368, -1.1910324096679688, 0.757902979850769, 1.352285385131836, -0.4535435140132904, -0.3368954062461853, 0.29146718978881836, -1.3212443590164185, 0.7311519980430603, 1.2230607271194458, -1.1710759401321411, -0.37362682819366455, 1.3003349304199219, -0.6084585189819336, -1.3447908163070679, 0.812143862247467, 0.8533378839492798, -0.638596773147583, 0.5903638005256653], [-0.08507156372070312, 0.5209766030311584, -0.9382866621017456, 0.5557506084442139, 0.28511130809783936, 0.2847422957420349, 1.4358090162277222, -0.32978418469429016, 1.716708779335022, -0.5316522717475891, -1.413635015487671, -0.4530040919780731, 0.25966665148735046, -1.233778476715088, -0.21744970977306366, -2.6963744163513184, 0.3119533061981201, 0.44429320096969604, 0.8996737599372864, -0.35884395241737366, -0.01919795200228691, -0.9398632645606995, 0.2604667842388153, 1.3257962465286255, -1.0342023372650146, 0.474984347820282, -0.6172602772712708, 2.7595205307006836, -0.2763122320175171, 0.2893376648426056, -0.9314832091331482, 1.951876163482666, -0.24730898439884186, -1.2888339757919312, -1.4526941776275635, 1.2509127855300903, 0.44580724835395813, -0.2820558249950409, 0.2691350281238556, -0.5549479126930237, -1.1384261846542358, -0.20725314319133759, 0.7930659055709839, -0.14980995655059814, 2.127371311187744, 0.4888840615749359, 0.20828649401664734, 0.21226458251476288, -0.31627681851387024, 0.3655151426792145, 0.17141184210777283, -0.42667776346206665, -0.442758709192276, 1.8228645324707031, -0.2400788962841034, -0.6020411849021912, 0.723567008972168, 1.2754931449890137, -0.28267544507980347, -0.01663273014128208, -1.4565988779067993, -0.47954532504081726, 0.3283086121082306, 0.7239112854003906], [-0.22146619856357574, -1.8298639059066772, -0.601844847202301, 0.4409514367580414, 0.9180186986923218, -2.0123977661132812, -0.7720123529434204, -0.22891972959041595, 1.9702441692352295, -0.5126529335975647, 0.21900694072246552, 1.2761269807815552, -2.041517496109009, -0.3210011422634125, -0.2975900173187256, -0.2803400158882141, -0.5669684410095215, -1.2745665311813354, 1.5227712392807007, 0.13578426837921143, -0.6807264089584351, 1.699569821357727, 1.1241626739501953, -1.272916555404663, 1.1973131895065308, -1.8851910829544067, 0.32213109731674194, 1.0194767713546753, -1.3753318786621094, -1.1799414157867432, -0.14417175948619843, 1.2165409326553345, 0.3305020332336426, -2.2240757942199707, 0.2394055724143982, 1.1187015771865845, -0.9539344906806946, -1.879446029663086, 0.3019162118434906, 0.06072293594479561, -0.6592453122138977, 1.8868939876556396, 0.2898551821708679, 1.7440402507781982, 0.8538233637809753, 1.2791763544082642, -0.00832268875092268, -0.20622365176677704, -0.4299161732196808, -1.1305493116378784, 0.4638952314853668, 1.843742847442627, -0.5804852843284607, 0.6213608980178833, 1.496341586112976, 0.2387770712375641, 1.7243263721466064, 0.15440276265144348, -0.895075261592865, -1.806006908416748, 1.5285148620605469, 0.8848358392715454, 0.5301832556724548, -0.6207287907600403], [-0.14518310129642487, -1.6362648010253906, -1.6347123384475708, -0.5605947375297546, -0.4289974272251129, 1.234649419784546, -0.5082383155822754, 0.20735608041286469, 0.7402878999710083, 0.1675158143043518, -0.33548054099082947, -0.8078188300132751, -0.030623849481344223, 0.6124095916748047, -1.6001123189926147, 1.2042540311813354, -0.7655697464942932, 1.9374297857284546, -0.06494171917438507, -0.2281314879655838, 0.5163454413414001, 1.1806001663208008, 0.6458227038383484, -0.6456615328788757, 0.4763963222503662, 1.123915433883667, 1.6421334743499756, -0.6913849115371704, -0.20422855019569397, -0.8910728693008423, -2.518631935119629, 0.1533239334821701, 1.2212152481079102, -1.8980295658111572, 0.37752971053123474, -0.7723724842071533, -2.278430938720703, 0.6864400506019592, -0.40545353293418884, -0.7848581075668335, -0.9767645597457886, -0.46523281931877136, -2.2030811309814453, 0.6965153813362122, -1.1089357137680054, 2.0434322357177734, 2.0816421508789062, -0.8829286694526672, -1.3147400617599487, -1.8805859088897705, -1.4181333780288696, 0.23069675266742706, -1.9602383375167847, 0.6850510835647583, -0.1363319605588913, 0.8342917561531067, 1.3103610277175903, 0.7358253002166748, 0.22215788066387177, -0.1417221873998642, -1.116942048072815, 1.2727751731872559, 0.07509144395589828, 1.7594937086105347], [-0.32006731629371643, -0.21516962349414825, -0.28213831782341003, -1.2798172235488892, -0.9824729561805725, 1.757857322692871, 0.47052332758903503, -1.1742368936538696, 1.8819206953048706, -0.17385777831077576, -1.012701153755188, -0.2322685867547989, 0.30971452593803406, 0.22215698659420013, 0.15516848862171173, 0.6445193886756897, 1.8567808866500854, 0.37340107560157776, -1.656316876411438, -1.3856161832809448, -0.9266127943992615, 0.11960216611623764, -0.7864330410957336, 1.4113808870315552, 0.2156348079442978, -0.9427183866500854, 0.5621750354766846, -0.7912927865982056, 1.0160256624221802, -0.36803510785102844, 0.9902932047843933, -0.07535024732351303, -1.0372626781463623, 0.6315737366676331, 0.7205073833465576, -1.852752447128296, -0.22668121755123138, 1.2525169849395752, 0.7083773016929626, 0.782699465751648, 0.16695430874824524, -1.1143895387649536, -0.2658848166465759, 1.0194265842437744, -0.5742186307907104, -1.7214540243148804, 0.4379402995109558, -1.0770939588546753, -1.5712834596633911, 0.18828485906124115, -0.6736326813697815, 0.7043880820274353, 1.641110897064209, 1.5056438446044922, 0.22910019755363464, 0.44146618247032166, 0.7417685985565186, -0.6608818769454956, 0.39550405740737915, -0.34757164120674133, -0.755465030670166, 0.9247518181800842, -0.7969450950622559, -0.686388373374939], [1.0037648677825928, 0.2740756571292877, 0.2726782560348511, -0.3342297375202179, -1.264143466949463, -0.5210639238357544, -0.32106274366378784, -0.15056636929512024, -1.3826606273651123, -0.6118113398551941, -1.8240904808044434, 0.5041258335113525, 0.9976012706756592, 1.2507297992706299, -2.5213539600372314, -1.8559620380401611, -1.958361268043518, 0.004258240573108196, -0.37903961539268494, 0.08256624639034271, 0.9000038504600525, -0.5310372114181519, 0.6221603155136108, 0.630849301815033, 0.7770451307296753, 0.5565571784973145, -0.16659735143184662, 0.7557668685913086, 2.353525400161743, 0.030489923432469368, -0.4556790292263031, -0.67305588722229, -1.255462884902954, -1.4537291526794434, 1.4736217260360718, 1.3935375213623047, -0.9537152051925659, -1.9783177375793457, -1.3208929300308228, 0.9527212977409363, 0.4604252576828003, -0.2238379716873169, 0.9533017873764038, 0.07118288427591324, -0.1260749250650406, 0.7119954824447632, 0.7892980575561523, 0.15802660584449768, 0.14297470450401306, -0.39634910225868225, -0.7610493302345276, 0.23738770186901093, -0.08071514964103699, -0.6282267570495605, -0.32866454124450684, -0.44001033902168274, -0.9762592911720276, 0.2721497118473053, 2.0991175174713135, 1.6082489490509033, 0.7354166507720947, -0.17463412880897522, 0.916833221912384, -0.5865351557731628], [0.19397804141044617, 0.5279886722564697, 0.6633236408233643, 0.1400015950202942, -0.6432065367698669, -0.6528146266937256, -1.8803715705871582, -1.6463481187820435, -1.097617745399475, -0.9733201265335083, -1.031534194946289, 1.4842559099197388, -1.6162896156311035, 0.7608085870742798, 1.2176570892333984, 2.022906541824341, -0.5203220844268799, -1.3352229595184326, -0.1142217293381691, 2.068993330001831, 0.9536837339401245, -0.34402716159820557, -0.27301260828971863, -0.2750716507434845, -0.8817880153656006, 0.4853837192058563, -0.7102953791618347, -0.7616817951202393, -1.4079896211624146, 0.1281479448080063, 0.3266282081604004, -0.6188639998435974, -1.1604390144348145, -0.883870542049408, -0.4647960066795349, -1.7736949920654297, -0.3575102388858795, -1.217734932899475, 0.6993986368179321, -1.4855722188949585, 0.6914548873901367, -1.301350712776184, -0.6745573282241821, 0.6748437881469727, 0.21905794739723206, -0.7190840244293213, 1.6509673595428467, -0.5130998492240906, 0.5937730669975281, 0.6074695587158203, -0.025245504453778267, -0.7681542634963989, 0.6369497179985046, 0.5176141858100891, -1.7493590116500854, 0.7623032927513123, -1.1899845600128174, -0.7947408556938171, -2.82767653465271, -1.3013155460357666, 0.2094157189130783, 0.1299687623977661, 0.4791620969772339, -0.5330938100814819], [0.017424684017896652, -1.1419941186904907, -0.6293752789497375, -0.1620783507823944, 0.19132573902606964, 0.5660067200660706, 0.3110748529434204, 0.30955004692077637, 0.08205728232860565, 1.9470518827438354, -0.8449277281761169, -0.07835642993450165, 1.4731353521347046, 1.1610652208328247, 0.9713546633720398, 1.621350646018982, -0.6411831974983215, 0.8876215815544128, 0.05208667367696762, -0.8423994779586792, 1.0523109436035156, 1.0018086433410645, -1.0365151166915894, 0.28029459714889526, 0.14791211485862732, -1.489245057106018, 0.712105393409729, 0.48594680428504944, -0.7836148738861084, 0.1461973637342453, -0.8441452980041504, 0.12297903001308441, -0.07774446904659271, 0.18175238370895386, -1.3988348245620728, 1.4185585975646973, 1.3762553930282593, 0.700139582157135, -0.8449930548667908, 0.03459940850734711, 1.2713282108306885, -0.3998630940914154, -0.702057421207428, -0.32027116417884827, -0.933139443397522, -0.22124554216861725, -0.42051178216934204, -0.3040914833545685, -0.5649374723434448, -0.362377792596817, 0.5871374011039734, -0.008645514026284218, -1.2845758199691772, 0.07141731679439545, -0.7812011241912842, 0.41762134432792664, 1.4995077848434448, -1.576167106628418, -1.285900354385376, 0.08201304823160172, -0.9206066727638245, -0.7200212478637695, 0.5153511762619019, 0.18330591917037964], [-1.9486298561096191, 1.0300580263137817, 0.12008284032344818, -0.7765270471572876, -2.088099718093872, -1.1492624282836914, 0.48849204182624817, -2.169356346130371, -0.6090213060379028, 1.1014752388000488, 1.2246227264404297, 0.8035911917686462, -0.39641237258911133, 0.7814345359802246, -0.6315311193466187, -0.21050848066806793, 0.6919806003570557, 0.6267475485801697, 1.3901309967041016, -0.8670070767402649, -0.08038003742694855, 1.0796706676483154, 0.1020406112074852, 0.059960972517728806, -0.996688187122345, 0.08841630816459656, 0.47332313656806946, 0.7259958982467651, 0.6386786103248596, -1.4675660133361816, 0.17629775404930115, 1.2763454914093018, 0.6161226034164429, 0.5807510614395142, -0.02680635266005993, 0.36323583126068115, -1.8048099279403687, 0.9829932451248169, 0.17258957028388977, 0.13080357015132904, 1.6504794359207153, 1.0468083620071411, 0.10892055928707123, -1.1367058753967285, 1.161076545715332, -0.2688332498073578, 0.11655537039041519, 0.3423381447792053, -0.36576664447784424, -1.6745072603225708, -0.5689123272895813, 0.5052719712257385, -0.7873809933662415, -0.16075336933135986, -1.3147592544555664, -2.7279610633850098, -0.2187395840883255, -0.9728051424026489, -0.33902373909950256, 0.4303939640522003, 1.6077319383621216, -0.49305459856987, 0.8447489738464355, 0.8794546723365784], [1.6830497980117798, -0.3867843449115753, -0.3876098096370697, -0.9281277060508728, 1.2870014905929565, -0.9544389247894287, -0.972344696521759, 0.16537067294120789, 1.036615014076233, -1.0155651569366455, 0.5011022090911865, -1.7545801401138306, 0.5088239312171936, 0.8487682342529297, -1.2871454954147339, -0.07180285453796387, 0.12304215878248215, 0.7133579254150391, 2.051515817642212, -0.19926053285598755, 1.1472889184951782, 0.2492511123418808, 0.2831900417804718, 1.2071475982666016, 1.5840870141983032, 1.5280367136001587, -3.1475942134857178, -0.5048527717590332, -1.050025463104248, 1.787462592124939, 1.3802615404129028, -1.3218640089035034, -0.2136303037405014, 1.031640648841858, 0.6179369688034058, 0.3453536927700043, 0.6774208545684814, -0.8314410448074341, -1.0781044960021973, 0.23000304400920868, 0.7029093503952026, 1.4384287595748901, 1.531013011932373, -0.10760282725095749, -0.2585236728191376, 0.2677159011363983, -0.1445102095603943, 1.7179820537567139, -1.893690586090088, -0.21746636927127838, 1.4258800745010376, -2.1665902137756348, -0.6831483244895935, 0.43269678950309753, -0.6562323570251465, -0.6794593930244446, -0.5946871638298035, -0.42874595522880554, 0.8262261152267456, 0.6504759788513184, 0.29631638526916504, 2.159022331237793, 1.574601650238037, 0.7092729806900024], [0.23781585693359375, 0.47241735458374023, -0.21363867819309235, 0.18692854046821594, -0.1453007459640503, 0.2321069836616516, 0.1375773549079895, 1.8848369121551514, -1.360471248626709, -1.6275858879089355, -0.6798852682113647, 1.7217072248458862, -0.625300943851471, 1.3608490228652954, -1.0918090343475342, -2.386194944381714, -0.5069004893302917, -1.1926846504211426, 0.14085105061531067, -0.34639981389045715, -2.048105239868164, -0.34885144233703613, 0.644177258014679, 0.4725489318370819, -0.6021295189857483, -0.4814693033695221, -0.5829557776451111, 0.2669430077075958, -0.6805163025856018, -1.095902681350708, 0.0701960101723671, -0.416358083486557, 0.12243537604808807, 0.17404423654079437, -0.32750725746154785, -0.30413246154785156, -1.6256011724472046, 1.8520435094833374, -1.3045467138290405, 1.2990442514419556, 1.095638394355774, 1.0054547786712646, -0.8841896653175354, 1.1975733041763306, -0.9127114415168762, -0.6677427291870117, 0.264066606760025, -1.3585360050201416, -0.8393270969390869, 0.504948616027832, -0.5585324168205261, 0.6320072412490845, -0.4553675651550293, -0.6832650303840637, -0.396070659160614, 0.5086619853973389, -0.7811021208763123, 0.44253018498420715, 0.1750027984380722, 1.303294062614441, -0.14557446539402008, -0.3362361192703247, 1.585283637046814, -0.42725178599357605], [-0.10415138304233551, -0.17183826863765717, -0.05599328130483627, 0.5723850131034851, 2.0063092708587646, 0.015275761485099792, -0.6514871716499329, 0.6468509435653687, -0.023228535428643227, -0.22440475225448608, -0.018396824598312378, 0.37054747343063354, -0.771745502948761, -0.10077888518571854, -0.4510098695755005, 1.5307306051254272, -0.1490856558084488, -0.6014028191566467, 2.137557029724121, 1.116076946258545, 1.0739272832870483, 1.2245780229568481, 0.09152797609567642, 0.08026038110256195, -0.8216890692710876, 0.36860597133636475, -0.02133749984204769, 0.18143330514431, 0.3948540985584259, 0.20208539068698883, 0.056958429515361786, 0.34035778045654297, 0.3415057063102722, -0.4127059876918793, 1.002406120300293, -0.8895400762557983, 0.2566570043563843, -0.449501097202301, -0.4911540150642395, 0.28337860107421875, 1.117268443107605, 0.1688016653060913, -0.8421568870544434, -0.20985528826713562, 0.5908369421958923, -0.6587183475494385, -2.3752403259277344, 0.3500862717628479, -0.2137853354215622, 0.3991796374320984, 1.279053807258606, -0.5765963196754456, -0.7069488763809204, 0.33455437421798706, 0.3912341892719269, -0.5211014151573181, -1.0475784540176392, 0.6981430053710938, -0.22327634692192078, -0.6584818959236145, 1.050512671470642, 0.296809583902359, -0.4558665156364441, 0.7775726318359375], [-0.09253330528736115, -0.8778989315032959, 0.7493760585784912, -0.9306634664535522, 1.9966121912002563, 0.22208943963050842, -0.8825114369392395, -1.9754586219787598, 0.7391842603683472, 0.07737895846366882, -0.031977809965610504, -0.45578354597091675, -1.310278296470642, 1.0480693578720093, -0.66935795545578, 1.1944797039031982, -0.3033362329006195, -0.37665849924087524, -0.6327268481254578, 1.204217553138733, -1.013884425163269, -1.049654245376587, -0.2421601414680481, 0.9408963918685913, 1.236672043800354, 1.0244426727294922, -1.3288006782531738, -0.8683459162712097, 0.7103984355926514, 0.005573654547333717, 0.34044018387794495, 0.6948410272598267, -1.7324724197387695, -1.2210160493850708, 1.1971606016159058, 1.106053113937378, -0.2739928960800171, 0.8717883229255676, 0.17607754468917847, 1.2519882917404175, 0.08123794943094254, -0.2027420699596405, 0.8956261277198792, -1.2991093397140503, -0.5695028901100159, 0.04971013218164444, -0.6225711703300476, -0.0359947569668293, 0.3895072042942047, -0.5683552622795105, 1.3724042177200317, 0.5527842044830322, 0.8939685225486755, 0.3741207420825958, -0.00021067216584924608, -1.1173148155212402, 0.09330376982688904, -0.04993220791220665, -0.7992621660232544, -1.6211978197097778, 1.1111518144607544, -1.2375813722610474, -0.7150077819824219, -0.6344053149223328], [-0.1918785572052002, 0.5564259886741638, 1.3850244283676147, 0.2629222869873047, 0.5917171239852905, -0.2483038306236267, 0.26742392778396606, 0.33725014328956604, 0.9636170864105225, -0.22255973517894745, 0.7496069669723511, -2.9538679122924805, 0.6577810049057007, 0.29743853211402893, -1.7524328231811523, 2.0069451332092285, -1.1404070854187012, 0.4363015294075012, -1.6225067377090454, -2.2634034156799316, 1.1337227821350098, 0.3156296908855438, 0.29320788383483887, -0.28295084834098816, -0.5054709911346436, 1.4990997314453125, 1.1589205265045166, 1.944791555404663, 1.141236662864685, 0.3019353449344635, -1.7684487104415894, -0.3087735176086426, 0.2300753891468048, -0.42228418588638306, -0.4196629524230957, -0.07589266449213028, -0.9158278107643127, 0.00655599357560277, -0.10803486406803131, -1.2151079177856445, -0.16286832094192505, -2.4132657051086426, -0.38515427708625793, -0.7033065557479858, 2.2631852626800537, -0.4864688217639923, 0.023073500022292137, 0.20202627778053284, 0.3397606313228607, -0.372639924287796, -1.7069402933120728, 0.5434858202934265, 1.0778779983520508, 0.14441530406475067, -0.6650872826576233, 1.780472755432129, 0.15666018426418304, 1.0033602714538574, -1.3131059408187866, -1.2230929136276245, -0.4932416081428528, 0.09297190606594086, -1.1955392360687256, 1.5600744485855103], [0.24302862584590912, 0.6954419016838074, 0.7138432264328003, -0.5474516749382019, -0.25375649333000183, -0.7268286347389221, -0.49890077114105225, 3.0788536071777344, 0.48073941469192505, -0.46638932824134827, 0.19876879453659058, 0.19546590745449066, -1.1394208669662476, -0.04823892563581467, -1.1662604808807373, -0.9322264790534973, 1.1054152250289917, -1.6168522834777832, -0.11457132548093796, -0.6977694630622864, -0.6562637686729431, -0.7400193214416504, 1.2984462976455688, 1.6267536878585815, 1.0809093713760376, -0.7977617979049683, 0.23598326742649078, -0.03132682293653488, 1.149256944656372, -0.6461175680160522, 0.05420392006635666, -0.3778708875179291, 0.39213985204696655, -0.2967390716075897, -0.5993135571479797, 0.07735694944858551, -0.5636400580406189, 0.13551262021064758, 0.0014960719272494316, 1.2408322095870972, -0.49534371495246887, -1.219106912612915, -2.3938851356506348, 0.10247179120779037, -2.549877882003784, 0.07603734731674194, 2.001969575881958, 0.442790687084198, -0.7545114159584045, 0.7574811577796936, 1.4136415719985962, 1.9063700437545776, 0.2539016604423523, 1.402557134628296, 0.651654839515686, -0.5024616122245789, 1.2930927276611328, -1.1810921430587769, -0.42972344160079956, 0.5596165657043457, -0.0487343892455101, 1.548370599746704, 0.903475284576416, -0.8537595868110657], [1.429352879524231, -0.6681401133537292, 0.49377331137657166, 0.6555547118186951, -0.8545230627059937, 1.6046385765075684, -0.750742495059967, 0.9386118054389954, -0.9700701832771301, 0.476772278547287, -0.904207706451416, 0.9190628528594971, 2.203728199005127, -0.5343399047851562, 0.33375051617622375, -1.6697018146514893, -1.381832242012024, 2.2750980854034424, 0.14334584772586823, -1.1234171390533447, -0.08192072808742523, -1.000378966331482, 0.48097798228263855, 0.3208986818790436, -0.1738509237766266, 0.1040882021188736, -0.5859596729278564, -3.009101390838623, 0.8407573103904724, -0.657346248626709, 1.612436056137085, -0.962633490562439, -0.33056989312171936, 0.8478224277496338, 1.2715140581130981, 0.7227744460105896, 1.1604104042053223, -0.05617370828986168, 1.0800038576126099, -0.5246838927268982, -0.3437363803386688, 0.2116715908050537, -1.6352986097335815, 0.008104970678687096, 0.18089574575424194, -0.6606308817863464, -1.5079119205474854, -0.11665914207696915, 0.8370547890663147, -0.8263233304023743, -0.4298401474952698, 1.3999196290969849, -0.04374788701534271, 1.4740040302276611, 0.48454681038856506, -0.5983846187591553, -1.8606541156768799, -0.1402154117822647, -0.7749874591827393, 0.3882908523082733, -1.752928614616394, 1.3459594249725342, 0.3883133828639984, -1.0222463607788086], [1.1201624870300293, 0.9180666208267212, 1.2536919116973877, 0.799428403377533, 0.7925719618797302, -1.6412261724472046, 0.8260143995285034, 1.0737965106964111, -1.1505650281906128, 0.47656652331352234, -0.7888782620429993, 1.7779144048690796, 0.6727268099784851, -0.6514595746994019, -0.5850746035575867, -1.4515944719314575, 1.384367823600769, -0.7799869775772095, 0.72969651222229, 1.005242943763733, -0.2672305405139923, 0.08578836172819138, 1.2993221282958984, 0.5956020355224609, -0.022607693448662758, -0.3963909447193146, 1.2762559652328491, -1.317130446434021, 0.004009269643574953, 0.25433608889579773, -0.40967288613319397, 0.20840178430080414, 1.1340856552124023, 0.8738569021224976, -1.8283960819244385, 0.03823612257838249, -0.8311097025871277, -1.0916624069213867, -1.2555643320083618, -0.834479033946991, -1.9195643663406372, -0.1804162561893463, -0.18774385750293732, 1.7038512229919434, 0.32714298367500305, -0.24272896349430084, -0.6944019794464111, 0.4296216070652008, -1.359811782836914, -0.5989299416542053, 0.6478472948074341, 1.1535252332687378, -0.9752056002616882, 0.9690470695495605, -0.9488312005996704, 0.8495238423347473, 0.9035091400146484, -0.1908470243215561, -0.3165399432182312, 0.4616382420063019, 0.3013295531272888, 1.4266082048416138, -3.0418307781219482, -1.1236284971237183], [0.3335743546485901, -0.21183285117149353, -1.412532091140747, 1.9443739652633667, -1.1744791269302368, 0.10487109422683716, 1.3095322847366333, 0.33512935042381287, -0.7009047269821167, -0.2266836017370224, -0.9316744804382324, -0.07432595640420914, -0.8881504535675049, 1.5257455110549927, 1.6635937690734863, 0.7863392233848572, 1.7254376411437988, -1.5205817222595215, -0.6116365790367126, 0.2792334258556366, 0.9733039736747742, 0.5153456926345825, 0.19102832674980164, 0.561457633972168, 1.2994160652160645, 0.3063046634197235, 0.5607515573501587, 1.7311395406723022, -1.078112006187439, 0.3856423497200012, 1.6297554969787598, -0.6255025267601013, 0.03762456402182579, -2.5137939453125, 1.0756653547286987, -0.9960142970085144, 0.03197324648499489, 1.5880627632141113, 1.0837736129760742, 0.18465612828731537, -0.9776319861412048, -2.59082293510437, 1.930978536605835, 0.8939003944396973, 0.44268396496772766, 0.637397825717926, 0.027764001861214638, 0.03038848377764225, 2.374616861343384, 0.3295527696609497, -2.3236143589019775, -0.4731883406639099, 1.5635061264038086, 0.10235798358917236, 2.313284158706665, 0.4069087505340576, 0.7828937768936157, 0.8149153590202332, -0.16554290056228638, -1.1683975458145142, 0.8070653080940247, -1.105386734008789, 0.6300342679023743, -0.8635131120681763], [0.4758951663970947, 0.9501171708106995, -0.2295151799917221, 0.8167172074317932, -1.047334909439087, 0.39289170503616333, 0.30870094895362854, -0.9469183087348938, -1.4044430255889893, -1.9538630247116089, -1.5628494024276733, 0.43612030148506165, 0.0642034262418747, 0.14634302258491516, -0.32641375064849854, 0.26412433385849, 0.8306729793548584, 0.9011862874031067, -1.6023855209350586, 0.10871925204992294, -1.0906131267547607, 0.745311439037323, 0.141431525349617, 0.6353816390037537, 0.779141366481781, 0.4692002832889557, 0.6466018557548523, 0.49086886644363403, 0.05154275521636009, 0.0017191946972161531, -1.002906322479248, 0.2578374445438385, -0.17458944022655487, 1.0198861360549927, -0.5611925721168518, 2.2177693843841553, -0.09884842485189438, 0.750434398651123, 1.2684258222579956, 1.0151851177215576, -0.32152441143989563, 1.5842442512512207, 0.1973028928041458, 0.0075953626073896885, 0.7233654856681824, -0.21050770580768585, 1.2676681280136108, 1.3712836503982544, -0.04281585291028023, -0.9279042482376099, -0.9143623113632202, -0.13075192272663116, -1.3916294574737549, -1.7570549249649048, 1.073786973953247, 0.5244706869125366, 0.9381835460662842, -1.6014704704284668, -0.2601964771747589, 1.033964991569519, 0.5190954804420471, -0.629886269569397, -0.06393497437238693, -0.40788957476615906], [-0.2771640121936798, 1.0053147077560425, 0.4323946535587311, 0.35836464166641235, 1.9826663732528687, 0.24982517957687378, 1.4640662670135498, 1.3901320695877075, -0.7998579144477844, 0.10545845329761505, 1.6370948553085327, 0.3752085268497467, -0.5300641655921936, 0.6890290975570679, 1.052716851234436, 0.35140782594680786, 0.22771471738815308, 0.4788152277469635, 0.9504914283752441, 1.212896466255188, 0.30840522050857544, 0.7764950394630432, -0.27706480026245117, 0.5233935117721558, -1.0338691473007202, 0.438738614320755, -1.988661766052246, 0.15454693138599396, 0.8055369257926941, 1.3466365337371826, -0.3985109329223633, 0.8805795907974243, 0.1713339239358902, -1.4600485563278198, 0.4724174737930298, 0.1385975480079651, 0.07458201050758362, -0.6311600208282471, -0.02136678248643875, 2.1666882038116455, 0.7243411540985107, 0.2442779541015625, 0.830765962600708, -0.8005874156951904, 0.2604354918003082, 1.3306710720062256, 2.3726561069488525, -1.6466622352600098, 0.2230474054813385, -0.22376668453216553, 0.3678688108921051, -0.8014220595359802, 0.7448004484176636, 0.3673872947692871, -0.5828290581703186, 1.2611514329910278, 0.10855738818645477, -0.3190106451511383, -0.7051566243171692, -0.14620904624462128, 0.7133803367614746, -0.21278119087219238, 0.09345432370901108, 2.4236392974853516], [-0.22332392632961273, -0.7003857493400574, -0.2862226366996765, -0.0001382599730277434, 0.12979184091091156, 1.483569622039795, -1.9400299787521362, 0.2248452752828598, -1.7210493087768555, -0.04183223843574524, -0.4325670301914215, 1.085514783859253, 0.41816213726997375, -0.22601604461669922, 0.2383507490158081, -0.46580666303634644, 2.294715642929077, 0.7470407485961914, 0.7792612910270691, 0.9545224905014038, -0.461622953414917, 0.7978981137275696, -0.6654815077781677, -0.26072147488594055, 0.02624308317899704, 1.531312346458435, -0.25040891766548157, -0.1590423583984375, -1.0778297185897827, -1.9498791694641113, 0.5981649160385132, 0.880574643611908, 0.6789225935935974, 1.7288002967834473, 2.0028584003448486, -1.3269604444503784, -1.0961506366729736, 0.5952479243278503, -0.0886855497956276, -0.28444820642471313, -0.8110710978507996, -1.0963134765625, 0.3368425667285919, 1.8565399646759033, 0.4820462763309479, 1.663016438484192, -0.8838170766830444, -0.4849395453929901, 0.19155901670455933, -0.13717085123062134, -0.19257690012454987, 1.1777602434158325, 1.2861250638961792, -1.2062413692474365, -0.3149895668029785, 0.45061543583869934, 0.9145123958587646, 0.30498167872428894, -0.46934792399406433, -1.511089563369751, -1.282427191734314, 0.345318466424942, 0.8944289684295654, -0.1762414425611496], [-0.5266894698143005, -1.2413235902786255, 1.2087863683700562, -0.11316192895174026, 0.8628555536270142, -0.5477952361106873, 0.4295462369918823, 2.3167412281036377, 0.32483577728271484, -0.47256267070770264, 0.11633514612913132, -0.1977776736021042, -3.5682718753814697, -0.07364557683467865, -1.4038665294647217, 0.0697731152176857, -1.296576976776123, -0.7815490365028381, -0.8006774187088013, 1.0016956329345703, 0.07944448292255402, -0.8808862566947937, -0.4844701588153839, -1.5334151983261108, 0.6940150856971741, 0.4321795403957367, -0.426312118768692, -2.423171281814575, -0.44185709953308105, 0.7632666826248169, -2.6390974521636963, 0.6171520352363586, -0.4240786135196686, 1.6684902906417847, -0.4968782067298889, -1.8170156478881836, -2.190162181854248, -0.4095042645931244, 1.6433711051940918, 0.06123977527022362, 0.16487684845924377, -0.034963395446538925, 0.764315128326416, -0.6343259215354919, -0.8647900223731995, -0.7623844742774963, -0.03239566460251808, 0.2611438035964966, 0.4270839989185333, 0.21418707072734833, 1.8639328479766846, -0.8999586701393127, 1.3151962757110596, 1.901376724243164, 1.360382318496704, 0.006399552803486586, 0.5190929770469666, 1.3369120359420776, -1.0843623876571655, -1.3743834495544434, -1.323525071144104, -0.293978214263916, -0.24710573256015778, 0.595957338809967], [-1.480896234512329, -0.3496030271053314, 1.1598619222640991, 1.6104487180709839, 0.4234345555305481, -0.08957087993621826, -1.1482607126235962, 0.3596876859664917, 0.3676467537879944, -0.9206439852714539, 0.22055910527706146, 0.05561477318406105, -1.7959939241409302, -0.6534876823425293, -0.7792974710464478, 0.20799700915813446, 0.3537057340145111, -0.3291410803794861, 1.886038064956665, -0.6038519144058228, -0.8925085663795471, 2.219261646270752, 0.942720353603363, 0.25599658489227295, 1.1446590423583984, -0.2847750782966614, -2.1599795818328857, 0.20178455114364624, -1.6557797193527222, 0.5693781971931458, 0.01033940352499485, 1.2053310871124268, -0.3954218626022339, -0.3659554421901703, -1.541530966758728, -1.9347658157348633, 0.3099641799926758, -0.5611101388931274, 0.7193281650543213, -0.5403269529342651, -0.12891080975532532, 1.2860898971557617, -1.6829299926757812, 0.30034852027893066, -0.8296031951904297, -0.7941762804985046, -0.7648451924324036, 2.063375234603882, 0.19084219634532928, 0.1249091625213623, -0.12046264111995697, 0.23310013115406036, 0.8364607691764832, 0.0054796002805233, -1.3748118877410889, 0.8301492929458618, -0.4529320001602173, 1.0162489414215088, 0.933669924736023, -1.268130898475647, -0.33154577016830444, -1.379629135131836, -0.42517417669296265, 0.5730960965156555], [-1.321997880935669, -1.0559724569320679, -0.05998784303665161, -0.31178247928619385, -1.4181878566741943, 0.8311573266983032, -0.5019307136535645, -0.4898245930671692, 2.3715975284576416, -1.5025386810302734, -0.7976949214935303, -1.2489153146743774, -0.370695561170578, -0.6758919358253479, 0.1836668699979782, -0.6734673380851746, -0.6512255072593689, -0.36622610688209534, 0.5364974141120911, -0.6666112542152405, -0.29341772198677063, -0.9495864510536194, -0.24463172256946564, -0.43068355321884155, 0.44383925199508667, 0.24261555075645447, 0.25708502531051636, -1.4908708333969116, 1.334406852722168, 0.07059811800718307, 1.3672748804092407, 0.5694132447242737, -0.6726098656654358, 3.1027450561523438, 0.05582798272371292, 0.5669782161712646, 0.8582043051719666, 0.7338087558746338, -0.37074384093284607, -1.3821065425872803, 0.4659430980682373, -1.59156334400177, 0.26309484243392944, 0.1586809903383255, 0.9212432503700256, 0.2709670960903168, -0.43503817915916443, -0.8617342710494995, 0.9717956781387329, -0.6176373958587646, -1.1065984964370728, 1.3283562660217285, -1.4193511009216309, -0.04021435230970383, 0.8136699199676514, 1.3951656818389893, -0.8137638568878174, -0.23252983391284943, 0.17995567619800568, 1.3746216297149658, 0.5397915244102478, 0.09089041501283646, -0.8413485288619995, 0.9854353070259094], [0.8997040390968323, 0.9879849553108215, 0.5713170170783997, -0.24696840345859528, -0.19665831327438354, -0.3952125608921051, 0.3575811982154846, 0.7038683891296387, -0.7987713813781738, -2.204469680786133, -0.3488506078720093, 1.7738624811172485, -0.5024548172950745, 0.018536098301410675, -1.8436064720153809, -0.1273760199546814, 0.21388986706733704, 0.2189558446407318, 1.4602051973342896, -0.29783371090888977, 1.9643980264663696, -1.3568382263183594, 0.014239277690649033, -0.3151683807373047, 0.06097909063100815, 0.12269696593284607, -1.2897882461547852, -0.14773042500019073, 0.3947838246822357, 1.3311512470245361, 1.1030521392822266, 1.860112190246582, 0.7840155363082886, -0.428732693195343, -1.5360829830169678, -0.1426696926355362, 1.406044363975525, 0.5255894660949707, -0.707798957824707, -1.173080563545227, 2.7492871284484863, -1.6812821626663208, -1.1545796394348145, 2.050711154937744, 0.5012129545211792, 0.8526820540428162, 0.7207650542259216, -0.5610532760620117, 0.01862635836005211, 0.45677468180656433, 2.1766393184661865, -0.6206905245780945, 0.12352179735898972, 0.3686409592628479, 0.3925713896751404, 0.11572380363941193, -0.9237598180770874, -0.41260892152786255, -0.7322648167610168, 0.6995624303817749, -0.7174321413040161, 0.2898227870464325, -1.5251445770263672, -1.544731616973877], [-0.4830946624279022, -0.9731082320213318, -0.39825916290283203, -1.0539137125015259, 1.56057870388031, -0.10322294384241104, -1.9216619729995728, -0.8159262537956238, 1.472577691078186, 0.07358144968748093, -0.30198293924331665, -0.7774360775947571, -0.02844928950071335, -0.373942494392395, -1.5006455183029175, 1.1828219890594482, -0.7126621603965759, -1.0625522136688232, -0.5166312456130981, 0.3667823076248169, -0.01655532233417034, -1.7842909097671509, -1.2819546461105347, 0.28681495785713196, 1.0752391815185547, -0.005150193348526955, -0.6802356243133545, -1.8144532442092896, -1.6943168640136719, -1.865944266319275, 0.3529190123081207, 1.4969441890716553, -1.2762441635131836, -0.6496002674102783, -1.0239932537078857, 0.6294729113578796, -0.07829646021127701, -1.0716794729232788, 0.9108657240867615, -0.4285311698913574, -0.5772819519042969, -0.08976402133703232, -0.0976228266954422, 0.22711552679538727, 0.34213030338287354, -0.13699305057525635, -0.39826682209968567, 2.3108127117156982, 0.5053920745849609, -0.2669225335121155, -1.1713064908981323, -0.20906661450862885, 1.121457576751709, 0.6140750646591187, -0.9973312020301819, -0.30799129605293274, 1.2776488065719604, -0.5433463454246521, 0.8922044038772583, -0.8504400849342346, 0.5220670700073242, -1.0822231769561768, -0.26573553681373596, -0.8079676628112793], [0.22477653622627258, 0.18514588475227356, -1.8588647842407227, 0.4860136806964874, 1.162613034248352, -1.8230918645858765, 0.9580041766166687, 0.4632505476474762, 0.8169647455215454, 0.43201690912246704, -0.7037813067436218, -0.878419041633606, 1.7788829803466797, 0.5193880200386047, -1.5508887767791748, -1.8399935960769653, -0.10855918377637863, -1.2445062398910522, -1.477358102798462, -0.35328409075737, -0.44880035519599915, -0.9778802990913391, 0.07824020087718964, 1.4451916217803955, -0.4134571850299835, 0.7376598715782166, 0.5134190320968628, 0.28848373889923096, -0.3519445061683655, 1.4509044885635376, -2.472278594970703, 1.038520336151123, 0.9805953502655029, -0.565430760383606, -0.9284454584121704, 1.2013651132583618, -0.349886029958725, -0.7546740770339966, 0.8431640267372131, -1.1008384227752686, -0.22307252883911133, -0.9652523994445801, -0.5541714429855347, 1.4637521505355835, -1.7128022909164429, 0.3658931851387024, -1.0903478860855103, 1.6565126180648804, -1.556880235671997, -0.20393773913383484, -0.701153039932251, 0.03509775549173355, 0.41614967584609985, 0.2726166248321533, 0.5596195459365845, 1.3324196338653564, -0.97077476978302, -0.08396723866462708, 0.36586031317710876, -0.1357761174440384, 0.8501551747322083, -0.16030265390872955, -0.4788769781589508, -0.1359962821006775], [-0.26815128326416016, -0.8699070811271667, -0.7893341779708862, -0.3460462689399719, -0.5584409832954407, -0.09012877941131592, 1.2277580499649048, -0.9694807529449463, -0.35355010628700256, -0.5205480456352234, 2.2241358757019043, -0.8832031488418579, 0.30745238065719604, -0.8784077763557434, -1.5706380605697632, 1.0743297338485718, -1.8108264207839966, -1.644011378288269, -0.8147952556610107, 2.0066123008728027, 0.3188849985599518, -0.7233914732933044, 1.293254017829895, -0.4355701506137848, 0.564436674118042, -1.2383499145507812, -0.08591847866773605, 1.4198228120803833, 2.0476202964782715, -1.729814052581787, 0.6989508271217346, -1.8574830293655396, -0.9432002902030945, -0.6615650057792664, -0.9851111769676208, 0.3545858860015869, -0.5323971509933472, 0.5612621903419495, -0.6439352035522461, -0.17983776330947876, 1.67389976978302, 0.5870153307914734, -0.41873544454574585, 0.7614550590515137, -1.2316008806228638, -1.0389370918273926, -0.18996405601501465, 1.0043855905532837, 0.6120196580886841, 1.193235993385315, 0.8373117446899414, -0.8431035280227661, -2.6551735401153564, 0.28276315331459045, -0.4760223925113678, -0.2673710584640503, 0.11468926817178726, 0.9667536020278931, 0.0572119802236557, -0.28475499153137207, -0.5369041562080383, -0.1467396318912506, -0.5145008563995361, 1.0710622072219849], [-1.1697429418563843, 0.0368276983499527, 1.1697360277175903, 1.3112396001815796, -1.78202223777771, 0.6057930588722229, -0.7856107950210571, 1.3098598718643188, -0.3491028845310211, -0.8369165062904358, -0.44872185587882996, 0.7982999086380005, -0.2292308360338211, -0.5355100631713867, -0.5566838383674622, -0.77303546667099, 0.5523545742034912, 0.8676775693893433, -0.977114737033844, 0.0991399735212326, -0.6244791746139526, -1.1892718076705933, 0.8177715539932251, 1.7113656997680664, 0.6803856492042542, -1.3568174839019775, -1.4000389575958252, 0.11552700400352478, -1.7223621606826782, 0.9066863059997559, -1.0596728324890137, 0.3279306888580322, 0.2960476577281952, 0.5347093343734741, 2.6389286518096924, -0.40396201610565186, 0.041038546711206436, 1.038712739944458, 0.28513187170028687, -0.7522333264350891, -0.2438926100730896, -0.18808571994304657, -1.4909782409667969, 0.10386055707931519, -0.767759382724762, -0.26895102858543396, -0.8654883503913879, -0.3902316093444824, 2.4028425216674805, 0.013384178280830383, -0.5978103280067444, -0.20861995220184326, 1.5892707109451294, -2.5260963439941406, 0.580727756023407, 0.5868950486183167, -0.7489754557609558, 0.43305960297584534, -0.2575587034225464, -0.0964331328868866, -0.036782409995794296, -0.6530842185020447, 1.7964470386505127, 2.026095390319824], [0.26911765336990356, 0.46569329500198364, -0.9705197811126709, 1.9347153902053833, 0.27145856618881226, 0.1551770269870758, -0.18302734196186066, -0.6887950897216797, -1.2309752702713013, -0.8786396384239197, -1.479557752609253, -0.7825868129730225, 1.0332417488098145, 1.9728940725326538, -0.38574641942977905, 0.03396062180399895, 0.847994863986969, -1.0317606925964355, -0.7101261615753174, 1.0439099073410034, 0.9131187796592712, 0.1424129456281662, -0.18629193305969238, -1.0947939157485962, -0.2190704345703125, -0.32954666018486023, 1.1499720811843872, -0.3312431871891022, 1.415915846824646, 0.28793323040008545, 0.5847476124763489, 0.21350619196891785, -0.42977336049079895, 0.7530262470245361, -0.09324561059474945, 0.5462697744369507, -0.05149568244814873, 0.8074607253074646, -0.7737814784049988, -1.3711775541305542, -0.5159390568733215, 1.1590243577957153, 1.1436940431594849, 1.2119390964508057, 1.8846112489700317, 2.366583824157715, -1.1831700801849365, -0.03683636710047722, 0.24328641593456268, -0.5092443823814392, 1.4434354305267334, 2.02335524559021, -0.1630939543247223, 0.2740708291530609, 1.6210111379623413, 0.12172127515077591, 1.258550763130188, 0.5612295866012573, 1.129191517829895, 1.2414968013763428, -0.17484557628631592, -2.328909158706665, 0.8472797274589539, -1.083217978477478], [-0.33968594670295715, 0.005566458683460951, -0.8390388488769531, 2.004345178604126, -0.08182612806558609, 1.9134446382522583, -1.6533327102661133, 0.6458463072776794, -1.1891969442367554, -0.935276448726654, 0.19166819751262665, -0.4134397506713867, -0.14152441918849945, 0.2742633819580078, 1.8744237422943115, 0.779327392578125, 1.072306513786316, 0.11431647837162018, -0.7494181990623474, -0.22997145354747772, 0.652851402759552, -2.4021010398864746, -0.7390604019165039, 0.08249850571155548, -0.47353455424308777, -1.3320162296295166, -0.6485472917556763, 1.0868277549743652, 1.5847023725509644, -0.5310707688331604, 0.6129770278930664, -0.3243376910686493, 1.3840833902359009, 2.3024659156799316, -0.5818678736686707, 0.6291011571884155, 0.4366823434829712, 0.3387015759944916, -0.8325101137161255, 1.1174925565719604, -0.9316748380661011, 1.6913477182388306, -1.267736554145813, -1.2000293731689453, 1.309756875038147, 1.3089215755462646, -0.9962835907936096, -1.0728412866592407, 0.9949646592140198, 0.28939852118492126, 0.1269262582063675, 0.9151942729949951, -0.6538174748420715, -0.22786006331443787, -0.47551611065864563, 0.8889802694320679, 2.0409493446350098, -0.20262138545513153, 0.5155182480812073, 0.4408029019832611, 0.36929741501808167, -1.3784053325653076, 0.5896158814430237, -0.2240675985813141], [0.7390322685241699, 0.30947014689445496, -0.24113468825817108, -1.3890365362167358, 1.316283941268921, 0.7480059862136841, 0.17535224556922913, 0.3886481523513794, 0.5583088397979736, 0.6215818524360657, 1.2359237670898438, 0.28101593255996704, 0.38309288024902344, 0.019383851438760757, 1.0748127698898315, 0.7918866872787476, -1.4859286546707153, -1.1144644021987915, -0.20421640574932098, 0.6373080015182495, -0.457114577293396, -0.6509840488433838, -1.0390315055847168, 0.14975321292877197, -1.3364742994308472, -0.22404560446739197, -0.18201564252376556, -0.231856569647789, 0.3572659492492676, 0.6389525532722473, 0.10468464344739914, 1.201500654220581, -1.7239503860473633, -0.9811055660247803, 0.6261816620826721, 0.415444016456604, -1.4373726844787598, 0.6796974539756775, 0.4548412263393402, -0.7982642650604248, -1.161878228187561, 1.177103877067566, -0.0073277004994452, 1.5357296466827393, -0.4016292691230774, 2.08206844329834, -1.2008576393127441, 1.6296519041061401, -0.5394940376281738, 0.10556495189666748, -0.8207658529281616, 0.5168762803077698, 1.3404362201690674, 0.2745234966278076, -0.32462456822395325, 1.4112520217895508, -1.4643220901489258, -0.9788647294044495, -0.5504275560379028, -0.39810678362846375, -1.1915614604949951, -0.7481964230537415, -0.726103663444519, -1.0156729221343994], [-0.34435632824897766, 0.8962653279304504, 1.7537150382995605, -0.8306830525398254, 2.289806842803955, -0.7683982849121094, -1.0783467292785645, -0.2304590493440628, -0.8667876720428467, 0.010320084169507027, -0.9599652290344238, 0.3599790632724762, -0.24080684781074524, -0.49043765664100647, -0.8233888149261475, 0.5589107275009155, -0.3085683286190033, -1.0200740098953247, 1.0224957466125488, -0.16560573875904083, 0.06529545783996582, 0.5574166178703308, -0.7783322334289551, -1.5106357336044312, -2.6206161975860596, -1.2225826978683472, 1.8749147653579712, -0.8820498585700989, 0.5877521634101868, 0.4909501075744629, 0.3798474669456482, -2.070441246032715, 1.9445936679840088, -1.3421928882598877, -0.3163570463657379, 0.1209946721792221, 1.1696604490280151, -0.3411479592323303, 0.7152183055877686, -1.3248854875564575, 0.556809663772583, 1.0059856176376343, -1.0043013095855713, -1.1983075141906738, -0.24531249701976776, 0.7368639707565308, -0.11211315542459488, 0.39126190543174744, -1.0436441898345947, 0.5145962238311768, 0.16173098981380463, 0.059032995253801346, -1.5307977199554443, -0.7529306411743164, 0.6443844437599182, -0.1893412470817566, 1.5842386484146118, -0.0021826468873769045, -0.7888084650039673, -0.390423983335495, -0.4303174316883087, -0.9333832263946533, 0.1898082047700882, -0.6759108901023865]]]], "dtype": "float32"}, {"__nd__": [[[[-0.8609941601753235, -0.15011711418628693, -0.009432380087673664, 0.627079963684082, -0.8873878121376038, -0.6780243515968323, -0.2767941355705261, -0.020799363031983376, -1.9006283283233643, -0.5451224446296692, -0.038970183581113815, -0.14682559669017792, -1.4015742540359497, 1.1327099800109863, 0.22871807217597961, 1.1517291069030762, -1.3879507780075073, -1.6342960596084595, 1.27351975440979, -0.7450204491615295, 1.5579816102981567, 0.9073446393013, 1.1559944152832031, 0.4617573022842407, -0.6661596298217773, -0.3570480942726135, -1.506670355796814, 1.1186929941177368, -1.7262548208236694, 0.31424418091773987, -0.4028685986995697, -0.7466201782226562, -1.12118661403656, -0.8747378587722778, 1.163649559020996, -0.059924252331256866, -0.2188754379749298, -0.2898283302783966, -2.4615530967712402, 1.1738632917404175, 0.7628827691078186, 1.6006735563278198, -1.6823328733444214, -0.6869992017745972, -0.47181254625320435, 2.7931878566741943, 0.9848871827125549, 0.08929044753313065, -3.0281474590301514, -0.342560350894928, 2.454758882522583, -0.64263516664505, -0.6821585893630981, 1.4110310077667236, -1.6400502920150757, 0.8704859018325806, -0.3885500431060791, -0.21650516986846924, 1.8835264444351196, 0.29780152440071106, -1.3334356546401978, -0.9532194137573242, -0.007192096207290888, -1.1205824613571167], [-0.19457688927650452, -0.19944307208061218, -0.19464504718780518, 1.8557798862457275, 0.23500680923461914, -0.046168550848960876, 0.11586213856935501, -1.4082331657409668, -1.2693164348602295, -0.7198173999786377, 0.6850296854972839, -0.7150759696960449, -0.06152363121509552, -0.6426714658737183, -1.8674570322036743, 2.4008948802948, -1.3897428512573242, 1.1881537437438965, -0.8483515381813049, 0.7348984479904175, -0.11237379908561707, -1.1314246654510498, -0.47255074977874756, -0.5599592328071594, -0.41205719113349915, 0.31893762946128845, -0.31566885113716125, -0.0863724797964096, 0.5062009692192078, 0.8849202990531921, -1.5477631092071533, 0.3791165351867676, -1.4148736000061035, 0.9451505541801453, 0.006779983639717102, 0.3965170383453369, -1.6775685548782349, 0.09787484258413315, 0.876636266708374, -1.1986517906188965, 2.21089243888855, 0.8052587509155273, 0.4328470230102539, 0.8424301743507385, -0.5205574035644531, 1.3579967021942139, 0.6365922093391418, 1.4861829280853271, 1.781632423400879, 0.04596126452088356, 0.7485219240188599, -0.134673610329628, -1.0345985889434814, -0.642089307308197, 0.6818094849586487, 1.5907974243164062, 1.2922242879867554, 1.351439356803894, 0.5089093446731567, 1.4851685762405396, 1.206158995628357, -0.06392846256494522, 0.4150228500366211, -0.1626569777727127], [-0.6788804531097412, 0.5633284449577332, 1.0792794227600098, 0.5104780793190002, 0.8662750124931335, 0.538988471031189, -0.5724259614944458, -1.9267702102661133, 0.056396737694740295, 0.7719241976737976, 0.06102722883224487, 0.49322712421417236, -1.0543280839920044, -0.20211271941661835, -1.4893463850021362, -0.237335667014122, -1.4185364246368408, 0.36525532603263855, 1.1855779886245728, 0.3297039568424225, 0.18802879750728607, -0.24469371140003204, 1.7280004024505615, 1.1113024950027466, -1.442112684249878, 0.899181067943573, -1.3397588729858398, -0.6415943503379822, -0.9988914132118225, -0.5720195174217224, 1.576799988746643, 0.9577687978744507, -1.1251015663146973, 0.16769176721572876, 0.5103022456169128, 0.7734143733978271, 0.6264649033546448, -0.09860202670097351, 1.7452553510665894, -0.5996032953262329, 0.27051156759262085, -1.6659408807754517, 0.4600701332092285, 0.6253476738929749, 1.8238487243652344, -0.6064818501472473, 1.3375072479248047, -1.1900852918624878, 0.36461544036865234, 0.3462907671928406, 1.2012110948562622, -1.1487438678741455, -0.2045390009880066, 0.7766141295433044, -0.2950698435306549, -2.180274486541748, 0.4037652015686035, 0.16888225078582764, 0.07069504261016846, -0.60124272108078, -0.41383394598960876, 0.18737539649009705, -1.52603018283844, -0.5964658260345459], [-0.12050395458936691, 1.4124664068222046, -0.5976600646972656, 0.45184919238090515, -1.7877347469329834, 0.30097150802612305, -0.49395355582237244, -1.2019065618515015, 0.8691343665122986, 0.25668203830718994, -0.9639866352081299, -1.2927258014678955, 0.9619650840759277, 1.9052363634109497, -1.0401939153671265, -0.4658101201057434, -0.46072185039520264, -0.055252838879823685, 0.46612319350242615, -1.0831584930419922, 0.06549226492643356, -0.97682124376297, -0.2806454598903656, 0.5105993747711182, 0.834696888923645, -0.8119999170303345, -0.7935555577278137, 1.2392529249191284, 0.30754411220550537, -1.4423210620880127, -1.8439207077026367, -0.9988576173782349, -0.890319287776947, 0.14006051421165466, 0.5691749453544617, 0.7250714898109436, -0.581264078617096, -0.6954302787780762, 0.813818097114563, 0.7423849701881409, 1.4858393669128418, 0.10462594777345657, 0.6077096462249756, -0.3084307312965393, 0.536361575126648, 2.4348645210266113, -0.43000954389572144, 0.13141103088855743, 0.7742368578910828, -0.341802179813385, -0.3449706435203552, -1.9768730401992798, 0.8733846545219421, 1.2251337766647339, -0.6175733804702759, -0.4337334930896759, -0.5839481353759766, -0.6620999574661255, -2.683363199234009, 1.2819262742996216, -1.2161020040512085, -0.39355069398880005, 0.2739914059638977, -0.03667047992348671], [0.3082776367664337, 0.5792457461357117, -0.865527868270874, 0.45259883999824524, -0.36487534642219543, -0.00306182773783803, -0.22115160524845123, 2.622239351272583, -0.18620814383029938, -0.5729960799217224, -0.5103104114532471, -0.43220096826553345, -0.7776411175727844, 0.46897202730178833, 1.697130560874939, -0.5154009461402893, -1.8036881685256958, 0.4947839081287384, -0.3780458867549896, 0.7970501780509949, 0.26137441396713257, -1.4624159336090088, 0.5627540946006775, 0.4908818304538727, -0.8123331665992737, -1.4469950199127197, 0.9430268406867981, 0.2418050467967987, 0.2677720785140991, -0.3766390085220337, -0.7889367341995239, -0.8135100603103638, 0.5225523710250854, 1.3070248365402222, 0.5346863865852356, 1.2281789779663086, -0.857502281665802, -3.286003351211548, -0.12723758816719055, -0.6238912343978882, -0.6958776712417603, 0.6051319241523743, 1.188579797744751, 2.3189196586608887, 1.3723547458648682, 0.5176283121109009, -0.9619875550270081, -0.4209851622581482, -0.9976224303245544, 0.7118245959281921, -0.578562319278717, 0.7324535250663757, 1.7624340057373047, 0.40783751010894775, 0.2222738265991211, 1.3477078676223755, -0.3785281777381897, -1.0299040079116821, -0.05257993936538696, -0.31578704714775085, 1.0901621580123901, 0.1192188709974289, -2.617530345916748, -1.4945420026779175], [-1.647995114326477, -0.5762707591056824, 0.7906367182731628, -0.1701706200838089, 0.17970652878284454, -1.093584656715393, 0.7615453600883484, 0.8753384351730347, -1.8959752321243286, -2.097151756286621, -0.28783535957336426, -0.24209322035312653, 1.9916326999664307, -0.7973762154579163, -0.17912058532238007, 2.342566728591919, 0.5084550380706787, 0.6688416004180908, -0.852534294128418, 1.1899585723876953, 1.0668517351150513, 0.1769048422574997, 0.4800119400024414, 0.052638109773397446, -0.5633858442306519, -1.9629980325698853, -1.363842248916626, 0.6885223388671875, 1.764074444770813, -1.4187196493148804, -0.4223383665084839, 0.775520384311676, -0.06509832292795181, 1.3173574209213257, -0.9089652895927429, -1.669783115386963, -0.21909993886947632, 0.23465590178966522, -1.0995217561721802, -0.1925870031118393, 1.3529396057128906, -0.4149326980113983, -0.7059462070465088, 0.3972107768058777, 0.14646108448505402, 0.0375778004527092, 1.6557766199111938, -1.0486252307891846, 0.47091972827911377, 0.01808183081448078, -1.096704363822937, 2.178779125213623, 1.114146113395691, -0.058277737349271774, -0.867801308631897, -2.0358564853668213, 0.11533834040164948, 0.19621166586875916, 0.9578876495361328, -0.16945497691631317, -1.8277922868728638, -1.2276122570037842, -0.7748575806617737, 1.7687320709228516], [-0.5862719416618347, -1.1502715349197388, -0.4996654689311981, 0.6395533084869385, -0.3015098571777344, 0.09236454218626022, -0.32840338349342346, -0.42172086238861084, -2.2506906986236572, -0.39653462171554565, 0.7217227816581726, 1.754983901977539, -0.7967022061347961, 0.4051900804042816, -0.6238709092140198, -1.1782801151275635, -0.71509850025177, 1.093689203262329, -1.8656196594238281, -0.25267118215560913, 0.2805350422859192, 0.5204113721847534, 0.20100511610507965, 1.2923102378845215, -1.3915091753005981, -1.0376639366149902, 0.3335755467414856, -0.9231131076812744, -2.407130718231201, 1.7855768203735352, 0.3567084074020386, 1.8603500127792358, -0.9558655023574829, 0.06046171858906746, 0.003747297450900078, 2.6380698680877686, 0.4333861172199249, -0.33232566714286804, 1.427937388420105, -0.07240460813045502, 0.6301149129867554, 1.0464013814926147, 1.168784737586975, -0.7704208493232727, 0.5630698800086975, -1.135978102684021, -0.31504103541374207, -0.6569116115570068, -0.026816964149475098, -0.41100919246673584, -0.030641280114650726, 0.9897439479827881, 0.6287127733230591, -0.7716151475906372, -1.7365446090698242, -1.3057985305786133, -0.010739759542047977, 0.3221130669116974, -2.9480605125427246, -1.2352321147918701, -0.09376067668199539, -1.5104049444198608, 0.5716115832328796, 0.7319576740264893], [-1.1896172761917114, -1.013407588005066, 1.3171809911727905, -1.183419108390808, 0.9532397389411926, -0.6619467735290527, 0.03382478281855583, -2.3134357929229736, 1.384177327156067, 0.5528699159622192, -0.35676848888397217, 0.25251054763793945, 0.2177688330411911, 0.748936116695404, 0.5656724572181702, 0.19861742854118347, 1.6152080297470093, 2.6062445640563965, -1.6518967151641846, 0.029417559504508972, 1.8191590309143066, -0.04655402526259422, -0.13820447027683258, -0.13056808710098267, -1.6975982189178467, 0.4198994040489197, -0.09184403717517853, -1.609731674194336, 0.4021090567111969, -0.5702330470085144, 1.2140758037567139, -1.683302640914917, 0.8716887831687927, -1.8131904602050781, 1.0983551740646362, -2.3438079357147217, 0.6889969706535339, 0.392916202545166, -1.6675907373428345, -0.773267924785614, -1.1881263256072998, 0.34909501671791077, -1.785142421722412, -1.0383933782577515, 1.5338491201400757, -0.27261197566986084, 1.1301443576812744, 0.14530442655086517, 0.11083845049142838, 0.8793917298316956, 1.4539172649383545, -0.12494268268346786, -0.12383660674095154, -0.5414720177650452, 0.6469157338142395, 0.7082810401916504, -0.25861456990242004, 1.3517307043075562, -0.5631132125854492, -0.5528649687767029, -0.29747146368026733, 1.136833667755127, -1.2722396850585938, 1.249783992767334], [-0.98343425989151, 0.34548231959342957, 0.3255954384803772, 0.7943130731582642, -0.1906488984823227, -1.977256417274475, 0.5662025213241577, 0.6679225564002991, -0.23226144909858704, 0.23339176177978516, -0.05897272378206253, 0.269382119178772, 0.7246993780136108, 0.29597410559654236, 1.0800951719284058, -1.5511595010757446, -1.0994662046432495, 0.22660911083221436, -0.5242422819137573, -0.6425315737724304, 0.08328843861818314, 1.7878308296203613, -1.3256723880767822, 1.5698665380477905, 0.5332415699958801, -0.2169705331325531, 0.5678173899650574, 1.3937044143676758, 1.1847635507583618, 1.0931304693222046, 0.4168425500392914, 0.823369026184082, -1.3003228902816772, -0.32361453771591187, -0.6474476456642151, -0.21772466599941254, 1.377820372581482, 0.5094512104988098, -0.2004449963569641, 0.18460215628147125, -1.0497727394104004, 0.47291404008865356, -1.2107430696487427, 0.8765990138053894, -1.1981464624404907, -2.318051815032959, -1.2560738325119019, -0.05067411810159683, 1.482652187347412, -0.4410242438316345, 0.2713017761707306, -1.2121772766113281, 0.040170662105083466, 0.8223455548286438, 1.01523756980896, 0.534240186214447, 1.6668190956115723, -1.2732512950897217, 1.8786152601242065, -0.4849957823753357, -0.6009510159492493, 0.7122545838356018, -0.4849376678466797, -0.7162975072860718], [-1.6583366394042969, -2.1029422283172607, -0.4704238176345825, 0.3759778141975403, 0.18726952373981476, -1.2338987588882446, 0.40737828612327576, -1.9298396110534668, -0.7442374229431152, 0.6297553777694702, -0.21580426394939423, 0.5441198945045471, -0.833926796913147, -0.5700023770332336, 2.616525888442993, -1.0951902866363525, -0.7194883823394775, 0.9909651279449463, -0.7562819123268127, -0.843856692314148, 0.36430248618125916, -0.7379270195960999, -1.9912878274917603, -1.5652118921279907, 0.4033840000629425, 1.830145239830017, -1.941773772239685, -0.9751691818237305, -0.07354728132486343, -0.021050695329904556, -0.09100182354450226, -1.873487114906311, -1.6050916910171509, 0.30155664682388306, 0.4996948540210724, 0.48351648449897766, 0.3878754675388336, 1.2255945205688477, 1.0547146797180176, 0.16254350543022156, 0.8501135110855103, 0.14290651679039001, -1.096758484840393, -1.581602692604065, 0.3921653926372528, 0.10200662910938263, 1.9294967651367188, -1.784531593322754, 0.48992958664894104, 0.4071837365627289, -1.199540615081787, -0.03523621708154678, 3.6405341625213623, 0.6596284508705139, -0.4257882535457611, 0.37693899869918823, 0.11944791674613953, 0.6829201579093933, -0.1258072555065155, -0.46660202741622925, 2.07370924949646, -0.37081679701805115, 1.3352190256118774, 1.1005440950393677], [-1.5581082105636597, 0.7401418685913086, -0.35922181606292725, 0.7460912466049194, -1.826540470123291, 0.6819437146186829, 2.5095653533935547, 1.2738194465637207, 1.051707148551941, 1.2054129838943481, 0.48250100016593933, -0.1715017557144165, 1.1473677158355713, 1.6926462650299072, 0.04360456392168999, -0.9036957621574402, -0.2358212172985077, -0.6153128147125244, -0.05346236005425453, -0.25123313069343567, -1.5856083631515503, -0.569490373134613, -0.11010681092739105, 1.336005687713623, 0.2515110671520233, 0.02658735029399395, -0.8693931102752686, 0.12364443391561508, 0.3739960193634033, -0.5113994479179382, 0.7756029367446899, -1.4864754676818848, 0.5409016013145447, -0.057125791907310486, 0.8114805817604065, 0.379454106092453, 0.9913303852081299, 1.6502344608306885, 0.4775276780128479, -0.2742144763469696, 0.4784913957118988, 0.03352978453040123, -1.2353702783584595, -0.9492419958114624, 0.15418535470962524, -1.2866406440734863, -2.184648275375366, -0.6901174783706665, 0.4333321452140808, 0.5305566787719727, 0.31024307012557983, -0.8600201606750488, -0.6336373686790466, -0.616808295249939, 0.7508389949798584, -1.0519148111343384, -1.7710309028625488, -1.334822654724121, 1.8014148473739624, 0.1772574484348297, 0.5469462275505066, 0.6241450309753418, 0.4522551894187927, -0.03337923809885979], [-0.02860010229051113, -0.25195515155792236, 0.6774765849113464, -0.9504308700561523, 2.1530356407165527, -1.2721620798110962, -0.8887531161308289, -2.085371732711792, -0.30079105496406555, 0.045929934829473495, -0.5328382253646851, -0.1540404111146927, 0.2873504161834717, -0.6409205198287964, -0.3240707218647003, -0.807428240776062, 1.8801416158676147, -0.37555357813835144, -0.3926601707935333, 0.8919740915298462, -0.7855996489524841, -0.4003245234489441, 0.5306426286697388, 0.458706796169281, 2.147965431213379, -0.8938829302787781, 0.09108258783817291, 0.2850065529346466, 1.6828843355178833, 0.25924673676490784, 0.3672807514667511, -1.0101850032806396, 1.0063027143478394, -0.20863837003707886, 2.128626585006714, 0.07144379615783691, -0.9725894927978516, -0.06460046768188477, -1.4167671203613281, 0.0008660722523927689, 0.0769042819738388, 1.5976955890655518, 0.3044659197330475, 0.5602725148200989, 0.7516233325004578, -1.717329740524292, 1.722368836402893, -0.6779810786247253, -1.8500158786773682, 1.1273608207702637, 0.5877912640571594, 0.1335136592388153, -0.2624480724334717, -0.8299168944358826, 0.6574797630310059, 1.0549336671829224, 0.46266695857048035, -2.2416787147521973, 1.0927635431289673, 0.6251264214515686, -1.8526912927627563, -1.262285828590393, 0.42986252903938293, -1.7901159524917603], [0.6160645484924316, -0.5396460890769958, 0.526379406452179, -1.2735203504562378, -0.5900858044624329, 1.1126210689544678, 0.7473640441894531, -0.3198562562465668, -0.15331606566905975, 0.22861053049564362, 0.028711259365081787, -1.4375767707824707, 1.1692832708358765, -1.2312111854553223, -0.4192042052745819, -0.023253843188285828, 0.04369448870420456, -1.1560078859329224, 0.2923375964164734, 0.7210593223571777, -0.13885344564914703, 0.10519217699766159, 0.32282084226608276, -1.8388782739639282, -1.1594377756118774, -0.4793325960636139, 1.3560727834701538, 0.6321308612823486, 0.9153612852096558, 0.6671977639198303, 1.5068496465682983, -0.1344733089208603, 1.4466853141784668, 0.7377961277961731, -0.20223593711853027, -0.06837432086467743, 0.5100730061531067, -0.08357337862253189, 0.6299890875816345, -0.839405357837677, 0.035417340695858, -0.34360766410827637, -0.456570565700531, 0.2890903055667877, -1.6410675048828125, -0.49836960434913635, -0.8670352101325989, 1.9323447942733765, -0.6094119548797607, -0.5916459560394287, -0.28739526867866516, -0.5615349411964417, -0.5023545026779175, 1.0799063444137573, 0.28953275084495544, -0.44995325803756714, 0.9818875193595886, 0.288505494594574, 0.6406441330909729, 0.7292418479919434, -1.055703043937683, 1.347907304763794, -1.2342900037765503, -0.33730995655059814], [-0.9248629212379456, -0.18127049505710602, 0.43056678771972656, -0.5666242837905884, 0.6202473640441895, 0.9441946744918823, -1.830360770225525, 1.1151301860809326, 0.49364784359931946, 1.2417628765106201, 1.400176763534546, 0.5498250722885132, -1.4516059160232544, -0.427239328622818, 0.8579938411712646, -0.056553665548563004, -0.06143098324537277, 0.49285268783569336, 0.6945996880531311, -0.4594910442829132, -1.145079493522644, 0.14572148025035858, 1.3365026712417603, -1.046030879020691, 0.5881221294403076, 0.4637109637260437, -1.5904319286346436, -1.063009262084961, 0.4075973629951477, 1.0817580223083496, -2.558511734008789, -0.40371060371398926, 0.04875130578875542, 0.7507105469703674, 0.936919629573822, -0.7680380344390869, 1.2123427391052246, 0.6508374810218811, -0.19693179428577423, 0.19483935832977295, -0.6752616167068481, 1.2198044061660767, -1.1933183670043945, -0.0976995900273323, -0.10365451127290726, -0.586179792881012, -0.23325012624263763, 1.4940028190612793, 0.055085740983486176, 1.1622086763381958, -1.3339207172393799, 1.0136783123016357, -0.05878245458006859, -0.21670182049274445, 0.08707612007856369, -0.6656832695007324, 1.3516700267791748, 1.0587072372436523, 0.36472728848457336, -0.27551379799842834, -0.5455202460289001, -0.9236972332000732, 1.166917085647583, 0.5860732197761536], [-0.9711307883262634, 1.8481262922286987, 0.4592859447002411, 0.47681549191474915, 2.7824864387512207, 0.7519311308860779, -0.012418105266988277, 0.4508572220802307, 0.01985216699540615, -0.6772535443305969, 1.1275449991226196, 0.053923193365335464, -0.03240324929356575, 2.3563928604125977, 0.017392583191394806, -0.038901180028915405, -0.5166664719581604, 1.4738860130310059, 0.31894928216934204, -0.004037695471197367, -0.7099297046661377, 0.9171367287635803, 1.436820149421692, -1.7777296304702759, 0.125285804271698, -1.2903227806091309, -1.703161597251892, 1.1749448776245117, 1.1520099639892578, 0.9679613709449768, 2.4216561317443848, -1.0674684047698975, 1.6464298963546753, -0.6687095165252686, -0.4395361542701721, 0.7768729329109192, -0.7966625094413757, 0.24009276926517487, -0.06103305518627167, -0.20870806276798248, 0.4429713189601898, 0.5353410840034485, -0.05814747139811516, 0.4720226526260376, 0.09850524365901947, -2.2205429077148438, 0.6759377121925354, 1.2931597232818604, 2.1798648834228516, 0.15027916431427002, 0.09567008167505264, 0.5631797909736633, 0.6195597052574158, -0.8820238709449768, 0.05811199173331261, 0.6185635924339294, 1.1821050643920898, -0.011693792417645454, -0.4504671096801758, 0.8019532561302185, -1.5940905809402466, -0.6270615458488464, 0.13364864885807037, 0.7725294828414917], [1.5740056037902832, -0.576130747795105, -1.392578363418579, -0.26608723402023315, -0.7939261198043823, 0.3009014427661896, 0.12354902178049088, -0.6104210615158081, 1.8234425783157349, -1.0662730932235718, -0.5079916715621948, 0.07489627599716187, 0.37130510807037354, 2.015672445297241, 1.3846168518066406, 0.04802244156599045, -0.43542078137397766, -0.7963864207267761, 2.887781858444214, 0.5845767855644226, 0.1641392707824707, 0.3360530436038971, -1.8137447834014893, -0.3290090560913086, 0.11767061054706573, 1.6969923973083496, 0.15022380650043488, 0.3747328519821167, 0.3718206286430359, 0.7782453894615173, 0.23693637549877167, -1.8122135400772095, 0.4997222125530243, -1.4146509170532227, 0.47370097041130066, 0.6360040903091431, -0.859306275844574, 2.030895233154297, 0.4448390007019043, 0.8123034238815308, 0.6239888072013855, 0.34306833148002625, -0.10577300190925598, -1.740973949432373, 2.137540817260742, -0.07372745871543884, -0.37136110663414, 0.33747977018356323, -0.8704718947410583, 0.4708505868911743, 0.34141167998313904, 1.3460767269134521, 1.1961561441421509, -0.023882171139121056, 0.4626084566116333, 0.8421021699905396, 0.7364696860313416, 1.3955241441726685, 0.9520552754402161, -0.5259753465652466, -0.4024616479873657, 0.3997083902359009, 0.640714704990387, -0.3448943495750427], [-0.4340203106403351, 1.6636483669281006, 0.5839421153068542, -0.492104709148407, -1.6859270334243774, -0.9069139361381531, -0.20905141532421112, 1.506596326828003, -0.29590848088264465, -1.1136517524719238, 0.23754063248634338, -1.9103485345840454, 0.03211243823170662, 0.27642151713371277, -1.5076448917388916, 0.5102057456970215, -0.1143907830119133, -0.18896284699440002, 0.47229060530662537, -0.2966029942035675, -0.18347576260566711, -0.8578735589981079, -1.4872076511383057, -1.1728169918060303, -0.5565829277038574, 0.11757944524288177, 0.5260685682296753, -0.28742215037345886, -0.45702794194221497, -1.143372654914856, -0.8792014122009277, -0.8833033442497253, -0.5316594243049622, 0.23867660760879517, 1.493965983390808, 0.36849021911621094, 0.649181067943573, -1.0401723384857178, -0.49790963530540466, -0.44743770360946655, -0.29260358214378357, 1.9717146158218384, 0.8852190375328064, 0.7852305173873901, -1.523681640625, -0.8166547417640686, -0.6661535501480103, 0.6465280055999756, -0.5283892154693604, 0.6806254386901855, -0.4343962073326111, -1.1377849578857422, 1.1767511367797852, 0.49657514691352844, -1.379784107208252, -0.026512794196605682, 0.5463854670524597, -0.9240636229515076, 0.739432692527771, -0.8708744645118713, -0.30930137634277344, 0.9090599417686462, -1.7986488342285156, -1.498754858970642], [0.15845073759555817, -0.24095773696899414, 0.04705774784088135, -0.4718211591243744, -0.027380822226405144, 0.48479247093200684, -0.576469898223877, 2.211784601211548, -1.5473337173461914, 1.3230066299438477, -1.3115838766098022, 0.531913697719574, 0.6151034235954285, -0.33813002705574036, 0.16844449937343597, 0.6667800545692444, 0.9649980068206787, -0.7112345695495605, -0.26760369539260864, -1.2830461263656616, -0.7011868357658386, -2.03017520904541, -0.713769793510437, -1.297074556350708, -0.7351078987121582, -1.2485777139663696, -1.5362884998321533, 0.06572955846786499, 0.8331454396247864, -1.1954110860824585, 2.435088634490967, -1.60995352268219, 1.0855321884155273, -0.5506886839866638, 0.24014389514923096, -0.09366986900568008, 0.4873543083667755, -1.2222107648849487, -0.26345837116241455, 0.6613724231719971, 0.5591975450515747, 0.18006496131420135, 1.0495827198028564, 0.3792625367641449, -0.3012714087963104, 0.11317525058984756, 2.6301634311676025, -0.6351652145385742, 1.0784441232681274, 0.5752024054527283, 1.2666289806365967, -0.6379013061523438, 0.5526360273361206, 0.5511738061904907, -1.1590391397476196, 0.1821756809949875, 2.082315444946289, -0.07004737854003906, -0.4720669388771057, 0.30930623412132263, 1.2855032682418823, -0.22153319418430328, 1.266597032546997, 1.0801777839660645], [-1.0018839836120605, 1.4148662090301514, 0.6986153721809387, 0.1844397634267807, 0.43226343393325806, -0.028443994000554085, 1.0858213901519775, -1.2674957513809204, 0.34474605321884155, 0.3161662220954895, -1.2046986818313599, 0.764057993888855, -1.3999780416488647, 0.1048261746764183, -0.8721356391906738, 1.7045038938522339, 0.5062938332557678, -0.2884189486503601, 0.6820970177650452, -1.8125003576278687, 0.2912265360355377, -0.24260486662387848, -1.1564375162124634, -0.914678156375885, 0.7474205493927002, -0.5893079042434692, 1.5089775323867798, 0.24956399202346802, -1.0757262706756592, -1.4174330234527588, -0.7461757063865662, 0.10490519553422928, -0.09436487406492233, -0.7222362756729126, 1.3426225185394287, -0.7516582608222961, -0.2901039123535156, 0.08246948570013046, 0.8976982831954956, -0.8801665306091309, -0.03185170143842697, -0.4748004078865051, -1.2391031980514526, -0.11449353396892548, -0.5598291158676147, -0.19806240499019623, -1.662652611732483, -0.18439234793186188, -1.207312822341919, -0.5089499354362488, 0.7872875928878784, -0.038626134395599365, -0.5731671452522278, -1.0829578638076782, -1.19191575050354, 0.7578422427177429, -1.0367443561553955, 0.36953237652778625, -1.9904708862304688, -1.5961859226226807, -0.5403679609298706, -1.4362214803695679, 0.48725423216819763, 1.0006216764450073], [0.34330329298973083, -0.08410774916410446, 0.9354888200759888, 1.63057541847229, -0.5514398217201233, -0.2759582996368408, 1.1416988372802734, 0.5051034688949585, 0.24037574231624603, 0.5734198093414307, -0.42552733421325684, 0.3767460286617279, 0.39147454500198364, 1.078134536743164, -0.4038057327270508, 0.6532843112945557, 0.5966017246246338, 1.441919207572937, -0.17366962134838104, -0.7475386261940002, -0.5937080979347229, -0.6423819661140442, -0.33320364356040955, 0.07390084117650986, 0.5523820519447327, -1.16652250289917, 0.8550591468811035, 0.8137215971946716, 0.5341237783432007, 0.29521480202674866, -0.4571966528892517, 0.7025370597839355, 0.014109944924712181, -0.5711453557014465, -1.0966089963912964, 1.1599011421203613, 0.8814579844474792, -0.8620210289955139, -0.31856706738471985, 0.4948911666870117, -2.5672202110290527, -0.394771009683609, -0.9620290994644165, 0.5000054836273193, -1.090509057044983, 0.04102909564971924, -0.8961396217346191, -0.17261774837970734, -0.4945348799228668, 0.2952403128147125, 0.24003306031227112, -0.9547406435012817, 1.0028010606765747, 2.9420101642608643, -0.4437711238861084, -0.709434986114502, 0.7630818486213684, 0.2449277639389038, -0.6850576996803284, 0.39265716075897217, 0.4425515830516815, -1.2422512769699097, 1.4290422201156616, 1.3569525480270386], [-0.9798168540000916, -0.33812007308006287, -2.280912160873413, 0.13627399504184723, 0.042157672345638275, -0.36434265971183777, -0.03750857338309288, -0.7798149585723877, -0.6382296681404114, -0.23364879190921783, -0.11479764431715012, 0.7604910135269165, 0.1875208169221878, -0.15566228330135345, -0.35331302881240845, 1.6366115808486938, 1.3899383544921875, -2.323988437652588, -0.4099929630756378, -3.0179789066314697, -0.7325862646102905, -0.6426939964294434, 1.7255533933639526, 0.47926953434944153, -1.2604426145553589, -0.37248584628105164, -0.0808861181139946, -2.4695487022399902, -1.6386921405792236, 0.1162753775715828, 2.2031567096710205, 0.09550394117832184, 0.3402015268802643, -0.7941574454307556, -1.4480576515197754, -2.3580305576324463, 0.37728801369667053, -0.6668384075164795, 0.4182550013065338, 0.7451024055480957, -0.9711495637893677, -2.0928597450256348, 1.0003999471664429, -0.6810095310211182, 0.6031563878059387, -0.19404815137386322, -0.29091721773147583, 1.4727566242218018, -0.8795251846313477, 1.27934730052948, 0.07030446827411652, -1.1612679958343506, -0.3890411853790283, 0.9065806865692139, 0.8083524107933044, -1.2395209074020386, -0.40450531244277954, -1.8219516277313232, 1.4857031106948853, -0.7122264504432678, -0.522602379322052, 0.49381759762763977, -0.6847280263900757, 0.4362568259239197], [-1.9891501665115356, 0.43914034962654114, -1.2966669797897339, -0.38993632793426514, 0.21196870505809784, -0.23807360231876373, -0.04579707235097885, -0.19422867894172668, 0.6059246063232422, 0.5899044275283813, -1.385717749595642, 1.7053593397140503, -1.5053752660751343, -0.5993422865867615, -0.051904935389757156, 2.7179250717163086, 0.11431092768907547, 0.6738414764404297, -1.0008955001831055, -0.04415381699800491, -2.017145872116089, -0.8483308553695679, 0.03188222274184227, 1.9299310445785522, -0.508131742477417, -0.7310633063316345, -0.38378357887268066, -0.6668629050254822, 0.175506129860878, -2.2830252647399902, -0.8603383302688599, -2.2735989093780518, 0.510013997554779, -0.9469299912452698, -0.9667568206787109, 1.554659366607666, -0.24313263595104218, -0.018026119098067284, -1.7282092571258545, -0.7536841630935669, -0.00381565373390913, -1.5302715301513672, -0.0825473964214325, -0.4644399583339691, 0.6495895385742188, -0.0036193104460835457, 0.8458698391914368, 0.33112409710884094, 0.9288843870162964, -0.47103720903396606, -1.394101858139038, -0.6583849191665649, 0.6387221217155457, 1.4898321628570557, -0.6125884652137756, 0.0893898755311966, -1.4202579259872437, 1.195510745048523, -0.33525457978248596, -0.9436970949172974, -2.46772837638855, 0.9493368268013, -1.952152132987976, 0.38433724641799927], [0.470353364944458, 0.7986135482788086, -0.24675142765045166, -1.6743065118789673, 0.07572401314973831, -0.47656551003456116, -1.988970398902893, 1.6036293506622314, 0.46049827337265015, 0.05639025941491127, 0.15449818968772888, -1.3251047134399414, -0.3101499676704407, 1.3539854288101196, 0.4794236123561859, -0.5135985612869263, -2.0382015705108643, 1.257407784461975, 0.5251697897911072, 0.09545674920082092, 0.13797442615032196, -0.09756286442279816, -0.23364180326461792, 1.7521790266036987, -0.562895655632019, 0.19962771236896515, -0.9512402415275574, -0.4216468632221222, 0.6350560784339905, -0.14913460612297058, 0.2314336597919464, -0.1530357152223587, 1.8750886917114258, 0.2803213596343994, 0.33257782459259033, -0.9444841742515564, -0.23077939450740814, -0.013710721395909786, -0.5716509222984314, 0.9159305095672607, 0.960040271282196, 0.41182810068130493, -0.04203091561794281, 0.3262907564640045, -0.5051831603050232, 0.15668605268001556, -0.14137278497219086, -0.2870062589645386, 0.578565776348114, 1.9692888259887695, 0.21544261276721954, -0.39065948128700256, 0.23195797204971313, -1.7099368572235107, 1.0234661102294922, -0.045127227902412415, -0.502909243106842, 1.8911834955215454, -0.268253356218338, -0.4123513102531433, 1.2428076267242432, 1.4955484867095947, -0.6294254660606384, 0.8670452237129211], [1.3392565250396729, 1.5229532718658447, -0.515247106552124, -0.02549774944782257, -0.12282241135835648, 1.0201568603515625, 0.6278494000434875, 0.5272753238677979, 0.884182333946228, -0.7200636267662048, 0.27298906445503235, -1.2749760150909424, 0.6103515625, 0.22779005765914917, -0.25653666257858276, 1.122174859046936, 0.14797312021255493, -1.241278886795044, -0.004892643075436354, 0.2728838622570038, -0.8738098740577698, -0.6612101793289185, -0.5255951285362244, 1.4405667781829834, -0.7065868377685547, 0.2615965008735657, 0.6090615391731262, -0.16378366947174072, 0.2180294245481491, 0.6999391913414001, -0.10793954133987427, -0.7378987669944763, -0.48469942808151245, -2.118199110031128, -0.8074826002120972, -1.4150646924972534, -2.2177815437316895, -0.20956262946128845, 0.06436782330274582, 0.09373217076063156, -0.3131406605243683, 0.394566148519516, 1.5177063941955566, 1.4313651323318481, 0.43524569272994995, 0.48004552721977234, 0.37030479311943054, -0.6785840392112732, 0.17501498758792877, 0.3120860159397125, 0.1674310863018036, 1.5024096965789795, -0.01041519083082676, 0.21094202995300293, -1.8852684497833252, -1.200434923171997, -0.5023892521858215, 1.4419552087783813, 0.06273450702428818, -0.4330284297466278, -1.3460139036178589, -0.566059410572052, -0.805744469165802, 1.0549733638763428], [-0.09640169143676758, -0.06836482137441635, -1.4857378005981445, 0.2677130401134491, -0.12024467438459396, 0.7965547442436218, 0.39665478467941284, -1.4134583473205566, 0.2901918292045593, -1.2035449743270874, -0.6994078159332275, -0.457853227853775, -1.8277883529663086, -0.5093980431556702, -0.8379797339439392, -0.373500257730484, -0.9449067711830139, -0.15120434761047363, 0.7355304956436157, 0.8638218641281128, 0.5545688271522522, -1.7860726118087769, -0.8202018737792969, 0.20829027891159058, 1.4319008588790894, -1.1076608896255493, -1.17812979221344, 0.3705402910709381, -1.4229129552841187, 2.246415376663208, 0.30337274074554443, 1.656524896621704, -0.172647625207901, -0.039644792675971985, 0.3096071183681488, -0.40427038073539734, -0.6583390235900879, 0.9374088048934937, -0.7917186617851257, 0.2060694396495819, -0.14840459823608398, 0.3801543116569519, -1.6403578519821167, -0.20480278134346008, -1.2685011625289917, -1.4244811534881592, -1.1522916555404663, -0.28897613286972046, 0.7845340967178345, 1.6468122005462646, 0.6470761895179749, -0.552727997303009, 0.34848612546920776, 0.7076624035835266, -1.0398211479187012, -1.9393463134765625, -0.8003149032592773, -1.8137060403823853, 1.5656296014785767, -2.1590962409973145, 0.07650851458311081, -0.2737148702144623, 0.2923944890499115, 0.8993817567825317], [0.7652823328971863, -1.8052120208740234, -0.807415783405304, 0.7404084801673889, -0.4544055461883545, -1.51747727394104, 0.0718006044626236, 0.13321228325366974, -0.24115337431430817, -0.4729920029640198, -1.2831065654754639, 1.3256330490112305, -0.5478408336639404, 0.5245829820632935, -0.12574851512908936, -1.2607439756393433, -1.676487922668457, -0.07528690993785858, 1.54152512550354, -1.110110878944397, -1.9993422031402588, -0.9192541837692261, 1.064671277999878, 0.315032422542572, -1.9255094528198242, 0.8574048280715942, 0.7425900101661682, 0.042559195309877396, -1.9273126125335693, -1.6034235954284668, -0.9070025086402893, 0.27025431394577026, -2.2758371829986572, 1.4545419216156006, -1.5622650384902954, 1.399534821510315, 0.2159566581249237, 0.1230497732758522, -0.8150312900543213, -1.141031265258789, 1.853493332862854, -0.32923465967178345, -1.2544211149215698, -1.1810390949249268, 0.20200984179973602, -0.13055875897407532, -2.179443597793579, 1.2064714431762695, 0.559045135974884, -0.80528324842453, -0.4012007713317871, 0.01408572867512703, 0.17365945875644684, 1.420430064201355, 1.0622525215148926, 0.18299640715122223, 2.140458345413208, -1.3296140432357788, -0.5383250117301941, 0.15914283692836761, -0.933242917060852, 1.0382643938064575, 0.8668243288993835, 1.0564221143722534], [1.1526826620101929, 1.3729923963546753, -0.4445532560348511, -1.8375109434127808, -1.3695464134216309, 1.0125150680541992, -1.1426798105239868, -0.7075168490409851, -0.4210744798183441, -0.5170444250106812, -1.2378020286560059, 0.24323637783527374, -0.022411765530705452, -0.46989062428474426, 0.3687976896762848, 1.0660613775253296, -0.8012211918830872, 0.1676711142063141, -0.030203135684132576, -0.9483445286750793, -0.5629733800888062, -0.23985591530799866, -0.3763582110404968, 0.31593605875968933, -1.4049898386001587, -1.4531441926956177, -0.6567898392677307, 0.26848483085632324, 0.8229092359542847, -1.007105827331543, 0.7403083443641663, 0.07119520008563995, 1.2246601581573486, -0.337922066450119, 0.001685812254436314, 0.18181215226650238, 1.978195071220398, 0.6175839900970459, -0.8754830360412598, 0.6865494251251221, -1.2725281715393066, 1.3039720058441162, 0.48090115189552307, -1.1240897178649902, 0.5988634824752808, 0.6175060272216797, 0.8332170844078064, 1.010625958442688, -1.271604061126709, -0.25751131772994995, -1.6755658388137817, 0.3361983001232147, -0.3276149034500122, 0.11863154172897339, 0.09042982757091522, -0.17142631113529205, 0.15267758071422577, 1.8318686485290527, -1.7139264345169067, -0.17531974613666534, -0.10033057630062103, 0.9005168676376343, 0.09077124297618866, 0.5579419732093811], [0.7843909859657288, 0.6948167681694031, -2.0314462184906006, 0.23465125262737274, -2.57932710647583, -0.9055960774421692, -1.133699655532837, 1.4693214893341064, -0.20011338591575623, -1.6889666318893433, 0.22027815878391266, 0.4992389678955078, 0.09350915253162384, -0.08731876313686371, 1.3275558948516846, 2.265838146209717, 1.3220068216323853, 1.9923704862594604, 0.46743083000183105, 0.7339954376220703, 0.8023068308830261, 1.628926157951355, 1.18031644821167, 0.5952181220054626, 0.8994151949882507, 0.8369032740592957, -0.28490638732910156, -0.6707897782325745, 0.010230700485408306, -0.12921354174613953, 0.7265486717224121, 0.11106313765048981, -0.05104559659957886, 0.4872540831565857, 1.4517704248428345, 0.26266348361968994, -0.6098127961158752, -1.07353675365448, 1.803892731666565, 1.2509137392044067, 0.8829399943351746, 0.6182246804237366, 0.3746352791786194, 0.8242540955543518, -0.6337827444076538, -1.9290517568588257, -1.364517331123352, 0.38696983456611633, -1.255441427230835, 0.04674877971410751, -0.25270575284957886, -1.8000158071517944, -0.30116379261016846, -0.8757171630859375, 1.1814532279968262, -0.007230921648442745, 0.8172934055328369, 0.05232727527618408, 0.10075519979000092, -1.622829794883728, -2.0052106380462646, 0.09711610525846481, 0.6352816820144653, -0.22444401681423187], [-0.30754274129867554, -0.2902122139930725, -1.0900622606277466, -0.05879015475511551, -2.3665666580200195, 1.5339916944503784, 0.004930903669446707, 1.3382997512817383, -1.9152119159698486, 0.5874946713447571, -0.6705492734909058, -2.1823058128356934, 1.4884023666381836, 0.29846465587615967, 1.2816355228424072, 0.7005516290664673, -1.0619646310806274, 0.12993302941322327, 0.4403252601623535, 0.5448601245880127, -0.7898416519165039, -0.42558416724205017, -0.32111692428588867, 0.9315316677093506, 0.048677705228328705, 0.46354520320892334, -0.0384262353181839, -0.3588472008705139, 2.221750497817993, 1.9026683568954468, -0.9763992428779602, -0.41072994470596313, 0.3185639977455139, 1.1540430784225464, 1.798502802848816, -0.8929242491722107, -0.4023638367652893, 0.6113358736038208, -0.9659222364425659, 0.7466058731079102, -0.42373475432395935, 0.6095975041389465, 1.2175931930541992, 0.9362069368362427, -0.9844310283660889, -0.7648908495903015, -0.209457665681839, -0.40277257561683655, 0.3835614323616028, -0.8304432034492493, 0.3642320930957794, -0.6297587752342224, 1.3046225309371948, 0.633732795715332, -0.05330631136894226, 0.29490870237350464, 0.1768614798784256, -0.03145362436771393, 0.8450637459754944, 1.2229779958724976, -0.14565658569335938, 0.060900457203388214, -0.6844211220741272, 0.4180063307285309], [0.6118108630180359, 0.2995256781578064, -0.13788357377052307, 0.7739046216011047, 0.1716562807559967, 0.819105863571167, -0.20753653347492218, -0.03915371373295784, -1.0047286748886108, 0.9102110266685486, -0.23160958290100098, -0.4228149652481079, -1.3500323295593262, 0.9207299947738647, 0.986057698726654, 0.10239401459693909, 0.5199608206748962, -0.1784714162349701, 0.548043429851532, 0.6267059445381165, -1.1585173606872559, 0.9599382281303406, 0.6237679123878479, -1.0163013935089111, -1.0785369873046875, -0.14948102831840515, 0.04403728246688843, 0.9004201292991638, 0.8218816518783569, 1.2414699792861938, -1.3911231756210327, 1.9253535270690918, -0.25454550981521606, 1.141419529914856, 0.3064817488193512, 1.0537240505218506, -1.0743471384048462, 1.307741403579712, -0.6782958507537842, -2.092778444290161, 0.35480743646621704, -0.31702905893325806, 0.4126327931880951, -1.4627490043640137, 0.231713205575943, 0.8462177515029907, 0.36046648025512695, -1.8980902433395386, 0.3503674268722534, -0.5205493569374084, -1.2048259973526, 1.9157906770706177, 0.0025993846356868744, 1.3909004926681519, 0.758996844291687, -1.322287917137146, 0.5202299952507019, -1.4686322212219238, 0.867871880531311, -1.1505647897720337, 0.4626287519931793, -0.6390694379806519, 0.6948075890541077, -0.24866925179958344], [-0.3112011253833771, -0.9355119466781616, 0.2125527709722519, 0.17107047140598297, 0.7546620965003967, 0.02194129303097725, -1.4043196439743042, 0.22900421917438507, 0.04973287135362625, 1.21427321434021, -0.7248159050941467, 0.2729591131210327, -0.1059306412935257, -1.0689963102340698, 0.06986317783594131, 0.9675253033638, -0.21370139718055725, 0.19467297196388245, 0.8574507236480713, -0.5618305206298828, -1.751671552658081, -0.03939196467399597, 0.3588458001613617, 0.7207197546958923, -0.23477238416671753, -0.3873354494571686, -0.6334671378135681, 0.1420404016971588, -1.3758727312088013, -0.28666549921035767, -1.522918939590454, -1.241211175918579, 0.6501504778862, 0.5009211301803589, 0.42477133870124817, 0.161962628364563, -0.7214124202728271, -0.012616447173058987, -0.8030598163604736, 1.274000644683838, -0.31398874521255493, 1.4382840394973755, 1.5051538944244385, 1.134734034538269, -0.4131622612476349, -1.3822944164276123, -0.08552704751491547, -0.13015927374362946, -0.5714408755302429, -1.1418769359588623, -0.4407491087913513, 1.0524568557739258, -0.6852760314941406, -0.3924803137779236, -0.05161238834261894, -0.4054715037345886, 0.9911495447158813, -1.8563456535339355, 0.47654643654823303, -0.6432037949562073, 0.4214431345462799, 1.8588166236877441, 1.202008605003357, 0.5052060484886169], [-1.1170125007629395, 1.1355643272399902, 1.0954134464263916, -0.27389323711395264, 0.5518791675567627, 0.6881526708602905, 0.1652613878250122, 0.4658907949924469, -0.39084622263908386, -0.6325910091400146, 0.8436254262924194, 1.2269221544265747, -0.2517942786216736, 0.8253708481788635, 0.010756353847682476, 0.9446482062339783, -0.9384887218475342, 0.36636248230934143, -0.823348879814148, -0.4940550625324249, 0.654971718788147, 0.2070535123348236, -0.07119093090295792, 0.45124006271362305, 0.8001043796539307, -0.7082247734069824, -0.7000550627708435, 0.484100341796875, -0.3971065282821655, 0.17689010500907898, -0.6072912812232971, -1.0154989957809448, -0.27220579981803894, -0.573400616645813, -1.5039072036743164, 0.7776762247085571, 0.5047609806060791, -0.09545613080263138, 0.009603188373148441, 1.5255266427993774, 0.9560952186584473, -0.9215779900550842, -2.5609824657440186, -0.9076777696609497, -1.700822114944458, -0.9836353063583374, -0.36441102623939514, -1.0813382863998413, -0.06243545189499855, -0.8167323470115662, 0.86146080493927, -0.20569264888763428, 0.912992537021637, -1.2421393394470215, 1.8001397848129272, -0.12235099077224731, -2.2072360515594482, -0.6562061905860901, -0.9749569296836853, 1.1779625415802002, 0.22158853709697723, -0.8175182342529297, 0.2570304274559021, -0.8304466009140015], [0.11475454270839691, -1.2152565717697144, -1.515424132347107, 0.6154403686523438, 0.46572718024253845, -0.32527899742126465, 1.0611188411712646, -1.9238625764846802, -1.4533518552780151, 0.6277576088905334, 0.6747186779975891, 1.799159288406372, -0.9613625407218933, 1.532821536064148, 0.17081178724765778, 0.39016932249069214, 1.2699223756790161, 0.36531776189804077, 2.0414950847625732, -0.40825915336608887, 0.09880437701940536, -0.2175176590681076, 1.2533056735992432, 1.2212272882461548, 0.17832060158252716, 0.44043466448783875, -0.07340186089277267, -0.19505511224269867, -0.0037249887827783823, 0.1830742061138153, -0.05076498165726662, -1.2543963193893433, 0.31750693917274475, 0.1292649358510971, -0.9512432217597961, 1.8775213956832886, 2.279794216156006, -0.4842633306980133, -0.21195019781589508, -0.3382132053375244, 1.423330545425415, 1.7909765243530273, 0.6910946369171143, 0.6490068435668945, 0.4878573417663574, 0.4099085330963135, 0.08921167254447937, 1.7271240949630737, 1.5417711734771729, 0.5487695336341858, -0.06057051941752434, -1.0828310251235962, -1.2581385374069214, -1.0365307331085205, -0.9629533886909485, 0.8876656293869019, 0.5775213837623596, 1.1129618883132935, -0.351121723651886, -0.4828121066093445, -0.698358952999115, -0.06479232758283615, 0.8140773773193359, -0.6796987056732178], [-0.7174848318099976, -0.5911429524421692, 0.29310745000839233, -0.19070865213871002, -0.5032816529273987, -1.0573183298110962, -0.654033362865448, 0.07565943151712418, 1.7718673944473267, -0.18198782205581665, -0.5841389298439026, 0.870072066783905, 1.567973017692566, 0.0830499455332756, -0.21279430389404297, -0.8324219584465027, 1.1624290943145752, 0.526221752166748, -0.38860100507736206, -0.09064056724309921, 1.2697893381118774, 0.6020615100860596, -0.08272095024585724, -2.1800830364227295, 1.552990198135376, 0.9434243440628052, -0.29033035039901733, -0.26992493867874146, -1.1111494302749634, -0.54305100440979, -1.3570979833602905, 0.1357099711894989, -0.9031050205230713, 1.0081173181533813, 0.0003634445311035961, 1.057176113128662, 2.7755837440490723, 1.2141178846359253, 1.3889384269714355, -1.0373311042785645, 2.140277624130249, 0.46471038460731506, 1.5505118370056152, 1.8743650913238525, 0.5801650881767273, 0.12817955017089844, -1.7267258167266846, -0.5591546297073364, 0.07032225281000137, -0.6644576191902161, 0.40414315462112427, 0.3112748861312866, 1.5730562210083008, 1.7554662227630615, -0.42198073863983154, 0.6379989385604858, -1.8753492832183838, -0.997255265712738, -1.1039069890975952, 2.8294410705566406, 0.9834110140800476, -1.072693943977356, 0.954922616481781, 0.1081949770450592], [0.04122775048017502, 0.5115620493888855, 1.3610304594039917, -0.31377938389778137, -1.2057290077209473, -0.3117404580116272, -1.1235357522964478, 0.6116927266120911, -0.8236618638038635, -1.088545322418213, -0.05519825965166092, -0.01455877535045147, 1.2350804805755615, 0.5910595059394836, -1.2647509574890137, -0.5220099091529846, 0.17765095829963684, 0.07120359688997269, 0.6342948079109192, 0.05725972354412079, 0.27632665634155273, 0.42390012741088867, 2.431511402130127, 0.8537560701370239, -1.0168944597244263, -0.2579403817653656, 0.35179808735847473, 0.03697571903467178, -0.6117492914199829, 1.6018441915512085, 0.22733238339424133, 0.24398861825466156, -2.2500522136688232, 0.5439459085464478, 0.2916530668735504, 1.9272451400756836, -0.4171484410762787, -0.20017103850841522, 1.683984637260437, -0.8263094425201416, 0.6816633343696594, -2.7278358936309814, 1.2215856313705444, 0.42682692408561707, 2.376539707183838, 0.3167867958545685, 0.06570195406675339, -1.1434553861618042, -1.2392594814300537, 1.8695279359817505, 2.0233287811279297, 0.022882474586367607, -1.0805271863937378, 0.0414489209651947, 0.6513153314590454, 0.9375309348106384, -0.40873950719833374, 0.2695348560810089, 1.3147248029708862, -1.0744671821594238, 1.261151671409607, 0.2425290048122406, 1.8902051448822021, -0.1947532296180725], [0.30909886956214905, 1.3638639450073242, 0.15800899267196655, 1.1415084600448608, 0.38285043835639954, -0.7293986082077026, 0.11349710822105408, -1.079743504524231, -0.4831397533416748, -0.7095288634300232, 1.0544366836547852, 0.2760942280292511, 1.450500726699829, -0.7172039151191711, -0.7955308556556702, 0.649182915687561, 0.1536673903465271, 0.6064352989196777, -2.3211722373962402, -0.9911668300628662, 0.2953895926475525, 0.9287929534912109, 1.1326501369476318, 0.4391041696071625, 0.7609303593635559, 0.46719491481781006, 0.20245325565338135, 2.3203094005584717, -1.87688148021698, 0.17144517600536346, 0.6810467839241028, -0.7366222143173218, -0.003589667845517397, -0.13870714604854584, 0.34763169288635254, 0.8748005628585815, -1.129648208618164, 0.21789894998073578, 0.6108153462409973, 0.7578855752944946, 0.07717940956354141, 0.05851786956191063, 1.4496164321899414, -0.768563449382782, -0.6883694529533386, -0.37227413058280945, -0.29160258173942566, -1.8413455486297607, 0.6490694284439087, 0.048380687832832336, 0.46004006266593933, 1.7941960096359253, -1.005407452583313, 0.35558250546455383, 0.490223228931427, -2.6925477981567383, 0.9802475571632385, 0.5705731511116028, 1.6092318296432495, 0.793434739112854, 1.954863429069519, 0.05389031395316124, -0.42253249883651733, -2.388720750808716], [0.808467447757721, -0.1608067899942398, 1.0701149702072144, -0.1316724419593811, 1.2505584955215454, -1.0967204570770264, -0.8750650882720947, -0.607662558555603, -0.5386390686035156, 1.4277135133743286, -0.9858190417289734, 1.756609320640564, -0.3528955578804016, -0.27201104164123535, 0.25310182571411133, 0.03896249830722809, -0.8675739765167236, 0.08800932765007019, -0.19824180006980896, 0.08925044536590576, -1.5173264741897583, -0.49662479758262634, -0.4891892075538635, -1.2007008790969849, -0.3516734838485718, 0.09225264936685562, -2.54811692237854, 0.7802625298500061, -1.121173620223999, -1.0307223796844482, -0.5190095901489258, 3.3659002780914307, -0.35259050130844116, 0.24641379714012146, -0.6263216733932495, 0.3832016885280609, -1.0480543375015259, -0.3023250997066498, -1.7970653772354126, -0.2760421633720398, -0.7992168068885803, -0.7960166931152344, 1.0274990797042847, -0.7567116022109985, -0.2094426453113556, 0.6758529543876648, -0.008353677578270435, 0.271879106760025, -1.9207937717437744, -1.2393932342529297, -0.0445169135928154, 0.9082952737808228, -0.15721182525157928, 1.0038985013961792, 0.5983516573905945, 0.37796270847320557, -0.3041527271270752, -1.7796926498413086, 0.9876689910888672, 1.1109908819198608, -0.044033393263816833, 0.6316856741905212, -1.8264634609222412, 1.5419334173202515], [-0.6191563606262207, -1.5000494718551636, 1.5484237670898438, -0.06670228391885757, -0.5315430760383606, -0.33284303545951843, 0.9460237622261047, 0.2818717956542969, -0.4650726616382599, 1.1272428035736084, 0.46758943796157837, 1.1840819120407104, -0.09902002662420273, -0.04518059641122818, 1.3329929113388062, 1.1293725967407227, -0.740399181842804, 0.32342228293418884, -0.0978354886174202, -0.1556035876274109, 0.47942110896110535, 0.8913142085075378, 0.6800698041915894, -1.2568843364715576, 1.1332707405090332, -1.168380856513977, -0.20633208751678467, -0.6925299763679504, -1.519075870513916, 0.4847067892551422, -1.455484390258789, 0.6790115237236023, -0.19737359881401062, -0.022221814841032028, -0.6966145038604736, -0.4423891603946686, 0.8546183109283447, 1.037636637687683, 1.0884289741516113, 0.2774149179458618, -0.8492420315742493, 0.4497920870780945, -0.10110946744680405, -1.414144515991211, -0.12459079921245575, 0.31719452142715454, -0.270394891500473, -0.5586395263671875, 1.4464325904846191, -0.9409893155097961, 1.6157082319259644, -1.1958433389663696, -1.6795809268951416, -0.25525417923927307, 1.1597360372543335, 0.6714361906051636, 0.000687349820509553, 1.1449097394943237, 1.268930435180664, -1.5074983835220337, -0.2263263314962387, 0.9941725730895996, -0.9378131031990051, -1.604009747505188], [0.6159108281135559, -1.1650813817977905, -0.5784628391265869, -1.1434283256530762, -1.331658959388733, 0.404064416885376, 0.873907744884491, 1.145570158958435, 1.072997808456421, -2.4562816619873047, 0.5226393938064575, -0.6463363766670227, 0.07902003079652786, -0.4499098062515259, 0.6605944037437439, -0.7061460614204407, 0.3051179349422455, 0.06091758981347084, 0.7248007655143738, -2.169790506362915, -0.4463646709918976, -1.870550274848938, 1.5110958814620972, -1.9539637565612793, -0.5614945292472839, -0.9537692070007324, 0.16498640179634094, -0.6609918475151062, 0.5715530514717102, 1.028543472290039, 0.6246948838233948, 0.18280260264873505, 1.0194377899169922, 0.3781987428665161, 1.422567367553711, -0.050325408577919006, -0.834004819393158, -0.2955620586872101, -0.6591817140579224, -1.1626062393188477, 0.17733432352542877, -0.29226401448249817, -0.38739779591560364, 0.21651212871074677, 0.7246502637863159, -0.3044813275337219, 0.4457964301109314, 0.6999878883361816, -0.13041873276233673, -0.18457943201065063, 0.9227281212806702, -0.0037887522485107183, 0.43384942412376404, -0.1079416275024414, -0.612297773361206, -0.8533393740653992, 1.846173882484436, -0.9050971865653992, -0.03577319160103798, 1.3519363403320312, 0.4909421503543854, -1.4161120653152466, 0.07803633064031601, -0.2579074501991272], [-0.3724299371242523, 0.9110836982727051, -0.20023493468761444, 0.460746169090271, -0.6196766495704651, 0.7294708490371704, 0.5873246788978577, 0.09058284759521484, 0.4962814748287201, 0.2955684959888458, 2.1434707641601562, 1.4691494703292847, 0.7679172158241272, 2.0412099361419678, 0.770025372505188, -0.864848256111145, 0.8722563982009888, -2.93861985206604, 0.3613435626029968, 1.1891916990280151, -0.07707833498716354, -1.5971766710281372, 0.26338431239128113, -0.6281943917274475, -0.5507739186286926, 0.3734142780303955, -0.6456824541091919, -0.9902441501617432, 1.1364065408706665, -0.3620980381965637, 0.011284456588327885, -2.0691111087799072, -0.47503820061683655, -0.34102222323417664, 0.5907107591629028, -1.1476683616638184, 0.5037907361984253, -0.2509766221046448, -1.3504372835159302, -1.9454821348190308, -0.5125536918640137, -0.6496124863624573, -0.011836743913590908, -0.41706323623657227, 1.543192982673645, 0.3455398678779602, -1.2428417205810547, -0.6037225723266602, -1.335440754890442, 0.6664366126060486, -0.6968203783035278, 0.9461724758148193, 1.8736761808395386, 0.05382952839136124, 0.5244238376617432, -1.1818615198135376, -1.0236027240753174, -2.5069921016693115, 1.2580701112747192, -0.2959096431732178, -0.07738550007343292, -1.1331467628479004, 0.5478939414024353, -1.5537453889846802], [-0.5639456510543823, 0.6367673277854919, -0.46152377128601074, 1.2260546684265137, -0.045221440494060516, 0.556056022644043, 1.4227508306503296, -0.4344165325164795, -0.5927632451057434, 1.0449748039245605, 2.393178939819336, 0.02532561682164669, -0.055768102407455444, 0.43969106674194336, -0.019156454131007195, 2.0353448390960693, 1.0237209796905518, -0.43588870763778687, -0.056075479835271835, 1.8998299837112427, -1.0902563333511353, -0.6472455859184265, 0.2813476026058197, 0.11703117191791534, -0.8056758046150208, -0.4892942011356354, -1.126136064529419, -0.8680275082588196, -0.4050848186016083, -0.9425171613693237, -1.996734857559204, -0.3384784758090973, -1.175569772720337, 0.34936752915382385, 0.8856779932975769, 0.0789167732000351, -1.722862958908081, -1.2624902725219727, -0.7738792896270752, -0.9827461242675781, 0.3970370292663574, -0.44293180108070374, -0.3437449038028717, -0.26840975880622864, -0.29897621273994446, 1.5651044845581055, -0.8360075354576111, -0.2965746819972992, -1.1498202085494995, -2.362218141555786, 1.3099836111068726, 1.3855326175689697, 0.9341571927070618, -2.130272626876831, -0.574094831943512, 1.467481017112732, -0.9808253049850464, -0.6571316123008728, 1.3032087087631226, -1.5445308685302734, 0.8174463510513306, -0.4353342652320862, 0.30576789379119873, 0.6802117228507996], [2.0752453804016113, 0.8727107644081116, -0.5422016978263855, 1.8042001724243164, -0.09491299837827682, 3.0066332817077637, -1.7117811441421509, -0.04236772283911705, -0.6960252523422241, 0.41797494888305664, -2.644639492034912, -0.4130401909351349, 2.5663137435913086, 0.2951798737049103, -1.529264211654663, -0.6979784965515137, -0.19295045733451843, -1.689523458480835, 1.682477355003357, 0.09995067864656448, 0.7352006435394287, 1.0491276979446411, 0.5530082583427429, -0.004408832639455795, 2.3703453540802, 0.35386398434638977, -1.115352749824524, 0.15804533660411835, 1.0308657884597778, -0.955102264881134, 2.129518747329712, 2.672090768814087, -1.1732656955718994, 1.875633716583252, 0.8700740933418274, 0.42004814743995667, 0.9300287365913391, -1.4718018770217896, -0.15123771131038666, 0.035219017416238785, -0.6608002781867981, -0.8449363708496094, 0.5012416243553162, 1.4127962589263916, -0.06184570491313934, -0.9671456813812256, -1.8918195962905884, -0.21134178340435028, -0.5305630564689636, -0.727064847946167, -0.1488020122051239, -0.32597631216049194, -0.5337146520614624, 0.13955584168434143, -1.3618024587631226, -1.5584509372711182, 1.4138938188552856, -1.1572538614273071, 0.00309908390045166, 1.6807087659835815, 1.6373251676559448, -0.8967577219009399, -2.0858592987060547, 0.8159651160240173], [0.8810242414474487, -1.4618922472000122, -0.21012865006923676, 0.16818265616893768, -0.02783331088721752, -0.504882276058197, 0.16506074368953705, -0.0033933683298528194, 0.3413495123386383, -0.9197287559509277, -0.16024132072925568, 1.012529969215393, -0.9190733432769775, -0.8470691442489624, -0.44629624485969543, -0.39511802792549133, -0.3530682325363159, 0.45236483216285706, -0.759836733341217, -2.0224766731262207, 0.9142218232154846, 1.0541588068008423, -0.21133093535900116, -0.8304995894432068, 0.43981876969337463, 0.8681476712226868, 2.263540506362915, -1.2189322710037231, 0.2813020348548889, 1.387292742729187, 0.001349621801637113, 2.1649506092071533, -0.8400098085403442, -0.21251347661018372, 0.0805029422044754, -0.5750049352645874, -0.7725919485092163, -0.13216133415699005, -0.9950690865516663, -0.7796642184257507, -2.0400149822235107, -2.062347650527954, -0.32905635237693787, -1.3548612594604492, 0.44177281856536865, 1.0068382024765015, 0.8792654275894165, 0.06780208647251129, -1.1206438541412354, 0.450284868478775, 2.0552890300750732, -1.4603934288024902, -0.30767762660980225, -1.487998127937317, -1.1545114517211914, -0.3292873501777649, 0.6792917251586914, -0.923757016658783, -0.8822462558746338, 0.2006796896457672, 0.713886559009552, 1.3058278560638428, 1.1130768060684204, -1.3039212226867676], [0.3689095079898834, -0.3960522711277008, 0.6720456480979919, 1.147079348564148, -0.4486348330974579, 0.1971801221370697, 1.4379719495773315, 0.5928248763084412, -0.3811401128768921, 2.194589853286743, 0.23080319166183472, 0.9188733100891113, -0.414408415555954, -1.116490364074707, 0.24217809736728668, -0.04230326786637306, 0.03826219215989113, -0.040426697582006454, -1.3486677408218384, -0.5835269093513489, 0.9111639857292175, -0.1716679483652115, 0.06435759365558624, -1.1294410228729248, 0.27148759365081787, -0.02080938033759594, 0.7121973633766174, 0.637839138507843, 1.2369821071624756, -1.2624844312667847, -1.050474762916565, -0.18273089826107025, 0.4880116879940033, 0.27283281087875366, 0.45246487855911255, -1.0703612565994263, -0.12123849242925644, 0.723555862903595, -0.667004406452179, 0.6777840256690979, -0.27999383211135864, -0.4766037166118622, 1.4122047424316406, -0.08482030779123306, -0.9789627194404602, -0.6203716397285461, 1.2267709970474243, 0.3729303181171417, 0.3584955036640167, -0.8286874294281006, -1.239548921585083, -0.03853988274931908, 0.1986072063446045, -0.3174205422401428, -0.6261149048805237, -0.6064178347587585, -0.36260393261909485, -0.9722535610198975, -0.2668003737926483, -0.13109809160232544, -1.5221797227859497, 0.5123972296714783, -1.6242938041687012, -0.19763828814029694], [-0.27479982376098633, -0.2952539920806885, -0.012559962458908558, -0.4748194217681885, -0.38793832063674927, 2.329162836074829, 2.5010757446289062, 1.9480907917022705, -1.8296338319778442, -0.6254345178604126, 0.11313922703266144, 1.4390134811401367, -1.1639117002487183, -1.1577173471450806, -0.2834480106830597, -1.502115249633789, 0.7753534913063049, -0.3296109139919281, -1.36520254611969, 0.6044477820396423, -0.6413833498954773, -0.05108675733208656, 1.2027981281280518, -0.8778378963470459, 0.026334356516599655, 0.7515720129013062, 1.207655668258667, 0.03376123681664467, 1.3070956468582153, 2.324777364730835, 0.10725492984056473, -1.0369231700897217, 0.48100146651268005, 1.4560534954071045, 0.38875243067741394, -0.5100186467170715, 0.5127940773963928, 2.169694185256958, -0.16254056990146637, 1.6539732217788696, -1.2820098400115967, -0.005670481827110052, -0.6432703733444214, 0.35174262523651123, -1.3798387050628662, 1.2135417461395264, 0.491231232881546, 0.37257328629493713, -0.8269971609115601, 0.8064707517623901, -0.04646063968539238, 1.1095207929611206, -0.7430678009986877, -2.832101345062256, 0.5615963339805603, -1.5140511989593506, 0.29037123918533325, -0.9656683206558228, 1.205609679222107, -0.9278545379638672, -2.974811315536499, 0.3403892517089844, -0.761677086353302, 0.6593905091285706], [-1.8849966526031494, -1.1582685708999634, 1.0486997365951538, 0.17560265958309174, 0.2677254378795624, -0.4146890640258789, 0.9274263978004456, -0.5813888311386108, 1.6023192405700684, 0.40939614176750183, 0.42021894454956055, -0.07055336982011795, -0.38389208912849426, -0.2564614415168762, 0.4639354646205902, -0.5637698173522949, 0.07567276805639267, -1.4309037923812866, -0.7408074140548706, -0.5140592455863953, 0.6617538928985596, 2.029252529144287, -0.2670580744743347, 0.7581695914268494, 0.12840402126312256, -0.178823783993721, 0.31987810134887695, 1.1661145687103271, -1.95503830909729, -0.5951545834541321, -0.16824465990066528, 1.2525602579116821, 0.8549128770828247, -0.3332792818546295, -1.1580051183700562, -0.5122654438018799, 0.5385524034500122, 0.5389228463172913, -1.7954424619674683, 1.2877495288848877, -0.7827849388122559, -0.0693700909614563, 0.5013530850410461, -2.1317238807678223, -0.19660374522209167, -1.363121509552002, -0.23721206188201904, -0.10510513931512833, -1.1020231246948242, -0.3689160645008087, 0.25519195199012756, 1.5711606740951538, 1.67361581325531, -0.33742690086364746, 0.12037506699562073, 0.19464173913002014, -1.0804551839828491, -1.4158241748809814, -0.8342002630233765, 2.41479754447937, -0.8607639670372009, -2.178919553756714, 0.7094528079032898, -1.3890328407287598], [-1.9451513290405273, 0.8970010876655579, -0.40539979934692383, 1.6812584400177002, 1.631026029586792, 0.6588044762611389, 0.9574552178382874, 0.7382605075836182, -1.6189278364181519, 0.17538489401340485, -0.1959039866924286, 0.9360573291778564, -0.9921343922615051, -1.251420259475708, -0.9100674390792847, -1.5718348026275635, 0.8753244876861572, -0.7420578002929688, 0.2154964655637741, 0.23919041454792023, 1.346744179725647, -0.5333706736564636, 0.8186134099960327, 0.5747475028038025, -0.13020059466362, -0.3974187970161438, -0.15231864154338837, 1.1394801139831543, -1.3723015785217285, 0.26810044050216675, 1.6421866416931152, 1.4988621473312378, 0.662877082824707, 0.33094310760498047, 1.4189887046813965, 1.1804072856903076, -1.062455654144287, 1.2497164011001587, 2.0862631797790527, -0.6275204420089722, -0.04544725641608238, 0.6226326823234558, -0.39824849367141724, -0.7465262413024902, 0.9742353558540344, -0.6015862822532654, 0.12228211015462875, -1.3466006517410278, 0.05034877359867096, -1.8105467557907104, -1.5454655885696411, -0.20317134261131287, -0.6393212080001831, -1.5027353763580322, 0.39899957180023193, -0.7906718254089355, -0.4322250783443451, 0.11115465313196182, 0.11514526605606079, -0.2169502228498459, 2.354750633239746, 0.403380423784256, -0.9742551445960999, 1.8365429639816284], [-0.41481274366378784, 0.5864725112915039, -1.6293436288833618, -0.15386004745960236, -0.5828061699867249, -1.428019404411316, 0.8350210785865784, -1.0644108057022095, 1.503809928894043, -1.0291489362716675, -0.5319022536277771, 0.76377272605896, 1.2147610187530518, 1.1687254905700684, -0.3272797167301178, -0.3790622055530548, 0.3201572299003601, 1.1071909666061401, 0.8359821438789368, 0.15003322064876556, 0.4181407392024994, -0.7784319519996643, 1.8457400798797607, 0.09650810062885284, 0.10873191058635712, 0.09417743235826492, -0.06150130555033684, -2.1112375259399414, 0.27390140295028687, -0.32741639018058777, 0.5784630179405212, 0.7080255746841431, -0.885945200920105, -0.13450585305690765, 0.6357126235961914, 0.1482158899307251, -1.1923800706863403, 0.287493497133255, -0.17370226979255676, -0.601767897605896, -0.4024966359138489, -0.07403373718261719, -0.049107588827610016, 0.3755008578300476, -1.199175477027893, -0.0897255688905716, 1.815133810043335, -1.0289448499679565, -0.2855048179626465, -0.6153491735458374, -0.419432133436203, 0.27729934453964233, 1.631639838218689, -1.3265941143035889, -0.6655867695808411, -0.6275885701179504, 0.7670319676399231, 1.4544044733047485, 1.196367859840393, -0.0746949166059494, 0.06395912170410156, -1.2261697053909302, -0.6198791265487671, 0.7356120944023132], [1.3103116750717163, 1.130373239517212, 1.9183663129806519, -0.05917997658252716, 1.1950466632843018, 1.5114414691925049, -0.7440593242645264, 0.8031266927719116, 1.8144962787628174, -1.1083885431289673, -0.5336318016052246, 0.18176871538162231, 0.9772328734397888, -0.478793740272522, 1.1394487619400024, 0.6380575299263, 0.5231723785400391, 1.8549944162368774, 0.20818418264389038, 0.6678658723831177, -1.1211631298065186, 1.3682129383087158, 0.02456558309495449, 0.7717444896697998, 1.5625534057617188, -1.25760018825531, -0.023960120975971222, 0.3164769113063812, 1.4434540271759033, -0.6316098570823669, 0.9648361802101135, -0.2668251395225525, 0.5049914717674255, 0.33793726563453674, 0.8295014500617981, -1.9578713178634644, -1.1993062496185303, 0.11447521299123764, 0.8794497847557068, -1.5135542154312134, 0.738150417804718, -1.756456971168518, 0.7410075068473816, 0.2003185898065567, 0.881604790687561, 0.34858080744743347, 1.7786288261413574, -0.6252471804618835, 0.8054002523422241, 0.44258415699005127, 0.662810206413269, 0.03678425773978233, 0.11207237839698792, -0.3352399170398712, 1.7573914527893066, -0.08312230557203293, -0.3581635355949402, -0.6461947560310364, 0.9738446474075317, 0.13388265669345856, -1.655456304550171, 0.5421707034111023, 1.3578546047210693, -0.10289496928453445], [1.0227376222610474, -1.2881356477737427, -0.10327010601758957, -1.4136325120925903, 0.2075352966785431, 1.4562113285064697, -2.0404436588287354, 0.0405195951461792, -1.818724513053894, -0.7946724891662598, -1.0624710321426392, 1.4738847017288208, 0.5537114143371582, -0.3752555549144745, 0.7524550557136536, -0.8406797647476196, -0.6848247647285461, -0.3791362941265106, -0.09475084394216537, 0.4235069751739502, -0.40263575315475464, 1.088089108467102, -1.991931676864624, 1.0241050720214844, -0.48700952529907227, -1.075655221939087, 0.46246978640556335, -2.5105574131011963, 0.07396726310253143, 0.8165263533592224, -0.106337770819664, -0.7525817155838013, -1.2872427701950073, 1.003143072128296, 0.5521280169487, -0.9982434511184692, -0.9294514656066895, -0.812057375907898, -1.5739988088607788, 0.3108440637588501, 1.040976881980896, 1.270742654800415, -1.1648201942443848, 1.828294038772583, -1.4005889892578125, 0.183798685669899, 1.6504443883895874, 0.3066258430480957, 0.034100454300642014, 0.10356540232896805, -1.4771143198013306, -1.1955738067626953, -0.1060781478881836, 0.8316100835800171, -1.554316759109497, -0.0011286974186077714, 1.275124192237854, -3.149486780166626, 1.5912941694259644, -0.6088531613349915, 0.5566829442977905, -0.09707579016685486, -0.2533472776412964, -1.0091984272003174], [0.33376970887184143, 0.0663294717669487, -0.8603149652481079, 0.46252956986427307, 0.6404593586921692, 1.232950210571289, -0.4317399859428406, -0.8107801079750061, -0.2915205657482147, 0.9063889384269714, -1.4943984746932983, 2.9715402126312256, 1.6306229829788208, -0.27076777815818787, -0.05455431342124939, 0.11349445581436157, -0.7910482287406921, 1.0479178428649902, 1.5982288122177124, 0.3830970227718353, -0.57923424243927, 0.39512377977371216, 1.003899097442627, 0.7300906181335449, -0.4363342821598053, -0.02579226903617382, -0.07510605454444885, -0.5789686441421509, -0.15245239436626434, -0.005331700202077627, -0.07780691236257553, -0.10936322808265686, -0.06393908709287643, -0.19512160122394562, -0.6943438649177551, 0.26441243290901184, 0.8375890851020813, 0.5953635573387146, -1.065389633178711, -1.518725037574768, -0.9042488932609558, -1.0524933338165283, 0.28222784399986267, 1.1134668588638306, -0.35087209939956665, -1.4095876216888428, -0.015079038217663765, -0.3927701413631439, -1.4235074520111084, 1.1639577150344849, -0.4819382429122925, -1.2432401180267334, -2.0642130374908447, -0.2909708321094513, -0.7488560676574707, 0.7084922790527344, 2.273231267929077, 0.4443088173866272, -0.6803225874900818, 0.4264613091945648, 0.123384028673172, -1.3397936820983887, -0.6489704251289368, 0.36130809783935547], [-2.5500402450561523, 0.0593600831925869, -0.44708317518234253, 1.922780990600586, -0.13001267611980438, 0.2701679766178131, -0.13978977501392365, 0.8319258689880371, -0.49119889736175537, -0.9459532499313354, 1.2683515548706055, 1.3550379276275635, -0.7006415724754333, 0.40099623799324036, 0.4529864192008972, -0.8718300461769104, 1.1005382537841797, 1.3513619899749756, 0.18530578911304474, -0.42000657320022583, 0.8832864761352539, 1.7829054594039917, 0.8096657991409302, -0.764134407043457, 2.3627641201019287, -0.4856039881706238, 1.34791100025177, 1.2572567462921143, 0.8516125679016113, -0.7281057834625244, 1.96077561378479, 0.7402046322822571, -0.5828654766082764, 0.8808435797691345, 0.769728422164917, 1.4068886041641235, -0.4184513986110687, 0.03220094367861748, -0.620189905166626, -0.6422900557518005, 0.5535131692886353, 0.5098422765731812, 0.12951967120170593, -0.9471237659454346, 0.6812707781791687, 0.7137030959129333, 0.8042600154876709, -1.041121006011963, 1.7659668922424316, -0.6387786269187927, 0.3415096700191498, -0.4731767177581787, -1.323021411895752, -0.032586753368377686, 0.30010920763015747, 0.4097374975681305, -0.5920242667198181, 0.948257565498352, -1.445298671722412, 0.3254736661911011, 0.28605103492736816, -0.0988285019993782, -0.34835192561149597, -0.7721251845359802], [-0.09545639902353287, 0.03936522454023361, -0.5447664260864258, 1.3223401308059692, -1.3168442249298096, 0.6686917543411255, -0.009989380836486816, 0.08918239921331406, 1.1537320613861084, -0.9914007186889648, 1.2140761613845825, 0.15601468086242676, -0.5778294801712036, -1.0987279415130615, 0.8459101915359497, -0.2463001161813736, 0.7370640635490417, 0.0263334009796381, 2.145782232284546, 0.13961835205554962, -0.3202841281890869, -0.04676685482263565, 0.5658379793167114, 0.13852539658546448, 0.7226247191429138, 0.31160813570022583, 0.46836891770362854, -2.006078004837036, 0.6997781991958618, 0.05344453081488609, 0.19232335686683655, -1.1939642429351807, -0.2993119955062866, 0.2802695631980896, 2.714205026626587, 0.09833969175815582, -0.9093295931816101, -0.09370822459459305, -1.9028620719909668, -0.35426950454711914, 0.9300833344459534, -1.9676746129989624, 1.4145820140838623, -0.5787371397018433, -0.12728345394134521, 0.12827248871326447, -1.5445125102996826, -1.1608572006225586, 0.8516387343406677, 0.9186169505119324, 0.5861223340034485, -0.052263159304857254, 0.5934892892837524, 0.40281328558921814, -0.8760268092155457, -1.1079895496368408, -0.2805614769458771, -0.644773542881012, -0.9752437472343445, -0.19188636541366577, 0.02107088640332222, -1.8643611669540405, 2.357577323913574, -0.6295901536941528], [0.03576013818383217, 0.40771979093551636, 1.1237431764602661, 0.16242486238479614, -1.1118096113204956, 0.06260990351438522, -1.4096260070800781, -1.148438811302185, -0.3112793266773224, 0.10832201689481735, 0.1251145899295807, -1.0221506357192993, -0.5795395970344543, 0.4799860119819641, -0.27828288078308105, 1.6520133018493652, 0.5269666910171509, 0.695902407169342, -1.0325225591659546, -0.48920533061027527, 0.46332332491874695, 0.5203660130500793, 0.24734610319137573, 1.1529632806777954, 0.9289100170135498, -0.5630271434783936, 0.7597222328186035, 1.094990849494934, 0.06309705227613449, -1.1331636905670166, -1.3582919836044312, 1.137624979019165, 1.3319449424743652, -0.468088835477829, -1.1474788188934326, -0.4646976888179779, -0.3015214502811432, 0.3789379894733429, -2.234739065170288, 0.717325747013092, -0.7238104343414307, -0.782516598701477, -0.14919064939022064, 0.7714409828186035, -0.7067002058029175, 0.18794749677181244, -1.8717397451400757, -0.5282163023948669, 0.6671987175941467, -0.7037101984024048, -0.3142637312412262, 0.9603285193443298, 0.30160725116729736, -0.09909806400537491, 0.45536962151527405, 0.7827193737030029, -0.6067171096801758, -1.2963919639587402, -0.2329123467206955, 0.22935646772384644, -1.9126524925231934, -0.3288615942001343, 1.0442510843276978, -0.12559857964515686], [-2.3121891021728516, -1.374606728553772, -0.33754727244377136, -0.7183440923690796, -1.108870506286621, 2.323183059692383, 1.6990880966186523, 0.7862929701805115, -0.11477068811655045, -1.121901035308838, -0.4844226539134979, 0.1855311393737793, -1.1653249263763428, -0.2505263686180115, -0.5687845349311829, -0.593293309211731, 0.24684999883174896, 1.17261803150177, 0.6257942914962769, 0.5201906561851501, -1.0423188209533691, 0.17159005999565125, 0.5885186195373535, 0.49548089504241943, 0.6461563110351562, 0.2510692775249481, -0.49132564663887024, -0.6528725028038025, -0.1590445637702942, 1.3546420335769653, -0.2332555651664734, -1.5307419300079346, 0.20895540714263916, 1.4689399003982544, 1.0197842121124268, -0.0008348080446012318, -1.7442798614501953, -1.3957347869873047, 0.24995779991149902, 0.03838165104389191, -0.6451588273048401, -0.29558128118515015, 1.2903650999069214, 0.42441466450691223, -1.7655638456344604, -0.4168924391269684, 0.10949353873729706, -0.4458369314670563, -1.6595529317855835, -1.111940622329712, 0.5999509692192078, -1.414027214050293, 0.8882745504379272, -0.22030656039714813, -1.3370740413665771, 0.2585800588130951, 0.4312557578086853, -1.3967801332473755, -0.8207620978355408, -2.002432107925415, -0.12835678458213806, -0.27591273188591003, -1.4049566984176636, -0.5891362428665161], [0.4513881802558899, -0.46890777349472046, -0.677514910697937, -0.33868077397346497, 0.3871837258338928, -0.1424637734889984, -0.25386372208595276, -1.5515477657318115, 0.7072212100028992, 0.8592431545257568, 0.24849256873130798, -0.08823608607053757, -0.3541697859764099, -0.664304256439209, 0.8527774810791016, 1.5283104181289673, -0.5272445678710938, 0.7312402129173279, 1.3770822286605835, 0.21076354384422302, -0.6622142195701599, -2.0754289627075195, -2.1429145336151123, -1.1223105192184448, 0.6667281985282898, 1.0331318378448486, 0.007500930223613977, 1.2619812488555908, -0.5115457773208618, -1.0008846521377563, -1.4116387367248535, 1.9333544969558716, 0.18991005420684814, 1.3058955669403076, 1.8971577882766724, 0.8510583639144897, -0.6205377578735352, -0.04833131283521652, 0.46814242005348206, -1.2355209589004517, 0.08145057410001755, 2.34244441986084, 1.3362082242965698, -1.2794522047042847, 1.224038004875183, -0.5530825853347778, -0.6098113656044006, 0.2697952687740326, 0.6830894351005554, 0.5931039452552795, -0.8897274732589722, 0.23314188420772552, 0.8981804251670837, 2.1362972259521484, -0.31657615303993225, 1.9548157453536987, -0.2843517065048218, 1.037500023841858, 1.5165842771530151, 0.15658366680145264, -0.06606659293174744, 0.35445070266723633, 0.9095021486282349, 0.7371720671653748], [-0.7080962657928467, -1.0739659070968628, -1.0165904760360718, 0.44992175698280334, 0.22007915377616882, 0.2134447991847992, 0.835246205329895, 0.9072328209877014, 1.4013726711273193, 0.45219144225120544, -0.922348141670227, 0.543283224105835, 0.24729032814502716, 0.6700810790061951, 1.086741328239441, 0.36514315009117126, 0.3695322275161743, -0.14489389955997467, 0.23531290888786316, 0.12668143212795258, -1.165354609489441, -0.3347018361091614, 1.5093263387680054, -0.19500401616096497, 0.07712869346141815, 1.29561185836792, -0.9421432018280029, 1.6077406406402588, 0.8014315962791443, 0.62322998046875, 0.2599838674068451, 0.03808487951755524, -0.29430755972862244, -0.38446611166000366, 2.0348594188690186, 1.0712708234786987, 0.8566870093345642, -0.17274212837219238, -2.761063575744629, -1.1438250541687012, 0.1360374540090561, 1.7090399265289307, 0.399230420589447, -1.2888706922531128, 1.3957222700119019, -0.2413668930530548, -1.6725244522094727, 0.0471830740571022, 0.6200734972953796, -0.8746693134307861, 0.8991299271583557, 0.6379418969154358, -0.6522542238235474, 1.528186559677124, 0.8467334508895874, -0.41054171323776245, -0.020861994475126266, -0.7118977904319763, -0.023422876372933388, 0.5790214538574219, 1.0889477729797363, 0.724554717540741, 0.1501987874507904, -0.8998992443084717], [-0.5653582215309143, 0.7859333157539368, 0.4240235686302185, -0.4831918179988861, -0.5737982392311096, 1.7704511880874634, -0.6778080463409424, -0.5469074249267578, 0.9923668503761292, -0.05749548599123955, -0.7591872811317444, 1.5918869972229004, 2.3319778442382812, 0.2676917016506195, 0.4087776839733124, -0.043835606426000595, -1.2103999853134155, 0.056526802480220795, -1.634891152381897, -0.19866392016410828, -0.9333381652832031, 1.486498475074768, 1.1348336935043335, -2.2230985164642334, -0.388461172580719, -1.8901488780975342, -0.8229814767837524, 0.918673038482666, 0.10419302433729172, -0.008819153532385826, 1.2958993911743164, -0.8489584922790527, -0.11361707001924515, -0.044294796884059906, -0.3375433087348938, -0.34018924832344055, 0.8379151821136475, 1.4651494026184082, 1.6225382089614868, 0.651883065700531, -1.057281732559204, -1.8372136354446411, 1.0550659894943237, 0.13945862650871277, 0.00939950067549944, 0.7824690341949463, 0.1850575953722, 1.4595377445220947, -0.2048601657152176, -0.9510674476623535, 0.25075745582580566, 0.6008156538009644, -0.783524751663208, -1.0981179475784302, -0.3388862609863281, -0.0066041769459843636, -0.6904881596565247, 0.6630582809448242, 0.09508609771728516, -0.0005757659091614187, 0.4175802767276764, -1.1895474195480347, 0.988233745098114, 0.01172696240246296], [-0.5012027025222778, -0.764214038848877, 0.9690467715263367, 0.14404402673244476, 1.207038402557373, -0.7724857926368713, 1.41107976436615, 0.29040294885635376, 0.042236801236867905, -1.2757014036178589, -0.09251739829778671, -1.2926617860794067, -0.963463544845581, -1.9441114664077759, -2.0140717029571533, -0.6164609789848328, -0.41707324981689453, -1.6370315551757812, -0.819602370262146, -0.33575108647346497, 0.8532006740570068, 0.43659311532974243, 1.2859903573989868, -0.12500372529029846, 1.044076919555664, 1.6260241270065308, -1.839584469795227, -0.2722674608230591, 1.8408217430114746, -0.7167345285415649, -0.3809562027454376, 0.600146472454071, -1.0385689735412598, -1.1094309091567993, 0.6498942375183105, -1.371132731437683, 0.6204437613487244, -0.5903692245483398, 0.04177367314696312, -1.4197674989700317, -0.05309247225522995, 1.9774270057678223, -0.18635201454162598, -0.9658868312835693, 0.2546224296092987, -0.6924352645874023, -0.3083241283893585, -0.8147568106651306, -0.5548837780952454, -0.24329443275928497, 1.0360437631607056, -0.8156667947769165, 0.9822230935096741, -1.8620774745941162, 0.21559742093086243, -0.24421505630016327, 1.2079946994781494, 1.466599464416504, 0.5690534710884094, 0.54730623960495, 0.5136016607284546, -0.4876340329647064, 1.1981241703033447, -1.2549690008163452], [1.3108165264129639, 0.47758567333221436, 0.774596631526947, -0.7805026769638062, -2.1058242321014404, 0.24049687385559082, 0.7499423623085022, 1.4525259733200073, -0.33139896392822266, 0.08629294484853745, -1.4590853452682495, -0.17318423092365265, 1.6531444787979126, 0.8806007504463196, -0.49370425939559937, 1.2524676322937012, -1.4715873003005981, 0.03580749034881592, 2.5122275352478027, 0.39851269125938416, -1.424814224243164, -0.01351818535476923, 1.4060533046722412, 0.4647812843322754, 0.20053578913211823, 1.9560487270355225, 0.25801771879196167, 0.48074182868003845, 0.03927184268832207, -0.45545411109924316, -0.972863495349884, -2.0268287658691406, -0.012674245052039623, -0.3457462191581726, -0.3456766605377197, 0.7798358201980591, 0.05376531928777695, -1.0178712606430054, -1.4015718698501587, 1.9757215976715088, 0.5515676736831665, -0.15075670182704926, -0.9505376219749451, -0.32804131507873535, 1.1400552988052368, 1.2786856889724731, -0.9319724440574646, 1.557060718536377, -0.7629086375236511, -1.2340033054351807, 1.5652475357055664, -1.1542458534240723, -0.25799086689949036, -1.8171225786209106, -0.8863343000411987, 0.7005734443664551, 0.6523162722587585, -0.8908857107162476, -0.37782493233680725, -0.01173443254083395, 0.603599488735199, 1.2763921022415161, -0.7325668334960938, 0.45093095302581787], [0.4464018642902374, -1.029329538345337, 0.2451678365468979, -1.128175973892212, 1.1239709854125977, -1.5939899682998657, 0.7662758231163025, 0.14300794899463654, -0.6775713562965393, -0.844024121761322, -1.0041025876998901, 0.4916532337665558, 0.4767625629901886, 2.2107982635498047, -0.03397368639707565, -0.9512922763824463, -0.4617565870285034, -1.0911500453948975, -0.3633188307285309, -0.2879779040813446, 0.8250557780265808, 0.0988662913441658, 0.8405275940895081, -0.5595009326934814, -0.4797143042087555, 0.43448394536972046, -0.06456194818019867, -0.9242256283760071, -0.3076302707195282, -0.18171906471252441, 1.5583175420761108, -1.6652858257293701, -1.8326373100280762, -0.9189223051071167, 0.029097339138388634, -2.064377784729004, -2.0863916873931885, -0.1729458123445511, 0.133118137717247, 0.5213601589202881, 0.08238071948289871, 2.1252858638763428, -0.7631502151489258, -0.018694305792450905, -1.0550895929336548, -0.9617021083831787, -0.8533378839492798, -0.9753504991531372, 0.45339053869247437, -0.7175770998001099, -0.58039790391922, 1.8948721885681152, -0.8685168027877808, -0.9146077036857605, -0.7106711864471436, 0.28418126702308655, -0.46521812677383423, -0.9134541749954224, 0.008356846868991852, 0.13275663554668427, -1.0869778394699097, 0.37080487608909607, 0.4024631679058075, -0.4770500957965851], [1.93439781665802, 1.5419366359710693, 1.653275966644287, 1.125451922416687, 0.9998370409011841, -1.2124890089035034, 0.8910882472991943, -2.02976131439209, -0.2801458537578583, 1.0199209451675415, 1.5936061143875122, -1.1208302974700928, 2.3624613285064697, -0.8718284368515015, 0.6350045800209045, -0.09385765343904495, -1.6840089559555054, -0.6551652550697327, 0.12366724759340286, 0.2218203991651535, -1.3219115734100342, 0.41295918822288513, 0.6191684603691101, -1.0377253293991089, 0.8009127378463745, -1.2894699573516846, -0.8713778257369995, -0.7529526948928833, -0.3976351022720337, -1.2234023809432983, 0.6282508969306946, -0.9500179290771484, -0.01855243742465973, -0.25434610247612, 0.15007179975509644, 0.29555025696754456, 1.1383521556854248, 0.24494044482707977, 0.6626471877098083, 0.8933883905410767, 2.4559309482574463, 0.4235485792160034, 0.6138654351234436, 1.2853947877883911, -0.13427351415157318, -1.0094181299209595, 1.577620506286621, 0.6563382744789124, 1.091517448425293, -0.2412586659193039, 1.3165271282196045, -1.267309546470642, -1.1670218706130981, -0.6579387187957764, -0.48251625895500183, -2.164396286010742, -0.947288990020752, -0.22480928897857666, -1.0921337604522705, 0.11800575256347656, -0.7597723603248596, -0.9242450594902039, -0.7987797856330872, 0.8724685311317444], [-0.8941072225570679, 0.8098932504653931, 0.600725531578064, 0.9019072651863098, -0.07378777116537094, 0.16740067303180695, 1.0432735681533813, 1.0609487295150757, 0.8744224905967712, 2.063659429550171, 1.1974226236343384, 0.656106173992157, -1.1153910160064697, 1.0673456192016602, 0.3485734462738037, -0.7296810746192932, 2.0515975952148438, 1.0283929109573364, -0.00880500115454197, -1.9551914930343628, 1.0139110088348389, -2.3663339614868164, 1.0444729328155518, -0.28555527329444885, -0.2900889813899994, -0.4666982591152191, -1.4900147914886475, -0.8208139538764954, 0.23409081995487213, 1.6047009229660034, -0.8449617028236389, 0.11262430995702744, 0.1395527869462967, -1.220945119857788, 0.8493637442588806, 0.6069941520690918, -0.5234270691871643, 0.7216691970825195, -1.4202200174331665, 0.661563515663147, 0.8084601759910583, 0.45627540349960327, -1.2240824699401855, -1.2422049045562744, -0.42253175377845764, 1.110840916633606, -1.4180526733398438, -0.7039557695388794, -1.604384422302246, 1.0473517179489136, -1.3240052461624146, -0.4676745533943176, -2.0250537395477295, 0.9546455144882202, 0.12380093336105347, 0.8754993081092834, 1.9927271604537964, -0.8995155692100525, 1.0640143156051636, -1.096097469329834, -0.6178561449050903, -0.48995259404182434, -0.5771908164024353, 0.14655955135822296], [0.34972187876701355, -0.3189953863620758, -0.8651215434074402, -0.12249861657619476, 0.5938822031021118, 0.3556195795536041, 0.39141449332237244, -0.7134013772010803, -0.37279582023620605, 0.20960068702697754, 0.12527191638946533, -0.7393592000007629, 0.40896058082580566, 0.6847224235534668, 1.5843040943145752, -0.4170544743537903, -1.2755670547485352, -0.7175373435020447, 0.9657551646232605, 1.8744828701019287, -0.47526609897613525, 0.3791808485984802, -1.7751779556274414, 1.3224550485610962, -1.5409419536590576, -0.027698390185832977, 0.5346412658691406, -0.27578407526016235, -0.651032567024231, -0.5792016386985779, 0.4347786009311676, -0.9066641926765442, -0.636473536491394, 0.6465104818344116, 0.18101286888122559, 1.3501559495925903, -0.5507976412773132, -1.2712410688400269, 0.9160007238388062, -0.17415611445903778, -0.5703323483467102, -0.1720518171787262, -0.12369741499423981, 0.919990062713623, -2.731694459915161, 0.2909432649612427, -1.4000639915466309, -0.248996302485466, 0.23513348400592804, -0.3001183271408081, -1.7387893199920654, 0.4422779679298401, 1.8751400709152222, 0.4836740493774414, 0.5183602571487427, 0.23363721370697021, 0.22752057015895844, -0.19368059933185577, -1.0351710319519043, 0.044266197830438614, -1.8722739219665527, 1.9178135395050049, 1.9247442483901978, 1.3617465496063232], [1.3522839546203613, 0.03549180179834366, 0.3622894585132599, 0.07393236458301544, 0.985001802444458, -1.5632482767105103, 0.41632702946662903, 0.7157418131828308, 0.06844360381364822, -0.14560821652412415, 0.7076338529586792, -0.37709158658981323, 0.7160173654556274, -0.5315360426902771, 2.0770273208618164, -1.1551204919815063, 1.1027562618255615, 1.6447948217391968, -0.30605635046958923, -0.2996275722980499, 0.2689623534679413, 0.9165694117546082, -0.6284868717193604, -0.7066304683685303, 1.0894925594329834, 0.9979049563407898, -0.43807724118232727, 0.5057196021080017, -0.15841248631477356, -0.9016200304031372, -1.1394540071487427, -1.3695852756500244, 1.6820740699768066, -0.43721452355384827, 2.052111864089966, 2.9760310649871826, -0.05323704704642296, 1.667794108390808, 0.5944598913192749, -0.6217449903488159, 0.3997234106063843, -0.47022902965545654, 0.9112271070480347, 0.21030569076538086, -0.06306059658527374, -0.22773173451423645, 1.4419070482254028, -0.7108374834060669, 1.496420979499817, 0.3963586688041687, -0.7216886878013611, -0.4022720456123352, -0.8218991160392761, 1.6840007305145264, 0.49502459168434143, 0.8187609910964966, -1.511488437652588, -1.4633451700210571, -0.5984435081481934, -1.0433412790298462, 0.6618570685386658, -0.09827402234077454, 0.4956114590167999, 1.7739747762680054], [1.467799186706543, -1.9068306684494019, 0.5483483672142029, -0.579601526260376, -0.9353941082954407, -0.9994873404502869, 0.5131279826164246, -1.8614437580108643, 0.044970255345106125, -1.213918924331665, 1.3419082164764404, 0.06248794123530388, -0.8784592747688293, 0.13074073195457458, 1.34990656375885, -0.027610531076788902, 0.06028888747096062, 0.32943016290664673, -0.997498631477356, 0.8445571660995483, 0.4993613660335541, -1.228886365890503, -1.2871123552322388, 0.8116814494132996, -0.3813421130180359, -0.4842938780784607, 1.8623803853988647, -1.0700771808624268, 0.7278150320053101, 1.0445644855499268, -2.651198387145996, -0.7819725871086121, 0.25571081042289734, 0.5527796745300293, 0.34691357612609863, 0.7261090278625488, -0.602790355682373, -1.3809863328933716, -0.6693698167800903, -1.5193053483963013, -1.2822601795196533, -0.6479061245918274, -2.519967555999756, 0.8234429955482483, -0.8934151530265808, -1.4879380464553833, -0.9040947556495667, -0.6683983206748962, -1.6650731563568115, -3.066906452178955, -2.2808752059936523, -0.49708667397499084, 1.0450979471206665, 0.040965158492326736, -0.5082697868347168, -0.37897399067878723, 0.89511638879776, -1.5901494026184082, -2.365772247314453, 0.5010021328926086, -0.4585014581680298, -0.9567897319793701, 0.42896318435668945, 0.533069372177124], [0.5841076970100403, 0.11381561309099197, -0.3309256136417389, 1.3542778491973877, 0.8359436392784119, 1.2256718873977661, 0.3244703710079193, 0.652742326259613, 0.13733255863189697, -0.18433867394924164, -1.5754446983337402, 0.7161546945571899, -2.1456005573272705, -1.6103135347366333, 0.08340688794851303, -0.2586977779865265, -2.019044876098633, 0.87861567735672, 0.5249361395835876, 0.6371968984603882, -0.7049925923347473, 0.7048792839050293, -0.7771500945091248, -0.9120750427246094, -0.8916873335838318, 1.030393123626709, 0.9468222856521606, -0.8306838274002075, -0.3362990915775299, -0.203016459941864, -3.015550136566162, 0.15097923576831818, -0.007341296877712011, -0.23569707572460175, -1.4343669414520264, 0.05294499546289444, 0.15074904263019562, -1.8442031145095825, 0.40306833386421204, -1.670563817024231, 0.712421715259552, -0.7232865691184998, 0.6912571787834167, 0.328292578458786, -0.05692553520202637, 0.17119430005550385, -0.9081233739852905, 0.949726939201355, 0.1631709635257721, -2.176473379135132, 0.34801095724105835, 0.5550147294998169, -2.0291717052459717, 1.7824612855911255, 1.1500797271728516, 0.14176923036575317, 0.31080204248428345, 1.0255714654922485, -0.9975578784942627, 0.06798041611909866, -0.6134620904922485, -0.01405664999037981, -1.6883126497268677, -1.2066482305526733], [2.120736837387085, -0.256905734539032, -0.7309820055961609, 1.4283943176269531, -0.589836061000824, 0.23515309393405914, -0.23949603736400604, -0.00786262284964323, -1.2518137693405151, -0.14655382931232452, -0.04953376576304436, 0.880676805973053, -0.6660484671592712, 2.0530004501342773, 0.3414429724216461, -1.3961763381958008, -1.337180733680725, -0.18403540551662445, 0.8565747737884521, 1.5300112962722778, -0.8897789716720581, 0.5084535479545593, 1.6609116792678833, 1.7780840396881104, 0.092067651450634, 1.0205632448196411, -0.20079892873764038, -1.1262763738632202, -0.4513060450553894, -0.4752683937549591, 0.5552480816841125, 0.7824349403381348, -1.4780206680297852, -0.7763067483901978, 0.6091905832290649, -0.24733874201774597, 1.18892240524292, -0.7830395698547363, 1.40541410446167, -0.7182444334030151, -0.6020670533180237, 1.3143033981323242, -0.36962124705314636, -0.9886749982833862, -0.0320381261408329, -0.26144593954086304, -1.0978679656982422, -1.7861517667770386, 0.2568224370479584, -0.057398732751607895, -0.35929998755455017, -0.438857764005661, 2.3877639770507812, -1.3548152446746826, -1.1541401147842407, 0.5556285381317139, 0.6733297109603882, -0.6437674760818481, 0.8057810664176941, -0.9534488320350647, 0.4309402406215668, 0.41070178151130676, 1.3942838907241821, 1.8288800716400146], [0.5853130221366882, 0.5850356221199036, -0.1637100875377655, -0.21550406515598297, 1.8985675573349, 0.31602150201797485, 1.3324694633483887, 1.9208804368972778, -1.7024251222610474, -1.138493537902832, 0.2998443841934204, -0.5204451680183411, 1.2856166362762451, 1.2608442306518555, 0.1428125500679016, 1.6140633821487427, -0.582897424697876, -1.2637220621109009, 0.6706479787826538, 1.067097783088684, -0.9232081174850464, -0.26569703221321106, -1.5944103002548218, -0.6217199563980103, -0.8937082886695862, 0.30654194951057434, -1.3904130458831787, -1.084514856338501, 0.9447709918022156, -1.3910998106002808, 0.25649920105934143, -1.004163384437561, 0.5283196568489075, 0.18724358081817627, -0.6374224424362183, 1.4074535369873047, -0.18583102524280548, 1.8053416013717651, 1.160414218902588, 1.1390869617462158, -0.5448748469352722, -0.23628638684749603, -0.1341399997472763, 0.4987773001194, 0.9422445893287659, -0.3930480480194092, 1.7637590169906616, -0.6656951308250427, 0.13188360631465912, 1.3431363105773926, -0.3686811327934265, 0.2327817678451538, 0.36321723461151123, 1.0763779878616333, 1.2238860130310059, -0.0019214439671486616, 0.9293211698532104, 0.3986620604991913, -0.4098755419254303, -0.11992411315441132, 0.06920907646417618, -0.19296278059482574, -0.8753606081008911, 1.134325385093689], [-0.21003279089927673, -0.33648237586021423, 0.1514013707637787, 0.2487620711326599, -0.16008679568767548, 0.6751432418823242, -0.5883389711380005, -0.799807608127594, 0.8711028099060059, -0.2025231271982193, -0.22736626863479614, 0.5433180332183838, -0.7765512466430664, 0.7362706065177917, -0.5198131799697876, -0.5653242468833923, 0.3483290672302246, -2.5085997581481934, 0.3864011764526367, -1.243316650390625, 0.7426258325576782, -0.48672741651535034, 0.10059954971075058, 0.1402086764574051, 0.9487861394882202, -0.3408289849758148, 1.713749885559082, 0.3351678252220154, -0.2561715245246887, -0.28141817450523376, -1.5846871137619019, -0.9317399859428406, 1.1967822313308716, 1.016569972038269, 1.055965542793274, -1.3790626525878906, -1.1644644737243652, -0.5165395736694336, -0.8705054521560669, 2.287879228591919, 0.17504054307937622, -0.5455229878425598, -0.4611225426197052, -2.3342368602752686, 1.2250208854675293, -0.7769735455513, -1.1506080627441406, 1.4279420375823975, 0.19242778420448303, 0.5455253720283508, -0.45382019877433777, -1.206497073173523, 0.11175242066383362, -0.5196855664253235, 1.0648401975631714, 0.9727636575698853, -1.1425039768218994, 0.380321204662323, 0.22271797060966492, -1.0662837028503418, 0.11027660220861435, 0.032280806452035904, -1.4966142177581787, 0.12237657606601715], [-0.25422340631484985, 0.701950192451477, 1.2177647352218628, -1.1268011331558228, -0.013546300120651722, 0.31275907158851624, 0.5687738656997681, -1.1898934841156006, -0.16418960690498352, -0.5824734568595886, -0.5669192671775818, -0.9595368504524231, 0.38470155000686646, -0.9219667315483093, 0.8427891731262207, 0.810286819934845, -0.5936633348464966, 1.2876229286193848, 0.19803079962730408, -0.44529640674591064, -1.0170872211456299, -0.8926827311515808, -0.23699767887592316, 1.4177722930908203, -2.3162410259246826, 1.0196199417114258, -0.45487427711486816, 1.4928467273712158, -0.743179202079773, 0.793684720993042, -1.141357660293579, -0.5935463309288025, -1.2858299016952515, -1.4705301523208618, -1.1905597448349, -0.4277353882789612, 0.6878349184989929, -0.5299373269081116, -0.5005412697792053, -1.2146096229553223, 0.38827717304229736, 0.7310364842414856, 0.3333947956562042, 0.5489187240600586, -0.5121756792068481, -1.0401349067687988, 0.2854667901992798, -1.1332731246948242, -0.5090759992599487, 0.4349803328514099, -1.16549813747406, -0.7679852843284607, 1.0566257238388062, -0.1490340232849121, -0.04712921380996704, -0.7003279328346252, 0.16443414986133575, -0.5374744534492493, -1.2063597440719604, -1.0472476482391357, 0.31372857093811035, -0.2371353954076767, -1.0542277097702026, 0.6535110473632812], [0.4583728611469269, 0.5926143527030945, 0.36087873578071594, -0.053582701832056046, 2.302281141281128, -0.35516077280044556, -0.20527340471744537, -1.6860928535461426, 0.43565088510513306, 0.6889960169792175, 0.1626112312078476, -0.49912944436073303, 0.37022650241851807, -0.5521215796470642, 2.119243860244751, 1.9632105827331543, -1.0389004945755005, -1.9221540689468384, -0.8198928236961365, 0.5070064663887024, -0.9354504346847534, -0.41649577021598816, 0.49717220664024353, 0.738944947719574, 2.826932430267334, 1.862541675567627, 1.146247148513794, -0.24903373420238495, 0.9277176856994629, 0.23396602272987366, 1.6382412910461426, -0.9314948916435242, -0.6244243383407593, -0.4350549876689911, -0.6577109098434448, 1.330175518989563, -1.683416724205017, 0.32447177171707153, 0.19816730916500092, 0.8737009763717651, -0.8750734329223633, -0.23750300705432892, -0.396048903465271, 1.04604172706604, -0.11385366320610046, -0.10084355622529984, -0.16253569722175598, -0.703630805015564, 0.9995502829551697, 0.37468355894088745, 0.8149960041046143, -0.408726304769516, -1.701230525970459, -0.28332990407943726, -0.31195247173309326, 0.9765189290046692, 0.7150800824165344, -0.17828615009784698, 0.23018763959407806, -0.5307809710502625, 0.44294437766075134, -0.8336727619171143, 0.06696221977472305, 0.45243212580680847], [-0.3318091332912445, -1.0690516233444214, 0.22845770418643951, -0.9197285175323486, 0.6161203384399414, -2.046556234359741, 0.8235908150672913, 1.0066542625427246, 0.2696872055530548, 0.049733418971300125, 2.724829912185669, 0.7119947671890259, -0.6176525354385376, -0.38772910833358765, 0.6043195724487305, 0.43050530552864075, 0.8353831768035889, -0.26817938685417175, -0.2700790464878082, 0.018779629841446877, 1.2288419008255005, -0.9482092261314392, -1.102502465248108, 0.5045685768127441, -1.7686727046966553, -1.9593333005905151, 0.5364948511123657, 0.07218822836875916, -0.276236355304718, -0.31920987367630005, 0.26141324639320374, 0.6778380274772644, 1.613512635231018, -1.516331434249878, 0.07001466304063797, 1.2978789806365967, -0.9259400963783264, 0.36484116315841675, -1.8820550441741943, -0.41921713948249817, -0.6669475436210632, -0.07581602036952972, 1.9813642501831055, -0.17344234883785248, 0.580377459526062, 0.1486366242170334, 1.9163089990615845, 0.8205577731132507, 0.7063415050506592, -0.9653791189193726, 0.06574132293462753, 2.640871047973633, 1.1112346649169922, 0.6299254298210144, -0.011690804734826088, 0.1520029604434967, 0.7164997458457947, -2.516998052597046, 0.476086288690567, 0.9963639378547668, 0.30503207445144653, 0.6929712295532227, 0.28915804624557495, -1.3913122415542603], [-0.4059399664402008, -0.34283140301704407, -1.0653942823410034, 1.5383676290512085, 0.7751965522766113, -0.5704441070556641, -0.5861291885375977, 1.168709635734558, 0.13970337808132172, 1.006745457649231, 0.30356910824775696, -1.1483628749847412, 0.11202479153871536, -1.0132695436477661, 1.040643334388733, 1.6586486101150513, -0.6681550741195679, 0.47141674160957336, -1.3716968297958374, -0.004803418181836605, 2.122255325317383, 0.12484677881002426, 0.07903819531202316, -1.6229244470596313, -2.582144021987915, 0.3673734962940216, -1.7128345966339111, -1.725342035293579, -1.0453522205352783, 1.70609450340271, -1.0335371494293213, 0.2752915024757385, -1.3177963495254517, 0.5323387980461121, -0.23618440330028534, 0.1084299311041832, 0.8803495764732361, -0.7335856556892395, 1.4543654918670654, 0.32534533739089966, -1.7478693723678589, 0.20778104662895203, -1.2144776582717896, 1.5542786121368408, -1.319130301475525, 1.0884140729904175, -1.1378127336502075, -0.4858059585094452, 0.8012730479240417, 0.04811757430434227, -0.6429338455200195, 0.581352949142456, 1.143889307975769, 0.17070353031158447, 1.2007204294204712, -1.093335747718811, -0.657322883605957, -0.9864184856414795, -0.4324803948402405, -0.33684009313583374, 0.2278905212879181, 0.4807978868484497, 0.4360286295413971, -0.5286811590194702], [0.505395233631134, -0.0653986856341362, -0.18079863488674164, 2.023613929748535, 0.9859813451766968, 0.6811830997467041, -0.18931829929351807, -0.22513814270496368, -0.9246428608894348, 0.44072142243385315, -1.495430588722229, -1.39975905418396, -0.05376788601279259, -1.3679852485656738, 1.061083197593689, -0.48611006140708923, -1.5613040924072266, 0.2056265026330948, 0.20673277974128723, -0.6603996753692627, -0.5807613730430603, 0.5361823439598083, 0.9605880379676819, -0.7432146072387695, -0.21799461543560028, -0.3079359233379364, 0.12302275002002716, -0.07146867364645004, -0.418417364358902, -1.5603306293487549, -0.25495606660842896, 0.4685962200164795, 0.7082637548446655, -1.559074878692627, 0.6912584900856018, -2.411423683166504, -2.1830785274505615, -2.2383382320404053, 0.20033608376979828, 1.3131576776504517, -0.3428388237953186, -0.16701383888721466, -0.6184821128845215, -1.894575834274292, -1.3685382604599, -0.8951249122619629, 0.18106329441070557, 0.6661587357521057, -0.5760992765426636, 0.8116394281387329, -0.47132736444473267, -1.421970248222351, -2.0300111770629883, -0.29682400822639465, 0.7231982946395874, 0.06608503311872482, -0.403990775346756, 1.6665998697280884, -0.0272569227963686, 0.8407000303268433, 0.723141074180603, 1.0444227457046509, -0.5174022316932678, -0.8768009543418884], [0.30210554599761963, -0.7893226146697998, 0.08900097012519836, 1.917421579360962, 0.5205017328262329, 0.8084481358528137, -0.35062962770462036, -0.06424654275178909, 0.123249851167202, 0.6892426609992981, -0.43564915657043457, -0.9325570464134216, 1.9231784343719482, -0.5787774324417114, 0.7261061668395996, 0.9956377744674683, 0.339813768863678, -0.5690827965736389, 0.23663640022277832, 0.10351777076721191, -0.2596134841442108, -1.243925929069519, 0.7112134099006653, 1.9547951221466064, -0.4616735279560089, -1.1810544729232788, 0.4134516716003418, -0.48642128705978394, -1.356255054473877, 0.841494083404541, 0.43241244554519653, 0.21859963238239288, 0.6634030938148499, -0.07660352438688278, -0.8915177583694458, -0.954971194267273, 1.8679802417755127, -0.45976772904396057, -0.05708080157637596, -0.6610610485076904, -1.146085500717163, -0.8129625916481018, 0.868587076663971, -0.17446444928646088, -0.5770675539970398, -0.06281173974275589, 0.4352196455001831, 0.9480218291282654, -1.9283828735351562, -0.605583667755127, 1.025640845298767, -0.16611462831497192, -1.3485815525054932, -1.3334623575210571, -1.1712855100631714, -1.215591549873352, 0.34047597646713257, 0.3598283529281616, -0.8133062720298767, 1.346325159072876, 1.8972039222717285, -0.3052586019039154, -0.3270758092403412, -0.02612609602510929], [-1.3343569040298462, -0.26909756660461426, 0.5389930605888367, -1.1805201768875122, -0.5216550827026367, 1.2670155763626099, 0.17676429450511932, -0.20625253021717072, -0.07195767760276794, 1.2371704578399658, -2.127246856689453, -0.3158944547176361, 0.35167819261550903, 0.7308357954025269, -0.35705557465553284, 0.3246941566467285, -0.049318891018629074, -1.3061919212341309, -2.0950024127960205, -0.002572382101789117, 0.546241044998169, -0.3271156847476959, 0.3906140923500061, -0.9410308599472046, -0.1782851219177246, 0.08092785626649857, -1.3768516778945923, 0.5381855964660645, 0.38155436515808105, 0.6230248808860779, -0.6951810717582703, -0.1944306641817093, 0.9718229174613953, 1.009572148323059, -0.3280677795410156, -1.1602171659469604, -0.022521477192640305, 0.8656354546546936, -0.8213815093040466, 0.24288597702980042, 0.49525147676467896, -0.8196353912353516, -0.6350938677787781, 0.9614449739456177, 1.0716211795806885, -0.930502712726593, -1.3263788223266602, -0.7926939129829407, -0.42222705483436584, -1.411550760269165, -0.5533972978591919, -1.3823598623275757, -0.26489636301994324, 0.34994256496429443, -1.8607863187789917, -0.3179096579551697, -0.591302752494812, -0.3376001715660095, -0.7819263339042664, -0.918785572052002, -0.06884203106164932, -0.8048574328422546, 0.005941078066825867, 0.590370774269104], [1.5972552299499512, 1.09614896774292, 0.826515257358551, 1.0327988862991333, -0.11814592778682709, -0.6737617254257202, 1.3881317377090454, 0.8219419717788696, 0.38791197538375854, -0.17855434119701385, 0.4441535472869873, 0.7744730114936829, 1.0555158853530884, 0.612097978591919, -0.21901848912239075, 1.0018895864486694, 0.3963104486465454, 0.6544942259788513, 0.16185200214385986, -0.2497040331363678, 0.005050400272011757, 0.5091014504432678, -0.3358408510684967, 1.0590794086456299, -3.1432900428771973, -0.6564884781837463, 2.1372358798980713, 0.26169756054878235, 1.283545970916748, 0.786903977394104, 1.1538243293762207, -1.5800297260284424, 0.4032692611217499, -0.43071579933166504, -2.304680585861206, 1.2214041948318481, 0.6873278617858887, 0.3258897662162781, -0.6151102185249329, 1.3756710290908813, -0.6648446321487427, 0.7352074384689331, 0.25871866941452026, -0.22317703068256378, -0.4256601631641388, 1.0671145915985107, -0.30612215399742126, 1.8323744535446167, -0.037556059658527374, 1.0937975645065308, 1.295662522315979, -0.16854670643806458, -0.5973784923553467, -0.9390676617622375, 0.5665014982223511, 0.9289935827255249, 0.00843917578458786, 1.1188832521438599, 1.0076875686645508, 0.9058177471160889, -0.16254767775535583, 0.5319737792015076, 0.3964800238609314, -1.36594557762146], [0.6967822909355164, 0.5798395276069641, 0.3768085241317749, 1.903669834136963, -0.012348228134214878, 0.6820009350776672, -0.21923434734344482, -1.1077899932861328, -0.8019884824752808, 0.3645685911178589, -0.5644626021385193, -2.218021869659424, -0.3570847809314728, -1.5445507764816284, 0.9884483814239502, 0.39179590344429016, -0.042862419039011, -0.7300788760185242, -0.5632995367050171, 0.943283200263977, 0.43693339824676514, 0.7391493916511536, -0.9027935862541199, 1.4121806621551514, 0.011055135168135166, -0.3847534656524658, -0.39265188574790955, -0.41121038794517517, -0.07494229078292847, -0.4672309458255768, -0.1049504205584526, -0.2632211446762085, -1.1704734563827515, -0.6189576983451843, -0.2142024040222168, 2.5396180152893066, 0.33496350049972534, -1.0607949495315552, 1.0376988649368286, 0.4319150745868683, -0.17677956819534302, 0.8504257202148438, -0.2242247313261032, -0.2806263267993927, -1.3237560987472534, 0.7943732738494873, -0.6770826578140259, -0.4081963002681732, -0.8786026835441589, 0.988672137260437, 1.5712027549743652, 1.9307233095169067, 0.672052800655365, 0.7170314788818359, 0.23189428448677063, 1.7377488613128662, 1.0228564739227295, -0.15182669460773468, -1.6738359928131104, -0.23617978394031525, 0.055772311985492706, -1.0669989585876465, 0.3514518439769745, 0.9088345170021057], [1.0166527032852173, 0.5224006175994873, -0.4288025200366974, 0.7733375430107117, 0.8118264675140381, -0.18714450299739838, 1.1595722436904907, 1.01797354221344, -0.06162629649043083, -0.9138138294219971, -1.4923547506332397, 1.5897008180618286, 1.1312839984893799, 0.1979120671749115, -0.38054683804512024, 0.33914878964424133, -0.19229795038700104, -1.4789000749588013, 1.3733474016189575, -1.537903070449829, 1.4657764434814453, 0.36350277066230774, -0.2795833647251129, 0.916519284248352, -0.9082390666007996, -0.39418384432792664, 0.8503212332725525, -0.9012966752052307, 0.03091156668961048, -0.1603393852710724, -0.7981769442558289, -0.010086056776344776, 1.7418206930160522, 1.53970468044281, -0.2693765461444855, -0.31563228368759155, 1.2167887687683105, -0.6182557940483093, -0.9670925736427307, 0.1871858686208725, 2.0115835666656494, -0.1087292954325676, -3.475947856903076, 0.18602153658866882, 0.8643973469734192, 1.2395042181015015, 1.262335181236267, -0.21413226425647736, -0.559376060962677, 0.7447746992111206, -0.31500473618507385, 1.5457043647766113, 1.449845790863037, 0.27968910336494446, -1.046510934829712, -0.47751423716545105, 0.8635820746421814, 0.08183740079402924, 1.0767979621887207, 0.9269798398017883, -0.33172720670700073, 0.27445465326309204, 1.3587629795074463, 0.8142853379249573], [1.1914753913879395, 1.6011656522750854, -1.6655563116073608, -0.8101516366004944, 2.11545729637146, -1.2120347023010254, 0.11852963268756866, 0.5760616660118103, -1.8749830722808838, 1.2323460578918457, -1.9216089248657227, -0.36825689673423767, -0.5036264657974243, -1.5833851099014282, 0.042667876929044724, 0.41497665643692017, -0.6391534805297852, 1.089640736579895, -0.21923266351222992, -0.5109055638313293, -0.27931228280067444, -1.1936500072479248, -0.34518563747406006, 0.7272857427597046, -1.2080891132354736, 1.6123450994491577, -0.06146502122282982, 0.6200621724128723, 1.2595254182815552, 0.00377110717818141, -0.5503183603286743, -1.4968172311782837, 2.500089645385742, -0.6784625053405762, 0.24774961173534393, -1.3005609512329102, 0.5215173959732056, -0.7478649616241455, 1.3769848346710205, -1.9866116046905518, -0.13805633783340454, -0.48114487528800964, 2.3858447074890137, -0.32410499453544617, -1.550410509109497, 0.24407093226909637, 1.4834266901016235, 0.4132935106754303, -0.14208129048347473, -0.1215391531586647, 0.6659466624259949, -1.295859456062317, -1.3730156421661377, -1.6394959688186646, 0.38064759969711304, 0.823693037033081, 0.5999870896339417, 2.494934558868408, 0.4208332598209381, -0.0933322012424469, -0.7402439713478088, -0.3723527491092682, 1.6003029346466064, 0.5840700268745422], [-0.004884947091341019, 0.21855831146240234, -1.7891091108322144, -0.0894491896033287, -0.8393489122390747, 0.990772545337677, 1.538403034210205, -1.324455976486206, -0.2291981726884842, 0.6105578541755676, 0.8581656813621521, 1.304073452949524, 1.395294189453125, 0.1693282425403595, 0.6142913699150085, -1.1601579189300537, -1.660159707069397, 0.2821255922317505, -1.8576240539550781, 0.006868930999189615, 0.8632990121841431, -0.9161115288734436, 0.9296631217002869, 2.637657880783081, 1.412995457649231, 0.02154974266886711, -0.3910911977291107, 1.668166160583496, -0.18617020547389984, 0.3370882272720337, 0.05539378896355629, 0.13880081474781036, -0.04019453376531601, 0.8305549621582031, -0.1980973333120346, 1.531655192375183, -1.2036210298538208, -0.892135500907898, -0.7216469049453735, -0.7261780500411987, -0.7559431791305542, -0.8488945960998535, -0.5128752589225769, -1.742367148399353, -0.6924026608467102, 1.5617557764053345, 0.6001638770103455, -0.8103824257850647, -0.22211141884326935, 0.24546915292739868, 2.159816265106201, 0.41315561532974243, 0.08120866119861603, 0.7865065336227417, -2.150418758392334, 0.6053624153137207, 1.4774378538131714, 1.402290940284729, -0.764650821685791, 0.5522282719612122, 0.5941541790962219, -0.6165063977241516, 1.6268717050552368, -0.16952434182167053], [0.04018242657184601, -0.19384905695915222, -0.9771081209182739, 0.982286810874939, -2.386009693145752, 1.9755288362503052, -0.678977906703949, 1.9442250728607178, -0.34049683809280396, 1.0958563089370728, 0.5857176184654236, -2.315175771713257, -0.7316522598266602, -1.3905415534973145, 0.3744174838066101, 1.7358843088150024, 1.2698798179626465, 0.5947152376174927, 1.2860103845596313, 0.3806666433811188, -0.6432562470436096, 2.034865140914917, -0.12476496398448944, 0.08401978760957718, -0.014583097770810127, -0.7342624068260193, -0.7532861828804016, -0.18716809153556824, 0.041794028133153915, 1.3655807971954346, -0.002932477043941617, 0.7575663924217224, -1.662050724029541, 0.08115002512931824, -0.5205295085906982, -0.17180949449539185, -0.0022731872741132975, 1.1913527250289917, -0.3200846314430237, 0.3742468059062958, -2.005176544189453, 1.563508152961731, -0.8940513730049133, 0.887417197227478, 1.9586679935455322, -0.4190971255302429, -0.9367745518684387, -0.33355623483657837, 2.1370439529418945, 0.3306739032268524, 0.4358776807785034, 0.4621857702732086, -1.1464827060699463, -0.6267904043197632, -0.7454879879951477, 1.0733616352081299, 0.2119133174419403, 0.2592199444770813, -1.2089924812316895, 1.101562738418579, -0.6162033677101135, 0.12912364304065704, -0.9405528903007507, 2.083763360977173], [-0.8067917823791504, 1.2952768802642822, 1.0671501159667969, 0.040805358439683914, -1.308885097503662, 0.36260953545570374, -0.1300930380821228, 0.5967763662338257, 0.5497336983680725, -0.1734393984079361, -0.9118545055389404, -0.5678903460502625, 0.2048555463552475, 1.2148102521896362, -2.034090757369995, -1.4839603900909424, 0.5609655380249023, 0.3867115378379822, 0.15059936046600342, -1.863242506980896, 1.3204079866409302, -1.0159589052200317, 1.2117036581039429, -0.03349500894546509, -1.223437786102295, 0.014616400003433228, -0.5883131623268127, 0.7475962042808533, -1.1105557680130005, -0.6398078203201294, -0.4522207975387573, -0.6607406735420227, -0.017554722726345062, 0.9521604776382446, -0.06759165227413177, 1.242012619972229, -0.4354826807975769, 0.276213139295578, 0.19114232063293457, 0.26735249161720276, -1.1425145864486694, -0.0011957659153267741, -0.5426633358001709, -0.30158552527427673, 0.5832274556159973, -0.6868138313293457, -0.009417304769158363, 0.6325491666793823, -0.43788906931877136, -0.27218130230903625, -0.14050418138504028, -1.3665337562561035, -0.9155199527740479, 1.5197330713272095, 1.1306201219558716, -0.8103635311126709, -0.8898109793663025, -0.7979600429534912, -0.2900325357913971, 0.6798127293586731, -0.030965009704232216, -0.16653680801391602, 0.16515135765075684, 1.8632701635360718], [0.9145824909210205, -0.8698582649230957, -0.8718473315238953, -2.164163827896118, 2.1479878425598145, 1.1648085117340088, 1.6057442426681519, -0.2304796427488327, -0.5068716406822205, -0.5083814263343811, 1.6250908374786377, -1.7455061674118042, 2.2579963207244873, -1.0856659412384033, -1.1513025760650635, -1.6018333435058594, -0.8927826285362244, -0.400590717792511, 0.2195139080286026, -2.036963701248169, 1.8057700395584106, -0.803435742855072, 0.7888723611831665, -0.022102076560258865, -0.6416066288948059, -0.02955278381705284, -0.15604622662067413, -0.9555599689483643, 0.8979517221450806, -0.49834197759628296, 0.9648338556289673, -0.2613745927810669, 0.5583673119544983, 1.414310336112976, 0.6888709664344788, -0.07322689890861511, -0.7136327624320984, -0.03564715385437012, 0.37635135650634766, 0.5581852197647095, 1.2081753015518188, -1.8532345294952393, 0.47582435607910156, 1.409637451171875, 0.4102310836315155, 0.6680739521980286, -1.099428653717041, -0.7115698456764221, 0.08021631836891174, -1.4316620826721191, -0.1259252429008484, 1.6593204736709595, 1.8535219430923462, 1.0284793376922607, 1.6839499473571777, -0.5186522603034973, 0.1297256499528885, -1.348806381225586, 0.50489342212677, -0.6713118553161621, 0.6699728965759277, -0.039565540850162506, -0.5690768361091614, -1.01865816116333], [-0.41246381402015686, 0.6243048310279846, -1.3062419891357422, 0.17367638647556305, -0.7354349493980408, -0.089577317237854, -0.3368726372718811, 0.09467384964227676, 0.9813435077667236, 0.10624085366725922, -2.067091226577759, 0.26947808265686035, 0.595878541469574, -0.00820303987711668, -1.3911463022232056, -0.8353191614151001, -0.550991952419281, 0.0040591564029455185, -1.8691189289093018, 0.5121005773544312, 0.06174816936254501, 0.19183315336704254, 2.0157694816589355, -0.7676076889038086, 0.3035108149051666, -0.3337770700454712, 0.18875299394130707, 1.7472779750823975, -0.9862965941429138, -0.2788180708885193, 0.5540761351585388, 0.3947121798992157, 0.04169752076268196, 0.9239892363548279, 0.8422405123710632, 0.34720325469970703, 0.8737581372261047, 0.06530430912971497, -1.7865533828735352, -0.3727751672267914, 1.529039978981018, 2.707465648651123, -0.04420693963766098, -1.87384033203125, -0.9926601648330688, 0.9480885863304138, 0.12658581137657166, 1.2321059703826904, -2.525031089782715, -1.516176462173462, 0.4456459879875183, -1.2453550100326538, 1.4192402362823486, -1.0280942916870117, 1.2066981792449951, -0.5317912697792053, -0.23313868045806885, 1.0998340845108032, 0.10562682151794434, 1.969333529472351, -0.7134814262390137, 0.5846468806266785, 0.21922093629837036, 0.26319143176078796], [0.043526627123355865, 0.24327100813388824, -0.6749482154846191, 1.868417739868164, -0.36272621154785156, 0.005623180884867907, 0.04851566627621651, -1.3001235723495483, -1.6306324005126953, 0.13386186957359314, -0.7006813287734985, -0.1448269635438919, 0.9482449889183044, -0.5180667638778687, 0.2939987778663635, -1.2438772916793823, -0.12918595969676971, 0.5968963503837585, -0.7988666892051697, -0.0516955591738224, -0.6025988459587097, -0.1825890839099884, -0.6081446409225464, -1.8989474773406982, -1.7706656455993652, 0.45506152510643005, -0.030096298083662987, 1.6802517175674438, 1.3614145517349243, 0.6499481797218323, 1.3664597272872925, 0.6375111937522888, -1.6748884916305542, -0.9780939817428589, -0.46173062920570374, -0.3664083182811737, -0.008064848370850086, 1.5282857418060303, 0.4634958505630493, 0.17450088262557983, -0.11959312111139297, -1.505395531654358, -0.6503943204879761, -1.38679838180542, -0.05633772909641266, -0.8868235945701599, -0.7914799451828003, 0.02474213019013405, -2.0469906330108643, -0.2865271270275116, -1.3159822225570679, 0.5689806938171387, 0.16841630637645721, 0.38313156366348267, 0.5222213864326477, 0.8748252391815186, 0.021020442247390747, 0.3643588721752167, -0.8561275005340576, -1.350069284439087, -0.07597119361162186, 1.0219584703445435, 1.2927759885787964, -1.2086341381072998], [0.04907900467514992, -0.8224460482597351, 1.6011238098144531, 0.3643190860748291, -0.7997329831123352, -0.6404417157173157, 1.194593071937561, -0.6871289014816284, 0.6785221099853516, -0.2819965183734894, 1.2347116470336914, 1.0268239974975586, 0.5660942196846008, 1.5649359226226807, 2.3502659797668457, -0.9495746493339539, -0.5159898400306702, -0.037249576300382614, -0.4626844823360443, 0.317221999168396, 1.7428267002105713, -0.6184020042419434, -0.8053791522979736, -0.7550656795501709, -1.2418489456176758, -0.8293339014053345, 0.42841753363609314, -0.6781027913093567, 1.8521519899368286, -0.3720146715641022, 0.49634164571762085, -0.6829363107681274, -1.9923802614212036, 0.15360642969608307, 0.6944147348403931, -0.0038429591804742813, 0.7732899188995361, -0.03820350393652916, 1.2581700086593628, 0.7298442125320435, 1.1676791906356812, -0.41212406754493713, -0.1845463514328003, 0.10892768949270248, -0.7018429636955261, -0.5351614952087402, -0.7187011241912842, -0.15215003490447998, 1.7362765073776245, 0.8684241771697998, 0.8993182182312012, 1.0484204292297363, 0.22831255197525024, -1.0735626220703125, 1.5127828121185303, -0.5980674028396606, -1.6982252597808838, 0.6517576575279236, -1.374651551246643, 0.05240365117788315, 0.0037898095324635506, -1.2891775369644165, 0.08511775732040405, -0.06956852227449417], [-0.874422013759613, 1.395383358001709, 1.0639888048171997, 0.5539552569389343, -0.9413099884986877, -0.9113966226577759, -1.7235959768295288, 0.23606516420841217, 1.720609188079834, -1.1709753274917603, -0.1856176108121872, 0.7300444841384888, -0.2822240889072418, -0.16606861352920532, 0.5850257277488708, 0.791560173034668, 0.5535784959793091, -0.6616950035095215, -1.3154627084732056, 0.9691208004951477, 0.33651331067085266, 1.04936945438385, -1.4595121145248413, 0.9197680950164795, -0.4641558825969696, -1.746077537536621, 1.3801460266113281, 0.5036856532096863, 0.07367119938135147, 1.2920196056365967, -0.037685856223106384, 1.1850392818450928, -0.5589489340782166, -1.308784008026123, -0.166367307305336, 0.22677169740200043, 0.6465296149253845, -0.5100405216217041, 0.9833064079284668, 2.0947442054748535, 0.05061604455113411, -0.3727762699127197, 0.5510053634643555, 0.5738046765327454, -0.47051939368247986, -1.1449229717254639, -0.27356478571891785, 0.17890408635139465, 0.25671762228012085, 0.14736686646938324, -0.361765593290329, 0.32950058579444885, -0.47770199179649353, -0.5199991464614868, -0.754864513874054, -1.3105897903442383, 0.6958805322647095, -1.5021430253982544, 0.1527373194694519, -0.20119450986385345, -0.10825267434120178, -2.346343755722046, 1.7179232835769653, 0.07306138426065445], [-0.46713441610336304, -0.7173401117324829, -0.7440845370292664, -0.5932953953742981, -1.4817575216293335, 0.4385657012462616, -0.14948585629463196, 0.3805440366268158, -0.12749938666820526, 0.8707192540168762, 1.6131948232650757, -1.2173563241958618, 0.300989031791687, 0.6286560893058777, 1.4995386600494385, -0.11608751863241196, 0.5435131192207336, -0.5006552338600159, -1.4943846464157104, 0.2843009829521179, 0.12759537994861603, -1.2366619110107422, -0.308594286441803, 1.0226738452911377, 1.3255298137664795, 1.0197420120239258, 0.25669020414352417, 1.8194046020507812, 1.8336578607559204, -0.888658881187439, -1.2407666444778442, -0.13767416775226593, 0.6992641687393188, -0.09900826215744019, 0.6053228974342346, 0.7148603200912476, 0.4667074680328369, 1.165372371673584, -0.8807390332221985, 1.160912036895752, 1.2457520961761475, -1.3735703229904175, 0.5773855447769165, -0.27087727189064026, -0.4132007956504822, -1.3869415521621704, 0.15334473550319672, 0.6209449172019958, 0.21322371065616608, -0.25893983244895935, -1.0161898136138916, -1.0171750783920288, -0.15956322848796844, 0.4612692892551422, -1.462225317955017, -0.5652431845664978, -1.2035627365112305, -0.2103457897901535, -1.0438021421432495, 0.10158542543649673, 1.156274437904358, -0.7886695861816406, -0.9319924116134644, -0.9337780475616455], [-1.3037090301513672, 0.7774506211280823, 0.5187252759933472, -0.2807759642601013, -0.5975967049598694, -0.7715334892272949, 0.8522353768348694, -0.9220377206802368, 0.30622774362564087, -0.48328760266304016, -0.7437645196914673, 1.7922114133834839, -0.9802421927452087, -1.5120021104812622, -0.8834823966026306, -1.2703551054000854, 0.31502100825309753, -0.3811202943325043, -2.2168502807617188, 0.5044117569923401, -0.225642591714859, 0.15435735881328583, 1.712415099143982, 1.9786856174468994, -1.0659466981887817, -0.32734215259552, -0.3233177959918976, 2.240508556365967, -0.17937511205673218, 0.8535439968109131, 0.3922337293624878, 0.19495514035224915, -0.17850331962108612, 0.5042813420295715, 1.8309804201126099, 0.05887471139431, -0.09258455038070679, -0.6426237225532532, 0.25009989738464355, 1.8012644052505493, -1.001170039176941, 0.44170692563056946, -0.7639282941818237, 0.8751208782196045, -0.6608076691627502, 2.0979607105255127, 0.03444235026836395, 0.7416257262229919, -1.7086714506149292, -0.19946986436843872, -1.2204490900039673, -0.36532682180404663, 0.16486620903015137, -0.5884003043174744, 2.821915864944458, -0.7965036630630493, -2.077737808227539, -0.9925617575645447, -1.336965799331665, 1.3458462953567505, 0.08787806332111359, 1.0219093561172485, -1.2840017080307007, -0.15939195454120636], [0.30380043387413025, -2.761730670928955, -0.7862859964370728, -0.8051031231880188, 0.9770186543464661, 0.35158875584602356, -0.5337450504302979, -0.9028921127319336, -2.898627996444702, -0.1544172763824463, 0.12756860256195068, -0.1398584246635437, -2.2621936798095703, -0.2179834097623825, 0.027148282155394554, -0.41096028685569763, -1.94290030002594, 0.8509342074394226, 0.20881937444210052, 0.09233219176530838, -1.3016371726989746, -0.6921453475952148, 0.11122746765613556, -0.4083329737186432, -1.7080625295639038, 0.4355294704437256, -0.2954254150390625, 0.8017998933792114, -0.9109144806861877, -1.2484475374221802, 0.6405401825904846, 0.3424624502658844, 0.27004382014274597, -0.8989235162734985, 2.1616289615631104, -0.5157077312469482, 0.6866754293441772, -0.038254622370004654, -0.5748516917228699, 0.972801685333252, 0.08124347776174545, 0.6675456762313843, -0.23892326653003693, -0.19427809119224548, 0.024178575724363327, -0.6746023893356323, 0.04980611428618431, 0.6289783716201782, 0.7253758311271667, 1.217496633529663, 0.8073192834854126, 1.2149430513381958, -1.1163272857666016, 0.3020041286945343, 0.417527973651886, -0.9339332580566406, 1.3225613832473755, -0.19478395581245422, 0.8606817722320557, -1.6789937019348145, -1.4534767866134644, -0.24180862307548523, 1.02107834815979, -0.21669107675552368], [-1.0563703775405884, 1.301498293876648, 1.2319777011871338, 0.08066315948963165, 1.0542960166931152, 0.0045817214995622635, -0.26760581135749817, 0.38230568170547485, 1.7489863634109497, 1.6507679224014282, -0.14573833346366882, 0.04427730292081833, -1.2120027542114258, 0.302021861076355, 0.24733054637908936, 0.3996061086654663, 1.0058997869491577, 2.5177762508392334, -0.30582574009895325, 0.2953214645385742, -0.5580648183822632, -0.7439193725585938, -0.22755427658557892, 1.2567987442016602, -0.11848924309015274, 1.0756887197494507, -0.458267480134964, 1.612310767173767, 0.27136990427970886, 0.613145649433136, 0.9845765829086304, -1.3495516777038574, -0.47312384843826294, -1.1128360033035278, 0.6639087796211243, 0.21772439777851105, -1.4589475393295288, 0.7999656200408936, 0.16187335550785065, -1.0471181869506836, -0.39329373836517334, -1.1312400102615356, 0.19366177916526794, -0.24784636497497559, -0.4578343629837036, -0.15832805633544922, -0.7034842371940613, 0.38901710510253906, 0.16825245320796967, -1.2609964609146118, 0.8639008402824402, -0.22253398597240448, 0.0944163128733635, -0.3132828176021576, -0.1903245598077774, 0.4888671636581421, 0.1818176507949829, 0.6602164506912231, 1.5337806940078735, -0.5922070741653442, -0.5105869174003601, 0.014863804914057255, -0.054567333310842514, -1.3856031894683838], [-0.734363317489624, 0.9059513807296753, -1.5957293510437012, 0.6010016798973083, 1.3349438905715942, -1.3868374824523926, -0.5596465468406677, 1.4474307298660278, -1.1520627737045288, 0.9585387110710144, -0.5042436122894287, 2.2075212001800537, 0.09905489534139633, -1.0153155326843262, 1.1372947692871094, 1.2418348789215088, -2.2820327281951904, -1.9371799230575562, -0.07016584277153015, -0.07950856536626816, 0.49009034037590027, 0.9255166053771973, 0.5029363632202148, -0.9217306971549988, -1.3969359397888184, -0.9558247327804565, 1.3011689186096191, 1.2037543058395386, -0.24249030649662018, 0.041499022394418716, 1.2330445051193237, -1.4265086650848389, -0.9862344264984131, 0.2054283320903778, 0.5735514760017395, -0.12195610255002975, 1.5564346313476562, -0.5108478665351868, 0.3277674615383148, 1.428613543510437, 0.10865827649831772, -0.4000702202320099, -0.12139160186052322, -0.37415969371795654, -0.2946457266807556, -0.10899737477302551, -2.225025177001953, 0.24864907562732697, -1.3839497566223145, -0.9937997460365295, 0.29202020168304443, -0.18871568143367767, -0.00595136871561408, -0.755165696144104, -2.1572492122650146, 0.6114093661308289, -0.7974226474761963, -0.7888069152832031, -0.3068757951259613, 0.06349875777959824, -0.544117271900177, -1.6108181476593018, 1.3494200706481934, 1.3196240663528442], [1.5362565517425537, 0.12890072166919708, 1.3653301000595093, -0.458173543214798, 0.7231281399726868, -0.7487580180168152, 0.48009711503982544, -0.2955031394958496, -0.6482736468315125, -0.025500774383544922, 1.074794888496399, 0.7005111575126648, 1.6469848155975342, 0.7399662137031555, 0.05572289973497391, -0.7317264676094055, -1.0127203464508057, 3.0237302780151367, 0.6199811100959778, -0.846321702003479, 1.027879238128662, -2.263965129852295, 0.2293001413345337, -0.6977052092552185, 0.7108986377716064, -1.3508827686309814, 0.6254924535751343, -1.1067440509796143, 1.1834057569503784, 0.25268009305000305, -0.983346700668335, 0.18865884840488434, -0.3417588174343109, -1.4217907190322876, -0.779961884021759, 1.0253342390060425, 0.5519366264343262, -0.15002089738845825, 0.9427855014801025, -1.3741505146026611, -0.09382867068052292, -0.2958497405052185, 1.6795588731765747, -0.25198256969451904, 0.966666579246521, -0.12324200570583344, -0.3539125323295593, -0.9520922899246216, 0.7034308910369873, 0.8005771636962891, -1.4837987422943115, -1.4458757638931274, 0.7503892183303833, 1.7234528064727783, 0.38218387961387634, -0.8481518626213074, -0.48669683933258057, -0.49503037333488464, 1.1420717239379883, 1.742713451385498, -0.14228560030460358, -0.9380002021789551, -1.1345692873001099, 0.15959204733371735], [0.09993114322423935, -1.258094072341919, -1.4419893026351929, 0.29641401767730713, 0.015826191753149033, 0.8001253604888916, 0.020613640546798706, 0.47068244218826294, 1.2694120407104492, 0.3118976652622223, 1.4592543840408325, 0.30888012051582336, 0.2936897873878479, -1.4431947469711304, -1.2223812341690063, 0.586530864238739, 0.05584997683763504, 0.49679774045944214, -1.8429416418075562, 0.9937787652015686, -1.8923245668411255, -0.33365559577941895, -0.8741418123245239, 1.2121753692626953, -1.7137062549591064, 1.3958979845046997, -0.5751709342002869, -1.4645612239837646, 0.31410765647888184, -0.7840055227279663, 0.7533607482910156, -0.8726822733879089, -0.5819804668426514, -0.3436517119407654, 0.30068880319595337, 1.0932716131210327, -1.1632165908813477, 2.791752815246582, -1.0403047800064087, -0.3871098756790161, -0.7711957097053528, -0.029247276484966278, 0.09499047696590424, -1.3331325054168701, 0.8675759434700012, 1.2068547010421753, 0.8848047256469727, -0.07760924845933914, -0.7612780332565308, 1.043471336364746, -0.5564148426055908, 1.119270920753479, -0.3083936870098114, 0.35776010155677795, -0.2575579881668091, 1.224195957183838, 1.2474539279937744, -0.6688761115074158, -3.3603742122650146, -0.20955374836921692, -1.1792255640029907, 1.8807189464569092, -0.047561291605234146, 1.383178949356079], [0.5465697646141052, -1.137089729309082, 0.49225422739982605, -0.5299551486968994, 0.32562053203582764, -2.048509120941162, 0.03799877315759659, -0.46653175354003906, 1.3602670431137085, -0.1974623203277588, -0.4316108524799347, 0.46852636337280273, -1.070572853088379, 1.1395974159240723, -0.5583735704421997, -0.10671043395996094, 0.7895920872688293, -0.2063743770122528, -0.20505401492118835, -0.32334786653518677, 0.14568151533603668, -0.572020411491394, -0.7715508341789246, 0.09495769441127777, 0.6961451172828674, -1.2165300846099854, 0.8069443702697754, 0.35858649015426636, -0.1250428706407547, 0.6595848202705383, 0.5730260610580444, 0.583447277545929, 1.5268791913986206, -0.03558201342821121, 0.07626855373382568, -2.188646078109741, 1.8906981945037842, -0.6874343156814575, 0.7938628792762756, -0.6513660550117493, -1.180342674255371, -0.28406423330307007, -0.4500158429145813, 1.3514243364334106, 0.6468241214752197, 0.07406904548406601, -0.11065544188022614, 0.6151583790779114, -0.6075339913368225, 0.43105724453926086, 1.347969651222229, -0.7436754703521729, 0.6997789144515991, -0.5033928155899048, -1.1207622289657593, -0.24380838871002197, 0.6096258759498596, 0.9350438714027405, -0.2738526463508606, 0.23456230759620667, 0.9000058770179749, -0.2283635437488556, 0.5388884544372559, 0.5786246061325073], [-1.2370936870574951, 0.575011134147644, -0.7287523150444031, -0.5231384038925171, -0.40852001309394836, 1.6487746238708496, -0.8851330280303955, 1.4397169351577759, -2.2982895374298096, -1.5332766771316528, -1.6496477127075195, -0.7047668695449829, -0.1330796629190445, -0.47890666127204895, 0.0689154863357544, 0.7956409454345703, -0.29439204931259155, -0.46126729249954224, 0.5421400666236877, -2.257571220397949, -0.6919622421264648, 1.5786738395690918, 1.7807340621948242, 0.24516306817531586, 0.4542742669582367, 0.3116445243358612, -0.5440704226493835, -0.19076022505760193, 1.523571491241455, -0.8879560828208923, -1.4461934566497803, 2.363508939743042, 1.2007391452789307, 2.1408817768096924, 0.15663421154022217, -0.13383711874485016, 0.9600292444229126, -0.7324337363243103, -1.2961862087249756, 0.008281310088932514, -0.691695511341095, -0.6883054375648499, 0.33784252405166626, -0.2942768931388855, -0.9270399212837219, -0.1848772019147873, -0.9386508464813232, 0.20859724283218384, 0.165914386510849, -0.7433449029922485, -1.2234535217285156, -1.5223205089569092, -0.34206080436706543, -0.15168525278568268, 0.37767502665519714, 0.3654419481754303, 0.3978510797023773, 0.7833568453788757, -1.9140923023223877, -2.4915382862091064, 1.1758097410202026, -0.0826018676161766, 1.279914140701294, -0.14334233105182648], [1.066376805305481, -2.0704562664031982, -1.1838537454605103, 1.1079494953155518, -0.0608203150331974, 0.26039156317710876, -0.8411173820495605, -0.07058295607566833, -0.44498252868652344, 0.38338765501976013, 0.16537795960903168, 1.449516773223877, 0.5136247277259827, -2.8006250858306885, 1.266814947128296, -0.1474982500076294, -1.0778112411499023, 0.4226544201374054, 0.9733863472938538, 1.053412914276123, 1.518525242805481, -0.038862086832523346, 2.8014748096466064, 1.6618640422821045, 0.18523366749286652, 0.22582127153873444, 0.4962223172187805, -1.1082221269607544, 2.5233607292175293, -0.8351536393165588, 2.1629507541656494, -1.0636436939239502, -2.32297945022583, 0.42556047439575195, -0.4905063509941101, 1.8608691692352295, 0.39924901723861694, -1.5666885375976562, -0.20124368369579315, 0.3209848701953888, 0.349188894033432, -0.06748588383197784, -0.6634464859962463, -0.9849941730499268, 0.5693144202232361, 0.5546875, -1.0397838354110718, 0.5070149302482605, 0.2871384620666504, -0.3298512399196625, -1.0477935075759888, -1.1879777908325195, 1.7741693258285522, 0.6977913975715637, -0.947014570236206, -0.5775466561317444, -0.4986828565597534, -0.9458605647087097, -0.2400059849023819, 0.01800432987511158, 0.07091805338859558, 2.027242422103882, -1.0018508434295654, 3.3707103729248047], [-1.7367558479309082, -0.23662027716636658, -0.8694736957550049, -1.1218606233596802, 0.5691784024238586, -1.0147944688796997, -1.0095168352127075, -1.622063398361206, -1.2473595142364502, -0.8506857752799988, -0.6178463697433472, -1.2594194412231445, -0.4843180775642395, 1.4254150390625, -0.005981266498565674, -0.22943276166915894, -1.5676058530807495, 0.8799077272415161, -1.0167990922927856, -0.8685080409049988, 0.33924105763435364, 0.5316929817199707, 0.04030558094382286, -0.5445590615272522, 0.3479527533054352, -0.1999480277299881, 0.6319544315338135, -0.8316119313240051, 1.1650093793869019, 1.318249225616455, -1.4826964139938354, -0.20750416815280914, -0.6505548357963562, 1.4565720558166504, 2.6436259746551514, -0.02616705931723118, 0.5921640992164612, -0.8107766509056091, 0.45435458421707153, 0.48133087158203125, -0.3101961612701416, -1.3689970970153809, 1.2533930540084839, 0.10972815006971359, 0.5272058844566345, 0.7231606841087341, -1.2311283349990845, 1.0428842306137085, 1.072709321975708, 1.7102488279342651, 0.19437207281589508, -0.24061734974384308, -0.041010547429323196, 0.630932092666626, -0.37172484397888184, -1.8801333904266357, -0.44868454337120056, 0.2714976668357849, 1.492276906967163, 0.15932893753051758, -0.717170774936676, -0.263488233089447, 0.15351933240890503, 0.41841644048690796], [-0.29695600271224976, 0.7988054156303406, 0.5135689377784729, 0.5874201655387878, -0.823798418045044, -0.2798752188682556, -0.5212326049804688, 0.5533875823020935, 0.2131120264530182, 1.274986982345581, 0.20519250631332397, -0.2314521074295044, 1.3680775165557861, -0.7487561106681824, 0.8916032910346985, -0.1472834199666977, -1.235011339187622, -1.4140260219573975, 0.42014142870903015, 0.8814316391944885, 1.3898760080337524, 0.4560760259628296, 1.1949211359024048, 0.5409116148948669, 0.751817524433136, 0.1478217989206314, -1.2983156442642212, 1.582228422164917, 0.4005346894264221, 0.18741992115974426, 0.7149266600608826, -0.46194058656692505, -0.21933022141456604, -0.9721445441246033, 0.20541167259216309, -0.32103395462036133, -1.430316686630249, 1.4801640510559082, -1.027588129043579, -0.6383963227272034, 0.007696489803493023, -0.3883376121520996, 0.8929221034049988, 1.017500877380371, 0.41763952374458313, -0.3965272903442383, -1.1204899549484253, -1.1083616018295288, 0.024355411529541016, 0.6204654574394226, -0.5809938907623291, -0.6535853147506714, -1.8974578380584717, -1.5387749671936035, 2.132167100906372, -0.785754382610321, 0.16799084842205048, 0.7754368782043457, 1.0840649604797363, 1.0709820985794067, 0.9774339199066162, -0.3599354028701782, 0.1538999229669571, -0.7452326416969299], [0.17857296764850616, 0.4564436674118042, -0.9710999727249146, 0.6058893203735352, 0.2945839464664459, -2.970165252685547, -0.23274318873882294, -1.8379244804382324, -1.7185922861099243, -0.07769764214754105, 0.6425272226333618, -1.9071450233459473, 0.859627366065979, 0.34108230471611023, -1.4245436191558838, -0.01994853839278221, 0.051336199045181274, 0.1736694872379303, 1.3523650169372559, -0.14389464259147644, 0.7897626757621765, 0.4509987533092499, -1.6573628187179565, 0.5267534255981445, -0.4321070611476898, 2.0859150886535645, -1.663769006729126, -1.5923056602478027, -1.987203598022461, -0.7691002488136292, 1.0509847402572632, -0.06061103567481041, -0.3989865481853485, -0.6185393333435059, 0.8540721535682678, -0.6972431540489197, -1.5003814697265625, 1.3559657335281372, -1.351997971534729, -0.2198459357023239, 1.0748199224472046, -1.7835382223129272, -1.8329026699066162, 2.778557538986206, -0.5312157869338989, 0.49257686734199524, -0.6913623213768005, -1.0786287784576416, 0.6880255341529846, -1.1179784536361694, 0.27266091108322144, 0.13905490934848785, -0.1537732630968094, 0.4491030275821686, -0.3360934853553772, -0.3155907690525055, -0.23192676901817322, -1.1616395711898804, -0.6728029251098633, 0.7051791548728943, -0.6752415299415588, 1.140262246131897, 0.48398277163505554, -0.15294621884822845], [1.5529488325119019, -0.9253983497619629, 1.7698065042495728, -1.7759320735931396, -0.07516676932573318, -0.16465960443019867, -0.30723780393600464, 1.2049423456192017, 0.03500944375991821, -1.0972168445587158, 1.4734535217285156, -0.5739838480949402, 0.6452459096908569, -0.7776419520378113, 0.7280372381210327, 0.21934717893600464, 0.9456629753112793, -0.5966982245445251, 0.3071727752685547, 2.1282155513763428, -0.28820812702178955, 0.05508539453148842, -1.230515718460083, -0.5747065544128418, 0.5819873809814453, 0.4004548192024231, -1.600464105606079, 0.33039799332618713, -0.7208067774772644, 1.498284101486206, 0.728325605392456, -0.6281037926673889, -0.1288161277770996, -1.3404213190078735, -1.9751583337783813, 0.8972629904747009, 0.9212237000465393, -0.41980549693107605, -0.4787452518939972, 1.1181575059890747, -1.365970492362976, -0.7171493172645569, 2.7186453342437744, 0.9575332403182983, -1.4664421081542969, 0.5580446124076843, -1.3464652299880981, -0.5828923583030701, -0.6347380876541138, -1.5864317417144775, -0.8896220326423645, 0.04315372556447983, -0.19721736013889313, -0.8412910103797913, -0.5959481596946716, 0.7981123328208923, -2.529526710510254, -0.44803711771965027, -0.04420707747340202, 0.4716286361217499, -1.56999671459198, -0.2882259786128998, -0.15090754628181458, 1.197930932044983], [-0.8095148801803589, 0.3573164939880371, -1.462467908859253, -2.2208938598632812, 0.28846412897109985, -0.7953085899353027, 1.4030150175094604, 0.1837613731622696, 0.5992822051048279, -0.9146637916564941, -0.7955540418624878, -0.2509218752384186, -1.2730759382247925, -0.8086388111114502, 0.8227060437202454, -0.09543731063604355, -0.4314264953136444, 0.3041805326938629, 0.3450852930545807, -0.0245656780898571, 0.45750465989112854, 1.60841965675354, 0.5458528995513916, -0.2501521408557892, -1.0492403507232666, 0.7030354738235474, -0.7347855567932129, 1.8344621658325195, -0.9370955228805542, -0.03523978590965271, -0.4105793535709381, -0.447910875082016, 0.32307717204093933, -0.24316175282001495, 1.648089051246643, -1.204368233680725, 1.1904507875442505, 0.25256556272506714, -0.013429329730570316, 0.15462180972099304, -0.711694598197937, -0.44433560967445374, -0.8159605264663696, 0.6283082962036133, -1.3778438568115234, 1.351399302482605, -0.897826075553894, -2.3207764625549316, -0.5844807624816895, 0.4813418984413147, -0.6016462445259094, -0.2615721821784973, -0.8038156628608704, -0.7871189117431641, 0.012508468702435493, -0.7958553433418274, 0.8838227391242981, 1.1692745685577393, 1.8659828901290894, -2.6519062519073486, -2.202908515930176, -0.30523329973220825, 1.2944533824920654, 1.0121886730194092], [-0.32095852494239807, -1.6255689859390259, 0.17250114679336548, -0.7104799747467041, -0.7991889715194702, 2.417587995529175, -1.2873047590255737, -1.8603503704071045, -0.19501996040344238, -0.09803470969200134, -0.09144808351993561, 0.854487419128418, 0.10436040908098221, 1.032448649406433, 0.32715293765068054, 0.055580876767635345, 0.4440949857234955, -1.7041385173797607, -1.6977262496948242, -0.5720036625862122, -1.8893568515777588, -0.18220645189285278, -1.4433104991912842, -0.6376425623893738, 0.6116742491722107, 1.528430700302124, 0.47882840037345886, 0.11352567374706268, -1.9224340915679932, 1.5875617265701294, 0.40829935669898987, -0.5629714131355286, 0.9451632499694824, 0.7915646433830261, -0.794975221157074, -1.8160043954849243, -0.6130518913269043, -0.15916848182678223, 1.0139317512512207, -0.7106021642684937, 0.585350513458252, -0.48666855692863464, -0.4176170229911804, 0.030644094571471214, 0.4761630594730377, 1.6731202602386475, -0.25828787684440613, 0.6917946338653564, 0.010250785388052464, 0.8620525598526001, 0.88663649559021, 0.4389054477214813, 0.3646552562713623, 0.24473917484283447, 1.310931921005249, 0.31243276596069336, 0.655719518661499, 1.011155366897583, -0.3947664499282837, 0.40882608294487, 2.1912107467651367, -0.4422031044960022, 1.106066346168518, 0.7446723580360413], [-0.13360752165317535, -0.05685325711965561, 0.5972168445587158, 0.7720091938972473, -0.9943912625312805, 0.292359322309494, 0.16724893450737, 0.9029004573822021, 0.2353508323431015, 2.0328824520111084, -0.019905414432287216, 2.4547245502471924, -0.061532218009233475, 0.5358840227127075, -0.12133169174194336, -0.6810283660888672, -1.6442281007766724, -1.435593843460083, 0.5780579447746277, -0.7310569882392883, 0.4205220341682434, -1.4260011911392212, 1.6496485471725464, 1.6317603588104248, 0.6968545317649841, -0.7534933686256409, 0.12493059039115906, 0.393312931060791, 1.335427165031433, 2.419312000274658, -1.8913198709487915, 0.5709818005561829, -1.3079423904418945, -0.11603137105703354, -0.20782873034477234, -0.8642544150352478, -1.1006501913070679, -0.7999312281608582, 2.430331230163574, -1.8449883460998535, 1.8824326992034912, 0.8863999843597412, 1.572619080543518, -0.021324656903743744, 0.45305585861206055, 0.37845325469970703, 0.15722963213920593, -0.056131888180971146, 0.27582091093063354, -0.8260664939880371, -0.3559795916080475, -1.3709038496017456, -0.6838638186454773, -1.2007088661193848, 1.0342023372650146, 0.12425700575113297, -0.4342650771141052, -1.1879321336746216, 0.08859772235155106, -2.1863555908203125, 0.46295592188835144, 0.2563091218471527, -0.8951283097267151, -1.0127160549163818], [1.6713860034942627, -1.1809754371643066, -0.6471982002258301, 2.8260369300842285, -0.39643508195877075, 2.563065528869629, -0.6331807971000671, -1.7479084730148315, 1.0844999551773071, -0.03458448871970177, 1.579530119895935, 0.8237033486366272, 1.7118353843688965, 0.8387899398803711, -0.7060061097145081, -0.22101734578609467, 1.41938316822052, -0.04835657402873039, -0.27731725573539734, -0.5942651033401489, -0.8865119218826294, -0.3330479562282562, -0.6575676202774048, -0.7690439224243164, -0.6691717505455017, 0.02430536225438118, 0.9566042423248291, 0.26288244128227234, -0.8256648182868958, -1.2070404291152954, 1.6471397876739502, 0.9762275218963623, 0.7336543798446655, 0.6225385069847107, 0.030869517475366592, 0.8094592094421387, 0.04652385413646698, -0.1109713539481163, -0.6898438334465027, -0.2439887672662735, -0.42125141620635986, -0.6709691882133484, 0.36253735423088074, 0.7858015298843384, 1.0733704566955566, 0.11213335394859314, -0.12380512803792953, -0.5957111716270447, -0.9984124302864075, -1.333154559135437, -0.8580363988876343, -0.4360155165195465, 1.1315540075302124, -0.3912014961242676, -1.0326191186904907, -0.8855122327804565, 0.14505843818187714, 0.8548082709312439, 0.36839354038238525, 0.12988172471523285, -0.4825378656387329, 0.7112765312194824, 0.41490045189857483, 0.18773120641708374], [0.3207043409347534, -1.1279574632644653, -0.23651869595050812, -0.6778494715690613, -0.2136365920305252, 1.0568313598632812, 1.4930241107940674, 1.0063035488128662, -0.2978646159172058, 0.051759231835603714, -1.5364958047866821, 0.9123384952545166, -1.6106411218643188, 0.3502085506916046, 0.15592268109321594, 0.4998950660228729, 1.3526124954223633, 1.9029836654663086, -2.2175567150115967, 1.293393850326538, -1.0248750448226929, -2.2450034618377686, 1.1012611389160156, -0.8271129131317139, 0.9231183528900146, 0.5883679986000061, 0.23035746812820435, 0.8098241686820984, -0.025223901495337486, 2.5448029041290283, -0.6604154706001282, 0.3917524516582489, 0.5970328450202942, -0.38920119404792786, 0.387016236782074, -2.210226535797119, -0.46932247281074524, 0.8507160544395447, 0.42223626375198364, -0.4909501075744629, 1.4082131385803223, -0.34312203526496887, 0.10824505239725113, 0.6686972975730896, 0.232025608420372, 0.7812273502349854, -1.0467958450317383, -1.6180499792099, -0.5747005343437195, 1.090307354927063, 1.384132742881775, 1.6907938718795776, -0.4237222671508789, 0.40048834681510925, -0.1578187197446823, 0.20642387866973877, -0.2249746322631836, 1.218246579170227, -0.12265283614397049, -0.8166540861129761, -1.1146137714385986, 0.5771986842155457, -1.0113940238952637, -0.3967589735984802], [0.4179064929485321, -1.867862343788147, -0.05703822150826454, -0.5256434679031372, 0.9697426557540894, 0.7696133255958557, -1.9362865686416626, 0.006058777216821909, 1.3031046390533447, 0.35847142338752747, 0.15680480003356934, 0.8488202691078186, -1.0448598861694336, 0.3286083936691284, 1.5234169960021973, -0.10175073891878128, 1.8806147575378418, -0.23288550972938538, -0.40469467639923096, 1.0126432180404663, 0.05939536169171333, 2.7864129543304443, 0.9984673261642456, 1.1240378618240356, -1.6515371799468994, 0.5373993515968323, 0.15975823998451233, 0.5858271718025208, 0.44831356406211853, 1.66183340549469, 1.4949629306793213, 0.361696720123291, -0.3643353581428528, -0.8280889391899109, 1.2517039775848389, 0.4393516778945923, -0.45382851362228394, 1.2975661754608154, -0.8732494115829468, 0.013655521906912327, 0.4681474566459656, -1.6047054529190063, -0.06480896472930908, 0.5599809885025024, -1.4981751441955566, -2.5766489505767822, -0.1876220554113388, 1.4871152639389038, -2.9315922260284424, -0.8148577809333801, -1.1249080896377563, 0.49407365918159485, -0.6717941164970398, -0.3811206519603729, 0.18853148818016052, -0.05907771363854408, -1.3450597524642944, 1.2179652452468872, 1.0138126611709595, -0.21680191159248352, 0.05716187134385109, -1.9301055669784546, -0.25784969329833984, 0.6269069910049438], [0.1535036861896515, -0.5244001150131226, 1.0317153930664062, -0.35470497608184814, -0.2993340492248535, 0.9575259685516357, -0.34917327761650085, 0.3584648072719574, -1.1643750667572021, 0.16585151851177216, -0.2293367087841034, -0.16893818974494934, -0.01151224598288536, 0.7531992793083191, -0.2879783809185028, 0.9128216505050659, 2.549445867538452, 0.01614130102097988, 0.19686761498451233, 0.5373396277427673, -0.32042670249938965, -1.4114656448364258, -0.8110475540161133, -0.7739713788032532, -0.8474761843681335, 1.6726113557815552, 0.41227760910987854, 0.6044064164161682, 0.29988881945610046, -0.17904944717884064, 2.0681395530700684, 0.25179821252822876, 1.7176498174667358, 0.7125656604766846, 0.4403105676174164, -0.5705955028533936, 0.24025435745716095, -0.06576474010944366, 0.3567968010902405, -1.196016788482666, 0.20192058384418488, -0.2659393846988678, 1.3818273544311523, 0.518867015838623, 0.8535693883895874, 1.0160248279571533, -0.19000007212162018, -0.5268358588218689, 0.5792979001998901, -0.5244265794754028, -0.14470404386520386, 0.39709746837615967, 0.6079323291778564, -1.1884398460388184, 1.100160837173462, 0.4504683017730713, -0.13025878369808197, -1.1060189008712769, -0.9269795417785645, -1.3503592014312744, -0.28433889150619507, 3.041618824005127, 0.8131256699562073, -0.25256818532943726], [1.1764068603515625, -1.128186821937561, 0.23887993395328522, -0.4172763228416443, -0.5363982915878296, 0.9088741540908813, 0.008220608346164227, -0.43167585134506226, -1.0683234930038452, 0.11247564852237701, 0.19706594944000244, 0.9906117916107178, -0.13566361367702484, 1.7796986103057861, -0.7176546454429626, 1.332140326499939, -2.5424108505249023, -0.29110345244407654, 1.367418646812439, -0.6977885365486145, 0.08248700946569443, -1.0475317239761353, -0.47207072377204895, 2.4329464435577393, -0.18515942990779877, 0.1434197872877121, -0.23591260612010956, 0.7148602604866028, -0.8059489130973816, 0.2675263285636902, -0.974941074848175, 0.9667526483535767, 0.8096144795417786, 0.37599456310272217, 1.1998107433319092, -0.03314906358718872, 0.007161327637732029, 0.42763781547546387, -1.5790823698043823, -0.02874184213578701, 0.2220137119293213, -0.7499131560325623, 0.41260451078414917, 0.9977807402610779, 0.6437070965766907, -0.6924907565116882, 0.08370981365442276, -1.2564233541488647, 0.4042339026927948, -0.13947615027427673, 0.18113866448402405, 2.2343244552612305, -0.23071864247322083, 0.42064788937568665, -2.041722059249878, -0.21287478506565094, -1.070767879486084, 0.15487952530384064, -0.491630494594574, -0.21382944285869598, 1.7031844854354858, 1.4785689115524292, 0.8773705959320068, 0.20368728041648865], [1.459252953529358, 0.9594060778617859, -0.8318894505500793, -0.7933963537216187, 0.014367143623530865, 0.1373152881860733, 0.3579334020614624, 1.2189196348190308, -1.150463342666626, -0.7033398747444153, 1.0266258716583252, 1.9949594736099243, 1.8068913221359253, 1.1151996850967407, 0.11718937754631042, -0.2712048590183258, -0.5809480547904968, 0.6608670949935913, 0.6210887432098389, 0.6325841546058655, -0.1561882644891739, 0.6942659616470337, 1.3963385820388794, 0.3437719941139221, 0.3271747827529907, 1.6707396507263184, -1.8460769653320312, -0.6843702793121338, -0.7208771109580994, 0.5413106083869934, 1.7541476488113403, -2.504462480545044, 1.12378990650177, -1.991450309753418, 0.8549008369445801, 0.04503511264920235, -0.06551005691289902, 2.5873615741729736, -1.2026424407958984, 0.39911627769470215, 0.0941317155957222, 1.3070785999298096, 0.7947197556495667, -0.6709235906600952, -1.3415186405181885, 1.4994419813156128, -1.8981530666351318, -0.550800621509552, -0.14675454795360565, -1.7316187620162964, 0.6135857701301575, 0.5742661952972412, -1.0742450952529907, -0.7832249999046326, -2.734890937805176, 0.30384233593940735, 2.1546497344970703, 0.1209862232208252, -0.04076196253299713, 0.5839212536811829, -1.4267228841781616, -0.6071045994758606, -0.388931006193161, 1.4151362180709839], [-1.3627641201019287, -0.6720439791679382, 0.5990293025970459, 0.26406633853912354, -1.8762454986572266, -0.3430440127849579, 0.6355305314064026, -0.7579640746116638, 0.2058330774307251, -1.1151992082595825, 0.03474660962820053, 0.5771248936653137, -0.2663734555244446, -1.0504093170166016, -0.39197203516960144, -0.6140419244766235, -1.6358239650726318, 0.1958445906639099, 1.800601601600647, 0.9122610688209534, 0.9851182699203491, 0.38253509998321533, -1.3592358827590942, -0.6354668140411377, -1.334029197692871, -0.7391136884689331, -1.2203917503356934, 1.7356700897216797, -0.24220265448093414, 1.3951624631881714, -0.7054555416107178, -0.37445083260536194, -1.057172179222107, 1.7052741050720215, 1.0353624820709229, -0.4260514974594116, 0.1593044251203537, -1.1576656103134155, -0.3819456994533539, -0.42417851090431213, 0.8158102631568909, -0.5361652970314026, 0.5921016931533813, 0.8099457025527954, -1.0098662376403809, 0.65488201379776, 0.7473362684249878, -0.07897777110338211, -0.7304158210754395, -0.25488102436065674, 0.7306855916976929, 0.7906107902526855, 0.05344472452998161, 0.18101096153259277, 0.17987845838069916, -0.4872247278690338, 1.2282651662826538, -0.3698543310165405, -0.8830918669700623, -0.5324918627738953, -0.4188120365142822, -0.12405899912118912, -0.7482366561889648, 0.4072718024253845], [-0.23294870555400848, -2.3369016647338867, 0.6269885897636414, -0.15464508533477783, 1.0441038608551025, -0.2562691569328308, 0.501711905002594, -0.317546010017395, -2.586008071899414, -0.3816041052341461, 1.6646614074707031, -0.521776020526886, 0.7026563286781311, -0.4288173019886017, 0.5506111979484558, 2.6051132678985596, 0.33379417657852173, -1.4955865144729614, 1.1797939538955688, 1.0495991706848145, -0.22830775380134583, 0.056958384811878204, -0.18950185179710388, -0.250120609998703, 0.6681492328643799, 1.2797905206680298, -1.2162837982177734, -0.7802640199661255, -0.20673173666000366, -0.7394206523895264, 0.8136209845542908, -0.18748775124549866, 0.18345953524112701, 1.9909018278121948, 0.49525752663612366, -1.173639178276062, -1.5239328145980835, 1.4469919204711914, 0.8702734708786011, -0.43812263011932373, 0.5646520853042603, -0.15998981893062592, -0.26819944381713867, 0.47825637459754944, -0.5657416582107544, -1.4943902492523193, 0.24914798140525818, -0.35281074047088623, -0.7578474879264832, -1.1471562385559082, -0.16270992159843445, -0.48120227456092834, 2.1497952938079834, -1.325579047203064, -0.9478158950805664, 0.2212432324886322, 3.095677614212036, -0.19503284990787506, 0.1449054330587387, -1.7236461639404297, -0.6526270508766174, -1.3118380308151245, -0.8547049164772034, -0.3293043375015259], [0.025851884856820107, -0.5130926370620728, 0.2980440855026245, -1.3365817070007324, 1.2583500146865845, -1.044524073600769, -0.8040090203285217, 0.042451076209545135, 0.5984050035476685, 0.34718501567840576, 0.027438469231128693, 3.277221441268921, 0.9560854434967041, -0.1328483670949936, -0.010401571169495583, 0.5838653445243835, -2.260904312133789, 0.11481228470802307, -2.2007811069488525, -0.9037875533103943, 0.3944060504436493, -0.9917799830436707, 1.1693750619888306, 1.6617907285690308, 0.07688021659851074, 0.2593720853328705, 0.6608116030693054, 0.4588015079498291, -0.8115717172622681, -0.2568308413028717, 0.9068431854248047, 0.19156482815742493, 0.5092130303382874, -1.0383118391036987, 0.3554699718952179, -0.389819860458374, 1.2844568490982056, -1.3076398372650146, 1.2263494729995728, -0.8227824568748474, 0.21284444630146027, 1.2499053478240967, 0.28813886642456055, -0.434491902589798, 0.5901608467102051, 1.665421724319458, -0.069933220744133, 1.1620477437973022, 1.9626197814941406, 0.6975309252738953, -1.3630542755126953, -1.0649960041046143, 1.2158268690109253, -0.2939613163471222, -0.8402816653251648, -0.7732163071632385, 0.20071439445018768, 1.0511014461517334, 0.06537923961877823, -0.923764705657959, -0.0669129267334938, -1.426625370979309, -0.04664638638496399, -0.28671297430992126], [-0.1610831767320633, -1.5293290615081787, -1.6634490489959717, 0.8761926889419556, 0.007376035675406456, 1.71419095993042, 0.05274689942598343, 0.6860668659210205, 1.0282739400863647, -0.4174596667289734, 0.5039734840393066, -0.38797324895858765, 0.03166288882493973, -0.5446835160255432, 0.9259328842163086, -1.4565004110336304, -0.5948640704154968, -0.1746935248374939, -1.082080364227295, -0.6622955203056335, -1.2054576873779297, -0.006250453181564808, 0.3507138788700104, -1.349205493927002, -0.006957752630114555, -0.6530828475952148, 0.8700154423713684, 0.5625391006469727, 0.2507643401622772, -0.7194854617118835, 0.2231452614068985, 0.12921276688575745, -1.2892100811004639, -0.5987897515296936, 0.3272407054901123, -1.219787836074829, -0.5034662485122681, 1.0792577266693115, 1.7160578966140747, -0.3113808035850525, -1.8697702884674072, 0.01735452003777027, 0.5860400199890137, 0.3449019491672516, 0.19956710934638977, -0.2852308452129364, 0.9899312853813171, 1.9532485008239746, 0.9731154441833496, 1.2336899042129517, -0.1368424892425537, 0.27921798825263977, 1.9663317203521729, 0.20310373604297638, 0.5383769273757935, 0.7993382215499878, 0.5481612086296082, 0.7845102548599243, 2.3019208908081055, -1.3711894750595093, 1.1230415105819702, 1.0086688995361328, 2.0843842029571533, 0.8940206170082092], [0.49262872338294983, 1.1923457384109497, 0.5105841755867004, -0.43210166692733765, 0.6282193064689636, -0.3580818474292755, -1.003801941871643, 1.385481595993042, 1.1316686868667603, 0.46240735054016113, 0.18387609720230103, -0.6783574223518372, -0.030150718986988068, 0.5645348429679871, -1.442503809928894, -0.8730551600456238, 1.1282800436019897, -0.045059096068143845, 0.1087261363863945, -0.23710350692272186, 1.5405304431915283, -0.09450069069862366, -0.6049054861068726, 1.6621887683868408, -2.2923922538757324, -0.44180166721343994, 0.6901525259017944, 0.5905964374542236, -0.40022921562194824, -2.3767802715301514, 0.553996205329895, -1.097917914390564, 0.19989660382270813, -0.33436766266822815, -0.08393669128417969, 1.0429370403289795, -1.5263166427612305, 1.6450364589691162, 0.8172888159751892, -1.7649238109588623, -0.6345736980438232, 0.728541910648346, -0.14234787225723267, -0.18158450722694397, -0.8395689725875854, 1.6358803510665894, -1.9514464139938354, -0.5516399145126343, 1.6760289669036865, 0.7841839790344238, -1.2483488321304321, 0.13904473185539246, 0.7196425199508667, 0.35378363728523254, -1.0885576009750366, 0.1277368813753128, -1.9115891456604004, -1.4122494459152222, -0.3822811543941498, -0.16862516105175018, -0.2783603370189667, 0.2934350371360779, 0.9156795740127563, -1.2708489894866943], [0.26140865683555603, 1.9643075466156006, -1.2032666206359863, -1.4818967580795288, -2.0438942909240723, -0.5299290418624878, -1.0663273334503174, 0.03977835550904274, 1.0284919738769531, -0.00905818585306406, -0.4376290738582611, -1.862233281135559, 0.39175400137901306, -1.056944727897644, 0.01609460450708866, 0.9548142552375793, -0.920415997505188, -0.7643380761146545, -0.6002958416938782, -0.11020267754793167, -0.8240562081336975, -2.213439464569092, 0.24328118562698364, 1.1811609268188477, -1.0569053888320923, 0.09369180351495743, 0.3656449317932129, 0.40294015407562256, 1.1232002973556519, -0.3214002847671509, 0.7368606925010681, 0.7238489985466003, -1.547739863395691, 0.4886891543865204, 0.08633606880903244, 0.7016929388046265, 1.5466063022613525, 0.029727650806307793, -2.2766671180725098, -0.5469000935554504, -0.8398984670639038, 2.0183799266815186, 0.8174607753753662, -0.4990832805633545, -0.5097984075546265, 0.4141654074192047, -0.19211776554584503, 1.5251774787902832, -0.004946267697960138, 0.5272394418716431, -1.56440269947052, 0.7076407074928284, -0.8418136239051819, -1.1595168113708496, -1.30933678150177, -0.8688982129096985, 0.5160943269729614, -1.592832088470459, -0.13542208075523376, -0.033049874007701874, -2.457393169403076, -0.8728991746902466, 0.9745274782180786, -0.1634645313024521], [-0.9232924580574036, 1.1266188621520996, 0.23086968064308167, 0.18879380822181702, -2.1021366119384766, -0.16433966159820557, 1.7818586826324463, -2.102834701538086, 0.7285051345825195, 1.4811688661575317, 0.0900622233748436, -0.7986087799072266, 0.24434231221675873, 2.0718095302581787, 0.9586799144744873, 0.6310733556747437, 0.4452206790447235, -0.4175925552845001, 0.33338791131973267, -0.8606146574020386, 0.36591383814811707, -1.450178623199463, 0.4547259509563446, 0.8248738050460815, 1.017525315284729, 1.759552001953125, 1.1370856761932373, -1.191982388496399, 1.0639656782150269, -0.21687288582324982, -0.20895308256149292, 0.4389629364013672, 0.05537561699748039, -1.357993483543396, -0.6238288283348083, 0.7883359789848328, 1.0178098678588867, 0.05745313689112663, -0.7602746486663818, 0.4378640651702881, -0.609843909740448, 0.1399553120136261, -0.8086043000221252, 0.47559407353401184, 0.4186386466026306, -0.6804068088531494, -0.3210844397544861, -0.043005626648664474, 0.12470009177923203, 2.493144989013672, -0.9265140891075134, 0.3886840343475342, 0.8332622051239014, -0.752185046672821, -0.7674623131752014, -0.2141781449317932, 1.7501693964004517, 0.06433078646659851, 1.3240598440170288, -0.10538922250270844, 0.06924259662628174, -0.6423800587654114, -0.7597391605377197, 2.021315813064575], [-0.472278356552124, 0.9210587739944458, -0.10490042716264725, 1.3876028060913086, -0.8146157264709473, 0.7798672914505005, 0.4143441915512085, -0.49871397018432617, 1.3692805767059326, 0.11356895416975021, 0.2509410083293915, 0.06176023557782173, 1.0245530605316162, 0.7631941437721252, -0.6087163090705872, -1.0820043087005615, -1.1068758964538574, -1.0510529279708862, 0.038067419081926346, 0.6226565837860107, -0.8878175020217896, 1.3584853410720825, -1.1044089794158936, 0.4199381470680237, 0.5431815981864929, -0.5445594787597656, 1.5295087099075317, -0.4776451587677002, 0.1557544320821762, 0.7186080813407898, 0.9120326638221741, -1.0427284240722656, -0.3084552586078644, -2.433398723602295, -0.3579578399658203, -0.8795235753059387, 0.32005783915519714, 1.6885260343551636, -0.2647489011287689, -0.387495756149292, -0.2605229318141937, -1.1301227807998657, -1.3189929723739624, 2.073637008666992, -0.007777446880936623, -1.0900648832321167, 0.07934873551130295, -2.147698163986206, -1.6244014501571655, -2.015496015548706, 0.23075270652770996, 0.05697610229253769, -1.3153804540634155, 1.5703771114349365, -0.21178124845027924, 1.1665345430374146, 1.5807496309280396, -0.07773891091346741, -0.8440236449241638, 2.582254409790039, -0.4798918664455414, -0.3315688967704773, -0.04108862206339836, 1.1663391590118408], [0.31934523582458496, -1.0848870277404785, -0.24856504797935486, 0.030134093016386032, -0.04336227849125862, -0.5892004370689392, 2.058056116104126, -0.9845559597015381, 0.364021897315979, 0.003735001664608717, 2.616274833679199, 0.1479412019252777, 0.42133793234825134, -0.768445611000061, 0.9665828943252563, -1.6248289346694946, -0.2453363984823227, 1.6180927753448486, -0.401513934135437, -1.1860228776931763, -0.2495875209569931, 0.8620966076850891, 0.1273026466369629, -0.42700085043907166, -0.30924734473228455, -0.9764204025268555, 0.4675157964229584, -0.5805388689041138, 0.2872184216976166, 0.8083944320678711, 0.8503120541572571, -0.025075599551200867, -0.7801860570907593, -1.2054963111877441, 0.6184399127960205, 1.5260705947875977, -0.9131954908370972, 0.21015791594982147, -0.49272990226745605, -0.20867504179477692, -0.04309474304318428, -0.10760588198900223, -0.449668824672699, 0.4318206310272217, -0.9806903600692749, -1.8974823951721191, 0.8925175070762634, -1.2982518672943115, -0.4168470501899719, 0.3526338040828705, -1.9430856704711914, -0.671495258808136, -0.042610276490449905, 0.3664422929286957, 1.3504128456115723, -0.7692698240280151, -0.5427442193031311, 0.9864699244499207, 0.16560572385787964, -1.4814125299453735, -1.5276378393173218, 1.3180207014083862, 0.5966073870658875, -0.43016889691352844], [-0.45967423915863037, -0.3125796318054199, 1.8161062002182007, -0.8050181269645691, -1.6531591415405273, -1.5353761911392212, 0.9208271503448486, 0.21687205135822296, 0.5261533856391907, 0.7388537526130676, 0.6362599730491638, -1.0837078094482422, 0.1376955360174179, 0.18329812586307526, 0.8703315258026123, -0.9988164305686951, -0.12921860814094543, -1.3927900791168213, -0.0940905287861824, -0.5806189775466919, -0.3077491521835327, -0.06280539184808731, 1.5802093744277954, 0.9852457642555237, -0.024296976625919342, 0.9452502727508545, 0.6921831965446472, -0.5647523999214172, 1.2320780754089355, 0.32773521542549133, 1.7613942623138428, 0.07726074755191803, -2.132930278778076, 0.2943436801433563, -0.2086576670408249, -2.2714967727661133, 1.1188373565673828, -0.19705140590667725, -1.0162686109542847, -1.058417558670044, 0.5397413372993469, 1.3187851905822754, 0.016630085185170174, 0.0034257511142641306, -1.0537699460983276, 1.3080936670303345, -1.1894659996032715, 0.0033996577840298414, 1.5380277633666992, -0.08673038333654404, 0.731199324131012, 1.2515918016433716, 1.7253614664077759, -0.0894366055727005, 0.4569752514362335, -1.3681442737579346, -0.8854318857192993, 0.8806261420249939, 0.4613291025161743, 0.2527140974998474, -0.6063361763954163, 1.1332929134368896, -1.0468586683273315, 0.021967094391584396], [-1.2945588827133179, -0.9509502649307251, -1.41805100440979, -1.6312752962112427, 0.10039098560810089, -0.19750039279460907, 0.9899503588676453, -0.44170892238616943, -1.311659812927246, -0.24719683825969696, 0.9530428051948547, 0.802229642868042, -1.420364260673523, 0.2192792147397995, 0.02255846932530403, 1.3461623191833496, 1.9100537300109863, -1.1190286874771118, -0.7215986251831055, -2.4013922214508057, 1.95980966091156, 0.0731176808476448, 0.7526775598526001, -0.392118901014328, 0.9478380680084229, -0.40559816360473633, 0.7732766270637512, -1.9373939037322998, -1.6244908571243286, 0.5440604090690613, 0.03477596491575241, 0.1964297890663147, 2.4358649253845215, 0.3219769299030304, -1.343108892440796, -0.84884113073349, -0.7505834698677063, -0.6866325736045837, 0.19419987499713898, 0.8283742666244507, 1.519545316696167, -0.14597579836845398, -0.4341820180416107, 0.5562828183174133, -0.46038907766342163, -0.3533300757408142, 0.47592270374298096, -0.8264771103858948, -0.4493313133716583, 0.7479580640792847, 0.10901373624801636, 0.8337416648864746, 1.460585117340088, 0.2840742766857147, -0.6662901043891907, -0.8067489862442017, 1.2794694900512695, 0.5473095178604126, -0.040908802300691605, 0.350512832403183, -0.39383408427238464, 0.37487393617630005, 2.1727044582366943, 1.7517179250717163], [-0.40735143423080444, 1.867780089378357, 0.6510009765625, 0.35904815793037415, 1.2746193408966064, 0.980691134929657, 1.0892252922058105, -0.3438686430454254, -1.3074694871902466, -1.5973913669586182, -2.1589555740356445, -0.6295725703239441, 1.0754363536834717, -0.22754433751106262, -1.915736198425293, -0.634995698928833, 1.0620368719100952, 0.19932974874973297, -0.37648382782936096, 1.0014077425003052, 0.4320701062679291, 1.0825167894363403, 1.5222011804580688, 1.1322622299194336, 1.51376211643219, -0.30577030777931213, -1.5480995178222656, 0.04497160390019417, 0.8172049522399902, 2.05846905708313, -2.0629830360412598, 0.14097458124160767, 1.2267314195632935, 0.8853117823600769, -0.021484624594449997, -1.6539913415908813, -0.35999056696891785, -0.0658491998910904, -0.9591450095176697, -2.4986865520477295, 1.3075400590896606, -1.310762882232666, -0.21728062629699707, 1.1963402032852173, -1.269018530845642, -0.4920147955417633, 0.6558135151863098, -2.2508556842803955, -0.802968442440033, -1.1844398975372314, 0.552250325679779, -0.46159565448760986, 0.7894304990768433, -1.7107373476028442, -2.8491885662078857, 0.7977275252342224, 0.44965875148773193, 0.8034518361091614, -0.06029624119400978, 0.7945998907089233, -0.4125795364379883, 1.0656172037124634, -0.6433924436569214, -0.08029308170080185], [0.6816491484642029, -0.21962901949882507, -1.284063458442688, -0.6394766569137573, 0.1418350487947464, -0.14263030886650085, -0.8923726677894592, -1.7472906112670898, -0.6210212111473083, -1.0818387269973755, -0.2866588532924652, 1.3972965478897095, 0.8433597683906555, 1.2050502300262451, 0.706662654876709, -0.831180989742279, -0.16352578997612, 1.2401372194290161, -0.5236477851867676, -0.327938050031662, 0.2471189796924591, 0.592638373374939, 0.4128921329975128, 0.5449996590614319, -1.7483395338058472, -0.288110613822937, 0.32938045263290405, -0.06500265747308731, -1.1403529644012451, -0.13908445835113525, 0.2584172487258911, -0.0784260705113411, 1.4998269081115723, 0.9511927962303162, 0.13416807353496552, -0.2678886353969574, 1.5573346614837646, 0.5396364331245422, -0.013145404867827892, -1.525140643119812, -0.6448701620101929, 1.1453344821929932, -0.004462961107492447, -0.16901808977127075, 0.9413184523582458, -0.27540960907936096, 0.7323861122131348, -0.7656540870666504, 2.044774293899536, 1.9721009731292725, 1.235985279083252, 1.8335541486740112, -0.9590830206871033, -0.6530590057373047, 1.5484005212783813, -0.24872970581054688, 0.5801858305931091, 1.027938961982727, -0.631706714630127, 0.15204665064811707, -1.5248013734817505, 1.0440967082977295, -0.9232856631278992, -1.2212116718292236], [2.7222959995269775, 0.09616231918334961, 1.0155060291290283, 0.39169907569885254, -1.1559864282608032, 0.06445315480232239, -0.07715444266796112, -1.1928842067718506, 2.425142526626587, -0.13823246955871582, 0.9907890558242798, 0.41218146681785583, 0.02552219294011593, 0.5718607902526855, 0.9955576658248901, 0.47482573986053467, -2.2093825340270996, 0.5895947217941284, -0.016352012753486633, 0.6444880366325378, -0.06812186539173126, 0.5772673487663269, -0.7096586227416992, -0.24283930659294128, 1.5854687690734863, 0.3962778151035309, -3.1806294918060303, 1.5953917503356934, 3.0886712074279785, -0.1432744860649109, 0.37209847569465637, 1.0776939392089844, 0.750749945640564, 0.5039607286453247, -0.43107321858406067, -0.6785136461257935, 0.6717005968093872, 0.29723435640335083, -0.5432270169258118, -1.7398816347122192, -0.12027183175086975, -1.2415882349014282, -1.4735163450241089, -1.0083110332489014, 0.0095055578276515, -0.47570475935935974, 1.626278042793274, -2.748379707336426, 0.033819396048784256, 1.3213704824447632, -1.0889729261398315, -1.8097304105758667, 0.5427585244178772, 0.8252854943275452, 2.057161808013916, -1.0812183618545532, 0.08113528788089752, 0.7401876449584961, 0.95648193359375, -0.07660795748233795, -1.0173321962356567, 0.6952733397483826, 0.04913558438420296, -0.7456725239753723], [-0.2272593230009079, -0.3416251540184021, -1.0719451904296875, -1.404611587524414, 0.11368847638368607, -1.003256916999817, 0.061366576701402664, -0.11907288432121277, 0.0954352393746376, 1.150038719177246, 1.1754777431488037, -0.1749659925699234, 1.0793392658233643, 0.4238120913505554, -0.4615277349948883, 0.5642168521881104, 0.3200283348560333, 0.021935511380434036, -1.6827038526535034, 1.4376049041748047, -0.4207655191421509, -0.9344727396965027, -0.49310681223869324, -0.1273449957370758, -0.5501734018325806, -0.3355152904987335, 1.118715524673462, -1.141642689704895, -0.28490135073661804, -0.6357662677764893, -0.11402161419391632, 1.3537521362304688, -1.335087537765503, -0.37205076217651367, -1.7716580629348755, 0.40529969334602356, 0.6640007495880127, 0.03127845376729965, -0.8012135028839111, 1.0788313150405884, 1.7269413471221924, -0.5540374517440796, -0.4680185317993164, -0.5215005874633789, 0.04088352620601654, -0.15202730894088745, -1.8159642219543457, -0.30807656049728394, 2.0824286937713623, 0.16075284779071808, 0.476529598236084, -2.558657646179199, -0.19908545911312103, -2.882478713989258, -0.3695748448371887, 0.17382396757602692, 0.6415718197822571, -0.7978668808937073, 1.154292106628418, -0.3618777096271515, 0.8213663101196289, -1.7357590198516846, -0.1293676197528839, 0.2827768921852112], [-0.3021748661994934, 0.900196373462677, -0.8659608960151672, -0.19890058040618896, -0.7721902132034302, 0.5565051436424255, -0.5534616708755493, -0.27984046936035156, 1.2644336223602295, -1.864699125289917, -1.1742852926254272, 0.17806342244148254, 0.43121254444122314, 0.20243297517299652, -1.928375244140625, 0.69499671459198, -2.1537835597991943, -0.7032938599586487, -1.6930179595947266, -0.62305748462677, 0.4247458279132843, 0.7347579002380371, -0.6653749346733093, -0.008875207044184208, -0.7786893248558044, -0.057435110211372375, -0.11072225868701935, 1.2018275260925293, -1.1390953063964844, 0.6333327889442444, 0.39537152647972107, 1.103344440460205, -1.2677898406982422, 0.44726622104644775, 0.2776564359664917, -1.3191955089569092, 1.1491886377334595, 0.11856792122125626, -0.39614251255989075, -0.3095162808895111, -0.7006464004516602, -0.41627052426338196, 0.7912026047706604, 0.19560684263706207, 0.6517636179924011, -0.6901257634162903, 0.05440203472971916, 0.235099658370018, -0.7119200229644775, 0.47128182649612427, 1.0153422355651855, 0.7091739773750305, 0.8678551316261292, -1.4911867380142212, 0.12187860906124115, 0.6905983686447144, 1.2942278385162354, 2.067962169647217, 0.3349590599536896, 0.663165807723999, -0.028497427701950073, -1.1219619512557983, 0.10552319139242172, 0.07159736007452011], [-0.13977409899234772, -0.43103423714637756, 1.4632147550582886, 1.298673391342163, 1.5422577857971191, 0.8932250142097473, 2.532642364501953, -1.5733330249786377, -0.27666333317756653, -0.4214509427547455, -2.012484550476074, 0.8524115681648254, -0.8580867052078247, 0.17553448677062988, -0.08352292329072952, 0.7895854711532593, -0.3153954744338989, 0.6155723929405212, 0.5560070872306824, 0.474367618560791, -0.010707362554967403, -1.821622371673584, -0.7150536179542542, 0.8940691947937012, 1.1054999828338623, 1.2881715297698975, -0.5813726186752319, 0.35339656472206116, -0.5830972194671631, 0.8725332617759705, 0.16251926124095917, -1.1880611181259155, 0.24883434176445007, -0.1258256733417511, -0.8189112544059753, 0.8508168458938599, -0.017686208710074425, -1.1583716869354248, 1.3562180995941162, -0.6540714502334595, -1.6792513132095337, -1.075523018836975, -0.37868374586105347, -0.21135710179805756, -0.3797217607498169, -1.0215317010879517, 1.6455912590026855, -1.522375464439392, -0.8849202990531921, 0.06246933341026306, -1.4423984289169312, -0.2517782747745514, -0.7309204936027527, 0.23614053428173065, 0.3398332893848419, 0.45400270819664, 0.35212570428848267, 1.4525127410888672, 1.0752450227737427, 0.11274418234825134, 1.2201558351516724, -1.7145898342132568, 2.3296852111816406, 1.4268051385879517], [-0.566196858882904, -0.10061483085155487, 0.9626700282096863, 1.9595153331756592, -0.05090819671750069, 0.14773254096508026, -0.512373149394989, 1.2862709760665894, 0.6324319839477539, 1.431924819946289, -0.4971475601196289, 1.1927109956741333, 0.6075164675712585, 0.1641705185174942, 0.6287446618080139, -0.4641386568546295, -0.2871372103691101, -2.191483497619629, 0.03766261041164398, -0.49880993366241455, -0.25617334246635437, -0.8522369265556335, 1.3934860229492188, -0.7035323977470398, -1.4938147068023682, -1.235739827156067, -0.3356633484363556, -1.4204761981964111, -1.226209044456482, -0.09651719033718109, 0.09095652401447296, 0.333528071641922, -0.011142400093376637, 0.9700918197631836, -0.045508213341236115, 0.15037567913532257, 0.236721009016037, -0.2569824755191803, 1.1775927543640137, 0.016114069148898125, 0.8838582038879395, 0.12765216827392578, -0.4837924540042877, 0.6762853860855103, 0.112917959690094, 2.090984344482422, -0.4629451632499695, 0.10325451195240021, 0.04570849984884262, 1.783962607383728, -0.8212841749191284, 0.40483883023262024, -0.08230536431074142, 0.9366636276245117, 0.6455982327461243, -0.6141735315322876, -0.12656208872795105, -0.7336418032646179, 0.7520329356193542, 0.8481870293617249, -0.19830754399299622, 1.7373079061508179, -0.2980801463127136, -0.22644324600696564]]]], "dtype": "float32"}, false], "e": {"__tup__": [{"__nd__": [[[[-0.09128519147634506, 0.04674171283841133, -0.04366074502468109, 0.23549789190292358, -0.24225100874900818, 0.04650949314236641, 0.23629386723041534, -0.17348000407218933, 0.10890591144561768, 0.08003214001655579, 0.06771275401115417, 0.22406606376171112, 0.19865524768829346, 0.16416436433792114, 0.2063625454902649, 0.15071043372154236, -0.1457626223564148, -0.08855988830327988, -0.02388809435069561, 0.03637794777750969, -0.0314570777118206, 0.010344898328185081, 0.14504550397396088, 0.165885791182518, -0.045174628496170044, -0.02414531633257866, 0.019304964691400528, -0.016699716448783875, 0.083043672144413, 0.165500670671463, 0.05831503868103027, -0.1900215595960617, -0.12416684627532959, -0.2260482758283615, 0.12486381828784943, 0.012743528001010418, -0.021073751151561737, 0.2673549950122833, -0.2653612196445465, -0.030984779819846153, 0.14810305833816528, -0.11772095412015915, -0.00904296524822712, 0.010037392377853394, -0.02842089906334877, -0.09913747012615204, -0.24962438642978668, -0.2455604076385498, -0.1298305243253708, -0.025165846571326256, 0.07452485710382462, 0.05706355348229408, 0.03046637959778309, -0.1596253514289856, -0.025914620608091354, -0.006396494340151548, 0.16479459404945374, 0.06651163846254349, 0.10141977667808533, 0.08717264235019684, -0.1538376808166504, -0.15509361028671265, 0.09465213865041733, 0.009656038135290146], [-0.08557779341936111, -0.08226408809423447, 0.18845431506633759, 0.051225461065769196, -0.20304341614246368, 0.07737251371145248, 0.16700449585914612, 0.04605874791741371, -0.12692545354366302, -0.0027343956753611565, 0.10899802297353745, 0.2814713418483734, 0.12286645174026489, 0.22674646973609924, -0.018083566799759865, 0.15086282789707184, -0.241907000541687, 0.03526756167411804, -0.011348869651556015, -0.14314185082912445, 0.08874189108610153, -0.08611004054546356, 0.36696353554725647, 0.3163089156150818, -0.18371832370758057, -0.14460702240467072, -0.11585600674152374, 0.025562256574630737, -0.06372804939746857, 0.22638988494873047, 0.0137562807649374, -0.016561442986130714, -0.17073804140090942, -0.051894575357437134, 0.09685449302196503, 0.18198901414871216, -0.015749819576740265, 0.17222394049167633, 0.027202017605304718, -0.06910362094640732, 0.013765765354037285, -0.2905558943748474, 0.06344049423933029, -0.01157840620726347, 0.09927275776863098, -0.02628662995994091, -0.080714151263237, -0.3131140470504761, -0.009275655262172222, 0.10356651246547699, 0.23340219259262085, -0.06232695281505585, -0.13318488001823425, -0.09263031929731369, -0.003247666172683239, -0.002060346305370331, 0.020902978256344795, 0.09627273678779602, 0.2041987031698227, -0.20963969826698303, 0.05828208848834038, 0.1284206509590149, 0.1903308480978012, -0.030596788972616196], [0.14244917035102844, -0.11333219707012177, -0.08453534543514252, 0.19344009459018707, 0.02597344107925892, 0.1848953813314438, -0.10331187397241592, -0.12145670503377914, 0.12544630467891693, 0.027743244543671608, 0.02121398039162159, 0.40382400155067444, 0.3576004207134247, 0.019948042929172516, 0.12989680469036102, 0.04240606352686882, -0.2338765561580658, -0.08076304942369461, -0.08267399668693542, -0.06528100371360779, 0.06388083845376968, -0.1010388508439064, 0.3856065571308136, 0.1676834672689438, 0.11102405190467834, 0.10644827038049698, -0.03540744259953499, 0.058681488037109375, 0.17770451307296753, -0.1768133044242859, 0.3611849248409271, 0.06872822344303131, -0.1376885324716568, -0.012711810879409313, 0.2567867338657379, 0.18832571804523468, 0.1147262379527092, -0.12945277988910675, -0.2358100265264511, -0.15073031187057495, 0.052414413541555405, -0.021406028419733047, 0.09504532814025879, 0.04766131564974785, -0.013136737048625946, 0.13995510339736938, -0.43075031042099, -0.13107994198799133, 0.06885109096765518, -0.18812718987464905, -0.13053952157497406, -0.043066684156656265, 0.27581271529197693, 0.06589477509260178, -0.19848467409610748, -0.15465179085731506, 0.11493032425642014, -0.21465960144996643, 0.03885869309306145, 0.08945539593696594, -0.10357620567083359, -0.2218562513589859, 0.032810013741254807, 0.13982588052749634], [0.034096818417310715, -0.003486157860606909, 0.022050397470593452, -0.14028076827526093, -0.06781808286905289, -0.060221780091524124, 0.22916953265666962, -0.111569344997406, -0.08414877951145172, -0.09317602217197418, -0.14033019542694092, 0.16445180773735046, 0.07870972901582718, -0.01455942913889885, 0.03157307952642441, 0.0470285527408123, -0.19700339436531067, -0.03941888362169266, -0.08728454262018204, 0.001958707580342889, 0.038865040987730026, -0.10199867188930511, 0.2933335602283478, 0.15852485597133636, -0.13906492292881012, 0.07808304578065872, -0.16540968418121338, 0.11063988506793976, 0.05525832995772362, 0.08028443157672882, 0.037940844893455505, -0.10225901007652283, -0.04115694388747215, 0.042758289724588394, 0.140422061085701, -0.0056838844902813435, 0.16916105151176453, -0.1265547126531601, -0.11635439842939377, -0.07855869084596634, -0.0159114021807909, 0.001292930101044476, -0.02212662063539028, 0.1946113407611847, -0.12195093929767609, 0.07715758681297302, -0.21129366755485535, -0.25179100036621094, -0.20683930814266205, -0.09681358188390732, -0.19263719022274017, -0.06386621296405792, 0.19211333990097046, -0.12906698882579803, -0.05548608675599098, -0.15540282428264618, 0.1256081759929657, -0.1073228195309639, -0.10651851445436478, 0.007942002266645432, -0.20047879219055176, 0.07990220189094543, 0.032709602266550064, 0.20362499356269836], [0.08161459118127823, -0.052184510976076126, 0.060560524463653564, 0.16849297285079956, -0.016439396888017654, 0.11026497185230255, 0.28120261430740356, 0.046024199575185776, -0.04265254735946655, 0.20263177156448364, 0.0604686364531517, 0.09772440791130066, 0.10643188655376434, -0.011673325672745705, 0.18787604570388794, 0.09726689010858536, -0.1139753982424736, 0.0038590694312006235, -0.2633945345878601, 0.047106191515922546, 0.06425647437572479, -0.15686261653900146, 0.04457223042845726, 0.021669385954737663, -0.2231549620628357, 0.013987486250698566, -0.015626806765794754, -0.014204109087586403, 0.1391177475452423, -0.08036858588457108, -0.0739416629076004, -0.2197198122739792, -0.018422216176986694, -0.08096285909414291, 0.2655622661113739, -0.1268490105867386, -0.0756223201751709, 0.04400555416941643, -0.08396259695291519, -0.024690164253115654, -0.13619472086429596, 0.07344036549329758, 0.20267732441425323, 0.04923560470342636, -0.1639856994152069, -0.07242649793624878, -0.014805348590016365, -0.05862933397293091, 0.038159940391778946, -0.04756688326597214, -0.12786084413528442, 0.006102015729993582, 0.22619964182376862, -0.1499931365251541, -0.1446646898984909, -0.13391068577766418, 0.02821454219520092, -0.0883263424038887, -0.10684758424758911, -0.1780836284160614, -0.36278384923934937, 0.09761346876621246, -0.2116335928440094, 0.2168794721364975], [-0.09866292774677277, -0.21138277649879456, -0.038945868611335754, 0.18006275594234467, -0.07273466885089874, 0.013343323022127151, 0.07565551996231079, -0.15382741391658783, -0.13726036250591278, 0.11567828059196472, 0.019932100549340248, 0.3078659772872925, 0.10894003510475159, -0.005333229433745146, 0.24002379179000854, 0.08374997973442078, -0.2648393213748932, -0.03938740864396095, 0.08618722856044769, -0.1635979413986206, -0.11016104370355606, -0.04479068145155907, 0.2752564549446106, 0.23287315666675568, -0.19606846570968628, 0.03746983781456947, -0.10230201482772827, -0.15462914109230042, 0.08502574265003204, 0.062426209449768066, 0.1823105812072754, -0.18743734061717987, -0.22066344320774078, -0.07823739945888519, 0.13684822618961334, -0.0042795659974217415, 0.08858276903629303, 0.019470330327749252, -0.14754383265972137, -0.19113174080848694, -0.0697438195347786, -0.32803454995155334, -0.014700879342854023, -0.07699998468160629, 0.13977880775928497, -0.11985888332128525, -0.023501604795455933, -0.3013606071472168, -0.1717466413974762, 0.0968199148774147, 0.04054675251245499, -0.12385333329439163, 0.2038525938987732, 0.07396557927131653, -0.09001219272613525, -0.1537683755159378, 0.0698329359292984, -0.06240985542535782, 0.17332424223423004, -0.1282425820827484, 0.010594683699309826, 0.22191724181175232, 0.1017749235033989, 0.11119172722101212], [-0.0212752316147089, -0.011751099489629269, -0.06086008995771408, 0.11374257504940033, 0.09711285680532455, 0.08968380838632584, 0.20846888422966003, -0.09758063405752182, -0.22211965918540955, -0.08042605221271515, -0.07206696271896362, 0.12925001978874207, 0.008301627822220325, -0.09433501213788986, 0.0070563144981861115, -0.04913294315338135, -0.13507895171642303, -0.09723865985870361, -0.0011531730415299535, 0.05955300107598305, 0.10828931629657745, -0.1260155588388443, 0.25879383087158203, 0.18546544015407562, -0.04589381441473961, 0.19995209574699402, -0.23689277470111847, 0.021285446360707283, 0.04985906556248665, 0.15465864539146423, 0.09581585228443146, -0.1586105227470398, -0.08844735473394394, -0.03250282257795334, 0.19908608496189117, 0.1414310485124588, 0.0479736365377903, 0.09343811124563217, 0.004689252469688654, -0.17840509116649628, 0.04568695276975632, 0.1249225065112114, 0.10078795999288559, 0.1762690395116806, -0.10287115722894669, 0.05250122398138046, -0.06662914901971817, -0.25412845611572266, -0.003662220435217023, -0.07639729976654053, 0.004856201820075512, 0.014182101003825665, 0.07153457403182983, -0.22981444001197815, -0.1636134684085846, -0.11300782859325409, 0.14718802273273468, 0.011414255946874619, -0.03044222854077816, 0.007241415791213512, -0.13106052577495575, -0.013947542756795883, 0.05983594432473183, 0.22914960980415344], [-0.07916203886270523, 0.11765705049037933, 0.10358767211437225, 0.06521166861057281, -0.30441075563430786, 0.10953742265701294, -0.022374087944626808, -0.08054392784833908, -0.05828672647476196, 0.11176541447639465, -0.22111830115318298, 0.3037485182285309, 0.21772348880767822, 0.21935684978961945, 0.05922528728842735, -0.021704992279410362, -0.37521883845329285, -0.05809430405497551, 0.0511636883020401, -0.27500396966934204, -0.016033152118325233, -0.06547150015830994, 0.23583360016345978, -0.0022132880985736847, -0.0896182432770729, -0.13327457010746002, -0.2628978490829468, 0.2176756113767624, 0.04782725125551224, -0.10627497732639313, -0.11697368323802948, 0.035538747906684875, -0.12774726748466492, 0.1190672516822815, 0.2724854350090027, 0.09297440946102142, 0.04047679156064987, -0.020987797528505325, -0.14506560564041138, 0.05491507798433304, 0.031397268176078796, -0.07820412516593933, 0.2394176423549652, -0.07720938324928284, -0.02370884269475937, 0.22800053656101227, -0.21386824548244476, 0.05028747022151947, -0.17335714399814606, -0.14954456686973572, -0.045026279985904694, -0.09456788003444672, 0.030995894223451614, 0.04465525969862938, -0.0226796455681324, -0.12172292172908783, -0.08839807659387589, -0.2755976915359497, -0.06164932623505592, 0.028917640447616577, -0.11904831975698471, 0.005944428034126759, -0.13803228735923767, 0.10661067068576813], [-0.06073280796408653, -0.13418278098106384, 0.09061045199632645, 0.05089400336146355, -0.06335338205099106, -0.009431296959519386, 0.23998989164829254, -0.20342108607292175, -0.017168408259749413, 0.04831443354487419, 0.003524319501593709, -0.012069891206920147, 0.1103031188249588, -0.15595658123493195, 0.387082576751709, 0.030807247385382652, -0.33511191606521606, -0.04863195866346359, -0.056294381618499756, 0.07011079788208008, -0.15463602542877197, 0.017049167305231094, -0.07802296429872513, 0.03244580700993538, -0.27312377095222473, -0.04345639795064926, -0.08926881104707718, 0.0905715599656105, 0.023257894441485405, 0.22187002003192902, -0.02545754797756672, -0.24284079670906067, -0.19583164155483246, -0.10819268971681595, 0.1609407216310501, 0.03463797643780708, -0.04982452467083931, -0.05440637841820717, 0.05887506529688835, -0.018283482640981674, 0.07279092818498611, 0.026188496500253677, 0.02722509577870369, 0.12759225070476532, -0.3299291133880615, -0.11046982556581497, -0.015670670196413994, -0.1950756013393402, -0.07717672735452652, -0.034684084355831146, -0.1606173813343048, -0.012747672386467457, 0.23694944381713867, 0.009471054188907146, -0.04089966043829918, -0.20031927525997162, 0.24731293320655823, -0.02307788096368313, 0.0006731230532750487, -0.22457097470760345, -0.2589092254638672, 0.019174350425601006, 0.16050738096237183, 0.22779516875743866], [-0.09551648795604706, -0.11576458811759949, -0.022632382810115814, 0.10119397938251495, -0.038843441754579544, -0.19193212687969208, 0.25960418581962585, 0.1062525063753128, 0.09926623106002808, 0.16625581681728363, 0.4590946137905121, 0.3882947564125061, 0.12982353568077087, 0.1271279901266098, 0.12730665504932404, 0.0395820252597332, 0.010598214343190193, -0.08367419987916946, -0.09350050985813141, -0.09218423813581467, 0.19468432664871216, -0.2786353528499603, 0.1650180071592331, 0.26791390776634216, -0.2333746999502182, -0.22637028992176056, -0.02446121722459793, 0.03201307728886604, -0.016471626237034798, -0.13927002251148224, 0.17143572866916656, -0.1449994295835495, 0.07069481164216995, -0.2392728179693222, 0.21894043684005737, 0.2177993506193161, -0.24834086000919342, 0.24979594349861145, -0.2976839244365692, -0.11515817791223526, -0.05383528023958206, -0.029932979494333267, 0.36411648988723755, -0.12015077471733093, -0.022882726043462753, 0.1959182620048523, 0.06901096552610397, 0.007776608224958181, 0.1435442417860031, -0.18972259759902954, 0.02801542915403843, 0.26387831568717957, 0.3331511914730072, 0.062374722212553024, -0.20605888962745667, -0.08525136113166809, 0.19697034358978271, -0.47428610920906067, 0.21952958405017853, 0.18267017602920532, -0.13294588029384613, 0.10974796116352081, 0.02502468042075634, -0.03611034154891968], [0.0005042640259489417, 0.003969673998653889, -0.010746072977781296, 0.023339146748185158, -0.10212132334709167, 0.036148861050605774, 0.16996987164020538, -0.22696423530578613, 0.0002322251966688782, 0.01865217462182045, -0.08353004604578018, 0.21919625997543335, 0.05617951229214668, 0.1088467612862587, 0.12166818231344223, 0.12520650029182434, -0.2790674865245819, 0.08839726448059082, -0.2775960862636566, -0.1934567093849182, 0.077694371342659, 0.002889308612793684, 0.1959606260061264, 0.27237051725387573, -0.12974099814891815, 0.05735529586672783, -0.049265503883361816, 0.05567795783281326, 0.03237379342317581, 0.0779181718826294, 0.1749303936958313, -0.03184942156076431, -0.02252250909805298, -0.09965790808200836, 0.2065621316432953, -0.02363242208957672, 0.08735471963882446, 0.06296925246715546, -0.17604365944862366, -0.049306828528642654, -0.180696502327919, -0.06652553379535675, -0.1988532543182373, -0.1417136788368225, -0.13722440600395203, -0.0982687845826149, -0.07984380424022675, -0.31162041425704956, -0.163910374045372, 0.11009776592254639, -0.05976136773824692, -0.020013563334941864, 0.12977951765060425, -0.18819688260555267, -0.03158220276236534, -0.2151739001274109, 0.1995924860239029, 0.05002700537443161, 0.07146618515253067, -0.14706698060035706, -0.015892766416072845, 0.13931959867477417, 0.0897124782204628, 0.1199900358915329], [0.09264465421438217, -0.026179958134889603, -0.1319107711315155, 0.19620108604431152, 0.05332781746983528, -0.005115231033414602, 0.06041018292307854, -0.3711010813713074, -0.25750255584716797, 0.12457139045000076, -0.12216786295175552, 0.21715818345546722, 0.12025555223226547, 0.15149009227752686, -0.07255373150110245, -0.05221802368760109, -0.2516707479953766, 0.04668426886200905, -0.030881714075803757, 0.15225185453891754, -0.06441047042608261, -0.1564844250679016, 0.4123682379722595, 0.27539336681365967, -0.08379953354597092, 0.2603068947792053, -0.2630425989627838, 0.07854432612657547, 0.007062396965920925, -0.04185774177312851, 0.20636630058288574, -0.0013872599229216576, -0.1482982635498047, 0.0038824365474283695, 0.3794076144695282, 0.12628056108951569, 0.132266566157341, -0.03539970889687538, -0.12046541273593903, -0.22688856720924377, 0.09831174463033676, 0.15441064536571503, 0.04608038067817688, 0.04956892132759094, -0.004594842903316021, -0.06024841219186783, -0.1360289603471756, -0.21159255504608154, -0.024021117016673088, -0.09499584883451462, 0.025998925790190697, -0.20455364882946014, 0.1598564237356186, -0.09543678909540176, -0.1536494940519333, -0.10513164103031158, 0.2454453557729721, -0.11687393486499786, 0.040422942489385605, 0.004128791857510805, -0.2323165088891983, -0.014485654421150684, 0.11764951050281525, 0.06397432833909988], [0.0739702507853508, -0.3540913760662079, -0.17217373847961426, 0.20923049747943878, -0.14168329536914825, -0.03773265704512596, 0.02402040921151638, 0.013985996134579182, -0.1300799697637558, -0.0008679766906425357, -0.24413295090198517, 0.4202422499656677, 0.008285684511065483, 0.11928068846464157, 0.10075663030147552, -0.19721533358097076, -0.4477837383747101, -0.0783139243721962, 0.14479993283748627, -0.3047238886356354, -0.3336063027381897, -0.12250498682260513, 0.3048653304576874, 0.10376492142677307, -0.3290128707885742, 0.03131544217467308, -0.0446980744600296, 0.04772982746362686, -0.2776668071746826, -0.10014303028583527, -0.2600007653236389, 0.15825693309307098, -0.3393547832965851, 0.36450818181037903, -0.015608739107847214, 0.2508354187011719, -0.0038950052112340927, -0.03779888153076172, -0.2651810646057129, -0.22370970249176025, 0.2749449610710144, -0.002981789642944932, -0.03858098387718201, -0.177034392952919, -0.034459322690963745, -0.02563241496682167, -0.4724854528903961, 0.12248292565345764, -0.04159614071249962, -0.16952744126319885, -0.09100005775690079, -0.016528503969311714, 0.09112920612096786, 0.12552711367607117, 0.19454529881477356, 0.06286703795194626, 0.3310427963733673, -0.2401263415813446, -0.13054995238780975, -0.05353428050875664, -0.15024936199188232, 0.18006807565689087, 0.050985418260097504, 0.24517948925495148], [-0.03499944508075714, 0.04092662036418915, -0.053921811282634735, -0.05736926943063736, -0.04563652351498604, -0.04372982308268547, 0.28744029998779297, 0.0845186859369278, -0.026791196316480637, -0.1837216168642044, -0.07615122199058533, -0.012753885239362717, 0.08383893221616745, 0.029225977137684822, 0.07207975536584854, 0.06288175284862518, -0.10565228760242462, -0.13890177011489868, -0.10989165306091309, -0.08366364985704422, 0.005273114424198866, 0.05650964751839638, 0.32809120416641235, 0.1339186280965805, -0.04940521717071533, 0.21792377531528473, -0.14838862419128418, 0.2585451304912567, -0.013426536694169044, 0.01841130293905735, -0.11535584181547165, -0.29305291175842285, -0.01562693901360035, -0.05651821941137314, 0.3511609137058258, -0.12112145870923996, -0.04193735495209694, 0.026668639853596687, -0.12828345596790314, 0.018118929117918015, 0.047754429280757904, 0.13185840845108032, 0.15009304881095886, 0.034846555441617966, -0.12325181812047958, 0.09239855408668518, -0.29890793561935425, -0.3191200792789459, 0.007585210260003805, 0.019067155197262764, 0.043888043612241745, 0.06468335539102554, 0.04692188650369644, -0.20264729857444763, -0.047340910881757736, -0.2426796406507492, 0.10742560774087906, -0.00766365323215723, 0.21933653950691223, -0.19329357147216797, -0.33986422419548035, -0.12406135350465775, 0.04583101347088814, 0.05443284288048744], [0.10380713641643524, -0.13412845134735107, 0.05839405953884125, 0.14663580060005188, -0.17031151056289673, -0.11349662393331528, 0.3793542981147766, -0.12536968290805817, -0.04920221492648125, 0.07553452998399734, 0.2686522901058197, 0.2705228924751282, 0.2601410746574402, 0.09536033123731613, 0.3055877983570099, -0.14186829328536987, -0.24829640984535217, 0.013837521895766258, -0.11132167279720306, -0.02932140789926052, -0.003382819937542081, -0.10781481117010117, -0.018336303532123566, 0.1819758117198944, -0.01929321698844433, -0.1119905337691307, -0.04817969724535942, 0.09014879167079926, 0.18626172840595245, 0.2658434808254242, 0.08388641476631165, -0.042735371738672256, -0.18021096289157867, -0.04178478196263313, 0.05397486314177513, 0.15872322022914886, 0.005823347717523575, 0.06141025573015213, 0.07877534627914429, -0.13930509984493256, 0.09938982874155045, -0.04570428654551506, 0.041327062994241714, 0.09842690825462341, -0.11381876468658447, -0.10520995408296585, -0.08826182037591934, -0.21677663922309875, 0.013683639466762543, 0.009675120003521442, 0.014545191079378128, -0.11911800503730774, 0.03513750806450844, -0.04115051403641701, 0.19305519759655, -0.02787299081683159, 0.21454817056655884, -0.05067821964621544, -0.12046640366315842, -0.2710949182510376, -0.22527965903282166, 0.1333768665790558, 0.02343691885471344, 0.07000800222158432], [-0.04451603442430496, -0.19929414987564087, -0.23979677259922028, -0.022491678595542908, -0.12932804226875305, -0.009828227572143078, 0.0892748087644577, -0.12482883036136627, -0.031232474371790886, -0.06841776520013809, 0.13793422281742096, 0.22424361109733582, 0.10785000026226044, 0.046864449977874756, 0.2290516495704651, 0.03734883293509483, -0.46867600083351135, 0.03591660410165787, -0.10073711723089218, 0.1436823606491089, -0.0074136508628726006, -0.10195352137088776, 0.00543519901111722, 0.26639947295188904, -0.32801568508148193, -0.19142232835292816, 0.0796620324254036, -0.15461638569831848, 0.05666639283299446, 0.04896768182516098, 0.029210714623332024, -0.014849287457764149, -0.39091116189956665, 0.006375303026288748, 0.1062706708908081, 0.2494884580373764, -0.011034801602363586, -0.14373907446861267, -0.11220072209835052, -0.032461173832416534, 0.025868378579616547, -0.11900367587804794, -0.021524712443351746, 0.05179379880428314, -0.35062694549560547, -0.06621905416250229, -0.22504577040672302, -0.230672225356102, 0.06034624204039574, -0.052521541714668274, -0.27608782052993774, -0.004766625352203846, 0.40520697832107544, -0.08592488616704941, -0.0474318228662014, -0.12826873362064362, 0.1690138876438141, -0.16951343417167664, -0.188585564494133, -0.009283720515668392, -0.3624991178512573, 0.051430705934762955, 0.284083753824234, 0.2782493233680725], [-0.025420185178518295, -0.08305668085813522, -0.2170088291168213, 0.06064869835972786, -0.1971324384212494, 0.29589012265205383, 0.153786301612854, 0.08034171909093857, -0.22285014390945435, 0.11942510306835175, -0.003799441969022155, 0.1290929913520813, 0.3032905161380768, 0.1912469118833542, 0.2277749478816986, 0.08859819918870926, -0.12756727635860443, -0.06888008862733841, 0.05495622754096985, -0.1570024937391281, 0.009256063960492611, -0.15546122193336487, 0.19737878441810608, 0.21832896769046783, -0.08276274800300598, 0.03048310987651348, -0.15810784697532654, -0.002091825008392334, 0.3396148979663849, 0.18145838379859924, 0.1313503235578537, -0.1999252736568451, -0.008420757949352264, 0.08639746904373169, 0.37035447359085083, -0.05163583159446716, 0.040057145059108734, 0.20555956661701202, -0.2799184322357178, 0.005275994539260864, 0.12234374135732651, -0.07208515703678131, -0.019770022481679916, 0.03106020763516426, -0.03380823880434036, -0.15676681697368622, -0.2946323752403259, -0.2637435495853424, -0.14288343489170074, -0.029141630977392197, -0.06533751636743546, 0.07533091306686401, 0.13636960089206696, 0.04624437168240547, -0.10235578566789627, -0.047268759459257126, 0.2300826609134674, -0.2166873812675476, 0.12244214117527008, -0.08471372723579407, -0.13261383771896362, 0.004616021178662777, -0.016062531620264053, 0.12202908843755722], [-0.07595115900039673, -0.03738097846508026, -0.11922218650579453, 0.1725810170173645, -0.14106954634189606, 0.2191709578037262, 0.17381757497787476, 0.08981034904718399, -0.06400976330041885, 0.0990428552031517, -0.016805298626422882, 0.18675225973129272, 0.22972670197486877, 0.1049889400601387, 0.23324933648109436, 0.1345529556274414, -0.1284893900156021, -0.03489274904131889, 0.004812358878552914, -0.04855194687843323, -0.07168645411729813, 0.05531277880072594, 0.385928750038147, 0.12324477732181549, 0.015882110223174095, 0.02437376044690609, -0.23022699356079102, -0.0006530046812258661, -0.03429775685071945, 0.17594094574451447, 0.17768117785453796, -0.21957530081272125, 0.003108543809503317, -0.11235801130533218, 0.21234139800071716, -0.001862685545347631, -0.08680414408445358, 0.27272170782089233, -0.16859939694404602, -0.0896228477358818, -0.0764693096280098, 0.03192989528179169, 0.03919118270277977, -0.03339935466647148, -0.03201589360833168, -0.014446129091084003, -0.3268463611602783, -0.17678368091583252, -0.014620611444115639, -0.09675352275371552, 0.1789785474538803, -0.03193526715040207, -0.1377248466014862, -0.15680886805057526, -0.14316725730895996, 0.024976948276162148, 0.2925073802471161, -0.01899711601436138, 0.09771697223186493, -0.03135111555457115, -0.09268873929977417, -0.04457568749785423, 0.0014309226535260677, 0.10302876681089401], [-0.12638621032238007, -0.06051529943943024, 0.16757670044898987, 0.05940789729356766, 0.2905789613723755, -0.019325178116559982, 0.2664400339126587, 0.026847783476114273, -0.02204430103302002, -0.22882939875125885, -0.06222836673259735, -0.10313576459884644, -0.06559701263904572, -0.3298545181751251, -0.18976572155952454, -0.022516217082738876, -0.1619715839624405, -0.5314382314682007, -0.2386990785598755, 0.03206593543291092, 0.09440019726753235, -0.007192763965576887, 0.3695935904979706, 0.19323532283306122, 0.10718440264463425, 0.3598911762237549, -0.4539579153060913, -0.16560085117816925, 0.5019320249557495, -0.11332022398710251, -0.06545217335224152, -0.20771096646785736, -0.3071451485157013, -0.23420369625091553, 0.22033092379570007, -0.1739596575498581, 0.07641780376434326, -0.12160845100879669, -0.10675329715013504, -0.4898248314857483, -0.09609730541706085, 0.2621000111103058, -0.037860073149204254, -0.014877931214869022, 0.050377875566482544, -0.09816218167543411, -0.22268258035182953, -0.42333000898361206, -0.16914209723472595, 0.07105205208063126, 0.12958596646785736, -0.1762804090976715, 0.28875547647476196, -0.33876854181289673, -0.028718791902065277, -0.06691877543926239, 0.30631735920906067, 0.09900552779436111, 0.16447468101978302, 0.010951340198516846, -0.05864250659942627, -0.19734013080596924, 0.3846602737903595, -0.21542201936244965], [0.02773255668580532, -0.022738954052329063, -0.19284799695014954, 0.18767565488815308, -0.06615571677684784, -0.027577025815844536, 0.13205952942371368, -0.03729717433452606, -0.13001105189323425, 0.11554919183254242, 0.023650938645005226, 0.2240782082080841, 0.16109660267829895, 0.024275092408061028, 0.1875532865524292, 0.001749510644003749, -0.31795191764831543, 0.11287474632263184, -0.12563137710094452, 0.03349507972598076, -0.009199680760502815, -0.25846871733665466, 0.23872803151607513, 0.4250437617301941, -0.11184025555849075, 0.008099102415144444, -0.07127870619297028, 0.11953070759773254, 0.036542680114507675, 0.026773842051625252, 0.010444185696542263, -0.12590250372886658, -0.1773943156003952, 0.05873052403330803, 0.06884387135505676, 0.166133314371109, -0.23604878783226013, 0.0009751451434567571, -0.1958932876586914, -0.2651810646057129, 0.0035775965079665184, -0.15491123497486115, -0.04995908588171005, -0.07498887181282043, -0.14598065614700317, 0.11239130795001984, -0.22316202521324158, -0.20987695455551147, 0.06266807019710541, 0.013922873884439468, 0.055425114929676056, 0.02296610362827778, -0.0012177240569144487, -0.005410987418144941, -0.1439904123544693, -0.0011709897080436349, 0.2049170881509781, -0.0253624077886343, -0.012694211676716805, -0.04283018782734871, -0.06464950740337372, -0.030104173347353935, 0.26628193259239197, 0.12031889706850052], [-0.0038998271338641644, -0.047454558312892914, -0.22124610841274261, 0.1130925714969635, -0.03804561868309975, 0.1282672882080078, 0.06748472899198532, -0.04141911119222641, -0.09485812485218048, -0.0770452618598938, -0.04658885672688484, 0.17595581710338593, 0.09101038426160812, -0.02108057588338852, 0.15870358049869537, 0.2275499552488327, -0.187380850315094, -0.008127901703119278, -0.03972739726305008, -0.18213099241256714, -0.015407999977469444, -0.13242945075035095, 0.12950611114501953, 0.03860259801149368, -0.155775785446167, -0.03717929497361183, 0.04379582777619362, -0.05198047310113907, 0.1447763890028, 0.041798900812864304, 0.018664032220840454, 0.03253483027219772, 0.04939226806163788, 0.16917799413204193, 0.2195858657360077, 0.026266587898135185, 0.09475818276405334, -0.09702171385288239, -0.15634779632091522, -0.1533229947090149, -0.08828741312026978, -0.16428768634796143, 0.04548674449324608, 0.0028060416225343943, -0.07418828457593918, -0.10598497837781906, -0.21550151705741882, -0.07331053167581558, -0.16641844809055328, -0.025800516828894615, -0.08394952118396759, -0.11747061461210251, 0.19802117347717285, -0.03580435365438461, -0.11942171305418015, -0.07325602322816849, 0.13382601737976074, -0.049842528998851776, -0.050436828285455704, -0.06787021458148956, -0.09062812477350235, -0.12357794493436813, 0.2237701416015625, 0.26429858803749084], [-0.03196174278855324, -0.01533020194619894, -0.05614814534783363, 0.1326611191034317, -0.0800594761967659, 0.07926924526691437, 0.03136616200208664, 0.11489555984735489, -0.14450208842754364, 0.08405641466379166, -0.0745311751961708, 0.21169625222682953, 0.08877918869256973, 0.03654951974749565, 0.21052555739879608, 0.13358381390571594, -0.2585732340812683, -0.07959122955799103, 0.013421550393104553, -0.05071985721588135, -0.03343330696225166, -0.04922473058104515, 0.2145550549030304, 0.06904978305101395, -0.1404590606689453, -0.04660690575838089, -0.12248151749372482, 0.17294344305992126, -0.04106995090842247, 0.13576997816562653, -0.07444095611572266, -0.16986636817455292, 0.02236139588057995, 0.028508907184004784, 0.12213184684515, -0.007284861523658037, -0.19317373633384705, 0.07281912118196487, -0.17242322862148285, -0.19919942319393158, -0.015535811893641949, -0.020094983279705048, 0.22620272636413574, 0.14742225408554077, 0.04318711906671524, 0.07704982906579971, -0.19790270924568176, -0.17186030745506287, -0.05293029174208641, -0.16926848888397217, -0.025217629969120026, 0.0232833344489336, -0.08511236310005188, 0.021631531417369843, 0.07858002185821533, -0.17673662304878235, 0.17047137022018433, -0.17154008150100708, 0.021569376811385155, -0.03950011730194092, -0.0360146127641201, 0.013564507476985455, -0.023276956751942635, 0.020860295742750168], [-0.039932530373334885, -0.0497220940887928, -0.025666873902082443, -0.005558698438107967, 0.0011789898853749037, 0.09434539079666138, 0.0835152417421341, -0.09482046961784363, -0.09147819876670837, -0.07867616415023804, -0.0746048241853714, 0.15624721348285675, 0.08296333253383636, 0.0061242119409143925, 0.07285556197166443, -0.014655690640211105, -0.20167213678359985, 0.018740158528089523, -0.07625145465135574, -0.014014304615557194, -0.02321905642747879, -0.13506150245666504, 0.10619132965803146, 0.05389949306845665, -0.07294867932796478, 0.07012596726417542, -0.20549964904785156, 0.07841410487890244, -0.10787840187549591, 0.21784338355064392, 0.03902637958526611, -0.07708553224802017, -0.1403181552886963, -0.05082639306783676, 0.17237818241119385, 0.11951687186956406, -0.026674220338463783, 0.11154678463935852, -0.061884067952632904, -0.06901377439498901, 0.027875326573848724, 0.01837906427681446, 0.08295206725597382, 0.1521649807691574, -0.08174271881580353, -0.034673623740673065, -0.19130872189998627, -0.13033853471279144, -0.05336133390665054, -0.03432042524218559, 0.06324152648448944, -0.002679463941603899, 0.13184025883674622, 0.001480558654293418, -0.11973422765731812, -0.1010875254869461, 0.10095226019620895, -0.1933193951845169, 0.032384078949689865, -0.10461565107107162, -0.17048929631710052, -0.12356919050216675, 0.08688104152679443, 0.21605387330055237], [-0.2137269675731659, 0.08665618300437927, -0.06517119705677032, 0.06911006569862366, 0.010540157556533813, -0.022361870855093002, 0.10586525499820709, -0.18751895427703857, -0.17004255950450897, 0.022539328783750534, -0.0693037137389183, 0.005664674565196037, -0.0684698075056076, 0.024999098852276802, 0.052987538278102875, 0.008203745819628239, -0.11303222924470901, -0.15493637323379517, 0.09549442678689957, -0.0922081246972084, 0.2025546431541443, -0.21765343844890594, 0.2693062722682953, 0.24567994475364685, -0.07576354593038559, 0.0012685440015047789, -0.1329522579908371, 0.15139086544513702, -0.1296931505203247, 0.02227182127535343, 0.14038576185703278, 0.04743930324912071, -0.08493729680776596, -0.028235919773578644, 0.4347042143344879, 0.14802467823028564, -0.1431039273738861, 0.04628463461995125, -0.39556196331977844, -0.03381766378879547, 0.11349006742238998, 0.3803800642490387, -0.16351497173309326, -0.20196954905986786, 0.014891833998262882, 0.08507039397954941, -0.009231893345713615, -0.173478901386261, -0.27235811948776245, -0.027464816346764565, 0.016173550859093666, -0.029575541615486145, 0.17798131704330444, -0.16639576852321625, -0.1714707314968109, -0.06933798640966415, 0.010847470723092556, -0.27804192900657654, 0.20311693847179413, 0.003955828957259655, -0.08073003590106964, -0.11015728116035461, 0.013911289162933826, 0.2691290080547333], [0.03316850587725639, -0.14700645208358765, -0.13352292776107788, -0.0019015315920114517, -0.20324155688285828, 0.06531830132007599, -0.0178629569709301, -0.08531632274389267, 0.005960062146186829, -0.1508970707654953, -0.1168307214975357, 0.23325198888778687, 0.24663206934928894, 0.3294123113155365, 0.24107538163661957, 0.04447527602314949, -0.08763954788446426, -0.07912847399711609, 0.14063458144664764, 0.05588751658797264, 0.11463966965675354, -0.02291961759328842, -0.037727098912000656, -0.08541068434715271, 0.01917891949415207, 0.28012266755104065, -0.03422081843018532, -0.008728639222681522, 0.02233383245766163, 0.07486847043037415, 0.09030737727880478, -0.279202401638031, -0.04400461167097092, 0.010818077251315117, 0.262076199054718, 0.17541144788265228, 0.18457596004009247, 0.3478657007217407, 0.09507618844509125, -0.020594514906406403, 0.3103633522987366, 0.05216839909553528, 0.1751372069120407, 0.05299044027924538, 0.13958126306533813, 0.018037253990769386, -0.24288512766361237, -0.09446784853935242, -0.1620033085346222, -0.061645399779081345, 0.04478510841727257, 0.0308370478451252, 0.3339112401008606, -0.005453924648463726, -0.07041113823652267, 0.02925723046064377, -0.025749780237674713, -0.11960167437791824, -0.041746314615011215, 0.0852772668004036, -0.031072068959474564, -0.04307296499609947, 0.2067093551158905, 0.12611237168312073], [0.007820663042366505, -0.23883414268493652, 0.029020659625530243, -0.06041713058948517, -0.07074400037527084, -0.23204809427261353, -0.01676553674042225, -0.17337745428085327, -0.04874389246106148, -0.1327296793460846, -0.14401698112487793, 0.019942177459597588, 0.04795507714152336, 0.009060272015631199, 0.1525576263666153, 0.027302231639623642, -0.2512315511703491, 0.17522278428077698, 0.04891011491417885, -0.07865605503320694, -0.24771177768707275, -0.23981399834156036, 0.20189282298088074, 0.18606752157211304, -0.36295515298843384, 0.11419499665498734, -0.08192108571529388, -0.038583654910326004, -0.08258448541164398, -0.1483127772808075, -0.2079191952943802, -0.15960673987865448, -0.39495107531547546, 0.10778798907995224, 0.04799409210681915, 0.20512966811656952, 0.04474933445453644, 0.0562552846968174, -0.4136103093624115, -0.23620447516441345, 0.16530393064022064, 0.0799451470375061, -0.24415504932403564, -0.1303291767835617, -0.02935130149126053, -0.17335201799869537, -0.18937043845653534, 0.022940069437026978, -0.0636017769575119, 0.08938909322023392, -0.047533534467220306, 0.05261915549635887, 0.15845276415348053, 0.012060977518558502, 0.06643851101398468, 0.07340342551469803, 0.4466601610183716, -0.3413841724395752, -0.08833423256874084, -0.16257944703102112, -0.37179121375083923, -0.08865021914243698, 0.18987861275672913, 0.22679097950458527], [-0.032922737300395966, -0.15324118733406067, -0.1669120341539383, 0.17301557958126068, -0.0530087910592556, 0.17859631776809692, 0.05092811584472656, -0.07689842581748962, -0.07741695642471313, 0.02867785654962063, -0.16276012361049652, 0.09132317453622818, 0.2697136402130127, -0.017594236880540848, 0.23472176492214203, 0.14765912294387817, -0.2560480833053589, 0.0292307510972023, -0.09875701367855072, -0.11472311615943909, -0.11204983294010162, -0.13594795763492584, 0.26179027557373047, 0.3274367153644562, -0.13016468286514282, -0.033325355499982834, 0.04797773063182831, 0.0846041813492775, -0.07834835350513458, 0.1457643061876297, -0.03819901868700981, -0.1488223373889923, 0.014137624762952328, 0.05365261435508728, 0.16091382503509521, 0.013111399486660957, 0.11867891252040863, -0.16250060498714447, -0.26497966051101685, -0.1626293659210205, -0.06444907188415527, -0.14548633992671967, 0.11330416053533554, -0.019088367000222206, -0.1049032211303711, -0.01389284711331129, -0.17638978362083435, -0.007042056415230036, -0.2501402199268341, -0.08199397474527359, 0.12332765012979507, -0.07491079717874527, 0.02233869396150112, -0.022420749068260193, -0.2572444677352905, -0.21786907315254211, 0.19654010236263275, -0.09432917088270187, -0.022975819185376167, 0.06708918511867523, 0.06423252075910568, -0.048320524394512177, 0.0515289269387722, 0.02876562997698784], [-0.006457351613789797, -0.09055862575769424, -0.08864499628543854, 0.1806105375289917, -0.07459701597690582, 0.15184803307056427, 0.1259090155363083, -0.17285898327827454, 0.01539029274135828, -0.02701621688902378, -0.0910438522696495, 0.1273512989282608, 0.07439225167036057, 0.0966438502073288, 0.11229962110519409, 0.08690749853849411, -0.027865689247846603, -0.014458023943006992, 0.04862988367676735, -0.09652998298406601, -0.08106371015310287, -0.10812026262283325, 0.25302058458328247, 0.17662931978702545, 0.10231135785579681, -0.06084584444761276, -0.13998469710350037, -0.04692687839269638, 0.08489430695772171, 0.09629658609628677, 0.12424340099096298, -0.08184365928173065, -0.09899269044399261, -0.03433115780353546, 0.3298357129096985, 0.07202192395925522, 0.014899988658726215, -0.019748657941818237, -0.22476701438426971, -0.08486849814653397, 0.09557479619979858, -0.08832347393035889, 0.01736324653029442, 0.11968348175287247, -0.027352087199687958, -0.1620284765958786, -0.22359229624271393, -0.1779300719499588, -0.17701353132724762, -0.025686606764793396, 0.15929865837097168, -0.06115506961941719, 0.08080939203500748, -0.04218113049864769, -0.2335156798362732, -0.0856780931353569, 0.14516402781009674, -0.1008094921708107, -0.06898736953735352, -0.004993425216525793, -0.09586925059556961, -0.21332859992980957, -0.010407600551843643, 0.11689532548189163], [0.06501774489879608, 0.05007552728056908, 0.259403795003891, 0.014576511457562447, 0.264872282743454, -0.11219850927591324, -0.03540623188018799, -0.12420684844255447, 0.05131753534078598, -0.09148910641670227, 0.015212972648441792, 0.1565512716770172, 0.039557505398988724, 0.38151687383651733, 0.08557748049497604, 0.19054526090621948, 0.024273879826068878, 0.30792751908302307, -0.2175530344247818, -0.21788275241851807, -0.05055766552686691, -0.024882351979613304, 0.053080733865499496, 0.07481825351715088, 0.17727336287498474, -0.30780765414237976, -0.06761279702186584, 0.3162035346031189, 0.15725496411323547, 0.08021613955497742, 0.22111879289150238, -0.049272678792476654, 0.48077499866485596, -0.14061766862869263, -0.09957001358270645, -0.2795676589012146, 0.17120322585105896, -0.05407363176345825, -0.06789616495370865, -0.14954769611358643, -0.3519590497016907, -0.18920215964317322, -0.012603143230080605, 0.27076345682144165, -0.017940325662493706, -0.11234130710363388, -0.1674562245607376, -0.03511330857872963, 0.2866549491882324, 0.06643874943256378, 0.1337154060602188, -0.034054867923259735, 0.3171568512916565, -0.07026994973421097, -0.0698917955160141, 0.08059212565422058, 0.1593855619430542, -0.00727952690795064, -0.04365481063723564, 0.033539220690727234, -0.3595988154411316, -0.19688263535499573, 0.3289787769317627, 0.23814193904399872], [0.05987429618835449, -0.04724166914820671, -0.10153213888406754, 0.15678884088993073, -0.050448860973119736, 0.03446422517299652, -0.015389831736683846, -0.11853136122226715, -0.17272956669330597, -0.07234400510787964, -0.10022229701280594, 0.2901631295681, 0.20567764341831207, -0.0029802361968904734, 0.24435338377952576, 0.07022786140441895, -0.2930508255958557, -0.07764053344726562, 0.18877242505550385, -0.1637117564678192, 0.003802333027124405, -0.006168427877128124, 0.2247980386018753, 0.17152416706085205, -0.0470815971493721, 0.05322444811463356, -0.034627269953489304, -0.24356494843959808, 0.12219192832708359, -0.07220181822776794, 0.10329382866621017, -0.10152378678321838, -0.08550109714269638, 0.10165250301361084, 0.3019650876522064, -0.008962302468717098, 0.07083439826965332, -0.06997137516736984, -0.30807650089263916, 0.026661686599254608, 0.23063315451145172, -0.1825992316007614, -0.07699096202850342, -0.0030444427393376827, -0.046874869614839554, 0.05445977300405502, -0.15560801327228546, -0.1668059080839157, -0.011014117859303951, 0.15775153040885925, -0.005209066905081272, -0.011056559160351753, 0.20482847094535828, -0.013732490129768848, -0.22961893677711487, -0.1923925131559372, 0.09217511862516403, -0.1651294082403183, 0.06216266378760338, -0.003163686254993081, -0.11598647385835648, -0.11517216265201569, 0.2607182562351227, 0.29119500517845154], [-0.10449091345071793, -0.03458020091056824, 0.10645173490047455, 0.0016084680100902915, -0.09535986185073853, 0.03964605554938316, 0.42820245027542114, -0.19151771068572998, -0.02223258838057518, -0.16526253521442413, 0.11451073735952377, 0.011150838807225227, 0.1268511563539505, 0.026359204202890396, 0.12765128910541534, 0.04995196685194969, -0.1796705573797226, 0.09528502076864243, -0.13526295125484467, -0.05086268484592438, 0.09632669389247894, -0.15720176696777344, 0.15817613899707794, 0.37131521105766296, -0.13819865882396698, 0.08400232344865799, -0.2466949075460434, 0.1269364356994629, 0.2392410933971405, 0.06778101623058319, -0.07814517617225647, -0.03295493498444557, -0.3521995544433594, -0.10629315674304962, 0.06063471734523773, 0.06782004982233047, 0.06805208325386047, -0.012990190647542477, -0.062112025916576385, -0.2973562777042389, 0.018693096935749054, 0.007079358212649822, -0.046217069029808044, -0.03815746679902077, -0.043900392949581146, -0.19221045076847076, -0.10006498545408249, -0.35518380999565125, 0.11058080196380615, 0.1436871439218521, 0.16175727546215057, -0.09886525571346283, 0.16287755966186523, -0.3389942944049835, -0.06988465785980225, 0.03138241171836853, 0.2066754698753357, 0.18021899461746216, -0.12671446800231934, -0.11816034466028214, -0.04326881468296051, -0.21373134851455688, 0.22835001349449158, 0.09924934804439545], [0.1734595000743866, -0.07607465237379074, 0.03213488310575485, 0.3831429183483124, 0.01722746342420578, 0.30468517541885376, -0.011442356742918491, -0.16069865226745605, 0.05683913081884384, 0.14376142621040344, 0.12082452327013016, 0.24306586384773254, 0.2969667613506317, 0.1383173167705536, 0.0715816468000412, -0.019014576449990273, -0.044552281498909, -0.208046093583107, 0.011642195284366608, -0.02573036588728428, -0.1590701937675476, -0.028644803911447525, 0.17649705708026886, -0.036533698439598083, -0.08309154212474823, 0.05300196632742882, -0.06121588498353958, 0.16636565327644348, -0.010639743879437447, -0.16264353692531586, 0.25827768445014954, 0.10103066265583038, 0.07394139468669891, 0.06483245640993118, 0.1674407571554184, 0.15008877217769623, -0.061099883168935776, -0.09275320172309875, -0.35965386033058167, -0.10788986831903458, -0.05077661573886871, -0.11157484352588654, 0.07951276004314423, -0.004210938233882189, 0.027881406247615814, 0.05523037537932396, -0.2976303994655609, -0.11505402624607086, -0.4126352369785309, -0.2827020287513733, 0.04027431458234787, -0.0005891037289984524, 0.11260555684566498, -0.045549530535936356, -0.007255850825458765, -0.11884981393814087, 0.09872620552778244, -0.17239807546138763, 0.2140682190656662, 0.07235176116228104, -0.11333544552326202, 0.06506667286157608, -0.16838501393795013, 0.044538918882608414], [0.1315549910068512, -0.03999902307987213, -0.032527435570955276, -0.07010845839977264, 0.010475429706275463, 0.04940245300531387, -0.14112688601016998, 0.05171353369951248, 0.05140317231416702, -0.056090097874403, -0.042552560567855835, -0.06164174899458885, 0.08805490285158157, 0.2657026946544647, 0.14518514275550842, -0.04788679629564285, -0.39001911878585815, 0.2363256812095642, 0.07681700587272644, -0.09107255935668945, 0.012481657788157463, -0.11870875209569931, 0.13268887996673584, 0.2674327790737152, -0.30299049615859985, 0.1299314647912979, -0.19696372747421265, 0.003684047143906355, -0.009536514058709145, 0.05159469321370125, 0.14207211136817932, -0.10999945551156998, 0.10909318923950195, -0.03294016420841217, 0.2603808045387268, -0.12071602791547775, -0.06904923170804977, -0.05429309234023094, -0.2801116704940796, -0.03955816477537155, 0.06255573034286499, 0.0479317381978035, -0.00599603122100234, 0.05109008401632309, -0.023551994934678078, 0.021283050999045372, -0.11304415017366409, -0.2728284001350403, -0.07325659692287445, 0.2307923585176468, 0.1218031644821167, 0.00031995936296880245, 0.160970076918602, -0.26522648334503174, 0.035205766558647156, -0.1281299889087677, 0.046969398856163025, 0.16865356266498566, 0.05845440924167633, -0.19460490345954895, 0.029800552874803543, 0.25814029574394226, -0.04909902438521385, 0.15696731209754944], [0.11056401580572128, 0.05515890568494797, 0.08098306506872177, 0.163617804646492, -0.1406959593296051, 0.07315806299448013, 0.06411058455705643, -0.18328920006752014, 0.11636576056480408, 0.04863525554537773, 0.05058782920241356, 0.056058600544929504, 0.09765709936618805, -0.06685630977153778, 0.29465022683143616, 0.05342946574091911, -0.3361296057701111, 0.03651997074484825, 0.11744832992553711, -0.023589204996824265, -0.05390980839729309, -0.051362041383981705, 0.14268219470977783, -0.01883319951593876, 0.07827640324831009, 0.11841559410095215, -0.4596036970615387, 0.22713902592658997, 0.2141222059726715, 0.06077875196933746, 0.09445461630821228, -0.1088516116142273, -0.07067417353391647, 0.012697112746536732, 0.28933751583099365, 0.10069547593593597, 0.03181073069572449, 0.10105311870574951, -0.1971512734889984, -0.1898447424173355, 0.022012386471033096, -0.3055332601070404, 0.1055733785033226, 0.006163276266306639, -0.05650945380330086, -0.09929589927196503, -0.09450888633728027, -0.5125678181648254, -0.03679623082280159, 0.17272436618804932, -0.12826815247535706, -0.25528690218925476, 0.09928052872419357, 0.026769675314426422, 0.08853446692228317, -0.33446621894836426, 0.07926331460475922, -0.035547930747270584, 0.06290802359580994, -0.14752545952796936, -0.06248614564538002, -0.2517539858818054, 0.26422736048698425, 0.10785548388957977], [-0.11257042735815048, -0.15832820534706116, -0.006992964074015617, 0.038511209189891815, -0.12057166546583176, 0.10313019901514053, 0.09825275093317032, -0.006528707221150398, -0.09367257356643677, 0.032485730946063995, -0.057120636105537415, 0.2169574797153473, 0.1974094957113266, -0.024693287909030914, 0.27628397941589355, 0.16991892457008362, -0.37299591302871704, -0.10934200882911682, 0.013856694102287292, -0.06013675779104233, -0.031642187386751175, 0.158779114484787, 0.30093324184417725, -0.01992795616388321, -0.1959657520055771, -0.10631050914525986, -0.10517854988574982, 0.08581110090017319, 0.013025268912315369, 0.15326012670993805, 0.0008910035248845816, 0.03437941521406174, -0.16972124576568604, 0.10053404420614243, 0.10081806033849716, -0.009370806626975536, 0.052748605608940125, 0.17096441984176636, -0.13391849398612976, -0.15272006392478943, -0.00321840588003397, -0.10540872812271118, 0.06840617954730988, -0.024248654022812843, -0.10844966769218445, -0.07451224327087402, -0.3041837811470032, -0.10572229325771332, 0.07683541625738144, -0.13409408926963806, -0.016600508242845535, -0.13773420453071594, -0.1164943277835846, -0.13342329859733582, 0.14785398542881012, -0.08780829608440399, 0.16207684576511383, 0.07458148151636124, -0.0005737107712775469, -0.2581782937049866, -0.1290307492017746, 0.012175365351140499, 0.11415189504623413, 0.005711421370506287], [-0.13006842136383057, 0.05807221680879593, 0.05321572348475456, 0.3116666078567505, 0.019650906324386597, 0.043353524059057236, 0.10467638820409775, -0.06737329810857773, -0.20349295437335968, 0.03717346489429474, 0.09493690729141235, 0.225693941116333, 0.1323089599609375, 0.19186918437480927, -0.004950075875967741, 0.3131759464740753, -0.3134404122829437, 0.07283762842416763, -0.15773852169513702, -0.013607236556708813, -0.09554462879896164, -0.04421620815992355, 0.2944185137748718, 0.052656516432762146, -0.10466045886278152, -0.05725225433707237, -0.09426335245370865, 0.1252720057964325, 0.05410214141011238, 0.08926205337047577, -0.011133892461657524, -0.12380210310220718, -0.15372423827648163, 0.04415765032172203, 0.17791572213172913, -0.0021679282654076815, -0.12155582755804062, -0.008599784225225449, -0.09263091534376144, -0.07392864674329758, 0.20876067876815796, 0.11224789172410965, -0.03351885825395584, -0.03246717527508736, -0.17757576704025269, 0.13624785840511322, -0.13860464096069336, 0.12423370778560638, 0.0619315467774868, -0.17273934185504913, 0.16417820751667023, -0.08439330011606216, 0.1054319441318512, -0.2206592857837677, -0.05309871584177017, 0.08679760247468948, 0.11431438475847244, -0.00013195400242693722, 0.1464415043592453, 0.04343505948781967, -0.08992888778448105, -0.09441576898097992, 0.09866126626729965, 0.07946695387363434], [-0.009416710585355759, -0.14655444025993347, -0.14498336613178253, 0.1559387892484665, -0.004612802993506193, 0.06511326879262924, 0.10026382654905319, -0.13049468398094177, -0.12227024137973785, 0.06971939653158188, -0.1076219379901886, 0.22241629660129547, 0.11794441938400269, 0.042066968977451324, 0.1622827798128128, 0.01521997433155775, 0.00615288270637393, -0.032115399837493896, 0.06955237686634064, -0.032101429998874664, 0.07027333229780197, -0.07760699838399887, 0.3859509825706482, 0.16402849555015564, 0.018266772851347923, 0.1063031554222107, -0.12425056844949722, 0.0702715665102005, 0.26497045159339905, -0.008063915185630322, 0.3374939560890198, -0.14980527758598328, -0.16320402920246124, -0.0010405790526419878, 0.18525061011314392, 0.21309538185596466, 0.2803449332714081, -0.13588900864124298, -0.24980135262012482, -0.20174236595630646, 0.06077592819929123, 0.269015371799469, 0.15820249915122986, -0.034144941717386246, 0.081048384308815, -0.044678330421447754, -0.20173144340515137, -0.10933323204517365, 0.036781564354896545, -0.13050924241542816, -0.014938319101929665, -0.12170161306858063, 0.09831037372350693, -0.12399433553218842, -0.21555863320827484, -0.039790619164705276, 0.12484506517648697, -0.03627875819802284, -0.09779486805200577, 0.047053951770067215, -0.032720621675252914, -0.038809292018413544, 0.07307160645723343, 0.19059497117996216], [-0.07999087870121002, 0.05349328741431236, 0.006666350644081831, 0.19322365522384644, -7.77949535404332e-05, 0.04980494827032089, 0.08973428606987, 0.13514351844787598, -0.0670938640832901, 0.04863474890589714, -0.05667408928275108, 0.13625352084636688, 0.031467583030462265, 0.1594633013010025, 0.167798712849617, 0.12856602668762207, -0.11759325116872787, -0.044884093105793, -0.09006431698799133, -0.06110125780105591, 0.18630672991275787, -0.07708010077476501, 0.24748282134532928, 0.08412346243858337, -0.20140396058559418, 0.10078012943267822, -0.15673577785491943, 0.018260862678289413, -0.05475495383143425, 0.2565944790840149, -0.1191876009106636, -0.19499333202838898, 0.04874240979552269, 0.1573610156774521, 0.18698205053806305, -0.02500804141163826, 0.02027377113699913, 0.06127209588885307, -0.09671860188245773, -0.043678563088178635, -0.014837149530649185, -0.1239008903503418, -0.10456669330596924, 0.1290242224931717, -0.18525901436805725, 0.11499500274658203, -0.24149192869663239, -0.10500093549489975, -0.14189377427101135, -0.025208255276083946, 0.03361798822879791, 0.029027292504906654, 0.13179726898670197, -0.19393345713615417, -0.017223050817847252, -0.15596424043178558, 0.006784286815673113, -0.06860039383172989, 0.14071033895015717, -0.07829108089208603, -0.1578700989484787, 0.02473755180835724, -0.034089501947164536, 0.059715062379837036], [0.09658122062683105, -0.0196505356580019, 0.11864224076271057, 0.06224922090768814, -0.11946253478527069, 0.10337938368320465, 0.20566418766975403, -0.13057337701320648, -0.019724303856492043, 0.06578943878412247, -0.19085712730884552, -0.03574042022228241, 0.008958146907389164, -0.14376790821552277, -0.020984575152397156, 0.12131460011005402, -0.24274398386478424, 0.060399070382118225, -0.13268287479877472, -0.007452033925801516, -0.09324285387992859, -0.15829846262931824, 0.2160986214876175, 0.0029302644543349743, 0.031074902042746544, 0.17977522313594818, -0.2728348970413208, 0.10593846440315247, 0.2984045147895813, -0.02302679978311062, -0.061754461377859116, 0.07075903564691544, -0.025814149528741837, -0.04441766068339348, 0.22484318912029266, -0.15560086071491241, 0.11685247719287872, -0.039329372346401215, -0.2586821913719177, -0.1675906628370285, -0.05864410474896431, 0.09870719164609909, 0.025643274188041687, 0.05065605416893959, -0.10490661859512329, -0.19135232269763947, -0.11187077313661575, -0.2382820099592209, -0.2014520913362503, 0.003498426405712962, -0.06397368758916855, -0.15158021450042725, 0.06991022080183029, -0.12015357613563538, -0.07203441858291626, -0.19758878648281097, 0.26909366250038147, 0.006598649546504021, -0.05394424870610237, -0.1095278263092041, -0.21839764714241028, 0.11867307126522064, -0.08700307458639145, -0.00037311919732019305], [-0.03401050716638565, 0.0004972612368874252, -0.07484614849090576, 0.012498341500759125, 0.19436942040920258, 0.20017817616462708, 0.18016117811203003, -0.17851336300373077, -0.015368198975920677, -0.0035205017775297165, 0.04713349789381027, 0.024648038670420647, 0.09316375106573105, 0.04301725700497627, 0.12133855372667313, 0.2279660552740097, -0.11719585210084915, -0.06707170605659485, -0.008127979934215546, 0.031819116324186325, -0.11319874227046967, -0.23953679203987122, 0.23372820019721985, 0.05586487054824829, 0.23341801762580872, 0.26669013500213623, -0.27303966879844666, 0.14687353372573853, 0.1873965710401535, -0.003817193442955613, 0.00792414415627718, -0.06310970336198807, 0.10077393054962158, 0.12111460417509079, 0.32076841592788696, 0.06124646216630936, -0.1839025616645813, 0.06183265149593353, -0.19249536097049713, -0.26972314715385437, 0.014913562685251236, 0.2559254467487335, 0.0958666205406189, 0.011486053466796875, 0.04543467238545418, -0.07747137546539307, -0.25779709219932556, -0.14895182847976685, 0.11534939706325531, -0.050282787531614304, 0.11570888757705688, 0.004848880227655172, 0.14885126054286957, -0.12463685125112534, -0.19577132165431976, 0.19742420315742493, 0.19802145659923553, 0.08725321292877197, 0.1624567061662674, -0.023674078285694122, -0.14420995116233826, -0.07747862488031387, 0.1932803988456726, 0.1670580357313156], [-0.04739900678396225, 0.02008647844195366, 0.08858165889978409, 0.1350032389163971, -0.18298611044883728, 0.1005285233259201, 0.08345036208629608, 0.008809436112642288, 0.1849246323108673, -0.15011195838451385, 0.060542382299900055, 0.28184062242507935, 0.044634412974119186, 0.13675566017627716, 0.10008127242326736, -0.013072436675429344, -0.009159372188150883, 0.16925904154777527, 0.04641122743487358, -0.09261618554592133, 0.004963950719684362, -0.05852022394537926, 0.07147776335477829, 0.03069683536887169, -0.023504476994276047, -0.05699692294001579, -0.025929342955350876, 0.18310101330280304, -0.022646289318799973, 0.04783371463418007, -0.053438276052474976, -0.15403804183006287, -0.10454531013965607, 0.007644405122846365, 0.03691596910357475, 0.16885751485824585, 0.026909710839390755, -0.039533425122499466, -0.12305393069982529, -0.02494008094072342, 0.07880544662475586, -0.18137064576148987, 0.23859666287899017, 0.07609464973211288, -0.031397730112075806, 0.1856938898563385, -0.18417535722255707, -0.027488255873322487, 0.011595165356993675, -0.14057330787181854, 0.02439434453845024, -0.0593777559697628, 0.037960249930620193, 0.029055345803499222, 0.0417877696454525, -0.07753237336874008, 0.04286009445786476, -0.14671488106250763, -0.047576144337654114, 0.08818838000297546, -0.41798749566078186, -0.15303373336791992, 0.06489385664463043, 0.09017658233642578], [-0.033084142953157425, -0.08455345779657364, -0.08077215403318405, 0.21924903988838196, -0.2069786936044693, 0.16371110081672668, 0.1697506457567215, -0.010757138952612877, -0.3699410855770111, -0.06331098824739456, -0.03671907261013985, 0.20602872967720032, -0.005223474930971861, 0.023014893755316734, -0.0338761992752552, 0.23753342032432556, -0.3556242883205414, 0.07592976838350296, -0.25675028562545776, -0.14658069610595703, 0.01851535215973854, -0.05076604336500168, 0.12978608906269073, 0.03497869148850441, -0.12433242052793503, 0.05063701421022415, -0.2409948855638504, 0.08234643936157227, 0.19887393712997437, -0.018388835713267326, -0.24489203095436096, 0.20176908373832703, -0.06724406778812408, 0.20833763480186462, 0.20963364839553833, -0.00866006687283516, 0.041678618639707565, 0.11425768584012985, -0.18544843792915344, -0.13825538754463196, 0.08526139706373215, -0.05902785062789917, 0.1478578746318817, 0.000744691351428628, -0.23240132629871368, -0.04127411171793938, 0.03133362904191017, -0.09194710850715637, 0.07707951217889786, -0.09243898093700409, -0.2281351536512375, -0.03559752553701401, 0.1527608335018158, -0.1725711077451706, -0.12288837134838104, -0.08804930746555328, 0.2353028506040573, 0.01727290078997612, -0.05615777522325516, -0.1914624720811844, -0.1907462179660797, -0.06791400909423828, 0.009780360385775566, 0.15634137392044067], [-0.0011951486812904477, -0.27269718050956726, -0.025687729939818382, 0.05841933935880661, -0.1645713895559311, -0.049413908272981644, 0.10503280162811279, -0.13996289670467377, -0.06428035348653793, -0.1370745301246643, -0.2235240787267685, 0.18212439119815826, 0.09001726657152176, 0.04858693853020668, 0.15458513796329498, 0.03449343517422676, -0.2817727029323578, -0.011633770540356636, 0.21893636882305145, -0.11180498450994492, -0.2008838653564453, -0.08170100301504135, 0.25911417603492737, 0.17314757406711578, -0.22839845716953278, 0.17303021252155304, -0.0876101553440094, 0.04707492142915726, -0.04675731807947159, -0.02962038293480873, -0.0805041566491127, -0.16386593878269196, -0.1946403682231903, 0.17711953818798065, 0.10944019258022308, 0.005981879774481058, 0.07939313352108002, 0.00406500231474638, -0.3553435504436493, -0.2002825140953064, 0.23796965181827545, -0.17176806926727295, -0.03253869339823723, -0.07082114368677139, -0.0159929096698761, -0.05494276061654091, -0.2657555937767029, -0.14202351868152618, -0.07397707551717758, 0.03352098912000656, 0.08879829198122025, -0.08173990249633789, 0.11085790395736694, 0.023674078285694122, -0.027274537831544876, -0.04414241388440132, 0.39086613059043884, -0.05060173198580742, -0.09760838747024536, -0.06850141286849976, -0.20072835683822632, -0.06693956255912781, 0.20437964797019958, 0.12304399162530899], [-0.07810456305742264, 0.007768342737108469, -0.13790959119796753, -0.007561704143881798, -0.0885835736989975, -0.09011543542146683, 0.10824781656265259, -0.09984856098890305, -0.12236581742763519, -0.01786559633910656, 0.08512874692678452, 0.12105879932641983, 0.06238902732729912, 0.024129265919327736, 0.07606029510498047, 0.11811364442110062, -0.27481672167778015, 0.07122383266687393, -0.09424930810928345, -0.010852977633476257, -0.015203162096440792, -0.18689803779125214, 0.13446703553199768, 0.23492276668548584, -0.2158430814743042, 0.008907178416848183, -0.2513613700866699, 0.01229548454284668, -0.06527537852525711, 0.10856937617063522, -0.11238398402929306, -0.15365692973136902, -0.29885149002075195, -0.0850306898355484, 0.19653846323490143, 0.055801667273044586, -0.1687193065881729, 0.07218576222658157, -0.41431233286857605, -0.2159600704908371, 0.09756164997816086, 0.172608882188797, -0.04339582100510597, 0.04933768883347511, -0.13878758251667023, 0.06238982081413269, -0.09434396028518677, -0.1111043244600296, -0.046149056404829025, -0.07704729586839676, -0.06550701707601547, 0.1253267526626587, 0.027001583948731422, -0.013840967789292336, -0.17095214128494263, -0.02132890745997429, 0.26896771788597107, -0.19696681201457977, 0.03299689665436745, -0.0003862381854560226, -0.30771490931510925, -0.0909784734249115, 0.06575367599725723, 0.016584359109401703], [0.058015260845422745, -0.10219266265630722, -0.11244510859251022, 0.03194347769021988, -0.029056183993816376, -0.11568815261125565, 0.2412346452474594, -0.22120119631290436, -0.07948309183120728, -0.004934036638587713, 0.07690882682800293, 0.27840086817741394, -0.07393374294042587, 0.25622090697288513, 0.38459211587905884, 0.020600415766239166, -0.005315958987921476, 0.06397061049938202, 0.044407326728105545, -0.009040779434144497, -0.04342787712812424, 0.035690516233444214, 0.3056614398956299, 0.2710137367248535, -0.14511938393115997, 0.15050268173217773, -0.08641265332698822, 0.14360865950584412, 0.04316633194684982, -0.035145726054906845, -0.08640503138303757, -0.27941274642944336, -0.08388170599937439, 0.02650846168398857, 0.20211927592754364, 0.08063425123691559, 0.1773904412984848, 0.0414680540561676, -0.2097843736410141, -0.21083080768585205, 0.09098005294799805, 0.09732364863157272, -0.023767685517668724, 0.015337436459958553, -0.14479267597198486, 0.2397838830947876, -0.25572481751441956, -0.22082261741161346, 0.21999125182628632, 0.03607773035764694, -0.03681614249944687, 0.13063451647758484, 0.1411370486021042, -0.16012190282344818, 0.02341340109705925, -0.02931004762649536, 0.05543709173798561, -0.0067793154157698154, 0.1914720982313156, -0.26514530181884766, -0.4484408497810364, -0.07626017183065414, 0.29462313652038574, 0.12760215997695923], [0.0034878214355558157, -0.15368813276290894, -0.12571467459201813, 0.0435250923037529, -0.07870396971702576, 0.19757474958896637, 0.18660758435726166, -0.03367926552891731, -0.15275532007217407, -0.02617821656167507, 0.05639517679810524, 0.10443825274705887, 0.12192673981189728, 0.08423643559217453, 0.2028845250606537, -0.11854815483093262, -0.3316177725791931, -0.08058289438486099, -0.024732673540711403, -0.06299570947885513, -0.028360292315483093, -0.07241453975439072, 0.2576247751712799, 0.2045360654592514, -0.24129419028759003, 0.010221055708825588, -0.1930796504020691, -0.06607166677713394, 0.09940344095230103, 0.08480729162693024, 0.026657721027731895, -0.019771087914705276, -0.19592785835266113, 0.027206018567085266, 0.23316510021686554, 0.06999809294939041, 0.02086867205798626, 0.07140516489744186, 0.2108432799577713, -0.211689755320549, 0.03737448900938034, 0.08303497731685638, 0.03160728141665459, -0.007992244325578213, -0.06554561853408813, -0.11073244363069534, -0.17219144105911255, -0.2668023705482483, 0.03777454420924187, 0.003082729410380125, -0.1726209819316864, 0.0011414465261623263, 0.30340898036956787, -0.2099236100912094, 0.1109236404299736, -0.07868041098117828, 0.1971246749162674, -0.02288631908595562, -0.10020780563354492, -0.2885642945766449, -0.11163913458585739, 0.040822017937898636, 0.1915895789861679, 0.20685555040836334], [0.004295949824154377, -0.16152727603912354, -0.15815959870815277, 0.2059212028980255, -0.08670467138290405, 0.10271581262350082, 0.09236016124486923, -0.13605615496635437, -0.11014717072248459, -0.1309787929058075, -0.05832784250378609, 0.3662090301513672, -0.018810447305440903, 0.16348375380039215, 0.07311151176691055, -0.018421057611703873, -0.054093338549137115, -0.517227828502655, -0.04428732767701149, -0.43103286623954773, 0.03613216429948807, -0.005638896021991968, 0.18290889263153076, 0.20393294095993042, -0.17637786269187927, -0.12901151180267334, 0.18545569479465485, -0.18308594822883606, -0.12971073389053345, 0.05461546778678894, -0.0846303403377533, 0.023990873247385025, 0.08580286055803299, 0.2001013457775116, 0.127369225025177, -0.27527767419815063, 0.006966129411011934, -0.1892937868833542, -0.14314649999141693, 0.21870945394039154, -0.056588996201753616, -0.24667921662330627, -0.1218942254781723, -0.24306263029575348, 0.08671115338802338, -0.05844346061348915, -0.3075127899646759, 0.06913885474205017, -0.11564312875270844, 0.12686410546302795, -0.06665047258138657, -0.1597655564546585, 0.14907975494861603, -0.09563873708248138, 0.18059416115283966, -0.027978908270597458, 0.013174720108509064, -0.13830682635307312, -0.014668626710772514, -0.2893049418926239, -0.16394439339637756, 0.09695781767368317, -0.09851588308811188, 0.1982766091823578], [0.11500760167837143, 0.06226438283920288, -0.09016748517751694, 0.011717763729393482, 0.025730831548571587, -0.21424490213394165, 0.20669236779212952, -0.16261790692806244, -0.194268599152565, 0.11006411910057068, 0.3865698277950287, -0.061849888414144516, 0.3550302982330322, 0.12446627765893936, 0.3997001051902771, -0.12129919230937958, -0.4316907823085785, 0.2462949901819229, -0.12121875584125519, -0.11983286589384079, -0.06922128796577454, -0.025220829993486404, 0.07656743377447128, 0.21098095178604126, -0.020883534103631973, -0.055059418082237244, -0.17877954244613647, -0.03154611587524414, 0.19972458481788635, 0.07553046941757202, 0.1826237142086029, -0.10788961499929428, -0.004236204549670219, -0.01172646414488554, 0.2497997283935547, 0.17996548116207123, -0.19714410603046417, -0.03821657970547676, -0.24544189870357513, 0.0016199341043829918, 0.10707620531320572, -0.020186318084597588, 0.00509730726480484, 0.18323978781700134, -0.3315418064594269, -0.18399178981781006, -0.0014666252536699176, -0.2590460777282715, 0.10998129844665527, 0.13655710220336914, 0.04118289798498154, -0.10962545871734619, 0.103468157351017, -0.043907102197408676, 0.07832536846399307, -0.13215266168117523, 0.07993917912244797, 0.10633576661348343, 0.041372403502464294, -0.12972328066825867, -0.250152587890625, 0.1277243047952652, 0.10008024424314499, 0.13928350806236267], [0.19108706712722778, 0.08346936851739883, -0.05581685155630112, -0.12037527561187744, -0.07123417407274246, 0.19230476021766663, 0.056988995522260666, -0.04832206293940544, -0.04228010028600693, -0.17972587049007416, -0.22256916761398315, -0.006560465786606073, 0.11180576682090759, -0.037978559732437134, 0.03758394718170166, 0.15733040869235992, -0.019790682941675186, 0.15350931882858276, -0.09659498929977417, -0.062011122703552246, -0.1365259289741516, -0.26672497391700745, 0.09369587898254395, 0.08611427992582321, -0.09756120294332504, 0.00875703152269125, -0.09220050275325775, 0.0009829018963500857, 0.24169304966926575, 0.13268053531646729, 0.007767730392515659, -0.20134761929512024, 0.08672808110713959, -0.08346860855817795, 0.22473804652690887, -0.1742040067911148, -0.07499048858880997, 0.025815173983573914, -0.21721167862415314, -0.12702417373657227, -0.12009204179048538, 0.11101585626602173, 0.07753793150186539, 0.10801729559898376, 0.024796031415462494, 0.0021178554743528366, 0.051431164145469666, -0.1258450746536255, -0.16422484815120697, 0.07183203846216202, 0.020917369052767754, 0.044486187398433685, -0.000285267160506919, -0.12879236042499542, -0.16219091415405273, -0.04985267296433449, 0.2464863508939743, 0.12131161987781525, -0.045448023825883865, -0.013581633567810059, -0.30045419931411743, -0.06413183361291885, 0.021770380437374115, 0.15833908319473267], [0.06841028481721878, -0.16123579442501068, -0.11274221539497375, 0.10496571660041809, -0.1495024561882019, -0.02344803512096405, 0.13244202733039856, -0.1349351704120636, -0.15109902620315552, 0.006857558619230986, 0.04211536794900894, 0.32184121012687683, 0.15118159353733063, 0.08637209236621857, 0.22118012607097626, -0.025015683844685555, -0.29932114481925964, 0.047558531165122986, -0.12622715532779694, -0.19687944650650024, 0.06150900200009346, -0.042128678411245346, 0.14813923835754395, 0.21681343019008636, -0.13484705984592438, -0.0925687626004219, 0.06676775962114334, -0.04920182377099991, 0.1508398801088333, 0.17717133462429047, 0.03300536051392555, -0.06966912746429443, -0.1760781854391098, -0.017790548503398895, 0.10065355896949768, 0.04122895747423172, 0.031225554645061493, -0.06262658536434174, -0.029085267335176468, -0.07836689800024033, 0.0027255075983703136, -0.15571753680706024, -0.003391082864254713, -0.007917916402220726, -0.10844949632883072, -0.018492022529244423, -0.09720080345869064, -0.10307939350605011, -0.05506567284464836, -0.06023620069026947, -0.03486010804772377, -0.13469986617565155, 0.14589408040046692, -0.06718719750642776, -0.10670100897550583, -0.10070624202489853, 0.10974518954753876, -0.047250766307115555, -0.05912055820226669, -0.09292399883270264, -0.07588702440261841, 0.11093907058238983, 0.1761515587568283, 0.13058669865131378], [-0.10816015303134918, -0.032678570598363876, 0.005276961252093315, 0.14316852390766144, -0.010398375801742077, 0.12931719422340393, 0.25724276900291443, 0.05336788296699524, 0.030234936624765396, 0.03359968587756157, -0.07425101101398468, 0.1630067676305771, 0.12165233492851257, 0.20041221380233765, 0.15423575043678284, 0.018596947193145752, -0.17738573253154755, -0.11565469950437546, 0.11333335936069489, -0.06987220793962479, -0.10105067491531372, -0.019622717052698135, 0.36346107721328735, 0.03499535843729973, -0.11180592328310013, 0.12208972871303558, -0.0012458438286557794, 0.20276108384132385, 0.14329971373081207, 0.07232633233070374, 0.13931913673877716, -0.2731279134750366, 0.056022852659225464, -0.02937137521803379, 0.2054532766342163, -0.054075635969638824, 0.014169237576425076, -0.0958111435174942, -0.33783018589019775, -0.18332539498806, -0.001737584825605154, 0.0428808331489563, -0.03494488075375557, 0.026444440707564354, 0.04068012535572052, -0.06506764143705368, -0.31616243720054626, 0.0035894401371479034, -0.08209151029586792, -0.20616768300533295, 0.09492448717355728, -0.03633716329932213, 0.02704305574297905, -0.04514351859688759, 0.009896616451442242, -0.04048191010951996, 0.19433313608169556, -0.025909869000315666, 0.10173249989748001, 0.048604659736156464, -0.06371625512838364, 0.059682175517082214, -0.08863015472888947, -0.024977408349514008], [-0.22928662598133087, -0.08917998522520065, -0.03448445349931717, -0.020647769793868065, -0.015028631314635277, -0.10166595876216888, 0.2665351629257202, -0.3207361102104187, -0.04801248013973236, 0.07208309322595596, -0.09871013462543488, 0.1800425946712494, 0.03754118084907532, 0.09943066537380219, 0.32921427488327026, 0.11549323052167892, -0.08051422983407974, 0.00011606881889747456, -0.06213484704494476, -0.07692660391330719, 0.09272914379835129, -0.09888949245214462, 0.1583784520626068, 0.11219363659620285, -0.045955248177051544, 0.1228162869811058, -0.07669618725776672, -0.119285948574543, 0.013559188693761826, 0.07503990828990936, 0.2287113070487976, -0.2555813193321228, -0.0353829525411129, -0.13955940306186676, 0.1959151178598404, -0.04111439734697342, -0.03218339383602142, 0.08016743510961533, -0.16547700762748718, 0.03043276071548462, -0.0035846505779772997, 0.05422802269458771, -0.15660417079925537, -0.02190818265080452, -0.1465052217245102, -0.12578025460243225, -0.026028672233223915, -0.2547331750392914, -0.08681318163871765, 0.09230433404445648, -0.1615777462720871, 0.0607609823346138, 0.25706392526626587, -0.12094171345233917, -0.154601588845253, -0.09962278604507446, 0.17049728333950043, -0.07319382578134537, 0.17660394310951233, -0.08454911410808563, -0.16822761297225952, -0.11508043110370636, 0.25164666771888733, 0.33778947591781616], [-0.21936818957328796, 0.02945311926305294, -0.27980780601501465, 0.0033704054076224566, 0.008568763732910156, 0.1579478234052658, 0.1628158688545227, 0.20613400638103485, -0.29718834161758423, -0.00905534066259861, 0.07117924839258194, 0.022113602608442307, 0.04001840204000473, 0.0829676017165184, 0.08015450090169907, 0.11277004331350327, -0.04982977733016014, -0.031733665615320206, -0.04949169605970383, -0.28112444281578064, 0.12476273626089096, -0.17246480286121368, 0.31269004940986633, 0.18524384498596191, -0.16485144197940826, -0.12429865449666977, -0.15547044575214386, -0.05592211335897446, -0.0032677354756742716, 0.23935338854789734, 0.19080734252929688, -0.06273198127746582, 0.23141495883464813, 0.14990267157554626, 0.2964523732662201, -0.06727369874715805, -0.11800049990415573, 0.025295227766036987, -0.048605553805828094, -0.02228318713605404, -0.025499282404780388, -0.003528605680912733, 0.0529475212097168, 0.10043678432703018, 0.07259276509284973, 0.0039882417768239975, -0.07485716044902802, -0.18083415925502777, 0.06165758892893791, 0.0413224920630455, -0.0047607384622097015, 0.10228659212589264, 0.142374649643898, -0.1861792951822281, -0.03951430693268776, -0.2438320517539978, 0.14283806085586548, -0.08252478390932083, 0.08629772812128067, -0.0077005671337246895, 0.01252757664769888, 0.06811098754405975, 0.016610760241746902, 0.414411723613739], [-0.26458555459976196, -0.04110325872898102, -0.007998132146894932, 0.11711405962705612, 0.15669041872024536, 0.11718308925628662, 0.19477255642414093, 0.06755083054304123, -0.12009131908416748, -0.0783141478896141, -0.07232658565044403, 0.20932970941066742, -0.01920219697058201, -0.1215653195977211, -0.16808083653450012, -0.17340552806854248, -0.09880013763904572, -0.19847413897514343, -0.16593274474143982, -0.15592288970947266, 0.3164992034435272, -0.0777326375246048, 0.3018619120121002, 0.1339607983827591, 0.040170084685087204, 0.13883142173290253, -0.15385492146015167, 0.15238946676254272, -0.011949167586863041, 0.03381161764264107, 0.2285473346710205, 0.12610407173633575, 0.062020059674978256, 0.04798755422234535, 0.33415061235427856, 0.1393919736146927, 0.011489517986774445, 0.2737945020198822, 0.07878697663545609, -0.21174922585487366, 0.08790597319602966, 0.09463991224765778, 0.01949108950793743, 0.03200313448905945, 0.09669940918684006, -0.14607100188732147, -0.2523043155670166, -0.3244251012802124, 0.010715986602008343, -0.2206909954547882, -0.07184260338544846, -0.12193702906370163, 0.11231464892625809, -0.2726743519306183, -0.12787598371505737, -0.19243767857551575, -0.025059010833501816, -0.10943535715341568, -0.012123160995543003, 0.07640969753265381, 0.2599560022354126, -0.045676715672016144, 0.13128839433193207, 0.2093905657529831], [-0.031228022649884224, 0.08361601084470749, 0.09058461338281631, 0.05759101361036301, -0.021302731707692146, -0.0833895280957222, 0.04094367474317551, -0.21413516998291016, -0.24797487258911133, -0.05074363574385643, -0.1056738942861557, 0.13039439916610718, -0.0743412896990776, 0.11845950782299042, 0.04245728999376297, 0.12993885576725006, -0.2614668309688568, 0.03186238557100296, 0.2266273945569992, -0.01764534041285515, 0.01183344703167677, -0.10493946820497513, 0.261216402053833, 0.20918835699558258, -0.10530578345060349, 0.021528374403715134, -0.3762216567993164, 0.15222735702991486, -0.11848060041666031, 0.1616010218858719, 0.024183515459299088, -0.23729665577411652, -0.11930285394191742, -0.1304226964712143, 0.3647015690803528, 0.0834728479385376, -0.09582080692052841, 0.07481321692466736, -0.3318885564804077, 0.10663426667451859, 0.14764219522476196, 0.24255378544330597, -0.057752810418605804, 0.0211102943867445, -0.11879587173461914, 0.11388686299324036, 0.04939758777618408, -0.1649315357208252, -0.39495304226875305, 0.12082240730524063, 0.2569125294685364, -0.1301877349615097, -0.06301247328519821, -0.03334591165184975, -0.1640896052122116, -0.020521080121397972, 0.10784335434436798, -0.18596993386745453, 0.2644664943218231, -0.12326354533433914, -0.3836953639984131, -0.0791887491941452, 0.024722909554839134, -0.004418644122779369], [0.03305204585194588, 0.11108043044805527, -0.0449381023645401, 0.021077154204249382, -0.09662896394729614, 0.09991398453712463, 0.06146764010190964, -0.13589122891426086, 0.05540086701512337, -0.044343866407871246, -0.04528152197599411, 0.1391131728887558, 0.29224690794944763, 0.15195558965206146, 0.0722736045718193, 0.2270127832889557, -0.2165287286043167, 0.06445712596178055, -0.22191810607910156, -0.12547822296619415, 0.010061665438115597, -0.17771975696086884, 0.17845764756202698, 0.3434295952320099, 0.15045420825481415, -0.07988245785236359, -0.06619128584861755, 0.18118351697921753, 0.0885239839553833, -0.012838025577366352, -0.1138734444975853, -0.13154906034469604, -0.029734548181295395, 0.07657530903816223, 0.10145247727632523, 0.03460658714175224, -0.2512180507183075, -0.051873259246349335, -0.20153667032718658, -0.0720958411693573, 0.06364454329013824, -0.1291714608669281, 0.0902140885591507, 0.07219456136226654, -0.07104402780532837, 0.2963215708732605, -0.18406276404857635, -0.12223419547080994, 0.13259243965148926, -0.060189660638570786, 0.16604083776474, -0.07769115269184113, 0.11085162311792374, -0.03370543569326401, -0.202548086643219, 0.038243964314460754, 0.08758392930030823, -0.07735243439674377, 0.010131278075277805, 0.12068022042512894, -0.1824420690536499, -0.1536339372396469, 0.2221573144197464, 0.10360340774059296], [-0.1533319354057312, -0.07713522017002106, -0.007873643189668655, -0.01233882550150156, 0.1058855652809143, 0.18671287596225739, -0.07041440904140472, 0.043022073805332184, -0.2130168080329895, -0.19624315202236176, 0.005749930161982775, 0.19340458512306213, -0.03830660134553909, 0.06352215260267258, 0.09581239521503448, 0.016511347144842148, -0.045699965208768845, 0.18892361223697662, -0.24237599968910217, -0.16339710354804993, 0.09401613473892212, 0.11181585490703583, 0.2247692495584488, 0.20536945760250092, -0.18591850996017456, -0.14704541862010956, -0.016200905665755272, 0.168283149600029, -0.19004005193710327, 0.15324437618255615, 0.0989217534661293, 0.13426189124584198, 0.2199562042951584, 0.035987548530101776, 0.3327762484550476, 0.13248124718666077, -0.015629976987838745, 0.07582946121692657, 0.012038784101605415, -0.03736690804362297, -0.04922632500529289, -0.07036145031452179, 0.12586015462875366, 0.03059377893805504, 0.07562185823917389, -0.08641892671585083, -0.20190952718257904, -0.1808210015296936, 0.13250632584095, -0.05236753076314926, -0.14084641635417938, 0.06407544761896133, 0.15442965924739838, -0.21741320192813873, -0.13620004057884216, -0.2257264107465744, 0.029833782464265823, -0.1665187031030655, -0.1236257404088974, -0.3456821143627167, -0.22622060775756836, -0.224275603890419, 0.1559118777513504, 0.34648269414901733], [-0.052178457379341125, -0.24861852824687958, 0.09771222621202469, 0.03662003204226494, 0.07456410676240921, -0.16831111907958984, 0.17182375490665436, -0.17034274339675903, 0.09122177213430405, 0.06813518702983856, -0.1382070779800415, 0.19018173217773438, 0.1000356450676918, 0.19557876884937286, 0.2154090702533722, -0.03881175071001053, -0.07952538132667542, 0.013670280575752258, -0.1003560870885849, -0.05431320145726204, 0.025703581050038338, -0.06076265871524811, 0.3457159996032715, 0.1786874532699585, -0.31074580550193787, 0.21261847019195557, -0.014354629442095757, -0.01867775060236454, -0.10482000559568405, -0.019477182999253273, 0.18728069961071014, -0.25142771005630493, -0.2303270697593689, -0.08372799307107925, 0.17251341044902802, -0.1146860346198082, -0.04466557875275612, 0.04901639372110367, -0.2957645058631897, -0.1825355738401413, -0.059866324067115784, 0.07299477607011795, -0.08830317109823227, -0.18155619502067566, -0.115847647190094, -0.038176629692316055, -0.2566361129283905, -0.21789325773715973, -0.014425991103053093, -0.10680709034204483, 0.02958524413406849, 0.19448497891426086, 0.18116699159145355, -0.06162960082292557, 0.006660098675638437, -0.13352416455745697, 0.0861327201128006, -0.1463058590888977, 0.0902201309800148, -0.03947848454117775, -0.1875297874212265, 0.036221012473106384, 0.035116806626319885, 0.047918159514665604], [0.04176601395010948, 0.15896347165107727, -0.21044476330280304, 0.0498778261244297, -0.0007751823868602514, 0.06005961820483208, 0.1324131190776825, -0.05389946699142456, 0.0011035747593268752, -0.029918374493718147, -0.13876873254776, 0.09879259020090103, 0.104393370449543, 0.01304809283465147, 0.20022296905517578, 0.3264317810535431, -0.1777312457561493, -0.01742338202893734, -0.08116549253463745, -0.07366706430912018, 0.09415798634290695, -0.26358795166015625, -0.0571260005235672, 0.02769630402326584, -0.14796799421310425, 0.14153817296028137, -0.1186366081237793, -0.11336539685726166, -0.06418471038341522, 0.2712823450565338, -0.08863995969295502, -0.25225552916526794, -0.02770020253956318, -0.1389034241437912, 0.22152699530124664, -0.2119285613298416, -0.09843487292528152, 0.12523891031742096, -0.022704685106873512, -0.1338898092508316, 0.03024538792669773, 0.017549345269799232, -0.15168675780296326, 0.1989927738904953, -0.17290954291820526, 0.13317908346652985, -0.19881956279277802, -0.19628584384918213, -0.06953690946102142, 0.08825536072254181, -0.008014777675271034, 0.17298422753810883, 0.1759844571352005, -0.1575169861316681, -0.2746049761772156, -0.1901157647371292, 0.18764851987361908, 0.050959449261426926, 0.06918482482433319, -0.11156447231769562, -0.28471681475639343, -0.21028989553451538, 0.19576333463191986, 0.3267560303211212], [-0.09091385453939438, -0.016235440969467163, -0.13570784032344818, -0.0657903254032135, -0.07804030925035477, -0.059803225100040436, 0.15303443372249603, -0.08509093523025513, -0.11033826321363449, -0.03846530616283417, 0.08559980988502502, 0.19206266105175018, 0.05190138518810272, 0.03478838503360748, 0.23793017864227295, 0.07129128277301788, -0.06540723145008087, 0.05096704512834549, -0.024990525096654892, -0.04652760177850723, -0.0854983776807785, -0.1287446767091751, 0.05046085640788078, 0.17995576560497284, -0.2323846071958542, 0.07556986808776855, -0.13730156421661377, -0.25544795393943787, 0.03429409861564636, 0.07471629977226257, 0.02484755404293537, -0.29917091131210327, -0.20955966413021088, -0.07772403955459595, 0.2884354889392853, -0.23715493083000183, -0.0022792054805904627, 0.06457361578941345, -0.28137725591659546, -0.26847130060195923, -0.01003950834274292, -0.09734584391117096, -0.12727560102939606, 0.2395622730255127, -0.2027566283941269, 0.04990645498037338, -0.10839953273534775, -0.27815183997154236, -0.010618883185088634, 0.08029157668352127, -0.1494988650083542, 0.06375353783369064, 0.27730461955070496, 0.0169992633163929, -0.1377345323562622, -0.21149978041648865, 0.15309898555278778, -0.14679290354251862, 0.27036672830581665, -0.2486410290002823, -0.37651243805885315, 0.011103421449661255, 0.08915352076292038, 0.09741581976413727], [0.2377823293209076, 0.04258362203836441, -0.22200311720371246, -0.04856090620160103, 0.04104117304086685, 0.05031257122755051, 0.10080157965421677, -0.1317167580127716, -0.04586608707904816, -0.16449281573295593, -0.10732343047857285, 0.1717323660850525, 0.22001653909683228, 0.20357367396354675, 0.32882818579673767, 0.06403845548629761, -0.19528023898601532, 0.07016027718782425, 0.08118598163127899, -0.06150314584374428, -0.010973094031214714, -0.07558952271938324, 0.12269463390111923, 0.21848934888839722, -0.1364365667104721, 0.10854211449623108, -0.11568456888198853, 0.16415850818157196, 0.03445214033126831, 0.11292576044797897, 0.06887554377317429, -0.349826842546463, 0.1956494152545929, -0.03159995749592781, 0.2634492814540863, -0.12656496465206146, -0.06168290972709656, 0.07302137464284897, -0.039308808743953705, -0.25774040818214417, 0.04100564867258072, 0.013549895957112312, 0.2152310311794281, 0.062206681817770004, 0.08496342599391937, 0.12309976667165756, 0.0018712058663368225, -0.22916767001152039, 0.1115126833319664, 0.16160458326339722, 0.17028465867042542, 0.20819231867790222, 0.008264521136879921, -0.18972069025039673, 0.04339755326509476, -0.0642881914973259, 0.13446879386901855, 0.23128120601177216, 0.13351796567440033, -0.03622369468212128, -0.32924529910087585, 0.03043494187295437, 0.17238016426563263, 0.05795443803071976], [0.1389545500278473, -0.050370652228593826, 0.05063403770327568, 0.2898942828178406, -0.05400876700878143, 0.028191380202770233, 0.2193179875612259, -0.08460216224193573, -0.08600075542926788, 0.09000162035226822, -0.05452736094594002, 0.09808436036109924, 0.16200582683086395, 0.1415719985961914, 0.14149197936058044, 0.10761134326457977, -0.16416184604167938, 0.10501708090305328, -0.11607572436332703, -0.13844801485538483, -0.10433278232812881, -0.16727006435394287, 0.18269133567810059, 0.10755940526723862, -0.08913995325565338, -0.16866132616996765, 0.21274642646312714, 0.15492668747901917, 0.14977669715881348, 0.09480253607034683, -0.02922825887799263, -0.04822687804698944, 0.09481453150510788, -0.04766285791993141, 0.057951703667640686, 0.1635589748620987, 0.0532602034509182, -0.2504062354564667, -0.23063147068023682, -0.06545118987560272, -0.326021671295166, -0.05543142557144165, -0.2237127274274826, -0.008913624100387096, -0.2587977945804596, -0.13876581192016602, -0.22112245857715607, -0.04958181455731392, -0.317822128534317, -0.11319491267204285, -0.12389164417982101, -0.20062090456485748, 0.22470054030418396, 0.17718395590782166, -0.1352216899394989, -0.1924288272857666, 0.31961092352867126, -0.1567985862493515, -0.12118888646364212, 0.12522518634796143, -0.01661727949976921, -0.1311965137720108, 0.1603357046842575, 0.2199317216873169], [-0.009495901875197887, -0.018436430022120476, 0.04608917981386185, 0.1320793628692627, -0.271481454372406, -0.004513352178037167, 0.19690576195716858, -0.012543082237243652, -0.03540387749671936, 0.1743669956922531, 0.008826125413179398, 0.4088931977748871, 0.09490685164928436, 0.20198851823806763, -0.013209608383476734, 0.0046822139993309975, -0.19146130979061127, 0.007642596960067749, -0.1573907434940338, -0.2406689077615738, 0.15705937147140503, -0.1634506732225418, 0.15207511186599731, -0.02916591428220272, -0.0380757674574852, 0.04640112444758415, -0.24114590883255005, 0.19048050045967102, -0.1208638921380043, -0.036534570157527924, -0.25548261404037476, -0.03695468604564667, -0.13705860078334808, 0.003839017590507865, 0.2749098837375641, 0.14096379280090332, 0.17198652029037476, 0.1633610874414444, -0.2024805098772049, 0.14172405004501343, 0.21240192651748657, 0.060733016580343246, 0.26367661356925964, 0.014316839165985584, -0.1051609143614769, 0.2462622970342636, -0.24966192245483398, -0.06660565733909607, -0.14334553480148315, -0.11191258579492569, -0.002203247742727399, 0.05643909052014351, 0.12682247161865234, 0.03432625159621239, -0.2648921608924866, -0.0389791876077652, 0.01019324455410242, -0.20200194418430328, -0.05443713814020157, 0.13121101260185242, -0.3378034234046936, -0.11311397701501846, -0.0776393860578537, 0.11474958062171936], [0.17879462242126465, -0.15300533175468445, 0.11479421705007553, 0.17302551865577698, -0.046701978892087936, 0.1286408007144928, 0.17177815735340118, -0.20491662621498108, 0.014116581529378891, 0.1025516465306282, 0.06394895911216736, 0.1348363310098648, 0.07654285430908203, -0.03123539499938488, 0.12581326067447662, -0.05445553734898567, -0.2850967049598694, -0.0100412517786026, -0.16442883014678955, -0.027161888778209686, -0.03774040937423706, -0.004871407989412546, 0.16773845255374908, -0.0167074091732502, -0.06673258543014526, 0.07420749962329865, -0.2017897367477417, 0.1117318794131279, 0.12942801415920258, -0.061905451118946075, 0.1354023516178131, 0.07706845551729202, 0.011046093888580799, -0.04054435342550278, 0.2654733657836914, -0.03391075134277344, 0.11667381972074509, 0.07628397643566132, -0.14864718914031982, -0.0861625224351883, -0.06423071771860123, -0.047982990741729736, 0.048062846064567566, -0.11154832690954208, -0.11583947390317917, -0.18794547021389008, 0.02431468293070793, -0.1479969322681427, -0.08734005689620972, 0.04137926176190376, -0.07024091482162476, -0.159056156873703, 0.12532949447631836, -0.15861466526985168, -0.024711742997169495, -0.2496500164270401, 0.049186352640390396, -0.05853644758462906, 0.0202472023665905, -0.14253993332386017, -0.2442501038312912, -0.026222633197903633, 0.032450590282678604, 0.128490149974823], [-0.06734434515237808, -0.05581909418106079, -0.046815015375614166, -0.09336194396018982, 0.025485828518867493, -0.12531884014606476, 0.022233692929148674, -0.11359135061502457, -0.1354379951953888, -0.012364223599433899, 0.03180399164557457, 0.30469557642936707, 0.15446680784225464, -0.0013485219096764922, 0.25424835085868835, 0.1814458817243576, -0.39000043272972107, 0.033130429685115814, -0.08146887272596359, -0.09563307464122772, -0.038396794348955154, -0.010605721734464169, 0.036269597709178925, 0.22567054629325867, -0.28941988945007324, -0.011950081214308739, -0.018838847056031227, 0.014210027642548084, -0.03367426618933678, -0.032402124255895615, 0.004317257087677717, -0.197872593998909, -0.21853642165660858, -0.014703692868351936, 0.12062246352434158, 0.05785504728555679, 0.17185339331626892, -0.047886621206998825, -0.22785253822803497, -0.10391385108232498, 0.22352924942970276, 0.0663442388176918, 0.0007198767852969468, 0.17625443637371063, -0.21108272671699524, -0.027996528893709183, -0.2351927012205124, -0.0772039070725441, 0.040582820773124695, -0.020882481709122658, -0.14372749626636505, -0.03075910359621048, 0.2679920196533203, -0.05609539523720741, -0.16641363501548767, -0.05944874882698059, 0.0875365361571312, -0.0634826272726059, -0.013851058669388294, -0.13635121285915375, -0.2966983914375305, -0.1547303944826126, 0.052671708166599274, 0.11090610921382904], [-0.04363667964935303, -0.06472006440162659, 0.1682547777891159, 0.07811709493398666, -0.19320999085903168, -0.10666418820619583, 0.08910178393125534, 0.02349206618964672, -0.15839645266532898, -0.07795770466327667, 0.015316368080675602, 0.33927664160728455, 0.07790571451187134, 0.039468467235565186, 0.0037077879533171654, 0.09584028273820877, -0.2780679166316986, 0.009605185128748417, -0.07749027013778687, -0.05615493282675743, 0.06109454482793808, -0.18837934732437134, 0.00875744316726923, 0.034226950258016586, -0.27224600315093994, -0.2332429736852646, 0.0011660746531561017, 0.1595822274684906, -0.037454765290021896, 0.13353656232357025, -0.20294946432113647, 0.09842811524868011, -0.1336251199245453, 0.009639917872846127, 0.034104350954294205, -0.021778076887130737, 0.023770824074745178, -0.02682516723871231, -0.11980791389942169, -0.03581679239869118, -0.024961154907941818, -0.09714791178703308, 0.16209149360656738, 0.010220494121313095, -0.04413264989852905, 0.06596715748310089, -0.1353551745414734, 0.0647057518362999, -0.05010410025715828, -0.13356146216392517, -0.1278148889541626, -0.019611775875091553, 0.08400045335292816, 0.0673036053776741, 0.014333708211779594, -0.19890576601028442, 0.14693069458007812, -0.24813686311244965, 0.08457190543413162, -0.03672710806131363, -0.29645663499832153, -0.061800580471754074, -0.09549178928136826, 0.12009770423173904], [0.18266081809997559, -0.13158150017261505, -0.22246840596199036, 0.16085907816886902, -0.05988174304366112, 0.1764327436685562, 0.0638282373547554, 0.06092589721083641, -0.39028507471084595, -0.0788784772157669, -0.08527041226625443, 0.31307360529899597, 0.010909241624176502, 0.3650209903717041, 0.20192821323871613, -0.16022644937038422, -0.3030199110507965, -0.015602256171405315, -0.038285743445158005, 0.22962981462478638, -0.10644201189279556, 0.057535041123628616, 0.25137996673583984, 0.4251067042350769, -0.23076818883419037, 0.07762181013822556, -0.10613901913166046, -0.0632367953658104, -0.15702293813228607, 0.08187108486890793, 0.17841096222400665, 0.12552930414676666, -0.13099408149719238, -0.017625512555241585, 0.25269362330436707, 0.16796645522117615, 0.2500435709953308, -0.023493925109505653, 0.27439624071121216, -0.16587866842746735, -0.14071208238601685, 0.1394895613193512, 0.0769025906920433, -0.018245059996843338, -0.11529509723186493, -0.07472208887338638, -0.34241172671318054, -0.43851524591445923, 0.03606868535280228, -0.07759858667850494, -0.20884662866592407, 0.014007281512022018, 0.5843282341957092, -0.19233942031860352, -0.23467566072940826, -0.07992611825466156, 0.10546088218688965, -0.10713697969913483, -0.05876549333333969, -0.265139639377594, -0.13946934044361115, 0.020595207810401917, 0.3645049035549164, 0.5543628931045532], [0.0670199990272522, 0.007672999985516071, 0.028910115361213684, 0.08462400734424591, -0.01758405566215515, -0.04347323253750801, 0.0783209502696991, -0.16440635919570923, -0.1311209797859192, -0.00011752552381949499, 0.04094028100371361, 0.12590569257736206, 0.222400963306427, 0.04918379336595535, 0.08819375187158585, 0.1487015336751938, -0.19224202632904053, 0.06429463624954224, -0.04877147451043129, -0.05352015048265457, -0.014350012876093388, -0.2325502336025238, 0.06559427082538605, 0.11130353063344955, -0.039472296833992004, 0.010743177495896816, -0.07246501743793488, -0.018735678866505623, 0.1671181470155716, -0.036201633512973785, -0.008900649845600128, -0.044914357364177704, 0.013914150185883045, -0.05112290754914284, 0.21722474694252014, 0.007909047417342663, 0.002068044850602746, 0.020209500566124916, -0.25190970301628113, -0.17257587611675262, 0.014150644652545452, -0.03261779993772507, 0.041172195225954056, 0.1113855391740799, -0.04025277495384216, -0.050277166068553925, -0.1578110307455063, -0.10208997130393982, -0.06158451735973358, 0.02649008296430111, -0.14798575639724731, -0.13361702859401703, 0.11856899410486221, -0.02737838216125965, -0.1271304488182068, -0.07918219268321991, 0.17139311134815216, -0.2089061439037323, 0.07645656913518906, -0.00031880856840871274, -0.19693772494792938, -0.21940359473228455, 0.05221359059214592, 0.0829378291964531], [-0.1010977029800415, -0.025740664452314377, 0.07515482604503632, -0.1586504578590393, -0.012638694606721401, 0.13764899969100952, 0.1894759088754654, -0.12440591305494308, -0.2856954336166382, -0.03819097578525543, -0.14803341031074524, 0.013349780812859535, 0.1694754660129547, -0.009114746935665607, 0.0743141621351242, 0.1568034142255783, -0.2656581699848175, 0.13108564913272858, -0.1035303920507431, -0.11438394337892532, 0.05093533545732498, -0.3077494502067566, 0.11989668756723404, 0.4662691652774811, -0.43560561537742615, 0.07817226648330688, -0.2076880931854248, 0.2478533834218979, -0.03508836403489113, 0.1646917164325714, -0.07740776985883713, -0.009517412632703781, -0.07489538192749023, -0.09492054581642151, 0.19050078094005585, 0.013284847140312195, 0.055708374828100204, 0.0562162920832634, -0.12711744010448456, -0.33046504855155945, 0.042706429958343506, -0.013743964955210686, 0.14397963881492615, 0.18603098392486572, 0.02134409360587597, -0.18924181163311005, -0.1168738603591919, -0.4470694065093994, 0.022687215358018875, -0.06392655521631241, -0.2869319021701813, -0.03937515243887901, 0.2545470595359802, -0.013683809898793697, -0.28768905997276306, -0.2666042447090149, -0.023704737424850464, -0.18882708251476288, -0.11968977749347687, -0.2616320550441742, 0.05844073370099068, -0.06338867545127869, 0.0418853722512722, 0.30059683322906494], [0.0013403635239228606, -0.2328437715768814, -0.1256156861782074, 0.2092101275920868, -0.042775433510541916, 0.2022256851196289, -0.005967143457382917, 0.04144227132201195, -0.17048750817775726, 0.01521280687302351, -0.02125687710940838, 0.24095091223716736, 0.17830327153205872, 0.06094937026500702, 0.20668819546699524, 0.10073608160018921, -0.2772219777107239, -0.22103074193000793, -0.02402404695749283, 0.035940758883953094, -0.12390504032373428, 0.08204486966133118, 0.2961723208427429, 0.27427464723587036, -0.19458049535751343, -0.008806157857179642, -0.06218603625893593, -0.0014170146314427257, -0.034749310463666916, 0.049516599625349045, 0.19952484965324402, -0.01658536307513714, -0.14027829468250275, 0.018762502819299698, 0.23188206553459167, 0.010398158803582191, 0.0529434010386467, 0.24486379325389862, -0.0011191420489922166, -0.18825151026248932, 0.020714815706014633, -0.12357043474912643, 0.03225256875157356, 0.027970412746071815, -0.059073738753795624, -0.1616818606853485, -0.18660305440425873, -0.1372339278459549, -0.13506895303726196, -0.22186000645160675, -0.08805350214242935, -0.02235637977719307, 0.17397540807724, -0.15813402831554413, -0.1650165319442749, -0.0733553096652031, 0.21652314066886902, -0.09796933084726334, 0.02652355097234249, -0.0533183291554451, 0.006245628464967012, -0.08194950968027115, 0.12493184208869934, 0.1700933426618576], [-0.06019346043467522, 0.016761932522058487, -0.074613057076931, 0.00047878624172881246, -0.17848099768161774, 0.08907200396060944, 0.14348432421684265, -0.0539126917719841, -0.04916435480117798, -0.07798698544502258, -0.15438993275165558, 0.12156181782484055, 0.18134064972400665, 0.07533634454011917, 0.18730950355529785, 0.17408029735088348, -0.2778203785419464, -0.02474038302898407, 0.08033892512321472, -0.01624435931444168, -0.10548225045204163, -0.0993933454155922, 0.13697074353694916, 0.21234655380249023, -0.16779300570487976, 0.021476324647665024, -0.2908155918121338, 0.059971194714307785, 0.1484934687614441, -0.018159881234169006, 0.0417603924870491, -0.13627611100673676, -0.13078829646110535, 0.042106743901968, 0.3077343702316284, 0.005978222005069256, 0.02162071131169796, 0.07878033816814423, -0.3358387351036072, -0.1886608749628067, -0.008523424156010151, 0.1448211669921875, -0.07999420911073685, -0.09639577567577362, -0.01599006913602352, 0.014200669713318348, -0.11623719334602356, -0.19048336148262024, -0.04663616046309471, -0.010978278703987598, -0.13369449973106384, 0.036653947085142136, 0.06380873173475266, -0.057347673922777176, -0.03391099348664284, -0.07998264580965042, 0.1633257269859314, -0.08734990656375885, -0.050000742077827454, -0.11704634875059128, -0.09478443115949631, -0.06053420156240463, 0.10635488480329514, 0.1051010861992836], [0.04004984721541405, -0.019534990191459656, -0.1693892925977707, 0.10262005031108856, -0.04254071041941643, 0.08443225920200348, -0.043388232588768005, -0.15298305451869965, -0.08228287100791931, -0.04997061565518379, -0.043938059359788895, 0.3106965124607086, 0.24656566977500916, -0.021163661032915115, -0.27955007553100586, 0.21462590992450714, -0.21777032315731049, -0.11137929558753967, -0.3819841146469116, -0.09195396304130554, -0.07057899236679077, -0.15401598811149597, 0.1430400311946869, 0.3394753038883209, -0.2148762047290802, 0.0677165687084198, -0.13498513400554657, 0.02632409892976284, -0.05772605165839195, 0.009274659678339958, 0.29059645533561707, 0.010183447040617466, -0.13460493087768555, -0.027910010889172554, 0.24016661942005157, -0.01741115190088749, -0.0413060337305069, 0.1517435908317566, -0.22841958701610565, -0.06108406186103821, 0.045891087502241135, -0.02947678044438362, 0.061950065195560455, 0.014891366474330425, 0.10597949475049973, 0.1786840260028839, -0.036831729114055634, -0.08133449405431747, -0.12382811307907104, 0.10062963515520096, -0.035485509783029556, -0.034789446741342545, 0.1050967201590538, -0.13322903215885162, -0.16855739057064056, -0.07102929055690765, 0.21543435752391815, -0.16525860130786896, -0.09973571449518204, 0.03847777843475342, -0.12319876998662949, 0.006528433877974749, 0.04894351214170456, 0.14531074464321136], [0.0972166508436203, 0.11923587322235107, 0.09657693654298782, 0.16376766562461853, -0.09648659080266953, 0.15861199796199799, 0.06671128422021866, 0.020530210807919502, -0.098849356174469, -0.038825176656246185, -0.07800153642892838, -0.02177499420940876, 0.1398528814315796, -0.023269113153219223, -0.046836476773023605, 0.15361513197422028, -0.18946175277233124, 0.02050611935555935, -0.175556018948555, 0.05278804525732994, -0.14477196335792542, -0.2993257939815521, 0.08953092247247696, 0.007363593205809593, 0.03897855430841446, 0.008770574815571308, -0.09797590225934982, 0.07024259865283966, 0.02174670435488224, 0.13483482599258423, -0.15498343110084534, -0.044579144567251205, -0.18281711637973785, 0.011270859278738499, 0.18365249037742615, -0.021883010864257812, -0.1695265769958496, -0.07662426680326462, -0.033518146723508835, -0.1909305453300476, 0.18876972794532776, 0.10958241671323776, 0.3069610297679901, 0.19410212337970734, -0.1515495926141739, 0.020060278475284576, -0.10803879052400589, -0.1260412335395813, -0.013482836075127125, 0.07889091968536377, 0.14662614464759827, 0.11083374172449112, 0.09545157849788666, 0.0578785240650177, -0.12343621999025345, -0.14977318048477173, 0.13264891505241394, -0.07913863658905029, 0.01847623661160469, -0.04147873446345329, -0.08874198794364929, -0.1409774273633957, -0.25012627243995667, 0.11330170184373856], [-0.07609493285417557, 0.19779884815216064, -0.06226447969675064, 0.17757964134216309, -0.02362222969532013, 0.03596055135130882, 0.057657886296510696, 0.02600201405584812, -0.013915643095970154, -0.01806233637034893, -0.008553784340620041, 0.19144988059997559, 0.17403094470500946, 0.05080906301736832, 0.14859063923358917, 0.1689772754907608, -0.1368103176355362, -0.14347656071186066, -0.11854395270347595, -0.02780892886221409, 0.02199999988079071, -0.023797238245606422, 0.22549106180667877, 0.20568357408046722, -0.05512082949280739, 0.002562132431194186, 0.006883957888931036, 0.23026320338249207, 0.0019167480058968067, 0.16625648736953735, 0.0925660952925682, -0.1823846399784088, -0.031522754579782486, -0.06088314577937126, 0.30459704995155334, 0.03693502023816109, -0.13119085133075714, 0.1063496544957161, -0.15956252813339233, -0.08193421363830566, 0.04222681373357773, -0.0006574969156645238, 0.21209479868412018, 0.0732189416885376, -0.04928391054272652, 0.16987936198711395, -0.17311479151248932, -0.14203625917434692, -0.011316217482089996, -0.05133242905139923, 0.0312761589884758, 0.09700070321559906, -0.059735607355833054, -0.13808953762054443, -0.1005503460764885, -0.23309077322483063, 0.03321511670947075, -0.09592271596193314, 0.08709666132926941, 0.08099892735481262, -0.12312977015972137, -0.3195800185203552, 0.12746751308441162, 0.13750573992729187], [-0.05804399773478508, -0.03270941972732544, -0.10151007026433945, 0.05047156289219856, 0.1838805079460144, 0.2394588589668274, 0.14496557414531708, 0.06547098606824875, -0.10143245756626129, -0.026844557374715805, 0.06569874286651611, 0.17557480931282043, 0.13509105145931244, 0.07514679431915283, 0.10395476222038269, 0.23599469661712646, -0.271797776222229, -0.05957783758640289, -0.12844303250312805, -0.0028586857952177525, -0.18663440644741058, -0.05364483222365379, 0.08975108712911606, 0.10762646049261093, -0.16828733682632446, 0.04321650415658951, -0.20305509865283966, -0.11829792708158493, 0.061567507684230804, -0.10748479515314102, 0.030900269746780396, -0.14624592661857605, -0.047397349029779434, 0.010598324239253998, 0.12251275777816772, 0.1444888412952423, -0.08047729730606079, 0.19853341579437256, -0.08025845140218735, -0.07128538936376572, -0.12713760137557983, -0.024539409205317497, -0.09011894464492798, -0.13382679224014282, -0.008584770374000072, -0.06946039944887161, -0.07495254278182983, -0.1267293393611908, 0.024630917236208916, 0.08585777133703232, -0.10032752901315689, -0.05030716583132744, 0.1379493921995163, -0.08075814694166183, 0.03639523312449455, 0.002065482782199979, 0.177656888961792, -0.001346340635791421, -0.02228640206158161, -0.1935221403837204, -0.09221186488866806, -0.08223120868206024, 0.16245484352111816, 0.17976853251457214], [-0.2397807240486145, -0.08766398578882217, -0.26440441608428955, -0.06269432604312897, -0.03412478044629097, 0.0602482408285141, 0.17057736217975616, -0.04880359023809433, 0.1666562855243683, -0.046712927520275116, 0.1533069908618927, 0.1165449395775795, 0.11184962838888168, 0.08673975616693497, 0.234585702419281, 0.02362247370183468, 0.009480844251811504, -0.023161081597208977, -0.15533918142318726, 0.040031127631664276, -0.00987130869179964, -0.1693253368139267, 0.04885273426771164, 0.0454183965921402, -0.04797326400876045, -0.04186512157320976, 0.008543990552425385, -0.08006163686513901, 0.005833467468619347, 0.0669785588979721, 0.1816445142030716, -0.005401174537837505, -0.11129014939069748, -0.1377621591091156, 0.2136593759059906, 0.009359275922179222, -0.06794484704732895, 0.32974424958229065, -0.09495317935943604, 0.05605720728635788, 0.10611876845359802, -0.03999675437808037, 0.0003905431367456913, -0.24606117606163025, 0.1210065484046936, -0.1962522268295288, -0.13321003317832947, -0.13254578411579132, 0.1900234967470169, 0.09773876518011093, -0.027975937351584435, -0.011818194761872292, 0.2975446581840515, -0.5011194348335266, -0.08352995663881302, -0.03768705204129219, 0.23368069529533386, -0.14713223278522491, 0.23185469210147858, -0.02686098963022232, 0.021316075697541237, -0.38244470953941345, 0.2827922999858856, 0.26082006096839905], [-0.212953582406044, -0.04472103342413902, -0.0649828091263771, 0.04999762773513794, -0.03998199850320816, -0.05283975228667259, 0.08439326286315918, -0.04455973953008652, 0.027910828590393066, -0.03990079462528229, 0.09693785756826401, 0.2577950358390808, 0.11285215616226196, 0.059982605278491974, 0.12938331067562103, 0.11884361505508423, -0.17963950335979462, -0.1340145319700241, -0.100489042699337, -0.11500626802444458, -0.0016404918860644102, 0.05408288910984993, 0.16710788011550903, 0.324685662984848, -0.23976781964302063, -0.11142421513795853, 0.11155960708856583, -0.08118599653244019, -0.013507071882486343, 0.00937061756849289, 0.2358372062444687, -0.23533830046653748, -0.09922391176223755, -0.1836436241865158, 0.12165160477161407, 0.2600163519382477, 0.005485442467033863, 0.19998659193515778, -0.15433768928050995, 0.0318460538983345, 0.00994351226836443, -0.2890142798423767, 0.08903783559799194, -0.020139023661613464, 0.07181816548109055, -0.06220221891999245, -0.08138465881347656, -0.20683491230010986, -0.06465420126914978, 0.07454707473516464, 0.02889115922152996, 0.056797534227371216, 0.10987463593482971, 0.009437862783670425, -0.10129932314157486, -0.019191015511751175, 0.07782711088657379, -0.21631792187690735, 0.05007419362664223, 0.07677862793207169, -0.13053637742996216, -0.13294096291065216, 0.3000848591327667, 0.17042241990566254], [0.27718114852905273, -0.15119391679763794, 0.10138166695833206, 0.3504501283168793, -0.22872768342494965, 0.1166902482509613, 0.30806463956832886, -0.15662388503551483, 0.1934145838022232, 0.052108556032180786, 0.3347374498844147, 0.21077851951122284, 0.306442528963089, -0.059056513011455536, 0.22943437099456787, 0.03200574591755867, -0.29734006524086, 0.10017004609107971, -0.021672625094652176, -0.19070833921432495, -0.1740526258945465, 0.022768257185816765, -0.006321270484477282, -0.11252373456954956, -0.12065913528203964, -0.20298275351524353, -0.08889885246753693, 0.06841732561588287, 0.18543802201747894, 0.16614000499248505, 0.18313990533351898, 0.09666912257671356, -0.04309318587183952, -0.0538599006831646, 0.07166646420955658, 0.000161944524734281, -0.030097488313913345, 0.14769336581230164, -0.16479124128818512, -0.12256071716547012, -0.08780432492494583, -0.16979816555976868, -0.03727300465106964, 0.06535585969686508, -0.10993875563144684, -0.09684258699417114, 0.03409422188997269, -0.26883402466773987, -0.12752997875213623, -0.05010692775249481, -0.012011009268462658, -0.2517140805721283, -0.06702955812215805, 0.08365499973297119, 0.14591170847415924, -0.1917407512664795, 0.28063729405403137, 0.023330749943852425, 0.14383380115032196, -0.09775006026029587, -0.3212008774280548, 0.23757068812847137, -0.006465580780059099, 0.0038704732432961464], [0.05378441885113716, -0.1616518795490265, -0.05920662730932236, 0.021221956238150597, -0.04589489847421646, 0.03677114099264145, -0.018007425591349602, 0.05182410404086113, 0.1115846335887909, -0.22185327112674713, 0.049447812139987946, 0.1466292440891266, 0.1336841732263565, -0.017249053344130516, 0.27836525440216064, 0.06639966368675232, -0.14468488097190857, -0.02668697014451027, 0.04059639573097229, -0.10858403891324997, 0.01539393700659275, -0.14216755330562592, 0.23660720884799957, -0.004680448677390814, 0.008066119626164436, -0.03413286432623863, 0.047712646424770355, 0.01365775614976883, 0.0898967981338501, 0.03586222603917122, 0.14458461105823517, -0.068661168217659, -0.08208006620407104, 0.010254275985062122, 0.24252034723758698, 0.17982031404972076, -0.10638099163770676, -0.031204648315906525, -0.22373226284980774, -0.1174367368221283, 0.048171184957027435, -0.0826307088136673, 0.08662115037441254, 0.0029378363396972418, 0.0595131516456604, 0.03456423804163933, -0.20128630101680756, -0.07475445419549942, -0.011430644430220127, -0.07565563172101974, 0.1051521748304367, -0.05481859669089317, 0.17166940867900848, 0.0074721272103488445, -0.1731906235218048, -0.09614348411560059, 0.12056604772806168, -0.34098929166793823, 0.07554896175861359, 0.11384781450033188, -0.1155526340007782, -0.29835566878318787, 0.0948135033249855, 0.10485594719648361], [0.10034243762493134, -0.10762919485569, -0.10943251103162766, 0.0689997673034668, -0.06766945123672485, 0.11077051609754562, 0.050308216363191605, -0.05580379068851471, -0.2571294903755188, -0.006285299081355333, -0.09557808935642242, 0.17479407787322998, 0.14181815087795258, 0.09089817851781845, 0.02779148332774639, 0.06091156229376793, -0.3274281919002533, 0.13331109285354614, 0.08647871762514114, -0.06115018576383591, -0.0003661663504317403, -0.20386023819446564, 0.19965890049934387, 0.3116000294685364, -0.1348702758550644, 0.05547460541129112, -0.20426544547080994, -0.006037991493940353, 0.08523658663034439, 0.16726140677928925, 0.051546577364206314, -0.0851082056760788, -0.03509256988763809, 0.047301556915044785, 0.28474462032318115, -0.08536306768655777, -0.07280135154724121, 0.05476502701640129, -0.20545662939548492, -0.20929966866970062, 0.0032986446749418974, -0.17187047004699707, -0.025045225396752357, 0.1986112892627716, 0.015266506001353264, -0.04316750541329384, -0.09456360340118408, -0.25167086720466614, -0.12952429056167603, 0.0527273528277874, 0.010568610392510891, 0.039295367896556854, 0.06308230012655258, -0.06274403631687164, -0.1365799456834793, -0.03311146795749664, 0.1312987506389618, -0.042093679308891296, 0.06136319041252136, -0.1700175553560257, 0.03926105052232742, 0.2576109766960144, 0.13666903972625732, 0.15672080218791962], [0.04348978400230408, 0.21475565433502197, -0.026313157752156258, -0.04122995212674141, -0.34244978427886963, -0.16931180655956268, -0.10071557015180588, -0.021496320143342018, 0.14154262840747833, -0.01404848787933588, -0.10190756618976593, 0.03342822566628456, 0.30499500036239624, -0.04701703414320946, 0.10975654423236847, 0.07374957948923111, -0.4080196022987366, 0.07481580227613449, 0.013925154693424702, -0.03396805748343468, -0.032790519297122955, -0.20858317613601685, 0.057647813111543655, 0.1734868288040161, -0.25549978017807007, -0.1434614658355713, 0.03038608655333519, 0.24651439487934113, 0.19257843494415283, 0.025282301008701324, 0.011871996335685253, -0.07776214182376862, -0.2729823589324951, 0.11233505606651306, 0.15127097070217133, 0.2043834775686264, 0.2315458357334137, -0.05607878044247627, -0.22883333265781403, 0.007020711898803711, 0.1191151887178421, 0.1554938554763794, 0.2667021155357361, 0.09495007991790771, -0.20045216381549835, 0.1661001741886139, -0.2506583034992218, 0.09117879718542099, -0.04647499695420265, 0.03794889152050018, -0.13447748124599457, -0.0375114381313324, 0.07770125567913055, 0.02605435810983181, -0.12590941786766052, -0.17199432849884033, 0.13828589022159576, -0.28400522470474243, -0.13741496205329895, 0.14588165283203125, -0.4114677309989929, -0.12306644767522812, 0.233860045671463, 0.10269556939601898], [-0.11746890842914581, 0.08977283537387848, -0.09049100428819656, -0.024681707844138145, 0.07604967057704926, -0.004368956666439772, 0.0228753462433815, -0.08774062246084213, 0.06039150804281235, 0.07899390161037445, 0.07751123607158661, 0.11641699075698853, 0.15345588326454163, 0.13863635063171387, 0.21715448796749115, 0.06495869159698486, -0.26724034547805786, 0.18369270861148834, -0.20202966034412384, -0.03513249009847641, 0.09989101439714432, -0.1646592617034912, 0.11000525206327438, 0.24248217046260834, 0.0023198004346340895, -0.02811024710536003, -0.09056338667869568, 0.21892401576042175, 0.11272739619016647, -0.009479086846113205, 0.11547482758760452, -0.06201324984431267, 0.09498750418424606, 0.04073712229728699, 0.2920404076576233, 0.06627945601940155, -0.11261487007141113, -0.11933428794145584, -0.3232990801334381, -0.13120129704475403, -0.0455622561275959, 0.1608498990535736, -0.09809110313653946, -0.15324750542640686, -0.08182086795568466, -0.02968267910182476, -0.11741849035024643, -0.09959414601325989, 0.03933025896549225, 0.0024006615858525038, 0.15022067725658417, -0.016218014061450958, 0.20014578104019165, -0.12142003327608109, -0.08785281330347061, 0.09621913731098175, 0.20092591643333435, 0.06793566793203354, -0.01620510034263134, 0.0007514347671531141, 0.021385816857218742, -0.021122140809893608, 0.078621506690979, 0.1923574060201645], [-0.04714880883693695, -0.05859909579157829, -0.021725041791796684, 0.027389077469706535, -0.08655185997486115, 0.12338471412658691, 0.13643811643123627, 0.016777820885181427, -0.1155892163515091, 0.044843558222055435, -0.09101637452840805, 0.21819521486759186, 0.11679784208536148, 0.09540268033742905, 0.08573947846889496, 0.11284414678812027, -0.08158581703901291, 0.09259744733572006, -0.057719044387340546, -0.14002759754657745, -0.0661165714263916, -0.10343890637159348, 0.19945180416107178, 0.2623216509819031, -0.1150948703289032, -0.022054068744182587, -0.11333463340997696, 0.03698328137397766, 0.1179373636841774, 0.21738621592521667, 0.006672108545899391, -0.190997913479805, 0.0792309120297432, 0.017607955262064934, 0.1938452124595642, -0.027873413637280464, -0.002142219338566065, 0.05104019120335579, -0.2633182406425476, -0.042128752917051315, 0.01622857339680195, -0.08451170474290848, -0.006602284032851458, 0.1188996434211731, 0.003273103153333068, -0.09462498873472214, -0.07613380253314972, -0.23139289021492004, -0.19670327007770538, 0.047193992882966995, -0.023510096594691277, -0.028358791023492813, -0.019811829552054405, 0.015954505652189255, -0.10885784775018692, -0.1111840233206749, 0.17424850165843964, -0.11469515413045883, 0.12769192457199097, -0.13315950334072113, -0.24142110347747803, 0.056560661643743515, 0.06775033473968506, 0.17527726292610168], [0.05666298791766167, 0.02129414677619934, 0.07560794800519943, 0.29574158787727356, 0.0430854968726635, 0.021347709000110626, 0.10715001076459885, -0.20839044451713562, -0.07294082641601562, 0.07785426080226898, -0.11263944953680038, -0.20462089776992798, 0.15706801414489746, -0.0913233608007431, 0.4973873198032379, 0.00894753448665142, -0.1651204228401184, 0.032292935997247696, -0.0682157576084137, -0.02963920496404171, 0.1428852528333664, 0.05957235395908356, 0.05455829203128815, 0.25259149074554443, -0.06275196373462677, 0.21600431203842163, -0.16797837615013123, 0.06548800319433212, -0.019289778545498848, -0.02704295702278614, -0.13661901652812958, -0.24709926545619965, -0.16837738454341888, -0.12731021642684937, 0.2364991158246994, 0.4659023582935333, 0.05719189718365669, -0.10380628705024719, 0.00827390793710947, -0.09242966026067734, 0.08195964992046356, 0.04878329113125801, 0.05439910292625427, 0.043746378272771835, -0.20689280331134796, -0.02013210579752922, -0.10857460647821426, -0.31515201926231384, -0.06279276311397552, 0.11512885242700577, 0.020229600369930267, 0.10256672650575638, 0.2214268296957016, 0.07809271663427353, -0.016833258792757988, 0.054951731115579605, 0.036277227103710175, -0.22429636120796204, -0.23311415314674377, -0.1304892897605896, 0.06827537715435028, -0.25099194049835205, 0.06226489320397377, 0.23711231350898743], [-0.06551751494407654, -0.10889473557472229, -0.10389824211597443, 0.14558562636375427, -0.017890535295009613, 0.12721413373947144, 0.20602236688137054, -0.05456705391407013, -0.13610288500785828, 0.0023209531791508198, -0.09056301414966583, 0.26603832840919495, 0.01771286502480507, 0.14514106512069702, 0.11326861381530762, -0.11313832551240921, -0.1361829936504364, 0.09058607369661331, -0.012984851375222206, -0.10140448808670044, -0.03367364779114723, -0.07358182221651077, 0.22504858672618866, 0.09703207015991211, -0.14027254283428192, 0.15237219631671906, -0.12972809374332428, -0.07860828936100006, -0.06523802876472473, 0.17175325751304626, 0.0875304564833641, -0.16615992784500122, -0.12372893840074539, -0.012721345759928226, 0.11140216141939163, -0.07731202989816666, 0.06432996690273285, 0.11912335455417633, -0.032793570309877396, -0.19304104149341583, -0.04780849441885948, -0.05020912364125252, -0.0666622743010521, 0.19691652059555054, -0.14657877385616302, -0.0646822452545166, -0.18078438937664032, -0.15215490758419037, 0.10648231208324432, -0.0029394265729933977, 0.040689319372177124, 0.04528498277068138, 0.06794718652963638, -0.06689292192459106, -0.028657278046011925, -0.10428343713283539, 0.21509361267089844, -0.018952909857034683, 0.11675573140382767, -0.1310196965932846, -0.21961677074432373, 0.0972394123673439, 0.0013935102615505457, 0.12815366685390472], [-0.09264963865280151, -0.21487444639205933, 0.13928453624248505, 0.03198126703500748, 0.00791194848716259, -0.1373380720615387, 0.0720110833644867, -0.08773214370012283, -0.17425255477428436, -0.04353957250714302, 0.08802089840173721, 0.18318846821784973, 0.13607728481292725, 0.0023451540619134903, 0.4156198799610138, 0.08277925848960876, -0.24583443999290466, -0.000904912652913481, -0.08785060793161392, -0.10374701023101807, -0.04253271594643593, -0.00011164244642714038, 0.03877675533294678, -0.052105795592069626, 0.05069364607334137, 0.06325879693031311, -0.20565511286258698, -0.09459361433982849, 0.06246531382203102, 0.03005583956837654, 0.09556148201227188, -0.17520157992839813, -0.11790722608566284, 0.02869163453578949, 0.15485715866088867, -0.09014160931110382, 0.04433978721499443, 0.14266405999660492, -0.09358857572078705, 0.009918301366269588, 0.20134221017360687, 0.07251966744661331, -0.06417711079120636, -0.009738474152982235, -0.23766878247261047, -0.06666383892297745, 0.03343580290675163, -0.16660672426223755, 0.14325764775276184, -0.06804804503917694, -0.10842783004045486, -0.07983652502298355, 0.3952116072177887, -0.06526748836040497, -0.14599914848804474, -0.12743735313415527, 0.23011481761932373, -0.14543962478637695, 0.07241940498352051, -0.06960837543010712, -0.10758073627948761, -0.23220176994800568, 0.11591441929340363, 0.15847066044807434], [0.2622291147708893, 0.2757144570350647, -0.1647506058216095, 0.19947847723960876, -0.07309696078300476, 0.21475481986999512, 0.24422380328178406, -0.033857788890600204, 0.09112177789211273, -0.03796639293432236, -0.011010745540261269, -0.0764436423778534, 0.17511318624019623, 0.15752822160720825, 0.06561212241649628, 0.16265834867954254, -0.04487049579620361, -0.29203876852989197, -0.05413977801799774, 0.043232858180999756, -0.16373790800571442, -0.08224862068891525, 0.1853901445865631, 0.18053984642028809, -0.2524050772190094, 0.06398874521255493, 0.00867037195712328, 0.15516218543052673, 0.1921621561050415, 0.2545410990715027, -0.09605050086975098, -0.11872856318950653, -0.16755498945713043, -0.18488402664661407, 0.07526235282421112, -0.07313773781061172, -0.27569812536239624, 0.05028415843844414, -0.19546619057655334, -0.21013705432415009, 0.1508660614490509, 0.1940721869468689, 0.28559261560440063, 0.14869128167629242, -0.018742019310593605, 0.21203742921352386, -0.10547945648431778, -0.21564622223377228, -0.11515625566244125, -0.06235278770327568, 0.05667925253510475, 0.26955145597457886, 0.09287447482347488, 0.030959146097302437, -0.181000754237175, -0.27957087755203247, -0.048751965165138245, 0.129812553524971, 0.054544467478990555, 0.0559888556599617, -0.2875443398952484, -0.19665421545505524, 0.0018451764481142163, 0.1703433245420456], [-0.003799977945163846, -0.028713328763842583, -0.09920378774404526, 0.005812667775899172, -0.17553465068340302, 0.15962627530097961, 0.05701894313097, -0.014282339252531528, -0.033272236585617065, -0.15202464163303375, -0.12040330469608307, 0.10073133558034897, 0.12584486603736877, 0.21431587636470795, 0.14711901545524597, 0.05889072269201279, -0.12716183066368103, 0.05192168056964874, 0.1242561936378479, -0.0355701819062233, -0.06539124250411987, -0.18090631067752838, 0.09657550603151321, 0.021875258535146713, -0.05568232759833336, 0.10969448834657669, -0.23802924156188965, 0.06971457600593567, 0.09424971789121628, 0.25255176424980164, -0.025544695556163788, -0.12572826445102692, -0.012710411101579666, 0.08092367649078369, 0.2837899625301361, 0.15009759366512299, -0.08065307140350342, 0.19769515097141266, -0.11639393866062164, -0.03152526542544365, 0.08477223664522171, -0.12657031416893005, 0.08443178981542587, -0.05334450677037239, 0.11970946192741394, -0.09674286097288132, -0.1753941923379898, -0.20166268944740295, -0.2240736484527588, 0.15131410956382751, 0.058600980788469315, -0.008894858881831169, 0.15086311101913452, 0.02886310964822769, 0.036519818007946014, -0.11671318858861923, 0.060570064932107925, -0.12130088359117508, 0.1335342824459076, -0.17095611989498138, -0.1495165079832077, -0.09220750629901886, 0.09743724763393402, 0.22354041039943695], [-0.17089436948299408, -0.03599665313959122, -0.3238106071949005, 0.13301797211170197, -0.08113037794828415, 0.1766292005777359, 0.12097993493080139, 0.1138961911201477, -0.06283850222826004, -0.013208695687353611, -0.23207730054855347, 0.1782711148262024, 0.2832269072532654, 0.2641788125038147, 0.10524623841047287, 0.24307353794574738, -0.10083355009555817, -0.1783314049243927, -0.10989393293857574, -0.24343103170394897, -0.12843602895736694, -0.03336452692747116, 0.47994616627693176, 0.20018550753593445, -0.20015515387058258, 0.0855187401175499, -0.20066748559474945, 0.2194814682006836, 0.054589517414569855, 0.048488687723875046, 0.04754401743412018, -0.07614287734031677, 0.05098845809698105, 0.08088776469230652, 0.38969215750694275, 0.011215336620807648, 0.11440476030111313, 0.0703747496008873, -0.4129938781261444, -0.2362596094608307, -0.0743199959397316, 0.028483781963586807, 0.15492229163646698, -0.25944191217422485, 0.13989216089248657, 0.02344166673719883, -0.39708834886550903, -0.1210816279053688, 0.035316064953804016, -0.09528759121894836, 0.18510845303535461, -0.010213945992290974, 0.07991915196180344, 0.14668215811252594, 0.0010533067397773266, -0.16634854674339294, 0.005852966569364071, -0.16955384612083435, 0.04230820760130882, 0.00511165289208293, 0.13764333724975586, 0.11648602038621902, 0.11490093171596527, -0.10885559767484665], [-0.04904428496956825, -0.05147062987089157, -0.12257443368434906, 0.01881985552608967, -0.06240512803196907, 0.09427531808614731, 0.2419196218252182, -0.006369709502905607, -0.034954316914081573, -0.06520875543355942, 0.03977283090353012, 0.2128542959690094, 0.126044362783432, 0.22074587643146515, 0.1840939074754715, 0.006919673178344965, -0.18457584083080292, 0.0442512109875679, 0.058699704706668854, -0.15194189548492432, -0.06546612083911896, -0.13363616168498993, 0.12357964366674423, -0.057529281824827194, 0.016622941941022873, 0.10260956734418869, -0.12587858736515045, 0.01573128253221512, 0.05349159613251686, 0.09589631855487823, 0.08652244508266449, -0.2654762864112854, 0.0748305469751358, 0.005926630459725857, 0.26696908473968506, 0.09367313981056213, 0.0576280876994133, 0.2542527914047241, -0.02063826657831669, -0.02002634108066559, 0.18895213305950165, 0.05146806687116623, -0.020189564675092697, 0.026843281462788582, -0.06329011917114258, -0.07035373151302338, -0.13707105815410614, -0.017203450202941895, -0.02563341334462166, 0.11934039741754532, 0.06064659729599953, 0.037848468869924545, 0.15761758387088776, -0.2373131811618805, -0.14597074687480927, -0.0034162395168095827, 0.23922328650951385, 0.02254132740199566, 0.08251218497753143, -0.09475123882293701, -0.16227714717388153, -0.14534319937229156, 0.12416001409292221, 0.18888725340366364], [0.025000669062137604, -0.024176595732569695, -0.0827956423163414, 0.084525465965271, -0.12608128786087036, -0.10317399352788925, 0.13446477055549622, -0.06192950904369354, -0.12303254008293152, -0.0650911033153534, 0.0217454731464386, 0.1224360391497612, 0.10975968092679977, 0.22237266600131989, 0.04284939169883728, 0.07910342514514923, -0.2233332395553589, -0.05282549560070038, 0.03386502340435982, 0.02391964942216873, -0.029127152636647224, -0.18872706592082977, 0.2204068899154663, 0.11219041049480438, -0.2528499364852905, 0.1500243991613388, -0.08449959754943848, -0.04211585596203804, -0.07617506384849548, 0.03049885667860508, -0.008634381927549839, -0.24676932394504547, -0.2665724456310272, -0.04316085949540138, 0.2474178820848465, 0.10767453163862228, -0.09560561180114746, 0.12641391158103943, -0.23465220630168915, -0.02124687470495701, 0.22367633879184723, 0.09467826783657074, 0.07832568138837814, -0.030191747471690178, -0.03599531203508377, 0.21274282038211823, -0.29863089323043823, -0.026398655027151108, -0.057533878833055496, -0.007169576361775398, 0.06887470930814743, 0.021454783156514168, 0.029187215492129326, -0.09297195076942444, -0.13397909700870514, -0.0760500431060791, 0.08188541978597641, -0.15161006152629852, 0.08791813254356384, -0.05147996544837952, -0.3334094285964966, -0.01999254897236824, -0.031972870230674744, 0.12885136902332306], [0.023326270282268524, 0.07389335334300995, 0.09839490056037903, 0.21687166392803192, -0.12975317239761353, -0.00836274866014719, 0.2818795144557953, -0.03688335046172142, -0.015873616561293602, 0.09807948768138885, 0.2118079960346222, 0.2102545201778412, 0.18642403185367584, 0.08228722959756851, 0.1597946584224701, 0.04592188075184822, -0.02545260265469551, -0.04902389273047447, -0.16136838495731354, -0.07985617220401764, -0.06499449163675308, -0.0939362570643425, 0.2044724076986313, 0.1908978968858719, -0.08682693541049957, 0.022758791223168373, -0.05187778174877167, 0.05034121870994568, 0.1058914065361023, 0.08906105160713196, 0.12489652633666992, -0.27776721119880676, -0.13261841237545013, -0.09680439531803131, 0.22443152964115143, 0.06244824826717377, -0.014357921667397022, 0.04087959975004196, -0.1355896145105362, 0.04791272431612015, 0.020503224804997444, -0.041644979268312454, 0.0628424733877182, 0.03496186062693596, -0.22146616876125336, -0.0062150293961167336, -0.2294379472732544, -0.1664043515920639, -0.021363476291298866, -0.05433221533894539, 0.019187351688742638, -0.003808763111010194, 0.09737157076597214, -0.11046598106622696, 0.002085941145196557, -0.22277523577213287, -0.04029737412929535, -0.16445831954479218, 0.14159856736660004, -0.04490553215146065, -0.18098074197769165, -0.057317107915878296, -0.020410196855664253, 0.04454435035586357], [-0.029483739286661148, 0.02088802494108677, -0.08767583221197128, -0.03240283951163292, -0.16737276315689087, -0.09162186831235886, -0.06289706379175186, -0.06529388576745987, -0.0003965080832131207, -0.11711621284484863, -0.17325040698051453, 0.23695988953113556, 0.08411171287298203, 0.04848487302660942, 0.15955665707588196, 0.24032354354858398, -0.1530144065618515, 0.01481857243925333, -0.20077380537986755, -0.14147700369358063, -0.008011951111257076, -0.11508384346961975, 0.046975716948509216, 0.24429064989089966, -0.24428701400756836, -0.06202750653028488, -0.01920696161687374, 0.04354548081755638, -0.029491964727640152, 0.08462313562631607, 0.1256786286830902, -0.0820780023932457, -0.16175901889801025, -0.08219391852617264, 0.17451317608356476, -0.0682377740740776, 0.13167360424995422, 0.07197275757789612, -0.13217101991176605, 0.006909532472491264, -0.01549141388386488, 0.06742401421070099, 0.009035347029566765, -0.10479828715324402, -0.03078126162290573, 0.05289291590452194, -0.1034826710820198, -0.1039186418056488, -0.0761546716094017, 0.14296869933605194, -0.08883333951234818, -0.015915164723992348, 0.2114921659231186, -0.039896897971630096, -0.13948781788349152, -0.08635818958282471, 0.21486257016658783, -0.12026162445545197, 0.16684702038764954, -0.09810832887887955, -0.20569859445095062, -0.2053060233592987, 0.22648881375789642, 0.13831335306167603], [0.13500507175922394, -0.1762799471616745, -0.1800033301115036, -0.009129652753472328, -0.11209756880998611, 0.1643131673336029, 0.0625557079911232, -0.12799806892871857, 0.0807483121752739, -0.10164633393287659, 0.13417693972587585, 0.17411364614963531, 0.2640163004398346, 0.2216353565454483, 0.10038281977176666, -0.03939107805490494, -0.1996346116065979, 0.13099446892738342, -0.04492175951600075, -0.22268180549144745, -0.0017871697200462222, -0.1833975911140442, 0.10376919060945511, 0.1869071125984192, -0.17021995782852173, -0.09470881521701813, -0.09244722127914429, -0.00146016338840127, 0.06549732387065887, 0.08709195256233215, 0.0901980772614479, -0.06993312388658524, 0.021356826648116112, 0.09574535489082336, 0.31774216890335083, -0.015028015710413456, 0.016388699412345886, 0.09248420596122742, -0.0881081223487854, -0.1226431205868721, -0.031792137771844864, -0.0840797871351242, 0.06299878656864166, 0.11843319982290268, 0.16381217539310455, -0.05857113376259804, -0.15634271502494812, -0.21223433315753937, 0.09807989746332169, 0.14683018624782562, 0.09807655960321426, 0.049903977662324905, 0.16364431381225586, 0.017968961969017982, 0.013558754697442055, -0.1247977614402771, 0.09136347472667694, -0.13806384801864624, -0.17512314021587372, -0.09183865040540695, -0.19065088033676147, 0.1123860776424408, 0.07775890827178955, 0.08062722533941269], [0.13819308578968048, -0.28594106435775757, -0.32941606640815735, 0.27968084812164307, -0.04109300673007965, -0.19342106580734253, -0.21122638881206512, 0.10357976704835892, -0.31039392948150635, 0.03492015600204468, -0.08986254781484604, 0.10672610253095627, -0.13685064017772675, 0.059442274272441864, 0.21458391845226288, 0.1331026554107666, -0.7902829647064209, 0.10512837022542953, -0.10860494524240494, -0.054888613522052765, 0.019016262143850327, -0.1882982701063156, 0.28324300050735474, 0.2006826400756836, -0.793459951877594, 0.11634255945682526, -0.18356601893901825, -0.29254499077796936, -0.2959285378456116, 0.19972974061965942, -0.2591983377933502, 0.15097621083259583, -0.6024329662322998, 0.15738175809383392, 0.0951913371682167, 0.2031899243593216, 0.20442727208137512, -0.21609808504581451, 0.21647410094738007, -0.17066366970539093, 0.0038026953116059303, 0.07312560081481934, -0.22125929594039917, 0.08413878828287125, -0.23809845745563507, 0.09623823314905167, -0.45643675327301025, -0.16960838437080383, 0.16234372556209564, 0.029643090441823006, -0.21205000579357147, 0.059739984571933746, 0.3566078245639801, -0.08336693048477173, 0.1885717213153839, -0.18981868028640747, 0.2626061737537384, -0.29868173599243164, -0.1209120973944664, -0.38653483986854553, -0.11730389297008514, 0.10401514917612076, 0.40331143140792847, 0.18876956403255463], [-0.08335116505622864, 0.0024814987555146217, 0.06689801812171936, 0.14898599684238434, -0.06642640382051468, 0.17953245341777802, 0.06743939965963364, -0.005804075859487057, -0.0623813159763813, -0.048243679106235504, 0.0319543182849884, 0.3004690706729889, 0.08746664971113205, 0.07614174485206604, 0.05527029186487198, -0.025802934542298317, -0.3471304476261139, -0.019718112424016, -0.12056730687618256, -0.0810355693101883, -0.13497328758239746, -0.03127173334360123, 0.20744135975837708, -0.035142142325639725, -0.09108243882656097, -0.0936991423368454, -0.13658958673477173, 0.14960162341594696, 0.14995376765727997, 0.1847749948501587, 0.17003847658634186, -0.002177497372031212, -0.12775187194347382, -0.03260914981365204, 0.12254015356302261, -0.03936775028705597, 0.013171565718948841, 0.06257427483797073, -0.04260462895035744, -0.09452877938747406, 0.05075426772236824, -0.06134681776165962, 0.061518196016550064, 0.01835491508245468, -0.12098436802625656, -0.1198873519897461, -0.04459298029541969, 0.025986744090914726, 0.08077956736087799, -0.08412781357765198, 0.03138487786054611, -0.06477946788072586, 0.1047588512301445, -0.18464165925979614, 0.03357413411140442, -0.09647729247808456, 0.24192112684249878, -0.055182114243507385, -0.005154476035386324, -0.02126188389956951, -0.0867726057767868, -0.06405909359455109, 0.08444070816040039, 0.14850659668445587], [-0.09702029079198837, -0.010454093106091022, -0.04152298346161842, -0.0351610966026783, -0.0855349525809288, 0.044459860771894455, 0.12843959033489227, -0.17187948524951935, -0.027708465233445168, -0.0997200459241867, -0.02646581642329693, -0.014633472077548504, 0.13282747566699982, -0.07075715810060501, 0.097137950360775, 0.2757415473461151, -0.23875507712364197, 0.2081945687532425, -0.1037798821926117, 0.021574996411800385, -0.04922666773200035, -0.14176148176193237, 0.05248557776212692, 0.05996136739850044, -0.25423508882522583, -0.02101735584437847, -0.2214578092098236, 0.0039049095939844847, 0.02310238592326641, 0.1713474988937378, -0.06591494381427765, 0.04441741853952408, -0.23050110042095184, 0.007105180062353611, 0.07079397886991501, -0.10298527777194977, 0.011144273914396763, 0.10520970821380615, -0.12412494421005249, -0.18290306627750397, 0.0633329302072525, -0.04839923232793808, -0.029477084055542946, 0.03265935555100441, -0.07393832504749298, -0.1621173769235611, -0.015996402129530907, -0.23321838676929474, 0.0914723351597786, 0.16204144060611725, -0.006804725620895624, -0.09973587095737457, 0.12878477573394775, -0.28252267837524414, -0.08508969843387604, -0.11265973001718521, 0.28026020526885986, 0.0001258951087947935, -0.028773920610547066, -0.22718849778175354, -0.1774945706129074, -0.12186366319656372, 0.14640064537525177, 0.21863329410552979], [-0.0764550119638443, -0.030408713966608047, 0.07828368246555328, 0.2636726200580597, -0.09836617112159729, -0.024218866601586342, 0.15317519009113312, -0.03985815867781639, -0.04069476202130318, 0.19647474586963654, 0.06055928021669388, 0.2826671898365021, 0.15400515496730804, 0.10730457305908203, 0.274316668510437, 0.38779494166374207, -0.2392003983259201, 0.009179048240184784, -0.12332925200462341, -0.13943062722682953, -0.03881869837641716, -0.0751057118177414, 0.02763536386191845, 0.0549810566008091, -0.243503138422966, -0.18882586061954498, 0.10636059939861298, -0.061447206884622574, 0.019262341782450676, 0.13784688711166382, -0.09981469064950943, -0.19453811645507812, -0.04157456383109093, -0.012222335673868656, 0.03765100985765457, 0.03516509383916855, 0.015683772042393684, 0.17398738861083984, 0.026104075834155083, -0.019351776689291, 0.0028919086325913668, -0.08346457779407501, -0.047567080706357956, -0.015416515059769154, -0.09180786460638046, 0.11647716164588928, -0.16112656891345978, -0.04128396138548851, 0.044077418744564056, 0.07406213879585266, 0.11511492729187012, -0.1376267969608307, -0.06045975163578987, 0.00718312943354249, 0.11025939136743546, 0.12012389302253723, 0.15085044503211975, 0.1862855851650238, 0.22525720298290253, 0.05059124901890755, -0.19351176917552948, 0.09734391421079636, 0.16364562511444092, -0.050621211528778076], [0.0006298601510934532, -0.22858408093452454, -0.002797928638756275, 0.04382479935884476, -0.04954373463988304, 0.09272005409002304, 0.10941421985626221, -0.07939466089010239, -0.015125419944524765, 0.14236702024936676, -0.11655786633491516, 0.27678248286247253, 0.14280621707439423, 0.04156043007969856, 0.11491791903972626, 0.06244964897632599, -0.3100142776966095, -0.05756191536784172, -0.19803091883659363, -0.21437375247478485, 0.007769162300974131, -0.05392766371369362, 0.22637398540973663, 0.06183742359280586, -0.1411457359790802, 0.033718813210725784, -0.06426993757486343, 0.2045098841190338, -0.12604095041751862, 0.05873224139213562, 0.02858993597328663, -0.06523851305246353, 0.06244712695479393, 0.0049349297769367695, 0.24580086767673492, 0.0025886893272399902, 0.06512812525033951, 0.013526331633329391, -0.186645969748497, -0.0011302524944767356, 0.05667298659682274, -0.007105674594640732, 0.09245792776346207, 0.0932832881808281, -0.14044144749641418, -0.04779310151934624, -0.2046346217393875, -0.11084955185651779, -0.03205796703696251, -0.13057073950767517, -0.10755397379398346, 0.07531273365020752, 0.07396861910820007, 0.1110628992319107, -0.06325814127922058, -0.24842017889022827, 0.14477135241031647, -0.18675565719604492, 0.03905855491757393, -0.003634594613686204, -0.11900139600038528, 0.017525453120470047, 0.02435733936727047, -0.06395228952169418], [-0.1089276596903801, 0.1668117642402649, -0.15980622172355652, 0.042060788720846176, -0.11382392048835754, 0.05370733141899109, 0.5335080623626709, -0.031662777066230774, -0.10112206637859344, 0.09606364369392395, 0.35344138741493225, 0.15351460874080658, 0.0290963314473629, 0.06493446230888367, 0.07222595810890198, 0.4666420519351959, 0.05577461048960686, -0.05943979322910309, -0.02158328890800476, 0.248801127076149, -0.25279244780540466, -0.07688893377780914, 0.35129767656326294, 0.21874134242534637, -0.3830122947692871, 0.006054210476577282, -0.31991928815841675, -0.0031068469397723675, -0.054826684296131134, -0.0824863612651825, -0.3225410282611847, -0.3402680456638336, -0.20999617874622345, -0.00372702325694263, 0.37524598836898804, -0.04721425101161003, -0.20734909176826477, -0.18312810361385345, -0.37153753638267517, -0.08806932717561722, 0.049007490277290344, -0.027905723080039024, -0.14833422005176544, 0.020749147981405258, -0.21679790318012238, 0.4392238259315491, -0.32829299569129944, -0.19043302536010742, -0.3305639922618866, -0.3236457407474518, 0.17191031575202942, 0.18015186488628387, 0.0621669664978981, -0.5540202260017395, -0.13590337336063385, 0.10225429385900497, 0.06671357154846191, -0.13861317932605743, 0.4790354371070862, -0.48121216893196106, -0.22221069037914276, -0.05661073699593544, 0.13809596002101898, 0.17936070263385773], [0.08655950427055359, -0.20270413160324097, 0.11571010947227478, 0.018099544569849968, 0.09225399047136307, -0.04803282022476196, 0.07964462041854858, -0.04787683114409447, -0.40087026357650757, -0.21522776782512665, 0.10329214483499527, 0.10548172146081924, 0.38718730211257935, 0.07292629778385162, 0.21592959761619568, 0.3949602246284485, -0.10581722110509872, -0.004107101820409298, 0.08302418887615204, 0.09236570447683334, -0.10969830304384232, -0.07558964192867279, 0.07349110394716263, 0.02497675269842148, -0.11603692173957825, -0.00983736664056778, -0.1492616832256317, -0.033344950526952744, 0.21851381659507751, -0.14219343662261963, 0.14273357391357422, -0.14876055717468262, -0.09739048779010773, 0.2250487506389618, 0.05924825370311737, -0.052125219255685806, -0.06575814634561539, 0.24786220490932465, 0.016266025602817535, -0.14389804005622864, 0.1903439313173294, -0.06477949023246765, 0.03590530529618263, 0.0701751708984375, -0.028019923716783524, -0.18998418748378754, -0.10373351722955704, -0.06952451914548874, -0.1007927879691124, -0.14098013937473297, -0.0873933956027031, -0.06045261770486832, 0.42924419045448303, -0.15040671825408936, -0.0693083330988884, 0.0521932952105999, 0.4727015793323517, -0.1601751446723938, -0.01538105122745037, -0.18698446452617645, -0.31358325481414795, -0.2182675153017044, 0.05047190561890602, 0.04090185835957527], [0.030882760882377625, -0.06379016488790512, -0.058529309928417206, -0.009312320500612259, -0.03443378955125809, -0.04555683583021164, 0.1135268360376358, -0.06296443194150925, 0.01680266112089157, -0.07408134639263153, 0.11652827262878418, 0.10976909846067429, 0.18768441677093506, 0.058407366275787354, 0.18228478729724884, 0.15490758419036865, -0.19938653707504272, -0.059111788868904114, -0.13222965598106384, 0.08374883979558945, -0.028821758925914764, -0.0704328715801239, 0.06092748045921326, 0.07619436830282211, -0.12647897005081177, 0.042857423424720764, -0.13728167116641998, -0.007330866064876318, 0.12031425535678864, 0.008852079510688782, -0.012546845711767673, -0.10568708926439285, -0.2523040473461151, -0.1151939406991005, 0.14941005408763885, 0.005235192831605673, 0.0011523913126438856, 0.06568916141986847, -0.10487981140613556, -0.09294025599956512, 0.13689514994621277, 0.11141978204250336, 0.025657419115304947, -0.03521396219730377, -0.07730084657669067, -0.030551891773939133, -0.16741156578063965, -0.11116783320903778, 0.024854883551597595, -0.033601313829422, -0.030135411769151688, 0.0006967206718400121, 0.19229917228221893, -0.23330046236515045, -0.12186028063297272, -0.019444817677140236, 0.11435096710920334, -0.0291140154004097, 0.07066207379102707, -0.08414224535226822, -0.26339295506477356, -0.21567070484161377, 0.15001915395259857, 0.1405380815267563], [0.03352658823132515, -0.1495811492204666, -0.03656364232301712, 0.13652418553829193, 0.07878868281841278, 0.33100470900535583, 0.054558128118515015, -0.08493565768003464, -0.0041849189437925816, -0.13573220372200012, -0.14011874794960022, 0.33503594994544983, 0.042785171419382095, 0.03767543286085129, 0.02681214176118374, 0.1576172411441803, -0.03007752075791359, 0.11959134042263031, 0.01977689377963543, -0.02544768527150154, -0.009236299432814121, -0.11890992522239685, 0.10498476773500443, 0.06084616109728813, 0.07881642132997513, 0.10384998470544815, -0.11024875938892365, -0.055298663675785065, -0.09489545971155167, 0.12842334806919098, 0.061091385781764984, 0.021642915904521942, 0.05410008877515793, 0.18932566046714783, 0.22873683273792267, 0.09488770365715027, 0.020059239119291306, -0.06397856771945953, 0.040159232914447784, -0.12330307066440582, -0.06090724840760231, -0.09822051227092743, 0.05464181676506996, 0.06985883414745331, 0.06113782525062561, -0.10853909701108932, -0.051237672567367554, -0.141104593873024, -0.1792861372232437, -0.028640348464250565, -0.10007397085428238, -0.057496532797813416, 0.20262782275676727, -0.10739690065383911, -0.18229518830776215, -0.06253229081630707, 0.1854381412267685, -0.01988081820309162, -0.004369434434920549, -0.028592471033334732, -0.10134194046258926, -0.10308815538883209, 0.29838651418685913, 0.3797987997531891], [-0.20299114286899567, 0.13816139101982117, 0.0004969722940586507, 0.20855693519115448, -0.09342233091592789, -0.020836642012000084, 0.22560344636440277, -0.05428070202469826, -0.11192822456359863, -0.011842671781778336, -0.04352399706840515, 0.184391587972641, 0.14170581102371216, -0.04415519908070564, 0.00631205877289176, -0.13037985563278198, -0.15395256876945496, -0.22449389100074768, 0.026287907734513283, 0.015704810619354248, 0.062187310308218, -0.036822449415922165, 0.2025919407606125, 0.3370485007762909, -0.2183794230222702, -0.0417013019323349, 0.11006378382444382, 0.03511572256684303, 0.00678908871486783, 0.13236725330352783, 0.2790074646472931, -0.15616369247436523, -0.23412255942821503, -0.06535055488348007, 0.2637304663658142, 0.1190296933054924, 0.009152158163487911, 0.163751021027565, -0.1260184943675995, -0.06091828644275665, 0.07376758754253387, 0.116025410592556, -0.0803806409239769, 0.13929039239883423, -0.11108513176441193, 0.0937034860253334, -0.213997483253479, -0.17384693026542664, -0.1922059804201126, -0.30000853538513184, -0.11550556123256683, 0.09462292492389679, -0.006581427995115519, -0.09269730746746063, -0.10960522294044495, -0.14737574756145477, 0.14105327427387238, -0.21215032041072845, 0.060675736516714096, 0.23413826525211334, -0.02113993465900421, -0.032244473695755005, -0.10584001243114471, 0.3592231571674347], [-0.03598949313163757, -0.14436978101730347, 0.0807490423321724, 0.16468441486358643, -0.23760803043842316, 0.10487876087427139, 0.2549892067909241, -0.14841848611831665, -0.005181474145501852, -0.04032494127750397, 0.13774612545967102, 0.09643568843603134, 0.18786613643169403, 0.1286526918411255, 0.3261539041996002, -0.06228413060307503, -0.10102993249893188, 0.04437737911939621, -0.057614121586084366, -0.0032952846959233284, 0.1510864645242691, -0.12527436017990112, 0.1091669574379921, 0.18665795028209686, -0.19017091393470764, -0.04870500788092613, -0.07460533082485199, -0.040752921253442764, 0.35386937856674194, 0.03311820328235626, 0.05262449011206627, -0.2655329406261444, -0.3333604335784912, 0.0680094063282013, 0.2920650541782379, 0.2378837913274765, -0.07131364196538925, 0.006490106228739023, 0.024226102977991104, -0.007713154889643192, 0.16555510461330414, -0.12043668329715729, 0.033216558396816254, 0.015091699548065662, -0.17808549106121063, -0.19999822974205017, -0.2338528037071228, -0.23103870451450348, 0.03156090900301933, 0.04036129638552666, -0.02779153548181057, -0.0954558253288269, 0.17083726823329926, -0.0842633917927742, 0.12658677995204926, -0.18263153731822968, -0.15930159389972687, -0.14970248937606812, -0.2305491715669632, -0.1289200633764267, -0.06364955753087997, -0.16086751222610474, -0.07955792546272278, 0.10290968418121338], [-0.12023507058620453, -0.03835274651646614, -0.0025181928649544716, 0.18728949129581451, -0.05964166671037674, 0.15151143074035645, 0.12586632370948792, -0.01625477895140648, 0.022453095763921738, -0.027452915906906128, 0.10292764753103256, 0.06162630021572113, 0.05384886637330055, 0.030282359570264816, 0.04018197953701019, 0.2076457440853119, -0.13654842972755432, 0.05163070186972618, -0.24543574452400208, 0.0061430539935827255, -0.009057258255779743, -0.21960866451263428, 0.11584688723087311, 0.01741856336593628, -0.16358470916748047, 0.04081312194466591, -0.16626760363578796, 0.11610528081655502, 0.08414187282323837, 0.09426046162843704, -0.1165412962436676, -0.14190027117729187, -0.15470248460769653, 0.019388409331440926, 0.13966022431850433, -0.011684920638799667, -0.12088024616241455, 0.13992156088352203, -0.000982856610789895, -0.2986651062965393, -0.04466638341546059, -0.05087568610906601, 0.027976248413324356, 0.0586472749710083, -0.12057975679636002, 0.03777480497956276, -0.10731128603219986, -0.03695031255483627, 0.12485292553901672, -0.1002417504787445, 0.04042666032910347, -0.03986382856965065, 0.1378048062324524, -0.09983432292938232, 0.06849756091833115, -0.03518560528755188, 0.149717777967453, -0.0041441465727984905, 0.17906895279884338, -0.07440808415412903, -0.03368216007947922, -0.07522845268249512, 0.10210436582565308, 0.014483478851616383], [0.18275727331638336, -0.013798849657177925, 0.08561451733112335, 0.26774534583091736, 0.012287825345993042, 0.23315513134002686, -0.1022525504231453, -0.15488673746585846, 0.1505480706691742, -0.017839789390563965, 0.02521885558962822, 0.18897774815559387, 0.28968578577041626, 0.08999312669038773, 0.16861633956432343, 0.07353679835796356, 0.06437994539737701, 0.301151841878891, -0.03686964884400368, -0.011470817029476166, -0.14154939353466034, -0.09413866698741913, 0.03756712004542351, 0.08242159336805344, 0.042550500482320786, 0.13873200118541718, -0.09703078120946884, -0.0047728559002280235, 0.009843946434557438, -0.11036926507949829, 0.26807281374931335, -0.19671958684921265, 0.006089626811444759, 0.05822557955980301, 0.21413646638393402, 0.06760528683662415, -0.05965161323547363, 0.04924843832850456, -0.08500722795724869, -0.1552368551492691, 0.07753784209489822, -0.18472333252429962, 0.1376204788684845, 0.09119197726249695, 0.054502714425325394, 0.05499119311571121, 0.010269347578287125, -0.3196398615837097, 0.012982473708689213, 0.057404305785894394, -0.0073256962932646275, -0.1016533151268959, 0.2352524697780609, -0.06876000761985779, -0.0751616507768631, -0.10965538769960403, 0.09808693826198578, 0.018347570672631264, 0.12416425347328186, -0.04653240367770195, -0.2095431387424469, 0.10864289104938507, 0.1388106495141983, 0.16883312165737152], [0.13252966105937958, 0.04763147234916687, -0.018907085061073303, -0.05812426656484604, -0.21370024979114532, 0.04136265814304352, 0.16119104623794556, -0.18522599339485168, 0.029240647330880165, -0.10189668089151382, -0.11442285776138306, 0.06502357870340347, 0.23817890882492065, 0.12144345790147781, 0.1721908301115036, 0.15328072011470795, -0.15028764307498932, 0.0745055228471756, -0.03173954784870148, -0.10142875462770462, -0.03185976296663284, -0.18496596813201904, 0.12410502135753632, 0.34046584367752075, -0.4731511175632477, 0.00276049692183733, 0.20044302940368652, 0.10686152428388596, 0.15816131234169006, 0.2130439132452011, 0.07361190766096115, -0.21108312904834747, -0.07356506586074829, -0.1142570748925209, -0.02077203243970871, 0.152247816324234, 0.21413880586624146, 0.15824025869369507, -0.2421833872795105, -0.11439191550016403, -0.029856931418180466, 0.04796856641769409, 0.015460709109902382, 0.008320385590195656, -0.011929531581699848, -0.09762115776538849, -0.22000959515571594, 0.006567368749529123, -0.29345372319221497, -0.009927503764629364, -0.1128794327378273, 0.04280487820506096, 0.07003786414861679, -0.1540829837322235, -0.05407099798321724, 0.00891608465462923, 0.2680197060108185, -0.007754098623991013, -0.07118362933397293, 0.03861019387841225, -0.15117986500263214, -0.09567888081073761, 0.10410955548286438, 0.13435472548007965], [-0.19159246981143951, -0.09503371268510818, 0.024983149021863937, -0.15672503411769867, -0.027779471129179, 0.13074755668640137, 0.07804785668849945, -0.26020070910453796, -0.0835404247045517, 0.11731685698032379, -0.14945532381534576, 0.11016155779361725, 0.26774927973747253, 0.1535484939813614, 0.3088616132736206, 0.06859435886144638, -0.3775443434715271, 0.10931260138750076, -0.1623634696006775, 0.018868321552872658, -0.002659304067492485, 0.025294765830039978, 0.16226573288440704, 0.11379597336053848, -0.3335014283657074, -0.016202980652451515, -0.16335821151733398, 0.013823766261339188, 0.06908506155014038, 0.009454569779336452, 0.21939828991889954, -0.22040516138076782, -0.12473605573177338, -0.05655992403626442, 0.16908253729343414, -0.04272972419857979, 0.1037822738289833, 0.07358437031507492, -0.13275043666362762, -0.03325846046209335, -0.06142360717058182, -0.11947977542877197, 0.02341741882264614, 0.13783201575279236, -0.2622188925743103, -0.10326622426509857, -0.2080831080675125, -0.1094028502702713, 0.04140600189566612, -0.07519745081663132, -0.127492755651474, 0.09761105477809906, 0.20971323549747467, -0.12400179356336594, -0.08500252664089203, -0.12959018349647522, 0.02503558248281479, -0.012675522826611996, -0.12764698266983032, -0.12434504926204681, -0.1489126980304718, 0.08602242916822433, 0.1332300454378128, 0.32860851287841797], [-0.010727053508162498, -0.10578811168670654, 0.05486896634101868, 0.02228526584804058, 0.07358347624540329, -0.06448917835950851, 0.0839352160692215, -0.22898022830486298, 0.0046180542558431625, 0.07107962667942047, 0.01809888705611229, 0.32091519236564636, 0.38676559925079346, -0.0040593198500573635, 0.180118128657341, 0.0642988383769989, -0.4002981185913086, 0.10851052403450012, -0.23998260498046875, -0.018997084349393845, -0.06770685315132141, -0.07924237847328186, 0.15424373745918274, 0.027322351932525635, -0.29116350412368774, -0.08946098387241364, -0.06764420121908188, 0.11784292012453079, 0.03344295546412468, -0.00284088053740561, 0.23366723954677582, -0.0958157479763031, -0.07229265570640564, -0.16851933300495148, 0.2033795416355133, 0.029841113835573196, 0.08072803169488907, -0.02634883113205433, -0.10194389522075653, -0.10895045846700668, -0.031940896064043045, -0.0003216527111362666, 0.06501587480306625, -0.06860464066267014, -0.00439418712630868, -0.018991831690073013, -0.06825467199087143, 0.0392242893576622, -0.10270082205533981, 0.020612381398677826, -0.1674375981092453, -0.05388760194182396, 0.156077042222023, -0.03849108889698982, -0.06074967607855797, -0.08683553338050842, 0.08128562569618225, 0.002113071735948324, 0.042128387838602066, -0.11952050775289536, -0.1944555938243866, -0.08302304893732071, 0.0755266323685646, 0.009497418068349361], [-0.034691162407398224, 0.010853678919374943, 0.04884781315922737, 0.20624539256095886, -0.023033712059259415, 0.026817047968506813, 0.13812007009983063, -0.07969314604997635, -0.2663266062736511, -0.004814195912331343, -0.09864414483308792, 0.14363764226436615, 0.16283543407917023, 0.11312314122915268, 0.09767010807991028, 0.06608105450868607, -0.3448069989681244, 0.03704720735549927, -0.043657321482896805, 0.04768512398004532, 0.04961476847529411, 0.009767835028469563, 0.09760697185993195, 0.3039912283420563, -0.07227738201618195, 0.05576429143548012, -0.1159130111336708, 0.1831914633512497, 0.09064247459173203, 0.2253705859184265, -0.001226173946633935, -0.04318331927061081, -0.020256392657756805, 0.021797584369778633, 0.14102335274219513, -0.03875403478741646, 0.05774308741092682, 0.11139596253633499, -0.009149887599050999, -0.18922147154808044, 0.09633652120828629, -0.06754075735807419, 0.06167618930339813, 0.25325947999954224, -0.14583469927310944, -0.09447245299816132, -0.16031406819820404, -0.13486447930335999, 0.05668093264102936, -0.13799308240413666, 0.039218027144670486, 0.054029613733291626, -0.006886123679578304, -0.10741830617189407, -0.05703301355242729, -0.07048646360635757, 0.18697841465473175, 0.09199526906013489, 0.04482394829392433, -0.013109459541738033, -0.05080679431557655, 0.13051259517669678, -0.0032748598605394363, 0.1069420725107193], [-0.029101120308041573, -0.07087180763483047, 0.037175387144088745, -0.03938499838113785, -0.27532458305358887, 0.1357847899198532, 0.2269720733165741, 0.0005212176474742591, 0.08065290749073029, -0.1810605674982071, 0.1426926702260971, 0.12701931595802307, 0.1395615190267563, 0.24995192885398865, 0.16378647089004517, -0.02425285242497921, -0.13835252821445465, 0.02432817779481411, -0.08907327800989151, -0.08756779879331589, 0.030372044071555138, -0.02096518501639366, 0.11460616439580917, 0.09964960068464279, -0.28072690963745117, -0.05202728882431984, -0.07291635125875473, -0.02091219834983349, 0.25850710272789, 0.17923255264759064, 0.2413596510887146, -0.16431878507137299, -0.23215845227241516, 0.03175904601812363, 0.19589638710021973, -0.14489609003067017, 0.10914237052202225, 0.20602883398532867, -0.03178052976727486, -0.18749694526195526, -0.045040033757686615, 0.0040170275606215, 0.101313516497612, -0.007929680868983269, -0.03798699006438255, -0.10560903698205948, -0.2875972092151642, -0.1390204280614853, 0.23058485984802246, 0.10580719262361526, 0.1217515841126442, 0.16852784156799316, 0.24039188027381897, -0.2715061604976654, 0.08811868727207184, -0.09386155009269714, -0.06330695748329163, 0.11499112099409103, -0.1157468855381012, -0.1492406129837036, -0.23442164063453674, -0.025498420000076294, -0.009246830828487873, 0.04380429536104202], [0.1433188021183014, 0.13030067086219788, 0.024573272094130516, 0.10266904532909393, -0.2928406894207001, 0.13453640043735504, 0.06488065421581268, -0.06392417848110199, -0.1776740998029709, 0.02156669646501541, 0.1814316064119339, 0.0408749133348465, 0.28157973289489746, 0.1632840484380722, 0.29434967041015625, 0.12321022152900696, -0.5232430696487427, 0.17156453430652618, -0.16931071877479553, -0.025228822603821754, -0.0881849080324173, -0.006899581290781498, 0.10275383293628693, 0.18626226484775543, 0.12593214213848114, -0.020152622833848, -0.2932223379611969, 0.08928622305393219, 0.4574544429779053, 0.20022305846214294, -0.005149306263774633, -0.10227211564779282, -0.03659083694219589, 0.05164853855967522, 0.11714987456798553, 0.05678199604153633, 0.2524379789829254, 0.11878391355276108, -0.14834396541118622, 0.06871035695075989, 0.2599599063396454, -0.08572354167699814, -0.12873202562332153, -0.03962111845612526, -0.31395742297172546, -0.29118186235427856, 0.0032944960985332727, -0.3019791543483734, 0.0015093922847881913, -0.028033539652824402, 0.04374486207962036, -0.2696986794471741, 0.25245732069015503, -0.14895957708358765, 0.1345796436071396, -0.22079703211784363, -0.13858076930046082, 0.011751187033951283, -0.07962566614151001, 0.15259812772274017, -0.1709616333246231, -0.0832991674542427, 0.06531956046819687, 0.05074600130319595], [-0.06533683836460114, -0.0408116951584816, 0.044911276549100876, 0.23560097813606262, -0.08887635916471481, -0.03169190511107445, -0.04283555597066879, 0.17580625414848328, -0.02066359668970108, -0.09183970838785172, 0.020220331847667694, 0.4168645739555359, 0.02830664999783039, 0.10677723586559296, 0.25865787267684937, 0.04675006866455078, -0.1108652800321579, -0.004426787607371807, -0.05364060774445534, -0.0362502820789814, 0.18523390591144562, -0.1559380739927292, 0.1403951495885849, 0.28647348284721375, -0.16815653443336487, -0.3302871584892273, 0.23720678687095642, 0.003945076372474432, 0.15129989385604858, 0.3582877516746521, -0.020537467673420906, 0.06940415501594543, -0.06573590636253357, -0.08129464089870453, 0.02815229631960392, 0.05210064351558685, 0.016903439536690712, -0.1096406951546669, 0.23884576559066772, -0.18156889081001282, 0.04688611626625061, -0.19777609407901764, 0.2994305491447449, 0.12591508030891418, 0.1322631537914276, -0.06223756819963455, -0.10023233294487, -0.15585549175739288, -0.02360176108777523, -0.08224401623010635, -0.16223308444023132, -0.017498308792710304, 0.10025166720151901, 0.04567331448197365, 0.0477876178920269, -0.25649118423461914, 0.08376360684633255, -0.3463743031024933, 0.058585915714502335, -0.09757734835147858, 0.023164859041571617, -0.12106060236692429, 0.022184643894433975, 0.10242900252342224], [0.01207643747329712, -0.14037665724754333, 0.041725579649209976, 0.2220471203327179, -0.0910101979970932, 0.029783405363559723, 0.1809791773557663, -0.03558870032429695, -0.006805744953453541, 0.19811835885047913, 0.053655292838811874, 0.3113323748111725, 0.000726737838704139, 0.07429228723049164, 0.2736523151397705, 0.024358198046684265, -0.1888313591480255, -0.05148174986243248, -0.18823546171188354, -0.017627080902457237, 0.03974723815917969, -0.20591400563716888, 0.20656171441078186, 0.14253249764442444, -0.3123469948768616, -0.0020492009352892637, 0.010244285687804222, -0.061547085642814636, -0.03080170415341854, 0.13011635839939117, -0.11160402745008469, -0.14957596361637115, 0.035430021584033966, 0.00454780925065279, 0.24556566774845123, 0.07597830891609192, 0.11346655339002609, 0.061556626111269, -0.06634127348661423, -0.07654985040426254, -0.06654740124940872, -0.036302756518125534, -0.0789266973733902, 0.12564630806446075, -0.24833141267299652, -0.03394298255443573, -0.16801050305366516, -0.13228052854537964, -0.12392210215330124, -0.04725712910294533, -0.27977463603019714, 0.06008078157901764, 0.16570745408535004, -0.042895346879959106, -0.03651963174343109, -0.21663443744182587, 0.21375875174999237, -0.22321049869060516, -0.04942532628774643, 0.006685464642941952, -0.2907944321632385, 0.05373384431004524, -0.0322147011756897, 0.1569676399230957], [-0.024651290848851204, -0.14670230448246002, 0.07006891816854477, 0.13249096274375916, 0.10764507949352264, -0.09041620045900345, 0.19832412898540497, 0.0018520215526223183, 0.17179851233959198, -0.04215945675969124, 0.016573939472436905, 0.09574229270219803, -0.03177058696746826, -0.05835491791367531, -0.05161810293793678, 0.032471444457769394, -0.14033551514148712, -0.06617433577775955, -0.28181061148643494, -0.07579656690359116, 0.26417768001556396, -0.1316799521446228, 0.22115808725357056, 0.08380824327468872, -0.26521411538124084, 0.253194123506546, -0.17359846830368042, -0.1632346361875534, 0.19205105304718018, 0.07570235431194305, -0.01120658591389656, -0.06654378026723862, -0.27311593294143677, -0.11616102606058121, 0.20406754314899445, -0.15273921191692352, -0.09593238681554794, -0.06166916713118553, -0.08384469151496887, -0.40565764904022217, -0.383548378944397, -0.024903658777475357, -0.1591378003358841, -0.01469497848302126, -0.13611222803592682, 0.09489498287439346, -0.2236417680978775, -0.29839199781417847, -0.04980902001261711, -0.11676955968141556, -0.09436837583780289, -0.06812454015016556, 0.3819423317909241, -0.25430914759635925, -0.03829227760434151, -0.20263837277889252, 0.07576644420623779, -0.18197594583034515, 0.019877217710018158, -0.04770638793706894, -0.04119060933589935, 0.07862342894077301, 0.2307494878768921, -0.20114143192768097], [0.0018076960695907474, 0.06977448612451553, 0.09734877943992615, -0.08939836919307709, -0.09375166147947311, 0.17192891240119934, 0.027255626395344734, -0.13311004638671875, -0.015163998119533062, -0.19449034333229065, -0.14194434881210327, 0.0852511078119278, 0.21413137018680573, 0.07566843926906586, 0.08151911944150925, 0.12901532649993896, -0.19174793362617493, 0.12821662425994873, 0.11463349312543869, 0.023071961477398872, -0.04008915647864342, -0.017897898331284523, 0.06747975945472717, 0.17521852254867554, -0.17316322028636932, -0.0898631140589714, -0.24946747720241547, 0.11656629294157028, 0.10845301300287247, 0.04373287409543991, -0.003466126974672079, -0.27163299918174744, -0.007011108566075563, 0.1314856857061386, 0.3152230978012085, -0.27427005767822266, -0.08583691716194153, -0.026880301535129547, -0.11613992601633072, -0.24750350415706635, 0.05434628203511238, 0.14081677794456482, 0.13023412227630615, 0.17995429039001465, 0.032161492854356766, -0.001957851927727461, 0.09780575335025787, -0.19236446917057037, 0.028424320742487907, 0.08866019546985626, 0.10221849381923676, 0.14992468059062958, 0.05432693287730217, -0.13800089061260223, -0.029573852196335793, -0.12392506748437881, 0.23817044496536255, 0.042857106775045395, -0.07315696775913239, -0.10120595991611481, -0.41649290919303894, 0.060836829245090485, -0.01908358745276928, 0.13916990160942078], [-0.013834703713655472, -0.2606738209724426, -0.17518192529678345, 0.2578734755516052, -0.06697189062833786, -0.04998078942298889, 0.27810341119766235, -0.1624695211648941, -0.13473455607891083, -0.08065284788608551, 0.10817738622426987, 0.2969610393047333, 0.055615443736314774, 0.311275452375412, 0.3275441825389862, -0.18762940168380737, -0.28451666235923767, 0.09559302777051926, 0.2231689840555191, -0.09987258166074753, -0.16491739451885223, -0.10980235040187836, 0.32452329993247986, 0.18489493429660797, -0.2593359649181366, 0.07706064730882645, -0.07358194887638092, 0.08174917101860046, 0.030413052067160606, -0.07536709308624268, 0.021075645461678505, -0.16891609132289886, -0.32276761531829834, 0.19205276668071747, 0.1465548425912857, 0.3294106721878052, 0.01300767157226801, -0.00794950034469366, -0.20273253321647644, -0.26346850395202637, 0.3952407240867615, 0.05211963132023811, -0.09556777030229568, -0.1829388290643692, 0.009976753033697605, -0.11549586057662964, -0.29594841599464417, -0.009296628646552563, 0.19069011509418488, -0.05043923109769821, 0.0970340222120285, 0.09606453776359558, 0.047000810503959656, -0.10842745751142502, 0.11263086646795273, -0.04642895981669426, 0.23001159727573395, 0.009335266426205635, -0.19912318885326385, -0.09576231986284256, -0.1322181522846222, -0.038256727159023285, 0.16098815202713013, 0.1394871175289154], [-0.0054006087593734264, 0.11068370938301086, -0.011689581908285618, 0.3559498190879822, -0.07786030322313309, -0.044237468391656876, 0.2014870047569275, -0.21461065113544464, -0.3438839018344879, 0.1402585655450821, 0.16639623045921326, 0.09256471693515778, 0.20015040040016174, -0.006116910371929407, 0.1163756251335144, 0.34784063696861267, -0.3096180260181427, -0.08135438710451126, 0.05668221786618233, 0.10900779813528061, -0.04249899089336395, -0.0978064239025116, 0.2657899856567383, 0.07024944573640823, -0.20441274344921112, -0.11590732634067535, -0.26048314571380615, -0.026659870520234108, 0.03500472754240036, 0.009904995560646057, -0.06206075847148895, -0.1804545521736145, -0.26344436407089233, -0.05360304191708565, 0.3037160038948059, 0.009670773521065712, -0.1438940018415451, 0.023097803816199303, -0.1802423745393753, -0.08652576804161072, 0.25751641392707825, 0.04016821086406708, -0.0010424887295812368, -0.047477588057518005, -0.13338086009025574, 0.17316415905952454, -0.1063309907913208, -0.09451921284198761, -0.15728306770324707, -0.15996551513671875, 0.21609747409820557, -0.06198874115943909, -0.05399255082011223, -0.2342744618654251, -0.1174851804971695, 0.03583240136504173, -0.004733305890113115, -0.006331885699182749, 0.17787817120552063, -0.08365349471569061, -0.11454489827156067, -0.038706716150045395, 0.11755730956792831, 0.06072521209716797], [0.09719102084636688, 0.013692932203412056, -0.08672047406435013, 0.10315590351819992, -0.10267705470323563, 0.1020800992846489, 0.10408161580562592, -0.1319074183702469, -0.04824870452284813, 0.014043386094272137, -0.1505560725927353, 0.18527498841285706, 0.2201876938343048, 0.1461736410856247, -0.018051642924547195, 0.09145887196063995, -0.09884517639875412, -0.08318767696619034, 0.010015866719186306, -0.1564270555973053, -0.03251536935567856, -0.055334802716970444, 0.35717877745628357, 0.13667415082454681, 0.03088635765016079, 0.1718304604291916, -0.1304301619529724, 0.2243925929069519, -0.04823823645710945, 0.04314831644296646, 0.04390576854348183, -0.11720621585845947, -0.05814596265554428, 0.08826827257871628, 0.2672731578350067, 0.10962963104248047, 0.013017094694077969, -0.06266661733388901, -0.2779290974140167, 0.01524683739989996, 0.08339932560920715, 0.09689263254404068, 0.1804177314043045, 0.06254951655864716, -0.011603952385485172, 0.09088614583015442, -0.34785550832748413, -0.10389817506074905, -0.10130654275417328, -0.15752628445625305, 0.103000208735466, 0.047301776707172394, 0.0027333784382790327, 0.005687342491000891, -0.04207345470786095, -0.18651582300662994, 0.024896537885069847, -0.08539236336946487, 0.15346597135066986, 0.11080677062273026, -0.0877995416522026, 0.038819365203380585, -0.051043491810560226, 0.024640193209052086], [0.3180791437625885, -0.03041299246251583, 0.2456234097480774, 0.15514324605464935, -0.00888283271342516, -0.11296553909778595, 0.2814042270183563, -0.3552434742450714, -0.05547620728611946, 0.15585234761238098, 0.3584597110748291, 0.07472608238458633, 0.32090887427330017, -0.13248109817504883, 0.33961987495422363, -0.02393489144742489, -0.4571995139122009, 0.060124438256025314, 0.022446023300290108, 0.012028580531477928, -0.2841593325138092, 0.010879985988140106, 0.2776488661766052, -0.15223544836044312, -0.03661319613456726, -0.1615460067987442, -0.19739305973052979, -0.06080994755029678, 0.005168266128748655, 0.209415465593338, 0.1350228637456894, -0.12977364659309387, -0.12717226147651672, -0.13105136156082153, 0.25024986267089844, -0.002384879859164357, 0.1731385737657547, 0.12113506346940994, -0.031698696315288544, -0.026167189702391624, 0.43682920932769775, -0.03375519439578056, 0.12715864181518555, 0.13708116114139557, -0.10646198689937592, -0.2969996929168701, 0.07186185568571091, -0.19583483040332794, 0.07164037227630615, 0.04340842738747597, 0.11227485537528992, -0.07559353113174438, -0.05852629616856575, -0.07044027745723724, -0.03712977096438408, -0.5135821104049683, -0.05431196466088295, 0.044063977897167206, -0.1153937503695488, -0.260703980922699, -0.33835580945014954, -0.07590937614440918, 0.02056286111474037, 0.07931818813085556], [0.00214611878618598, 0.1316450834274292, -0.15472844243049622, -0.0057035814970731735, 0.029242290183901787, 0.11294252425432205, 0.0860142782330513, 0.06937550008296967, -0.11634662002325058, 0.004752393811941147, -0.014652851969003677, 0.10487044602632523, 0.23763291537761688, 0.07070967555046082, 0.33381909132003784, 0.2966660261154175, -0.3316929042339325, -0.056659258902072906, -0.12801389396190643, 0.0034846151247620583, -0.09510204195976257, -0.10327280312776566, 0.04090358689427376, 0.21775934100151062, -0.2610819637775421, -0.023008910939097404, -0.003299516858533025, 0.23685677349567413, -0.04059072211384773, 0.07312392443418503, 0.039570145308971405, -0.2476712316274643, -0.06903628259897232, 0.07052957266569138, 0.1474752128124237, 0.0538504496216774, 0.05056546628475189, -0.04653042554855347, -0.05938684567809105, -0.1358904242515564, -0.024448206648230553, 0.022732384502887726, -0.004378040321171284, -0.01938004232943058, -0.1864875853061676, 0.045165084302425385, -0.17908798158168793, -0.1166902557015419, -0.14648903906345367, 0.0036875891964882612, 0.024653278291225433, 0.03301743417978287, 0.2466811090707779, -0.09213925153017044, -0.1488429754972458, -0.03690514713525772, 0.1481257677078247, 0.007071303203701973, -0.12646235525608063, -0.09442979842424393, -0.10817041248083115, -0.13513050973415375, 0.2325800508260727, 0.307931512594223], [-0.11465442180633545, 0.012724916450679302, 0.0238402858376503, 0.18471357226371765, -0.15259993076324463, 0.08821514248847961, 0.34491097927093506, -0.05909441038966179, -0.08906418830156326, -0.08247888088226318, 0.19170397520065308, 0.07442309707403183, 0.09295263141393661, 0.2002004235982895, 0.10807454586029053, 0.1565174013376236, -0.07747562974691391, -0.0244031623005867, 0.008115451782941818, 0.01966264098882675, 0.04914793372154236, -0.16232864558696747, 0.16961108148097992, 0.11765089631080627, -0.1815994679927826, -0.11839848011732101, -0.28962811827659607, -0.09791300445795059, 0.1137174442410469, 0.0006755823269486427, -0.25308701395988464, -0.0831875279545784, -0.306486576795578, 0.056093037128448486, 0.25372228026390076, 0.027384551241993904, -0.16187968850135803, -0.09207456558942795, -0.16285526752471924, -0.11011749505996704, 0.13554473221302032, -0.010678582824766636, -0.04684014245867729, 0.004308031871914864, -0.07314377278089523, 0.11493321508169174, -0.16412560641765594, -0.17954964935779572, -0.1524176448583603, -0.18479619920253754, 0.14510096609592438, 0.11960161477327347, 0.2008286863565445, -0.326308012008667, -0.09839608520269394, 0.0363512821495533, 0.0009059645817615092, -0.12789680063724518, -0.05667617544531822, -0.13307583332061768, -0.11473266035318375, -0.15987153351306915, -0.008589623495936394, 0.17878971993923187], [-0.10624667257070541, 0.09258822351694107, -0.1425636112689972, 0.003916269633919001, 0.08782083541154861, 0.06114968657493591, 0.010866465978324413, -0.07012268155813217, -0.2637302875518799, 0.009512651711702347, -0.10849031805992126, 0.13303031027317047, 0.13760033249855042, 0.06114097312092781, 0.027389638125896454, 0.027297036722302437, -0.15732146799564362, -0.14680954813957214, -0.17017582058906555, -0.1433001309633255, 0.10885047912597656, -0.17312820255756378, 0.2499304860830307, 0.26048585772514343, -0.07355563342571259, 0.06335614621639252, -0.01669810153543949, 0.05531826987862587, 0.18215595185756683, -0.03838421404361725, 0.1419035643339157, -0.16070982813835144, 0.07049338519573212, 0.043788447976112366, 0.313088059425354, 0.09408005326986313, 0.042739205062389374, 0.05532627925276756, 0.038316015154123306, -0.10391180962324142, -0.04304315894842148, -0.01728641614317894, 0.15524865686893463, -0.056492067873477936, 0.003049860242754221, 0.008035116828978062, -0.1719711869955063, -0.11903680860996246, 0.016043223440647125, -0.010309855453670025, -0.16222766041755676, -0.14905133843421936, 0.17607849836349487, -0.11712251603603363, -0.2006126344203949, -0.2117619663476944, 0.042246658354997635, -0.058657899498939514, 0.05102013424038887, 0.04267582669854164, 0.023392731323838234, -0.11373244971036911, 0.14849799871444702, 0.2634676694869995], [-0.0736742913722992, -0.1422712802886963, 0.006499262526631355, -0.06444968283176422, 0.05441167205572128, 0.12038790434598923, 0.13600482046604156, -0.1722433716058731, -0.06742946803569794, 0.004907236434519291, -0.0017467211000621319, 0.25710055232048035, 0.11119939386844635, 0.020624307915568352, 0.06310942769050598, 0.10427293181419373, -0.26494842767715454, 0.18222840130329132, -0.06150588020682335, -0.17859141528606415, -0.04244875907897949, -0.1716238260269165, 0.1541191190481186, 0.21474935114383698, -0.26810571551322937, -0.019001103937625885, -0.09340670704841614, 0.08424023538827896, -0.014941214583814144, 0.10366307944059372, 0.1859750896692276, -0.060837019234895706, 0.0958642065525055, -0.0995175912976265, 0.2829274833202362, -0.030366405844688416, 0.06188912317156792, -0.015048319473862648, -0.11143781989812851, -0.17212451994419098, -0.10435706377029419, -0.16510793566703796, 0.1384427845478058, 0.11890632659196854, -0.04303692653775215, -0.17749740183353424, -0.08030130714178085, -0.14805877208709717, -0.05502212047576904, -0.0469479076564312, -0.08151430636644363, -0.05784473940730095, 0.08660272508859634, -0.055574752390384674, -0.20309396088123322, -0.2022588551044464, 0.036217495799064636, -0.038076575845479965, -0.027815742418169975, -0.20142462849617004, -0.06763742119073868, -0.046135708689689636, 0.13289238512516022, 0.12311125546693802], [0.03112594410777092, 0.07180455327033997, -0.15494617819786072, 0.01595211774110794, -0.05621745437383652, 0.11166395992040634, 0.10587942600250244, -0.06469231098890305, -0.026166576892137527, 0.05953572317957878, -0.07240675389766693, 0.1098853126168251, 0.2897535562515259, 0.028512150049209595, 0.16670066118240356, 0.1926160454750061, -0.3281865119934082, 0.00034502489143051207, -0.11734524369239807, -0.05699615180492401, -0.17157773673534393, -0.226126030087471, 0.06415453553199768, 0.1303432285785675, -0.08841060847043991, 0.001218561315909028, -0.06783633679151535, 0.23763975501060486, 0.1955840140581131, 0.15183843672275543, 0.05904665216803551, -0.06605406850576401, -0.004680297337472439, -0.036274269223213196, 0.26262399554252625, -0.030419031158089638, -0.1081383153796196, 0.0487624816596508, -0.05318708345293999, -0.09366351366043091, 0.028741924092173576, 0.046629633754491806, 0.2875903248786926, -0.03755183145403862, 0.0072264415211975574, -0.0536523200571537, -0.07997970283031464, -0.058171968907117844, 0.04552604630589485, 0.05561158433556557, 0.12847161293029785, -0.0021449001505970955, 0.07869401574134827, -0.05006843060255051, -0.19977957010269165, -0.056306347250938416, 0.08019417524337769, 0.13242056965827942, 0.23410144448280334, 0.015711139887571335, -0.14644837379455566, -0.16557466983795166, 0.20892156660556793, 0.19259564578533173], [0.20718009769916534, -0.03497910499572754, -0.14541104435920715, 0.10241134464740753, -0.14942382276058197, 0.17005027830600739, 0.1141129806637764, 0.05072790011763573, 0.13623353838920593, -0.17490144073963165, 0.1705782264471054, 0.029322464019060135, 0.22590802609920502, 0.002709595952183008, 0.24739181995391846, 0.19812680780887604, -0.09359799325466156, -0.09562192112207413, -0.012040819972753525, -0.15048432350158691, -0.16823455691337585, -0.33949804306030273, 0.08178798854351044, -0.007947896607220173, -0.210420623421669, -0.08048828691244125, 0.058533865958452225, -0.010903161019086838, 0.039879053831100464, 0.13877223432064056, 0.005190499126911163, 0.013328124769032001, 0.04854094982147217, -0.03267434611916542, 0.3419150710105896, -0.06226477771997452, -0.24078874289989471, 0.048341017216444016, -0.11588991433382034, -0.2698940336704254, 0.05139068141579628, 0.09558836370706558, 0.16362501680850983, 0.055374473333358765, 0.13681913912296295, 0.03885629028081894, -0.053781766444444656, -0.08139501512050629, 0.05702056363224983, 0.057295139878988266, 0.1002148687839508, 0.15901964902877808, 0.13976451754570007, -0.02639642544090748, -0.19581475853919983, -0.09321825951337814, 0.1901656985282898, -0.05617860332131386, 0.16874077916145325, 0.10913563519716263, -0.15580828487873077, -0.25028935074806213, 0.12903711199760437, 0.1502474546432495], [-0.15428891777992249, 0.08235195279121399, 0.03220916911959648, 0.23732946813106537, -0.12065087258815765, -0.0010800841264426708, 0.26707887649536133, 0.009799308143556118, 0.12130961567163467, 0.08278810232877731, 0.1766991764307022, 0.21667985618114471, -0.02268718183040619, 0.10970699042081833, 0.06508155912160873, 0.06155020743608475, -0.004074440337717533, -0.016926731914281845, -0.11684230715036392, -0.21754944324493408, 0.14577007293701172, -0.2943251430988312, 0.24200209975242615, 0.12739793956279755, -0.1508599817752838, -0.10714646428823471, -0.18163450062274933, 0.06649130582809448, -0.039578020572662354, 0.21907906234264374, -0.05728548765182495, -0.07178821414709091, -0.17877836525440216, -0.2685701251029968, 0.31627121567726135, 0.06695299595594406, -0.1937454640865326, 0.1462244987487793, -0.32442307472229004, 0.022180311381816864, 0.19669657945632935, 0.1842290610074997, -0.013879485428333282, 0.012823187746107578, -0.1627904623746872, 0.338318794965744, -0.1048288568854332, -0.08247943967580795, -0.25037291646003723, -0.08240677416324615, -0.009996121749281883, 0.14321953058242798, -0.03999429568648338, 0.09067373722791672, -0.10547646135091782, 0.06820934265851974, 0.19659551978111267, -0.25843796133995056, 0.1053602546453476, 0.14938873052597046, -0.2197498083114624, -0.19453459978103638, -0.010220006108283997, 0.05866166576743126], [-0.02146509848535061, -0.05166758596897125, 0.025757646188139915, 0.11046215146780014, 0.0044073122553527355, 0.02536371536552906, 0.08490368723869324, -0.05081788823008537, -0.11098075658082962, 0.02797034941613674, -0.03282351419329643, 0.1013999879360199, 0.028347352519631386, 0.03070262260735035, 0.18513190746307373, 0.041196174919605255, -0.0993540808558464, -0.05411810800433159, 0.01942477747797966, -0.02735544927418232, -0.10405104607343674, 0.028721751645207405, 0.20967420935630798, 0.09311345964670181, -0.2001742571592331, -0.006981203332543373, -0.15341515839099884, 0.06276015192270279, 0.07752992957830429, 0.0950097069144249, -0.14098134636878967, -0.060740936547517776, -0.16700109839439392, 0.05954902619123459, 0.23043935000896454, -0.0013672384666278958, 0.02118060737848282, -0.006692662835121155, -0.14946486055850983, -0.2610146105289459, 0.053376805037260056, 0.09316348284482956, 0.1138942539691925, 0.07129991799592972, -0.044020164757966995, -0.06617729365825653, -0.1570310741662979, -0.2664557099342346, -0.0023415936157107353, 0.05940885841846466, -0.06938279420137405, 0.07811744511127472, 0.05384896695613861, 0.05765266716480255, 0.04308198392391205, -0.13836455345153809, 0.21692392230033875, -0.25261810421943665, 0.11983215063810349, -0.21313826739788055, -0.3131241798400879, -0.04708811640739441, 0.0207244660705328, -0.07209334522485733], [0.0028104574885219336, -0.2104838639497757, -0.20459072291851044, -0.009433645755052567, -0.11614633351564407, 0.07728345692157745, 0.11709345132112503, 0.0027272263541817665, 0.09036149829626083, -0.06086428463459015, 0.041964489966630936, 0.19716164469718933, 0.18746830523014069, 0.0096008675172925, 0.12483401596546173, 0.008618426509201527, -0.2827211022377014, -0.02596493996679783, -0.049634210765361786, -0.2347594052553177, -0.12010378390550613, -0.3039341866970062, 0.3490850627422333, 0.10822837054729462, -0.21651598811149597, -0.014172974042594433, -0.08616000413894653, -0.03404640406370163, 0.15629763901233673, 0.07101748883724213, 0.208200141787529, -0.04182576388120651, -0.21995729207992554, 0.045691389590501785, 0.15225186944007874, 0.033616699278354645, -0.10670637339353561, 0.09853530675172806, -0.15362520515918732, -0.2495976686477661, 0.09574317187070847, -0.17881125211715698, 0.045633796602487564, -0.039566610008478165, 0.06571116298437119, -0.02231195755302906, -0.035947296768426895, -0.15334241092205048, -0.014848784543573856, 0.02434157393872738, -0.00782499648630619, -0.011257817968726158, 0.10161631554365158, -0.022336121648550034, 0.11301642656326294, -0.07464004307985306, 0.32576438784599304, -0.19743219017982483, 0.10740795731544495, -0.20179437100887299, -0.07498947530984879, -0.02504481002688408, 0.16195107996463776, 0.09274806827306747]]]], "dtype": "float32"}, {"__nd__": [[[5.4057536125183105, 5.341392993927002, 5.379648208618164, 5.284099578857422, 5.252420902252197, 5.2874016761779785, 5.318326950073242, 5.35424280166626, 5.280433654785156, 5.5599799156188965, 5.182994842529297, 5.416094779968262, 5.2649641036987305, 5.401705741882324, 5.512894153594971, 5.50407075881958, 5.262062072753906, 5.259099960327148, 5.4342217445373535, 5.226678848266602, 5.150973796844482, 5.272641658782959, 5.146462917327881, 5.643019199371338, 5.668093681335449, 5.5899763107299805, 5.078649997711182, 5.359960079193115, 5.829004287719727, 5.189174652099609, 5.649960041046143, 5.352635860443115, 5.222195148468018, 5.49534273147583, 5.381554126739502, 5.19862699508667, 5.226426124572754, 5.366671562194824, 5.253686904907227, 5.538792610168457, 5.592543125152588, 5.462086200714111, 5.601975917816162, 5.294092655181885, 5.62804651260376, 5.351057529449463, 5.594883441925049, 5.463444232940674, 5.211041450500488, 5.000007152557373, 5.273276329040527, 5.337090969085693, 5.357470512390137, 5.586363792419434, 5.387570858001709, 5.440739154815674, 5.666996002197266, 5.165087699890137, 5.510788440704346, 5.372124195098877, 5.386765003204346, 5.277244567871094, 5.585468769073486, 5.1117987632751465, 5.446645259857178, 5.336373805999756, 5.451465129852295, 5.189075469970703, 5.350935935974121, 5.364633560180664, 5.274087905883789, 5.388164520263672, 5.325350284576416, 5.393206596374512, 5.466583251953125, 5.849306583404541, 5.713338375091553, 5.471903324127197, 5.309850215911865, 5.1180033683776855, 5.350902080535889, 5.426681995391846, 5.0145955085754395, 5.520869255065918, 5.234375, 5.492612838745117, 5.497753620147705, 5.305887699127197, 5.532834529876709, 5.402626991271973, 5.48424768447876, 5.250932693481445, 5.304427623748779, 5.357319355010986, 5.846662521362305, 5.136931419372559, 5.422964572906494, 5.49345588684082, 5.174980640411377, 5.511257648468018, 5.484004020690918, 5.096328258514404, 5.314453601837158, 5.616788387298584, 5.321011066436768, 5.3671555519104, 5.406774044036865, 5.520817756652832, 5.494844436645508, 5.200485706329346, 5.30009651184082, 5.584266185760498, 5.550596237182617, 5.601685047149658, 5.459168434143066, 5.398799419403076, 5.27583122253418, 5.651593208312988, 5.211354732513428, 5.332790374755859, 5.436461925506592, 5.1489410400390625, 5.353031635284424, 5.278176784515381, 5.2089691162109375, 5.106616020202637, 5.5479655265808105, 5.521016597747803, 5.17120361328125, 5.409467697143555]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[[1.5740818977355957, -1.07892906665802, -4.7174763679504395, -2.7429604530334473, -2.8573999404907227, 1.6254069805145264, 2.80435848236084, -1.3148393630981445, 0.25372546911239624, -9.595868110656738, 1.2929844856262207, -3.5016469955444336, 1.5271356105804443, -0.3827640116214752, 2.487880229949951, 3.608469009399414, 3.6334547996520996, -2.671996593475342, 1.3149378299713135, 5.608603000640869, 2.7075254917144775, -1.1416242122650146, 2.4829485416412354, -3.1934516429901123, 2.455451011657715, 4.644442558288574, 4.372570037841797, -0.3386026620864868, 0.7843685150146484, -5.001470565795898, 2.3644843101501465, -2.379215717315674], [1.0205638408660889, -1.2549514770507812, -3.318948745727539, -4.1979756355285645, -10.64285945892334, 3.384002685546875, 4.383073806762695, -0.38486865162849426, -0.22207346558570862, -1.6091117858886719, -0.6144877672195435, -5.698424339294434, 3.3472275733947754, 1.9507198333740234, -1.6863552331924438, -2.5953257083892822, 3.6252050399780273, 5.343790054321289, -3.0323500633239746, -0.7678179740905762, -4.904158115386963, -6.11306095123291, -2.254106044769287, -0.6040751934051514, 4.2495808601379395, -0.6271096467971802, 2.144261360168457, -2.845869541168213, -1.0867266654968262, -5.416757583618164, -0.6539786458015442, 0.625508189201355], [-1.3965656757354736, -0.28102046251296997, 0.15726079046726227, -0.7257308959960938, -3.5659608840942383, -2.156144142150879, -1.0903453826904297, -5.021176338195801, -2.563877582550049, 2.3942697048187256, -4.977482795715332, -2.2595841884613037, -3.600773811340332, -0.2501068711280823, 3.098273754119873, 0.36016136407852173, 1.5931122303009033, -2.681403398513794, 2.73044490814209, -2.213939905166626, 0.8570964932441711, 2.254324197769165, 4.3683905601501465, 4.297125816345215, -7.658788204193115, -4.819316864013672, 2.556553840637207, -0.16844706237316132, 2.2614054679870605, 2.4175355434417725, 2.2784066200256348, -0.5705776214599609], [4.877921104431152, -2.216352939605713, 5.925745964050293, -0.7223111391067505, 0.6446807980537415, -1.6435282230377197, 4.72078275680542, 1.4487338066101074, -2.11753249168396, 2.7044239044189453, -4.0517072677612305, 0.8744527101516724, -1.2218232154846191, -5.382803916931152, -1.8372842073440552, -1.4519901275634766, -1.8910928964614868, 3.9013285636901855, 1.9308793544769287, 1.2003082036972046, -1.7408668994903564, 4.343044281005859, 2.831632614135742, 1.6259589195251465, 2.928225040435791, 1.563584804534912, 0.6335110664367676, 4.1579155921936035, -0.03573773801326752, 1.0641934871673584, 0.9144948720932007, 2.1838717460632324], [1.727855920791626, 7.105114459991455, 6.264497756958008, 2.263204336166382, 3.3826093673706055, -0.1954536736011505, 2.721047878265381, 4.413736343383789, 2.5566651821136475, -2.5735127925872803, 4.278875350952148, 6.655950546264648, 2.223942518234253, -0.16357338428497314, -0.4162489175796509, 2.962160587310791, 4.0828166007995605, -3.754784345626831, -2.1766271591186523, 3.5657200813293457, 3.5313777923583984, 8.252578735351562, 5.402468681335449, 1.1240403652191162, -3.491299867630005, -5.473763942718506, -0.2951301336288452, -2.15468168258667, -0.01649649627506733, 2.9149067401885986, -0.5417860150337219, -0.986740231513977], [1.7209839820861816, -1.4189422130584717, -1.5341384410858154, -7.651529312133789, -0.8575961589813232, -2.1102025508880615, -1.9723048210144043, 3.514495611190796, -0.6138926148414612, 2.472867727279663, -1.47352135181427, 0.49000364542007446, 1.0475974082946777, 2.2112255096435547, -2.3118081092834473, 1.629430890083313, -1.42534339427948, 0.47830307483673096, 0.0913727805018425, 0.14934051036834717, -1.7719240188598633, -1.5597615242004395, 5.77877140045166, -5.108079433441162, 3.385758876800537, -3.2853403091430664, 0.11608165502548218, -0.6429153680801392, 1.9944627285003662, -4.748310089111328, -5.822744369506836, -0.5856543183326721], [3.156536102294922, 0.4769900143146515, -1.7412917613983154, -3.705132484436035, 0.20479145646095276, 2.6049773693084717, 2.9186463356018066, 1.998085856437683, 0.17388015985488892, 1.349608302116394, 3.341552495956421, 4.543821334838867, 0.03594823181629181, 2.2602357864379883, -6.325078010559082, -3.074974536895752, 1.5522100925445557, 0.6196382641792297, 0.2295381724834442, 1.7948945760726929, -1.0131738185882568, 2.467629909515381, 1.819108486175537, -6.171854019165039, -5.1633758544921875, 3.743870973587036, -0.9797489047050476, 0.6697155237197876, -1.6115922927856445, 0.2530386745929718, 2.055300712585449, -3.7518460750579834], [-0.6824663877487183, -2.839240074157715, 2.4382400512695312, -4.761370658874512, -0.3826853632926941, -2.7291488647460938, 2.5901103019714355, 0.5003658533096313, 3.6546473503112793, 0.6710828542709351, -0.36677396297454834, -3.549874782562256, -1.31624174118042, -4.216681957244873, 1.5412318706512451, 5.293603420257568, 5.319218635559082, -2.9342641830444336, -2.1973342895507812, -1.1199299097061157, 4.654021263122559, -4.199152946472168, -0.9155265092849731, -3.481265068054199, -0.32285076379776, -0.7818655967712402, 1.274012565612793, 0.5460766553878784, 8.071629524230957, -1.8981311321258545, -0.39099204540252686, -0.027264941483736038], [4.020947456359863, 0.35218310356140137, 1.4630666971206665, 5.026364326477051, -4.753389835357666, 1.160413146018982, 1.4965484142303467, -0.5880230665206909, -1.9933757781982422, 3.361140251159668, 0.7285520434379578, 2.093945026397705, 1.4421627521514893, -1.029129147529602, -0.33190783858299255, -0.05299157649278641, 3.1131997108459473, 1.4114961624145508, -4.365852355957031, 3.643932342529297, 0.9953765273094177, -1.3616418838500977, 1.6254998445510864, -2.3727622032165527, -2.307642936706543, -1.312637209892273, 0.8400748372077942, 1.6972321271896362, 3.7181332111358643, -3.13338565826416, 3.007936954498291, -2.4126944541931152], [1.2671502828598022, -5.139763832092285, -0.16123399138450623, 0.021895425394177437, -3.3795511722564697, -1.737645149230957, 2.6032052040100098, -2.944166421890259, 0.7967740297317505, 2.5093812942504883, 3.832581043243408, -0.06928934156894684, -1.0129826068878174, 2.2276294231414795, 7.067752838134766, 2.812986373901367, 5.894896984100342, -3.0947978496551514, -3.6383395195007324, 4.7423906326293945, -0.16838733851909637, 5.704293727874756, 4.691391468048096, 4.068520545959473, -1.2180259227752686, 1.4869393110275269, -9.176648139953613, -2.4565558433532715, -0.49191418290138245, 0.8114640712738037, 2.052886486053467, 2.388917922973633], [-1.8605350255966187, 1.703894019126892, -1.8983899354934692, -1.4027159214019775, -3.5943284034729004, 0.7256196737289429, 0.4080684781074524, 1.0456547737121582, -3.566453218460083, -3.264397621154785, -1.437193512916565, -3.7562804222106934, -3.7686543464660645, -0.10984833538532257, -5.97430419921875, -2.135571241378784, -0.14549149572849274, 1.4096078872680664, 0.4974622130393982, -2.2133853435516357, -5.314774513244629, 4.059648036956787, 1.6008048057556152, -1.2258293628692627, 1.0728474855422974, 0.7743116617202759, -4.079195976257324, 0.046956419944763184, -0.2593701183795929, -5.301091194152832, 0.018579570576548576, 1.6568715572357178], [-0.9728049039840698, -1.2162246704101562, 3.8466029167175293, -0.4007216691970825, 5.60402774810791, -1.6767313480377197, -4.055995941162109, -4.3197174072265625, 3.2575621604919434, 0.293517529964447, 2.746344566345215, -2.7383828163146973, -4.099077224731445, -0.1478257179260254, -3.333019733428955, 2.5112075805664062, 4.151156902313232, -0.6027580499649048, 2.9176158905029297, -0.9409631490707397, 6.950554847717285, 1.1260902881622314, 2.6022214889526367, 2.1889686584472656, -1.5950006246566772, 1.7795565128326416, -0.725368082523346, -6.142361640930176, 2.8344967365264893, 0.17863570153713226, 1.9545912742614746, -2.696240186691284], [-5.127673625946045, -0.31289035081863403, 1.3411433696746826, -2.519404888153076, -2.4456076622009277, 1.3090145587921143, 3.308361768722534, 0.9617988467216492, -0.23088842630386353, 2.4016785621643066, 1.3431035280227661, 4.638869762420654, -0.07829651236534119, -0.14080208539962769, -5.5455169677734375, 1.111198902130127, -3.1358392238616943, 3.3246192932128906, 1.9059287309646606, -0.05855419859290123, -1.7376219034194946, 2.8889217376708984, 5.2175984382629395, -1.0997819900512695, 6.079755783081055, -3.3309223651885986, 6.884180068969727, -3.553788185119629, -4.497341632843018, 4.416660308837891, -0.765038013458252, 0.28081274032592773], [5.481734752655029, -1.3526577949523926, -0.22689007222652435, 0.29495757818222046, 1.608469009399414, 1.5159683227539062, 1.0392537117004395, 0.404363214969635, -4.0920305252075195, 3.436795234680176, -6.1138916015625, -1.616969347000122, 2.2458417415618896, -1.773144006729126, -0.5069020986557007, 1.413671612739563, -5.039857387542725, -6.6158246994018555, -1.460678219795227, -1.8565459251403809, 0.9600679278373718, 3.077962875366211, 1.8874452114105225, -1.327904462814331, -1.459651231765747, -0.08289039134979248, -0.7603855133056641, 2.3187851905822754, 3.2675695419311523, -1.1784883737564087, -2.1913769245147705, -0.8866093754768372], [0.5355671644210815, -2.256718397140503, 1.5116298198699951, -0.7679139375686646, -1.5443289279937744, 6.42435359954834, 2.8808517456054688, 3.241011619567871, 1.9092302322387695, -0.12373174726963043, 0.44974184036254883, -0.021269412711262703, -4.333266258239746, 1.5991253852844238, 0.08818238973617554, 5.192226409912109, 2.3781518936157227, 5.115747451782227, -2.835172176361084, 1.6605172157287598, 3.8712050914764404, -2.8372669219970703, 2.0762057304382324, 5.467483997344971, -0.910690426826477, 8.25836181640625, 3.113664150238037, -2.5893445014953613, 3.584566116333008, -2.402082920074463, -0.18929773569107056, -1.620927095413208], [-0.4580514430999756, -1.0924264192581177, 3.0712552070617676, 1.061582326889038, -4.028149604797363, -1.640052080154419, 1.9107674360275269, 0.16197213530540466, -3.180438995361328, 1.928499698638916, 4.994260787963867, 0.51401686668396, 1.5234278440475464, 0.6423723101615906, 2.793572187423706, -1.6129143238067627, 0.5578751564025879, -0.6446722745895386, 0.49462807178497314, -1.5705280303955078, -1.6093189716339111, 2.1171250343322754, 5.31795597076416, 4.628623962402344, -2.1017913818359375, 2.1768014430999756, 1.829895257949829, 7.694632530212402, -0.7353938817977905, -2.6100072860717773, 3.4371237754821777, -0.1557139754295349], [3.150580644607544, 1.1739094257354736, 3.6038453578948975, 0.8586156964302063, -4.883047103881836, 3.4511938095092773, -0.2294074147939682, 0.4322817325592041, 1.6718792915344238, 1.7629029750823975, -0.987470269203186, 0.29232531785964966, -2.3574492931365967, 3.0121827125549316, -1.9843671321868896, 0.6402207612991333, -3.469857692718506, -3.5618810653686523, 2.5228612422943115, -4.377840518951416, 0.26146239042282104, -1.967752456665039, 4.038102626800537, 5.311824798583984, 1.1895778179168701, -2.8053879737854004, 6.987707138061523, -0.8539899587631226, -0.41867226362228394, -0.7516220211982727, -1.720346450805664, -4.365320682525635], [-3.5332982540130615, 1.9867701530456543, -4.258862018585205, 1.041611671447754, -2.55887508392334, -0.7163487672805786, 0.9857938885688782, 0.9316351413726807, 2.6471972465515137, 3.803109884262085, 1.4808435440063477, 1.94602370262146, -2.220141887664795, 0.2220594584941864, -0.7906248569488525, 1.2426081895828247, 0.8651970624923706, 7.480924129486084, 2.1573386192321777, 6.100029945373535, -1.3923324346542358, -3.321073293685913, 5.890407562255859, 1.166025161743164, 3.037092685699463, -1.276792287826538, -0.7977893352508545, 5.55472469329834, 1.5223667621612549, -5.015482425689697, 3.4503278732299805, -0.17848633229732513], [5.761567115783691, 0.30426767468452454, 2.248542308807373, -1.8120265007019043, 2.8896992206573486, 3.9286818504333496, -3.0321173667907715, -1.0526645183563232, -0.9306600093841553, -3.9006285667419434, -1.5929005146026611, 5.68020486831665, -7.73294734954834, -0.4333801567554474, 3.046534538269043, 0.27133703231811523, -1.282433032989502, 2.9827301502227783, 1.8694992065429688, 2.690103769302368, 3.2787013053894043, -1.7959797382354736, 5.105741500854492, 6.9709930419921875, 3.1070146560668945, -4.169272422790527, 5.953140735626221, 2.825498104095459, -4.387328147888184, 1.157470703125, 0.3700914978981018, 1.0400793552398682], [0.8842154741287231, 1.2590535879135132, 1.6494247913360596, 0.20900237560272217, -2.278616428375244, -3.7191429138183594, -4.522856712341309, -0.9514369368553162, -1.55995512008667, -3.0725135803222656, -2.6295835971832275, -6.334561347961426, 0.6623568534851074, 3.164442539215088, -3.0839791297912598, 4.346813201904297, -1.4097563028335571, -4.246262550354004, 2.6459875106811523, -1.0312001705169678, 1.246798038482666, 7.364524841308594, 0.46801912784576416, -0.584628164768219, 1.021541714668274, -0.870833158493042, 1.242713451385498, -0.3634716272354126, 0.0421217642724514, 1.9415347576141357, -1.0849007368087769, 1.1259628534317017], [-0.3181145191192627, -3.975144863128662, -2.568535566329956, 1.6572020053863525, 6.43064022064209, -3.4254417419433594, -0.9062057733535767, -4.363699913024902, -2.655848503112793, 0.2133442908525467, 4.017032146453857, 6.811911582946777, -1.106550693511963, 0.4927634000778198, -3.0167288780212402, 1.2729368209838867, 2.3469433784484863, -0.2780815362930298, -3.6240999698638916, -0.9301632642745972, -2.4141342639923096, 0.8956988453865051, 1.5016261339187622, 1.9127122163772583, -3.289283037185669, 2.917416572570801, -1.810113787651062, -3.6748838424682617, -1.8161747455596924, -1.999804973602295, -3.6648309230804443, 3.740445137023926], [-1.8684706687927246, 5.289854526519775, -0.3346625566482544, -0.6647554039955139, 3.2425360679626465, -2.84806752204895, -1.051276683807373, -4.713921070098877, 0.2281186580657959, 0.8494210839271545, -5.100956916809082, 1.4896223545074463, 2.2441282272338867, 4.169708251953125, -1.3495557308197021, -3.9194118976593018, -1.4814672470092773, -1.6054167747497559, 0.556687593460083, 0.8362793326377869, -7.675185203552246, 1.663161039352417, -2.1319918632507324, 0.7041254043579102, -6.922650337219238, -1.247147798538208, 4.617153644561768, -2.9902806282043457, 2.409574270248413, 1.4744497537612915, 3.5089244842529297, -0.14754682779312134], [0.5157145857810974, -1.6319220066070557, -7.5955424308776855, 5.473915100097656, -3.1258039474487305, -4.844374179840088, 4.360935211181641, 0.06684708595275879, 1.3523180484771729, -2.905503988265991, 1.0945889949798584, 1.2352163791656494, 0.4413336515426636, -4.8416218757629395, 2.2172203063964844, 1.2848734855651855, -9.392069816589355, -2.979734420776367, 3.9586708545684814, -0.8823747038841248, -3.6278576850891113, 2.302474021911621, -1.5027995109558105, 2.1059141159057617, 2.639146566390991, -1.3379616737365723, 1.6126155853271484, 1.8475241661071777, 2.6456356048583984, -6.261810302734375, -0.633203387260437, 1.971695065498352], [-3.573289394378662, -4.739480495452881, -1.0666706562042236, -3.011218786239624, -0.30845487117767334, 2.0851731300354004, -0.21688464283943176, 2.73643159866333, 1.3828036785125732, -0.021269099786877632, -5.699021816253662, 6.361003875732422, 4.10821008682251, -4.208477973937988, 1.778247356414795, -2.777953863143921, 6.791131019592285, -6.614663600921631, -2.690929651260376, 0.9770873785018921, 1.566370964050293, 1.1951677799224854, 0.8428726196289062, 1.0226242542266846, 1.4968448877334595, -2.8533530235290527, -3.816523790359497, 2.2228193283081055, 0.6995238661766052, 0.02040187083184719, 0.6099002361297607, 1.5919268131256104], [3.7191872596740723, -0.43378156423568726, 0.16025114059448242, 2.15502667427063, -0.4112708866596222, -0.36363914608955383, 2.2276878356933594, -5.701439380645752, 2.6985061168670654, -3.5995309352874756, 0.7083133459091187, -0.44283270835876465, -0.6692137122154236, -2.367124080657959, -1.2637161016464233, -0.025518085807561874, -6.687725067138672, 1.5606927871704102, 3.7482404708862305, -0.1552453637123108, -1.3869690895080566, -3.9847605228424072, 3.4898505210876465, -1.4150787591934204, -2.101630926132202, -3.4208269119262695, 4.339860439300537, -2.6567044258117676, 2.50399112701416, -2.7428479194641113, 4.377326965332031, -0.21398185193538666], [2.026456117630005, 6.825160980224609, 0.09643429517745972, 0.6538752317428589, -3.0985145568847656, 1.455104947090149, -3.9068593978881836, 4.912999629974365, -3.85975980758667, -6.054117679595947, -1.7180507183074951, 6.3989481925964355, 7.629963397979736, -7.737987041473389, -1.1611186265945435, 2.4578187465667725, 1.201385498046875, -3.5937066078186035, 4.101601600646973, 4.716037750244141, -1.8614366054534912, 6.759798049926758, 3.7265560626983643, 1.1246190071105957, 2.5352983474731445, 0.6396315693855286, -0.10109449923038483, 0.120725616812706, 0.0855870321393013, 0.7490410208702087, 0.9134640097618103, -1.2971844673156738], [-0.3177488446235657, 3.736006259918213, -2.090351104736328, 4.075812816619873, -1.4805171489715576, 3.9807629585266113, -5.69731330871582, -0.4577396512031555, 1.8623210191726685, 1.2612051963806152, 2.1583876609802246, 2.3192129135131836, -4.322612762451172, 2.963928699493408, -3.4259486198425293, 2.620029926300049, -4.041436195373535, -2.4377219676971436, 2.1485092639923096, 0.5036560297012329, -0.8654515743255615, 1.7247837781906128, 4.835208415985107, 2.004883289337158, -2.8251254558563232, -6.630851745605469, 3.615868091583252, -0.36035072803497314, -1.7568719387054443, 0.36041495203971863, -1.511011004447937, -1.648066520690918], [-0.8807696104049683, 1.8012690544128418, -2.029376983642578, -2.313833713531494, 3.342834949493408, 0.5991684198379517, -2.0888686180114746, -5.02842378616333, 1.3472234010696411, 0.7138468623161316, -2.794128656387329, 0.5084645748138428, 2.8197262287139893, 6.529142379760742, -2.393399477005005, -3.925121307373047, 1.6746962070465088, 0.17494603991508484, -5.0788984298706055, -0.3697702884674072, 1.5797977447509766, -1.9818792343139648, 4.134018898010254, -3.9908246994018555, -0.5931181907653809, -2.332395553588867, -3.187732696533203, -3.877614974975586, -7.253462791442871, -2.1155409812927246, -1.0353386402130127, 1.44697904586792], [-5.310915946960449, 1.861020803451538, 1.8220667839050293, -1.6802124977111816, -4.465070724487305, -1.4575833082199097, 4.38222074508667, 2.659998655319214, 4.225668430328369, 4.173520565032959, 0.8425961136817932, 2.1769495010375977, -0.43868428468704224, -5.041501998901367, 1.9956257343292236, -1.3650386333465576, -0.6236505508422852, 1.971719741821289, 0.5844987034797668, -1.8625526428222656, -5.830735683441162, 1.4709546566009521, -2.4236154556274414, -8.246110916137695, 1.4923033714294434, -1.5303966999053955, -1.8118877410888672, -0.26538488268852234, -1.612295150756836, -1.8676918745040894, 1.4256675243377686, -0.7858543395996094], [2.6299614906311035, -0.9622535705566406, 1.3081974983215332, 6.938412666320801, -5.518848896026611, 0.16934047639369965, 0.2826501131057739, 4.010766506195068, -0.8730921149253845, 2.4305477142333984, 6.946713447570801, 4.4374237060546875, 0.326376348733902, -2.5978193283081055, 1.1731541156768799, 4.301287651062012, 0.05505765229463577, -3.0885634422302246, -1.6434993743896484, 1.5483522415161133, -4.302244663238525, 0.7157719731330872, -2.621265172958374, -0.19587329030036926, -0.2933564782142639, -0.17723935842514038, -5.552369594573975, -0.14376911520957947, -0.8369688987731934, -2.9263010025024414, -0.6555765867233276, 2.7956764698028564], [2.138559579849243, 1.520447015762329, -7.286187171936035, 0.9878606796264648, -2.0251646041870117, 4.7740936279296875, 1.2812113761901855, 7.62857723236084, -2.0707850456237793, 2.445953369140625, 1.658169150352478, 2.6188297271728516, -2.5924253463745117, 2.915505886077881, -5.436689376831055, -5.103245735168457, -4.775353908538818, -0.4213009476661682, 0.016658280044794083, 2.795518398284912, 0.9193021059036255, 7.91276741027832, -0.9027023315429688, -0.7653119564056396, 0.4778996407985687, 0.5656095147132874, 2.6596240997314453, -2.9449853897094727, -1.4874327182769775, 6.612875938415527, -4.986213207244873, -3.451063871383667], [3.148643970489502, -0.34062981605529785, 0.16325779259204865, -0.3348763585090637, -4.834024429321289, -0.6605551242828369, 4.511794090270996, -3.7532505989074707, 2.304762601852417, -2.64127254486084, 0.2354537397623062, -4.957441329956055, -0.4977509379386902, -4.042383670806885, 4.271184921264648, 9.440196990966797, -0.7727618217468262, -0.5148907899856567, -6.426266670227051, 1.2887839078903198, -1.9459564685821533, -0.3128998875617981, -0.7881685495376587, -0.6626108884811401, 1.0468112230300903, -2.887704849243164, -2.073719024658203, -2.3679006099700928, -1.949871301651001, 1.0841096639633179, 0.3147716224193573, -6.046769142150879], [0.29619842767715454, 2.693240165710449, 6.932620048522949, 0.7595480680465698, -2.005688428878784, 1.3605576753616333, -0.8440356254577637, -1.5536446571350098, 3.133620500564575, -0.6580483913421631, 1.8127200603485107, -1.266557216644287, -0.32169926166534424, -0.2906455993652344, -3.011178493499756, -0.3732445240020752, 0.25072261691093445, 3.9042627811431885, -4.158222198486328, -6.174924850463867, -3.5954184532165527, -1.9601001739501953, -0.9881948232650757, 2.1250200271606445, 4.877486705780029, 1.4204788208007812, 0.5670484304428101, 0.6224616169929504, 1.4275033473968506, 2.620713710784912, -3.904956340789795, 1.5856819152832031], [-2.0082690715789795, -0.9641447067260742, 2.423889636993408, -1.7704341411590576, -3.6593620777130127, 0.18918442726135254, -2.6124284267425537, -0.181741863489151, -1.7236604690551758, -4.2640204429626465, 1.0598113536834717, -3.255465507507324, -0.14838522672653198, -5.370611667633057, 2.587418556213379, 2.890820026397705, -2.2851059436798096, 3.026014566421509, -5.1442999839782715, -1.9556071758270264, 0.5951380729675293, -5.876123428344727, 0.10037199407815933, 3.836650848388672, -2.7888381481170654, 2.5966687202453613, -4.514674663543701, 1.641059398651123, -3.361884117126465, -1.5023113489151, 0.41029196977615356, 5.19586706161499], [2.212183952331543, 0.9399807453155518, 0.39707738161087036, 1.3539096117019653, -0.80089271068573, -6.862940788269043, 4.556224346160889, 5.147145748138428, -1.7909762859344482, -1.9887230396270752, -2.3730647563934326, 0.8847965598106384, 2.7099175453186035, -1.339190125465393, -3.5337259769439697, 2.180603265762329, -0.7648054361343384, -0.7445482015609741, -4.5811662673950195, -2.0805726051330566, -0.9611679911613464, 2.241819381713867, -3.1073641777038574, 4.574007511138916, -2.8853137493133545, -3.1820359230041504, 4.250918388366699, 2.3613288402557373, 0.21343061327934265, 1.8429011106491089, 2.1830577850341797, 2.0952682495117188], [-1.7052580118179321, 2.190472364425659, -3.0018820762634277, -0.5051891803741455, 8.337968826293945, 4.362154483795166, 2.141679286956787, -1.2003037929534912, 1.3150486946105957, 1.3453093767166138, -4.851165771484375, -3.8944437503814697, -3.383011817932129, 0.6787256002426147, -2.9392666816711426, 2.7068140506744385, -1.982892394065857, 1.3563449382781982, -0.5921940803527832, 7.024899959564209, -1.664396047592163, -4.057587623596191, -0.6787232756614685, -1.052317500114441, 5.0014801025390625, -5.851200580596924, -3.8980917930603027, -0.323108971118927, 1.0135483741760254, -1.0785762071609497, -0.30946096777915955, -5.056252956390381], [0.10931743681430817, -1.0786690711975098, 2.305349826812744, -2.328150987625122, 0.7956646084785461, 0.34436139464378357, -3.841656446456909, 6.168635368347168, 5.185699939727783, 5.261334419250488, 1.6208205223083496, 2.126884937286377, -3.918639659881592, -4.894430637359619, -4.15356969833374, 3.2570128440856934, -0.8261276483535767, -5.4110846519470215, 3.7836947441101074, 4.62491512298584, -1.3305010795593262, -5.749178409576416, -2.2768688201904297, -0.27664440870285034, -2.1512277126312256, 1.8328452110290527, 2.488913059234619, -0.3031727075576782, 1.987436056137085, 2.983436346054077, 0.04890386760234833, 3.5873680114746094], [-3.0901694297790527, 0.4245622754096985, 3.163456916809082, 0.19438102841377258, 4.2223286628723145, -0.02037702687084675, 3.162914276123047, 0.23597648739814758, 3.619558811187744, -2.186386823654175, -1.3943259716033936, -4.445235252380371, -1.2706023454666138, 0.9982626438140869, 0.07953642308712006, 1.2958970069885254, 2.6068930625915527, 0.6039534211158752, 3.7664079666137695, 1.4994053840637207, 3.0716161727905273, -2.5618603229522705, -2.4223198890686035, 4.444324970245361, 1.6432719230651855, -2.397970676422119, -2.715930223464966, -4.745846748352051, 0.5755723714828491, -2.302137851715088, -1.596540927886963, -4.498280048370361], [-1.6588129997253418, -2.32010817527771, 3.7242302894592285, 2.1294898986816406, -0.14123883843421936, -1.8004672527313232, 0.016361407935619354, 2.531771659851074, 2.1259214878082275, -5.486084461212158, 1.338894248008728, -1.270066261291504, -0.4908972680568695, -7.648800373077393, 0.7487324476242065, -3.319359540939331, -0.7545463442802429, 3.044674873352051, 2.4420359134674072, 1.3799166679382324, 0.023238668218255043, 1.9740275144577026, 6.174806594848633, -1.594423532485962, 2.5846900939941406, -1.482535481452942, -3.2022035121917725, 4.016496658325195, 3.398430347442627, -2.333883762359619, 0.7403728365898132, 3.068101406097412], [-2.553964138031006, 2.4274418354034424, 0.3939613401889801, -0.987903356552124, 1.1009163856506348, 3.418257236480713, -1.212792158126831, 0.4671803116798401, 1.948585033416748, 0.4951862096786499, -1.559644103050232, -0.0643230527639389, -5.605167865753174, 2.2553439140319824, -5.591001510620117, 2.32878041267395, 3.5368947982788086, 4.197024822235107, -1.4058668613433838, 4.307085037231445, 0.08520737290382385, 0.12030830979347229, 0.324577659368515, 2.2793993949890137, 8.268987655639648, 2.003706455230713, -0.6852712035179138, 5.299979209899902, -4.1118245124816895, -1.2555338144302368, 3.023270845413208, 0.032313838601112366], [-5.077888488769531, 6.730001449584961, -4.174417495727539, 8.410526275634766, -0.06745409965515137, 0.25192099809646606, 1.6225252151489258, 0.4746333360671997, -2.014604091644287, -2.104870557785034, -0.8674124479293823, -1.874971866607666, -0.8879415392875671, -2.3859403133392334, 1.2586240768432617, -1.6294968128204346, 1.919501781463623, 2.273468017578125, 0.40765368938446045, 0.5727555751800537, 0.46716079115867615, -0.19763243198394775, 0.603759765625, -0.397757887840271, 3.3703901767730713, -0.0770907998085022, 2.3083972930908203, -1.6304779052734375, -3.1916441917419434, 1.8130552768707275, 0.00577327236533165, 5.820884704589844], [-3.5502710342407227, 2.171779155731201, -2.3615856170654297, 1.151323676109314, 2.19230580329895, -0.29019811749458313, -3.8882737159729004, 1.837615728378296, -3.991408109664917, -3.169196605682373, 2.157529830932617, -6.371611595153809, -1.4146605730056763, -2.926389694213867, -3.5238943099975586, -4.553923606872559, -2.8006701469421387, -1.9700212478637695, -1.5681159496307373, -1.801100492477417, 0.648129940032959, -2.5046372413635254, -0.8231474161148071, 3.115858554840088, -1.0670301914215088, -2.2490005493164062, 2.175260066986084, -0.4969364404678345, 0.691157877445221, -3.243699073791504, -2.6826252937316895, 2.1463136672973633], [0.8611751794815063, -1.426138162612915, -1.261475920677185, 0.8648298978805542, -0.17153790593147278, 0.46664541959762573, -0.44752323627471924, -0.6832708120346069, -2.8207805156707764, 1.2819623947143555, 2.69785737991333, 0.7706936597824097, -1.3098609447479248, 3.1028923988342285, 3.8044791221618652, 3.5366954803466797, -1.7593828439712524, 0.1938190758228302, -1.6941032409667969, -4.55255651473999, 1.6392772197723389, 4.511989593505859, 4.019391059875488, -1.5553154945373535, -3.592221736907959, -6.716006755828857, 1.1059556007385254, -2.6150741577148438, 3.3629274368286133, 4.077401638031006, 4.399981498718262, -5.654549598693848], [-1.4072582721710205, -4.2127227783203125, 1.281450629234314, 2.5191004276275635, 1.645134687423706, 3.5315744876861572, -0.010269735008478165, 0.7369121313095093, 4.367707252502441, -0.6372014284133911, 1.248490810394287, 4.563384056091309, 3.657658576965332, -5.346635818481445, 0.7321979999542236, 1.442711353302002, -0.6042143106460571, 2.447312355041504, -9.029671669006348, -4.9815521240234375, 0.16484293341636658, -2.035646915435791, -1.2258707284927368, -2.072965145111084, -0.01798282377421856, 2.283529758453369, 0.5776784420013428, 5.948842525482178, 1.4378141164779663, 1.4308693408966064, -0.21149080991744995, -3.4042317867279053], [-0.6220728158950806, -1.9473419189453125, -0.5055801868438721, 5.902798652648926, 3.691235065460205, -4.99252986907959, -3.554986000061035, 3.18552827835083, 2.9607043266296387, -2.106269359588623, 2.3772172927856445, -0.579426646232605, -1.646876573562622, -1.1545190811157227, 3.051344871520996, 6.763308525085449, 0.9091176986694336, -1.2186187505722046, -0.5328010320663452, -3.1652894020080566, 0.5008477568626404, 0.9223853945732117, -1.724100112915039, -0.8419812917709351, -0.754236102104187, -2.0158519744873047, 0.904269814491272, -0.7884831428527832, -1.9197282791137695, -1.0111894607543945, -1.9242708683013916, -1.839287519454956], [-0.4284313917160034, 0.9609304666519165, -0.9509652853012085, -6.811960220336914, 0.1724943220615387, -0.37577900290489197, -3.3439579010009766, -5.3100905418396, -0.5216306447982788, 1.8443108797073364, 0.1828393191099167, 5.499227046966553, -5.890387058258057, 1.2499735355377197, -1.404986023902893, -3.571796178817749, -2.4007182121276855, 0.6255999207496643, 2.470123291015625, 3.1707444190979004, 0.99493408203125, -0.26167258620262146, 3.056464433670044, 3.641587257385254, -1.00538969039917, -2.023083209991455, -1.5136377811431885, -3.2617807388305664, -1.8486403226852417, -1.6672815084457397, 1.9867069721221924, 0.8018329739570618], [1.7591203451156616, -2.1202166080474854, -4.381522178649902, 1.8369646072387695, -2.3160483837127686, -1.818556547164917, -3.595500946044922, -3.146336555480957, -2.975531816482544, -2.5730462074279785, -1.2771198749542236, -2.1834096908569336, 1.3097659349441528, -0.5469507575035095, 4.222051620483398, 0.2112216055393219, 1.4466770887374878, 4.997049331665039, 1.3948827981948853, -2.8065264225006104, 2.581859588623047, -0.7081704139709473, -0.0053860219195485115, -0.9048538208007812, -0.5021749138832092, 0.39314210414886475, 3.3388071060180664, 2.8954012393951416, -1.9589338302612305, 0.699948787689209, -0.0889921560883522, 2.608285903930664], [-6.209402084350586, 3.0160770416259766, 6.010730743408203, -3.5100598335266113, -2.6730549335479736, 1.9950528144836426, -2.136368751525879, -2.583651542663574, -0.6090798377990723, 0.5527955889701843, -3.8493611812591553, 3.927640914916992, 1.943763256072998, 6.561246871948242, -3.4563369750976562, 1.1310358047485352, 2.2088146209716797, -2.022590160369873, -1.4982068538665771, 4.512165546417236, 3.1110541820526123, -0.10149744898080826, -3.8049869537353516, 1.4821161031723022, 1.51841402053833, -2.7596490383148193, 0.1628103107213974, 1.9579815864562988, 0.3223372995853424, -2.812835216522217, 0.8589622974395752, -1.241644263267517], [-1.2534875869750977, 1.0393903255462646, -0.06915037333965302, -0.6733517050743103, 0.8627291917800903, 0.6884915828704834, 1.6102643013000488, -0.5819214582443237, 2.0061678886413574, -4.1936726570129395, 3.503934144973755, -0.33476534485816956, -1.3632543087005615, 0.7828715443611145, 3.3417258262634277, 0.1581379771232605, -0.6838599443435669, 2.7510876655578613, -0.2115371972322464, 1.4062104225158691, 1.5929534435272217, 0.826689600944519, 1.8307673931121826, -2.485473155975342, 4.4823479652404785, -2.6384050846099854, -0.6659888029098511, 0.9160707592964172, -2.737626552581787, 1.2598133087158203, 5.616254806518555, -3.136810302734375], [-1.3664871454238892, -2.895275831222534, 0.8110873103141785, -0.716240644454956, -3.160775899887085, -2.569199562072754, 0.20186801254749298, -4.259422302246094, -0.7942155003547668, -4.607834339141846, -1.4983627796173096, -7.019175052642822, 0.8435381054878235, 0.9061111807823181, -2.74147891998291, 0.055874694138765335, 0.48743095993995667, -5.973605155944824, -4.467782020568848, -0.20362450182437897, -3.502351999282837, -0.6282289028167725, 3.4147374629974365, -1.4327601194381714, -0.7598846554756165, 3.878051519393921, 1.2115964889526367, -0.8719936609268188, -5.432926177978516, -1.9748790264129639, -0.4204025864601135, -3.308530569076538], [-1.3457810878753662, -0.575911283493042, -2.048227548599243, 2.714306116104126, 6.3515777587890625, -0.27161383628845215, 3.3999338150024414, -3.1449594497680664, 1.7653172016143799, -0.20103329420089722, 0.3577876091003418, 2.425530433654785, 0.9037218689918518, -0.4340824484825134, -1.472213625907898, 0.31982362270355225, 1.1028586626052856, 2.0254313945770264, -3.9100964069366455, 3.585753917694092, -2.3733468055725098, -0.44629281759262085, -5.489026069641113, -3.8096444606781006, -0.5499257445335388, 0.9858019351959229, -2.623885154724121, 1.9996594190597534, 2.3300342559814453, -2.336857318878174, -0.07774618268013, -0.7451418042182922], [0.8860716819763184, -4.270318508148193, -2.0598416328430176, -3.2603249549865723, 1.3160632848739624, -1.7633979320526123, 1.145362377166748, 1.516771674156189, 4.986997127532959, 2.1162784099578857, -0.5541220903396606, -4.657299518585205, 7.221199989318848, 1.1279189586639404, 2.29744029045105, 2.462388753890991, -0.22029514610767365, -1.5766750574111938, 1.4478427171707153, 3.1259679794311523, -1.6588444709777832, -0.8358243703842163, 1.191064715385437, 0.5729117393493652, -1.0969693660736084, -0.2152446210384369, -1.7949790954589844, -1.8631618022918701, -0.911657452583313, -3.8615734577178955, -0.9287927150726318, -2.716158151626587], [-0.9381959438323975, 0.6955850124359131, 3.359499454498291, 0.4227762222290039, -2.835986852645874, -3.335059404373169, 4.777005195617676, 3.6491267681121826, -0.510540246963501, -4.201523780822754, 7.7768940925598145, -5.240930557250977, 0.9721589684486389, -2.3808298110961914, -4.786680698394775, -4.970365524291992, 2.4978809356689453, 0.5731697082519531, 5.809435844421387, -1.5427613258361816, 1.7422868013381958, 3.8022546768188477, 2.6138453483581543, 0.7913503646850586, 5.348912239074707, 0.08866523206233978, 3.6543822288513184, 0.9333494901657104, 0.6180750131607056, -1.3269126415252686, -1.0443614721298218, 0.5178180932998657], [-6.045280456542969, -0.9147375226020813, 0.2979765236377716, 2.674828290939331, 3.2571423053741455, -0.6538834571838379, -3.3743581771850586, 0.9247696399688721, 2.7254810333251953, 8.326858520507812, -4.701809883117676, 6.674025535583496, -3.2638773918151855, -7.876232147216797, 3.4903311729431152, -6.1438679695129395, -0.4454817473888397, -7.51133394241333, -0.9289507269859314, -1.1634286642074585, -2.3378424644470215, -4.122922420501709, 1.312736988067627, -1.143973469734192, 2.3173182010650635, -1.8377312421798706, 0.7547086477279663, -0.3534567058086395, 2.3348662853240967, -1.5701794624328613, 0.05464255437254906, -4.951084136962891], [-1.077683925628662, -1.5784032344818115, -1.8251557350158691, -5.246016025543213, -0.6719544529914856, -0.6007193922996521, 1.924272060394287, 0.07562756538391113, 0.5074971914291382, 1.5529277324676514, -1.9569811820983887, 1.4643152952194214, -2.2214536666870117, 2.040461778640747, 6.935126304626465, -2.547724723815918, 1.025533676147461, 2.8383913040161133, 3.4398186206817627, 2.0480127334594727, -4.181032180786133, -0.25381040573120117, 0.310030996799469, 2.0575952529907227, 3.194972515106201, 3.8716936111450195, -5.030060768127441, -4.362898349761963, -3.8104612827301025, 5.222499370574951, -1.2057063579559326, -3.6668009757995605], [-1.9083802700042725, -2.6540236473083496, 0.378100723028183, 7.162975311279297, -5.608392715454102, -1.6474788188934326, 0.9832647442817688, -1.0560190677642822, 7.162088871002197, -1.5713655948638916, -1.8647561073303223, -2.3679208755493164, 5.068358421325684, -9.144060134887695, -1.2432821989059448, -1.2339478731155396, 1.5111597776412964, 3.638596534729004, 0.057314369827508926, 3.2131526470184326, 1.1995798349380493, -0.03503495827317238, 2.5498921871185303, -3.923995018005371, 1.1217588186264038, 2.5800652503967285, 2.9233040809631348, 0.5665876865386963, -3.036817789077759, -0.11154607683420181, -4.715720176696777, -3.1766631603240967], [-6.980904579162598, -1.261634111404419, -2.8134491443634033, -5.813322067260742, -0.28913354873657227, 4.444057464599609, -0.13562320172786713, -2.4251351356506348, -0.18096545338630676, 2.593064069747925, 1.2819817066192627, -5.897872447967529, 2.3729028701782227, -3.5967817306518555, -5.115445613861084, 4.6568145751953125, 2.214271068572998, 2.9862325191497803, -1.972102165222168, -2.382450819015503, 0.448504239320755, -2.2782044410705566, -0.7262363433837891, 3.33797550201416, -2.7420525550842285, -0.19828495383262634, 1.1567721366882324, 1.1351150274276733, 5.631467819213867, -4.213871002197266, 1.475716233253479, 2.431464910507202], [-3.4406142234802246, 2.4976534843444824, -2.081740140914917, -1.3672146797180176, -4.269661903381348, -0.8355355262756348, 5.250900745391846, -3.074294328689575, -0.7845300436019897, 2.224256753921509, 1.0217838287353516, -2.673403263092041, -3.562422752380371, -3.2451109886169434, -0.27965253591537476, 1.5233126878738403, 2.5948681831359863, -3.567777156829834, -1.9539644718170166, -3.930724620819092, -6.139894008636475, -3.0554301738739014, -1.402580976486206, 3.0512447357177734, 4.639428615570068, 0.8312783241271973, 2.8722236156463623, -1.3642491102218628, -0.13396765291690826, -0.7578826546669006, -2.583754539489746, 9.932683944702148], [0.96616530418396, 1.2230385541915894, -1.7901337146759033, 1.4782750606536865, -3.542916774749756, 3.019084930419922, 3.308101177215576, -0.09436608850955963, 1.0548756122589111, 1.1822848320007324, 2.271754264831543, 10.169708251953125, -5.18096923828125, -4.433060646057129, -2.1463496685028076, 3.29286527633667, -0.9462499022483826, -2.473106861114502, -1.2675998210906982, 0.4430590271949768, -0.09717908501625061, -1.398238182067871, -2.395642042160034, -3.8145790100097656, -1.180626392364502, -5.211106300354004, -2.681830406188965, -3.85945463180542, -0.026245299726724625, 5.204365253448486, 2.7536699771881104, -1.4587246179580688], [2.0630459785461426, 4.204236030578613, 2.258416175842285, -2.1192617416381836, -0.17880746722221375, 1.1081502437591553, -2.5112881660461426, 2.428271770477295, -4.258047580718994, 5.86349630355835, 1.314812421798706, -0.5631524324417114, -0.37447914481163025, 4.786513328552246, -2.811069965362549, 1.5356624126434326, 1.6736476421356201, -4.797300815582275, -2.155346393585205, -1.2195672988891602, 1.9106965065002441, -1.2413755655288696, -3.6392822265625, -0.22264021635055542, 4.714881896972656, 2.5257649421691895, 4.272497653961182, -0.11472905427217484, -2.485206127166748, 1.6619371175765991, 8.179295539855957, 2.3867266178131104], [-4.70218563079834, 2.7753539085388184, -2.6697912216186523, 3.9535579681396484, -1.494220495223999, 1.4083703756332397, -0.9381030797958374, 3.6588120460510254, 2.5700302124023438, 3.7731094360351562, 0.19556275010108948, 1.3376139402389526, -1.8733259439468384, -0.6918412446975708, -4.175928115844727, 3.84015154838562, 2.213897228240967, 1.5395972728729248, -8.284760475158691, 2.654134750366211, -2.996865749359131, -6.62034797668457, -6.231784820556641, -4.323112487792969, -1.7031333446502686, -0.31165552139282227, 4.219728946685791, 1.1825844049453735, 0.1336185187101364, -1.4047727584838867, 2.363980531692505, 3.618593692779541], [1.9408538341522217, 2.2569169998168945, -0.8244525790214539, -6.916443824768066, 0.24743011593818665, -4.613053321838379, -1.61898672580719, -1.96147620677948, 0.9750013947486877, -1.7874873876571655, 1.6356085538864136, 3.7637224197387695, 1.6703764200210571, -3.3544211387634277, -1.1927821636199951, -1.0441663265228271, -0.5918536186218262, 0.859259843826294, -1.1658066511154175, -0.8695030212402344, 0.4885893166065216, 1.8528764247894287, 0.6586716771125793, 0.6140453815460205, 0.38564950227737427, -4.04409122467041, 1.07453453540802, 0.7529005408287048, 2.019476890563965, 0.5292182564735413, -0.016321582719683647, 0.3748709559440613], [1.1414905786514282, 1.9114162921905518, -1.4258694648742676, -0.9370142221450806, 2.0440542697906494, 1.8348712921142578, 2.878605604171753, 1.085026502609253, -1.9740984439849854, -2.0444893836975098, 0.15450243651866913, -1.492323637008667, 7.331063747406006, 0.05469095706939697, 1.5543272495269775, 1.2554281949996948, -0.045382849872112274, -3.620345115661621, 0.8341615200042725, 5.241700649261475, -4.39587926864624, 1.573190689086914, 0.5822967290878296, 1.8698234558105469, 1.7784258127212524, -6.846599578857422, 4.953760147094727, 0.7138347029685974, 1.2471415996551514, -1.7639200687408447, -4.886994361877441, -2.2024917602539062], [1.0902032852172852, -1.6452945470809937, 1.5284518003463745, 3.8072669506073, 1.5601320266723633, -2.2384281158447266, -5.207094669342041, 1.214935541152954, 3.362997531890869, -2.712405204772949, 4.629376411437988, 1.4911788702011108, 0.3579946756362915, -1.6363524198532104, -0.5429416298866272, 1.963435173034668, -3.2765564918518066, -0.5369014739990234, -4.397542953491211, -0.18205754458904266, 1.831788182258606, 5.951707363128662, 1.4205187559127808, 0.9346348643302917, 4.64590311050415, 2.2029800415039062, -1.3770465850830078, -2.1484222412109375, -3.6876301765441895, -8.688633918762207, 2.9247636795043945, -2.398900032043457], [-1.3772735595703125, 1.56551992893219, -4.444889068603516, 4.629214763641357, -0.5432264804840088, -3.373671054840088, 1.038666009902954, -0.5155030488967896, -6.832327842712402, 3.7400240898132324, 5.311681747436523, -1.163705587387085, 8.617145538330078, 2.170417070388794, -2.9725282192230225, -4.271693229675293, 4.464055061340332, -0.03233965486288071, -0.5514593124389648, 2.7278521060943604, 5.3989152908325195, -4.949373245239258, -0.5803793668746948, 0.2763734459877014, -1.3549352884292603, -1.914586067199707, -3.154308319091797, -0.7111390829086304, 1.2450262308120728, 1.4871429204940796, -5.532793998718262, 3.050072193145752], [1.6153781414031982, 1.660542368888855, -0.8613237142562866, 1.003056526184082, 0.09927919507026672, -5.927712440490723, 1.7222918272018433, -1.2135196924209595, 0.07951607555150986, -3.7179784774780273, -0.9753773212432861, -6.3339643478393555, 2.3705530166625977, -0.7863348722457886, 0.2661854922771454, 0.6998711824417114, -6.339584827423096, 3.679706573486328, -3.68198561668396, 1.2765220403671265, 1.1274816989898682, 0.14220134913921356, -1.8133108615875244, 1.114738941192627, 3.9967567920684814, 2.4962778091430664, 2.073831796646118, 3.6449856758117676, 2.2796149253845215, -1.4864948987960815, 3.687096118927002, -2.151491641998291], [1.0395675897598267, 4.3368024826049805, 1.0774158239364624, 0.6306087374687195, 0.9737519025802612, 1.6369524002075195, -3.1144614219665527, -3.6677820682525635, -0.9279369115829468, 6.0082597732543945, -0.14874544739723206, -1.434297800064087, -6.221691131591797, -1.5582807064056396, -4.4187541007995605, 3.3117880821228027, -0.4550829529762268, 2.267943859100342, -0.036560144275426865, -1.2160919904708862, -2.131603956222534, -0.32174792885780334, 0.5738272666931152, -4.14589786529541, 4.1772308349609375, -2.8903446197509766, -0.48437994718551636, 6.203155517578125, 0.3312026262283325, 2.928781509399414, -2.562847852706909, -0.7807971239089966], [2.353886365890503, 3.441190719604492, -0.3006608188152313, -1.5973725318908691, -4.353313446044922, 0.7609045505523682, -4.217337608337402, 2.1504392623901367, -3.030925750732422, -0.6347151398658752, -1.508089542388916, 0.7808818817138672, 3.4398930072784424, 5.348154067993164, -1.5909335613250732, -3.8855230808258057, -0.8870080709457397, 1.6702816486358643, 1.212088704109192, -5.33015251159668, -1.9706008434295654, -0.020303549244999886, -0.5246797800064087, -3.8952808380126953, 4.130328178405762, 1.4153220653533936, 1.5454418659210205, 2.2848403453826904, -3.371140956878662, -0.7176448106765747, -3.309515953063965, 2.6740455627441406], [-2.876828193664551, 0.8423154354095459, -1.4907855987548828, -1.0772206783294678, 6.85533332824707, 1.7967991828918457, -0.16907083988189697, 1.2399871349334717, -2.6454195976257324, 1.5000860691070557, -2.6122822761535645, -0.9795292615890503, 0.3542254865169525, 7.387241363525391, -0.4289626479148865, -3.6510043144226074, -1.514155626296997, -0.39123979210853577, 2.01680588722229, 0.7945226430892944, -0.009613641537725925, -0.5340428948402405, -4.7607011795043945, -3.950381278991699, -1.2101771831512451, -4.509147644042969, -2.7999513149261475, -1.6187355518341064, -1.6219391822814941, -3.5171656608581543, 0.27160266041755676, 3.1381120681762695], [-3.9073901176452637, -2.6130189895629883, 2.825038433074951, -0.9450893402099609, -1.669158935546875, 0.43291908502578735, -1.5092101097106934, -0.3981984853744507, 2.56646728515625, 1.9713374376296997, 3.287263870239258, -3.5749433040618896, 1.8889299631118774, -4.537827014923096, 0.6920405030250549, -0.7742002010345459, 1.0246264934539795, -0.5556143522262573, 2.531796455383301, -1.7540602684020996, 2.249307155609131, -0.4175010919570923, -3.6648592948913574, -1.6410043239593506, -6.19287109375, -3.116605758666992, 0.458479106426239, 2.7468957901000977, -2.5983402729034424, 0.250037282705307, -0.056035153567790985, 1.053908348083496], [-2.051506519317627, 0.19287090003490448, -5.297565460205078, 3.5138349533081055, 2.911247491836548, 1.772553563117981, 3.9176883697509766, -1.655653715133667, -1.8346033096313477, 2.0102169513702393, 6.834514617919922, 1.3068122863769531, -0.21356502175331116, -0.6420313119888306, -0.34364786744117737, 0.40691447257995605, 5.355067729949951, -3.502969741821289, 0.22590681910514832, 0.44980156421661377, 4.698951721191406, -1.5378212928771973, -1.9691652059555054, -2.7128355503082275, 0.5560157299041748, 0.4048234224319458, 1.3495526313781738, -5.426268100738525, -2.044947385787964, -0.707400381565094, -0.985764741897583, 2.0793704986572266], [9.442314147949219, 5.377944469451904, -4.482048988342285, -1.0052073001861572, -1.2468839883804321, -2.671203136444092, 1.574865460395813, 1.8945865631103516, -3.400723934173584, -1.7294337749481201, 2.5494256019592285, 1.6841329336166382, 1.4041383266448975, 1.287421703338623, 0.2150038182735443, -5.320234298706055, 1.3682701587677002, 7.9336748123168945, 0.7027354836463928, -2.489741802215576, -2.6911444664001465, -0.974437952041626, 3.307809829711914, 1.704921841621399, -1.5027347803115845, 1.4323060512542725, -2.8127334117889404, -0.6479383111000061, -0.7265783548355103, -1.6354032754898071, 1.0711356401443481, 1.8922115564346313], [1.1354092359542847, 6.829675197601318, -4.991093158721924, 4.586978912353516, -3.078807830810547, -2.5574965476989746, 3.196928024291992, -1.6900852918624878, -3.6289379596710205, -3.675446033477783, -0.05518288537859917, -2.9880261421203613, -3.680701732635498, 1.1363592147827148, -2.9705660343170166, 0.8748405575752258, 0.2774661183357239, -3.969587802886963, 1.8131892681121826, 0.24516943097114563, 1.2872087955474854, 0.2555760145187378, -0.8067904710769653, 1.9901973009109497, 3.1650967597961426, 1.361871361732483, -0.03426139056682587, 0.2509801983833313, 5.040112495422363, -2.048290491104126, 4.316432952880859, 0.13064615428447723], [1.8716272115707397, -5.751576900482178, -1.8794949054718018, -0.651127815246582, -0.7918720245361328, 0.36259159445762634, -1.781477689743042, -3.4190897941589355, 0.38524025678634644, 0.4682052731513977, -0.4672277569770813, 0.8136796951293945, 1.702918291091919, -3.7403807640075684, 4.756082057952881, 1.8457506895065308, -4.369776725769043, -5.317258358001709, 2.2549843788146973, 1.8337101936340332, -1.9801323413848877, -5.73890495300293, -1.1219546794891357, -1.1775866746902466, 0.7086477875709534, -0.45646291971206665, 2.087313175201416, 2.1599690914154053, 4.814838409423828, -2.0022802352905273, -2.4327173233032227, -0.534595251083374], [-1.999651551246643, 0.9798401594161987, -2.6679322719573975, 4.926373481750488, 4.065367221832275, -3.0125834941864014, 2.2922616004943848, 3.692476749420166, -2.166062355041504, 1.9797227382659912, -0.2282501459121704, 3.221038818359375, -3.1328043937683105, -6.856316566467285, -1.9111576080322266, 0.8565928936004639, 0.876350998878479, -3.0468297004699707, -4.568204879760742, 0.5040351748466492, -1.1057219505310059, -3.1208460330963135, 0.4235842823982239, -4.509768486022949, 0.2964991331100464, 3.157921314239502, 2.3298115730285645, 5.8884382247924805, 3.7471632957458496, 5.283708572387695, 0.3518204987049103, -5.329610824584961], [2.4678597450256348, -6.539388656616211, 4.33452844619751, 1.5332655906677246, 0.19693712890148163, 0.9244973659515381, -5.309835910797119, 1.102832555770874, 1.2716292142868042, 3.3958678245544434, 1.183095932006836, -3.246464967727661, -2.9378490447998047, -0.8561253547668457, 5.770906925201416, 1.661707878112793, -3.6269750595092773, 1.4278123378753662, 1.463942527770996, 3.3082330226898193, -0.4470790922641754, -4.662693023681641, 4.805016040802002, 5.345124244689941, 1.317718267440796, -3.6596226692199707, 7.586091995239258, -1.8450424671173096, 0.40850090980529785, -3.1541543006896973, -4.857168197631836, 1.5186691284179688], [-0.5730297565460205, -3.499638080596924, -3.6158366203308105, -5.89291524887085, 0.6624365448951721, -3.1533350944519043, 0.3706929385662079, 1.2811386585235596, -0.8764777183532715, 2.498594284057617, -2.545790910720825, 2.1271719932556152, 4.120357036590576, -0.9315986633300781, 5.034734725952148, 0.947519063949585, -2.0393433570861816, 6.190157890319824, 3.4438483715057373, 0.36571457982063293, -1.8912873268127441, 0.7295142412185669, -0.14721956849098206, -1.309335470199585, -1.8728015422821045, -1.2117031812667847, -2.5180482864379883, -0.006724500562995672, 4.459944725036621, -0.9885473251342773, -2.3346729278564453, 5.762839317321777]], [[-1.5909004211425781, -3.7157678604125977, 0.8880023956298828, 2.2620341777801514, -4.412073135375977, -3.3685033321380615, -3.4966564178466797, 1.2061941623687744, -0.11365576088428497, 2.8036580085754395, -0.7341527938842773, -0.03828901797533035, -5.03509521484375, -1.3863425254821777, -0.12530921399593353, 0.673287034034729, -0.676970899105072, -4.147431373596191, 0.14630773663520813, 0.3338308334350586, -0.15555809438228607, 0.3114904463291168, -0.3287312984466553, -2.5234367847442627, -2.5174560546875, -7.125150680541992, -6.055445671081543, -0.22272750735282898, 0.7173326015472412, -2.2022628784179688, -0.09092840552330017, 4.0317487716674805], [4.117249011993408, -0.7604435682296753, 2.6497135162353516, -3.920275926589966, 0.7485450506210327, 1.699686884880066, 0.4049677848815918, 2.6085264682769775, -0.9015127420425415, -0.5879979729652405, -2.282285213470459, -2.7925050258636475, 2.9468657970428467, 4.010231018066406, 2.5740292072296143, -1.774064302444458, 0.35928255319595337, 1.4308080673217773, 0.4318732023239136, 1.4191573858261108, 4.48839807510376, 1.850738286972046, -2.51094913482666, -3.1094789505004883, 7.550069332122803, 4.789184093475342, -4.6670660972595215, 0.9050912261009216, 0.07852129638195038, 1.781491756439209, 1.0180307626724243, -3.6528360843658447], [-1.0021889209747314, -3.539149284362793, -0.4934607744216919, 2.8954873085021973, 0.5141849517822266, 1.419560194015503, -0.9870349168777466, -7.519234657287598, -5.216369152069092, -4.312478542327881, 1.416170358657837, 3.1454451084136963, -0.7180020809173584, 5.254806995391846, -2.3080716133117676, -3.606147050857544, -1.8500150442123413, -0.5561661124229431, 0.6228407621383667, -0.5441545248031616, 0.24007947742938995, 6.938772201538086, 1.2255852222442627, -1.3894445896148682, -5.348974227905273, 2.930370569229126, -4.402362823486328, -2.663776397705078, 0.9498826265335083, -2.7140755653381348, -1.1070117950439453, 1.565610647201538], [-1.493277668952942, 3.0579535961151123, 3.9637603759765625, -0.4765781760215759, -0.837565004825592, -2.7138102054595947, 1.6002943515777588, 3.9847846031188965, -1.3685246706008911, -0.2896503508090973, -6.628084182739258, 4.365962028503418, -0.05322416126728058, -3.4277687072753906, 2.627044677734375, -2.8566248416900635, -0.6099957227706909, -2.468319892883301, -4.767696857452393, 1.001471996307373, -2.043241500854492, 2.3298559188842773, -4.136617183685303, 2.9613192081451416, -1.3991414308547974, -4.0638651847839355, -4.496734142303467, 0.41494694352149963, -0.9124979972839355, -0.8110054135322571, 0.01393219642341137, -0.9586266875267029], [0.9427145719528198, 3.630354404449463, -1.1442118883132935, 1.3964383602142334, 3.5208640098571777, 2.4229369163513184, 3.852108955383301, -0.41240546107292175, 1.278218388557434, 0.14108817279338837, 5.622176170349121, 1.4761409759521484, -1.0796473026275635, 0.7240077257156372, -0.1788773536682129, 1.824931025505066, 5.874063014984131, 0.2613178491592407, 0.23244726657867432, 0.15452376008033752, -5.260089874267578, 1.7491652965545654, -0.9920846223831177, -4.096066951751709, -0.13476185500621796, -3.812001943588257, -4.225392818450928, 0.4890873432159424, 0.5041848421096802, -1.7913233041763306, 3.2565228939056396, -0.998760461807251], [-5.8449201583862305, 0.022539548575878143, -0.3017050623893738, -1.6860840320587158, 0.8244786262512207, 1.9513825178146362, -2.196206569671631, -1.9469149112701416, 0.8441882133483887, 6.069432258605957, -0.13466642796993256, 1.7880903482437134, -3.410377025604248, -0.5457662343978882, 3.326936721801758, 3.0206058025360107, -3.549680471420288, 6.710021495819092, -2.243779420852661, -2.128376007080078, -3.651088237762451, 1.3002852201461792, 5.928380489349365, 1.866567611694336, -1.5525767803192139, -0.25373977422714233, -0.19479413330554962, -0.1667306274175644, 2.477923631668091, -0.5868277549743652, 0.5145242214202881, -1.421316385269165], [1.8953368663787842, 2.427013874053955, 1.5506994724273682, -5.870829105377197, 1.5164427757263184, 2.468228578567505, 4.7333292961120605, 2.7919130325317383, -1.2903836965560913, -4.110301971435547, -3.4677765369415283, 1.0601481199264526, -2.4057164192199707, -5.038388729095459, -1.8250197172164917, 5.908809185028076, -0.8106088042259216, -0.6382365822792053, 0.716727077960968, -5.243896484375, 2.608241558074951, 0.7111771106719971, 2.3853952884674072, -1.8545916080474854, -2.9865221977233887, -0.33083754777908325, -8.294037818908691, 1.0996003150939941, 2.6713852882385254, -2.6757688522338867, -6.2051897048950195, 5.155814170837402], [1.1781822443008423, 1.1887896060943604, -1.4124971628189087, -4.012016296386719, -0.9357966184616089, 1.3832385540008545, -0.7491551637649536, -1.933178186416626, -2.3796885013580322, -1.5278596878051758, 0.7110462188720703, -1.1954269409179688, -2.5344502925872803, -1.5408174991607666, -0.3398906886577606, -1.4968990087509155, 5.724600791931152, 0.3445253372192383, 4.984489440917969, -5.9328107833862305, -4.947798252105713, 3.7678422927856445, -2.315683364868164, 0.200558602809906, 0.45874738693237305, -0.4237450361251831, -1.0098482370376587, -2.321352005004883, -2.910390853881836, 2.6468851566314697, 1.0058858394622803, 0.6093153953552246], [5.833116054534912, -1.0234382152557373, 1.08759605884552, -3.6014981269836426, -0.8975352048873901, -1.2653307914733887, -2.0414798259735107, 1.3575596809387207, 0.6055610179901123, -3.7603683471679688, -0.03430004045367241, 1.2073445320129395, 2.8162283897399902, -4.609068393707275, -0.39876481890678406, 2.4721755981445312, -1.2215185165405273, -4.993290901184082, 1.2676916122436523, 0.27583664655685425, -2.355865478515625, 1.4416033029556274, -1.4108350276947021, -0.6007382273674011, -0.7934791445732117, 2.858388900756836, 2.6878879070281982, -0.6726583242416382, 0.19756516814231873, 0.16961854696273804, 5.604495525360107, -2.9798660278320312], [3.8516664505004883, -1.4617400169372559, 4.903913974761963, -2.388664722442627, 3.048861026763916, -3.0097079277038574, -1.38835871219635, -0.14749063551425934, -1.5737123489379883, 3.568425178527832, 5.799846172332764, -1.2492197751998901, 0.5245335698127747, 1.1870958805084229, 1.577497959136963, 1.5275893211364746, 2.0083274841308594, -4.157738208770752, 0.9397092461585999, 0.10952822864055634, -2.0281107425689697, -2.2270851135253906, -0.1326393187046051, 0.15695452690124512, 0.25111836194992065, 0.3211279809474945, 1.6422319412231445, 1.0755252838134766, -2.0755879878997803, -0.3236558139324188, 4.430736541748047, -5.849205017089844], [6.0303239822387695, -6.157020092010498, 2.024441719055176, 2.2612736225128174, -4.244951248168945, 1.111630916595459, -2.1170544624328613, 0.43096357583999634, 4.788919925689697, -0.24562382698059082, -0.2352428436279297, 0.2942761480808258, -3.566131591796875, 5.168117523193359, 0.47137290239334106, 4.181875228881836, 3.218477725982666, 3.2867937088012695, -3.565199375152588, -5.166693210601807, -4.159480094909668, -4.7228288650512695, 2.857614278793335, -1.6255908012390137, 2.3082308769226074, 2.3128793239593506, -2.8598690032958984, 4.391525745391846, 1.4383347034454346, 1.591957926750183, -1.863846778869629, -3.7781989574432373], [2.0720887184143066, -1.679842472076416, -2.8627514839172363, -2.9371023178100586, -2.127965211868286, -7.347105503082275, 7.868799209594727, 2.635564088821411, -5.576377868652344, 2.2249629497528076, 3.6067163944244385, -5.609652519226074, -0.3584638237953186, 1.4807966947555542, -1.5605919361114502, -1.257360816001892, -2.2264671325683594, 3.0537238121032715, 0.3534286916255951, -5.443085670471191, -3.8714170455932617, -4.943474769592285, -0.2387450635433197, 3.026224136352539, -2.770490884780884, -0.5728129148483276, -0.5991336703300476, 0.2918677031993866, -2.143662452697754, 0.1953754723072052, -1.7518231868743896, -4.886336326599121], [3.4578418731689453, 0.5485994219779968, 3.028636932373047, 3.6730337142944336, 3.7610700130462646, -2.034456729888916, 1.829313039779663, 2.1395325660705566, 3.7716946601867676, -1.7727116346359253, -0.8180492520332336, -0.37158942222595215, -2.0432279109954834, -1.936746597290039, 0.34588363766670227, -1.2627484798431396, -3.317819595336914, 1.0880842208862305, 6.504294395446777, 0.39085155725479126, -4.091182231903076, 7.168091773986816, -2.60347318649292, -0.6432116627693176, -2.400070905685425, -3.454503059387207, -3.1850271224975586, 1.4863049983978271, -3.4973652362823486, -1.8446364402770996, -3.6218323707580566, -1.3538835048675537], [-0.5159468650817871, 1.8138656616210938, -4.312700271606445, 0.8757412433624268, -2.8164985179901123, -1.2955193519592285, 3.2508559226989746, -1.056035041809082, 1.403940200805664, 0.8442286849021912, -4.797239303588867, 1.6563137769699097, 0.4218447804450989, -0.6062951683998108, 4.847346782684326, 5.538443565368652, -0.2397555112838745, -1.3678441047668457, 4.051869869232178, -6.962689399719238, -2.1012115478515625, 0.6003798246383667, -1.4349395036697388, -0.18684381246566772, 3.0221800804138184, 1.4765825271606445, -0.4331161379814148, -0.9540807008743286, -3.887819766998291, 2.530651092529297, -1.2421780824661255, -2.804283857345581], [-3.8265609741210938, -3.435291290283203, -0.8212364315986633, 4.999583721160889, 1.4211281538009644, -2.991396427154541, 0.6441270709037781, -1.130730390548706, 2.461902618408203, 3.9358341693878174, -4.85167932510376, 1.790228247642517, -3.125282049179077, 1.7656340599060059, 1.6706926822662354, -2.8382046222686768, 2.5908260345458984, -1.1787389516830444, 4.0569257736206055, 2.298539638519287, -0.2807256579399109, 2.924229621887207, 1.1379958391189575, 0.4675792157649994, 0.23600366711616516, -3.997066020965576, -1.136788010597229, 0.7767429351806641, -3.35311222076416, 3.09915828704834, 5.126155376434326, -4.150018692016602], [1.7235233783721924, -5.335977554321289, 5.288599014282227, 5.157834053039551, -3.9806478023529053, -1.133087396621704, 3.426724910736084, -3.709554672241211, 3.707213878631592, 1.867131233215332, -1.4997810125350952, 0.28104934096336365, 1.6146957874298096, 1.0980796813964844, 5.232293605804443, 0.8553507328033447, -0.9699915647506714, -1.6896002292633057, -3.3044686317443848, 0.1496829092502594, 3.9368700981140137, -2.7165234088897705, 0.6878268122673035, -5.418246269226074, -1.4903037548065186, 2.989686965942383, 1.7437341213226318, -0.04218094423413277, 6.126805305480957, -2.036166191101074, 2.075777053833008, 2.8729240894317627], [-1.8916301727294922, -0.7540274262428284, 5.962119102478027, 3.7467663288116455, 6.212747097015381, 1.5285812616348267, -1.0948307514190674, 0.39119577407836914, -3.6974236965179443, -1.493241548538208, -5.08422327041626, 1.6435959339141846, 1.1741838455200195, -1.3993419408798218, -0.41418251395225525, -3.7960360050201416, 2.3995068073272705, 0.4068174958229065, 0.9061657190322876, -5.952030181884766, 0.7244837880134583, 0.1907694786787033, 4.113283634185791, -2.7677056789398193, -0.9498691558837891, -4.207883358001709, -0.7124525308609009, 0.6450075507164001, -5.74699592590332, 1.3696690797805786, 2.057448148727417, -1.3458998203277588], [-2.1772561073303223, 2.2440738677978516, 0.7325031757354736, 6.381896495819092, 1.7386232614517212, 1.6564276218414307, 1.7631919384002686, -2.6878490447998047, 0.1711948812007904, 1.4226003885269165, -0.9089373350143433, -0.44654160737991333, -2.13622784614563, -2.8383736610412598, 2.059108018875122, 3.0983681678771973, -4.381636142730713, -4.023996353149414, -1.0142879486083984, -1.58866286277771, 8.133258819580078, -1.7468485832214355, -0.5565000176429749, 4.73768949508667, 1.4059089422225952, 0.45207738876342773, -8.597082138061523, -1.8858897686004639, -0.21248573064804077, -0.14575576782226562, -2.282066822052002, -0.4544888734817505], [-0.01406068541109562, -2.1214914321899414, 0.3125191032886505, -4.60129976272583, 1.5779647827148438, 4.204225540161133, -1.2015650272369385, -2.3271780014038086, -4.01441764831543, 1.6135364770889282, 4.904958724975586, 0.44184350967407227, 3.325035572052002, 1.8610092401504517, -0.7041007280349731, -4.280372619628906, 2.189952850341797, -7.509591102600098, 2.070828437805176, -4.488973617553711, -0.18616822361946106, -1.499635934829712, 0.942044198513031, -3.120335340499878, 2.5790135860443115, 5.242402076721191, -0.08623148500919342, 1.6895625591278076, -1.5537745952606201, 1.1783194541931152, 0.23086035251617432, 3.208577871322632], [-1.680304765701294, 3.3573718070983887, 1.3856596946716309, -5.747647285461426, 3.38002347946167, 0.9424152374267578, -1.120413899421692, 0.4773862957954407, -2.366443634033203, -1.1872224807739258, 2.672478199005127, -1.8920420408248901, 6.103295803070068, 5.361012935638428, -5.132232666015625, -0.9284675121307373, 1.9344007968902588, -0.522070050239563, 6.1578521728515625, -1.632167100906372, 0.5602289438247681, -1.8371176719665527, 0.31585144996643066, -1.7093700170516968, -1.89228093624115, -2.774156093597412, 2.7966830730438232, -3.56266450881958, -4.33648157119751, 3.412203311920166, 1.9786949157714844, 0.25025424361228943], [-4.246419429779053, 0.38423261046409607, 5.173341274261475, 3.8016486167907715, -1.05219566822052, -3.9791159629821777, 1.4275367259979248, 1.5884504318237305, 3.8642985820770264, 3.399207592010498, 2.7042899131774902, -4.14022159576416, 2.0094523429870605, -4.607493877410889, -3.3168575763702393, 2.302048683166504, -2.6564688682556152, 4.141846656799316, 3.744722366333008, -0.27788975834846497, 2.7855870723724365, 5.324163436889648, -3.1240360736846924, -2.721285581588745, 5.336949825286865, 0.1836572289466858, -1.435727834701538, -0.4347444176673889, -1.9090287685394287, -4.745913505554199, 4.422685623168945, -1.5288437604904175], [2.9430410861968994, 1.0661563873291016, -2.9825387001037598, 5.750181198120117, 0.7879687547683716, -2.429241418838501, -5.457295894622803, 0.8103494644165039, 0.8664618730545044, -3.542708158493042, 2.7519822120666504, 0.9497402906417847, 0.09004196524620056, 1.4767961502075195, -1.5702295303344727, 1.3690273761749268, -1.4693553447723389, -1.282914161682129, -5.145830154418945, 2.049487590789795, -3.0591979026794434, 1.94293212890625, -2.1423003673553467, 1.3300507068634033, -1.486603021621704, 0.5074906945228577, -1.8761820793151855, 8.152145385742188, -3.8072454929351807, 0.7718154788017273, -3.5990676879882812, -1.038018822669983], [-1.3181779384613037, -0.04784267395734787, -1.694819688796997, -4.151036262512207, 4.059841632843018, 0.41432178020477295, -3.639406681060791, -0.6182262897491455, -4.319730758666992, 2.8192124366760254, 1.3165231943130493, 2.591118097305298, -2.4234189987182617, -0.03196195885539055, 0.8042845726013184, -0.715505838394165, -0.4057462811470032, -0.2252933531999588, 0.955925703048706, -3.1602258682250977, 0.4729679226875305, 2.8209187984466553, 3.4077420234680176, 0.4013756811618805, 0.8838165998458862, -0.3051055073738098, -2.3203799724578857, -0.5877479314804077, 1.735571265220642, 2.0293588638305664, 0.362362802028656, -5.141350746154785], [-2.10351300239563, 0.4174967408180237, 0.3624686002731323, 2.3330509662628174, 4.795388221740723, -1.4746196269989014, -2.412494421005249, -6.397543907165527, -2.042057991027832, -1.7441277503967285, 2.7153985500335693, 4.408743858337402, -2.585165500640869, -0.595885694026947, 0.33259040117263794, 4.515226364135742, 1.5463461875915527, -2.231492280960083, -3.8558506965637207, 2.961223840713501, 0.5182497501373291, 0.7340327501296997, 0.5967194437980652, -2.503096342086792, -3.9300947189331055, 0.3499513566493988, -0.3388514518737793, -1.8913328647613525, 1.6246392726898193, -2.147045850753784, 1.6535943746566772, -2.968935489654541], [0.30502623319625854, -6.215575218200684, -0.08553052693605423, 2.371307611465454, -0.30193543434143066, -5.058393478393555, -3.7776124477386475, -0.7819581031799316, 0.51694655418396, -0.7913880348205566, -1.253330945968628, -3.1948671340942383, 1.9303847551345825, 2.379952907562256, 0.3800256848335266, -0.10144465416669846, -2.3319711685180664, -4.625265121459961, -3.0569348335266113, -4.793699741363525, 6.143500328063965, 1.7197104692459106, 0.9702901840209961, 2.0671794414520264, -2.8109612464904785, 0.7864010334014893, -0.6830383539199829, 3.1882617473602295, -1.2954365015029907, 0.9666736721992493, 1.285766363143921, -4.357966899871826], [-6.166046619415283, 0.9484034776687622, 2.9724478721618652, 3.2794947624206543, 1.361068844795227, -1.1523795127868652, 4.077326774597168, -3.545576572418213, -3.0972323417663574, -0.7532846331596375, -2.736480236053467, 0.5318314433097839, -1.0199811458587646, -5.64888334274292, 1.734675645828247, 0.8927692174911499, -0.8264294862747192, -2.2714014053344727, -1.5079540014266968, -4.712454795837402, 1.18144953250885, 2.7455592155456543, 0.8285466432571411, 2.6688714027404785, -0.06778498739004135, -6.839652061462402, 1.0595749616622925, 3.70285964012146, 0.17194809019565582, 3.923549175262451, -1.0415213108062744, -3.871488094329834], [-2.201282262802124, -1.2360557317733765, -0.9262102246284485, -2.622412919998169, 0.620695948600769, 4.142825603485107, 0.1334768533706665, 1.9049346446990967, -0.08636455237865448, 1.9484184980392456, 0.5609606504440308, 5.792416572570801, -1.4203801155090332, -7.594540596008301, -6.097698211669922, -0.2674449682235718, -3.2143850326538086, -2.906369686126709, 2.548584461212158, -5.360899448394775, 2.6234962940216064, 1.898282766342163, 2.4987664222717285, -4.628375053405762, 1.8455064296722412, -3.6501612663269043, 0.3183479309082031, -0.16523222625255585, 0.5453628301620483, 2.3008337020874023, -1.5464279651641846, 5.993741035461426], [1.0309092998504639, -1.8246188163757324, 2.1910667419433594, -3.1163907051086426, 3.4775524139404297, -2.2422118186950684, 4.020875453948975, 2.2632226943969727, 2.6063685417175293, 0.3067407011985779, 1.9863460063934326, 1.4947148561477661, -2.6707968711853027, -2.079044818878174, 2.495316743850708, -2.3125267028808594, -1.602747917175293, 1.8224176168441772, -0.23598554730415344, -4.8603034019470215, -1.1952340602874756, -3.3907549381256104, -0.37016186118125916, 1.3878458738327026, 0.9031661152839661, -1.901642084121704, 1.0249370336532593, -2.6657848358154297, 3.472156524658203, -2.7489521503448486, 0.5539653301239014, 3.2779688835144043], [1.628463864326477, -4.376368522644043, 0.20485366880893707, -5.066538333892822, 5.6420464515686035, 0.8562178015708923, 3.8563780784606934, -1.6036105155944824, -1.148932933807373, 1.912088394165039, -6.912851333618164, -3.5729260444641113, 2.756561756134033, -0.08832754194736481, 3.8372440338134766, -0.9481971859931946, 2.8722782135009766, 3.0871849060058594, 0.09458065778017044, 4.340865135192871, 1.2226598262786865, -5.028749465942383, -1.9885330200195312, 3.4027538299560547, -4.1954755783081055, -3.725931167602539, -4.056022644042969, 0.6370581388473511, -0.16144366562366486, -2.825242042541504, 1.558990478515625, -4.375372886657715], [2.7218194007873535, -0.1367901861667633, -1.9441255331039429, 1.8650546073913574, -2.091202735900879, -0.0585322231054306, 5.223823547363281, 0.646891176700592, 1.3083170652389526, 1.4395065307617188, 0.6674771904945374, -1.9125312566757202, 3.246973991394043, -1.518620491027832, 4.348074913024902, -0.9073289036750793, -3.592158794403076, -0.6749902963638306, 1.4408819675445557, -1.5390925407409668, 3.4373466968536377, 2.9799017906188965, 0.5308941602706909, -2.6136622428894043, -5.2632222175598145, -0.6370145678520203, 3.0050933361053467, 1.0713876485824585, -1.4186066389083862, -3.6553196907043457, 3.4230587482452393, -4.509125709533691], [6.788120269775391, -2.785498857498169, -1.0157383680343628, 0.3999488949775696, -2.4178545475006104, 2.953850746154785, 1.6522042751312256, 3.048553943634033, -3.238218307495117, 0.40671128034591675, 1.613436222076416, -2.245046615600586, -1.2206895351409912, 1.4171775579452515, 0.2694963216781616, 3.1795716285705566, -0.3822350800037384, -0.27113020420074463, 3.9139342308044434, -2.4350292682647705, 0.04330611228942871, 1.0282809734344482, 1.9682550430297852, -0.03290369734168053, -1.9050110578536987, 7.231176853179932, -0.20204460620880127, 1.4971758127212524, 5.655954360961914, -2.344897747039795, 2.6904916763305664, 0.032048583030700684], [2.6878561973571777, 2.0319290161132812, -2.4566783905029297, -0.645179271697998, 2.4402060508728027, -1.516705870628357, -3.8109447956085205, 1.564774751663208, -0.7079555988311768, -0.8509548902511597, -1.5215452909469604, -3.088622570037842, 0.9746322631835938, 1.9462119340896606, -0.8569760322570801, -3.1105685234069824, -1.3070300817489624, 0.2718600630760193, 2.8490445613861084, 2.0983211994171143, -4.789042949676514, 0.7937924861907959, -0.19305700063705444, 4.272570610046387, 1.044869065284729, -3.468421459197998, -0.7227088809013367, 1.732520580291748, 3.0312013626098633, 0.8012559413909912, -0.7152097821235657, -2.7035436630249023], [3.8945653438568115, 1.2682337760925293, 2.146862030029297, 1.3555972576141357, 0.19995862245559692, -0.516939640045166, 0.5047632455825806, -2.253939151763916, 4.045507431030273, 2.117494583129883, 2.278886556625366, -5.03179931640625, 4.801222324371338, 3.156829357147217, 4.962725639343262, 0.1903238594532013, 1.2789306640625, 3.202202320098877, -0.21239912509918213, -3.3261213302612305, -7.781099796295166, -0.8502265214920044, -6.353896141052246, 6.4942626953125, -0.6164371967315674, 0.6576966047286987, 3.584646701812744, -3.050818920135498, -6.531638145446777, -4.595792293548584, 5.1343913078308105, -1.446570873260498], [-0.031382784247398376, 2.0021791458129883, 0.7045875787734985, 2.8175909519195557, -5.591386318206787, 2.1624362468719482, 1.736323356628418, 3.854429244995117, 0.20245778560638428, 0.3737444281578064, -1.5872851610183716, -0.6602733731269836, 1.4300192594528198, 1.7726941108703613, 1.8879919052124023, -1.9239258766174316, -0.6775524616241455, 4.911344528198242, 4.981873989105225, 4.19268798828125, 1.2377394437789917, -3.2218070030212402, 0.6621789336204529, 2.2014875411987305, 0.03748604655265808, -2.3301877975463867, -4.903472423553467, 7.356152057647705, 6.926633834838867, -1.912879228591919, 5.332121849060059, 1.8210268020629883], [3.4165525436401367, 0.3562343418598175, -1.834228754043579, 0.5808151364326477, 1.1850404739379883, -0.9439505338668823, 4.510730743408203, -3.5999608039855957, -1.3558413982391357, -2.4859533309936523, -1.5621575117111206, -0.7153039574623108, 3.1464385986328125, -1.0953398942947388, 4.056573867797852, -0.9468564987182617, 1.8073235750198364, -3.054439067840576, 5.128353595733643, 0.9814158082008362, 3.977963447570801, -0.9622054696083069, 1.1487343311309814, 3.8592002391815186, 0.9723153114318848, 6.159906387329102, 3.08203125, 2.5382885932922363, -1.6443560123443604, -2.4399118423461914, 3.1579580307006836, -1.1002767086029053], [3.3847036361694336, 1.701033592224121, -1.4715192317962646, -5.327037811279297, 0.5803175568580627, 2.330286979675293, 1.0285708904266357, -4.363192081451416, 2.162855863571167, 3.637803077697754, 1.0694386959075928, 2.5413269996643066, -1.9020411968231201, 1.1826233863830566, -3.9473042488098145, -2.7760300636291504, 5.906117916107178, -0.3476850390434265, -2.1849844455718994, -4.624963283538818, 4.524065971374512, 5.246277809143066, -2.285266876220703, -2.1746320724487305, 1.5476751327514648, -1.8101732730865479, 3.245616912841797, 3.963742256164551, -1.8281381130218506, 3.4408514499664307, 3.5570077896118164, 3.0403003692626953], [2.6286838054656982, -0.5500751733779907, 5.605551242828369, 3.579817056655884, 1.2463628053665161, -4.259091377258301, 0.4040922522544861, -2.094604253768921, -3.286318302154541, -4.558185577392578, 1.6335028409957886, 6.499850273132324, -1.5540893077850342, 4.912929058074951, -3.145021677017212, -3.689751625061035, -0.6217061281204224, 2.1699459552764893, -1.928185224533081, 0.9956926703453064, 2.2255258560180664, 1.4487650394439697, -1.9095371961593628, -1.6483039855957031, -7.074889183044434, -2.5912530422210693, 0.7576628923416138, 1.0390208959579468, 0.7850030064582825, 2.4565484523773193, -2.37723445892334, 1.4340198040008545], [-3.4792284965515137, 0.5398949384689331, -2.7954745292663574, 2.5743556022644043, 1.5478341579437256, 4.679221153259277, -1.082579255104065, 2.8570730686187744, 0.7480762004852295, 2.784355401992798, 4.184531211853027, 1.565307855606079, 3.2709596157073975, 1.9476324319839478, -0.9916414022445679, -4.90038537979126, -1.5325441360473633, -6.499528884887695, 1.1363537311553955, -2.5338330268859863, 0.6615030765533447, -2.765402317047119, 0.3211359977722168, -0.12963047623634338, 0.7676414251327515, 0.3300456702709198, -3.0469675064086914, -1.748490333557129, -1.3369896411895752, -2.0389046669006348, -0.7143111228942871, -3.1243486404418945], [2.5776546001434326, -1.4241493940353394, -1.360624074935913, 0.4372641444206238, 2.925109624862671, -0.33361583948135376, -0.4882073402404785, -1.4595143795013428, -1.109545111656189, -0.7465990781784058, 2.1419854164123535, 0.10960429161787033, -5.109199047088623, -0.8574467897415161, 1.2703202962875366, -1.8005043268203735, 2.963217258453369, -3.573568344116211, 2.041342258453369, 1.5599031448364258, 8.004045486450195, -0.21760571002960205, -6.834031105041504, 1.604449987411499, 1.5218505859375, -0.8936804533004761, -0.016397766768932343, 3.234421491622925, 3.8242573738098145, -2.6694421768188477, -0.9582833647727966, 3.3653159141540527], [2.770163059234619, -0.2335088700056076, 2.739832639694214, -1.4907286167144775, 4.138805866241455, 5.803178310394287, 6.274548053741455, -4.849743366241455, -0.7869096994400024, 3.708686113357544, 1.0536774396896362, 1.471628189086914, -4.030521869659424, 2.562488555908203, 3.381422281265259, 0.17193223536014557, 2.4295783042907715, 0.2807691991329193, 5.87294864654541, -0.8361256122589111, 9.564458847045898, -1.7547751665115356, -0.5832006335258484, -1.4379453659057617, 0.5046748518943787, -2.0841317176818848, -2.5267467498779297, 3.050959825515747, 1.8494335412979126, -0.9165116548538208, 0.4513140916824341, 0.839748740196228], [-4.324372291564941, 3.015101432800293, -1.2159758806228638, -1.1622790098190308, 2.709589958190918, 9.94611644744873, -4.647522926330566, -2.0346784591674805, -1.279677152633667, -0.5892741680145264, 3.998690128326416, 1.4526317119598389, -2.9552085399627686, 4.5406060218811035, 5.784357070922852, 0.9627343416213989, -1.2132941484451294, -2.359376907348633, -0.9424804449081421, -2.5075273513793945, 4.254214286804199, 0.9932196140289307, -0.4732495844364166, -3.3948404788970947, 0.7653431296348572, -1.8685691356658936, 0.7017757892608643, -0.2871896028518677, -3.721158027648926, -1.828008770942688, 0.4315827190876007, -3.1189863681793213], [-1.4759094715118408, -1.3019613027572632, 4.486741542816162, -1.345053791999817, -3.8958587646484375, -4.261483192443848, 0.44903093576431274, 0.004348237067461014, -1.0143773555755615, -2.2157933712005615, -5.2789201736450195, -6.217881202697754, 4.094380855560303, -1.8349666595458984, -1.9905141592025757, 1.9405252933502197, 2.4000701904296875, 3.0963544845581055, -5.140285491943359, -0.6509684920310974, -0.35378405451774597, -0.7842202186584473, 4.970911502838135, 2.8906075954437256, -2.002082109451294, 2.207207202911377, -3.0382261276245117, -2.335562229156494, -1.0385127067565918, 1.1735918521881104, -1.0673086643218994, 0.41950759291648865], [-1.245619773864746, -2.9968104362487793, 1.4765604734420776, -3.5683436393737793, 0.6363672018051147, -3.760064125061035, -0.801480770111084, 0.9427293539047241, 1.7498905658721924, 2.4075303077697754, -1.636523962020874, -1.062038779258728, -7.736575126647949, -1.162506103515625, 1.3095427751541138, -2.8912153244018555, 1.9037983417510986, -3.7716257572174072, 1.2843575477600098, 3.164163589477539, 3.135950803756714, -1.374097466468811, 5.157192707061768, -2.3291523456573486, 2.110435962677002, -6.463194847106934, -2.2046022415161133, 4.623374938964844, -1.8666290044784546, -0.9877905249595642, -2.1232922077178955, 5.340989589691162], [-2.9202959537506104, 2.495333433151245, 0.1468765288591385, 3.258305549621582, -0.12858246266841888, 0.6297985315322876, 1.534501314163208, -2.6521575450897217, 4.581395149230957, -1.5273656845092773, -2.053706169128418, 0.8715274930000305, -4.428952693939209, 1.8399481773376465, 2.0606513023376465, 4.3130316734313965, 1.4901456832885742, 1.5917460918426514, 2.066171169281006, 1.2371021509170532, 5.083567142486572, 4.063209533691406, -0.8478336334228516, 0.8194690942764282, 0.8732040524482727, -5.794510364532471, 2.7019875049591064, -5.63296365737915, -4.215960502624512, -3.3257250785827637, 0.7237248420715332, -1.1487255096435547], [-3.003974676132202, 0.9599379301071167, -5.546306610107422, -0.39774540066719055, -0.26602089405059814, -0.3488401770591736, 5.012046813964844, -0.9392228126525879, 1.017143964767456, 0.9692872762680054, -0.8455550670623779, -3.8411874771118164, 2.6295313835144043, -3.834895610809326, -1.4208729267120361, -1.9906589984893799, 2.241940498352051, 6.852631568908691, -2.360352039337158, 1.0742788314819336, -1.1103835105895996, -0.5261861085891724, -5.4233598709106445, -2.352954387664795, -2.795213222503662, -3.0233638286590576, 1.247422695159912, -2.90252685546875, 4.995428085327148, 2.3037610054016113, 2.815556526184082, 0.47403502464294434], [-2.57606840133667, 1.1620113849639893, -0.9232973456382751, 1.5220005512237549, 5.648766994476318, -0.27275094389915466, -0.013801326975226402, -2.2796764373779297, -2.0502400398254395, 1.7902016639709473, 3.236699104309082, -1.2376905679702759, 3.8672842979431152, -1.1565730571746826, -1.9844038486480713, 1.263580560684204, -2.7889299392700195, -3.4001216888427734, -1.5591691732406616, 1.6509087085723877, -1.1239266395568848, 0.3594624102115631, -1.1738805770874023, -0.48546749353408813, -1.0740067958831787, 1.610168218612671, 2.071646213531494, 0.0979209840297699, 1.1560359001159668, -0.9917868375778198, 0.04048086330294609, 5.4854607582092285], [3.39798903465271, 1.3218088150024414, -2.124889850616455, 3.8429393768310547, -2.938790798187256, -5.331993103027344, -5.788037300109863, -1.7409693002700806, 6.637955188751221, 1.5402753353118896, 3.9175143241882324, 0.5535951852798462, 2.2366747856140137, 5.404886722564697, 1.0918598175048828, 1.361515998840332, 0.3392271101474762, 1.074986219406128, 1.465759515762329, -0.5593959093093872, -5.020951271057129, -0.6075880527496338, 2.1350247859954834, -1.223806619644165, 2.1591663360595703, -0.7912260293960571, 0.1926569938659668, 6.53639030456543, 3.655843734741211, -5.37938928604126, -1.759753942489624, -1.427870273590088], [1.2447457313537598, -3.0452370643615723, -0.17341595888137817, 1.6189146041870117, 0.401712030172348, 3.0177135467529297, -0.4175340235233307, 0.8130581974983215, -1.0288746356964111, -1.2513245344161987, -0.9284909963607788, -5.8529558181762695, 2.274757146835327, 5.271358013153076, 1.6711044311523438, -4.201584339141846, -2.8041670322418213, -4.341634750366211, 5.668891429901123, 0.17796394228935242, 6.296786308288574, -1.3520557880401611, 6.237583160400391, -0.8415133953094482, -3.5846338272094727, -0.31963521242141724, -2.2465052604675293, -1.6671433448791504, 0.696052074432373, -6.304305553436279, 0.107664093375206, -4.371480941772461], [-1.5976366996765137, -4.983591556549072, -1.1313936710357666, 2.8071987628936768, 4.929191589355469, -1.5946422815322876, 2.0026516914367676, -2.0284149646759033, 2.0598230361938477, -1.2819780111312866, -3.884685516357422, 5.189270496368408, 0.9314923286437988, 4.083776473999023, 7.142427444458008, -1.778770923614502, 2.19698429107666, -3.525092601776123, -2.052353620529175, 0.4584682881832123, 0.019476745277643204, -0.6763217449188232, 2.5450077056884766, 0.8332867622375488, 0.09855207055807114, 1.6292771100997925, -0.04780488461256027, 3.3300485610961914, 0.2774713337421417, -6.272798538208008, -7.372142314910889, -1.0274808406829834], [0.3848230540752411, 3.5325865745544434, -1.5920753479003906, 1.253031611442566, 3.831835985183716, -0.8768550753593445, -1.9350652694702148, -2.4451208114624023, -0.671378493309021, -8.2018461227417, 0.48044291138648987, 0.8990788459777832, 4.184659957885742, -4.051130771636963, -0.6252350211143494, 4.522789001464844, -3.6610913276672363, 1.9374279975891113, 4.436075210571289, -1.6897859573364258, 1.3859632015228271, 0.9623366594314575, -1.3945176601409912, -4.950973033905029, 2.229245185852051, -3.401599407196045, 5.512835502624512, 5.195618629455566, 1.8065311908721924, -0.1778058558702469, -0.5919627547264099, 1.506468415260315], [-0.8751637935638428, 0.426198810338974, 2.077705144882202, 2.6437137126922607, -3.3813095092773438, -3.7026493549346924, 1.2586445808410645, -2.8882334232330322, 2.7358384132385254, -2.1550185680389404, 4.752608776092529, -0.03968760743737221, -2.7920103073120117, 0.3429471254348755, -0.922590970993042, -0.012089550495147705, 1.5231502056121826, 2.379903793334961, 5.693842887878418, -6.708476543426514, 2.729811668395996, 1.4670275449752808, 7.035043716430664, -5.975419998168945, -2.2989683151245117, -4.456551551818848, -0.654869019985199, 0.864123523235321, 1.6880009174346924, 0.22057881951332092, 1.8407878875732422, 2.841439723968506], [1.2614920139312744, -5.78964376449585, -1.2635658979415894, -0.591025710105896, 4.349707126617432, 2.3930230140686035, -0.8629454970359802, -4.430427074432373, 1.7598917484283447, 2.2755062580108643, -1.9102710485458374, -2.4787778854370117, -2.2171101570129395, 6.561485767364502, 2.917236089706421, -1.1599371433258057, 0.04278484359383583, 1.8870129585266113, -2.608677864074707, 0.06350883096456528, -0.9248734712600708, -2.1308062076568604, 3.038527011871338, -5.891140937805176, 1.453130841255188, -0.24594494700431824, -8.254985809326172, 0.6792744398117065, 4.037353992462158, -4.942160606384277, -3.7575523853302, 1.73284113407135], [7.037683486938477, 6.954706192016602, -0.8004283905029297, 4.21628475189209, -2.958756446838379, -2.3551621437072754, 4.439449310302734, 1.6044292449951172, 1.1108795404434204, -1.5797706842422485, 2.4754014015197754, 5.899805545806885, -0.6880216002464294, -4.132093906402588, 1.8714755773544312, -1.6312155723571777, 0.6367906332015991, -0.1119077056646347, -0.6099896430969238, -2.9957242012023926, 1.109245777130127, 1.508624792098999, 5.022871494293213, -2.9978160858154297, 2.543250560760498, 5.229339599609375, 4.692417621612549, -1.217706322669983, -2.711064338684082, -1.4448655843734741, -2.1821441650390625, 2.1272075176239014], [-3.4564108848571777, 3.660437822341919, -3.8968584537506104, 0.3086708188056946, 0.8616873621940613, -0.19569283723831177, 3.3570377826690674, -1.7506451606750488, -1.8371284008026123, 0.37025171518325806, -2.629354238510132, 2.627572536468506, 2.9761006832122803, -1.7652921676635742, -0.4549538493156433, -2.2717347145080566, 1.7102669477462769, -1.0139696598052979, 2.2962088584899902, -4.019134521484375, 0.7289136052131653, 0.4732274115085602, 2.019552707672119, 0.4964190423488617, 7.072479248046875, -1.757617712020874, -2.534053087234497, 0.4067235589027405, -3.9435172080993652, 1.121599555015564, 6.492347717285156, 1.8665939569473267], [2.68577241897583, -3.3523316383361816, -2.294515371322632, 1.7787584066390991, 1.8908863067626953, 2.739758014678955, 1.9702870845794678, -0.8173901438713074, -1.978722095489502, -0.015045644715428352, 0.9258058071136475, 1.3288142681121826, -0.8302825689315796, -0.8308852910995483, 3.2864606380462646, -2.163553476333618, -3.184652328491211, -3.0610649585723877, 0.7806885838508606, -2.740544319152832, 2.2990784645080566, 0.6236984133720398, 6.882343769073486, 2.877288341522217, 4.595876216888428, 3.2962539196014404, -4.342047214508057, 1.939195990562439, 7.235411643981934, 2.146638870239258, 1.4548704624176025, -4.825012683868408], [4.472848415374756, -3.330625057220459, -1.5057382583618164, -4.6559648513793945, 0.4217418432235718, -1.6577017307281494, -2.1489686965942383, -4.167422294616699, -6.006421089172363, 6.442610740661621, -3.612135887145996, 0.46147406101226807, 2.153468132019043, -2.263033628463745, -2.8883473873138428, -0.27193981409072876, -3.2612690925598145, -0.9620452523231506, -2.348902940750122, -1.6278820037841797, 0.9132188558578491, -3.2586097717285156, -0.5425394177436829, 0.8327609896659851, 0.7148218154907227, -1.292965054512024, 2.2088782787323, -1.6017812490463257, 4.495852947235107, -3.504974126815796, 1.37699294090271, 4.458983421325684], [3.5969748497009277, -1.7276206016540527, -1.2348828315734863, -5.615776062011719, -3.4523634910583496, -3.2608165740966797, -0.16056674718856812, 4.361184120178223, 1.2150980234146118, -2.7377123832702637, -3.0941338539123535, -3.4406919479370117, 1.3497436046600342, 6.423224449157715, -1.1766606569290161, -3.0333518981933594, 1.5260379314422607, 4.33819580078125, -4.050140380859375, -4.160444259643555, -2.382613182067871, -3.6408257484436035, 5.490968227386475, -0.01499196421355009, -2.309934616088867, 7.547002792358398, 0.9112895727157593, -1.134503960609436, 0.7279680967330933, 0.4831709861755371, -0.9599395990371704, -1.4246704578399658], [-2.6343228816986084, 0.7351105213165283, 1.2962220907211304, 0.7532727718353271, 1.911420464515686, -0.17650289833545685, -3.0446341037750244, -4.813257217407227, -0.32550889253616333, 5.514944553375244, 1.742475986480713, -1.3048033714294434, 2.229820489883423, -6.679718017578125, 1.5670297145843506, -3.297456979751587, -1.9483671188354492, -0.7119777202606201, -5.208515167236328, -3.992290735244751, -1.4460060596466064, -1.341761827468872, -6.059812545776367, -3.1735596656799316, 3.6134414672851562, -0.5542347431182861, -1.0788344144821167, 4.703268051147461, 2.877591133117676, 0.639687716960907, -1.9284939765930176, -4.814787864685059], [6.225360870361328, -4.493470191955566, 1.0021167993545532, 1.0312981605529785, 3.1668291091918945, 4.936585426330566, 4.374667167663574, -0.8985146284103394, -0.41229480504989624, 1.5563149452209473, 1.4554433822631836, -4.463372230529785, 6.071887969970703, -1.1660676002502441, 2.2737369537353516, 1.3085548877716064, 0.4467626214027405, -2.685166120529175, -3.6078643798828125, -3.8766839504241943, -5.790895938873291, 5.367929458618164, -3.398944854736328, -0.7006896734237671, 4.398126125335693, 3.2654693126678467, -0.6305988430976868, -0.8223250508308411, 1.756354808807373, -1.1021651029586792, -0.45806148648262024, -0.9377044439315796], [-0.04859338700771332, 3.430105209350586, -3.0477294921875, 2.4301018714904785, 0.11330027878284454, 2.1035706996917725, 1.2382076978683472, 3.687669038772583, -2.871316432952881, 4.683629512786865, -2.407351493835449, -2.4269416332244873, -0.723169207572937, 0.1162877231836319, -1.7729690074920654, -4.2496137619018555, 1.1670887470245361, 1.2968368530273438, 4.317259788513184, -2.4871275424957275, 1.124764084815979, -1.3358598947525024, -0.7407059073448181, -0.2738516628742218, 0.9493687152862549, -3.5368552207946777, 3.5341548919677734, 4.755587577819824, -5.066678524017334, 4.724529266357422, 1.3494421243667603, 1.7598885297775269], [-4.074277400970459, -5.350057125091553, 0.38853681087493896, -1.1768758296966553, -0.19969135522842407, 3.2472589015960693, 3.9672794342041016, 0.938798189163208, -3.3128914833068848, -0.3233168125152588, 1.9480547904968262, 2.8215746879577637, -4.035521984100342, -1.943333625793457, -2.7190709114074707, -2.6229546070098877, 1.087540864944458, -2.847726583480835, -0.012943867594003677, 2.211674213409424, -2.2267584800720215, -1.6236037015914917, 5.301218509674072, -2.3045778274536133, -0.8384600877761841, 1.3382209539413452, 3.8641910552978516, -5.878907203674316, 2.3427164554595947, -6.782412528991699, 3.3440890312194824, -4.487837314605713], [-1.0152957439422607, -0.7567481994628906, 2.8343734741210938, 3.7428460121154785, 0.9990106821060181, -2.1779696941375732, 0.28934040665626526, 3.3448758125305176, 0.8226647973060608, -2.286067008972168, 1.9951010942459106, -2.7431230545043945, -0.40256887674331665, 0.22815203666687012, -1.9651949405670166, -0.19710609316825867, -2.9859728813171387, -2.897263765335083, -3.957914352416992, -1.035675287246704, 5.011940002441406, -3.8538174629211426, -1.254981517791748, -0.33799493312835693, -0.36025670170783997, 1.1941401958465576, -4.776893615722656, -3.259260654449463, -0.3429691195487976, -1.331455945968628, 1.1463327407836914, 1.467170000076294], [1.3428202867507935, 4.517214775085449, 0.37720751762390137, 2.4303011894226074, 3.406240940093994, -1.7075374126434326, -0.46160078048706055, -0.2719639837741852, 0.14577415585517883, -1.3993531465530396, 7.23837947845459, -6.231900215148926, 1.5995121002197266, -0.3565037250518799, 0.4138703942298889, 5.515104293823242, -0.8067007064819336, 2.578235149383545, -1.8295100927352905, 1.7989366054534912, 0.6116941571235657, -2.5033199787139893, 2.432354688644409, 1.7900843620300293, -4.320713043212891, 0.007906343787908554, -0.13838912546634674, -1.0546458959579468, -1.0050958395004272, -1.1416571140289307, -4.593318939208984, 2.407207727432251], [-1.76987886428833, -1.4236578941345215, -3.2154088020324707, -2.4814467430114746, -5.896327018737793, 3.234835147857666, -1.2887099981307983, 2.087191581726074, 1.6931748390197754, -1.4164491891860962, -0.5188040733337402, -3.062136173248291, 1.0055177211761475, -1.356644868850708, 3.016751289367676, -1.2799994945526123, -3.7286806106567383, -1.3895608186721802, 5.279251575469971, 2.612239360809326, 2.723879337310791, -2.197410821914673, -0.7824506163597107, -2.145528554916382, -1.8770723342895508, 2.2633073329925537, 5.9939398765563965, 1.2451812028884888, -0.15297797322273254, -2.674794912338257, 0.5221198201179504, -0.1574787050485611], [0.5233314037322998, -2.596193552017212, -3.235003709793091, -0.3647052049636841, 0.5714209675788879, 2.571014404296875, 3.2548680305480957, 1.9834952354431152, 0.9014254808425903, -0.2712221145629883, 1.8681310415267944, 3.4034619331359863, 4.3658037185668945, 2.975980758666992, 6.611148834228516, -4.003474235534668, 4.061454772949219, 0.21367639303207397, 0.7729827165603638, -6.063356399536133, 3.8089370727539062, -1.4777958393096924, -1.002105474472046, -4.441869258880615, 8.49231243133545, 0.5968992114067078, -4.3813958168029785, 1.885317325592041, 2.0050930976867676, -2.6613540649414062, 0.0924963429570198, 0.8738585114479065], [4.8557634353637695, 0.03660115599632263, 1.1771644353866577, 1.4468390941619873, -1.0869413614273071, -1.9479429721832275, -1.5542500019073486, -1.3758004903793335, 2.372474193572998, -0.02554144337773323, -3.1055941581726074, 0.41421639919281006, 5.8352952003479, -2.9919941425323486, 0.11029548943042755, 4.270012378692627, 4.3934173583984375, 3.5272011756896973, 3.2494444847106934, -4.451934814453125, 3.010105609893799, 1.302944302558899, -2.032503128051758, 5.7344584465026855, -1.836159110069275, 5.957298755645752, 2.3355538845062256, 0.6737245321273804, 1.462080955505371, 2.452120780944824, 6.617790222167969, 1.742647647857666], [-0.1047767624258995, -1.2356905937194824, -0.5485997796058655, -5.977962493896484, 3.302950143814087, 2.416810989379883, -5.40046501159668, -0.6217235326766968, 3.859307289123535, 0.3402100205421448, -1.4310290813446045, -0.9973102807998657, 0.780733585357666, -2.048640012741089, 3.6168131828308105, -0.40262192487716675, 2.417830467224121, -1.867638349533081, -7.025174617767334, -0.20084509253501892, -1.1868150234222412, -4.2331132888793945, -1.7701494693756104, 0.4083523750305176, -1.3754700422286987, 0.573087751865387, 0.43054938316345215, 1.1827292442321777, -0.5539311766624451, -4.676055431365967, -0.29737401008605957, 1.798940896987915], [3.115711212158203, -4.592288970947266, 1.1221098899841309, -2.598195791244507, 0.8626416921615601, 1.9822415113449097, -1.8169028759002686, 3.487072467803955, 2.420121192932129, -1.1268562078475952, -1.1862483024597168, 1.4247746467590332, 0.7871065735816956, 1.407194972038269, 2.488624095916748, -1.5320923328399658, -3.159597873687744, -2.2588067054748535, -4.995879173278809, -0.24188625812530518, 3.450873374938965, 0.6656237244606018, -1.025395154953003, 0.9769999980926514, -1.883123517036438, -1.8401098251342773, -1.0138518810272217, 2.3460476398468018, 0.18290317058563232, 0.2853162884712219, -3.675172805786133, 1.5422968864440918], [-5.298676013946533, -5.1461310386657715, -1.8912444114685059, 0.7191907167434692, -1.3569704294204712, 4.3292012214660645, 1.2500333786010742, 2.5906078815460205, 1.494929313659668, 0.5095458030700684, 4.296440124511719, -0.45730024576187134, -2.7159407138824463, -1.606508731842041, 2.025345802307129, 2.5390262603759766, -3.855353593826294, 1.972303032875061, 1.6576271057128906, 10.007200241088867, -0.005569105502218008, -2.1528449058532715, -4.1183762550354, 1.2775877714157104, -0.5524146556854248, 1.0310249328613281, -0.44790977239608765, 1.0051137208938599, 0.5302303433418274, -6.146791458129883, -4.30097770690918, 5.437117099761963], [1.9482126235961914, -1.0794507265090942, 2.4451229572296143, 1.1275826692581177, 0.9667099714279175, 6.9595537185668945, 5.1446452140808105, 3.2537989616394043, -2.2074763774871826, -0.9060860872268677, -0.9626314640045166, 4.042869567871094, -3.404974937438965, -1.4371107816696167, 5.209365367889404, 1.3350895643234253, -1.2482411861419678, 0.9429236054420471, 0.9273392558097839, -3.5694522857666016, 1.2920660972595215, -0.48203474283218384, -2.564793109893799, -0.8956682682037354, -0.1277695596218109, -0.18679164350032806, -2.0384185314178467, -1.3382596969604492, 1.6782023906707764, 0.51540607213974, -1.7286561727523804, -2.129976272583008], [-2.1934192180633545, 1.4880914688110352, 3.3112471103668213, 4.465405464172363, 1.142895221710205, 0.926582932472229, 0.5160837769508362, 2.2942066192626953, -3.095111131668091, 2.480205535888672, -4.2868547439575195, 1.9057071208953857, -2.8822569847106934, 5.669055938720703, 1.0782181024551392, 0.8532218933105469, 2.3006298542022705, 0.43901634216308594, -1.0655808448791504, 1.2160594463348389, 2.8678340911865234, 1.849148154258728, -2.4465017318725586, -2.3733465671539307, 3.2012648582458496, 2.039835214614868, -1.3153806924819946, 0.30582261085510254, -0.825157880783081, -2.6071317195892334, 6.805662155151367, -0.9203929901123047], [4.531821250915527, -3.955077648162842, -1.1986300945281982, 3.679687023162842, -0.011076866649091244, 2.539794445037842, 2.63110089302063, 0.40506720542907715, -2.231552839279175, -4.669726371765137, 4.550093173980713, 0.8948602676391602, -4.406961441040039, -5.2054009437561035, -2.318955421447754, 2.065350294113159, -3.7304954528808594, -3.445934772491455, -1.6919792890548706, 5.7861528396606445, 0.6580917239189148, 2.088557243347168, 5.359218597412109, -0.7698028087615967, -3.043383836746216, 1.7244603633880615, -2.6770453453063965, -0.2805204689502716, -2.173638105392456, 2.2859368324279785, 3.913079023361206, -0.8368228673934937], [-0.5176247954368591, -0.8204841613769531, 0.7818934917449951, 0.7684184312820435, -4.2577409744262695, 0.8632943630218506, 4.464613437652588, 1.3317351341247559, 0.41043269634246826, 0.6602498292922974, 3.9213762283325195, -1.9728882312774658, 2.0139341354370117, 2.0284998416900635, 9.069068908691406, 2.337294101715088, 1.6439831256866455, 3.4647789001464844, 3.5420055389404297, 0.9643793106079102, -0.7599163055419922, 0.6333620548248291, -1.8983960151672363, -1.442112684249878, -0.8174401521682739, -0.5541723370552063, -2.9544425010681152, 2.7682065963745117, -2.393646478652954, 0.4435938000679016, -5.435869216918945, 7.034026622772217], [1.8442211151123047, -2.3866257667541504, 2.966775417327881, -4.105749130249023, 1.1097941398620605, 0.2883197069168091, 3.166325569152832, -0.24080321192741394, -1.366525650024414, 0.38407713174819946, -3.809231758117676, 1.6926183700561523, 1.8720293045043945, -0.1879827082157135, 6.052155017852783, 1.5287915468215942, 4.504755020141602, 4.133981227874756, 4.815608978271484, -7.3611040115356445, -2.3080222606658936, -5.256860256195068, -6.851406097412109, 3.214843511581421, -3.751049280166626, 0.8676458597183228, -4.416191101074219, -0.4582170844078064, 1.5625014305114746, -4.366325855255127, 0.3546423017978668, 1.6376218795776367], [-3.9855546951293945, -0.20655983686447144, -1.5826480388641357, 1.9320298433303833, 1.529794454574585, -3.433011054992676, 0.8628518581390381, -3.0958378314971924, -2.3591628074645996, -1.754896879196167, -1.1603587865829468, 3.63860821723938, -0.9114241003990173, 3.3142261505126953, 4.047170639038086, -5.839757919311523, 2.7022576332092285, -1.653336524963379, 3.956051826477051, 2.4708635807037354, -3.6362738609313965, 0.3694283068180084, -1.6747455596923828, 0.3775399625301361, 0.3374360203742981, 1.3965957164764404, -1.0950305461883545, -0.5428563356399536, -0.19442802667617798, -1.8491168022155762, -0.519354522228241, -0.6418067216873169], [-2.868213653564453, -2.2042348384857178, -5.738832473754883, 3.0807390213012695, 0.6024678349494934, 2.1411633491516113, 0.8883966207504272, -4.964779853820801, 0.5425267219543457, 2.7044196128845215, -5.354883193969727, -0.8956909775733948, 0.8604142069816589, 1.5021789073944092, 4.609410285949707, -3.283463478088379, -0.8215601444244385, 4.031027793884277, -2.3981289863586426, -5.908694744110107, 1.5574347972869873, 2.31546950340271, 1.5320656299591064, -3.9220633506774902, -4.0462141036987305, 4.194518089294434, 2.83803129196167, -2.9486241340637207, 3.4257266521453857, 1.1573302745819092, 1.361485242843628, 3.3310036659240723], [-6.0135908126831055, 1.4120137691497803, 0.07039128243923187, 3.3329553604125977, -3.4121453762054443, -1.8913416862487793, -0.41553473472595215, -1.083755612373352, -0.9445397257804871, -2.2020139694213867, 0.4555462896823883, -3.762037754058838, 0.48987138271331787, 1.9661705493927002, -1.4711105823516846, 3.0529003143310547, -1.966994285583496, 3.3616943359375, -4.943892478942871, -2.918527841567993, 3.4767258167266846, -0.6531456708908081, -7.620584964752197, 0.6569546461105347, -5.764699459075928, 0.25671282410621643, 3.6373214721679688, -2.6760058403015137, 1.9348969459533691, -4.081223011016846, 2.971595525741577, 1.8711154460906982]], [[1.9846925735473633, 1.1493854522705078, -2.3345823287963867, -3.3366658687591553, -2.7283148765563965, 0.3977205157279968, 1.327407717704773, 0.08873917907476425, 5.323955535888672, -3.028285503387451, -1.2891672849655151, 2.783989191055298, -2.3375000953674316, -0.637976348400116, -3.8454933166503906, 1.9766067266464233, -2.8387463092803955, -0.39877161383628845, 1.5194675922393799, -0.5969974398612976, 6.873723983764648, 0.048092059791088104, -1.1282974481582642, 2.148768901824951, -2.9320216178894043, -0.055925674736499786, -0.27648597955703735, 0.3520350456237793, -0.8973180055618286, 1.7318719625473022, 4.218247890472412, -1.9012739658355713], [-1.1532423496246338, -0.12421229481697083, -0.007189027965068817, -0.4907230734825134, 0.12948821485042572, 3.357180595397949, 0.7097852230072021, -1.9025763273239136, -2.3417351245880127, 1.3954031467437744, -3.5389087200164795, -2.1719422340393066, 1.3812873363494873, -0.8432838320732117, 0.8299328088760376, -0.6976039409637451, -0.6221226453781128, -2.8688254356384277, -2.496856451034546, -4.40314245223999, 3.8832764625549316, 2.0678350925445557, 2.2856085300445557, -0.735539972782135, 8.057882308959961, -1.2486642599105835, 6.509955406188965, -1.6371660232543945, 1.1059718132019043, -0.5713061690330505, -2.233264446258545, -2.1565310955047607], [1.4693467617034912, -0.9538506269454956, 2.6155178546905518, -0.7946232557296753, 3.3368425369262695, -2.4163215160369873, -3.7397425174713135, -2.9614763259887695, 1.2083725929260254, -3.4639978408813477, -2.590902328491211, 5.6497344970703125, 4.1133270263671875, 4.375800609588623, -1.3523310422897339, -1.3091975450515747, 1.1937755346298218, 1.1249254941940308, 0.6497058272361755, 4.7375030517578125, -2.887030601501465, -1.270112156867981, 0.6722504496574402, 5.7572407722473145, 6.797079563140869, -1.3784449100494385, -1.1596392393112183, 3.3511269092559814, 3.649397373199463, 5.903848648071289, 5.222424507141113, -4.985901355743408], [-2.82080078125, 3.0266637802124023, -0.8942651748657227, -1.6586308479309082, -1.761642336845398, 0.9138109087944031, 0.5105929374694824, -1.7257572412490845, -2.2972662448883057, 1.1435000896453857, 4.375758171081543, -2.9298512935638428, 3.282355308532715, 2.1189961433410645, 4.399718284606934, 3.2058815956115723, 0.26100701093673706, -2.0059468746185303, 0.1069144457578659, 3.655090808868408, 1.216853141784668, -1.236301064491272, 1.380934715270996, -1.250240683555603, 2.6885311603546143, -3.0853471755981445, -0.7589329481124878, -1.2023131847381592, -2.9271223545074463, -0.7594538927078247, -1.7805429697036743, 1.8389664888381958], [0.6214593648910522, 1.405698299407959, -1.8631727695465088, -1.8536021709442139, -0.4750892221927643, -0.5818578004837036, 0.7014069557189941, 0.785759687423706, 4.092011451721191, 0.2611713707447052, -4.750701427459717, 0.5001979470252991, 4.083963871002197, 1.1470023393630981, -5.617004871368408, 1.706777572631836, -3.932718276977539, 2.767374038696289, 2.274235248565674, 2.5596795082092285, 0.11589880287647247, -6.95260763168335, 2.98783540725708, -2.2267870903015137, 5.161414623260498, 3.149954319000244, 4.781288146972656, 0.2463037669658661, 5.526837348937988, -3.2561264038085938, -2.668750524520874, 2.836242437362671], [2.6542298793792725, 2.088362455368042, -1.6050915718078613, 3.177185535430908, 4.7597880363464355, 1.4169977903366089, 1.2802579402923584, 1.8147557973861694, -5.350906848907471, 1.9845603704452515, 3.020359754562378, -1.6696360111236572, -0.14689034223556519, 0.4727691113948822, 3.0505776405334473, 0.08889248222112656, -3.902177095413208, 3.976924180984497, -1.2078032493591309, 0.9853754043579102, -0.33555182814598083, 3.929875373840332, -4.164003372192383, -4.803948402404785, -0.7529323697090149, 0.48483675718307495, -2.3211257457733154, 0.9489457607269287, -0.7061108946800232, -1.6406961679458618, 2.4536170959472656, -0.31405723094940186], [1.7159940004348755, 0.3270413279533386, 1.9815340042114258, -1.617384910583496, 3.7086148262023926, 9.519609451293945, 2.625494956970215, 0.9915856122970581, 2.3844289779663086, -5.833053112030029, -0.7425258159637451, 1.449607253074646, 1.290116310119629, 4.89439058303833, -0.008643114008009434, -0.8092991709709167, -4.958181381225586, -1.0993854999542236, 1.4050936698913574, 0.09055927395820618, -1.6139726638793945, 7.188115119934082, -2.157313346862793, -0.008163711056113243, 1.941704273223877, 3.9552040100097656, 2.913464069366455, 0.5863458514213562, 2.159607172012329, 0.16349908709526062, 3.717796802520752, -0.1321989893913269], [-2.1439781188964844, 4.57758092880249, 3.4010403156280518, 2.5475919246673584, -1.3670837879180908, 5.2615156173706055, 3.1447713375091553, -0.6200822591781616, -2.0816454887390137, -0.20317433774471283, 4.812477111816406, 2.506769895553589, -2.4624686241149902, -0.4625810384750366, -4.402327537536621, -1.6566274166107178, -1.3435965776443481, 1.185245394706726, -0.3416692018508911, -4.99488639831543, -0.5250417590141296, -0.9090766310691833, 1.6206040382385254, 0.5337360501289368, -1.246437907218933, -2.9504380226135254, -7.973654747009277, -1.1220018863677979, 7.407824516296387, -4.104190826416016, -2.4736692905426025, 1.8900375366210938], [-3.0883090496063232, -7.132442951202393, 1.3927829265594482, 4.181460380554199, -5.274542808532715, 1.4319252967834473, 1.482041835784912, -3.375607490539551, -0.1464611440896988, -1.4645678997039795, -0.2687506675720215, 0.3420400023460388, -3.907457113265991, 1.2808431386947632, 1.719886302947998, -2.9923818111419678, 0.6198322772979736, -2.3745274543762207, 1.3420354127883911, -1.5467307567596436, 1.70225191116333, 2.385596513748169, 1.0756349563598633, 1.3295429944992065, 0.8203201293945312, 2.521944046020508, 0.10866788774728775, -4.70836877822876, 2.3424670696258545, -5.004032611846924, 0.024578846991062164, 0.4547153413295746], [2.233654260635376, 0.8000102043151855, 1.4547781944274902, -4.273171901702881, -0.8952382206916809, 8.538647651672363, 1.4948983192443848, -0.47375619411468506, -2.8593873977661133, -3.2015132904052734, -0.8731838464736938, 5.293628692626953, 7.823988914489746, 3.176494598388672, 1.2001097202301025, 2.582797050476074, 0.3443818986415863, -3.7403934001922607, 1.1751760244369507, -1.861452579498291, -1.8583835363388062, -4.591121196746826, 1.528996229171753, -2.838862895965576, 3.2442662715911865, -2.02158784866333, -1.898132562637329, 0.7724852561950684, 0.9536198973655701, 2.284942150115967, 0.024355515837669373, -3.248155117034912], [-5.440356254577637, 0.493613064289093, -0.3008710443973541, 5.613525390625, 2.306041955947876, -0.3258189558982849, 5.762118339538574, 0.0013043646467849612, 3.0053019523620605, 2.1245806217193604, -2.9756014347076416, -3.604846477508545, -0.9370445609092712, 0.5800722241401672, 2.7846407890319824, -3.7274341583251953, -1.1087279319763184, -1.0636521577835083, -0.46695512533187866, -2.810074806213379, 0.5466969013214111, -2.0828912258148193, -0.5337698459625244, 2.151732921600342, -7.393073081970215, 5.0627241134643555, 2.332247734069824, 3.138901710510254, 1.3756250143051147, -2.9851365089416504, 8.962589263916016, -0.6621414422988892], [1.5674099922180176, 5.471785068511963, -0.7181218862533569, 5.354279518127441, -1.1398468017578125, -0.1382642686367035, 3.5145621299743652, -2.4766993522644043, 0.198472261428833, 6.259331703186035, -1.8039517402648926, -4.513486385345459, 1.7325692176818848, -3.957705020904541, 3.250783920288086, 3.8720321655273438, -1.0163581371307373, -5.531357765197754, -0.5966135263442993, -8.169507026672363, 1.0608289241790771, 6.553460121154785, -0.27733856439590454, -3.873081684112549, 1.4200382232666016, 2.5944366455078125, 1.4502522945404053, 1.9762871265411377, 0.20008257031440735, -0.8440011739730835, 1.7240856885910034, -4.258688926696777], [-2.4134011268615723, 0.8915523290634155, -2.661355972290039, -3.5224034786224365, -0.899584174156189, 0.38538557291030884, 5.695087432861328, 1.6824581623077393, -2.9197163581848145, 2.724107503890991, 2.619971752166748, -4.377826690673828, 5.027605056762695, -1.4164479970932007, 1.6263794898986816, -3.7798614501953125, -0.3386816382408142, -1.1325771808624268, 0.4070969223976135, -3.719221830368042, -0.7608771324157715, 0.39335015416145325, -0.592352569103241, 0.19733794033527374, -5.877288818359375, 2.382295608520508, -4.062526702880859, -0.19816476106643677, 5.674912452697754, -0.8178808093070984, -0.12553583085536957, -1.005374789237976], [-0.37587660551071167, -0.3253375291824341, 0.03790881484746933, -2.5630695819854736, 1.0672321319580078, -0.6265913248062134, 0.4396625757217407, -4.33634090423584, 7.748811721801758, 3.3825182914733887, -0.6172444820404053, -2.4353504180908203, -0.026318278163671494, -4.190879821777344, -0.2850213944911957, -3.429965019226074, 1.2004776000976562, 1.1110265254974365, -4.555370330810547, -2.4897351264953613, -1.8051419258117676, -1.4720600843429565, -0.8826688528060913, 2.6751551628112793, -1.525747537612915, -1.405804991722107, -2.3354430198669434, -3.275951862335205, 1.1230865716934204, 2.822993040084839, 4.307523727416992, -0.7823969125747681], [1.9854086637496948, 5.0036420822143555, -2.4107837677001953, 2.415731191635132, -5.8956451416015625, -3.9605841636657715, 2.316866397857666, -2.028465747833252, 4.464660167694092, -1.034112572669983, 0.1485045701265335, 4.733886241912842, 5.110123634338379, 1.207245945930481, 0.23259566724300385, -1.4376137256622314, -0.5144543647766113, -1.131790041923523, -0.16644372045993805, -0.450508713722229, -1.640348196029663, -1.8932186365127563, 1.4432512521743774, 2.5817573070526123, -2.0006093978881836, -1.3215354681015015, 0.4039200246334076, -0.08341920375823975, -5.783356666564941, 0.6907485723495483, -0.4853416979312897, 0.759886622428894], [-4.016794204711914, 0.2861393392086029, -5.057651519775391, -0.05338682234287262, -2.016103506088257, 2.0249757766723633, -0.9341853857040405, 4.24808406829834, -6.524494171142578, -1.4503223896026611, 7.632171630859375, -5.807382583618164, 3.857574939727783, 0.9888405203819275, 3.4182300567626953, 3.7495007514953613, -4.4106550216674805, -0.7357031106948853, -2.193751811981201, -2.0042076110839844, -1.6039373874664307, 2.190131187438965, -0.6478440165519714, -1.5603306293487549, -6.268450736999512, -1.2795295715332031, -0.8250235319137573, 0.5976597666740417, 6.7830915451049805, 2.8065502643585205, -1.7478100061416626, 3.5832972526550293], [-1.4446187019348145, 4.9933695793151855, 0.1589219868183136, 2.4672510623931885, 1.4003105163574219, 0.6640732288360596, -1.3350417613983154, -1.634885311126709, -0.9517119526863098, -1.4888025522232056, -3.020048141479492, -8.421223640441895, -0.6480321288108826, -1.2185401916503906, 0.3150534927845001, -1.2847975492477417, 1.5212502479553223, 6.204181671142578, -0.012642857618629932, 0.7440199851989746, 6.09537410736084, -0.9013458490371704, 4.680523872375488, 0.40709683299064636, -0.3389454782009125, 3.740868091583252, -2.8190290927886963, -0.5836132764816284, -1.855942964553833, 0.2219642847776413, -2.573219060897827, -1.4473718404769897], [-2.566152572631836, -4.109943389892578, -2.3074846267700195, -3.9970273971557617, 0.04317906126379967, 1.402624249458313, -3.1191415786743164, -1.9348795413970947, -1.1069273948669434, 0.9876846671104431, -3.804551124572754, -4.171045303344727, 5.0342183113098145, -5.301811218261719, -0.790144681930542, -0.8361340761184692, 1.7356393337249756, -0.9596117734909058, -5.069713592529297, -0.6964937448501587, -2.909390926361084, 2.250338077545166, -3.2383551597595215, 2.062009334564209, -5.053020477294922, 2.7137398719787598, 1.3970108032226562, -5.681372165679932, -2.6176767349243164, -3.088670253753662, -2.5733819007873535, -3.92238450050354], [-0.637855589389801, 2.7143094539642334, -2.4631943702697754, -0.7587932348251343, 2.553666353225708, 0.20884470641613007, 1.0983353853225708, -3.004568338394165, 0.7006814479827881, 0.7906839847564697, 1.6698787212371826, -1.5898065567016602, 1.0819807052612305, 3.443361282348633, -5.531005859375, -0.24192818999290466, -0.13215607404708862, 0.09037673473358154, 1.4405517578125, 2.388575553894043, -0.30931904911994934, -0.7029222249984741, 1.152376413345337, 2.1614718437194824, 1.3276721239089966, 3.171473979949951, 5.253808975219727, -0.6008760929107666, -8.517034530639648, 1.3743599653244019, -0.2978052496910095, -2.423973798751831], [-2.9005885124206543, 2.9571080207824707, 0.17573334276676178, -4.359992027282715, 3.4065849781036377, -2.192873239517212, -1.446877121925354, -4.642179489135742, 3.8797802925109863, 3.1003339290618896, -0.7197420597076416, 3.6907243728637695, -1.535631537437439, 2.1159040927886963, 0.3477979600429535, -4.162824630737305, -0.2640812397003174, -4.575231075286865, -0.807664155960083, -2.041996479034424, 1.4188754558563232, -1.1766140460968018, 2.377366781234741, 2.8182506561279297, 3.4123029708862305, -1.8367486000061035, -0.7459215521812439, -6.422557353973389, 2.5513482093811035, -1.7570276260375977, -5.501365661621094, 0.8231124877929688], [-0.2985122799873352, 1.2953139543533325, 1.4406590461730957, 0.6285563111305237, -3.6686246395111084, 1.6164425611495972, -7.166211128234863, -1.6425950527191162, 4.994106292724609, 1.515880823135376, -5.231723785400391, 0.8725131750106812, -1.0003368854522705, -0.6724039316177368, 0.5740427374839783, 0.9672000408172607, 5.685955047607422, -0.8590613603591919, 3.077526092529297, -2.294991970062256, 0.8309187889099121, -2.5483503341674805, 1.1976304054260254, -3.746431589126587, 2.181990623474121, -3.422799825668335, -1.1558644771575928, -0.24995505809783936, 1.1738044023513794, -1.803877592086792, 1.2269270420074463, 4.089224815368652], [-4.482406139373779, 0.207439586520195, -4.286166667938232, 0.37953680753707886, 3.445645332336426, 4.2971954345703125, -1.819504737854004, 0.03151059150695801, 3.313380241394043, -1.2140624523162842, 1.7971687316894531, 2.8292148113250732, -3.887378454208374, -0.5488675236701965, -7.388373374938965, 2.6998891830444336, -0.36756640672683716, -6.327743053436279, -2.332066535949707, -1.998366355895996, 3.802886486053467, -2.2487072944641113, 1.0648969411849976, 3.025400161743164, 2.201260566711426, -0.09640192985534668, 1.7439494132995605, -1.1457628011703491, -3.69952392578125, 3.450023651123047, -1.8766846656799316, -2.092061758041382], [-7.798724174499512, 0.858640730381012, 1.7960550785064697, -1.2578675746917725, 1.8803935050964355, -4.911158561706543, 3.360210418701172, 2.613877058029175, 1.0702354907989502, 0.17274251580238342, -1.9211422204971313, -4.9978461265563965, -2.883445978164673, 0.8551038503646851, 4.252148628234863, -2.6426539421081543, -2.850297451019287, 2.4671742916107178, 1.5282108783721924, 5.749897003173828, 0.9843059778213501, -0.12499278783798218, 0.11184965074062347, 0.03380845859646797, -1.1697791814804077, -3.138436794281006, 5.120865821838379, -2.4156086444854736, 3.7974836826324463, 1.2141950130462646, 1.316703200340271, -0.6664999723434448], [1.0420559644699097, 1.2658486366271973, 1.8866095542907715, -8.362512588500977, -5.204604148864746, 3.7388112545013428, -5.886575222015381, -3.2347302436828613, 3.1078925132751465, 2.957850456237793, 3.0556769371032715, -0.1550542712211609, 1.3525559902191162, 4.314272880554199, -0.5074692368507385, -0.6353793144226074, -1.3452448844909668, 0.14867159724235535, 2.4013776779174805, 1.5928635597229004, -0.031079255044460297, -4.579897403717041, -5.378480911254883, -0.8576030731201172, -1.0590566396713257, -1.1346302032470703, 0.02685580775141716, 1.8372280597686768, -1.8768310546875, 7.3651123046875, -0.6038681268692017, 5.169616222381592], [-0.10970734059810638, -1.1224422454833984, -2.713292360305786, -2.5682425498962402, 3.0317273139953613, -2.485783576965332, 3.4837679862976074, 3.046621322631836, 0.04839271679520607, -1.3334797620773315, 1.9601352214813232, 2.0954532623291016, 1.0431262254714966, -2.1368722915649414, -3.8960886001586914, -4.6830878257751465, 3.345766544342041, 1.0023279190063477, -0.9989553093910217, 0.43248671293258667, -2.95820689201355, 1.5836336612701416, -2.0722973346710205, -0.4644351303577423, 3.2764298915863037, -3.603738307952881, 1.1629129648208618, 0.2085418701171875, 1.6843891143798828, -1.3904958963394165, 3.4901628494262695, 2.1003589630126953], [1.7162513732910156, -5.263110637664795, -6.1454243659973145, 0.4457658529281616, -2.282449245452881, -1.5349388122558594, 3.364177703857422, -0.8119674921035767, 0.8466143608093262, 5.453615188598633, 5.0161590576171875, 0.6147958040237427, 2.767313241958618, 1.6801432371139526, -0.5292178988456726, 5.420865535736084, -1.798933744430542, 2.290576934814453, -2.1492769718170166, 1.8539726734161377, -6.447151184082031, -2.592648506164551, -3.3862802982330322, -1.6195063591003418, 2.673935651779175, 0.28375235199928284, 7.324350357055664, 0.922329306602478, 5.887966156005859, -2.3293404579162598, 2.8768973350524902, 1.0461229085922241], [-6.847501754760742, 2.7573440074920654, 3.888885974884033, -4.644316673278809, -6.2793779373168945, 2.7352805137634277, 4.78408670425415, 0.00417357450351119, 0.4699312448501587, -1.1794575452804565, 4.511204719543457, -3.1848154067993164, -2.0059757232666016, -2.5424132347106934, 2.4481306076049805, -1.693818211555481, -5.6073713302612305, -5.899352073669434, -4.446417808532715, 0.6792183518409729, 1.4990562200546265, -2.272021770477295, 2.147921323776245, -2.0464401245117188, -7.326669692993164, 1.7891016006469727, 1.8884108066558838, 1.9689278602600098, 2.429605484008789, 5.717826843261719, -4.6717529296875, -0.6521778106689453], [-1.6420915126800537, 1.1319783926010132, -3.2523388862609863, 2.347033977508545, -1.7039905786514282, 3.8304553031921387, -0.1293286681175232, 0.614364504814148, 0.16112908720970154, -2.8008246421813965, 1.036316156387329, -2.9654488563537598, -1.9104262590408325, 2.868257999420166, -2.2718305587768555, 2.6192872524261475, -2.7720634937286377, -0.38390523195266724, -2.8357715606689453, -3.7454752922058105, 2.4992733001708984, 0.3960447311401367, -3.8145980834960938, 3.6276602745056152, -3.293450355529785, -0.4443414509296417, 1.0998659133911133, -0.7425801753997803, 0.17454884946346283, -3.0957260131835938, -0.9128522872924805, -0.06074019894003868], [-1.6500881910324097, -3.02083683013916, 2.610572338104248, 1.9482643604278564, 2.2940332889556885, -1.9898946285247803, -6.051019191741943, -0.362081915140152, -2.5228841304779053, -1.2768718004226685, -1.721332311630249, -2.163484573364258, 1.3509334325790405, -2.9202613830566406, 7.468022346496582, -9.100997924804688, 2.1424612998962402, -1.3220772743225098, 5.694909572601318, -1.3596123456954956, -5.282676696777344, -0.2605140507221222, -0.11447151750326157, 5.066770553588867, 6.062803268432617, 2.282255172729492, -2.2463719844818115, -1.0003623962402344, 1.7694220542907715, -2.6313652992248535, -0.6579486727714539, 8.543563842773438], [2.10103440284729, 0.8298633098602295, -2.840843677520752, 0.4494125247001648, -2.7875020503997803, -0.2946488559246063, 0.08343277126550674, 1.943054437637329, 2.3110809326171875, 1.0949018001556396, -1.1410127878189087, 0.14415611326694489, 4.725317001342773, -0.19553428888320923, 2.4844810962677, -0.34425240755081177, 3.4678802490234375, 3.1323962211608887, -3.611786127090454, 0.40990185737609863, 0.14194202423095703, 0.25206559896469116, 8.034127235412598, -0.9102878570556641, -2.750734329223633, 2.3496556282043457, 4.767199993133545, -3.7977209091186523, 0.874687135219574, -3.0529136657714844, -0.4572015702724457, 0.7512936592102051], [1.2221275568008423, -4.367308616638184, 8.188743591308594, 3.6107277870178223, -2.8123176097869873, -1.2141199111938477, 2.0078585147857666, 2.3924851417541504, -0.2461610734462738, -2.4287242889404297, -2.0523107051849365, 0.05306107923388481, 1.7476253509521484, -0.7045568227767944, -7.25501823425293, -5.1046833992004395, 1.2077056169509888, 0.3415052890777588, -3.5552706718444824, -0.26430365443229675, 1.23343825340271, 2.616985559463501, 7.400824546813965, -3.374253749847412, 2.105426549911499, 0.7414901256561279, 4.118515491485596, 3.5209150314331055, -4.408116340637207, 1.844545602798462, -2.8969950675964355, 1.204978585243225], [1.6006159782409668, 3.3543806076049805, 0.8934625387191772, 1.1729724407196045, -0.5388165712356567, -1.0907639265060425, 0.8616689443588257, -1.6181459426879883, 0.6908375024795532, -1.56658935546875, -2.934504270553589, -0.2800559997558594, 3.2894747257232666, -2.5742485523223877, -0.6850916743278503, -0.829395055770874, -1.3124072551727295, -0.4328967332839966, -3.9586892127990723, 3.364750862121582, -6.8775434494018555, 0.5956469774246216, 3.5431573390960693, 2.3953752517700195, 3.041078567504883, -2.124774932861328, -2.573270797729492, 2.4244942665100098, -2.4274024963378906, -2.062845230102539, 0.037127867341041565, 0.8143644332885742], [-3.540897846221924, -0.32460349798202515, -1.4022024869918823, 5.537961483001709, -1.6684200763702393, 3.6268310546875, -0.5411379933357239, -2.284740924835205, 2.370482921600342, 2.3254621028900146, 1.3591868877410889, 0.33232349157333374, 0.17927034199237823, -2.955493927001953, 3.620293617248535, -1.2835988998413086, -0.8705587387084961, -0.33817166090011597, -2.18078875541687, 1.6962193250656128, -4.920935153961182, -1.398756504058838, 2.7423131465911865, 0.5829799771308899, 2.2641639709472656, -0.41230228543281555, 2.41609787940979, 0.267051100730896, -0.5857080221176147, 5.42671012878418, 0.3811105489730835, 2.952907085418701], [-6.092750072479248, -1.6524136066436768, 4.034083366394043, 0.26383963227272034, -4.565206050872803, -0.10264592617750168, -2.038998603820801, 0.41391894221305847, -3.3237829208374023, 0.41277647018432617, -1.8176311254501343, 1.1999740600585938, 2.289407968521118, 0.808960497379303, 0.8059363961219788, -2.393599510192871, -0.49711811542510986, 6.377577781677246, 0.16954410076141357, 1.4412685632705688, 2.5020370483398438, -1.1675103902816772, 0.6666451692581177, -2.2317049503326416, -1.8926420211791992, 2.024254083633423, 4.050646781921387, 1.5073304176330566, 1.9119490385055542, -0.46471962332725525, 1.1833933591842651, 1.5778402090072632], [-0.2907370924949646, -3.2276721000671387, -2.6872193813323975, -0.8435051441192627, 2.9712345600128174, 2.8152475357055664, -0.34094929695129395, 3.873824119567871, -0.2686913311481476, 1.7589998245239258, 1.8961546421051025, 3.44526743888855, -0.44221776723861694, -2.071564197540283, 1.0663444995880127, 0.3560004234313965, 5.593707084655762, -5.919460773468018, -2.249424934387207, 0.9080599546432495, 2.158566951751709, 0.35699301958084106, -3.7112224102020264, 1.7592148780822754, -4.306096076965332, 0.03442251309752464, 2.8943629264831543, -0.806383490562439, 2.411731243133545, -1.6179065704345703, 1.0383436679840088, -4.68445348739624], [-2.5373175144195557, 2.5598201751708984, -4.49627685546875, 0.8133472204208374, 4.241185665130615, -1.2259913682937622, 1.3768739700317383, -0.036469802260398865, -4.0543365478515625, 2.199496030807495, -0.4090719223022461, -1.5184202194213867, 2.1606593132019043, 0.46197375655174255, 2.8949167728424072, -3.304494857788086, 3.2465362548828125, -2.289618492126465, 2.386737823486328, 1.1191308498382568, 3.125648021697998, 0.15977196395397186, -4.189886093139648, 7.3079833984375, 2.307208299636841, 5.308269500732422, 4.049339294433594, 1.5328717231750488, 2.3783507347106934, -3.1819167137145996, -3.1250624656677246, -3.192444324493408], [5.510378837585449, 2.3893508911132812, -1.3247261047363281, -4.505648136138916, 1.2829930782318115, 4.176706314086914, 2.153524398803711, -1.6131293773651123, 1.3916306495666504, 2.7186427116394043, 3.5811591148376465, 4.966919422149658, 0.6598601341247559, 1.2757014036178589, 1.2270920276641846, -0.09862305968999863, -1.91232168674469, 1.6795231103897095, 2.9930803775787354, -4.189553737640381, -1.124029517173767, -1.3273577690124512, -0.6081777811050415, 0.5911402702331543, 5.472211837768555, 5.192455291748047, 1.7681697607040405, 1.5094587802886963, 4.752617835998535, 1.5187854766845703, -0.6224017143249512, 1.7405450344085693], [-1.0150645971298218, -0.5810155272483826, 6.778757095336914, 4.605123519897461, 1.585242509841919, -3.6165096759796143, -1.1658002138137817, -0.14147403836250305, -2.441983222961426, -1.2906311750411987, -3.512392044067383, -4.310425281524658, -7.262957572937012, -3.018033742904663, -1.112105131149292, -1.7825528383255005, -0.921515941619873, 0.30603140592575073, 3.6455397605895996, -1.0350416898727417, -2.0314557552337646, 1.9700286388397217, 7.695000171661377, -2.1308436393737793, 5.036050319671631, 3.6363422870635986, -1.6873894929885864, 3.233546733856201, 0.8241700530052185, -5.701997756958008, 0.48820987343788147, 1.1412501335144043], [2.871373414993286, -0.9784700274467468, -0.5538895726203918, 1.5526854991912842, -2.6284170150756836, 1.0617313385009766, 2.593722105026245, -2.8414230346679688, 0.14425155520439148, 1.7984137535095215, -3.8199427127838135, 3.0020384788513184, 5.196432113647461, -3.8176369667053223, 0.46088191866874695, -0.4079357385635376, 4.477173805236816, 0.44181394577026367, 1.2801487445831299, -0.6013001799583435, 0.07161077111959457, -0.1618393063545227, 1.1534569263458252, -0.7294541597366333, -2.987299919128418, -0.26100417971611023, 5.946944236755371, -1.8170864582061768, 2.0655531883239746, 1.6765174865722656, 1.3672995567321777, 1.147413969039917], [5.487674236297607, 3.4608027935028076, -0.9371739625930786, -0.2859356105327606, 4.0823187828063965, -1.3489453792572021, 2.7561702728271484, 2.5628721714019775, 1.6967356204986572, 0.34453362226486206, -8.083937644958496, 3.4038925170898438, 1.1273947954177856, 3.976579189300537, -2.025331735610962, 0.033559273928403854, -2.6148769855499268, 3.5119495391845703, -4.2964301109313965, -1.5115337371826172, 1.9611897468566895, 0.42987149953842163, -1.8329987525939941, 0.4129156172275543, -0.3810955286026001, -3.3025078773498535, -1.275446891784668, 8.074295043945312, -0.9222573637962341, 2.432072639465332, 1.7728238105773926, 0.8416589498519897], [2.2105884552001953, -3.532622814178467, -0.9477115869522095, 0.4862211346626282, -0.309945285320282, 5.678112506866455, -1.710511565208435, -3.94990873336792, -1.927849531173706, 3.167227268218994, 2.9580206871032715, -2.834829568862915, -0.7143917679786682, -3.8807079792022705, -5.067803382873535, -2.5773701667785645, -0.19341957569122314, -3.834568500518799, 3.0120797157287598, -2.7729408740997314, 3.379169464111328, 2.378990411758423, -4.957892417907715, 1.9797402620315552, -3.2600934505462646, 0.6064102649688721, -0.5055031180381775, -1.77455472946167, 5.296154022216797, -2.7343380451202393, 3.1730289459228516, 3.494229793548584], [-3.485250473022461, -3.448045253753662, -2.091629981994629, 2.4815213680267334, -1.6262394189834595, 2.750619411468506, 2.5018560886383057, 2.011692762374878, 5.2808732986450195, -0.28240400552749634, -6.659914016723633, 1.019561767578125, -1.2949373722076416, 1.7759366035461426, 1.1272776126861572, -2.3475959300994873, -7.53004264831543, 1.822224497795105, -1.8638412952423096, 0.9763548374176025, -0.17692215740680695, 1.8827292919158936, -2.477999687194824, 0.31899452209472656, 2.8937110900878906, 3.149364471435547, 1.1678322553634644, -2.098193645477295, 4.103621959686279, 3.1385602951049805, 1.8497672080993652, -6.011244773864746], [-1.326778769493103, -1.742383360862732, -3.573431968688965, 3.9766740798950195, -1.367152214050293, -6.08358907699585, 3.4536445140838623, -3.3697667121887207, 2.6856064796447754, -1.1541858911514282, -4.832317352294922, 5.182607650756836, 0.6249071955680847, -1.3241041898727417, -0.6318895816802979, -2.914780855178833, -0.2781720459461212, 5.051122665405273, -0.5034170746803284, -0.07333362102508545, 2.867861747741699, 0.4161252975463867, 2.665235996246338, -2.7919068336486816, 3.0268983840942383, -8.222354888916016, -3.9169063568115234, 4.954987525939941, -0.7195985913276672, -4.739194869995117, -2.143800735473633, 5.940229415893555], [1.7012475728988647, 0.2247173935174942, -2.701962947845459, 5.928129196166992, -5.354870796203613, -5.092952728271484, 3.1379213333129883, 1.3066173791885376, 2.2230679988861084, -0.13660189509391785, -0.2837597727775574, -2.289296865463257, -2.5947425365448, -3.2457408905029297, 1.883371353149414, 0.48610925674438477, 1.934985637664795, 1.6417675018310547, 1.081082820892334, -2.5303375720977783, -0.20628806948661804, 5.940520286560059, -2.1350417137145996, 4.428842544555664, -5.211132049560547, -5.503695964813232, -0.528899073600769, -0.7092033624649048, -4.259849548339844, -1.7926356792449951, 2.904794216156006, -3.0750670433044434], [-3.1219067573547363, -2.1094486713409424, -0.4962519407272339, -0.013782517053186893, 1.609574794769287, -2.9174792766571045, 2.8575448989868164, -3.669403553009033, 0.25223299860954285, 6.390542030334473, 2.397108793258667, 1.516194462776184, 0.5501542091369629, 2.8201680183410645, -0.8810844421386719, 1.4955865144729614, -1.8345637321472168, -1.635980486869812, -0.8205734491348267, 0.2858094274997711, -2.091991901397705, 0.08238371461629868, -3.2681102752685547, 2.1951065063476562, 3.0056142807006836, 3.426823854446411, -0.5066164135932922, -1.0853341817855835, 2.345322608947754, -0.4833438992500305, -2.9257912635803223, -0.4383874833583832], [3.946831703186035, -1.5664340257644653, 3.1717896461486816, -0.7615118026733398, 0.5573905110359192, 1.92659330368042, -1.8532507419586182, 3.1383512020111084, -4.342597007751465, -3.8745553493499756, 2.6250436305999756, -0.7113502025604248, 2.4848251342773438, -1.6290531158447266, 0.9741881489753723, 0.9974411129951477, 3.1086266040802, -0.11923888325691223, -1.2222602367401123, 1.6584503650665283, -0.5816507339477539, -0.6673516035079956, 2.3795363903045654, -0.9657338261604309, -0.48282957077026367, 1.7828059196472168, 1.9298958778381348, -0.4281495213508606, -0.3442135751247406, -5.3348259925842285, -1.4759211540222168, 0.46178120374679565], [-1.2510185241699219, 1.2534818649291992, 3.9484941959381104, -1.0567340850830078, -2.3476948738098145, 1.1390128135681152, 1.4316705465316772, 0.27315273880958557, -3.8977737426757812, 1.3212162256240845, 2.896566152572632, 3.839672327041626, -3.9887266159057617, -2.9768247604370117, -2.15560245513916, -0.22866350412368774, 4.641917705535889, -1.018226146697998, 0.24827221035957336, -2.3912436962127686, -3.4693117141723633, 0.049837276339530945, 1.239802598953247, -7.879836082458496, -1.3297109603881836, 4.808180332183838, -1.819117784500122, -4.448209762573242, 0.3578433394432068, 0.556039035320282, 0.7840105891227722, -2.253115653991699], [-2.8880317211151123, 1.4651120901107788, -6.932130813598633, -2.335756778717041, -0.14426732063293457, 6.52839469909668, 2.0312576293945312, -6.183128356933594, -3.905611753463745, -3.917508125305176, -5.940481185913086, 4.917728424072266, -1.9042603969573975, 0.8303811550140381, -4.6205644607543945, -2.001305103302002, -4.8208770751953125, 0.1774502694606781, 2.9251656532287598, 2.3996076583862305, -3.0090930461883545, -2.8293302059173584, 3.9123072624206543, 1.83022141456604, 0.05936902016401291, -3.4031662940979004, -2.8319480419158936, 0.593471884727478, -1.998490810394287, -0.8537688255310059, 2.9727530479431152, -7.183529376983643], [-0.9754606485366821, -0.1806565523147583, -2.9536380767822266, -4.165924072265625, 4.451637268066406, 3.209412097930908, -1.9367022514343262, -4.364931583404541, 1.6908448934555054, -3.7399868965148926, -2.3693888187408447, -2.814274311065674, 3.2041492462158203, 3.122037887573242, 2.3985190391540527, -0.6500007510185242, -5.107419967651367, -0.6721276044845581, -3.985391139984131, 0.6556599736213684, -2.165811061859131, 0.310812771320343, 1.8986932039260864, 6.806801795959473, 1.448368787765503, -1.2974164485931396, 4.384037971496582, 3.6629111766815186, 0.38658401370048523, 0.13622814416885376, 2.6014208793640137, 4.856595039367676], [-4.815182685852051, -6.246281623840332, -0.16962499916553497, -1.4707144498825073, 1.2622511386871338, 1.648097038269043, 2.1818137168884277, 2.932154417037964, 2.067836284637451, 4.159701347351074, -3.4607748985290527, 0.3602885603904724, -2.240112066268921, -1.1826759576797485, 0.994817852973938, -1.1327146291732788, -0.35796505212783813, 1.2881460189819336, 2.594940662384033, 5.621476173400879, -0.2064652144908905, 3.274712562561035, -2.98846173286438, 1.8608375787734985, 5.017205238342285, -4.341773509979248, -1.2412234544754028, 4.763280391693115, -5.972380638122559, 0.1078091710805893, -2.834777593612671, 3.0214014053344727], [1.2694276571273804, 7.824086666107178, 0.8788695335388184, 0.4753059446811676, -0.435319185256958, 1.8814027309417725, -4.238008499145508, -6.368699073791504, 4.833003997802734, 1.568450927734375, -4.580552577972412, 0.747774600982666, -0.10946495831012726, 1.4099676609039307, -2.017228603363037, 0.5271368026733398, -4.026345252990723, 5.410444259643555, 1.88420569896698, 3.053084373474121, -2.846680164337158, -1.6901066303253174, -2.8452937602996826, -0.7644673585891724, 1.9558106660842896, 1.2999452352523804, -5.357605457305908, -0.5282655954360962, 0.27715542912483215, 1.191558599472046, 0.7455774545669556, 0.8226615786552429], [-2.1202120780944824, 0.3574819266796112, -2.063403367996216, -0.8279476165771484, -2.6092865467071533, 3.410707473754883, 1.6928837299346924, -3.7318930625915527, 0.31944623589515686, -3.192889451980591, -3.578516960144043, -0.8071644306182861, 0.5739394426345825, 4.34268856048584, -3.17655611038208, 2.8342926502227783, -1.7153217792510986, 4.373711109161377, -1.7822918891906738, 3.9831953048706055, -2.8335812091827393, 0.9778520464897156, -0.6455792188644409, -2.243030309677124, 0.8819900751113892, -1.333483099937439, -1.3039436340332031, -1.2740198373794556, 4.341907501220703, -2.7963778972625732, 1.97633957862854, 2.5839967727661133], [2.7256741523742676, 0.42913341522216797, 1.0545469522476196, -2.912261486053467, -6.632840156555176, -2.8825063705444336, -3.1485238075256348, -3.2374234199523926, 0.32810038328170776, 5.371277809143066, 4.644773006439209, 0.40269994735717773, 6.462912559509277, -1.2587001323699951, -6.500278472900391, -8.025065422058105, -2.723376989364624, -0.6667037606239319, -2.0727577209472656, -5.686492919921875, -1.1003621816635132, 2.7958385944366455, -2.027947425842285, -1.0256905555725098, 0.749302864074707, -3.4907102584838867, 3.3036954402923584, 5.136983871459961, 2.684370756149292, -0.7664221525192261, 4.050065994262695, 2.565185070037842], [0.055020637810230255, 6.06400203704834, 4.128066062927246, -2.828594446182251, 5.147036552429199, -0.6016783118247986, -1.71907377243042, 1.6999340057373047, 3.9483096599578857, -0.9313740134239197, -5.3160905838012695, -2.263944149017334, 0.9389156103134155, -3.5955681800842285, -3.8385987281799316, -4.1495890617370605, -3.10678768157959, -6.034989356994629, 3.1366090774536133, 0.7151508927345276, 1.2574774026870728, -2.812023401260376, 2.835449457168579, -4.426772594451904, -1.4765375852584839, 2.8955633640289307, 3.368746757507324, -1.172971248626709, -4.525631427764893, 4.92045783996582, 6.089348316192627, -1.4487066268920898], [-2.037506341934204, -3.92861270904541, 0.9315307140350342, 1.5741920471191406, 3.451033592224121, -4.333762168884277, 2.8365931510925293, -2.052851915359497, -0.160905659198761, 2.4538040161132812, 3.041837692260742, -1.0102165937423706, 4.17855167388916, -1.5000193119049072, 2.085810661315918, 2.9914417266845703, -4.596220016479492, 8.37983512878418, -4.026167392730713, 6.364830493927002, -3.9385623931884766, 2.9599711894989014, -1.0285627841949463, 0.34665319323539734, 0.6588359475135803, -1.637434720993042, 0.1513652205467224, -2.987203598022461, 0.9061000943183899, -7.076887130737305, -8.536273002624512, -2.3724098205566406], [2.096616744995117, -3.300873041152954, 2.4666340351104736, 4.464118003845215, 1.5173392295837402, 1.8790737390518188, 2.897587776184082, 1.8156899213790894, 1.23508620262146, -3.5430164337158203, 0.29507219791412354, 1.3602864742279053, -2.356353282928467, -2.574313163757324, -3.9032628536224365, 6.973536014556885, 1.7839276790618896, 0.9397653937339783, -3.285956382751465, -1.0956854820251465, -3.5896401405334473, 2.3662233352661133, -1.800284743309021, 2.5642342567443848, 1.9791219234466553, -0.588620662689209, -1.6468702554702759, 2.3475019931793213, -4.586616039276123, 0.0240086168050766, 0.4806710481643677, -1.0501055717468262], [3.4991297721862793, 0.8906859159469604, 1.0333127975463867, 1.2915678024291992, 1.8555033206939697, -4.170872688293457, -0.23390565812587738, -0.2637418508529663, 0.1381371021270752, 3.541861057281494, 0.5570528507232666, -2.0157458782196045, -3.8336684703826904, 0.2952817976474762, -3.525446891784668, 4.62993049621582, -0.5197603702545166, -4.2312774658203125, 0.1655043363571167, 3.8250010013580322, -0.9776797294616699, -3.0794613361358643, -4.982733726501465, 1.807783603668213, 1.3687152862548828, 2.645906448364258, -5.15576696395874, -3.242868423461914, 3.3259730339050293, 2.379725694656372, 2.419090986251831, -0.5825595855712891], [-5.315284729003906, 1.7697019577026367, 1.8360376358032227, 7.052582740783691, -1.1671559810638428, -0.37114259600639343, -3.6657063961029053, -0.3989214599132538, 0.10997726023197174, -3.219226837158203, 1.2517337799072266, 1.3796910047531128, -0.27113741636276245, 1.482675552368164, -4.772716999053955, -1.562829852104187, -0.07521668821573257, -4.800173282623291, -3.4078288078308105, 1.4922598600387573, -2.088097333908081, 1.305826187133789, 1.0171927213668823, 3.6268019676208496, 2.550630569458008, -5.3109846115112305, -0.18368622660636902, 1.5588438510894775, -1.7943724393844604, 2.0227155685424805, -0.997362494468689, -5.839046955108643], [0.7020887732505798, 1.1687345504760742, -1.19060218334198, 1.9341893196105957, -4.1363043785095215, 2.3140206336975098, -0.6815482974052429, -3.1722679138183594, -1.8157234191894531, 2.108370304107666, 3.239431381225586, -2.264389991760254, -2.443419933319092, 2.105774402618408, -1.3972246646881104, 0.05373614653944969, -0.0700620487332344, 2.2267816066741943, -3.5696561336517334, 0.6589457988739014, 3.270998239517212, -3.05098295211792, 1.1701176166534424, 1.9496006965637207, 1.781083583831787, 0.09585555642843246, 2.235203266143799, -0.6798688769340515, -3.0952601432800293, 1.417871117591858, -4.8114776611328125, -1.573021411895752], [-1.666237235069275, 0.9401931762695312, 1.1864924430847168, 5.340425491333008, 4.660513877868652, 1.2887544631958008, 2.3411343097686768, 1.0354052782058716, -1.6610221862792969, 6.69903564453125, 0.12345334887504578, 0.4560781717300415, -0.723686695098877, 0.7018051147460938, -1.0714939832687378, -0.10368971526622772, 0.368825227022171, 5.263022422790527, 0.09230154752731323, -0.26194053888320923, 5.64209508895874, 6.154626846313477, -0.9965147972106934, 0.4055721163749695, -1.686856985092163, 3.2173547744750977, -0.2746403217315674, -4.492327690124512, 2.3654401302337646, -1.7381224632263184, 1.3211687803268433, 0.4682304859161377], [-1.3386406898498535, -1.468720555305481, -4.181264400482178, 4.451776504516602, -1.893101692199707, 2.1375629901885986, -3.552457332611084, -4.782870769500732, -1.0193960666656494, -0.6263799071311951, 2.91188383102417, 3.0148820877075195, -1.387689232826233, 3.9520201683044434, 6.311634063720703, 2.8834943771362305, 5.844718933105469, -1.5816092491149902, 5.212322235107422, 2.505772829055786, -9.76888656616211, -1.6882591247558594, 2.4020752906799316, 2.5356521606445312, -2.756641387939453, -0.2539065480232239, 0.7031350135803223, -4.679224014282227, -6.231492519378662, -0.7192496061325073, -0.3140246570110321, -0.17670637369155884], [-1.5884804725646973, 0.9948869943618774, 2.7962355613708496, 3.603940010070801, 2.89131498336792, -5.152840614318848, -1.1842677593231201, -6.62992000579834, -1.4042625427246094, 0.6990282535552979, -0.04553776979446411, 1.0523667335510254, -3.041684627532959, -0.5702548027038574, -1.107985258102417, -2.469935894012451, 0.013099577277898788, -1.9993715286254883, -5.795534133911133, -5.314638614654541, 0.1449112892150879, -1.661672592163086, 0.42414265871047974, -4.430839538574219, 0.4633892774581909, -1.3006370067596436, 0.7343118190765381, 0.7052935361862183, 0.1739995926618576, -2.3065073490142822, -3.177640199661255, 2.0901222229003906], [1.4380840063095093, 5.085842132568359, -0.4061543047428131, 2.0342977046966553, -3.229194164276123, -4.032907009124756, 5.556903839111328, 4.306837558746338, 2.0049118995666504, -2.3498523235321045, -0.1115003451704979, 1.079620122909546, -1.74293851852417, 1.18513822555542, 4.772881031036377, -0.5614949464797974, 5.633890151977539, -3.125694751739502, -3.9144654273986816, 1.4481836557388306, -0.3454586863517761, -5.343136787414551, -1.6078903675079346, 3.826550245285034, 2.4692721366882324, -1.029414415359497, 4.001867771148682, -0.6261193752288818, 5.955896854400635, -3.207904815673828, 2.001161575317383, -2.7900242805480957], [-0.8995298147201538, 0.7547893524169922, -1.4509915113449097, -2.355722188949585, 1.7096166610717773, 0.1553080528974533, -4.2971296310424805, -2.559404134750366, 5.788492202758789, -3.5922653675079346, 2.071035385131836, -5.7379631996154785, -1.1296031475067139, 4.647610664367676, -3.9619710445404053, -0.10805370658636093, 0.5251511335372925, -0.3735969066619873, -2.8343639373779297, 2.4540855884552, -5.542485237121582, -3.727570056915283, -5.179619312286377, -0.46183258295059204, 5.225523471832275, 0.17459334433078766, 1.761307954788208, -0.36964476108551025, 1.0771031379699707, 3.402261972427368, -7.16249942779541, -2.226942777633667], [0.5600201487541199, 1.635585069656372, -2.1508593559265137, -3.218648672103882, -2.761794328689575, 0.16870561242103577, 0.9612112045288086, 2.7174501419067383, 1.8602087497711182, 0.4622418284416199, -0.3537673354148865, 0.6202656030654907, -3.937069892883301, -0.433408260345459, -1.2579962015151978, -5.4185028076171875, 0.6708532571792603, 3.2975900173187256, -1.618901014328003, -1.564009428024292, 6.865419387817383, -3.229952573776245, -3.1751766204833984, -2.818779230117798, 0.8977677226066589, 3.7270803451538086, -1.042802333831787, 3.0574750900268555, -9.115689277648926, -0.9554091095924377, -3.038267135620117, -3.6130504608154297], [-2.0536458492279053, -3.0488195419311523, -4.310558319091797, 0.21620772778987885, -1.3123607635498047, 8.04892349243164, 1.5880622863769531, -3.295621871948242, 2.0431745052337646, 2.846637010574341, 2.1315698623657227, 2.6209964752197266, -0.9272609949111938, -2.3762290477752686, -1.1615346670150757, 5.1560187339782715, 2.1440048217773438, 2.239030361175537, -2.115226984024048, -4.995373725891113, 1.3080036640167236, 2.7312254905700684, -0.946672797203064, -2.590972661972046, 0.30984485149383545, -3.439894676208496, -1.8836127519607544, 1.5119366645812988, 2.021707534790039, 1.0902880430221558, -1.3916783332824707, 0.13641947507858276], [-3.561488151550293, 0.3335581123828888, -1.634049415588379, -3.7998290061950684, -3.5462350845336914, 0.30261802673339844, -2.817620038986206, -1.9206401109695435, -0.7880855798721313, 6.513586521148682, -0.6256570219993591, 0.8811162710189819, 4.631234169006348, 0.31614601612091064, 1.3245232105255127, -6.921529769897461, -1.5010459423065186, 3.5150628089904785, 6.813323974609375, 2.5985519886016846, -2.3859469890594482, -4.824532985687256, -0.25903984904289246, -0.024991683661937714, -1.6552517414093018, 1.829422950744629, 0.027423614636063576, 4.8718061447143555, -2.789182662963867, 4.134057998657227, -2.011983871459961, -3.260444164276123], [-0.9672673344612122, 3.3469200134277344, -3.2364253997802734, -1.030510425567627, -0.4960998296737671, -1.1446841955184937, -0.3012397289276123, 3.019148111343384, -2.657498836517334, -0.7806262969970703, 5.400996208190918, 2.484405040740967, -1.6058857440948486, 0.6966395378112793, -2.0860767364501953, 0.7948175668716431, 4.771714687347412, -2.4008686542510986, 4.975434303283691, -5.401052474975586, 3.924467086791992, -1.8145830631256104, -2.160818576812744, 1.8040820360183716, -7.695988655090332, -5.1847357749938965, 6.687127113342285, -3.186795234680176, -9.287290573120117, 4.059589385986328, -0.35556137561798096, 2.6351077556610107], [-1.5211734771728516, 2.4294652938842773, -3.706315755844116, -1.3580807447433472, -1.457977056503296, -0.9578115940093994, 4.266292572021484, 1.774505853652954, 7.613840103149414, -1.107257604598999, 2.1520965099334717, 1.0621745586395264, -4.4937262535095215, -0.6166282892227173, 0.37629270553588867, 0.49779072403907776, 1.929840087890625, -1.8618910312652588, 0.15048767626285553, -1.9261053800582886, -1.4599480628967285, 0.9594717025756836, -1.180461049079895, -7.448665618896484, -3.163821220397949, 1.8046629428863525, 3.7038052082061768, -1.1988577842712402, -1.4196269512176514, -7.3729705810546875, -9.404890060424805, 1.5090004205703735], [0.725157618522644, -2.918912410736084, 1.7008026838302612, 3.272265672683716, -0.40590333938598633, -1.1310476064682007, -1.9769386053085327, 0.5564705729484558, 0.9234135150909424, 1.957334280014038, -3.0112929344177246, 2.812241315841675, 3.0002832412719727, 7.428668975830078, 3.0078420639038086, -3.3235366344451904, 1.6430878639221191, -3.260735511779785, 5.557911396026611, 1.3876674175262451, -0.06202412396669388, -6.123320579528809, 0.09813190251588821, -1.8398208618164062, 1.8367021083831787, 4.298736572265625, -1.6308515071868896, 4.714710235595703, -0.41929930448532104, -4.753244876861572, -1.0330017805099487, 1.8031134605407715], [-0.5164211988449097, -5.956484794616699, 0.8845609426498413, 2.5401034355163574, 1.6047029495239258, 0.9116967916488647, 3.426809787750244, 3.008425235748291, -0.06308764219284058, -0.6500407457351685, 5.436263561248779, -2.662196159362793, 0.5411344766616821, 2.7009034156799316, -0.6176321506500244, 2.9458744525909424, -0.09092310070991516, 0.21809121966362, -1.2837263345718384, 4.669785499572754, -0.8039804697036743, -2.042626142501831, -4.035757541656494, -2.2747082710266113, 0.6099419593811035, 6.222426414489746, -1.6327550411224365, -3.2015540599823, -2.9038009643554688, -2.618774175643921, -6.483207702636719, 3.323190212249756], [-8.34762191772461, -0.6900559067726135, 5.8861799240112305, -3.373936414718628, 1.1260666847229004, 1.9720134735107422, -2.9879841804504395, 0.6463747620582581, 5.888208866119385, -3.447200298309326, -0.09033097326755524, 1.027405023574829, -0.17531031370162964, -1.2708477973937988, 2.6552557945251465, 0.9900792837142944, -5.023350715637207, 4.237396240234375, -1.8113518953323364, 0.11252649128437042, -0.4726828336715698, -3.6879687309265137, -5.44616174697876, -2.0472891330718994, 3.900355815887451, -1.9310256242752075, 2.288806676864624, 3.246722459793091, 2.492109775543213, -2.080385208129883, -0.7416573166847229, 0.3134913742542267], [-3.6275906562805176, -1.784149169921875, -0.8921734094619751, -2.0787298679351807, -0.857947826385498, -0.02441541478037834, 2.246391534805298, -3.4753541946411133, -0.7383450865745544, 5.9473066329956055, -1.1540992259979248, 3.4930100440979004, -1.0976943969726562, -0.37107986211776733, -1.8311930894851685, 0.6157146692276001, -0.1552182286977768, -4.528010368347168, 4.19663143157959, 0.8911510705947876, -1.7926429510116577, -3.5661821365356445, -3.1257870197296143, -3.5013771057128906, 1.9099841117858887, 0.49490708112716675, -3.9625253677368164, -3.8206257820129395, 1.1904408931732178, -1.4175100326538086, -1.6752530336380005, -1.3174643516540527], [-0.05070815235376358, -2.2018959522247314, 0.2757136821746826, -10.622332572937012, -2.1472859382629395, -3.223505973815918, -0.1769391894340515, -3.7129931449890137, 1.4255023002624512, -1.5712192058563232, -4.290954113006592, -1.2355682849884033, -5.759806156158447, 4.261322498321533, -0.9071298837661743, -1.2834051847457886, -0.14903150498867035, -2.283752679824829, 4.837238311767578, 0.2893029749393463, -1.5315749645233154, 2.701014995574951, -1.322980284690857, 1.3461588621139526, 1.5853021144866943, 1.187612533569336, 8.324974060058594, 2.948413133621216, 0.5677502155303955, 1.3517537117004395, -3.9460225105285645, 1.6122794151306152], [0.8073680400848389, -4.130064964294434, -3.0193748474121094, 0.6234420537948608, -2.4370481967926025, -2.2895126342773438, -1.8991219997406006, -2.869640350341797, -0.14065101742744446, 2.221898317337036, -0.18357455730438232, -0.970465898513794, -4.2959208488464355, 1.5420622825622559, -0.38074594736099243, 0.7947322130203247, -1.177261233329773, 1.9891608953475952, -5.811008453369141, 0.891000509262085, -3.274592876434326, 1.443943977355957, 1.9815927743911743, 3.026784896850586, 2.1404457092285156, -1.332064151763916, 1.8308727741241455, -2.733349323272705, 1.2169766426086426, 1.0522490739822388, -3.4509615898132324, 0.5048713684082031], [3.9981117248535156, -0.6697492599487305, -1.1444971561431885, 1.0052025318145752, 1.9386645555496216, 5.760891914367676, -5.132078647613525, -1.4852046966552734, -2.684831142425537, -1.8963534832000732, -4.014525413513184, -0.642518937587738, 3.831293821334839, -3.16202449798584, -1.1654452085494995, -0.9265705347061157, -2.187695026397705, 5.376506805419922, 3.6642093658447266, 1.31691575050354, -1.38643479347229, 2.688257932662964, -0.8538491725921631, -0.2830846607685089, -4.409258842468262, 2.6261508464813232, 3.0335168838500977, -0.9298422336578369, 1.1117948293685913, -1.6884663105010986, 0.6576305627822876, -4.129956245422363], [-1.0142064094543457, -2.393104076385498, 0.008252890780568123, 6.646003246307373, 0.03809951990842819, 1.2742942571640015, 0.28612977266311646, -1.0217769145965576, -1.6295573711395264, -0.17071577906608582, 2.771314859390259, -2.4224276542663574, -1.4777084589004517, 0.33334654569625854, 0.6026997566223145, 3.5682854652404785, 5.20562744140625, -0.7385093569755554, -2.169975519180298, -0.38362014293670654, 7.291224479675293, -0.05317136272788048, 3.4012091159820557, 5.888044834136963, 2.7800722122192383, 5.759666442871094, 2.750365972518921, 3.6069138050079346, 0.026670794934034348, 1.7569215297698975, -5.2354583740234375, -5.2919158935546875]]]], "dtype": "float32"}, {"__nd__": [[[[4.082848072052002, 1.1817128658294678, 3.2721643447875977, -1.6719063520431519, -0.006638919934630394, 2.2908875942230225, 3.4376513957977295, -6.4964752197265625, -1.1987287998199463, 3.075051784515381, 2.425241470336914, -1.421149492263794, 3.6891653537750244, 1.4433273077011108, -3.523961067199707, 1.222242832183838, 2.158247470855713, 1.2440167665481567, -3.380216360092163, 2.484689950942993, -1.5301721096038818, -3.427309036254883, 2.0105478763580322, -4.231311798095703, 0.4233185648918152, 1.029991865158081, 2.9248785972595215, 3.8091859817504883, -3.1342928409576416, 0.9204282760620117, -2.1227526664733887, 5.130087375640869], [-5.899553298950195, -0.29541972279548645, 1.1030175685882568, 1.0750324726104736, 3.4451301097869873, -2.505037784576416, 1.3074604272842407, -0.009485220536589622, -3.0675320625305176, 1.0159354209899902, 1.5527689456939697, -0.8760500550270081, -3.1140060424804688, -1.93912935256958, -0.7253614664077759, -4.747500419616699, 4.974807262420654, 1.5780832767486572, -3.0080959796905518, 0.39172813296318054, -0.12750476598739624, 7.134016990661621, -0.4661855697631836, 2.600480556488037, -2.5842247009277344, 0.41162872314453125, -0.9836828112602234, -5.043288230895996, 0.11562155932188034, 2.6757640838623047, -8.765251159667969, 6.776937961578369], [-0.4554542899131775, -0.5010578632354736, -2.4189724922180176, -1.396175742149353, -1.2178599834442139, -2.368320941925049, -2.3136444091796875, 0.6667144298553467, 1.5638620853424072, 7.911018371582031, -1.7016777992248535, 0.6694830656051636, 0.8191797733306885, 1.081484079360962, -3.6998987197875977, -0.9223359823226929, 1.6527070999145508, -1.350396990776062, 3.841913938522339, -3.0007174015045166, 2.510230541229248, 0.4531424641609192, 1.1254079341888428, -4.124333381652832, -2.5329575538635254, 3.4418225288391113, 1.4215540885925293, 4.8418169021606445, -0.5189368724822998, -6.885768890380859, -2.771240234375, -0.14676432311534882], [-1.7284691333770752, -0.8935035467147827, -2.1106133460998535, 3.1495203971862793, 0.3734591007232666, 0.5827778577804565, 5.04139518737793, -3.7557413578033447, -5.76469612121582, -3.5251829624176025, -1.5768574476242065, 1.7091472148895264, 3.311849594116211, -1.411470651626587, 1.6688423156738281, -3.6439056396484375, 3.42250919342041, -4.630035400390625, -6.314593315124512, -2.4348931312561035, 0.3500952422618866, 3.770191192626953, -1.2139209508895874, -7.358982086181641, -0.31301379203796387, -4.031119346618652, -1.8996706008911133, 0.6097521781921387, 1.4031362533569336, 1.419450283050537, -3.863128662109375, -3.8133840560913086], [2.8081531524658203, -0.9047044515609741, -2.369861602783203, 0.48944228887557983, -2.971179485321045, 5.847516059875488, 0.048619676381349564, -4.661726474761963, -3.8636107444763184, -3.4201576709747314, -1.5790584087371826, -6.984667778015137, 2.6406288146972656, -5.688961982727051, -4.510456562042236, -0.44411373138427734, 0.31731024384498596, 0.9416821002960205, -1.2521228790283203, 8.946266174316406, 0.3649599254131317, -1.9150855541229248, 1.2667624950408936, 0.014453164301812649, 6.855908393859863, -3.632120132446289, 1.495254397392273, -0.4545266628265381, 0.3410925567150116, 1.2139127254486084, -3.477358341217041, -0.43585944175720215], [2.9552948474884033, -0.3333064913749695, -2.391248941421509, -2.2380547523498535, 1.4685924053192139, 0.1654028594493866, -0.5196196436882019, -2.728666067123413, -0.0585038885474205, -2.6328132152557373, 2.605825424194336, -0.513462245464325, 3.007107734680176, 0.6563299894332886, -4.474534511566162, 0.7394230365753174, 2.342832565307617, 1.2186808586120605, -6.5200042724609375, -1.9880787134170532, -1.0632363557815552, 3.8872053623199463, 3.1483006477355957, 0.20304018259048462, 0.5561517477035522, 1.0750205516815186, -1.1062594652175903, 3.579033136367798, 0.8789354562759399, -4.660683631896973, 2.893207550048828, 3.6507067680358887], [-2.444990873336792, -1.1666622161865234, 0.05701649934053421, -4.3465728759765625, -1.0936861038208008, 5.16391658782959, 2.792145013809204, 1.4560554027557373, 2.4410617351531982, -0.016943147405982018, 1.785083532333374, -0.0626530796289444, -6.2820024490356445, 2.6745333671569824, -0.3685725927352905, 3.9098517894744873, 1.7595252990722656, -0.5027791857719421, -1.8285300731658936, 5.612358570098877, -0.8893965482711792, -1.6756083965301514, 4.597065448760986, 0.23649980127811432, -1.6066285371780396, -5.845714569091797, -3.159444808959961, -1.1802711486816406, -1.910796880722046, -2.71785569190979, -1.7029062509536743, 0.9818091988563538], [1.989456295967102, -1.579932689666748, -2.121908664703369, 1.0045487880706787, 3.2633509635925293, -1.0618078708648682, 1.2187769412994385, -3.2369155883789062, -1.2321891784667969, -1.7385778427124023, -0.014345110394060612, -3.2163233757019043, 0.41599515080451965, 1.1347427368164062, 1.4618455171585083, -2.4963228702545166, -0.36412477493286133, 1.8431141376495361, 3.6873459815979004, 0.7948591709136963, -2.614405870437622, 1.2715399265289307, -2.907898187637329, -1.898827314376831, -4.367501258850098, 0.6260675191879272, 6.589442253112793, 2.5606377124786377, 0.2419809103012085, 0.0449649952352047, 1.2983016967773438, 3.602123737335205], [-2.247610569000244, 0.5807459354400635, 1.8442423343658447, -1.7699954509735107, 4.623912811279297, -2.91800856590271, 5.269430160522461, -3.829472541809082, 0.188167542219162, 4.160799980163574, -4.538819313049316, 3.443817138671875, -0.22262704372406006, -0.14165933430194855, -1.5267562866210938, -3.391780376434326, 5.591779708862305, -2.920384407043457, 4.962594509124756, 3.714010715484619, -0.9964883923530579, 1.226159930229187, -0.8482116460800171, -4.000411510467529, 0.4303428530693054, 3.269033432006836, -0.46570926904678345, -2.410581111907959, 2.999854564666748, 0.902051568031311, 3.1125688552856445, -1.3913077116012573], [-1.411505937576294, -1.2610433101654053, -1.2752180099487305, -7.761661529541016, -2.362136125564575, -0.7966669797897339, 4.968105792999268, 2.5426173210144043, -0.532450258731842, 3.464444875717163, -5.838378429412842, 5.780442237854004, -0.28374814987182617, -3.1521785259246826, -5.672266006469727, 7.213381767272949, 1.004137635231018, 1.553189992904663, -3.6436619758605957, -4.7758331298828125, -6.276598930358887, -0.7965421676635742, 0.26903611421585083, 4.764512062072754, 2.5654187202453613, -0.5889197587966919, -0.29273343086242676, 2.5936279296875, 5.227899551391602, 4.152378559112549, 3.8416953086853027, -5.473745346069336], [2.8115105628967285, -9.378477096557617, -1.0090014934539795, -4.155331611633301, 0.0963999330997467, -2.2132954597473145, 6.1768903732299805, 3.610325574874878, 6.73038911819458, -3.754359245300293, -0.4173857867717743, 4.675694942474365, 0.738088846206665, -1.576064944267273, 0.12163804471492767, -3.310337543487549, -2.3526430130004883, 1.7901217937469482, 0.3979196548461914, 2.4507288932800293, -0.9002321362495422, 0.15049679577350616, -0.2479366660118103, -0.10209151357412338, -4.053435325622559, -3.637802839279175, 10.05878734588623, 2.9751172065734863, -4.548534393310547, -1.4426019191741943, 1.899566650390625, 0.1364401876926422], [-1.9313626289367676, 1.338701844215393, 0.8046258687973022, -1.1927142143249512, -1.389948844909668, 0.4588168263435364, -2.704448699951172, 0.2573004364967346, 0.7151690721511841, -1.0582656860351562, 8.314395904541016, 1.9526996612548828, -2.0306785106658936, 3.643253803253174, 1.0089877843856812, 5.134908676147461, 2.9861481189727783, -1.3912179470062256, 0.07818663120269775, 4.934138298034668, 1.0200719833374023, -0.579127848148346, -1.3238310813903809, 9.106996536254883, 5.6664838790893555, -2.29136323928833, 1.571845531463623, 1.782156229019165, 2.9911458492279053, 0.5352141857147217, -1.5684210062026978, 3.8715200424194336], [3.2580513954162598, -1.7454346418380737, -1.4084062576293945, 2.992915153503418, 5.5992536544799805, -5.386171817779541, 1.2307088375091553, -0.9638494849205017, 1.7679437398910522, -3.3383007049560547, 1.68517005443573, -8.017487525939941, 2.742948055267334, 0.13800734281539917, 0.4286491870880127, -0.8665258884429932, 1.6449174880981445, 0.21619075536727905, -5.079162120819092, -3.1912741661071777, 1.8176376819610596, 2.9644477367401123, 2.934596538543701, -1.5933423042297363, -0.9167792201042175, -4.669768333435059, 10.000429153442383, 2.9974184036254883, 1.263606071472168, -0.9019322991371155, 3.692500114440918, 0.20196875929832458], [0.7448582649230957, -0.5987536907196045, 1.4040664434432983, -0.24865466356277466, -0.11900635063648224, 6.718472480773926, -5.993018627166748, 4.518617630004883, 2.589728355407715, -1.9805134534835815, 2.173795461654663, -4.297878265380859, 1.8842484951019287, 7.484036445617676, -1.359399437904358, -2.9863734245300293, -2.857339859008789, -5.682916641235352, 0.6632583141326904, 1.5882396697998047, 4.0305705070495605, -1.0346641540527344, -0.9671921730041504, 0.3314914107322693, -4.3807244300842285, -1.6403477191925049, 0.7104026079177856, 1.453096628189087, -1.5390081405639648, -1.9432915449142456, -4.640937805175781, 0.7688241004943848], [0.010574069805443287, -1.3043835163116455, 3.049638271331787, 0.05079387128353119, 1.6754651069641113, -4.455000877380371, 1.572545051574707, 1.2881828546524048, 0.9736160039901733, -2.7649738788604736, 0.9120998382568359, -4.523341655731201, 1.5313055515289307, -2.399001359939575, 0.19262591004371643, -3.7825984954833984, 5.841324806213379, -2.0545716285705566, 2.181236982345581, -3.018251895904541, -2.914546251296997, -1.9823343753814697, -0.7019575834274292, 0.27793630957603455, -1.71470308303833, 0.9556640982627869, -4.375698089599609, -0.23122796416282654, 6.100207328796387, -5.048779487609863, 5.879253387451172, -2.2066211700439453], [-6.287771701812744, 3.1054279804229736, -1.3711868524551392, -2.289346218109131, 2.4491639137268066, 0.42380914092063904, 0.4299340844154358, -4.522289752960205, -1.0143818855285645, 0.15830105543136597, 6.485416889190674, 5.174188613891602, 0.05309036001563072, 3.0433008670806885, -1.1273255348205566, -1.2396574020385742, -3.7292580604553223, 3.921998977661133, -4.415485382080078, -2.944690227508545, -2.6746842861175537, 0.3824129104614258, 1.084213137626648, 0.9328206777572632, 0.9619401097297668, 0.6057720184326172, 3.1829957962036133, -0.1310766637325287, -2.193683624267578, 2.7890260219573975, 1.7439074516296387, -5.631196975708008], [4.601574897766113, 0.3612043559551239, 0.6819988489151001, 2.611828327178955, 4.902050018310547, -0.6823751926422119, 4.730929851531982, -5.834968090057373, -2.839407205581665, -0.6839452981948853, -2.2847490310668945, 0.17562821507453918, -4.494652271270752, -1.9200986623764038, -3.1134490966796875, 2.105910301208496, 5.36796760559082, -3.709054946899414, -3.513078212738037, -2.9465255737304688, -0.022820718586444855, -2.2875986099243164, -2.3499832153320312, 2.044426441192627, 2.660104274749756, -2.364455461502075, 0.3984593152999878, -0.005269232206046581, -5.26580286026001, -1.6249297857284546, 6.751892566680908, -0.552024245262146], [2.6103572845458984, -4.746410846710205, 0.4111476540565491, 1.0528461933135986, -1.3336124420166016, -2.337418556213379, -0.4473488926887512, -1.5940771102905273, -4.201540946960449, -5.040279388427734, 2.2411670684814453, -1.2586009502410889, 0.6559461355209351, -3.8512139320373535, 2.4017391204833984, 1.698027491569519, -3.7914724349975586, -3.9995760917663574, -2.9575064182281494, -1.5109524726867676, -1.9326592683792114, 4.918001651763916, 3.503653049468994, 1.6778208017349243, -3.745980978012085, -2.7805263996124268, 3.976363182067871, -0.22466051578521729, 5.720229625701904, 1.357776165008545, 1.456066608428955, 1.5115309953689575], [-0.8958448171615601, 1.6396676301956177, 0.7985584735870361, 0.13763439655303955, -1.5607126951217651, 0.47299617528915405, 4.437937259674072, -0.9899313449859619, 3.489023208618164, 0.027189254760742188, 1.8177645206451416, -4.89717960357666, 1.6228539943695068, 7.0398101806640625, -2.010275363922119, -2.2788729667663574, 1.722680926322937, -4.58321475982666, -2.0645599365234375, 0.08912990987300873, 1.3249133825302124, -2.897110939025879, 3.8338708877563477, -0.17713716626167297, -2.2472782135009766, -0.16932639479637146, -0.4140332043170929, -0.436629056930542, -4.761228561401367, 0.16746456921100616, -0.7124892473220825, 3.5603435039520264], [-1.514984130859375, -1.661789059638977, -3.4144768714904785, 0.8240349292755127, -0.005774166435003281, 3.745673179626465, -4.504445552825928, -0.10704260319471359, -1.467543601989746, 0.9449237585067749, -0.22249536216259003, 0.4998723864555359, 4.061773300170898, -2.4648780822753906, -2.498955488204956, -0.2047448754310608, -1.1605980396270752, 2.944833755493164, 2.6169204711914062, 0.904331624507904, -2.0344769954681396, -5.676697731018066, 4.989645004272461, 1.135693907737732, 1.0198978185653687, -1.1289607286453247, 3.848320484161377, 1.7100231647491455, 5.1501898765563965, -0.5192846059799194, 0.9567834734916687, 0.40256476402282715], [-1.020066499710083, 6.09611701965332, -4.258563995361328, -0.058023132383823395, 3.617983818054199, -0.17950141429901123, 2.6021728515625, 4.003164291381836, 1.8320724964141846, -1.3684271574020386, 1.7742741107940674, -2.972334623336792, -2.066193103790283, 5.592067718505859, 4.71578311920166, 5.182400703430176, -3.322827100753784, -1.6317672729492188, -0.7874520421028137, -4.347725868225098, 0.30949074029922485, -1.623687744140625, -3.7770373821258545, 0.30804896354675293, 0.32699573040008545, 2.4640328884124756, -2.258929491043091, 0.09013095498085022, 4.4278883934021, 1.6702945232391357, 0.4842531681060791, -0.6428920030593872], [8.540916442871094, -1.4794648885726929, 3.0116286277770996, -0.8007332682609558, -1.9049030542373657, 0.9183788299560547, -1.4283592700958252, 1.0837920904159546, 0.287980854511261, -1.4631397724151611, -4.661592483520508, 3.530154228210449, -4.55652379989624, 1.1371387243270874, -3.7237465381622314, -2.135514497756958, -4.731897354125977, 1.508255124092102, -2.123223304748535, 2.193927526473999, 0.7150285243988037, 7.641990661621094, -2.0348167419433594, -1.804007649421692, 0.2893212139606476, 1.7440738677978516, 4.243913173675537, 2.2419605255126953, 0.7546490430831909, 0.189470574259758, 2.214315176010132, 12.328957557678223], [0.21620821952819824, -1.525542974472046, -4.911378860473633, -0.4628666639328003, 3.955845832824707, 0.39890819787979126, -1.4467437267303467, -3.388167142868042, 4.795977592468262, 3.2492001056671143, -0.3118773102760315, -4.560800552368164, 5.6136651039123535, -6.067605018615723, 4.58135986328125, -0.06414473801851273, 3.7049665451049805, -0.34817805886268616, -2.9819283485412598, 2.237753391265869, 3.351555347442627, -2.3744428157806396, 1.8150932788848877, -2.974017381668091, -2.0306684970855713, 0.7637383937835693, 2.391469717025757, 6.922695159912109, -1.324535846710205, -0.23617887496948242, -1.2070224285125732, -3.995556592941284], [2.020470142364502, 1.7665653228759766, 2.1325583457946777, 3.019214630126953, 0.17079895734786987, -1.5156679153442383, -1.387711763381958, -2.3659372329711914, 3.343639850616455, -3.961167335510254, 2.227813959121704, -3.7910304069519043, 0.1558666080236435, 3.17059063911438, -4.581209659576416, 1.2571730613708496, -4.516613006591797, -0.1222793310880661, 5.1533966064453125, -1.9737372398376465, 1.5113729238510132, 3.987724542617798, 4.306159496307373, -4.947240829467773, -0.10889838635921478, 1.347731113433838, -2.5812206268310547, 0.785974383354187, 0.2984253168106079, 0.8317745923995972, -0.47133368253707886, 0.2903195023536682], [1.3021841049194336, -2.886059522628784, 1.996624231338501, 3.5260753631591797, 0.692330539226532, 0.450001060962677, -0.4648875296115875, -2.498166084289551, -1.0185295343399048, -2.708113193511963, -3.3509521484375, -2.871147632598877, 0.6887354254722595, 2.7172372341156006, 1.2488335371017456, -1.1260191202163696, 1.9001909494400024, -0.10405711829662323, -1.8087108135223389, -4.628749370574951, 2.4829838275909424, -1.8428313732147217, 0.16523459553718567, -5.0696330070495605, 3.0895333290100098, -0.9821497201919556, 2.4880502223968506, 3.695370674133301, -3.046597957611084, -0.9684113264083862, -3.030052661895752, 3.3426103591918945], [4.969454765319824, -0.21905142068862915, -0.40948522090911865, -2.6639695167541504, -0.5359580516815186, 1.6486668586730957, -1.3233134746551514, 1.7263517379760742, 3.799931049346924, -0.001844990998506546, 1.3519471883773804, 0.1198868453502655, -0.057738445699214935, 1.8660374879837036, -0.14749173820018768, 0.3764663338661194, 0.6337278485298157, -1.7582809925079346, 0.5040998458862305, -5.686694145202637, 0.48903554677963257, 3.790088653564453, -0.15115591883659363, 1.461417317390442, 0.2766924798488617, 0.2006952464580536, 3.4815258979797363, -0.2903754711151123, 4.89865779876709, 1.0255990028381348, -0.91798996925354, -3.0783276557922363], [3.7600417137145996, -0.5840345621109009, -0.5627188682556152, -1.3783336877822876, -4.681708335876465, 1.9516186714172363, -0.8157016038894653, 0.06644181162118912, -2.4283335208892822, -2.42592716217041, -0.28452572226524353, -0.48256951570510864, -2.03529691696167, -0.5031200647354126, -1.024820327758789, 0.9169130325317383, 2.1220641136169434, 1.0496826171875, 1.7428076267242432, 1.7289588451385498, -0.3967263400554657, 2.1955037117004395, 4.414372444152832, -0.5462267994880676, -1.849794864654541, -1.5418176651000977, 0.6769781112670898, 4.798197269439697, 2.7612738609313965, 2.2992403507232666, -1.7171634435653687, -2.40852689743042], [0.6429854035377502, 0.18696025013923645, 0.11395537108182907, -2.1017885208129883, 1.9694955348968506, -3.4894304275512695, -1.779921293258667, 3.8606882095336914, -0.21903634071350098, -5.5733842849731445, 2.67885684967041, -4.752292633056641, 1.0346379280090332, -1.9458335638046265, -1.4382379055023193, 1.119017243385315, -2.5221407413482666, -2.800013780593872, 1.5230494737625122, 6.3131866455078125, -0.30238500237464905, -1.001820683479309, -4.651979446411133, -2.5268912315368652, 2.654369592666626, -1.159519076347351, -2.0182347297668457, 2.328720808029175, -3.640653133392334, 0.5381081104278564, 8.684895515441895, 0.7819710373878479], [6.005199432373047, 2.3792591094970703, -1.441221833229065, 0.10997380316257477, 2.7237191200256348, 2.586508274078369, 0.6965605020523071, -0.3357164263725281, -3.0110840797424316, -0.8242912292480469, 5.88385009765625, 0.25874996185302734, -4.216752052307129, -2.206422805786133, 2.401028871536255, 1.8672263622283936, -0.3090325593948364, -0.6911267638206482, -5.267224311828613, 6.151700973510742, 2.8813259601593018, -1.4988408088684082, 1.0301599502563477, -2.605259895324707, 1.602623701095581, 3.1355881690979004, 1.1116087436676025, -0.022218449041247368, -1.2962013483047485, -0.058015428483486176, -0.5940330624580383, 2.0009703636169434], [-5.139797210693359, 5.1587724685668945, -2.9162790775299072, -4.561790943145752, -0.7555508613586426, 3.0202689170837402, -1.0549064874649048, -1.0421736240386963, -0.813622236251831, -2.003091812133789, -3.6094858646392822, 7.455658912658691, 0.24525156617164612, -4.0397515296936035, 1.1285674571990967, 7.457907676696777, -4.931844234466553, -1.299854040145874, 3.0354201793670654, 2.4322633743286133, -0.6546965837478638, 1.8737850189208984, -0.4350890517234802, 1.4560061693191528, 3.1096551418304443, 3.608370780944824, -0.7158867716789246, 3.14870023727417, -1.5947527885437012, -1.6705718040466309, -2.5613203048706055, -5.097278118133545], [1.9473721981048584, -2.3248438835144043, -1.4345157146453857, -0.3953763246536255, -3.5793471336364746, -0.341365247964859, 0.22656060755252838, 4.05225133895874, -1.0267812013626099, 2.2885830402374268, -4.871471405029297, 3.6362078189849854, 2.183467388153076, 1.5235977172851562, 0.030481915920972824, -1.5815503597259521, 0.49539265036582947, 2.3426742553710938, -5.544154167175293, -2.8177833557128906, -1.6195576190948486, 4.195183753967285, -2.350233554840088, 0.8069305419921875, 3.3160293102264404, -0.2635404169559479, -1.3453307151794434, 1.9799437522888184, 0.2806471884250641, 1.4006435871124268, 1.7292627096176147, -4.008506774902344], [0.6584640145301819, 5.26961612701416, -3.742302894592285, -0.7202125787734985, -1.5208323001861572, 5.47413444519043, -0.10833395272493362, -0.9321557283401489, -0.13962581753730774, 3.397037982940674, -0.001278612995520234, -0.4952867925167084, -4.828762531280518, -3.582139015197754, -0.662946343421936, -2.7127225399017334, -0.6974357962608337, 0.44056880474090576, -7.801339149475098, 1.2129158973693848, -0.5256441831588745, -0.3417457342147827, 1.253182053565979, -4.05816125869751, -0.15629902482032776, -2.306398868560791, 0.609965980052948, -2.4914252758026123, 0.032775990664958954, -4.870298385620117, -2.2331738471984863, -0.9038984775543213], [-2.1127445697784424, -1.7650775909423828, -4.831971645355225, 2.659080982208252, 2.4061522483825684, 1.5665441751480103, 2.0710251331329346, -2.5251781940460205, 1.5687017440795898, 4.197900772094727, 2.1102678775787354, 3.205435276031494, 0.5774370431900024, 1.6611011028289795, -1.9769729375839233, -3.8702878952026367, -3.507143020629883, -3.075019359588623, -1.6368488073349, 2.6296401023864746, 1.4373997449874878, 0.20814098417758942, 1.358994960784912, -2.000476360321045, -1.4663200378417969, 6.539338111877441, -3.6092395782470703, -1.5027433633804321, 2.6727001667022705, 0.11105944216251373, 1.0931042432785034, -0.45688149333000183], [-2.3290903568267822, 3.4532017707824707, -0.11128680408000946, 0.6454193592071533, 0.7062026262283325, 0.9478839635848999, -2.311359405517578, -1.7734706401824951, -2.108520030975342, 5.447185516357422, -3.046627998352051, -0.16611967980861664, 0.48378440737724304, -0.6447667479515076, -1.5330744981765747, 2.293246030807495, 0.057125069200992584, -2.9060587882995605, 2.7829668521881104, -1.2313809394836426, -2.933380603790283, -0.3638780415058136, -1.566009521484375, 0.5177065134048462, 1.0441620349884033, 8.786115646362305, -0.2538023591041565, -0.07857629656791687, 1.8696236610412598, 2.177607536315918, 1.6803474426269531, 1.000749111175537], [2.6652259826660156, -3.6846764087677, 3.441729784011841, 4.705564498901367, 2.7452445030212402, 1.9591891765594482, 3.6019210815429688, -2.247020721435547, -0.48200950026512146, -5.75570011138916, -3.6584694385528564, 2.8716838359832764, -1.7650556564331055, 4.448798656463623, -2.6730241775512695, 1.255515694618225, -0.6305149793624878, 0.9739209413528442, -0.0037314570508897305, 8.252114295959473, 4.800993919372559, 0.24933132529258728, 1.1316301822662354, 4.26206111907959, -10.654600143432617, -4.451882362365723, -1.6798174381256104, 4.337930202484131, -1.0421264171600342, -2.998082160949707, 4.061272144317627, -3.4481325149536133], [-2.3375847339630127, 1.4991371631622314, -2.2962026596069336, -5.588129043579102, -3.7199583053588867, 2.513982057571411, 1.112658977508545, -0.49209409952163696, -2.2912497520446777, 4.626106262207031, -2.350477695465088, 1.6296360492706299, 0.6169167757034302, 3.1151626110076904, -2.9198150634765625, 0.15001240372657776, 4.001979827880859, 1.6742538213729858, 1.993388056755066, -2.4958126544952393, -6.237983703613281, -0.19125835597515106, -2.358808994293213, 1.3415918350219727, 2.5602033138275146, 1.8372430801391602, -1.2747002840042114, -0.2757759094238281, 1.9255095720291138, -3.032369613647461, -1.0073779821395874, -1.723606824874878], [6.497068405151367, 2.1647472381591797, -1.2699857950210571, -2.8098952770233154, -1.754807710647583, 0.8216215372085571, 0.3609834313392639, 0.6617928147315979, -4.432283878326416, -1.8677890300750732, -6.190256595611572, -0.8337251543998718, 0.4240712821483612, 1.2443616390228271, 0.23383009433746338, -0.5264431834220886, -1.4232341051101685, -0.04405267536640167, -0.10638982057571411, 0.3142164647579193, -5.990235328674316, -2.7738094329833984, -0.2138260155916214, -2.8561224937438965, -0.35508155822753906, -4.841038227081299, 0.42645055055618286, -1.2187073230743408, -1.368358850479126, 1.9757561683654785, -1.3680462837219238, 2.7476797103881836], [-2.4042739868164062, 0.2983349859714508, -2.2955238819122314, -0.5705797672271729, -2.160733699798584, 1.2003792524337769, -3.2075047492980957, 0.06335531175136566, 3.1454150676727295, -1.1481471061706543, -0.20844602584838867, 1.8964591026306152, -2.261716842651367, -0.04116912558674812, -1.6741876602172852, 5.305294513702393, 5.65725564956665, 0.8538675308227539, -2.2558069229125977, -2.5375776290893555, -1.3409713506698608, -3.7306017875671387, 0.9416661858558655, -3.5300588607788086, -2.0557312965393066, 1.179624319076538, 1.2175509929656982, -3.259747266769409, -6.697111129760742, 1.6515158414840698, -1.254908561706543, -5.2188286781311035], [-1.99504816532135, 2.3137624263763428, -0.12531785666942596, -2.048854112625122, 1.0152316093444824, 0.8163573741912842, 0.9385879039764404, -1.284179925918579, 5.1108078956604, 1.3025250434875488, -2.2987663745880127, -3.712770938873291, -2.984485149383545, -2.9086451530456543, -3.1395184993743896, -6.195951461791992, 2.594265937805176, 3.0114126205444336, -9.102079391479492, -1.245847463607788, -2.3030295372009277, -3.940621852874756, 0.24190784990787506, 1.3371139764785767, -2.1445603370666504, 1.8927873373031616, 2.6233291625976562, -1.3515839576721191, -1.3171933889389038, 2.5773510932922363, -5.750484466552734, 2.2745697498321533], [-2.0635149478912354, 1.2690576314926147, 0.06541749835014343, 2.0157463550567627, -1.2077903747558594, 1.8768343925476074, 3.75827693939209, -4.047959327697754, 5.669077396392822, 2.2928895950317383, -0.19521984457969666, 3.2930779457092285, -2.7395620346069336, -9.129582405090332, -2.1388261318206787, -2.2259204387664795, 0.5757405161857605, 2.3585872650146484, -2.4522440433502197, 3.5100855827331543, 0.9994643926620483, -4.68143367767334, 0.5538426637649536, -4.167113304138184, 0.3861895799636841, -3.212597608566284, 1.876854419708252, -3.7010629177093506, -2.5556511878967285, -5.462161540985107, 1.75334894657135, -0.9025808572769165], [3.257683277130127, 1.4508440494537354, 6.014735698699951, 0.4289388656616211, 3.239393949508667, 2.060885429382324, 4.481030464172363, -1.1158101558685303, -2.3158884048461914, 2.217836618423462, -2.4582228660583496, 1.1244539022445679, -3.843855142593384, 1.457524061203003, 3.624919891357422, 2.4681239128112793, 3.8179006576538086, -1.200595498085022, 0.7771340608596802, 3.8736438751220703, 0.2272622287273407, -2.5485002994537354, -1.1297565698623657, 1.56227445602417, 2.2153570652008057, 3.397704601287842, 1.0403759479522705, -3.809886932373047, 1.0163289308547974, 1.7879537343978882, 3.9114925861358643, 0.8382335901260376], [0.4686589241027832, -0.6769170761108398, -4.235410690307617, -3.8650379180908203, 5.336213111877441, 0.5113089680671692, 0.814173698425293, 1.7382612228393555, -0.3188236355781555, 0.7609598636627197, -0.2006586790084839, -0.10877566039562225, 2.012141704559326, 0.13042856752872467, -1.3018524646759033, -3.704619884490967, 3.224330425262451, -0.3085174858570099, -1.2319090366363525, 0.5012872219085693, -1.6380424499511719, -2.8524065017700195, -0.0609794519841671, 1.4467154741287231, -3.8544039726257324, -1.7903075218200684, -2.442556381225586, 0.38131940364837646, 0.7105865478515625, 0.8654632568359375, -2.526366710662842, -2.6196441650390625], [-3.8692569732666016, -5.098736763000488, 3.262723922729492, -0.06551594287157059, -2.8202643394470215, -0.5593934655189514, -0.5392589569091797, 1.7035291194915771, -7.2997331619262695, 2.6844987869262695, -4.878120422363281, -1.7239619493484497, -4.542088031768799, 0.6831090450286865, -0.3555713891983032, -0.2867189049720764, -0.4178091883659363, 5.751926898956299, -4.6346821784973145, 2.662612199783325, 0.2558812201023102, 0.31143254041671753, -2.4667179584503174, -3.6998167037963867, -3.70883846282959, 1.3711738586425781, -1.1908079385757446, 3.010215997695923, -0.7720520496368408, 3.3720555305480957, 2.5171637535095215, 4.301060199737549], [0.1773090958595276, -1.0745856761932373, -0.9136438965797424, 4.920310974121094, 4.443960189819336, 6.4308085441589355, 0.325890451669693, -1.0336079597473145, 3.819241762161255, 0.6752926111221313, 0.3168830871582031, -3.082493782043457, 4.845563888549805, -2.2274646759033203, 5.494974136352539, 0.8751070499420166, 11.578838348388672, 1.532015085220337, 0.3883150815963745, -2.0231375694274902, 1.2773334980010986, 3.389765739440918, -0.33056342601776123, 1.842144250869751, 0.4664551913738251, 1.8082568645477295, -3.405149459838867, 0.04679478332400322, 2.155975818634033, 6.5816473960876465, 4.273222923278809, -0.5289491415023804], [-1.5288212299346924, 2.8727009296417236, -3.2543559074401855, -4.706138610839844, 2.6234936714172363, -0.8702064752578735, 2.8544821739196777, -1.7359038591384888, 0.5643370747566223, 0.42920583486557007, -3.485733985900879, -3.6792397499084473, 0.7406736612319946, -4.060488700866699, 4.084572792053223, -0.15540894865989685, 3.796699047088623, -2.8300600051879883, 1.4040138721466064, -5.240840911865234, -3.884713649749756, -2.486518621444702, -0.8503550291061401, 0.8906254768371582, -2.5444653034210205, -0.8848355412483215, -1.2062264680862427, -3.6332435607910156, 0.5891570448875427, 6.4632568359375, 1.0165895223617554, 0.32148507237434387], [-0.47476112842559814, 0.6751046776771545, 3.7913410663604736, 0.4490855634212494, -5.7784423828125, -0.26981812715530396, -1.4198154211044312, 1.269174337387085, -5.377895355224609, 1.6161997318267822, -5.884495735168457, -3.5712482929229736, 0.7307859063148499, 1.8571066856384277, -1.0665380954742432, -0.615720808506012, 3.245659351348877, -5.2622199058532715, 3.802252769470215, -1.318669319152832, 1.0103729963302612, 2.0756783485412598, -1.4490084648132324, 0.5546935796737671, -0.12448257207870483, -0.4900611639022827, -1.6203856468200684, 2.1752052307128906, 1.6086723804473877, 2.824145555496216, -5.289624214172363, 5.145413398742676], [-2.5921669006347656, -2.572678327560425, 3.9579625129699707, 3.755280017852783, 3.5287833213806152, 5.8357696533203125, 1.6324381828308105, 3.3756344318389893, 6.073055267333984, -0.45724809169769287, 2.510939836502075, -6.766366004943848, -4.900562286376953, -2.869095802307129, -3.443148136138916, -1.7624714374542236, -0.12910887598991394, 4.205470085144043, -0.27173495292663574, 2.4858851432800293, 1.9852070808410645, -0.550620436668396, 3.7981860637664795, 0.260271817445755, 1.6671276092529297, 1.2333528995513916, 1.8620266914367676, -4.31584358215332, 1.7843215465545654, -1.2961761951446533, 3.1554558277130127, -1.2997196912765503], [1.509710669517517, -2.5808117389678955, 5.082805633544922, -3.6622509956359863, -1.3501992225646973, 8.762149810791016, -3.4095067977905273, -4.888080596923828, -0.2616979479789734, -1.3670616149902344, 3.9910075664520264, -0.6648651361465454, 1.73597252368927, -3.123569965362549, 1.0400450229644775, 5.571606159210205, 0.32237422466278076, -1.7411574125289917, 3.3411519527435303, 5.829097747802734, -0.8245630860328674, 0.5119071006774902, 3.2115468978881836, 1.6048433780670166, 1.9009735584259033, 1.3115171194076538, -4.578819751739502, -1.5867352485656738, -3.307091474533081, -0.29927128553390503, 2.1384029388427734, -2.8261637687683105], [3.238941192626953, -4.176250457763672, -0.5806010365486145, 2.654505729675293, -7.363090515136719, -1.4892354011535645, -5.745562553405762, 1.623844027519226, -1.911739468574524, -7.095098495483398, 0.9928291440010071, -0.45719707012176514, 2.6066877841949463, 1.305313229560852, 0.6290264129638672, -2.0987486839294434, -1.1115654706954956, 3.229081630706787, -0.5829435586929321, 3.937720775604248, -4.258347511291504, -2.888791561126709, -3.0473008155822754, -3.038689136505127, 2.2123308181762695, -0.8204759359359741, 1.1304378509521484, -3.896145820617676, 0.2723499834537506, -1.0760847330093384, 3.4138784408569336, 0.7150947451591492], [3.0791382789611816, 0.0843312218785286, 2.494777202606201, 0.0883975476026535, -5.878420829772949, -1.0203903913497925, -0.42029869556427, -1.039145827293396, -2.382835626602173, -1.0400278568267822, 0.6307507157325745, -1.9004509449005127, 0.1665024757385254, -0.8891415596008301, 1.4889534711837769, 6.108536720275879, 0.9066805839538574, -3.817155361175537, 6.62572717666626, 2.463386297225952, 2.2101550102233887, 1.8916836977005005, 5.4960246086120605, -0.09542694687843323, 3.2528886795043945, 6.977964878082275, -2.2545838356018066, 3.134239912033081, 3.0391201972961426, -2.9406607151031494, 4.40813684463501, -1.554842472076416], [1.6187069416046143, -1.7755379676818848, -1.028764247894287, -1.1838775873184204, -4.72944974899292, -1.8555049896240234, -6.389786720275879, -2.3390674591064453, 0.5474612712860107, -0.41572338342666626, 1.209714651107788, 0.5809919834136963, 2.3554296493530273, -0.47206658124923706, -4.752516269683838, -2.632425546646118, 2.0668418407440186, -3.2675223350524902, -4.670644760131836, 0.6099047660827637, 0.38408905267715454, 2.725426435470581, -0.06515620648860931, -1.2071548700332642, -4.4211225509643555, 5.687575340270996, 4.259798526763916, 0.699615478515625, 0.7149031758308411, -5.3754167556762695, 4.521518707275391, 4.030246734619141], [-8.23186206817627, -6.517337799072266, -4.648236274719238, -5.640750885009766, -3.029357671737671, 3.8465170860290527, 0.5718458890914917, 4.766761779785156, -0.576732337474823, -1.3275916576385498, -0.9678641557693481, -0.013414199464023113, -2.5756337642669678, 1.0270520448684692, 1.152159571647644, 1.8572219610214233, 3.1270852088928223, 0.18602976202964783, -4.629787921905518, 0.9107445478439331, -2.9865286350250244, -0.43270745873451233, -4.6122002601623535, -2.07822322845459, -0.5886922478675842, 1.5685282945632935, 1.8020544052124023, 4.869854927062988, 3.524354934692383, 1.299112319946289, 2.9330573081970215, 2.1805620193481445], [-2.067368984222412, -1.2593410015106201, -1.2615270614624023, -2.9938783645629883, 6.216579437255859, 2.890435218811035, -0.1774943321943283, -2.863693952560425, -4.995133399963379, 1.5430552959442139, 2.7576732635498047, -0.5751074552536011, -1.2701385021209717, 4.531420707702637, -2.660315752029419, 2.4612011909484863, -2.49916934967041, 0.8081747889518738, 6.345636367797852, -3.132349967956543, -2.9103450775146484, 1.9941656589508057, -6.608539581298828, 2.9617650508880615, 1.7939174175262451, 5.161443710327148, 0.1919746845960617, 1.7062021493911743, -4.304436683654785, 3.953014850616455, 0.43066826462745667, -1.2182406187057495], [5.041832447052002, -1.748986005783081, -0.36304962635040283, -2.5159525871276855, 0.34991490840911865, 1.6367976665496826, 4.327685832977295, -3.200883150100708, 2.278965711593628, -2.372488498687744, -3.598712921142578, 1.8158040046691895, -1.1113224029541016, 6.663280487060547, -1.2293729782104492, 3.531487226486206, -3.2797858715057373, 1.1476484537124634, -2.0447397232055664, -0.9437520503997803, 1.1260638236999512, -3.1194376945495605, -1.996669054031372, -1.827269196510315, 0.2614392936229706, -1.3462351560592651, -1.1928513050079346, -2.4626879692077637, -2.545691967010498, 1.8282604217529297, -3.9123945236206055, -0.06699119508266449], [1.3190393447875977, -0.5557462573051453, -0.4773659110069275, 0.3954538106918335, -1.077627420425415, -0.36464017629623413, 2.053179979324341, -1.7012676000595093, 1.8767585754394531, 0.6771728992462158, -0.4465431869029999, 3.033147096633911, 0.9457231760025024, 1.257615566253662, -3.1270179748535156, 0.5515486598014832, 2.980318307876587, -1.4947025775909424, 0.9681200981140137, 2.7095794677734375, -7.161123275756836, 0.0011958896648138762, -2.5629968643188477, 2.8587801456451416, 2.8596630096435547, 0.6265687942504883, -1.5884075164794922, -3.796027183532715, -0.8913198709487915, -3.1088786125183105, -1.9097082614898682, -1.1939921379089355], [1.1874456405639648, -4.076961994171143, -2.681610107421875, -2.166116952896118, 4.860110759735107, -2.1499247550964355, -2.1453092098236084, -1.644895315170288, -3.2013473510742188, -4.290996551513672, 3.2200684547424316, -0.745587170124054, -0.7980365753173828, 0.1724955290555954, -3.821255922317505, -2.6372947692871094, -3.7859253883361816, -2.8029892444610596, 4.722513675689697, 4.8211164474487305, -4.849032402038574, -3.3771190643310547, -2.3657805919647217, -0.02173551172018051, 4.151949882507324, -4.6749372482299805, 0.5381791591644287, 3.4615182876586914, 0.591462254524231, -2.1964776515960693, -5.772793769836426, -0.4436931610107422], [2.1300976276397705, -2.020536422729492, -5.003413200378418, -2.6070847511291504, -2.7741284370422363, 0.6055534482002258, -4.076369762420654, 2.398951768875122, -2.529095411300659, 1.729996681213379, -4.729589462280273, 1.5152053833007812, -1.2514209747314453, 4.513991355895996, 2.2061471939086914, 1.7809488773345947, 4.374743461608887, -5.435375690460205, -3.996589422225952, -3.3507533073425293, -3.1520397663116455, -1.4320147037506104, -2.4919180870056152, -1.574971318244934, -2.2645576000213623, 0.14743468165397644, 0.3039216101169586, -1.8571369647979736, 1.851947546005249, 1.4462251663208008, 1.187028169631958, -6.853541374206543], [-3.5591626167297363, 3.52054500579834, 1.7495877742767334, 1.6859509944915771, -1.0916153192520142, 4.840854167938232, 0.6707985401153564, -3.9357523918151855, 3.356943368911743, 1.4516078233718872, -0.1855354756116867, -0.21299773454666138, 0.49039405584335327, 2.7871580123901367, 1.1624706983566284, 1.7568027973175049, 0.09860219061374664, -0.059140339493751526, -0.9460470676422119, -3.9838390350341797, 1.4885327816009521, -0.5521446466445923, -2.922579050064087, 0.11922714114189148, 4.926026821136475, 0.9591466784477234, -5.0421648025512695, 1.8900184631347656, -0.9882709980010986, -0.7037093043327332, -3.5956578254699707, -1.575317621231079], [4.707786560058594, -2.569675922393799, -3.7046191692352295, 4.198866367340088, -3.8880608081817627, 0.3891776204109192, -1.1533408164978027, 0.4329177439212799, -1.2371246814727783, 2.7861530780792236, 2.849400758743286, -1.5906014442443848, 1.644824743270874, -6.162025451660156, 3.15463924407959, -3.5699594020843506, 7.466264247894287, -3.1664986610412598, 1.6613763570785522, 4.413135051727295, 5.884830951690674, 0.8603471517562866, -1.6999272108078003, -1.759879231452942, -1.7151832580566406, 4.747354507446289, 4.573929786682129, -1.137014389038086, -1.1635291576385498, -4.8290252685546875, -4.62291955947876, -3.14945650100708], [2.111860752105713, 2.8270320892333984, -2.0556511878967285, 0.41368651390075684, -3.6576740741729736, 1.9520291090011597, -1.9128608703613281, 0.5608769059181213, -1.9812953472137451, -4.323064804077148, 4.381880283355713, -0.04411481320858002, 2.6187758445739746, 2.0621118545532227, -1.5458695888519287, 2.179863452911377, 2.8913588523864746, 1.3774418830871582, -0.16467724740505219, -3.439361572265625, -1.8357341289520264, 1.9333429336547852, -4.286313533782959, 2.020970344543457, 2.154801368713379, 3.943479537963867, 5.020689487457275, 5.5648627281188965, -3.5222747325897217, -2.6043758392333984, 4.370880126953125, -0.9760742783546448], [-0.7007604241371155, -1.5508875846862793, -0.5433886051177979, 0.1811543107032776, 3.992319345474243, 2.1088531017303467, -2.9569523334503174, 3.544053077697754, 0.5946862697601318, 2.329392910003662, -4.945305824279785, 4.248304843902588, 2.8729898929595947, 1.7801710367202759, 0.4348669946193695, 1.3231277465820312, 1.3156077861785889, 0.3013591170310974, 5.349989891052246, 1.7283357381820679, -1.2888153791427612, -0.781536877155304, 1.6420392990112305, -4.213703632354736, -1.6161515712738037, 0.004188667517155409, 2.6096489429473877, 3.9374163150787354, 3.7383008003234863, -2.78946590423584, 1.1024339199066162, -3.2296993732452393], [4.857896327972412, -3.55812406539917, -6.395217418670654, 0.0697675347328186, -1.8339741230010986, -5.7819743156433105, -2.3009390830993652, 0.46350300312042236, -1.6862902641296387, -3.530925750732422, -2.36120343208313, -2.7734375, 1.5681219100952148, -4.17459774017334, -3.651822090148926, -3.564955234527588, -3.359400987625122, -4.0081634521484375, -4.935800075531006, -3.6893229484558105, 3.123732805252075, 1.3313475847244263, 1.4499469995498657, 6.374825477600098, 2.1103615760803223, 2.511072874069214, -0.274411678314209, -3.359165668487549, 0.13172230124473572, -1.0949559211730957, 4.187222480773926, 7.137509822845459], [-5.623664855957031, 0.2835541069507599, -0.3401612937450409, -1.3256250619888306, -3.3507461547851562, -1.257336139678955, 3.1473333835601807, 2.568122148513794, 6.320145606994629, 2.330378532409668, 5.481161594390869, -1.489332675933838, 3.5664420127868652, -0.3066994249820709, 0.011705349199473858, 3.646010398864746, -1.92030930519104, 1.252189040184021, -4.192223072052002, 2.9219586849212646, 2.708420753479004, -2.6917617321014404, 2.532409191131592, -1.3845503330230713, 3.37328839302063, 0.294506698846817, -1.2377324104309082, 1.97996985912323, 1.8221418857574463, 1.5783100128173828, 0.17999890446662903, -2.5410590171813965], [0.7352480292320251, -2.3354709148406982, -1.0056875944137573, 1.8433324098587036, 5.4064621925354, -1.1798197031021118, 2.5646538734436035, -2.4556784629821777, 0.8577961921691895, -1.6214077472686768, -8.574798583984375, -0.3429770767688751, -5.463812351226807, -0.2804381549358368, -0.06577716767787933, 2.6163601875305176, 3.3621058464050293, 2.4092090129852295, -4.834543228149414, -2.403726100921631, -1.146323561668396, -4.6278557777404785, -3.5748190879821777, 4.701620578765869, 1.122174620628357, -0.15387192368507385, -3.5017151832580566, -2.4520316123962402, 5.378681182861328, -1.0727896690368652, 0.4631139039993286, -0.4761900007724762], [5.559560775756836, 3.015550136566162, 6.460623264312744, 0.17290233075618744, -3.4985511302948, 0.49260586500167847, -0.04233166202902794, 5.538055419921875, -2.5219240188598633, 1.549112319946289, -3.10221529006958, 6.867682456970215, 3.6006696224212646, 0.1205606460571289, 2.7480101585388184, -0.9615200757980347, -5.674745559692383, 1.165500521659851, -0.9655589461326599, 0.5673670172691345, -2.6115729808807373, 1.6281919479370117, 1.1784123182296753, 4.777050971984863, 0.4464370012283325, 3.803412437438965, -4.305877685546875, -0.3529033064842224, -0.9804074764251709, -5.592004299163818, -0.8346688747406006, -1.188462257385254], [-1.364652156829834, -1.2863370180130005, 2.372713088989258, 3.0863285064697266, -3.6823253631591797, -4.4640021324157715, -2.869422197341919, 4.0072922706604, -0.7276661396026611, 0.27846255898475647, -2.177326202392578, -3.256319522857666, -0.13461437821388245, 0.34496384859085083, -4.348289966583252, 5.424208641052246, -2.299928665161133, -4.605645179748535, 1.0057122707366943, 3.7174293994903564, 1.5212149620056152, 2.0037786960601807, 0.760793924331665, 2.539191961288452, -0.41432440280914307, 0.17788416147232056, 2.5671935081481934, -4.348226547241211, -2.244539976119995, 2.3157753944396973, -1.9952802658081055, -2.683173656463623], [0.7367266416549683, -3.044830322265625, -1.3919264078140259, 0.5919011831283569, 6.494006156921387, 1.927316427230835, -2.134312391281128, 3.1809754371643066, -2.151522397994995, 2.5979223251342773, 2.2558095455169678, 0.30983108282089233, -7.779303550720215, 7.75501823425293, 0.6130284070968628, -3.106327533721924, -4.280301570892334, -4.664672374725342, 3.5993123054504395, 1.5758185386657715, -0.3525415062904358, 1.3442169427871704, 0.7838660478591919, 0.16194477677345276, 1.455841302871704, 1.1556427478790283, 3.8614120483398438, 0.49553167819976807, 1.5061094760894775, -2.446312189102173, -1.2377948760986328, -1.1658382415771484], [5.0731353759765625, -1.332542061805725, -0.9750611186027527, 1.6789079904556274, 0.9770594835281372, -8.368910789489746, -2.5601260662078857, -0.7041313648223877, -4.874220848083496, -4.882245063781738, -1.2659318447113037, -6.123966217041016, 0.792458176612854, 0.646874189376831, -0.2641158103942871, 2.023761749267578, 0.21631622314453125, 4.473531723022461, -0.2638058364391327, 3.918306827545166, -0.6273279190063477, 2.4683446884155273, 1.3716734647750854, 1.5325652360916138, 0.2961825728416443, -4.167596817016602, -3.88655424118042, -4.291620254516602, 0.340643048286438, -0.04787285625934601, -6.835575103759766, -4.868530750274658], [-3.3341410160064697, -3.0757598876953125, 6.499408721923828, 3.909210205078125, -3.351315498352051, -5.179799556732178, 7.288780212402344, -2.045975685119629, -1.8970794677734375, 0.5980170965194702, -2.346632242202759, 2.8842663764953613, 3.534238815307617, -2.9615540504455566, 3.459235668182373, -2.844451904296875, -1.6163538694381714, 2.2394723892211914, 0.29086220264434814, -2.797426700592041, 4.835222244262695, 2.103893756866455, -2.1688191890716553, 1.8061325550079346, 1.512078881263733, 1.464843511581421, 3.995309352874756, 7.272176742553711, 0.6613498330116272, 0.08558081835508347, -4.2714409828186035, -5.10788106918335], [-2.6047301292419434, 2.9977517127990723, 3.9454400539398193, 3.2487168312072754, -2.049438238143921, -1.8335697650909424, -3.3873724937438965, 2.2243189811706543, 0.693681001663208, -0.8865005970001221, -4.211538791656494, 2.576953172683716, 2.5653209686279297, -4.420513153076172, 2.097062110900879, -0.5705671906471252, -1.1427180767059326, -0.1489926278591156, -5.171663284301758, 0.846566915512085, 3.1770236492156982, -2.360180377960205, 0.4898841381072998, -2.4599099159240723, 4.238317489624023, -1.3183883428573608, -0.5610339045524597, -0.12091657519340515, 2.731489658355713, 2.4151792526245117, 2.486863136291504, -0.039958782494068146], [-0.9425076842308044, -1.9162280559539795, -0.930637001991272, 0.9722241163253784, 2.4810314178466797, 2.743171453475952, -2.6403067111968994, -1.0850557088851929, 1.4254329204559326, -4.482449531555176, 1.855926513671875, -4.424503803253174, 0.8581846952438354, -3.781064033508301, 1.6914477348327637, 0.5538240671157837, -3.261770009994507, 5.082406044006348, 0.5633921027183533, -0.9983235597610474, 6.212098121643066, 2.0564658641815186, -2.7127132415771484, 0.6831585764884949, 2.7924773693084717, -3.206148147583008, 0.8111555576324463, -0.8325772285461426, -2.421419143676758, 3.3947718143463135, 0.896892786026001, -2.0158705711364746], [0.3686666488647461, -2.4406795501708984, -3.047046184539795, 0.6081908941268921, 0.40321648120880127, -0.5820304155349731, -4.309917449951172, 3.3042140007019043, 1.5479174852371216, -0.3817833662033081, -5.121330738067627, 6.427194595336914, -3.7714524269104004, -3.697922706604004, -0.13012006878852844, 2.624289035797119, 0.28048157691955566, 0.6893787384033203, -0.22467108070850372, -1.8410693407058716, -2.5775222778320312, 3.47214674949646, 6.414158821105957, -4.4632439613342285, 4.512900352478027, 1.273634910583496, 4.2992753982543945, 2.892636299133301, 1.7741570472717285, 2.1197776794433594, 3.160834789276123, -1.277379035949707], [-2.1421117782592773, 0.31504935026168823, -0.1310705840587616, -2.0686182975769043, 0.6962231397628784, 2.0634982585906982, 1.0398285388946533, -1.894721508026123, 0.5373265743255615, -3.8521318435668945, 1.0876930952072144, 4.24075174331665, 0.6300495862960815, -3.1984028816223145, -0.6434388160705566, -0.05881770700216293, -0.8118047118186951, -1.85239839553833, -2.008469820022583, 4.119906902313232, -0.466178297996521, -1.96175217628479, -3.9295687675476074, -6.689216136932373, 0.5832664966583252, 0.6112490296363831, 0.20006272196769714, -0.6814196109771729, -3.5660195350646973, 1.5458545684814453, -0.6517175436019897, -1.7335996627807617], [-4.324911594390869, -3.2392115592956543, 4.248637676239014, -4.621151924133301, 1.8484463691711426, -3.6985888481140137, 3.199819564819336, 4.007435321807861, 3.4548349380493164, 5.533155918121338, 1.9488675594329834, 3.127223014831543, 0.3190067708492279, 3.2677836418151855, 1.46823251247406, 4.8035888671875, 7.146759986877441, 1.0778377056121826, -3.6044464111328125, 1.298775315284729, 3.742400646209717, 2.110945224761963, -1.1558488607406616, 2.3482186794281006, 1.8960692882537842, 3.5911481380462646, 3.0279314517974854, -7.443331718444824, -0.6143462061882019, -1.246140480041504, -1.0143303871154785, -3.0577940940856934], [-3.1137290000915527, -0.6881728768348694, 3.3048977851867676, 5.4656548500061035, -0.01125368196517229, 1.2936999797821045, -0.35520023107528687, -2.0975451469421387, 0.40756282210350037, 1.231825828552246, -2.5675039291381836, 3.8572475910186768, -0.8440338373184204, -2.3959641456604004, 1.82484769821167, -1.3465520143508911, -2.9255971908569336, -4.123559951782227, -1.3290345668792725, -4.828721046447754, 1.7086998224258423, 3.0662577152252197, -3.724897861480713, 7.041484832763672, -8.415569305419922, -0.3215107023715973, 6.0997209548950195, 2.422307252883911, 2.7175116539001465, 1.8001034259796143, 6.028634071350098, -0.6726281046867371], [-2.649468421936035, -1.9494333267211914, 6.761736869812012, -1.797684907913208, -0.5435172915458679, -1.1802853345870972, -5.144203186035156, -4.269551753997803, 2.995375633239746, -1.1271109580993652, -4.102353096008301, 3.7228856086730957, 4.04164981842041, -0.5724716186523438, -2.790677785873413, 0.9614258408546448, 5.492260932922363, 3.6841211318969727, 2.134228467941284, -0.3602951169013977, -0.972663938999176, 4.337921142578125, 6.433917999267578, 0.8441053032875061, 0.47504156827926636, -5.613509654998779, -0.20451900362968445, -2.0077571868896484, 0.16411903500556946, 0.4938884973526001, -4.695420265197754, 0.2899995446205139], [-1.0047982931137085, -1.1933387517929077, 2.0233304500579834, 1.2910313606262207, -3.1962480545043945, 1.7518179416656494, -2.6550023555755615, 5.5304179191589355, 0.6766986846923828, 1.7865228652954102, -0.1874242126941681, 0.2688087224960327, 2.750019073486328, -3.1779050827026367, -4.539133548736572, -2.9738123416900635, -4.5557684898376465, -2.3080673217773438, 1.6626759767532349, -1.8295161724090576, 0.9206653833389282, 1.8208794593811035, 2.5807981491088867, 2.038137197494507, 3.2867307662963867, 1.9646711349487305, 3.5617079734802246, 1.7233796119689941, -5.25216007232666, -1.7446987628936768, 2.8963749408721924, 0.6975476741790771]], [[-0.16120806336402893, 0.5289976596832275, -2.890960931777954, -2.0481786727905273, 0.3753766417503357, -3.974857807159424, 2.675978660583496, -4.0106658935546875, 0.6372587084770203, -1.9817368984222412, 5.677577972412109, 1.1568392515182495, -0.012771278619766235, 0.591143786907196, -4.0344319343566895, -4.595839500427246, 0.7694438695907593, 1.6694934368133545, -2.305691719055176, -0.5335081219673157, -1.1320769786834717, 4.3917412757873535, 2.5006210803985596, 0.6720036268234253, -3.0424294471740723, 0.1765792965888977, -0.6353006362915039, -0.35251355171203613, -4.314988136291504, 5.193282604217529, 5.267182350158691, 1.051853895187378], [0.5046054720878601, 0.5349839329719543, -1.470740556716919, -1.6050013303756714, 2.4138741493225098, -2.2908434867858887, 0.17681321501731873, -2.211740255355835, 0.2133779078722, 1.5259602069854736, 2.7496654987335205, 2.617846965789795, -0.7863101959228516, -5.346925735473633, -1.212712049484253, 3.5616652965545654, -3.4270567893981934, -1.856785535812378, -4.352687835693359, 3.3637261390686035, 0.48460516333580017, 7.06599235534668, -2.8982415199279785, 2.0037925243377686, -4.606573104858398, 2.2189483642578125, -2.0053887367248535, 1.7095847129821777, -3.082921028137207, -0.43272531032562256, -2.628969192504883, 3.1327366828918457], [3.5891904830932617, -0.4198886454105377, 0.7120980620384216, -1.4344134330749512, -0.16232973337173462, 4.56909704208374, -4.428959846496582, -0.7008485794067383, -0.22139716148376465, -2.0173420906066895, -4.898642063140869, -1.1202504634857178, -4.0037126541137695, 3.0787224769592285, 5.7888712882995605, -2.9512975215911865, -0.7282320261001587, -3.227205276489258, 1.537175178527832, 1.726595163345337, 1.5708364248275757, -1.7210919857025146, 1.394203543663025, -0.5421826839447021, -2.7932779788970947, -0.40256085991859436, -1.2371582984924316, -1.170902967453003, -4.9557342529296875, -0.06922087073326111, -3.647434711456299, 0.4334457516670227], [-1.4111385345458984, -2.6208763122558594, -2.662269353866577, -2.5975687503814697, -1.011975884437561, 6.634683609008789, 0.5715213418006897, -3.2682251930236816, 1.7911899089813232, 3.0766239166259766, 1.0639857053756714, -3.2158148288726807, 2.0188279151916504, 2.4836950302124023, -2.0380847454071045, -0.25463831424713135, -4.549310684204102, -0.4733470380306244, 3.9263386726379395, 0.7104555368423462, 0.8407003283500671, -1.5043456554412842, -3.4142518043518066, -2.9748523235321045, 1.3960673809051514, -0.20891998708248138, 4.047247409820557, 1.3697829246520996, -1.603930950164795, 9.925119400024414, -0.7399705648422241, 1.5012459754943848], [-4.0315117835998535, 1.2131458520889282, -3.5538830757141113, 2.549588680267334, -0.4913104772567749, -1.531603217124939, -4.041234970092773, -2.1704444885253906, 2.889392375946045, -6.488497734069824, 1.2762386798858643, -2.9357311725616455, -7.625906467437744, -0.15072759985923767, -0.3480756878852844, 2.831270217895508, 2.3387272357940674, 2.097707986831665, 4.3944244384765625, 0.65479576587677, 3.674682140350342, -5.402895450592041, -0.4858618378639221, 0.9710172414779663, 2.6635842323303223, -1.5952486991882324, -1.0369043350219727, -3.063778877258301, 4.9714579582214355, -4.204189300537109, -2.144420623779297, 1.9647648334503174], [-2.1392500400543213, -7.665411472320557, 6.629123687744141, -2.734748125076294, 3.910257339477539, 3.474994659423828, 2.6767706871032715, 1.7293528318405151, -5.799864768981934, 1.4413695335388184, 6.709817886352539, -2.0345559120178223, -5.6777520179748535, 1.7671161890029907, 0.318987101316452, -2.7270796298980713, 1.8451719284057617, 5.279662609100342, 2.2980241775512695, 0.1878584325313568, -4.894026756286621, -1.0842818021774292, -0.196417897939682, 1.6556979417800903, 2.9730026721954346, 0.8572724461555481, -0.28289657831192017, 2.576727867126465, -4.606816291809082, -1.2871904373168945, 2.2950830459594727, 4.198709487915039], [1.5267250537872314, 3.304910659790039, -3.2613773345947266, 2.685663938522339, 0.3285275399684906, 1.3724615573883057, 2.512040615081787, 0.37641966342926025, -3.6060938835144043, -0.1682662069797516, -0.9421787261962891, 3.586510181427002, -3.94846248626709, -0.8107616901397705, 2.8742427825927734, -0.693838357925415, -1.2126109600067139, -1.5323227643966675, -0.25341594219207764, -0.2154509425163269, -0.5342222452163696, -1.700168251991272, 1.520424485206604, -3.3865180015563965, 0.2900613248348236, -2.272134780883789, -0.036916255950927734, 2.503432035446167, -1.5328491926193237, 2.240769863128662, 2.1411798000335693, -2.403566360473633], [0.5907437801361084, 2.8545680046081543, -2.133465528488159, 5.810845375061035, 5.386893272399902, -2.0158472061157227, 3.4517698287963867, 2.2443528175354004, -2.6727285385131836, 4.3802080154418945, -2.278672695159912, -1.5992412567138672, -3.4223475456237793, 0.0629863291978836, -1.2721936702728271, -0.18269985914230347, 0.33439600467681885, -4.017972469329834, 5.830777168273926, 5.170761585235596, 1.316266417503357, 1.1487690210342407, 4.901457786560059, 1.222465991973877, -0.3615694046020508, -0.9440436363220215, 1.9082376956939697, -2.3725926876068115, -0.024792153388261795, 1.3887046575546265, 1.2898310422897339, 1.5183666944503784], [1.950329303741455, -0.13180728256702423, -4.909365177154541, 2.1021358966827393, 6.539724826812744, 1.3948214054107666, 4.190610885620117, 0.6585124731063843, 2.225492238998413, 0.8195338249206543, 1.7166520357131958, -2.036170482635498, 0.42642873525619507, 1.1325669288635254, 5.86555290222168, 2.7326581478118896, 1.8091788291931152, 2.104167938232422, -1.9054838418960571, 1.2924747467041016, 0.3842189311981201, 8.525662422180176, 0.638583242893219, -4.170654296875, 1.701674461364746, -0.8272632360458374, -0.45066624879837036, -2.2573437690734863, 0.4685228168964386, 1.5110280513763428, -0.18225154280662537, -2.57108998298645], [1.641939640045166, -5.1864118576049805, -3.8634915351867676, 1.7634336948394775, 2.156639337539673, 1.7793607711791992, 3.6170153617858887, -3.1637327671051025, -1.0424786806106567, 2.4429752826690674, -0.7090071439743042, -4.680123329162598, -5.044391632080078, 1.316057562828064, 0.7131052613258362, -0.20250704884529114, -5.441278457641602, 0.8854902982711792, 2.186948299407959, -1.9502230882644653, 2.3395919799804688, -0.21413232386112213, -1.5376228094100952, 8.276389122009277, 2.6308412551879883, 3.5873794555664062, 2.2107815742492676, 3.1403896808624268, -4.574453353881836, 7.349542617797852, -1.7238988876342773, 0.9915491342544556], [-0.7688126564025879, 1.6585605144500732, -1.2846565246582031, 1.5512993335723877, 1.4121729135513306, 5.187065124511719, -1.9935276508331299, -3.6700778007507324, -0.6407146453857422, 0.5072744488716125, -5.805858135223389, -1.1971817016601562, 7.973512649536133, 3.6855578422546387, 2.5461783409118652, -1.291278600692749, 1.1024603843688965, 0.03941015899181366, -9.112052917480469, 1.1168488264083862, -2.0386362075805664, -4.178877353668213, 4.225937843322754, -4.117674350738525, 0.4188404381275177, 1.299221396446228, -2.95392107963562, -3.956458568572998, -1.2875936031341553, 2.7344117164611816, -0.5407388210296631, 4.7729058265686035], [1.3311420679092407, -1.5341806411743164, 2.929328203201294, 6.561809539794922, -3.1244492530822754, -0.22016774117946625, -1.985119104385376, 2.9487600326538086, -6.749669551849365, 2.0513339042663574, 3.4774937629699707, 2.9422688484191895, -4.554428577423096, 2.5043821334838867, -6.856715202331543, 5.080885410308838, -0.20405830442905426, 2.353959083557129, 1.3117973804473877, -4.806591033935547, -1.700610637664795, 1.6380856037139893, -1.0852830410003662, 3.5473380088806152, 3.641037940979004, 1.8296821117401123, -3.2981395721435547, -3.041316032409668, 1.8454506397247314, 0.2828153073787689, -0.3880288004875183, -1.2141060829162598], [-6.517402648925781, -4.218130588531494, 2.610438823699951, 3.819138526916504, -0.7655278444290161, -5.1629109382629395, -1.6296792030334473, -3.360025405883789, -1.0461674928665161, 2.156888246536255, -2.29569673538208, 1.4980989694595337, 1.8304699659347534, 1.3865190744400024, -1.7137725353240967, -1.699638843536377, 5.007778167724609, -1.2645736932754517, 1.433802843093872, -0.21936574578285217, 0.7880142331123352, 0.4471687078475952, 5.282291889190674, 2.729870080947876, 3.6045022010803223, -0.23386862874031067, 2.5620439052581787, -4.016124725341797, 0.8802642822265625, -7.953943252563477, 5.382035255432129, -1.480352759361267], [5.8065667152404785, -1.6291441917419434, 1.6302598714828491, 7.659389495849609, -4.518212795257568, 0.0003957102890126407, 3.6722264289855957, -1.689689040184021, -8.154868125915527, 3.726400375366211, -0.20439644157886505, -1.934234619140625, -1.4289438724517822, -0.8853391408920288, -1.5499627590179443, 1.367006540298462, 1.9239685535430908, 4.799724102020264, 6.062824249267578, 0.13468226790428162, 2.999539375305176, -1.7124624252319336, 7.606308460235596, 0.9582877159118652, 5.526447772979736, -3.6320347785949707, 5.234930038452148, 0.06532151252031326, 1.6867400407791138, -1.1946654319763184, 1.5539846420288086, -4.965236663818359], [1.9330830574035645, -5.205793380737305, -1.0415295362472534, -1.054534912109375, -2.197556495666504, -0.020438911393284798, -2.802990674972534, 1.8959729671478271, -1.8427751064300537, 0.4726544916629791, -3.094425678253174, -2.975491762161255, -0.20556014776229858, -1.8291124105453491, 2.0944981575012207, 0.8446813821792603, 4.159258842468262, -4.996431350708008, -1.728214144706726, -2.4464616775512695, 5.55368185043335, -2.288886308670044, -0.8542886972427368, -3.7960832118988037, 1.0302482843399048, 0.5556015968322754, 9.539952278137207, -2.4417498111724854, 0.1838509887456894, 4.170523643493652, 1.6259490251541138, 1.4328641891479492], [2.561588764190674, -0.1464233547449112, -2.8293609619140625, -1.000121831893921, -0.6600872278213501, 1.0854427814483643, 1.0683507919311523, 0.8092766404151917, 0.9084266424179077, 2.33868408203125, 4.021100997924805, 0.08040942251682281, 2.6333749294281006, 1.7202658653259277, -2.971491813659668, 1.5682679414749146, -3.9097604751586914, -0.5540086030960083, -5.624467372894287, 0.35624587535858154, 0.7798463106155396, 1.0852694511413574, -3.8242528438568115, 1.1960744857788086, 5.3791117668151855, 0.10929954051971436, -2.606780767440796, 2.3597042560577393, 1.6643948554992676, -2.9394733905792236, -3.047579526901245, -2.1198036670684814], [-0.9358253479003906, -2.445718765258789, 4.506167411804199, -2.924325704574585, 2.1195454597473145, 0.4806842803955078, 3.2973251342773438, -0.36111390590667725, -3.9289865493774414, 0.02580489031970501, -3.6609067916870117, 3.2500767707824707, 0.6483578681945801, -1.1612783670425415, -3.661046028137207, -6.404421806335449, -0.006382088642567396, 0.7448053359985352, 4.062782287597656, 1.0495637655258179, 0.7470066547393799, -1.8841533660888672, -1.4152016639709473, -0.47484567761421204, -3.279857635498047, -3.643935441970825, -2.109074354171753, -2.524902105331421, 3.7104392051696777, -0.8903580904006958, -3.310781955718994, 0.556540310382843], [1.563093900680542, -0.5135241150856018, 3.673426628112793, 0.14907528460025787, -1.4519490003585815, -0.7412827014923096, -8.5364351272583, -1.5699419975280762, -1.083035945892334, 3.098052978515625, 1.7540448904037476, -0.5193458795547485, 3.5441017150878906, -2.044621706008911, -0.4141118824481964, 0.7207412719726562, -3.6532721519470215, -2.9903810024261475, 2.768364191055298, -7.089407444000244, 1.824452519416809, -0.8107688426971436, -1.4862966537475586, 0.5412671566009521, 0.45912235975265503, 0.9364859461784363, -1.5830130577087402, 1.8908690214157104, 1.7917288541793823, 1.3476614952087402, -0.6292132139205933, -2.2180919647216797], [-3.5618772506713867, 0.3119894862174988, 1.4857099056243896, 2.47188401222229, 2.2587037086486816, -1.6792857646942139, 1.4720650911331177, 1.1824668645858765, 0.409662663936615, 1.521902084350586, -1.632568359375, -1.7679566144943237, -2.2938055992126465, -2.614008665084839, -1.4524552822113037, -0.605979323387146, 0.6332221627235413, -6.213771820068359, -4.3845062255859375, -2.606992721557617, -4.437155723571777, 4.604308128356934, 2.494846820831299, -1.5265862941741943, -5.740097999572754, 1.706923007965088, 0.5752689242362976, 4.9559173583984375, -5.494935989379883, -2.914193630218506, -0.8811575174331665, -0.4468676447868347], [-4.47418212890625, 2.416630506515503, 0.5319563150405884, 7.77109432220459, -3.8034725189208984, 1.8633522987365723, -1.252293586730957, 0.27904149889945984, 0.24650096893310547, -2.456784963607788, -0.36099404096603394, 0.4791048765182495, -2.0340139865875244, -1.4510111808776855, 5.5762786865234375, 2.7649550437927246, 2.809082508087158, -0.2967977523803711, -1.5295913219451904, -5.505439758300781, -2.214682102203369, 2.8085689544677734, -1.6430200338363647, 4.095191955566406, 0.6613301038742065, 1.1693600416183472, -2.8036997318267822, -3.95756196975708, 2.7376651763916016, 2.353914976119995, -0.07740489393472672, 1.481520175933838], [1.6952736377716064, -0.3234582245349884, 3.1593854427337646, -2.4626407623291016, 2.7856671810150146, 3.496140480041504, -2.3045456409454346, 4.148249626159668, 2.0606274604797363, 1.852858543395996, 6.230940341949463, 1.9386245012283325, 0.8309462070465088, -0.9057161808013916, -4.407606601715088, -3.492042303085327, -0.4008873701095581, 1.6051104068756104, 1.224797010421753, -1.4991148710250854, 1.8992109298706055, -2.285116672515869, 7.031767845153809, -2.79571795463562, 0.3822985589504242, 0.25731584429740906, -2.637202739715576, -4.81995153427124, 5.172137260437012, -1.0304508209228516, -1.7292160987854004, 1.6810765266418457], [2.274233818054199, -1.2930549383163452, 4.478499889373779, 2.2457823753356934, 4.271485805511475, -0.4165326654911041, -3.24210262298584, -0.4723256826400757, -5.2224321365356445, 2.9348318576812744, -1.2714422941207886, -3.8683600425720215, 0.008419515565037727, -2.5825231075286865, -0.4505162835121155, -0.8870478272438049, -1.9483455419540405, 4.671777725219727, -0.6057109236717224, 2.002957344055176, 1.6643192768096924, 2.845641613006592, 0.13877815008163452, -5.271145820617676, 4.7716383934021, 1.8942550420761108, -1.2470325231552124, 0.04108139127492905, 5.6282958984375, -2.3491954803466797, -2.5374755859375, 1.829822063446045], [-3.3639564514160156, -5.487216472625732, -0.18459051847457886, -0.18204109370708466, -3.192917823791504, 0.23836469650268555, -0.07954438030719757, 2.566009998321533, 4.093930721282959, -3.42887806892395, -1.597167730331421, -0.9054148197174072, -0.660452663898468, -0.7571176886558533, -2.1833086013793945, -2.4962172508239746, -1.7050347328186035, 0.8679187297821045, 2.491610288619995, 1.2845985889434814, 6.017012596130371, -2.3821210861206055, 2.6833982467651367, -3.5902254581451416, 5.635617256164551, 1.5587434768676758, 6.632001876831055, 1.2799654006958008, -1.1810734272003174, -0.8606357574462891, -1.167191982269287, -7.319021701812744], [2.9744505882263184, 0.14605757594108582, 1.3364719152450562, 1.200869083404541, 1.9875215291976929, 1.9163864850997925, -4.574985027313232, -3.618196964263916, 4.227992057800293, 2.828399658203125, -1.7478077411651611, 0.9960753321647644, -8.463680267333984, -1.5005626678466797, 4.322508811950684, 2.237128734588623, 1.2325453758239746, -2.87369704246521, 1.6949915885925293, 4.638792991638184, -2.998274326324463, -0.27223023772239685, 0.9562082290649414, 2.286120891571045, -3.7335312366485596, 1.1645296812057495, -1.3529319763183594, -0.8778266906738281, 2.7742788791656494, -3.955387830734253, -1.5330779552459717, -2.362773895263672], [2.582132339477539, 1.5659871101379395, 1.2991331815719604, 3.2464776039123535, -1.1633812189102173, 1.6545770168304443, -3.1583824157714844, -2.5330331325531006, -0.6179069876670837, 1.3367533683776855, -0.13996416330337524, 0.03642497956752777, -3.636410713195801, -7.684108734130859, 0.4035230875015259, 0.84885174036026, -1.9023346900939941, 0.33775216341018677, 0.17958030104637146, 1.2839266061782837, -1.0467534065246582, -4.326409339904785, -1.1378090381622314, 1.941152572631836, 1.6799229383468628, 3.771634340286255, 5.887755393981934, -2.5942864418029785, -0.4912547469139099, 0.7384657263755798, 0.9745421409606934, 1.346390724182129], [1.476317286491394, -2.627210855484009, 3.041828155517578, -1.8223003149032593, 0.9506020545959473, 7.767602920532227, -1.1530178785324097, 5.37252140045166, -3.097820281982422, -1.7889671325683594, 2.6357293128967285, -4.393914222717285, 1.5301475524902344, -4.315459251403809, 2.123603343963623, 3.5073890686035156, 1.7046492099761963, 2.5673584938049316, 1.7492985725402832, 3.089352607727051, -0.5424810647964478, 1.3051490783691406, -1.0579534769058228, -0.3963302969932556, -2.1259663105010986, -1.8902474641799927, -5.657355785369873, -3.0707831382751465, -2.7220475673675537, 1.2144362926483154, -0.363981693983078, -7.312014579772949], [-1.4621825218200684, 0.14148566126823425, -1.0137463808059692, -9.417093276977539, 0.7411528825759888, -1.7718454599380493, -5.385890960693359, 3.015840768814087, 3.1923651695251465, -2.27241849899292, -3.017777442932129, -0.4736211597919464, -6.782781600952148, 2.2248196601867676, 0.6468015909194946, -0.5270960330963135, -1.4722614288330078, 1.6725969314575195, 3.671731472015381, -2.6805074214935303, 0.2786734104156494, -0.2960788607597351, -4.088283538818359, 2.7774057388305664, 0.004363256972283125, 4.507180213928223, 4.697580337524414, 1.0050510168075562, -4.238462448120117, 0.08286002278327942, -1.4228684902191162, 6.805606842041016], [-1.0594788789749146, 0.2901025414466858, -2.9160773754119873, 9.529704093933105, 0.7905101180076599, 0.14924347400665283, 3.5956270694732666, 0.3287973403930664, -10.01111125946045, 0.8331060409545898, 0.27371475100517273, -1.917818546295166, -0.8512399792671204, 2.4767932891845703, -4.298888206481934, -0.5144013166427612, -2.4169044494628906, -0.6534255743026733, -0.14088459312915802, 3.5799078941345215, 3.359116554260254, 3.171384811401367, -0.6190195679664612, -1.8194890022277832, 1.9637200832366943, -1.8193892240524292, 4.482809543609619, -1.1244382858276367, -0.5841730833053589, -3.567263126373291, 3.4400854110717773, 2.331629991531372], [-4.4810895919799805, -0.9530904293060303, 0.0177862998098135, 1.2476933002471924, 3.0553858280181885, -1.201406717300415, 0.9341266751289368, 3.342991352081299, 7.0997514724731445, 0.6390995979309082, 4.44251823425293, -4.350769996643066, 0.4455223083496094, 0.5588893890380859, 1.4366365671157837, 4.471704483032227, 5.795607089996338, 3.6574769020080566, -7.344587326049805, -0.8005766868591309, 4.357427597045898, 7.2319793701171875, -1.038773775100708, 0.5407484769821167, -2.5676510334014893, 0.3955978751182556, -1.5638773441314697, -0.9802683591842651, 3.2552475929260254, 1.6664338111877441, -1.3712184429168701, -0.485784113407135], [1.6515142917633057, 2.70820951461792, 3.472446918487549, 1.770738959312439, -0.8296639919281006, 7.156912803649902, -0.9805635213851929, -0.10259737074375153, -2.2394461631774902, 4.853715419769287, 0.39944154024124146, 1.6898114681243896, 1.3376226425170898, -2.401093006134033, 5.382804870605469, -2.2874977588653564, 1.4297571182250977, -0.872592568397522, 1.460020661354065, 1.107442855834961, -1.322300910949707, -0.6394263505935669, 0.1378040909767151, 2.3436636924743652, 2.3592796325683594, 3.987875461578369, 2.6996474266052246, -0.7858903408050537, 1.9024646282196045, -5.829169273376465, 3.138768196105957, 3.416149139404297], [-0.840621292591095, -2.0728516578674316, 1.2149732112884521, 3.09942626953125, 0.5858880877494812, -2.4099669456481934, -1.9893300533294678, 2.4008970260620117, -0.5826665163040161, 3.443300485610962, 4.9306206703186035, 0.20597463846206665, -0.034757308661937714, 4.578670501708984, 4.226864814758301, 0.353787899017334, 1.9578888416290283, -4.192805290222168, -2.293769121170044, 1.9230211973190308, -4.125599384307861, 2.8774003982543945, -2.2032270431518555, -4.616392135620117, 1.7019249200820923, -2.363771915435791, 0.05840126425027847, -0.7045742869377136, 1.711031198501587, -1.3250417709350586, 4.492395877838135, 5.344766616821289], [1.3586487770080566, 2.9852540493011475, -1.5719459056854248, -0.13383501768112183, -2.6270883083343506, -4.919201850891113, 7.223127365112305, -0.5574814677238464, 3.879567861557007, -1.4635417461395264, 2.6935372352600098, -2.2106430530548096, -1.322949767112732, -1.1909629106521606, 3.361520767211914, 6.576391696929932, -1.3727967739105225, 1.4010584354400635, 3.4308600425720215, 4.911870002746582, -7.969156265258789, 4.985404014587402, 5.641718864440918, -0.10526856780052185, -0.522987961769104, -0.8737962245941162, 0.23508064448833466, 2.252779483795166, 0.052646055817604065, 2.582559585571289, 0.8106086254119873, 3.0245399475097656], [3.404723644256592, 6.078911781311035, 2.5326271057128906, 4.147672176361084, -4.043663024902344, 0.7549145221710205, 4.335188388824463, -0.0322672575712204, 4.061377048492432, 3.98998761177063, 3.2619075775146484, -1.2205307483673096, -0.841687798500061, 1.8287262916564941, -1.985277771949768, 2.5064291954040527, -0.15159295499324799, -2.973794937133789, 0.022799106314778328, -3.097956657409668, -2.2496843338012695, 0.7084841728210449, 3.6061699390411377, 3.253023624420166, 3.1958062648773193, -5.589572906494141, 6.339835166931152, 4.502289772033691, 5.042252540588379, -3.6072874069213867, 3.088892698287964, 0.15065152943134308], [2.259404182434082, -3.4147088527679443, 5.990156650543213, -1.031432867050171, 3.268739700317383, 4.896737575531006, -1.982106328010559, -0.12392456829547882, -6.072989463806152, 0.6167710423469543, 3.068863868713379, -0.17787885665893555, -0.9665098190307617, -1.0531584024429321, -5.302310943603516, 1.030879020690918, 0.19500204920768738, 2.5813870429992676, -1.6581182479858398, -1.8819663524627686, 8.35836124420166, 2.2956440448760986, -1.1681995391845703, 0.2256900668144226, -3.796470880508423, 5.795194625854492, 1.985447645187378, -2.2530312538146973, -0.9341599941253662, -0.5157366991043091, 0.7224850654602051, -0.975725531578064], [3.229793071746826, -3.9125797748565674, 2.604064464569092, -3.3998031616210938, 2.806140899658203, 8.129461288452148, -4.2816901206970215, -5.3448805809021, 3.0041263103485107, 1.5782750844955444, -3.973207712173462, -6.170788764953613, 1.9799394607543945, 2.3388710021972656, 3.057553768157959, 0.8172702789306641, -2.0527305603027344, 3.220248222351074, -6.420858383178711, 2.7433342933654785, 5.806879043579102, -1.181549310684204, 1.2842613458633423, 0.16299553215503693, 2.6206836700439453, -1.2246077060699463, 1.9814282655715942, 2.900575876235962, -0.34346476197242737, 1.4887304306030273, 4.488473415374756, -3.780588150024414], [2.8270015716552734, -3.9739747047424316, 1.1945066452026367, -1.8280482292175293, -1.272532343864441, 1.5561597347259521, 0.3332803547382355, 4.115268230438232, 2.312798500061035, -0.9915410280227661, 3.4769439697265625, 1.290244460105896, 3.8516626358032227, -1.067114233970642, 0.5010029077529907, 4.917430400848389, -0.8236960172653198, 2.3395133018493652, -2.695167064666748, -1.0224453210830688, 2.3345963954925537, -0.4305633306503296, 3.9824938774108887, -3.9023001194000244, -1.0146681070327759, 1.5145256519317627, -0.30747807025909424, 0.5865792632102966, -0.6630003452301025, 4.487675666809082, 4.309486389160156, 0.03790631145238876], [-4.172649383544922, -1.2800357341766357, 0.91314697265625, -4.32761287689209, 3.473247528076172, -3.1674976348876953, -0.9164847135543823, 1.7875332832336426, -5.203787803649902, -0.9314488172531128, 4.396063327789307, 1.5118074417114258, -2.662597417831421, 3.8806629180908203, -3.338932991027832, -0.1088472306728363, -1.8666054010391235, 0.1822754442691803, 2.732804298400879, -1.172227382659912, -3.0590596199035645, 0.6945139765739441, -4.598721981048584, 3.2973599433898926, -2.040412187576294, -1.921309471130371, 1.8236992359161377, 0.48938214778900146, 0.33516985177993774, 1.5334522724151611, 0.2927943170070648, -4.175685882568359], [0.6276792287826538, 3.230516195297241, -0.9594501256942749, 3.605742931365967, -1.7222568988800049, 4.1443281173706055, -3.720024585723877, 1.081446886062622, -0.9682891368865967, -3.8304200172424316, 2.6865127086639404, 6.208109378814697, 1.7518200874328613, -1.1948354244232178, -1.5923473834991455, 0.5270036458969116, 1.063594102859497, -4.311589241027832, -0.7648582458496094, 1.972917079925537, 0.43364065885543823, -3.5502476692199707, 2.947403907775879, -0.07013329863548279, -0.014029981568455696, -0.041605591773986816, 2.9652724266052246, -1.1966303586959839, -0.20457731187343597, -2.0931813716888428, 4.45324182510376, -2.9036240577697754], [1.8006274700164795, 2.2538228034973145, -4.3758320808410645, -4.2942399978637695, 0.49531126022338867, 1.0446830987930298, -1.1352262496948242, 0.2591943144798279, -4.022994041442871, -0.03383389860391617, 3.6046857833862305, -2.2145962715148926, 2.9030935764312744, 4.7286834716796875, 0.8699767589569092, -2.3636136054992676, 9.670369148254395, -0.7416941523551941, 4.276486873626709, -1.1314958333969116, 3.6730260848999023, 3.093935489654541, 3.4452056884765625, -2.8805341720581055, 0.053934093564748764, -0.7495412230491638, -1.7496553659439087, -1.429728388786316, 2.18556809425354, -0.2884935438632965, 2.4274115562438965, 2.3570830821990967], [-6.414263725280762, 6.75065803527832, -2.3720037937164307, 0.782692551612854, -8.02419376373291, 0.46494001150131226, -1.5994625091552734, 2.515852451324463, -4.2416253089904785, -3.525857925415039, 4.228790283203125, -6.9451518058776855, 2.807766914367676, -0.24335035681724548, 2.7703895568847656, -0.18585342168807983, 1.8463754653930664, 2.413712501525879, 0.714081883430481, -2.036318778991699, 3.827110767364502, 0.0757768303155899, -0.5380188226699829, 4.735808849334717, -1.920921802520752, -4.409576416015625, 0.8868054151535034, 3.242443084716797, 1.2763166427612305, 2.443753957748413, 0.7276642918586731, -7.192193984985352], [2.820168972015381, 2.882875680923462, -1.8480422496795654, -0.8628729581832886, 5.02038049697876, 2.9213457107543945, -2.1404061317443848, 2.933424711227417, -0.3675292432308197, 2.0982537269592285, 2.9343297481536865, 1.8536657094955444, -5.440694808959961, 4.076833724975586, 0.11162323504686356, 2.8116657733917236, 1.6528655290603638, -1.4878159761428833, 0.9410679340362549, 2.3537070751190186, 5.310084819793701, -5.83274507522583, 2.0343575477600098, -2.528132438659668, 1.5598243474960327, -2.4559476375579834, -1.7650349140167236, 4.304663181304932, 0.5748166441917419, 1.4155988693237305, 3.7521677017211914, -8.387459754943848], [1.117086410522461, -1.4901080131530762, -10.473505020141602, -4.046039581298828, 1.020193338394165, -6.270196914672852, -7.68889045715332, -4.361185073852539, 5.677047252655029, 1.8068640232086182, -0.26556453108787537, -2.3988003730773926, 0.6789987087249756, 1.794954776763916, -0.6365237236022949, -1.9688054323196411, -0.5012763142585754, 1.3022074699401855, -1.9914326667785645, -0.49441078305244446, -3.7722291946411133, -0.7422728538513184, -1.845862627029419, -2.533782482147217, -2.4073662757873535, 1.0477540493011475, 3.6714582443237305, -2.1888327598571777, -1.1002423763275146, -0.01602543517947197, -2.396963119506836, 5.539316654205322], [-3.960526943206787, 2.623131036758423, -0.944964587688446, -1.7900581359863281, 3.5679430961608887, 4.490617752075195, -2.6244232654571533, 0.15164247155189514, 1.3291890621185303, -2.0153024196624756, -2.7530980110168457, 1.15492844581604, 0.5475928783416748, -1.4425331354141235, 3.926210403442383, 6.462234020233154, -3.7912583351135254, -3.6491167545318604, -0.5432566404342651, 0.455462783575058, 3.812631368637085, 3.7162857055664062, 5.503422737121582, -0.9499305486679077, -2.2594990730285645, -3.64675235748291, -1.7879228591918945, -3.5168871879577637, 4.687076091766357, 0.9933925867080688, 6.049684524536133, 0.10123883187770844], [3.0892181396484375, -2.155299186706543, -1.0742063522338867, 1.0699098110198975, -4.073904037475586, 1.1128946542739868, 1.470301866531372, 2.150869369506836, -3.6228697299957275, -2.974823474884033, 2.996236801147461, -0.6188589930534363, -1.0730417966842651, 1.914186954498291, 0.8828858137130737, 2.477297782897949, 4.177574157714844, 0.5868088006973267, -1.3555138111114502, 1.3373541831970215, -3.5134334564208984, 3.484630584716797, -2.8779330253601074, 0.04921706020832062, 0.08336124569177628, -1.5492172241210938, -4.283384799957275, 6.202930450439453, -4.063199996948242, -1.7075684070587158, 0.7582763433456421, -2.332643508911133], [-7.555585861206055, -1.7364667654037476, 1.7273502349853516, 1.1219793558120728, 3.497359275817871, -4.719679355621338, 2.729339361190796, -0.1961473822593689, 3.667515993118286, 4.127540588378906, -1.6161913871765137, 0.8891587257385254, -5.447251796722412, 1.6332911252975464, -0.04500719904899597, 3.1704344749450684, 0.5671986937522888, -4.886089324951172, 2.593876838684082, 0.8800705671310425, -4.856387138366699, 2.8375895023345947, 3.5060644149780273, 0.09663327038288116, -1.413758635520935, -1.7886600494384766, 0.7238808870315552, 1.2074835300445557, 1.9293062686920166, 0.5542210340499878, -0.46546152234077454, 0.6996716260910034], [-1.589164137840271, 0.17101804912090302, -0.05837466940283775, -0.019380958750844002, 3.0417585372924805, -5.496212959289551, -2.28619384765625, 1.8704612255096436, 2.346196413040161, -7.691576957702637, 0.6590433120727539, 0.4864153265953064, -4.685873508453369, 1.444380521774292, -6.404936790466309, -1.678721308708191, -0.3809909224510193, -3.8150458335876465, 1.4620227813720703, -3.4842443466186523, 3.310764789581299, -6.431707382202148, -1.9121954441070557, -0.85503089427948, -0.8618382215499878, 0.4349263906478882, 1.0465881824493408, 0.014661626890301704, -0.5038360357284546, -4.69716215133667, 1.108388066291809, 4.995915412902832], [-6.4561662673950195, -1.1521331071853638, 5.3770623207092285, 3.912287950515747, -3.982354164123535, -1.8526639938354492, -1.9466054439544678, -2.2805418968200684, -3.9858646392822266, -0.8679555058479309, -5.1589860916137695, 0.2657627463340759, -5.343739986419678, -1.3789546489715576, -3.930084705352783, -4.801289081573486, 4.7156782150268555, -2.339107036590576, -0.6934918761253357, -2.8880937099456787, 8.525025367736816, -0.9005654454231262, -3.42958402633667, -2.6635959148406982, -1.9203053712844849, 1.1195623874664307, -2.0106124877929688, 1.2408461570739746, 0.16856902837753296, 4.040893077850342, -1.9229196310043335, -2.313650369644165], [-8.413623809814453, 5.257896900177002, -3.476841926574707, -1.5894739627838135, -1.95198392868042, 0.9362964034080505, 0.872954785823822, -5.324188232421875, 0.9888803958892822, -2.8183653354644775, 3.982557773590088, -0.12105877697467804, -3.5215680599212646, 1.781947374343872, -7.534032821655273, -0.47360846400260925, 2.1865556240081787, 2.481449842453003, 2.740893840789795, -5.377313137054443, 0.5108065009117126, -3.4182324409484863, 0.6644949316978455, -2.5934362411499023, 1.8845155239105225, 2.0535359382629395, 5.719027042388916, 0.7344346046447754, 0.044670455157756805, 1.4509994983673096, 3.0305213928222656, -2.0023269653320312], [5.82288932800293, -2.3922834396362305, 0.7380324006080627, 1.1515978574752808, -0.7450097799301147, 2.731597423553467, 0.08197257667779922, -1.8386867046356201, -3.915820598602295, 2.5274267196655273, 1.7650648355484009, -1.819993019104004, -4.957054138183594, 3.1652169227600098, -2.8253397941589355, 2.7846295833587646, 0.3973775804042816, -4.178946495056152, -4.766287803649902, 4.015927314758301, -0.01615971140563488, -3.1037914752960205, 3.1680097579956055, 4.346543788909912, 1.2060989141464233, 1.4600963592529297, -2.692847490310669, 2.9844605922698975, 0.9229626059532166, 2.3983967304229736, 2.7724554538726807, -0.9197570085525513], [-3.8022618293762207, -1.3929885625839233, -3.750851631164551, 0.25373777747154236, 0.5644793510437012, 3.8088274002075195, -0.015794023871421814, 0.4396715462207794, 0.7584287524223328, -0.28577548265457153, -1.1468186378479004, 0.1567572057247162, 1.5627777576446533, 4.796546936035156, 1.9514038562774658, 0.46127164363861084, 1.3888046741485596, 2.6875295639038086, 0.941267728805542, -1.7504992485046387, -1.8791717290878296, -2.2284154891967773, -2.749722480773926, -1.9591931104660034, 1.7123620510101318, -3.3096203804016113, 3.848978042602539, 2.5301527976989746, 1.2890729904174805, -0.05846264585852623, 3.9321675300598145, -4.536070346832275], [-3.7047107219696045, -3.230748176574707, -1.1368556022644043, 0.7005796432495117, 2.836142063140869, -0.5004931688308716, 4.269417762756348, -0.7243736386299133, -2.0993494987487793, -6.04901123046875, 0.7674262523651123, -0.5734363794326782, 3.011781692504883, 5.27506160736084, -4.849268436431885, 2.748450756072998, 0.6548930406570435, -0.47310584783554077, 5.465461730957031, 0.3524019718170166, -3.921603202819824, 1.047292947769165, -0.048653677105903625, -4.036287307739258, 3.6259589195251465, 3.0248537063598633, 1.7262682914733887, 0.3891274631023407, -4.594334125518799, 0.6945285797119141, -0.7533268332481384, 1.3777844905853271], [-1.2370734214782715, -0.8378275036811829, 0.5269808173179626, -3.4056954383850098, 0.2080506682395935, -4.0370097160339355, 0.9918632507324219, -0.12532250583171844, -1.7808541059494019, 3.405228853225708, -5.016843318939209, 0.501168429851532, -3.045745849609375, 3.0403928756713867, 2.0537118911743164, -1.456908941268921, -0.7598322629928589, 3.1141209602355957, -1.0653761625289917, -0.4820912182331085, -0.45867007970809937, -0.7572325468063354, 1.5355126857757568, 2.1068668365478516, -1.3100883960723877, -4.010943412780762, -1.3441505432128906, 2.303546667098999, -1.2573920488357544, 0.7474557161331177, 2.521580696105957, 0.4786358177661896], [-1.4597233533859253, -1.833328127861023, -1.0497946739196777, 1.4765187501907349, 4.025448799133301, -2.7996363639831543, 2.82084059715271, 0.7638236284255981, -3.4621031284332275, -1.762643814086914, 1.8845109939575195, -1.6970880031585693, -3.537191867828369, 3.896946430206299, 5.1563720703125, 1.0238083600997925, 0.8422320485115051, -1.57712721824646, 2.4003446102142334, -4.7116522789001465, 1.6184649467468262, 5.560961723327637, 5.4178361892700195, 5.816438674926758, -1.7899221181869507, 0.4368114173412323, -0.26568904519081116, -0.4424533545970917, 2.518315553665161, 3.5406668186187744, 1.4202125072479248, -0.5069417357444763], [-5.645669937133789, 2.527165651321411, 0.5359906554222107, -1.5195741653442383, 2.704317808151245, -2.665966033935547, -1.6183544397354126, -1.6304187774658203, 0.5875453948974609, 1.537752628326416, 1.474823236465454, 3.043473720550537, 1.0867846012115479, 2.248837471008301, 4.749948501586914, 1.5218825340270996, 0.7360300421714783, -3.004582405090332, 5.214811325073242, -2.6803536415100098, -5.521607875823975, 0.5206174254417419, 3.921278715133667, -2.1402695178985596, -3.029763698577881, 1.4509416818618774, 1.693134069442749, 4.2623515129089355, 0.9551594853401184, 4.339221000671387, -0.13264761865139008, -0.8435620665550232], [1.3500896692276, 3.1119112968444824, 3.983680248260498, -7.102927207946777, -3.8680014610290527, -0.24097713828086853, 0.2439591884613037, -4.943179607391357, 1.8487855195999146, 0.543572723865509, -0.8767942786216736, -3.877199172973633, -5.293838024139404, 3.0841403007507324, 3.733336925506592, 1.3712304830551147, -2.985696792602539, 3.6049561500549316, -3.3026504516601562, 4.803650379180908, -3.4086835384368896, -5.188661098480225, -3.842805862426758, -2.4456746578216553, 2.550597667694092, 3.7586488723754883, 0.5325303077697754, -3.7695631980895996, -2.2262563705444336, -0.18040674924850464, -0.19276557862758636, -1.4263134002685547], [-1.3900001049041748, 5.538523197174072, 1.956639289855957, 6.776826858520508, -1.0333689451217651, 1.908355951309204, -1.500842809677124, 3.7672438621520996, 1.668547511100769, -1.7390859127044678, 1.655297040939331, 2.439520835876465, 1.3049951791763306, -5.299347877502441, -2.5574774742126465, 2.261160373687744, 6.735734939575195, 0.11904584616422653, -2.0462236404418945, 0.21852648258209229, -2.3334293365478516, -6.703821659088135, -2.8335587978363037, -3.4049277305603027, 0.29300230741500854, -1.9211043119430542, 0.8545787334442139, -2.7611262798309326, 2.827712059020996, -2.873298406600952, -4.664707183837891, 4.490851402282715], [2.6578235626220703, -5.152067184448242, 0.7719365358352661, 0.46802133321762085, 0.32477378845214844, 0.8321928977966309, -1.7062044143676758, 1.15004563331604, -0.7603771090507507, 2.3542404174804688, 5.422268390655518, 1.6645543575286865, 1.8679001331329346, -5.083057403564453, -5.272442817687988, -1.5948951244354248, 3.8710694313049316, 0.9447914958000183, -1.3446890115737915, 0.9744935631752014, -2.5237865447998047, -1.405495524406433, 0.44023606181144714, -1.0896267890930176, 0.9312366843223572, 2.124901294708252, 0.18017974495887756, 4.134829521179199, 0.4755430817604065, 0.5489490628242493, -5.493515968322754, 6.866726875305176], [0.5538985729217529, 3.6137986183166504, 1.7190039157867432, -0.021895719692111015, -1.1711688041687012, -1.2750314474105835, -1.478562831878662, 0.5570232272148132, -2.0454964637756348, 3.1751394271850586, 0.8608613014221191, 1.3362395763397217, 1.9231319427490234, -0.5296585559844971, -3.6348459720611572, 2.137010097503662, 6.97747802734375, -0.9833623766899109, 3.2967958450317383, -2.5113534927368164, 4.378582954406738, -1.3068575859069824, -4.035299301147461, 0.4314924478530884, -0.26021671295166016, -3.498739242553711, -0.9063900709152222, 0.24007165431976318, 0.7036340832710266, -0.6147421002388, -2.1531357765197754, -1.8652997016906738], [-1.112425684928894, 2.897189140319824, 3.0810437202453613, -3.6276893615722656, -0.8302991390228271, 1.1462024450302124, -1.467634677886963, -4.174815654754639, 2.355617046356201, 1.7017954587936401, 5.33380651473999, 0.6869471073150635, 0.024975411593914032, 2.568554639816284, -1.0406365394592285, 0.838525652885437, 0.5287555456161499, 3.7451071739196777, 0.5329246520996094, -2.98844051361084, 1.3690160512924194, -5.700620651245117, 0.4450278878211975, -2.505824327468872, -0.8340588212013245, -3.2514281272888184, -0.6991519331932068, 1.7455358505249023, 3.8461623191833496, -3.928471326828003, 1.277919054031372, 1.6969473361968994], [-5.460424423217773, 4.255375862121582, 3.594644069671631, 2.0888214111328125, 3.468867778778076, -7.617303848266602, 3.4681553840637207, -4.934971332550049, -0.6371244192123413, 4.329564094543457, -1.7204341888427734, 1.439896821975708, -1.3561877012252808, 1.01935613155365, 1.6169581413269043, 0.9518647193908691, 0.4597129225730896, -3.979726791381836, 0.8930662870407104, 1.2600866556167603, 1.5122089385986328, 2.991943836212158, 2.5516927242279053, 0.43587926030158997, 1.9026565551757812, -3.8869900703430176, -6.520616054534912, 2.331239700317383, -0.9718338251113892, 3.420559883117676, -2.479928970336914, 2.7643191814422607], [0.7044786810874939, -5.90667200088501, -2.3273379802703857, -5.222405433654785, 3.4422249794006348, 4.333808898925781, -0.2066328525543213, 1.9121421575546265, -5.592658042907715, 1.542959213256836, 0.43235141038894653, -3.230433940887451, -1.881015658378601, 1.9622337818145752, -3.7297487258911133, 2.9004368782043457, -0.5607700347900391, -3.7799439430236816, 2.532050848007202, 7.091851234436035, -8.79431438446045, 3.642878532409668, 4.64641809463501, 0.4914959669113159, -4.646956920623779, 2.9382169246673584, -2.710996389389038, -2.2051682472229004, 2.447251319885254, 2.6738290786743164, -6.81560754776001, -4.307868957519531], [1.607187032699585, 1.5559204816818237, -0.18348130583763123, -1.1863830089569092, 0.7643753886222839, -3.234102487564087, -6.494311332702637, -2.067924976348877, 2.5422914028167725, 3.2938032150268555, -3.3329124450683594, -0.6270078420639038, -0.5328104496002197, 4.972884178161621, -3.8636434078216553, -1.382096290588379, 4.2165374755859375, -5.044506072998047, -0.9638838768005371, -0.34119996428489685, 2.1248745918273926, 1.5805611610412598, -0.2356981337070465, 0.24879804253578186, 3.7855358123779297, 0.061480019241571426, -0.5378745794296265, 1.9832310676574707, -2.4394872188568115, 6.028014183044434, -0.3936932682991028, -2.970865249633789], [-3.1863980293273926, 1.4235954284667969, -1.7982962131500244, 1.8480446338653564, -1.3784592151641846, 2.9813179969787598, -0.2711271047592163, 0.46748536825180054, 0.5642364025115967, 3.645789384841919, 0.7404848337173462, 1.7082099914550781, -1.4918389320373535, -0.36528274416923523, -8.445672035217285, -3.5462236404418945, 4.752158164978027, -3.1584384441375732, -3.0556797981262207, 4.851260662078857, 3.0149903297424316, -4.189302921295166, -0.9555636048316956, 2.456984519958496, 3.052504777908325, -2.6805806159973145, 2.225266933441162, -0.9753258228302002, 1.257444143295288, 1.4132448434829712, 1.4589767456054688, -0.3049320578575134], [3.8951797485351562, -0.09702904522418976, -1.5466021299362183, -1.1288509368896484, -0.40843096375465393, 3.962334632873535, 0.6525634527206421, -4.357764720916748, 4.617212295532227, -4.399605751037598, -4.950139999389648, 1.423437476158142, -2.3868062496185303, -1.9455809593200684, -1.982132911682129, -7.444867134094238, 4.323670864105225, -0.049610480666160583, 0.45731908082962036, 5.855461597442627, 4.766987323760986, -2.093902349472046, 2.0923402309417725, -1.8228145837783813, -4.060177326202393, -3.2369027137756348, 1.2701644897460938, -5.558849334716797, 2.120668411254883, 3.983278512954712, 0.9019631147384644, -2.980862855911255], [3.232224464416504, 0.5997143983840942, 0.11805307120084763, -3.021775960922241, -4.470969200134277, 0.7560455799102783, -4.445350646972656, -1.92685866355896, -5.919763565063477, -1.5795232057571411, -2.5810844898223877, 2.9758007526397705, 2.6800222396850586, -8.477531433105469, -3.2424964904785156, 4.3957366943359375, 0.7882242202758789, -3.049800395965576, -1.1289621591567993, -9.189053535461426, -0.053933680057525635, -2.317938804626465, -1.6123276948928833, 0.23552784323692322, -0.4526362717151642, 2.05067777633667, 2.295729637145996, 3.390622615814209, -0.4095362424850464, 5.405971050262451, -0.022423040121793747, -1.2499147653579712], [-2.9058260917663574, 4.323450088500977, -1.0362504720687866, 4.888402938842773, -0.48248910903930664, -2.6772613525390625, 0.4052235782146454, 2.0432257652282715, -0.06410741060972214, -2.0244531631469727, 2.007547378540039, 2.181361675262451, 2.1207735538482666, -2.469327926635742, -0.5839987397193909, 4.302372932434082, 2.630680561065674, 3.252936601638794, -1.4425092935562134, 1.2801196575164795, 3.522777557373047, -2.2751035690307617, 0.9715327620506287, -0.7871831655502319, -0.1603093147277832, -2.9181079864501953, -3.427644968032837, -2.1643152236938477, -2.3570990562438965, 1.1415060758590698, -0.8874561786651611, -0.2103247493505478], [-0.8509863615036011, 3.085602283477783, 1.8890531063079834, 0.17860504984855652, 1.0267367362976074, 3.5457253456115723, -0.9172314405441284, -2.4107983112335205, 0.7360527515411377, -0.9541807174682617, -0.9553601741790771, 1.4560203552246094, -7.833241939544678, 2.2838497161865234, 2.802680015563965, 3.330965995788574, -0.9744657278060913, -0.6869306564331055, -4.010447978973389, 2.946903944015503, 5.20025634765625, 1.07892906665802, 1.3842411041259766, -3.665597438812256, -1.2516541481018066, -3.942023992538452, -1.304371953010559, -4.183399677276611, -4.453469276428223, -1.6461385488510132, -5.267215251922607, -1.431944489479065], [-1.6177115440368652, -4.533572673797607, -6.241966724395752, -2.049844980239868, -3.091081380844116, 0.7040201425552368, 2.252187967300415, 0.44078922271728516, 2.549482822418213, -6.082932472229004, -2.5645060539245605, 0.8018679618835449, 4.408548355102539, 0.985640823841095, 0.35679391026496887, 4.288274765014648, 2.1041176319122314, -1.2840014696121216, 4.77146053314209, -2.2704734802246094, 2.351271390914917, -5.472066402435303, 1.4684069156646729, 4.600183963775635, 6.564519882202148, -1.8099188804626465, 1.2089518308639526, 5.905885696411133, 0.9981930255889893, 4.447664260864258, -4.775282859802246, -1.281862735748291], [-4.217841148376465, 4.347922325134277, -1.3879690170288086, -4.764883518218994, 3.5981192588806152, -3.526444435119629, -0.741085946559906, -2.1913161277770996, -0.1523541361093521, -2.745861530303955, 0.167550191283226, 1.5069737434387207, 1.3083136081695557, 0.6096453070640564, -0.7514961361885071, 4.160820960998535, 5.858897686004639, -3.8947277069091797, 3.093703508377075, -2.5596251487731934, -0.0781167596578598, -0.37028563022613525, 1.5892850160598755, -0.383594810962677, -1.5592525005340576, 0.5688873529434204, -4.535843372344971, -0.7718784213066101, 2.289064884185791, -3.988466262817383, -2.312150478363037, 3.392519474029541], [-2.822521209716797, -3.679375648498535, -0.34391993284225464, -3.4003772735595703, -2.359920024871826, 3.7638721466064453, -4.525581359863281, -0.7092127799987793, 1.2323544025421143, 2.565767288208008, 0.8767774701118469, -1.718037486076355, -0.9429308176040649, -0.6376069188117981, 2.5976409912109375, -1.0180400609970093, -2.645442247390747, -2.1124677658081055, 7.922210693359375, 2.4897775650024414, -3.4246323108673096, -5.829867362976074, -2.118342876434326, -1.251655101776123, 3.9416561126708984, 0.830093264579773, -0.45966365933418274, 4.772794246673584, -2.37587833404541, -0.22450602054595947, 0.5730286836624146, 1.3097413778305054], [-0.23390352725982666, -4.402569770812988, -2.072126865386963, 2.6899447441101074, 0.6806667447090149, -3.0804669857025146, 0.33343571424484253, -5.54816198348999, 1.4430835247039795, -2.5590779781341553, -0.44510388374328613, 2.0198216438293457, -0.17889952659606934, 4.557835102081299, 2.195892810821533, 0.8833827972412109, -2.550600290298462, 1.4275383949279785, -5.537962436676025, -2.882248640060425, 0.29246559739112854, -0.7140158414840698, -0.3149174749851227, -1.4822807312011719, 0.07345539331436157, -4.7002339363098145, -1.9331536293029785, 3.3377232551574707, 0.26630717515945435, 1.4623264074325562, -3.8337454795837402, 1.150031566619873], [-5.511098384857178, -5.130341529846191, -2.0917434692382812, -0.16948387026786804, -1.4831337928771973, -3.0441091060638428, 1.3659296035766602, -1.422902226448059, -3.3492422103881836, 3.959818124771118, -1.086891770362854, -0.011373472400009632, -6.3090362548828125, -3.021252393722534, 5.926496505737305, 0.48005127906799316, -3.444397449493408, -6.856929302215576, -0.8794040679931641, -1.1225075721740723, 0.6334970593452454, 0.8955445289611816, -0.036606259644031525, 5.8761467933654785, -0.3914697468280792, -3.129580020904541, 1.1706923246383667, 2.2530057430267334, 1.5002679824829102, 1.4251340627670288, 1.429762840270996, -6.622118949890137], [0.34559324383735657, 2.356710195541382, 0.5103129148483276, -6.669589042663574, 7.6866230964660645, 2.838695526123047, -2.6026744842529297, 1.6358438730239868, 0.8587292432785034, -1.2416419982910156, -0.3272608518600464, -0.34899258613586426, -1.3650844097137451, 5.096840858459473, -0.2808281481266022, 1.5777583122253418, 0.5691604614257812, 0.37532103061676025, -2.026364326477051, 3.8596572875976562, -1.0218580961227417, -4.3412322998046875, 0.12546557188034058, 1.9352078437805176, 1.1937040090560913, -1.349668264389038, -1.659531831741333, -5.518644332885742, -2.353865623474121, 2.0934548377990723, 1.1901936531066895, -0.7388482093811035], [-0.8654950261116028, -2.4600536823272705, 4.109706878662109, -1.8052071332931519, 0.6431975364685059, 0.5935726165771484, 8.14065933227539, 1.7018232345581055, -6.859683036804199, -3.659757137298584, 1.4334280490875244, 2.852289915084839, 1.4999308586120605, 0.28178414702415466, 3.2048914432525635, 1.3947138786315918, 0.585563063621521, -1.7404217720031738, -1.5119233131408691, 1.4367269277572632, 0.32233670353889465, 0.9722100496292114, 2.9931013584136963, -3.6730663776397705, -1.3653532266616821, 6.85842227935791, -0.09831036627292633, -1.5019559860229492, -0.23367086052894592, 2.9839775562286377, 0.6193661093711853, 2.3191468715667725], [2.4629647731781006, 0.28495490550994873, -1.442252278327942, -2.98759126663208, 2.9714386463165283, -1.3184171915054321, -1.9233559370040894, -0.029588259756565094, -2.237213134765625, 0.7019464373588562, -2.2005491256713867, 1.356698751449585, 3.6060757637023926, 4.387096405029297, 3.807988166809082, 2.80741286277771, -0.45366305112838745, -3.596257209777832, -4.625642776489258, -0.7251766324043274, 3.2609875202178955, -2.192166805267334, 2.122408390045166, -4.983796119689941, 3.1114206314086914, 4.274726390838623, 0.5789765119552612, 0.8987996578216553, 1.9217069149017334, -2.153898000717163, 1.0712318420410156, -2.0800678730010986], [3.028108835220337, 1.141497015953064, -2.325759172439575, -0.6760349869728088, -0.07344355434179306, 0.06366360187530518, -4.69519567489624, 2.4717719554901123, 1.3932710886001587, -0.2854744791984558, 5.1571245193481445, -1.8500722646713257, 0.3103804290294647, 3.22672963142395, 2.545607328414917, 1.1558187007904053, 2.9476521015167236, 5.095318794250488, 0.7416598200798035, -0.9962736368179321, -1.2423981428146362, -0.1801452338695526, -0.14752696454524994, -2.0899922847747803, 3.0107779502868652, -0.7832174301147461, -1.4562197923660278, -0.34916406869888306, 7.15949821472168, 2.035339117050171, -3.424842357635498, 2.5700201988220215], [-1.3985190391540527, -1.66192626953125, 0.2558702826499939, 2.220996141433716, 0.4325578808784485, 2.7376415729522705, -2.5343854427337646, -0.5949710011482239, -0.9614076018333435, 5.1634016036987305, -0.9546892046928406, -4.250992774963379, 4.553475856781006, -1.9891958236694336, -1.9033143520355225, -3.1329479217529297, -3.9927613735198975, 2.320267677307129, -3.567763566970825, 1.1485669612884521, -2.525941848754883, -2.552006483078003, 5.533604145050049, 3.8256750106811523, 1.6930170059204102, 11.611572265625, 1.3639254570007324, 2.1579651832580566, 7.307621955871582, 3.3291540145874023, -0.24034738540649414, -0.6402009725570679]], [[3.2903647422790527, -0.5638503432273865, 0.45112547278404236, 2.063894510269165, -0.0038627395406365395, -0.15576626360416412, 3.129406213760376, -2.195831298828125, -6.282834053039551, 4.58929967880249, 1.6296782493591309, -2.2633066177368164, 4.264714241027832, 0.9570165872573853, -3.0061287879943848, -4.382049560546875, -2.8166937828063965, -2.7372145652770996, -1.2868397235870361, 0.6403208374977112, 3.219329595565796, 0.7039819955825806, 1.3773237466812134, -2.3027265071868896, 3.8627190589904785, -5.114696979522705, 1.4028987884521484, -2.286146879196167, -0.8858221769332886, -0.8190754652023315, 0.8799571990966797, 5.656177997589111], [-6.650306701660156, 1.7224504947662354, -1.3642771244049072, 5.174086570739746, 2.045987844467163, 3.540743827819824, -2.250474452972412, -3.3136868476867676, -0.5648691654205322, -7.292864799499512, -2.5564043521881104, 1.0212688446044922, -6.987041473388672, -1.9493927955627441, 2.772334575653076, -0.9178509712219238, 2.1814076900482178, -1.459313154220581, 2.107433319091797, 2.7272984981536865, 2.6637046337127686, 0.4206221103668213, 3.4824635982513428, 6.1849470138549805, -3.011326789855957, -7.373315334320068, -3.1493825912475586, -0.08944401144981384, 0.6440150737762451, 0.5841981768608093, 6.695753574371338, -0.8497880101203918], [1.0912584066390991, -0.5024089813232422, -3.3130321502685547, 2.8977279663085938, 0.5879632234573364, -3.3525872230529785, 3.846341133117676, -2.3660640716552734, 1.6473150253295898, -0.4527149796485901, 4.694726943969727, 3.095594882965088, 1.7338589429855347, -4.323337554931641, -2.057516574859619, 2.4019036293029785, 1.495308518409729, -3.993164539337158, -4.431994438171387, 3.998645305633545, 1.9955313205718994, -1.4430840015411377, -0.9839107990264893, -2.367072820663452, 3.25730562210083, 0.910358190536499, -3.2090930938720703, 0.39763855934143066, -3.9965951442718506, -4.2146148681640625, -5.0154128074646, -5.7749457359313965], [0.31169816851615906, 1.3454563617706299, -4.4708967208862305, -0.28296804428100586, -1.1525068283081055, -3.8482141494750977, 6.176126003265381, -0.016612330451607704, -2.1134891510009766, 0.2952013313770294, 3.859316349029541, -6.685791492462158, 0.5778899192810059, -0.3165636658668518, -1.3728878498077393, 1.3486380577087402, -7.172153472900391, 0.5787894129753113, 1.902895450592041, 4.254205226898193, 1.1394754648208618, 0.05706696957349777, -0.5939283967018127, -0.16647599637508392, 4.635820388793945, -0.7663669586181641, 3.3305697441101074, 3.6263186931610107, -0.39990097284317017, 0.8558342456817627, 2.282090902328491, -5.042051315307617], [-1.3194797039031982, -1.4985657930374146, -0.7173189520835876, -1.9067176580429077, 2.3332087993621826, -0.6525071263313293, -3.373338222503662, 4.5779924392700195, -1.0319554805755615, 3.8717732429504395, 0.8166280388832092, 3.9817962646484375, -2.666905164718628, -1.2371108531951904, 3.60636043548584, -2.016602039337158, -0.421747088432312, -2.4852237701416016, 0.20460426807403564, -0.44825246930122375, -1.3601268529891968, 1.158219814300537, 2.5276875495910645, 2.671893358230591, 5.623933792114258, 4.182727336883545, 0.22525492310523987, 0.7985575199127197, 3.2266106605529785, 1.3342392444610596, 4.351660251617432, 1.0361233949661255], [-1.1197144985198975, -0.30334100127220154, 4.297691822052002, 2.720674753189087, -1.625932216644287, -0.6920289993286133, -4.432088851928711, -4.197142601013184, -1.3358051776885986, -0.8943352699279785, -0.9274556636810303, -2.129180431365967, -3.0721025466918945, -0.8239821195602417, 1.5640164613723755, -5.909055709838867, -2.474364995956421, -4.586948871612549, -1.4968125820159912, 1.7893121242523193, -1.5061771869659424, -3.498414993286133, 5.393877029418945, 3.6183667182922363, 2.3494534492492676, 2.925931215286255, 2.6076724529266357, 0.20110966265201569, 2.0178613662719727, -1.639245629310608, 1.30861496925354, 1.1466835737228394], [3.1086106300354004, 0.4795737862586975, 2.237330675125122, 5.055685997009277, -1.463536024093628, 2.7475833892822266, -3.5919384956359863, -0.684779167175293, -1.048738718032837, 4.323604583740234, -4.115908622741699, -7.434803009033203, 3.3204898834228516, -0.8112464547157288, -7.979671478271484, 0.9378929138183594, 0.436603307723999, -2.0137829780578613, -1.3925588130950928, 3.4653217792510986, -0.830507755279541, 2.9929332733154297, 0.825682520866394, 2.449807643890381, 4.664673805236816, -0.10078144073486328, 1.0803070068359375, 3.176870584487915, 0.820071816444397, -1.1009923219680786, -0.8587008714675903, 4.652997016906738], [3.186150312423706, 4.703166484832764, -0.9040144085884094, -1.1225966215133667, 1.2333663702011108, -1.34524667263031, -2.025214672088623, -4.256538391113281, -3.777099609375, 5.751996994018555, 0.3748469948768616, 1.3756734132766724, -7.613772869110107, 9.031509399414062, 2.2047290802001953, -0.3218027353286743, -1.6889479160308838, -7.6006364822387695, 3.589308738708496, -1.1567491292953491, -0.504867434501648, 3.3386385440826416, -6.4101948738098145, 1.1316874027252197, -3.058764934539795, -1.9977118968963623, 0.14503906667232513, 0.1585972011089325, -1.2149169445037842, -2.1279306411743164, 3.8294801712036133, -1.7548000812530518], [-2.2609000205993652, -1.9928839206695557, 0.8518382906913757, 0.4541880488395691, 4.139899730682373, 8.568086624145508, -0.3814060091972351, 12.413936614990234, 2.7587316036224365, -0.9484645128250122, 1.9840426445007324, -2.119523525238037, -2.1679086685180664, 4.020359039306641, -3.461636781692505, 2.292090654373169, 3.8442635536193848, -0.4548203945159912, 5.063533306121826, -1.9223995208740234, 3.618572473526001, -1.6350493431091309, 0.1867227554321289, -0.56898033618927, -3.536039113998413, 0.762155294418335, -0.4174325466156006, 0.29434120655059814, 1.8872971534729004, 3.9028267860412598, 1.1371757984161377, 1.4840415716171265], [-0.3678724765777588, -6.852273941040039, 4.516129493713379, 2.874833345413208, -4.040536403656006, -0.41650888323783875, -1.7494323253631592, -1.0873113870620728, 4.4700541496276855, 0.30778104066848755, -1.6887255907058716, -2.1517364978790283, 2.1329703330993652, 1.785930871963501, 6.499167442321777, -0.18058577179908752, 1.1298696994781494, -6.674035549163818, 0.4568827450275421, 3.269864559173584, 1.9532065391540527, -1.8976428508758545, -0.8796882033348083, 0.5584703087806702, 4.653406620025635, -2.3269307613372803, 2.0572285652160645, -2.8813364505767822, -2.708094835281372, 3.1379199028015137, -2.817239999771118, -0.04062722623348236], [2.2047784328460693, -1.4857738018035889, -2.2244935035705566, 1.263687014579773, 1.5505242347717285, 1.1293628215789795, -0.6966643333435059, 7.807747840881348, 0.9256147742271423, -7.06942081451416, 2.8394899368286133, 3.4980955123901367, -1.6671435832977295, 1.5671828985214233, -2.3150668144226074, 5.877656936645508, -5.187948226928711, -0.5609655380249023, 5.042440414428711, 1.5268309116363525, -1.2517499923706055, -5.804360389709473, -0.04563833028078079, -2.342127799987793, -0.43396735191345215, 1.1957601308822632, -0.7854243516921997, -3.9197094440460205, 3.7421751022338867, 5.822713851928711, 5.906833171844482, 0.043110497295856476], [4.0029497146606445, -2.091360092163086, 1.640841007232666, -2.2875046730041504, 1.0442087650299072, -1.2143406867980957, -0.6517544388771057, 0.6863529086112976, -3.3254122734069824, -3.1579737663269043, -0.3764835298061371, 1.4352819919586182, -5.047842502593994, 0.7859351634979248, -3.1997857093811035, -4.184109687805176, -6.560641288757324, -5.768799781799316, 4.185532569885254, 2.6249032020568848, -1.292728304862976, 0.8330509662628174, -5.20496940612793, 0.454262912273407, -3.0286989212036133, -2.3804306983947754, -5.958349227905273, 2.0701181888580322, 4.778948783874512, -2.265524387359619, 2.7365503311157227, 0.08035159111022949], [1.6546071767807007, 3.2890615463256836, -2.9046037197113037, 1.1680469512939453, 1.3166122436523438, 3.3329880237579346, -1.7086341381072998, 1.5854648351669312, 1.923480749130249, 1.3071668148040771, -7.297876358032227, 2.3309640884399414, -0.26286768913269043, -4.264880180358887, -0.5226539373397827, -4.231560230255127, -0.5244636535644531, -0.6413460969924927, 0.10692673921585083, 1.8080213069915771, -5.307135105133057, -0.43781018257141113, -2.5047123432159424, 2.239872694015503, -0.8523010611534119, 1.5090386867523193, -0.5859479904174805, -1.6869590282440186, -0.9701565504074097, -1.1977403163909912, -0.25706833600997925, -3.25430965423584], [0.7268182635307312, 0.3451038599014282, 1.8634798526763916, 0.14817474782466888, -0.5662573575973511, -0.2296472191810608, 2.3456218242645264, -1.1596721410751343, -4.969829559326172, 2.3460941314697266, -0.6638165712356567, 2.9753947257995605, 3.224313735961914, -0.8703230619430542, -0.9047576189041138, 0.6793403029441833, 4.048203468322754, 2.1844491958618164, 0.7492848634719849, -3.234410285949707, -2.0883548259735107, -0.3757023215293884, -4.699812412261963, 3.8431172370910645, 3.394055128097534, -6.373836994171143, -4.786347389221191, 1.619149088859558, -0.5341348052024841, 1.7628810405731201, 3.8772735595703125, 3.4709177017211914], [-4.587104320526123, 0.19468532502651215, -6.143856525421143, -0.7675691843032837, -1.7648019790649414, -1.9363980293273926, 0.3095811605453491, -2.2816243171691895, -0.28726136684417725, 0.8213887214660645, 3.237361431121826, -0.588187575340271, 5.265089988708496, 2.438708543777466, 1.569324016571045, 3.1554055213928223, 1.1495286226272583, -3.6734166145324707, 4.131705284118652, -2.8577682971954346, 4.8067498207092285, 2.643033504486084, -2.9314820766448975, 1.4704439640045166, 4.595165252685547, -0.8331992626190186, -5.975532531738281, -1.2610747814178467, -1.0728189945220947, 0.45249518752098083, -1.4434269666671753, -5.541267395019531], [1.0831232070922852, -4.120677471160889, -0.5606160759925842, -0.26105213165283203, 2.249030590057373, -0.3357856571674347, -3.3102729320526123, -5.369655132293701, 1.531509518623352, 1.6846177577972412, 3.3008241653442383, -0.7916942238807678, 0.3820692300796509, -3.1803741455078125, 4.302910804748535, 0.6616950035095215, 3.775228977203369, -1.0089994668960571, -0.8108391165733337, 0.6165173649787903, 1.4285346269607544, -4.192752361297607, 6.099418640136719, -6.475417137145996, 3.8159689903259277, 1.636998176574707, -0.3437626361846924, -0.9536200761795044, -0.3580309748649597, 0.603733479976654, 4.811100006103516, -0.15236301720142365], [-3.4686827659606934, 4.35320520401001, 0.43968313932418823, 1.734130620956421, 6.027716159820557, 0.4586185812950134, -0.570246696472168, 0.17090977728366852, -5.107893943786621, -0.1476864069700241, 0.40443331003189087, 0.3931772708892822, 0.9267377853393555, -2.895819664001465, 1.8951902389526367, 0.08147574961185455, -2.2689547538757324, 2.2988884449005127, 1.7691655158996582, -5.528378486633301, 3.7690558433532715, 3.2015271186828613, -2.6744236946105957, -0.41721153259277344, 1.6603035926818848, 2.096392869949341, 8.038436889648438, -0.9098955392837524, -1.112520694732666, 1.1490943431854248, 0.3924756646156311, 5.909887790679932], [-0.5137817859649658, 4.649614334106445, 3.13797664642334, 0.2681739926338196, -4.207741737365723, -0.8265348672866821, 2.684166669845581, 2.1638882160186768, -0.2627429962158203, -2.8736915588378906, 1.0704822540283203, -1.7477856874465942, 0.9886701107025146, -0.4311124086380005, 2.114306926727295, -2.661839485168457, -2.7547993659973145, 3.6989121437072754, 3.7301745414733887, 4.534488677978516, -3.151052951812744, 0.660271942615509, 0.5031922459602356, 3.272876739501953, -0.6486560106277466, -0.966300368309021, -0.28346577286720276, -3.0400428771972656, 2.4667108058929443, 1.8831562995910645, -2.6426024436950684, 4.599886417388916], [-0.9562757015228271, -0.5043999552726746, -5.402444839477539, -1.068164587020874, -3.1023178100585938, -1.9792239665985107, -0.21715500950813293, 5.297872543334961, 5.6185102462768555, -1.4686503410339355, 2.048785924911499, -3.3570961952209473, 1.354901671409607, 5.026670932769775, 3.210312843322754, 0.6971592307090759, -2.319610595703125, -4.375665187835693, 2.9474377632141113, -2.6749041080474854, 2.8886566162109375, 1.6028225421905518, 0.5535836815834045, -1.3469326496124268, 1.7556467056274414, -1.5719716548919678, -2.759310483932495, 0.5677174925804138, -2.769627809524536, -2.347156524658203, -1.8012700080871582, -1.7384973764419556], [-2.933651924133301, -5.778049468994141, -5.811802864074707, 2.6042873859405518, -1.7373600006103516, 2.924757480621338, -2.555755138397217, 5.363081455230713, 0.654162585735321, 3.425168991088867, -1.584854006767273, -1.0755959749221802, 5.381725311279297, -0.03619267791509628, 1.2424371242523193, 0.8782210350036621, 6.614767074584961, 0.6861993074417114, 3.247032642364502, 3.574439525604248, -2.41042160987854, -1.3069757223129272, 0.13898585736751556, -1.606881022453308, -2.079627513885498, 4.339850425720215, 3.1510062217712402, 4.8033905029296875, 0.5180315375328064, -2.8176608085632324, -0.11270506680011749, -1.7815428972244263], [5.5148701667785645, 3.6350255012512207, 0.16234137117862701, -0.9575788974761963, -1.2899656295776367, -0.5805764198303223, -1.5328285694122314, -3.097311019897461, -2.3339645862579346, 0.38009244203567505, -0.7663065791130066, -3.538930892944336, -3.6975350379943848, 0.49138346314430237, 5.01099157333374, 0.35892757773399353, -3.831663131713867, -2.7336063385009766, -4.508792877197266, 1.291599154472351, -0.17006653547286987, 3.5432381629943848, 2.046781539916992, -3.3604736328125, -3.1987967491149902, -2.254150390625, -3.2376232147216797, -3.266402006149292, 5.782777786254883, 0.7405255436897278, -1.473491907119751, -4.202010631561279], [0.08325725048780441, -1.3559859991073608, 2.615389823913574, -1.9895415306091309, 2.489178419113159, 3.7331807613372803, -3.58474063873291, -3.405397415161133, 1.4970221519470215, 1.7906162738800049, -3.2523231506347656, 0.2674654722213745, 2.0492825508117676, 2.833493232727051, -0.15610229969024658, -2.5828585624694824, -0.6017411947250366, 0.9349887371063232, -0.21545957028865814, -0.23903314769268036, 1.73380708694458, -0.007354592904448509, -0.7716488242149353, -2.860975980758667, -4.230503082275391, 2.526078224182129, 3.996887683868408, 1.4259965419769287, 1.3866322040557861, -0.9034818410873413, -3.199582815170288, -1.4243876934051514], [3.121201515197754, -2.845080614089966, -3.918517827987671, -2.3750357627868652, 4.006750106811523, -0.6595304012298584, -4.888230800628662, 1.9989320039749146, -3.1979706287384033, 0.47768545150756836, 2.1611244678497314, -5.337227821350098, 2.7452468872070312, -0.5106155872344971, -1.084405779838562, 5.157140254974365, 0.4399827718734741, 6.453847885131836, 2.048490285873413, 1.7485027313232422, -5.462062835693359, 2.424739360809326, -6.697444438934326, 3.481172561645508, 1.2959064245224, 0.07674600929021835, 0.5371776819229126, -2.5560719966888428, -1.0075386762619019, -2.9393622875213623, -0.49028706550598145, 3.6322860717773438], [1.7628660202026367, 0.7939937114715576, 0.9373250007629395, -5.9255900382995605, 2.090913772583008, 1.117393970489502, 3.3439812660217285, 4.356952667236328, 0.8604085445404053, -5.218747138977051, 4.295170783996582, 0.502104640007019, -1.7365503311157227, -3.7370667457580566, 2.486785888671875, 4.161857604980469, -4.037360191345215, -1.441697597503662, 1.800115942955017, 3.2769269943237305, -4.994548797607422, -4.366988658905029, 3.373445510864258, 2.614591121673584, -1.0243518352508545, -0.4369167387485504, -7.350910663604736, 0.11593946814537048, 0.6412944793701172, -1.6368067264556885, 1.9606502056121826, 1.86051607131958], [0.5529554486274719, 0.8360962271690369, 1.6730983257293701, -0.010195968672633171, -1.668633222579956, -2.153164863586426, -6.173004627227783, -1.5996698141098022, 3.110440492630005, -6.705643653869629, 2.291074752807617, -1.247383952140808, -1.9123270511627197, 2.4615750312805176, 1.181673526763916, -0.60526442527771, 1.214078426361084, 5.05430793762207, -1.2345564365386963, 2.8850905895233154, 0.45319899916648865, -0.8336250185966492, 3.081833839416504, -0.9522041082382202, -1.911972999572754, -2.5546627044677734, -3.1941871643066406, 2.4906692504882812, -3.3237247467041016, -4.823380470275879, -0.8380807042121887, -1.293839931488037], [2.0696065425872803, -5.711480140686035, -0.9662209153175354, 6.003539085388184, 0.43539369106292725, -1.0208568572998047, -3.0317916870117188, 0.5846362709999084, -0.2410568743944168, 0.33656731247901917, -5.585976600646973, 2.258321762084961, -1.0388861894607544, -0.5977499485015869, -4.215751647949219, 2.534395217895508, -0.5955865979194641, 0.4807173013687134, 1.636174201965332, -2.163022041320801, -7.3549957275390625, 5.0637288093566895, -6.598272323608398, -1.8477290868759155, -1.821699619293213, 0.7955020666122437, 1.7615947723388672, 6.311001777648926, -1.4076387882232666, 0.22497281432151794, -0.28920021653175354, 1.9580336809158325], [3.4209861755371094, 4.318428993225098, -6.843540191650391, 1.2544116973876953, -0.9058921337127686, 0.4662318229675293, -0.2626904547214508, -0.43399474024772644, 0.48479264974594116, -3.180199384689331, 3.5338830947875977, -0.9571001529693604, -1.6729624271392822, -0.11173982918262482, -0.542937159538269, -7.828815460205078, -0.05831321328878403, 0.32266271114349365, 0.18901139497756958, -0.8298338055610657, -1.3300771713256836, -0.610270619392395, -3.1272411346435547, 0.3079988956451416, -0.7441802620887756, -0.10588165372610092, -4.875507354736328, -2.6518778800964355, -2.5552000999450684, -6.6751604080200195, 0.10574467480182648, -4.095032215118408], [3.1022777557373047, -0.6888442635536194, 0.24041935801506042, -2.4767985343933105, 1.4899482727050781, -1.6313419342041016, 2.2276241779327393, -5.419520378112793, -2.4358489513397217, -0.3071150779724121, -2.4620625972747803, 6.399102210998535, -4.553093910217285, -3.2054100036621094, 2.1626949310302734, 1.2147600650787354, -2.8307321071624756, 0.9269442558288574, 1.039587140083313, -2.494466543197632, -1.4805800914764404, 2.9251790046691895, 2.7738726139068604, -3.3755087852478027, -4.439754486083984, 3.608159065246582, -0.0831371620297432, -3.6742687225341797, -2.089865207672119, -8.406196594238281, -0.5478888154029846, 2.813061237335205], [0.838499903678894, 2.6874964237213135, -2.9123029708862305, -1.5971124172210693, -3.4736247062683105, -1.765364408493042, 1.5781629085540771, -0.2582434415817261, -1.245212197303772, -2.4983999729156494, -1.1454969644546509, 1.157972812652588, 0.4047139883041382, 5.914182662963867, -0.8604692220687866, -2.3363728523254395, 3.1149258613586426, 1.1177870035171509, 2.132002115249634, 2.3568742275238037, -3.543632984161377, -0.916601836681366, 4.006049156188965, 3.8506851196289062, -2.582965135574341, 1.064435601234436, 2.6786749362945557, 0.14790213108062744, 0.28200021386146545, 4.173036098480225, 2.927320718765259, -0.3189278542995453], [-3.287407398223877, -1.2636250257492065, 1.1550266742706299, 2.0532469749450684, 1.1880888938903809, 5.880280017852783, 1.675126552581787, 1.307422399520874, 1.6014107465744019, -1.7682340145111084, 1.3779423236846924, 0.6057850122451782, 1.6795527935028076, 0.36375248432159424, -0.4016903042793274, 0.29007047414779663, -4.074366092681885, -5.883982181549072, 0.176232248544693, -0.11159368604421616, 4.69025993347168, 2.2114059925079346, 0.3282391428947449, -3.351184368133545, 2.9220223426818848, 2.5871047973632812, -2.0267696380615234, 2.80696702003479, 3.9528098106384277, 0.4801675081253052, -8.515697479248047, 0.49975091218948364], [-6.671529769897461, -2.5795743465423584, 6.570487022399902, -3.5183463096618652, 6.155483722686768, -1.5938594341278076, 0.23844951391220093, -4.439953804016113, 2.398193597793579, -2.0793826580047607, 1.6136162281036377, -0.9201171398162842, -2.7056384086608887, -3.631352663040161, 0.7817705869674683, -0.8503044247627258, -0.5533657073974609, -0.9938849210739136, 0.08009282499551773, -0.7709312438964844, -0.891291618347168, -3.7435121536254883, -0.08898437023162842, -1.2329432964324951, -1.1087098121643066, 3.816880226135254, -5.31489896774292, 4.290676593780518, 3.363713264465332, -2.370030641555786, -4.634673118591309, -3.3206212520599365], [5.932611465454102, 0.4204048812389374, -0.7262694239616394, -0.3891277313232422, -5.122251987457275, -2.1551475524902344, -2.815833568572998, 1.5305888652801514, -4.702404022216797, -2.8222198486328125, -0.1183810606598854, -0.750396728515625, -1.3701975345611572, 5.206914901733398, 1.1468673944473267, -2.3413186073303223, 2.159792900085449, 2.3706445693969727, 6.034399032592773, -0.040170032531023026, -2.8218326568603516, -0.6984424591064453, -0.749869167804718, -1.057173252105713, 0.4501842260360718, -0.19376254081726074, 4.643376350402832, 3.6836729049682617, 0.3686506152153015, 0.5209639668464661, -1.5742182731628418, -1.5567858219146729], [0.32550114393234253, 3.295928955078125, 0.49822354316711426, -2.934307098388672, -4.01322603225708, 2.548543691635132, 3.8057780265808105, 0.27185481786727905, 1.5039736032485962, -5.061863899230957, -2.2408337593078613, 0.40312594175338745, 0.48775067925453186, -0.05895525589585304, -6.664040565490723, -0.7210620641708374, 4.990906238555908, 1.2564388513565063, 3.9979262351989746, -3.001082181930542, -2.262634754180908, -0.4653397500514984, 1.8145334720611572, -3.6104049682617188, 1.958418607711792, 2.198699951171875, -7.600241661071777, -1.2830885648727417, -2.402921199798584, 0.7895776033401489, -4.067319393157959, -1.8323160409927368], [-2.2642247676849365, 0.016451146453619003, 3.938472032546997, -1.6644089221954346, -1.3000822067260742, -3.330756664276123, 1.112142562866211, 1.7470595836639404, -0.7808538675308228, -1.333587408065796, -0.9196217060089111, 2.53372859954834, -5.878519058227539, 0.9808777570724487, 0.7535736560821533, 1.742513656616211, -1.7302393913269043, -1.1762332916259766, -1.032863974571228, -3.7036781311035156, 5.304971694946289, 1.7688703536987305, -0.9297679662704468, -0.763310432434082, -0.95942223072052, -2.8975942134857178, -0.3306644856929779, 2.8845856189727783, 0.9508514404296875, 5.159221649169922, -0.548997163772583, -7.180710792541504], [-0.8864818811416626, -0.9131927490234375, 1.144104242324829, 1.8137202262878418, 0.17335373163223267, -5.098724365234375, 3.414620876312256, 0.0678233951330185, -2.0136897563934326, 1.2443941831588745, 0.4972081184387207, -1.4142794609069824, 1.2758512496948242, -2.1898977756500244, -0.4691088795661926, -2.4232430458068848, 1.2605621814727783, 3.2462189197540283, 0.49665266275405884, 0.5401948690414429, -2.2473294734954834, -2.110682964324951, 0.5194263458251953, -2.9271488189697266, -2.970551013946533, 1.7633217573165894, 0.7459461688995361, -2.193143367767334, 1.3928276300430298, 5.157773971557617, 7.548505783081055, 2.806004047393799], [3.1824545860290527, 1.2821495532989502, -0.9991965293884277, 3.6483099460601807, -2.4957079887390137, 7.373212814331055, -2.3187508583068848, -1.2006374597549438, 2.2731220722198486, -7.098232269287109, 0.7276118993759155, -1.5233262777328491, 1.8729984760284424, -0.46167343854904175, -1.4090908765792847, 0.5434682965278625, 0.40541476011276245, 0.9944449067115784, -1.900460958480835, -1.5117762088775635, 2.072084665298462, -0.09299744665622711, 5.357106685638428, -0.010344213806092739, -0.8125158548355103, -1.6673498153686523, 2.92297625541687, 3.5147945880889893, -5.424384117126465, 2.518632411956787, -1.2126657962799072, -2.730684518814087], [4.520312309265137, 0.8628954887390137, 0.9921610951423645, -3.705841064453125, -1.2863985300064087, 5.759360313415527, 0.43401867151260376, 0.6759851574897766, -0.5391724109649658, 1.411043405532837, -4.804659843444824, 4.072865009307861, 1.6463295221328735, -3.735063076019287, -3.1689512729644775, -2.9821853637695312, -1.247818946838379, -0.99632728099823, -2.3975491523742676, 4.2671918869018555, 3.5745255947113037, 3.5453672409057617, 2.2896902561187744, 2.2152698040008545, -2.7210497856140137, 1.0579099655151367, 4.483217239379883, 0.6637357473373413, 1.1006240844726562, -4.338956832885742, 0.5049685835838318, -3.651517152786255], [1.7008941173553467, -2.0223751068115234, -0.8556536436080933, 2.8382391929626465, -0.5087239742279053, -0.8584089875221252, 3.1094892024993896, 1.0460833311080933, -0.5628471374511719, 5.394824504852295, 1.3894537687301636, 0.028664592653512955, -3.8334453105926514, -6.33264684677124, -1.7085638046264648, -0.3454148769378662, -0.35451486706733704, -5.795241832733154, -3.3883233070373535, -0.8223854899406433, 1.3036441802978516, 4.046328067779541, -2.810178518295288, -5.746524810791016, -1.7177921533584595, 3.726567506790161, -4.063220977783203, 0.05765199288725853, 4.61817741394043, 1.6661431789398193, -1.2679425477981567, 3.276212453842163], [-1.1740052700042725, -0.771416187286377, -4.982157230377197, -2.622044801712036, -0.5787121057510376, -4.027279853820801, 2.989192485809326, 0.6152824759483337, -1.761815071105957, -0.20224629342556, -0.17830964922904968, 0.5648844242095947, -1.8598276376724243, -0.15397068858146667, -1.6871259212493896, -1.682485818862915, -7.014791488647461, -2.193840265274048, 0.3590494990348816, -0.33155837655067444, -1.588108777999878, -2.571751356124878, 7.566374778747559, 1.5856071710586548, -4.482068061828613, -1.4875850677490234, -1.9914422035217285, 2.7681849002838135, 0.5375423431396484, 4.240602493286133, 0.7239037752151489, -0.6205129623413086], [4.84597635269165, -1.9988139867782593, 0.6144818663597107, 1.2013623714447021, -0.3389504849910736, 0.5761582255363464, 1.0404222011566162, -0.70894455909729, 3.1552562713623047, -2.518399238586426, 0.896425724029541, 1.7860193252563477, -0.20031902194023132, -0.2255471646785736, -3.3131103515625, 2.763559341430664, -1.075673222541809, 1.0277159214019775, 0.6386021375656128, -3.452444553375244, 2.2075376510620117, -4.870424270629883, -0.5058107376098633, -1.0393750667572021, -1.9758764505386353, -6.995602607727051, -1.3937323093414307, 2.8880276679992676, 2.4739034175872803, 1.9588826894760132, -1.6158767938613892, 1.4939451217651367], [-3.313218355178833, 2.2569353580474854, -0.36604511737823486, -7.3357086181640625, -1.541792869567871, 0.7586270570755005, 3.618600368499756, -3.407236099243164, 2.6853466033935547, 2.3610053062438965, 0.6527475714683533, -0.7704405784606934, 1.04574453830719, -3.7745110988616943, -4.788258075714111, -3.2143893241882324, 0.2677120566368103, 3.8175466060638428, -1.983534336090088, -2.6496853828430176, -4.725131988525391, -4.847792625427246, 1.1818232536315918, -2.8284919261932373, -0.4067894518375397, 1.944324254989624, -2.3955650329589844, 5.618995666503906, 0.5206436514854431, 4.843045711517334, 2.410201072692871, 1.983628749847412], [4.16593074798584, -2.190027952194214, 2.76223087310791, -0.7043013572692871, -1.4361557960510254, 1.1192290782928467, 0.2896820902824402, -3.1788439750671387, -1.0254740715026855, -0.31685322523117065, -4.827019214630127, -1.492905616760254, 3.782390594482422, 4.002876281738281, -1.1686222553253174, 1.7437766790390015, 0.5113071203231812, 0.516400933265686, 0.47283974289894104, -0.9939624071121216, 1.2979609966278076, -1.2148942947387695, -3.34055233001709, 2.1804776191711426, 0.5330634117126465, 3.8752520084381104, 0.3779962956905365, 0.14519362151622772, -2.0791091918945312, -1.2843586206436157, 1.4514765739440918, 4.098219394683838], [0.8638553619384766, 1.843491554260254, 1.7797291278839111, -0.3870113492012024, 1.359301209449768, 4.982675075531006, 0.5275406837463379, -6.259612083435059, 3.2567920684814453, -0.23460960388183594, -3.042222023010254, 0.3532126247882843, -5.253091812133789, 4.903998374938965, -1.5137794017791748, -3.2187447547912598, -0.6317071318626404, 2.3017141819000244, 0.46871423721313477, -3.2674198150634766, -0.9392449855804443, 2.3638689517974854, -2.628098487854004, 0.34645533561706543, -4.592992305755615, -2.294508695602417, -1.874131679534912, -1.4998232126235962, -1.7547343969345093, -1.9045218229293823, -4.845373153686523, -0.30605995655059814], [4.106274604797363, 2.6312222480773926, 0.9459034204483032, -0.33695948123931885, -1.2245813608169556, 0.381756067276001, -0.2050502747297287, 3.5999715328216553, 2.9411799907684326, 6.493328094482422, -4.919089317321777, 0.8198509812355042, -0.35116174817085266, -6.917065620422363, -3.993734121322632, 0.8409100770950317, 6.404303550720215, 5.892705917358398, 0.2271493375301361, -6.269172191619873, 0.9846384525299072, -1.1894190311431885, 0.6720781922340393, 0.43543481826782227, 2.734302520751953, -8.032770156860352, 3.359570026397705, 1.264962911605835, -1.3489184379577637, 5.16326904296875, -1.2439473867416382, -1.101783037185669], [-0.6560198664665222, -0.3556373715400696, 2.172746419906616, 1.0015467405319214, 1.289528727531433, 0.4885365962982178, 3.128011465072632, -1.847550630569458, -2.016124725341797, -3.544732093811035, 3.609348773956299, 2.7702994346618652, 1.6053917407989502, 2.0449695587158203, -2.8273708820343018, -0.7148892879486084, 1.2450817823410034, -0.6409703493118286, 0.8887140154838562, 0.581592321395874, 0.8754964470863342, -2.7788968086242676, 1.977405309677124, -5.277690887451172, -3.255375385284424, 0.3620067834854126, 0.4297793507575989, -5.463973522186279, 1.5698668956756592, -1.2939611673355103, 2.5849926471710205, 0.7165530920028687], [-2.102597713470459, -2.6822850704193115, 0.33148878812789917, 0.990633487701416, -1.8069171905517578, -1.5378289222717285, -0.5822338461875916, 3.0188608169555664, 2.170266628265381, -4.615019798278809, 1.9120376110076904, 2.683570146560669, -2.8108863830566406, -1.8201758861541748, 3.851285219192505, 3.2938380241394043, -4.496986389160156, 0.9199410676956177, -3.357647180557251, -2.030515670776367, -5.636829853057861, 5.184055328369141, 0.6547659635543823, -2.0761020183563232, 3.4789257049560547, -2.6160972118377686, 0.16587495803833008, -1.007174015045166, 3.12607741355896, 1.7528711557388306, -1.0572930574417114, 3.3609132766723633], [3.916114568710327, 0.34665989875793457, 1.5639352798461914, 3.3473386764526367, -4.086630821228027, 0.21691319346427917, 0.41099995374679565, -0.8489788770675659, 0.7155056595802307, 0.1966930478811264, -6.208574295043945, 0.9796015024185181, 4.494277000427246, -1.200883150100708, -1.5605993270874023, 0.024042252451181412, -1.615372657775879, -1.2520861625671387, -1.6432857513427734, -0.2685389518737793, -5.613016128540039, -4.473854064941406, 4.324051380157471, -3.1192588806152344, -0.004999768920242786, 0.4989846348762512, -0.02052227407693863, -0.5115543603897095, -0.59282386302948, -3.474355697631836, -1.1161974668502808, 1.2940646409988403], [-3.84342885017395, 5.47988224029541, -2.509000301361084, 0.7642205357551575, -6.717408180236816, -3.384153127670288, -1.7635889053344727, 5.971174240112305, 0.4742768406867981, 3.271113872528076, -0.8267710208892822, 4.292666912078857, 6.058311462402344, -5.957840442657471, -1.6515257358551025, -0.9622644782066345, -3.7092747688293457, 1.8079874515533447, 0.027455437928438187, 2.9042325019836426, 1.3256803750991821, -0.751899242401123, 2.5045549869537354, 1.4707335233688354, -3.9136905670166016, 0.20377042889595032, -1.6395494937896729, -2.7029356956481934, 2.547786235809326, -2.6302590370178223, -2.6017675399780273, 7.816988468170166], [-2.8316617012023926, -2.551577568054199, -2.476323127746582, -2.290236711502075, 2.8143558502197266, 1.4371849298477173, 3.3829572200775146, 0.9263297319412231, -1.5905742645263672, 0.04131478816270828, -3.6354451179504395, 2.914632797241211, 3.8841824531555176, -6.824840068817139, -2.864492177963257, -0.7732290029525757, 1.468306541442871, -2.2350964546203613, -2.5159244537353516, 0.2901201844215393, 0.6093471050262451, -5.631241798400879, 2.9538979530334473, -4.385168552398682, -3.4924569129943848, -1.6794966459274292, 1.4530236721038818, -1.338035225868225, 5.906443119049072, 1.3119316101074219, -0.8086130619049072, 2.7248027324676514], [-1.5462393760681152, -1.7770832777023315, -1.1538681983947754, -3.3367116451263428, -3.0676679611206055, 4.462741374969482, -2.250720500946045, -3.0301036834716797, -4.23920202255249, -1.1986243724822998, 1.8716872930526733, 0.31049713492393494, -2.2748405933380127, 0.7289352416992188, 0.9725220203399658, 2.061169147491455, 1.9864881038665771, -1.7587151527404785, 1.670051097869873, 4.472877502441406, 1.495411992073059, 6.788877487182617, 0.2321476936340332, 0.749569296836853, -7.4556121826171875, -0.8106206655502319, 1.2483928203582764, 0.3836040496826172, -5.548373699188232, -1.396199345588684, 6.243528366088867, 5.9098334312438965], [1.652679204940796, -0.1301998496055603, 0.5435197353363037, 4.240243434906006, 2.2561755180358887, 2.368607997894287, 4.059374809265137, 1.4300791025161743, 5.103100299835205, 0.25260987877845764, -1.1590436697006226, -3.987391233444214, -5.0020670890808105, -5.994488716125488, 1.9926016330718994, 1.2345507144927979, 1.0387637615203857, -0.05107063800096512, -1.6191027164459229, -4.594466686248779, -0.8189547657966614, -4.7694220542907715, -1.6331148147583008, 0.15215569734573364, -2.966865301132202, -1.0488646030426025, -4.4429521560668945, -0.9729839563369751, 0.3479612171649933, -3.688828945159912, -0.2023504674434662, 3.337221145629883], [-0.8788944482803345, -3.836878538131714, -2.816227436065674, 4.389126777648926, 1.5250400304794312, 0.0460735559463501, -1.9431984424591064, -1.8297858238220215, 0.22296202182769775, 4.956997871398926, -0.7482689619064331, -0.9232970476150513, 3.8792552947998047, 3.3537116050720215, 1.2339776754379272, -2.456204414367676, 4.173399448394775, -2.2630491256713867, -3.4686989784240723, 0.7059146165847778, 1.645932674407959, -1.716285228729248, 1.959336519241333, 0.9830617904663086, 3.9665443897247314, 3.221618175506592, 0.39997565746307373, -5.830474853515625, 3.285038709640503, -0.9467738270759583, 0.6804476976394653, -1.302294373512268], [-2.3210055828094482, 2.2586212158203125, -2.7479684352874756, -5.765928745269775, 1.2340779304504395, -1.030047059059143, 0.9943069219589233, -2.1873741149902344, 3.271009922027588, 0.7275252938270569, 2.7865495681762695, -3.9692394733428955, 3.9738030433654785, 1.001962661743164, -1.8719902038574219, 0.09792298078536987, 1.628488302230835, 3.6181702613830566, -4.28515100479126, -2.69126033782959, 2.446986675262451, 2.4852828979492188, 5.529113292694092, 5.0629353523254395, 3.2687809467315674, -0.61362624168396, -4.380805969238281, -0.7630141973495483, -3.2338356971740723, 3.272975206375122, 2.6328482627868652, 4.989017009735107], [-2.475299835205078, -2.757814645767212, 6.816717624664307, -1.1615955829620361, 2.380110263824463, -1.4223809242248535, 1.08688485622406, -0.623546838760376, -0.6047686338424683, -2.086225986480713, -5.237143516540527, 2.5028882026672363, -9.954166412353516, -1.6791682243347168, -0.4617842137813568, -3.7431063652038574, -0.9828286170959473, -3.882457733154297, -1.3187685012817383, 2.866068124771118, 1.6465892791748047, 1.3384051322937012, -0.881420373916626, -0.4848189353942871, 2.0136351585388184, 2.8836333751678467, -5.669347286224365, 2.5752172470092773, -3.670477867126465, 2.1766276359558105, -6.572690963745117, 0.8316457271575928], [-1.4090436697006226, -4.126187801361084, 0.6373989582061768, -0.13979531824588776, 5.001772880554199, 3.6201963424682617, -0.5790808200836182, -2.3227405548095703, -2.204822540283203, -2.1582751274108887, -1.014288306236267, 1.8667476177215576, -5.055936336517334, -1.0423539876937866, -2.5503501892089844, -2.144327163696289, -2.7873339653015137, 1.57863450050354, -4.8711771965026855, -2.460982322692871, 2.626763105392456, -4.258568286895752, -0.39286208152770996, -3.159986972808838, -1.4438228607177734, -2.2378578186035156, 0.22060555219650269, -0.9253699779510498, 0.8550151586532593, 1.8994907140731812, 0.8498774766921997, 0.23347927629947662], [1.2116931676864624, 0.213701069355011, 2.3203744888305664, -1.57417631149292, -0.3487522006034851, -1.8112154006958008, 1.5570091009140015, 1.157213807106018, -0.6235005855560303, 3.203148365020752, 2.0483341217041016, -2.7379610538482666, -4.366369724273682, -4.5596747398376465, 0.34303513169288635, 4.5486602783203125, 3.288846969604492, 1.2218647003173828, -4.260782241821289, 1.2267502546310425, -5.196572303771973, 1.262808918952942, -1.919090986251831, -7.296245574951172, -4.235445976257324, -2.0334699153900146, 2.445507049560547, -2.575572967529297, -0.7467361092567444, -1.1001100540161133, 0.966957688331604, 0.3882347345352173], [0.7492550015449524, -1.7590022087097168, 1.122572898864746, -6.430196762084961, -4.548074722290039, -1.9179039001464844, 4.948665142059326, -0.692420244216919, 1.8055896759033203, 5.232551574707031, 0.09193883091211319, -0.8671338558197021, 2.8853042125701904, 4.263826370239258, 0.9462568759918213, -2.779383659362793, 2.5080132484436035, -4.656192779541016, -1.2192944288253784, -5.504154205322266, 0.7510684132575989, 5.171817779541016, 0.6296347975730896, 2.3768310546875, -0.5850507616996765, 0.994924783706665, -2.0274367332458496, 0.8346871137619019, 3.8674912452697754, -2.018962860107422, 5.03216552734375, 1.1028904914855957], [4.4735565185546875, 2.5221147537231445, 0.8031672835350037, 0.31859809160232544, -2.361398458480835, 4.506828308105469, -0.45694267749786377, -1.5094072818756104, -5.037637710571289, -1.811229944229126, -0.8534156084060669, -3.2947301864624023, 0.8288257122039795, -2.922638416290283, -0.5749455690383911, 2.8015830516815186, -1.2884173393249512, 0.12462953478097916, -5.892862319946289, 1.5465567111968994, 1.5661756992340088, -0.2463454008102417, -2.405893325805664, -0.15477357804775238, -3.671154022216797, -0.9299814701080322, 2.054741382598877, -3.256049394607544, -1.223542332649231, -0.021780885756015778, 2.4840056896209717, -0.334261029958725], [1.527808427810669, -3.625976085662842, 3.314329147338867, 3.0310885906219482, -2.239880323410034, -2.8361587524414062, 4.105456352233887, -2.8424012660980225, 2.765275001525879, 0.7111400961875916, 3.3651719093322754, 1.516817331314087, 0.6231563091278076, 2.5203824043273926, 2.76340389251709, -4.3713531494140625, -0.49054551124572754, -1.9638481140136719, 2.2506392002105713, 0.3853883743286133, 5.684413909912109, -4.506003379821777, -3.379591464996338, -1.5008294582366943, 1.0355095863342285, 0.7017317414283752, 2.826028347015381, 0.937312126159668, 2.313227891921997, -0.6706559062004089, -0.28533628582954407, 3.109992504119873], [3.8683512210845947, -0.3147827386856079, 3.0090172290802, -1.8807995319366455, -3.496791362762451, 2.001420021057129, -1.2791775465011597, 1.1586341857910156, -3.371058225631714, -1.779853105545044, 1.2986814975738525, -1.0369200706481934, 2.2678322792053223, -5.014707565307617, -2.165968418121338, -5.1226654052734375, -2.560619354248047, -1.859398603439331, -0.49704235792160034, 2.2205758094787598, 1.3843278884887695, 2.5753443241119385, -0.6655532121658325, -3.0695507526397705, 0.4104905426502228, -6.6530351638793945, -2.1436235904693604, -0.4050082564353943, -7.804749488830566, 3.068230152130127, -1.843539834022522, 0.7630389332771301], [0.018354302272200584, -2.4493162631988525, -1.9133951663970947, 4.207622528076172, -1.652269721031189, -2.710789918899536, 0.4131619334220886, -2.0152015686035156, 0.013753781095147133, -0.15412668883800507, 2.038705587387085, 5.730133533477783, -0.8401800394058228, -0.14558102190494537, 4.5748090744018555, -0.052407827228307724, 4.871124267578125, -1.7244482040405273, 0.6121120452880859, -2.2246079444885254, 2.914839267730713, -1.0825320482254028, -0.03521169349551201, 0.49876508116722107, 2.8623204231262207, -5.076140880584717, 2.509369373321533, 1.784782886505127, -2.3227100372314453, -3.8708324432373047, -0.9265738725662231, 0.36357027292251587], [6.2593674659729, 1.825263261795044, -1.2433345317840576, 2.136113166809082, -3.224679946899414, 1.520721673965454, 2.3125977516174316, 0.0726131945848465, 6.1020283699035645, 3.425916910171509, -4.755827903747559, -1.3814165592193604, -0.07845832407474518, -0.1490752249956131, -1.9336134195327759, -0.8433825373649597, 0.5052100419998169, -1.4450756311416626, -1.4102256298065186, 1.141874074935913, 1.848020076751709, 2.8851945400238037, -2.1600141525268555, 3.4178173542022705, -0.6985630393028259, -1.580189824104309, 3.7023746967315674, -1.160668134689331, 3.550563335418701, 3.1404638290405273, -0.7127300500869751, 2.610605239868164], [-1.156746745109558, -9.707162857055664, -1.8544034957885742, 4.676395893096924, 0.8845130205154419, 2.339242935180664, -2.3931994438171387, -7.076099395751953, 0.93438720703125, -0.8526914119720459, 3.9181032180786133, 4.083014488220215, 0.8035006523132324, -1.325478196144104, -0.19366925954818726, -0.5398297905921936, -1.4854962825775146, 3.712594985961914, 1.6437392234802246, 3.801476240158081, 2.9653804302215576, 1.8224537372589111, -10.4910306930542, 3.271906852722168, -0.906217098236084, -2.3461809158325195, 3.695542812347412, 7.164377212524414, 0.5151950120925903, 2.8769569396972656, 0.21130821108818054, 2.134949207305908], [-2.681546926498413, -2.3783342838287354, 0.20670101046562195, 7.5930047035217285, 1.1445101499557495, -5.444999694824219, -0.18089428544044495, -1.7536264657974243, -0.20882001519203186, -1.0404207706451416, -5.668505668640137, -1.0678069591522217, -5.623828887939453, 2.0031137466430664, 3.44374418258667, 2.8969902992248535, -4.933149337768555, 0.04934724420309067, -4.867980480194092, 2.1171302795410156, 1.2733436822891235, 0.7932144403457642, -0.44656091928482056, 1.9759944677352905, -7.899415016174316, 3.7527942657470703, 2.529144763946533, 0.7553856372833252, 1.9434623718261719, 0.7816382646560669, -0.23649193346500397, 6.087430000305176], [5.311832904815674, -3.5872786045074463, 0.6108705997467041, -2.8616018295288086, 1.1752015352249146, -2.075404167175293, 1.660576581954956, -3.3544013500213623, -3.1641993522644043, 3.517153263092041, -1.2124749422073364, 0.59221351146698, 2.8710978031158447, 2.411881923675537, -0.8337991237640381, -0.28988000750541687, -1.429565191268921, 6.380434513092041, 1.9746983051300049, 5.097237586975098, 2.6203525066375732, 0.8058537244796753, -5.597491264343262, 1.4968215227127075, 2.168142080307007, -3.104649066925049, -4.569060325622559, 2.4219727516174316, 0.9305320978164673, 2.769680976867676, 2.98987078666687, -5.33155632019043], [-2.930920124053955, -0.2696021795272827, -4.191467761993408, 0.5614674687385559, 2.002089738845825, -4.5096435546875, 1.0069119930267334, -3.5088348388671875, -3.815335750579834, -0.9231146574020386, 0.24513375759124756, 0.30151379108428955, 8.493142127990723, 4.632897853851318, -1.3091392517089844, -5.021050453186035, -1.745019793510437, 1.3488757610321045, 0.3980783224105835, -0.497944176197052, 5.569580078125, -0.9670016765594482, -2.8135976791381836, -5.167328834533691, 1.5257236957550049, 3.9632925987243652, -3.3315324783325195, -4.942685604095459, 0.8533211946487427, -1.902624249458313, 0.7540696263313293, -4.919212818145752], [-2.981807231903076, -1.7337498664855957, 0.6870107650756836, 3.897829532623291, -2.4909884929656982, 1.371370792388916, 0.6651296615600586, -4.262661933898926, -0.29164645075798035, -1.3488500118255615, 5.617791175842285, -0.44880837202072144, -0.16497009992599487, 0.9221783876419067, 1.74147367477417, -1.4296290874481201, -3.4742298126220703, 0.47519832849502563, 3.3584976196289062, 2.748424530029297, 1.7035386562347412, 0.034693192690610886, 0.6938714981079102, -5.160898685455322, -5.987144947052002, 0.6679811477661133, 4.051751136779785, 2.098046064376831, -4.60524320602417, 4.145544052124023, 5.992504119873047, 3.995260715484619], [1.835097074508667, -5.154715538024902, -2.269676685333252, 3.41859769821167, -0.7416133880615234, 3.765047550201416, 0.9117056131362915, 3.2332868576049805, 3.204991340637207, -0.9108314514160156, -0.15284889936447144, -7.610808849334717, 0.5505145192146301, -0.03646919131278992, -6.023968696594238, -1.9174742698669434, -2.7699975967407227, 2.1337976455688477, 0.3902720808982849, 3.4752306938171387, -0.7105640172958374, -2.0796830654144287, -4.309770584106445, -0.13113099336624146, 0.9786128997802734, 5.569043159484863, 0.06706666946411133, 2.2395033836364746, -0.43834441900253296, 0.8510596752166748, 0.9225219488143921, 3.827047348022461], [0.011466900818049908, 0.187049999833107, 1.820417881011963, 0.3905384838581085, 4.026963233947754, -1.2828395366668701, -1.589573860168457, 3.490595817565918, -0.7958236932754517, 4.224023818969727, 2.1105728149414062, 7.959936618804932, 4.4419708251953125, 2.64094614982605, -1.500382661819458, 3.558424472808838, -2.1109936237335205, -5.961085796356201, -0.6350964307785034, -4.038801670074463, -0.839530885219574, -1.5737063884735107, -0.24620778858661652, 0.6586419343948364, 2.0957834720611572, 5.119449615478516, 3.783048152923584, 0.8516601920127869, -2.726470947265625, 1.132891297340393, 2.737830638885498, 0.8494901657104492], [4.430607795715332, -4.330771446228027, 2.126199245452881, -2.05265474319458, -2.7147748470306396, 6.075852870941162, -0.5135796070098877, -2.9503159523010254, -6.862771511077881, -1.0579955577850342, -2.5355594158172607, 2.7795827388763428, 1.743156909942627, 2.4322190284729004, -0.9588629007339478, -1.0076713562011719, -0.8367244005203247, -3.7677154541015625, -3.570718288421631, 1.8619472980499268, -3.0785715579986572, 1.1712875366210938, 1.1367727518081665, 1.32212495803833, -1.1905808448791504, 3.095031261444092, 1.7241034507751465, 0.7913730144500732, -1.1682268381118774, -2.245972156524658, -3.7056007385253906, -0.554519534111023], [-2.8996410369873047, -2.384805679321289, 1.7208588123321533, -2.6543641090393066, -0.653279185295105, 0.13100475072860718, -0.2521182894706726, -2.9557127952575684, 6.257291793823242, -4.829813480377197, 0.27713489532470703, 1.2630290985107422, 1.1920713186264038, 0.027627568691968918, 6.203511714935303, -0.6328571438789368, 1.2703150510787964, 5.407212257385254, 5.754988670349121, -0.5813258290290833, -1.5017101764678955, -6.215958595275879, 3.9224703311920166, -4.263265609741211, -1.4887712001800537, 4.777743816375732, -1.4459255933761597, 1.7945351600646973, -0.3587675094604492, -2.8445229530334473, 1.236451506614685, -3.1106796264648438], [-2.688852310180664, 2.4506943225860596, -7.6354079246521, -1.548295497894287, 1.070608377456665, 2.1565585136413574, 3.937896966934204, -0.37400227785110474, -3.89359712600708, 2.9329135417938232, 2.2621285915374756, -1.2127928733825684, -0.2357645183801651, -2.809511184692383, -2.536118507385254, 1.999913215637207, 6.72501277923584, 2.3199052810668945, -1.4941478967666626, 0.6876978278160095, 1.5288108587265015, 3.218801975250244, -4.363258361816406, 4.045206546783447, -0.5616657137870789, -5.526285171508789, 8.133378982543945, 2.452270984649658, -0.8223165273666382, 0.5707404613494873, 0.3071235418319702, -0.49102240800857544], [-3.3815412521362305, -1.207955002784729, 1.2350208759307861, -2.5387892723083496, -0.6182974576950073, 3.9372432231903076, -1.872344970703125, -1.40345299243927, 0.07762574404478073, -2.96148943901062, 4.185368061065674, 2.143075942993164, 3.620692729949951, -1.868667483329773, 1.9505006074905396, -1.9032182693481445, 2.884925603866577, -4.41362190246582, 0.29162144660949707, -1.734897494316101, 5.040663242340088, -6.452352046966553, -1.9979428052902222, 0.7766276597976685, -6.4269561767578125, -5.278428554534912, 0.5635995864868164, -1.9429235458374023, -1.6651722192764282, -0.46359968185424805, -5.397531509399414, -0.4813162386417389], [-1.8935818672180176, 2.126542091369629, 3.341815948486328, -0.7968279719352722, 1.593750238418579, 0.44857093691825867, 2.4289941787719727, 1.9711321592330933, 6.422422885894775, 0.26739948987960815, -3.3381507396698, 0.08988741040229797, 0.43668466806411743, -0.9890135526657104, 2.8278112411499023, -7.590947151184082, -1.5694372653961182, -2.475055694580078, -0.5107591152191162, 3.4612350463867188, 4.062366962432861, -4.319214820861816, 1.6034263372421265, -0.3993278741836548, -9.02606201171875, 3.5815725326538086, -1.283574104309082, 2.732792377471924, -0.0031396313570439816, 3.3135905265808105, -2.090496063232422, 0.6345000267028809], [-1.7160556316375732, -2.404902458190918, 1.9031190872192383, 0.43165814876556396, -1.1729823350906372, -2.3573734760284424, 7.721436500549316, -3.193885087966919, -1.9160566329956055, -6.820903778076172, -2.2641632556915283, -4.493125915527344, -0.956660270690918, -4.718498706817627, -1.8704743385314941, -0.24292494356632233, 4.448134899139404, -2.8092148303985596, -1.763613224029541, -2.7179982662200928, 2.3461568355560303, -1.4828072786331177, -3.019078254699707, -0.14115193486213684, -0.6496133804321289, 5.440633296966553, 0.7542154788970947, 4.470710277557373, -1.8718304634094238, -1.2707372903823853, -0.30854251980781555, -1.673706293106079], [-4.112884521484375, 0.23146206140518188, 3.4459428787231445, -2.955991744995117, 5.972338676452637, -0.09328728169202805, -1.4450514316558838, -5.3508381843566895, 1.4123234748840332, -4.922296047210693, 2.125736713409424, 1.5054141283035278, 0.9049370288848877, -5.747117519378662, -1.2640717029571533, -3.308267593383789, -0.17249637842178345, 3.991941452026367, -1.9483528137207031, 3.31357741355896, 0.1189771369099617, 1.9869722127914429, 2.1518776416778564, -7.407599449157715, 3.456469774246216, 6.327694892883301, -0.73219233751297, -1.1473528146743774, -1.6270878314971924, -4.929184436798096, 1.9239566326141357, 0.7513453960418701], [0.26064425706863403, 2.7824623584747314, 0.5149409174919128, -5.2183732986450195, -1.7047510147094727, -2.299351692199707, 0.8497601747512817, 2.661571502685547, 2.2415928840637207, 4.946346282958984, -0.5953096151351929, 1.7435635328292847, 0.7577621340751648, 0.7537742257118225, 1.6614543199539185, 1.8641459941864014, 2.237182855606079, -0.052956897765398026, -1.6242560148239136, 0.8942917585372925, -4.204188346862793, 1.017625331878662, -2.540510416030884, -1.667041301727295, -5.653570175170898, 0.9955019950866699, 0.3659895360469818, -3.405366897583008, 2.5514156818389893, 0.4420139789581299, 0.23332953453063965, 2.2187142372131348]]]], "dtype": "float32"}, {"__nd__": [[[[0.34189072251319885, 0.5715025067329407, 0.08587130159139633, 1.6457091569900513, 1.5743330717086792, -2.462388753890991, 0.039485860615968704, 1.1979976892471313, -0.4485374093055725, 0.08299200236797333, 0.7270336747169495, -0.8429952263832092, 0.683402419090271, 1.921859622001648, -0.7909533977508545, 0.03858182579278946, 0.5776016712188721, 1.7049071788787842, 0.19753096997737885, 0.5681702494621277, 1.3612632751464844, -0.49022969603538513, 1.8556007146835327, 0.5310510396957397, -1.0524189472198486, 0.1147909089922905, 1.4623761177062988, 0.7221479415893555, -0.8624618649482727, 1.2722402811050415, 0.4840214252471924, 0.3296871781349182], [-0.19942563772201538, -0.8696134090423584, -0.7225436568260193, 1.2252514362335205, -0.9418306350708008, -1.1361148357391357, -0.7243167757987976, 0.8390597701072693, -0.5370333790779114, 0.999799907207489, -0.3945271670818329, 0.047310903668403625, -0.746005117893219, 0.8038243651390076, 0.11625395715236664, 0.7427152991294861, 1.6811178922653198, 0.2128942608833313, -1.74677574634552, 0.9544232487678528, 0.8509791493415833, -0.5559995770454407, 0.5033033490180969, 0.5289571285247803, -0.2506024241447449, 1.251832127571106, 1.0320905447006226, 0.13853546977043152, -1.0822055339813232, -0.6442305445671082, -1.104439377784729, 0.6277080178260803], [-0.9215275645256042, 0.3304184079170227, 1.6334888935089111, 0.13532254099845886, -0.6831117272377014, 0.14460258185863495, -0.04740210622549057, 1.5361911058425903, 1.4620323181152344, -1.7024099826812744, 1.0086109638214111, -0.06354810297489166, -0.6078841686248779, -1.1682100296020508, -0.6079948544502258, -0.7024245262145996, 0.06019890680909157, -0.1933276653289795, 0.06511704623699188, -1.1595150232315063, 0.46952882409095764, -0.48991337418556213, -1.3569427728652954, -1.4472452402114868, 0.5060343146324158, 0.785309910774231, -2.817788600921631, 2.061626672744751, 0.2860957980155945, -0.11940024793148041, -1.0392035245895386, 0.8632451295852661], [-0.13584421575069427, -0.24650231003761292, -0.5988087058067322, -0.1814688742160797, -0.8562909960746765, -0.17007744312286377, 0.31220197677612305, -0.37715786695480347, 0.05206003785133362, 0.17494423687458038, 0.10980090498924255, -0.8557449579238892, 0.3450125455856323, 0.6134687066078186, -2.1987524032592773, 1.1386123895645142, 1.0825927257537842, -0.8712880611419678, 0.028437523171305656, 0.11274409294128418, -1.7062524557113647, 1.6913304328918457, 2.1657116413116455, 0.09521343559026718, 1.8937357664108276, -1.5373716354370117, 0.206444650888443, -0.8283812999725342, -0.266730934381485, 0.5519465804100037, 0.5907037854194641, -0.3498551845550537], [0.22072602808475494, -0.20325537025928497, 0.22392705082893372, -0.2324187308549881, 0.31707504391670227, -0.1879194676876068, 0.15739354491233826, -0.10521551966667175, 0.0340883731842041, -0.6064615845680237, -2.749551773071289, -0.2004283368587494, -0.053520165383815765, 1.0658899545669556, 1.4109820127487183, -0.3845811188220978, -1.9984629154205322, 0.03991061449050903, -0.17308105528354645, -0.44412311911582947, -0.18741737306118011, 0.8238999247550964, 1.2713171243667603, -1.6944222450256348, -0.6001970767974854, -0.7175788283348083, 0.3962775766849518, 0.08196983486413956, -1.2527929544448853, -0.18393340706825256, 0.42000049352645874, -1.3193609714508057], [1.5352721214294434, 0.8209229111671448, 0.3787679374217987, 0.6082656383514404, 1.4072437286376953, 0.8382600545883179, 1.7755330801010132, -0.14217109978199005, 0.7951644062995911, 0.06469433754682541, -1.800402283668518, 0.47224804759025574, -0.20737694203853607, -0.4305610656738281, 0.137931227684021, -1.923751950263977, -0.1093154028058052, 0.009439181536436081, 0.17426279187202454, -1.2021547555923462, 0.8229929804801941, 0.5982517600059509, 0.7167443037033081, 1.0324695110321045, -1.1909594535827637, 0.733782947063446, -1.3979257345199585, -0.6285489797592163, -0.0027341877575963736, 1.032860279083252, 0.8736456036567688, 0.8889526724815369], [2.1093146800994873, 0.3080073595046997, -1.236533284187317, -0.06447184085845947, 0.25652188062667847, 1.3105649948120117, 0.5901199579238892, -0.36480507254600525, -0.3969254195690155, 0.5383952260017395, 1.4933905601501465, -0.03372237831354141, -0.4038812518119812, 0.467032790184021, -0.2456388771533966, -2.264709711074829, -0.5061616897583008, 0.7860112190246582, 1.4727789163589478, 0.07146836817264557, 0.29542458057403564, 0.2061242014169693, 1.0552250146865845, 0.029345035552978516, 0.5923371315002441, -0.6180022954940796, 0.6086592078208923, 0.7264031767845154, -1.3934818506240845, -1.7004530429840088, 1.5830265283584595, 0.745482861995697], [-0.32594001293182373, 0.5537081360816956, 1.4356977939605713, 2.3961026668548584, -1.1794377565383911, -0.11095401644706726, 0.5246511697769165, 1.2611685991287231, -1.8960766792297363, -0.08326248824596405, -0.47679224610328674, -1.8306533098220825, -2.241699695587158, -1.6556185483932495, 1.2423248291015625, 0.057515498250722885, 0.09010852128267288, 1.2463264465332031, 0.018220407888293266, 0.4363175928592682, -0.9029082655906677, 0.4446091055870056, 0.21164007484912872, 1.0081297159194946, 0.09387611597776413, 0.3753935694694519, 0.8625594973564148, -0.5910314321517944, -0.9577705264091492, -0.5596253871917725, -0.09012659639120102, -0.3716319501399994], [-0.2152116745710373, -0.19475682079792023, 0.7737764120101929, 0.8613322377204895, -0.8327805399894714, -0.2132788747549057, 2.086291551589966, 0.3563128411769867, -0.7539901733398438, -0.3210175335407257, -0.4791378974914551, 0.6949278116226196, -0.1722840666770935, -0.07993900030851364, 0.6148501634597778, -0.9890445470809937, -0.9665935635566711, 0.03406210243701935, -0.7912296056747437, -0.3602786362171173, -0.5153635144233704, 1.68931245803833, 1.343549132347107, -0.1505281776189804, 0.3956831097602844, -0.6202987432479858, 0.9837680459022522, 0.34436675906181335, -0.2155366986989975, 1.0321178436279297, 0.2928653359413147, -2.454099655151367], [-0.693781316280365, -0.12416265904903412, -2.350827217102051, -0.01463928073644638, 1.2252318859100342, 2.7251150608062744, -0.5644174218177795, 0.7347211837768555, 0.8655773401260376, -1.4860382080078125, -0.7390479445457458, -0.07013893127441406, 0.22558017075061798, -0.7061031460762024, -1.3589603900909424, -0.7301806807518005, -0.6842411160469055, 0.2680431008338928, -0.19793352484703064, -0.16153638064861298, 0.037983741611242294, -0.9108991622924805, 0.6626234650611877, 2.4293646812438965, -1.7514050006866455, -0.08707385510206223, -0.4675924479961395, -1.8502687215805054, 0.14847075939178467, -0.1879417598247528, -1.044184923171997, -1.358386516571045], [-0.33282506465911865, 0.10415247082710266, -1.2410603761672974, -0.0623590424656868, -0.21534566581249237, 1.4377188682556152, 0.2571670711040497, 1.1327574253082275, 0.5451841354370117, -0.9505146741867065, -1.5678848028182983, 1.693710207939148, 0.9135432243347168, 0.452117919921875, -0.09216827154159546, 0.3932669162750244, 0.44738852977752686, 0.055975012481212616, 1.3535575866699219, 1.1869992017745972, -0.4008539319038391, -1.2018282413482666, 1.5570902824401855, -2.4678382873535156, -0.9764511585235596, -1.194251298904419, 0.17212936282157898, -0.6663609743118286, -1.5705463886260986, -0.8688217997550964, -0.6247740387916565, -0.019997747614979744], [-1.0661503076553345, 1.0788217782974243, -0.21167396008968353, -1.216233253479004, -0.23207278549671173, -0.8980621695518494, -1.2557995319366455, -0.6570694446563721, -0.18155640363693237, 0.7549111247062683, 1.6196556091308594, 0.4789790213108063, 0.3878628611564636, -1.7117022275924683, 1.8394420146942139, 0.17502734065055847, -0.71942138671875, 0.16055408120155334, -0.9976663589477539, 0.5205702185630798, 0.1323380470275879, 1.8453079462051392, -0.63233882188797, 0.7799303531646729, -1.0994220972061157, -2.192880392074585, -0.04918181523680687, 0.63170325756073, 0.9544729590415955, 0.21316389739513397, 0.0524354912340641, -0.32798027992248535], [0.575922429561615, 0.6058874726295471, -0.013775328174233437, -0.4505111277103424, 0.4614506959915161, 0.1806371510028839, 1.0084846019744873, -0.11931474506855011, -0.017512759193778038, 0.6576445698738098, 1.1887096166610718, -0.1607130914926529, 1.927668571472168, -1.131550669670105, 0.0394805371761322, 0.46149855852127075, -0.6213470101356506, 0.7956560850143433, 0.4735505282878876, 0.5773553848266602, -2.2616419792175293, -1.584018349647522, 1.0099431276321411, 1.550673246383667, 0.5997054576873779, -1.799037218093872, -1.5878816843032837, 0.14165110886096954, -0.0764869898557663, -0.5784828662872314, 1.8578684329986572, 0.08762943744659424], [-1.4726532697677612, -0.7566120028495789, -0.014636429026722908, 0.5124388933181763, -0.71326744556427, -0.5990278720855713, 0.03689441457390785, -1.0455143451690674, -0.8116407990455627, -0.9372609257698059, 0.3354862928390503, 2.013169765472412, 0.6214315891265869, -1.6645112037658691, -0.42306873202323914, -1.2611957788467407, 0.9353778958320618, 2.96757435798645, 0.2653729319572449, -1.1214081048965454, 0.008401358500123024, 1.0912926197052002, -0.4992225766181946, 0.9878436326980591, 0.8343965411186218, -0.5131781101226807, 0.11790376156568527, 0.566750705242157, -0.2686093747615814, -0.07852664589881897, -0.35433927178382874, -0.6806026101112366], [1.204359531402588, 1.2543771266937256, -0.37366393208503723, 0.49083560705184937, -0.6074585318565369, 1.4784694910049438, 0.4885781705379486, -0.08220832794904709, -0.6337570548057556, 0.6865285038948059, -1.1591989994049072, 0.7502439618110657, -0.7445134520530701, 0.3616181015968323, 1.1794582605361938, -0.3344362676143646, -0.4776292145252228, -1.4251235723495483, 0.3266410827636719, -0.2941056787967682, -0.6167474389076233, 1.4987092018127441, -0.8073790669441223, 1.136056900024414, -1.505167007446289, -0.06493556499481201, 2.0869314670562744, 0.37008559703826904, 1.5233715772628784, -1.33572256565094, -0.40961748361587524, -2.70778489112854], [0.655110776424408, 0.445064514875412, 0.66416996717453, -0.4277374744415283, -0.500506579875946, 0.7734801173210144, -0.3099401593208313, -0.1818406581878662, -1.5078859329223633, 0.7121669054031372, -1.6130119562149048, 0.7165208458900452, -0.025013355538249016, -1.4124058485031128, -1.0489593744277954, 0.5185120701789856, -0.8981474041938782, 0.33310312032699585, 1.399914026260376, -0.08557257056236267, -1.6797215938568115, 0.240682452917099, 0.008162671700119972, -0.03031526319682598, 0.007952798157930374, 1.0842835903167725, -2.4149837493896484, -0.7070449590682983, 0.8267074823379517, 0.04840346425771713, 0.36203742027282715, 1.0245951414108276], [-1.7556382417678833, -0.1039726510643959, 2.0293233394622803, 0.3085464537143707, -0.6122655868530273, -1.365920066833496, -1.697692632675171, -1.2313165664672852, -0.6831545829772949, -0.9955800175666809, -1.5016329288482666, 1.313981056213379, 0.5063976645469666, 0.56172776222229, 1.1092349290847778, -0.39100438356399536, -0.5586248636245728, 0.33770477771759033, 0.29520082473754883, -0.27006959915161133, -0.32342728972435, -0.3357577621936798, 1.8366602659225464, -0.13156749308109283, 1.1260912418365479, -1.144210696220398, -1.2474888563156128, 0.4221658408641815, 1.0608789920806885, -0.8784908056259155, -2.164846658706665, 0.5036840438842773], [-0.4819280207157135, -1.5895932912826538, 0.8737636804580688, -0.22482351958751678, 0.7256497740745544, 2.6982786655426025, 0.0443950854241848, -1.1451988220214844, 0.593112587928772, 0.6150074005126953, 0.37920141220092773, -0.0861988216638565, 1.6190258264541626, -0.12026965618133545, 0.6080506443977356, 1.7713571786880493, 0.5665700435638428, -0.24403776228427887, -0.3383930027484894, -0.10632462054491043, -0.26481086015701294, -1.2609682083129883, 1.7379565238952637, -0.011217352002859116, -1.1102728843688965, -0.8284322619438171, 0.3538070619106293, 2.0649731159210205, 0.4385344386100769, -1.1981357336044312, 3.977051019668579, 0.7328529953956604], [1.5132015943527222, 0.5867382884025574, -0.10935956239700317, -1.1428109407424927, 0.52546226978302, 1.1711349487304688, -0.144235759973526, 0.30174580216407776, -1.306127905845642, -1.7292574644088745, 0.536788284778595, 1.5060884952545166, 0.8592108488082886, -0.983529806137085, 1.124712347984314, -0.46457043290138245, -0.16140495240688324, -1.2200793027877808, -0.22141438722610474, 0.18358536064624786, -0.4003615379333496, -1.3371022939682007, -0.42799729108810425, 0.8108909726142883, -0.45881927013397217, -0.9907506704330444, -1.9277559518814087, 0.5695332288742065, 0.5824297070503235, 0.9587136507034302, 0.3369942903518677, 0.5272183418273926], [0.6293147206306458, 0.603283166885376, 0.21209096908569336, 1.5438069105148315, 0.4425232410430908, -0.8460249304771423, 0.5762245059013367, 0.06733423471450806, -0.6617624759674072, 0.9898955821990967, 0.44238409399986267, 1.634063482284546, -0.48834729194641113, 0.2635839879512787, -1.626990795135498, -1.3346199989318848, 0.4908026158809662, -0.5687694549560547, -1.9040435552597046, 0.35335293412208557, -0.5775220394134521, 0.30552148818969727, -0.7794919013977051, 1.409822940826416, 0.1253659576177597, -0.6633008122444153, -0.19927431643009186, 0.9503493905067444, -0.3393859267234802, 0.054589156061410904, 0.5934586524963379, 1.3896998167037964], [0.7588149309158325, -1.0801680088043213, -1.0272904634475708, -0.9454492926597595, -0.644093930721283, -1.2481114864349365, 0.8847510814666748, 0.008119593374431133, 0.8650590181350708, -0.4208938479423523, -0.2668778598308563, 0.2667646110057831, -0.35656073689460754, 0.9932460188865662, 1.6472078561782837, -0.1301874965429306, 0.8052482008934021, -0.02642039954662323, -0.14041535556316376, 1.5873265266418457, -0.608677864074707, -1.452004313468933, -0.7273433208465576, -0.1685822457075119, 0.4843730330467224, 0.715425431728363, 1.1477153301239014, 0.1607896387577057, -1.6830769777297974, 0.8716603517532349, -0.7442623972892761, -0.6149120330810547], [0.3651686906814575, -0.995581328868866, 1.890604853630066, -0.05553578585386276, 0.314579576253891, 0.0077275787480175495, -0.11947716027498245, -0.011430017650127411, -0.3464224636554718, 2.4955780506134033, -0.9640748500823975, 0.5508983731269836, -0.5033341646194458, -0.9739103317260742, 0.7973788976669312, 0.5819509625434875, 0.11449041962623596, -0.2665265202522278, 1.612603783607483, 1.1750019788742065, -1.6771457195281982, 0.8893355131149292, -1.2640610933303833, 0.7964828014373779, -0.5701454281806946, -0.42024022340774536, -0.35533419251441956, -1.2215540409088135, -1.1660103797912598, 0.4361669719219208, 0.38885876536369324, -0.024517305195331573], [-0.15593461692333221, -1.5219374895095825, 0.024429989978671074, -0.6979153156280518, -2.0607357025146484, 0.9357070922851562, -0.1217619776725769, 0.46685752272605896, 0.7651455402374268, 1.0948846340179443, -0.4725550413131714, -1.3733761310577393, 0.33703866600990295, 0.3047436475753784, 0.6425678133964539, 0.22731268405914307, -0.7816311717033386, 2.517749071121216, 0.05951012298464775, 1.67654550075531, 0.17670150101184845, -0.05381062999367714, -0.39391571283340454, -0.8409661650657654, 0.2231927365064621, -0.19128742814064026, -0.9119453430175781, -0.517381489276886, -0.2709074020385742, 2.1260793209075928, -1.0108968019485474, -1.1734124422073364], [0.2589409351348877, 1.3067158460617065, -0.5847401022911072, 0.5889137387275696, 1.4291285276412964, -0.7099202871322632, -1.7568100690841675, 1.262376070022583, 0.6275153160095215, -0.611011803150177, 0.11523529887199402, -0.6707085371017456, 0.18079873919487, -1.7392592430114746, -0.4751569926738739, 0.10495825856924057, 0.727057158946991, 2.3157124519348145, -1.556588888168335, -0.31407251954078674, 0.800003707408905, 0.052441515028476715, 0.6849414706230164, 0.6035596132278442, 0.03143991902470589, -0.10221467912197113, 0.6167492866516113, 1.010319471359253, 1.0560133457183838, 1.284088134765625, 0.7592863440513611, 1.6447430849075317], [-0.5006524324417114, -1.8228261470794678, 0.7704375982284546, 0.5093534588813782, -1.5307859182357788, 0.4942097067832947, -1.535481572151184, 0.11105074733495712, 0.760775625705719, 1.022481083869934, -0.6031432747840881, 0.968619167804718, -1.1894882917404175, 0.9156067371368408, 1.0378988981246948, 0.6632151007652283, -0.6126096248626709, -0.6488856077194214, -0.18286095559597015, 0.20443397760391235, -1.4873638153076172, 1.396327257156372, 0.9270323514938354, -0.4450890123844147, 0.9797146916389465, 0.5522221922874451, -1.748331069946289, 0.433173269033432, 1.3540775775909424, 0.5478472113609314, 0.30019429326057434, -0.39638474583625793], [1.4831598997116089, 0.8879082798957825, 1.1506515741348267, -0.25232598185539246, 2.1605372428894043, 0.16367758810520172, -1.2199026346206665, 0.14035122096538544, 0.958756148815155, -0.946729838848114, -1.7417523860931396, 1.7215243577957153, -0.38930779695510864, -1.1750924587249756, -2.1470673084259033, 0.578565776348114, 0.8457817435264587, -1.0282692909240723, -0.918615460395813, -0.9142808318138123, 0.7849351167678833, 0.44563183188438416, -0.7900456190109253, -0.5444279909133911, 0.1283418834209442, 0.016070222482085228, -0.47019630670547485, 0.11627628654241562, 0.35877639055252075, -0.14105406403541565, -1.440048336982727, -1.5430206060409546], [-0.06795115023851395, 1.1664845943450928, -0.10809683054685593, 1.73617422580719, -0.838025689125061, 0.9477564692497253, 0.2845006585121155, 2.1352570056915283, 0.8399931192398071, 0.09981204569339752, -0.7908647060394287, -0.9474865198135376, -1.0649008750915527, -0.24102863669395447, -0.10846657305955887, 0.6603665351867676, 0.3887975811958313, 0.7491105794906616, -0.24094894528388977, 0.47083717584609985, 0.37873220443725586, -0.9451884031295776, -0.43684375286102295, 1.0974527597427368, -0.7785720825195312, -0.07958900183439255, 1.0147793292999268, 0.5861701369285583, -0.655072033405304, 0.8148055076599121, -0.6770128011703491, -0.02813676930963993], [-2.0669031143188477, 0.0233627837151289, 1.430524230003357, 0.5377381443977356, -1.3100597858428955, 0.6012245416641235, -1.290670394897461, -0.20351488888263702, -1.806589126586914, 0.7846503257751465, -1.1443685293197632, -0.40804052352905273, -2.668126106262207, 0.6361088156700134, 0.3434188961982727, -2.172844171524048, 0.862136960029602, -1.2964729070663452, -0.23017345368862152, 1.1211638450622559, 0.7427281141281128, 0.5485944151878357, 0.20863085985183716, -0.3490850627422333, 0.011747184209525585, -0.35656386613845825, 0.5720657706260681, -1.0104738473892212, 0.07026766240596771, -1.1638660430908203, 1.6288361549377441, -0.3632696270942688], [-0.7722612619400024, -0.79313063621521, -0.0375915989279747, 0.2819039821624756, 0.9213038682937622, 0.4903656542301178, -0.0009601346682757139, 1.4821010828018188, 0.23522107303142548, -1.0061941146850586, 1.7295873165130615, -2.259799003601074, -0.6260185837745667, 0.3377305567264557, -0.9699799418449402, 1.1592934131622314, 0.9784621596336365, 0.5063039660453796, 0.6218147873878479, 1.6064471006393433, -0.21188099682331085, -0.6035124063491821, -1.117445468902588, 1.1554192304611206, 0.9288808703422546, -1.7188043594360352, -1.0659246444702148, -0.7511075139045715, -0.07064619660377502, 1.0808879137039185, 0.2600552439689636, 0.2223554104566574], [0.5293795466423035, -0.08603313565254211, -0.9054056406021118, 1.6663588285446167, 1.3532565832138062, -0.4665587544441223, -2.582747220993042, 1.9578757286071777, -0.7872250080108643, 0.18207095563411713, -0.7542944550514221, 0.2060844898223877, 0.45393985509872437, 0.11162510514259338, -0.6034878492355347, -0.39066725969314575, -0.593655526638031, 1.6802825927734375, -1.1664395332336426, -1.8158851861953735, -2.5422489643096924, 0.3988090455532074, 0.24069511890411377, -1.102874994277954, -0.14837850630283356, 0.6421526670455933, 0.1887436807155609, -0.9451321363449097, -0.7781926393508911, -0.7129665613174438, 1.0283796787261963, 0.5489229559898376], [0.43897324800491333, 1.150504231452942, 1.5042643547058105, -1.3854731321334839, -0.1280009150505066, 0.46528735756874084, -0.42097994685173035, 0.7356122136116028, -0.5793905854225159, -1.5346641540527344, 0.8990098237991333, 1.545648455619812, 0.09181120991706848, 0.25568056106567383, -1.2793992757797241, -1.059462547302246, -0.5544089674949646, 0.4515337646007538, -0.9530197978019714, -0.14041395485401154, 0.49771395325660706, 1.9497768878936768, -1.039076566696167, -0.1611318290233612, 0.15708507597446442, 2.5689690113067627, 1.4384069442749023, 0.7071603536605835, 0.9926146864891052, -0.6205856800079346, 1.1324790716171265, -0.40073680877685547], [-0.9340636730194092, -1.7365467548370361, 0.053812090307474136, -1.8811813592910767, -1.4852869510650635, -0.5114762187004089, -1.1656819581985474, -1.1164366006851196, 1.1708096265792847, 1.1224521398544312, 0.19905820488929749, 0.30199167132377625, -0.5535085201263428, 1.3942840099334717, 1.310617208480835, 0.7809773087501526, -0.11234604567289352, 1.1746810674667358, -0.6884804368019104, 0.5633128881454468, 0.8556456565856934, -0.034316498786211014, 0.12938304245471954, 0.9335018396377563, -0.9566879272460938, -1.968153715133667, 1.130240797996521, 0.7962126135826111, 0.4499173164367676, -0.30446359515190125, -0.755881667137146, 2.1506717205047607], [-0.858748197555542, -2.7203447818756104, 0.372484415769577, 0.83722323179245, 0.09587310999631882, 1.0129711627960205, 0.7216952443122864, -3.0447144508361816, -0.07509315013885498, 0.6649642586708069, 0.7682871222496033, -0.29602253437042236, -2.070404052734375, -0.7152009010314941, 1.164006233215332, 1.5595312118530273, -0.6067069172859192, -0.6121916174888611, -0.11885835975408554, 0.12774412333965302, 0.46625322103500366, 0.28363582491874695, 1.7887887954711914, -0.982265055179596, 1.3674191236495972, 0.23647990822792053, 0.916198194026947, -0.3451601564884186, -1.508888840675354, -0.5993755459785461, -0.36411890387535095, -1.171570897102356], [-0.7488905191421509, 0.3098718523979187, 1.1059303283691406, -2.00165057182312, -1.3213577270507812, 0.6478949189186096, 1.7564455270767212, -0.9634630680084229, -0.6483416557312012, 0.6030964255332947, 0.2558492422103882, 0.658944845199585, 0.9530982375144958, 2.7589194774627686, 2.5715925693511963, 0.3439534306526184, 1.2007206678390503, -0.7614859342575073, 0.274244487285614, -2.5366406440734863, 0.3949381411075592, 0.471037358045578, -0.7877777814865112, -0.8216314315795898, 0.4773404598236084, -0.931025505065918, -1.0783464908599854, 1.4443461894989014, -0.36076414585113525, -0.35977470874786377, 0.4245776832103729, -0.1140948086977005], [0.19189077615737915, 1.6193721294403076, 0.34280136227607727, -2.356160879135132, 0.47705715894699097, 1.066179871559143, -1.082621693611145, -0.5491679906845093, 0.4852353036403656, -0.035370949655771255, 0.7071108222007751, 0.3508354723453522, -0.29594868421554565, -1.283186912536621, -0.1582205891609192, -0.40165847539901733, 0.8282621502876282, -0.009329527616500854, -1.5276004076004028, 0.2669494152069092, 0.5945324897766113, -0.7502774000167847, 0.6262670159339905, 0.35938018560409546, -0.7350742816925049, -2.5188684463500977, 1.5187715291976929, 0.030703511089086533, -1.8899351358413696, -0.3714394271373749, 0.9282782077789307, 0.21382461488246918], [-1.7435216903686523, 0.3393896520137787, -1.2254644632339478, -1.6253118515014648, -2.0756380558013916, -0.5904630422592163, 1.034995436668396, -1.2962918281555176, 0.1900593489408493, 0.6486093997955322, -2.118377923965454, -0.28591272234916687, 0.9329866766929626, 0.005649695638567209, -0.4643673300743103, -0.00022611155873164535, 0.756065845489502, 0.5982376337051392, 0.31199413537979126, -0.05307751148939133, 0.8619118928909302, 0.7434318661689758, 1.0269131660461426, 1.1740446090698242, 1.3345762491226196, -1.6378803253173828, 0.4362671971321106, 1.2218555212020874, 1.2464015483856201, 0.22328980267047882, -1.3187661170959473, 0.5414074659347534], [0.4229590892791748, -1.92510187625885, 0.6107704043388367, 0.29352834820747375, -0.026883866637945175, -0.16248908638954163, -0.9085167646408081, 0.5435821413993835, -1.1521203517913818, -1.5860979557037354, 0.14551794528961182, 1.171261191368103, 1.450162649154663, -0.5106344819068909, 0.2837994694709778, -2.6547303199768066, -0.261844277381897, -0.6486362218856812, 0.004377465695142746, 0.01310353446751833, 1.440700650215149, -0.36497750878334045, 1.7905536890029907, -0.6281605958938599, 0.8861426115036011, 0.6243175268173218, 0.6275638937950134, 0.6144945621490479, -0.15129917860031128, -0.8282627463340759, 0.09220754355192184, -0.16981515288352966], [-0.7448033094406128, 0.4112263321876526, -1.0723183155059814, 1.867561936378479, 1.5295696258544922, -0.34264975786209106, -0.7635787129402161, -0.4786226749420166, 0.49639761447906494, 0.19804291427135468, 0.2623600363731384, -3.12155818939209, -0.03686629608273506, 2.0857605934143066, -0.6348637938499451, -0.47433042526245117, -0.354257732629776, -3.0861494541168213, -0.7907734513282776, 0.5214277505874634, -0.30951759219169617, -0.001580076408572495, 1.0326675176620483, 1.6198502779006958, -0.06420650333166122, 0.6716284155845642, -2.3781487941741943, -2.28695011138916, 0.6338881254196167, 0.6796230673789978, 1.6363131999969482, 1.7837414741516113], [-1.476891040802002, -1.296851634979248, 0.8212013244628906, 0.037737078964710236, 0.14807075262069702, 0.16627369821071625, 1.46436607837677, -0.24563202261924744, 0.7344419360160828, 0.293470561504364, -0.455515056848526, -0.9945163130760193, 0.41092029213905334, -1.0123039484024048, -1.689407229423523, 0.49751314520835876, 0.9070130586624146, -1.3757975101470947, -1.3779150247573853, -1.7995432615280151, -0.672675371170044, 1.0481656789779663, 0.7281335592269897, 1.4417935609817505, -1.2671422958374023, -2.05788254737854, -0.3799152970314026, 1.8548169136047363, -0.5478790402412415, 0.8506197333335876, 0.3032130300998688, 1.5829172134399414], [-1.485756516456604, -0.8904823064804077, 0.476935476064682, 0.04071784019470215, 0.3635435402393341, -0.6659257411956787, -0.14704138040542603, -0.07617536932229996, -0.031204672530293465, 0.7609609961509705, -0.4942476153373718, 0.013280855491757393, -1.1960259675979614, 0.10179067403078079, -0.1529332995414734, -2.255105972290039, -1.190459132194519, 0.295131653547287, 0.9587689638137817, 2.9611310958862305, -0.005867746192961931, -0.4921151101589203, -0.7968171238899231, -0.9323426485061646, 0.28291308879852295, 0.791454553604126, -1.3054249286651611, -0.7627397179603577, -1.2971301078796387, -0.5421085953712463, 0.47071221470832825, 0.7472925186157227], [0.34819066524505615, -0.4234071969985962, 0.6796231269836426, -1.9941214323043823, 0.35132500529289246, 0.17613938450813293, -1.497726559638977, 0.4616611897945404, 1.5116486549377441, 1.4758672714233398, -0.49720001220703125, 1.545196771621704, -0.38095930218696594, 1.241713523864746, -0.40235641598701477, -0.9384681582450867, -0.9206933379173279, -0.6250498294830322, 0.49438977241516113, -1.0755106210708618, 0.3850553333759308, 0.08330413699150085, 0.2162182331085205, 0.17098842561244965, 0.8922516107559204, -0.08659908175468445, -0.9265816807746887, -0.3684680759906769, 0.5367026329040527, 1.0001451969146729, -0.11942626535892487, 1.06727135181427], [-1.0359392166137695, 1.1861953735351562, 0.3781801462173462, 1.4812668561935425, 0.37281879782676697, 1.5560706853866577, -2.0667741298675537, -0.977454662322998, 0.36262640357017517, 0.789067268371582, -0.22247907519340515, 0.9196086525917053, -0.4464615285396576, 0.4303831160068512, 0.5079982876777649, 0.12413148581981659, -0.14304442703723907, -0.5960589051246643, -0.6541104316711426, -0.438328355550766, -0.37126076221466064, -0.010819205082952976, -2.967888832092285, -0.31884315609931946, -0.8190603852272034, -1.4779847860336304, -1.8808457851409912, 1.8791929483413696, -0.5525700449943542, 0.8085348606109619, 0.6300826668739319, -0.3828110694885254], [0.3769216239452362, -0.858649730682373, 1.1418341398239136, -0.8249638676643372, 0.37407299876213074, -1.2068246603012085, -1.5135443210601807, -0.41693732142448425, -0.18415850400924683, -0.11831176280975342, 0.4425140619277954, 0.409841924905777, -0.03293859213590622, -1.6838757991790771, 0.5686746835708618, 1.3510210514068604, 1.6656861305236816, 1.5381306409835815, 0.6371671557426453, 0.6415850520133972, 0.11085496842861176, -1.4327974319458008, 1.0895559787750244, 1.1972882747650146, 0.5307844281196594, 0.16469840705394745, 1.739642858505249, 0.13731886446475983, -0.7349346280097961, -1.3137460947036743, 1.2145934104919434, 2.503209352493286], [-0.44290581345558167, -0.061660509556531906, 0.22935695946216583, 1.0843784809112549, 0.9265649318695068, 0.11193765699863434, -1.0788103342056274, 0.6406851410865784, 0.8979173898696899, -0.7409557700157166, 0.7479881644248962, -1.633423924446106, 0.44621723890304565, -0.3334989547729492, 0.2660105526447296, -0.7983017563819885, 0.6044158935546875, 0.3505368232727051, 0.3724680244922638, 0.4125664234161377, 0.7278571128845215, 0.07318959385156631, -0.13240361213684082, 1.2829842567443848, -0.8656532168388367, -0.7035695910453796, -2.772325038909912, 0.18930481374263763, -0.7259487509727478, -0.931797981262207, 0.7211571931838989, -0.8079866170883179], [-0.3449610471725464, 0.8449773788452148, -0.4749167263507843, 0.24902091920375824, -1.3788204193115234, 0.3015816807746887, -0.7955322265625, 0.9083253145217896, 0.807000458240509, -1.7093420028686523, 1.65532386302948, 2.189754009246826, -0.5598270893096924, -0.8477560877799988, 1.1816552877426147, 1.10755455493927, 1.5074012279510498, 1.1173917055130005, 0.5421192049980164, -0.8512872457504272, 0.9864708781242371, -0.8007000088691711, 0.20064496994018555, 0.06512515991926193, 1.5592502355575562, -2.8093416690826416, -0.9569706320762634, 0.01725144311785698, -0.8142551779747009, 0.18397358059883118, -0.25043192505836487, 0.07665372639894485], [1.424572467803955, 0.3383369445800781, 1.230833649635315, -1.2904975414276123, -0.14743545651435852, 0.49559900164604187, -0.18291205167770386, -2.680515766143799, -0.5683591365814209, 1.57133948802948, -0.8934771418571472, 1.5168726444244385, -0.22134961187839508, -1.3453750610351562, 0.6586381196975708, 0.04942392557859421, -2.1496803760528564, -2.7849276065826416, -0.9236950278282166, 0.25714513659477234, 0.42989227175712585, -2.0018274784088135, -0.6081332564353943, 0.39994606375694275, 1.9375190734863281, 0.0877501592040062, -0.4781213700771332, 1.0732533931732178, 1.3943238258361816, 0.5046392679214478, -0.4112503230571747, 0.1438658982515335], [-1.0862106084823608, -1.0311087369918823, -0.4811255931854248, 0.19936630129814148, -0.06797647476196289, -0.4000921845436096, 0.7518233060836792, 0.14090359210968018, -0.014731641858816147, 0.7566006183624268, -1.3711596727371216, 1.6041737794876099, 0.5403024554252625, -1.4781124591827393, -0.6008950471878052, -0.6650543808937073, 1.779331088066101, 1.538030743598938, -0.3276040554046631, -0.5512456297874451, -0.39862093329429626, 0.204972505569458, -0.048002563416957855, -0.24407583475112915, -1.1721266508102417, 0.29143163561820984, 0.09749972075223923, -0.11759612709283829, -0.34958791732788086, -0.906743049621582, -0.7160290479660034, 0.4205840826034546], [-0.99906325340271, -0.43886590003967285, -0.3078240156173706, 0.2707514762878418, 0.06368719786405563, -0.1844659447669983, -1.035165548324585, -0.9899120926856995, 0.009769363328814507, 0.30156221985816956, -0.26602545380592346, -0.167107954621315, 0.2507094144821167, 1.0247352123260498, 0.3728756010532379, -0.42385971546173096, 0.8414936065673828, -0.42438173294067383, -1.2420856952667236, 1.0242793560028076, -0.6498561501502991, -0.21032948791980743, 0.7436299920082092, -0.2566283941268921, 0.6156405210494995, 1.0956780910491943, 1.2719846963882446, 1.1189689636230469, 0.39646482467651367, 0.07972047477960587, -2.0438332557678223, -1.7229042053222656], [-0.022174792364239693, 2.8485381603240967, -0.5558702349662781, 2.158398151397705, -1.0925654172897339, -0.7284455299377441, -1.640465497970581, 0.7834779024124146, -0.5561875700950623, 0.9961797595024109, -2.332911252975464, 1.582621455192566, -2.477365016937256, 0.580578625202179, -1.1213901042938232, 0.7601367831230164, 0.9493891000747681, 0.9708676934242249, 2.154282569885254, 1.3541470766067505, -0.1878635287284851, -0.09200803935527802, -0.11947324126958847, 0.5043262243270874, 0.2654077410697937, 1.3128998279571533, 1.4082601070404053, 0.4117138087749481, 0.6599071621894836, 0.06457188725471497, -0.2348048835992813, 1.175047516822815], [-2.6278493404388428, 0.2756817936897278, 0.916204571723938, -0.014499496668577194, -0.6961582899093628, -0.08963581174612045, -2.6420164108276367, -0.6093639135360718, -0.07143234461545944, -0.6789434552192688, -1.7302316427230835, -0.050977613776922226, 0.7364917397499084, -1.399902582168579, -0.26221951842308044, -1.476999282836914, -0.4348025321960449, -1.5175036191940308, 0.20056335628032684, 0.3865108788013458, -1.3547961711883545, 0.3632664382457733, 0.7343054413795471, -0.41318073868751526, -0.553525984287262, 0.07626908272504807, 0.14042715728282928, -2.156481981277466, -1.3622064590454102, -1.1712838411331177, 1.3227416276931763, 0.4240199029445648], [-0.9203968644142151, 0.930023729801178, 1.1158589124679565, -0.6056987047195435, 0.8601780533790588, 0.6980375647544861, -0.1132030040025711, -0.3194539546966553, -2.2307116985321045, 1.1003950834274292, 0.3632282316684723, 1.2690247297286987, -0.5060675144195557, -1.8268123865127563, -0.1833920031785965, 0.8625593781471252, -0.3896278142929077, -0.4591483175754547, 1.4398183822631836, -0.594189465045929, 0.4098353981971741, -0.277598112821579, -0.6629204750061035, -1.2383127212524414, -0.16067710518836975, -1.1037046909332275, -0.16601824760437012, -1.1558767557144165, -0.1785425841808319, -0.17216448485851288, 0.44183361530303955, 0.33763712644577026], [-0.17187519371509552, -0.5034577250480652, -0.3701331615447998, 0.0777205303311348, -0.41080084443092346, -0.05656333267688751, -0.39263346791267395, 0.14277179539203644, -0.17526257038116455, -0.6216850280761719, -1.4646469354629517, 0.6442455649375916, 0.28991952538490295, -1.0935567617416382, -1.439234733581543, -0.9519203901290894, 0.6719069480895996, -0.13014636933803558, -0.5577662587165833, 0.0824127197265625, -0.45520615577697754, -0.5307952761650085, -0.3084788918495178, -0.5740203857421875, 0.6748086214065552, -0.002122175646945834, 0.29926103353500366, 0.327199250459671, 1.2575949430465698, 1.7370448112487793, -0.07087071239948273, -0.7824697494506836], [-0.21464207768440247, 1.4155528545379639, -0.07475229352712631, -1.8214004039764404, -0.15034349262714386, -0.8876276612281799, 0.49687644839286804, 0.30287668108940125, 0.5733152627944946, -0.8882160782814026, -0.31898194551467896, -0.4716034531593323, -0.9350882172584534, -0.42474764585494995, 0.4554556906223297, -0.5609433650970459, 0.7120067477226257, -0.760087251663208, -1.4133142232894897, -0.837465226650238, 0.539603590965271, -1.0361716747283936, -1.812545657157898, -0.6027175784111023, 0.6486953496932983, 0.38879188895225525, -0.09080240875482559, -1.4093550443649292, 1.1108119487762451, 1.2677549123764038, -0.06087573617696762, 0.2478720247745514], [-0.052512381225824356, -0.14721672236919403, -0.27322837710380554, 0.3412388563156128, -0.6196261644363403, 1.7428572177886963, -0.40522220730781555, -0.06560254842042923, -1.1181319952011108, 0.4416213929653168, -0.47177791595458984, 1.3650586605072021, 0.13368470966815948, -0.8481137752532959, -1.463982343673706, 0.4620347023010254, -0.09721075743436813, -1.2870603799819946, 0.8080105781555176, -0.8581635355949402, -0.5076606273651123, -0.058071959763765335, -0.3623098134994507, -1.8060532808303833, -0.8310779333114624, 0.15470564365386963, -1.0269519090652466, 0.006219116039574146, -0.4342784285545349, 0.38677510619163513, 0.5656264424324036, 0.36271795630455017], [-0.16119319200515747, 0.25492388010025024, -0.2585428059101105, -0.2709941565990448, -0.3614242970943451, -1.0292888879776, -0.5849317908287048, 0.06397520750761032, 1.5049799680709839, 0.47740447521209717, 0.3688012659549713, -0.13814862072467804, -0.6678940653800964, -0.28629934787750244, 1.2378242015838623, -0.43761736154556274, 2.8523976802825928, 1.8340802192687988, -2.3384759426116943, -1.4981402158737183, -0.9694117903709412, 0.3014836013317108, -0.5627893209457397, 0.37156572937965393, 1.081685185432434, -1.1052157878875732, -0.8588558435440063, 1.031385898590088, 0.3307323455810547, -1.5212266445159912, -1.576654076576233, 0.0512472540140152], [0.5556343793869019, 0.20143522322177887, 1.9671857357025146, -0.5963864326477051, -0.12224322557449341, 1.4944671392440796, 0.924957275390625, -0.3741711974143982, -0.9574475288391113, 0.6093029975891113, 0.4746551513671875, 0.0523470863699913, 0.5969816446304321, -0.7375473976135254, 0.0172518789768219, 1.2613474130630493, -0.4209878444671631, -0.8643335700035095, 0.8130972385406494, -0.5477657914161682, 1.0046963691711426, -0.816648542881012, -0.8875867128372192, -0.6513080596923828, -1.7396240234375, 0.3631953001022339, 0.09372204542160034, -0.6029646396636963, 0.41321638226509094, 0.028259359300136566, -0.26352691650390625, 0.9228013157844543], [1.8236916065216064, -1.2430205345153809, -0.20331494510173798, -0.277188777923584, 0.6614463329315186, 0.40844812989234924, 0.8441921472549438, -1.205163836479187, 0.9810516238212585, 0.4489339590072632, -1.9487264156341553, 2.7714083194732666, 1.8593300580978394, -0.9303602576255798, 2.2175211906433105, 0.3108360767364502, -1.0432478189468384, 1.5949370861053467, -1.0797300338745117, -0.8383436799049377, 0.9791423678398132, -0.29618218541145325, 0.9616852402687073, -0.15295332670211792, 0.32298743724823, -1.4683507680892944, 2.2685439586639404, 1.344150185585022, -0.032886963337659836, 1.727838158607483, 0.7689599990844727, 1.289690375328064], [0.40815469622612, 0.4002254009246826, 1.6681325435638428, 0.2429938167333603, 2.551295042037964, -0.3026285171508789, 0.2462923526763916, -0.3784761428833008, 0.456855446100235, 1.6565262079238892, -0.20068396627902985, 1.16024911403656, 0.10397583991289139, 0.20833197236061096, 0.35943838953971863, -1.372889757156372, 0.36823922395706177, -2.7457475662231445, -1.2724428176879883, 0.2502228021621704, 0.2631302773952484, -0.05696883425116539, -0.5455571413040161, -1.262337565422058, -0.7015675902366638, 2.903428792953491, 1.8331642150878906, -0.13441921770572662, 0.39511606097221375, -1.3903509378433228, -1.9705270528793335, -0.8090054392814636], [0.4406334161758423, -0.44065967202186584, 1.3440477848052979, -1.1703802347183228, 0.581749677658081, 0.2595878541469574, -0.4877352714538574, -1.9728180170059204, 0.032169342041015625, -0.7048941850662231, 1.5415844917297363, -3.0201663970947266, 1.3706647157669067, 1.1003810167312622, -0.0549759604036808, -0.8613465428352356, -1.3113062381744385, 1.810799241065979, -2.2560648918151855, 0.017110200598835945, -0.2970438599586487, 0.19193574786186218, 0.5654800534248352, -1.3372901678085327, -0.024511702358722687, 1.3242188692092896, 2.090200901031494, -0.9854613542556763, 1.0398445129394531, -0.06563660502433777, 0.9228528141975403, -1.067291259765625], [-0.3081144094467163, -0.1672864556312561, -0.6822019219398499, -1.4001630544662476, -1.816454529762268, 1.8798502683639526, 0.8067488074302673, -0.9089137315750122, -0.28940239548683167, -0.8426047563552856, 1.8317205905914307, 1.4505888223648071, -1.3304531574249268, 0.5547866821289062, -1.0909342765808105, 0.4333329498767853, -0.4223465621471405, -1.2420251369476318, -1.178518533706665, -0.3100733160972595, 0.782793402671814, 0.7938161492347717, -0.191572904586792, 0.8031854629516602, 0.7651450037956238, 1.12185800075531, 0.9402270913124084, -0.7244419455528259, 2.1777546405792236, -0.08422939479351044, 0.4100191593170166, -0.864589273929596], [1.2007008790969849, 0.6616017818450928, -2.3331832885742188, 0.16607418656349182, -0.8031362295150757, -0.6521973609924316, -1.5002261400222778, 0.4833756983280182, -0.41829973459243774, -0.24238552153110504, 0.07640638947486877, -0.9806775450706482, -1.0347694158554077, 1.9461240768432617, 0.12848523259162903, -0.5755515694618225, -0.8324374556541443, -0.7348056435585022, -1.3831641674041748, -0.7847175002098083, -0.4807351231575012, -0.9148511290550232, 0.2689271569252014, -0.33026599884033203, -0.05203187093138695, 0.6582362055778503, 0.5126184821128845, 1.345334529876709, -0.7057939171791077, -0.9721064567565918, 0.22962434589862823, -0.43629270792007446], [0.9710761904716492, -0.31478947401046753, -0.6639446020126343, 0.0333406962454319, 0.10152984410524368, -0.382972776889801, 0.00818725861608982, -0.2951337993144989, -0.6055367588996887, -0.9298328161239624, 0.6210390329360962, 1.8682955503463745, 0.023118581622838974, 1.3603423833847046, -1.0010356903076172, 0.022028598934412003, 1.48000168800354, 0.4092880189418793, -0.723778486251831, 1.260082721710205, 0.6033643484115601, -0.7911810874938965, 2.24870228767395, 1.0870524644851685, -0.6588748097419739, -1.0797195434570312, -0.8166055083274841, -0.5561383366584778, -0.07027141749858856, 1.770898699760437, -0.06425611674785614, -0.35681086778640747], [0.9313486218452454, -0.11785364151000977, 0.5221008658409119, 0.20635975897312164, 1.2827633619308472, 0.9276958107948303, 1.1086903810501099, 1.380497694015503, 0.8081998229026794, 0.17031770944595337, -0.39007461071014404, -1.2711749076843262, -0.6883053183555603, 1.005599021911621, 0.23556263744831085, -0.8845797181129456, 0.5639789700508118, 1.3048150539398193, 0.7862098813056946, -1.2579344511032104, -1.6978321075439453, 0.5212313532829285, 1.3098890781402588, -1.0686569213867188, -2.348691701889038, 0.4503352642059326, 0.8711392879486084, 0.8128902912139893, 1.6789424419403076, 1.2962634563446045, 0.19630268216133118, 0.8546290993690491], [-0.9367495775222778, -0.5004578828811646, 0.8855875730514526, 1.0079782009124756, -0.17275470495224, 0.2931622266769409, -0.3115993142127991, 1.702184796333313, 0.8576502203941345, 0.24934813380241394, -0.004029184114187956, 2.7686960697174072, -0.5595628619194031, -1.3642828464508057, -0.24369902908802032, 0.27224940061569214, -2.055081844329834, -0.18457727134227753, -0.2959216237068176, 0.4837774932384491, 3.385902166366577, -0.9562315940856934, 0.24745741486549377, 0.10740439593791962, 2.1145730018615723, 0.33510032296180725, -0.5032321810722351, -1.096550703048706, 0.2335827499628067, 1.9934462308883667, 0.9948340058326721, 1.8217812776565552], [-0.4374159872531891, 0.08655936270952225, -0.24116624891757965, 1.068995714187622, -0.395412802696228, 0.012725351378321648, 0.5833459496498108, 1.3191426992416382, 1.3054471015930176, -0.7779517769813538, -1.3440057039260864, 1.3558149337768555, -0.06397603452205658, -0.16035740077495575, -1.1779518127441406, -1.8057467937469482, -1.173866629600525, 1.68364417552948, -0.9243337512016296, -0.003954753745347261, -1.0803091526031494, 1.2200217247009277, -0.5392380356788635, -0.18233732879161835, -2.3727564811706543, 0.8956965804100037, -2.0149528980255127, -0.7016196846961975, -0.36120039224624634, -0.6418914198875427, -0.42323920130729675, -0.1398966759443283], [-0.7128523588180542, -1.5619739294052124, 2.539124011993408, 0.8577925562858582, -0.3089381158351898, -1.67274808883667, -0.6169226765632629, -1.3875596523284912, 0.7767595052719116, -0.29034867882728577, -1.3645118474960327, -0.15606704354286194, 0.3803360164165497, -0.24589340388774872, 0.7050063610076904, 1.0768983364105225, 0.34954655170440674, -0.19343997538089752, -0.46734800934791565, 1.3781474828720093, 0.48789742588996887, 0.1436241716146469, -1.1413699388504028, 0.9474368691444397, -1.3818833827972412, -1.8623688220977783, 0.3753502368927002, 0.5875545740127563, -0.011678459122776985, 0.8266002535820007, -0.697741687297821, -1.0432153940200806], [1.5322439670562744, -0.5063666105270386, 1.608213186264038, -0.09965862333774567, -1.616746187210083, 2.1741597652435303, -0.6054256558418274, -0.5721672773361206, -1.56593918800354, 0.9732491970062256, -0.1254677176475525, 0.29406365752220154, 0.5777813196182251, -0.2823900580406189, 0.5285979509353638, -0.19618584215641022, -0.9757732152938843, -0.2416202574968338, 0.5733701586723328, 0.05888855829834938, -2.312598705291748, 0.5431029796600342, -1.1859134435653687, 0.26878106594085693, -1.5971490144729614, 0.44381436705589294, 1.8076452016830444, -0.062264684587717056, 0.21689265966415405, 1.4409316778182983, 0.8522610068321228, -0.21609053015708923], [0.4219715893268585, 0.24807482957839966, -0.0872105285525322, -1.0532236099243164, 1.282588243484497, 1.4901058673858643, -0.14591766893863678, -1.0987505912780762, 2.5314199924468994, 0.7195445895195007, -0.15906168520450592, 1.8320361375808716, 2.27689790725708, 2.682530164718628, 0.8559288382530212, -1.4112849235534668, 1.4773776531219482, -1.1838910579681396, 0.1583895981311798, 1.7920043468475342, 0.4270997941493988, 0.7888360619544983, -0.3141491115093231, 1.185195803642273, 0.7950929403305054, 0.46512076258659363, -0.9863974452018738, -0.20650531351566315, 0.4056253731250763, 1.0264190435409546, -1.1111143827438354, 0.27760934829711914], [1.5383528470993042, -1.6307063102722168, 1.613861083984375, -1.621856689453125, -1.3331936597824097, -1.5962798595428467, -0.8185980916023254, -1.3475526571273804, -0.37155792117118835, -1.1539239883422852, -0.41080552339553833, 0.46057194471359253, 1.3515757322311401, 1.052319049835205, -0.6799539923667908, 0.8880164623260498, 0.09741612523794174, -0.04649396985769272, 1.8259165287017822, 1.0789152383804321, -0.5807420611381531, 0.4152485728263855, -1.4008781909942627, 0.710432231426239, 2.394273281097412, 0.49668681621551514, 0.41848447918891907, -0.15625400841236115, -1.0913599729537964, 0.06406623870134354, -0.4434283375740051, -1.058816909790039], [-1.5794187784194946, 1.099402904510498, -0.43910104036331177, 0.18195296823978424, 1.4534392356872559, -0.7902934551239014, -2.146422863006592, 0.536202073097229, 0.22317840158939362, 1.3024390935897827, -1.2304946184158325, -0.05904797464609146, 0.4717642664909363, -0.9599934816360474, -0.8037918210029602, 0.026666589081287384, 1.352622151374817, 0.5887072086334229, -0.7521533966064453, -0.17555853724479675, -0.1619000881910324, -2.271599769592285, 1.1448791027069092, 1.2146532535552979, 0.07158825546503067, -0.07544584572315216, -0.25900712609291077, 1.0969418287277222, -0.39401453733444214, -1.341517686843872, -0.7871851921081543, -1.0722520351409912], [-0.09126804769039154, -1.2769949436187744, -0.5037842988967896, -0.03294297680258751, -1.7415807247161865, 0.5779376029968262, 0.8243860006332397, 0.4308176338672638, -0.7994381189346313, -0.5994872450828552, 1.6379612684249878, 0.6694223284721375, 1.0886789560317993, -1.0110834836959839, -0.5844062566757202, 1.5941842794418335, -0.4645017385482788, -1.317670464515686, -1.0748809576034546, -1.1000310182571411, -1.8479905128479004, 0.36485984921455383, 0.9045242667198181, 1.136533498764038, -0.6189607381820679, 1.141056776046753, -0.09864845126867294, 0.007313817739486694, -0.04200618341565132, 1.0950907468795776, -0.4805987775325775, -0.7108759880065918], [-0.5476034879684448, 0.2031564861536026, -0.8681782484054565, -2.1148219108581543, 1.094218134880066, -0.37372928857803345, -1.4983313083648682, 0.7069079279899597, 0.6031376719474792, 2.188950538635254, 0.8652517199516296, -0.4449304938316345, 0.9000455737113953, 0.6824729442596436, 0.3266356885433197, -0.21903806924819946, -0.00327212642878294, -0.1267019510269165, -0.3467741310596466, 1.7192736864089966, -0.45631399750709534, -1.8374754190444946, -0.1670692414045334, -0.18427304923534393, 1.574345350265503, -2.1466455459594727, 1.557369589805603, -0.3437877893447876, -0.3987281024456024, -0.2608186900615692, 1.175909399986267, -0.878741443157196], [-0.04037512466311455, -1.1336581707000732, 0.534956157207489, 0.9657946228981018, 0.4393496513366699, 0.6500840783119202, 0.06702394783496857, 0.6368730664253235, -0.7663578391075134, -0.37009167671203613, 1.8178995847702026, 0.5829350352287292, 0.8131067156791687, 0.7409153580665588, -0.14360636472702026, 0.3335675299167633, 0.5625416040420532, 0.8385946750640869, -1.584241271018982, 0.8282057642936707, 0.38515743613243103, -0.8901056051254272, 0.062223974615335464, -1.2140616178512573, -0.11371710151433945, 1.5907514095306396, -0.5731516480445862, -1.483697533607483, -0.6589580774307251, 1.6345021724700928, 0.8213841915130615, 0.7654567360877991], [0.9304370284080505, 0.9183906316757202, 1.3644442558288574, 0.2043197602033615, -1.3141980171203613, -0.21977682411670685, -1.4349424839019775, -0.2894361615180969, 0.2386016994714737, 0.18486438691616058, -0.5039978623390198, 0.9830898642539978, 1.3053499460220337, 0.8552958965301514, -0.5780125856399536, 1.4567670822143555, -1.5706992149353027, -0.24152755737304688, -0.7548067569732666, 1.758053183555603, 1.7624939680099487, 0.6948472857475281, 0.5097546577453613, -0.41542184352874756, -0.004222426097840071, 0.5023278594017029, 1.3640086650848389, -1.1370878219604492, -0.28961095213890076, -1.1458311080932617, 1.032572865486145, 1.9072628021240234], [-1.0448216199874878, 0.14904843270778656, 1.87552809715271, 0.1136893704533577, -0.5321781039237976, -0.45305925607681274, -0.2778809070587158, 0.3077528178691864, -1.079427719116211, 2.2422754764556885, -0.8025888800621033, 2.2291347980499268, 0.053676340728998184, -0.015546352602541447, -0.6779781579971313, -0.9409863352775574, -0.5426410436630249, 0.7790219187736511, 1.6108587980270386, -0.4659620523452759, 1.0882141590118408, -1.193869948387146, 1.5916813611984253, -0.1966085135936737, -1.2284349203109741, -0.014996157959103584, -0.22272156178951263, -1.0609257221221924, -1.5698028802871704, 0.7486162185668945, 0.5885319709777832, -2.3124115467071533], [1.7309490442276, 1.3520969152450562, -1.1308600902557373, -1.0607630014419556, -0.2675088942050934, -1.1732409000396729, -0.18317410349845886, 0.9526832103729248, 0.5435091257095337, -1.3131197690963745, 0.8371027708053589, -1.9063631296157837, -1.903931975364685, 1.8812804222106934, -0.4770844578742981, 0.7398884296417236, 0.009737118147313595, 0.892106294631958, 0.5392447113990784, 1.5936334133148193, 0.5792828798294067, 1.3201528787612915, 0.9528424143791199, -0.3116743862628937, 0.7977144718170166, -0.036882735788822174, 0.2127613127231598, 1.3002727031707764, -0.386862188577652, 1.8727835416793823, 2.2422375679016113, -0.8570887446403503], [-1.450933575630188, -1.0266233682632446, 1.5140330791473389, 0.7173722982406616, 0.03492417186498642, -2.2917463779449463, -0.8444253206253052, 0.41530275344848633, -0.6181867718696594, -0.9034979343414307, 1.0932555198669434, -0.8310167193412781, 2.0561366081237793, -0.03838540241122246, 0.8894363045692444, 0.5037909150123596, 0.06533389538526535, -2.024794816970825, 0.8852874040603638, 1.3569681644439697, -0.28608864545822144, -0.6548627614974976, 1.2002227306365967, -0.12484066933393478, 1.0557368993759155, -0.004880726337432861, -1.3115633726119995, 0.0365615114569664, 0.780006468296051, -0.5072860717773438, -0.10722244530916214, 0.3114369809627533]], [[0.05745861679315567, -0.39159736037254333, 2.538038492202759, -1.2514461278915405, 0.7949472069740295, 0.5737931728363037, -0.46556830406188965, -0.45836764574050903, -0.06775569915771484, 0.15679869055747986, -1.8408395051956177, -1.0409754514694214, 0.06624629348516464, -1.2262307405471802, -0.3256818950176239, 1.337916374206543, -0.003493302734568715, -0.41294366121292114, 0.22222203016281128, 1.7685619592666626, 1.0887503623962402, 0.3674977719783783, -0.8936050534248352, 0.2360985279083252, -0.6739560961723328, 0.7547433972358704, -1.5936989784240723, -0.9507530331611633, -0.9088459014892578, -0.1470852941274643, -0.17228643596172333, -1.0747990608215332], [1.4590442180633545, -0.2352975755929947, -3.156097888946533, -0.2478177696466446, 1.831616997718811, 0.6606425046920776, 0.450676828622818, 0.2594643235206604, 0.3505282700061798, 0.6764709949493408, -1.0097521543502808, -0.408964067697525, -0.13853251934051514, -0.15548598766326904, -0.25332772731781006, -0.49257034063339233, 0.3295596241950989, 1.525518774986267, -0.4412343204021454, -0.3490268886089325, -0.514924168586731, -0.3676002621650696, 1.2397441864013672, -0.35574138164520264, -0.24889372289180756, 0.2555566132068634, 0.7755697965621948, -0.59053635597229, -0.3489075005054474, 0.4594837725162506, 0.001443111919797957, -0.7191582322120667], [0.8022241592407227, 0.2423688769340515, -3.0669314861297607, -1.2191085815429688, 0.2172158658504486, -0.32931438088417053, -0.5154613256454468, 0.575752317905426, 0.5919797420501709, 1.229830265045166, -1.8216674327850342, 0.6806825995445251, -0.337365061044693, 1.2319269180297852, -0.30272313952445984, 0.7474873065948486, -2.0214977264404297, 1.059464931488037, -0.6469670534133911, -0.21412323415279388, 0.15490864217281342, -1.047279715538025, -1.9639517068862915, -1.189972162246704, -0.6901481747627258, 1.0671403408050537, 0.7004940509796143, 0.026785245165228844, -0.22155164182186127, 0.22472470998764038, -1.2401905059814453, 0.8313539028167725], [0.21664173901081085, -0.8186172246932983, -1.1815460920333862, 1.9124683141708374, -0.17149661481380463, -1.2379812002182007, -0.9374825954437256, -0.20010685920715332, 1.0505962371826172, -0.5813777446746826, 0.6060060262680054, -0.04606284946203232, 2.933854341506958, 0.3680623471736908, -0.3384169936180115, -0.28565114736557007, -0.4747503399848938, -0.7142051458358765, -0.17995771765708923, 0.5542895793914795, 1.5257433652877808, -0.23784321546554565, -0.38111454248428345, -0.40189558267593384, -0.09972186386585236, -0.03447553515434265, -1.0792099237442017, 2.9612722396850586, -0.46816933155059814, -1.3340235948562622, 1.219015121459961, -0.7423838973045349], [1.617471694946289, 1.690124273300171, -0.7445847988128662, 0.23738013207912445, -2.1541385650634766, -0.6879192590713501, -1.6933989524841309, 0.6121497750282288, 0.5118671655654907, -1.6130714416503906, 1.3563897609710693, -0.544778048992157, -0.3365117907524109, 0.25722843408584595, -1.866626501083374, 0.6548836827278137, -0.27049633860588074, 0.5796120166778564, -0.8968465924263, -0.7500922083854675, 1.0403708219528198, -0.9165122509002686, -0.4996570944786072, -0.44190341234207153, 0.056689757853746414, 0.15589724481105804, 2.1956210136413574, -0.2923431098461151, 0.06643117964267731, -0.8751795887947083, 0.012225603684782982, -0.3187554180622101], [0.24555163085460663, -0.16683954000473022, 0.0010949175339192152, 0.7254179120063782, 0.21006274223327637, 0.00337150227278471, 0.10555768013000488, 1.714106559753418, 0.01580164022743702, 0.4170542359352112, 0.447619765996933, 0.7093056440353394, 0.7795944213867188, 0.5184453725814819, 2.187808036804199, -0.5253689885139465, 1.4004619121551514, -0.9582607746124268, -0.031446292996406555, 1.985408902168274, 2.09264874458313, -0.9736249446868896, -1.4118492603302002, -0.02643568255007267, -1.2032239437103271, -1.5069692134857178, 0.09885998070240021, 0.20192450284957886, 0.32664647698402405, 0.7201982140541077, 1.7222250699996948, -0.4840347170829773], [0.9337219595909119, 3.177098274230957, 0.20657573640346527, 0.8992052674293518, -1.7785112857818604, 0.5948821306228638, 1.0207000970840454, 0.09027646481990814, 2.9349868297576904, 0.762900710105896, -0.3020918667316437, 0.1658109426498413, 0.597766101360321, 0.714906632900238, 0.45665809512138367, -0.03510677069425583, -1.4755109548568726, -1.3337039947509766, -1.3091471195220947, -1.2622578144073486, -0.04420972988009453, -1.252566933631897, -0.031758103519678116, 1.108459234237671, 2.5161261558532715, -0.16121631860733032, 0.564705491065979, -0.48534777760505676, -1.4513088464736938, -0.8302565217018127, -2.7495815753936768, -2.2726454734802246], [0.7820242047309875, -1.9401190280914307, -0.5909103751182556, -2.554708242416382, 2.9459404945373535, -0.001869490253739059, -0.7548274993896484, -2.182344913482666, -0.3667755722999573, -0.7073397636413574, 0.11497180163860321, -0.5849893689155579, -1.116852045059204, -0.6669294834136963, -0.4341989755630493, 0.532673180103302, 0.03346005082130432, 0.8906976580619812, 1.5499905347824097, 0.6636330485343933, 0.9125568866729736, 1.1182758808135986, -0.04562178999185562, 0.6917404532432556, -0.16072723269462585, -0.7004702687263489, -2.022226333618164, -0.474323034286499, -0.29047468304634094, 0.8318144083023071, 1.6457077264785767, -1.996287226676941], [1.0997265577316284, 0.869598925113678, -0.32397007942199707, -0.244699165225029, -2.1118741035461426, -0.7167454957962036, -1.4640772342681885, -0.8873229026794434, 0.7499648332595825, 0.7667569518089294, -0.26599740982055664, 1.501643180847168, -0.3053618371486664, -0.2825440466403961, 1.1213148832321167, -0.1743551343679428, -1.4102768898010254, 1.2710777521133423, -0.11913561820983887, 1.5504968166351318, 1.1159048080444336, -0.9321011900901794, -0.8321805000305176, 0.12370853871107101, -0.8195551633834839, -0.8654436469078064, 0.797035813331604, 2.321195363998413, 0.009750597178936005, 0.18126808106899261, 0.1108187884092331, -0.3983425199985504], [0.40057897567749023, -0.48791661858558655, -0.3259706199169159, -0.5518956780433655, 0.44996634125709534, 1.213258147239685, 1.9277750253677368, -0.6665780544281006, 0.991141140460968, 1.4145528078079224, 0.002915501594543457, 1.9013409614562988, -0.3749679923057556, -0.8427785634994507, 2.3034565448760986, -0.22245891392230988, -0.7895076870918274, -0.6010323166847229, 0.6224137544631958, -0.7123314142227173, 2.4322028160095215, -0.16975289583206177, -0.07238028198480606, -0.8090347051620483, 1.0454297065734863, -1.3461700677871704, -2.020620584487915, 1.330075740814209, 0.266664981842041, -0.5976071953773499, 0.40564340353012085, -0.9096671342849731], [-1.6540602445602417, 0.5519795417785645, -0.3939078450202942, -0.3353058397769928, -0.1068662777543068, -0.9343933463096619, 0.22062845528125763, -1.036030888557434, 0.5597707629203796, -0.0774410143494606, -0.3325105309486389, 0.6798438429832458, 0.4767109155654907, -0.1043345257639885, -0.846712052822113, -0.0947537049651146, 0.2845519185066223, -0.12353645265102386, 1.4285670518875122, 0.7119596004486084, -0.5244534015655518, -0.7937299013137817, -1.850503921508789, -1.8315645456314087, -0.9621914029121399, 3.030548334121704, -0.08442538231611252, 0.023492202162742615, -0.6141815781593323, -0.6908472180366516, 0.12717212736606598, -0.5751243829727173], [0.21211418509483337, 0.8601006865501404, 1.9434928894042969, -0.35606855154037476, 1.3605785369873047, -0.8865732550621033, 1.822899341583252, 2.404850721359253, 1.3781131505966187, -0.26793691515922546, 1.4766312837600708, -2.2608001232147217, 1.6107070446014404, 0.8030943870544434, -1.3487722873687744, -0.33088040351867676, 1.4414714574813843, 0.8804078102111816, -0.33752331137657166, -0.3105837106704712, 2.4602086544036865, 0.35690009593963623, -0.7491953372955322, 0.11743919551372528, 2.559966564178467, 0.09535317867994308, 1.0735098123550415, -0.2675182819366455, 0.6442960500717163, 0.4364687204360962, -0.21857060492038727, 0.38833707571029663], [-0.4589204490184784, 1.8334733247756958, -0.8022178411483765, 0.2787112891674042, -0.30704444646835327, 2.58054780960083, 1.0820521116256714, 0.18811701238155365, 0.028431281447410583, 0.6834818720817566, -0.5610370635986328, 0.16676190495491028, -0.9852075576782227, 1.7071949243545532, 0.06072867289185524, -0.33658742904663086, 1.104563593864441, 0.15254104137420654, 1.0913110971450806, 0.6931753158569336, 0.7150461673736572, 0.8882802724838257, -1.6623866558074951, -1.4311984777450562, -2.240539073944092, -0.021107634529471397, -1.983756184577942, 1.0391687154769897, 0.49645233154296875, -0.6288328766822815, 0.7292250394821167, -1.1456743478775024], [-0.7035516500473022, -0.15643158555030823, -1.0701780319213867, 1.4137336015701294, -0.7969662547111511, 0.1172783300280571, -0.23340728878974915, -1.9827712774276733, -0.3041591942310333, -1.729308843612671, 0.659328818321228, 0.011438002809882164, 0.13766713440418243, -0.17426766455173492, 0.053794704377651215, 0.96091228723526, 0.3350794315338135, -0.5964973568916321, -1.3853830099105835, -0.29211363196372986, -1.4772964715957642, -0.04097910225391388, 0.8855229020118713, -1.769636869430542, 0.3008655607700348, 0.37897130846977234, -2.090130090713501, 1.3989512920379639, 2.0536015033721924, -1.2113292217254639, 1.2199052572250366, 0.7721588611602783], [1.2148993015289307, 0.41911885142326355, 0.3114015460014343, 0.7246869802474976, -0.49301835894584656, -0.16494619846343994, -1.2663030624389648, -1.5586590766906738, 0.5860909223556519, -0.16760589182376862, -1.1927719116210938, 1.1854897737503052, 1.897822380065918, 0.9993840456008911, -0.0736192986369133, -0.2679479718208313, -0.08247606456279755, -0.8026198148727417, -0.03189077600836754, 1.344323754310608, 0.11521454155445099, -0.23863591253757477, 0.7227092385292053, -0.6229273676872253, -1.0664130449295044, 0.08178307116031647, -0.3149729073047638, -0.11215761303901672, 0.6194060444831848, 0.8910598754882812, 0.24479146301746368, 0.9629653096199036], [0.9766963720321655, 0.35736602544784546, -0.25204500555992126, 1.4089770317077637, 1.1461083889007568, 1.4701220989227295, 0.753043532371521, -0.8091511130332947, 0.4133719503879547, -0.7033621668815613, -2.2594733238220215, 1.17739999294281, 0.22460931539535522, -1.2355531454086304, -0.9570878148078918, 0.027408765628933907, 1.9572339057922363, 0.5500854253768921, 1.012237310409546, -1.8067262172698975, 0.9457811117172241, -1.5786713361740112, 0.3844917416572571, 0.9460258483886719, -0.48486071825027466, 0.857574999332428, -2.1736178398132324, -1.2348212003707886, 0.4679464101791382, -0.1373274028301239, -0.03346655145287514, 0.44186922907829285], [0.2391958385705948, 1.1851210594177246, -0.7155849933624268, 0.6316489577293396, 0.17851026356220245, -0.8575097918510437, 0.6244370341300964, 0.25578275322914124, -0.5349664688110352, -0.39555105566978455, 0.531484842300415, -0.6414492726325989, 0.7077358961105347, 2.102820634841919, 1.1662076711654663, -0.4208575487136841, -0.09182360768318176, 0.1597556620836258, 1.472306728363037, 0.6143051981925964, 1.9661747217178345, 0.8383888006210327, 0.715403139591217, -0.9958157539367676, 0.4114437401294708, -0.2559868395328522, -0.39522942900657654, -0.9094682931900024, 2.0736148357391357, -0.5008658766746521, 0.7874552607536316, -0.5719119906425476], [-0.6163979172706604, -0.11484221369028091, 1.542250156402588, 0.40537843108177185, 1.039957046508789, 1.7246174812316895, 0.16569197177886963, 0.2982231080532074, 0.2859777510166168, -1.1759862899780273, -1.4830195903778076, 1.4532554149627686, -1.4513598680496216, -1.5744413137435913, -0.5393019318580627, 0.7385107278823853, -1.5754209756851196, -0.6944990754127502, -0.6521968245506287, 1.0599299669265747, -0.12188147008419037, 0.3187260925769806, 0.38677966594696045, 0.8763281106948853, 1.1569641828536987, 1.6088080406188965, 1.2509862184524536, 0.5362577438354492, -0.7656359672546387, 0.4463338553905487, -1.1492245197296143, -1.0203790664672852], [-0.23247130215168, 1.117478370666504, -1.355271577835083, -0.7743080854415894, 1.3391841650009155, 0.5088476538658142, 0.9217522144317627, 1.144810438156128, -0.059683121740818024, -0.4460128843784332, 1.7103501558303833, -0.7931304574012756, -1.3807674646377563, 2.5797832012176514, 0.20076154172420502, -0.855000913143158, 2.172065019607544, -0.5323585271835327, -1.2401070594787598, 0.32935845851898193, -1.5185402631759644, 0.36263373494148254, 0.20010682940483093, -1.6574835777282715, -1.2050111293792725, 0.0307511817663908, -0.6900284290313721, -1.3245525360107422, -0.7235110402107239, -2.260707139968872, -0.07200299948453903, -0.5826461911201477], [-0.5810114145278931, 1.6888948678970337, 0.1208442822098732, -0.8180487155914307, -0.8069213032722473, 0.01177359651774168, 0.23158486187458038, -0.016399335116147995, -0.22420699894428253, -0.7686256766319275, 0.2207254320383072, 1.1643028259277344, -0.5654524564743042, 0.4916013479232788, -0.2806204855442047, -1.81390380859375, -0.9657745361328125, 0.10140781849622726, -3.1571264266967773, -0.7715733647346497, -1.4013383388519287, -0.4887202978134155, 0.7194441556930542, 1.178422451019287, 0.4459761381149292, -0.25629064440727234, 0.07751531898975372, 0.8139781951904297, -0.18488754332065582, 0.5343347787857056, -1.3513556718826294, 2.380089044570923], [-2.170949935913086, 3.2020485401153564, 0.27541303634643555, -3.077662467956543, 1.6919670104980469, -0.21451435983181, 1.1594442129135132, 0.39118191599845886, -1.7004495859146118, -0.7723657488822937, 0.15304093062877655, -0.33927637338638306, -1.3218542337417603, 0.19675341248512268, -0.7589219212532043, -0.20913241803646088, 0.8780568242073059, 0.7164003849029541, 2.0366642475128174, 0.8811690211296082, -0.3204891085624695, -0.4659917652606964, -2.1309869289398193, -0.2928393483161926, -1.3999247550964355, 1.647139072418213, -1.0471159219741821, -0.047685086727142334, -0.7053342461585999, 0.6559478044509888, -0.7258604168891907, 0.644479513168335], [0.3144395351409912, -1.8868038654327393, -1.04766047000885, -0.4756060540676117, 0.6004794239997864, 0.8978151082992554, 0.3771637976169586, -0.652616024017334, 0.9278468489646912, 0.30065128207206726, 0.41457366943359375, 0.37060660123825073, -0.06339428573846817, 1.769330382347107, -0.9208088517189026, 1.0447427034378052, -1.6667485237121582, -0.7039284110069275, 0.34936583042144775, -1.133594274520874, -0.755734384059906, 0.6622918844223022, -1.3305728435516357, 1.3486626148223877, 0.056958846747875214, -2.54559063911438, 0.4149625301361084, -0.5195440649986267, 0.4145708680152893, 0.6528471112251282, 0.604964017868042, 1.3057981729507446], [0.17804589867591858, -0.02867007441818714, 0.45702075958251953, -0.48906150460243225, 1.2537814378738403, -2.0705113410949707, 0.14332573115825653, -0.2801693379878998, 0.5281335711479187, 0.27955907583236694, 1.6935468912124634, -0.2179943025112152, 0.941124439239502, -1.2659285068511963, -1.0719220638275146, -0.8546602725982666, 0.9580546617507935, -1.0304393768310547, 1.9172186851501465, -0.1259605884552002, 1.8393689393997192, 0.1732778549194336, -2.301222801208496, 0.788821816444397, -0.20980989933013916, 0.15636056661605835, 0.10072554647922516, -0.04630229249596596, 1.805062174797058, 0.8746294975280762, -0.5293863415718079, 0.1940668225288391], [-0.2798258066177368, 0.9421362280845642, 0.7133397459983826, 1.6154199838638306, -0.2764311134815216, -0.8593108654022217, -0.8386478424072266, -0.7017888426780701, -0.8853208422660828, -0.7405017614364624, 0.6388862133026123, 1.293127417564392, -1.0706825256347656, -1.000933051109314, 0.043597765266895294, -1.2891967296600342, 0.9953427910804749, -0.6008973717689514, 0.6334832906723022, 3.2192001342773438, 0.27648869156837463, -0.42140617966651917, -0.09456063061952591, 0.2745855748653412, -1.4128252267837524, 1.710909366607666, -0.6006910800933838, -0.10890381783246994, -1.3567752838134766, 0.5051413774490356, 0.1101139560341835, 0.23851032555103302], [0.17697574198246002, 0.2585203945636749, 0.6263049244880676, -1.1655521392822266, 0.525734007358551, -0.3094877600669861, -0.753066897392273, -0.43337002396583557, -0.7209256291389465, -0.31332871317863464, -0.6301127076148987, 1.1665565967559814, -0.23163366317749023, -0.102385014295578, -1.0316632986068726, 0.8792935013771057, 1.1298489570617676, 0.8835168480873108, 0.6107137799263, 0.6137901544570923, -0.6697482466697693, 0.1163894385099411, -2.1884334087371826, -1.171887993812561, -0.0269977618008852, 0.40469396114349365, -1.2360470294952393, -0.00024820075486786664, 0.7785645723342896, 1.889032244682312, 0.012420016340911388, 0.9378560185432434], [0.6173591613769531, -0.8926946520805359, 1.295112133026123, 0.5418484210968018, 0.6590859889984131, 0.3881649672985077, -0.4903823435306549, 0.9086830615997314, 0.06957495957612991, -1.5491220951080322, -0.7125048041343689, 0.7556427121162415, -0.12153276056051254, -0.8630083799362183, -1.0172430276870728, 0.8725448250770569, 0.8575006127357483, 2.192080020904541, 0.37886175513267517, -0.6178993582725525, 1.388859510421753, -1.3517035245895386, -0.42568230628967285, -1.02853262424469, 0.35268014669418335, -0.6698186993598938, -1.645200490951538, -0.8051798343658447, -1.371355652809143, 0.3424064815044403, 0.815639078617096, -1.2340972423553467], [-1.551608681678772, 0.1578715443611145, 0.03118535690009594, 0.9222462773323059, 0.07671817392110825, -0.3302721381187439, 0.6952747702598572, -0.8152212500572205, -0.9272048473358154, 1.5067288875579834, 0.2084370106458664, 0.5811865925788879, 0.020707953721284866, -0.974819540977478, -0.4091647267341614, 0.9443650841712952, -1.6824638843536377, -0.3105393350124359, -0.18312086164951324, 0.6502806544303894, 0.3322696387767792, -1.5021569728851318, 0.4206683933734894, -0.9698169231414795, 1.0054038763046265, 0.5188028812408447, -1.4248008728027344, 0.00027392050833441317, -0.39598050713539124, -0.9870599508285522, -0.9408462643623352, -0.255565345287323], [0.30717307329177856, -0.49978023767471313, 0.03696060553193092, -0.8815259337425232, 1.9870694875717163, 1.370139241218567, 0.16690562665462494, 0.052383147180080414, -0.10062728822231293, 2.0692203044891357, -0.08129354566335678, -0.06311320513486862, 0.45838749408721924, 1.2769715785980225, -1.769144058227539, 0.7939491271972656, 2.392744779586792, 0.5928371548652649, 0.11922820657491684, -2.4236321449279785, 1.437590479850769, -0.5734139084815979, -0.1770627647638321, 0.6444782018661499, 0.31172722578048706, 0.5232762098312378, -0.7777605652809143, 0.36100709438323975, -0.4635542035102844, 0.327714204788208, -1.5580815076828003, 0.9362763166427612], [-0.8683375120162964, -1.1089972257614136, 1.4910385608673096, 0.16317863762378693, -0.059395212680101395, 0.38003435730934143, -2.0920047760009766, 0.9065418243408203, 0.44492393732070923, 1.4111815690994263, 0.8820065259933472, -0.5976410508155823, -0.36560988426208496, -1.0742038488388062, -0.2728349268436432, -1.4918333292007446, 0.4597710371017456, 0.7713724970817566, 0.7063494920730591, 0.9296097159385681, -1.7907847166061401, -0.15161430835723877, -0.20721465349197388, -0.7922582626342773, -1.1401898860931396, 0.035435330122709274, 1.6509884595870972, 1.8635234832763672, 1.6244555711746216, 0.1566000133752823, -0.7834862470626831, 0.978134036064148], [1.319031834602356, -0.9891396164894104, -0.055340662598609924, 0.6671687364578247, 0.8505173325538635, 0.32036736607551575, -0.23561249673366547, -0.04205814376473427, 0.35747823119163513, -0.5548762083053589, 1.7044860124588013, -0.41003936529159546, 0.226413756608963, 1.8126343488693237, -0.29708558320999146, -0.19935089349746704, 0.17501512169837952, 0.2706921696662903, -0.14858947694301605, 0.9044851660728455, 0.038483962416648865, -0.17718152701854706, 0.13970276713371277, -0.70166015625, -0.6812143921852112, -0.7781274318695068, -0.1904652863740921, 0.636093258857727, 2.252300262451172, -1.107607364654541, 0.06466799229383469, 0.2977313697338104], [0.5459553599357605, 0.7618905901908875, 0.733354926109314, -0.8468989133834839, 0.4640793204307556, -1.875380277633667, -0.023239286616444588, 0.7913143038749695, 0.6599404811859131, -1.8323462009429932, 0.22563986480236053, -2.08846378326416, 0.09778756648302078, -1.6829290390014648, 0.033967070281505585, -1.3318886756896973, 0.9759306311607361, 0.1417897790670395, 0.5356337428092957, -1.1115292310714722, 3.1457719802856445, -0.34269750118255615, -1.354557991027832, 0.3011437952518463, 1.152390480041504, 0.5363689661026001, -1.1617181301116943, 0.35611990094184875, -0.5742045640945435, 0.32828259468078613, -0.7706795334815979, -2.315248489379883], [-0.7818138003349304, 0.40439537167549133, 0.08082518726587296, -1.389529824256897, -0.6704220175743103, -0.3687446415424347, 0.5267260670661926, 0.39351800084114075, -0.2803869843482971, -0.6034753918647766, 0.216911181807518, -0.202506884932518, -1.6590536832809448, 0.3553279638290405, -0.8248475193977356, 0.7803587317466736, 1.1816205978393555, 1.2624683380126953, -0.9510632157325745, -0.39044415950775146, 2.2291643619537354, -0.21916311979293823, 2.3219034671783447, 1.9345786571502686, 1.3869613409042358, -0.5349769592285156, 0.6555594205856323, 0.40971484780311584, 0.8919728398323059, 0.5600920915603638, -0.8187218904495239, -0.3953436613082886], [0.2949639558792114, 0.9059205055236816, 0.04409266635775566, 0.09621458500623703, -1.3967732191085815, 0.81859290599823, 0.04864732176065445, -0.08375894278287888, -0.6398032307624817, 1.6016031503677368, -1.1311782598495483, -0.8228703141212463, 0.0334559828042984, 0.5947107672691345, 1.9294664859771729, -0.6036455035209656, -1.0497032403945923, -1.9154413938522339, 0.005093592219054699, 0.4963876008987427, -1.469506025314331, 0.9780173301696777, -0.008920556865632534, -0.8583258390426636, 1.2396364212036133, 1.2428470849990845, -1.9784785509109497, 1.5039904117584229, 0.05625727027654648, 0.0704694539308548, -0.4615054428577423, 0.8808163404464722], [-0.1593206524848938, 0.4198072850704193, -0.3653857707977295, 0.3438751697540283, 0.6706519722938538, 0.2326720952987671, -0.37410107254981995, 0.8859485983848572, -0.40119197964668274, -0.11018027365207672, -0.9576559066772461, -0.14152757823467255, 0.6570643186569214, 1.9513351917266846, -0.9032593369483948, 0.7348713874816895, 3.5313117504119873, 1.22830069065094, 0.589857280254364, -0.5775770545005798, 2.0383687019348145, 1.5670336484909058, 1.1998528242111206, -0.878892719745636, 0.6365963220596313, 0.11269953846931458, -1.3510721921920776, 0.05437615513801575, -1.4106640815734863, -0.1927388459444046, 1.895799160003662, -2.0733799934387207], [0.09454620629549026, 0.7165354490280151, -0.45168086886405945, 0.09617270529270172, 0.18187448382377625, -0.6831576824188232, 2.1890649795532227, 0.24483805894851685, 0.31757283210754395, 0.2871754467487335, -1.845909595489502, 0.18975503742694855, 1.169670581817627, -0.5489078760147095, -0.2816029191017151, -1.023207187652588, 0.9495954513549805, 1.0885981321334839, 1.3882957696914673, -0.07762076705694199, 0.5471177697181702, -0.39872318506240845, -0.3371838331222534, 0.7619943618774414, 1.2625707387924194, -0.6783643960952759, 1.715003490447998, -1.0314433574676514, 1.1839845180511475, 0.1328032910823822, 1.6123924255371094, -2.183514356613159], [-1.16994047164917, 0.46437183022499084, 0.8298454284667969, 1.2906559705734253, -1.6471354961395264, 0.37313076853752136, -1.0262553691864014, 0.6544686555862427, -0.5651106834411621, -0.1333579123020172, 0.5972112417221069, 0.8056214451789856, -0.14347252249717712, -1.0461299419403076, 1.6132105588912964, -2.318437099456787, 0.5460091233253479, 0.865175187587738, 2.392760753631592, -1.3258063793182373, -0.24584893882274628, -0.4428301155567169, -1.3744380474090576, 1.091915249824524, -0.4602644443511963, 0.9414375424385071, 0.42040762305259705, -0.02729448303580284, -0.6891528367996216, -0.28693172335624695, 0.935023307800293, 0.4238334894180298], [-1.0889867544174194, 0.40451380610466003, -0.413895845413208, -2.2698566913604736, -0.9455966949462891, -0.19966961443424225, -0.7048031091690063, -0.2037106305360794, -1.9919346570968628, 0.46852537989616394, -0.6361100673675537, -0.6943325400352478, -0.26149019598960876, 1.8539377450942993, -0.1296297311782837, 0.8564984798431396, -0.4598633646965027, 1.929843544960022, 1.4094306230545044, 1.9040528535842896, 0.5136304497718811, -0.6118904948234558, 0.11077062785625458, 0.4132220447063446, -1.1730389595031738, -0.0433485209941864, -0.035008206963539124, -0.020532939583063126, -0.1321703940629959, 1.2923654317855835, 1.4217867851257324, 1.5889521837234497], [2.159064292907715, 0.015016479417681694, 0.9917343258857727, -0.8133506178855896, 0.7438374757766724, 0.21879442036151886, -0.26882418990135193, 0.20451343059539795, 0.6557489037513733, -1.2072314023971558, -0.01165497861802578, -0.15009427070617676, 0.018448522314429283, 0.1646617203950882, -0.39707592129707336, -1.4473066329956055, 0.20528250932693481, -0.369402140378952, -0.038437630981206894, 0.297916978597641, 0.09095744043588638, 0.3356306254863739, 0.48057225346565247, -0.5416931509971619, 0.27653050422668457, 0.14002633094787598, 0.23583823442459106, 1.4885385036468506, -0.9418604969978333, 0.3388306200504303, 1.5210767984390259, 0.77070552110672], [0.49356546998023987, 1.1211795806884766, 0.03613635525107384, 0.2777599096298218, -0.807272732257843, 0.43761366605758667, -0.02134140394628048, 0.5023964047431946, 1.2368407249450684, 0.6247028112411499, -0.10513986647129059, -0.10639394074678421, 1.4787908792495728, -0.7371299862861633, -1.16706120967865, 0.08359474688768387, 1.9142566919326782, 2.160445213317871, 0.9750053286552429, -0.6400812268257141, 0.14358869194984436, -0.3873692452907562, 0.9414507746696472, 1.61886727809906, -0.3606760501861572, -0.5498679876327515, -0.5742186903953552, -0.40403223037719727, 0.1332186907529831, 0.37060022354125977, 0.15920116007328033, 0.3043054938316345], [-0.04694204404950142, -0.5216652154922485, -2.2723562717437744, 0.19086028635501862, -1.4146404266357422, 0.828019917011261, -0.7645683288574219, 0.4952007234096527, 0.6518917083740234, -0.8429659008979797, -1.9592827558517456, -0.9467748999595642, 1.65350341796875, -0.9420929551124573, 0.058485906571149826, 0.43295371532440186, 1.3211580514907837, 0.13960714638233185, -1.6810762882232666, -0.24392005801200867, -0.7977874875068665, -0.25759157538414, -0.3819011449813843, -0.3154822885990143, 0.4996698498725891, 0.56655353307724, 0.3905601501464844, 1.3886573314666748, -0.27123281359672546, -0.8719919919967651, -0.7524840831756592, 0.4834778308868408], [0.22129562497138977, 0.5849900245666504, 1.352007269859314, 0.2827601134777069, 1.7842351198196411, 0.7455700039863586, -0.14432398974895477, 0.4796229600906372, 1.0551611185073853, 1.3184338808059692, -1.4066684246063232, 1.1793873310089111, -0.43335506319999695, -1.242267370223999, 0.06002092361450195, 0.34515687823295593, -0.7680299878120422, 0.03837836906313896, -0.3611316680908203, -1.3169982433319092, 0.5303946137428284, 0.8523558974266052, 2.509089231491089, 0.5930187106132507, 0.9502560496330261, -1.6055883169174194, 1.0628159046173096, 0.9844052195549011, 0.16000041365623474, -1.3758857250213623, 0.17106173932552338, 1.4111111164093018], [-1.433546543121338, 0.0570528544485569, 0.7229508757591248, 0.4376555383205414, 0.12356968224048615, 0.8346157670021057, -0.8101657629013062, 1.0494316816329956, -0.7845001220703125, 1.12664794921875, -0.37854406237602234, 0.4033076763153076, 0.7977113723754883, 0.8290542960166931, -0.8009737730026245, -0.0689486414194107, -0.4510643482208252, -0.3670535981655121, 0.40488865971565247, -0.7013331055641174, -0.9419824481010437, 0.11507085710763931, 0.18401044607162476, -0.9275893568992615, -0.19738471508026123, -0.8082672953605652, -0.2633915841579437, 2.3815481662750244, 0.5383689999580383, 0.46629440784454346, 0.5807893872261047, -0.3361319303512573], [-0.1507624238729477, -0.37587955594062805, 0.6395069360733032, -2.207131862640381, 0.45669445395469666, -0.9724564552307129, -0.0892532616853714, -1.3314553499221802, -0.16319233179092407, 0.9500318169593811, -1.4576880931854248, 1.02952241897583, 0.8096708655357361, -0.1669931560754776, -0.4777083992958069, -0.29684990644454956, -1.4184564352035522, -0.10087040066719055, -0.35443082451820374, -1.436767339706421, 0.37128695845603943, -0.11906265467405319, -1.9621881246566772, -2.1555728912353516, -1.495233416557312, -1.099780797958374, 2.4223339557647705, 0.7716870903968811, -0.24022071063518524, -0.9004610776901245, -1.3552767038345337, 0.7152756452560425], [1.4523742198944092, -0.3595563471317291, 0.9570735096931458, -1.1706725358963013, -1.3035802841186523, 0.6814089417457581, -0.13785001635551453, 1.3770204782485962, 2.0873279571533203, 1.0545984506607056, 0.784757673740387, -0.1840307116508484, 0.8811779618263245, 3.390221118927002, -0.4006175398826599, -0.1673031896352768, -2.076179265975952, -2.2796781063079834, -0.923674464225769, 1.3745882511138916, -1.0687363147735596, -0.8613353371620178, -1.1614203453063965, 1.6382502317428589, 1.7742618322372437, 0.0963355153799057, 0.21089130640029907, 0.67976975440979, 0.9633113741874695, -0.060164086520671844, 0.4858609437942505, -0.49476954340934753], [0.36473390460014343, -0.0937199518084526, 0.6706255674362183, 0.2223474234342575, 1.105973243713379, 1.6551882028579712, 0.38696885108947754, 1.6951133012771606, 0.7220515608787537, -0.7863675951957703, -0.7542635798454285, -0.27585241198539734, 0.45082300901412964, 0.2100716531276703, -1.542292594909668, 0.7681123614311218, -1.0685240030288696, -1.5397112369537354, 0.6173673272132874, -1.6345314979553223, 0.10880221426486969, 1.2888635396957397, 0.23270727694034576, -0.12091083824634552, 0.9238020777702332, -0.05261249840259552, 0.8827338218688965, -0.354458749294281, -1.5125919580459595, 0.85078364610672, -1.1303179264068604, -1.3153626918792725], [-0.06855721026659012, 0.08136209845542908, 0.47757577896118164, 1.1892493963241577, 1.212519645690918, -0.7655601501464844, 0.44727036356925964, -1.6819806098937988, -0.7366029024124146, 0.05968999117612839, -0.22410666942596436, 0.3277539908885956, 1.4594444036483765, -2.0224878787994385, -0.2369185835123062, 1.1971157789230347, 0.5294433236122131, -0.43797481060028076, 0.6541176438331604, -0.028991669416427612, -0.3088698387145996, 0.28735870122909546, -1.5368595123291016, 0.6178008317947388, -1.4184534549713135, -0.4402860999107361, -0.20448024570941925, -1.9785341024398804, -1.55986750125885, 0.623843789100647, -0.4609440863132477, 1.4179718494415283], [0.27062737941741943, 0.20117224752902985, -0.4416639506816864, -1.3876738548278809, 0.17178431153297424, 0.6424252390861511, 0.7587774991989136, 0.24251459538936615, 1.2173840999603271, 1.1998984813690186, 1.1693828105926514, 0.7837139368057251, -1.2432581186294556, 0.254218190908432, 1.131430745124817, 0.37659239768981934, 1.4287704229354858, 0.3543184995651245, -1.504825234413147, 0.2228313833475113, 0.0376194566488266, 0.8774088621139526, -0.7847161293029785, 0.5208687782287598, 1.1385096311569214, 0.13144072890281677, 1.4447823762893677, 0.1248026043176651, 0.4097926616668701, 0.575739324092865, 0.951712965965271, 0.9360693097114563], [-0.605816125869751, 0.46292975544929504, 0.9110429883003235, 0.9956591725349426, 1.1306710243225098, -1.846449851989746, 0.07258172333240509, 0.2195870578289032, -0.9206883311271667, 1.9832559823989868, 0.14028067886829376, -0.4597799479961395, -1.0760583877563477, 0.9491770267486572, -2.002532958984375, 2.3007936477661133, -1.658612847328186, 0.11873897910118103, 0.11310464888811111, 0.8171535730361938, -2.3516552448272705, -0.195826917886734, -0.12219732254743576, 0.4949471950531006, -0.8204894661903381, 0.6150234937667847, -0.8662862777709961, 0.8928594589233398, 0.6194996237754822, 0.9022164940834045, -0.8258804678916931, -0.02629697509109974], [-0.5718469023704529, 0.46255606412887573, -1.1706589460372925, 1.7722710371017456, 0.4925006926059723, 0.5863653421401978, 0.13574975728988647, 0.35628998279571533, 0.007017267402261496, -1.0672717094421387, -0.09085550159215927, -0.32150912284851074, -0.15382231771945953, -0.4062926471233368, -0.9794384837150574, -0.24902217090129852, 1.0650333166122437, -2.269735097885132, 1.257123589515686, -1.8275315761566162, -0.6117634177207947, -0.08862199634313583, 1.0204801559448242, 0.6416119337081909, 0.6696189641952515, -0.1807752251625061, 1.2937626838684082, -0.6297298073768616, 0.7334487438201904, 1.4167486429214478, -1.0670676231384277, 0.19488155841827393], [-0.30222874879837036, 0.8852489590644836, -0.4286436438560486, 0.04867884889245033, -0.6055684685707092, -1.1334847211837769, 0.9342895746231079, 0.12741050124168396, -0.08861315995454788, 1.0250672101974487, -1.270991325378418, -0.18676617741584778, 0.2579696476459503, -0.8167481422424316, -0.18837188184261322, 1.3408030271530151, -0.8329671621322632, 2.638267755508423, 0.19043198227882385, -1.4256720542907715, 1.5344667434692383, -0.027331072837114334, 0.11393376439809799, -1.2511484622955322, -0.12774068117141724, -0.831932008266449, 1.0524218082427979, 0.18091000616550446, 0.1939586102962494, 1.6613456010818481, 2.436497688293457, 0.3859170079231262], [-0.11113253235816956, 0.3663008511066437, -4.119348049163818, -1.449593186378479, 0.05724945291876793, -0.16617777943611145, -0.9971147775650024, 0.39156797528266907, 0.010716710239648819, 0.334428071975708, 0.3079783320426941, -0.2965479791164398, 1.3821399211883545, 0.09435362368822098, -0.9206075668334961, -0.2910071909427643, -0.0701080933213234, 0.7555830478668213, -0.5311746001243591, 1.117661714553833, 0.6976538300514221, -1.5324835777282715, 1.2056282758712769, 0.6327650547027588, -1.2167447805404663, 1.6089404821395874, 0.1792234480381012, -0.16462701559066772, -0.032642245292663574, 0.5869954824447632, -0.6983356475830078, 1.3739327192306519], [0.0864967331290245, -0.6399928331375122, 1.3822590112686157, -0.9576977491378784, -0.667277991771698, 0.7443285584449768, 0.5951129198074341, 1.1496630907058716, -2.2787396907806396, 0.49082979559898376, -1.0805047750473022, 0.8128526210784912, -0.04908367618918419, 0.1088772565126419, 0.5365179181098938, -0.1362873911857605, -0.7318035364151001, 0.45240962505340576, 1.5125412940979004, -0.8231010437011719, -0.050433795899152756, -0.707042932510376, -0.3226117491722107, 0.45168381929397583, 2.169368267059326, -1.0491479635238647, -1.1006693840026855, 0.5275861024856567, -0.4983130991458893, 1.7057174444198608, -0.2913321256637573, 1.1888517141342163], [-0.03777708485722542, -0.40875494480133057, -0.8199049830436707, -0.749214768409729, -1.6560657024383545, 0.4529232978820801, 2.867237091064453, 0.7486062049865723, -0.9969856142997742, 0.43161541223526, 0.11968468874692917, -1.5600497722625732, 1.1148149967193604, 0.4265999495983124, 0.8702564835548401, -1.8131177425384521, -1.5500080585479736, -1.6012552976608276, 1.2709182500839233, 0.0640086680650711, -1.8639976978302002, 0.5677279233932495, 0.21950842440128326, 2.607790946960449, -0.2659876346588135, 0.6749314665794373, 0.783623456954956, -0.3439975678920746, 0.4276285767555237, 0.717376708984375, -0.37573137879371643, -0.3514226973056793], [-0.34477880597114563, -0.36453011631965637, -1.0474354028701782, -1.5058139562606812, -0.6982682943344116, 0.350939005613327, -0.33976033329963684, -0.3181194067001343, -0.7389556169509888, 1.558907151222229, -0.03105899505317211, -1.2851929664611816, 0.1952352523803711, 0.7644236087799072, -0.1448584645986557, 0.5486491918563843, 0.23077000677585602, 1.1212749481201172, -0.3052453100681305, -0.0974966362118721, 0.744864284992218, -0.4652019143104553, 0.9982240200042725, 1.071879506111145, -0.14219500124454498, -1.1974060535430908, -0.523038387298584, 0.5037913918495178, 2.1370747089385986, 1.9389339685440063, -0.8899799585342407, -0.23277609050273895], [-1.3699043989181519, 1.0081061124801636, -1.6329262256622314, 0.5683538317680359, -0.6025815010070801, 2.7499163150787354, 0.3060685098171234, -0.031120676547288895, -1.4140076637268066, -3.459223444224335e-05, 1.1530375480651855, -1.1225064992904663, -0.5784980654716492, -2.592888355255127, 1.4970370531082153, 0.2763207256793976, -1.2152137756347656, 0.08417078852653503, -1.3687913417816162, -1.4926203489303589, -0.63689786195755, -0.17329636216163635, 1.014203667640686, -1.8738276958465576, -0.3095071017742157, -0.5170163512229919, -1.3229174613952637, 0.8064014911651611, 1.2689874172210693, -1.1438852548599243, -1.4589712619781494, -0.7641181349754333], [0.723442792892456, 1.6777169704437256, -0.6554312705993652, -0.3962533473968506, 0.29700398445129395, -1.059428095817566, -1.1547280550003052, -1.074009895324707, -0.3814321756362915, -1.786799430847168, 0.9333876967430115, 0.5926946997642517, -0.49286431074142456, -1.1489499807357788, 0.9733209013938904, -0.1868649572134018, -1.9134771823883057, -0.0902286022901535, -0.5726615786552429, -0.6078272461891174, -0.05188167095184326, -2.0389444828033447, -1.1757270097732544, 1.213140845298767, 1.825069785118103, -0.9284049868583679, 0.2670665681362152, -0.6202585697174072, 0.6662293076515198, -0.611585795879364, -1.4712077379226685, -0.45870131254196167], [0.7725290060043335, 0.6427143216133118, -1.8829950094223022, 0.3616318702697754, 0.23973998427391052, -0.04915764555335045, 0.2711029052734375, 0.10235921293497086, -1.2746397256851196, 1.098441481590271, 1.673581838607788, 0.22090047597885132, 0.09310396015644073, -0.08545584976673126, -0.14135676622390747, -0.03159848973155022, -0.20579136908054352, 0.37697672843933105, -0.9027908444404602, 2.117823839187622, 0.05876196175813675, -0.1386561393737793, 1.5194538831710815, -0.035318002104759216, 1.283945918083191, 2.318873405456543, 0.33275938034057617, -1.2719452381134033, -0.44236448407173157, 0.11451832950115204, -0.6575176119804382, 0.30837196111679077], [-1.1709692478179932, -1.2728763818740845, -0.24037688970565796, 1.6181323528289795, -0.32465851306915283, -0.40416473150253296, 0.525206983089447, 1.073972225189209, 0.17226867377758026, 0.6731857061386108, 1.5325630903244019, -0.3259124755859375, 0.23076653480529785, 0.032976262271404266, -0.38628044724464417, -0.1675208956003189, -0.023391123861074448, 1.5278438329696655, -0.7760737538337708, 1.0709110498428345, 0.10642649233341217, -1.0557266473770142, 0.03319048136472702, 0.05045175924897194, -1.8520004749298096, -0.386033833026886, -0.8509529829025269, 0.3171099126338959, -0.11518514156341553, 0.8539466261863708, 2.2769224643707275, -0.918595016002655], [0.9036626815795898, 0.46108895540237427, -1.1319515705108643, 2.3324222564697266, 0.8132686614990234, 0.9187514185905457, -1.9313594102859497, -1.0882508754730225, 0.289699524641037, 0.7780974507331848, -1.6104837656021118, -1.5681034326553345, -1.610844612121582, 0.8829956650733948, 2.8246726989746094, -1.1680104732513428, -0.8424462080001831, 0.3674653172492981, -1.0727723836898804, 1.5493860244750977, 0.75002121925354, 0.33748435974121094, 0.3900129199028015, -1.3422154188156128, -0.40052592754364014, 1.3697694540023804, -2.0470385551452637, 0.0685938224196434, 2.046133041381836, -1.4114587306976318, -0.6792126893997192, -1.1187262535095215], [-0.7043177485466003, 1.466707468032837, -0.2699489891529083, -0.754770040512085, -1.5546035766601562, -0.28519243001937866, -0.6162558197975159, -0.7093762755393982, -1.6405717134475708, -0.8430147171020508, -1.1715391874313354, -0.6594907641410828, 2.596536636352539, 0.11329303681850433, 0.21194328367710114, 0.3043944835662842, 0.5753132700920105, 2.074834108352661, -0.4984772503376007, 1.9725736379623413, -0.42408469319343567, -0.5853457450866699, -0.5136088132858276, -0.13860483467578888, -0.29296666383743286, 0.39876726269721985, 0.8154488801956177, 1.7015316486358643, 0.866942822933197, 0.13808514177799225, -0.45920002460479736, 2.461989402770996], [2.7619714736938477, 0.9994170665740967, -0.8693909645080566, 1.1534229516983032, -1.3120144605636597, -0.1058090403676033, 1.1605050563812256, -0.45086225867271423, 0.030805401504039764, -0.234359472990036, 0.8183170557022095, 0.2327970266342163, 0.7235733866691589, 0.654289722442627, 0.3167276382446289, -0.6010192632675171, -0.08652135729789734, 0.5161341428756714, -0.7068254947662354, 1.2588268518447876, -0.24668505787849426, 0.262195348739624, 0.5143377780914307, -0.5035656094551086, -0.5552788972854614, -1.2301161289215088, 0.6857351660728455, 0.6909632682800293, -1.222104787826538, -1.2936890125274658, -1.3109654188156128, -1.4498801231384277], [-1.779539704322815, -1.4485398530960083, -0.46452730894088745, -0.23989561200141907, 0.3218657672405243, -1.199181079864502, 0.3589162230491638, 0.46427059173583984, 1.0732728242874146, -0.2449287474155426, -0.07618222385644913, 0.09231972694396973, 0.2445068359375, -0.5274561047554016, 0.08543457090854645, 1.3035776615142822, -0.34752702713012695, -0.32154977321624756, 0.08324764668941498, 1.3853261470794678, 1.0517566204071045, -0.023485276848077774, -0.24328702688217163, 0.7485128045082092, 1.1587923765182495, -1.3526201248168945, 0.057098522782325745, 0.7364445328712463, 0.3886318504810333, -0.9273498058319092, -0.24500858783721924, 0.2900266945362091], [0.41757217049598694, -1.1841856241226196, 0.6421510577201843, 0.43748706579208374, 2.202298641204834, 0.14280419051647186, -1.1550028324127197, -0.09286465495824814, -0.2861030101776123, 0.4099739193916321, -1.8251632452011108, -2.1908352375030518, 0.07315108925104141, -1.2341452836990356, -0.9938411712646484, 0.39099398255348206, -1.2001689672470093, -0.9697657227516174, 0.567298412322998, 0.6859157085418701, -0.8801729083061218, -1.5875016450881958, -0.018664220348000526, 0.24754494428634644, 0.6983855366706848, 0.6059991717338562, 0.629525899887085, 1.3049932718276978, 0.34433701634407043, -1.4825791120529175, -0.13395266234874725, 0.6183956265449524], [0.059687934815883636, 0.6723896265029907, -0.6265442967414856, -1.8765177726745605, -1.3403294086456299, -0.9483065009117126, 0.4930121898651123, 1.6377850770950317, 0.3890913724899292, 1.2887227535247803, -0.40692946314811707, 0.08262249082326889, -0.8623753190040588, 2.200348138809204, 0.631106436252594, -0.08904330432415009, -0.07307690382003784, 0.261763334274292, 0.17927664518356323, 1.8297373056411743, 0.16551895439624786, 0.32325857877731323, 0.0843726098537445, 1.138897180557251, 0.721762478351593, -1.0401710271835327, 0.0681803971529007, 0.6382131576538086, 0.8558490872383118, 1.729925513267517, 0.2172849029302597, -0.1516823172569275], [0.7988386750221252, 0.14122898876667023, -0.1578902006149292, -0.5242502093315125, -0.317217618227005, 1.1715929508209229, 0.8304758667945862, 0.8082641363143921, 1.7300339937210083, 1.1372126340866089, 0.47861412167549133, 0.9823330044746399, -0.3356061577796936, -0.7847197651863098, -0.8896917700767517, 0.020384304225444794, 1.111224889755249, 0.3210633099079132, -1.4892888069152832, 1.535786747932434, -0.05554845929145813, 0.425282746553421, 0.5005847215652466, 0.18358910083770752, 0.5450844168663025, -1.3044167757034302, -0.6546623706817627, 0.7900117039680481, 1.7989716529846191, 0.3095383048057556, 0.6159294247627258, 0.7829006314277649], [-0.44231873750686646, -0.20798222720623016, -0.8637152910232544, 0.3097583055496216, 0.6172316074371338, 1.2190465927124023, 1.5653735399246216, -0.3823544383049011, -0.3380080759525299, 0.36774900555610657, -0.2979118227958679, 0.7789182066917419, -1.2687650918960571, 0.7853047251701355, 0.7454054951667786, 0.2011091262102127, 0.6626692414283752, -0.08880612254142761, -1.1741230487823486, 0.5911548733711243, 1.7914385795593262, 0.74043869972229, 0.8647770881652832, -1.4938654899597168, 0.7914609313011169, -0.19520212709903717, -0.2524343430995941, 0.1238911896944046, 0.36118343472480774, 0.5938025116920471, -0.7428876161575317, 3.431082248687744], [0.5560271143913269, -0.7331259846687317, 0.005650984589010477, -0.43511733412742615, 0.9447568655014038, -0.8831132054328918, -0.8457037210464478, -0.493748277425766, -1.7169677019119263, 0.15119022130966187, -1.259023666381836, 2.645662546157837, 0.12474558502435684, -0.9825194478034973, 0.46574893593788147, 1.623361587524414, 0.6076727509498596, 1.5557053089141846, 1.4946017265319824, -0.14603421092033386, -1.4276591539382935, -1.6285638809204102, -1.196470022201538, -0.6731594800949097, 1.353289246559143, 1.0577949285507202, 0.9149465560913086, 0.328509122133255, 0.37661096453666687, -1.8324552774429321, 0.4203445315361023, -0.8800461888313293], [-0.8563318848609924, 1.1028642654418945, 1.3935658931732178, 0.6392776966094971, 0.04715314134955406, -1.2036480903625488, -1.2016016244888306, 0.1706492304801941, 0.8692869544029236, 1.098386287689209, -0.15419213473796844, 1.8616855144500732, 0.2594924867153168, 0.19323022663593292, -0.579330563545227, 1.3576314449310303, -0.710828423500061, 0.66352778673172, -2.13328218460083, 1.0032932758331299, -0.09794677793979645, -0.06835594773292542, 0.1941961795091629, -0.9495899081230164, 2.3937079906463623, 0.37572136521339417, 0.3007502257823944, 0.12457910925149918, -1.5497682094573975, -0.383127361536026, 0.35966384410858154, 1.1615768671035767], [0.36154308915138245, -2.6023366451263428, -0.03866063058376312, -0.30460241436958313, -1.343047022819519, 0.5751636028289795, 0.9982055425643921, -0.5032409429550171, -0.6163613200187683, -0.2763019800186157, 0.14429226517677307, 0.4667200446128845, 0.6593965291976929, -0.4691191017627716, -0.6218068599700928, -1.0571383237838745, -0.1736278086900711, 0.3417900502681732, 1.4809215068817139, 0.007383949588984251, 0.776991069316864, -0.03496792912483215, -0.9157341718673706, 0.37029847502708435, 1.531313180923462, 0.5579145550727844, 0.6086817979812622, -0.25652164220809937, -0.06076985225081444, 0.2029002159833908, 1.4236825704574585, 0.4079379737377167], [-1.2690880298614502, 0.9256107211112976, 0.8888503313064575, 0.4802606701850891, 0.10667088627815247, 1.0897573232650757, 0.2391013503074646, 0.05399897322058678, -1.5210024118423462, -0.2755855619907379, -1.1912704706192017, -0.729079008102417, -0.10452617704868317, -0.9389362931251526, -1.317487120628357, 0.4037707448005676, -0.14705544710159302, -1.8220936059951782, 0.7831956744194031, 1.277189016342163, 0.2260458916425705, -0.8032744526863098, 0.3619749844074249, 0.3342811167240143, 1.3508987426757812, 0.871441125869751, -0.2038995623588562, 0.5389184355735779, 0.4526011347770691, -1.369864821434021, 1.5331318378448486, -0.7046785354614258], [-0.412153035402298, 0.696436882019043, 1.4139050245285034, -0.9841524958610535, 1.0284069776535034, -1.716631531715393, -0.0065397038124501705, 0.22407609224319458, -0.10098793357610703, -0.6484308838844299, -1.7431046962738037, 0.046862516552209854, 0.9881755113601685, 0.6874929070472717, -1.9405860900878906, 0.9976247549057007, -0.5766624212265015, -0.8399358987808228, 1.6295768022537231, -0.041787564754486084, -0.008690555579960346, 0.27727869153022766, 0.4355219900608063, -0.20924806594848633, 0.2528274357318878, 1.5994845628738403, -3.044663906097412, 0.14815320074558258, 0.07002948224544525, 0.6731809973716736, 0.5470933318138123, -1.7886232137680054], [-1.370944619178772, 1.3874562978744507, 1.2884641885757446, -0.849351704120636, -1.9553520679473877, 0.5654689073562622, -0.40355420112609863, -0.7599831819534302, 0.4322148561477661, 0.41036033630371094, -0.1691855490207672, -1.6996045112609863, -0.7644228935241699, -1.3691743612289429, -2.315279245376587, -0.3409225642681122, 0.14942151308059692, 0.7170342803001404, 0.2084558755159378, 0.24601492285728455, 0.11411810666322708, 1.8067002296447754, -0.8282616138458252, -0.1444275677204132, 0.38625413179397583, -0.886931836605072, -0.8552778959274292, -1.5653399229049683, 0.6471427083015442, -0.712735652923584, 1.1685659885406494, -0.7581971883773804], [0.9673264026641846, 0.8117170929908752, 0.4577048122882843, 1.3846189975738525, -0.041931405663490295, -1.1072882413864136, -0.7036515474319458, -0.7909972667694092, -0.6187267899513245, -0.8916478157043457, 0.7910619974136353, 0.7839093208312988, -0.12322426587343216, -0.12362077832221985, 0.39566704630851746, 0.5089229941368103, 0.46004411578178406, -0.3159237205982208, -0.18974223732948303, 0.3237287998199463, -1.79902982711792, 0.19719983637332916, -0.9404017925262451, -1.1826672554016113, 0.9092352986335754, 1.1350606679916382, -0.8307101130485535, 1.4108741283416748, 1.3357353210449219, 1.8933438062667847, 0.5795800685882568, -1.2056734561920166], [-0.06572476029396057, -2.8212130069732666, 0.5274049639701843, -0.042812515050172806, -0.8240911364555359, -0.9918552041053772, 1.086578130722046, 2.3874549865722656, 0.5735535621643066, -0.33821985125541687, -0.1938313990831375, -0.21467967331409454, -0.12740270793437958, 0.6048633456230164, -0.07110359519720078, 2.424865484237671, 0.2325158715248108, -0.8202218413352966, -1.5252059698104858, -1.23261559009552, -0.31792718172073364, 0.24230726063251495, 1.463119626045227, -0.5587934255599976, 1.1656689643859863, 0.9962406754493713, -0.30127647519111633, 0.6390269994735718, 0.24841760098934174, 0.46227896213531494, -0.9416986703872681, 1.4979774951934814], [-1.1154849529266357, -0.07018831372261047, -1.5042648315429688, -0.8258868455886841, 0.41374117136001587, 0.9834182858467102, -0.1828227937221527, 0.8527356386184692, 0.8663709163665771, -0.35654154419898987, 0.7278257608413696, -0.6951546669006348, -0.2032020390033722, -0.7856940031051636, 1.0161436796188354, -0.0068348608911037445, -0.48314112424850464, -0.07281351834535599, -0.8395459651947021, -1.198549747467041, -0.5994735956192017, -0.6112658381462097, 0.7160404324531555, 0.5715383887290955, -1.5082913637161255, -0.8114249110221863, -0.6578834056854248, 0.716648280620575, -0.14154689013957977, 0.42056065797805786, 0.3709162771701813, -0.27422913908958435], [-0.2844386398792267, -0.5054111480712891, 0.20589132606983185, -0.29725608229637146, -0.33830952644348145, -2.407712936401367, -0.27577561140060425, 0.8603191375732422, 0.8385656476020813, -1.1644030809402466, -0.5210042595863342, -0.6011080741882324, -0.22645515203475952, 0.025737112388014793, 0.7164391875267029, 2.5355823040008545, 0.20180563628673553, -0.16115620732307434, -0.7583320736885071, -3.003652811050415, 1.2847222089767456, -1.0954524278640747, -0.3083604872226715, -0.06542008370161057, -0.17877131700515747, -0.4923784136772156, 1.1667349338531494, 0.4052158296108246, 0.20892778038978577, -3.0077922344207764, -0.8665962219238281, -0.32772737741470337], [0.4359409213066101, 0.12041500955820084, 0.33023011684417725, -1.5793299674987793, -1.7626454830169678, 0.5514583587646484, -0.38748013973236084, -0.7516180872917175, -0.1631532460451126, 0.6518006324768066, -1.0312151908874512, 0.2165687382221222, -1.5314685106277466, -2.2125842571258545, 2.6955018043518066, 0.26634466648101807, 2.0084805488586426, -0.43864744901657104, 0.3730107545852661, -0.4934808313846588, -1.5148041248321533, -0.2818613350391388, -0.24990224838256836, -0.034637272357940674, 0.6781110763549805, 0.6835705637931824, 1.2819736003875732, -0.463665246963501, 0.3469151556491852, 0.6744967699050903, 1.1788148880004883, 0.7587233781814575]], [[-0.7713785171508789, 0.22343406081199646, -0.11667456477880478, -0.6509381532669067, 1.8312207460403442, -0.6138396859169006, 1.995156168937683, -0.6161959767341614, -0.8783025145530701, -1.0582467317581177, 1.8495588302612305, 0.007976903580129147, 1.7538994550704956, 2.221278190612793, 1.0912153720855713, -1.6345044374465942, 0.793724000453949, -1.2049834728240967, -0.5494323968887329, -0.3787364363670349, 2.435338258743286, 0.10047660768032074, 0.4973605275154114, -0.9030768275260925, -0.40301308035850525, 0.3019244968891144, 0.06313247233629227, 0.5673832893371582, 0.7016668915748596, 2.2695798873901367, 0.1972828209400177, 0.8354983925819397], [-0.35007810592651367, 0.6223260760307312, 0.2573120594024658, -0.14319145679473877, -0.12602737545967102, 1.537570595741272, 0.4175148010253906, 0.4030625820159912, -1.2824286222457886, 0.3114335834980011, 3.202193021774292, -0.7062605023384094, -0.9249488711357117, -1.3497570753097534, -0.6272128224372864, -0.9566219449043274, 0.3331548273563385, -0.2448762208223343, 1.8384082317352295, -2.8990256786346436, -0.35381418466567993, 0.2235993891954422, 0.9622626304626465, 0.10818588733673096, 0.9603262543678284, -1.1621832847595215, -0.98609459400177, 0.3373306095600128, 0.29977312684059143, 0.43694034218788147, 0.5397254228591919, 0.5500810146331787], [-0.04861065000295639, 0.4128851890563965, -0.45709866285324097, 1.2034705877304077, -0.9688860774040222, 0.5133678913116455, -0.9200504422187805, -0.19374896585941315, 0.8286169767379761, 0.3528951406478882, 0.04465724155306816, 0.6296355128288269, 1.446246862411499, 0.9193735718727112, -0.8636066913604736, -0.09563030302524567, -0.06403111666440964, -1.2156226634979248, -0.11699104309082031, -0.7295717597007751, 0.800588071346283, -1.4705754518508911, 0.20700514316558838, 0.057707056403160095, -0.014716469682753086, -0.32910245656967163, 0.6565617918968201, 0.950999915599823, 1.1110457181930542, 0.5255821347236633, -0.8936140537261963, 0.05815611407160759], [-0.32997238636016846, 1.1510756015777588, 0.16568078100681305, -0.5012379884719849, -0.4462374448776245, -0.10536351799964905, -0.5582216382026672, 1.653485894203186, 0.7789797186851501, 0.18496933579444885, 0.05545690655708313, 0.05998248979449272, 1.0488736629486084, -2.0911874771118164, 2.3355705738067627, 1.3824824094772339, 0.8644337058067322, -0.651228129863739, 0.3432198464870453, 0.5130545496940613, -0.45651304721832275, 0.021009428426623344, 1.7919151782989502, 0.23289339244365692, 2.0497024059295654, -0.6762035489082336, -0.20595000684261322, -0.06020674109458923, -0.0019481723429635167, 0.3967036008834839, 0.5601879358291626, 0.36063796281814575], [-0.9454878568649292, 1.5669955015182495, 0.25973382592201233, 0.06748414784669876, 0.38960713148117065, 1.7872453927993774, 0.11887907981872559, -0.24647930264472961, 0.28818243741989136, 1.167826533317566, 1.1645435094833374, 1.040757179260254, -0.9044137001037598, 1.2241806983947754, -0.5115131735801697, 0.7640354633331299, 0.6714394688606262, -0.9679993391036987, -0.9424712061882019, 1.6584560871124268, 0.7327854037284851, -0.5559465885162354, 0.8688182234764099, -1.2292616367340088, 0.02473561465740204, 0.1717173308134079, -2.8083834648132324, 0.590696394443512, 0.5610836744308472, 0.07985076308250427, -0.26230841875076294, -0.011768764816224575], [-0.2584317624568939, -2.2060165405273438, -0.881074070930481, -0.9826215505599976, -0.22793082892894745, 0.7858145236968994, -0.5988827347755432, 0.555521547794342, -1.4236668348312378, -0.6171319484710693, 0.43406200408935547, -0.34068983793258667, -0.6335623264312744, -0.8476018309593201, 1.3101060390472412, -0.7570011019706726, 0.10645797103643417, 1.3833547830581665, 1.787214994430542, -1.5353491306304932, 1.4841721057891846, 0.24050241708755493, 1.0855720043182373, -0.33040323853492737, 0.17469219863414764, 0.761864960193634, 0.7805221080780029, 0.5704853534698486, -0.14870063960552216, 0.5452752709388733, -0.21320995688438416, 0.9524301886558533], [0.7715468406677246, -0.6083022952079773, 1.3769450187683105, 1.0892852544784546, -0.75345778465271, 1.1233853101730347, -1.6264262199401855, 0.7016897797584534, -1.1304768323898315, -0.01664150506258011, 1.0593693256378174, -0.8015260100364685, 0.98591548204422, -0.35551613569259644, 0.8169575333595276, -2.175813674926758, 0.05626218020915985, -1.7755529880523682, -0.5572887659072876, 1.2067707777023315, -0.7441467046737671, 0.6700294613838196, -1.397461175918579, 0.06846745312213898, -1.1721687316894531, 0.635610044002533, 0.3443467617034912, 0.37415578961372375, -0.2591634690761566, -0.8591368198394775, -0.7553908824920654, 1.4951368570327759], [-1.5669586658477783, 0.33051714301109314, 0.17016452550888062, -1.0605789422988892, -0.20433777570724487, -0.2098410278558731, -0.4528612792491913, 0.3994746208190918, 1.3699191808700562, -0.6614096164703369, 0.6145507097244263, -0.7474327683448792, -0.5987377166748047, -0.3217144012451172, 1.1383082866668701, 1.1061484813690186, 0.029240120202302933, -1.37104070186615, 1.2432985305786133, 0.4658292233943939, 0.5316457748413086, 1.0418665409088135, -0.40313610434532166, 0.8213452696800232, 1.366356611251831, -0.47106045484542847, -1.2620543241500854, -0.32598191499710083, 0.18768228590488434, 0.1616198718547821, -0.6817886233329773, -1.4157922267913818], [1.848336100578308, -1.191507339477539, -0.6193476319313049, -0.2230190485715866, -1.6629303693771362, -0.7248212099075317, 1.0637969970703125, 0.10980276018381119, -0.6242247819900513, -1.3498964309692383, -0.14822696149349213, 0.9588039517402649, 0.23703518509864807, 0.6209202408790588, -2.175753593444824, -1.088254451751709, -0.7880445718765259, -0.4111204743385315, 0.5699619054794312, -0.31110960245132446, 0.6153833866119385, -0.630567729473114, -0.6772530674934387, 0.36189067363739014, 1.3477272987365723, -0.3861984312534332, -1.230357050895691, 0.6133469343185425, -1.2281032800674438, 1.7006100416183472, -1.0655614137649536, -1.3174808025360107], [1.4030035734176636, 0.9870775938034058, -0.2867872714996338, -1.463051438331604, -0.34609588980674744, 0.538180410861969, 0.551541268825531, 0.7897809147834778, 0.17063026130199432, -0.050509802997112274, -1.473402976989746, 0.7965177297592163, -0.6159272193908691, 0.5543360710144043, 0.13542971014976501, -0.21328149735927582, -0.29263558983802795, -1.293815016746521, -0.9906918406486511, -0.9096913933753967, -1.3787978887557983, -0.7494104504585266, 0.14753228425979614, -1.9779438972473145, -1.1780314445495605, -1.4453188180923462, 0.8337213397026062, 0.08749562501907349, -0.007612342480570078, 0.4904296100139618, -0.3673118054866791, -0.19879405200481415], [2.757538080215454, 0.7493932843208313, -0.28288835287094116, -0.2734910845756531, 1.2437278032302856, -0.721250057220459, -0.9828550219535828, -0.17067104578018188, -0.12439562380313873, -0.14456480741500854, 0.05389036610722542, 2.574878692626953, -0.3408949375152588, -0.13552458584308624, 0.5280792117118835, 0.4997105598449707, -1.9454911947250366, -0.4325263500213623, 0.5670034885406494, 0.9638186097145081, -0.686722457408905, 0.3609025180339813, -0.5089329481124878, 0.9642487168312073, 1.390231966972351, 0.15797767043113708, 0.2994343340396881, -0.6520748138427734, 0.40812981128692627, -0.4897189438343048, -1.1696851253509521, 0.8604279160499573], [-1.2002302408218384, 1.1053229570388794, 0.6990070343017578, 0.2538059651851654, 1.5202480554580688, -1.5112133026123047, -0.44065412878990173, 0.05828441306948662, 0.5961799025535583, 1.238419532775879, 1.3466721773147583, -0.39020830392837524, -1.4757572412490845, -0.24811075627803802, 1.2182466983795166, 1.981363296508789, -0.20367859303951263, 0.513149619102478, 0.04374129697680473, -0.4198016822338104, -0.4600572884082794, -0.4277191460132599, -1.0233211517333984, 0.5078672766685486, -0.08502070605754852, -0.3427426815032959, 1.1640233993530273, -0.3235565721988678, -0.6830931901931763, -0.4420125186443329, -0.4383760988712311, 0.4540596902370453], [0.2228783667087555, 0.9940091371536255, -0.16056333482265472, 1.3384616374969482, -0.7610649466514587, -0.7677697539329529, 0.858095645904541, -0.3917253613471985, 1.394553780555725, 1.2241963148117065, -2.01301646232605, -1.2833338975906372, 0.8895232677459717, 0.7687597274780273, 0.06937672942876816, 0.06383085995912552, -1.118613839149475, 1.775818109512329, -1.440794825553894, -1.32120680809021, -1.0428200960159302, 0.9003942012786865, 1.4486528635025024, -1.0595967769622803, -2.224306583404541, -0.4105982184410095, -0.6496056318283081, 1.549116849899292, 0.3886535167694092, -0.3978451192378998, 1.1004458665847778, -0.350383460521698], [-0.8154924511909485, 0.17179135978221893, -1.065019965171814, 0.6127496957778931, 0.5093249678611755, 0.19833756983280182, -0.5517974495887756, 0.2350708395242691, 0.12369710206985474, 0.1605791598558426, 1.436372995376587, 0.8051887154579163, 0.4049898087978363, -1.661622405052185, -0.5565491914749146, -0.09186815470457077, 0.377148300409317, 0.29173532128334045, -1.0864365100860596, -0.8210675120353699, 0.1952352523803711, 0.4999423921108246, -0.1375097632408142, -0.7311725616455078, 0.4486176669597626, -0.2881668508052826, 1.7466163635253906, -1.7510926723480225, -1.1879550218582153, 0.7717546820640564, 0.35660576820373535, 0.3128308653831482], [0.013881340622901917, -0.4170786142349243, -1.4795852899551392, 2.641281843185425, 0.7451174259185791, -0.7307285070419312, 0.5184412598609924, -0.42083069682121277, -1.1017411947250366, -0.30712729692459106, 0.6167430877685547, -0.5851898193359375, 0.8934365510940552, 1.7996366024017334, -0.4442542791366577, 1.9213989973068237, -0.942935585975647, 1.2078008651733398, 2.0878922939300537, -0.5852863788604736, -0.18078197538852692, -0.08359870314598083, -2.1923441886901855, -0.4088748097419739, -0.8583415746688843, 1.7436293363571167, -1.064672827720642, 0.11446554213762283, -1.8148785829544067, -1.205903172492981, -0.030971884727478027, 0.485163152217865], [1.0637092590332031, 0.6330089569091797, -0.9981475472450256, 1.8475860357284546, 0.24684427678585052, -1.8263754844665527, -1.1908228397369385, 1.7937555313110352, -0.6649338006973267, -0.9960284233093262, -1.092726230621338, 0.8950160145759583, 0.4110985994338989, 1.4496357440948486, -0.3955954313278198, 1.923750400543213, -0.2726860046386719, 0.039155736565589905, 0.6094951629638672, 1.193705439567566, 0.5337008237838745, -1.9701852798461914, -0.22639693319797516, -0.50486159324646, 0.09059998393058777, 1.5672638416290283, -0.7345809936523438, -0.2592151463031769, -1.5024573802947998, -0.9576352834701538, -1.925135850906372, -0.7129324078559875], [-0.5128256678581238, 0.8041374087333679, -1.4850157499313354, -0.5041527152061462, 0.28458091616630554, 2.0586025714874268, 0.5940200686454773, 0.07869232445955276, 0.39365893602371216, -0.6369661688804626, 1.2020083665847778, 1.0518909692764282, 1.2084962129592896, 0.36197593808174133, -0.3089364767074585, 1.023625135421753, -0.1904500275850296, -0.8795551061630249, -0.21042576432228088, 1.1474170684814453, 0.896755039691925, 2.4722743034362793, -0.5865059494972229, -1.5069845914840698, 1.2214747667312622, -0.0270368754863739, 0.11532249301671982, -0.4441728889942169, -1.2066091299057007, -0.5391182899475098, -2.5161683559417725, -0.40732407569885254], [-0.38025376200675964, -2.382831573486328, 0.5763428807258606, -0.8416930437088013, -0.10476046055555344, -0.4472358226776123, -0.26038438081741333, 0.24381382763385773, -0.5838993787765503, -1.006765604019165, -0.16929322481155396, 0.7970706820487976, 0.3687424659729004, 1.146480917930603, -0.52962726354599, -0.7059024572372437, -0.10587577521800995, -1.1722197532653809, 0.8676092028617859, -0.1493692845106125, -0.8369441032409668, 1.331677794456482, 0.5284180641174316, 0.26336315274238586, 0.025471217930316925, -0.4772294759750366, -1.146913766860962, -0.05788581445813179, 1.1209338903427124, 0.13480131328105927, 0.7499953508377075, 1.0509557723999023], [0.5817117691040039, 0.8478255867958069, -1.302790880203247, -0.7730226516723633, -0.6057371497154236, 0.6309098601341248, -0.45689430832862854, -0.06884042918682098, -0.8463038802146912, 1.227105975151062, -0.25091981887817383, 0.8042747378349304, -0.21855022013187408, -0.5309653878211975, -0.3642626702785492, 1.1427122354507446, -0.5563648343086243, 0.5062369108200073, 0.022821512073278427, -0.024614423513412476, -0.6198036074638367, -0.22897294163703918, -1.6825182437896729, -1.1693366765975952, -1.8592253923416138, -0.08243914693593979, 1.3849250078201294, 0.931348979473114, 0.8990535736083984, -0.8224048018455505, 0.8201887011528015, 0.6772635579109192], [-1.2908401489257812, -0.6866270899772644, -0.4880399703979492, 0.8871422410011292, -0.3011549115180969, -0.7872869968414307, 0.10328745096921921, -0.5885282754898071, -0.6198603510856628, 0.2990664541721344, 1.1715245246887207, 1.2385975122451782, -0.31261181831359863, 2.0694894790649414, -1.557870626449585, 0.9990781545639038, -1.1121773719787598, 0.6802557706832886, 0.1411152482032776, -0.8873862028121948, 0.9385293126106262, -0.38894689083099365, 2.3606932163238525, -0.8910303115844727, 2.316164255142212, -1.2091761827468872, -0.20086725056171417, -0.023476190865039825, 0.5343303084373474, 0.6038948893547058, -2.7993967533111572, -0.9039674997329712], [1.1945812702178955, -0.38724929094314575, 0.6858965754508972, 0.3962780833244324, -0.28283387422561646, -0.6759811639785767, 2.759605646133423, -0.48848244547843933, 0.9252641797065735, 0.6789382696151733, 0.3502451479434967, -0.9267939925193787, 1.1583329439163208, 0.8161296248435974, -0.2287142425775528, -2.1870806217193604, 2.8764851093292236, -0.5900743007659912, -0.8051034808158875, -1.174669623374939, -0.04655388370156288, 0.9069337248802185, -0.4968722462654114, -1.8625121116638184, -1.4370205402374268, -1.2332288026809692, 1.6979544162750244, -0.27388796210289, 1.6055042743682861, -1.6510587930679321, 0.28369808197021484, 0.6910306811332703], [0.291401743888855, 0.7639490365982056, 0.9371494054794312, 1.2951018810272217, 0.08914333581924438, -1.5999377965927124, 1.8719786405563354, 2.4694130420684814, -0.967109203338623, -2.2023816108703613, -2.6272928714752197, 0.13385429978370667, -0.5606155395507812, -0.1898457556962967, -0.34403157234191895, 1.86599862575531, 0.4090639352798462, -0.5041844844818115, -0.8482745289802551, 0.18056553602218628, 0.9081627726554871, 0.6773595213890076, 1.568444013595581, 1.37460458278656, 1.3306406736373901, 0.9322927594184875, 0.7846049666404724, 1.7404717206954956, -0.3674524426460266, -0.7979192137718201, 0.22945380210876465, -0.1882629543542862], [0.30126771330833435, 0.08963892608880997, 0.10887755453586578, -1.465309500694275, -0.8341270089149475, -0.2681334316730499, 0.800207793712616, 0.43193039298057556, 0.6628292202949524, -1.0179879665374756, -1.8617140054702759, -0.31176459789276123, -1.091267704963684, -0.8163251280784607, -0.13837173581123352, -0.5977528691291809, -0.6363115906715393, -0.11596648395061493, 0.008152300491929054, 0.9535180330276489, -0.3295568823814392, -1.202132225036621, 1.3767516613006592, -0.8066769242286682, -0.43550795316696167, 0.41316866874694824, 1.0479308366775513, -2.0662355422973633, -1.9045320749282837, -3.022613525390625, -0.342405766248703, 1.1651536226272583], [0.6790107488632202, 0.6946365833282471, -1.0281407833099365, -0.6481346487998962, -0.01307877991348505, 0.8287391662597656, -0.20997372269630432, -0.49392449855804443, -1.4274544715881348, -0.9949690103530884, -0.48245862126350403, -1.6732840538024902, 0.5922377705574036, 0.4781915247440338, -0.12956120073795319, 0.3350423574447632, -2.3720178604125977, 0.16929295659065247, -1.186360239982605, 0.2305336594581604, -1.4479870796203613, 0.7770633101463318, 0.00024397496599704027, 0.8221709728240967, 0.5475000739097595, -0.5918369293212891, -1.0559719800949097, -1.8548563718795776, -0.3955472707748413, -0.9199278354644775, -0.9709778428077698, -0.11864478886127472], [-0.8724110722541809, -0.22910736501216888, 0.47213923931121826, -1.3840937614440918, 0.12667928636074066, 0.08393819630146027, 2.3446390628814697, 2.1367926597595215, -0.3928564190864563, -0.8435924053192139, -0.43965527415275574, 1.1770716905593872, 0.425699919462204, 0.31424278020858765, -0.8458221554756165, -0.2657042145729065, 0.22570741176605225, 0.42105528712272644, -0.05512469261884689, 1.3365422487258911, 2.699164867401123, 0.35937923192977905, -0.7616913914680481, 0.38765424489974976, 1.8340927362442017, 0.024067025631666183, -2.042837381362915, -0.3231697082519531, 1.060760259628296, -0.7823047637939453, -0.5144880414009094, -0.019767766818404198], [-0.4998011291027069, 0.9029010534286499, 0.7342731356620789, 0.32311007380485535, -1.0801056623458862, -0.7132835388183594, -0.10390862822532654, 0.425428569316864, -0.4697421193122864, -1.056659460067749, -0.6331418752670288, -1.0126831531524658, 1.0796887874603271, 1.1359490156173706, -0.18975968658924103, 0.6468948721885681, 0.41548269987106323, 1.5945961475372314, -0.7447066307067871, -0.777550995349884, 0.3992573916912079, -0.5625028610229492, 0.13765259087085724, 1.436510682106018, -0.3547673225402832, 0.16858826577663422, -0.23772269487380981, -1.235567331314087, 0.7613363265991211, -0.4427943527698517, 0.5793875455856323, -0.7683439254760742], [-1.871010661125183, -0.12481577694416046, -1.480699896812439, 0.7998395562171936, 1.8816982507705688, 1.4597126245498657, -0.8999819159507751, 0.029042938724160194, -1.5797263383865356, -0.11010682582855225, 0.4234621524810791, 0.5465670824050903, 0.989536702632904, 0.10305099934339523, -0.8317745923995972, -1.1947795152664185, 1.2047042846679688, -1.2551383972167969, 0.3260827362537384, 1.7158043384552002, -0.4521368741989136, 0.3403678834438324, 1.5224344730377197, 1.2231464385986328, 0.8380191922187805, -0.37427300214767456, 2.2182772159576416, -0.06827250868082047, -1.1640535593032837, 0.20440112054347992, -1.2926578521728516, -0.9844195246696472], [1.261118769645691, -2.3373923301696777, 0.22022317349910736, -0.021852148696780205, -2.5189268589019775, -0.7873755097389221, 0.1294303983449936, -0.9402698278427124, -2.496493339538574, 0.20252059400081635, -0.9695917367935181, -1.5226585865020752, -1.3256539106369019, -0.32640913128852844, -0.7610647678375244, -1.2769006490707397, 0.6648937463760376, -0.11230308562517166, 1.8822427988052368, 0.9658526182174683, 0.35753634572029114, -0.3450861871242523, -2.175469398498535, -0.35879039764404297, 0.07440222054719925, -0.22961583733558655, -1.2040019035339355, -0.6941867470741272, 0.7819393873214722, 0.7346673011779785, -1.0463461875915527, 0.39373543858528137], [1.067439079284668, -1.8371390104293823, -0.17811836302280426, -0.2514107823371887, -0.42773327231407166, 0.9854638576507568, -0.6786842346191406, 0.5980397462844849, -0.2977936565876007, 0.6997649669647217, 0.5188602805137634, -1.4361056089401245, -0.8472863435745239, 0.6309012174606323, 2.3048882484436035, 0.9966965913772583, 0.6157786250114441, -1.3398284912109375, 0.39689335227012634, 0.41088947653770447, -0.914351224899292, 0.08033319562673569, -0.26799917221069336, 0.7705201506614685, -0.9577265977859497, -0.8070557117462158, 0.8016475439071655, -0.12171456962823868, 0.7257034182548523, -0.3281504809856415, 1.401725172996521, 1.201716661453247], [0.8240758776664734, -0.7368817329406738, 0.17058897018432617, 0.2117966264486313, 0.3318346440792084, -1.0354191064834595, 0.2373727411031723, 0.67633455991745, 1.367410659790039, 0.3964402675628662, -0.3808271288871765, 2.2134668827056885, 0.6427183151245117, -0.6568636894226074, 0.8264996409416199, -0.005497075617313385, -0.08571431040763855, -0.01471811905503273, 0.2153690606355667, 0.23582741618156433, -0.8289918899536133, -0.6122223138809204, 0.482281357049942, 0.6048235893249512, 0.2728550434112549, -0.13559940457344055, -2.382875442504883, 0.49508875608444214, 1.929980993270874, 0.08680375665426254, -0.6451829671859741, 1.3567860126495361], [-0.8004206418991089, -0.0383802130818367, -0.15127599239349365, -0.022351099178195, -1.0386542081832886, 1.184805154800415, -1.0451875925064087, -1.4693838357925415, 0.20673690736293793, 0.49321675300598145, -0.26045921444892883, -0.10103289783000946, 0.6017962098121643, 2.037583351135254, 0.7499025464057922, 0.25923192501068115, 0.12415286153554916, -0.16402114927768707, 0.37344539165496826, -0.700779139995575, 0.025195667520165443, 0.1485632359981537, 0.34667733311653137, 1.3401916027069092, -0.522648274898529, -2.3280327320098877, -1.7576887607574463, 1.9092719554901123, -1.914894461631775, -1.2062066793441772, 0.33939671516418457, -0.3689691126346588], [-0.3171524107456207, 0.09116515517234802, -2.594688892364502, -0.1222962886095047, 0.7119452953338623, 0.05706185847520828, -1.6398195028305054, 1.6934866905212402, 0.4356526732444763, 0.3351050913333893, -0.5366387963294983, 0.7599347233772278, 0.4392542839050293, 0.4667280316352844, 1.108690619468689, -2.315725326538086, -0.386581152677536, -0.7849421501159668, 0.25192689895629883, 1.9926812648773193, -3.020779848098755, 1.312065839767456, -0.3206010162830353, -0.3587004542350769, -0.9519296288490295, -0.7852867841720581, -0.9992215633392334, -0.07313643395900726, 0.5504921078681946, 0.2103516012430191, -1.3799777030944824, 0.48038485646247864], [0.23810291290283203, -1.786021113395691, -1.697747826576233, 2.055302143096924, -0.6639370322227478, 1.0202513933181763, -0.30100390315055847, 1.2190898656845093, 0.428374320268631, 1.6263000965118408, 0.020617784932255745, -0.9966953992843628, -1.0223373174667358, 0.5697574019432068, -1.2256128787994385, -1.525299310684204, -0.9624378085136414, -0.035273049026727676, -0.2941390872001648, -0.47500094771385193, -1.3305277824401855, 0.56629478931427, -1.6157777309417725, -1.0251672267913818, -1.7770707607269287, 1.141512155532837, 1.39284086227417, -1.8330440521240234, -0.33884215354919434, 0.33067184686660767, 0.35447192192077637, -0.16708751022815704], [-0.3793458640575409, 0.3186335861682892, 1.0496262311935425, 0.649139404296875, 0.31095021963119507, 0.5144075155258179, 2.0436062812805176, 1.6924582719802856, 0.646020770072937, -1.692136526107788, -0.7221372723579407, 0.07026128470897675, -0.3186858892440796, -0.7727658152580261, -2.1025893688201904, -1.2857168912887573, -0.7984532117843628, 0.32220298051834106, -0.23528099060058594, 0.17219294607639313, 2.9207923412323, 0.17635473608970642, -0.7082566618919373, -2.6267008781433105, -0.746971070766449, -0.35301586985588074, 1.9907481670379639, -0.6847571730613708, -1.3908754587173462, -0.9393792152404785, -0.7048850059509277, -0.39128121733665466], [0.9156213402748108, 0.01291128434240818, 0.3756002187728882, 0.07780392467975616, 1.0964195728302002, -1.8462215662002563, -0.12767170369625092, -0.40066516399383545, -0.8309407830238342, -0.9232213497161865, -0.8831107020378113, 0.7420789003372192, 0.6080892086029053, 0.5482624769210815, -0.2885531783103943, -0.47152531147003174, -1.6846901178359985, -1.0098235607147217, -1.6257777214050293, 0.042263053357601166, -0.7087224125862122, -0.12220889329910278, -0.2300516813993454, 2.171027898788452, -0.35638681054115295, -0.3717210292816162, -0.8742733597755432, 0.5088931322097778, 0.08908668905496597, -0.08293727785348892, 1.2585347890853882, -1.266449213027954], [2.2354838848114014, -0.3971482515335083, 1.2794044017791748, -0.695273220539093, 0.8828217387199402, -0.19727662205696106, -0.9013927578926086, -0.10764922946691513, -1.5704801082611084, 0.9527880549430847, 0.22031526267528534, -0.7247215509414673, 1.8127397298812866, -0.8133761286735535, -0.2742311656475067, -0.735916018486023, -0.8448011875152588, 0.4038180410861969, 0.4321598708629608, 0.2985958158969879, 1.5507900714874268, -2.1472578048706055, 0.5435701608657837, 0.870071530342102, 0.14958956837654114, 0.6563766598701477, 0.05736558139324188, -0.9982694983482361, 0.3395829200744629, 0.8571848273277283, -0.5575345158576965, -0.44055038690567017], [2.190936803817749, 0.7625468969345093, -0.5264710783958435, -0.577934205532074, 1.2240715026855469, 1.442480444908142, -0.9079880118370056, -0.547446072101593, 1.3629857301712036, -1.3064780235290527, 0.852103054523468, -0.0991179347038269, -0.4832713007926941, 0.1785602867603302, -1.106501579284668, -0.802272379398346, -0.7943491339683533, 1.6625392436981201, 0.7854037880897522, -2.622753620147705, -0.0048051560297608376, 0.03650926053524017, 0.14352621138095856, -0.26363474130630493, -0.46314728260040283, 0.29680800437927246, -1.1512324810028076, -0.3639933168888092, -0.2509446144104004, 0.7300338745117188, -0.6847761273384094, 0.9314407110214233], [0.975689709186554, 1.4800927639007568, -0.3788556456565857, 0.5841239094734192, 1.0229191780090332, -0.5285987257957458, 0.49881309270858765, 0.761311948299408, 0.7511870861053467, -0.1416303813457489, -1.0963541269302368, 1.2395143508911133, 1.424926519393921, -2.260901927947998, -0.18791420757770538, -0.14795595407485962, 0.02659749612212181, -0.33381739258766174, 1.3249354362487793, -0.5788910984992981, 2.3607730865478516, 1.3064879179000854, 0.5866420269012451, -0.7206926345825195, -0.9093590378761292, -0.12063496559858322, -0.1213502511382103, 1.2369356155395508, 0.01917128823697567, -1.8995170593261719, 0.17208360135555267, 0.250005841255188], [2.8330483436584473, -1.0844210386276245, 1.1036418676376343, 1.4450360536575317, 1.206472396850586, -0.07938579469919205, -1.0277493000030518, 2.1032874584198, -1.4804998636245728, -1.3231256008148193, -0.9738070368766785, 0.37446966767311096, 0.24016304314136505, -0.6810616254806519, -0.11789407581090927, 1.8903872966766357, 0.31403931975364685, 1.8473213911056519, 1.0678794384002686, 0.7498714923858643, -0.7381635308265686, 0.3613027036190033, -2.1542601585388184, 0.07489379495382309, 1.6388049125671387, 0.7392095327377319, -0.8234479427337646, -0.661686897277832, 0.5354667901992798, 0.30769994854927063, 1.632282018661499, -1.169467806816101], [0.06439842283725739, 1.8014668226242065, 0.062474120408296585, 2.127791166305542, 0.47890713810920715, 0.2645047605037689, 0.1774989366531372, 0.6308827996253967, 0.39135053753852844, -0.7436482310295105, 0.11148364096879959, 0.021087966859340668, 0.025848088786005974, -1.0354797840118408, -0.5936017632484436, -0.2512968182563782, -0.8801618218421936, -0.13131946325302124, 0.5436980724334717, -0.11304552853107452, -2.3185505867004395, 1.121300458908081, 0.4402143955230713, -0.05813322961330414, -1.8451409339904785, 0.06483858823776245, -0.9711576104164124, -3.153662919998169, -0.5405937433242798, 2.1817896366119385, -2.148475408554077, -0.7708455920219421], [-0.7065116763114929, -0.7273155450820923, -1.7874332666397095, 1.416559100151062, 0.7269187569618225, -0.45185187458992004, 0.08614151924848557, 0.5256916880607605, -0.02023077942430973, 0.6027935743331909, -1.4718824625015259, 0.05616271495819092, 0.8755918145179749, -0.19186419248580933, 0.46064695715904236, 0.7462198138237, 0.3368299901485443, -0.7525820732116699, -0.7214231491088867, 1.6876556873321533, 1.3504480123519897, -0.49104443192481995, 0.02669808454811573, -0.3782689869403839, 0.19085073471069336, 0.43506020307540894, -0.042713671922683716, -1.467297911643982, -0.14999154210090637, 0.738303005695343, 0.8835421204566956, -0.5323970913887024], [1.661162257194519, 0.21832343935966492, -0.026391929015517235, -0.2729233503341675, -0.21567033231258392, -0.21207153797149658, 1.470192790031433, 0.9991374611854553, -1.038028597831726, 0.8680484294891357, 0.8469834923744202, -0.8341507315635681, 0.13424751162528992, -0.8228889107704163, -0.035886406898498535, -0.24447160959243774, 0.25973743200302124, 0.6702456474304199, 0.636989951133728, -0.12360942363739014, 1.1629765033721924, -0.6171424984931946, 0.592934250831604, -0.5554004311561584, 0.2989753186702728, 0.7661052942276001, 1.0074886083602905, 0.5331015586853027, 0.842030942440033, -1.8158941268920898, 1.5373753309249878, 0.02329961210489273], [-0.5324879884719849, -0.2150951325893402, 0.5449315309524536, -0.34264665842056274, 1.6628459692001343, -0.5517287254333496, 2.3105576038360596, 0.9966996312141418, -1.9019111394882202, 2.1690428256988525, 0.1687685251235962, 0.10261660069227219, 0.08774546533823013, 1.3648804426193237, -1.1845595836639404, -0.7020111083984375, 0.16989745199680328, -0.7587714791297913, -0.5612469911575317, -0.6246499419212341, -1.2674747705459595, 0.7994272112846375, -0.4622659385204315, 1.335375189781189, -0.8711265921592712, -0.6023257970809937, 0.6406376957893372, 1.1029342412948608, 0.1531788408756256, -0.6373487710952759, -1.0801663398742676, 1.3902267217636108], [1.169027328491211, -0.19196704030036926, 0.04489898681640625, 0.694252073764801, -0.5471863150596619, 0.6918684244155884, 1.3386054039001465, -0.037764739245176315, 0.020327961072325706, -2.5249741077423096, -0.5369466543197632, -0.5175484418869019, 0.7641002535820007, 0.8911949992179871, 0.29256004095077515, -0.706493616104126, -0.5309458374977112, -0.6569247245788574, -1.042683720588684, 0.0038913125172257423, 0.5725634694099426, -3.272634506225586, -1.298791527748108, -1.1023774147033691, -0.42305368185043335, -1.3919137716293335, -0.8514147996902466, -1.256610631942749, 0.93915194272995, -1.4173554182052612, -0.5348299741744995, -0.6541938781738281], [-0.6550361514091492, 1.8025187253952026, -0.08176197111606598, 0.683530867099762, -0.660516619682312, -0.3150152266025543, 1.4124077558517456, 0.8230503797531128, -2.790762186050415, 0.6755790114402771, -1.5061454772949219, -1.2750577926635742, 0.25393909215927124, 2.1610403060913086, -1.2254562377929688, 1.0721561908721924, 0.16103345155715942, 1.9432618618011475, -0.3844916522502899, -1.197126865386963, -1.3978723287582397, -1.8670684099197388, 0.16577324271202087, 0.3343196511268616, 1.3696573972702026, -0.04987618699669838, -1.3504890203475952, -2.2182669639587402, -0.05143492668867111, 0.945448100566864, -1.0283135175704956, 1.360874056816101], [0.2206907868385315, -1.1149064302444458, -0.49813777208328247, 0.5920795202255249, -0.2086717039346695, 0.12004755437374115, 1.3313158750534058, -0.3446517586708069, -1.4511427879333496, -0.7029072046279907, 0.5189297795295715, -0.8470867872238159, -0.6534460783004761, 1.1500903367996216, 0.022795040160417557, 1.2860180139541626, -0.3001996576786041, 1.2774646282196045, -0.12645180523395538, -1.4828163385391235, 0.23501095175743103, -1.462838888168335, 0.5584267377853394, -1.1219645738601685, 1.6734665632247925, -1.2875359058380127, -0.39266350865364075, 0.1336902230978012, -0.2622756063938141, -0.6848534941673279, 0.7858656644821167, -2.1246464252471924], [0.9097276926040649, -0.16951967775821686, 0.4857514500617981, 1.0978842973709106, 0.28325873613357544, 0.03185795620083809, -0.21975143253803253, 0.019894447177648544, 1.2863054275512695, -0.10069555789232254, 1.3004257678985596, -1.3065340518951416, -0.362562894821167, 0.06208033114671707, -1.558210015296936, -0.49003902077674866, 0.7912743091583252, 0.07400105148553848, -0.9438027143478394, 0.4688221216201782, 0.48324984312057495, -0.7044481039047241, 0.0933275893330574, 1.3149980306625366, 1.0904215574264526, -0.5277577042579651, -0.44416069984436035, -1.1334809064865112, 0.19716766476631165, 0.843905508518219, 0.21679982542991638, 0.706036388874054], [0.7311467528343201, 0.6032859086990356, 0.5620760917663574, 0.5037553906440735, 1.0396883487701416, -0.09449364989995956, -0.6196549534797668, 0.8907255530357361, -2.6362147331237793, 0.5799340605735779, 1.2304279804229736, 0.4812762141227722, 0.19405235350131989, -0.8323546648025513, -0.23229065537452698, 1.0548955202102661, 1.1144763231277466, 0.8184325098991394, 0.9438264966011047, -2.485257863998413, -1.4729883670806885, -0.1965576559305191, 1.1321144104003906, -0.3154008984565735, 1.3281482458114624, 1.180164098739624, -1.1149907112121582, 1.949102759361267, 0.11805138736963272, 0.7321335077285767, 0.25516000390052795, 0.04980158433318138], [-0.44505438208580017, -1.0383977890014648, -1.072317361831665, 0.9876390099525452, 0.8103282451629639, -0.8675587177276611, 2.183789014816284, 0.17720527946949005, 0.2175697237253189, 0.24664482474327087, 0.4123851954936981, -1.3869789838790894, -0.06245715543627739, -0.5854735970497131, 2.0416147708892822, -0.3839993476867676, -0.44836926460266113, -0.563845157623291, 0.9542076587677002, -0.22863145172595978, 1.0351406335830688, 0.628866970539093, 0.3059433102607727, 0.04631727188825607, -0.16152384877204895, 1.285995602607727, 0.007464495021849871, -1.5729435682296753, -0.7427433729171753, 0.08996808528900146, 1.4075169563293457, 0.004512782674282789], [-1.8887643814086914, 1.5254621505737305, 1.3609670400619507, 1.6990891695022583, -1.2932074069976807, -0.2758767902851105, 0.9868009090423584, -0.7276604771614075, -1.3871307373046875, -0.49409592151641846, 0.06271298974752426, 0.08620528131723404, -2.1171164512634277, -0.8029508590698242, 1.4945876598358154, -0.5141119360923767, 0.26966428756713867, 1.247205376625061, -0.059066444635391235, 1.157543420791626, -1.040450096130371, -0.3081650733947754, 1.5671995878219604, 0.21714335680007935, 0.5943920612335205, 0.5392933487892151, 0.49396008253097534, 0.6534106135368347, -0.9379897713661194, -2.593808650970459, 0.4661615490913391, 0.9295217394828796], [-0.3739297389984131, -2.1367416381835938, 0.28778502345085144, 0.44372013211250305, -0.3727276921272278, 0.3421919047832489, 2.3520357608795166, -0.2848047912120819, 0.06517070531845093, -0.8215293884277344, -0.36027809977531433, -0.08616199344396591, 0.768478274345398, -0.16363032162189484, 1.4099355936050415, -0.2159726470708847, -0.3609892725944519, -0.953003466129303, 1.3771381378173828, -2.3032569885253906, -1.86354660987854, 1.3166121244430542, 0.1006852313876152, 0.9360901117324829, -0.3786761164665222, -1.3229618072509766, 0.23041962087154388, -0.39492255449295044, -2.134636878967285, 0.8729210495948792, 0.3035079836845398, 1.8634696006774902], [-0.0720994845032692, -1.2905179262161255, 0.8450891971588135, -1.0899232625961304, -1.3763760328292847, -0.3349149823188782, 2.6834635734558105, 0.8295577168464661, -1.2032440900802612, 2.576542377471924, -0.03467995300889015, 0.18131394684314728, 0.648751437664032, 0.5005941390991211, -0.2728522717952728, -2.3090763092041016, 0.840924084186554, -0.48450055718421936, -0.6772913932800293, 1.5566133260726929, 0.85193932056427, 0.3838389813899994, 1.5360839366912842, 1.1218369007110596, 0.09709686785936356, 0.5821535587310791, -1.9535201787948608, 0.5478850603103638, -0.8472505211830139, -2.745723247528076, 0.414516419172287, -0.6654677987098694], [0.699662446975708, 0.22474229335784912, -1.703479528427124, -1.0572302341461182, -0.9591154456138611, 1.4505897760391235, -0.20100131630897522, 1.10610830783844, -1.9362688064575195, 0.3946833312511444, -0.34524133801460266, 0.47689419984817505, -0.0873769074678421, -1.2394057512283325, 1.530396819114685, -0.030735740438103676, -1.0562859773635864, -1.3921452760696411, 0.646168053150177, 0.12347523868083954, 0.26092636585235596, 2.3292129039764404, 1.6145473718643188, -0.07450070232152939, -1.4382199048995972, 0.9143241047859192, 0.6924814581871033, 1.018134355545044, 0.13815663754940033, -0.9343705773353577, -0.7347488403320312, -1.449945330619812], [-0.43399256467819214, -0.020586835220456123, -0.9489880800247192, 0.9465277194976807, 0.6858167052268982, 0.6870816946029663, -0.47663256525993347, 1.343522310256958, 1.1495147943496704, 0.6281341314315796, 0.0412144660949707, -0.24981987476348877, 0.3794427812099457, 2.0818467140197754, -0.7684985399246216, -1.0133707523345947, -0.3526917099952698, -1.096977710723877, 1.2945659160614014, 0.26941266655921936, -1.1668109893798828, 0.1068878024816513, 1.1829092502593994, 0.9246903657913208, -0.5997015833854675, -0.4622207283973694, 0.010337521322071552, -0.4400954246520996, 0.05602873116731644, 0.0026145686861127615, 0.5178667902946472, -0.7931477427482605], [-0.5571958422660828, 0.19217747449874878, -0.8496295213699341, 0.4474022090435028, 0.4538445472717285, 1.4437330961227417, -0.8644317388534546, 0.2742654085159302, 1.4747819900512695, -0.8084388375282288, -0.15548476576805115, 0.5908932089805603, -0.3858623504638672, -1.1032110452651978, 0.6562048196792603, 1.3172674179077148, 1.3702272176742554, 0.46603190898895264, -2.04038143157959, -0.3329157829284668, -0.5651463866233826, 0.05892731621861458, -0.39025765657424927, 0.7621387839317322, -0.3172820806503296, 0.9341319799423218, 1.2362260818481445, 1.3210153579711914, -1.3467994928359985, -0.8173839449882507, 0.4037844240665436, 0.31673741340637207], [1.3534879684448242, 0.3273827135562897, -0.09383121877908707, 1.0161393880844116, 0.38863709568977356, 0.9123904705047607, -0.03359733521938324, 0.21211647987365723, -0.13659022748470306, 0.43413543701171875, 1.1792221069335938, 0.739534318447113, 0.9512099027633667, 0.24683533608913422, -0.7219911217689514, -0.3370714485645294, -0.2190571427345276, 0.03881913796067238, -0.419880211353302, 0.9928954839706421, -0.026507370173931122, -0.7648158073425293, -0.0205118115991354, -0.09277592599391937, -0.8666670322418213, -1.7734590768814087, 0.5369676351547241, -1.2446388006210327, 1.696071982383728, 2.1272521018981934, -1.8408262729644775, 0.49156272411346436], [0.1448003202676773, -0.6389775276184082, 0.5885940194129944, 0.037893664091825485, -1.4154658317565918, 0.03511491417884827, 0.7282084822654724, -0.11111371219158173, 0.3916401267051697, -0.35307052731513977, -0.18549013137817383, 0.6128372550010681, 1.1208709478378296, 0.17155568301677704, -0.09398186206817627, 0.25953471660614014, 1.5106724500656128, -0.041034843772649765, -0.8804125189781189, 1.7989903688430786, 0.6009550094604492, -0.5375129580497742, 0.7620046734809875, 0.9988715648651123, 0.5515997409820557, 0.4538605213165283, 1.4902780055999756, 0.058192215859889984, -0.1789548397064209, -0.8984202146530151, 1.6084673404693604, 1.6703240871429443], [0.18292884528636932, 0.42353495955467224, 2.095430612564087, -0.9194278717041016, -0.25007951259613037, -0.2932890057563782, 1.297723650932312, -1.4472631216049194, 0.2940976619720459, 0.28708016872406006, 0.0860525518655777, 0.20661966502666473, 1.3068962097167969, 1.294973373413086, 0.5835243463516235, -2.8491344451904297, 0.3908616900444031, -1.586448073387146, 0.2325713187456131, 0.5195074677467346, -0.19713032245635986, -1.0442479848861694, 0.43078354001045227, -0.2605118155479431, -0.0811787024140358, 0.591032087802887, -0.1896217167377472, 0.2588353455066681, 1.9822524785995483, 1.0731637477874756, 1.2362712621688843, 0.9139280319213867], [-0.32682231068611145, 1.3479948043823242, -1.1866885423660278, -1.127902626991272, 1.600206971168518, -1.9557067155838013, 0.11553183197975159, -0.6000049710273743, -1.7323509454727173, 1.3628315925598145, -1.3682456016540527, 0.5831400156021118, 1.3647669553756714, 1.1582714319229126, -0.3497847020626068, 0.3198602497577667, 0.22231827676296234, 0.8593891859054565, -1.692446231842041, -0.9174206852912903, 0.01710938848555088, 0.5888741612434387, -0.5886762738227844, 0.40398913621902466, -0.4658632278442383, 0.014158129692077637, -0.6637168526649475, -1.679715871810913, 0.7952995300292969, -0.951919436454773, 1.1003210544586182, 0.7526447772979736], [-0.7221693992614746, -0.6703388094902039, 0.7344760894775391, -1.2562954425811768, 0.6909629106521606, -0.22252926230430603, -2.0109896659851074, -0.14258551597595215, 1.1264129877090454, 0.7458651661872864, -0.44034188985824585, -1.1594228744506836, 0.3046707808971405, 0.16416755318641663, -0.5145334005355835, -0.6022846102714539, 0.5126225352287292, 1.0948550701141357, -0.7999024987220764, -1.4338041543960571, 0.2594251334667206, 1.6438847780227661, -1.0595791339874268, -1.9251468181610107, -0.3005240857601166, -0.7514546513557434, 2.1474037170410156, 1.2540996074676514, 0.3120298385620117, -0.010911663994193077, -2.264075756072998, 0.3782125413417816], [-0.22181160748004913, 0.5837751626968384, -2.005516290664673, 0.6607048511505127, -0.09581641852855682, 0.6493037343025208, -1.7549439668655396, -1.6158037185668945, 0.9707072973251343, 0.7874680161476135, 1.43282151222229, 1.2322510480880737, -2.144505262374878, -0.26562491059303284, -1.5029101371765137, 0.26179900765419006, -0.6426296830177307, -1.1116803884506226, -0.2771846354007721, -0.1723337471485138, 0.9657750725746155, 1.7303165197372437, -0.046802718192338943, -0.128306582570076, -1.045084834098816, 0.48321548104286194, -0.5393341779708862, 1.6307260990142822, -0.27007967233657837, 0.6081240177154541, 0.24959231913089752, -0.7315556406974792], [0.706512987613678, 0.07771838456392288, -1.4985865354537964, 0.6223552823066711, -0.49106520414352417, 0.7422060370445251, -1.3310645818710327, 0.5227838754653931, -1.0139551162719727, 1.421137809753418, 0.8663108348846436, -0.33474820852279663, -1.138365626335144, -1.3322720527648926, -1.9715462923049927, 0.16886025667190552, 0.5033713579177856, -1.583305835723877, -0.8505299687385559, -1.1021735668182373, -0.11564701795578003, 2.2234997749328613, -1.4805448055267334, 1.3572367429733276, -1.128167986869812, 0.557230532169342, -1.284820318222046, 1.2962464094161987, 0.28435346484184265, 1.309372067451477, -0.27530771493911743, 0.27739694714546204], [-0.8449336290359497, 0.42724594473838806, -1.6520025730133057, 0.6742703914642334, 0.4430648982524872, 0.030493920668959618, 0.5406231880187988, 0.7877962589263916, 1.8120152950286865, 0.7230842113494873, 0.3639538288116455, 0.625523030757904, 1.0938751697540283, 1.432355284690857, -1.920454740524292, 1.2203454971313477, -0.8821301460266113, 0.29626893997192383, -0.17986896634101868, 0.3116101622581482, -0.7743507623672485, 0.11864681541919708, 0.5553421974182129, -3.1037092208862305, -3.689521551132202, 0.04602651298046112, 0.5085964202880859, -0.13383212685585022, 1.1426618099212646, 1.3119479417800903, 0.2590206265449524, -0.2678303122520447], [-1.220711588859558, 0.5801412463188171, -1.236432433128357, -1.057796835899353, -0.11458791792392731, -2.3895442485809326, -0.6234742999076843, -0.5017339587211609, -1.828962802886963, -0.8164450526237488, -0.8306844830513, 0.5403810739517212, -0.4486229419708252, -0.540351152420044, 0.09263621270656586, 0.5357553362846375, 0.9477531313896179, -0.5100089907646179, -0.06842304766178131, -2.4520270824432373, -0.454633891582489, 1.3538652658462524, 0.5295325517654419, -0.7705007195472717, 0.389588326215744, -1.22062087059021, -1.680524468421936, -0.2581155598163605, 0.5340737104415894, 1.8271375894546509, 0.5880431532859802, -0.5666369795799255], [-0.5507103800773621, -0.5399883389472961, 2.1093766689300537, 0.7274957299232483, -0.011705110780894756, -1.387169361114502, 0.3799051344394684, -0.7960999011993408, -0.0006309510208666325, -0.9184029698371887, 0.717447817325592, -0.04886632785201073, 0.11778277903795242, -1.2476496696472168, 0.49771836400032043, -0.19880062341690063, 0.21920686960220337, -0.042171914130449295, 0.711395263671875, -1.1138709783554077, 0.9291900992393494, 0.41287219524383545, -1.0453336238861084, -0.5075083374977112, 1.4676823616027832, 1.6805018186569214, 0.2373693585395813, 1.489391565322876, 1.0049172639846802, 0.16099153459072113, 0.46439337730407715, 0.43712031841278076], [-0.7966095805168152, 0.49918726086616516, 0.049594175070524216, 0.06744220852851868, 0.6291970014572144, -1.299035906791687, 1.19087815284729, 1.6639890670776367, 0.8021652698516846, 0.43003910779953003, 0.2502046823501587, 0.014101123437285423, 0.8003556728363037, -1.2678263187408447, 0.3808448016643524, 0.22051016986370087, 2.5735673904418945, 1.3699314594268799, -1.5201034545898438, -0.06638319790363312, -0.20136474072933197, -0.34899237751960754, 0.1171545460820198, 1.1529213190078735, 0.971682071685791, 0.1979091912508011, -0.8889890313148499, -1.5586071014404297, 0.9251163005828857, 0.44233545660972595, -0.022402560338377953, -1.5876281261444092], [-0.6195414662361145, -1.71767258644104, -1.1951810121536255, -1.0499178171157837, -1.0268832445144653, 0.9008820056915283, -0.24006080627441406, -1.2723581790924072, 0.7266142964363098, -1.2222626209259033, 0.45464643836021423, -0.18122026324272156, -0.3356814980506897, 0.15214478969573975, 0.08528222888708115, -0.20843929052352905, 1.5511549711227417, 2.5699377059936523, 1.1903575658798218, 0.2385248988866806, 0.912209153175354, 2.076319932937622, -0.3403349816799164, 0.834865391254425, 2.2944719791412354, -1.6687859296798706, 0.585138738155365, 0.5388273000717163, -0.9737915992736816, 1.7454428672790527, -1.0406264066696167, -1.5322345495224], [1.4489212036132812, 0.8061133027076721, -1.0960689783096313, -0.005200361832976341, -0.866320013999939, 0.5420259833335876, 0.4845140278339386, 0.21545207500457764, -1.0509263277053833, 0.39823800325393677, 0.6996511816978455, 0.33814194798469543, 0.03381975367665291, 1.3089368343353271, -0.2751498818397522, 0.18748755753040314, 0.5816088914871216, 0.4111906588077545, -0.81694096326828, 0.2372233122587204, -0.04937337338924408, -1.8078855276107788, -0.29432013630867004, -1.6787841320037842, 1.4751530885696411, -1.1820529699325562, 1.7804224491119385, 0.9773982763290405, 0.01517750695347786, -0.09904517978429794, -1.006420373916626, 0.19187162816524506], [0.5503708124160767, 1.4234470129013062, 0.6125202178955078, 0.4164329767227173, -0.8138216733932495, -0.11768463999032974, 1.4713536500930786, -1.167231559753418, -0.8069116473197937, 0.37406352162361145, 1.3295174837112427, 0.831794023513794, 0.5332877039909363, 0.13757753372192383, 1.4964723587036133, 0.4385153651237488, 0.45060351490974426, -1.18468177318573, -0.8790557384490967, 0.8920586705207825, 1.1041181087493896, -1.541858434677124, 1.1106146574020386, 0.40907418727874756, -0.007899777963757515, -1.9807082414627075, -1.0649034976959229, 1.2643496990203857, 0.49342378973960876, 0.8287286162376404, 0.02587011829018593, 1.1596295833587646], [-0.1035381332039833, -1.858677864074707, 0.21675248444080353, 1.1398805379867554, -0.3433447480201721, 0.06452907621860504, 0.43145596981048584, -0.619328498840332, 0.12940721213817596, 2.527212142944336, -0.5217247009277344, 0.1807810217142105, 2.261627435684204, 1.056288242340088, 0.4333755373954773, -0.7006973028182983, -2.6093666553497314, 0.8046517372131348, -0.9751756191253662, 0.4342803955078125, 0.7291592359542847, -0.45451620221138, 0.91407310962677, 0.022812167182564735, -0.7321230173110962, 0.015106518752872944, 1.3002499341964722, 0.3066841959953308, -1.1383113861083984, -0.6163679957389832, -0.5671773552894592, -0.1362098604440689], [-1.0342377424240112, -0.12412084639072418, -0.5010703802108765, -0.9147288203239441, -0.6235945224761963, -0.09488991647958755, 1.5431296825408936, 0.4307909607887268, -0.7674880623817444, 0.4682770073413849, 2.5617382526397705, 0.8314627408981323, -0.21431991457939148, 1.5132514238357544, -0.5699501037597656, -0.45013248920440674, -0.2934210002422333, -0.2501550316810608, -0.5874916911125183, -0.46882542967796326, 0.8010461926460266, -0.7125102877616882, 0.23813652992248535, -0.06681928038597107, 0.04568397253751755, 0.0938224270939827, -1.5577726364135742, 0.3654744327068329, 0.11004946380853653, -1.1402721405029297, 1.3967180252075195, -0.29832589626312256], [1.346243143081665, -0.38092929124832153, 1.7994389533996582, 0.7020378708839417, 0.4853302538394928, -0.007308902684599161, -1.1733494997024536, 0.5478293895721436, 0.4399369955062866, -0.5250414609909058, 0.5108863711357117, 0.3633997440338135, -0.8527909517288208, -0.5353876948356628, -2.0782275199890137, 0.37552180886268616, 1.461580753326416, -0.6600872278213501, -0.2264687716960907, 0.7648992538452148, -0.507652223110199, -0.696972131729126, -0.04644007980823517, 0.5323697328567505, -0.7765728235244751, 0.16211847960948944, 0.5452938675880432, 0.11384040117263794, 0.6054355502128601, 1.7962580919265747, -0.1370140016078949, 0.08746177703142166], [2.1014041900634766, -0.21108362078666687, 0.24147072434425354, 0.2417367398738861, 0.7770925760269165, -2.425467014312744, 0.4129942059516907, -2.4082539081573486, -0.22001634538173676, -0.49217984080314636, 0.24813881516456604, -0.5737740397453308, -0.15241824090480804, 0.24705424904823303, 0.8198866844177246, 0.054902635514736176, 0.27003780007362366, -0.03120783530175686, 0.46694284677505493, 1.635738492012024, -0.6432850956916809, 0.42306873202323914, 0.5298457741737366, -0.5947970151901245, -0.18544593453407288, -0.8614287972450256, -0.22248485684394836, -0.5155665874481201, -0.5486945509910583, -0.0012082434259355068, 0.17059272527694702, 0.3163153827190399], [-0.7531431913375854, -0.021024400368332863, -0.1775103509426117, -1.2858649492263794, -0.8538829684257507, 0.8437349796295166, 0.07975292950868607, 0.04892972484230995, 1.1738625764846802, -0.9382849335670471, -0.2607002258300781, 1.304696798324585, -1.0124106407165527, -0.18214385211467743, -0.47683754563331604, -0.4503769278526306, -0.749295175075531, 0.5278265476226807, -0.7850350737571716, -0.6246291995048523, 0.5940201878547668, 0.4639817178249359, 2.4262185096740723, 0.02875850722193718, -0.9890874028205872, 1.3128923177719116, 1.27548348903656, 0.4562080502510071, -0.6832616329193115, -0.9329550862312317, -0.5082907676696777, 0.7691438794136047], [-0.6040568351745605, -0.5513824224472046, 0.3125101625919342, 0.12237128615379333, 0.7344156503677368, -0.2746475338935852, -0.5499457716941833, 0.20526227355003357, -0.18165597319602966, -0.1430005580186844, -2.2226123809814453, -0.14341561496257782, -0.8949347734451294, 0.5660054087638855, 0.6786641478538513, -1.9672294855117798, 1.1684966087341309, -1.393849492073059, 0.4320559799671173, -0.28774940967559814, -1.1507128477096558, 1.8060888051986694, 0.4445870816707611, 0.16076794266700745, 0.02086832746863365, 0.47673705220222473, 0.48028042912483215, 0.9369737505912781, -0.09058617800474167, 0.4298591911792755, -1.1486717462539673, 0.5034553408622742], [0.9422429203987122, 0.729718804359436, -0.1510843187570572, -0.5530616044998169, -0.03094351291656494, 1.4418545961380005, -0.28168603777885437, 0.3233042061328888, 1.4446463584899902, -0.052356328815221786, 0.10028119385242462, -0.606731116771698, 0.21029677987098694, -1.0485498905181885, -1.2490782737731934, 0.9242472052574158, 0.5196225643157959, 0.8147774338722229, -0.5256654620170593, 0.7370202541351318, -0.21977467834949493, -1.6858916282653809, 0.16138242185115814, -0.22994652390480042, -0.05208255350589752, 1.7523878812789917, 0.8785924315452576, 1.0528531074523926, -1.9849478006362915, -0.03184356540441513, 0.5521082282066345, 0.7051547169685364], [1.579470157623291, -0.8535735607147217, -1.2442768812179565, 1.4080148935317993, 0.42162173986434937, -0.9499906301498413, -0.5787562131881714, 1.3678556680679321, -0.5202975273132324, -0.5654399394989014, 0.46845510601997375, 1.1565622091293335, 1.055383563041687, -0.5893287658691406, -0.23957392573356628, 1.9682451486587524, -0.8379736542701721, 1.1673351526260376, 0.08583726733922958, 0.35619282722473145, 0.4236695170402527, 1.9953045845031738, 2.339919090270996, 1.6989504098892212, 0.6530430316925049, 1.23473060131073, 0.634133517742157, -2.669062376022339, 1.3363577127456665, 0.5364185571670532, -1.2916756868362427, -0.648989200592041]]]], "dtype": "float32"}, true], "e": {"__tup__": [{"__nd__": [[[[0.34189072251319885, 0.5715025067329407, 0.08587130159139633, 1.6457091569900513, 1.5743330717086792, -2.462388753890991, 0.039485860615968704, 1.1979976892471313, -0.4485374093055725, 0.08299200236797333, 0.7270336747169495, -0.8429952263832092, 0.683402419090271, 1.921859622001648, -0.7909533977508545, 0.03858182579278946, 0.5776016712188721, 1.7049071788787842, 0.19753096997737885, 0.5681702494621277, 1.3612632751464844, -0.49022969603538513, 1.8556007146835327, 0.5310510396957397, -1.0524189472198486, 0.1147909089922905, 1.4623761177062988, 0.7221479415893555, -0.8624618649482727, 1.2722402811050415, 0.4840214252471924, 0.3296871781349182], [0.34189072251319885, 0.5715025067329407, 0.08587130159139633, 1.6457091569900513, 1.5743330717086792, -2.462388753890991, 0.039485860615968704, 1.1979976892471313, -0.4485374093055725, 0.08299200236797333, 0.7270336747169495, -0.8429952263832092, 0.683402419090271, 1.921859622001648, -0.7909533977508545, 0.03858182579278946, 0.5776016712188721, 1.7049071788787842, 0.19753096997737885, 0.5681702494621277, 1.3612632751464844, -0.49022969603538513, 1.8556007146835327, 0.5310510396957397, -1.0524189472198486, 0.1147909089922905, 1.4623761177062988, 0.7221479415893555, -0.8624618649482727, 1.2722402811050415, 0.4840214252471924, 0.3296871781349182], [-0.2269556224346161, -0.8238609433174133, -0.6327187418937683, 1.1836978197097778, -0.9319652915000916, -1.0872880220413208, -0.6985087394714355, 0.8656383156776428, -0.46081840991973877, 0.8967768549919128, -0.34103143215179443, 0.043083805590867996, -0.7407383322715759, 0.7286407351493835, 0.0886411964893341, 0.6876184940338135, 1.619319200515747, 0.19740788638591766, -1.677695631980896, 0.8738285303115845, 0.8364365696907043, -0.5534799695014954, 0.432381808757782, 0.45361387729644775, -0.2217559516429901, 1.234045147895813, 0.8853131532669067, 0.2118542194366455, -1.0300384759902954, -0.6242200136184692, -1.101951241493225, 0.6366877555847168], [0.33635783195495605, 0.5580818057060242, 0.07904832810163498, 1.6411895751953125, 1.5501307249069214, -2.4490373134613037, 0.03239001706242561, 1.194821834564209, -0.44855424761772156, 0.09071704000234604, 0.7167856693267822, -0.834439754486084, 0.6696496605873108, 1.9102305173873901, -0.7824919819831848, 0.0447792150080204, 0.5875840187072754, 1.6903210878372192, 0.17950518429279327, 0.5710157752037048, 1.356172800064087, -0.4908372163772583, 1.8417552709579468, 0.5302022099494934, -1.0443544387817383, 0.12558096647262573, 1.4566044807434082, 0.7173153162002563, -0.8640112280845642, 1.2539438009262085, 0.46870145201683044, 0.33266526460647583], [-0.19842439889907837, -0.859801173210144, -0.7205951809883118, 1.2030997276306152, -0.940483570098877, -1.12090265750885, -0.7079945802688599, 0.8199079036712646, -0.5277568697929382, 0.9868108034133911, -0.38658541440963745, 0.033090364187955856, -0.7288247346878052, 0.8008268475532532, 0.07979930192232132, 0.7489495277404785, 1.6716928482055664, 0.1958215981721878, -1.7188211679458618, 0.941169261932373, 0.8107101917266846, -0.5206106305122375, 0.5294814705848694, 0.5221269130706787, -0.21683534979820251, 1.207910180091858, 1.0190889835357666, 0.12330935895442963, -1.0693641901016235, -0.6253941655158997, -1.077745795249939, 0.6123142242431641], [-0.9215084314346313, 0.3304184079170227, 1.633471131324768, 0.1353287696838379, -0.6830902099609375, 0.14459195733070374, -0.0473947711288929, 1.536176085472107, 1.4620147943496704, -1.7023910284042358, 1.0085819959640503, -0.06355039775371552, -0.6078745722770691, -1.1681832075119019, -0.6079832315444946, -0.7024237513542175, 0.06019037961959839, -0.19331766664981842, 0.06511679291725159, -1.1595041751861572, 0.46953055262565613, -0.4899033010005951, -1.356909155845642, -1.4472298622131348, 0.5060165524482727, 0.7852994203567505, -2.817749500274658, 2.0616023540496826, 0.28608226776123047, -0.1193908080458641, -1.039183497428894, 0.8632320761680603], [-0.8458346128463745, 0.34477540850639343, 1.5676878690719604, 0.18620659410953522, -0.5852838754653931, 0.08003439009189606, -0.01659286767244339, 1.4994239807128906, 1.3953261375427246, -1.6221970319747925, 0.9573489427566528, -0.07811859250068665, -0.5639862418174744, -1.0662165880203247, -0.6021171808242798, -0.6998234987258911, 0.07263240218162537, -0.1349639743566513, 0.07117771357297897, -1.1092720031738281, 0.500190258026123, -0.4724194407463074, -1.2301729917526245, -1.350900650024414, 0.4352128803730011, 0.7639436721801758, -2.669811487197876, 1.9802964925765991, 0.247563898563385, -0.06184447184205055, -0.9642683267593384, 0.8478200435638428], [0.9443821907043457, 0.3098272383213043, -0.1536739021539688, 0.013418308459222317, -0.1041150912642479, 0.8453952074050903, 0.3480035662651062, 0.3477458655834198, 0.30240198969841003, -0.3032004237174988, 1.288830041885376, -0.05774374678730965, -0.46671295166015625, -0.1366978883743286, -0.4074495732784271, -1.6301442384719849, -0.27119770646095276, 0.4007737338542938, 0.9242427349090576, -0.3862188756465912, 0.3366245627403259, -0.03481252118945122, 0.1733737736940384, -0.5184440016746521, 0.5716161727905273, -0.1046360582113266, -0.672318160533905, 1.2020326852798462, -0.7503127455711365, -1.0710692405700684, 0.5890139937400818, 0.7730527520179749], [0.1927039921283722, 0.28156578540802, -0.12133090198040009, 0.9816165566444397, 0.7517995834350586, -1.620288372039795, 0.1410091668367386, 0.6373305320739746, -0.2614983022212982, 0.08108159154653549, 0.3669510781764984, -0.810782253742218, 0.5387842655181885, 1.4567304849624634, -1.1333458423614502, 0.3545088768005371, 0.6206188797950745, 0.8099144101142883, 0.12800754606723785, 0.3692750334739685, 0.3239623010158539, 0.26064392924308777, 1.9194620847702026, 0.29930537939071655, -0.10455992072820663, -0.43777379393577576, 1.001118540763855, 0.19849054515361786, -0.6859312057495117, 0.979962944984436, 0.5164591073989868, 0.04769318178296089], [2.0681493282318115, 0.28704649209976196, -1.2273015975952148, -0.04146046191453934, 0.2352198362350464, 1.2669466733932495, 0.566752016544342, -0.34334373474121094, -0.399368554353714, 0.5465952157974243, 1.4596021175384521, -0.03225584328174591, -0.4099683463573456, 0.4729936420917511, -0.23917357623577118, -2.2111051082611084, -0.4671694338321686, 0.7757636308670044, 1.4153518676757812, 0.08714423328638077, 0.30534660816192627, 0.19256265461444855, 1.0453758239746094, 0.03829227387905121, 0.577237606048584, -0.5846242904663086, 0.6161143183708191, 0.7158670425415039, -1.3878724575042725, -1.6815088987350464, 1.5351073741912842, 0.7433894276618958], [1.5210598707199097, 0.8072464466094971, 0.36967727541923523, 0.6103070378303528, 1.3883802890777588, 0.8238438367843628, 1.7544680833816528, -0.13525614142417908, 0.784196138381958, 0.06964000314474106, -1.7893638610839844, 0.4676975607872009, -0.21087317168712616, -0.4192945957183838, 0.1391846239566803, -1.9031122922897339, -0.10077565163373947, 0.011426862329244614, 0.16169722378253937, -1.1852834224700928, 0.8208765983581543, 0.5901679396629333, 0.7161285281181335, 1.0235964059829712, -1.1820720434188843, 0.7336767315864563, -1.3772324323654175, -0.620934247970581, -0.013011676259338856, 1.0171364545822144, 0.8595003485679626, 0.8830450773239136], [-0.22094829380512238, -0.2112228125333786, 0.7083205580711365, 0.8596606254577637, -0.8324078321456909, -0.25411760807037354, 1.952101230621338, 0.36732766032218933, -0.7412081360816956, -0.2617274820804596, -0.4598888158798218, 0.668178379535675, -0.19027727842330933, -0.05805807560682297, 0.6047517657279968, -0.9132410287857056, -0.8621745109558105, 0.04196969047188759, -0.8296794295310974, -0.3027118444442749, -0.45751214027404785, 1.6034622192382812, 1.2960182428359985, -0.1171935647726059, 0.359359472990036, -0.5596821904182434, 0.9777467250823975, 0.3386106789112091, -0.24028313159942627, 0.9608908891677856, 0.23697979748249054, -2.3185102939605713], [-0.6694885492324829, -0.10882597416639328, -2.2762768268585205, -0.017833853140473366, 1.1285083293914795, 2.63861346244812, -0.5092390775680542, 0.7614158987998962, 0.8440321683883667, -1.4500354528427124, -0.7946493029594421, 0.04825460538268089, 0.27174827456474304, -0.6283055543899536, -1.2738796472549438, -0.6547872424125671, -0.6082772612571716, 0.25382429361343384, -0.09375925362110138, -0.07100807875394821, 0.008538655936717987, -0.9303905963897705, 0.7226830124855042, 2.100551128387451, -1.6993296146392822, -0.161405548453331, -0.4246116876602173, -1.7707194089889526, 0.03303629904985428, -0.23366385698318481, -1.0159642696380615, -1.2684967517852783], [-0.14416280388832092, -0.24082350730895996, -0.5791267156600952, -0.17813068628311157, -0.8520162105560303, -0.16378892958164215, 0.3077490031719208, -0.35755401849746704, 0.06640322506427765, 0.15478308498859406, 0.11734876781702042, -0.8466432094573975, 0.33583587408065796, 0.5943554639816284, -2.181731700897217, 1.1181813478469849, 1.0703407526016235, -0.8626305460929871, 0.028588108718395233, 0.0999717265367508, -1.6829490661621094, 1.666967749595642, 2.1295130252838135, 0.08316293358802795, 1.8757339715957642, -1.5131430625915527, 0.1764524281024933, -0.8013784885406494, -0.2610042691230774, 0.5444072484970093, 0.5730879306793213, -0.339430034160614], [0.33290180563926697, 0.6050798892974854, -0.9066101312637329, -0.5802052021026611, 0.15197357535362244, 0.4805986285209656, -0.38526055216789246, -0.36464065313339233, -0.1471586376428604, 0.39241090416908264, 1.2814409732818604, 0.19414564967155457, 0.030665550380945206, -0.661355197429657, 0.5646746754646301, -0.9745144248008728, -0.6242316365242004, 0.44023793935775757, 0.15226148068904877, 0.24682003259658813, 0.19050906598567963, 0.8139991164207458, 0.24343585968017578, 0.6586892008781433, -0.4565799832344055, -1.267408013343811, 0.1809086948633194, 0.3728651702404022, -0.14397601783275604, -0.6512451171875, 0.5729399919509888, 0.005570112261921167], [0.7373786568641663, 0.8085042238235474, 0.15571917593479156, 0.13265833258628845, 0.5339100956916809, 0.5718356966972351, 0.8393622040748596, -0.021376216784119606, 0.3333772122859955, 0.16572272777557373, -0.6738592982292175, 0.45666494965553284, 0.10000582784414291, -0.6003074049949646, 0.40135878324508667, -0.8646718859672546, -0.2811645269393921, -0.05684550479054451, 0.15572570264339447, -0.46655526757240295, 0.014208394102752209, 0.44636136293411255, 0.2627391815185547, 0.709658682346344, -0.8500697612762451, -0.16702871024608612, -0.7923353314399719, -0.05429770424962044, 0.27726268768310547, 0.21423381567001343, 0.4956335723400116, 0.09773218631744385], [-1.4695615768432617, -0.7537165880203247, -0.02226998284459114, 0.5100497007369995, -0.7070673108100891, -0.5881307125091553, 0.034828703850507736, -1.038961410522461, -0.8057477474212646, -0.9383789896965027, 0.3317275643348694, 2.006260871887207, 0.6202877759933472, -1.660575270652771, -0.4250390827655792, -1.2583662271499634, 0.9297561645507812, 2.9571499824523926, 0.2641763687133789, -1.1168932914733887, 0.008318096399307251, 1.0843600034713745, -0.49475550651550293, 0.990343451499939, 0.8251960277557373, -0.5127084851264954, 0.11607926338911057, 0.5589497089385986, -0.26764166355133057, -0.07918277382850647, -0.3563838601112366, -0.6821430921554565], [1.9757018089294434, 0.3383139967918396, -1.1927852630615234, -0.11003679782152176, 0.23671790957450867, 1.2223511934280396, 0.5145056843757629, -0.3726843297481537, -0.38448864221572876, 0.5418477654457092, 1.4954019784927368, -0.01335938274860382, -0.3719731569290161, 0.376736044883728, -0.16419140994548798, -2.1637182235717773, -0.5140625834465027, 0.7591890096664429, 1.370683193206787, 0.08740397542715073, 0.2888779938220978, 0.2693409323692322, 0.9842169880867004, 0.05960794910788536, 0.522418737411499, -0.6782315373420715, 0.5765369534492493, 0.7218543291091919, -1.2959964275360107, -1.6204313039779663, 1.5157058238983154, 0.7007407546043396], [-0.6896677017211914, 0.5786154866218567, -0.7376444935798645, -0.6238629817962646, -0.22353710234165192, 0.29855474829673767, -0.4806031882762909, 0.2590484023094177, 0.19008146226406097, -0.11956542730331421, -0.014569224789738655, 1.1012381315231323, 0.6567118167877197, -0.6012396812438965, 0.848990797996521, 0.2852218449115753, -0.12182415276765823, 0.10742636770009995, 0.20819415152072906, 0.8610124588012695, -0.14088992774486542, 0.2829257845878601, 0.4911331832408905, -0.8838898539543152, -1.0341295003890991, -1.6801196336746216, 0.06361404061317444, -0.032995179295539856, -0.33935612440109253, -0.3424566090106964, -0.2950933575630188, -0.16922461986541748], [-0.29610416293144226, -1.1067297458648682, 0.6599957942962646, -0.24670572578907013, 0.6224240064620972, 2.065495252609253, 0.24372895061969757, -0.9292013049125671, 0.419268935918808, 0.5713064074516296, 0.5446576476097107, -0.030987348407506943, 1.648910641670227, -0.38136839866638184, 0.4558345675468445, 1.3976448774337769, 0.3325626254081726, 0.07929879426956177, -0.14998172223567963, 0.0009815304074436426, -0.6689189672470093, -1.248216986656189, 1.511795997619629, 0.3454131782054901, -0.6907494068145752, -1.0186735391616821, -0.056096240878105164, 1.6159509420394897, 0.3077778220176697, -1.0319167375564575, 3.3917369842529297, 0.5513136386871338], [-0.19941619038581848, -0.8696032762527466, -0.7225361466407776, 1.2252466678619385, -0.9418177604675293, -1.1361030340194702, -0.7243034839630127, 0.8390533328056335, -0.5370270609855652, 0.9997942447662354, -0.39453577995300293, 0.047313954681158066, -0.7460014224052429, 0.8038161396980286, 0.11625344306230545, 0.7427007555961609, 1.6811059713363647, 0.2128933072090149, -1.746762752532959, 0.95441073179245, 0.850976824760437, -0.555992841720581, 0.5033044219017029, 0.5289592146873474, -0.25060686469078064, 1.251828670501709, 1.0320745706558228, 0.13853085041046143, -1.082197904586792, -0.6442211866378784, -1.1044280529022217, 0.627709686756134], [-0.22708189487457275, -0.10930114984512329, 0.8488079905509949, 1.0344817638397217, -0.8718536496162415, -0.2008763998746872, 1.90714430809021, 0.45857447385787964, -0.8842048645019531, -0.29320740699768066, -0.4797723889350891, 0.4083136022090912, -0.4070200026035309, -0.259829044342041, 0.6847333908081055, -0.8690628409385681, -0.8466117978096008, 0.1718824803829193, -0.697613000869751, -0.26965075731277466, -0.560279130935669, 1.54688560962677, 1.2140213251113892, -0.01893080398440361, 0.3611208200454712, -0.5059399008750916, 0.9672939777374268, 0.23736834526062012, -0.29894107580184937, 0.850680410861969, 0.24945852160453796, -2.2149767875671387], [-0.4795288145542145, -1.5623680353164673, 0.8397824168205261, -0.22220362722873688, 0.7105199694633484, 2.678004503250122, 0.04781763255596161, -1.108585000038147, 0.5923322439193726, 0.5898491144180298, 0.3479119837284088, -0.0576062873005867, 1.6076709032058716, -0.11107780784368515, 0.5968037843704224, 1.7492033243179321, 0.5646528601646423, -0.23921038210391998, -0.3112049400806427, -0.0855383574962616, -0.267001211643219, -1.2600109577178955, 1.7350388765335083, -0.05068439990282059, -1.1081141233444214, -0.8343029618263245, 0.35089021921157837, 2.021071434020996, 0.4062435030937195, -1.1928375959396362, 3.9030842781066895, 0.7207493185997009], [-0.16206803917884827, -0.241298109292984, -0.6791980266571045, -0.17282232642173767, -0.7593087553977966, -0.03394892439246178, 0.2727506160736084, -0.323894739151001, 0.08968294411897659, 0.09703714400529861, 0.06912188977003098, -0.8175312876701355, 0.3388954997062683, 0.5508020520210266, -2.1551530361175537, 1.048452377319336, 0.9968509078025818, -0.8153970241546631, 0.017045069485902786, 0.10016629099845886, -1.6224191188812256, 1.5685503482818604, 2.093249559402466, 0.20379894971847534, 1.7206016778945923, -1.4678597450256348, 0.17515265941619873, -0.8748942613601685, -0.24726130068302155, 0.5184674263000488, 0.5130414366722107, -0.399625688791275], [-0.3009951710700989, 0.06450854986906052, -1.100968360900879, -0.06343203783035278, -0.14331962168216705, 1.418280005455017, 0.26007431745529175, 0.9983746409416199, 0.5190999507904053, -0.8235834240913391, -1.379339575767517, 1.5405668020248413, 0.9550000429153442, 0.3589228093624115, -0.06816398352384567, 0.43865966796875, 0.41998085379600525, 0.08249148726463318, 1.218428611755371, 1.0998570919036865, -0.4472212493419647, -1.1962718963623047, 1.5262750387191772, -2.1951956748962402, -0.9173064231872559, -1.185157299041748, 0.1266247183084488, -0.5117964148521423, -1.4173380136489868, -0.8520691394805908, -0.3463689982891083, 0.03175884112715721], [0.21035797894001007, -0.22315271198749542, 0.232868030667305, -0.2307043820619583, 0.3235061764717102, -0.14473602175712585, 0.1558358371257782, -0.1202721819281578, 0.04202287644147873, -0.5870243906974792, -2.6995327472686768, -0.19707292318344116, -0.028972312808036804, 1.0469080209732056, 1.3955237865447998, -0.3534012734889984, -1.9576467275619507, 0.03521166741847992, -0.17696641385555267, -0.4382392168045044, -0.18892017006874084, 0.7919825315475464, 1.276169776916504, -1.665329933166504, -0.6074603199958801, -0.719001054763794, 0.39483049511909485, 0.11154462397098541, -1.2264094352722168, -0.1987503319978714, 0.4725658595561981, -1.28644597530365], [-1.4174232482910156, -0.7384524941444397, -0.03330478444695473, 0.5028576850891113, -0.6973795890808105, -0.5703532099723816, 0.043720949441194534, -1.0338144302368164, -0.8044329881668091, -0.9137221574783325, 0.3532923460006714, 1.9801355600357056, 0.6055222153663635, -1.6323403120040894, -0.4193201959133148, -1.2746962308883667, 0.9124979376792908, 2.9324567317962646, 0.28202223777770996, -1.1020028591156006, 0.012961392290890217, 1.0777448415756226, -0.47534510493278503, 0.9730668663978577, 0.829198956489563, -0.5153235197067261, 0.12518762052059174, 0.5692870616912842, -0.28419259190559387, -0.1021222248673439, -0.3243677020072937, -0.6576367616653442], [1.5125728845596313, 0.5866594910621643, -0.10924028605222702, -1.1414132118225098, 0.5258989930152893, 1.169379472732544, -0.14416132867336273, 0.3021485209465027, -1.305634617805481, -1.7282848358154297, 0.5368162989616394, 1.5049375295639038, 0.8590536117553711, -0.9820916652679443, 1.1237634420394897, -0.4643157124519348, -0.1610592007637024, -1.218639850616455, -0.2212001383304596, 0.18374624848365784, -0.399549275636673, -1.336577296257019, -0.4268520474433899, 0.8107210993766785, -0.4590502381324768, -0.9901628494262695, -1.9261189699172974, 0.5695797204971313, 0.5817493200302124, 0.9588375091552734, 0.33706894516944885, 0.5270971655845642], [-0.5946453213691711, -0.13048191368579865, -1.712201714515686, 0.1519685536623001, 0.7987204790115356, 2.1265618801116943, -0.04618347808718681, 0.6645129323005676, 0.5514088273048401, -1.2497246265411377, -0.6842520833015442, 0.0887647420167923, 0.1492524892091751, -0.5740230083465576, -0.9678657650947571, -0.7610757350921631, -0.7169762849807739, 0.21185091137886047, -0.29283905029296875, -0.19309577345848083, -0.07722992449998856, -0.40229150652885437, 0.7897537350654602, 1.8626710176467896, -1.3081721067428589, -0.19676974415779114, -0.19544446468353271, -1.3917593955993652, 0.06727013736963272, 0.03915208950638771, -0.7782922983169556, -1.5382704734802246], [-1.053236484527588, 1.0032355785369873, -0.20460186898708344, -1.155927300453186, -0.18583562970161438, -0.8408660888671875, -1.2048908472061157, -0.572212278842926, -0.16488182544708252, 0.6849280595779419, 1.6233282089233398, 0.3698791265487671, 0.3476504683494568, -1.62894606590271, 1.7267131805419922, 0.21376349031925201, -0.651411771774292, 0.17425481975078583, -0.9322614073753357, 0.5633198022842407, 0.11857140809297562, 1.7463979721069336, -0.6501492261886597, 0.7942528128623962, -1.0183073282241821, -2.1729226112365723, -0.0891452506184578, 0.577246367931366, 0.9127501249313354, 0.24641825258731842, 0.06264571100473404, -0.3054109513759613], [-0.08455384522676468, -0.4796615242958069, 1.228554368019104, -0.1370631605386734, -0.07836250960826874, -0.09447283297777176, -0.175227090716362, -0.06503293663263321, -0.5126219987869263, 0.7520077228546143, -0.2402634620666504, 1.089103102684021, -0.11956870555877686, -0.8032115697860718, 0.03141159564256668, -0.19542115926742554, 0.24604718387126923, 0.6814467310905457, 0.5934996008872986, 0.33238744735717773, -0.6964138746261597, 1.090057134628296, -0.9496345520019531, 0.6269416213035583, -0.06407830119132996, 0.2578764855861664, 0.19380708038806915, -0.32357847690582275, -0.4919290840625763, 0.041290368884801865, 0.29114946722984314, -0.23393282294273376], [0.09719996899366379, -0.1981452852487564, 0.04329099878668785, -0.20073780417442322, 0.3690309226512909, 0.045915424823760986, 0.051589906215667725, -0.06295899301767349, 0.08920759707689285, -0.6816679835319519, -2.519718647003174, -0.15492509305477142, -0.012231312692165375, 0.8958268165588379, 1.154011845588684, -0.41091567277908325, -1.8343125581741333, 0.07011581212282181, -0.15906094014644623, -0.4094756245613098, -0.1716616153717041, 0.6400206089019775, 1.231531023979187, -1.2906180620193481, -0.65059494972229, -0.6746653318405151, 0.28116825222969055, -0.07034898549318314, -1.0705056190490723, -0.20374302566051483, 0.24325713515281677, -1.2656137943267822], [0.177663192152977, 0.41722729802131653, -0.22090135514736176, 1.1825904846191406, 1.2881145477294922, -1.374709129333496, -0.13188953697681427, 0.9794166088104248, -0.29119986295700073, -0.0794752985239029, 0.32074958086013794, -0.5506148338317871, 0.5157715678215027, 1.2127079963684082, -0.8205407857894897, -0.024022212252020836, 0.2511626183986664, 1.3108664751052856, 0.1856268048286438, 0.40063855051994324, 0.8777733445167542, -0.4245991110801697, 1.4802956581115723, 0.7516301870346069, -1.0346404314041138, 0.1290075033903122, 0.8137653470039368, 0.23257513344287872, -0.5203716158866882, 0.9309591054916382, 0.22717022895812988, 0.09567683190107346], [-0.750092089176178, 0.006594957783818245, 0.8619420528411865, 0.43419623374938965, -0.15003199875354767, 0.7903231382369995, -0.21877212822437286, -0.07325670123100281, -0.6470839381217957, 0.3770126700401306, -0.7830514311790466, -0.3340493142604828, -1.3376572132110596, 0.25860828161239624, 0.182333841919899, -1.339418888092041, 0.5262741446495056, -0.5896384716033936, -0.020700648427009583, 0.44164136052131653, 0.5335052609443665, 0.2936771810054779, 0.33983513712882996, 0.20607011020183563, -0.2903763949871063, -0.288640558719635, -0.13100868463516235, -0.5486209392547607, 0.044995732605457306, -0.32803425192832947, 1.4258801937103271, 0.11622364819049835], [-0.33994048833847046, 0.07664371281862259, -1.6745692491531372, -0.1330612748861313, 0.9602124094963074, 1.9927828311920166, -0.11018159240484238, 0.5204459428787231, 0.616091251373291, -0.8876565098762512, -0.2589552104473114, -0.01991201937198639, 0.7094781398773193, -0.769707202911377, -0.9292081594467163, -0.3608839809894562, -0.6173908710479736, 0.3983515501022339, 0.04632527381181717, 0.09231715649366379, -0.5935269594192505, -1.10189950466156, 0.7935870289802551, 1.98484206199646, -1.0916061401367188, -0.5903830528259277, -0.7371782064437866, -1.2631402015686035, 0.016476215794682503, -0.32198959589004517, -0.24465766549110413, -0.9130809903144836], [0.23886774480342865, -0.19836091995239258, 0.20987997949123383, -0.23083163797855377, 0.31646260619163513, -0.17351296544075012, 0.16152751445770264, -0.10772915929555893, 0.029960960149765015, -0.595421552658081, -2.7086892127990723, -0.1988140046596527, -0.05689764767885208, 1.0601311922073364, 1.3950438499450684, -0.40264400839805603, -1.9840726852416992, 0.0471060648560524, -0.1572628766298294, -0.43914735317230225, -0.18275710940361023, 0.8179404139518738, 1.2692188024520874, -1.67779541015625, -0.5887317657470703, -0.7166436910629272, 0.39833179116249084, 0.08817870169878006, -1.2541155815124512, -0.1985217034816742, 0.43116456270217896, -1.2994399070739746], [-0.9516074657440186, 0.40512028336524963, 0.8128235340118408, -1.4053645133972168, -1.0230872631072998, 0.32319891452789307, 0.6728208065032959, -0.739136278629303, -0.6561295986175537, 0.6127121448516846, 0.33796101808547974, 0.5008862018585205, 0.33957746624946594, 1.4033147096633911, 1.9841511249542236, -0.052954740822315216, 0.7107973694801331, -0.571956992149353, -0.042263370007276535, -1.3216739892959595, 0.3458462953567505, 0.7369106411933899, -0.5980823040008545, -0.3749570846557617, 0.05842792987823486, -1.0880670547485352, -0.629696786403656, 0.8925966024398804, -0.04930184409022331, -0.334754079580307, 0.4633679687976837, -0.18589235842227936], [1.0932785272598267, 1.2662405967712402, -0.3037915825843811, 0.13810668885707855, -0.46313223242759705, 1.4086532592773438, 0.3075912892818451, -0.1424681693315506, -0.49165642261505127, 0.5910638570785522, -0.8804534673690796, 0.6861756443977356, -0.680387020111084, 0.16871672868728638, 0.9950197339057922, -0.3799152970314026, -0.32086774706840515, -1.2042649984359741, 0.12399017810821533, -0.2134190797805786, -0.45502376556396484, 1.197535753250122, -0.5951533317565918, 1.014137864112854, -1.3602155447006226, -0.3663918673992157, 1.9812921285629272, 0.3363201916217804, 1.0476269721984863, -1.213204264640808, -0.2132500559091568, -2.284205436706543], [1.1789809465408325, 1.2292639017105103, -0.36332815885543823, 0.4856770634651184, -0.5785577893257141, 1.468195915222168, 0.4640418291091919, -0.07930158078670502, -0.6128648519515991, 0.6744084358215332, -1.135140061378479, 0.729751706123352, -0.7174628376960754, 0.3394896686077118, 1.1590256690979004, -0.31411656737327576, -0.45748862624168396, -1.3816460371017456, 0.303920179605484, -0.291341096162796, -0.6004489660263062, 1.4612079858779907, -0.7714399695396423, 1.119857668876648, -1.4866756200790405, -0.07234369963407516, 2.0570995807647705, 0.38882580399513245, 1.5075054168701172, -1.3108429908752441, -0.3605074882507324, -2.637629270553589], [-0.9783612489700317, 1.0234291553497314, -0.21602796018123627, -1.1312260627746582, -0.2062484323978424, -0.8323137760162354, -1.2213038206100464, -0.6285345554351807, -0.19645346701145172, 0.7079217433929443, 1.526645541191101, 0.4718838334083557, 0.36843565106391907, -1.594584584236145, 1.7404729127883911, 0.09532580524682999, -0.7118246555328369, 0.19292747974395752, -0.9124853014945984, 0.47098931670188904, 0.1042063981294632, 1.746106743812561, -0.5352216362953186, 0.7292422652244568, -1.0138071775436401, -2.1001389026641846, -0.04835924506187439, 0.6070659756660461, 0.8733127117156982, 0.13800859451293945, 0.06389287859201431, -0.28512492775917053], [-0.1997625231742859, -0.8690434694290161, -0.7207239270210266, 1.2229294776916504, -0.9398517608642578, -1.1346484422683716, -0.7229312658309937, 0.8374396562576294, -0.535950779914856, 0.9979380369186401, -0.39697593450546265, 0.04696596786379814, -0.7453626990318298, 0.8035671710968018, 0.11733121424913406, 0.7401586771011353, 1.6757445335388184, 0.21254770457744598, -1.7437329292297363, 0.9535151124000549, 0.8491807579994202, -0.5542389750480652, 0.5035430788993835, 0.526078462600708, -0.2507660388946533, 1.2490390539169312, 1.030030608177185, 0.13829413056373596, -1.082221269607544, -0.6434488296508789, -1.1018816232681274, 0.6258478164672852], [-1.253912091255188, -0.7784844040870667, -6.52787639410235e-05, 0.4964773952960968, -0.6020640730857849, -0.5481643676757812, 0.13033777475357056, -0.7243422269821167, -0.5849905610084534, -0.5490003228187561, 0.16177134215831757, 1.3997403383255005, 0.481462299823761, -1.2771238088607788, -0.45515725016593933, -0.8018141388893127, 0.9251189827919006, 2.1220364570617676, -0.11919383704662323, -0.9080154895782471, -0.039764340966939926, 0.8273648023605347, -0.19843722879886627, 0.9673612713813782, 0.47360435128211975, -0.5285825729370117, 0.11939041316509247, 0.6359695196151733, -0.38868579268455505, -0.05977443605661392, -0.29064053297042847, -0.2880071699619293], [-0.4792771637439728, -1.5842911005020142, 0.8718730807304382, -0.22533081471920013, 0.7246590852737427, 2.692265748977661, 0.04601631313562393, -1.142688274383545, 0.5911319255828857, 0.6149477362632751, 0.3797819912433624, -0.08594989776611328, 1.618698000907898, -0.12271751463413239, 0.6058514714241028, 1.7682121992111206, 0.5638112425804138, -0.24195191264152527, -0.3361990451812744, -0.10505764186382294, -0.269197016954422, -1.2606089115142822, 1.7357231378555298, -0.008272193372249603, -1.1062135696411133, -0.8295445442199707, 0.34896546602249146, 2.059814214706421, 0.4376240074634552, -1.1962281465530396, 3.970850706100464, 0.7314372658729553], [-0.1653832346200943, -1.4954322576522827, -0.017187252640724182, -0.6852146983146667, -2.000370502471924, 0.9667930006980896, -0.1299401819705963, 0.471560001373291, 0.7662743330001831, 1.047434687614441, -0.4764508605003357, -1.3491706848144531, 0.3350028693675995, 0.2864287495613098, 0.6057255864143372, 0.20972029864788055, -0.7789979577064514, 2.4755892753601074, 0.054833199828863144, 1.642449140548706, 0.1740836352109909, -0.06806980073451996, -0.37481793761253357, -0.7817400693893433, 0.18790346384048462, -0.18895556032657623, -0.9039764404296875, -0.5403648018836975, -0.2632012367248535, 2.0826025009155273, -1.009803056716919, -1.175893783569336], [0.5016037225723267, 0.5699292421340942, -0.08973504602909088, -0.3228914141654968, 0.5074505805969238, 0.12992450594902039, 0.9034325480461121, -0.13840025663375854, 0.025167593732476234, 0.615382194519043, 1.1136900186538696, -0.3265959620475769, 1.7803633213043213, -0.913816511631012, 0.02358192577958107, 0.3986722528934479, -0.5851176381111145, 0.5582244992256165, 0.39110851287841797, 0.5886064171791077, -2.1239635944366455, -1.4899157285690308, 0.986203670501709, 1.5295867919921875, 0.5592451691627502, -1.6190282106399536, -1.5941481590270996, 0.001017155242152512, -0.05827248841524124, -0.484431654214859, 1.8074992895126343, 0.17602358758449554], [1.965984582901001, 0.3098771274089813, -1.0794198513031006, -0.08706776052713394, 0.20575983822345734, 1.2470651865005493, 0.5006585717201233, -0.3389427959918976, -0.4747506082057953, 0.550696611404419, 1.240759253501892, 0.024373268708586693, -0.3754301071166992, 0.3210759162902832, -0.3007300794124603, -2.042485237121582, -0.5353253483772278, 0.7516217827796936, 1.4406577348709106, 0.06677740812301636, 0.13523176312446594, 0.20718026161193848, 0.9618169665336609, 0.016736922785639763, 0.5391578674316406, -0.4869364798069, 0.37942400574684143, 0.6032807230949402, -1.2262204885482788, -1.5537396669387817, 1.4798705577850342, 0.759819746017456], [-0.2671966850757599, -1.0546897649765015, 0.6902377009391785, 0.5407822728157043, -1.122044324874878, 0.3677023947238922, -0.7604697346687317, 0.22238780558109283, 0.27513134479522705, 0.814017117023468, -0.2460099309682846, 0.34441736340522766, -0.7004191279411316, 0.1870727390050888, 0.8662857413291931, 0.5378862619400024, -0.5277701616287231, -0.06101525202393532, -0.02782837487757206, 0.33850792050361633, -1.5194376707077026, 0.6787988543510437, 0.8207016587257385, 0.10768972337245941, 0.7772573828697205, 0.06915008276700974, -1.3645979166030884, 0.22564469277858734, 0.7512490153312683, 0.23037219047546387, 0.5138998031616211, -0.31906816363334656], [-0.19348271191120148, 1.299417495727539, 0.06312434375286102, -1.703470230102539, 0.2741619348526001, 0.3370591700077057, -1.1905913352966309, -0.4281793534755707, 0.1824556589126587, 0.22901734709739685, 0.8147400617599487, 0.3782399296760559, -0.014071936719119549, -1.2885228395462036, 0.3982973098754883, -0.22592048346996307, 0.2563024163246155, 0.1732592135667801, -1.2996994256973267, 0.18336057662963867, 0.238068088889122, 0.13040868937969208, 0.22491148114204407, 0.398329496383667, -0.7456048130989075, -2.1631641387939453, 0.9267920851707458, 0.1684560477733612, -0.8917673230171204, -0.20723800361156464, 0.6243131756782532, 0.07842659950256348], [-1.9409657716751099, 0.03911910951137543, 1.3887382745742798, 0.49340489506721497, -1.2682559490203857, 0.6052353382110596, -1.2409735918045044, -0.2024686634540558, -1.785490870475769, 0.7803221940994263, -1.163273572921753, -0.3530946373939514, -2.537259340286255, 0.5399944186210632, 0.27916419506073, -2.0462591648101807, 0.7855063676834106, -1.215057373046875, -0.15885940194129944, 1.06332266330719, 0.6296432018280029, 0.5322144627571106, 0.20000144839286804, -0.3337762653827667, 0.009771428070962429, -0.2897796034812927, 0.43817591667175293, -0.9914655685424805, 0.10296183079481125, -1.1074081659317017, 1.562004804611206, -0.30124884843826294], [1.4910894632339478, 0.5824657082557678, -0.1129080280661583, -1.119524598121643, 0.5285890698432922, 1.1544559001922607, -0.1435929536819458, 0.2939606010913849, -1.286781907081604, -1.7051632404327393, 0.530078649520874, 1.4686199426651, 0.854882538318634, -0.9608062505722046, 1.1081558465957642, -0.45963579416275024, -0.16599620878696442, -1.2204951047897339, -0.2220832258462906, 0.18500792980194092, -0.40673738718032837, -1.3226512670516968, -0.40845078229904175, 0.8126771450042725, -0.45101433992385864, -0.9835774898529053, -1.9224717617034912, 0.5503749251365662, 0.5767903923988342, 0.9488361477851868, 0.3490373194217682, 0.5292235016822815], [-0.42453572154045105, -0.8979746699333191, -0.04652249068021774, 0.23447579145431519, -0.5239852070808411, 0.10255104303359985, 0.642873227596283, -0.9063073396682739, -0.104254812002182, 0.2649056613445282, 0.13737022876739502, -0.4005281329154968, -0.45652610063552856, 0.11701114475727081, -0.7235053181648254, 0.6481437087059021, 0.1572006642818451, -0.5466716885566711, -0.08365961909294128, 0.25453850626945496, -0.8174596428871155, 1.146436095237732, 1.6978223323822021, -0.29080966114997864, 1.3749916553497314, -0.7801637053489685, 0.398352712392807, -0.5055259466171265, -0.6539260745048523, 0.26380547881126404, 0.28227370977401733, -0.8089584708213806], [-0.1561093032360077, -1.5199271440505981, 0.02331477776169777, -0.6969907879829407, -2.0585813522338867, 0.9361097812652588, -0.12159521132707596, 0.4671316146850586, 0.7646723389625549, 1.092893123626709, -0.47348394989967346, -1.370148777961731, 0.3373672366142273, 0.30487385392189026, 0.6419234871864319, 0.22728566825389862, -0.7803098559379578, 2.5147204399108887, 0.06049748510122299, 1.6756329536437988, 0.17601627111434937, -0.054666515439748764, -0.3923695981502533, -0.8420449495315552, 0.22196133434772491, -0.19214732944965363, -0.9106349349021912, -0.5170683860778809, -0.27181941270828247, 2.1228199005126953, -1.0104199647903442, -1.1724848747253418], [-0.37530502676963806, -0.23919928073883057, -0.34137701988220215, 0.056937988847494125, 0.16774824261665344, -0.14783817529678345, 0.6016175150871277, 0.1366395503282547, -0.0016959888162091374, 0.6118320226669312, -0.5021389126777649, 0.8589411973953247, 0.8416950702667236, -1.2908351421356201, -0.30152589082717896, -0.26230597496032715, 0.8624805808067322, 1.2001882791519165, -0.15171709656715393, -0.1856943517923355, -0.8618614673614502, -0.25191736221313477, 0.2764352858066559, 0.42019668221473694, -0.5803706049919128, -0.3668517470359802, -0.23756663501262665, 0.06436547636985779, -0.059797488152980804, -0.6669116020202637, 0.17129814624786377, 0.24058514833450317], [-1.4842841625213623, -0.8896911144256592, 0.47727257013320923, 0.04164968058466911, 0.36214596033096313, -0.6653943657875061, -0.14447523653507233, -0.07568623870611191, -0.032026227563619614, 0.7597173452377319, -0.4942232370376587, 0.014062097296118736, -1.1948388814926147, 0.10158833861351013, -0.15203571319580078, -2.2536134719848633, -1.1901915073394775, 0.29484254121780396, 0.9567384719848633, 2.9572806358337402, -0.006447731517255306, -0.48959803581237793, -0.7943413853645325, -0.931430459022522, 0.2830367088317871, 0.7898076176643372, -1.3027695417404175, -0.7614532113075256, -1.2958701848983765, -0.5402855277061462, 0.47049278020858765, 0.743605375289917], [-0.3653285801410675, 0.8559539914131165, -0.4548807442188263, 0.07461626082658768, -1.3033034801483154, 0.20041854679584503, -0.6545484662055969, 0.8134309649467468, 0.7668688893318176, -1.5795304775238037, 1.4173803329467773, 1.9462188482284546, -0.5442389845848083, -0.7874435186386108, 1.0864602327346802, 0.9516586661338806, 1.4128649234771729, 0.9738755822181702, 0.3984983563423157, -0.8291498422622681, 0.9365485906600952, -0.75694340467453, 0.10059399157762527, 0.04644922912120819, 1.4825918674468994, -2.548855781555176, -0.8536199331283569, -0.04384230822324753, -0.635427713394165, 0.2594204246997833, -0.25916483998298645, 0.08154905587434769], [-1.221327304840088, -1.0129764080047607, 0.38762426376342773, -0.10477287322282791, -0.11223641037940979, -0.3506776690483093, -0.1414545178413391, 0.030318692326545715, 0.12518668174743652, 0.8236805200576782, -0.4946126937866211, -0.25923585891723633, -0.8928670883178711, 0.14358319342136383, 0.006357737351208925, -1.7641772031784058, -1.111884593963623, 0.7307254672050476, 0.7799981832504272, 2.7020888328552246, 0.02958192303776741, -0.40341612696647644, -0.7135217785835266, -0.9159722924232483, 0.2693810760974884, 0.5955306887626648, -1.2247278690338135, -0.7128642201423645, -1.095674991607666, -0.017834166064858437, 0.1798882633447647, 0.3661857545375824], [-0.19025014340877533, -0.48748740553855896, -0.4429444372653961, 0.076265349984169, -0.3489318788051605, 0.04551352187991142, -0.3972160816192627, 0.16439670324325562, -0.13724417984485626, -0.651086151599884, -1.434882640838623, 0.6190550327301025, 0.2864282429218292, -1.0768738985061646, -1.4363209009170532, -0.9441342353820801, 0.6212323307991028, -0.11589588224887848, -0.5462902784347534, 0.07369805127382278, -0.4368067979812622, -0.5433800220489502, -0.27286237478256226, -0.4590483009815216, 0.5838714241981506, -0.0065479883924126625, 0.26997190713882446, 0.24737656116485596, 1.2138901948928833, 1.6626657247543335, -0.10619350522756577, -0.800255537033081], [-1.301261305809021, -1.0879298448562622, 0.6048130393028259, 0.1235945075750351, -0.05279069393873215, 0.07883608341217041, 1.1311613321304321, -0.0909833163022995, 0.6458995342254639, 0.19245658814907074, -0.2938670217990875, -0.6724456548690796, 0.2612488865852356, -0.8699521422386169, -1.3302619457244873, 0.5555903911590576, 0.9974878430366516, -1.0619816780090332, -1.2457592487335205, -1.5215662717819214, -0.43124687671661377, 0.7959367632865906, 0.671142578125, 1.2700719833374023, -0.9677585959434509, -1.884105920791626, -0.32148313522338867, 1.5900201797485352, -0.5897272229194641, 0.6931259632110596, 0.15554361045360565, 1.3906201124191284], [-1.4612704515457153, -0.8668214082717896, 0.3897721767425537, 0.039065927267074585, 0.39008891582489014, -0.5614773631095886, -0.1599358469247818, -0.051249440759420395, -0.0035596038214862347, 0.6917502880096436, -0.501711905002594, 0.010578254237771034, -1.1521698236465454, 0.07703689485788345, -0.19006338715553284, -2.208005666732788, -1.1747989654541016, 0.2941678762435913, 0.9230474829673767, 2.8647687435150146, -0.0045228637754917145, -0.5049610733985901, -0.7517416477203369, -0.8286777138710022, 0.22025342285633087, 0.7643215656280518, -1.2796015739440918, -0.7962390184402466, -1.2524961233139038, -0.5311607718467712, 0.4241132140159607, 0.6825310587882996], [-0.5897677540779114, -0.12548334896564484, 0.1736958622932434, -1.1217015981674194, -1.2789642810821533, 0.8832576870918274, 0.08280666172504425, -0.8264415264129639, -0.21622970700263977, -0.5449910759925842, 0.7339794635772705, 1.3460557460784912, -0.6907418966293335, 0.7437618970870972, -0.2841772139072418, 0.09182046353816986, -0.37784692645072937, -0.7986053228378296, -0.5961576700210571, -0.5180341005325317, 0.472790390253067, 0.43060043454170227, 0.22840382158756256, 0.4091462790966034, 0.8303779363632202, 0.3609006106853485, 0.13697928190231323, -0.3148787319660187, 1.5842201709747314, -0.15120601654052734, -0.20167143642902374, -0.31010377407073975], [-0.18579991161823273, -0.5085713267326355, -0.360818475484848, 0.07616131752729416, -0.40364155173301697, -0.062307000160217285, -0.38903769850730896, 0.13952787220478058, -0.17215287685394287, -0.6068549752235413, -1.4534869194030762, 0.6363884806632996, 0.27576547861099243, -1.0807757377624512, -1.4257832765579224, -0.9613118171691895, 0.6544490456581116, -0.12672020494937897, -0.5446277856826782, 0.10744832456111908, -0.4504310190677643, -0.5285542011260986, -0.3115635812282562, -0.5742039680480957, 0.6681846380233765, 0.0019528227858245373, 0.28421565890312195, 0.3188410699367523, 1.231576681137085, 1.7136691808700562, -0.06579569727182388, -0.7639013528823853], [-0.5423505306243896, -0.10512590408325195, -2.0452466011047363, -0.006716239731758833, 1.0795109272003174, 2.4790360927581787, -0.40721046924591064, 0.6650509834289551, 0.7775176167488098, -1.3187553882598877, -0.7760961055755615, 0.12086659669876099, 0.28330278396606445, -0.5668423175811768, -1.1695438623428345, -0.6029269099235535, -0.513594388961792, 0.23576848208904266, -0.07879135757684708, -0.066802479326725, 0.02249251678586006, -0.878217339515686, 0.7694467902183533, 1.9342314004898071, -1.6216012239456177, -0.17089010775089264, -0.45577704906463623, -1.6069773435592651, 0.02857842855155468, -0.17321516573429108, -0.8187902569770813, -1.1081057786941528], [0.2874685525894165, -0.12262266129255295, 0.571320652961731, -0.3049463629722595, 0.22949433326721191, 0.14737462997436523, 0.31035757064819336, -0.15880949795246124, -0.1635333001613617, -0.3641471862792969, -2.106802225112915, -0.15003715455532074, 0.07619543373584747, 0.7063990831375122, 1.133099913597107, -0.056554824113845825, -1.6839545965194702, -0.14031051099300385, 0.023472411558032036, -0.4647344946861267, 0.050113558769226074, 0.4969022572040558, 0.8410937786102295, -1.4864161014556885, -0.8271688222885132, -0.5022090077400208, 0.33594202995300293, -0.05453450605273247, -0.9207078814506531, -0.14165204763412476, 0.2838099002838135, -0.8724483251571655], [0.5545300245285034, 0.35926854610443115, 0.1461493968963623, 0.005508352071046829, 0.6731895208358765, 0.2743394076824188, 0.45756322145462036, -0.25918081402778625, -0.6894490718841553, -0.05322901904582977, -0.13181859254837036, 1.2973458766937256, -0.20024870336055756, -0.0767853707075119, -0.44331294298171997, -0.2764679193496704, 0.49270936846733093, 0.044004105031490326, 0.1532406508922577, 0.031471025198698044, 0.6009349822998047, -0.24760764837265015, 0.9579479098320007, 0.38513705134391785, -0.6550622582435608, -0.5712776184082031, -0.7724964022636414, -0.7465948462486267, -0.07767116278409958, 0.9775877594947815, 0.3416532278060913, 0.1897202879190445], [0.37669453024864197, -0.47379836440086365, 1.1340094804763794, -0.9446448683738708, 0.42815566062927246, 0.131353497505188, -0.49701619148254395, -1.701481580734253, -0.01824677176773548, -0.542319655418396, 1.348612904548645, -2.7146449089050293, 1.1688947677612305, 1.0657768249511719, -0.06896477937698364, -0.6914986968040466, -1.0146251916885376, 1.6288617849349976, -2.1757917404174805, 0.10050763189792633, -0.2156676948070526, 0.14607882499694824, 0.581214964389801, -1.1513503789901733, -0.017652882263064384, 1.274011254310608, 1.9661874771118164, -0.881587028503418, 0.8336921334266663, -0.10614462941884995, 0.7396826148033142, -0.9071049690246582], [0.5682077407836914, 0.6030685305595398, -0.010041351430118084, -0.4469267725944519, 0.4555853307247162, 0.18131153285503387, 1.0002838373184204, -0.11976788938045502, -0.023680150508880615, 0.656274676322937, 1.1808773279190063, -0.15922820568084717, 1.9113736152648926, -1.1233315467834473, 0.039267368614673615, 0.4528844356536865, -0.6144434809684753, 0.788772702217102, 0.4700525999069214, 0.5797852277755737, -2.2491955757141113, -1.5765548944473267, 1.0088403224945068, 1.5442917346954346, 0.5964939594268799, -1.7937815189361572, -1.5803550481796265, 0.13732513785362244, -0.0760253444314003, -0.5776829123497009, 1.8550264835357666, 0.08574520796537399], [0.08986946195363998, 0.1287962794303894, 0.1003970056772232, 0.9038017988204956, 1.0162726640701294, -1.9287998676300049, -0.18050837516784668, 0.7033104300498962, -0.09641381353139877, 0.2920481264591217, 0.5468088984489441, -0.5493044257164001, 0.42037633061408997, 1.7141244411468506, -0.35165250301361084, 0.11559450626373291, 0.40632665157318115, 1.421929121017456, -0.02314450405538082, 0.5287339687347412, 1.1956005096435547, -0.3880712389945984, 1.425126314163208, 0.5514311194419861, -1.0207600593566895, -0.18882198631763458, 1.363918662071228, 0.6657740473747253, -0.5642519593238831, 0.8610340356826782, 0.1532101184129715, 0.5906423330307007], [-0.12989626824855804, -0.5894372463226318, -0.12256341427564621, -0.8461413383483887, -1.3840750455856323, 1.237205147743225, 0.14831139147281647, -0.6623285412788391, -0.2527617812156677, -0.42392581701278687, 0.9549765586853027, 1.3118407726287842, -0.8379749655723572, 0.29647064208984375, -0.44463208317756653, 0.033036209642887115, -0.5189342498779297, -1.1215331554412842, -0.7797471284866333, -0.1241350844502449, 0.45961794257164, 0.5345557332038879, 0.18261262774467468, 0.40144073963165283, 0.8648964166641235, 0.8475525975227356, 0.3622777462005615, -0.2298264503479004, 1.6157723665237427, -0.035351548343896866, 0.2671227753162384, -0.591071367263794], [1.2044429779052734, -0.5249513387680054, 1.4307336807250977, -0.04103534296154976, -1.511999249458313, 1.8674495220184326, -0.5299771428108215, -0.6190536618232727, -1.4771238565444946, 0.76304692029953, -0.07562339305877686, 0.4737999439239502, 0.5774044990539551, -0.4306841790676117, 0.42593398690223694, -0.30711519718170166, -0.7657037377357483, 0.09627098590135574, 0.5345730781555176, -0.07097064703702927, -2.0480737686157227, 0.5928342342376709, -1.1139438152313232, 0.3400339186191559, -1.3314458131790161, 0.341495543718338, 1.6172195672988892, -0.0006858176784589887, 0.1683671772480011, 1.2750033140182495, 0.7182207703590393, -0.2613036334514618], [-0.1628027707338333, -1.50642728805542, 0.026502011343836784, -0.6886317133903503, -2.036633014678955, 0.9227907061576843, -0.1210232600569725, 0.45343026518821716, 0.752036988735199, 1.0660884380340576, -0.46471911668777466, -1.3428211212158203, 0.34208017587661743, 0.29357871413230896, 0.6293180584907532, 0.21496711671352386, -0.7662790417671204, 2.5040032863616943, 0.06107543036341667, 1.6499580144882202, 0.1721280962228775, -0.05024804174900055, -0.3875039517879486, -0.8350739479064941, 0.22428615391254425, -0.19114768505096436, -0.8953233361244202, -0.5091545581817627, -0.2721036970615387, 2.0916733741760254, -0.9995547533035278, -1.163230538368225], [0.4400864839553833, -0.4408794641494751, 1.3433865308761597, -1.1696062088012695, 0.5816256403923035, 0.25949767231941223, -0.4875633716583252, -1.9715778827667236, 0.032167527824640274, -0.7047109603881836, 1.5412787199020386, -3.0193557739257812, 1.36968994140625, 1.1000109910964966, -0.05520390346646309, -0.8605526685714722, -1.3103197813034058, 1.810051679611206, -2.2549355030059814, 0.0177005585283041, -0.29692620038986206, 0.19168922305107117, 0.5651081800460815, -1.3363631963729858, -0.02413724921643734, 1.323193907737732, 2.0889673233032227, -0.9852358102798462, 1.0391443967819214, -0.06541790068149567, 0.9223846793174744, -1.0667121410369873], [-0.3935995399951935, 0.032151754945516586, -0.21650511026382446, 1.0281381607055664, -0.36760106682777405, 0.017251886427402496, 0.5389003753662109, 1.2695451974868774, 1.2295129299163818, -0.7900053858757019, -1.2862244844436646, 1.3523896932601929, -0.010687162168323994, -0.14811822772026062, -1.120305061340332, -1.8166531324386597, -1.1156384944915771, 1.5886337757110596, -0.8867600560188293, 0.007947288453578949, -0.9844753742218018, 1.1597806215286255, -0.4529137909412384, -0.17575961351394653, -2.2492783069610596, 0.8749253749847412, -1.924617886543274, -0.660827100276947, -0.3470584452152252, -0.6188393235206604, -0.40491387248039246, -0.13579092919826508], [-2.3424878120422363, 0.21219418942928314, 0.7941566705703735, -0.026963990181684494, -0.6475457549095154, -0.13783712685108185, -2.4094395637512207, -0.5842100381851196, -0.0908026471734047, -0.6915066838264465, -1.55973482131958, 0.08067142963409424, 0.67082679271698, -1.174536108970642, -0.2598765790462494, -1.3531855344772339, -0.29884466528892517, -1.3601999282836914, 0.1393958330154419, 0.4551948606967926, -1.215172290802002, 0.25554412603378296, 0.8054144978523254, -0.31652721762657166, -0.5265586376190186, 0.010091139003634453, 0.09168262779712677, -1.996719241142273, -1.2702146768569946, -0.9554710984230042, 1.173435926437378, 0.35807132720947266], [0.5520400404930115, 0.1940726786851883, 1.9551599025726318, -0.5917796492576599, -0.12685967981815338, 1.488816738128662, 0.9193186163902283, -0.3712131381034851, -0.948837161064148, 0.6117032170295715, 0.4710802733898163, 0.05032780393958092, 0.5949681401252747, -0.7308491468429565, 0.016706014052033424, 1.2526280879974365, -0.41925305128097534, -0.8508798480033875, 0.8031405806541443, -0.5374669432640076, 0.996792733669281, -0.812021791934967, -0.8823714852333069, -0.646941602230072, -1.7278672456741333, 0.3577072322368622, 0.08998233079910278, -0.5962350964546204, 0.40919986367225647, 0.03423534333705902, -0.259297639131546, 0.9159867167472839], [-0.565625011920929, 0.15714487433433533, -1.0623575448989868, -1.8311108350753784, 1.1037577390670776, 0.03739437460899353, -1.3694264888763428, 0.7077385187149048, 0.6366153955459595, 1.6974087953567505, 0.6498479247093201, -0.39721471071243286, 0.8090335130691528, 0.49825501441955566, 0.09828739613294601, -0.2829219698905945, -0.09136542677879333, -0.0734325498342514, -0.32580211758613586, 1.466765284538269, -0.3931795358657837, -1.705098271369934, -0.05221216753125191, 0.16121801733970642, 1.1341171264648438, -1.8707270622253418, 1.2839001417160034, -0.5439001321792603, -0.326023668050766, -0.24699684977531433, 0.8786678910255432, -0.9412325024604797], [0.6913033723831177, -0.17801079154014587, 0.9365045428276062, -0.19463185966014862, -1.0591554641723633, 1.1427263021469116, -0.5207812190055847, -0.46530863642692566, -1.065339207649231, 0.8775155544281006, 0.11131278425455093, 0.5508130788803101, 0.48249000310897827, -0.6367506980895996, 0.5440340042114258, -0.2238582819700241, -0.556625485420227, -0.000343704508850351, 0.05286746844649315, 0.11821939051151276, -1.5172240734100342, 0.7259672284126282, -0.9065680503845215, 0.3479413688182831, -1.3464114665985107, -0.17157460749149323, 1.1410328149795532, 0.1187959536910057, 0.250510036945343, 0.8628041744232178, 0.5096563696861267, -0.08513295650482178], [0.29819247126579285, -0.04271519556641579, -0.938888669013977, 0.05189381167292595, -0.47026753425598145, -0.24403998255729675, -0.7405739426612854, 0.2724541425704956, -0.21215279400348663, -0.4457249045372009, -0.6431185603141785, 0.11512772738933563, -0.17211613059043884, 0.06650299578905106, -0.5614075064659119, -0.5759352445602417, 0.1745755523443222, -0.15823467075824738, -0.6072452664375305, -0.104420006275177, -0.2839682102203369, -0.6412511467933655, 0.1777716875076294, -0.24860522150993347, 0.30909061431884766, 0.08651374280452728, 0.40269267559051514, 0.6057673692703247, 0.15925171971321106, 0.2680647373199463, 0.07271918654441833, -0.2822868824005127]], [[0.05745861679315567, -0.39159736037254333, 2.538038492202759, -1.2514461278915405, 0.7949472069740295, 0.5737931728363037, -0.46556830406188965, -0.45836764574050903, -0.06775569915771484, 0.15679869055747986, -1.8408395051956177, -1.0409754514694214, 0.06624629348516464, -1.2262307405471802, -0.3256818950176239, 1.337916374206543, -0.003493302734568715, -0.41294366121292114, 0.22222203016281128, 1.7685619592666626, 1.0887503623962402, 0.3674977719783783, -0.8936050534248352, 0.2360985279083252, -0.6739560961723328, 0.7547433972358704, -1.5936989784240723, -0.9507530331611633, -0.9088459014892578, -0.1470852941274643, -0.17228643596172333, -1.0747990608215332], [0.10324230790138245, -0.38649171590805054, 2.3520359992980957, -1.2186620235443115, 0.8288106918334961, 0.576630175113678, -0.43563857674598694, -0.43491920828819275, -0.05409218370914459, 0.1737741231918335, -1.81369149684906, -1.0203304290771484, 0.05955706164240837, -1.1912541389465332, -0.32331839203834534, 1.2781223058700562, 0.007386084645986557, -0.34962254762649536, 0.20054979622364044, 1.6993895769119263, 1.036365270614624, 0.3434853255748749, -0.823917806148529, 0.21676570177078247, -0.6600711345672607, 0.7384371161460876, -1.5163053274154663, -0.9389863014221191, -0.8905551433563232, -0.12727132439613342, -0.166611447930336, -1.0631818771362305], [0.07016108930110931, -0.3900463283061981, 2.4858033657073975, -1.2424726486206055, 0.8040669560432434, 0.5743485689163208, -0.4573954641819, -0.45170265436172485, -0.0638580247759819, 0.1617029905319214, -1.8334105014801025, -1.0349092483520508, 0.06431843340396881, -1.2160654067993164, -0.3250299394130707, 1.3214200735092163, -0.0010107159614562988, -0.395267516374588, 0.21608304977416992, 1.7491611242294312, 1.074196457862854, 0.36058560013771057, -0.874808669090271, 0.23046335577964783, -0.6701628565788269, 0.7503603100776672, -1.5719736814498901, -0.9472965002059937, -0.9036760926246643, -0.1415759176015854, -0.17099517583847046, -1.0711567401885986], [1.4585766792297363, -0.23495779931545258, -3.1560332775115967, -0.24850891530513763, 1.8304682970046997, 0.6599382758140564, 0.449989378452301, 0.25968918204307556, 0.3506999611854553, 0.6768645644187927, -1.0103298425674438, -0.4081890285015106, -0.13867393136024475, -0.15449921786785126, -0.25336289405822754, -0.4916878342628479, 0.32788708806037903, 1.5251868963241577, -0.4413805603981018, -0.3489305078983307, -0.5144473910331726, -0.3680836260318756, 1.2374647855758667, -0.35633471608161926, -0.24920770525932312, 0.2561340630054474, 0.7755159139633179, -0.5900972485542297, -0.34881702065467834, 0.4593166410923004, 0.0005598115385510027, -0.7180553078651428], [0.05780205875635147, -0.3913813531398773, 2.5369277000427246, -1.251165509223938, 0.7948189973831177, 0.5736878514289856, -0.4655531644821167, -0.4581664502620697, -0.06764255464076996, 0.1567072570323944, -1.8404241800308228, -1.0408399105072021, 0.06617988646030426, -1.2259411811828613, -0.32581520080566406, 1.337593913078308, -0.003471112111583352, -0.41257721185684204, 0.22202399373054504, 1.7680280208587646, 1.0885192155838013, 0.3672742247581482, -0.8932667970657349, 0.23595169186592102, -0.6738279461860657, 0.7546170353889465, -1.5930110216140747, -0.9506406784057617, -0.9086759090423584, -0.14706739783287048, -0.17224466800689697, -1.0746783018112183], [0.25225314497947693, -0.16660091280937195, -0.015218716114759445, 0.7200609445571899, 0.21768006682395935, 0.006561018992215395, 0.1066574901342392, 1.7059876918792725, 0.017706118524074554, 0.41766801476478577, 0.4401865005493164, 0.7029340863227844, 0.774451732635498, 0.514711856842041, 2.17356276512146, -0.5246158838272095, 1.394201397895813, -0.9448739290237427, -0.03382977470755577, 1.9723998308181763, 2.0787367820739746, -0.9703375697135925, -1.3978008031845093, -0.028257111087441444, -1.19780695438385, -1.497070550918579, 0.10287084430456161, 0.1976090371608734, 0.32294026017189026, 0.7181956768035889, 1.7125779390335083, -0.48525097966194153], [1.4593188762664795, -0.23101432621479034, -3.1507906913757324, -0.2468579113483429, 1.8226869106292725, 0.6575741171836853, 0.445866197347641, 0.2602728307247162, 0.350912868976593, 0.6715073585510254, -1.0046476125717163, -0.409141480922699, -0.1389877051115036, -0.1544264554977417, -0.2568725347518921, -0.4899151921272278, 0.3279816508293152, 1.5233871936798096, -0.44225814938545227, -0.34989362955093384, -0.5114365816116333, -0.368880957365036, 1.2355729341506958, -0.35601988434791565, -0.24826732277870178, 0.25542640686035156, 0.7786776423454285, -0.5898139476776123, -0.347983717918396, 0.4565269649028778, 0.0013261105632409453, -0.718111515045166], [0.18689441680908203, -0.23699556291103363, 0.7922618985176086, 0.10881556570529938, 0.39246460795402527, 0.18117046356201172, -0.07272983342409134, 1.0361725091934204, -0.010126299224793911, 0.3357860743999481, -0.2663033604621887, 0.1632317453622818, 0.5572637915611267, -0.025790207087993622, 1.403387427330017, 0.055937740951776505, 0.9621942639350891, -0.7880659699440002, 0.04764479771256447, 1.9175336360931396, 1.7793625593185425, -0.555194616317749, -1.2500872611999512, 0.05538906529545784, -1.0379759073257446, -0.8012024164199829, -0.4292270839214325, -0.1573609560728073, -0.058864809572696686, 0.4494144916534424, 1.1310856342315674, -0.6683141589164734], [1.4105119705200195, -0.23493102192878723, -2.958400249481201, -0.27973923087120056, 1.7895478010177612, 0.6567135453224182, 0.41990000009536743, 0.2349264770746231, 0.3408447802066803, 0.6590937376022339, -1.0364962816238403, -0.4286132752895355, -0.1299857199192047, -0.1895056515932083, -0.2545148432254791, -0.42968955636024475, 0.31421777606010437, 1.4550917148590088, -0.4202719032764435, -0.2787952721118927, -0.45937538146972656, -0.3445649743080139, 1.1641514301300049, -0.33375832438468933, -0.2589835822582245, 0.27181151509284973, 0.695109486579895, -0.6012519001960754, -0.36944064497947693, 0.4365568161010742, -0.009012497961521149, -0.7329884171485901], [0.2455519586801529, -0.16684158146381378, 0.0011064528953284025, 0.7254032492637634, 0.21006834506988525, 0.00337445386685431, 0.10555378347635269, 1.7140889167785645, 0.01580212078988552, 0.4170515835285187, 0.44760748744010925, 0.7092950344085693, 0.7795878648757935, 0.518435001373291, 2.187790870666504, -0.5253579616546631, 1.4004511833190918, -0.9582551121711731, -0.031443335115909576, 1.9854042530059814, 2.0926408767700195, -0.9736154079437256, -1.41184401512146, -0.026432882994413376, -1.2032181024551392, -1.5069561004638672, 0.0988488644361496, 0.20191794633865356, 0.3266386389732361, 0.7201932668685913, 1.7222132682800293, -0.4840405583381653], [0.06124573573470116, -0.03555954992771149, -0.2703971862792969, 0.3734179139137268, 0.144236221909523, -0.06980305165052414, 0.17793354392051697, 1.1004693508148193, 0.1992534101009369, 0.43511340022087097, 0.19171595573425293, 0.7634382247924805, 0.5755640864372253, 0.3825880289077759, 1.5757242441177368, -0.3513389825820923, 0.8680968880653381, -0.6769042611122131, 0.15378537774085999, 1.453847050666809, 1.6333316564559937, -0.8990269899368286, -1.3962446451187134, -0.3944415748119354, -0.9642312526702881, -0.7117761373519897, 0.001982317538931966, 0.23580026626586914, 0.1592097133398056, 0.3866666257381439, 1.211503267288208, -0.44442853331565857], [0.3899247348308563, -0.46586763858795166, -0.2986641228199005, -0.4704873561859131, 0.4358540177345276, 1.1324996948242188, 1.8068606853485107, -0.5101696252822876, 0.9271512031555176, 1.3469520807266235, 0.030215000733733177, 1.816491961479187, -0.2981649935245514, -0.7541798949241638, 2.28883957862854, -0.23979118466377258, -0.6453077793121338, -0.6224049925804138, 0.578849732875824, -0.5346425175666809, 2.4084081649780273, -0.2202509343624115, -0.1604447066783905, -0.7562727928161621, 0.8995203971862793, -1.3521461486816406, -1.8806917667388916, 1.2526503801345825, 0.26916128396987915, -0.5111504197120667, 0.48900601267814636, -0.8813130259513855], [1.2046219110488892, 0.46370017528533936, -1.2139389514923096, -0.2962449789047241, -0.7712342739105225, -0.2715374827384949, -0.8510992527008057, -0.5570322275161743, 0.6010491847991943, 0.7067634463310242, -0.4902462363243103, 0.8601843118667603, -0.2709003686904907, -0.25051450729370117, 0.6583763957023621, -0.25779515504837036, -0.8365315794944763, 1.341825246810913, -0.1835169494152069, 0.9382085204124451, 0.6027103662490845, -0.7116002440452576, -0.1697131097316742, -0.014047729782760143, -0.6272546648979187, -0.5115271210670471, 0.7289477586746216, 1.3514177799224854, -0.10867069661617279, 0.28219425678253174, 0.10956723242998123, -0.5325276851654053], [0.4240771234035492, -0.3733403980731964, -0.31252390146255493, -0.5080719590187073, 0.3555741310119629, 1.168470859527588, 1.8548003435134888, -0.6495295166969299, 1.0385112762451172, 1.386976957321167, -0.008343206718564034, 1.8488966226577759, -0.3464394211769104, -0.7933833003044128, 2.2367424964904785, -0.21684835851192474, -0.8157311677932739, -0.5940073132514954, 0.5609272718429565, -0.6942752599716187, 2.348306179046631, -0.2086958885192871, -0.08210814744234085, -0.7456994652748108, 1.0571106672286987, -1.3074947595596313, -1.9129996299743652, 1.2973992824554443, 0.21788527071475983, -0.5932377576828003, 0.31961461901664734, -0.9378347992897034], [-0.1846870630979538, 0.9995176196098328, -0.755520761013031, -0.34740152955055237, 0.41179201006889343, 2.0097954273223877, 0.6760942339897156, -0.33577507734298706, -0.05889676883816719, 0.37607571482658386, -0.4116179645061493, 0.0006405434105545282, -1.0142614841461182, 1.182489275932312, -0.04863268882036209, -0.14446023106575012, 0.8678275346755981, 0.31563809514045715, 1.1925970315933228, 0.6866150498390198, 0.7586415410041809, 0.9390673041343689, -1.3050330877304077, -0.9620572328567505, -1.780842661857605, -0.17123103141784668, -1.992244005203247, 0.7047184109687805, 0.3225835859775543, -0.3060668408870697, 0.9317495226860046, -1.3335994482040405], [1.0581427812576294, 0.4404132664203644, 0.26515015959739685, 0.6689842939376831, -0.47249072790145874, -0.1728021800518036, -1.1693012714385986, -1.5149810314178467, 0.578648567199707, -0.15462970733642578, -1.1440399885177612, 1.1499383449554443, 1.7981126308441162, 0.9533337950706482, -0.10945979505777359, -0.2599863111972809, -0.05202966183423996, -0.7596386671066284, 0.05015600472688675, 1.3064407110214233, 0.09029410779476166, -0.2533835768699646, 0.573225736618042, -0.690154492855072, -1.0734974145889282, 0.2230558693408966, -0.32207685708999634, -0.09295757114887238, 0.558964729309082, 0.7979938387870789, 0.24453270435333252, 0.866273045539856], [-0.07898218184709549, 1.2146497964859009, -0.6820234656333923, 0.49470314383506775, -0.008869935758411884, 0.25882965326309204, 0.6763463020324707, 0.2318500429391861, -0.24023555219173431, -0.013068623840808868, 0.14481915533542633, -0.20414584875106812, 0.18018926680088043, 1.6851787567138672, 0.7789556980133057, -0.34845787286758423, 0.3928758502006531, 0.03500451520085335, 1.1509312391281128, 0.7153543829917908, 1.380458116531372, 0.5897586941719055, -0.2987535297870636, -1.098321795463562, -0.5677024722099304, -0.09596694260835648, -0.8419399857521057, -0.137994647026062, 1.3020412921905518, -0.46753665804862976, 0.7919575572013855, -0.7088807225227356], [0.478521466255188, -0.34475740790367126, -0.34746307134628296, -0.5008960962295532, 0.29292064905166626, 1.0831396579742432, 1.688377857208252, -0.5731226801872253, 0.9583090543746948, 1.2081953287124634, 0.09036503732204437, 1.7234984636306763, -0.36736753582954407, -0.7665424942970276, 2.013050079345703, -0.1651657521724701, -0.7464852929115295, -0.5152506232261658, 0.519679069519043, -0.709051251411438, 2.333120822906494, -0.21412229537963867, -0.09905549883842468, -0.7774651050567627, 0.9839559197425842, -1.2375491857528687, -1.731736421585083, 1.2142938375473022, 0.25218477845191956, -0.6069054007530212, 0.3768358528614044, -0.8682174682617188], [0.20632518827915192, -0.36078041791915894, -0.2984868586063385, 1.0744613409042358, 0.12288139760494232, -0.30939584970474243, -0.21595561504364014, 1.0599802732467651, 0.3446039855480194, 0.04780568554997444, 0.41344505548477173, 0.5098648071289062, 1.3543394804000854, 0.3928103744983673, 1.300089716911316, -0.4017385244369507, 0.7058505415916443, -0.8724790215492249, -0.10018957406282425, 1.509549617767334, 1.8238874673843384, -0.6957025527954102, -1.018667221069336, -0.10921212285757065, -0.7740700244903564, -0.9276002645492554, -0.21744059026241302, 1.0551154613494873, 0.042465437203645706, 0.08301730453968048, 1.4499205350875854, -0.5776576995849609], [0.21626922488212585, -0.8173906803131104, -1.1727343797683716, 1.9048746824264526, -0.16918125748634338, -1.2336673736572266, -0.9361960291862488, -0.20065468549728394, 1.0477858781814575, -0.5796144008636475, 0.6002376675605774, -0.048462316393852234, 2.9268486499786377, 0.36449870467185974, -0.3382088840007782, -0.28184640407562256, -0.4735881984233856, -0.7134026288986206, -0.17883071303367615, 0.5571626424789429, 1.524764060974121, -0.23630675673484802, -0.38220810890197754, -0.40045633912086487, -0.10102497041225433, -0.03264978155493736, -1.0803378820419312, 2.951627492904663, -0.4689261019229889, -1.3311254978179932, 1.2156963348388672, -0.7431456446647644], [-0.4582955241203308, 1.8299118280410767, -0.7988630533218384, 0.27915725111961365, -0.30536991357803345, 2.573208808898926, 1.0815175771713257, 0.18859969079494476, 0.029838141053915024, 0.679495632648468, -0.5570358037948608, 0.16345509886741638, -0.9805455207824707, 1.7038062810897827, 0.058872323483228683, -0.33510905504226685, 1.1040891408920288, 0.15264157950878143, 1.086630940437317, 0.6907624006271362, 0.7148807048797607, 0.8864983916282654, -1.658286452293396, -1.4294655323028564, -2.231325626373291, -0.020480085164308548, -1.9797807931900024, 1.0378286838531494, 0.4983501136302948, -0.6280490756034851, 0.7284888625144958, -1.1415073871612549], [1.2651174068450928, -0.07814328372478485, -2.925762176513672, -0.29014262557029724, 1.7697169780731201, 0.6517499089241028, 0.5063607692718506, 0.3477787375450134, 0.30529746413230896, 0.5372058749198914, -0.7181239128112793, -0.4360804259777069, -0.2737833559513092, 0.1395270973443985, -0.20958639681339264, -0.5278944373130798, 0.5517902374267578, 1.2857106924057007, -0.5160372257232666, -0.287730872631073, -0.6125363111495972, -0.2980476915836334, 1.1149709224700928, -0.48827627301216125, -0.35813361406326294, 0.23643308877944946, 0.5822793841362, -0.6790259480476379, -0.3826337456703186, 0.1494160145521164, -0.00711863674223423, -0.6923080682754517], [-0.9768964052200317, 0.8563757538795471, 1.1816085577011108, -0.5871663093566895, 1.1854393482208252, 1.0461441278457642, 0.4981567859649658, 0.40048566460609436, -0.19098876416683197, -0.9913711547851562, -0.8828455805778503, 0.7884082794189453, -1.253786563873291, -0.9519399404525757, -0.6117569208145142, 0.42013415694236755, -0.7663736343383789, -0.22356966137886047, 0.10268247127532959, 0.9403247833251953, -0.048875756561756134, 0.08623576164245605, -0.37096938490867615, 0.5158044099807739, 0.500579833984375, 1.5074185132980347, 0.5997977256774902, 0.3492681682109833, -0.6788334250450134, 0.4934478998184204, -0.9805678129196167, -0.4985446035861969], [-0.21527743339538574, 0.8992692232131958, 0.5682779550552368, 1.5448682308197021, -0.19838419556617737, -0.7999312281608582, -0.7891885042190552, -0.6652029752731323, -0.838483452796936, -0.6867095828056335, 0.5767704844474792, 1.2288061380386353, -1.0359549522399902, -0.9669848680496216, 0.03252171352505684, -1.2586283683776855, 0.9705944657325745, -0.5212181806564331, 0.593762993812561, 3.0841963291168213, 0.24749886989593506, -0.41832226514816284, -0.04643271490931511, 0.24961550533771515, -1.3701943159103394, 1.6551506519317627, -0.550557017326355, -0.125774085521698, -1.3176355361938477, 0.5022389888763428, 0.10657884925603867, 0.20170065760612488], [-0.601564884185791, -0.11063356697559357, 1.5280907154083252, 0.4048551917076111, 1.0286729335784912, 1.7084224224090576, 0.15914785861968994, 0.28430798649787903, 0.2889576852321625, -1.163764476776123, -1.4749985933303833, 1.4496581554412842, -1.4255337715148926, -1.5520597696304321, -0.5313063263893127, 0.7281718850135803, -1.5598887205123901, -0.6950523853302002, -0.645246684551239, 1.0581120252609253, -0.11651334911584854, 0.3141372501850128, 0.3870425522327423, 0.8612684607505798, 1.1388812065124512, 1.591821551322937, 1.2329752445220947, 0.5314118266105652, -0.7533384561538696, 0.4458516836166382, -1.1360079050064087, -1.0057094097137451], [-0.3354003429412842, 1.2914320230484009, -0.9017462134361267, -0.7731629610061646, 0.6705870628356934, 0.3576551079750061, 0.7093724012374878, 0.7767346501350403, -0.0996527448296547, -0.543430745601654, 1.2477864027023315, -0.1931096315383911, -1.1208953857421875, 1.9290943145751953, 0.056642163544893265, -1.1348044872283936, 1.2016208171844482, -0.34362542629241943, -1.8196061849594116, -0.012042592279613018, -1.4755666255950928, 0.09778036177158356, 0.3589177131652832, -0.7901972532272339, -0.6858111023902893, -0.0561407133936882, -0.4597959816455841, -0.6626840233802795, -0.5516021847724915, -1.4052786827087402, -0.46274057030677795, 0.3128618001937866], [-2.1702964305877686, 3.201167583465576, 0.27507004141807556, -3.0764429569244385, 1.6914995908737183, -0.21474525332450867, 1.1590417623519897, 0.39106979966163635, -1.699870228767395, -0.7722936868667603, 0.15315040946006775, -0.33925652503967285, -1.3209419250488281, 0.19698096811771393, -0.7586472630500793, -0.2091657519340515, 0.877727210521698, 0.7161048650741577, 2.0362343788146973, 0.8810868859291077, -0.3199440836906433, -0.4658140242099762, -2.13038969039917, -0.29292988777160645, -1.399513602256775, 1.6466602087020874, -1.0470454692840576, -0.047285690903663635, -0.704998791217804, 0.6554996967315674, -0.7253820896148682, 0.6441197395324707], [-0.8169336915016174, 1.3116220235824585, 0.11446762830018997, -0.927678108215332, 0.8530439138412476, -0.10087370127439499, 0.5718933939933777, 1.1098216772079468, -0.7412393689155579, -0.09992178529500961, 0.31873705983161926, 0.24106158316135406, -0.13981790840625763, 0.3820401430130005, 0.8804838061332703, -0.37718456983566284, 1.1402575969696045, -0.2162991762161255, 0.879352331161499, 1.4846925735473633, 1.0306423902511597, -0.7411240339279175, -1.694435954093933, -0.15669703483581543, -1.2604581117630005, -0.1115238219499588, -0.41345566511154175, 0.08266139775514603, -0.1121295690536499, 0.6705760359764099, 0.6322062015533447, 0.007424835581332445], [0.5053229331970215, -0.27708742022514343, 0.6991890668869019, 0.5683419704437256, 0.5166811943054199, 0.019094007089734077, -0.16014638543128967, 0.715224027633667, -0.10947920382022858, -1.2072014808654785, -0.34405648708343506, 0.34179189801216125, 0.12409578263759613, 0.01568390615284443, -0.37038636207580566, 0.4893639087677002, 0.5760980248451233, 1.589927077293396, 0.7027089595794678, -0.25280314683914185, 1.5597673654556274, -0.7029095888137817, -0.08777307718992233, -1.0188392400741577, 0.3699856102466583, -0.5470642447471619, -1.2747491598129272, -0.836001992225647, -0.35078954696655273, 0.09259679913520813, 0.80714350938797, -1.0377827882766724], [0.8768134713172913, 0.7390431761741638, -0.40436112880706787, -0.06871335953474045, -1.882622241973877, -0.6002033948898315, -1.283966064453125, -0.9839689135551453, 0.6196017265319824, 0.49115607142448425, -0.13877269625663757, 1.29429030418396, -0.24580581486225128, -0.2420409768819809, 0.9530833959579468, -0.044399410486221313, -1.1414309740066528, 1.0291602611541748, -0.2547490894794464, 1.2830091714859009, 0.8167413473129272, -0.8155077695846558, -0.634107232093811, -0.08821992576122284, -0.6772286891937256, -0.7000346779823303, 0.4444263279438019, 2.1599347591400146, 0.2374550700187683, 0.015885252505540848, 0.2097775936126709, -0.25328847765922546], [-0.5522222518920898, 0.0022059602197259665, -0.5892892479896545, 1.130748987197876, -0.4498162865638733, -0.041215188801288605, 0.09260059893131256, -1.2811979055404663, -0.035464316606521606, -1.493977665901184, 0.7920371294021606, -0.35002097487449646, 0.3714028596878052, -0.013237073086202145, -0.16964195668697357, 0.7525573372840881, 0.5099525451660156, -0.35968735814094543, -1.2153608798980713, -0.2913685142993927, -0.8483994007110596, 0.02166747860610485, 0.6239774227142334, -1.4671167135238647, 0.6559351682662964, 0.3304396867752075, -1.5827378034591675, 1.1323590278625488, 1.8306562900543213, -0.9496941566467285, 0.9882822632789612, 0.7098451256752014], [0.47111043334007263, -1.130443811416626, -0.2972155511379242, -1.4663642644882202, 1.948340892791748, -0.07093663513660431, -0.5714645981788635, -1.5510715246200562, -0.35804852843284607, -0.6660836935043335, 0.1048312783241272, -0.10289356857538223, -1.0146331787109375, -0.5937597751617432, -0.335096150636673, 0.24808678030967712, 0.15723663568496704, 0.5706213712692261, 1.1106867790222168, 0.9022416472434998, 0.8410325050354004, 0.63467937707901, 0.04255242273211479, 0.5625033974647522, -0.21091146767139435, -0.21091172099113464, -1.4591820240020752, -0.32793861627578735, -0.33834269642829895, 0.6463388800621033, 1.0736515522003174, -1.3465415239334106], [1.1646991968154907, -0.7371556758880615, -0.04127010330557823, 0.5989235043525696, 0.5630322098731995, 0.36937692761421204, -0.191400408744812, 0.003656964050605893, 0.22859543561935425, -0.2701813876628876, 1.3279647827148438, -0.4278489649295807, 0.21864831447601318, 1.6269766092300415, 0.040179651230573654, -0.2578409016132355, 0.0642421767115593, -0.025584260001778603, -0.12670712172985077, 0.8846870064735413, -0.08067010343074799, -0.06205850839614868, 0.07639329135417938, -0.7000454664230347, -0.4650512635707855, -0.5564103722572327, -0.3967564105987549, 0.7271682024002075, 1.9320337772369385, -0.9129530191421509, 0.0483781099319458, 0.34332722425460815], [-0.3372781574726105, 0.22124110162258148, -0.6664911508560181, 0.9380764961242676, -1.0054290294647217, 0.3694404363632202, -0.13231763243675232, -1.2937699556350708, -0.42177432775497437, -0.5302858948707581, 0.021622173488140106, -0.28942546248435974, 0.10049678385257721, 0.10951025784015656, 0.7246573567390442, 0.3952785134315491, -0.1620025932788849, -1.0656436681747437, -0.8818131685256958, -0.0046338667161762714, -1.4679348468780518, 0.3237345218658447, 0.5618717670440674, -1.4382349252700806, 0.6335424780845642, 0.6839306950569153, -2.041963577270508, 1.4333438873291016, 1.3381093740463257, -0.7512016892433167, 0.6121994256973267, 0.8090383410453796], [-0.026029018685221672, -0.3919191360473633, -0.6580996513366699, 0.5332632660865784, -0.09073615074157715, 0.5680493116378784, 0.5990260243415833, -1.2418525218963623, 0.27439871430397034, -0.3632868528366089, 0.5222904086112976, 0.6870447397232056, -0.06117842346429825, -0.19731208682060242, 0.8763163685798645, 0.36054232716560364, -0.11822932958602905, -0.4827701151371002, -0.4355085790157318, -0.3032797574996948, 0.23468291759490967, -0.09591974318027496, 0.4141964614391327, -1.2522262334823608, 0.4629037380218506, -0.4326103627681732, -1.8372255563735962, 1.2663534879684448, 1.3655381202697754, -0.944088339805603, 0.7706121206283569, 0.0417289175093174], [0.05825839936733246, -0.3916492760181427, 2.5278780460357666, -1.243180751800537, 0.7919711470603943, 0.5692262053489685, -0.4661218225955963, -0.4580835700035095, -0.065090611577034, 0.15526798367500305, -1.8351715803146362, -1.0372999906539917, 0.07347900420427322, -1.2213153839111328, -0.32551082968711853, 1.3329288959503174, -0.004183335695415735, -0.41330793499946594, 0.22156274318695068, 1.7650220394134521, 1.088935136795044, 0.3657127022743225, -0.8913317322731018, 0.23428548872470856, -0.6721348762512207, 0.7523354291915894, -1.5910816192626953, -0.941922128200531, -0.9065595865249634, -0.1490376740694046, -0.16860967874526978, -1.0733332633972168], [0.22207462787628174, 1.1453602313995361, -0.6696281433105469, 0.5917033553123474, 0.19337862730026245, -0.811747670173645, 0.5824887752532959, 0.2616000771522522, -0.532197892665863, -0.37834328413009644, 0.4639514088630676, -0.6337190270423889, 0.6954931020736694, 2.061138391494751, 1.092847466468811, -0.36763978004455566, -0.002641078317537904, 0.18698783218860626, 1.4361642599105835, 0.6021318435668945, 1.9522982835769653, 0.8448641896247864, 0.7068491578102112, -0.9732217788696289, 0.39870715141296387, -0.2347247302532196, -0.4304667115211487, -0.8823114037513733, 1.9455609321594238, -0.48190703988075256, 0.806023895740509, -0.6051933169364929], [-0.6011489629745483, 2.0379528999328613, 0.5362175107002258, -2.2424824237823486, 1.3471670150756836, -0.04947137087583542, 0.6409448385238647, 0.3214520215988159, -0.8415245413780212, -0.9298807978630066, 0.08485782146453857, -0.262809157371521, -0.8365616798400879, 0.1774943768978119, -0.6281460523605347, -0.6513147354125977, 0.6295035481452942, 0.3204802870750427, 1.2786319255828857, 0.6672802567481995, -0.16942663490772247, -0.1766817718744278, -1.1780531406402588, -0.37686267495155334, -0.7864205241203308, 1.102648377418518, -0.5794751644134521, 0.5053942203521729, -0.7879375219345093, 0.5391265153884888, 0.08299840241670609, 0.6837804913520813], [1.617163062095642, 1.6898467540740967, -0.7444182634353638, 0.2374289482831955, -2.1536667346954346, -0.6877524852752686, -1.6930893659591675, 0.611946702003479, 0.5117653012275696, -1.6126573085784912, 1.3560632467269897, -0.5445347428321838, -0.3363504111766815, 0.25722062587738037, -1.8663188219070435, 0.6547592878341675, -0.27025651931762695, 0.5795271992683411, -0.8965862989425659, -0.7498565912246704, 1.0403227806091309, -0.9163583517074585, -0.4994699954986572, -0.4418337345123291, 0.05664670839905739, 0.15591149032115936, 2.1949946880340576, -0.2922731637954712, 0.06633606553077698, -0.8749701380729675, 0.012255419045686722, -0.31870216131210327], [-0.3337290585041046, 0.2200261503458023, -0.738894522190094, 0.9404316544532776, -0.5213382244110107, 0.03741107136011124, 0.19879218935966492, -0.9936401844024658, 0.02946012280881405, -0.895748496055603, 0.03685244545340538, 0.02482525259256363, 0.5495476722717285, -0.15971535444259644, -0.19171035289764404, 0.45653387904167175, 0.8742794990539551, 0.20125529170036316, -0.4108864665031433, -0.28444474935531616, -0.5933198928833008, -0.05977759510278702, 0.6747695803642273, -0.7614286541938782, 0.35582610964775085, 0.011839032173156738, -1.155514121055603, 0.6166346669197083, 1.37973153591156, -0.6569084525108337, 1.1650729179382324, 0.015239945612847805], [0.2204235941171646, 0.5866328477859497, 1.303825855255127, 0.2748521864414215, 1.7479934692382812, 0.7141790390014648, -0.09901408106088638, 0.47522231936454773, 1.0392074584960938, 1.2976497411727905, -1.4165291786193848, 1.1583095788955688, -0.40122494101524353, -1.2215664386749268, 0.05221470072865486, 0.3190760016441345, -0.7373791337013245, 0.062093451619148254, -0.3271749019622803, -1.289268970489502, 0.5296734571456909, 0.8221820592880249, 2.440016031265259, 0.5913779139518738, 0.9518687725067139, -1.5796769857406616, 1.0747355222702026, 0.9416930079460144, 0.17925593256950378, -1.3414191007614136, 0.19577090442180634, 1.3380589485168457], [-1.6528996229171753, 0.5509779453277588, -0.3928700089454651, -0.33467987179756165, -0.10706377029418945, -0.9321660399436951, 0.2190394550561905, -1.0340110063552856, 0.5595306158065796, -0.07623553276062012, -0.33188673853874207, 0.6785350441932678, 0.4758140444755554, -0.10464209318161011, -0.8458853363990784, -0.09580522775650024, 0.2850644886493683, -0.12274247407913208, 1.4275102615356445, 0.7119591236114502, -0.5250586271286011, -0.7926977276802063, -1.849013328552246, -1.8304383754730225, -0.9624718427658081, 3.027078151702881, -0.08362149447202682, 0.02530556544661522, -0.6121642589569092, -0.6902211308479309, 0.1265682876110077, -0.5739989876747131], [0.19384565949440002, 0.04936027154326439, -1.5161566734313965, -0.40850430727005005, 0.3191732168197632, -0.25775402784347534, -0.1901354342699051, 0.2695891261100769, 0.06923845410346985, 0.8912902474403381, -0.7007105350494385, 0.5473315119743347, -0.1253042370080948, 0.42055293917655945, 0.08397184312343597, 0.5022478103637695, -1.069644808769226, 0.33873578906059265, -0.26374295353889465, 0.46496710181236267, 0.626791775226593, -0.9495285153388977, -1.1163986921310425, -0.7449535131454468, -0.3618369996547699, 0.32369086146354675, -0.04111958295106888, 0.00901935063302517, -0.17098848521709442, 0.058262407779693604, -0.3675247132778168, 0.11527280509471893], [1.5865535736083984, 1.6600041389465332, -0.7244247794151306, 0.20616208016872406, -2.1155967712402344, -0.6821460127830505, -1.666345477104187, 0.5823142528533936, 0.5031499862670898, -1.5649309158325195, 1.311306118965149, -0.5163474678993225, -0.32437407970428467, 0.2526051700115204, -1.8324706554412842, 0.6356351375579834, -0.28421729803085327, 0.5729861259460449, -0.881373405456543, -0.7446986436843872, 1.0289183855056763, -0.9017174243927002, -0.5206572413444519, -0.46258044242858887, 0.028278442099690437, 0.13662318885326385, 2.181478500366211, -0.2646217942237854, 0.0648406371474266, -0.8696157932281494, -0.002949515823274851, -0.3070610761642456], [-0.853766918182373, -1.042364478111267, 1.3444243669509888, 0.17146597802639008, -0.10158934444189072, 0.40360140800476074, -2.0040805339813232, 0.8900959491729736, 0.41010844707489014, 1.3297994136810303, 0.7474485635757446, -0.5648562908172607, -0.2636692523956299, -1.0009338855743408, -0.2751607596874237, -1.377707600593567, 0.44554609060287476, 0.7142277956008911, 0.6208756566047668, 0.8417989611625671, -1.7161654233932495, -0.14859186112880707, -0.20216529071331024, -0.7779883742332458, -1.0568684339523315, 0.0190399382263422, 1.5407651662826538, 1.867384433746338, 1.5213836431503296, 0.13581719994544983, -0.7306628823280334, 0.9103125333786011], [0.9534333944320679, -0.33175691962242126, -1.7642844915390015, -0.5179826617240906, 1.8704668283462524, 0.9208770394325256, 0.3224412500858307, 0.177664116024971, 0.1637859046459198, 1.2211418151855469, -0.6033985614776611, -0.2809835970401764, 0.11435278505086899, 0.42524173855781555, -0.8811841607093811, 0.034121789038181305, 1.1900206804275513, 1.110200047492981, -0.1893065869808197, -1.215782880783081, 0.3374044895172119, -0.4497959613800049, 0.6087636351585388, 0.06261967867612839, -0.0016004751669242978, 0.37031182646751404, 0.10022857040166855, -0.1751270443201065, -0.3969399034976959, 0.39931702613830566, -0.6506306529045105, -0.050655465573072433], [0.2823832929134369, 0.8997411727905273, 0.04903360828757286, 0.09868520498275757, -1.3857009410858154, 0.8186917304992676, 0.042403385043144226, -0.07550986856222153, -0.6408483386039734, 1.598118543624878, -1.1256887912750244, -0.8139486312866211, 0.039002448320388794, 0.5964047312736511, 1.9095759391784668, -0.5997478365898132, -1.0453238487243652, -1.9041461944580078, 0.007996812462806702, 0.4876633584499359, -1.465624451637268, 0.971725344657898, -0.007504101376980543, -0.8588038682937622, 1.229182481765747, 1.227911353111267, -1.9659793376922607, 1.5103603601455688, 0.059767983853816986, 0.07335327565670013, -0.4539269804954529, 0.8719434142112732], [0.7529497742652893, 0.2424435019493103, -2.895866870880127, -1.1355661153793335, 0.21605713665485382, -0.3530197739601135, -0.48260360956192017, 0.5078395009040833, 0.5804657936096191, 1.1450281143188477, -1.6655834913253784, 0.6352511644363403, -0.2753114402294159, 1.117674708366394, -0.32636237144470215, 0.6984349489212036, -1.8489735126495361, 0.9888595938682556, -0.5682873725891113, -0.21676728129386902, 0.17184461653232574, -0.9861960411071777, -1.8859076499938965, -1.112907886505127, -0.657184362411499, 1.0084898471832275, 0.6184043288230896, 0.043832942843437195, -0.12492325156927109, 0.2172052562236786, -1.157065987586975, 0.8018503189086914], [-0.052854496985673904, 0.8621883988380432, -0.24269309639930725, 0.5031836032867432, 0.14828674495220184, 0.7095041871070862, 0.04639118164777756, 0.29161426424980164, -0.06834696233272552, -0.1241682916879654, -0.24300707876682281, 0.48964276909828186, -0.6256603598594666, 0.3946109414100647, -0.3425988554954529, -0.31189051270484924, 0.2381952852010727, -0.40384548902511597, 0.5164994597434998, 0.9347401857376099, 0.26097315549850464, 0.2504044473171234, -0.6527279019355774, -0.5231205224990845, -1.0078803300857544, 0.7255632281303406, -0.46888595819473267, 0.10349950939416885, -0.7418239712715149, 0.13407552242279053, -0.17809361219406128, -0.43032461404800415], [-0.35595014691352844, 0.4646456837654114, 0.7157250046730042, 0.8023975491523743, 0.8581448793411255, -1.583527684211731, -0.021914735436439514, -0.021714165806770325, -0.7441413998603821, 1.4442548751831055, 0.12508174777030945, -0.31302180886268616, -0.659032940864563, 0.527682900428772, -1.723762035369873, 1.9245127439498901, -1.2890980243682861, 0.08378137648105621, 0.07723607122898102, 0.49557676911354065, -1.7232329845428467, -0.18980145454406738, -0.37879395484924316, 0.33176085352897644, -0.8287684917449951, 0.3793891370296478, -0.41547611355781555, 0.489016056060791, 0.3106408417224884, 0.6612929105758667, -0.7352783679962158, 0.1418815702199936], [-0.593206524848938, 0.4357578754425049, 0.7543697953224182, 0.9925131797790527, 0.9445487260818481, -1.6583772897720337, 0.049079544842243195, 0.06790987402200699, -0.8651354908943176, 1.6991093158721924, 0.1596188247203827, -0.4317265450954437, -0.9721214175224304, 0.8582591414451599, -1.7972885370254517, 2.1493070125579834, -1.4895391464233398, 0.050293732434511185, 0.008012060075998306, 0.726125180721283, -2.244807481765747, -0.17091983556747437, -0.04671556502580643, 0.3246927261352539, -0.7045589685440063, 0.6021265983581543, -0.951012372970581, 0.9258920550346375, 0.7039283514022827, 0.7386608719825745, -0.6771101951599121, 0.040371283888816833], [-1.6145187616348267, 0.5267791152000427, -0.4016619622707367, -0.33238500356674194, -0.09615208208560944, -0.911608874797821, 0.24212515354156494, -1.0187857151031494, 0.5612555146217346, -0.06960742175579071, -0.3394996225833893, 0.6715236902236938, 0.47759929299354553, -0.08818686008453369, -0.8417325615882874, -0.09163003414869308, 0.26973700523376465, -0.117699533700943, 1.4161407947540283, 0.6837929487228394, -0.5160992741584778, -0.7736493349075317, -1.829343318939209, -1.7700923681259155, -0.9285110831260681, 2.9313864707946777, -0.06077319756150246, 0.00686701200902462, -0.5847145318984985, -0.6675769686698914, 0.14740560948848724, -0.5705119967460632], [0.008075104095041752, 0.5557047128677368, -0.21268883347511292, 0.315346360206604, -1.0684144496917725, 0.5059909820556641, -0.03967701271176338, -0.5832382440567017, -0.4836187958717346, 0.4409463703632355, -0.5197427868843079, -0.45876652002334595, -0.02966877445578575, 0.3422285318374634, 1.0942023992538452, -0.05615726485848427, -0.443562388420105, -1.2370882034301758, -0.40241920948028564, 0.2306300401687622, -1.2091975212097168, 0.5738474726676941, 0.262919545173645, -0.9610000252723694, 0.9200136065483093, 0.8463246822357178, -1.78139066696167, 1.3384228944778442, 0.6516698598861694, -0.1789397895336151, 0.0199690293520689, 0.765387237071991], [-0.5277569890022278, 0.793532133102417, 0.5012729167938232, 0.7581286430358887, 0.7319967746734619, -0.6890182495117188, 0.3007831871509552, 0.20239315927028656, -0.6167429089546204, 1.6030535697937012, -0.07046929001808167, -0.31166014075279236, -0.9674307703971863, 1.0510993003845215, -1.4501837491989136, 1.5952688455581665, -0.8954944014549255, 0.14967457950115204, 0.3541434705257416, 0.7648826241493225, -1.48744797706604, 0.05859171599149704, -0.4779260754585266, 0.05751581862568855, -1.1276808977127075, 0.4446480870246887, -1.1287723779678345, 0.8570664525032043, 0.5381165146827698, 0.49925899505615234, -0.43593868613243103, -0.3112178444862366], [-0.046349361538887024, -0.4030552804470062, -0.8226372599601746, -0.7181009650230408, -1.6383618116378784, 0.4499548375606537, 2.82415771484375, 0.7173670530319214, -0.9851459860801697, 0.4045060873031616, 0.12420304864645004, -1.5372146368026733, 1.099442481994629, 0.4166257679462433, 0.8560040593147278, -1.7772220373153687, -1.5213418006896973, -1.5914318561553955, 1.2414380311965942, 0.05371985211968422, -1.8537395000457764, 0.55901700258255, 0.22959378361701965, 2.55269193649292, -0.2563430666923523, 0.6676686406135559, 0.7535049915313721, -0.32503780722618103, 0.4458373486995697, 0.6974797248840332, -0.36025920510292053, -0.3371943235397339], [-0.6777556538581848, -0.6152645945549011, 0.053167421370744705, 0.12016661465167999, 0.2830239236354828, 0.8033066391944885, -0.36495962738990784, 0.32529959082603455, -0.10654248297214508, 0.7390457391738892, -0.11684942990541458, 0.41089165210723877, 0.503683865070343, 1.169485330581665, -0.7888762950897217, 0.32730039954185486, -0.8719711899757385, -0.49477073550224304, 0.37717559933662415, -0.7523301243782043, -0.7875750064849854, 0.3109595775604248, -0.33620572090148926, -0.08280584961175919, -0.11405998468399048, -1.3725039958953857, -0.011455152183771133, 1.1812361478805542, 0.5090696215629578, 0.5325160026550293, 0.5675185322761536, 0.2899191975593567], [-1.4799424409866333, 0.15493737161159515, 0.34337908029556274, 0.5648104548454285, 0.06410190463066101, 0.4393778443336487, -0.21746790409088135, 0.28620463609695435, -0.789376974105835, 1.142869234085083, -0.11833792924880981, 0.4139406383037567, 0.46782708168029785, 0.04499673470854759, -0.5761156678199768, 0.2731613218784332, -0.8484744429588318, -0.31546223163604736, 0.19425056874752045, -0.218871608376503, -0.48725757002830505, -0.47558608651161194, 0.19058360159397125, -1.0209882259368896, 0.15514591336250305, -0.16404606401920319, -0.6788120865821838, 1.4101760387420654, 0.20079313218593597, -0.13687899708747864, -0.025763986632227898, -0.3373055160045624], [-0.6129984855651855, -0.11833672225475311, 1.534909725189209, 0.4041156768798828, 1.038987636566162, 1.7224313020706177, 0.1667361557483673, 0.2952363193035126, 0.28754857182502747, -1.172157645225525, -1.4793057441711426, 1.450554609298706, -1.4468446969985962, -1.566635012626648, -0.5403764843940735, 0.7385348677635193, -1.5725573301315308, -0.6934047937393188, -0.6486274600028992, 1.0527150630950928, -0.12240242213010788, 0.31798285245895386, 0.38288676738739014, 0.8774020671844482, 1.1532310247421265, 1.5988529920578003, 1.2464474439620972, 0.5324329733848572, -0.7619062662124634, 0.44632238149642944, -1.14423668384552, -1.0140119791030884], [1.0992072820663452, 0.8675317764282227, -0.32363083958625793, -0.23882952332496643, -2.1000208854675293, -0.708918035030365, -1.4560481309890747, -0.8868676424026489, 0.7487303018569946, 0.761380672454834, -0.27295491099357605, 1.5003103017807007, -0.30342742800712585, -0.28578194975852966, 1.113731026649475, -0.17356367409229279, -1.3982224464416504, 1.268473744392395, -0.1150844544172287, 1.5384302139282227, 1.1152032613754272, -0.9343019723892212, -0.8278729319572449, 0.1265459656715393, -0.818218469619751, -0.8594034910202026, 0.7864258289337158, 2.3082876205444336, 0.011378023773431778, 0.18013004958629608, 0.11039681732654572, -0.3954066336154938], [0.21560558676719666, -0.6808521151542664, -0.8296565413475037, 0.7490045428276062, 0.152305006980896, -0.14782902598381042, 0.15874560177326202, -0.6335757374763489, 0.8114482760429382, 0.02559707872569561, 0.3617746829986572, 0.6116451025009155, 1.2654345035552979, -0.15136116743087769, 0.6038539409637451, -0.08275148272514343, -0.4411102831363678, -0.5686606764793396, 0.05955646187067032, -0.007475373335182667, 1.4992501735687256, -0.1341460794210434, -0.12591971457004547, -0.6161736845970154, 0.34662216901779175, -0.47442206740379333, -1.5465306043624878, 2.030133008956909, 0.05244578793644905, -0.9388399124145508, 0.9136887192726135, -0.6856433749198914], [-0.45475253462791443, 1.832245111465454, -0.8021726608276367, 0.2797321379184723, -0.3081814646720886, 2.5767669677734375, 1.0820475816726685, 0.18728716671466827, 0.0284610353410244, 0.6822259426116943, -0.5591292381286621, 0.16681598126888275, -0.9828894734382629, 1.7056251764297485, 0.06099064648151398, -0.3369687497615814, 1.1030242443084717, 0.15291237831115723, 1.0890346765518188, 0.6938393712043762, 0.7138708829879761, 0.8873978853225708, -1.6595981121063232, -1.4298551082611084, -2.2381951808929443, -0.022622134536504745, -1.9801784753799438, 1.0386618375778198, 0.49431294202804565, -0.6295450329780579, 0.7265908718109131, -1.1459026336669922], [-0.06848358362913132, 0.08142068237066269, 0.4773555397987366, 1.1886272430419922, 1.212256669998169, -0.7652289867401123, 0.44730234146118164, -1.6814801692962646, -0.7361541390419006, 0.059923216700553894, -0.22378315031528473, 0.3278346061706543, 1.4587942361831665, -2.021900177001953, -0.2366315871477127, 1.1968955993652344, 0.5296624898910522, -0.43775931000709534, 0.6536304354667664, -0.028942594304680824, -0.30875980854034424, 0.2874777913093567, -1.5366482734680176, 0.6177477240562439, -1.4178308248519897, -0.44013839960098267, -0.204107403755188, -1.9779971837997437, -1.5593857765197754, 0.6238061785697937, -0.4606109857559204, 1.4177974462509155], [-0.7855695486068726, -0.8852876424789429, 1.2889868021011353, 0.037749409675598145, -0.0900329053401947, 0.2705971896648407, -1.8728309869766235, 0.7888596057891846, 0.36363381147384644, 1.1338200569152832, 0.8319787383079529, -0.5232546329498291, -0.451690137386322, -0.9774385094642639, -0.26128268241882324, -1.2773120403289795, 0.41007548570632935, 0.7680341005325317, 0.5336570143699646, 0.7710012197494507, -1.4425740242004395, -0.23562686145305634, -0.07860535383224487, -0.5279480218887329, -0.8442633748054504, -0.03960351645946503, 1.5252420902252197, 1.6552960872650146, 1.5257011651992798, 0.14740456640720367, -0.8103926181793213, 0.821305513381958], [0.17809291183948517, -0.028759809210896492, 0.45632901787757874, -0.48770761489868164, 1.2525503635406494, -2.0693180561065674, 0.1426260769367218, -0.2805991470813751, 0.528083086013794, 0.27932456135749817, 1.6919955015182495, -0.21740342676639557, 0.9419012069702148, -1.264675259590149, -1.0712552070617676, -0.8539832234382629, 0.9568798542022705, -1.0301244258880615, 1.9155455827713013, -0.12518125772476196, 1.838435173034668, 0.17274557054042816, -2.299222230911255, 0.7877119183540344, -0.20982544124126434, 0.1563478261232376, 0.09999912232160568, -0.04534950107336044, 1.8035918474197388, 0.8736099004745483, -0.5286820530891418, 0.19394870102405548], [0.4935417175292969, 1.1210451126098633, 0.0359516441822052, 0.2775897681713104, -0.8074872493743896, 0.4370003044605255, -0.02147560566663742, 0.5019571185112, 1.236417293548584, 0.6247748136520386, -0.10544085502624512, -0.10601808875799179, 1.4780621528625488, -0.737031102180481, -1.1662876605987549, 0.08378182351589203, 1.912858009338379, 2.160268783569336, 0.9745733141899109, -0.6397210359573364, 0.1441391110420227, -0.38744038343429565, 0.9408067464828491, 1.6178576946258545, -0.36073532700538635, -0.5499568581581116, -0.5735722184181213, -0.40325430035591125, 0.13321149349212646, 0.37080317735671997, 0.15965378284454346, 0.30413684248924255], [0.79880690574646, 0.14123088121414185, -0.1578664481639862, -0.524223268032074, -0.31720277667045593, 1.1715830564498901, 0.8304451704025269, 0.8082506656646729, 1.7299867868423462, 1.1371673345565796, 0.47857987880706787, 0.9823302626609802, -0.335605651140213, -0.7847056984901428, -0.88967365026474, 0.020384108647704124, 1.1112031936645508, 0.3210585415363312, -1.4892438650131226, 1.5357513427734375, -0.05553831532597542, 0.42528146505355835, 0.5005768537521362, 0.18359142541885376, 0.5450814962387085, -1.3043596744537354, -0.654640257358551, 0.7899975776672363, 1.7989065647125244, 0.3095366656780243, 0.6159170866012573, 0.7828629612922668], [-0.35123804211616516, 0.5342971086502075, -0.13157311081886292, 0.1828646957874298, 0.16098527610301971, -0.2750602960586548, 1.3404053449630737, 0.44013452529907227, 0.02087077684700489, 0.5101296305656433, -1.4195352792739868, 0.2563856542110443, 1.0552836656570435, -0.16859211027622223, -0.4318191111087799, -0.7463417053222656, 0.5550975799560547, 0.6715503334999084, 1.1199092864990234, -0.23229734599590302, 0.12504132091999054, -0.26739829778671265, -0.22000470757484436, 0.2580417990684509, 0.8287017941474915, -0.6498118042945862, 1.1439003944396973, -0.08675827085971832, 0.9765335321426392, 0.20800268650054932, 1.3040908575057983, -1.6527611017227173], [0.2107165902853012, 0.6373689770698547, -0.8745869398117065, -0.11346326023340225, 0.18986110389232635, -0.6197234988212585, 1.74277663230896, 0.2954847812652588, 0.36070504784584045, 0.4380386471748352, -1.8399511575698853, 0.27262023091316223, 0.9213262796401978, -0.26215991377830505, -0.28554484248161316, -0.7325172424316406, 0.4690646529197693, 1.082103967666626, 1.0566165447235107, -0.10359986126422882, 0.48245540261268616, -0.5071232914924622, -0.5987767577171326, 0.44421374797821045, 0.9402520656585693, -0.3896082639694214, 1.5392465591430664, -0.8583648204803467, 0.9523066282272339, 0.14597204327583313, 1.1437437534332275, -1.6854372024536133], [0.5250840187072754, 0.20649957656860352, -0.15625795722007751, 0.6277165412902832, -0.2606273889541626, -0.13387268781661987, -0.2491600662469864, 1.4844729900360107, 0.11095350980758667, 0.01421279739588499, 0.6305660009384155, 0.45726731419563293, 0.5545027852058411, 0.4639759957790375, 1.3700604438781738, -0.29011619091033936, 1.060294270515442, -0.6440426111221313, -0.20695340633392334, 1.4391776323318481, 1.8706120252609253, -0.9570448398590088, -1.2163965702056885, -0.11002068966627121, -0.9443593621253967, -1.1635950803756714, 0.5171049237251282, 0.10005596280097961, 0.2689046561717987, 0.3975442051887512, 1.3682434558868408, -0.45099109411239624], [0.6130481958389282, -0.874244749546051, 1.2854102849960327, 0.5353334546089172, 0.6492220163345337, 0.3846769332885742, -0.4849785268306732, 0.9011204838752747, 0.07095026224851608, -1.538132905960083, -0.7076680064201355, 0.7580792903900146, -0.12120527774095535, -0.8530704379081726, -1.0086222887039185, 0.8603931665420532, 0.8454756140708923, 2.175142526626587, 0.3631852865219116, -0.6178481578826904, 1.3709317445755005, -1.3470244407653809, -0.4210609197616577, -1.0150049924850464, 0.35784149169921875, -0.6646655201911926, -1.629229187965393, -0.7953088879585266, -1.3620021343231201, 0.33733564615249634, 0.799957275390625, -1.2198400497436523], [0.3026791214942932, -0.48311853408813477, 0.03281693905591965, -0.8745116591453552, 1.9694063663482666, 1.3732106685638428, 0.17318527400493622, 0.054628197103738785, -0.09650446474552155, 2.05598783493042, -0.08256015926599503, -0.06166481599211693, 0.4467598497867584, 1.2744684219360352, -1.7516096830368042, 0.7846126556396484, 2.3806304931640625, 0.5892791748046875, 0.12263797223567963, -2.3991785049438477, 1.4320975542068481, -0.5614592432975769, -0.18622834980487823, 0.6314966082572937, 0.29900187253952026, 0.5181549787521362, -0.7806625962257385, 0.3649544417858124, -0.456017404794693, 0.32139116525650024, -1.538974404335022, 0.9216710329055786], [-0.409445196390152, 0.48680970072746277, -1.1545124053955078, 1.738472580909729, 0.4059745669364929, 0.552725076675415, 0.18434756994247437, 0.31778597831726074, 0.008216166868805885, -1.0253992080688477, -0.048070117831230164, -0.2938924729824066, -0.1119876429438591, -0.3553597331047058, -0.9154874682426453, -0.2659473419189453, 1.0087738037109375, -2.127723455429077, 1.1605980396270752, -1.6763849258422852, -0.5917277932167053, -0.07279229909181595, 0.9945732951164246, 0.5852652788162231, 0.6094780564308167, -0.23057222366333008, 1.261418342590332, -0.5655962824821472, 0.6365717649459839, 1.2840107679367065, -1.075596570968628, 0.11428025364875793], [-0.7802849411964417, 0.40373238921165466, 0.08108317852020264, -1.3878862857818604, -0.6699783205986023, -0.3681207597255707, 0.5261743664741516, 0.3948127031326294, -0.279356986284256, -0.6021729111671448, 0.2172602117061615, -0.2018585205078125, -1.6563630104064941, 0.35644635558128357, -0.8225013017654419, 0.779045045375824, 1.1806353330612183, 1.2595956325531006, -0.950356125831604, -0.3881015479564667, 2.2279086112976074, -0.2199394702911377, 2.317917585372925, 1.9330042600631714, 1.3852014541625977, -0.53543621301651, 0.654969334602356, 0.4096605181694031, 0.891555905342102, 0.5599733591079712, -0.8164061307907104, -0.3954569697380066], [0.25195157527923584, 0.2764184772968292, -0.5146515369415283, 0.4630424380302429, -0.12743480503559113, -0.47599202394485474, 0.6428895592689514, 0.18867191672325134, -0.5024691820144653, -0.28294089436531067, 0.43770408630371094, -0.3084663152694702, 0.6753014922142029, 1.4130715131759644, 0.8535671234130859, -0.5486457943916321, -0.019842876121401787, 0.12149913609027863, 1.3141623735427856, 0.5969561338424683, 1.688012719154358, 0.4911205768585205, 0.21852639317512512, -0.6410204768180847, 0.5117430686950684, -0.1648208647966385, -0.17642492055892944, -0.6666808724403381, 1.4782575368881226, -0.27507928013801575, 0.960679829120636, -0.36062613129615784], [-0.44998684525489807, 1.5386966466903687, -0.8059234619140625, 0.010832774452865124, -0.4074768126010895, 2.1431524753570557, 0.8132612109184265, 0.09305069595575333, -0.14517514407634735, 0.6840975284576416, -0.5276293754577637, -0.049141667783260345, -0.6332175731658936, 1.4926552772521973, 0.030367396771907806, -0.19627434015274048, 0.9524035453796387, 0.3573170602321625, 0.8383525013923645, 0.6588972210884094, 0.6553186178207397, 0.6451585292816162, -1.2680119276046753, -1.0553780794143677, -1.8622187376022339, -0.1219758465886116, -1.6315410137176514, 0.9919485449790955, 0.6853250861167908, -0.2806065082550049, 0.46505653858184814, -0.833692729473114], [-1.64996337890625, 0.5429446697235107, -0.3734915256500244, -0.3211784064769745, -0.10269399732351303, -0.9020583629608154, 0.20175835490226746, -0.9979163408279419, 0.5352062582969666, -0.05542687326669693, -0.3333539664745331, 0.6748073101043701, 0.4825655221939087, -0.0872744768857956, -0.8458365201950073, -0.09429018944501877, 0.2710723280906677, -0.12796401977539062, 1.4098106622695923, 0.6861445903778076, -0.5320534110069275, -0.7771204710006714, -1.813285231590271, -1.8149912357330322, -0.9482061862945557, 2.960303783416748, -0.0876825824379921, 0.0666237398982048, -0.5931015014648438, -0.6696740388870239, 0.13545654714107513, -0.5707384347915649], [-0.07215223461389542, -0.5331073999404907, -2.0970590114593506, 0.18039289116859436, -1.3147164583206177, 0.804493248462677, -0.7932433485984802, 0.4997268319129944, 0.6277307868003845, -0.7193926572799683, -1.8241218328475952, -0.9181119799613953, 1.5548810958862305, -0.9123314023017883, 0.01681448146700859, 0.3782253861427307, 1.2886899709701538, 0.16601938009262085, -1.5689555406570435, -0.22483691573143005, -0.8098894357681274, -0.25641635060310364, -0.3735745847225189, -0.31482699513435364, 0.433871865272522, 0.5455567240715027, 0.4160788953304291, 1.3841135501861572, -0.20622020959854126, -0.8121536374092102, -0.7612966299057007, 0.5057083964347839]], [[-0.7713785171508789, 0.22343406081199646, -0.11667456477880478, -0.6509381532669067, 1.8312207460403442, -0.6138396859169006, 1.995156168937683, -0.6161959767341614, -0.8783025145530701, -1.0582467317581177, 1.8495588302612305, 0.007976903580129147, 1.7538994550704956, 2.221278190612793, 1.0912153720855713, -1.6345044374465942, 0.793724000453949, -1.2049834728240967, -0.5494323968887329, -0.3787364363670349, 2.435338258743286, 0.10047660768032074, 0.4973605275154114, -0.9030768275260925, -0.40301308035850525, 0.3019244968891144, 0.06313247233629227, 0.5673832893371582, 0.7016668915748596, 2.2695798873901367, 0.1972828209400177, 0.8354983925819397], [-0.7713785171508789, 0.22343406081199646, -0.11667456477880478, -0.6509381532669067, 1.8312207460403442, -0.6138396859169006, 1.995156168937683, -0.6161959767341614, -0.8783025145530701, -1.0582467317581177, 1.8495588302612305, 0.007976903580129147, 1.7538994550704956, 2.221278190612793, 1.0912153720855713, -1.6345044374465942, 0.793724000453949, -1.2049834728240967, -0.5494323968887329, -0.3787364363670349, 2.435338258743286, 0.10047660768032074, 0.4973605275154114, -0.9030768275260925, -0.40301308035850525, 0.3019244968891144, 0.06313247233629227, 0.5673832893371582, 0.7016668915748596, 2.2695798873901367, 0.1972828209400177, 0.8354983925819397], [-0.35007810592651367, 0.6223260760307312, 0.2573120594024658, -0.14319144189357758, -0.12602739036083221, 1.537570595741272, 0.41751477122306824, 0.4030625820159912, -1.2824286222457886, 0.3114335834980011, 3.202193021774292, -0.7062605023384094, -0.9249488115310669, -1.3497570753097534, -0.6272128224372864, -0.9566219449043274, 0.3331548273563385, -0.24487623572349548, 1.8384082317352295, -2.8990256786346436, -0.35381415486335754, 0.223599374294281, 0.9622626304626465, 0.10818588733673096, 0.9603262543678284, -1.1621832847595215, -0.98609459400177, 0.3373306095600128, 0.29977312684059143, 0.43694034218788147, 0.5397254228591919, 0.5500810146331787], [-0.6731154322624207, 0.39920955896377563, -0.07237246632575989, -0.5804157853126526, 1.3457010984420776, -0.4938313364982605, 1.455997109413147, -0.1870715469121933, -0.532643735408783, -0.7959239482879639, 1.4763586521148682, 0.03195218741893768, 1.6165167093276978, 1.394384503364563, 1.2759408950805664, -1.0415010452270508, 0.7869617342948914, -1.1029120683670044, -0.37449949979782104, -0.2220812737941742, 1.863233208656311, 0.049482010304927826, 0.7298389077186584, -0.6709796786308289, 0.05914120748639107, 0.10661143064498901, 0.02712853066623211, 0.4602927267551422, 0.5811245441436768, 1.883223295211792, 0.239121675491333, 0.729793131351471], [-0.3307364284992218, 1.149489164352417, 0.16518759727478027, -0.5014362931251526, -0.44234269857406616, -0.10615980625152588, -0.5538574457168579, 1.6495171785354614, 0.7761412262916565, 0.18288305401802063, 0.05854779854416847, 0.05993622541427612, 1.0500271320343018, -2.083667278289795, 2.333289861679077, 1.377286434173584, 0.8642856478691101, -0.6521947383880615, 0.3416493237018585, 0.5115389823913574, -0.4515186548233032, 0.021092763170599937, 1.7896438837051392, 0.2309095859527588, 2.045413017272949, -0.6745029091835022, -0.20554795861244202, -0.059093885123729706, -0.0007059212075546384, 0.39988991618156433, 0.5595108270645142, 0.3614293336868286], [-0.360210120677948, 1.0875153541564941, 0.14632855355739594, -0.5114676356315613, -0.2902179956436157, -0.14018377661705017, -0.3832976222038269, 1.497962236404419, 0.665441632270813, 0.09980425983667374, 0.1783706545829773, 0.05643073469400406, 1.0971734523773193, -1.7956956624984741, 2.25026798248291, 1.1757713556289673, 0.8595756888389587, -0.6891738772392273, 0.28205549716949463, 0.45194530487060547, -0.25837624073028564, 0.026430588215589523, 1.703202247619629, 0.15506333112716675, 1.8816359043121338, -0.6091863512992859, -0.1875111609697342, -0.017195357009768486, 0.04627273231744766, 0.5250117182731628, 0.5353028178215027, 0.39316433668136597], [-0.9000740051269531, 1.516478419303894, 0.2730552554130554, 0.06558778136968613, 0.34758657217025757, 1.726918339729309, 0.07900060713291168, -0.17884203791618347, 0.26698049902915955, 1.1181224584579468, 1.151384711265564, 0.9745106101036072, -0.827316403388977, 1.095238447189331, -0.42016375064849854, 0.7229964733123779, 0.6646490097045898, -0.9650352001190186, -0.8805626034736633, 1.5843141078948975, 0.6725568771362305, -0.5169817805290222, 0.8606976866722107, -1.1620562076568604, 0.06702213734388351, 0.14584891498088837, -2.680680274963379, 0.5688670873641968, 0.5326411128044128, 0.07768935710191727, -0.24170929193496704, 0.024254318326711655], [-0.35212838649749756, 0.620290219783783, 0.25545498728752136, -0.14568370580673218, -0.11649288982152939, 1.527043342590332, 0.42515984177589417, 0.3980945646762848, -1.280440330505371, 0.3047221302986145, 3.1954801082611084, -0.7027600407600403, -0.9118552803993225, -1.3323078155517578, -0.6187644600868225, -0.9599143862724304, 0.3353898227214813, -0.24952556192874908, 1.8267399072647095, -2.8866543769836426, -0.3401504158973694, 0.22299528121948242, 0.9599807262420654, 0.10324353724718094, 0.9536423683166504, -1.1549723148345947, -0.9809128642082214, 0.3384603261947632, 0.30172139406204224, 0.4458748400211334, 0.5380159020423889, 0.5514786243438721], [-0.2596483826637268, -2.168508529663086, -0.8659776449203491, -0.9714901447296143, -0.22657842934131622, 0.7957820296287537, -0.5854039192199707, 0.5534989237785339, -1.4217910766601562, -0.60481858253479, 0.47076818346977234, -0.34553664922714233, -0.6374251842498779, -0.8542576432228088, 1.2844161987304688, -0.759646475315094, 0.10946446657180786, 1.3617604970932007, 1.7878910303115845, -1.5534279346466064, 1.4598000049591064, 0.24027830362319946, 1.0839356184005737, -0.32458755373954773, 0.18511004745960236, 0.7363510131835938, 0.7570935487747192, 0.5673931837081909, -0.1427529752254486, 0.5438389778137207, -0.20322628319263458, 0.9470930099487305], [1.7809875011444092, -0.9604276418685913, -0.581874668598175, -0.34903645515441895, -1.5027451515197754, -0.5960680842399597, 1.017721176147461, 0.1718304306268692, -0.5450797080993652, -1.2155942916870117, -0.26489368081092834, 0.9347279667854309, 0.1647726446390152, 0.6271917819976807, -1.9171783924102783, -1.0038127899169922, -0.7251554131507874, -0.506612241268158, 0.40410539507865906, -0.37189310789108276, 0.4302583932876587, -0.6375757455825806, -0.5844223499298096, 0.11727209389209747, 1.0795835256576538, -0.48672595620155334, -1.0116512775421143, 0.5606862306594849, -1.0881820917129517, 1.582611322402954, -0.9854874014854431, -1.1870265007019043], [-0.3499622941017151, 0.6189232468605042, 0.25593483448028564, -0.14426825940608978, -0.12626168131828308, 1.536340594291687, 0.416289746761322, 0.4032192528247833, -1.282226800918579, 0.310138076543808, 3.1984713077545166, -0.7057269215583801, -0.9244900941848755, -1.3489230871200562, -0.6248849034309387, -0.9561532139778137, 0.3327820301055908, -0.24315457046031952, 1.8381859064102173, -2.8968746662139893, -0.3515344262123108, 0.22365432977676392, 0.962130606174469, 0.10778437554836273, 0.9595025777816772, -1.1598396301269531, -0.9841172695159912, 0.3375384211540222, 0.29913800954818726, 0.4371204674243927, 0.5386056900024414, 0.550179660320282], [-1.4808813333511353, 0.3602459132671356, 0.18243156373500824, -1.0089032649993896, -0.22219888865947723, -0.19006426632404327, -0.4705544412136078, 0.46294429898262024, 1.3138707876205444, -0.6128800511360168, 0.5920079350471497, -0.7079018950462341, -0.5003136396408081, -0.4066654145717621, 1.1915342807769775, 1.0832406282424927, 0.0700172632932663, -1.3403851985931396, 1.1792895793914795, 0.47516724467277527, 0.47051382064819336, 0.9866610765457153, -0.3069521188735962, 0.7840608358383179, 1.371130108833313, -0.46895134449005127, -1.192257046699524, -0.3046550750732422, 0.17437870800495148, 0.16256238520145416, -0.622449517250061, -1.2973036766052246], [-0.5739380717277527, 0.9033761024475098, 0.34017446637153625, -0.13010185956954956, 0.4560929536819458, -0.8318070769309998, -0.276536762714386, 0.697014331817627, 0.5180476307868958, 0.506677508354187, 0.707349956035614, -0.07872127741575241, -0.17340068519115448, -0.8478770852088928, 1.3955001831054688, 1.3491803407669067, 0.22586746513843536, -0.10660300403833389, 0.18861250579357147, -0.0235441867262125, -0.26475268602371216, -0.23929595947265625, 0.22501108050346375, 0.33008405566215515, 0.9028642177581787, -0.4605869948863983, 0.3592795133590698, -0.10453272610902786, -0.3953816890716553, 0.1794397532939911, -0.05373012274503708, 0.283144474029541], [0.20006872713565826, 0.9696934819221497, -0.16580922901630402, 1.2974343299865723, -0.7338104248046875, -0.7123199701309204, 0.8301483392715454, -0.367972195148468, 1.3261456489562988, 1.1909674406051636, -1.8771390914916992, -1.2457048892974854, 0.8481452465057373, 0.7003065347671509, 0.05354516580700874, 0.04248463362455368, -1.0727430582046509, 1.7216496467590332, -1.37201988697052, -1.3421703577041626, -1.0098278522491455, 0.8816344141960144, 1.4204909801483154, -1.0343306064605713, -2.1320502758026123, -0.41909122467041016, -0.6236217021942139, 1.4870493412017822, 0.3673699200153351, -0.3671988546848297, 1.078015685081482, -0.32379060983657837], [-0.04571860283613205, 0.41249147057533264, -0.45989111065864563, 1.2040119171142578, -0.9601466655731201, 0.5049730539321899, -0.9069197177886963, -0.1932159811258316, 0.8219133019447327, 0.3523150086402893, 0.039005719125270844, 0.6200277805328369, 1.4385545253753662, 0.9197387099266052, -0.8573721647262573, -0.08830279111862183, -0.07029975950717926, -1.1973592042922974, -0.11525964736938477, -0.731147825717926, 0.7877081036567688, -1.456087589263916, 0.20198987424373627, 0.04883818328380585, -0.02585301734507084, -0.323869913816452, 0.6475861668586731, 0.9466323256492615, 1.0952528715133667, 0.5169216394424438, -0.8829107284545898, 0.05820639058947563], [2.1666667461395264, -0.4505749046802521, -0.48995354771614075, -0.2384330779314041, -0.5680537819862366, -0.7188742160797119, 0.28940635919570923, 0.015998518094420433, -0.4287334382534027, -0.8881052136421204, -0.07002469897270203, 1.5502122640609741, 0.029418818652629852, 0.32155975699424744, -1.1329759359359741, -0.47342443466186523, -1.2064546346664429, -0.41781195998191833, 0.5700263381004333, 0.16873081028461456, 0.1215788945555687, -0.2558687627315521, -0.5990450382232666, 0.5834785103797913, 1.3645472526550293, -0.18189957737922668, -0.6533485651016235, 0.13669151067733765, -0.6114217042922974, 0.8706583380699158, -1.0901390314102173, -0.4913942515850067], [0.003115968080237508, 0.21432849764823914, 0.5807129740715027, 0.23515768349170685, -0.3199683129787445, 1.400978446006775, -0.23349300026893616, 0.49738559126853943, -1.2167290449142456, 0.18963803350925446, 2.4786274433135986, -0.7132484316825867, -0.2979999780654907, -1.0138555765151978, -0.15172195434570312, -1.3131262063980103, 0.23878635466098785, -0.7182832360267639, 1.06264328956604, -1.550689697265625, -0.4478392004966736, 0.38351309299468994, 0.2061735838651657, 0.07564845681190491, 0.2867269515991211, -0.566952645778656, -0.5418227910995483, 0.34221336245536804, 0.10325486212968826, 0.01985815353691578, 0.09391363710165024, 0.8390274047851562], [0.22287391126155853, 0.9939887523651123, -0.16056829690933228, 1.3384590148925781, -0.7610665559768677, -0.767734706401825, 0.858047604560852, -0.3917172849178314, 1.3945285081863403, 1.2241696119308472, -2.012955904006958, -1.2832869291305542, 0.8895354270935059, 0.7687618136405945, 0.06935678422451019, 0.06382432579994202, -1.1185859441757202, 1.7757378816604614, -1.4407553672790527, -1.3211846351623535, -1.0427759885787964, 0.9003372192382812, 1.4486092329025269, -1.0595674514770508, -2.224250555038452, -0.4105898141860962, -0.6495733261108398, 1.5490965843200684, 0.3886636793613434, -0.397826611995697, 1.1003925800323486, -0.35036733746528625], [-0.32967132329940796, 1.1500837802886963, 0.1651778221130371, -0.49953705072402954, -0.44663938879966736, -0.10453347116708755, -0.5585697889328003, 1.6515181064605713, 0.7787386775016785, 0.18496088683605194, 0.055717021226882935, 0.060353755950927734, 1.0491046905517578, -2.0879721641540527, 2.332282781600952, 1.3806508779525757, 0.863361656665802, -0.651914119720459, 0.3427382707595825, 0.5120908617973328, -0.45528602600097656, 0.02004118636250496, 1.7897262573242188, 0.23262864351272583, 2.0473036766052246, -0.6756249070167542, -0.20518936216831207, -0.059305980801582336, -0.001104913535527885, 0.3965507745742798, 0.5584468245506287, 0.36036211252212524], [-0.695481538772583, -1.211561918258667, -0.5249314904212952, -0.8777713775634766, -0.18210577964782715, 0.3889594078063965, -0.4946766495704651, 0.46305903792381287, -0.4170384705066681, -0.6062721014022827, 0.4931640028953552, -0.47378426790237427, -0.5675232410430908, -0.5464076995849609, 1.1675944328308105, -0.00032495238701812923, 0.04262997582554817, 0.38323280215263367, 1.569908857345581, -0.7809883952140808, 1.067118763923645, 0.49524280428886414, 0.44231754541397095, 0.049520235508680344, 0.5409478545188904, 0.34693828225135803, -0.016898030415177345, 0.23899118602275848, -0.08494854718446732, 0.34666070342063904, -0.3710981011390686, 0.09695571660995483], [1.083166241645813, 0.658508837223053, -0.8876460194587708, 1.465516209602356, 0.17001251876354218, -1.5127078294754028, -0.9920769333839417, 1.661510944366455, -0.5851333141326904, -0.8724811673164368, -1.0725386142730713, 0.8537545800209045, 0.2933969497680664, 1.310382604598999, -0.32970377802848816, 1.6282236576080322, -0.2664220631122589, -0.1232999861240387, 0.43767061829566956, 0.9286577105522156, 0.30818888545036316, -1.7928847074508667, -0.18505463004112244, -0.6522836089134216, -0.048609986901283264, 1.2081471681594849, -0.5577043294906616, -0.2104402482509613, -1.3125344514846802, -0.7872259020805359, -1.7239041328430176, -0.6251952648162842], [1.7812973260879517, -1.133672833442688, -0.6041934490203857, -0.19328026473522186, -1.6249420642852783, -0.6696553230285645, 1.0148926973342896, 0.10758589208126068, -0.6052775382995605, -1.2927403450012207, -0.09754793345928192, 0.9299550652503967, 0.24398036301136017, 0.5992490649223328, -2.1261913776397705, -1.0647823810577393, -0.7595569491386414, -0.42355790734291077, 0.5750375986099243, -0.3541732430458069, 0.604124128818512, -0.6338334083557129, -0.6380739212036133, 0.3526516854763031, 1.315231442451477, -0.3944924473762512, -1.1894607543945312, 0.6147950887680054, -1.1609692573547363, 1.6575077772140503, -1.0396074056625366, -1.262651801109314], [-0.3571646511554718, -0.7521383166313171, 0.38709691166877747, -0.6805257201194763, -0.25963616371154785, -0.2725439667701721, -0.38850894570350647, 0.8839026689529419, 0.026470160111784935, -0.45382657647132874, -0.03388375788927078, 0.44823265075683594, 0.6639184951782227, -0.34489911794662476, 0.7671923637390137, 0.23755772411823273, 0.33799222111701965, -0.9269793033599854, 0.6397284865379333, 0.12328996509313583, -0.6597683429718018, 0.7269816398620605, 1.1050068140029907, 0.24795383214950562, 0.9516600966453552, -0.5741671919822693, -0.7190508246421814, -0.05500289052724838, 0.6040846705436707, 0.2564276456832886, 0.661841869354248, 0.7332489490509033], [-0.6771245002746582, 0.5184763073921204, 0.24854446947574615, -0.6431660652160645, -0.16929130256175995, -0.3694843649864197, 0.19382035732269287, 0.8863140940666199, 0.6801773905754089, -0.8745589852333069, -0.4017511010169983, -0.2982318103313446, -0.5887407064437866, -0.14348141849040985, 0.6542811989784241, 1.028577208518982, 0.050595737993717194, -1.1760081052780151, 0.43557608127593994, 0.1724437028169632, 0.2826806902885437, 0.662851095199585, 0.09525033086538315, 0.45946115255355835, 0.9258045554161072, -0.34711888432502747, -0.48688116669654846, 0.16511785984039307, 0.03941471502184868, 0.02214176580309868, -0.43804481625556946, -0.9493560791015625], [-0.8079174757003784, 0.1740802526473999, -1.0462654829025269, 0.5874339938163757, 0.5116462111473083, 0.18864619731903076, -0.5211663842201233, 0.22888416051864624, 0.11982621997594833, 0.14472709596157074, 1.4168059825897217, 0.7901197075843811, 0.40283700823783875, -1.6199100017547607, -0.5384005904197693, -0.10483678430318832, 0.3733023405075073, 0.27563589811325073, -1.0733343362808228, -0.8023959994316101, 0.20968884229660034, 0.4832197427749634, -0.12173562496900558, -0.7322713136672974, 0.43490901589393616, -0.27826744318008423, 1.72262704372406, -1.7303900718688965, -1.1749608516693115, 0.7563093900680542, 0.3491019606590271, 0.3225804567337036], [-0.1197853535413742, 0.3178207278251648, 0.044002603739500046, -0.8331272006034851, -0.6373385787010193, -0.2888108789920807, 0.2733410894870758, 0.676740825176239, 0.5212242603302002, -0.454299658536911, -0.8333855867385864, 0.0212095994502306, -0.30549386143684387, -0.8185885548591614, 0.44735124707221985, 0.2509657144546509, -0.22734856605529785, -0.1745181828737259, 0.13257017731666565, 0.5581323504447937, -0.19346293807029724, -0.7019646763801575, 1.6446621417999268, -0.4890882074832916, 0.7333185076713562, -0.1570199579000473, 0.4774349331855774, -1.1474519968032837, -0.9632048606872559, -1.4362860918045044, -0.39799851179122925, 0.6228185892105103], [1.1365580558776855, -0.44945284724235535, 0.667599081993103, 0.3462275564670563, -0.2753814458847046, -0.6554057598114014, 2.6347241401672363, -0.46157190203666687, 0.8561027646064758, 0.6110223531723022, 0.32866787910461426, -0.8740659356117249, 1.1245421171188354, 0.8194032907485962, -0.2329210788011551, -2.1195268630981445, 2.7410080432891846, -0.5987353920936584, -0.7462595105171204, -1.1339179277420044, -0.07618744671344757, 0.9177019000053406, -0.4560351073741913, -1.7733845710754395, -1.3728526830673218, -1.1994208097457886, 1.5871046781539917, -0.27402904629707336, 1.5726398229599, -1.582745909690857, 0.2897931635379791, 0.6983379125595093], [0.13308271765708923, -0.43454495072364807, -0.8677800893783569, 0.011774707585573196, -0.1966332644224167, 0.28623491525650024, 0.12924394011497498, 0.10196562111377716, -0.8520698547363281, -0.5563857555389404, 0.257040798664093, 0.5932378768920898, 0.3221309781074524, 0.16885985434055328, -1.2332364320755005, -0.7406550049781799, 0.009240005165338516, -0.6556650400161743, 0.55047607421875, 0.31033769249916077, 0.08351201564073563, -0.10697240382432938, 0.06499453634023666, 0.5100876092910767, 0.8435953259468079, -0.39034298062324524, 0.17957574129104614, 0.3237858712673187, -0.8319793939590454, 0.7629007697105408, -0.848108172416687, -0.9097246527671814], [-0.2602614462375641, -2.195919990539551, -0.8780189752578735, -0.9798170924186707, -0.22628112137317657, 0.7884860634803772, -0.5969582200050354, 0.5533782839775085, -1.4190822839736938, -0.6123610734939575, 0.43600374460220337, -0.33699092268943787, -0.634283721446991, -0.8420551419258118, 1.3052278757095337, -0.7529338598251343, 0.1079651415348053, 1.3770534992218018, 1.7799060344696045, -1.5268062353134155, 1.4821462631225586, 0.23837213218212128, 1.0849878787994385, -0.33281055092811584, 0.17428569495677948, 0.760276198387146, 0.7709271311759949, 0.5705336928367615, -0.14680184423923492, 0.5440272688865662, -0.21333980560302734, 0.9498494267463684], [-1.2804954051971436, -0.6884357333183289, -0.4876617193222046, 0.8850438594818115, -0.301339715719223, -0.783804178237915, 0.09944304078817368, -0.5821022987365723, -0.6195244789123535, 0.29836875200271606, 1.1660714149475098, 1.2300361394882202, -0.31336888670921326, 2.0640292167663574, -1.5454484224319458, 0.9997076988220215, -1.105796217918396, 0.6736104488372803, 0.14289133250713348, -0.880506694316864, 0.9327426552772522, -0.38974496722221375, 2.3487586975097656, -0.88576740026474, 2.3035526275634766, -1.2040507793426514, -0.19908902049064636, -0.024221401661634445, 0.5321832895278931, 0.5992379784584045, -2.7861411571502686, -0.8975560665130615], [0.768016517162323, -0.6065651178359985, 1.3733928203582764, 1.0851926803588867, -0.7476421594619751, 1.1194548606872559, -1.6182239055633545, 0.698738694190979, -1.1299099922180176, -0.019019637256860733, 1.0610347986221313, -0.7996501326560974, 0.9874699711799622, -0.349800169467926, 0.8175998330116272, -2.174436330795288, 0.05790846422314644, -1.7739719152450562, -0.557067334651947, 1.2029613256454468, -0.7368646264076233, 0.6686950325965881, -1.3929870128631592, 0.06625932455062866, -1.170323371887207, 0.6348510980606079, 0.34374305605888367, 0.37459278106689453, -0.2569911479949951, -0.8520200252532959, -0.7532082796096802, 1.493589997291565], [0.7527101039886475, -0.552023708820343, 1.2983421087265015, 1.0665411949157715, -0.7419179677963257, 1.0692853927612305, -1.5397642850875854, 0.6565704941749573, -1.0689101219177246, -0.00014143649605102837, 0.9575488567352295, -0.8222649693489075, 0.9752039313316345, -0.3146323263645172, 0.7830432057380676, -2.078697919845581, -0.007035915274173021, -1.6518455743789673, -0.5833696722984314, 1.122188925743103, -0.7552846670150757, 0.6749067902565002, -1.299350380897522, 0.04977291077375412, -1.1700317859649658, 0.5907105803489685, 0.29978910088539124, 0.3730508089065552, -0.24250879883766174, -0.8440159559249878, -0.7097782492637634, 1.4231239557266235], [0.7318997979164124, -0.2529752850532532, -0.5430791974067688, -1.095735788345337, -0.268063485622406, 0.5220205187797546, 0.021684173494577408, 0.7460072040557861, -0.49337485432624817, -0.31002169847488403, -0.6793394684791565, 0.35421329736709595, -0.572382390499115, 0.049540769308805466, 0.5477702021598816, -0.31776344776153564, -0.13285255432128906, -0.19493721425533295, 0.17645984888076782, -1.0553356409072876, -0.17741084098815918, -0.42350998520851135, 0.4948122501373291, -1.2532206773757935, -0.5755332112312317, -0.445240318775177, 0.711363673210144, 0.25690096616744995, -0.1333082616329193, 0.4361118674278259, -0.3796864449977875, 0.22162985801696777], [-0.7468245029449463, -0.14173521101474762, -0.19145138561725616, 0.8260678052902222, -0.12352166324853897, -0.7026001214981079, 0.6393795013427734, 0.3638940453529358, -0.6093934774398804, -0.5191144347190857, 0.08361948281526566, 0.8953760862350464, -0.2137683779001236, 1.2268967628479004, -1.052308440208435, 1.2080897092819214, -0.569292426109314, 0.16175317764282227, -0.15979677438735962, -0.3562822639942169, 0.8951689600944519, 0.2381770759820938, 1.7928004264831543, -0.3027768135070801, 1.8793050050735474, -0.46568530797958374, 0.10055554658174515, 0.4297879934310913, 0.10084222257137299, 0.08263468742370605, -1.8661037683486938, -0.6178725957870483], [-0.7306621670722961, -0.7761626243591309, -0.5113499164581299, 0.6890299916267395, -0.5441599488258362, -0.7756362557411194, 0.274260550737381, -0.46376317739486694, -0.6199960708618164, 0.004952038172632456, 0.9357684850692749, 1.1882978677749634, -0.2141457200050354, 1.8105489015579224, -1.6675512790679932, 0.6265150904655457, -1.0539058446884155, 0.48485392332077026, 0.21758180856704712, -0.7843571305274963, 0.8807931542396545, -0.4321826994419098, 1.817981243133545, -0.6671727299690247, 2.1425864696502686, -1.0620341300964355, -0.3844265639781952, 0.09036867320537567, 0.2201407253742218, 0.7993810176849365, -2.4893157482147217, -0.977454662322998], [-0.33234289288520813, -0.14055569469928741, -1.319214940071106, 1.539940595626831, 0.4577367305755615, -0.015655692666769028, 0.46947261691093445, -0.3149826228618622, -0.6323832869529724, -0.29611527919769287, 0.8552980422973633, 0.1277899146080017, 0.7809431552886963, 1.4631996154785156, -0.5806761980056763, 1.5402649641036987, -0.7711472511291504, 0.5774557590484619, 1.1809051036834717, -0.18116562068462372, 0.2762468755245209, 0.5309889316558838, -1.0482505559921265, -0.7666963934898376, 0.18848304450511932, 0.811639666557312, -0.6223753094673157, -0.052610844373703, -1.2798898220062256, -0.7428512573242188, -1.112711787223816, 0.030781712383031845], [-0.9363169074058533, 1.560463547706604, 0.25202423334121704, 0.0650007501244545, 0.3902292847633362, 1.7812633514404297, 0.11844955384731293, -0.24413686990737915, 0.28825798630714417, 1.158339023590088, 1.1609044075012207, 1.0433013439178467, -0.8946786522865295, 1.2172058820724487, -0.5078086256980896, 0.7630130052566528, 0.6632058620452881, -0.965839684009552, -0.9361461997032166, 1.6542489528656006, 0.7288378477096558, -0.5435332655906677, 0.8606621026992798, -1.224808931350708, 0.031028475612401962, 0.17046919465065002, -2.7922205924987793, 0.5845123529434204, 0.5556184649467468, 0.0768136978149414, -0.2721361219882965, -0.010654224082827568], [-0.25920870900154114, -2.202467203140259, -0.8797618746757507, -0.9809536933898926, -0.22931119799613953, 0.7864322066307068, -0.5996322631835938, 0.5522523522377014, -1.4210684299468994, -0.6153037548065186, 0.43294891715049744, -0.34035879373550415, -0.6315181255340576, -0.8429461717605591, 1.3091330528259277, -0.7554609179496765, 0.10649733245372772, 1.3806778192520142, 1.7848327159881592, -1.5338099002838135, 1.4816941022872925, 0.2403631955385208, 1.0841810703277588, -0.32770979404449463, 0.1735040247440338, 0.7568894028663635, 0.7763887047767639, 0.572586715221405, -0.15151000022888184, 0.5424039363861084, -0.21237410604953766, 0.9503298997879028], [1.8633137941360474, 0.6144967079162598, -0.48256373405456543, -0.43446770310401917, 1.0976250171661377, 1.122460126876831, -0.7934717535972595, -0.5413935780525208, 1.1093071699142456, -1.163197636604309, 0.7974622845649719, 0.03936544805765152, -0.4215032756328583, 0.34101927280426025, -1.1002904176712036, -0.645768404006958, -0.8554724454879761, 1.4636218547821045, 0.6276389360427856, -2.365000009536743, 0.03640514612197876, -0.0030138918664306402, 0.29730552434921265, -0.20295965671539307, -0.24450664222240448, 0.15033307671546936, -1.0624066591262817, -0.2980145812034607, -0.17429864406585693, 0.6824262142181396, -0.7587978839874268, 0.6922777891159058], [-0.3401966691017151, 1.1447845697402954, 0.504732608795166, 0.8488774299621582, -0.5926541090011597, -0.407672643661499, -0.025878148153424263, 0.48390689492225647, -0.20461758971214294, -0.9440963268280029, -0.37658098340034485, -0.6744537949562073, 0.7567856907844543, 0.4478369653224945, -0.32003238797187805, 0.36626332998275757, 0.03649802878499031, 1.067770004272461, -0.3773692548274994, -0.5857011079788208, -0.3801476061344147, -0.05333803594112396, 0.21650053560733795, 0.9481345415115356, -0.7691605091094971, 0.12808364629745483, -0.40334951877593994, -1.7949202060699463, 0.3398680090904236, 0.33666765689849854, -0.21543703973293304, -0.7446690797805786], [0.009851488284766674, 1.3103599548339844, 0.10146232694387436, 0.4375392496585846, 1.2407644987106323, -0.9616708755493164, 0.07978574186563492, 0.4491078555583954, 0.6795204877853394, 0.4696231484413147, -0.00926427822560072, 0.5134938955307007, 0.1384253352880478, -1.3642140626907349, 0.4374101758003235, 0.7926722168922424, -0.07515037059783936, 0.038929283618927, 0.75391685962677, -0.5055626630783081, 1.1059353351593018, 0.5366127490997314, -0.1298571228981018, -0.17479181289672852, -0.5435911417007446, -0.2179671674966812, 0.44862082600593567, 0.5435318350791931, -0.2921392023563385, -1.2501686811447144, -0.0999632179737091, 0.3418751060962677], [0.3662492036819458, 0.9470012784004211, -0.14756736159324646, 1.2079392671585083, -0.6166424751281738, -0.7477864623069763, 0.7481061816215515, -0.3277108669281006, 1.2954562902450562, 1.094282865524292, -1.8423793315887451, -0.9871604442596436, 0.8015533089637756, 0.663467288017273, 0.0952516719698906, 0.07978323847055435, -1.1315561532974243, 1.5908033847808838, -1.277598261833191, -1.142290711402893, -0.9728006720542908, 0.8304470181465149, 1.3002768754959106, -0.9297949075698853, -1.9401130676269531, -0.3755757808685303, -0.5930495858192444, 1.3730727434158325, 0.39223313331604004, -0.39466363191604614, 0.9206720590591431, -0.25141239166259766], [-0.09379984438419342, 1.4084407091140747, -0.021971749141812325, 1.6357558965682983, 0.4440290629863739, 0.21847383677959442, 0.29372960329055786, 0.6457208395004272, 0.24713221192359924, -0.6507847309112549, 0.23707571625709534, 0.1609353870153427, 0.12235874682664871, -0.9052209854125977, -0.5718898177146912, -0.28083449602127075, -0.6169623732566833, -0.08317550271749496, 0.33364051580429077, -0.08687148243188858, -1.5603055953979492, 0.9485222101211548, 0.27341321110725403, -0.11251679062843323, -1.292881727218628, 0.024332713335752487, -0.7526654005050659, -2.670422077178955, -0.4339030086994171, 1.8086411952972412, -1.7097574472427368, -0.5551732778549194], [-0.3545190095901489, 0.5585514307022095, 0.17486727237701416, -0.06890885531902313, -0.05914613604545593, 1.4953233003616333, 0.3964942693710327, 0.3694252669811249, -1.2409526109695435, 0.17783816158771515, 2.924726724624634, -0.6450923085212708, -0.7714059948921204, -1.1929014921188354, -0.5981528162956238, -0.9524217247962952, 0.335618793964386, -0.30050715804100037, 1.6527708768844604, -2.5800983905792236, -0.31942522525787354, 0.08756166696548462, 0.8929787278175354, 0.10957901179790497, 0.8960475325584412, -1.1343814134597778, -0.8411595821380615, 0.2534172236919403, 0.26377803087234497, 0.350686252117157, 0.4186372756958008, 0.43370354175567627], [-0.15475493669509888, -0.23736977577209473, -1.1483618021011353, 2.0046095848083496, 0.3873882591724396, -0.4217319190502167, 0.1974715292453766, -0.4188764691352844, -0.6321473121643066, -0.20196053385734558, 0.46726134419441223, -0.4231714904308319, 0.8183843493461609, 1.5419763326644897, -0.25877928733825684, 1.5235607624053955, -0.6947042942047119, 0.6911174058914185, 1.6835849285125732, -0.5322560667991638, -0.0009925527265295386, -0.08252458274364471, -1.6048438549041748, -0.13799646496772766, -0.6066341400146484, 1.03685462474823, -0.9881200790405273, 0.32644301652908325, -1.4171959161758423, -0.9788877367973328, -0.10950395464897156, 0.24626490473747253], [0.46802833676338196, 0.17046253383159637, -0.281314879655838, -0.7358303666114807, -0.02820923924446106, 0.2159353643655777, 0.24777287244796753, 0.2299627661705017, 0.05179493874311447, -0.7220446467399597, -0.47233083844184875, -0.1072799414396286, -0.28681352734565735, 0.14372961223125458, -0.5570366978645325, -0.3594089448451996, -0.7901862859725952, 0.4725361764431, 0.07550669461488724, -0.06234501302242279, 0.012552459724247456, -0.3435956835746765, 0.5960245728492737, -0.2421289086341858, 0.2916008234024048, -0.02252301201224327, -0.3992670774459839, -1.104828953742981, -0.47944745421409607, -0.87484210729599, -0.8634011149406433, 0.4940796494483948], [0.3357153832912445, -1.08990478515625, -1.3513790369033813, 1.693294882774353, -0.4432087242603302, 0.6546680331230164, -0.07173393666744232, 1.068111777305603, 0.23298420011997223, 1.2561849355697632, -0.2643836438655853, -0.6681014895439148, -0.5722362995147705, 0.178314208984375, -1.0475590229034424, -1.1616960763931274, -0.7024635672569275, 0.019227443262934685, 0.016734294593334198, -0.49144893884658813, -0.7002690434455872, 0.5319235324859619, -1.1849037408828735, -0.8870525360107422, -1.4236642122268677, 0.8410196304321289, 0.9378312826156616, -1.3331466913223267, -0.23705250024795532, 0.01775439828634262, 0.21557126939296722, -0.005346406251192093], [-0.3494075834751129, 0.6219689249992371, 0.2574867606163025, -0.14285123348236084, -0.12574993073940277, 1.5372209548950195, 0.4172127842903137, 0.40341562032699585, -1.2824574708938599, 0.3110947012901306, 3.2012922763824463, -0.7060359716415405, -0.9246960878372192, -1.3496062755584717, -0.6271030306816101, -0.9560202956199646, 0.33314409852027893, -0.2444283366203308, 1.838231086730957, -2.898251533508301, -0.3538980185985565, 0.22363123297691345, 0.9616101980209351, 0.10817378014326096, 0.9604535102844238, -1.1617801189422607, -0.9860575795173645, 0.3371255397796631, 0.29982230067253113, 0.4369094967842102, 0.539956271648407, 0.549716055393219], [-0.156357079744339, -1.7884888648986816, -0.6891831159591675, -0.8683146238327026, -0.20506080985069275, 0.8049511909484863, -0.449039489030838, 0.53872150182724, -1.3333312273025513, -0.4926376938819885, 0.6227925419807434, -0.37199848890304565, -0.572607696056366, -0.8622326254844666, 1.0381014347076416, -0.7322152256965637, 0.0927482396364212, 1.1226916313171387, 1.6544398069381714, -1.4655717611312866, 1.2432299852371216, 0.18273654580116272, 1.005615234375, -0.2909304201602936, 0.23379573225975037, 0.5758917331695557, 0.6102961301803589, 0.44376665353775024, -0.1341431587934494, 0.39556702971458435, -0.151278555393219, 0.8621835708618164], [0.8440877199172974, 0.9324571490287781, -0.01640121452510357, -0.9387060403823853, -0.5410153269767761, 0.1810140311717987, 0.366182804107666, 0.6671741604804993, -0.015072017908096313, -0.3152611255645752, -1.1969285011291504, 0.3178633451461792, -0.154440775513649, 0.7305486798286438, 0.01982850208878517, 0.03496547043323517, -0.11626643687486649, -0.4854009747505188, -0.9048554301261902, -0.8699689507484436, -0.8597948551177979, -0.692331075668335, 0.1803237795829773, -1.0421355962753296, -0.8953521847724915, -1.005326747894287, 0.5293481945991516, -0.27193042635917664, 0.20484019815921783, 0.2386331409215927, -0.15470899641513824, -0.36001235246658325], [-0.4581996202468872, -0.10590001940727234, 0.4350893497467041, -0.14390450716018677, 1.4020296335220337, -0.5682953596115112, 2.1254851818084717, 0.8516262173652649, -1.5399688482284546, 2.048011064529419, -0.07342201471328735, -0.03795347362756729, 0.1779596358537674, 1.2792671918869019, -1.0339094400405884, -0.605162501335144, 0.04251527041196823, -0.5054419636726379, -0.6503801345825195, -0.6568677425384521, -1.2036036252975464, 0.7888321280479431, -0.266726553440094, 1.0660735368728638, -0.9898936748504639, -0.5637565851211548, 0.5021826028823853, 1.1015764474868774, 0.1712617427110672, -0.590390145778656, -0.8298850655555725, 1.1843658685684204], [0.07633776217699051, -0.346456378698349, 0.33991822600364685, 0.2614475190639496, 0.21049945056438446, 0.010381780564785004, 0.7264034748077393, 0.6338228583335876, -0.8108346462249756, 0.662624716758728, 0.2048712968826294, -0.40460988879203796, -0.1312549114227295, 0.41173577308654785, -0.48367372155189514, -0.7239939570426941, 0.05139845237135887, -0.37366801500320435, -0.30046701431274414, 0.06179717555642128, -0.5770590305328369, 0.10950197279453278, -0.1831139326095581, 0.38475796580314636, -0.31174272298812866, -0.03855506703257561, 0.34143954515457153, -0.09736176580190659, -0.08066587150096893, -0.6803591847419739, -0.24023443460464478, 0.8799788951873779], [-0.7580989599227905, 0.036161575466394424, -0.4449015259742737, -0.24365843832492828, 1.6131457090377808, -0.5814158320426941, 1.6184812784194946, -0.3911387622356415, -0.7095689177513123, -0.7311904430389404, 1.1958813667297363, 0.017226016148924828, 1.5808327198028564, 1.7456943988800049, 0.9670300483703613, -1.1662558317184448, 0.7035964131355286, -1.116089940071106, -0.5832664370536804, 0.028183648362755775, 2.2208633422851562, -0.015801353380084038, 0.40417811274528503, -0.7995514273643494, -0.2863788604736328, 0.3281715214252472, 0.04237843677401543, 0.16701661050319672, 0.5338238477706909, 1.9673081636428833, 0.33198633790016174, 0.5665493011474609], [0.5843470096588135, 0.18882854282855988, -0.6588954925537109, 0.7796469926834106, 0.3979056477546692, -0.7145417928695679, 0.06514061987400055, -0.021044539287686348, 0.2774466574192047, 0.37953945994377136, -1.1331572532653809, 0.2587277889251709, 0.5238890051841736, 0.19544710218906403, 0.2603302597999573, 0.2774640917778015, -0.8664188981056213, 0.1367865353822708, -0.6615815758705139, 0.2740069329738617, -0.08581949770450592, 0.20190514624118805, 0.3298838436603546, 0.0089411037042737, -0.34251299500465393, 0.03197995200753212, -0.2520294785499573, -0.07022896409034729, 0.14524365961551666, 0.019149500876665115, 0.48542729020118713, -0.2139470875263214], [0.28767815232276917, 0.10218392312526703, 0.08690523356199265, -1.3616974353790283, -0.8393584489822388, -0.2377871572971344, 0.7334166765213013, 0.40763071179389954, 0.6692557334899902, -0.9647595286369324, -1.7876935005187988, -0.27522096037864685, -0.9927508234977722, -0.7489291429519653, -0.1665131151676178, -0.5782492160797119, -0.6140872240066528, -0.1586516946554184, 0.00329887168481946, 0.8881680965423584, -0.2856837809085846, -1.212540626525879, 1.3313347101211548, -0.7731072902679443, -0.4191651940345764, 0.3843318223953247, 1.0327131748199463, -1.949077844619751, -1.787465214729309, -2.884854555130005, -0.3637945353984833, 1.1221646070480347], [-0.49963414669036865, 0.9025689959526062, 0.7340838313102722, 0.3231969475746155, -1.079958438873291, -0.7130300998687744, -0.10384748131036758, 0.4252800941467285, -0.4696067273616791, -1.0564061403274536, -0.6329854726791382, -1.012412190437317, 1.0796891450881958, 1.1357715129852295, -0.18974114954471588, 0.6467149257659912, 0.4153173267841339, 1.5940513610839844, -0.7444535493850708, -0.7776103615760803, 0.39919111132621765, -0.5625619888305664, 0.13767710328102112, 1.4362270832061768, -0.35466939210891724, 0.1684163361787796, -0.23757708072662354, -1.2352283000946045, 0.7611555457115173, -0.4425645172595978, 0.5791586637496948, -0.7680989503860474], [-0.6902400255203247, 0.8679759502410889, 0.3435424864292145, -0.16583463549613953, 0.936769425868988, -0.9558068513870239, -0.4247088134288788, 0.19325067102909088, 0.758945107460022, 0.4071582555770874, 0.7818446755409241, -0.03303958475589752, -0.8467777371406555, -0.426677405834198, 1.0001424551010132, 1.379529595375061, -0.3069382607936859, -0.2203984409570694, 0.5641949772834778, -0.01858259178698063, 0.04592287540435791, 0.24034781754016876, -0.6474362015724182, 0.5474778413772583, 0.4377724826335907, -0.3070388734340668, 0.24196091294288635, -0.22932706773281097, -0.24596694111824036, -0.3911719024181366, -0.5395787954330444, -0.07230417430400848], [-0.34984999895095825, 0.6222137808799744, 0.2573918402194977, -0.14323599636554718, -0.12594446539878845, 1.537406325340271, 0.41738831996917725, 0.4030205309391022, -1.28243887424469, 0.31147700548171997, 3.2018847465515137, -0.7062505483627319, -0.9246941804885864, -1.3496922254562378, -0.6271705031394958, -0.9565991759300232, 0.3330458402633667, -0.2448137402534485, 1.8382673263549805, -2.898714303970337, -0.3536233901977539, 0.22338633239269257, 0.9622159600257874, 0.10824090242385864, 0.9602377414703369, -1.1620033979415894, -0.9859739542007446, 0.33721691370010376, 0.29977208375930786, 0.4369727075099945, 0.5396130084991455, 0.5499943494796753], [2.2089908123016357, -0.38876259326934814, 1.2721856832504272, -0.6873724460601807, 0.867755115032196, -0.19372771680355072, -0.8874233365058899, -0.11057961732149124, -1.5491015911102295, 0.9445125460624695, 0.21995072066783905, -0.7128114104270935, 1.8054059743881226, -0.7947822213172913, -0.26829102635383606, -0.7421327233314514, -0.8316560387611389, 0.384305864572525, 0.42704012989997864, 0.2953087389469147, 1.535279393196106, -2.1310436725616455, 0.5414503812789917, 0.8587614297866821, 0.1479596197605133, 0.6495980620384216, 0.058450229465961456, -0.9802761077880859, 0.3488084375858307, 0.8540931940078735, -0.5494288802146912, -0.42885246872901917], [-0.5077800154685974, 0.8018323183059692, -1.4777700901031494, -0.5023053884506226, 0.2832610607147217, 2.0470619201660156, 0.5975905060768127, 0.08181649446487427, 0.3915386497974396, -0.6300005912780762, 1.1935559511184692, 1.050766110420227, 1.2078962326049805, 0.35540473461151123, -0.3088070750236511, 1.014220952987671, -0.18810097873210907, -0.8773412108421326, -0.20711882412433624, 1.143385648727417, 0.9004669785499573, 2.464963436126709, -0.5796340703964233, -1.4999070167541504, 1.2138539552688599, -0.026237547397613525, 0.11078450828790665, -0.43782320618629456, -1.2027071714401245, -0.5462759137153625, -2.5035979747772217, -0.4061678647994995], [-1.6724576950073242, 1.3351484537124634, 1.179197907447815, 1.4618386030197144, -1.1432878971099854, -0.08929304033517838, 0.8687722086906433, -0.5955473780632019, -1.319258213043213, -0.3802056312561035, 0.38383179903030396, 0.00501289265230298, -1.9486510753631592, -0.7763243317604065, 1.2518277168273926, -0.49128416180610657, 0.2547091245651245, 1.0280334949493408, 0.1308320015668869, 0.7312261462211609, -0.9275174736976624, -0.23669640719890594, 1.4712365865707397, 0.20099233090877533, 0.6199007630348206, 0.32594379782676697, 0.3400108814239502, 0.5994179248809814, -0.7619441151618958, -2.1947007179260254, 0.4330524802207947, 0.8515527844429016], [-0.5718141794204712, -0.14881275594234467, -0.37441524863243103, 0.15573619306087494, -0.60850590467453, 1.064609169960022, -0.8662620186805725, -0.7523108720779419, 0.38958460092544556, 0.1687462031841278, -0.23778624832630157, -0.03445061296224594, 0.2196488082408905, 1.1101467609405518, 0.5058997869491577, 0.28043627738952637, 0.39864587783813477, -0.07603329420089722, 0.012826265767216682, -0.4449847638607025, -0.15420609712600708, 0.1050318256020546, 0.10959622263908386, 1.027286171913147, -0.43687787652015686, -1.2240498065948486, -0.831562340259552, 1.388683795928955, -1.4083868265151978, -0.8622279167175293, 0.27279067039489746, -0.17615251243114471], [-0.1699916422367096, 0.5779452323913574, -1.8139500617980957, 0.40110525488853455, 0.09178867191076279, 0.33140480518341064, -1.4516984224319458, -1.3473914861679077, 0.5290427803993225, 0.868707537651062, 1.0240418910980225, 1.0179349184036255, -1.610883116722107, -0.08248531073331833, -1.2372058629989624, 0.2822818160057068, -0.45464739203453064, -0.87944096326828, -0.4461010694503784, -0.2699855864048004, 0.749414324760437, 1.5347096920013428, -0.1536572128534317, 0.012916063889861107, -0.9620749354362488, 0.3795742690563202, -0.5259684920310974, 1.1382687091827393, -0.09287959337234497, 0.39303332567214966, 0.3789602220058441, -0.44533446431159973], [0.18770091235637665, 0.0857405737042427, 0.07067892700433731, -1.262279748916626, -0.8085430860519409, -0.07627470791339874, 0.5911110639572144, 0.23691730201244354, 0.5583497285842896, -0.7689621448516846, -1.6235531568527222, -0.24904294312000275, -0.8520352840423584, -0.4173820912837982, -0.04086941480636597, -0.49130672216415405, -0.5234474539756775, -0.16299596428871155, 0.015001008287072182, 0.6967549920082092, -0.3157905042171478, -0.9951108694076538, 1.1810916662216187, -0.5553574562072754, -0.4642393887042999, 0.0381803959608078, 0.6972057223320007, -1.4975625276565552, -1.8076196908950806, -2.663470983505249, -0.2709476351737976, 0.9460246562957764], [-1.371496319770813, -0.12120608985424042, -1.3096517324447632, 0.8674460053443909, 1.4437124729156494, 1.2027416229248047, -0.7576349973678589, 0.459361732006073, -0.6750727295875549, 0.15569502115249634, 0.29295724630355835, 0.2674937844276428, 0.7627928256988525, 0.7316550016403198, -0.819037675857544, -1.1369905471801758, 0.6705600619316101, -1.178313136100769, 0.6113609075546265, 1.2060267925262451, -0.6822000741958618, 0.26368796825408936, 1.3453824520111084, 1.0754668712615967, 0.33428218960762024, -0.37352606654167175, 1.5063968896865845, -0.20977704226970673, -0.7537976503372192, 0.14469319581985474, -0.6968392133712769, -0.9013530015945435], [-0.4925415515899658, 0.3289622962474823, -1.350111961364746, 0.6844344139099121, 0.2792958915233612, 0.13790659606456757, 0.6717157363891602, 0.6468454003334045, 1.496260643005371, 0.16723692417144775, 0.20534656941890717, 0.43501242995262146, 1.0323575735092163, 1.312057614326477, -1.5282858610153198, 0.8831557631492615, -0.817294180393219, 0.13218051195144653, -0.3139054477214813, 0.25344809889793396, -0.545220673084259, -0.4424493908882141, 0.2412278950214386, -2.7318356037139893, -3.1096482276916504, -0.19409212470054626, 0.2643023431301117, -0.32991138100624084, 1.09392511844635, 0.8426358103752136, 0.11208004504442215, -0.3284012973308563], [-0.7171989679336548, -0.7264739871025085, -1.7625534534454346, 1.4065388441085815, 0.7077783942222595, -0.45822763442993164, 0.08644673228263855, 0.5049123167991638, -0.0312633253633976, 0.5967937707901001, -1.4225713014602661, 0.07796113938093185, 0.8534650802612305, -0.15039731562137604, 0.42348048090934753, 0.7505109906196594, 0.31006622314453125, -0.7259812951087952, -0.7051195502281189, 1.6395061016082764, 1.342504858970642, -0.48883649706840515, 0.06945034861564636, -0.3877294063568115, 0.23027892410755157, 0.4049798548221588, -0.045603636652231216, -1.4398711919784546, -0.13706043362617493, 0.7356479167938232, 0.8154366612434387, -0.5389588475227356], [-1.4924030303955078, 1.210275650024414, 1.0214771032333374, 1.3885446786880493, -1.2413203716278076, -0.18153882026672363, 0.9522669911384583, -0.628563642501831, -1.1976591348648071, -0.5898148417472839, 0.10308211296796799, 0.19131898880004883, -1.7327194213867188, -0.611952006816864, 1.0889581441879272, -0.48256832361221313, 0.18238511681556702, 1.0277385711669922, 0.0043812096118927, 1.0175971984863281, -0.7767501473426819, -0.1758156716823578, 1.2415707111358643, 0.1568828672170639, 0.7094942331314087, 0.391893595457077, 0.3315148949623108, 0.5890210270881653, -0.9641830325126648, -2.0719473361968994, 0.17608217895030975, 0.6357083916664124], [1.138067603111267, -1.9087111949920654, 0.11387116461992264, 0.10524635761976242, -2.1851038932800293, -0.5607448220252991, 0.042071886360645294, -0.7950054407119751, -2.069035053253174, 0.2506875991821289, -0.7608853578567505, -1.1743184328079224, -0.9606684446334839, -0.2066693902015686, -0.7453210353851318, -1.0716420412063599, 0.5273883938789368, -0.1760435700416565, 1.5599901676177979, 0.7957949638366699, 0.33047768473625183, -0.4362396001815796, -1.859044075012207, -0.325422078371048, -0.034206315875053406, -0.3165806531906128, -0.8991039991378784, -0.5523306131362915, 0.8398576974868774, 0.7463811635971069, -1.0191043615341187, 0.3899698257446289], [-1.0161858797073364, -0.11900390684604645, -0.681149423122406, 0.3234729468822479, 0.2252039760351181, -1.106784701347351, 0.10761486738920212, -0.5871794819831848, -0.9251825213432312, 0.5928215980529785, 0.46205660700798035, 1.0549139976501465, 0.15138673782348633, 1.8117358684539795, -1.2177542448043823, 0.8061613440513611, -0.7396275401115417, 0.7242697477340698, -0.3690725564956665, -0.8936594724655151, 0.676763653755188, -0.11874322593212128, 1.5374189615249634, -0.5339852571487427, 1.5359587669372559, -0.8704327940940857, -0.32700926065444946, -0.48135247826576233, 0.6051849126815796, 0.1723625808954239, -1.7120375633239746, -0.4425866901874542], [1.4445629119873047, 0.8042705059051514, -1.0933353900909424, -0.003891828004270792, -0.8662285208702087, 0.5408651828765869, 0.4817577004432678, 0.21488291025161743, -1.0448352098464966, 0.39686816930770874, 0.695709764957428, 0.3386184275150299, 0.0359538197517395, 1.3057470321655273, -0.2760280668735504, 0.18613547086715698, 0.5789839625358582, 0.4071618318557739, -0.8144697546958923, 0.23580601811408997, -0.04799782484769821, -1.8063385486602783, -0.29157453775405884, -1.6737757921218872, 1.4699947834014893, -1.1785669326782227, 1.776396632194519, 0.9744988679885864, 0.016227733343839645, -0.10028816014528275, -1.0055023431777954, 0.19272352755069733], [-0.8008447885513306, -0.038612380623817444, -0.15215688943862915, -0.024159634485840797, -1.037597894668579, 1.181908130645752, -1.039365291595459, -1.4650779962539673, 0.2046450972557068, 0.4930882453918457, -0.25432902574539185, -0.09904620796442032, 0.5999926924705505, 2.036351442337036, 0.7468376755714417, 0.25786450505256653, 0.1231570765376091, -0.1640619933605194, 0.3713151812553406, -0.7002705931663513, 0.026811307296156883, 0.1465991884469986, 0.3464764654636383, 1.3366810083389282, -0.5214929580688477, -2.322610855102539, -1.7569949626922607, 1.9056929349899292, -1.9102246761322021, -1.2058454751968384, 0.3416826128959656, -0.3689447343349457], [-0.2021602839231491, -0.08018367737531662, -0.984693169593811, 1.6573405265808105, 0.5222138166427612, -0.5740194320678711, 0.34896498918533325, -0.10011008381843567, -0.3899271786212921, -0.21218933165073395, 0.39595484733581543, -0.37652695178985596, 0.7122777700424194, 0.951834499835968, -0.10692746192216873, 1.3718256950378418, -0.42446285486221313, 0.6346381902694702, 1.5192110538482666, -0.41634833812713623, 0.14143076539039612, 0.13792045414447784, -1.3870285749435425, -0.07970088720321655, -0.422550767660141, 0.9671822190284729, -0.9619637131690979, 0.10282570868730545, -1.142508864402771, -0.917143702507019, -0.08426722884178162, 0.03220703452825546], [-0.3729073107242584, 0.10032311826944351, -2.458378314971924, -0.15566027164459229, 0.6706671118736267, 0.050135448575019836, -1.577471375465393, 1.633054256439209, 0.48197823762893677, 0.2932029068470001, -0.4810522198677063, 0.6848280429840088, 0.39246615767478943, 0.44408825039863586, 1.0957473516464233, -2.152801513671875, -0.36738553643226624, -0.8132967352867126, 0.3035229444503784, 1.9112621545791626, -2.848113775253296, 1.2903534173965454, -0.3126211166381836, -0.2960788309574127, -0.8463318943977356, -0.7686828374862671, -1.0022214651107788, -0.08692571520805359, 0.5305590033531189, 0.2062356173992157, -1.3334842920303345, 0.38710421323776245], [-0.6753988862037659, -0.0376623161137104, -0.9162192940711975, -0.48930513858795166, -0.1742779165506363, -1.4789806604385376, 0.13828915357589722, -0.41797003149986267, -1.6357650756835938, -0.6837983727455139, -0.35043445229530334, 0.049241337925195694, -0.46598997712135315, 0.055077098309993744, 0.05908559635281563, 0.6724823713302612, 0.5590779781341553, 0.07936540991067886, -0.10827966034412384, -2.020681858062744, -0.19115351140499115, 0.3961716890335083, 0.5408586263656616, -0.8644266724586487, 0.7657538056373596, -1.1957919597625732, -1.188034176826477, -0.11455415934324265, 0.25665032863616943, 0.8422990441322327, 0.6366087794303894, -1.0459401607513428], [2.1240532398223877, 0.7569892406463623, -0.5017055869102478, -0.546714723110199, 1.161474347114563, 1.3670252561569214, -0.8419217467308044, -0.5122542977333069, 1.336337924003601, -1.2715775966644287, 0.7542458176612854, -0.11680865287780762, -0.46903783082962036, 0.17434757947921753, -1.0719327926635742, -0.7680230736732483, -0.7879107594490051, 1.6266494989395142, 0.7317103743553162, -2.5396058559417725, -0.016079090535640717, 0.037743568420410156, 0.18712322413921356, -0.2685911953449249, -0.4742201566696167, 0.2922217845916748, -1.1032639741897583, -0.34205806255340576, -0.26039889454841614, 0.6589608192443848, -0.6482381224632263, 0.9061813950538635], [-0.09844636917114258, -0.5011097192764282, 0.41724008321762085, 0.034895941615104675, 0.5810405015945435, -0.24463911354541779, -0.4980746805667877, 0.16972406208515167, -0.30703118443489075, 0.14308211207389832, -1.607361078262329, -0.09727360308170319, -0.27178630232810974, 0.3474023938179016, 0.4238952100276947, -1.6217381954193115, 0.7517166137695312, -1.0473343133926392, 0.36091098189353943, -0.15654735267162323, -0.5741650462150574, 0.9216272234916687, 0.47797971963882446, 0.2977740466594696, 0.04738187789916992, 0.4394367039203644, 0.26303449273109436, 0.6295334100723267, 0.0808178260922432, 0.3900238871574402, -0.9847797751426697, 0.33136290311813354]]]], "dtype": "float32"}, {"__nd__": [[[3.636291742324829, 16.438325881958008, 5.430905342102051, 8.471108436584473, 15.508615493774414, 18.274669647216797, 13.825794219970703, 8.081819534301758, 11.654465675354004, 19.643856048583984, 14.075851440429688, 12.585124969482422, 20.088354110717773, 15.142659187316895, 15.469602584838867, 10.753422737121582, 15.131817817687988, 17.508485794067383, 21.563945770263672, 14.76950454711914, 27.140878677368164, 21.053518295288086, 21.891902923583984, 20.32874870300293, 19.35087013244629, 16.726076126098633, 19.79448890686035, 24.000144958496094, 13.262304306030273, 13.687175750732422, 17.717365264892578, 18.683979034423828, 13.092185020446777, 8.659392356872559, 24.230194091796875, 27.17523193359375, 14.213168144226074, 14.471176147460938, 17.540836334228516, 18.807071685791016, 18.148069381713867, 17.00556755065918, 23.100812911987305, 24.057109832763672, 16.570955276489258, 15.401747703552246, 17.13662338256836, 18.36810302734375, 16.97831153869629, 19.57784652709961, 16.421777725219727, 19.546695709228516, 19.321210861206055, 28.11127281188965, 17.942564010620117, 30.179906845092773, 26.51325798034668, 21.87598419189453, 25.03980827331543, 21.00809097290039, 28.68159294128418, 13.849579811096191, 24.859195709228516, 18.496244430541992, 22.530014038085938, 25.470401763916016, 16.274995803833008, 19.028635025024414, 20.157455444335938, 16.097213745117188, 24.695478439331055, 20.23765754699707, 20.969989776611328, 21.98787498474121, 25.322298049926758, 20.491844177246094, 12.935892105102539], [-5.99871826171875, -8.010419845581055, 15.695980072021484, 10.683627128601074, 9.26723575592041, 8.3645658493042, 13.549537658691406, 13.4605712890625, 5.80389928817749, 15.23119831085205, 7.431952953338623, 18.027368545532227, 13.433697700500488, 13.0377836227417, 21.80647087097168, 16.730363845825195, 14.3930082321167, 16.533035278320312, 18.04314613342285, 19.205814361572266, 18.499473571777344, 18.989301681518555, 11.475446701049805, 18.010725021362305, 19.689075469970703, 16.737802505493164, 22.545024871826172, 15.287100791931152, 20.691730499267578, 13.767906188964844, 19.063426971435547, 7.247564792633057, 16.661449432373047, 18.60480308532715, 15.972084045410156, 21.34514808654785, 19.09720230102539, 15.586197853088379, 20.334501266479492, 18.752601623535156, 25.387067794799805, 19.0732364654541, 15.474156379699707, 20.187488555908203, 16.88636589050293, 15.886622428894043, 27.354820251464844, 20.80868148803711, 16.103662490844727, 14.717491149902344, 22.027381896972656, 25.055240631103516, 18.195486068725586, 16.21783447265625, 23.865205764770508, 17.5230655670166, 18.416576385498047, 22.897369384765625, 23.92755889892578, 19.904441833496094, 28.20939064025879, 21.45560073852539, 17.98737907409668, 24.473337173461914, 22.734153747558594, 25.015146255493164, 22.39643669128418, 15.906509399414062, 20.845869064331055, 18.33480453491211, 22.339248657226562, 24.500295639038086, 23.47164535522461, 17.523048400878906, 16.38507652282715, 25.810218811035156, 21.99557876586914], [-10.441723823547363, 16.18501853942871, 13.716423988342285, 12.133302688598633, 9.919816017150879, 12.528600692749023, 5.563387393951416, 11.480813980102539, 17.14763641357422, 7.998967170715332, 17.662837982177734, 15.858148574829102, 7.090999126434326, 9.353863716125488, 11.474505424499512, 15.643688201904297, 11.033308982849121, 16.904970169067383, 15.562766075134277, 13.139970779418945, 14.845531463623047, 18.771793365478516, 18.891965866088867, 13.262327194213867, 13.837063789367676, 23.066768646240234, 17.166872024536133, 11.400272369384766, 29.660802841186523, 16.488372802734375, 21.897249221801758, 16.27165985107422, 14.52296257019043, 12.855749130249023, 18.64706039428711, 18.087068557739258, 15.402942657470703, 27.281169891357422, 14.980218887329102, 16.81975555419922, 22.87759017944336, 15.749471664428711, 17.375049591064453, 18.79107093811035, 13.749099731445312, 11.586795806884766, 21.222497940063477, 32.741668701171875, 14.366455078125, 16.483203887939453, 18.823139190673828, 10.643501281738281, 26.33159065246582, 15.623608589172363, 26.70246124267578, 23.103479385375977, 19.987255096435547, 25.10254669189453, 14.547870635986328, 19.623716354370117, 22.27326011657715, 18.041841506958008, 15.625798225402832, 15.335308074951172, 20.23028564453125, 18.821678161621094, 24.880401611328125, 24.501340866088867, 22.466747283935547, 23.264612197875977, 25.16303062438965, 27.59670066833496, 15.847896575927734, 22.458751678466797, 14.73642349243164, 22.667133331298828, 20.047502517700195]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[[0.16645976901054382, -0.7796664834022522, -0.8263030052185059, -0.9469125270843506, -0.3119489550590515, -0.40285882353782654, 0.1077691912651062, -1.23544442653656, -0.7272970676422119, -0.1728445142507553, 0.7950834035873413, -1.092186450958252, 1.3965355157852173, 1.6942213773727417, -1.8071948289871216, -0.37792932987213135, 0.9108468294143677, 1.3323016166687012, 0.7738511562347412, -1.3518781661987305, 1.4302513599395752, -0.6094102263450623, 1.0803240537643433, -0.9098564386367798, 0.6467806100845337, -0.4649302661418915, -0.6908559203147888, -1.3476814031600952, 0.15516966581344604, -0.3131853938102722, -0.4120500385761261, -0.8111737966537476, -1.5824695825576782, -1.6716537475585938, 0.541368305683136, -1.1121790409088135, 1.0702157020568848, 0.20404835045337677, 0.9135958552360535, 0.5027125477790833, 0.22723397612571716, 0.27085599303245544, -1.3347822427749634, 1.0771375894546509, -0.8454411625862122, -2.56870436668396, 0.5395628809928894, -0.816694974899292, 1.8673053979873657, 0.4251197874546051, -1.321951985359192, -0.6929566264152527, 0.4547458291053772, -0.20428839325904846, 0.4642336070537567, 0.5909931659698486, -0.4044553339481354, -0.3801858127117157, -0.9049007296562195, 1.2420673370361328, -0.007600613869726658, 0.4704030156135559, -1.6406378746032715, 0.06010650098323822, 0.12566038966178894, -0.5075125694274902, 0.8473456501960754, 0.5653030872344971, -0.3963894546031952, 0.47141334414482117, -1.2270808219909668, -0.004805751610547304, -1.3304468393325806, 0.8949440717697144, 0.39885541796684265, 0.5373623371124268, -0.3698216378688812, -1.4513506889343262, 0.4062083661556244, -0.47847166657447815, -0.20770801603794098, -0.34859928488731384, -0.16618672013282776, -0.9127791523933411, 1.753400206565857, 0.7550128102302551, -0.5132474303245544, 2.6615817546844482, 0.15395419299602509, -0.0002825050614774227, -0.330282062292099, 0.01775188557803631, 0.592464804649353, -0.32497164607048035, -0.20458215475082397, -0.4289951026439667, -2.7594056129455566, 2.6154096126556396, 0.06658577173948288, -0.47499847412109375, 0.16133557260036469, -1.27767014503479, -1.6173784732818604, 2.1495466232299805, 0.4270859360694885, -1.2553473711013794, 0.507304847240448, -0.44783440232276917, -1.3126349449157715, -1.078972339630127, -1.1165403127670288, -2.0122570991516113, -0.23538091778755188, 1.2021241188049316, -0.11268365383148193, -0.09710409492254257, -1.4558424949645996, 0.4019334018230438, -0.8522088527679443, 0.23471415042877197, -0.7093002796173096, -0.7738845348358154, -1.4008909463882446, 0.6584088802337646, 0.11488695442676544, -0.0005553894443437457, 1.2487741708755493, -1.318288803100586], [0.141489177942276, 0.8550299406051636, 0.0027527527417987585, 0.16567882895469666, 1.209114909172058, -0.3692576289176941, 0.05669734999537468, -0.9900895953178406, 0.1193060427904129, 0.4161837100982666, 0.676249623298645, -2.1001648902893066, 0.9493546485900879, 1.076552391052246, 0.8685345649719238, 1.0335400104522705, 1.808281421661377, -1.5873221158981323, -0.24487602710723877, 0.37693533301353455, 0.7089977860450745, -0.82635897397995, -0.7455736398696899, 0.32963261008262634, -0.8488976359367371, 0.38696563243865967, 1.2069841623306274, -0.4545031785964966, 0.8324726223945618, 0.19327333569526672, -1.3881934881210327, 0.7915263772010803, -0.6030990481376648, 2.914642095565796, 1.5403789281845093, 1.693419098854065, 0.718900740146637, -0.7637272477149963, 0.7287774085998535, 0.1997727006673813, -0.6177738308906555, -0.43329665064811707, -0.8975925445556641, -0.14324502646923065, -1.2255077362060547, 0.06779266893863678, -0.006488476879894733, 0.2518111765384674, 0.8103564977645874, -0.0782608836889267, 1.0197137594223022, 0.08499769866466522, 0.8091713786125183, 1.508864164352417, 1.6647529602050781, -0.11946119368076324, -0.9427960515022278, -0.12862034142017365, 0.7306740283966064, -0.038892339915037155, 0.792524516582489, 0.18385863304138184, 0.14841949939727783, 0.43927228450775146, -1.8506014347076416, -1.286590576171875, -0.46253159642219543, -1.1243510246276855, -0.32005152106285095, -0.5253484845161438, -0.1033099889755249, 0.48157137632369995, -0.11192773282527924, -0.34706294536590576, 0.3127013146877289, -0.18782281875610352, -0.30966222286224365, 0.6248055100440979, 0.7488421201705933, 0.06502500176429749, 0.4358768165111542, 0.7456426620483398, -0.8469083309173584, 1.5401196479797363, -1.841785192489624, 0.030081525444984436, 0.3510938882827759, -0.23259809613227844, 0.05747446417808533, 0.9998859763145447, 1.5408629179000854, -0.6552034616470337, -0.1414206176996231, -0.332385778427124, -0.38654041290283203, -1.0341911315917969, 0.9572256803512573, 1.223023772239685, 1.2139344215393066, -0.4408715069293976, -0.6068252325057983, 0.9575883150100708, -1.9405163526535034, 0.9394313097000122, 1.2660883665084839, 2.0553536415100098, -0.16899217665195465, 0.39584362506866455, -1.0685937404632568, 0.5244969129562378, -0.08558975160121918, -0.4220709800720215, 2.3148536682128906, 0.5515385866165161, -0.3257550299167633, 0.031116729602217674, 0.7649251222610474, -0.31755322217941284, 1.6536964178085327, 0.27914345264434814, -1.735944151878357, -0.5796083807945251, 0.1885089874267578, 0.23973055183887482, 1.0290417671203613, 1.2357525825500488, 0.41452890634536743, -1.1835486888885498], [-0.03929924592375755, 2.0339982509613037, 0.20669332146644592, -0.6396231055259705, -0.655633807182312, -0.45374882221221924, -0.5290180444717407, 0.34673917293548584, 1.0394842624664307, 0.2544306218624115, 0.699026882648468, 0.21842622756958008, -0.34586891531944275, -2.2645034790039062, 0.471287339925766, -0.32811036705970764, 1.393481969833374, 1.319053053855896, -0.10661054402589798, 1.3259931802749634, 1.762200117111206, -0.418197363615036, 0.7197379469871521, 0.019754936918616295, -2.3173646926879883, -0.8720777630805969, 0.27378565073013306, -0.9538936018943787, 0.5110589861869812, -0.3644756078720093, -0.42383337020874023, 1.0272010564804077, -1.5091757774353027, -0.04955184832215309, -0.7258148193359375, -0.21810992062091827, 0.10497431457042694, -1.993193507194519, -0.7260849475860596, -0.16537199914455414, -0.29540392756462097, 0.3989240527153015, -0.5937755107879639, -0.5309692025184631, -0.11253968626260757, -0.6135295629501343, -1.7703578472137451, 0.25895005464553833, -0.2740655839443207, -0.4352995455265045, -0.6671940088272095, 0.46695345640182495, -0.3971329927444458, -0.6019662022590637, -0.951003909111023, -0.6432358622550964, 0.1110243871808052, -0.09988170117139816, -1.8288254737854004, -2.015596628189087, 0.5116882920265198, -0.8919594883918762, 2.2652180194854736, -0.26272764801979065, -0.347671240568161, 2.3127689361572266, 0.6709262728691101, 0.6326205730438232, -0.21998684108257294, -0.48307332396507263, -0.13001586496829987, -3.110945701599121, 0.17809276282787323, 0.9498947262763977, -0.5844858288764954, 0.6843580007553101, 0.40327319502830505, -0.017202546820044518, -1.4482768774032593, -0.1252305954694748, -1.5015435218811035, 1.1013270616531372, 0.617887556552887, -0.12797854840755463, 0.7610763907432556, -0.24426305294036865, -0.11612068116664886, -1.8269789218902588, 0.636407732963562, 0.4162820279598236, -0.04049598053097725, 0.28977909684181213, 0.606256365776062, 0.9783321022987366, 0.3553454577922821, -1.5222580432891846, 0.4623263478279114, -1.8871386051177979, 0.5876609683036804, 0.255524605512619, 0.7386438846588135, -1.3915570974349976, -0.1677524596452713, -0.11141591519117355, -0.9103076457977295, 0.6416849493980408, 2.3157525062561035, 0.6040016412734985, 0.22819527983665466, -0.7216394543647766, -1.750934362411499, -0.8312961459159851, 1.6957173347473145, 0.03472551330924034, 0.766441822052002, 0.1944746971130371, -0.5362729430198669, 1.0922036170959473, 0.34047672152519226, 1.4331077337265015, -1.1296732425689697, 0.8296632766723633, -1.0660353899002075, 0.8597016930580139, -0.8174484372138977, 0.7630545496940613, -1.8811641931533813, 2.5588791370391846], [-1.0629534721374512, -0.9237795472145081, -0.930732786655426, 0.6292055249214172, -0.45866355299949646, 0.1843208223581314, -0.3498823940753937, -1.7557013034820557, -0.0028026362415403128, -1.1573927402496338, -1.7151399850845337, -1.5337344408035278, -0.5152877569198608, 1.8883814811706543, -0.6795533895492554, 0.27708736062049866, 1.8643163442611694, -0.10700421035289764, 2.1651201248168945, 0.7517788410186768, 1.0619372129440308, 1.2556871175765991, 0.07109816372394562, -0.21033519506454468, 0.14827872812747955, -0.782109797000885, 1.2830857038497925, -2.847913980484009, 1.2215698957443237, 1.0566991567611694, 1.2191940546035767, -0.649839460849762, -1.7332441806793213, 2.136915445327759, 1.862624168395996, 0.7888302206993103, -1.5709877014160156, -0.3220936357975006, -0.4470337927341461, -1.7143090963363647, 0.13052161037921906, 0.6387179493904114, -1.518239140510559, -0.27186837792396545, 0.05777180567383766, 0.9992484450340271, 0.3487069010734558, 0.049583859741687775, 0.08165502548217773, -2.7217304706573486, 0.34857797622680664, 0.12423378974199295, 0.18519921600818634, -0.820268452167511, 0.8248895406723022, -0.4774251878261566, 0.4122334420681, 0.02045418508350849, 1.0216282606124878, 1.0444252490997314, 1.0653531551361084, -1.295820713043213, -2.521186351776123, 0.4771566689014435, -0.28366684913635254, 0.9357770681381226, 0.7487754225730896, 2.0369110107421875, 1.2947964668273926, -0.07630449533462524, -0.2838274836540222, -1.4307941198349, -1.472350001335144, 1.2191412448883057, 1.1450103521347046, -1.179611325263977, 1.5140435695648193, -0.1755860447883606, -1.2198529243469238, 0.0889800563454628, 2.033329725265503, 0.455202579498291, -1.6650971174240112, -0.9887540340423584, 0.8809448480606079, -1.0191646814346313, 1.1667983531951904, 0.8188389539718628, -1.2423183917999268, 1.224828839302063, -1.1853407621383667, 0.01700822450220585, 0.14445656538009644, -0.4329153299331665, -0.42914262413978577, 1.6269875764846802, -2.412860155105591, -1.4678293466567993, -0.3375471830368042, 0.6801741719245911, 1.5792657136917114, -0.49712586402893066, -0.7333205342292786, -0.7770422697067261, -1.9067480564117432, 1.207780361175537, 0.5968537926673889, 0.3561839461326599, 0.028303081169724464, 0.2825780510902405, -0.8818649649620056, -0.6595504283905029, -0.19087465107440948, -1.6012569665908813, 1.0516334772109985, 0.5901572704315186, -0.9047498106956482, -0.7333002090454102, 0.1758519411087036, 0.543663501739502, -1.185427188873291, 0.5483598709106445, 0.34425681829452515, 0.6290805339813232, -0.6346705555915833, 0.05482669174671173, 2.3647654056549072, -0.6560408473014832], [1.3904799222946167, -0.07526888698339462, -1.6137511730194092, 0.144635871052742, -0.3232896625995636, 0.4313335418701172, -0.8935742974281311, -0.28142011165618896, 0.7295412421226501, 0.665468692779541, -1.8862667083740234, 0.1600969135761261, -0.43902280926704407, -0.40307798981666565, 0.7500718235969543, -0.844903826713562, -0.15359416604042053, -0.185334250330925, 0.8556719422340393, -0.26930367946624756, 0.8615041971206665, -0.23309850692749023, -0.28621286153793335, 1.9522581100463867, 1.9052590131759644, 2.009690523147583, 1.742729663848877, -0.5657076835632324, -0.9536154270172119, -1.4795840978622437, 0.8706649541854858, 1.6514240503311157, -0.11901530623435974, 0.6212575435638428, -1.1161447763442993, -0.7724332213401794, -3.541881561279297, -0.16175420582294464, 1.0626968145370483, 0.6402493715286255, -0.6554163098335266, 0.7640407085418701, 0.6429395079612732, 0.6274697184562683, 0.21035335958003998, 1.5422227382659912, -1.6611762046813965, -0.651978075504303, 0.17373396456241608, 0.8405548930168152, 0.7250489592552185, 0.15456454455852509, 1.1488882303237915, 0.04682615399360657, -1.524538516998291, -0.053456682711839676, 1.6343164443969727, -0.9829875230789185, -1.176990032196045, -0.5566912889480591, 0.7798228859901428, -0.7973806858062744, 1.0629899501800537, 2.270267963409424, -0.055301032960414886, 0.04029882699251175, -0.7296891212463379, -0.3910241425037384, -0.4811434745788574, 0.672146737575531, 1.0692335367202759, -0.6379397511482239, -0.7908941507339478, 0.7226111888885498, -1.074572205543518, 0.4100989103317261, -0.015138731338083744, 0.7292391657829285, 0.8001604080200195, -0.42974457144737244, -0.4921253025531769, -1.0906462669372559, 1.1209601163864136, 0.2504824697971344, 1.7569117546081543, -0.3928358852863312, -0.8710166811943054, -0.6625537872314453, 1.0387389659881592, -0.250009149312973, -0.9577900767326355, -0.28974422812461853, 0.45892247557640076, 2.0849788188934326, -1.8673104047775269, 1.3981518745422363, 1.3242508172988892, -0.027397096157073975, -0.7060176134109497, 0.28279003500938416, -0.7481266856193542, 0.5423043966293335, 0.6345872282981873, -1.5605579614639282, 0.79045170545578, 0.2362217754125595, -1.0602141618728638, -0.9127445816993713, 0.29159632325172424, -0.594506561756134, 0.8782057762145996, -0.18935196101665497, -0.9699833989143372, -0.3160822093486786, 0.5367415547370911, 0.8310648202896118, -0.3729764521121979, 0.03168212249875069, -0.1506405472755432, -0.5422921776771545, 0.11289983242750168, 0.02859359048306942, 0.1181986853480339, 0.5961824059486389, 0.06022927165031433, -0.5567064881324768, 0.5302567481994629, 0.35329338908195496], [0.41448837518692017, 0.8571105599403381, -0.12922728061676025, -1.4925469160079956, 0.8544774055480957, -2.021230697631836, -0.7998220920562744, 1.0014950037002563, 1.358157753944397, 1.8044880628585815, 0.1395399123430252, 0.6809146404266357, 1.7764359712600708, -0.10086124390363693, 0.535930335521698, -1.591916799545288, 0.05772078037261963, 2.084305763244629, 1.279725432395935, -1.6519012451171875, -0.2748146653175354, 0.017554938793182373, 0.17597417533397675, -0.27384257316589355, -0.321086049079895, 2.424560308456421, 0.13830602169036865, -0.7500147223472595, 0.6893857717514038, -1.9532749652862549, -1.8244895935058594, -0.3372504413127899, -0.38055282831192017, 1.153554916381836, 0.8570706844329834, -0.10151307284832001, -1.5487775802612305, -0.9246623516082764, -0.712398111820221, 0.4896868169307709, 2.014347791671753, -0.022828493267297745, 1.9089452028274536, 0.9798198342323303, 0.6777448058128357, 0.3718840479850769, -0.5317946076393127, 1.074995994567871, 1.7825425863265991, -0.7205621600151062, -1.8710039854049683, 0.9535892605781555, 0.37718772888183594, 1.82895028591156, -0.2837980389595032, 1.4570084810256958, -1.1269079446792603, -0.9407751560211182, 1.3600987195968628, -1.1436762809753418, 0.18481720983982086, -0.7480332255363464, -0.2947516143321991, -1.2466455698013306, 1.5022796392440796, -2.5543980598449707, 0.40805360674858093, -0.3613626956939697, 0.12445927411317825, -1.0053274631500244, -1.200157880783081, -0.24098993837833405, 0.2697512209415436, 0.5316081643104553, -1.5378302335739136, -0.12950092554092407, -1.5828142166137695, -0.8323721289634705, -2.3311893939971924, 0.2181459665298462, 0.9184338450431824, -0.6153121590614319, 0.8155749440193176, -1.0333194732666016, 1.4960354566574097, 0.848369836807251, 0.6280941963195801, -0.007306843996047974, -0.6441685557365417, -0.956628680229187, -0.7564312219619751, -0.9308457374572754, -0.7423632740974426, 0.5114534497261047, 0.6419897675514221, 0.5414959788322449, -0.8209984302520752, 1.5258805751800537, 0.12373429536819458, -1.948543667793274, -0.6146001219749451, -0.994766354560852, -1.3803502321243286, -0.04051525890827179, -0.7702617049217224, -0.08497578650712967, 0.7027732729911804, 0.7701096534729004, -0.5834960341453552, 2.099370002746582, -0.9043054580688477, 0.42383694648742676, 0.7770076394081116, 0.5354408025741577, 0.33100733160972595, 0.9230484366416931, 1.0153781175613403, 0.660763144493103, 0.8317955732345581, -2.4826595783233643, 0.8746310472488403, 1.5457395315170288, -1.491392731666565, -0.21020373702049255, 0.9973211884498596, 0.3215487003326416, 1.2983944416046143, -0.15236778557300568], [-0.16397103667259216, 1.1192420721054077, -1.4688565731048584, -0.41702166199684143, -0.5705860257148743, -1.0541929006576538, -0.9732851982116699, 1.0585283041000366, 0.41500911116600037, -0.363669216632843, 0.6992782354354858, 0.7398461699485779, -0.4496983587741852, 0.10174261033535004, -0.3683897852897644, -0.430033415555954, 0.8903207182884216, -1.104231834411621, -1.1587097644805908, 0.7259805202484131, 0.7386483550071716, -0.35192301869392395, 0.7130617499351501, 0.8770725131034851, -1.9395592212677002, 0.29487696290016174, -0.6103076934814453, 1.2494051456451416, 1.172940731048584, -0.7343142628669739, 0.2930006980895996, -1.1471402645111084, 0.7438784241676331, 0.9299336671829224, 0.0684250071644783, 0.5184774398803711, -0.0968373566865921, -1.8025741577148438, 0.5627432465553284, -0.1009594276547432, 1.2556699514389038, -0.6069186329841614, 1.0636595487594604, -1.103516697883606, -0.8960185647010803, 1.0822254419326782, -0.6931696534156799, 0.9056976437568665, 0.6163303852081299, -0.21253345906734467, 0.13282912969589233, -1.2984074354171753, 0.6128883957862854, -1.0543477535247803, 0.12832556664943695, -0.7217082381248474, 1.7730263471603394, 1.0585308074951172, 0.7035625576972961, -2.022491693496704, 0.3381615877151489, -0.18233831226825714, -2.453721046447754, 1.3479338884353638, -0.4683704674243927, -0.8022887706756592, -1.4613971710205078, 0.3919416069984436, -1.2449889183044434, -0.3621062934398651, -1.0628740787506104, -0.9422309398651123, -0.7071382403373718, 0.7873538136482239, 1.931033968925476, -0.3688587546348572, -0.13802814483642578, 0.09171193838119507, 0.7408416271209717, 0.3710159361362457, -1.5811668634414673, 1.7000508308410645, 0.5579409003257751, 0.39765071868896484, -0.11239079385995865, 0.2736510634422302, -0.3957139849662781, 1.0840402841567993, -1.8900262117385864, -0.4646260142326355, -0.3364981710910797, -0.8651144504547119, -0.6427839398384094, -1.0183502435684204, -0.7251688838005066, -0.04476175457239151, -0.4494534134864807, -1.4595344066619873, 0.8202058672904968, -0.476352721452713, 0.7294418215751648, -1.0242786407470703, 0.9661521315574646, 0.039103925228118896, 1.2151875495910645, -1.8909156322479248, -0.44301068782806396, -1.5382083654403687, -0.08975248783826828, -0.8668372631072998, -2.10467529296875, 0.2978867292404175, 0.08420669287443161, 0.962722897529602, -0.4261602759361267, -0.45531702041625977, 0.2202938199043274, -0.3277531862258911, -0.3418315649032593, 0.35134536027908325, 0.9624243378639221, 0.01246469747275114, 1.4245959520339966, -1.4273227453231812, -0.5076351761817932, -0.24859251081943512, 0.040906310081481934, 0.15995220839977264], [0.2675222158432007, -0.5468982458114624, 0.0883062332868576, -0.564264178276062, -0.19377686083316803, -0.6562811136245728, 0.03557693213224411, 0.9719159603118896, -1.5377002954483032, 0.09150815010070801, -0.2386857569217682, -0.6358447074890137, -0.5810344219207764, -1.1569689512252808, 0.8279263377189636, -1.9869462251663208, 0.6300548911094666, -1.063463807106018, -0.5681928396224976, 0.08545807003974915, -0.033008500933647156, 0.6648340821266174, 0.037066541612148285, 1.462424874305725, -0.1669950932264328, -0.12059486657381058, 0.44272381067276, 0.9421482682228088, 0.9711173176765442, -0.5174579620361328, 0.47140881419181824, -0.6531749367713928, -0.6260677576065063, -0.8658745884895325, 2.0609800815582275, 0.7064918279647827, 1.1248708963394165, -0.930415689945221, -0.5928887128829956, -0.49547281861305237, -0.6579880714416504, 1.9293553829193115, 1.0236320495605469, -0.7309693098068237, -0.8284226059913635, 0.6414288878440857, 0.6284979581832886, 1.0775913000106812, 2.3512797355651855, -1.5761864185333252, -0.19205248355865479, 1.1229662895202637, -0.23597899079322815, -0.8668398857116699, 0.16504713892936707, 0.21431371569633484, -0.5778136849403381, 0.2682607173919678, 0.8843136429786682, 0.4220656752586365, -0.28438299894332886, -0.19878007471561432, 0.5508587956428528, -0.3046160936355591, -1.0077478885650635, -1.2988991737365723, -0.5348479747772217, 1.7408711910247803, -0.8864789009094238, -0.6767852902412415, -0.5344098210334778, 0.3217843472957611, -0.2709801197052002, 1.1425267457962036, 0.21551772952079773, 2.021862745285034, -1.327821135520935, -0.30195823311805725, -1.5287202596664429, 0.7759039998054504, -1.2110823392868042, -1.464290976524353, -1.3645128011703491, 1.397144079208374, -0.6016418933868408, -0.17813582718372345, 0.7752432227134705, 0.6459044814109802, 0.8852590918540955, 1.4466805458068848, -1.1667392253875732, -1.0901432037353516, -0.1757233589887619, 1.658294677734375, -0.8791334629058838, 2.082122564315796, 2.048222541809082, 1.1534591913223267, -0.11304153501987457, -0.07084726542234421, 0.8140977025032043, 0.6006114482879639, -0.5290310382843018, 0.17505432665348053, 0.11076321452856064, 0.5443836450576782, 0.6945832371711731, -1.1628154516220093, 0.9239756464958191, 1.1849026679992676, 0.6066630482673645, 0.5624355673789978, 0.5132530927658081, -0.27410101890563965, 0.5823625922203064, 0.3357366621494293, 1.4430102109909058, -2.2825779914855957, -0.08085213601589203, 0.733568549156189, -0.08619911223649979, 0.9237032532691956, -0.23213639855384827, 0.29067450761795044, 0.17532707750797272, -0.7005651593208313, -0.3910169303417206, 0.9141446352005005], [-1.463091492652893, -0.33930546045303345, 1.395564317703247, -0.3842620849609375, 0.3075859844684601, 1.2175599336624146, -0.5817527770996094, -0.2264779806137085, -0.5953996181488037, 0.9127190709114075, -0.3317844569683075, -0.9041547775268555, -1.8745126724243164, -0.5324464440345764, 2.313295364379883, 1.34690260887146, 0.2808198928833008, -0.7927132248878479, 1.0682467222213745, 1.282680630683899, -0.9650316834449768, -1.9791523218154907, 0.7934505343437195, -1.4821354150772095, 0.5614113807678223, -0.07498138397932053, 0.7366666197776794, -1.224882960319519, -0.170779749751091, 0.7748299241065979, 1.1673223972320557, 0.16050449013710022, 0.23009756207466125, -1.0060386657714844, -0.9656670093536377, -1.5834323167800903, 0.27680861949920654, 0.028479207307100296, 0.5387198328971863, -1.0658416748046875, -1.9004377126693726, -1.096711277961731, -0.10634983330965042, -1.3604753017425537, 0.25304898619651794, 0.12282827496528625, -0.022524110972881317, -0.4413033723831177, -0.884247362613678, -0.7773006558418274, 1.9558560848236084, 0.380684494972229, 0.9350997805595398, -1.0403188467025757, -1.2008239030838013, 1.143317699432373, 1.1212399005889893, -1.4930219650268555, 0.4026321470737457, 0.36117175221443176, 0.5818193554878235, 0.29405927658081055, -1.5334455966949463, 0.35969483852386475, -0.5754395127296448, 0.03990982845425606, 0.22887077927589417, -0.7764625549316406, 0.25443896651268005, 0.3709472417831421, -1.8269761800765991, 0.1588076949119568, -0.4086263179779053, -0.8203548192977905, 1.6094064712524414, 0.37289097905158997, -1.1002051830291748, 0.90171217918396, -0.2647051513195038, 1.1693812608718872, 1.202786922454834, -1.3752435445785522, 1.8367893695831299, -0.7620449662208557, -1.4812963008880615, 0.7431697249412537, -0.8456105589866638, -0.22512514889240265, -2.160719633102417, 0.3211410939693451, 1.9539189338684082, -0.16566798090934753, 0.24954672157764435, 0.6779429912567139, -0.07038786262273788, 0.47524866461753845, 0.9355440735816956, -0.8820394277572632, -0.344774067401886, 1.451852560043335, 0.4789065420627594, -0.3703792095184326, -0.41517603397369385, 0.0027417896781116724, 0.9557102918624878, 1.1290351152420044, -1.0248790979385376, 1.4989368915557861, -1.1111714839935303, 0.488249272108078, 1.3518301248550415, 0.13598649203777313, 0.9765852093696594, -1.0531872510910034, 0.5742712616920471, 0.09571687877178192, 0.14362095296382904, 0.2835579514503479, 0.19530834257602692, 0.5585578680038452, 2.2613534927368164, -0.14038756489753723, 0.38564056158065796, -0.6871531009674072, -2.0360379219055176, 0.2140105962753296, 1.4624872207641602, 3.061319351196289], [-0.23284503817558289, -0.12071380764245987, 2.819528818130493, -0.16212597489356995, -1.1123567819595337, -1.3952598571777344, -0.5820633769035339, 1.1942418813705444, 0.6733360290527344, -0.48937535285949707, -0.9225777983665466, -0.054702527821063995, -1.5035619735717773, 0.2075241357088089, -1.837691068649292, 1.021958589553833, -0.3187946379184723, -1.8087663650512695, -0.39385318756103516, 0.2954408526420593, 0.25524717569351196, -0.06716255098581314, 0.2615477442741394, -1.192541241645813, 1.6078166961669922, 0.8525657057762146, -0.8981068730354309, -0.30304720997810364, 1.839898943901062, 1.050887107849121, 0.9745334386825562, 0.46367350220680237, -0.13955573737621307, 2.3851566314697266, -0.27957355976104736, -0.5797490477561951, 0.2257402390241623, -0.42696064710617065, 0.005314243491739035, 1.4249083995819092, -0.7614731788635254, -0.8167778253555298, -1.5165894031524658, -0.1847391575574875, 0.755584180355072, -0.5600859522819519, -1.1258500814437866, 0.1752089560031891, 1.962782859802246, 0.14476101100444794, 2.925135850906372, -0.18719670176506042, -0.1521734893321991, -0.681620180606842, 0.20459380745887756, -1.4943139553070068, -0.49359840154647827, 0.7234270572662354, 0.5499576926231384, 0.7918232679367065, 0.05354676395654678, -0.13484281301498413, -0.32588571310043335, -0.0454702228307724, -0.6810845136642456, -0.6338960528373718, -1.4401121139526367, -1.4105788469314575, 0.30756640434265137, -0.09052371233701706, -1.4031333923339844, -2.747178077697754, 0.6859263777732849, 1.2442982196807861, -0.9643498659133911, 0.2733002305030823, 0.7858842015266418, 0.52186518907547, 1.6075892448425293, 0.5211896896362305, 1.4078742265701294, -0.05341620370745659, -0.5846129059791565, 1.1210392713546753, 1.1461914777755737, 0.8940764665603638, -1.3567900657653809, 0.4206782877445221, -1.1341731548309326, 1.4670131206512451, -0.43475332856178284, 0.4791204035282135, -0.24408045411109924, -0.84947669506073, 1.2022417783737183, 0.3942874073982239, 1.0055351257324219, -0.397399365901947, -0.023301495239138603, 0.5161518454551697, -0.01636015810072422, 0.7728440165519714, -0.22197109460830688, 0.08917433768510818, 1.4632161855697632, 1.5976600646972656, -0.740610659122467, -0.8063002824783325, -0.9511967301368713, 0.9303422570228577, -0.8884350657463074, 0.3251672685146332, -1.0183125734329224, -0.02589854598045349, 0.033950358629226685, -1.4128849506378174, 0.025210611522197723, -0.14214590191841125, -0.300307959318161, -1.4008965492248535, 2.9932830333709717, 0.15614639222621918, 1.342860460281372, -0.6759633421897888, 1.8815884590148926, 0.25628089904785156, 0.536596417427063, -0.5358067154884338], [0.2670542001724243, -0.2200223207473755, -0.5814688801765442, -0.8871918320655823, 1.413560152053833, 2.938455104827881, 1.3100768327713013, 0.6240108013153076, -1.4998589754104614, -0.36791691184043884, 0.24030143022537231, -0.6018553972244263, 0.0930374413728714, 1.0755079984664917, 0.9413526058197021, -1.0615320205688477, 1.618620753288269, 0.740317702293396, 0.4153319299221039, -0.32343417406082153, -1.5275697708129883, -0.2697623074054718, -0.6035343408584595, -1.0109397172927856, 1.7856311798095703, -0.28996992111206055, -0.5176048874855042, -0.2620450258255005, 0.9350919127464294, 0.0573011077940464, 2.5208542346954346, -0.12596838176250458, 1.8811477422714233, 0.7642302513122559, -1.4176238775253296, 0.25879764556884766, 0.5865342617034912, -0.5045294761657715, -1.6723620891571045, -0.20196878910064697, -0.4298498332500458, 0.05731701850891113, 1.196286916732788, -1.4062775373458862, 1.1209285259246826, -0.49157536029815674, -1.3117913007736206, -0.3821149170398712, 1.7774653434753418, -0.5254608392715454, -1.1518596410751343, 1.7679489850997925, -2.39015531539917, -1.4060389995574951, 0.8882520794868469, -0.31509238481521606, 0.9359185099601746, 1.6082760095596313, 0.9685095548629761, -0.6712290644645691, -1.1799523830413818, -0.42522284388542175, -1.4076869487762451, -1.0188086032867432, -0.9922245144844055, -0.7531033754348755, -0.5297845005989075, 0.7853405475616455, 1.4046422243118286, 1.8014204502105713, -0.3846305310726166, -1.2346978187561035, -0.1470959186553955, -0.508781373500824, 0.5778115391731262, 1.1276792287826538, 1.6681971549987793, 2.770979404449463, -0.17527826130390167, 0.9618249535560608, 0.8688139915466309, -1.4211821556091309, -2.096590995788574, 0.18814097344875336, 1.5404859781265259, 0.2756615877151489, 1.4320462942123413, 0.45158591866493225, -0.573513925075531, -0.7591789364814758, -0.5897089838981628, -0.17927424609661102, -0.7743828892707825, 0.9970371723175049, 0.24724097549915314, 0.05300584435462952, -0.16259874403476715, -1.1417436599731445, 0.16371993720531464, 0.9095252156257629, -0.046519286930561066, 0.5566685199737549, -1.7435005903244019, -0.6679979562759399, -0.590516984462738, 0.10106998682022095, -1.7864494323730469, -0.1130824014544487, -1.101407527923584, 0.37805694341659546, -1.3373639583587646, 0.8985453248023987, 0.4246453046798706, -1.34909188747406, 0.018621770665049553, 0.49771058559417725, 0.6123108267784119, 0.3717776834964752, 1.002475619316101, -0.7806758284568787, 1.5047078132629395, -0.6085063815116882, 0.42612409591674805, -0.1473000943660736, 2.571592092514038, 2.0921177864074707, -1.64878511428833, 0.011716349981725216], [-1.306963562965393, -1.9602035284042358, 0.9253818988800049, -1.4455958604812622, -0.28535178303718567, -1.217704176902771, 0.4522098898887634, -1.4693517684936523, -0.01677258126437664, -0.6890560388565063, -0.16469208896160126, 0.9314926862716675, -0.26251766085624695, 0.3470485508441925, -0.15776091814041138, 0.6630868315696716, 0.5309807658195496, 1.3668675422668457, -1.974403977394104, 1.506300449371338, -0.5805759429931641, 0.05141114071011543, -1.0860412120819092, 0.39239367842674255, 0.3895379900932312, -1.0671837329864502, -1.2160900831222534, -0.2678990066051483, 0.7412527203559875, 0.8288561105728149, -0.20605753362178802, 0.562516450881958, -0.011687961407005787, -0.1076224073767662, -0.3841315805912018, 0.07356373220682144, 0.16489633917808533, 0.21379321813583374, 1.406839370727539, -0.03979808837175369, -1.9864501953125, -0.39292988181114197, -1.205816388130188, 1.0370333194732666, 0.22144433856010437, -0.42650872468948364, -1.610066294670105, -0.46131300926208496, -0.45366182923316956, 0.14731189608573914, -1.1663388013839722, -0.9915819764137268, -0.9131576418876648, 1.20692777633667, 1.5639220476150513, 0.21474075317382812, -0.9729694128036499, 1.7029880285263062, -0.5144850611686707, -0.09488225728273392, -0.14622657001018524, -0.2472199648618698, 0.30745112895965576, 1.8071891069412231, -1.9993067979812622, 0.24507738649845123, -0.35510849952697754, 0.1985345184803009, 0.7193554043769836, -0.09644609689712524, -0.334011971950531, 0.6606950163841248, -0.37912309169769287, 0.38824865221977234, -0.31229111552238464, 0.6766048073768616, 0.38320091366767883, 0.998417317867279, -0.06475228071212769, -0.5863348841667175, 1.1804461479187012, 0.7722926139831543, -2.3587121963500977, 0.2453172206878662, 0.14564388990402222, -2.141514539718628, -0.0637897327542305, 1.1222184896469116, 0.004085123538970947, -0.1845870018005371, 0.03378372639417648, -0.17674027383327484, 1.3713637590408325, 0.36145222187042236, -1.3927277326583862, 0.5604926347732544, -0.06351201236248016, 0.7481682300567627, 1.0212889909744263, 0.11082711815834045, 0.5022432208061218, -2.390796184539795, -0.573467493057251, -0.6399176716804504, -0.24155378341674805, -2.185398578643799, -0.9001892805099487, -0.24213159084320068, -0.41248732805252075, -0.006959992460906506, 2.264918327331543, 1.0753029584884644, 0.2091842144727707, -1.5359792709350586, 0.20788626372814178, 1.0378470420837402, -1.515216588973999, -0.09804835915565491, -0.06987283378839493, 0.06745413690805435, 2.023386240005493, -0.28030163049697876, -0.3374093174934387, -1.2123042345046997, 1.1330657005310059, -0.04656272754073143, -1.7310137748718262, 1.330167531967163], [0.9921790361404419, 0.8647368550300598, 0.5246359705924988, 2.444819450378418, 0.3647453784942627, 0.9492043852806091, 0.25839605927467346, 0.4809114336967468, -1.0487719774246216, -0.6024733185768127, 1.0463275909423828, 0.6989907026290894, 1.9847767353057861, -0.7698477506637573, 0.4975000321865082, 1.5585596561431885, -1.5208170413970947, -0.7308191657066345, -1.369088888168335, 0.6265148520469666, -1.4897065162658691, 0.07883980125188828, 0.173787459731102, 0.5608760118484497, -0.5606931447982788, 0.5795437097549438, -0.20016814768314362, 1.9430569410324097, 1.248046636581421, -2.4218008518218994, 1.4454447031021118, -0.665442943572998, -0.14692342281341553, -0.09052559733390808, -0.8986829519271851, 0.6877244114875793, -0.7406618595123291, 0.07524513453245163, -1.6786596775054932, -0.25943872332572937, -1.4702523946762085, -0.04201936721801758, 1.4736011028289795, 0.1414313167333603, -0.8706753253936768, -0.9840275049209595, 0.135115385055542, 1.2838847637176514, 1.2217786312103271, 1.6732796430587769, 0.330465704202652, -1.6388167142868042, 0.5195659399032593, 1.6298490762710571, -0.006809896323829889, -0.9927067756652832, 0.7229192852973938, -0.70738685131073, 0.7359158992767334, -1.4980286359786987, 0.2787262797355652, 0.845645010471344, -0.3374907970428467, 1.3287601470947266, -0.44667378067970276, 1.1127856969833374, -1.0277270078659058, 3.4837563037872314, -2.1951301097869873, 1.6097891330718994, -0.9013720750808716, 0.05994350090622902, -0.0902307853102684, -0.28027012944221497, 0.6422827839851379, 1.0724107027053833, -0.04506808891892433, -0.3872721791267395, 0.05223740264773369, 0.27165427803993225, 1.3579071760177612, 1.9339821338653564, -1.3746644258499146, -0.4535374641418457, 0.2662446200847626, 0.27655029296875, 0.500028669834137, 0.07976574450731277, -0.20861543715000153, 1.7023944854736328, -0.5372734069824219, 1.0726085901260376, 0.12682446837425232, 1.2331476211547852, 0.09754443168640137, -2.0940966606140137, 0.24759286642074585, 3.0159082412719727, 0.4504343867301941, -0.04890657588839531, 0.6651402115821838, 1.361184000968933, -0.4207436740398407, -0.5397427082061768, -0.39159855246543884, 0.14662866294384003, -0.6451525688171387, 0.8474573493003845, 1.7186250686645508, -0.549227774143219, 0.589483916759491, -1.1210017204284668, 1.2656501531600952, -0.011187169700860977, -0.7355507016181946, 1.2153064012527466, 1.31139075756073, -0.9468504786491394, -1.0978270769119263, 2.139514923095703, 1.4310245513916016, -0.8567702770233154, -0.7135807275772095, 0.6400172710418701, 0.38942524790763855, -0.1766078919172287, -0.49901288747787476, 0.47655823826789856], [-1.472974419593811, 1.6396313905715942, 1.251745343208313, -0.0035025146789848804, 1.59357488155365, 0.7433995008468628, -0.6627029776573181, 1.5126140117645264, 0.7298761606216431, -0.0009865418542176485, -1.0639545917510986, -1.070137858390808, -0.7460008859634399, -0.14081190526485443, 0.459547758102417, 0.4622756242752075, 0.7100421786308289, 0.6868484020233154, -0.5448289513587952, 0.07068022340536118, 2.886289358139038, -0.5774379968643188, -1.0734647512435913, -1.6500521898269653, 0.7091169357299805, -0.6181187033653259, 0.6263404488563538, -1.1355522871017456, 0.15217643976211548, 0.6408262848854065, -1.1763582229614258, 0.5628709197044373, 0.08657371997833252, 1.2649211883544922, 0.20625117421150208, -0.9482261538505554, 1.0898737907409668, -0.7098212838172913, 0.12064149975776672, 0.9319888949394226, -1.3350321054458618, -0.0230492502450943, -0.7757586240768433, -1.3591227531433105, 0.5665276050567627, 1.676397681236267, 0.22276905179023743, -0.13207900524139404, 0.7670737504959106, 0.3478017747402191, 1.2857091426849365, 0.6258623003959656, 0.611012876033783, 0.2537340223789215, 2.028822183609009, 0.8686655163764954, -0.20242474973201752, 0.11856774985790253, -1.768773078918457, 3.0699915885925293, -2.484553813934326, 0.5662955045700073, -0.05080844834446907, -1.1795438528060913, -1.1202924251556396, 2.032362699508667, -0.603329062461853, -0.8124350905418396, -0.2289751172065735, 0.8273072838783264, 0.07880597561597824, -0.3586333096027374, 0.056973326951265335, 0.9581425786018372, 1.5892090797424316, 1.0915416479110718, 0.11030810326337814, -0.7835885286331177, -0.039064254611730576, -1.0704835653305054, 0.0633176863193512, -0.8261792659759521, 0.14233016967773438, 0.006516309920698404, -0.32837730646133423, 0.2807946801185608, 0.6404628157615662, -0.3767111599445343, 0.24174238741397858, 0.7568717002868652, -1.2070443630218506, 0.6944278478622437, 0.691042423248291, -1.9090940952301025, 0.6298859119415283, -2.4161221981048584, -0.6831101775169373, -0.10949866473674774, 1.1824769973754883, 1.7066917419433594, 1.3237900733947754, -0.9690808057785034, 1.2184432744979858, -0.9342465996742249, 1.7243921756744385, 2.003659963607788, 1.1365294456481934, 0.157729834318161, 1.8635286092758179, 0.012257878668606281, -0.9293609261512756, -2.3081412315368652, -1.1741880178451538, 0.7398511171340942, -0.604300320148468, 0.1208852231502533, -0.009644605219364166, -0.2862582504749298, 0.5268048048019409, -1.326704978942871, -0.28710997104644775, -1.1491384506225586, 1.0427148342132568, 1.2214337587356567, -0.90980064868927, 0.40274032950401306, 0.14848096668720245, 1.3102165460586548], [-0.06870529800653458, 0.832541286945343, 0.5255059003829956, 1.4107942581176758, 0.32829123735427856, 0.8337521553039551, -0.08782224357128143, -0.17469987273216248, -0.06648606806993484, 1.3600000143051147, -0.2791856527328491, -0.12501539289951324, -1.0809381008148193, 0.44387874007225037, 2.4094929695129395, -0.04004555568099022, 1.0190684795379639, -1.1449552774429321, 0.15392988920211792, -0.6276165843009949, 1.8613204956054688, 0.7774943709373474, -1.4966586828231812, -0.12345156073570251, 0.5420424938201904, 0.30274778604507446, -0.04401404410600662, -0.443697065114975, -1.0949159860610962, -0.7629398703575134, -0.6066473126411438, -0.5133665204048157, 1.147464394569397, -2.986473560333252, -1.467569351196289, -0.4809492528438568, -0.07474659383296967, -0.9582744240760803, -1.6971875429153442, -0.032238081097602844, 0.7006251215934753, -0.495883971452713, 0.04352777451276779, 1.21579110622406, 1.2365694046020508, -0.21294592320919037, -1.3015426397323608, 0.6723359227180481, 0.6043132543563843, 0.3531018793582916, 0.39224568009376526, 0.42397749423980713, 0.14728060364723206, 0.8264627456665039, 2.5694375038146973, -0.7779784202575684, 1.905781865119934, -0.150084987282753, 0.011083410121500492, -0.9419400095939636, 1.1629751920700073, 0.8443896174430847, 0.15026043355464935, -0.10345611721277237, -0.8581024408340454, -1.3641003370285034, 0.5607635378837585, -0.12811265885829926, -2.0059571266174316, 1.5153534412384033, 1.9129904508590698, -0.5980049967765808, 0.5108945369720459, 0.3194870054721832, -2.0316903591156006, -0.1320236772298813, -1.0871877670288086, -0.03988192230463028, 0.060793936252593994, -2.3824756145477295, -0.8588078618049622, -0.37620145082473755, 0.4487755298614502, 0.28164422512054443, 1.489968180656433, 0.6167133450508118, 1.031092643737793, -0.3220355212688446, -0.013401017524302006, 0.18144914507865906, 0.10787375271320343, -0.8200123310089111, -0.8563986420631409, -0.9252695441246033, -0.3640357553958893, -0.7702327370643616, 0.2591665983200073, -0.4753405749797821, -1.4213486909866333, 0.9355193972587585, -1.1808120012283325, 0.33111292123794556, -0.26821407675743103, 1.2187482118606567, 0.44871386885643005, 0.7721498012542725, -1.2271126508712769, 0.26225054264068604, -2.268465042114258, -1.517711877822876, 1.5590933561325073, -0.026218954473733902, -0.668291449546814, -0.2994866967201233, 1.2343248128890991, -0.9188591241836548, -1.0907961130142212, -0.9609959125518799, 1.4984325170516968, -0.5112735033035278, 0.40664035081863403, 0.15918545424938202, -0.06494087725877762, -0.3262578547000885, -1.5277378559112549, 1.0522987842559814, -0.13169953227043152, -0.12859806418418884], [0.7978882789611816, -1.3789674043655396, 0.6922679543495178, 0.5232976675033569, 0.5884200930595398, -2.4285244941711426, 0.2735840976238251, 0.23593458533287048, 0.4086494445800781, 0.10223233699798584, 0.22700056433677673, 0.19523587822914124, 0.14159949123859406, -0.8766283988952637, 0.7582885026931763, -1.2150371074676514, -1.9134905338287354, -0.7034333944320679, -0.10194286704063416, -2.13938570022583, 1.1548017263412476, 1.0652351379394531, 0.6843648552894592, 0.4051402509212494, -0.49703505635261536, -0.010006151162087917, -1.089890956878662, -1.603427529335022, -0.43675386905670166, 0.6000321507453918, 0.240483820438385, 0.61839359998703, -0.6191868185997009, -1.092318058013916, -0.6929867267608643, -1.0558820962905884, 0.6449313759803772, 0.6695308089256287, -0.1810368448495865, -0.5631405711174011, -1.2070188522338867, 0.6954261660575867, -0.045047577470541, -0.8125913739204407, -0.2820632755756378, 0.39367780089378357, -0.05386313050985336, -0.4934472143650055, -0.6390866041183472, 0.2658054530620575, -0.9425702095031738, -0.8124885559082031, 0.7117690443992615, -1.8560032844543457, 0.7421696782112122, -0.22676941752433777, -1.6113165616989136, -1.1146308183670044, 1.3675026893615723, 0.8005049228668213, -0.16587504744529724, -0.18543003499507904, 1.1860554218292236, -0.1727064847946167, 0.7010725140571594, -0.013674694113433361, 0.6344565749168396, 0.8136265873908997, 2.1483681201934814, 1.0325785875320435, 0.13882608711719513, 0.23808173835277557, -0.00841658003628254, 0.6066790819168091, -1.3860218524932861, -0.813719630241394, 0.012438139878213406, 2.4709787368774414, -0.4235704243183136, 1.029791235923767, 0.4197891652584076, -0.053231723606586456, -0.8600826263427734, -0.1609908938407898, 1.380123257637024, -0.19120821356773376, -0.1309104859828949, 2.492230176925659, -0.198593407869339, -1.1234149932861328, 1.0273470878601074, 0.002970722271129489, -1.4873727560043335, -1.038163185119629, 1.3729406595230103, -0.5913184881210327, 0.7650123238563538, -0.17719508707523346, -1.8053343296051025, 0.6179206967353821, 2.16522479057312, 0.04453602805733681, -1.7286789417266846, -0.7620642781257629, 0.2827359735965729, -0.7311488389968872, -1.8714016675949097, -0.2970942556858063, -0.21232622861862183, 1.8043410778045654, 0.6346738934516907, -0.46657881140708923, -0.030741065740585327, 0.4791513681411743, 0.49937018752098083, 0.4887854754924774, 0.25050151348114014, -0.01281601469963789, -0.6923884153366089, -0.9297695159912109, 0.8051661849021912, 0.2938907742500305, -0.4693216383457184, 0.23656046390533447, 0.27556490898132324, 0.6574369072914124, 1.1649444103240967, 1.499237060546875], [0.26854783296585083, -0.5835586786270142, 1.3630579710006714, -0.8993228673934937, 1.5727890729904175, 1.0337687730789185, -0.09008762240409851, -0.11395896971225739, -0.6968522071838379, -0.24559761583805084, -1.5744085311889648, -0.4367493987083435, -0.7589775919914246, 1.0017472505569458, -1.0004783868789673, -0.6274225115776062, -0.4153216779232025, -0.2848585844039917, -0.4037826955318451, -0.20256422460079193, -0.10414652526378632, -0.10756659507751465, -1.3069819211959839, -0.9166663885116577, 0.8120843172073364, -0.280674546957016, -1.0496256351470947, -0.7779460549354553, -0.578204333782196, -1.8225231170654297, 0.2945680320262909, -1.2798384428024292, 0.3768779933452606, 0.088563933968544, -1.0579149723052979, -1.2116189002990723, 1.6536277532577515, -0.8189451098442078, 0.8465924859046936, 0.029197324067354202, 0.016544261947274208, 1.7358355522155762, -0.016974816098809242, -0.6828070282936096, -1.0779612064361572, 0.5210530161857605, -0.7115615010261536, -0.9720169305801392, 0.03440411388874054, -1.0747685432434082, 0.05883829668164253, -1.0384831428527832, -2.5096704959869385, 0.7497062087059021, -0.600033164024353, -0.4898940920829773, 0.7139356136322021, -0.1744285672903061, -2.2147202491760254, 0.6827930212020874, 1.6472489833831787, 1.1032824516296387, 2.2136096954345703, 0.5003492832183838, -0.7573933601379395, -0.580712080001831, -0.10648852586746216, 1.5356909036636353, -0.8478933572769165, 0.9212251305580139, -0.4336564838886261, 0.8937962651252747, -0.1480083465576172, 1.459311842918396, -0.18089833855628967, 0.6450970768928528, -0.30719324946403503, 0.11225499957799911, -2.3618743419647217, -0.06157557666301727, 0.22893649339675903, 1.2493975162506104, -0.04693455994129181, 0.19014720618724823, 0.6065824627876282, 0.43751972913742065, 1.018289566040039, 0.27854788303375244, -0.7709009647369385, 0.2803884446620941, -1.7307075262069702, -0.24164019525051117, 1.108078956604004, -0.17126618325710297, -0.3368704915046692, -0.6982493996620178, -0.5532238483428955, 0.9974921941757202, 0.688835620880127, 0.3316625654697418, 0.5222712159156799, -1.0973920822143555, 0.5847430229187012, 1.8610135316848755, 1.2838841676712036, -0.48175686597824097, 1.6005269289016724, -0.7697511911392212, -3.9247217178344727, -0.15138471126556396, -0.8704191446304321, -0.2342561036348343, 0.8729218244552612, -0.15310567617416382, -2.7152769565582275, 0.25390923023223877, -0.26303794980049133, -0.4301000237464905, -0.4921099543571472, -0.5517998337745667, -0.4542214274406433, -0.7885878086090088, -1.0913827419281006, 0.15942001342773438, 0.2952488660812378, -1.0123436450958252, 0.5286434292793274, -1.2714706659317017], [0.8637117743492126, -0.5634410381317139, -0.7118630409240723, -0.8965769410133362, -0.46061471104621887, -0.430079847574234, -0.3293941617012024, 0.005914445035159588, -0.912467896938324, 1.1356706619262695, 1.712143898010254, -0.42497676610946655, -2.272965431213379, 0.2719370126724243, -0.33488860726356506, -0.13482195138931274, -0.7205213308334351, 1.053147315979004, 1.2018100023269653, -1.4155919551849365, -1.1790680885314941, -1.4642813205718994, -0.3518093228340149, -1.6883236169815063, -1.1407448053359985, 0.3482314944267273, -0.5011566877365112, 1.0466444492340088, -0.07891485095024109, -0.06369957327842712, -2.2511532306671143, 3.329529047012329, 0.22408562898635864, -0.002767466474324465, -2.182992935180664, -1.3481123447418213, -0.9565910696983337, -2.283299446105957, 0.11663536727428436, 2.3886008262634277, 0.4851551055908203, -1.7194551229476929, 0.6609434485435486, -1.7500969171524048, 0.3783549964427948, -0.607964277267456, -0.3417111039161682, -1.8533164262771606, -1.8492417335510254, 0.1259000301361084, -0.5711790323257446, 0.6661249995231628, -0.7377358675003052, -1.3008923530578613, 1.4258506298065186, 1.136009931564331, 1.0543357133865356, 0.5447662472724915, -0.8526400923728943, -0.16108274459838867, -0.5783483982086182, 1.5533703565597534, -1.074468731880188, -0.007121883798390627, -1.2242845296859741, -0.09486021846532822, 0.27347514033317566, -0.6002178192138672, 0.29831111431121826, -0.0308800358325243, 1.1739604473114014, -0.3362771272659302, -0.5219237804412842, -0.257226824760437, -0.5058392882347107, -0.5356073975563049, 1.0104069709777832, 0.6832772493362427, -0.9999386072158813, -0.35106539726257324, -0.5226361155509949, -0.8197982311248779, -0.8250935077667236, -0.3935721218585968, 1.969791293144226, -0.05180284380912781, 0.5909028649330139, 0.5938344597816467, 0.09828513860702515, 0.3105546236038208, -1.1557732820510864, 0.9245538711547852, 1.3001450300216675, -0.34048354625701904, 0.5386307239532471, 2.5842843055725098, 0.4027649164199829, -1.1105215549468994, -0.40276822447776794, 0.11768028140068054, 0.7264045476913452, 1.3006998300552368, 0.9851318597793579, 0.21894784271717072, 0.5079259276390076, 0.9834497570991516, 1.8524080514907837, -0.36262595653533936, 0.8732903599739075, -0.7393267154693604, -0.4290591776371002, -1.5631682872772217, 0.5913392901420593, -0.059820957481861115, 0.8530718088150024, -1.1373234987258911, -0.9181964993476868, -1.204859733581543, -0.5683965682983398, -1.2073324918746948, 1.1669459342956543, 1.0451387166976929, 0.14147530496120453, -1.0373187065124512, -0.2709278464317322, -0.36675626039505005, 0.31534910202026367, 0.7904052138328552], [-0.6240403056144714, -0.21046486496925354, 0.506129264831543, -0.5058980584144592, -0.22661340236663818, 0.9477885365486145, 1.6784522533416748, -0.15853212773799896, 0.4190511405467987, 0.7164610028266907, -0.7623429894447327, -1.430346131324768, 2.1662728786468506, -0.43714067339897156, -1.7335309982299805, 1.3743127584457397, -0.3102155029773712, -0.12959757447242737, 1.502623200416565, 3.2647461891174316, 0.4430169463157654, -0.9346495270729065, -0.8134276270866394, 0.05305001139640808, -0.6151158213615417, 0.9810833930969238, -0.01041326206177473, 0.9993608593940735, -0.8941992521286011, -0.5993123054504395, 1.1138112545013428, -0.5215204954147339, 1.342118740081787, 0.45359745621681213, 1.3332687616348267, 1.134613275527954, 0.26296165585517883, -0.13233141601085663, 0.8038467168807983, 1.3379756212234497, 0.7356703877449036, -0.4525506794452667, 0.5162137746810913, -0.3820311725139618, -0.5576781630516052, 0.6536092758178711, -0.05208746716380119, -0.7801228761672974, -0.3529086112976074, -0.4970950186252594, 1.5255229473114014, 0.5453003644943237, -0.8449046611785889, -0.3060430884361267, -0.04642930254340172, 1.9637900590896606, -0.4942515790462494, 0.32484668493270874, 0.6375439763069153, -1.2654556035995483, -0.7936863303184509, -1.0977979898452759, -0.3130068778991699, 1.3718267679214478, 0.21159829199314117, -1.3120516538619995, 0.7222775816917419, 0.06451196223497391, 1.1237181425094604, -2.0377180576324463, 0.9324113130569458, -0.3162487745285034, -0.3030637502670288, 1.1795769929885864, -1.3766276836395264, -1.1547726392745972, -0.6822791695594788, -1.3995827436447144, 0.39439719915390015, 0.004186989273875952, 0.10737341642379761, 1.2560744285583496, 0.7411314845085144, -0.6156545281410217, -0.1576671451330185, 1.5256829261779785, 0.5243778824806213, 1.0766962766647339, 0.15493685007095337, 0.5877522230148315, 0.9231917262077332, -0.08368825167417526, 0.8431071639060974, -1.1260535717010498, -0.23625265061855316, 0.08953883498907089, -0.896342933177948, 0.13797485828399658, -1.9033156633377075, 0.08435113728046417, -0.7731508016586304, -0.23814095556735992, 0.9136863946914673, 0.8599515557289124, -0.35107192397117615, 0.4565178155899048, -0.8762264251708984, -1.0724364519119263, -0.5950224995613098, -0.3740074336528778, 1.4826200008392334, -0.5845832228660583, 0.894855260848999, 0.2783845365047455, 0.9702607989311218, -0.2547272741794586, 1.183251976966858, 0.7914993762969971, 0.646220326423645, -1.0085662603378296, 1.0094342231750488, -0.030919227749109268, -0.49041053652763367, 1.4839465618133545, 0.24539314210414886, -0.5368548631668091, -0.9499886631965637, -0.518910825252533], [-0.5456423163414001, 1.664440393447876, 0.18414513766765594, -1.15936279296875, -0.15124815702438354, 0.19630886614322662, 1.6233068704605103, -1.0133332014083862, 0.30007433891296387, -1.3340892791748047, 0.6178898811340332, -0.8679494261741638, 0.8290774822235107, -1.646986722946167, -1.8474228382110596, -0.7044829726219177, -2.4323368072509766, 0.4816753566265106, -0.5438736081123352, 0.16950149834156036, 1.8021730184555054, 1.525212287902832, 1.3762903213500977, -0.5410506129264832, 0.21500809490680695, -0.45238420367240906, -0.23541219532489777, 0.6723018884658813, 0.053793858736753464, 0.593540370464325, 1.2419288158416748, -0.16983141005039215, 0.7072165608406067, 1.0545965433120728, -0.014377167448401451, 0.09373750537633896, -0.2641269266605377, -0.5929582715034485, -1.0684258937835693, -3.4635965824127197, 2.273430109024048, -0.21936579048633575, -1.0917625427246094, -1.8108793497085571, 0.9652503728866577, -0.7984970211982727, 0.005601434502750635, 1.9869956970214844, -0.2608696222305298, -1.4879659414291382, 1.2108043432235718, 0.3618794083595276, 0.8107913136482239, -0.6798706650733948, 1.3464895486831665, -0.9795598387718201, 2.4012598991394043, -0.4072081446647644, 0.6366130709648132, 0.20190276205539703, 1.2705717086791992, -0.09897895902395248, -1.1267164945602417, 0.14316390454769135, 0.7071601152420044, -0.9566690325737, -0.3313392400741577, 0.46449652314186096, 1.1113674640655518, -0.26410073041915894, -0.9400784373283386, -0.14537809789180756, -0.1596907079219818, 1.4025517702102661, -1.5100775957107544, -0.6495106816291809, -1.9694255590438843, 0.5250283479690552, 0.5269120335578918, -1.4055891036987305, 1.1810259819030762, 0.5058395266532898, 0.14658480882644653, -2.1117124557495117, 0.0793621763586998, -1.2892520427703857, -1.1979520320892334, 1.8735630512237549, 0.24761459231376648, 2.681715726852417, -0.8115295767784119, -0.012980441562831402, 0.5102389454841614, -0.24995851516723633, 1.4486833810806274, -1.0779750347137451, 0.5498847961425781, -0.11566606909036636, 0.8533449769020081, -0.7199779748916626, -0.4491797089576721, 0.012058485299348831, 0.5256789922714233, -0.7260875701904297, -0.46777376532554626, -0.8842552304267883, -0.6587981581687927, 0.381300687789917, -0.0004010849806945771, -0.2030758261680603, 0.9848193526268005, 0.18255172669887543, -1.798806071281433, -0.6350245475769043, 0.16799397766590118, 0.042230356484651566, -0.2703937292098999, -0.3656681776046753, -0.8336976766586304, -0.3362385332584381, 0.13804373145103455, -0.09303615242242813, -0.8342670202255249, 2.526121139526367, 0.12191122025251389, 0.9596279859542847, 0.7496444582939148, -1.2054260969161987], [0.9151011109352112, -0.4507705271244049, 0.28476640582084656, -0.21856696903705597, -1.498531460762024, -1.8956698179244995, -0.831644594669342, -0.053299978375434875, -0.20749616622924805, 1.534379243850708, -1.0371345281600952, 0.4291808307170868, 0.046482473611831665, -1.92323899269104, 0.27940264344215393, -1.24164879322052, -1.2828747034072876, 2.8607137203216553, -0.04323940724134445, 0.5999277234077454, 0.24810190498828888, -1.2830978631973267, -1.3558896780014038, 1.1551356315612793, 0.9624594449996948, -1.5707799196243286, 0.6163813471794128, -0.31922784447669983, -0.11420751363039017, -0.34434226155281067, -0.04691753163933754, -0.8653374910354614, 1.0467036962509155, -0.2550733685493469, -2.023949146270752, -0.7417131662368774, 1.4850616455078125, 1.0973914861679077, 0.6900307536125183, 0.7154483795166016, 1.1172693967819214, -0.008911870419979095, -0.812774121761322, -0.4121258556842804, -0.3697149455547333, 0.4879123866558075, 0.7633127570152283, 0.04518193006515503, -0.703368604183197, -0.5314984917640686, -0.8964125514030457, -1.6091485023498535, 2.5601158142089844, -2.586632013320923, 0.3696538507938385, 0.011868314817547798, -0.619381308555603, -2.7776732444763184, 1.3315976858139038, -0.8900500535964966, -0.5141595602035522, -0.12061765789985657, 2.032414197921753, 1.1510990858078003, 1.5974340438842773, 0.4786715805530548, -0.7297386527061462, -0.4941837191581726, 0.6442826390266418, -0.2364269345998764, 0.000923464831430465, 0.6419551372528076, -3.0925772190093994, 1.5446470975875854, -0.23873655498027802, -0.9829468131065369, -0.5911034345626831, 1.485723614692688, 0.5851049423217773, -0.317557692527771, -1.1346113681793213, -0.3437970280647278, 0.70928555727005, -0.36125198006629944, -0.07560665905475616, -0.34381020069122314, -0.36183232069015503, 1.7364853620529175, -0.24735744297504425, -2.418694496154785, -0.6857518553733826, -1.472087025642395, 0.5208050012588501, 0.6544174551963806, 1.212244987487793, -0.2027435153722763, 0.08879412710666656, -1.023587942123413, 0.7525932192802429, -1.4129269123077393, -0.5796654224395752, 1.0493056774139404, -0.1986919343471527, -1.497876524925232, 1.8926483392715454, 0.24006371200084686, -0.1125931590795517, -1.5562156438827515, -0.08590684831142426, -0.2005160003900528, 0.9568389654159546, 0.5708240866661072, -0.41442182660102844, 1.2815991640090942, -2.124220609664917, -1.0071121454238892, -0.4531465470790863, 0.48342350125312805, 0.31144464015960693, -1.5463508367538452, -0.524411141872406, 1.228169560432434, -1.943964958190918, -1.301527500152588, -0.5927699208259583, 0.23046383261680603, 1.0449440479278564, 1.2567663192749023], [-0.41605931520462036, 0.1166289672255516, 0.3473359942436218, 0.5338017344474792, 0.0003049895167350769, -0.29573437571525574, -0.7732028961181641, -0.7919755578041077, 0.43211376667022705, -0.5825009346008301, 0.36675912141799927, 0.8816836476325989, -1.9798377752304077, 0.45507410168647766, -1.2155059576034546, -0.29405149817466736, 1.8471437692642212, 1.2199167013168335, 0.2861855626106262, -1.141546368598938, 0.8434405326843262, 0.4183465242385864, -0.7846126556396484, -2.376236915588379, 0.7472097277641296, 0.611394464969635, 0.6587230563163757, 0.08632223308086395, 0.08617702126502991, -1.063872218132019, -1.0399950742721558, 2.3672592639923096, -0.7647821307182312, 2.041008949279785, -1.2578693628311157, -0.2751697599887848, -0.367464154958725, 0.09076828509569168, -1.1312134265899658, 0.4170667827129364, 0.5328802466392517, 0.16909712553024292, 0.8006576895713806, -1.0319854021072388, 1.0246686935424805, -0.253523051738739, -0.9053620100021362, -0.07096538692712784, 0.01484595239162445, -0.3276137709617615, -0.7173085808753967, 0.8374667167663574, -0.8999177813529968, -0.8077486157417297, 1.0465490818023682, -1.0720570087432861, 0.9060636162757874, 0.9237143993377686, 1.390122413635254, -1.3472356796264648, 0.5306243300437927, -0.3325846791267395, 0.3707636892795563, 0.8563594222068787, 1.0903509855270386, 1.5496175289154053, -1.231410026550293, 0.6126658320426941, 0.16637545824050903, -1.3258239030838013, 2.9105899333953857, 0.42890477180480957, -1.9545527696609497, -0.56484055519104, 0.5226293206214905, -0.5901474952697754, 0.3224203288555145, -0.3486742675304413, -0.8287733197212219, -1.1824337244033813, 0.12735724449157715, 0.4530867636203766, 0.35775071382522583, 1.2681093215942383, 0.07015549391508102, 1.7108184099197388, 0.08601851016283035, 0.1119294986128807, 0.0279325433075428, -0.8242200016975403, 0.5790569186210632, -0.5788823366165161, 0.8258219361305237, 0.532162070274353, -0.4187465012073517, -0.6146209239959717, 0.20172131061553955, -0.8503963947296143, -0.4571817219257355, -1.137848973274231, -1.9438148736953735, -0.7668781280517578, -2.3547451496124268, -0.1892760992050171, -0.8214132189750671, -0.11638624966144562, 1.2511576414108276, 1.154828429222107, 1.9124619960784912, -0.6478136777877808, 0.20135030150413513, 0.20971748232841492, -1.1585807800292969, -0.12453311681747437, -1.329304814338684, 2.7970805168151855, -0.38359805941581726, 0.3000757098197937, 0.13232913613319397, -1.5300015211105347, -0.43958204984664917, -0.9610115885734558, -0.24571681022644043, -1.4973453283309937, 0.5823794007301331, 0.17657923698425293, -0.06434978544712067, -1.103550672531128], [-0.8700684309005737, 1.414958119392395, -1.7421281337738037, 0.34483614563941956, 0.641846776008606, 1.897504448890686, 0.23602497577667236, -1.775092363357544, 0.07395678013563156, 0.05102606117725372, 1.640903353691101, -0.3491530120372772, -0.9047980308532715, 0.353562593460083, -0.04435756430029869, -1.2726250886917114, -0.5746304392814636, 1.1265305280685425, -0.7380877733230591, 0.8157268166542053, 0.8547766804695129, 2.035759925842285, -0.7718938589096069, -1.8615312576293945, 1.5504566431045532, 0.2268228381872177, -3.2531850337982178, -0.9820020794868469, -0.20509563386440277, 0.8218915462493896, -0.23817428946495056, -0.5658931732177734, 2.014139175415039, -1.9083672761917114, 0.6635690331459045, 0.05693851411342621, -0.2930133640766144, 0.5767407417297363, -1.6607897281646729, 0.8867477178573608, -1.884169101715088, -0.10914265364408493, -1.6496165990829468, 1.080363154411316, -0.0619865357875824, 2.5578672885894775, 1.4041805267333984, 0.49619027972221375, -0.6229349970817566, 1.3081520795822144, 1.4874520301818848, 0.16287024319171906, 1.6210103034973145, 0.7771556377410889, 1.3534996509552002, -0.5250604152679443, -0.887333869934082, -1.1659045219421387, 1.5405486822128296, -0.4032413959503174, -2.0734195709228516, -2.4348902702331543, 0.6272187829017639, -0.15010935068130493, -1.3690636157989502, -0.7115556597709656, 2.877830982208252, 0.029496589675545692, -1.058611512184143, -0.40473759174346924, 0.7517854571342468, 1.0328696966171265, 1.4055291414260864, -0.24895671010017395, -0.7134857177734375, 0.3548282980918884, 0.678120493888855, 1.3625553846359253, 0.13949602842330933, -0.11863134056329727, 0.9383296966552734, 0.45447441935539246, 0.7966874837875366, -1.710526943206787, -0.6316858530044556, 0.01695900224149227, -1.812772274017334, -0.5219054222106934, 0.7836413383483887, 0.6277738809585571, -0.982420027256012, -0.49877703189849854, 1.4568181037902832, 0.07881154119968414, 2.301586151123047, 0.1466686725616455, -1.0003700256347656, 1.885169506072998, 0.7817040085792542, -0.07373326271772385, -1.0204498767852783, -0.9028832316398621, -2.067718744277954, 0.8779489994049072, -0.46375772356987, 1.2162052392959595, 1.2598811388015747, 0.7665552496910095, 1.2293837070465088, 0.16012173891067505, 0.13190755248069763, 1.0924437046051025, 0.8983631730079651, 1.3410283327102661, -0.5609164237976074, -0.5917702913284302, -2.1357529163360596, 0.15908363461494446, 0.8670307993888855, -0.11991851031780243, -0.9692139029502869, 1.3372766971588135, -0.5910388231277466, -0.7789045572280884, -0.4318172037601471, -0.5220358967781067, 0.5210968255996704, 0.6261730790138245], [1.578050971031189, 1.6051087379455566, 0.5234881043434143, -0.9347590208053589, 0.17590753734111786, -0.7165223956108093, -0.6265032291412354, 0.12357635796070099, -1.1541298627853394, -0.42770010232925415, 0.01627613790333271, -1.088800311088562, -1.171532154083252, 0.007882464677095413, 1.1460704803466797, 1.0916986465454102, 0.6069109439849854, -0.3818344175815582, 0.8863186836242676, -0.13473156094551086, 0.2495027482509613, -0.7762143015861511, -0.8405139446258545, 1.2385954856872559, -0.28461185097694397, 0.057260047644376755, 0.6696600914001465, 0.23301294445991516, 0.11147332191467285, 0.867121160030365, 0.8812742233276367, 0.8986640572547913, -0.26513490080833435, 0.3042724132537842, 0.4692017734050751, -0.22961527109146118, 0.46350178122520447, 0.22040021419525146, -0.22099588811397552, -0.07252881675958633, -1.0558375120162964, -0.5590139627456665, -1.101830244064331, 0.19727247953414917, -0.9174159169197083, -0.3723101019859314, 0.1324959248304367, 0.19461528956890106, -0.5013344287872314, 0.6015561819076538, -1.6730642318725586, -0.4433470666408539, -1.3263334035873413, 0.06613828986883163, 0.7553791403770447, -0.2612289488315582, 0.37618133425712585, -0.7181805372238159, -1.4229013919830322, 1.1883633136749268, -1.2548654079437256, -1.7451035976409912, -0.4243948459625244, 1.3670108318328857, -1.2830767631530762, 0.690121054649353, -1.3719334602355957, 0.26020193099975586, 1.0222125053405762, 1.2919930219650269, -0.37457627058029175, -2.2802021503448486, 0.29075291752815247, 2.3970844745635986, -0.3413233458995819, -0.3124413788318634, 0.2458219975233078, 0.40513619780540466, -0.13961470127105713, -0.4606243371963501, -0.6407105326652527, 1.7302050590515137, 0.7327360510826111, 0.6725483536720276, -1.1168687343597412, -0.9165241122245789, -0.33824390172958374, 0.47018539905548096, 2.0114939212799072, -1.0927300453186035, -0.9369211196899414, -0.7085698246955872, -0.15910427272319794, -0.1473979353904724, 0.19999493658542633, -0.5675011873245239, 1.071317195892334, -1.033430576324463, 1.0056437253952026, 0.2541760206222534, -0.5823671221733093, 0.02255222760140896, -0.3843916356563568, -1.9682047367095947, 1.464282751083374, 1.890425443649292, 0.47070276737213135, -1.0539088249206543, 0.7455453872680664, 0.838848888874054, 0.6279107928276062, 0.24393314123153687, -0.9753575921058655, -0.559710681438446, 0.38932573795318604, -0.7537139058113098, 0.4571392238140106, 2.2766692638397217, 0.8702280521392822, 0.7793641686439514, 1.7332197427749634, -0.153607577085495, -0.6786016225814819, 1.008318543434143, 2.0123777389526367, -0.2614864408969879, -0.5157981514930725, 1.4810456037521362], [0.42487794160842896, -0.36494699120521545, 0.2554110586643219, -0.2440936118364334, 0.08753982931375504, 1.1873588562011719, 0.7284203171730042, -0.6394630074501038, 0.5077733993530273, 0.36647555232048035, 0.18276600539684296, -0.858853280544281, 0.27210256457328796, -0.734241783618927, -0.7745010256767273, 0.4029766023159027, -0.9391124844551086, 0.8525406122207642, 1.4999375343322754, 0.23543186485767365, -0.666694700717926, 0.34198713302612305, 0.4516129195690155, 0.1522861123085022, -0.4187016785144806, -1.5355517864227295, 0.16770251095294952, -0.6418178677558899, 0.568936288356781, 1.251255989074707, 0.18764261901378632, -0.8191140294075012, 0.09839886426925659, 1.0151138305664062, -1.2648890018463135, -0.8246701955795288, 0.4327200949192047, 0.34215208888053894, -1.1505106687545776, -0.6086514592170715, -0.6804804801940918, 0.4887107312679291, 1.5348292589187622, 0.08467049896717072, -0.3756607174873352, 0.07414396107196808, 1.0520167350769043, 0.5987600088119507, -0.9397822022438049, 1.4985859394073486, -0.02915285713970661, -0.5196946263313293, 0.18079189956188202, 0.18747113645076752, -0.15985791385173798, 0.5115857124328613, -0.18751321732997894, -1.0887738466262817, -0.6898888349533081, 0.9696062803268433, -0.8196423649787903, 0.9037543535232544, 1.405297040939331, 0.7294726371765137, 0.26200398802757263, -1.852731704711914, -0.8700895309448242, -1.3887507915496826, -0.8977693319320679, 0.12087226659059525, 2.217658042907715, 0.019570892676711082, 1.85797917842865, -0.26142606139183044, -0.1631600707769394, -0.0383438915014267, -1.2990593910217285, 0.832457423210144, -1.272341251373291, 1.6922401189804077, 1.012807846069336, 0.7253071665763855, -0.7354503273963928, -0.9093625545501709, -0.20078283548355103, 0.8941116333007812, 0.5969080328941345, 0.8545751571655273, 1.3457043170928955, 0.005054289009422064, -1.2551817893981934, -1.668062686920166, 0.8284473419189453, 1.0832875967025757, 1.3616857528686523, 1.3116401433944702, 0.5421398282051086, 0.05321048945188522, -0.985689103603363, 1.139991283416748, -0.1947752684354782, -1.5522524118423462, -0.11843584477901459, 1.0709967613220215, 0.5300486087799072, -0.950320303440094, -0.5544983744621277, -1.1818455457687378, -0.5652075409889221, 0.2080865353345871, 0.6970005631446838, -0.13991230726242065, -2.000781536102295, -0.6408918499946594, -0.024910707026720047, 0.6988942623138428, 1.1738587617874146, -0.4486656188964844, 1.1608924865722656, 0.27762600779533386, -0.30706870555877686, -0.9592968225479126, -1.2312583923339844, -0.9473833441734314, -0.7374189496040344, -0.10250674933195114, 0.18418480455875397, -1.871814489364624], [0.2468363344669342, 0.053279612213373184, 0.340225487947464, 0.11604540795087814, -0.14377640187740326, -0.5176903605461121, 2.604588031768799, -0.19661174714565277, -1.1596204042434692, -0.1627694070339203, -0.09509105980396271, 0.574900209903717, 0.01885763742029667, -0.143661767244339, 0.8140203952789307, 0.3465723693370819, 0.9721576571464539, -0.5621687173843384, 0.3689563274383545, -1.0076408386230469, 0.7074407935142517, -0.27799174189567566, -0.07982685416936874, 1.3957685232162476, 0.0033457118552178144, -2.099541664123535, 0.6754762530326843, 0.08692286163568497, -2.1430957317352295, -0.0754108875989914, 1.1362502574920654, 0.12957507371902466, 2.3268415927886963, -1.0339020490646362, -0.07965832203626633, -1.4117867946624756, 1.0608283281326294, -0.7088497877120972, 2.289085626602173, -0.7493299245834351, 2.76381516456604, -2.558434247970581, -0.24235810339450836, 0.3202678859233856, 1.5863808393478394, -1.4138782024383545, -1.2353389263153076, -0.19029375910758972, -1.4370626211166382, 0.13120035827159882, -1.2667745351791382, 0.8437095284461975, 2.150347948074341, -0.43610554933547974, 0.8600219488143921, 0.20567956566810608, -0.18463453650474548, -0.9064985513687134, 0.808921217918396, -0.467656672000885, -0.5648214817047119, 0.460563063621521, -0.055424559861421585, -0.12509149312973022, -0.792807936668396, 0.5243627429008484, -0.33882805705070496, -0.18933677673339844, -1.1684192419052124, -0.960971474647522, 0.3642007112503052, -0.6664093732833862, -0.7473752498626709, -0.4614355266094208, 0.08760245144367218, 0.2068072259426117, 0.17940647900104523, 1.104819655418396, 0.06832817196846008, 1.0294325351715088, 2.214988946914673, 0.07433348894119263, -1.3591660261154175, 0.4495828449726105, 0.7902487516403198, 1.152513861656189, 1.0211812257766724, -0.6247653365135193, -0.8762954473495483, 0.0038310543168336153, 0.956236720085144, -0.3203750252723694, 0.0649946928024292, 0.5033581852912903, 0.9656445980072021, -1.0400166511535645, 0.7276548147201538, -0.6220586895942688, -0.23170402646064758, 0.25978201627731323, -0.7857716083526611, 1.0012295246124268, 0.9579629898071289, 0.11547505110502243, 1.971004843711853, 0.7990183234214783, -0.7391692996025085, 0.1482321321964264, 0.09031357616186142, -1.3297768831253052, 0.17222513258457184, 0.2394295185804367, 0.8633263111114502, 0.9784157872200012, 1.871938943862915, -0.21759437024593353, -0.29346728324890137, 0.004811150021851063, -1.3130173683166504, 0.9117283821105957, 1.2240289449691772, 0.23963628709316254, 0.39537352323532104, -1.2940822839736938, -0.9666228294372559, 0.7913864254951477, -2.1527099609375, -0.03443404659628868], [-0.6274388432502747, -1.607143521308899, -1.322769045829773, -0.12560057640075684, -1.0127676725387573, -0.4232165217399597, -2.9976038932800293, 0.08336775749921799, 0.5133656859397888, -0.616707980632782, -0.2324158251285553, -0.8330416679382324, 1.2218573093414307, -0.17342163622379303, 0.4037173092365265, -0.5100600123405457, 1.299536108970642, 1.6221323013305664, -0.5064210891723633, 1.1988037824630737, -1.0021954774856567, -0.30953529477119446, -0.396138459444046, 0.06553211063146591, 0.5765855312347412, 0.03711535781621933, -0.09978698194026947, 1.9238735437393188, -0.9287847280502319, -0.18320342898368835, 0.09578472375869751, -1.225650668144226, -0.7525392770767212, 0.42094528675079346, -0.08413517475128174, -0.9977922439575195, -0.12540696561336517, 0.513420045375824, 0.9086066484451294, -0.3909425735473633, 0.6197791695594788, 1.3737956285476685, -2.4594838619232178, -0.3663850426673889, -0.6232901215553284, 1.3918485641479492, 0.6340073347091675, 0.1981520652770996, -0.062224261462688446, -0.360385000705719, 0.623247504234314, -1.005469560623169, 1.3782317638397217, 1.0163018703460693, -2.884413957595825, 0.2991637885570526, 1.094923496246338, -0.34394344687461853, -0.22863197326660156, 0.15398874878883362, -1.09946870803833, 0.08906421065330505, -1.5766605138778687, -1.1273019313812256, -0.6799819469451904, 0.3197017014026642, 0.08545680344104767, 0.06396964192390442, 0.7449778914451599, 0.6112513542175293, -0.5978949666023254, 0.5855894684791565, -0.13301865756511688, 0.22617775201797485, -1.809990406036377, 0.6918586492538452, 1.9598186016082764, 1.0725113153457642, -0.18665651977062225, -1.5780853033065796, 0.6216319799423218, 1.009446144104004, -0.6396331787109375, 0.028412021696567535, 0.51496821641922, 0.07439027726650238, -0.1112370491027832, -0.6400556564331055, -2.67345929145813, -1.600573182106018, -0.7600539922714233, 0.8742141723632812, -0.7445136308670044, -0.17444676160812378, 0.016978736966848373, -0.6275251507759094, -0.08340834826231003, 0.17466874420642853, 1.1087766885757446, -1.8347479104995728, 0.32035714387893677, 0.15218321979045868, -0.3449096381664276, 0.1840960681438446, -0.3519551753997803, 0.9806450009346008, -0.7997519373893738, 0.3430997133255005, -0.1674395352602005, -0.2628692090511322, 0.051312170922756195, -0.5044036507606506, -0.5421342253684998, 0.08966793864965439, -1.9492148160934448, -0.053603824228048325, 0.18931598961353302, 1.1181666851043701, -0.3922339677810669, -0.3115570843219757, 0.05107862502336502, 0.0332900695502758, 0.7466110587120056, 0.7207363843917847, 1.1763014793395996, -0.3928396701812744, -0.5130630731582642, 1.126436471939087], [0.1538511961698532, -0.08923507481813431, 1.2405887842178345, -0.9078108072280884, 1.102328896522522, 1.4615198373794556, -0.9258290529251099, -0.9008080363273621, -1.1094787120819092, -0.256911963224411, -1.6664018630981445, -1.0711617469787598, 0.022853797301650047, 0.8607597351074219, 2.4019553661346436, -0.7552635669708252, 0.04849710687994957, 1.5148708820343018, -1.396424412727356, -0.982775866985321, 1.4423340559005737, 1.0844758749008179, 2.113861560821533, 0.24939638376235962, 0.19714702665805817, -0.2878178358078003, -0.8604151606559753, -0.04576229304075241, 1.0836697816848755, -0.9385611414909363, -0.046193867921829224, -1.051875114440918, -1.1030402183532715, -1.4309215545654297, -1.4200356006622314, 0.6823433637619019, 1.0008955001831055, -2.683499336242676, 1.8436044454574585, 0.9449781179428101, 1.7765870094299316, 0.9076188206672668, -1.2240606546401978, 0.574205756187439, 1.5022422075271606, 0.03052734211087227, 0.5043678283691406, -1.2761099338531494, 0.671663224697113, 2.1003262996673584, 0.4451102316379547, -0.09213961660861969, -0.04693731293082237, 0.41140878200531006, 0.5179262757301331, 1.5812863111495972, -0.8392258286476135, -1.4527101516723633, -0.6911905407905579, 0.23420199751853943, 1.6996757984161377, 0.02410932630300522, 0.18648767471313477, 0.46302324533462524, -1.2534921169281006, 1.4480656385421753, 0.7451951503753662, -1.5081079006195068, 0.008269510231912136, 0.513023853302002, 1.165763258934021, -0.23810532689094543, 0.022920330986380577, -1.2585622072219849, -0.7422873377799988, -0.3216494917869568, -0.5105631351470947, -0.027670912444591522, -0.9230669140815735, 0.5463266372680664, 0.940696656703949, 0.6376075148582458, 0.14652837812900543, -0.9364750981330872, -2.186995029449463, -1.7472999095916748, -0.8785793781280518, 1.2085243463516235, -0.028984583914279938, -0.14241257309913635, -0.3450586795806885, 0.1918552815914154, 0.17208531498908997, 0.3878284692764282, 0.7859777808189392, -1.2194480895996094, -0.6534337997436523, 1.0864163637161255, -0.5758987069129944, -1.6564345359802246, 1.466372013092041, -1.856846809387207, 1.0099949836730957, 0.16263455152511597, 1.2937703132629395, 1.7225254774093628, -0.1576150357723236, 1.265484094619751, 1.7248133420944214, 1.3469831943511963, -0.8480134606361389, 0.10868668556213379, 0.396572470664978, 0.6514348983764648, 1.7985442876815796, -0.5905788540840149, -0.1635681539773941, -0.15913039445877075, -1.0325429439544678, -0.17813092470169067, -0.10625213384628296, -1.1153271198272705, -1.383966326713562, 0.474170446395874, -0.4521418511867523, 0.6029205322265625, 0.49103474617004395, -0.20408949255943298], [-0.19370751082897186, 0.23358429968357086, -0.17112424969673157, 0.18496917188167572, -0.17442351579666138, -0.5387067794799805, 0.28234240412712097, -0.2980351448059082, 1.0942707061767578, -1.7060339450836182, 0.25971415638923645, -1.2494654655456543, -0.970350444316864, -0.46305352449417114, 0.8121052980422974, -0.8617339134216309, -1.0072755813598633, 1.6201307773590088, 0.15894749760627747, 0.11267604678869247, -0.48619183897972107, 1.9046097993850708, -0.09974169731140137, -1.428709626197815, 0.6336506009101868, 1.0290721654891968, -0.5351504683494568, -1.5952622890472412, -0.4425209164619446, 0.20918279886245728, -0.4082754850387573, -2.376133680343628, 0.09447740763425827, -0.46535640954971313, 1.4146976470947266, -1.1820992231369019, -0.3652912378311157, -1.3705819845199585, -0.8323559165000916, 0.32305386662483215, 0.30698251724243164, -0.38667675852775574, -0.2755919098854065, -0.2540944814682007, 1.1362240314483643, 0.04787631332874298, -0.7070040106773376, 0.551426887512207, 0.2572147250175476, -0.27968573570251465, 0.040938060730695724, 0.6279622912406921, -0.5126293897628784, 0.44050565361976624, 0.7406377792358398, 0.013285787776112556, -0.5403379797935486, -0.28143569827079773, 0.6001559495925903, 0.16059179604053497, -0.21284182369709015, 0.7848066687583923, -1.9146367311477661, -1.2424794435501099, 0.4904683530330658, -0.7303866147994995, 2.317798376083374, 0.26617059111595154, -0.7936609387397766, 0.7184363603591919, 2.2266249656677246, -0.07132332772016525, -0.17888502776622772, 1.883541464805603, 0.7757576107978821, -1.151917576789856, -0.2942371964454651, 0.6894807815551758, -0.832275927066803, -1.1181693077087402, -0.12920893728733063, 0.6063894033432007, 0.9947320818901062, 0.017657440155744553, -0.37301120162010193, 0.3180147707462311, 0.36281150579452515, -2.5637192726135254, -1.1283334493637085, 0.9566464424133301, -0.49152159690856934, 1.1333682537078857, -0.254434198141098, -0.004428253509104252, 0.3022451400756836, 0.11320552974939346, 1.486973524093628, -0.21649780869483948, -0.9258148670196533, -1.5746371746063232, 0.23858670890331268, -0.786963164806366, 1.0934388637542725, -1.0530840158462524, -0.7152805924415588, 0.39589351415634155, -0.13827110826969147, -0.8807269334793091, -2.706622838973999, -0.3467399477958679, 0.012378810904920101, -1.2653789520263672, -1.1183431148529053, -0.17234590649604797, -0.4029488265514374, 0.03627679497003555, 0.1360020786523819, 1.062670111656189, 0.4313783645629883, -0.9986879825592041, -0.6271594166755676, -0.4139935076236725, 0.08757008612155914, 0.6266525387763977, -0.9039835929870605, 1.5250122547149658, -2.077131986618042, 1.4266749620437622], [-1.4725033044815063, -0.37462398409843445, -1.096411943435669, -0.5354627966880798, 0.5833747386932373, -0.1344083994626999, -0.3090050518512726, -0.3671010434627533, 0.41609033942222595, 0.6744286417961121, -1.1253963708877563, -0.8819975852966309, 1.1096863746643066, 1.2881333827972412, 0.4505789279937744, 0.1651414930820465, -0.24860040843486786, 0.5694010853767395, 1.942913293838501, -0.16273869574069977, -1.2190691232681274, -0.8283613324165344, 1.148108720779419, 0.32500165700912476, 0.309415340423584, -1.690868854522705, 0.4208385646343231, -1.1722095012664795, -0.9158534407615662, 0.9528030753135681, -1.4874286651611328, 0.4477635324001312, 0.9867525100708008, 0.975498378276825, -1.2750835418701172, -0.8028157353401184, -0.7453256249427795, 1.9670716524124146, -1.641740083694458, 2.0124552249908447, -1.1052390336990356, -0.9889119267463684, 0.21874235570430756, 0.04032205045223236, -1.2613219022750854, 0.40428227186203003, -0.27358394861221313, -0.18658632040023804, 0.5675622224807739, -1.1232277154922485, -1.2493664026260376, -0.2523592412471771, 0.20394927263259888, 0.45211875438690186, 0.6528593897819519, -0.38307297229766846, -0.7758510112762451, 0.4212236702442169, -0.8182919025421143, -1.09302818775177, 0.23314917087554932, -1.1059035062789917, -0.6562215685844421, -1.8674033880233765, -0.34538352489471436, -1.0893235206604004, -0.009790698997676373, 1.4483163356781006, 0.5403105616569519, -0.7605062127113342, 0.30814459919929504, -0.18983671069145203, -1.0277636051177979, 0.32590943574905396, -0.6687202453613281, -2.6450231075286865, -1.7993375062942505, 1.0301151275634766, 2.4634454250335693, 0.2754090428352356, 1.4143010377883911, -1.0565208196640015, 2.653944969177246, -0.40137919783592224, 1.5202168226242065, 3.5486831665039062, -0.008420588448643684, 0.13954676687717438, -0.19718517363071442, 0.4873703420162201, -0.3909350037574768, 0.45886898040771484, 1.0402592420578003, 1.3512942790985107, -0.4023953378200531, -0.6205596923828125, 1.5029454231262207, 0.2423701137304306, -1.4095420837402344, 1.9327489137649536, -1.097220540046692, -0.3011210560798645, -1.8484822511672974, 0.16189992427825928, -0.21399155259132385, -1.7484865188598633, -0.16904988884925842, 0.46477699279785156, -0.23156189918518066, 0.3847779333591461, 0.3556172251701355, 0.7437216639518738, -0.3291805684566498, -0.7799001336097717, 0.3082689046859741, -0.29521894454956055, -0.13369818031787872, -2.5390331745147705, -1.3961849212646484, -1.6925640106201172, 1.710188865661621, -1.8517471551895142, 0.77420574426651, 0.2906695008277893, 1.296507477760315, 0.3436649441719055, 0.053446605801582336, -2.5228216648101807], [-1.9198894500732422, 0.41455259919166565, -0.08474881947040558, -0.5403406023979187, -2.6622259616851807, -0.34002262353897095, 0.9128557443618774, 1.4575350284576416, -0.5325766801834106, -0.09757732599973679, -1.0526305437088013, -0.7059447765350342, 0.9979599714279175, -0.8373388648033142, -1.189668893814087, 0.10147963464260101, -0.04794107750058174, 1.733328104019165, -0.8232753276824951, -1.0254756212234497, -0.7799858450889587, -1.041075348854065, -1.1508830785751343, 0.5322564840316772, 0.7223126292228699, 0.25473839044570923, -2.7346134185791016, 0.4159649610519409, 0.3592749834060669, -0.9363369345664978, -0.16676132380962372, -0.14314545691013336, 0.34827378392219543, -0.0383874773979187, -0.22876767814159393, 0.9014831185340881, -0.9151932001113892, 0.7095649838447571, -1.3075066804885864, 0.8545470833778381, -0.53973388671875, -0.42114660143852234, 1.1729702949523926, 0.9484580755233765, -0.6199305653572083, 0.23942941427230835, 0.9597511291503906, -0.5009697079658508, 1.2366307973861694, 0.6737414002418518, -2.064798355102539, 1.7483634948730469, 1.121863842010498, -0.42758455872535706, -0.5242192149162292, -0.8222946524620056, -0.5871585607528687, 0.5494403839111328, -1.7947527170181274, -0.3595326542854309, 0.08908743411302567, -1.3001139163970947, 3.1843507289886475, -0.9222233891487122, 0.0688532292842865, -0.48282891511917114, -0.10982462018728256, 0.0015815083170309663, -0.10678955912590027, -0.8588021993637085, 0.6646711230278015, -0.17521430552005768, 0.37633830308914185, 1.4153947830200195, -0.9758466482162476, 0.8724913597106934, -0.48653310537338257, -0.8023619055747986, -0.2880612015724182, 0.5171684622764587, -1.0199129581451416, 1.7331669330596924, -2.190643787384033, -0.24668185412883759, -0.4115430414676666, -0.3335725963115692, -0.42846035957336426, 1.2138566970825195, -0.035702791064977646, -0.7678945660591125, 1.1618099212646484, 0.4927469491958618, 0.5047468543052673, 0.7532986998558044, -1.9169268608093262, 0.9444218873977661, 0.39918920397758484, -0.3379903733730316, -1.397332787513733, -2.7244696617126465, -0.11369507014751434, 0.09031987935304642, -0.7486540675163269, 0.8737185597419739, -1.5267307758331299, -0.043041981756687164, 1.177108645439148, -1.3634779453277588, 0.5628495812416077, 0.4512535631656647, -1.812340497970581, -0.7181523442268372, 0.033530063927173615, 0.8214390277862549, 0.2548167407512665, 0.46713730692863464, -0.009057154878973961, -0.22613726556301117, 0.43913599848747253, -0.6512491703033447, -0.35767173767089844, -1.088140606880188, -0.20288361608982086, -0.3982156813144684, -1.3918957710266113, 0.2802954316139221, 0.9418576955795288, -1.2804228067398071], [-0.1901327669620514, -0.4488738775253296, -0.4994257688522339, 0.054775334894657135, -0.1421598345041275, 1.1736565828323364, 1.0723015069961548, -0.17721280455589294, 1.0014954805374146, 0.7474344372749329, -0.7675376534461975, 0.018732482567429543, 0.17295336723327637, 0.5734231472015381, -0.07012218236923218, 0.23900818824768066, -0.3719480037689209, 0.5308155417442322, -1.0612365007400513, -1.6903849840164185, -0.7530977725982666, -0.2785506844520569, -0.06006133183836937, -1.8621000051498413, -0.7457119226455688, 1.4050043821334839, -0.5745449066162109, 0.664769172668457, -0.20432308316230774, -0.21805046498775482, 0.33953073620796204, -0.11656280606985092, 1.716031789779663, -0.4303004741668701, -0.4990738034248352, 1.7614665031433105, -0.02742646262049675, 0.3850892186164856, -0.3579384982585907, -0.3327937424182892, -0.0692669227719307, -1.0454447269439697, -1.380238652229309, -1.7925125360488892, -0.573053240776062, -0.24074819684028625, -1.0637519359588623, 0.5364131331443787, -0.20063993334770203, -0.5171406269073486, -0.02243507094681263, -0.016806060448288918, -0.10342885553836823, 1.4587754011154175, 0.058678723871707916, -0.5694965720176697, 0.13122132420539856, 1.4475339651107788, 1.518528938293457, 0.7771109938621521, 0.5756694674491882, -0.6046982407569885, 0.9049955606460571, 0.29300111532211304, 0.8660764694213867, -0.29541170597076416, 1.7751322984695435, 0.29273128509521484, -0.4914402961730957, 0.7122462391853333, -0.028103407472372055, 0.5705562233924866, -0.799809455871582, -1.0889934301376343, 0.43637219071388245, 0.15233725309371948, 0.01680581457912922, -0.7650397419929504, -0.6740007996559143, -0.6061060428619385, 0.9448053240776062, -0.15485039353370667, -0.3324231505393982, -1.580260157585144, -0.052276913076639175, 0.7381782531738281, -0.48853155970573425, -3.0801141262054443, -1.258810043334961, -0.05554515868425369, -1.471347451210022, 1.1921918392181396, 1.390493631362915, 1.7976164817810059, -0.872565507888794, -0.8094500303268433, -0.2397756427526474, 0.7069688439369202, 0.027216799557209015, -0.36227113008499146, 1.841645359992981, -0.8394349217414856, -0.8474541902542114, 0.6854895949363708, 0.7074121236801147, -0.315998375415802, 0.0065093375742435455, -1.2843111753463745, -0.6105110049247742, -1.1953545808792114, 0.6640849113464355, 0.9566494226455688, 1.6843417882919312, -0.2518274486064911, 0.5737338662147522, 0.12577587366104126, -0.395834356546402, -0.5992955565452576, -0.8613819479942322, -0.6311213374137878, 1.9840584993362427, 0.2759529948234558, -0.14569726586341858, -0.029934272170066833, -1.2242200374603271, -0.9191611409187317, 1.0100444555282593, -2.0759778022766113], [0.9899914860725403, -1.1229652166366577, -1.9237157106399536, 1.0184125900268555, 0.2413368672132492, 0.6572151780128479, 0.2357618659734726, 0.19861610233783722, 0.2941279113292694, 0.7951178550720215, 1.0349668264389038, -0.5509276390075684, 0.5265527367591858, -0.4494320750236511, -1.2361745834350586, -0.45663148164749146, -0.48207786679267883, -1.395626425743103, -0.2804896831512451, 0.4796575605869293, -0.4079260528087616, -1.419130802154541, -0.3679109811782837, -2.185882568359375, 0.4111601412296295, -1.2212340831756592, -0.2571050524711609, -0.3442732095718384, -1.2458114624023438, 0.4070153534412384, -0.6078320741653442, -1.7731924057006836, 2.6946120262145996, -1.1544227600097656, 0.7269459366798401, 1.2295371294021606, -0.6644273996353149, 0.4206767678260803, -0.21320298314094543, 0.5025100111961365, -1.2080329656600952, 1.0817677974700928, 0.07872083783149719, 0.5431749224662781, -0.7358145713806152, 0.08198107033967972, 1.6370595693588257, 0.05977346748113632, -1.1999295949935913, -0.447884738445282, 0.2155490517616272, 1.0436211824417114, 0.9204865097999573, -0.6577793955802917, 0.312130331993103, 1.0179377794265747, -1.1546266078948975, 0.15958108007907867, 0.01616000384092331, -0.4071137607097626, -0.8082236647605896, 0.6349255442619324, -0.42777732014656067, 0.6200073957443237, -0.3061229884624481, -0.34937915205955505, 1.2443219423294067, 1.0679223537445068, 0.4781631827354431, -0.20528513193130493, -0.6973828673362732, -0.49688878655433655, 2.037721633911133, 0.5153260231018066, 0.9385684132575989, -0.5863085985183716, -0.41003721952438354, 0.4387998580932617, -1.384212613105774, -0.5722935795783997, 0.44365596771240234, 1.218387484550476, -0.9347249269485474, -0.18816393613815308, -0.6048218011856079, 0.02561378665268421, 0.6732500195503235, -0.26837167143821716, 1.098645567893982, 0.23587758839130402, 0.6403576731681824, 0.7490358948707581, 0.04961555823683739, -0.45706912875175476, -0.10279713571071625, 0.7336423397064209, -1.4922832250595093, -1.6190427541732788, -0.5110970735549927, 1.3534198999404907, -0.1177082508802414, -0.36590930819511414, -0.3020349442958832, -0.47165974974632263, 0.17362560331821442, 2.001039981842041, 2.000150680541992, 0.7420900464057922, -0.763991117477417, 0.7792625427246094, 1.010519027709961, -0.8540642261505127, 1.6764808893203735, -0.05807187780737877, 1.4210220575332642, 2.111083745956421, 1.4668095111846924, -1.4691847562789917, -0.5002274513244629, -0.48060286045074463, 0.2100522369146347, -0.7160753607749939, -0.7036962509155273, 0.4675616919994354, 0.663483738899231, 0.19234949350357056, -1.7130717039108276, -0.8285744190216064], [1.1856003999710083, -0.31456536054611206, 2.6405367851257324, -0.753820538520813, -2.3810195922851562, -0.20463448762893677, -1.8085733652114868, -0.2628451883792877, 0.5008989572525024, 1.018507480621338, -1.1533982753753662, -1.1066218614578247, -1.6602823734283447, 0.4020591378211975, 0.9314975142478943, -0.7480811476707458, 0.7422714829444885, 1.4489208459854126, -0.5592473745346069, 1.4163572788238525, -0.8440901041030884, 0.34532663226127625, 0.549530565738678, -0.22997204959392548, -0.3077322542667389, -1.3281666040420532, -0.3958607017993927, 1.5369749069213867, 0.9788640141487122, 0.5950571298599243, -1.7446836233139038, 1.3940097093582153, -1.375888466835022, 0.9433449506759644, 0.3546000123023987, 1.489395260810852, 1.1879456043243408, 1.3144725561141968, 0.6865386366844177, 0.42528393864631653, 0.9153549671173096, 0.18146905303001404, 0.806348443031311, -0.21534086763858795, -0.3255920708179474, -0.4858081340789795, 1.018334984779358, -0.353659063577652, 0.4413222372531891, -0.22380521893501282, -1.1877492666244507, 1.0264712572097778, 1.2149548530578613, 0.558421790599823, 0.7602254152297974, -1.414231300354004, 0.5049942135810852, -1.3207374811172485, 0.4619005024433136, 0.4172057807445526, -0.18442381918430328, 1.4127449989318848, -0.7693163156509399, -0.45827072858810425, 1.3486765623092651, 0.7803219556808472, 2.334261417388916, -1.4728912115097046, 0.5140743851661682, 1.3813246488571167, 1.4280225038528442, -0.482512891292572, -0.056538939476013184, -1.78908371925354, 0.22514469921588898, 0.3903634548187256, -1.1223042011260986, 0.13808415830135345, -0.9377469420433044, -1.2644597291946411, 0.5500348806381226, 0.2741517424583435, -1.8367013931274414, -0.06578317284584045, -1.086816668510437, -0.7103167176246643, 0.11499569565057755, 0.9679512977600098, -0.16160117089748383, 0.9332260489463806, -0.9906485080718994, -0.5765321254730225, 0.6759836077690125, -0.874412477016449, 0.2942553758621216, 0.9605197310447693, -0.8671549558639526, -0.7157217860221863, 0.07223162055015564, -0.3171168565750122, 0.4043387174606323, -1.1254373788833618, -0.7235537171363831, 0.3401181995868683, 0.2311050146818161, -0.583497166633606, -1.0765727758407593, -0.6666091680526733, -0.05011637136340141, -0.432096004486084, 1.839487075805664, 2.0165083408355713, 0.6129191517829895, 1.4225621223449707, -0.015841173008084297, 0.6238124966621399, -0.9714077115058899, -1.4688881635665894, 0.09036053717136383, 0.5396735668182373, -0.5390112400054932, -1.5330528020858765, 0.7488744854927063, 0.09653747081756592, -0.1302146315574646, 0.8065512776374817, 0.6043930053710938, -0.8359994292259216], [0.2796873152256012, -0.3863425552845001, 0.8768535852432251, -1.2093054056167603, 0.3567578196525574, 0.9190802574157715, 1.58589506149292, -0.6771866679191589, -1.3637217283248901, -0.7758160829544067, 0.4361077845096588, 0.31178903579711914, -0.0064718108624219894, -0.6877328157424927, -1.8122323751449585, 0.7459917068481445, -0.3360813856124878, -1.0691254138946533, 0.5209332704544067, -0.06208211928606033, 0.870891273021698, -1.023848295211792, 0.7811698317527771, -0.2816009521484375, -0.03697121888399124, -0.19863887131214142, -0.18402141332626343, -0.03824642673134804, -0.7017601132392883, 0.012736587785184383, 0.37753280997276306, -1.323301076889038, -0.6765731573104858, -0.8915882706642151, -0.5139161348342896, -0.24950096011161804, 0.9661176800727844, -0.49359622597694397, -2.847991466522217, 2.234457492828369, -0.8269256949424744, -1.2302213907241821, 0.3011927008628845, -0.21650336682796478, -0.606300950050354, 1.1633844375610352, 1.178027629852295, 0.903892457485199, -1.02777898311615, -2.0495052337646484, 0.9263551831245422, 0.9418164491653442, 0.37514999508857727, 0.14262646436691284, -0.48260942101478577, -0.9052759408950806, -0.6543604731559753, 1.164638876914978, 0.03066667728126049, 0.1785750389099121, 1.4614229202270508, -0.026526687666773796, 0.7928511500358582, 0.9279248118400574, -0.5429759621620178, 0.14182305335998535, 0.730116069316864, -0.7937976717948914, 0.23938068747520447, -1.0183806419372559, 0.19060686230659485, 0.08292220532894135, 1.0674067735671997, 1.3493486642837524, -0.0628974437713623, -1.0050711631774902, -0.6881654262542725, -1.2334603071212769, 1.443682312965393, -0.021033836528658867, -0.3700190782546997, 1.6514253616333008, 0.8623595237731934, -0.4928732216358185, -0.3453405499458313, 0.0213896706700325, -1.6125731468200684, 0.5524425506591797, -0.2515576183795929, -1.7885648012161255, -1.5175594091415405, 0.4497985541820526, -0.1728472262620926, 1.3640238046646118, -0.9754076600074768, -0.7717016339302063, 0.3971574008464813, -0.8226012587547302, 0.3368799090385437, 0.32819217443466187, 0.443484365940094, -1.4660439491271973, 0.5165079832077026, -0.04509030655026436, 1.1406855583190918, -0.20268242061138153, -1.2632039785385132, 0.7812726497650146, 0.494829386472702, -0.47093528509140015, -1.062820553779602, -1.0059077739715576, -0.7880871891975403, -1.0864462852478027, 0.9838167428970337, -0.1169564425945282, 0.2878274619579315, 0.967662513256073, -1.7750288248062134, -0.41757333278656006, -0.10038995742797852, -0.22592946887016296, -0.34600597620010376, -1.9587531089782715, 0.5479505062103271, 1.48384690284729, 0.6640840768814087, -0.10285729914903641], [0.7336841821670532, -1.4744929075241089, 1.0200345516204834, 0.5701086521148682, 0.8614378571510315, -0.6395109295845032, -0.45498713850975037, 1.503236174583435, -0.8028820753097534, 0.6171901226043701, 0.15307533740997314, -0.030110837891697884, 1.571851372718811, 0.5312876105308533, 1.0188871622085571, -0.014781899750232697, -0.14639829099178314, -0.14421366155147552, -1.2623562812805176, 0.6422503590583801, 0.34214749932289124, 0.6143215298652649, 1.3133625984191895, 1.0491104125976562, 0.5454161763191223, 1.0057697296142578, -0.046396806836128235, 0.6856133937835693, 0.16598999500274658, 0.7531173229217529, 1.295379638671875, 0.6490516066551208, 1.5981924533843994, -1.4300707578659058, 0.2845071256160736, 0.7699711322784424, 0.6726207137107849, -0.4851084053516388, 1.1108053922653198, 0.547779381275177, -0.34604161977767944, 0.45924368500709534, -0.3931429386138916, -0.1355823576450348, 1.785595417022705, 0.46824902296066284, -0.8247441053390503, -1.102907419204712, 1.0911377668380737, -1.8284714221954346, -0.5276522040367126, -0.04761626943945885, 0.026745328679680824, -0.10206688940525055, -1.0864167213439941, 0.9517083168029785, -1.7147232294082642, 0.3868878185749054, -0.799785852432251, 0.6702248454093933, -0.05910150334239006, 0.435457319021225, 2.2082762718200684, -0.08986827731132507, 0.09145817160606384, 0.5209294557571411, 1.2873475551605225, -0.5321903228759766, 0.22212831676006317, -1.504014492034912, -0.4308798909187317, -1.3205485343933105, 0.5799448490142822, 0.2429393231868744, -0.7339587807655334, -0.026535743847489357, 1.0280345678329468, 1.036107063293457, 1.855919361114502, -0.7578868865966797, 0.9790053367614746, 0.6457633972167969, -0.07173612713813782, -1.8923410177230835, -1.1278021335601807, 1.9694722890853882, -0.3614162504673004, -0.7747893929481506, -1.4700701236724854, 1.5680649280548096, -1.3220386505126953, -1.6327552795410156, 0.6381244659423828, -1.8686586618423462, -1.3108738660812378, 0.4151456654071808, -0.2838475704193115, 1.5701085329055786, 0.3401535451412201, 0.8762182593345642, -0.1572505086660385, 0.8456769585609436, -0.3313983380794525, 0.8756502270698547, -0.9357067942619324, 0.37224170565605164, 0.07530327886343002, 1.1499266624450684, 1.7845258712768555, 2.3041558265686035, 0.15503528714179993, 1.5513020753860474, 0.2542558014392853, 1.6365786790847778, -0.9158918261528015, 0.030172303318977356, 2.2282490730285645, 0.4464324712753296, -1.055478572845459, -1.4240494966506958, 0.009099021553993225, -0.6147053837776184, 0.16915616393089294, -0.3418574631214142, -0.145549014210701, 0.10947579145431519, 0.5727149248123169, -1.488897681236267], [-0.5639359951019287, 0.8746024370193481, -1.3608689308166504, 0.7745780944824219, 0.5071927309036255, -1.390533447265625, 1.1705894470214844, -1.1938297748565674, -0.05069233104586601, 1.9317914247512817, 0.3499886989593506, -1.479583501815796, 1.7324029207229614, -0.192457914352417, 1.7053481340408325, 0.20450431108474731, -0.5293995141983032, -1.0588042736053467, 0.897063136100769, -0.44802945852279663, 1.0287176370620728, -2.5128884315490723, -2.2609810829162598, 0.1522020548582077, 0.9723466038703918, 0.6413052082061768, 2.9544589519500732, -0.884043276309967, -1.9688193798065186, 1.6289377212524414, -0.538697361946106, -0.005508764181286097, 0.5800997614860535, -1.9611128568649292, -0.29949867725372314, 0.21348616480827332, -0.2147553712129593, -0.12294274568557739, -0.9357461929321289, -2.199681043624878, 0.6430783271789551, -0.49959200620651245, -0.02264624647796154, 1.5315711498260498, 1.276706337928772, 0.698335587978363, 1.1272708177566528, -1.9350264072418213, -0.4224611222743988, -0.7057338953018188, -1.9084924459457397, -0.4651876389980316, -0.021279562264680862, 0.14017565548419952, 0.17471513152122498, 0.26350119709968567, -1.3437418937683105, -0.9709797501564026, -0.6772864460945129, -2.3733723163604736, -1.2397534847259521, 0.6270242929458618, -2.63619327545166, 0.0686701238155365, 1.150231957435608, -0.33599311113357544, 1.4545141458511353, -0.07088124006986618, 0.33788037300109863, 0.07587616890668869, 0.4357287883758545, -0.23045913875102997, -0.3887168765068054, 0.7667233347892761, -0.17539915442466736, 0.3031613230705261, 0.0762908011674881, 0.6431199312210083, -0.7991947531700134, 0.09629826247692108, -0.8400478363037109, 1.8218713998794556, 0.5565800070762634, -1.2478740215301514, 0.6173622012138367, 0.1261710226535797, -0.5146782994270325, 1.7858071327209473, -1.243574619293213, -0.3059237599372864, 0.6339061260223389, -1.0470378398895264, -0.7405730485916138, -2.1322522163391113, -0.8744155168533325, -0.9561091065406799, 0.08981549739837646, -0.905957043170929, -0.5073744654655457, -0.13778537511825562, 0.02483982965350151, -0.677537739276886, -1.2881966829299927, 0.4617960453033447, -0.44326546788215637, 0.09852459281682968, -1.4472098350524902, -0.37052926421165466, 2.477579355239868, -0.8699653148651123, 0.16061821579933167, -1.1278384923934937, -1.027970790863037, 0.3331771790981293, 0.0658007487654686, 0.2557043433189392, 1.4701868295669556, -0.4944313168525696, -0.5397863388061523, 1.0226094722747803, 0.11999961733818054, 1.3746825456619263, -0.4095096290111542, 0.5183557868003845, -1.244674801826477, -1.4758906364440918, -0.6752811074256897, -1.4381035566329956], [-1.6731058359146118, 0.3693508207798004, 0.040992915630340576, 0.31064772605895996, -0.8445476293563843, -1.5651097297668457, -0.5405738353729248, -1.8919613361358643, 0.4299277067184448, 1.4961791038513184, -0.050539351999759674, -1.575021743774414, 1.7548776865005493, -0.05507620424032211, -1.0502259731292725, -1.3536370992660522, 1.3990532159805298, -0.1771678477525711, -0.9504903554916382, -0.9912020564079285, -0.5970665812492371, 3.1826303005218506, 0.3929380178451538, 0.21308712661266327, 0.3282671570777893, -0.2537611424922943, 0.7005942463874817, -0.25325244665145874, 2.277825117111206, -0.7158039808273315, 1.1672003269195557, 1.2125623226165771, 0.4530086815357208, -1.968698501586914, 0.06417136639356613, -1.3465148210525513, -0.6982949376106262, -1.5141737461090088, -1.212783932685852, 0.09132762253284454, 0.3346197307109833, 0.20471623539924622, 0.8616278767585754, -0.8572742342948914, -0.4126124680042267, 0.5850249528884888, 1.4527124166488647, 0.37725695967674255, -0.9988368153572083, 0.12897109985351562, -1.2086070775985718, 0.6103009581565857, -1.6736174821853638, -1.1171245574951172, -0.8887084722518921, -0.1914742887020111, -0.12829868495464325, 0.5339182019233704, -0.24981297552585602, 0.8723686933517456, -0.6628562211990356, 0.3524533808231354, 1.7423206567764282, -0.7610405683517456, -0.6769206523895264, -0.28543204069137573, -1.21780526638031, 0.009581808000802994, -0.6765879392623901, -0.16671006381511688, 0.8515228629112244, -0.02391492761671543, -0.9108685255050659, 0.4217967391014099, -0.12406887114048004, 0.5218291282653809, 1.256524682044983, -0.2326863557100296, -0.19424757361412048, 0.13867376744747162, -1.2717041969299316, 0.853678822517395, 1.1528825759887695, 0.18407592177391052, -1.669908881187439, 1.0769785642623901, -1.5833284854888916, 0.8796425461769104, 0.7943134307861328, -1.7388801574707031, -1.1865092515945435, 0.44065728783607483, 0.15993255376815796, 0.16864313185214996, -1.1008607149124146, 0.4616568088531494, 0.9140660762786865, 0.6839014887809753, 0.3073429465293884, -0.7570011615753174, 2.509981870651245, 1.3760591745376587, 1.3606603145599365, 0.6942917108535767, -0.2106119990348816, 0.32312673330307007, 0.5846848487854004, -1.699700117111206, -0.2517925202846527, 0.11671354621648788, -1.7423866987228394, -0.3268510103225708, 0.6603276133537292, 1.6877901554107666, 1.2430497407913208, -0.49505650997161865, 0.6300610899925232, -1.8316280841827393, 0.2040257304906845, 0.979208767414093, -0.8478860259056091, 0.20709823071956635, -0.021524161100387573, -0.36404192447662354, -0.6840983629226685, -0.054308533668518066, 1.085364818572998, -0.14219310879707336], [-0.3919001519680023, 1.3420851230621338, 0.3293163478374481, 0.23119305074214935, 2.563582420349121, -1.1084471940994263, -0.4628356993198395, 0.9160056114196777, -0.550599992275238, 0.5944064855575562, 0.29037874937057495, -0.5820532441139221, 1.1951897144317627, -0.5805153250694275, -0.21305020153522491, 1.3828130960464478, 0.9474074244499207, 0.34302130341529846, -1.0228208303451538, 0.9329887628555298, -0.7718012928962708, -0.1437506228685379, 1.1924747228622437, -0.4535330533981323, 0.3378363251686096, 0.33983999490737915, -1.11311936378479, 0.6373689770698547, 0.8307434916496277, -2.1282262802124023, -1.127209186553955, 1.878928542137146, 0.3144093453884125, 0.05602674558758736, -0.28957027196884155, -0.451437771320343, 1.3348876237869263, 1.1185998916625977, 1.3223179578781128, -0.9190117120742798, -0.2035364806652069, 0.6414893865585327, 1.5726144313812256, 0.8748091459274292, -0.47024035453796387, -0.5315657258033752, -1.1008492708206177, 0.434455543756485, -0.48250481486320496, -0.8436448574066162, -1.805203914642334, -0.976763904094696, 1.1479084491729736, 0.5896468758583069, -0.876056432723999, -2.0720717906951904, -0.14605608582496643, 0.48916980624198914, -1.6590838432312012, 0.8095748424530029, 2.4474735260009766, -1.0447808504104614, 1.2461268901824951, 1.3231991529464722, 0.8655582666397095, -0.07500442862510681, -0.3715154528617859, -1.7700012922286987, -2.29765248298645, -0.633063793182373, -0.09066054224967957, 2.5562937259674072, -0.12713591754436493, -2.2788398265838623, 1.6045377254486084, 0.29301682114601135, 0.39780816435813904, 0.2033470869064331, -0.19537612795829773, -0.27021071314811707, 0.8309417963027954, -2.2805087566375732, -1.3044140338897705, 0.95355224609375, 0.4586898386478424, 0.6293593645095825, -0.26989302039146423, 0.24428333342075348, 0.12104635685682297, 1.4129315614700317, 1.3125380277633667, 1.3993111848831177, 0.21310879290103912, 0.8904106020927429, 0.28677868843078613, -0.9423378109931946, 1.1666254997253418, -1.2838724851608276, 0.09136970341205597, -1.100787878036499, 1.3009538650512695, -0.31226757168769836, -0.39353904128074646, 1.148636817932129, 0.3467913269996643, -0.05980671942234039, -0.023880714550614357, -0.05710991844534874, 1.110853910446167, -1.9944080114364624, -0.5542829632759094, -2.083711624145508, 0.3163544833660126, -1.5882189273834229, -0.7584748864173889, 0.06890656799077988, 1.9011478424072266, -0.1871599406003952, -0.6036882400512695, 0.2444934844970703, -0.3169492483139038, -0.5959410071372986, 0.7541359066963196, -1.9327034950256348, 1.1205335855484009, 0.11552251875400543, 0.09910840541124344, -0.5247908234596252], [-1.472511649131775, 0.7960940003395081, 0.8497072458267212, -0.666856586933136, -0.10506559908390045, -0.08025584369897842, 0.6082459092140198, -1.931323528289795, 0.8240402340888977, -2.685760736465454, -0.5525092482566833, -1.0332599878311157, 0.8368307948112488, 0.5322599411010742, -0.885802686214447, -1.1793367862701416, -1.2159913778305054, -0.7926428914070129, 2.028022527694702, -0.17595274746418, -0.05887727439403534, -0.1821860671043396, -1.1750675439834595, 1.2361695766448975, -0.9041595458984375, -0.11843313276767731, 0.5838908553123474, -1.8154211044311523, 1.4454271793365479, 1.1241858005523682, -0.416156142950058, 1.2002798318862915, 0.5886219143867493, 0.48628947138786316, 0.9837878942489624, -0.6510759592056274, -1.6674493551254272, 0.4498615562915802, 0.6886163353919983, -0.12496606260538101, -1.1578607559204102, -2.4114387035369873, -0.27778327465057373, 0.2706751227378845, 1.2044389247894287, 0.8665550351142883, -0.9323880076408386, -0.7453240156173706, -1.6805555820465088, -0.4621979892253876, -0.5481151342391968, 0.9994248151779175, 1.1608364582061768, -0.2045152634382248, 1.5945923328399658, -0.1311589628458023, 0.011154836975038052, 1.218719482421875, 0.5559274554252625, -0.36742478609085083, -0.8609364628791809, 0.614876389503479, -1.1598546504974365, -0.6391969919204712, 1.3543566465377808, -0.478058785200119, -0.4751444458961487, 0.6605287790298462, -0.177917018532753, 1.589232087135315, 2.785993814468384, -1.103306531906128, -0.6505317091941833, -0.6455624103546143, -0.257833868265152, 0.09285061806440353, -0.19786207377910614, -0.6044294834136963, 1.4230265617370605, -0.25573208928108215, -2.2032692432403564, -0.5900909304618835, 0.37997451424598694, -0.10058177262544632, 0.7154501080513, -1.4589112997055054, -1.3393893241882324, 0.8288702368736267, 1.533469796180725, -0.171351358294487, -2.0883774757385254, -1.1425867080688477, 0.6521583795547485, -2.957347869873047, 1.0029840469360352, 0.08561010658740997, 1.7986544370651245, 1.4564998149871826, 0.5861301422119141, 0.6597746014595032, -1.0722603797912598, -0.6754500865936279, -1.1425776481628418, 1.4135453701019287, -0.43163377046585083, -1.779502034187317, 1.358249545097351, -1.1452820301055908, -0.42445263266563416, 0.2677222490310669, 0.6939049959182739, -0.2261868268251419, -2.2417125701904297, 0.5454994440078735, 0.9930899143218994, 1.231784701347351, 0.9384070038795471, 0.5401118993759155, -1.4611705541610718, 0.8722484707832336, -0.4469132721424103, -0.05394724756479263, 2.3067071437835693, -0.352940171957016, -0.7076181173324585, 0.8915274739265442, 0.2105930596590042, 2.7090835571289062], [0.5936917066574097, -1.995584487915039, -0.32018938660621643, -0.5211277604103088, -2.099508285522461, 0.12663285434246063, 1.1491974592208862, 1.9057371616363525, 0.229170024394989, -0.14686396718025208, -0.026593292132019997, -0.5261412858963013, -0.8828490972518921, 0.8570272326469421, -1.9603759050369263, -0.2400141805410385, 1.101834774017334, -0.34439989924430847, -0.5758424401283264, -0.006795973517000675, -0.7990838289260864, 0.519005537033081, -1.019334077835083, -0.8951175808906555, 0.11506175249814987, -0.8019490242004395, 0.25956523418426514, 0.9160979986190796, -2.622746229171753, -0.4049224257469177, 1.264976978302002, 1.6337705850601196, 1.5398173332214355, 1.1935385465621948, -1.0787020921707153, -0.00975360069423914, -0.5987217426300049, -1.3658555746078491, -0.3092205226421356, 1.013375997543335, 0.1254226565361023, 0.6884677410125732, -1.6938220262527466, 0.20052871108055115, -0.04908118396997452, -1.1798105239868164, -1.6179829835891724, -0.4259459674358368, 0.7445982694625854, 0.45886245369911194, 2.5373315811157227, 1.2740012407302856, 0.6699477434158325, 0.49363821744918823, 0.17898453772068024, -0.50239497423172, 1.3552292585372925, -0.7497888803482056, 1.0783963203430176, -0.20448365807533264, -0.02797294221818447, -0.06662731617689133, -1.4926377534866333, -0.9086435437202454, -1.1687804460525513, -0.6720724105834961, -1.1886283159255981, -1.9110914468765259, 0.4750613868236542, 0.056784238666296005, 0.013784744776785374, 1.0503698587417603, 1.864749789237976, 0.6451888680458069, -0.17551031708717346, 1.548828125, 0.23108121752738953, 0.4263985753059387, 1.5509796142578125, 0.2975936233997345, 0.4294230341911316, 0.011149133555591106, 2.2346646785736084, -0.7908686399459839, -0.531379759311676, -1.2436574697494507, -1.8224471807479858, -0.8083191514015198, 0.41396480798721313, -1.0594632625579834, -1.6402640342712402, -1.093843936920166, 0.6042660474777222, -0.7715296149253845, -0.145433709025383, -1.5712592601776123, -0.22386030852794647, -0.04959459602832794, 0.35975101590156555, -1.240766167640686, 1.277409315109253, -1.0276697874069214, 0.9161064624786377, 0.2847878038883209, 0.005424422677606344, -0.15918852388858795, 1.3401936292648315, -1.807462215423584, -1.002191424369812, -0.24706541001796722, 0.3219757080078125, 0.45528772473335266, 1.073638916015625, -1.9226727485656738, -0.31967172026634216, -0.6359643340110779, 0.19815553724765778, 1.3021740913391113, 2.8853189945220947, 0.6919077634811401, -1.6967706680297852, -0.08045801520347595, 0.106073759496212, 0.5685175657272339, 0.7219733595848083, -0.037174925208091736, -1.0719256401062012, -1.2811909914016724], [-0.36785808205604553, 1.3572494983673096, 0.6494893431663513, 0.3253714144229889, 0.8132730722427368, 0.6230942606925964, -1.336963415145874, 0.5189900994300842, -0.7761486768722534, -1.5785714387893677, -1.6022536754608154, -0.0655428022146225, -0.9804638028144836, -0.5981336832046509, 0.404509574174881, 0.45436063408851624, -0.8214945197105408, -0.8584975004196167, -0.6076399087905884, -0.2918790876865387, -0.43851399421691895, 0.05681497976183891, -0.06707942485809326, 1.0598171949386597, 0.365496426820755, -0.8078019022941589, -0.6386595368385315, 1.1133216619491577, -0.8633801341056824, 0.8247316479682922, 0.7329520583152771, 1.330203890800476, -0.6237198114395142, 0.7030730843544006, -0.9499509930610657, 0.9257525205612183, -0.30081117153167725, 0.6215003728866577, -1.0591648817062378, -1.2874494791030884, 0.3121730387210846, 0.16782309114933014, 1.6293648481369019, -0.7127652764320374, -1.9079992771148682, 2.018401622772217, 0.7201225161552429, 0.7712156176567078, -1.212619662284851, 1.6168462038040161, 0.9642391800880432, -0.4680330157279968, 0.9563915133476257, -0.5385735034942627, 0.793447732925415, 2.1749534606933594, -1.8942201137542725, 1.0816099643707275, -0.4813731014728546, 0.6165364980697632, -1.1981507539749146, -0.3211478590965271, -0.5779552459716797, -1.0144708156585693, -1.111608624458313, 0.8914740681648254, -0.9150356650352478, 1.0968776941299438, 1.132476806640625, -0.14604505896568298, 1.863071322441101, -1.3797972202301025, 1.3624545335769653, 0.4176049530506134, 1.9120877981185913, 0.5804213881492615, -0.4719272255897522, 0.6270432472229004, 0.01332502719014883, 0.44683408737182617, -1.824522614479065, -0.3322223722934723, -0.5723066926002502, -0.1291513890028, -0.9719017148017883, 0.8003218173980713, 0.5109608173370361, 0.4823233187198639, -0.5741078853607178, 1.4339061975479126, 2.9815306663513184, 0.9673134088516235, 0.7715405225753784, -1.8850176334381104, 0.006563153583556414, -0.3504931330680847, 0.13726887106895447, -2.1321685314178467, 1.6212785243988037, -1.3671051263809204, 1.6682169437408447, -1.8034358024597168, -1.104479432106018, 2.560302734375, -0.9118468165397644, -1.261342167854309, -0.05061109736561775, -0.6406623721122742, 1.7660633325576782, -0.2134363055229187, 0.408291220664978, -0.9947643876075745, 0.42175713181495667, 1.3337819576263428, 0.41359591484069824, -1.2805488109588623, -0.7124820351600647, -0.5712237358093262, 0.10184629261493683, 0.34918975830078125, 2.026456117630005, -0.3072567582130432, 0.06620319187641144, -1.9979989528656006, -0.21325281262397766, 1.3693420886993408, -0.5043970942497253, 0.1647839993238449], [-1.2522817850112915, -1.4962373971939087, -0.05742860212922096, 0.30942046642303467, -1.30872642993927, -1.2593672275543213, -1.3077675104141235, 0.3324015438556671, 0.44745588302612305, 0.9485743641853333, -0.561363935470581, 0.022173084318637848, -0.7623745203018188, -0.5788546800613403, -0.271051287651062, 0.08588577806949615, 0.13185349106788635, 0.1655108779668808, -1.3105558156967163, -0.24300776422023773, 0.32149845361709595, 0.45392733812332153, 0.010252157226204872, -0.8361029624938965, -0.8013210892677307, -1.4171369075775146, 0.12027829885482788, -1.2693604230880737, 1.0851943492889404, -1.0215096473693848, 1.2734662294387817, 0.022449076175689697, 0.3599143326282501, -1.0045502185821533, -0.7836637496948242, -0.3887428939342499, 0.8817140460014343, 0.1695587933063507, 0.25333088636398315, 0.17082665860652924, 0.008535772562026978, -0.15359020233154297, 0.4950428009033203, 1.5550720691680908, 0.6781081557273865, -2.125046968460083, -0.3707062005996704, -0.2171424776315689, -0.6833792328834534, 0.028396833688020706, 0.9601507782936096, -0.03592994436621666, -0.7715969085693359, 0.919642984867096, -0.26030683517456055, -1.1814111471176147, -1.956844687461853, 0.1937447041273117, 0.23382824659347534, -0.732896089553833, 0.41045308113098145, 0.7245689630508423, 0.7542321085929871, -1.2588270902633667, -0.02676851861178875, 0.2987362742424011, 1.252665638923645, -0.05857057124376297, -0.036920852959156036, 1.6028642654418945, -1.6160905361175537, 0.8561744093894958, 1.5085572004318237, -1.0957964658737183, -0.5873112678527832, 1.5816209316253662, 0.709645688533783, -1.010372281074524, 1.081483006477356, 0.05548088625073433, -0.6342303156852722, 0.8590922355651855, 0.4483849108219147, -0.6279517412185669, -0.33338549733161926, -0.20897217094898224, 1.8202290534973145, -1.0085062980651855, 0.7876390814781189, 0.2306242138147354, -0.08999364823102951, -0.9568307399749756, 0.9416673183441162, -0.9029923677444458, -0.8751033544540405, -1.4750432968139648, 1.1613448858261108, -0.19128386676311493, -2.629866123199463, -0.7776963114738464, -0.79190593957901, 0.6520723700523376, -1.5788254737854004, 0.07636868953704834, -1.3605754375457764, 0.8537936806678772, -1.455592155456543, 0.9939358830451965, 0.4598334729671478, 2.0493414402008057, -0.44791510701179504, 0.453498512506485, 2.570173978805542, 0.22400543093681335, -0.8515016436576843, -0.5841779708862305, -0.19341431558132172, 1.97860586643219, 0.4987533390522003, 0.5946993827819824, -0.7943584322929382, -0.6532827615737915, 0.07643508911132812, 0.9443089962005615, -0.8302971124649048, -0.0007889711996540427, 0.15819989144802094, 0.6786532998085022], [-0.009532518684864044, -0.4452308118343353, -1.6327619552612305, -1.1366413831710815, -0.32296472787857056, -1.6567118167877197, -1.5051648616790771, -0.09641170501708984, -1.293779969215393, 0.6842365264892578, -2.2552707195281982, 0.18768946826457977, -1.9784438610076904, 0.23667065799236298, 0.025520868599414825, -0.9812087416648865, 0.5120146870613098, 1.7942113876342773, -0.9366904497146606, -0.6261053681373596, -0.9087240099906921, 2.3961896896362305, -0.6966184973716736, -0.7348085641860962, -1.3258001804351807, 1.1796953678131104, -1.9396671056747437, -0.9691775441169739, -0.9428562521934509, -0.8738605380058289, -0.03877265378832817, -1.2755378484725952, 2.8187522888183594, -1.0533015727996826, 0.5988950729370117, -1.2035951614379883, -2.4016621112823486, -0.45111754536628723, -1.01201593875885, -2.0658462047576904, -0.512964129447937, 1.346877932548523, -0.9899036288261414, -2.1510932445526123, -0.7864883542060852, -1.9400599002838135, -1.5827934741973877, -0.5413808226585388, -0.631676971912384, -0.8334502577781677, 0.17315176129341125, -0.28533437848091125, 0.7647317051887512, 0.035844266414642334, 0.20342804491519928, 0.1829940527677536, 0.33022600412368774, -0.595811128616333, -2.433638572692871, 1.5934687852859497, 0.4711982309818268, -2.366185426712036, 1.5164806842803955, 1.1359585523605347, 0.23806095123291016, 0.3307739794254303, 0.7040612697601318, 2.310772657394409, -0.6476824283599854, 0.944195568561554, 1.0059846639633179, 0.796737790107727, 0.4096029996871948, -1.1026479005813599, -0.3147958219051361, -1.4212396144866943, -0.6212930679321289, -0.4834135174751282, 0.4736748933792114, 0.1546056568622589, -0.6711461544036865, -2.120877504348755, 1.2764520645141602, -0.1737534999847412, -0.26961350440979004, -0.4828360974788666, -1.3352577686309814, -0.7499148845672607, -0.15352080762386322, -0.6196227669715881, -0.268195241689682, -0.20723791420459747, -0.2538011074066162, -0.19880340993404388, 0.991253137588501, -1.4252674579620361, -0.6460822224617004, -0.04652842879295349, 0.7028089761734009, 0.9504801630973816, 1.8845641613006592, 3.1088624000549316, -0.7991144061088562, -0.6392495632171631, -0.883208692073822, 1.0890694856643677, 0.03235961124300957, -0.4440251886844635, 0.5831862092018127, 0.5021213293075562, -1.5575206279754639, -0.7830783724784851, 0.3062538802623749, -1.8741497993469238, -0.18454083800315857, -0.5370057821273804, 1.35548734664917, 1.1976574659347534, -0.49869510531425476, 2.524007797241211, -0.6232147812843323, 1.193995714187622, -2.0689892768859863, -0.20671656727790833, 0.19258278608322144, -0.8787558674812317, 0.7384040355682373, 0.17347638309001923], [-0.3596988916397095, 1.1070493459701538, -0.0230043213814497, 0.1559179723262787, 1.1371396780014038, 0.4606454074382782, 0.32454243302345276, 0.8819397687911987, 0.2951471209526062, 1.5598329305648804, 0.30836576223373413, 0.775579571723938, -0.6051952242851257, 2.505821704864502, 0.9124706387519836, -0.6168447136878967, 1.0952339172363281, 0.30436742305755615, 0.0718718096613884, -0.6777225136756897, 0.3749063014984131, -1.3284605741500854, -2.1558728218078613, -0.257404625415802, -1.4321494102478027, 1.5057258605957031, -0.9765030145645142, -1.168776035308838, 1.5400053262710571, 0.2653059661388397, 0.2409282922744751, -0.7117756009101868, 0.03613550588488579, 0.647203266620636, -0.19760195910930634, 0.0690615326166153, 0.2134489119052887, -0.7546752095222473, 1.7629987001419067, -2.10660719871521, -1.0798766613006592, 1.1139878034591675, 0.9326800107955933, -1.731654167175293, -0.38924968242645264, -0.6933252215385437, -0.6657894849777222, -1.3254626989364624, -0.11887285858392715, 0.7092525362968445, 0.553609311580658, 0.32968536019325256, -0.7667607665061951, 0.9031248688697815, 0.8548274636268616, 0.322339802980423, -1.369911789894104, -1.5340524911880493, 1.1041728258132935, -1.3356691598892212, 1.2291431427001953, -1.3209435939788818, -1.5558663606643677, -2.48472261428833, 1.2141042947769165, -0.3814588189125061, -0.36698710918426514, 0.734639048576355, -0.9009731411933899, 1.0765949487686157, -0.26766765117645264, 0.1116936057806015, -0.8882728815078735, -0.3275621831417084, -0.9067197442054749, 1.1654149293899536, -1.2790390253067017, 0.7420019507408142, 0.8049805164337158, -0.35037463903427124, 1.9813461303710938, -0.19289715588092804, 1.2956538200378418, -0.34039971232414246, -0.7115021347999573, 0.10198518633842468, 0.19397294521331787, 0.25863760709762573, -0.7171831727027893, 0.012067945674061775, -0.8206999897956848, -1.0189785957336426, 0.9898332357406616, 1.020476222038269, -0.691469132900238, 0.22225113213062286, 0.9034929275512695, 0.21921177208423615, -0.6579693555831909, -0.8269597291946411, -1.578620195388794, -1.013044834136963, 0.2071485072374344, -0.3452224135398865, 1.353558897972107, -1.4315811395645142, 0.3561863303184509, 0.8542613983154297, 0.7624182105064392, 0.005536389071494341, -1.0860955715179443, -0.4958783686161041, 0.952024519443512, -0.5471547842025757, -0.7747569680213928, -0.25010648369789124, -1.7599791288375854, 1.1242702007293701, 0.9574222564697266, 0.8897852897644043, 1.106575608253479, -0.06460593640804291, 0.4569207727909088, 2.271125555038452, 1.7848727703094482, 0.30251893401145935, 0.1575188785791397, -0.19814248383045197], [0.6456185579299927, -0.539757251739502, 0.7744359970092773, -0.8524174690246582, -0.8859398365020752, 1.1360962390899658, -0.5593147873878479, 0.2822081744670868, 0.49455970525741577, -0.8839071393013, 1.9871876239776611, 1.0094189643859863, 0.6197077631950378, 0.3849538564682007, -1.4134862422943115, 0.0901210755109787, -1.110120415687561, 0.692180335521698, -0.08884774148464203, -0.3839009404182434, 0.6205169558525085, 1.3178880214691162, 1.1026405096054077, 1.2792972326278687, -0.7307828068733215, 0.367565780878067, -0.9292941093444824, -0.2963634729385376, 0.4493202567100525, 0.6162105202674866, 0.6620950698852539, -0.8039612770080566, 1.1710827350616455, 1.0858601331710815, -0.39968982338905334, -0.3837735950946808, 0.4216068387031555, 0.06148025020956993, 0.8684347867965698, 0.13583646714687347, -0.4661908745765686, -0.1366734355688095, -0.5230688452720642, -0.46262988448143005, -0.22892990708351135, 0.23021343350410461, -0.15352144837379456, 0.8725185394287109, 1.873731017112732, 1.0687675476074219, -1.2700345516204834, -0.15866732597351074, -0.8034136295318604, -0.6305639147758484, 1.0674086809158325, -1.4517924785614014, 0.9562642574310303, -1.7721408605575562, -0.37198343873023987, -0.9755247831344604, 1.7502198219299316, -1.1775263547897339, -1.039737582206726, -0.026779869571328163, 1.0863887071609497, -0.24569053947925568, 0.8940441012382507, 0.9158337116241455, -0.16803933680057526, 0.802277684211731, -0.4642969071865082, 2.514415740966797, 0.2570870816707611, -0.28186818957328796, 0.12665244936943054, -1.436264991760254, -0.17235587537288666, -0.37422341108322144, 1.0252050161361694, -1.635263442993164, -0.274554580450058, 1.6481462717056274, 0.08019126951694489, 0.20491032302379608, 0.8016024231910706, -1.575745701789856, -0.12346617132425308, 1.403111457824707, 0.16309483349323273, 1.2982709407806396, 0.375777006149292, -0.7409965991973877, 2.537994623184204, -0.8547258973121643, -0.7390671968460083, -0.45233145356178284, -0.7070327997207642, 1.109237551689148, 0.014587651938199997, -0.1125558540225029, -0.3904537856578827, -0.6230855584144592, -0.738224446773529, 0.2873242497444153, 1.1402504444122314, 0.6596077084541321, 1.5736844539642334, -0.2035801261663437, 0.38946500420570374, -0.42238888144493103, 1.1200149059295654, 2.1186580657958984, 0.16888341307640076, -0.3940259516239166, -0.6177241206169128, -0.5353872776031494, -0.6180147528648376, -1.897303581237793, -0.10977286100387573, 1.2784706354141235, 0.03458859398961067, -0.33572709560394287, 1.0195677280426025, 0.15332554280757904, -0.4420856237411499, -0.4108291268348694, -1.432580590248108, -0.8713791966438293], [0.4726777672767639, 0.5210220217704773, -1.047614574432373, 0.9570956826210022, 0.3195536136627197, -0.3754461407661438, 1.0604443550109863, -0.3126758635044098, 0.56830233335495, 0.03131049498915672, 0.10494966804981232, 0.942914605140686, 0.14599251747131348, -1.9657968282699585, 1.0441464185714722, -0.24571752548217773, 1.3851336240768433, 3.143756628036499, -0.14369559288024902, -0.7996442914009094, -0.6283888816833496, 0.9835662841796875, 1.3758741617202759, 1.3002561330795288, -0.7668859958648682, 1.1435142755508423, 1.2526271343231201, 0.13802248239517212, -0.3822133243083954, -0.038992807269096375, 0.2548760175704956, -0.38704901933670044, -1.594040870666504, -1.0562901496887207, -0.5357204675674438, 1.3756968975067139, -0.828992486000061, -0.43761417269706726, -0.5054015517234802, 0.34818243980407715, -1.2276667356491089, 0.4390082359313965, -0.050878796726465225, -0.30329057574272156, 0.3707634210586548, 1.5857020616531372, 1.4071910381317139, 0.7849496006965637, -1.95063054561615, 0.07629004120826721, -1.1451972723007202, 0.43381965160369873, 1.5882354974746704, -0.7077680826187134, -0.9102115035057068, -0.05871555209159851, 0.47424009442329407, 1.5320907831192017, 1.6827396154403687, 1.7908134460449219, -0.5984236598014832, 0.11016334593296051, -2.5158026218414307, 0.41838276386260986, 0.9717050790786743, 0.8334836959838867, -0.6114739179611206, -0.9102168083190918, -1.886955738067627, -1.60923171043396, -0.026829199865460396, -0.8774142265319824, -0.3745099902153015, 1.5719908475875854, 0.15693166851997375, -0.6402508020401001, 2.099125623703003, 0.11183447390794754, 1.3450933694839478, 1.501045823097229, 0.07699564844369888, 0.5913078188896179, -0.41236093640327454, -0.24586240947246552, 0.6777713894844055, -0.8339372277259827, -0.06462545692920685, -2.623598098754883, 0.6704747676849365, 0.6073135137557983, -0.930277407169342, 1.0183063745498657, -0.8180912137031555, 0.43081068992614746, -0.5502648949623108, 1.8508598804473877, -0.819236159324646, 0.40743958950042725, 0.09755948185920715, -0.2647336423397064, -1.9569898843765259, 2.067389726638794, -1.0429856777191162, 0.007950380444526672, 0.03982210159301758, 0.8642575144767761, -0.04796258732676506, -0.11299337446689606, -0.05404245853424072, -0.3245251476764679, 0.30266913771629333, 0.5053713917732239, -0.860893189907074, -1.417863368988037, 2.274977445602417, 0.1217988133430481, -0.15627634525299072, -0.6489564180374146, 1.9161456823349, -0.10059937089681625, -0.44436946511268616, 1.9357285499572754, -0.06909209489822388, 0.12288964539766312, -1.3597217798233032, -0.001035612542182207, -1.8488634824752808, -0.5191466212272644], [-1.415280818939209, 0.8845162987709045, 0.9142616391181946, -0.19771768152713776, 1.159206509590149, 2.753512144088745, 1.693436861038208, 0.38727831840515137, -1.4263147115707397, 0.06638500094413757, 2.0065746307373047, 1.4127322435379028, -0.4012987017631531, 0.5791175961494446, 0.23976904153823853, 1.113588809967041, 0.3250243067741394, 0.2165168970823288, -0.28501778841018677, 0.9733018279075623, 0.2317948043346405, -1.0840344429016113, 0.30887216329574585, 1.6001112461090088, -1.4629331827163696, 0.4787735641002655, 1.565564513206482, 0.9308180212974548, 0.5880523920059204, 0.03433043509721756, 0.7871996164321899, 1.409568428993225, -0.41823121905326843, 0.7688316702842712, -0.6102166771888733, -0.9441985487937927, 0.8311604857444763, 0.43618136644363403, 0.5003306269645691, -0.8557100296020508, 0.39192894101142883, -1.0721491575241089, 0.5106508135795593, 0.42874109745025635, 0.49962422251701355, 0.32081860303878784, 2.471285104751587, -0.6112947463989258, -0.6884999871253967, 0.25093498826026917, 1.148877501487732, 0.8954142332077026, 0.23892705142498016, -1.107881784439087, 1.0308952331542969, -0.48471203446388245, 1.064192771911621, 0.7541201114654541, -0.11430089175701141, 0.7811789512634277, 0.06894978135824203, -1.182228684425354, -0.5924782156944275, 0.45703646540641785, -0.4126916229724884, -0.578967809677124, 1.8285560607910156, -0.13100597262382507, -0.713786244392395, 0.3405801057815552, 1.2961134910583496, 1.607985496520996, 0.3781440556049347, 0.244619220495224, -0.0660274475812912, -0.411389023065567, 1.4050753116607666, 0.31166744232177734, -0.11714734137058258, -0.3176513612270355, 0.17851731181144714, 1.629177212715149, 0.6762007474899292, -1.4196871519088745, 0.8416861295700073, -0.9296753406524658, -0.5182963013648987, -0.0745992362499237, -1.565600037574768, -0.32351332902908325, -0.7031340003013611, -1.0631563663482666, 1.4073628187179565, 0.9048648476600647, 0.0008405859698541462, 0.2463127076625824, 0.34590643644332886, -0.023055927827954292, 1.4072576761245728, 1.1701680421829224, -0.251362144947052, -0.10615986585617065, 0.1743934601545334, -1.3958336114883423, 0.49980098009109497, 0.8215203285217285, -0.8538366556167603, 1.5339884757995605, 1.7359000444412231, 0.9414673447608948, 1.4481886625289917, -1.8396352529525757, 1.3763759136199951, 0.6007605791091919, 1.5279301404953003, -1.3811051845550537, -0.3187032639980316, 0.6200750470161438, 0.6051135659217834, -0.5296551585197449, 0.9875541925430298, -1.3001763820648193, 0.8176578879356384, -1.0626202821731567, -0.17429284751415253, -0.02546403557062149, 1.3573315143585205, 1.690287709236145], [-1.9878227710723877, 0.21688905358314514, 1.8018068075180054, -1.252289891242981, 1.3343013525009155, -1.25541090965271, -1.2142471075057983, 1.5361214876174927, -0.6222901344299316, 1.6969828605651855, -0.5986052751541138, -0.047721389681100845, -1.0277035236358643, 0.17487363517284393, 2.7637369632720947, -1.576986312866211, -0.3160938024520874, 0.15842126309871674, 1.774552822113037, 0.06373217701911926, -0.2604248821735382, 0.5875069499015808, 0.29025182127952576, 0.5541000366210938, -0.43420833349227905, 1.0521225929260254, 0.24900202453136444, -1.686700463294983, -1.0932923555374146, 0.5423730611801147, -0.02179579623043537, -0.49837788939476013, 0.9375857710838318, -2.03083872795105, 1.1329494714736938, 1.4194133281707764, 0.3124949336051941, 0.029241735115647316, -0.20651394128799438, -1.0859909057617188, 1.5439155101776123, 0.014746353961527348, 0.8037877678871155, 0.8617105484008789, -0.5101728439331055, -1.8667937517166138, 1.0197211503982544, 0.8329210877418518, -0.7853653430938721, 0.04167521744966507, -0.46919703483581543, -1.0524293184280396, -1.027079463005066, -1.4650129079818726, 0.38786378502845764, 0.02163691073656082, 0.5755714774131775, -0.9474571347236633, -1.2950317859649658, 3.0937247276306152, -0.8457635045051575, 0.1351766139268875, 1.345225214958191, 0.30991145968437195, -1.3944175243377686, 0.8620533347129822, 0.45894289016723633, -1.1485319137573242, -0.8709040284156799, -0.8617464303970337, -0.35962972044944763, -0.4978070855140686, -1.0667163133621216, -0.2610572874546051, -1.8947299718856812, -0.7089227437973022, 1.2244372367858887, -0.47539371252059937, -0.12791670858860016, 1.119938850402832, -0.4890095591545105, -0.9648857712745667, 0.8858379125595093, -1.4550426006317139, -1.53593909740448, 2.2206976413726807, -1.00324285030365, 0.3489343225955963, -0.27960848808288574, 0.16139213740825653, 0.1470032036304474, 2.097181558609009, -0.23346558213233948, 0.942489743232727, -0.0387464240193367, -0.5036332011222839, 0.30871298909187317, 1.3427996635437012, 1.5311222076416016, 0.33220374584198, -0.1456623077392578, -0.15643806755542755, -0.6804634928703308, -0.5126059651374817, -0.1392173171043396, -0.5576075315475464, 0.8991868495941162, -0.3719547986984253, 0.020747000351548195, 1.03604257106781, -2.0126140117645264, -0.1833060085773468, -0.22265945374965668, -0.3499648869037628, -0.6455694437026978, 1.0395381450653076, -0.26025691628456116, -2.179481267929077, 0.4808438718318939, -0.6554259061813354, 1.0517091751098633, -0.7797898650169373, -0.3574601709842682, -0.9982448220252991, -0.5284889936447144, 0.3423956632614136, -1.4433122873306274, 0.28517070412635803], [0.7784955501556396, 0.06793449074029922, -0.514803946018219, 1.331040620803833, 0.6693274974822998, 1.3944635391235352, -0.4193365275859833, 0.3750245273113251, 0.27992403507232666, 0.08563654124736786, -0.5577784776687622, 2.3270347118377686, -1.9273803234100342, 0.22081933915615082, 0.023359280079603195, 1.8201943635940552, 0.33452701568603516, 1.9923365116119385, 0.12290316075086594, 1.6074111461639404, 0.43743160367012024, 0.421169638633728, 0.7148253917694092, -1.6367747783660889, 0.5629153251647949, 0.21044035255908966, 1.53421950340271, -0.8626327514648438, 0.9280507564544678, 0.3566395938396454, 0.3296961486339569, -0.8781200051307678, -0.996089518070221, -1.2843825817108154, -0.8517541289329529, -0.38297998905181885, -1.0934514999389648, 0.7792148590087891, -1.0136911869049072, -0.09668731689453125, -0.1596229374408722, -1.3892699480056763, -0.06546958535909653, 0.5426138043403625, -0.24493308365345, 0.2917236387729645, 1.0256479978561401, -0.33978691697120667, 2.6427242755889893, 0.7371924519538879, 0.7144659757614136, -0.3518907129764557, -2.6949050426483154, 1.0277752876281738, 1.9425814151763916, 1.1583667993545532, 0.467945396900177, 1.8076575994491577, 0.3004154562950134, -0.3461633026599884, -0.7304587960243225, -1.0191978216171265, 1.0061969757080078, -1.3748302459716797, -0.5858948826789856, -0.9778991341590881, -1.0076172351837158, -1.1516367197036743, -1.7885501384735107, -2.1727418899536133, -0.22582454979419708, 1.481898307800293, -0.7734870910644531, 0.028352344408631325, 1.2930208444595337, 0.20581991970539093, 0.19922184944152832, -0.3265048563480377, -0.41344431042671204, 0.07525963336229324, 0.3256140351295471, 0.5643637180328369, 0.42473194003105164, 0.9777933359146118, -0.7761995196342468, 0.0543951578438282, 0.39148303866386414, 0.5050889253616333, -1.3428211212158203, -0.6147419810295105, 0.7915239930152893, -0.21834832429885864, 0.66702800989151, 1.3423709869384766, 2.464043617248535, 0.4503685235977173, -0.34307172894477844, -1.7675305604934692, 2.531020402908325, 0.28978022933006287, -0.3604061007499695, 0.06502076238393784, 0.39454522728919983, 1.7053064107894897, 1.1606862545013428, -0.7904422879219055, -0.039714764803647995, 0.9206833243370056, -0.33008861541748047, 0.9425384402275085, 0.06697609275579453, 0.098091259598732, 0.09958183020353317, -1.5139659643173218, -0.8738493919372559, -0.587346613407135, 1.5046621561050415, -0.9312242865562439, -0.3490847051143646, -0.331252783536911, -1.567242980003357, 0.18536487221717834, -1.1711833477020264, 0.6282371878623962, 0.02187344804406166, 0.5647205114364624, -0.6663323640823364, 0.6167566180229187], [0.7988808155059814, -0.44050949811935425, 2.091951608657837, -0.49058547616004944, 0.8543189167976379, -2.111143112182617, -1.4340397119522095, 1.0720478296279907, -0.9066331386566162, 0.9806036353111267, -0.2696410119533539, 0.5468979477882385, -0.25154024362564087, -0.2871292233467102, -0.020183298736810684, 0.2715936005115509, -0.2020585536956787, -1.314071774482727, 0.0043203155510127544, 0.9296016693115234, -0.23451964557170868, 0.6992291212081909, -0.32510945200920105, -2.112910747528076, 0.061845000833272934, -0.6495487093925476, -0.38500526547431946, 0.5479701161384583, -0.11261434108018875, -1.3785431385040283, -1.326564908027649, -0.10062939673662186, -0.23908467590808868, 0.1640227735042572, -1.3116906881332397, -0.3551985025405884, 0.7709200382232666, -0.7577233910560608, -1.1265195608139038, -0.22616198658943176, -0.180565744638443, -0.8688139915466309, -0.9664681553840637, 1.1345144510269165, -1.0361841917037964, -0.8496327996253967, -1.288258671760559, 0.4552094340324402, 0.19029663503170013, 0.16432631015777588, -1.4564933776855469, -0.5264115929603577, -0.17727898061275482, 0.5742905735969543, -1.4096777439117432, -1.5545035600662231, 0.48186251521110535, 0.21856966614723206, 1.3922067880630493, -0.6853811740875244, 0.8810788989067078, 0.9330341219902039, 1.2671318054199219, -0.513321578502655, -0.06073705106973648, 0.06390943378210068, -0.6726316213607788, 0.5977107882499695, 1.1977595090866089, 0.661842942237854, 1.185810923576355, 0.018404614180326462, -0.6562920212745667, -0.9600383639335632, 0.07507740706205368, 0.37497082352638245, -0.3723040223121643, -0.5196231007575989, 0.18024417757987976, 0.4574210047721863, 1.0468865633010864, -1.3459076881408691, 0.898708164691925, -0.8494552969932556, -0.9479081034660339, -0.7203173637390137, 2.3545825481414795, -0.026523590087890625, -1.5342442989349365, 0.654104471206665, 0.6639343500137329, -0.1365469992160797, -0.662879467010498, 0.7198631763458252, -0.27428361773490906, -0.27803337574005127, -1.3410981893539429, 0.24598906934261322, -0.785373866558075, -2.1981921195983887, -0.040368568152189255, -1.034152626991272, -0.7936230301856995, -0.8129600882530212, 0.8182250261306763, 1.2297935485839844, 1.2919857501983643, -0.828790009021759, -2.0504584312438965, 0.8392096757888794, 0.6871976852416992, -0.8653042912483215, -2.3552958965301514, -1.0752205848693848, 1.172631859779358, 0.02190658077597618, 2.0872085094451904, -1.6941394805908203, -1.0910613536834717, -0.43259182572364807, 1.0019581317901611, -0.835748016834259, 0.35468146204948425, -0.1480976939201355, -0.15695568919181824, -0.5447918176651001, 2.3781473636627197, -0.7291768193244934], [0.6577692627906799, -0.013740727677941322, 0.44440990686416626, 1.7954440116882324, 1.1902966499328613, -0.07094230502843857, 0.34035614132881165, -0.592871367931366, 0.9502624273300171, 1.798869252204895, 0.43588975071907043, 1.7883672714233398, -0.9968492388725281, -0.653390645980835, -1.8266143798828125, -0.8766779899597168, 1.0557669401168823, 1.5034376382827759, 0.6348139047622681, 0.3732481896877289, 0.05823742225766182, 0.485832542181015, 0.4680902659893036, 0.9275346994400024, 1.832047462463379, -2.1615068912506104, 0.8795389533042908, 1.417637825012207, 2.467489242553711, 0.6321797370910645, -1.1579601764678955, -0.7004044651985168, 0.17361566424369812, 1.4585695266723633, 1.2158770561218262, 0.12480560690164566, -0.5212849378585815, -1.3136154413223267, -1.3358038663864136, 0.14361871778964996, -0.8096481561660767, 2.7943027019500732, -1.1124340295791626, 0.22740747034549713, 0.3692381978034973, -0.9909529089927673, -0.3962445855140686, 0.6290474534034729, -0.5213626623153687, 0.023114938288927078, 1.0895243883132935, -0.18077655136585236, 1.2677119970321655, 0.9480105042457581, 0.3089624047279358, -0.9266735315322876, -0.16929779946804047, -0.4885329604148865, -0.10024656355381012, 0.10826744884252548, -0.2855609953403473, -0.9676158428192139, -1.0821136236190796, -1.4864659309387207, 0.003897774266079068, 0.6895285844802856, -0.9702147841453552, 1.2561159133911133, 0.3164065182209015, 0.2708031237125397, -0.25131818652153015, 0.23468048870563507, 1.2924805879592896, -2.1387429237365723, 0.46762412786483765, -0.21494072675704956, 0.6789039373397827, -1.9310495853424072, -0.26341935992240906, -0.8317246437072754, -1.677965521812439, 0.658356785774231, -0.4109041392803192, 0.4586942493915558, 0.5143179297447205, 0.39303475618362427, 0.10844244807958603, -0.5850390195846558, 0.2697064280509949, 1.3547782897949219, -0.14443831145763397, -0.8620790839195251, -2.650566339492798, -2.32016658782959, 1.4532831907272339, 1.704384446144104, -0.6823781132698059, 1.184882640838623, 0.35110917687416077, 0.8791022896766663, 0.7141924500465393, -1.4905346632003784, -0.3810674250125885, 0.6917700171470642, -1.8409370183944702, 0.831173837184906, 0.2085004299879074, -1.2885607481002808, 0.4674087166786194, -0.33812883496284485, 1.048611044883728, 0.15774130821228027, 0.40368974208831787, 0.5072720050811768, 1.5490894317626953, -0.31806427240371704, 1.8552916049957275, 2.062481164932251, 0.8946250081062317, 0.5643910765647888, -1.5413730144500732, -0.9645097851753235, 2.494030475616455, -0.7790225148200989, 0.47526946663856506, -0.16771095991134644, -0.2542126476764679, -0.8117716908454895], [1.2662464380264282, 1.0901645421981812, -1.3589574098587036, -1.7910584211349487, 1.0794516801834106, -0.5570110082626343, 0.568062424659729, 1.4055900573730469, 0.21650750935077667, 0.5594049096107483, -0.5986489057540894, 0.3367898166179657, -0.7871056795120239, 1.3503423929214478, 0.5771210193634033, 0.9132013916969299, -1.6106822490692139, -1.7338529825210571, -0.9820176362991333, -0.43683871626853943, -2.7377195358276367, 0.032608699053525925, 0.3795975148677826, 0.24227522313594818, 0.9179980158805847, 0.3314916789531708, -0.5611608028411865, 1.3105002641677856, 0.15647824108600616, -0.2852634787559509, -1.2953294515609741, 0.8394254446029663, 2.303550958633423, -0.48326656222343445, -0.7838610410690308, -1.277180790901184, -0.2677053213119507, 1.0649183988571167, 0.10847318172454834, 1.1138454675674438, -0.20575295388698578, -0.09336713701486588, -0.6696023941040039, 0.2418612241744995, -0.619568943977356, -1.382031798362732, 0.9454168677330017, 0.24625328183174133, 0.3028644323348999, -1.2503259181976318, -1.7344495058059692, 0.5779421329498291, -0.1571209877729416, 1.1773909330368042, 0.004358052276074886, -0.3736761510372162, 0.06083473190665245, -0.46976062655448914, -0.5558640360832214, -0.49109092354774475, 0.5504609942436218, 1.0776984691619873, -0.23774453997612, -0.8429698348045349, -0.02153443731367588, -1.5125142335891724, -0.14189885556697845, 0.6503129601478577, -0.029282087460160255, 1.6369093656539917, 0.8965772986412048, -0.9338070750236511, -0.07308816909790039, 0.9460850358009338, -0.31136006116867065, 4.070145606994629, 0.785622775554657, 0.8079837560653687, -0.3032340705394745, 0.14927656948566437, 0.6247091293334961, -0.2576581537723541, 1.0416841506958008, 1.0017236471176147, -0.4204999506473541, -1.1296359300613403, 1.841766595840454, -0.6996138095855713, 0.39814433455467224, -0.23065820336341858, 0.8454810976982117, -0.7934810519218445, 1.1359115839004517, -0.8806864023208618, -0.38388791680336, 1.2982639074325562, 0.5222964882850647, 1.1186439990997314, -1.4035454988479614, 0.6133154630661011, 1.0989716053009033, -0.6576332449913025, -0.13555990159511566, 1.4488177299499512, 0.4235641658306122, -0.16496747732162476, 1.7625445127487183, -0.48511263728141785, -0.4337952435016632, -0.062342483550310135, 0.43106022477149963, -0.6090080738067627, 1.7725387811660767, -0.17337580025196075, -0.5735013484954834, -0.8563918471336365, 1.7006216049194336, -0.05187631770968437, 1.5671805143356323, 0.3861352205276489, 0.26188522577285767, 0.10616467148065567, -0.45711657404899597, -0.6901145577430725, 0.2552095055580139, -0.9392465949058533, -0.596758246421814, 0.9419976472854614], [-1.7508245706558228, -0.956864058971405, 0.6317293643951416, -0.012971495278179646, 0.057790279388427734, -0.28188633918762207, 0.5832607746124268, -0.00045191915705800056, -0.0830286517739296, 0.6831918358802795, -1.050714135169983, -0.47773435711860657, -0.3836902379989624, 0.22430497407913208, -0.6331962943077087, 1.10831880569458, -0.43067967891693115, 1.9737939834594727, -0.8698214292526245, -1.0764912366867065, -0.7997627258300781, 2.140493392944336, 2.02362322807312, -0.4823063611984253, 0.6407912969589233, -0.7305670976638794, 0.16981089115142822, -1.0069761276245117, -0.5503832697868347, -0.8337185978889465, 0.18292655050754547, 0.35014232993125916, -0.685144305229187, 1.060929775238037, -1.8061848878860474, 0.38261836767196655, -0.8499014377593994, 2.299259662628174, 0.48022907972335815, 0.2148895263671875, 0.9956347942352295, 1.240902066230774, 1.659227728843689, -2.7718253135681152, 0.05439144745469093, 0.10349059849977493, 0.47748124599456787, -0.7080819606781006, -0.572860062122345, -0.4435499608516693, -2.332512617111206, -1.1165099143981934, 0.13550961017608643, 1.0003539323806763, -1.0439199209213257, 0.3835500478744507, -0.85617995262146, -0.36270225048065186, 0.07690777629613876, 0.33636894822120667, 0.7654846906661987, 0.11771558225154877, 0.33875614404678345, -1.1341465711593628, -1.7227040529251099, -0.4318149983882904, -0.739555835723877, -0.45221224427223206, 0.5401580929756165, -1.5149927139282227, -1.7711451053619385, -1.1497148275375366, 2.628751516342163, -2.8730571269989014, 0.6733559370040894, 1.7157790660858154, -0.027079185470938683, -0.4488610029220581, -0.18847540020942688, 0.49531978368759155, -1.0923608541488647, -0.16271403431892395, -0.1556365042924881, 1.528696894645691, -0.014735018834471703, 2.008845806121826, 0.6268107295036316, -0.35507214069366455, -0.9440014362335205, 0.6806111335754395, 0.7568939924240112, 0.33085644245147705, -2.0350840091705322, 0.7812901735305786, 0.267792284488678, 0.8005865812301636, 1.1816184520721436, -0.507118284702301, 0.1267491579055786, -0.9691620469093323, -1.0630313158035278, 0.27640044689178467, 0.935296356678009, 0.7894397974014282, -0.13952027261257172, 2.598541021347046, -1.2405755519866943, -0.5007990002632141, 1.4761724472045898, -0.7484810948371887, 2.113210678100586, -0.6720487475395203, 1.3629517555236816, -0.6519812941551208, 1.2894755601882935, -0.7524855136871338, -0.27804243564605713, -1.653923511505127, 0.8805137872695923, -1.0565918684005737, -0.2638576328754425, -1.8460383415222168, -1.5575978755950928, 0.03298642113804817, -0.3049897253513336, 0.8633067011833191, 0.8732309937477112, -1.6233896017074585], [-0.10043763369321823, 0.15467588603496552, 0.3472708761692047, -0.9308657646179199, 0.3561696410179138, -1.1978559494018555, -0.3412090539932251, -0.5798500776290894, -1.2602367401123047, 0.5975593328475952, -1.6559205055236816, -2.151400089263916, -0.16209107637405396, -0.5710640549659729, -1.7813992500305176, -0.7340369820594788, -1.1258076429367065, -0.20446407794952393, -0.15207017958164215, 0.2846486270427704, -0.1169123724102974, 0.08513978123664856, 0.003533073700964451, -0.1827986091375351, -0.5116174817085266, 1.5893081426620483, -1.3316600322723389, 2.0675647258758545, -1.1738289594650269, -0.28083011507987976, -0.4499233365058899, 0.9309671521186829, -0.15386269986629486, 0.13404837250709534, 0.5292466282844543, -1.2282487154006958, -0.8672329187393188, 0.23752282559871674, 0.18135185539722443, 0.2690700888633728, -1.0050296783447266, -0.19732555747032166, 0.049127381294965744, 0.9325467348098755, -1.497273564338684, -2.909860134124756, 0.07945498824119568, -0.5534110069274902, -0.3732602298259735, -1.0415524244308472, -0.9367225766181946, -0.25688010454177856, 0.30656424164772034, 0.490330308675766, -2.6308884620666504, -0.8764863610267639, -0.31632378697395325, 0.48274263739585876, -0.29925280809402466, -1.5698901414871216, 0.3023394048213959, 0.9003807306289673, -1.625346064567566, 0.24620361626148224, 0.7705094218254089, 0.26967522501945496, 1.4307174682617188, 0.7844010591506958, 0.07184715569019318, -0.7661309838294983, -0.45573025941848755, -0.47090569138526917, -0.7352893352508545, 1.507068395614624, 0.7126128673553467, 0.251169890165329, -0.14427635073661804, 1.2434698343276978, 0.13955003023147583, 0.539944052696228, -0.6729288101196289, 1.335092544555664, -0.8382384777069092, 1.005664348602295, -1.2493407726287842, -0.08309168368577957, -0.10497617721557617, 0.4953141510486603, -0.610289454460144, 0.45955032110214233, 0.8123517036437988, 1.3759887218475342, -0.28291916847229004, -0.4774700999259949, 0.6415925621986389, 1.0055752992630005, 0.20550432801246643, -0.00816735252737999, -0.5810315608978271, 1.1439735889434814, -1.525044322013855, -0.9170241355895996, 0.541400134563446, -1.0513900518417358, 1.5602153539657593, 0.08232704550027847, -1.599181056022644, 1.4351075887680054, -0.43232256174087524, -1.3702113628387451, 1.1816391944885254, -0.7230983972549438, -1.4459654092788696, -0.2810973823070526, 0.8805646300315857, 0.6570196151733398, -0.9977238178253174, 0.01945527456700802, -0.27177971601486206, -0.9606083631515503, 0.6792412996292114, -0.6359763145446777, -1.465147614479065, -1.0898699760437012, 1.1015516519546509, 1.850250482559204, -0.4694526791572571, 0.30878347158432007], [-0.3228274881839752, 0.46419110894203186, -0.8673985600471497, 0.898249089717865, 1.1106805801391602, 0.24748271703720093, 0.21506960690021515, -0.23862873017787933, -0.7308666110038757, -1.1344853639602661, 0.06944833695888519, 0.739017903804779, -0.53364497423172, -1.3458915948867798, 0.5516777038574219, -1.487263560295105, 1.2385894060134888, 2.2067384719848633, 2.1726467609405518, 0.9339789152145386, -0.3205645680427551, -0.16749049723148346, 0.5122427940368652, -1.372936725616455, -0.9516602158546448, 0.11223627626895905, -0.5270904898643494, -1.5280505418777466, -0.1689440906047821, 0.6321263909339905, 1.0579781532287598, -0.22224019467830658, -1.6210190057754517, 0.007989342324435711, 0.07619871199131012, 0.5702911019325256, -1.5553207397460938, 0.47598955035209656, 0.3051699995994568, -0.6113201379776001, 1.4513531923294067, -0.2719718813896179, -1.30986487865448, 0.36050161719322205, 1.8237190246582031, -0.5721250176429749, 0.5984007716178894, -0.24113847315311432, -0.37865254282951355, -0.9597896337509155, -1.0151951313018799, 0.8851743936538696, 0.1458902359008789, -0.1397145390510559, 0.25271913409233093, 1.2867403030395508, 0.47068288922309875, -0.27743324637413025, -1.2563512325286865, -0.44723573327064514, -1.9380334615707397, -0.22250869870185852, 0.6347906589508057, -0.639567494392395, -1.4570980072021484, -0.7794401049613953, 0.24037230014801025, -0.17895975708961487, 0.814548671245575, -0.3583177328109741, 0.7175281047821045, 1.7098482847213745, -0.17341820895671844, -1.101914644241333, 1.0187939405441284, 0.36618170142173767, -1.1224114894866943, -0.7624340057373047, -0.6923097372055054, -1.3235411643981934, -1.27741539478302, -1.2093671560287476, 0.6786129474639893, 1.9734392166137695, 1.4127023220062256, -0.8450702428817749, -0.23637957870960236, -0.5003357529640198, -0.33003172278404236, -1.2405779361724854, 0.6144420504570007, 0.07145082205533981, -1.037983775138855, -0.8153330087661743, 0.3213020861148834, -1.789711833000183, 1.3902063369750977, -0.6714192628860474, -0.6984304785728455, -0.5394465327262878, -0.5887278914451599, -0.7706291079521179, 0.7240012884140015, -1.639458417892456, 0.0855536237359047, -0.7651957869529724, 0.23113329708576202, -1.0225921869277954, 2.2592968940734863, 0.46536073088645935, 0.048894695937633514, -0.5024337768554688, 0.47351744771003723, -1.1108731031417847, 1.452593445777893, 0.8878452777862549, -0.015028415247797966, -0.467648446559906, -0.4504026472568512, -0.6878039836883545, 0.78526771068573, 0.4978429079055786, 0.25307583808898926, 1.8919792175292969, -3.4727423191070557, -0.44098415970802307, 0.2118709534406662, -0.25824445486068726], [1.0916978120803833, -0.9969479441642761, -0.22349055111408234, 0.21351245045661926, 0.7231088280677795, 0.26312676072120667, -0.2535722255706787, 0.5708814263343811, 0.3426154851913452, -0.37746599316596985, -0.056947898119688034, 0.6322729587554932, -0.25559836626052856, -0.8864083886146545, -2.1903648376464844, 0.42004624009132385, -1.7070989608764648, -1.139178991317749, -1.2470309734344482, -0.5977568030357361, -1.5387163162231445, 0.598463237285614, 0.41735175251960754, 0.33817747235298157, -0.16233929991722107, 0.928428590297699, 0.15425661206245422, 1.4944219589233398, -0.11948744207620621, 0.26133477687835693, -0.8768449425697327, -1.1801377534866333, -0.6782549023628235, -0.2171647548675537, -0.33793821930885315, 1.284053087234497, -0.20329037308692932, 1.0533274412155151, 0.040821343660354614, -0.1588037610054016, -1.3756043910980225, 0.5818753838539124, 0.8271087408065796, -0.5429081320762634, -0.5161476731300354, 0.3458133637905121, -0.49899303913116455, 0.10620395094156265, -0.649306058883667, 1.331743836402893, 2.145559549331665, -0.5490795969963074, -0.8843675255775452, -1.1741938591003418, 1.3976330757141113, 0.12644687294960022, 0.20731203258037567, -0.3834514021873474, 0.7771092057228088, 0.8509825468063354, 0.8849782943725586, -0.24120189249515533, -0.0370221883058548, -0.4314071536064148, -1.5489087104797363, -1.4812843799591064, 1.5885224342346191, -0.6257050633430481, -1.798685073852539, 0.02869066782295704, -0.8920248746871948, -0.5877160429954529, -1.1820225715637207, -1.2811238765716553, 0.7045457363128662, 0.5358037352561951, -0.3870636224746704, -0.05087084695696831, 1.0944472551345825, -0.9616573452949524, -1.6407451629638672, 0.7678155899047852, -0.9362630248069763, 0.8564590215682983, 1.599166989326477, 0.7849360108375549, 1.21181321144104, 0.5622570514678955, -0.6103745102882385, -0.5302972197532654, 0.23870228230953217, 1.3753619194030762, -0.08885704725980759, -0.7899497747421265, -0.8294513821601868, 0.42855820059776306, -0.6341509222984314, -0.03756032884120941, 0.6870814561843872, -1.9699492454528809, 0.23908524215221405, -0.460532009601593, 0.6196141242980957, -1.2454873323440552, -1.0659514665603638, -2.3529207706451416, 2.273806095123291, -0.4961378872394562, 0.50517737865448, -0.6640245318412781, -1.2831740379333496, 0.1102583110332489, -1.6316094398498535, 0.8627516031265259, 1.196542739868164, 0.5398112535476685, -0.7863456010818481, -1.0175296068191528, -0.2879568338394165, 0.2854112684726715, -1.8212047815322876, -1.7891584634780884, -0.22335729002952576, 1.1953661441802979, -0.5732731223106384, 0.2809734344482422, -1.393864631652832, -2.657000780105591], [0.10801074653863907, 0.7130325436592102, -0.0530523806810379, 0.5994076132774353, -0.261074036359787, -0.3050922751426697, 0.02161373384296894, 0.8931178450584412, -0.4625117778778076, 1.4010485410690308, -0.9729540348052979, -0.3741503059864044, -1.216395378112793, -0.7557680010795593, 1.190603494644165, -0.6163904666900635, -0.4530095160007477, -2.3233699798583984, 0.1146908700466156, 0.21329261362552643, -0.6601165533065796, -1.7951695919036865, -1.2313343286514282, -0.3505789637565613, 0.8104461431503296, 0.5941327214241028, -0.30585891008377075, 1.1298112869262695, 1.2502079010009766, -0.21211108565330505, 0.9070236682891846, -1.2193032503128052, 0.7286275625228882, -0.8317590355873108, 0.6038010716438293, -0.43010714650154114, -0.4054848253726959, 1.6512258052825928, 0.7655615210533142, 1.0215367078781128, 0.48028963804244995, -1.084513545036316, 1.1983689069747925, 0.21388104557991028, 0.22341884672641754, -0.18550609052181244, -1.6477060317993164, 0.040500886738300323, 0.9105761647224426, 0.962282121181488, -0.09929393976926804, -1.4018157720565796, -1.03939688205719, 0.05610695853829384, -1.0468708276748657, -0.6801742911338806, 0.541314959526062, 0.09238624572753906, -0.2652692496776581, 0.43786439299583435, 1.407408356666565, 0.44500431418418884, -2.659130334854126, -0.017200538888573647, 0.973388135433197, 1.5649969577789307, -1.1697224378585815, -1.2624635696411133, -0.33058303594589233, -1.2478218078613281, -0.39127665758132935, 0.4432879388332367, 0.3426819443702698, -0.06905169039964676, -1.1570783853530884, -1.0418577194213867, -0.5737116932868958, -0.3752514123916626, 1.0064916610717773, -0.45286107063293457, -0.6424327492713928, -1.2212949991226196, -1.2668201923370361, 0.5936151146888733, 0.8458366394042969, -0.7712200284004211, -0.43798375129699707, 0.28258708119392395, -0.049002230167388916, 2.0916671752929688, 0.7007589340209961, -0.17973218858242035, 2.1023151874542236, -1.1151305437088013, -0.6073752045631409, 0.4479610323905945, -0.016557011753320694, 0.9217451810836792, 0.13231097161769867, 0.9127604365348816, -0.8077518343925476, 0.036144208163022995, 0.20840097963809967, -0.2802910804748535, 0.9102115035057068, -0.1855277121067047, 0.9584811329841614, 0.953410267829895, 0.9167531132698059, 2.196934223175049, 0.04968128353357315, -1.5366718769073486, -0.07018526643514633, 0.13994251191616058, 1.0343053340911865, 0.9406998753547668, -0.0198203232139349, -0.7890512943267822, 1.0150901079177856, 0.3495604991912842, 0.275351881980896, -1.6520347595214844, -0.06493756175041199, 0.39513304829597473, 0.7304436564445496, 0.2816801071166992, -0.6489008665084839, -0.1797885149717331], [-2.5680153369903564, -2.037867546081543, -0.41712409257888794, -0.7546194195747375, -0.10132108628749847, 0.0866854339838028, 0.07840820401906967, -1.7643916606903076, -0.3792417347431183, -0.25129422545433044, 1.222120761871338, -0.9219468832015991, 1.436276912689209, 0.43920406699180603, 0.3754327595233917, 0.08543969690799713, -0.32513248920440674, 2.5589303970336914, -0.5213510990142822, -1.3414056301116943, -0.889634907245636, 1.016410231590271, 0.6664161682128906, -1.1219074726104736, -1.4771705865859985, -0.6648755073547363, 0.02353198267519474, -1.3261470794677734, -0.8507477045059204, 0.11760615557432175, 1.8056585788726807, -0.9710943102836609, 0.8792626857757568, -0.17992399632930756, 1.6551686525344849, -1.1364175081253052, 0.1413184255361557, -0.5021072030067444, -0.9261966347694397, -3.1199119091033936, -2.690019369125366, 0.33249977231025696, 0.5262916684150696, 0.938129723072052, 0.7775787115097046, 0.36693498492240906, -0.7273526191711426, -0.043429095298051834, -1.2022238969802856, 0.1775670051574707, -1.3522123098373413, 1.2766978740692139, -0.7523849606513977, -0.2943413257598877, 0.11028508841991425, -0.19638486206531525, -1.2542625665664673, 1.1954528093338013, 0.7090699672698975, -0.5085800290107727, 0.23618990182876587, 1.2477632761001587, 0.882868766784668, 0.9108479022979736, 0.05128949508070946, -0.9047630429267883, 1.0084831714630127, 0.19161652028560638, -0.9692471027374268, -0.4949021637439728, 0.21590428054332733, 1.1044656038284302, -1.3081212043762207, -0.39662662148475647, 0.8786845207214355, -0.2789864242076874, -0.24861785769462585, -0.3859303593635559, -0.24099856615066528, 0.7988790273666382, 0.6191287040710449, -1.432895302772522, 1.2052322626113892, 0.6737126111984253, 0.6332087516784668, 0.1187625527381897, -0.06471513956785202, 0.3192063271999359, 2.0573246479034424, 0.04986030235886574, 0.144041046500206, -0.990277886390686, 0.4722279906272888, 0.8362482190132141, -1.7695330381393433, -0.023056620731949806, 1.2392566204071045, 0.3095199465751648, -1.1142984628677368, 0.14111563563346863, -1.0738584995269775, -0.7967439889907837, -0.10559234768152237, -0.6896714568138123, -0.1308225691318512, -1.2143110036849976, -0.8036483526229858, 0.3263472318649292, 0.12025447934865952, 1.386400580406189, 1.6441173553466797, -1.806006908416748, 0.5293245315551758, 0.4515140652656555, 1.681785225868225, -0.36694443225860596, 1.7191407680511475, -0.04262359067797661, -1.0075756311416626, 0.001571682165376842, 2.0400023460388184, 0.37984421849250793, -0.6695928573608398, -0.13454271852970123, 0.07750750333070755, 0.8433167338371277, 0.017012521624565125, 0.5639075040817261], [-0.29301294684410095, -0.5117207765579224, -0.047153353691101074, 1.7328238487243652, 0.08541475236415863, 0.6510291695594788, -0.40550172328948975, 0.35074982047080994, -0.705888032913208, 0.12290439009666443, 1.1274052858352661, 0.046385280787944794, -0.5246190428733826, -0.3676616847515106, 0.13669134676456451, -1.7332091331481934, 1.5656076669692993, -1.2361010313034058, -0.9648048281669617, -1.1510639190673828, -1.4550650119781494, 0.46454906463623047, -0.06023162603378296, -0.4295594096183777, 1.5138622522354126, -0.9090766310691833, -1.0867257118225098, 0.92716383934021, -1.750387191772461, 0.561937689781189, 0.4844038188457489, 0.13580548763275146, 0.7589899301528931, 0.8393384218215942, -0.9832984805107117, 0.4513595998287201, -0.4574872851371765, 2.0952272415161133, 0.6545657515525818, -1.2986078262329102, -0.5154010653495789, -0.03464307263493538, -0.2984195947647095, -0.9722693562507629, 0.29489797353744507, 0.3570913076400757, -0.2185557335615158, -0.06119030341506004, 1.0813679695129395, 1.8962981700897217, 0.04870949313044548, 0.16487663984298706, -0.1818239837884903, 0.2303534895181656, -0.9653350710868835, -0.1801760047674179, 1.216741681098938, -1.2131232023239136, -0.23107433319091797, 0.8616904616355896, -1.5203876495361328, 0.04837630316615105, 2.1967613697052, -1.2653664350509644, 0.566852331161499, 2.3821020126342773, -0.2351081371307373, -0.31377112865448, 0.18908289074897766, 2.2227184772491455, 0.8944053053855896, 0.9791341423988342, 0.3130338191986084, 2.13767147064209, 0.033665262162685394, 0.05538913607597351, 0.7065449953079224, -0.511384129524231, -0.7354069948196411, 0.09568088501691818, -0.40034523606300354, 0.36217615008354187, 2.421133279800415, -0.6287115216255188, -0.7453709840774536, -0.5469157099723816, -0.3196083903312683, -0.4113878011703491, -0.7592533826828003, 0.2674902081489563, 1.9545667171478271, 0.23809416592121124, -0.2714060842990875, -1.4414410591125488, 2.057878255844116, -0.1410675346851349, 1.3094687461853027, -0.0805668756365776, 1.2908639907836914, -1.4427214860916138, 1.016168236732483, -0.5322314500808716, -0.891626238822937, -0.6613464951515198, -0.2567649781703949, -0.20527751743793488, -0.706437885761261, 0.7191603183746338, -1.335182785987854, -0.9018867015838623, 0.7278051376342773, 1.3186781406402588, -0.8280376195907593, 1.3206373453140259, 1.2530893087387085, -0.7005187273025513, -0.24555830657482147, -0.13307641446590424, -0.5757432579994202, 1.186238408088684, 0.06268863379955292, 1.2488938570022583, -0.30982667207717896, -0.4489828646183014, 0.10797980427742004, 0.5539706945419312, -0.8249711990356445, -0.11233014613389969], [1.1040349006652832, -0.5243716239929199, 1.4322072267532349, 0.4569249749183655, -0.4487699568271637, 0.17324723303318024, -0.0687679871916771, -1.358426570892334, -0.10597898811101913, 1.6815649271011353, -1.6760889291763306, 2.7447893619537354, 0.946868896484375, 1.0260368585586548, -1.4537453651428223, 0.8721384406089783, 0.8656963109970093, -0.024273410439491272, -1.1188230514526367, 1.1032370328903198, -0.7998170256614685, 0.7325130701065063, -0.12844157218933105, 0.8576726317405701, -0.4519490599632263, -0.44523099064826965, 0.28366097807884216, 0.9397870898246765, 0.09579409658908844, 0.2680710554122925, -0.16539564728736877, 1.109381914138794, 1.0708749294281006, 1.42513906955719, 1.0189690589904785, 1.221760630607605, -0.7238667607307434, -1.5043613910675049, 2.3804893493652344, -0.7936645746231079, 1.0955044031143188, -2.1288206577301025, 0.19773364067077637, -1.0260370969772339, 0.17603816092014313, -0.9981356263160706, 1.2656633853912354, 0.5233103036880493, 0.016920307651162148, 1.1201368570327759, 0.1758001744747162, -0.3588109612464905, 0.17508229613304138, 0.6890761256217957, -0.3201308250427246, -1.9027191400527954, -1.2587506771087646, -1.3284797668457031, 0.19777067005634308, -0.14335757493972778, -0.4947666525840759, 0.5207374095916748, 0.6297401189804077, -1.76055908203125, -1.3416942358016968, 0.5835238099098206, 0.4683546721935272, -0.197986900806427, 1.341530203819275, 0.16480320692062378, -0.3471849262714386, -0.948932409286499, 2.880214214324951, -1.6284608840942383, 0.8762430548667908, -0.08830355107784271, 0.9501641392707825, 1.436347246170044, 1.7509007453918457, -0.13047920167446136, 1.3113245964050293, 0.12155944854021072, 0.16543686389923096, -0.14446493983268738, -0.6802000403404236, 2.08709454536438, 0.5513028502464294, 1.0083240270614624, -0.9542283415794373, -0.12066148221492767, -0.8596326112747192, -0.3567759394645691, -1.0620092153549194, 1.9133741855621338, 0.4483244717121124, -1.3951828479766846, -1.2735614776611328, 0.8775023221969604, -1.2593275308609009, 1.9421395063400269, 0.683829128742218, -1.5794801712036133, -0.3866443634033203, -0.7245423197746277, -0.4570971727371216, 0.33714258670806885, -0.3624008893966675, -0.023401740938425064, 0.9672571420669556, 1.1289637088775635, -0.9439542889595032, -1.8003513813018799, 1.0255147218704224, -1.9262453317642212, -0.935064971446991, -1.7902894020080566, 0.12978090345859528, -0.3672528862953186, -1.4687758684158325, -0.1961117386817932, -0.16031105816364288, 2.061436653137207, -1.3364535570144653, 0.6995668411254883, 0.6872332692146301, -0.0477355532348156, 0.3015505075454712, 1.8931896686553955], [-2.1983509063720703, 1.1538547277450562, 1.0162713527679443, -0.3592570424079895, -0.13957037031650543, 0.8601691722869873, 1.76197350025177, -0.9526141881942749, -1.644717812538147, -0.2312958985567093, 1.2998793125152588, 1.8000966310501099, -0.6814799308776855, -1.6013813018798828, 1.0255106687545776, -0.7615025043487549, -0.7465910315513611, 0.7168673276901245, -0.7155747413635254, -0.33915212750434875, 1.5398582220077515, -0.3230360746383667, -0.14374490082263947, -1.6642054319381714, 1.6935667991638184, 0.2715281844139099, -0.8109509348869324, 1.1482956409454346, -0.8115381002426147, -0.7922084331512451, -0.765957772731781, 0.9670405387878418, -1.0740227699279785, -1.849827527999878, -0.21414127945899963, 1.0094060897827148, -1.5300472974777222, -0.16588255763053894, -0.2865840792655945, 0.6082922220230103, -0.02027895487844944, 1.2144538164138794, 0.7152758240699768, 0.842643678188324, 1.3478617668151855, 0.5189858078956604, -0.727276623249054, 0.6888085007667542, -0.8507715463638306, -1.1151814460754395, -0.5512176752090454, 0.4321081340312958, -0.6673133969306946, -1.3775891065597534, 0.5707737803459167, -1.6190955638885498, -0.4691448509693146, 0.8407585024833679, 0.05809151753783226, -1.1415201425552368, 0.3115180730819702, 0.5790929794311523, -0.6861457824707031, 1.4762464761734009, 0.32634177803993225, 0.12426389008760452, 0.582820475101471, -1.3375314474105835, -3.1785600185394287, 0.8664745688438416, -0.8708583116531372, 0.3358452022075653, -0.329986035823822, -0.20127429068088531, 0.9530785083770752, -2.0573740005493164, -1.1060863733291626, 0.9036985635757446, -0.5616366863250732, -1.073302149772644, -0.3723292052745819, 2.3551297187805176, -1.9002593755722046, -0.8899940252304077, -0.6540616750717163, 0.789135754108429, 0.4240608215332031, -1.4186675548553467, -2.4246091842651367, -0.7986140251159668, 2.233645439147949, 1.0668505430221558, -0.7642045021057129, -0.008999284356832504, 0.1415405571460724, 1.715382695198059, 0.402036190032959, -0.4350023567676544, 0.7580145597457886, 1.7158257961273193, 1.004608392715454, -0.3716204762458801, -1.581382155418396, 0.007871906273066998, 3.203674077987671, 1.1320146322250366, -0.6340521574020386, 0.9178913235664368, -0.21136924624443054, -0.9749010801315308, 0.8350928425788879, 1.4024080038070679, -0.5702731609344482, -0.938101589679718, -1.5607727766036987, 0.2652847170829773, 0.7162383794784546, -0.24023203551769257, 0.0946064218878746, 1.409201979637146, 0.06939224153757095, 0.4243139922618866, -0.5689350962638855, 0.474308043718338, -0.10022949427366257, -1.319820761680603, 0.3665134608745575, -1.9272319078445435], [-1.6582722663879395, 1.8152028322219849, -0.8217580318450928, 0.6998056173324585, -0.6405123472213745, -0.03367982059717178, 0.05124767869710922, -0.2915644645690918, 0.5066749453544617, 1.5039217472076416, -0.9094992280006409, 0.5888862013816833, -0.8750404715538025, -0.44437485933303833, 0.7754142880439758, -0.27317574620246887, -0.6424074172973633, -1.5989406108856201, 0.6638787984848022, -0.677864670753479, -0.36373287439346313, 0.7201403379440308, -0.08849870413541794, 1.7611021995544434, -0.08499530702829361, 1.046735405921936, -1.4295039176940918, -0.060980424284935, -0.5307530760765076, 0.1138404831290245, -0.20457175374031067, 0.10633433610200882, -1.0202077627182007, -1.339804768562317, 1.4740476608276367, 2.1815779209136963, -0.41643011569976807, -1.0354689359664917, 0.19631071388721466, -0.2086462676525116, -0.07703003287315369, 1.7313833236694336, 1.6146749258041382, 0.05755836144089699, 1.2111090421676636, 1.3770891427993774, 0.7339426875114441, -0.2487035095691681, 0.8421690464019775, 1.342926263809204, 0.24869868159294128, 0.1911277025938034, -1.9358267784118652, 0.3147481679916382, -1.9807602167129517, -0.5554928183555603, 0.940895140171051, 0.46175438165664673, -0.7238889336585999, -0.14996781945228577, -0.1627904176712036, 0.2059730440378189, -0.6835381984710693, -1.0956575870513916, -1.1233103275299072, -2.090264320373535, 0.6867832541465759, -0.627177894115448, -0.3991186320781708, -1.2601816654205322, -0.25876694917678833, -0.2462170273065567, -0.5453529357910156, -1.4101014137268066, -1.9251059293746948, 0.01014865655452013, -0.5446144938468933, -0.38922497630119324, 0.9221302270889282, 0.22901371121406555, -0.04554038867354393, -0.5302571654319763, -2.632070302963257, 1.3873857259750366, 0.028066476806998253, 0.8532500863075256, -0.361503928899765, 1.396324872970581, -1.0692970752716064, 0.6754258871078491, -0.6880345344543457, 1.2246527671813965, -0.6642816066741943, -0.40864405035972595, -1.4214485883712769, 1.2322494983673096, 1.8689771890640259, 0.1566045731306076, -0.017896397039294243, -0.15692563354969025, -0.2910943031311035, -1.0981318950653076, 0.5697553753852844, -0.38005006313323975, 0.5772393941879272, 0.47284063696861267, 0.03903411328792572, -0.7270011305809021, 0.6365675330162048, -1.4784015417099, 0.7007612586021423, -0.5761800408363342, -1.6002103090286255, -0.14450719952583313, 0.7289884090423584, 2.4061715602874756, -1.6616342067718506, -0.8750489950180054, -1.3145428895950317, 0.17384234070777893, 1.1477675437927246, 0.05658344551920891, -1.4749457836151123, -0.5522444248199463, 1.905466914176941, -0.5551458597183228, -0.04572694003582001, 1.0609052181243896], [2.741379976272583, -0.9791149497032166, 2.380375862121582, -0.6947084665298462, 0.055026426911354065, 1.1701452732086182, 0.8707839846611023, 0.009831253439188004, -0.07206220924854279, -1.0966804027557373, 0.9575424194335938, 2.1386525630950928, 0.479726642370224, -0.8201584815979004, 0.9629904627799988, 0.07783959060907364, -0.2623443305492401, -0.2218075692653656, 1.5520292520523071, 0.2836545407772064, 1.459025502204895, 0.8040663003921509, 0.6577613949775696, -1.5588117837905884, -0.7766106128692627, -0.840302050113678, -0.9426904320716858, 0.6771324276924133, -1.2609245777130127, 0.12965820729732513, -0.9488754868507385, 0.8691423535346985, -0.3697461783885956, 0.12400899082422256, -0.4659135043621063, -0.07665155827999115, -0.30288636684417725, 0.22490432858467102, 0.9891066551208496, -0.5610127449035645, -0.6568452715873718, 0.8902923464775085, 1.2456741333007812, 0.758974015712738, 0.49324801564216614, 2.178778886795044, -0.9024634957313538, 0.2717544734477997, -2.4827420711517334, 0.774318277835846, -0.5282202959060669, -0.06665422022342682, -0.7824929356575012, 1.4662600755691528, -0.5428904891014099, -1.0598323345184326, -1.3170734643936157, 0.8903560638427734, 1.419588565826416, -0.6855489611625671, 0.41322919726371765, 1.1435234546661377, -0.3148006498813629, 0.28298497200012207, -0.4718025028705597, -1.0023542642593384, -0.4741540551185608, -0.4219019114971161, -0.897763729095459, 1.134598970413208, 0.7491405010223389, 0.1461089700460434, -0.4753847122192383, -0.6134268045425415, -0.7072392702102661, -0.9308850169181824, 0.36159831285476685, 0.367247611284256, -0.6542856097221375, -0.2039899379014969, -0.2466590851545334, 0.5067643523216248, 0.2553035318851471, -0.1716197282075882, 0.3808414340019226, -1.004284381866455, 0.1904747486114502, 0.33589744567871094, 1.2550795078277588, -1.0775015354156494, -3.0990545749664307, 2.316504955291748, -0.21717733144760132, 0.21526047587394714, 0.42221370339393616, 0.4178839325904846, -0.7156935930252075, 0.26238346099853516, 0.6276099681854248, 1.7868646383285522, 0.7548448443412781, 0.3007771968841553, 1.3394086360931396, 0.18020343780517578, 1.3239959478378296, 1.1248993873596191, -0.11915937066078186, 0.06728798896074295, -1.4921865463256836, 0.37820714712142944, 0.9042598009109497, 1.361154317855835, 0.5836473703384399, -0.7761393785476685, -0.5672307014465332, -0.9024691581726074, -1.7208980321884155, 1.232243299484253, 1.7319310903549194, 1.347485899925232, -0.4501558542251587, 1.1041189432144165, -1.4131876230239868, -0.2998499572277069, -1.0551605224609375, -1.1180018186569214, 0.030750280246138573, 1.6568613052368164], [0.3437540829181671, -0.7898485660552979, -0.268258661031723, 1.5138757228851318, 0.7229534387588501, -0.6721785664558411, 0.3048588037490845, 0.4733315110206604, -0.6852912902832031, -0.7540512681007385, 0.4344664514064789, -0.1708626002073288, -2.4984042644500732, 1.0701267719268799, -0.6908255815505981, 0.10243135690689087, -1.337996006011963, 0.21564747393131256, -0.641186535358429, -0.6367049813270569, -0.4395933449268341, 0.29530128836631775, -0.0658746287226677, -0.6244124174118042, 0.4628995954990387, -0.41788291931152344, 1.550713062286377, 0.1628231555223465, 0.4980614483356476, -0.554551899433136, 0.18310457468032837, 0.9433995485305786, -0.3050979673862457, 0.07995833456516266, -1.6364734172821045, -0.7581316232681274, 0.5062814950942993, -0.9917079210281372, 0.6854583024978638, -0.662527859210968, -2.616527795791626, 0.9623667001724243, -1.235597014427185, -0.8317034244537354, -0.2707887291908264, 1.1341984272003174, 0.4274579584598541, 1.3814857006072998, -1.1582101583480835, 0.8006638288497925, 0.12621887028217316, -2.8219010829925537, -0.5391626358032227, 2.1585185527801514, 1.1869016885757446, 0.7095823884010315, 0.43835169076919556, 0.16247934103012085, -1.2733159065246582, -0.9390549063682556, -0.5493858456611633, 0.7592384219169617, 1.1851396560668945, 0.3303222060203552, -0.7792635560035706, -1.0599706172943115, 3.1631083488464355, -1.1105729341506958, -0.14751969277858734, 0.528886616230011, -0.38550591468811035, -0.9866107106208801, -0.5575032234191895, -0.5044329166412354, 0.4320509433746338, 0.005583429709076881, -1.611415982246399, 0.7773807644844055, 0.9284956455230713, 0.8047935366630554, -2.6487302780151367, 1.1694484949111938, 0.5059714317321777, 1.0313717126846313, 0.2129773497581482, 2.823315143585205, 1.5163590908050537, 0.2970964014530182, 0.2840355336666107, 0.5277242660522461, 0.03906045854091644, 1.7034130096435547, -0.27788007259368896, 0.9636680483818054, 0.6807608008384705, 0.2976062595844269, -0.6141433715820312, 0.7273126840591431, 0.41988080739974976, 0.13559018075466156, 0.06095098331570625, -1.3390684127807617, -0.8671966195106506, -1.1212064027786255, -1.3663712739944458, -2.700394630432129, 0.5598551630973816, -0.3459327518939972, -0.37582969665527344, 0.6277584433555603, 2.0763700008392334, 0.28870537877082825, -1.460693120956421, 0.325285941362381, -0.13247597217559814, -0.9823231101036072, 1.4935551881790161, -0.6199339628219604, -2.390984535217285, 0.014821229502558708, -0.07190252840518951, 0.26133015751838684, 0.9883559942245483, -1.3369762897491455, -0.25387540459632874, 0.9315527081489563, -0.5999749302864075, 0.40444138646125793]]]], "dtype": "float32"}, {"__nd__": [[[[-0.994406521320343, -0.4068412482738495, -1.980467677116394, -0.2520065903663635, -0.08447200804948807, -0.8371936678886414, -0.5554162263870239, 2.1056530475616455, -0.34393009543418884, -0.4218425750732422, -0.9615739583969116, -0.2665683627128601, 0.6473016142845154, -0.8416646122932434, -0.2094932347536087, -0.18271546065807343, 1.1790416240692139, -0.49511367082595825, 1.1865273714065552, -1.3398666381835938, 0.09736552834510803, 0.5881792306900024, -1.2445929050445557, 0.5006079077720642, 1.0952988862991333, 0.27261972427368164, 1.6576223373413086, -1.4918655157089233, 0.9379808306694031, -1.6438535451889038, 1.223467230796814, -1.1794687509536743, 0.14580830931663513, 0.34987813234329224, 0.545820951461792, 1.2735278606414795, -0.12282083928585052, -0.1275598704814911, -0.8039188385009766, -1.0931892395019531, 1.3677904605865479, 0.43393364548683167, 0.8838222622871399, 0.09003198146820068, 0.40381693840026855, -0.6220930814743042, -0.957719624042511, -2.4216582775115967, 0.539472222328186, 0.37584567070007324, 1.4115334749221802, -0.9329919815063477, -1.497185230255127, -1.5941965579986572, -0.342168390750885, -1.5270631313323975, 0.09941620379686356, 1.2935978174209595, 0.7982949614524841, 0.46097975969314575, 0.48615798354148865, 1.1620252132415771, 0.6265010237693787, 0.031642042100429535, 0.8473230004310608, 0.08624465018510818, 0.4207962453365326, -1.0868662595748901, -0.40067800879478455, 0.34896203875541687, 2.3464858531951904, 0.5252850651741028, -0.02636692486703396, -0.9240217208862305, 0.09634452313184738, -1.883488416671753, -0.8783166408538818, 1.6370956897735596, 0.17806367576122284, 0.5404555797576904, 0.22589124739170074, 0.7057872414588928, 0.6086617708206177, 0.9938701391220093, 1.3195544481277466, -0.3100302815437317, 0.34754741191864014, -0.12445548921823502, -0.2707962691783905, -1.20740807056427, -1.5952668190002441, 0.7503791451454163, -0.692531168460846, 0.38839221000671387, 0.5097962021827698, 1.0173277854919434, -1.651349425315857, 1.3724561929702759, 0.10963144898414612, -0.29150471091270447, 0.03823770582675934, 0.839251697063446, -0.4407368302345276, -0.6505641341209412, -0.33931636810302734, -0.9203778505325317, -0.04494885727763176, 0.2461172491312027, 0.6185469627380371, -0.37478262186050415, 0.1786646693944931, 1.2674375772476196, 0.6656549572944641, -1.1635798215866089, 0.6251600384712219, 1.5172606706619263, -0.9777970314025879, -1.1470084190368652, 0.037047192454338074, -0.49608394503593445, 0.8534521460533142, 0.7812005877494812, -1.8895963430404663, 0.8900980949401855, -0.05547809973359108, -1.606281042098999, -0.18639203906059265, -0.27389809489250183], [-0.18028143048286438, 0.2978173494338989, 0.49785342812538147, 0.6834580302238464, 0.6326103806495667, 0.8489252328872681, -0.9047983884811401, -1.509171724319458, 1.1545591354370117, -0.8632229566574097, 1.9393495321273804, -0.5046247243881226, 0.12776707112789154, 0.00625717593356967, -0.5449613928794861, 0.42505648732185364, -1.7368415594100952, -1.1328113079071045, -0.7860301733016968, 0.3078652024269104, 0.04621842876076698, -0.5668267607688904, 0.8024720549583435, -1.0545721054077148, -1.5531302690505981, -1.0716197490692139, 0.49461543560028076, 0.4181952476501465, 0.4543006420135498, -0.5718299746513367, 0.769222617149353, 2.11457896232605, -0.2606867551803589, -0.11044256389141083, -0.44410157203674316, 1.3496400117874146, 0.31952714920043945, 0.18913908302783966, -1.352250576019287, -0.7209278345108032, 0.1675838679075241, -0.33068302273750305, 0.1131337359547615, -0.5696310997009277, 0.2616907060146332, -0.25847873091697693, -0.7696475982666016, -2.330352306365967, 0.33761632442474365, -0.055222537368535995, -1.1031715869903564, 0.8928857445716858, 0.7178862690925598, -0.9209778904914856, -0.47589826583862305, -0.6036482453346252, -0.3075607419013977, -1.8943933248519897, -1.1099085807800293, -0.9210747480392456, -0.3032209575176239, 0.2377922087907791, -0.6912420988082886, 3.6906793117523193, -0.02613556943833828, -1.393922209739685, -0.7780729532241821, -0.23055323958396912, -0.4724963307380676, 0.3167039752006531, -0.88743656873703, -0.3367496430873871, 0.7020028233528137, 0.5635460019111633, 0.3990597426891327, 0.0563453808426857, -0.6283668875694275, -0.23151792585849762, 1.068333387374878, 0.3208121061325073, -0.9756996035575867, -0.6251290440559387, 1.0145196914672852, 0.24332840740680695, 0.13783566653728485, -0.4019991159439087, -1.0231711864471436, 0.5456242561340332, 0.5467824935913086, -1.5284696817398071, 1.4593981504440308, -0.8507512807846069, -2.05306339263916, 1.114534616470337, -0.6205233335494995, 0.3928340971469879, -1.5438790321350098, 0.6009175777435303, 0.172918438911438, -1.9774686098098755, -0.6535962224006653, -0.7683398723602295, 0.32093796133995056, -0.08286558091640472, -0.24058176577091217, -0.14035183191299438, -1.103072166442871, -0.29251930117607117, -1.1061862707138062, -1.3209747076034546, -1.5572348833084106, 0.9450667500495911, 0.4478721618652344, 0.796742856502533, -0.7537128925323486, -0.11116430163383484, -1.8927526473999023, -0.759122908115387, 0.29214656352996826, -1.2560092210769653, -0.9037812948226929, 0.41455402970314026, 0.6646965742111206, -0.28138095140457153, -0.3180556297302246, -0.017029428854584694, 0.209851935505867, 0.32088619470596313], [-1.6933127641677856, 0.42321762442588806, -0.9769367575645447, 0.5622163414955139, 0.8835405707359314, -0.5687614679336548, -0.634947657585144, -1.219275951385498, -0.8793613910675049, -0.6498399972915649, 0.5481042861938477, -0.7916718125343323, -0.4007355570793152, -0.10349873453378677, -1.1339912414550781, -0.2709956169128418, 0.09455127269029617, 1.3276646137237549, 1.0906054973602295, 1.049460530281067, -0.09691660106182098, 0.07160107046365738, 1.5863099098205566, -0.3486379384994507, 1.1699529886245728, -1.3210170269012451, 1.8616323471069336, -0.7331673502922058, 0.5838020443916321, 0.8583634495735168, 0.19432830810546875, -0.03642013296484947, -0.41624945402145386, -0.6197614073753357, -0.7137199640274048, 0.9188348054885864, 0.5750817060470581, 0.4796449840068817, -1.0389444828033447, 0.6013351678848267, 0.6077513694763184, -0.09813591837882996, -0.8809558153152466, 0.11068196594715118, 2.349468469619751, -0.4451676309108734, 0.6981242299079895, -1.336846113204956, 1.3844966888427734, -1.2651852369308472, 0.21986281871795654, 0.14620304107666016, 1.007230520248413, -1.3914194107055664, 0.4023311734199524, -0.09859346598386765, -0.5181511640548706, 1.1950151920318604, 1.5238606929779053, -0.1172771155834198, 0.7115648984909058, -1.7119258642196655, -0.21287362277507782, 0.38869959115982056, -0.8244101405143738, -0.4499734938144684, -0.26568228006362915, -1.0105533599853516, 0.03646741434931755, -0.048956457525491714, -0.35033610463142395, 1.1754144430160522, 0.5228841304779053, 0.020527847111225128, -1.2783193588256836, 0.6518869400024414, 1.8200594186782837, 0.05838145315647125, 3.042189598083496, -0.7885617613792419, 0.6768757700920105, -1.369642972946167, -1.211410403251648, 1.3718147277832031, 0.004472718574106693, 0.5798130035400391, 0.19580677151679993, -1.235945224761963, 1.8247259855270386, -0.4999452829360962, -0.5017623901367188, -0.2301454395055771, 0.28006836771965027, -0.16889232397079468, 0.22008536756038666, 1.1277214288711548, 1.638187289237976, -0.9926971793174744, 1.9531757831573486, 0.28918659687042236, 0.04286601394414902, 0.9285078048706055, -0.01654602587223053, -0.6324818134307861, 0.678495466709137, 0.09563172608613968, -0.028822749853134155, 0.4270050525665283, -2.611849069595337, 0.6606674194335938, 1.257602572441101, 0.4327421188354492, -1.2346831560134888, -0.564290463924408, -0.678918182849884, 0.7750512957572937, -1.1325740814208984, 0.786247968673706, -1.117336630821228, 1.4937282800674438, 0.8431747555732727, -0.941972017288208, 1.5886329412460327, -1.7479937076568604, 0.7930091023445129, 0.0639399066567421, -0.24367769062519073, -0.41950708627700806], [0.44209179282188416, 1.0034518241882324, -0.4192466139793396, -0.6192532777786255, 0.3482933044433594, -2.065617084503174, -0.07979857176542282, -1.0941197872161865, -0.39437687397003174, -0.15895810723304749, 0.32523301243782043, 0.5571488738059998, -0.5987406373023987, -1.1854238510131836, -1.4698638916015625, 1.7093820571899414, 1.4126362800598145, 0.623802900314331, 1.3456909656524658, 1.8704689741134644, -1.0884696245193481, 0.9580815434455872, -1.034763216972351, -0.5543341040611267, -0.05720147117972374, 0.7430153489112854, -1.1676490306854248, -0.2162720263004303, 0.053630322217941284, -0.1454901546239853, -0.6872789859771729, 0.508690595626831, 0.34140902757644653, -0.4512450695037842, 1.480522871017456, -0.8122150897979736, -0.8067114353179932, -0.7597025632858276, -1.2077821493148804, -0.35103270411491394, 0.21962328255176544, 0.4336315393447876, -0.3338621258735657, 1.2233214378356934, 0.7260864973068237, 0.979951024055481, -0.9878820776939392, -0.024194536730647087, 1.0469425916671753, 0.5088273882865906, -1.168656349182129, -0.664576530456543, 0.4576384723186493, 0.3071238100528717, 0.16424760222434998, 2.6663384437561035, -0.06281585991382599, 0.07470131665468216, -2.3105697631835938, -0.49478697776794434, 2.3120930194854736, 0.38071778416633606, -0.28891927003860474, -0.6269356608390808, 0.22967414557933807, 1.1796479225158691, 1.556530475616455, -0.3375287353992462, 0.9967153668403625, 0.9532979130744934, 1.2010446786880493, 0.7762871384620667, 1.0832469463348389, -0.14705874025821686, -0.23771962523460388, 0.004910382442176342, 1.1410008668899536, 0.9728612303733826, 0.6373966336250305, -0.9348790645599365, -0.15836898982524872, 0.6579309105873108, 1.1329456567764282, -1.1491587162017822, 0.4448489248752594, 2.3452188968658447, 0.6154770851135254, 0.10305298864841461, -0.6565616726875305, -0.2382473349571228, 1.3168679475784302, -0.05991409718990326, -0.8690091371536255, 2.21500301361084, -0.15322858095169067, 0.22940650582313538, -0.1829020231962204, 1.022171974182129, 0.9823786616325378, -1.3298667669296265, -0.4711229205131531, 0.43147388100624084, -1.6119990348815918, -0.7356777787208557, -0.04765122011303902, -1.2287038564682007, -0.9861786961555481, -1.1864196062088013, -1.6580229997634888, -0.7819713950157166, 0.97185218334198, -0.148467555642128, 2.1737866401672363, -0.30256590247154236, -0.1393808275461197, -0.18551066517829895, -0.15284386277198792, -0.018322426825761795, -0.38795167207717896, 1.2543673515319824, 0.29684901237487793, -0.4566499888896942, -1.3803361654281616, 0.056428320705890656, -0.1671651005744934, -0.10141409188508987, 0.9594359993934631, -0.6129286885261536], [0.4390237033367157, 0.07683170586824417, -1.1433428525924683, 1.4408175945281982, 1.8211184740066528, 0.5906322598457336, -0.6081050634384155, 0.1453673094511032, 0.48662030696868896, -1.8187055587768555, -0.14914536476135254, 0.35040366649627686, 0.5078704357147217, 0.7763280868530273, -0.1400732398033142, -0.142971470952034, -1.7168340682983398, 0.3897039592266083, 0.029660740867257118, 0.6793814301490784, -1.8221033811569214, 1.9549180269241333, -0.5668562054634094, -0.1031608060002327, -1.5184276103973389, 2.016488552093506, -2.303386926651001, 0.1508706957101822, 0.6377153396606445, -1.1723477840423584, -1.1820271015167236, -0.048501960933208466, -1.2950665950775146, -0.15134195983409882, -0.40368974208831787, -0.9240890741348267, -0.604176938533783, 1.9203189611434937, -0.12189217656850815, -0.7607669234275818, 0.7802517414093018, 1.1296106576919556, -0.3597147762775421, 2.064908504486084, -1.0523966550827026, -1.2193303108215332, 1.2230937480926514, 0.17614144086837769, 0.3897266387939453, -0.8496190309524536, 0.5221542119979858, -1.0601609945297241, -0.13123691082000732, -0.26596498489379883, -0.6075239777565002, -0.075517937541008, 1.6440105438232422, 1.112934947013855, -1.2932910919189453, -0.48623716831207275, -0.6894109845161438, -1.1242272853851318, -0.23639392852783203, 2.382861375808716, 0.42689403891563416, 1.430508017539978, -0.667004406452179, -0.12391315400600433, -0.020608603954315186, 0.6487070918083191, -0.40995314717292786, 1.3666280508041382, 2.1585888862609863, 0.9655409455299377, -0.2822088599205017, 0.10926491767168045, 1.254436731338501, 0.8829963207244873, 0.13253673911094666, 0.6821079254150391, 1.1510614156723022, -3.1790668964385986, 0.6367741227149963, 0.4930177330970764, -0.07502797245979309, 0.32192960381507874, -0.7172624468803406, 1.1011195182800293, -1.670663833618164, 0.6059969663619995, -0.7558441162109375, 0.07462606579065323, -1.1218054294586182, 0.568199098110199, 0.5764091610908508, -1.8085843324661255, 0.3044757544994354, 0.54731684923172, -0.31873026490211487, 0.6265537738800049, -1.0226763486862183, -1.2134037017822266, -2.0470216274261475, -3.1594250202178955, -1.9357337951660156, 1.7335933446884155, 0.6992381811141968, 1.0005340576171875, 0.6487101912498474, -0.3359936773777008, -2.261070489883423, 0.8472875952720642, 0.4195796549320221, 0.48159193992614746, -1.4566349983215332, 0.13739658892154694, 0.3065187335014343, -0.8922414183616638, 1.806283712387085, 1.5026576519012451, -0.16025564074516296, -1.1561356782913208, 1.8057328462600708, -0.4333609938621521, -1.1866133213043213, 0.18412691354751587, -0.15615996718406677, 1.6037496328353882], [-0.7682524919509888, -1.5567034482955933, 0.6177017688751221, -0.3517138957977295, -0.9806820154190063, 1.2547760009765625, 0.9645511507987976, 0.3768078684806824, -0.368914932012558, 0.06890404224395752, 0.7601719498634338, -0.3521364629268646, -1.9279417991638184, -0.8714902400970459, 0.32444652915000916, -0.908278226852417, 1.0465617179870605, -0.3422200679779053, -0.3326338529586792, -0.5576034784317017, -2.358860492706299, 0.2843162715435028, 1.3118280172348022, -0.6425861716270447, -0.815031111240387, -0.25294941663742065, 0.662865161895752, -1.051125407218933, 1.6228291988372803, 0.11302896589040756, 0.32140907645225525, 0.2597598433494568, 0.06436318904161453, 0.03112640604376793, -0.34642478823661804, -0.7306835055351257, -0.32530108094215393, 0.13018599152565002, -1.540842890739441, 0.8658403158187866, 0.08430317044258118, -1.4182648658752441, -0.5790418982505798, 0.296822726726532, 1.2514981031417847, 0.45541682839393616, 0.28780391812324524, -0.3653797209262848, 1.1295654773712158, -1.4858996868133545, 0.38610729575157166, 0.7201474905014038, 0.634201169013977, 0.9058691263198853, 0.7623560428619385, 1.3249938488006592, 1.2638918161392212, -1.1700384616851807, -0.5278461575508118, -0.0010659119579941034, 0.09023270756006241, 0.9296277165412903, -0.8212964534759521, -0.10016542673110962, -0.03729876130819321, -0.0672663226723671, -0.30953341722488403, 0.2037385106086731, 1.223915934562683, 0.39749911427497864, 1.0247946977615356, 0.06726468354463577, -0.7768639922142029, 1.0099745988845825, -0.06989825516939163, 0.2068527638912201, 1.2123273611068726, -0.9031869769096375, -0.028557302430272102, -0.45099809765815735, 0.7713350057601929, -0.11604597419500351, -0.5613935589790344, -0.812619149684906, 0.5384907126426697, -0.4399917721748352, 0.5138977766036987, -0.13392145931720734, 1.344438910484314, -0.5144646167755127, -0.7281360626220703, -0.21868737041950226, -1.1916866302490234, -1.6765557527542114, -0.47864964604377747, 0.4371398091316223, -0.005587449762970209, -0.020107895135879517, 0.2808556854724884, 0.16348649561405182, 0.02181197516620159, -0.9717084765434265, -0.9282868504524231, 0.3685970902442932, -0.4605969786643982, -1.0499027967453003, -0.39074575901031494, 0.27332815527915955, -0.31782376766204834, 0.21449138224124908, 0.22208885848522186, -0.3317390978336334, 0.4970960021018982, 0.06361247599124908, -0.48702049255371094, 0.07121942192316055, 1.6604466438293457, 0.7098793387413025, -1.553624153137207, -1.2665042877197266, 0.3680686056613922, -0.6796995997428894, -0.8561299443244934, 1.1916450262069702, -1.6227906942367554, -0.31750568747520447, 0.2516712546348572, 1.662892460823059], [-1.3061320781707764, 0.33355143666267395, 0.14409880340099335, -1.6530771255493164, -0.3902346193790436, 0.8577656745910645, 0.5215307474136353, 1.43726646900177, 2.2032558917999268, -1.6822768449783325, -0.1197248250246048, -1.5962927341461182, -1.343575119972229, -0.2160598337650299, -0.49345868825912476, -0.4692080318927765, -0.1911785900592804, 0.0936707928776741, -0.8635247945785522, 1.0480338335037231, -0.12801304459571838, -0.793804943561554, -0.1013658195734024, 0.24426856637001038, 0.08991982787847519, 0.47045910358428955, -1.5158277750015259, -0.10776398330926895, 0.41084054112434387, 0.30898213386535645, -1.1674225330352783, -0.3052259385585785, 0.7206926941871643, -0.9102506637573242, -1.9393376111984253, 0.11070511490106583, 0.41206100583076477, 1.6224303245544434, 0.6253008246421814, -1.0247777700424194, 0.3998413383960724, 0.07567457854747772, 0.8388710021972656, 1.0372018814086914, 0.7277411222457886, 1.161795735359192, -0.2888715863227844, -1.568516492843628, -0.8388490676879883, 0.6858978867530823, -1.505948543548584, -1.5952359437942505, 0.24771417677402496, -0.9943512678146362, 0.14075958728790283, -1.823066234588623, 0.11730211973190308, -0.019530504941940308, -0.28032955527305603, 0.06395704299211502, 1.2370986938476562, -1.5461974143981934, 0.9701997637748718, -0.9528255462646484, -2.0758464336395264, -0.4363473057746887, 0.14461001753807068, -0.8293349742889404, -1.5202808380126953, -0.5113932490348816, 0.512545645236969, 0.012992131523787975, 0.41254135966300964, 0.6682782173156738, 1.0888209342956543, 0.6516631841659546, 0.17205508053302765, -1.1998765468597412, -0.7276132702827454, 1.2610278129577637, -0.8852018117904663, 0.2417888045310974, -0.02779906429350376, 1.0287060737609863, 1.3985493183135986, -0.624214231967926, -1.4053411483764648, 0.47287797927856445, 1.205397367477417, -0.26832515001296997, 0.5392817258834839, 2.4376392364501953, -0.8345246911048889, 0.32098090648651123, 0.5189949870109558, 1.5339571237564087, -0.35406389832496643, -1.5589354038238525, -0.45276424288749695, 0.4264123737812042, 1.455061674118042, -0.018174922093749046, 0.747037410736084, 1.661027193069458, -1.0393071174621582, -0.7073232531547546, 1.2762397527694702, -0.5771364569664001, 0.023709489032626152, -1.0988354682922363, -0.1695852428674698, 0.7790156006813049, 0.6675528287887573, 1.172788143157959, -1.2996609210968018, -0.048455581068992615, 0.9017625451087952, -0.07534559071063995, -0.7448972463607788, 1.49455726146698, 1.8153756856918335, -0.960482120513916, -0.513763964176178, 1.145054817199707, -0.8645215630531311, -1.1242361068725586, 1.155049204826355, 0.9340051412582397], [2.391587257385254, 1.9756860733032227, 0.0016493676230311394, -0.3025396466255188, -0.06860390305519104, 0.17232009768486023, 1.1767700910568237, 1.458891749382019, -1.4870336055755615, 0.7702479362487793, 0.4407326877117157, 0.9623379707336426, -0.31979700922966003, -0.49951162934303284, 0.18554501235485077, -0.03763442113995552, 1.580552101135254, -0.8184816241264343, 0.38410061597824097, 1.1267688274383545, 0.18248991668224335, -0.07236044853925705, -1.602659821510315, 2.1780834197998047, -0.2584942877292633, 0.7344841361045837, -0.6327525973320007, -0.1858031004667282, -0.4893063008785248, 2.3299789428710938, -1.5819199085235596, -1.8586891889572144, -0.9626763463020325, -0.7273115515708923, 1.4743733406066895, -0.27704548835754395, -0.9687450528144836, 0.5266499519348145, 0.33526331186294556, -0.22395111620426178, 0.2244274616241455, 1.4681495428085327, 0.057023052126169205, -0.4767444431781769, 0.6877739429473877, -1.089622974395752, 0.2791656255722046, -1.56940758228302, 1.0579825639724731, -0.8420439958572388, -0.31110844016075134, -0.26786738634109497, 0.583392858505249, -0.5062184929847717, -0.08368424326181412, -1.328066349029541, 1.6346842050552368, -0.6937186121940613, -1.0407354831695557, 1.267403483390808, -0.9765177369117737, 0.7164223790168762, 3.4274978637695312, 0.93417888879776, 0.26850566267967224, 1.9017189741134644, -0.7267833948135376, -2.0414109230041504, -0.7392891645431519, -1.318708896636963, -1.2676584720611572, -0.2890867590904236, -0.20484387874603271, -0.6194301843643188, -1.4724071025848389, -0.46037405729293823, -0.04092627763748169, -0.9520472884178162, 0.6076838374137878, 0.5907251238822937, -0.519084632396698, -3.710846424102783, -0.8078455924987793, -0.547023594379425, 0.06851258128881454, 0.29717814922332764, 0.33668580651283264, 0.4548070430755615, 0.4225788116455078, -0.37221214175224304, -0.1813349574804306, 3.1750235557556152, -0.8846403360366821, -1.1034762859344482, -0.12398799508810043, -1.2122586965560913, -0.0783400908112526, 1.646765947341919, 0.2302054464817047, 0.08728346973657608, -1.7811753749847412, 0.7852354645729065, -0.18853677809238434, 1.289476752281189, 0.3794120252132416, -2.65914249420166, -0.6911197304725647, -0.3947595953941345, -0.2676306962966919, -0.34821614623069763, 1.0844172239303589, 0.44912806153297424, 0.16016972064971924, -0.4019176661968231, 1.320601463317871, -0.3105834722518921, 0.641583263874054, -0.6044027805328369, -0.22708699107170105, -0.9377771615982056, 0.0692397877573967, 0.9183283448219299, -1.2724097967147827, -0.3994315266609192, -0.4998420774936676, 2.2274515628814697, 0.09807132929563522, 1.0667767524719238], [-0.7197646498680115, 0.9972180128097534, -1.4438881874084473, 0.13977673649787903, 0.8330501914024353, -0.8381057381629944, 2.016857385635376, -0.9621341228485107, -0.6394383907318115, -0.4707227647304535, -0.052466701716184616, 1.0802215337753296, 1.1092597246170044, -0.6753592491149902, -0.7534511685371399, 0.795702338218689, 0.6848406791687012, -0.44751614332199097, 0.8095104098320007, -0.9360788464546204, 0.3570660948753357, 1.0505752563476562, 1.1298041343688965, 0.6768174767494202, 1.1715189218521118, -1.4914110898971558, 0.3242810070514679, 0.5527009963989258, -0.7163796424865723, 1.5777705907821655, -1.0424152612686157, -0.022284116595983505, 0.8163949847221375, 1.1429214477539062, -0.24137282371520996, -1.3055561780929565, 0.16251413524150848, -0.7876980304718018, -1.3094011545181274, 0.6443914771080017, 1.3669874668121338, -1.0851771831512451, 0.8734190464019775, -0.10975158214569092, -0.8765046000480652, 0.10564663261175156, 0.9953652620315552, 0.18108607828617096, 0.12725648283958435, 0.5927224159240723, -0.0615646168589592, -0.2702762186527252, -1.6309987306594849, -0.46539729833602905, -0.8421180248260498, -2.381568670272827, -1.2682732343673706, 1.4091148376464844, -1.459267258644104, -2.2580206394195557, 1.3286921977996826, 0.8632431626319885, -2.0093939304351807, -0.16312427818775177, 1.1199393272399902, -0.15890897810459137, -0.9916239976882935, -0.19261638820171356, 0.6824967265129089, -1.037848949432373, -0.4845007658004761, 0.048018310219049454, -0.4668179452419281, 0.684235155582428, 0.2274351567029953, -0.31560394167900085, 0.2671355903148651, 0.10325152426958084, -1.3386839628219604, 0.9398358464241028, 0.8735795021057129, -1.8272477388381958, -0.4085339903831482, 1.2116971015930176, 1.1985599994659424, 0.1287613809108734, -0.2747071087360382, 1.4231358766555786, 0.31536319851875305, -0.4984278380870819, 0.12873488664627075, 1.545072078704834, -0.36860328912734985, 0.3635915219783783, -1.1516841650009155, 0.48622846603393555, 1.0159090757369995, -0.513247013092041, 0.0058461702428758144, -0.8281778693199158, -0.09264125674962997, -0.7877241969108582, 0.5358235239982605, -0.4941544234752655, -1.373207449913025, -0.7122500538825989, 0.7621457576751709, 1.0171756744384766, 0.2666998505592346, 1.8961318731307983, -1.1135246753692627, 1.2549560070037842, 0.9073604345321655, 1.3989614248275757, -0.08341539651155472, 2.1423306465148926, 0.2864651083946228, -0.37248358130455017, -1.0258252620697021, -0.07921543717384338, -0.015267148613929749, 1.2627990245819092, -1.5547351837158203, 0.8467506766319275, 1.3133049011230469, 0.41126441955566406, 0.8314367532730103, 0.17418941855430603], [0.8539913296699524, -0.5763395428657532, -0.557177722454071, 0.5702784061431885, 0.25282153487205505, -0.10853306949138641, -0.17592521011829376, 0.5176522135734558, -2.0626282691955566, 0.8505414724349976, -0.33201441168785095, 0.6431323885917664, -0.7791153788566589, -0.8341319561004639, 0.5606589913368225, -1.7439192533493042, -0.9250786304473877, 1.0855828523635864, 0.38011664152145386, -0.8123482465744019, 0.35706546902656555, -0.4692259430885315, -1.0874145030975342, -0.8190761804580688, -0.8401944637298584, -1.1101922988891602, -1.524398922920227, -0.4051629900932312, 0.5789771676063538, -0.030734872445464134, 0.3618709146976471, 0.5589852929115295, -0.5277854204177856, -0.8401235938072205, 0.4870148003101349, -1.1002217531204224, -0.32064226269721985, -0.4271916449069977, -0.4654935300350189, 0.38121503591537476, 0.8745037317276001, 0.21291795372962952, -0.4624141752719879, 1.5046782493591309, 0.17721012234687805, -1.3961466550827026, -0.2033976912498474, 0.9421955347061157, -1.7576205730438232, -1.5723861455917358, -0.9885537624359131, 0.12454812973737717, -1.7152984142303467, 0.7259324789047241, 0.16854240000247955, -0.27276402711868286, 0.5099396109580994, -1.7866021394729614, -1.8121337890625, -1.5371248722076416, 0.07408367842435837, -0.26223552227020264, 1.0015745162963867, -0.8370662927627563, 1.216240644454956, 1.3988089561462402, 0.1142251119017601, -0.38607436418533325, -0.9497193098068237, -0.7292903065681458, 1.881195306777954, 1.7986539602279663, 1.34943687915802, 0.9930604696273804, -0.14596585929393768, 2.2668066024780273, -0.43874484300613403, -0.3503182828426361, 0.38685548305511475, -1.5951606035232544, 0.35197487473487854, -0.7151744365692139, -0.29346761107444763, -0.1259298473596573, 0.6810545921325684, -0.7450608611106873, -0.5726998448371887, 1.4095600843429565, -0.6848372220993042, -1.0745835304260254, 1.8040845394134521, -0.8381813168525696, -0.42116445302963257, -0.7725841403007507, 1.2053314447402954, 1.0312787294387817, -0.8766430020332336, -0.7923845052719116, 0.4213688373565674, -1.9692323207855225, -1.400921106338501, 2.0217578411102295, -0.16517366468906403, 1.1175262928009033, 2.282766819000244, 0.91678786277771, -0.9431180953979492, 1.4682374000549316, 0.24278390407562256, 2.3534932136535645, 0.04023262858390808, 0.6797999143600464, 1.506501317024231, -1.7433267831802368, -1.5818241834640503, 0.8464564085006714, 0.4329484701156616, 0.0068863751366734505, -0.16020017862319946, 1.0166932344436646, 0.0049236975610256195, 1.1907378435134888, -1.2314397096633911, 0.3050054609775543, -0.5281581878662109, -0.06322836875915527, 1.2220736742019653, 0.07268267124891281], [0.9542728066444397, -0.3898140490055084, 0.7024174332618713, 1.3923043012619019, 0.8550028800964355, -0.7465100884437561, -0.6877236366271973, 1.386273980140686, -1.9507107734680176, 1.615426778793335, 1.3342598676681519, 0.28238967061042786, -1.7512538433074951, 0.04770532622933388, 1.8544394969940186, -0.2022283524274826, -1.1468545198440552, 0.5522738695144653, 2.3958756923675537, 0.5101298093795776, -0.30822157859802246, 1.0281435251235962, -1.019168496131897, 0.03918032348155975, 0.8280670642852783, 0.5617462992668152, -0.5791510939598083, -0.6400579810142517, 0.21101365983486176, -0.26721131801605225, 0.07890179008245468, 1.009329915046692, -0.09282509237527847, 0.8565773367881775, -0.7715792655944824, 0.09499969333410263, 0.25074800848960876, -0.08180122822523117, 0.6041517853736877, -0.33511239290237427, -0.6256875991821289, 1.1190513372421265, -0.4635929465293884, 0.9947167038917542, 0.335146963596344, -0.6875610947608948, 0.822553277015686, 0.49229586124420166, 0.050027940422296524, 1.425240159034729, 0.4349643886089325, 1.7513600587844849, -1.5152506828308105, -1.0143133401870728, -0.18863436579704285, -1.849029779434204, -0.506775438785553, 0.26070162653923035, -1.287155270576477, 1.7590210437774658, 1.0750807523727417, 0.5510050058364868, 2.310901403427124, 1.3807978630065918, 0.014588707126677036, -0.37123212218284607, -1.3317134380340576, 1.5216388702392578, -1.7240827083587646, 1.5035419464111328, -0.8634809255599976, 0.49692538380622864, -0.8230417966842651, 0.47973397374153137, -0.083064503967762, -1.8453279733657837, 0.43124058842658997, -1.5400753021240234, 0.02925933711230755, -0.6746187210083008, 0.9080939888954163, 1.0938687324523926, -1.070507287979126, 0.789872407913208, -0.9350431561470032, 0.7322237491607666, -0.8535253405570984, 0.07201581448316574, -0.2935611307621002, -0.6900187134742737, -1.7977921962738037, -0.37572675943374634, 0.47955846786499023, 0.0051586925983428955, 0.14063939452171326, -1.215379238128662, -0.2306426465511322, 0.9367480278015137, -1.1102510690689087, 0.9118679165840149, 1.5791242122650146, -1.4779956340789795, 0.46160417795181274, -0.5338300466537476, -0.26759833097457886, -1.4044369459152222, 0.0025712342467159033, 0.22319930791854858, -0.05766652151942253, -0.49873268604278564, 0.015157033689320087, 0.04049261286854744, 0.5735563635826111, 1.8003917932510376, 0.44640710949897766, 2.441253900527954, 1.4667916297912598, -0.02948756329715252, -1.943575382232666, -0.7154174447059631, -0.32582929730415344, 0.39367881417274475, 0.58747798204422, -0.09561346471309662, -1.0024784803390503, 1.275078296661377, 0.7065540552139282, -1.41436767578125], [-0.8202041387557983, -0.8796519041061401, -1.1427000761032104, -0.11544344574213028, -1.0931143760681152, 0.6361966729164124, 0.6217252016067505, 0.8125725984573364, 0.020264949649572372, 1.337627649307251, 1.0247337818145752, -1.8897725343704224, 0.8109166026115417, -1.5623502731323242, -0.19336958229541779, 0.5023397207260132, -0.7451272010803223, 0.15623855590820312, -0.16870029270648956, -0.2696543037891388, -1.450914740562439, 1.058676838874817, -0.8933558464050293, -1.3633971214294434, -0.7678887248039246, 0.5090414881706238, 1.7374430894851685, 0.3953452706336975, 2.05705189704895, 0.2274082452058792, -1.502366542816162, -0.16771143674850464, 2.815389633178711, 1.729441523551941, 0.6295249462127686, 1.1797951459884644, -1.8745455741882324, -0.9904633164405823, 1.0474857091903687, 1.8480366468429565, -0.17038753628730774, -0.26316821575164795, 0.044488366693258286, 0.30122336745262146, -0.5550765991210938, 0.030611926689743996, 0.009583989158272743, 0.0769536942243576, 0.860273540019989, 0.23828645050525665, -0.5001657605171204, -1.7899662256240845, -1.2883607149124146, -2.633415937423706, 0.26997309923171997, -1.6290297508239746, -1.3115088939666748, -0.1294226050376892, -0.3397902548313141, 1.350171685218811, -0.2963230609893799, -0.18564730882644653, 0.45424559712409973, 1.5447709560394287, -0.7265677452087402, -1.1283400058746338, 0.5329709649085999, 0.4494515657424927, 0.3006865978240967, 0.14030949771404266, -0.14933033287525177, 0.0035477301571518183, -1.4289716482162476, 0.8010203242301941, 1.1666032075881958, 0.07205105572938919, 0.3668006956577301, 0.3541785478591919, 0.22771510481834412, -0.3732392489910126, -0.6826647520065308, 0.664203941822052, 0.41061705350875854, 0.9423884749412537, 0.4756529629230499, -0.1919095516204834, 0.9526939392089844, -1.4030885696411133, 0.7248067855834961, -0.7133177518844604, -0.07871173322200775, -1.0467820167541504, -0.12867774069309235, -0.12801454961299896, -0.7053382992744446, 0.589253306388855, -0.3079194128513336, -1.4450693130493164, 0.6446696519851685, 1.9515023231506348, 1.39337956905365, -0.37418413162231445, -1.3285787105560303, 0.24650003015995026, 1.2642978429794312, 0.9052721858024597, 0.8906553387641907, -0.6986395716667175, -1.0008599758148193, 1.16765296459198, -0.6194518208503723, -1.8271421194076538, 1.2767398357391357, 0.6128111481666565, 0.6400160789489746, 1.1354098320007324, 2.616556406021118, -0.10502084344625473, 0.9087066650390625, -0.2653506100177765, -0.47723495960235596, -0.4141988754272461, -0.8150194883346558, -0.2473696768283844, 0.4630706310272217, -1.366988182067871, -0.774387776851654, -0.006935819052159786], [2.091247081756592, 0.03822999820113182, 0.09322403371334076, 0.09052707999944687, 1.1129778623580933, 1.012304663658142, 0.15185555815696716, 0.22398793697357178, -0.5072363018989563, 0.4892488718032837, 1.5655797719955444, -1.3593412637710571, 0.39770597219467163, -1.8752272129058838, -0.8731133341789246, 1.6619559526443481, -0.5630715489387512, 0.3992708921432495, -2.0683798789978027, 0.6677330136299133, -0.6486217379570007, -0.8010468482971191, 0.9610394239425659, -0.42520493268966675, -0.044378861784935, 0.22909076511859894, 0.43242353200912476, 1.0329952239990234, -0.8643993735313416, 1.3328076601028442, -1.0486878156661987, -0.6387039422988892, -0.37010458111763, -0.746933102607727, -1.0951236486434937, 0.4368893504142761, -0.3842640221118927, -0.3803119957447052, 1.3296557664871216, 0.7060995101928711, -0.209059938788414, -0.1974126398563385, -0.5452382564544678, -0.6620417833328247, -0.17867235839366913, -0.09029008448123932, 1.157299280166626, -1.1540073156356812, -0.815180242061615, 0.01048672292381525, -0.5237924456596375, -0.5964692831039429, -0.3076581358909607, -0.5513714551925659, -2.0530903339385986, -0.24413666129112244, -0.71145099401474, 0.07084245979785919, 0.9077396392822266, -1.481673002243042, 1.4778192043304443, 0.45938974618911743, 0.2907489240169525, 0.05861350893974304, 1.298436164855957, 0.5970262885093689, -0.22184298932552338, -0.24529892206192017, -0.6962481737136841, 0.6441311836242676, 1.2854021787643433, -2.1205973625183105, -1.1074773073196411, -1.6223750114440918, 0.6889652609825134, -1.8059546947479248, 0.7261375188827515, 0.08867454528808594, 3.0821590423583984, -0.12891806662082672, 0.3921273350715637, -0.8793492317199707, -0.4543263912200928, -0.33893612027168274, 0.11499097943305969, 0.07476775348186493, -1.6111080646514893, 3.393841028213501, -0.3737257719039917, -1.9396097660064697, 0.4720406234264374, -1.0093870162963867, -1.2543765306472778, -0.6471360921859741, -0.8445690870285034, -1.669595718383789, 0.031376488506793976, -0.19082139432430267, -1.1778289079666138, -0.5925804972648621, -0.07329921424388885, 0.35436490178108215, 1.780543565750122, -0.5484281182289124, 0.9410504698753357, 0.34187811613082886, -0.45335400104522705, 0.363389253616333, -0.8415864109992981, -0.5307361483573914, -0.07302634418010712, -0.0599660649895668, 2.198267936706543, 2.0007259845733643, -1.1427770853042603, 0.6011754274368286, 0.09472639858722687, -2.2320501804351807, 0.5866905450820923, 0.6306706070899963, -0.6018154621124268, 1.0079516172409058, 1.1355911493301392, 0.6261435151100159, 0.5848338603973389, 0.930123507976532, -1.6399247646331787, -0.5435543060302734], [-1.6045145988464355, -0.7030771970748901, -0.9263298511505127, -1.1867825984954834, -1.2001924514770508, -0.7979555130004883, 0.9870906472206116, -0.24546372890472412, -1.6534051895141602, 0.15821793675422668, 0.16138355433940887, 0.20489402115345, 0.44492822885513306, -0.05106443911790848, -0.7746819853782654, -1.1818665266036987, 1.5467075109481812, -1.5962973833084106, 2.177635431289673, -0.5866482853889465, 0.3890873193740845, 0.29985517263412476, -1.7392268180847168, -1.2930772304534912, 1.082005262374878, -0.34025537967681885, 0.40421807765960693, -0.49464336037635803, -0.6067739129066467, 1.188402771949768, 0.02633831463754177, 2.651364326477051, -1.6074906587600708, 0.34190186858177185, -1.4587403535842896, -1.7475124597549438, 0.42332082986831665, -0.29967013001441956, -1.7282366752624512, -0.5566805005073547, 0.6303423047065735, 0.8451522588729858, 1.5835647583007812, 0.43708372116088867, 2.310451030731201, -0.16233868896961212, -0.7150064706802368, -1.6835319995880127, -1.0190272331237793, -0.292473703622818, 0.6044326424598694, -0.6104199290275574, 0.09285090863704681, 0.05851314216852188, -0.5211585760116577, -0.45679569244384766, -0.6059449911117554, -2.5842909812927246, -1.2368272542953491, 2.5407888889312744, 0.2677333652973175, -1.0821154117584229, 0.18318119645118713, 0.8128333687782288, 0.6070601344108582, 0.8975717425346375, -0.04796690493822098, -0.3898701071739197, -0.4232577979564667, -0.8077391982078552, 2.615684747695923, 1.222469449043274, 0.8461290597915649, 0.4202651381492615, 0.3847610652446747, 0.6756250262260437, -0.15113836526870728, -0.3305671513080597, 0.7197791337966919, 0.9902016520500183, -2.853252410888672, 0.27091845870018005, 0.09760147333145142, -0.7140917778015137, 1.061211109161377, 1.2700707912445068, -0.47226420044898987, -0.5599177479743958, -0.2587275803089142, 1.280247449874878, 1.2454230785369873, 1.0499536991119385, 1.0401912927627563, 1.0785558223724365, -1.3695838451385498, 0.027369512245059013, -0.47564059495925903, 0.8606321811676025, 0.9283017516136169, -0.07456372678279877, 0.6557344794273376, 0.1064935028553009, -0.9014173150062561, 0.768517792224884, -0.7478811740875244, 0.183996319770813, -0.23594534397125244, -0.38018831610679626, -0.8036994934082031, 0.006874856539070606, 0.050256356596946716, -0.4590367376804352, -0.2883344292640686, -1.0026445388793945, 0.5001196265220642, 1.4511120319366455, -1.6588743925094604, -1.2152938842773438, -1.2878717184066772, -2.3026158809661865, -0.9483657479286194, 1.9441646337509155, -0.8223626017570496, 0.5058388113975525, -1.1695911884307861, 0.663810133934021, 0.4677867889404297, -0.6488352417945862], [0.43018946051597595, 0.8050545454025269, 1.2444102764129639, 0.2712423801422119, 0.7192504405975342, -1.8637707233428955, -0.7390731573104858, 0.7747891545295715, 0.31504422426223755, 2.1368136405944824, 0.6793531179428101, 0.0210584606975317, -0.5355980396270752, -0.9742786884307861, 0.3013761639595032, 1.8048371076583862, 0.8206915259361267, 1.1951003074645996, 0.20579977333545685, 1.0099239349365234, 0.4667547345161438, 1.0119547843933105, 0.24506618082523346, -0.7284841537475586, 0.8513269424438477, -0.4895782172679901, -0.1979847401380539, 0.3241904675960541, -0.12951703369617462, -0.3829096257686615, 0.6292078495025635, -0.5180523991584778, 0.6549236178398132, 1.036399006843567, 0.6170071363449097, 0.10611553490161896, -0.23027542233467102, 0.07874774932861328, 0.8266898393630981, -0.5219005942344666, -0.7855936884880066, -1.2310645580291748, -0.5512088537216187, -0.04981331527233124, 0.8298503756523132, -0.01799045316874981, 0.5860951542854309, 0.19811825454235077, 0.3241845667362213, 0.5169546604156494, -0.26145726442337036, 1.9882766008377075, 0.3107605278491974, 0.6043930053710938, -0.9897828102111816, 1.4389110803604126, 0.35044732689857483, 1.8995822668075562, -1.035636305809021, -1.871994137763977, 1.0315957069396973, -0.38312748074531555, -0.4787053167819977, 0.7687069773674011, 0.8778560161590576, 0.6269287467002869, 1.1307450532913208, 1.1871790885925293, -2.1623220443725586, -2.0679967403411865, 1.0109224319458008, -0.5530626177787781, -0.8786615133285522, 0.14960278570652008, 0.081000417470932, 0.9217708110809326, -1.5708059072494507, 0.33228984475135803, -0.8046041131019592, 0.9541138410568237, -0.4414931833744049, -0.7276315689086914, 0.020335176959633827, -1.794744610786438, 0.36562252044677734, -0.17471538484096527, -0.7468021512031555, -2.705676317214966, 1.479102373123169, 1.0050171613693237, -1.3121634721755981, 2.079484462738037, 0.3537202775478363, 1.741031289100647, -1.5105832815170288, 0.7466421723365784, -0.23731817305088043, 1.387984037399292, -1.590211272239685, -2.1002979278564453, 1.6159369945526123, -0.6756808757781982, -0.3662058413028717, -0.4315011501312256, -1.7983946800231934, -0.823930025100708, -0.016313698142766953, -0.946174144744873, -0.3854817748069763, 1.310228943824768, 0.17401422560214996, 1.14030122756958, -0.9349667429924011, -0.2755141258239746, -1.7585810422897339, -0.30492785573005676, 0.19311508536338806, -0.6873257160186768, -1.1790969371795654, -0.04675683006644249, -0.9400594234466553, -0.6444670557975769, 0.3045850396156311, 1.081449270248413, -1.1086583137512207, -1.6917634010314941, 0.08434233069419861, -0.514321506023407], [-0.11164865642786026, 1.008945345878601, 0.010792013257741928, 1.148722767829895, 1.4834558963775635, 0.1351846158504486, 1.904523491859436, -0.018507156521081924, 0.2951720356941223, 0.215485081076622, -0.08928031474351883, -0.5638422966003418, -1.0130047798156738, -1.2949531078338623, -0.8130958080291748, -0.5584852695465088, 0.5946081280708313, 1.5809271335601807, -0.17865324020385742, -0.254618376493454, 0.7985125780105591, -1.2727845907211304, 0.3859705626964569, 0.04001646116375923, 0.6063783168792725, -0.25647348165512085, -0.18826404213905334, -1.926421880722046, -0.8828127980232239, -1.3304808139801025, -0.8381348252296448, -0.37384921312332153, -0.43698662519454956, -0.9471997022628784, 1.1992011070251465, -0.23183102905750275, -0.7923871278762817, -2.099642038345337, 0.7973463535308838, -0.817144513130188, -0.10050545632839203, -0.7807942032814026, -0.7534099817276001, 0.211581289768219, 0.7160400748252869, 0.8275715112686157, 0.5455266237258911, 0.6385046243667603, 0.20529457926750183, 0.5178174376487732, 0.5827816724777222, -0.2890881597995758, 0.09227380901575089, 0.6499740481376648, -0.3294658958911896, -0.5289451479911804, -0.08276182413101196, -0.18061482906341553, 0.5925630927085876, 0.8368919491767883, 0.5962210893630981, 0.3921259939670563, 0.9261351823806763, -0.7391988635063171, -0.45786339044570923, 0.1342829018831253, -1.028151273727417, -1.9156379699707031, -0.560606837272644, 1.318501591682434, -0.285856157541275, -0.26175767183303833, -0.5492496490478516, 0.8745394349098206, 1.7064460515975952, -0.1510544866323471, 1.314896583557129, -0.6375548243522644, -0.6314364075660706, 0.378905326128006, 0.10080201923847198, -0.3057044744491577, 0.2535804212093353, -1.3892414569854736, -1.2260469198226929, -0.36977913975715637, -0.02181493677198887, 1.1263885498046875, 0.31033164262771606, -0.04981502890586853, -0.8001378178596497, -1.9393033981323242, -1.0860792398452759, -1.182375192642212, -0.42247524857521057, 0.4739973247051239, -0.6585684418678284, 0.5157188773155212, -0.6329993009567261, -0.575152575969696, 0.09104235470294952, 0.009529315866529942, -1.992321252822876, -0.7399047613143921, 1.6575357913970947, 1.3735493421554565, -2.036278486251831, -0.22261297702789307, -1.3708059787750244, 0.16706490516662598, -1.5820038318634033, -0.9024915099143982, -0.1842993199825287, 1.630219578742981, 0.9410544633865356, 0.4770784080028534, 0.19851532578468323, -0.2760610282421112, -0.6287084817886353, -0.7358864545822144, -0.7711931467056274, -0.054367560893297195, -1.3105769157409668, 0.43440866470336914, -0.5229379534721375, -0.7865303754806519, -1.0285651683807373, 1.0223900079727173], [1.751304268836975, 0.4632939398288727, 0.13928787410259247, -0.7332205772399902, 1.5744930505752563, -1.119988203048706, 0.4846305251121521, -0.6114526391029358, 0.10669129341840744, 1.4942666292190552, -1.9238770008087158, -1.4818061590194702, -0.5068303346633911, -1.20636785030365, -1.1828651428222656, -0.970336377620697, -0.7844420075416565, -0.2146785408258438, 0.4343675673007965, 0.3258852958679199, 1.5146516561508179, 0.3609124422073364, 0.8053544163703918, 0.5462105870246887, -0.47492027282714844, -0.15720364451408386, -0.3680291771888733, -0.9113684296607971, -0.9151478409767151, -0.5964924693107605, -0.8472215533256531, 0.4781840443611145, -0.9260722398757935, 0.1739213764667511, -0.8051585555076599, 0.9661552309989929, -0.5146222114562988, -0.2346138060092926, -1.0962661504745483, -2.499699831008911, 1.1181355714797974, 0.35977011919021606, -1.8545632362365723, -0.6732572913169861, -0.09639538824558258, -2.7368152141571045, 1.0184738636016846, -1.8062399625778198, 1.8681282997131348, 1.025716781616211, 0.40148836374282837, -1.1038296222686768, -1.624351143836975, -0.08969701081514359, -1.4080593585968018, 2.2839584350585938, 1.1442822217941284, 0.30482032895088196, -0.9526920914649963, -1.2456257343292236, 0.6273626685142517, -0.762270987033844, -1.1017752885818481, -0.888054609298706, 0.4524727463722229, -0.6773629188537598, 1.1905723810195923, 0.41449663043022156, -1.3904035091400146, -1.0536459684371948, -0.17078816890716553, 1.0462149381637573, -0.02553212083876133, -0.11207640916109085, 1.5390238761901855, 0.09859883040189743, 0.8576289415359497, -0.441915363073349, -1.2262824773788452, 0.6546577215194702, 1.0382248163223267, -0.9587202072143555, -0.7102597951889038, 2.3738298416137695, -1.2712109088897705, 1.2028212547302246, 0.05829634889960289, 0.107853002846241, -0.41799989342689514, -0.6412139534950256, -0.7888858914375305, -0.1458578258752823, -0.2525940239429474, 0.223715677857399, -0.8752003312110901, -0.5184212327003479, 0.021897973492741585, -0.02719574235379696, -1.1823513507843018, 0.7617840766906738, 1.0393465757369995, -0.45319825410842896, 3.214487314224243, -0.10916293412446976, 1.028131365776062, 1.6107040643692017, 2.6383373737335205, -1.3731017112731934, -0.1793879121541977, 1.038079023361206, -0.2843702733516693, -1.6710726022720337, -0.7284792065620422, 0.17014139890670776, -0.6100860238075256, -0.6088848114013672, -1.179422378540039, 0.08145169168710709, 0.3648431897163391, 1.0389373302459717, -0.9624821543693542, -1.0868855714797974, -1.216737151145935, -2.3428120613098145, -0.138511061668396, 0.9250965714454651, -0.30615493655204773, -0.21465136110782623], [0.3077363669872284, 0.3774964511394501, -0.5656652450561523, 1.4255216121673584, -0.10326287150382996, -0.7871987819671631, 1.1841578483581543, -1.5554473400115967, 0.7063611149787903, 0.2512276768684387, -0.21567854285240173, -0.4564155042171478, 1.4298607110977173, 1.822524905204773, -0.10370828211307526, -0.872386634349823, -1.994450569152832, -0.4100962281227112, 0.8211028575897217, 0.8080126047134399, 1.7548221349716187, -0.035428278148174286, -1.6344544887542725, -2.264890432357788, -0.7278223633766174, -0.4354225695133209, -0.7240187525749207, 1.830461025238037, 1.6698719263076782, 0.5310937166213989, -0.9071977734565735, 0.7522992491722107, -0.5618546605110168, 0.10367749631404877, 0.4708033502101898, 0.8348466157913208, 0.040892165154218674, -0.5422483086585999, 0.21560899913311005, -0.7758252620697021, 1.3707847595214844, 1.4348564147949219, 0.06821949779987335, 0.6230103373527527, 1.5014009475708008, -1.6059153079986572, -0.737679660320282, 0.6989887952804565, -1.1552034616470337, 1.1835442781448364, 0.19723790884017944, -0.24596092104911804, 1.7601293325424194, -0.2925906181335449, -0.37437403202056885, -2.0821657180786133, 0.5917727947235107, 0.4299096167087555, 2.5411458015441895, 0.11017757654190063, -0.24873527884483337, 0.04101766645908356, -0.21227136254310608, 1.7535884380340576, -0.060099661350250244, 0.46361374855041504, -1.5831551551818848, 0.581960916519165, -1.686307430267334, -0.7490678429603577, -0.18035367131233215, 2.1830315589904785, -0.6394964456558228, 0.5465701818466187, 1.8806501626968384, -0.115545354783535, 1.245845913887024, 0.5309759378433228, 0.8110558390617371, -1.3907984495162964, -0.29849737882614136, 0.8260720372200012, 1.1303294897079468, -1.6996134519577026, -0.12610594928264618, -1.173427939414978, -1.203769326210022, 0.996865451335907, -0.9071740508079529, -0.5704501867294312, 0.691612184047699, -0.957933783531189, -0.828704297542572, -0.1589859575033188, 0.4509809911251068, 0.8747227191925049, -0.13469819724559784, -0.691411554813385, -2.1997640132904053, -0.010018828324973583, 0.9872894883155823, 2.2357141971588135, -0.16186268627643585, 0.03715578839182854, 0.672568142414093, 0.9831428527832031, 0.7222497463226318, 0.008166735991835594, 1.36588454246521, -0.575992226600647, 0.7500364184379578, -0.1362694948911667, 0.8738962411880493, -0.4231715798377991, -0.562169075012207, -0.8438923358917236, -0.8206202387809753, -1.7086690664291382, -1.2418261766433716, 0.44942474365234375, 0.05467278137803078, -0.11594043672084808, -0.5503945350646973, -0.34368136525154114, -1.6161377429962158, 0.18262876570224762, -0.49771779775619507, -0.1250685602426529], [0.5334286093711853, -0.16836433112621307, -0.14267197251319885, 0.3376745581626892, -0.5746557712554932, -1.8162164688110352, -0.14243318140506744, 1.16176438331604, -0.7582201361656189, 0.05981367826461792, 0.6323797702789307, -0.6745541095733643, 1.1270644664764404, 1.7245031595230103, 1.2527785301208496, -0.8885126709938049, -0.5499637126922607, -1.3195911645889282, 1.0263162851333618, -0.9786677360534668, -1.3612374067306519, -1.7533384561538696, -1.7192163467407227, 0.8066490888595581, 1.3978441953659058, 0.7405182719230652, 1.6187328100204468, -0.456699401140213, 0.20612609386444092, -0.04412534460425377, -0.10104922205209732, 1.774912714958191, 0.6040031313896179, 0.5884663462638855, -1.7718991041183472, 0.123679019510746, 0.409975528717041, 0.11810550093650818, -1.0194584131240845, 0.3443351984024048, 1.9019619226455688, 0.8728243708610535, -0.7383536100387573, 0.4176124036312103, -0.37411877512931824, -0.011176942847669125, -0.503487229347229, -0.23197013139724731, 0.49513009190559387, 0.4094802141189575, 0.6339474320411682, 0.07937170565128326, -1.3293323516845703, -2.02644944190979, 0.7058342099189758, 0.4988991320133209, 0.1771606206893921, -0.0015012682415544987, 1.5148892402648926, -1.8122618198394775, 0.9803628325462341, -0.9322230815887451, -1.450047254562378, 0.3451915979385376, 1.2205959558486938, 1.1756840944290161, -0.5245463252067566, -1.6749016046524048, -2.3773751258850098, 0.18372035026550293, 0.7526654005050659, -1.285306692123413, -0.61038738489151, 1.893193244934082, -0.5230436325073242, 0.7695716023445129, 0.0050487336702644825, -0.5572097301483154, 0.7912631034851074, -0.41041433811187744, -0.0019781943410634995, 0.545993447303772, 0.17861658334732056, 0.4808329641819, -0.35368111729621887, -0.9376927614212036, -1.0971096754074097, -0.44635701179504395, -1.3051927089691162, -1.0315876007080078, -0.5258466601371765, -0.9541263580322266, -1.0989240407943726, 0.07394625246524811, 0.0779217928647995, -0.030403347685933113, -1.838844656944275, -1.087660551071167, 0.07864347845315933, 0.015009966678917408, 0.26844680309295654, 1.0015759468078613, -0.20391623675823212, -0.19163325428962708, -0.5414531826972961, -0.34705761075019836, 1.0982409715652466, 0.1405632495880127, 0.7895407676696777, -0.6791138648986816, 0.461288183927536, 0.06799790263175964, -0.07104622572660446, -1.2083758115768433, 1.1575329303741455, 0.3778230845928192, -0.1432752013206482, -1.311691164970398, -1.0320302248001099, -0.4467870891094208, 0.6305316686630249, -1.417705774307251, -1.2243033647537231, -1.2502288818359375, 0.3499741852283478, -1.4746992588043213, -0.13042734563350677, -0.17901752889156342], [0.1690187007188797, -2.50972318649292, 0.34520068764686584, -0.016963046044111252, 0.5361335873603821, -0.1269112527370453, 1.4671705961227417, -0.07386862486600876, 0.21568696200847626, 2.292734384536743, 1.8282707929611206, 0.4100337624549866, -0.7705366611480713, -0.7724101543426514, -2.239227771759033, 1.2162595987319946, 0.02150094322860241, 1.575023889541626, 2.1138086318969727, -0.9159252643585205, 0.9130299687385559, 0.26308754086494446, 0.004949009045958519, -0.8897666335105896, -0.02767694927752018, -0.258944034576416, -0.9658499956130981, -0.16509009897708893, -0.5614053606987, -0.4097212553024292, 0.10039077699184418, -0.7353733777999878, -0.3236772418022156, -1.6915775537490845, 0.959796667098999, -1.0491515398025513, -0.4754303991794586, 1.4498190879821777, -1.6320961713790894, -0.9618552923202515, 0.15347453951835632, -0.18108457326889038, 0.6459929347038269, 0.379412978887558, -0.3510054051876068, -0.8560985326766968, -1.1585109233856201, 0.04743647202849388, 1.3287808895111084, 0.07624006271362305, 0.7768069505691528, -0.23295868933200836, 0.10460019111633301, 0.6635404825210571, -0.6056078672409058, 0.31396448612213135, -0.13095451891422272, -1.4812207221984863, 0.02156429924070835, -0.024996265769004822, -1.3207886219024658, -0.9595275521278381, -1.1335959434509277, 0.8028727173805237, 0.7898058295249939, 0.3527783751487732, -1.8060169219970703, 0.8254246115684509, 0.45838791131973267, -1.480842113494873, -1.2592378854751587, 0.21637345850467682, 0.8051910400390625, -1.3522990942001343, 3.6030185222625732, -1.4500579833984375, 1.0979939699172974, -0.973041296005249, 1.6891469955444336, -1.0664206743240356, -0.16430269181728363, 0.07070135325193405, 2.7269294261932373, -0.0747562050819397, 1.32365882396698, -1.4711037874221802, 1.5944325923919678, 0.3825658857822418, -0.12814635038375854, 0.9370618462562561, 2.3259992599487305, 1.4378094673156738, 0.18950028717517853, 0.5731092095375061, -0.4320423901081085, 1.0920965671539307, 0.7992693185806274, -1.8258355855941772, -0.036301594227552414, -2.0083775520324707, 0.7282710075378418, -0.5449680685997009, 0.4088094234466553, 0.14888311922550201, 0.947512149810791, 1.0821974277496338, -0.001695021754130721, -1.0324227809906006, -0.718594491481781, 1.2225836515426636, -0.501060962677002, 0.6504253149032593, -0.058109983801841736, -0.4424404203891754, 0.12057071924209595, -1.1402201652526855, 0.6077578663825989, -2.284590482711792, 0.00277212867513299, -0.3084234893321991, -0.07613667845726013, 1.350696086883545, -0.31637123227119446, 0.8083821535110474, -0.25729474425315857, 1.3131853342056274, 0.06910919398069382, -1.3053314685821533], [0.6600170731544495, 0.9696051478385925, 0.3810513913631439, 1.0114635229110718, -0.6019378304481506, 0.16309204697608948, 0.29401224851608276, -1.078275203704834, 0.40640759468078613, -1.361088514328003, -1.4140058755874634, -1.1648744344711304, 2.4175143241882324, -1.1648896932601929, -1.1725863218307495, 1.202641487121582, 1.2979696989059448, 0.04200836271047592, -0.01602954789996147, 0.12211731821298599, -1.3966981172561646, -1.949466586112976, 0.8099983334541321, -0.16343720257282257, -1.8484838008880615, -0.09853263944387436, 0.09323637187480927, -0.517092764377594, -0.6544161438941956, 0.12467799335718155, -1.020127296447754, 0.5224604606628418, 0.46200546622276306, -0.21219158172607422, -1.19185471534729, -0.5580061674118042, -0.1055714562535286, 1.423824429512024, -2.060500144958496, 1.0051225423812866, -0.41056784987449646, -0.2849760055541992, 0.9253109693527222, -2.659198045730591, -1.4849961996078491, -0.7208852171897888, 0.38012367486953735, 1.022829294204712, 0.7692892551422119, 0.935997486114502, 0.9953160881996155, -0.04750165715813637, -2.316464900970459, -0.42886969447135925, 0.054908256977796555, -0.06271832436323166, -0.7749615907669067, -0.41264864802360535, 0.553253173828125, -2.1641762256622314, -1.2317352294921875, -0.8590582013130188, 0.05208394303917885, -0.516474187374115, 1.7196543216705322, -0.08224198967218399, -1.2917405366897583, 1.9559658765792847, -1.3816323280334473, 0.4311860203742981, 0.08118003606796265, -0.5417254567146301, -0.5732836127281189, 0.1192793920636177, 1.1819829940795898, -0.3498484492301941, -0.4107869565486908, 0.7898049354553223, 0.23064038157463074, -1.1404798030853271, -0.2637304663658142, -1.4669172763824463, -0.3525876998901367, 0.7963137626647949, 1.6653000116348267, 0.4888550937175751, 0.5835220813751221, -1.75661039352417, -0.11715787649154663, -0.8558459877967834, 0.958246111869812, -0.1703169196844101, 1.1954857110977173, -0.08055359125137329, 0.8352581858634949, -1.0540273189544678, 1.1690069437026978, -1.2318378686904907, 0.4956916272640228, 1.352148175239563, 0.21519364416599274, -0.19624720513820648, 1.6531645059585571, 1.422203779220581, -0.40233877301216125, -0.12796907126903534, -0.9530673623085022, 0.08305077999830246, -1.0150314569473267, 2.0662808418273926, -0.5382589101791382, 0.4720365107059479, 0.676276445388794, 0.017249204218387604, 0.04167243838310242, -0.010725724510848522, -0.1391528844833374, -0.8263906240463257, -0.604882538318634, -1.5216450691223145, -1.279968500137329, 0.38735830783843994, -1.135086178779602, 0.8796607851982117, 1.5157302618026733, 0.23281820118427277, -0.7550962567329407, 0.5136340260505676], [-0.8943437933921814, -0.8512485027313232, 0.8007248044013977, -1.1175975799560547, -0.19211697578430176, 0.7803015112876892, 1.2427794933319092, 0.2556931972503662, -0.991030752658844, 1.3492499589920044, -0.13901367783546448, -0.688478410243988, -0.9432506561279297, 1.733550786972046, -0.33123642206192017, -0.8950965404510498, -0.7926121354103088, 0.008930384181439877, -0.34247657656669617, -0.6196960806846619, 1.0339572429656982, 1.188344955444336, -1.1579017639160156, -0.2071533203125, 1.9001363515853882, 0.7590616345405579, -0.521491289138794, 1.7473695278167725, 2.2674520015716553, -0.580156683921814, 1.6672444343566895, -0.23997214436531067, 0.13708218932151794, 0.7274700999259949, -0.9938122034072876, -0.039906784892082214, -0.026424622163176537, -0.2827197015285492, -0.291572630405426, -0.32615068554878235, 0.870094358921051, 0.10547401010990143, 0.9679774045944214, 0.656994104385376, 0.8254827260971069, -0.48794111609458923, -2.1665191650390625, -1.1760194301605225, 1.0945624113082886, -0.2309955656528473, -0.04549277573823929, 0.4964717626571655, 0.8784719705581665, -0.1037694588303566, 1.077447533607483, 0.00976090133190155, -0.8615154027938843, -0.4023829996585846, -0.8074449896812439, 0.7804920673370361, -1.9263628721237183, -0.25512829422950745, -2.2610347270965576, 1.740277886390686, -1.9471094608306885, -0.5379323959350586, 0.29076361656188965, 0.2559199631214142, 0.6355281472206116, -0.21874266862869263, 0.7776060700416565, 1.4390095472335815, 0.426066130399704, 1.4913146495819092, 1.5719884634017944, -0.29524847865104675, 0.489707887172699, 0.32855039834976196, 0.2920646071434021, 0.05014872923493385, -2.969860792160034, -0.7305744290351868, 1.3360118865966797, -0.019278092309832573, 0.6370548009872437, -0.02973151206970215, -1.1992390155792236, 0.23938824236392975, -1.769374966621399, 0.24550491571426392, 0.5043678879737854, 0.05317128449678421, -1.3705226182937622, 1.5257519483566284, -1.147389531135559, 0.2185237854719162, -0.7104631662368774, -0.7540221810340881, 0.5617884397506714, 0.16069120168685913, -0.5037268400192261, 0.14293502271175385, 2.0373687744140625, -1.96161949634552, -0.6666670441627502, -0.3257240056991577, 1.375793218612671, 1.0899133682250977, 0.230281263589859, 0.8235902786254883, -1.89937424659729, -0.20650002360343933, -0.6913382411003113, -0.9415494799613953, -0.5307968258857727, 1.8460513353347778, -1.3418512344360352, -0.7022117972373962, -3.1121952533721924, 1.517175555229187, 0.8530138731002808, -0.447136789560318, -2.16674542427063, -0.243112251162529, 0.3950909674167633, 0.680239200592041, -1.9510502815246582, 0.7602066397666931], [-0.14407773315906525, -0.7027798295021057, -0.024199234321713448, -0.14847254753112793, -0.5745260119438171, -0.37926140427589417, -0.201696515083313, -0.9424559473991394, -1.7624008655548096, -0.13779504597187042, 0.058077313005924225, -0.30903360247612, 2.3213841915130615, 0.44200071692466736, -1.9152402877807617, 0.08092573285102844, -1.9174596071243286, -0.44741541147232056, -0.47411054372787476, 0.7758057117462158, -0.4581354558467865, -0.02709471434354782, -1.5690675973892212, -0.12180830538272858, 0.3882145881652832, 0.7485036253929138, -0.5896517634391785, 0.7976345419883728, 1.7012779712677002, -1.4318249225616455, 1.0734989643096924, -0.8543310761451721, 0.12264072895050049, -0.3722136318683624, 1.2099792957305908, 0.3674476146697998, 1.1658298969268799, -0.5623104572296143, 0.7613005638122559, 1.2183918952941895, 1.227411150932312, -0.2123694270849228, -0.31955188512802124, -0.5890840291976929, 0.24685139954090118, -0.6689465045928955, 0.7803186774253845, -0.6350199580192566, 0.13137218356132507, 1.0782965421676636, -0.8243578672409058, 2.046769380569458, 0.4795973300933838, 0.5033782124519348, -0.09077954292297363, -0.8281724452972412, -1.189591884613037, -0.11085345596075058, 0.25399115681648254, -0.779827356338501, -0.19571854174137115, 2.777392625808716, 0.9922673106193542, 0.5632205009460449, 0.26877930760383606, -1.5296822786331177, -1.0590795278549194, -0.4548034369945526, -1.8500542640686035, -0.10333477705717087, 0.18696311116218567, -0.9538331031799316, -1.0881460905075073, 0.7126323580741882, 0.3855746388435364, -0.574472188949585, -1.0291327238082886, 1.7131214141845703, -0.6861549019813538, -0.23930230736732483, -0.21259979903697968, -0.7704156041145325, -0.514406681060791, -0.09294301271438599, -1.0677913427352905, 0.47588050365448, 1.1964784860610962, -1.3547108173370361, -1.3800259828567505, 2.1421334743499756, 1.4168908596038818, 0.20619021356105804, 0.47212591767311096, 0.9109686017036438, -0.6931756734848022, 0.01120662409812212, -0.09040089696645737, 0.8922623991966248, 1.6602245569229126, -0.010401391424238682, 1.0945097208023071, -0.4115789830684662, 0.5744607448577881, -0.5800568461418152, 1.3316096067428589, 0.006248993333429098, 2.763949155807495, -1.298140287399292, 1.0009093284606934, -0.6026968955993652, -0.5687159299850464, -1.069434642791748, 0.4321536719799042, 0.708530843257904, -1.011365532875061, 0.015256263315677643, -0.1717052310705185, -0.5135038495063782, -0.5319547653198242, -0.48000285029411316, -1.3392761945724487, -0.807663083076477, 0.029532935470342636, -0.6220003962516785, 0.326986700296402, -0.9351699352264404, -1.1119873523712158, 0.6737672686576843], [-0.43234091997146606, 1.1492438316345215, -1.716366171836853, -0.4511163830757141, 0.6051326990127563, -0.8849266171455383, -0.7345191240310669, -1.0400571823120117, 0.23854941129684448, -1.278882622718811, -0.3344130218029022, 0.026203395798802376, -1.3000668287277222, -0.6770753264427185, -0.2944181561470032, 0.08374693989753723, 0.5821255445480347, -0.06763250380754471, 0.13439081609249115, -0.14440298080444336, -1.4027748107910156, -0.4787752032279968, -0.15536971390247345, 1.3204443454742432, -0.7010921835899353, -1.5627437829971313, 1.970062494277954, 0.865233302116394, 0.7611559629440308, -1.4591310024261475, 1.2197459936141968, 0.3110319674015045, 0.7203412652015686, -0.8099783062934875, 0.9343849420547485, -1.0299617052078247, 0.44644981622695923, -0.15865710377693176, 0.4994105398654938, 0.6903713345527649, 2.058523654937744, -0.8617464303970337, 1.1232107877731323, -0.027083417400717735, 0.20616205036640167, -0.1066742092370987, -0.5108306407928467, -1.4098951816558838, 0.030060257762670517, 1.2398682832717896, 0.11797414720058441, 0.7873071432113647, -0.1861293762922287, 1.240054965019226, 0.4247852861881256, -0.31905272603034973, -0.9521795511245728, -1.115993618965149, -2.484867811203003, -0.17088095843791962, -0.6316164135932922, 1.5420613288879395, -1.1403119564056396, 0.24772794544696808, 0.03366852551698685, -0.6455511450767517, 0.26242566108703613, -0.9495189785957336, -0.045835960656404495, -0.029091039672493935, -0.3517557680606842, -1.07912015914917, -0.19807299971580505, -0.15435567498207092, 0.8524442315101624, -0.5833519101142883, -1.3768788576126099, -0.18929129838943481, -1.2583038806915283, -0.2805113196372986, 0.8823221325874329, 0.4307722747325897, 1.0323636531829834, 0.5834793448448181, -0.520725667476654, 1.359131097793579, -0.8172135949134827, -0.28194180130958557, 2.5284087657928467, 0.9163568019866943, 0.9631589651107788, -0.1502452939748764, 1.1608843803405762, -0.3404189646244049, 0.7516852617263794, 0.01219184510409832, 0.1661650836467743, 2.0270678997039795, -0.6688241958618164, -2.2550482749938965, -0.7813853621482849, 0.8290851712226868, -1.6836045980453491, 0.10539378225803375, -1.7602403163909912, -1.8823004961013794, 0.41762644052505493, 2.3779168128967285, -0.6169632077217102, -0.4353184401988983, -0.8950303792953491, 1.255114197731018, -0.43096107244491577, -0.28158828616142273, -1.1062054634094238, 0.6403093934059143, -0.386600136756897, 0.5252866148948669, -2.5259649753570557, -0.10095129162073135, -1.4474841356277466, 0.47510018944740295, -0.9548288583755493, -1.3933424949645996, -0.6725009679794312, -1.1849418878555298, 0.9144546389579773, -0.2192644625902176], [-0.6140868663787842, 0.7463349103927612, -0.22590869665145874, 0.7913715839385986, -0.21250596642494202, 1.6334185600280762, -0.6405211687088013, -0.3497883975505829, 0.3837812840938568, 0.2228737622499466, -0.3887883126735687, 0.8073573112487793, 0.7149900197982788, -1.6746149063110352, 0.8076805472373962, -1.5181655883789062, -0.3753613531589508, 1.8639795780181885, -0.22899718582630157, 0.7655280828475952, 0.8855005502700806, 1.667808175086975, -1.298040509223938, 0.3445892632007599, -0.6160724759101868, 1.9998332262039185, 1.5849413871765137, 0.6806790232658386, -0.8483291864395142, -1.0324065685272217, -1.0489827394485474, 0.6432576775550842, -0.9322773218154907, 1.9750710725784302, -0.48059165477752686, -1.1968473196029663, 0.07285383343696594, 1.19502854347229, -0.3179095983505249, -0.7627723813056946, -0.6389206647872925, 1.2779541015625, -1.1981596946716309, 2.051915407180786, -0.7841141223907471, 1.7426860332489014, 1.6119099855422974, -0.21906104683876038, -1.323128581047058, 1.2021145820617676, 0.7382863759994507, -0.7651596665382385, -0.08224273473024368, 1.0973613262176514, 2.3433914184570312, -0.5179913640022278, -0.35575950145721436, 2.405881881713867, 0.13058263063430786, -1.3362714052200317, -0.7219541668891907, -1.2547986507415771, -1.2371517419815063, -0.24547524750232697, -0.999056339263916, 1.4456208944320679, -0.2642972469329834, -0.7869678735733032, 0.21985170245170593, 0.8754696846008301, 0.5628618597984314, 1.280774712562561, 0.2979702949523926, 0.36470505595207214, -0.8943614363670349, 0.0910663902759552, 1.8233766555786133, 0.06940024346113205, -1.4578981399536133, -0.9714756011962891, -0.29937922954559326, -0.2456308752298355, -1.1696840524673462, 2.4229817390441895, -0.30175405740737915, -1.8136048316955566, 0.6970560550689697, 2.2708003520965576, 1.3459858894348145, -0.40635204315185547, -0.3424980640411377, 0.5205355286598206, -1.4828846454620361, -1.3523184061050415, 1.2861099243164062, -0.19981355965137482, -0.00997912883758545, -1.6210567951202393, 1.1668728590011597, 0.2857690453529358, 1.6587481498718262, 0.9605543613433838, 0.5103550553321838, 1.3070096969604492, -0.2489522397518158, -2.0997908115386963, -0.2472183108329773, -1.2894519567489624, -1.188159465789795, -0.01853795349597931, 0.2857813239097595, 2.11344313621521, 2.6378395557403564, -0.84867924451828, 1.326833724975586, -0.6866725087165833, -0.04101508855819702, 1.2281142473220825, -0.7655645608901978, 1.692438006401062, 1.0305182933807373, -1.9621880054473877, -0.8031471371650696, 0.6717237830162048, -0.4510568678379059, -1.0940499305725098, 1.061368465423584, -0.03670226410031319], [-1.5707265138626099, -1.737463355064392, -1.0874614715576172, -0.46646925806999207, -0.35744887590408325, 0.8057549595832825, 1.4232921600341797, 0.28055498003959656, -0.0032437017653137445, -0.25995004177093506, -0.7769685983657837, 0.2792685329914093, -0.06282445043325424, 1.3843368291854858, -1.9920850992202759, 0.20135775208473206, -0.7287841439247131, -0.44173407554626465, 0.03968128561973572, 1.453596591949463, -0.8715915679931641, -0.31308481097221375, -0.9781773090362549, -0.5994173884391785, -1.7339978218078613, 0.7976399660110474, 0.053039830178022385, 2.6772475242614746, -0.9794405698776245, 0.022965438663959503, 1.1337727308273315, -0.8895830512046814, 0.11526888608932495, -0.6220961809158325, 0.5209278464317322, 0.9744558334350586, -1.2407947778701782, -3.1820640563964844, -0.9295853972434998, -0.3884434103965759, -0.33854323625564575, 0.656183660030365, 0.10669700056314468, -0.5872064232826233, 1.2841089963912964, 1.1471080780029297, 0.9398382902145386, -0.18827617168426514, 0.46234118938446045, -0.5709863901138306, 0.35136619210243225, 0.46408724784851074, 0.05637001246213913, 1.4007455110549927, -0.880326509475708, 0.005673401989042759, 1.869793176651001, -0.13997532427310944, -0.9344554543495178, 0.5807690024375916, 1.851540207862854, -1.3870089054107666, -1.2504740953445435, 0.15063650906085968, -0.15509124100208282, 1.2504758834838867, 0.4481644630432129, -0.20258194208145142, 0.6926933526992798, 0.33994221687316895, 0.6993900537490845, -1.8201030492782593, 0.6099914908409119, 0.41120585799217224, -0.1630593240261078, -1.6305054426193237, 0.2606087327003479, 1.83645761013031, -1.8058851957321167, 0.1763588786125183, 1.3620502948760986, 0.16525718569755554, 0.1500525325536728, -1.117846965789795, -1.5330477952957153, -1.040753722190857, -0.6061933636665344, -1.3225092887878418, -0.3633299469947815, -1.194923758506775, -0.1324204057455063, -1.1777395009994507, -0.12557494640350342, 1.427794098854065, -0.3745281994342804, -0.011366153135895729, 0.7925261855125427, -0.029875129461288452, -0.48965921998023987, 0.3969077467918396, 0.05746491253376007, 0.7010413408279419, -1.7798703908920288, 0.9394984841346741, 1.1393556594848633, -0.24855098128318787, -0.91377192735672, 2.313239336013794, 0.7125122547149658, -0.06739449501037598, -0.9040926098823547, 3.5580427646636963, 0.030653774738311768, -0.6239261031150818, 0.8092226386070251, -0.8754948377609253, 1.4347172975540161, -0.7552576661109924, -1.1418706178665161, -0.46552127599716187, -0.6836767196655273, -0.693626344203949, -1.910348892211914, 0.04018787294626236, -3.4781908988952637, -1.1596540212631226, -1.9182556867599487, -0.87637859582901], [-2.0271553993225098, 0.8027477264404297, -0.14424070715904236, -0.5886495113372803, 0.38249367475509644, 0.9989504218101501, -0.8545860648155212, -0.6170210242271423, -0.29720163345336914, -0.9333170652389526, -0.17208197712898254, 1.2789429426193237, -0.26653823256492615, 0.9241009950637817, 1.5253790616989136, -0.8090770840644836, 0.34208905696868896, -0.30983129143714905, 0.895336925983429, 0.5766947865486145, 0.7109934091567993, 0.34742996096611023, -0.15562549233436584, -0.14106087386608124, -0.19919779896736145, 0.4639033079147339, -0.545640230178833, -0.5384092330932617, 1.8035812377929688, 1.0094866752624512, -0.32351627945899963, -0.4599982500076294, -1.2971453666687012, 1.8036245107650757, -0.6255441308021545, 0.4624236226081848, 0.06398047506809235, 1.2662385702133179, 0.5698865056037903, 0.9582415819168091, -1.4343219995498657, 1.7579807043075562, 1.846619963645935, -0.13506516814231873, -1.551738977432251, 3.844719648361206, 0.42294928431510925, 0.47810301184654236, -1.0075218677520752, -0.28341859579086304, -0.6136904954910278, -0.3670800030231476, 0.5108323693275452, 1.647557258605957, 0.7064258456230164, 0.14849022030830383, 1.2069470882415771, 0.11699726432561874, -0.26634180545806885, -0.9242733120918274, -1.0992486476898193, -0.36616504192352295, 0.7704501152038574, -0.05963223800063133, 0.42587584257125854, 0.7366383671760559, -0.12907809019088745, 1.1624531745910645, 0.6963824033737183, 0.19531482458114624, -1.0248693227767944, 0.8984017372131348, -1.3969484567642212, -0.6094592809677124, -0.07958413660526276, 0.6176288723945618, 0.7496747374534607, -1.0898903608322144, -0.3524160385131836, 1.236958384513855, 0.12887419760227203, -0.6902201175689697, -0.3981515169143677, -1.7241225242614746, 1.9083737134933472, 0.5921896696090698, -1.3020070791244507, 1.4363784790039062, -1.5816137790679932, 1.4820706844329834, -0.9913613796234131, -1.4556530714035034, 1.6024937629699707, 0.664740800857544, -1.0330939292907715, -0.6950850486755371, -1.0397425889968872, -0.05324213206768036, -0.34587523341178894, 0.5260248780250549, -0.7183492183685303, 0.3507060408592224, -0.6696536540985107, -2.1642162799835205, -0.10947470366954803, 0.47727295756340027, 1.5251489877700806, -0.4356071352958679, 1.2269370555877686, -1.3131424188613892, 1.1550934314727783, 0.06316516548395157, -0.29007652401924133, -1.3886926174163818, 2.4185123443603516, -0.2714104652404785, 1.4978173971176147, -0.044447459280490875, 0.6298116445541382, 1.2942503690719604, -1.0662845373153687, 1.3127654790878296, 1.6346772909164429, -1.607390284538269, -0.08131302893161774, 1.3766188621520996, 0.25017163157463074, -1.5382812023162842], [1.7515182495117188, -1.497429370880127, 1.1949838399887085, -0.01881534978747368, -0.31274059414863586, -1.4316656589508057, 1.0863096714019775, -0.4455978274345398, 0.874057412147522, -0.9774696230888367, -1.3675576448440552, -0.9506800770759583, 0.5249572396278381, 1.0837221145629883, 0.9395876526832581, 1.2105765342712402, -0.3006207048892975, -0.09806262701749802, -1.503342628479004, 0.8652187585830688, -1.1468896865844727, 0.25301074981689453, -0.06420983374118805, -0.30598124861717224, -0.47510120272636414, 0.16174961626529694, -0.003265351289883256, -0.18624083697795868, 0.1290394812822342, -0.23147006332874298, 1.7218822240829468, 0.6268042922019958, -0.2100965976715088, 1.718488335609436, 0.7998342514038086, -0.012864500284194946, -2.2529077529907227, -0.017726436257362366, -1.0554600954055786, -0.8857407569885254, -0.3289116621017456, 2.021087408065796, -0.10612869262695312, -0.5525121688842773, 1.1331819295883179, 0.13793841004371643, -0.836999773979187, 0.552573561668396, 0.2588334381580353, -0.9370191097259521, -0.37404078245162964, -0.11649643629789352, -0.18368607759475708, -0.9433177709579468, 0.05700128898024559, 0.3191021680831909, 0.4271266460418701, -0.7407305836677551, 0.08234154433012009, -1.2193742990493774, 0.8966989517211914, -0.42882078886032104, -0.02495361492037773, -0.11473539471626282, -0.24331040680408478, -0.15313564240932465, -0.40619713068008423, 0.31528738141059875, -0.5369192361831665, 0.6970126032829285, 0.12835416197776794, 0.19832883775234222, 0.8922441601753235, 0.3823131024837494, -0.7805653214454651, 1.3329052925109863, -0.4930690824985504, 0.46574217081069946, 0.7177577018737793, -0.9157903790473938, 0.3332958519458771, 0.5679868459701538, 0.39654189348220825, -0.5382434129714966, 1.2860482931137085, 0.6716096997261047, -0.739683210849762, -1.2492350339889526, -0.44959282875061035, -0.6482848525047302, 1.214621663093567, -0.20933526754379272, 0.006894663907587528, 0.6297329664230347, -0.15848639607429504, -1.4347418546676636, -0.012543591670691967, 2.0689306259155273, 1.1141736507415771, 2.203807830810547, -0.41652053594589233, 1.7563532590866089, -0.6960921287536621, -0.5870475172996521, -0.43231910467147827, -1.2267340421676636, -0.25208383798599243, 1.290508508682251, 0.7767112851142883, -0.11087339371442795, 0.5007882118225098, -0.9874315857887268, 0.6074182987213135, -0.10542720556259155, -0.23401419818401337, -0.4650067090988159, 1.4263514280319214, -0.21160584688186646, -0.0729372501373291, 1.7193827629089355, -0.8725672364234924, 0.712701141834259, -0.20612628757953644, -1.39212965965271, -0.8354218602180481, -2.4019863605499268, -0.3007877469062805, -1.2754664421081543], [-0.2068035751581192, 1.160287618637085, -0.8861807584762573, -0.8395784497261047, -0.6261517405509949, -0.4361953139305115, -0.6422572731971741, -0.21498559415340424, -0.6485071778297424, -1.358985424041748, -0.7775151133537292, 1.8316166400909424, 0.0895468071103096, 0.42003268003463745, -1.5552453994750977, -0.030001094564795494, 0.008786396123468876, 0.7489911317825317, 1.936044692993164, -0.756026029586792, -1.6629629135131836, -0.5060412883758545, -0.42480388283729553, 0.1233091875910759, 1.0568791627883911, 0.33773303031921387, -1.0708415508270264, 0.04165535047650337, 1.174120545387268, -1.8746262788772583, -0.8731805682182312, 0.062085602432489395, -0.283333420753479, -0.3207632303237915, -0.6241697072982788, -0.8120424747467041, 0.5001199245452881, -0.03822750225663185, 0.3729054629802704, -0.7117310166358948, -1.0234771966934204, 0.7394225597381592, 1.0575374364852905, -0.40099671483039856, 1.8112643957138062, -1.6476117372512817, 0.4356260299682617, 0.3630128502845764, 0.32050445675849915, -1.357841968536377, 0.4770466387271881, 0.5821018218994141, 0.31149137020111084, 0.30995309352874756, 1.0484592914581299, 0.6895374655723572, -0.5231663584709167, -0.2624247968196869, 0.635952889919281, 0.6840107440948486, 1.1120554208755493, -0.6773186922073364, 0.7769914865493774, -0.32514283061027527, 0.35645949840545654, -0.20036935806274414, -1.8902859687805176, -0.524603545665741, 0.024397369474172592, -0.4354139268398285, -0.7092210650444031, 0.8566162586212158, 0.737023115158081, -0.2855263650417328, 0.23331287503242493, -0.2596161365509033, -1.4457275867462158, 1.6132484674453735, 1.001134991645813, 0.32783374190330505, -1.0641053915023804, 1.2903369665145874, -0.5766648650169373, -0.5952643752098083, -0.4830210506916046, -0.8947365283966064, -0.001734933815896511, 1.2716991901397705, -1.0819698572158813, 0.06918488442897797, -0.17922982573509216, -0.08828737586736679, 0.23930609226226807, -0.034656986594200134, -1.144987940788269, 0.4070817530155182, -0.5863572359085083, -0.3179575502872467, 1.528380274772644, 1.0375092029571533, 1.5056581497192383, -1.595988392829895, 1.0544090270996094, 0.6423718333244324, -0.0631033256649971, -1.5623432397842407, -0.7027712464332581, 0.8772457242012024, 0.5227280855178833, 0.9600414633750916, -0.17572727799415588, -0.04123429208993912, 0.886994481086731, 1.5218710899353027, -0.3822292983531952, 0.05556602403521538, 0.5504563450813293, -1.1485252380371094, -1.347357153892517, -0.3118760883808136, -0.3489847481250763, -1.400341510772705, 1.0153099298477173, 1.4609397649765015, 1.1000475883483887, -0.20376534759998322, -1.2582507133483887, -0.1511353850364685], [0.161670982837677, -0.14490395784378052, -2.1770212650299072, -0.636091411113739, 0.784709632396698, 0.9372953176498413, 1.4539787769317627, 0.009215456433594227, -0.9328352808952332, 0.43436476588249207, 0.45599278807640076, -1.5318666696548462, 0.27274760603904724, -0.7093356251716614, 0.3918328881263733, 1.1959818601608276, -2.116534471511841, 0.009050947614014149, -1.191855788230896, 0.4875319302082062, 0.042701125144958496, 2.4496073722839355, -1.2838550806045532, 0.6361275315284729, 0.3249286711215973, 0.4326359033584595, -0.5140994191169739, -0.8508847951889038, 1.1536387205123901, -0.4306560754776001, 0.9452899694442749, 1.517390251159668, -1.4708232879638672, -1.5263255834579468, -0.678555428981781, 0.06584886461496353, -0.5532499551773071, -1.5149520635604858, 0.6317638158798218, 0.29562652111053467, 1.274932861328125, 0.10502764582633972, -1.5186859369277954, 0.7379623055458069, 0.2446897327899933, 0.7036125659942627, -0.9571954607963562, -0.31285563111305237, -0.7485828399658203, -0.2824031114578247, -0.3111860156059265, -1.1701525449752808, 0.8382193446159363, 0.30586326122283936, -0.4110056757926941, 0.696488618850708, -1.31427800655365, 0.1219056025147438, 0.7929601073265076, -0.14468196034431458, -1.3419996500015259, 0.2002965211868286, -1.9916390180587769, -1.3773131370544434, 0.016856597736477852, -0.5559500455856323, 0.5215370655059814, 1.3803684711456299, -1.3427608013153076, 0.490643709897995, -1.2249951362609863, -0.28194233775138855, -0.49894979596138, -1.504148006439209, 1.5576252937316895, 0.6063662767410278, 0.41210561990737915, -0.2050819844007492, -0.6563809514045715, -0.2996971011161804, -1.8691881895065308, 0.5042209029197693, -2.2511863708496094, 0.1094740554690361, 0.9059085845947266, -0.10523249953985214, 0.06160452961921692, 1.4635882377624512, -2.756657838821411, 0.6950775980949402, -1.2148892879486084, -0.6284158825874329, 0.24426788091659546, -1.534887433052063, -0.8859089016914368, -1.1879260540008545, -0.9963801503181458, 0.7745469212532043, 0.5693991780281067, -1.3274471759796143, -0.06203512102365494, -0.01192467100918293, -0.987187385559082, 0.12590628862380981, -0.11967536807060242, -0.2761215567588806, 0.4172503352165222, 3.054219961166382, 1.0416454076766968, -0.7741469740867615, -0.6174221038818359, -1.3354076147079468, -1.3084584474563599, 1.196030616760254, -0.6278941035270691, -0.6394375562667847, 0.6216391921043396, -0.41925248503685, 2.823700428009033, -1.1069554090499878, 1.0618089437484741, -0.2484760731458664, 0.6823870539665222, -1.434417963027954, 0.07355162501335144, 1.0706473588943481, 0.4099828898906708, 0.3535187542438507], [-0.9504514336585999, -0.3489457368850708, -0.8682082891464233, -0.1548958271741867, 0.11020819842815399, 0.2805488109588623, 2.7138736248016357, 0.6807806491851807, 0.23567438125610352, 0.9602375626564026, 0.973319947719574, 0.015881558880209923, 1.3574329614639282, 0.023668544366955757, -0.24720601737499237, 1.4849697351455688, -0.8489294648170471, 0.04295264929533005, -1.7125720977783203, -0.9270415902137756, 0.9634454250335693, 0.6447610855102539, -0.4442962110042572, 0.13590021431446075, -2.4352798461914062, 0.9762057662010193, -0.08844056725502014, 1.4273223876953125, 1.5981743335723877, 0.31458258628845215, 1.5149186849594116, 1.0069845914840698, 0.551778256893158, -0.12748879194259644, 0.4934449791908264, -0.34685343503952026, 0.36718884110450745, -2.5159735679626465, -0.027875909581780434, -0.27386903762817383, -1.3141698837280273, -1.765194296836853, 1.2067643404006958, -1.3198686838150024, 1.7032192945480347, 0.5442086458206177, -0.7421891689300537, -0.17444460093975067, 0.07093390822410583, -0.7727276086807251, 0.09782733023166656, 0.50816410779953, 0.6990344524383545, -0.050685230642557144, 2.236070156097412, -0.3396334946155548, 1.2879489660263062, -0.3153949975967407, -0.15046508610248566, -0.46353599429130554, -0.3096262514591217, 1.689217448234558, 0.47237876057624817, 0.6636879444122314, -0.4333335757255554, -1.4281471967697144, -0.9653188586235046, 0.7114900350570679, 0.08045827597379684, 0.5683651566505432, 0.8167611360549927, -0.9646905660629272, 1.0444484949111938, -0.7711750864982605, -1.73200261592865, 1.2093958854675293, -1.4127646684646606, 1.1653305292129517, 0.767428457736969, -1.461305022239685, 1.7155015468597412, -0.02082885429263115, -0.6241352558135986, 1.3480170965194702, 0.17514966428279877, 0.6770642399787903, -0.6950476169586182, -0.08686556667089462, -0.5245723724365234, 0.4066616892814636, 0.29733312129974365, -1.9967446327209473, 1.6702932119369507, -1.5104444026947021, 2.016937017440796, 1.3605382442474365, 0.7563709020614624, -0.21978123486042023, 0.12434912472963333, 2.178210735321045, 0.4075036346912384, -0.05212569981813431, 0.6227918863296509, -0.14643879234790802, -1.1702983379364014, -0.38652247190475464, -1.2431639432907104, -1.813309907913208, 0.9593245983123779, -0.5856059789657593, -1.1837022304534912, -0.10365170240402222, -0.3966343104839325, 0.26945140957832336, 2.604215621948242, 0.9124467372894287, -0.6191838383674622, 1.5028634071350098, -0.13721448183059692, -2.0916481018066406, -0.555766224861145, -0.5682842135429382, 0.02809559740126133, 0.6190409660339355, 1.2410188913345337, -0.2645025849342346, 0.05553395301103592, -0.03990945219993591], [0.7386738657951355, 1.192438006401062, -0.02941344864666462, -0.14983174204826355, 0.9176080226898193, -0.08389190584421158, 1.1578267812728882, -0.6510147452354431, -1.0536253452301025, -1.2801165580749512, 0.8935281038284302, 0.9565981030464172, -0.09366964548826218, 1.4640538692474365, -0.1679726094007492, -0.7874616384506226, 0.374421626329422, 0.007062797900289297, 0.8086742162704468, 1.2168751955032349, 0.5061208605766296, 1.3874841928482056, 0.3789145350456238, -0.22998446226119995, -0.9029055237770081, -1.669738531112671, 0.8843964338302612, -0.2331179976463318, -0.2811223566532135, 0.5969957709312439, 0.4208976924419403, 0.06852118670940399, 0.7892094254493713, 1.2671619653701782, -0.43280962109565735, -1.0191171169281006, -2.957637310028076, 1.1527650356292725, -0.607014000415802, 0.49193382263183594, -1.1247708797454834, -0.23639295995235443, -0.07107464224100113, -0.8326095342636108, -0.33223116397857666, -0.5660890340805054, 1.2768760919570923, -1.0153908729553223, -2.5175039768218994, 0.8878321647644043, -1.0728657245635986, -2.2788000106811523, -1.4664173126220703, -1.8700937032699585, 1.5803731679916382, -0.6595649719238281, 0.896245539188385, -0.255886971950531, -0.050796303898096085, 0.8085988163948059, 0.6072136163711548, -1.4572478532791138, 0.2556931972503662, 0.09771127253770828, 1.2037832736968994, -1.3097765445709229, 0.012056342326104641, -0.4122110605239868, -0.20507879555225372, 1.1168490648269653, 0.6679513454437256, -0.4487674832344055, -1.7600634098052979, -0.762373149394989, -1.8385951519012451, 1.3341426849365234, -2.3284144401550293, 1.9422961473464966, 0.12279704213142395, -1.3930155038833618, 2.506063461303711, 1.3470555543899536, -0.9440853595733643, -1.7566683292388916, 0.10897163301706314, 0.4050484001636505, 0.38574209809303284, 0.6841772794723511, 0.0987776592373848, -0.1885831356048584, 0.7986656427383423, -1.1750118732452393, -0.040977347642183304, -0.6512038111686707, -2.082923650741577, 0.21948795020580292, 0.6221696138381958, -1.670628547668457, -0.6409616470336914, 1.600191593170166, -0.58402419090271, -1.6642447710037231, 0.7750014662742615, -0.41105782985687256, -2.645657539367676, -0.18484963476657867, 0.86714768409729, -0.45837634801864624, -2.0766656398773193, -2.2268121242523193, -1.2283871173858643, 0.7838187217712402, 0.29433220624923706, 0.00823289342224598, 1.0628857612609863, 1.0981935262680054, -0.4184287488460541, -0.9962877631187439, 0.743206262588501, 0.17984646558761597, 0.6924263834953308, -0.47100213170051575, -0.41490432620048523, -0.5995801687240601, -0.3501666486263275, 0.6989657282829285, -0.34087803959846497, 0.514953076839447], [0.954552173614502, 0.2817003130912781, -1.8591296672821045, 0.11707066744565964, -0.776728630065918, -0.7829762101173401, 0.9626268148422241, 0.6779994368553162, -1.1734485626220703, 0.06560210883617401, -1.7879612445831299, -1.3058701753616333, 0.9581066966056824, -1.0136240720748901, 0.264873206615448, 0.46208181977272034, -1.016409993171692, -0.7335991263389587, 0.7285773754119873, -0.18537990748882294, -2.000044345855713, -0.041166406124830246, -0.14493274688720703, 0.4084113538265228, -0.9729576110839844, -0.4500369131565094, 1.6250654458999634, -0.38585609197616577, -1.1245555877685547, 0.27554216980934143, 0.7702082395553589, 1.062578797340393, -0.3166092038154602, -0.5339944362640381, -1.2445636987686157, 0.6150048971176147, 1.212770938873291, -0.09326168894767761, 0.7384211421012878, -0.7532533407211304, -0.8736510276794434, -0.9441811442375183, -0.7773253917694092, -1.1094163656234741, -0.4263404607772827, -0.08603805303573608, -0.6909886002540588, 0.546692430973053, 0.42922085523605347, 0.09530419856309891, 0.7694107890129089, -0.6879841685295105, 0.8106511831283569, -0.05795198678970337, -0.8715855479240417, 0.10630882531404495, 0.2747823894023895, -1.1925462484359741, -0.330390989780426, 1.659159541130066, -0.5744438767433167, 0.6865015625953674, -0.16050328314304352, 0.8805598616600037, 0.1975875347852707, 0.9803181290626526, -0.06376311928033829, 2.036562204360962, 0.2681950032711029, -1.1035118103027344, -0.04635436460375786, 0.9261860251426697, 2.8511455059051514, 0.3635294437408447, -0.2606126070022583, -0.7797002196311951, 0.3234023451805115, -0.13820582628250122, -0.8757036328315735, -1.128904104232788, 0.04196454584598541, -1.3830199241638184, 0.04005468264222145, -0.2511933743953705, 0.9016466736793518, -2.481854200363159, -1.7411390542984009, 0.10861540585756302, -0.43746379017829895, -0.6833099722862244, 0.1747928410768509, -2.3105688095092773, 1.5838301181793213, 0.6624014973640442, -1.2665272951126099, -2.3329570293426514, 0.05593324825167656, -0.3840520977973938, 0.46461060643196106, 0.9325466156005859, -0.5020517706871033, -0.12004287540912628, 2.1203105449676514, 1.0845355987548828, -0.49527856707572937, -0.12706446647644043, 1.5094400644302368, -0.05915338546037674, -1.1195224523544312, 0.015368013642728329, 1.578104019165039, -0.47808825969696045, 0.5559245347976685, 0.5255018472671509, 1.2301565408706665, -1.9502841234207153, 1.5630271434783936, -1.915480375289917, -1.536124587059021, 0.24690718948841095, -0.3559167981147766, 1.622166395187378, -0.8450373411178589, -0.36644139885902405, -1.644887089729309, -0.2754181921482086, -1.0351349115371704, 0.31836676597595215], [1.438698649406433, 0.8203530311584473, 0.32090604305267334, -0.04309554025530815, 0.7732214331626892, -0.4621564745903015, -0.8825855851173401, -0.3844287693500519, 1.230128288269043, -0.18637919425964355, 0.42631062865257263, 0.2240392565727234, -1.231762409210205, -0.5327777862548828, -2.8588640689849854, -0.0026941413525491953, 0.6689296364784241, 0.9625792503356934, -1.0393644571304321, -0.09961232542991638, 1.4319355487823486, 0.7866136431694031, -0.6774757504463196, -0.24963736534118652, 0.7612595558166504, 0.8894112706184387, 0.1748933345079422, -1.0201259851455688, -0.9648531079292297, 3.566561460494995, 1.5478893518447876, -0.764117956161499, -1.1116725206375122, -0.7079845070838928, 0.0622907355427742, 0.4288294315338135, -0.059903401881456375, -0.8702148795127869, 0.24673226475715637, 0.24127288162708282, 0.47409164905548096, -0.872714638710022, 2.440329074859619, 0.5421764254570007, -1.4281939268112183, 0.0974699929356575, -1.5456750392913818, 0.45017698407173157, 1.16562819480896, -1.228800892829895, -1.0029053688049316, -0.6466426253318787, 0.2983923852443695, -1.5156441926956177, 1.9430391788482666, -0.7032290101051331, 2.2370142936706543, -2.0567500591278076, -3.476977825164795, 0.504268229007721, -1.904829740524292, 0.1679932028055191, -0.22892843186855316, 0.6331666111946106, 1.1441354751586914, 0.935961902141571, 0.2876882255077362, 0.7032037377357483, -0.17285969853401184, 0.12808211147785187, -0.9915329217910767, 0.7000906467437744, -0.3885161876678467, 0.6820678114891052, -3.0113635063171387, 0.7588586807250977, -0.8443509340286255, 0.8578799366950989, 1.8550699949264526, -1.7398885488510132, -0.013686822727322578, -0.01869560033082962, -0.34469524025917053, 1.460763692855835, -0.9338086843490601, -0.27602720260620117, 1.270512342453003, 0.3337201774120331, -1.5892950296401978, -1.0985329151153564, 0.5968955159187317, 1.9206560850143433, 2.130798816680908, -0.30553513765335083, 0.6816486120223999, -2.4479517936706543, -2.7851853370666504, 0.3862638473510742, 0.5600831508636475, 0.04649057611823082, 0.46673014760017395, 1.971990942955017, 0.06898759305477142, 1.033581256866455, 2.2913618087768555, 2.131927490234375, -0.644770622253418, -0.6722733378410339, 0.638599157333374, -0.5319429636001587, 0.5245454907417297, 2.2187225818634033, 0.27398115396499634, 0.06245483085513115, 0.7402900457382202, 0.9156484007835388, -0.6431739330291748, -0.3989195227622986, -1.2206835746765137, -0.4070679247379303, 0.9570764899253845, -1.3598754405975342, -0.20277319848537445, 0.3171631991863251, 0.9452939033508301, -0.43875446915626526, -1.877411127090454, -0.03624683991074562], [1.5195101499557495, -2.195685625076294, -0.2682654857635498, 0.30559390783309937, -0.47561395168304443, 0.7205755114555359, 1.8881334066390991, -1.340309500694275, -0.4554998278617859, 0.28823158144950867, -0.18827861547470093, -0.01588287577033043, 1.017991304397583, -2.8123061656951904, -0.48751187324523926, -0.44330111145973206, 0.918580174446106, 2.3477022647857666, 0.6804558038711548, -1.3203517198562622, -0.21059376001358032, -1.323438286781311, 1.2807258367538452, -0.38495901226997375, -1.9838207960128784, -0.669797956943512, 0.953528881072998, 0.35423797369003296, 1.8958370685577393, 0.7996727228164673, 0.12725761532783508, 0.5737823843955994, 0.02782365493476391, 2.7890117168426514, -1.0625687837600708, 0.5262203216552734, 1.0983372926712036, 1.864631175994873, 0.6796178817749023, -0.18673421442508698, -0.17620433866977692, -0.451524555683136, -0.34672993421554565, 1.152069091796875, -1.8907493352890015, -0.20086315274238586, -0.25582024455070496, 0.18714381754398346, -0.16929522156715393, -0.43438929319381714, -0.6481812596321106, 0.7887609004974365, -1.1479519605636597, 0.9044115543365479, -0.14561805129051208, 0.37542611360549927, 0.13761721551418304, 0.6367407441139221, 0.759610652923584, -1.2888935804367065, 0.39204832911491394, -0.6305518746376038, 0.043187014758586884, -1.6806342601776123, 0.8383194804191589, 1.179890513420105, -0.7006257772445679, -0.8554018139839172, 0.16920751333236694, -1.9473834037780762, -0.39301058650016785, 1.6668859720230103, 0.766484797000885, 1.1047768592834473, 0.3200540244579315, 0.8356351852416992, -0.8727692365646362, -0.6165716052055359, 0.5882608294487, 1.524669885635376, 0.290678471326828, -1.0956382751464844, 1.2850724458694458, -0.40173110365867615, 0.32356399297714233, -0.7362921237945557, -0.1225091814994812, -0.19891780614852905, -1.1285585165023804, 0.18329887092113495, -1.0175241231918335, -0.522063672542572, 0.6450590491294861, 0.4893054962158203, -0.28619205951690674, -0.6365630030632019, -0.24342894554138184, -0.8205826282501221, -0.548628568649292, 0.03977815434336662, 0.6792519688606262, 0.20725999772548676, -0.8127212524414062, 0.4992172420024872, -0.02339005284011364, -0.5153632760047913, 0.2732943892478943, -0.6490875482559204, 0.2643324136734009, -0.02794123813509941, 0.23035570979118347, 1.361102819442749, 0.8853314518928528, -0.9303845167160034, 0.8719321489334106, 0.8926370143890381, -1.8742890357971191, -0.019859720021486282, 0.2347477823495865, 2.3677563667297363, -1.5852010250091553, -1.4729982614517212, -0.37434712052345276, 0.8303306698799133, 0.665464460849762, 0.10581642389297485, -1.0164464712142944, -0.7732000946998596], [-1.2319680452346802, -0.5011633038520813, -0.9986931681632996, 0.40331214666366577, -0.8877977728843689, 1.5151013135910034, 0.819187581539154, 0.39990106225013733, -0.3042687177658081, 1.6035007238388062, -0.18284857273101807, 0.24417002499103546, 0.7529416084289551, 0.053557008504867554, -0.4230797588825226, 0.33122044801712036, 0.0997297614812851, 1.4630886316299438, -0.8820030689239502, 1.6644939184188843, -0.553918182849884, -0.2081577181816101, 0.9417393803596497, -0.4578722417354584, -0.2255469113588333, 0.6834947466850281, 0.5672763586044312, -0.516403317451477, -0.915014922618866, -0.40961918234825134, 0.840954065322876, -1.3538075685501099, 0.6359532475471497, -0.988678514957428, -1.065046787261963, 1.528716802597046, -2.165496826171875, 0.024852579459547997, 0.5910462737083435, 0.4071759581565857, 0.8228639960289001, -1.78099524974823, 0.16889961063861847, 1.0698765516281128, -0.2819305658340454, 0.10347089916467667, -0.5030701756477356, -0.9256959557533264, -0.47204384207725525, 1.2462729215621948, 0.2799973785877228, -0.278199702501297, -0.542862057685852, 1.8578587770462036, 0.599433183670044, -0.30379053950309753, 1.3783302307128906, -2.273030996322632, 0.7736059427261353, 0.5202115178108215, -0.36281418800354004, 0.22627165913581848, -1.5757709741592407, -0.42554375529289246, 0.01801106333732605, -0.16271431744098663, -0.9878712892532349, -0.045874446630477905, 1.372492790222168, -0.9497580528259277, 0.08369023352861404, 1.0798063278198242, -0.805180013179779, 0.21247079968452454, 0.5396915674209595, 1.773816704750061, -0.10074535757303238, -0.14597627520561218, -0.18832021951675415, 0.4258759021759033, -0.7959497570991516, 0.2937675714492798, -1.0050493478775024, -2.3707542419433594, 0.28671586513519287, 0.9693140387535095, 1.1021591424942017, 0.4088354706764221, -0.9469950795173645, -0.3224947452545166, -2.336446523666382, -0.6406112313270569, 0.3216796815395355, -0.5988966822624207, 1.1362982988357544, -0.3078690767288208, -0.6445437073707581, 2.0369298458099365, -1.66745924949646, -0.6335304975509644, 1.706507682800293, -0.7288461327552795, -0.6354728937149048, 0.7386946678161621, 0.7241512537002563, -0.812760055065155, -0.3666994869709015, 1.4818822145462036, -0.3710783123970032, 0.8826034665107727, -0.17819072306156158, 1.949528455734253, 2.0116779804229736, 0.6607112884521484, -0.7536194920539856, 0.46709656715393066, 0.6454713940620422, 0.3013378977775574, -0.664910614490509, 0.33160021901130676, -0.7166945934295654, 0.5394099354743958, 0.06136492267251015, -0.3905806541442871, -0.6855801343917847, 0.722575306892395, -0.26958587765693665, -0.8665470480918884], [-1.5682514905929565, -0.23443445563316345, -1.8839802742004395, -0.17750674486160278, -0.3389938771724701, -1.288370966911316, 0.4835273325443268, 0.6399559378623962, 0.4686857759952545, -1.0426486730575562, -0.7486772537231445, -1.5630300045013428, 1.335618495941162, -1.9697688817977905, -1.443729043006897, 0.6166243553161621, -2.1496641635894775, -2.080655097961426, -1.0714900493621826, 0.6431719660758972, -0.0843890979886055, -1.7857009172439575, 0.08167336881160736, 0.4294424057006836, 0.018479537218809128, 1.1763617992401123, -1.2279361486434937, 0.551609456539154, 0.3718712329864502, 0.7049567699432373, 0.4004061222076416, 0.15578314661979675, 0.5982838273048401, 0.5815566778182983, 0.9395365118980408, 0.5957834124565125, -0.6378984451293945, -0.771572470664978, -1.8854163885116577, -0.6887620687484741, -1.277078628540039, 0.2196044623851776, -0.9051893949508667, 0.8489904999732971, -0.8377835154533386, -0.6868203282356262, 1.791207194328308, 0.8020040392875671, 0.6750782132148743, -1.0561782121658325, 0.6388795971870422, 1.1421523094177246, 0.014468695968389511, 0.49151289463043213, -0.7371454238891602, 0.43695658445358276, 2.9874260425567627, 0.9530885815620422, -0.9405568838119507, -0.7966809868812561, -0.33858686685562134, -0.4348042905330658, -1.3830997943878174, 0.9723997116088867, -2.2707464694976807, -0.9574527740478516, -0.4031847417354584, 1.075304627418518, -0.5818077325820923, -0.9734857678413391, 0.030875127762556076, 0.11751881241798401, -0.9552804231643677, 1.5869075059890747, 0.8682848215103149, 0.7635883688926697, -0.3992990255355835, 0.5921600461006165, 0.3143782913684845, -0.3733942210674286, -1.0824780464172363, -0.4019746780395508, -1.6348974704742432, -0.20633509755134583, -1.2548109292984009, -0.9275240898132324, 0.01153018418699503, -0.6020462512969971, -0.7428133487701416, 0.4471062123775482, -1.5689587593078613, -0.614703893661499, -0.8813499212265015, 1.0478591918945312, -0.7286162376403809, 0.6580093502998352, 1.1281111240386963, 0.5803523063659668, 0.7220855355262756, -0.6332791447639465, -1.215328574180603, 0.7070215940475464, -0.5796021223068237, 1.3991026878356934, -1.900403380393982, -1.0066914558410645, -0.26799505949020386, 0.03760617971420288, -0.15875861048698425, -1.006394863128662, 0.23290088772773743, 1.9523741006851196, 2.0822908878326416, 0.15969474613666534, 0.8198275566101074, 2.4096970558166504, 0.6765022873878479, 0.7438507676124573, -1.2459195852279663, 0.39148178696632385, -0.08583124727010727, 1.273015022277832, 0.791256308555603, -1.3192861080169678, -1.8798561096191406, 0.0956440269947052, 1.9403153657913208, -0.3441712260246277], [-0.629420280456543, -1.1361405849456787, 0.17663216590881348, 1.1012537479400635, -1.0711581707000732, -1.2408616542816162, -1.6846693754196167, 0.5608338117599487, 1.2667427062988281, 0.43015992641448975, 0.09598474204540253, -1.1345720291137695, -0.033486515283584595, -1.263847827911377, -0.914185643196106, 0.4352882504463196, -1.9751631021499634, -0.3461029529571533, 0.17363359034061432, -0.8043458461761475, 1.327532172203064, 0.03676725924015045, -0.5108823776245117, 0.5066511631011963, 2.3063807487487793, 1.362084150314331, 1.5573152303695679, -0.3077976405620575, 0.9744178652763367, -0.9914727210998535, 1.5023744106292725, 0.4685671031475067, 2.1865103244781494, 0.6982588171958923, 0.5789566040039062, 0.7040379643440247, 2.0199010372161865, -0.49814724922180176, -0.5316125750541687, -1.6640667915344238, 0.6094294190406799, 0.9258586168289185, -1.2008038759231567, 0.1577886939048767, 0.09283563494682312, 0.41294705867767334, 0.6217053532600403, -1.1378746032714844, -0.7289358973503113, -0.3441762328147888, -0.76749587059021, -2.3874003887176514, -0.4943317472934723, 1.41854989528656, -0.49887168407440186, 0.08075176924467087, -0.11943667382001877, 1.499179482460022, -0.0998331606388092, -0.8167039155960083, -0.5562276244163513, -0.34879711270332336, 0.3352568447589874, 0.4900796115398407, 0.465385377407074, 0.07137464731931686, -1.0038663148880005, -0.5620058178901672, -0.07322521507740021, -0.39028313755989075, -0.6287879347801208, 0.7610068917274475, 0.40418386459350586, -2.1359307765960693, -0.5793265104293823, 0.15685643255710602, 0.21809785068035126, 0.48713839054107666, 1.349861979484558, -1.9972485303878784, 0.5692694187164307, -2.0249075889587402, -0.21646621823310852, 0.6154884696006775, 0.6591922640800476, 0.3189486563205719, 0.4002273380756378, 1.0205037593841553, -0.22350077331066132, -0.5325731635093689, 0.19665220379829407, -0.4228173792362213, -0.5630363821983337, -1.0162070989608765, 0.81069415807724, 0.5284463763237, 1.7941371202468872, -3.1294069290161133, 0.15207834541797638, -1.5302338600158691, -1.409619927406311, 1.3670408725738525, -0.8607966899871826, -1.2764959335327148, -0.04247003421187401, -0.012094379402697086, 0.5311349034309387, -0.9446564316749573, 0.05893012881278992, 0.9131984710693359, -1.1914517879486084, -1.0725140571594238, -0.18462885916233063, -0.6960274577140808, -0.4390765130519867, -0.7618710994720459, -0.371811181306839, -1.394632339477539, -0.34568026661872864, 0.2202925831079483, 0.24295364320278168, 0.39634379744529724, -0.9505308866500854, -0.6172521710395813, -0.49485042691230774, 2.2018136978149414, -0.5807939767837524, 0.2823925316333771], [-0.22971637547016144, -1.2619444131851196, 0.5789401531219482, 1.7506986856460571, 0.9818741083145142, -0.4483156204223633, 0.10810483992099762, 1.5117863416671753, 0.0872206911444664, 0.11517304182052612, -1.683634638786316, 0.6630678772926331, 0.7950093150138855, -0.23181505501270294, 0.016046937555074692, 1.8377543687820435, 0.3881436586380005, 0.981226921081543, 0.5648940205574036, -0.39100924134254456, 0.36930859088897705, 0.46042031049728394, -0.7708202600479126, 0.5919193029403687, -0.48250409960746765, 1.7245817184448242, -1.042699933052063, 0.8968596458435059, 1.8834906816482544, 0.5529060363769531, 0.515598475933075, 0.8974131345748901, -1.076676845550537, -1.6145334243774414, -1.5064001083374023, 0.9068999290466309, 0.9254301190376282, -1.5630872249603271, -0.370819091796875, -0.6189379692077637, -0.6315321922302246, -1.4978857040405273, -0.015487290918827057, -0.3558805286884308, 0.7100165486335754, -0.16198313236236572, -0.9861911535263062, 1.4531242847442627, -0.07736034691333771, -0.5336530208587646, 1.4286000728607178, -0.5422702431678772, -1.68394935131073, -1.4726715087890625, -2.8890140056610107, -0.9729245901107788, 1.0252265930175781, -1.584572434425354, -1.452921986579895, -0.6601799726486206, 0.6018240451812744, -1.2112635374069214, -0.047069091349840164, 0.9889756441116333, 0.7946714758872986, 0.21539849042892456, -0.1570543646812439, 0.8746930360794067, 1.2407793998718262, -1.3812716007232666, -0.577777624130249, -0.5306621789932251, -0.4253462255001068, -1.9924989938735962, 1.850792407989502, -0.5695311427116394, -0.6228575110435486, 0.4389733672142029, 1.8235145807266235, -0.41833755373954773, -1.1756237745285034, -1.7706767320632935, 1.4317984580993652, 0.5829503536224365, -0.19433815777301788, -1.0034029483795166, -1.1807998418807983, -0.6437569856643677, 0.7024688720703125, -2.6227331161499023, -1.0261658430099487, -0.31329330801963806, -0.7768880724906921, -0.989013671875, -0.17992204427719116, 0.019238729029893875, -0.6400699019432068, -0.3927760422229767, -0.585793137550354, -1.430518388748169, -0.0012658071937039495, -0.5575740337371826, 0.7431819438934326, -1.1376816034317017, -0.970440685749054, -0.4790925979614258, 0.8446062207221985, -0.7293152809143066, 0.5402666926383972, -0.029501132667064667, -0.3681747019290924, 0.413370817899704, -0.7153274416923523, 0.00405444111675024, -1.712652325630188, 0.9497698545455933, 1.9036837816238403, -0.3424089550971985, 0.13181325793266296, 0.43258944153785706, 0.512749195098877, 0.15752573311328888, 2.606879472732544, -1.740877389907837, 0.13567259907722473, 1.7372945547103882, 0.12849275767803192, 1.2872624397277832], [-0.5582450032234192, -0.8971915245056152, 0.19236710667610168, 1.3576909303665161, 1.3602616786956787, -0.10921461135149002, -1.2427449226379395, -0.026466023176908493, 1.1375386714935303, 0.06444525718688965, 0.04929177835583687, -1.2585108280181885, -0.8705415725708008, -0.1696132868528366, 0.03175215423107147, -0.41885945200920105, 0.661470890045166, -0.16510236263275146, -0.11449632048606873, -0.24520182609558105, -0.4676111340522766, 2.0267605781555176, 1.2383674383163452, 0.13961274921894073, 0.09449475258588791, 0.9028087854385376, -0.6878843307495117, 1.2245880365371704, 0.08683968335390091, 0.7834835052490234, 0.6861312985420227, -0.2784681022167206, -0.7508904933929443, -0.987123966217041, 0.14003853499889374, -0.8685562610626221, -0.6860131621360779, -0.6321741342544556, 0.46548953652381897, -0.6019435524940491, 0.7841459512710571, 0.6456992626190186, 0.7175876498222351, 0.6031041741371155, 2.2495594024658203, 0.28264880180358887, -0.3702046871185303, -0.20994523167610168, -0.30616891384124756, -0.7529919743537903, 0.5176761746406555, 0.08498027920722961, 0.6028757691383362, 0.7232896089553833, -0.408399760723114, -0.06768220663070679, -1.7478258609771729, -1.0973286628723145, -1.5923737287521362, 0.34437716007232666, 1.9054725170135498, 0.8119161128997803, 0.07907354831695557, 0.7514013051986694, 0.36558911204338074, 1.5057809352874756, 1.4935394525527954, -2.0373330116271973, 0.2444431036710739, -1.0127695798873901, 1.7454185485839844, -1.5076566934585571, 1.384084939956665, 1.92647385597229, 0.39389604330062866, 0.30029866099357605, -0.13416002690792084, 0.8346138596534729, -0.3311014771461487, -0.34392276406288147, -1.4950417280197144, 1.4428632259368896, -0.9974954724311829, 0.9690550565719604, -0.7910650372505188, 1.400533676147461, -1.0393002033233643, -0.8309222459793091, 0.29181620478630066, 2.661216974258423, -1.6554286479949951, -0.8462566137313843, -0.587817370891571, -0.5827310085296631, -0.4069680869579315, -1.6596733331680298, 1.0826510190963745, -0.47426334023475647, -0.90449458360672, -0.8369935750961304, 0.10887155681848526, -0.18647883832454681, -0.1823042929172516, -2.3599653244018555, -0.39533156156539917, 0.8819923996925354, -0.9876863956451416, -0.39432770013809204, 1.9775636196136475, -1.584229588508606, 1.1478461027145386, 0.5632034540176392, -0.0038390501867979765, 0.1161252111196518, 0.3203166425228119, 0.37834224104881287, -0.8470014333724976, 0.23194363713264465, -0.5018287301063538, -0.3702811598777771, 0.17752689123153687, -0.7769520282745361, -0.22723941504955292, -0.7222941517829895, 2.015122890472412, -0.5741305351257324, -0.1757691651582718, 1.0465346574783325], [0.254565566778183, 0.28548237681388855, -0.7227015495300293, -2.581860303878784, -0.6662696003913879, -0.44650155305862427, 1.5566461086273193, -0.01049090176820755, -1.161368727684021, -1.6217246055603027, 0.24751627445220947, -1.20486581325531, 0.49091988801956177, 0.5088335275650024, 0.5037723183631897, 1.1489884853363037, 0.03746230900287628, 0.6686147451400757, 0.8301501870155334, 0.8893154859542847, 0.8615342974662781, 0.33116546273231506, 1.4816621541976929, -0.9752182960510254, 1.3114101886749268, 1.1206846237182617, 0.7696340680122375, -1.07391357421875, -1.491422176361084, -0.03697612136602402, 0.630610466003418, -0.2724563181400299, 0.04490389674901962, -1.4286948442459106, -0.964106559753418, 0.1392311006784439, -0.23454906046390533, 1.2510234117507935, -1.3036589622497559, -0.054779306054115295, -1.7799707651138306, -0.17074453830718994, -0.7411270141601562, -1.1800034046173096, -1.871574878692627, 0.3625597655773163, -0.3058573305606842, -1.3310154676437378, -0.24054279923439026, 1.8100214004516602, -0.7996423840522766, -1.0596978664398193, 0.02541615627706051, -1.2196731567382812, 0.28580108284950256, 0.10675576329231262, 1.6262801885604858, -0.96442049741745, 0.8637958765029907, -1.4062552452087402, 1.3273991346359253, -0.4686499536037445, 0.15153859555721283, 0.8979418873786926, 2.7035157680511475, -0.19201503694057465, -0.06136477366089821, -1.0486934185028076, 0.6601629257202148, 0.5637297630310059, 0.6823993921279907, -0.7721797227859497, -1.1985704898834229, -1.3210227489471436, -0.7288723587989807, 0.5924860239028931, -0.42749494314193726, 0.9891964793205261, 1.193375587463379, -1.0632380247116089, -0.17491938173770905, 0.3264205753803253, 0.5309898257255554, -0.1416396051645279, 0.504212498664856, -0.691218376159668, 0.00979992002248764, 0.7127690315246582, 1.0768824815750122, -2.737025499343872, 1.0777814388275146, 0.07067754119634628, 0.32788780331611633, 1.473061203956604, -0.1486678570508957, -1.1054719686508179, -0.5821313858032227, -0.23635223507881165, -0.6137195229530334, -0.27104392647743225, 0.5609451532363892, -1.5693366527557373, 0.4033616781234741, 1.775929570198059, 0.5754356980323792, 0.3754960000514984, -0.09421414881944656, -0.9303967356681824, -0.13178598880767822, -0.9804901480674744, 0.6580807566642761, 0.0770607516169548, 0.5564152002334595, 1.3717830181121826, -0.7247553467750549, -0.6758816242218018, -1.4581563472747803, -1.478597640991211, -1.6530739068984985, -0.46249914169311523, -0.652526319026947, 0.9344229102134705, 0.5880244970321655, 0.4557458460330963, 0.6893650889396667, -0.31157970428466797, 2.295269727706909, 1.172532320022583], [-0.7624676823616028, -1.306099772453308, 2.1047186851501465, 1.3856700658798218, 0.705527663230896, -0.11974696815013885, -0.41588324308395386, -2.3127288818359375, -1.1722285747528076, -2.1567530632019043, -0.48108217120170593, -0.10414370149374008, 0.15037992596626282, -0.9985541105270386, 1.456343173980713, -1.510533332824707, 0.25523048639297485, -0.4449324309825897, -0.6031901836395264, -0.4889718294143677, 0.7562559247016907, -1.1635069847106934, 1.4348472356796265, 1.1490720510482788, -0.49770891666412354, 0.41183939576148987, -0.21973754465579987, -0.772776186466217, 1.7269573211669922, -0.19379200041294098, -0.49609673023223877, -0.9595471024513245, -0.3701818585395813, 0.9804286956787109, -1.2609856128692627, 0.6162962317466736, -0.22377635538578033, 0.4358828067779541, 0.9912943840026855, 1.0085561275482178, 1.9046220779418945, 0.7192875146865845, -2.6099562644958496, 0.9480555057525635, 1.8170499801635742, 1.153024435043335, 1.4660539627075195, 0.027633991092443466, 0.40288791060447693, -1.4599547386169434, 0.31662851572036743, -1.181803584098816, 0.4398813545703888, -1.1278365850448608, -0.07750476151704788, -0.4508480429649353, 0.31151634454727173, 0.031623613089323044, 0.6823246479034424, 0.3876837491989136, 0.4057590663433075, 0.49023497104644775, -1.260013461112976, -1.4764108657836914, 0.4579014182090759, -0.7618382573127747, 0.3374722898006439, -0.2769048810005188, 1.2743219137191772, 0.9479695558547974, 0.015470394864678383, -0.42564767599105835, 2.163738250732422, -0.6202074289321899, -0.305890828371048, 0.14559398591518402, -0.4885862469673157, -0.16269339621067047, 0.04773600026965141, 1.5272349119186401, 0.04566950350999832, -0.2577281594276428, 0.2194252759218216, 0.7575132846832275, 0.37537524104118347, -0.7365542054176331, -0.985914945602417, -1.3094791173934937, -0.9012805819511414, -1.32639741897583, -1.5894930362701416, -0.004800537601113319, 0.12613114714622498, -0.07351849973201752, 2.1876540184020996, 1.5889185667037964, 1.0298599004745483, 0.9382508397102356, -0.1384928971529007, 0.42271295189857483, -0.7905836701393127, 0.5028563737869263, -0.25669699907302856, -1.4307715892791748, 0.9620907306671143, -0.23948922753334045, 0.6546138525009155, 1.2364767789840698, 2.7679872512817383, -1.0430736541748047, -0.35978806018829346, 1.227628469467163, -0.15074194967746735, 0.9083672165870667, -0.8745850324630737, 0.10549040138721466, 0.2329523265361786, -1.710247278213501, -0.28114426136016846, 0.42286136746406555, 0.596194326877594, 0.015983328223228455, -0.19813205301761627, 0.1288335621356964, 0.7358420491218567, 0.22115017473697662, 0.4968300759792328, 0.5930346846580505], [0.9931551218032837, -0.23966652154922485, 1.4596420526504517, 0.21562503278255463, 0.21465715765953064, 0.4460853338241577, 1.0320558547973633, -0.14263330399990082, 0.0061794305220246315, 0.6349649429321289, 0.09901322424411774, 0.8834913969039917, -1.0708667039871216, 1.3522875308990479, -0.9002640843391418, 0.976470947265625, 0.12464965134859085, 0.443160742521286, -1.2272545099258423, 0.865048885345459, 2.017758846282959, -0.29466018080711365, 1.6320217847824097, -0.6969448924064636, 0.9628888964653015, 0.44710662961006165, 0.9467028379440308, -0.13864991068840027, 1.5179482698440552, 0.3258805572986603, -0.44345882534980774, -0.42662397027015686, -0.7007322907447815, 0.508176863193512, 0.12022051960229874, 0.2951304316520691, -1.3397873640060425, 0.15358124673366547, -1.3519258499145508, 0.6489111185073853, -0.9256578683853149, -0.009222500026226044, -0.527958869934082, 0.4200361669063568, 0.8060449957847595, -0.6677191257476807, -0.8492127656936646, 0.23331402242183685, -1.3967052698135376, 0.7898445129394531, 1.9803401231765747, -0.36537492275238037, -0.676880955696106, -0.24116413295269012, -0.2515610158443451, -1.222174048423767, -2.274325132369995, 0.7110032439231873, 1.2067787647247314, -1.0907663106918335, 0.15838982164859772, -0.9958813190460205, -1.2468193769454956, -0.8687998652458191, -1.483119249343872, 0.4996839761734009, 0.3305114507675171, -2.051494836807251, -1.0408154726028442, -0.07309255748987198, -0.12167707830667496, -0.4397740364074707, 0.3068176209926605, 0.2955425977706909, -0.8163309693336487, -1.7296192646026611, 0.16207748651504517, 0.27284494042396545, -1.7837061882019043, -1.7260984182357788, 0.11864356696605682, -0.14735639095306396, -0.2851017415523529, -0.6251189708709717, 0.11647329479455948, 2.9735984802246094, 1.0673106908798218, -0.24124674499034882, -1.6136205196380615, -0.7469962239265442, 0.15173213183879852, -0.582009494304657, -0.15205107629299164, -0.30881816148757935, -0.05316873639822006, 0.1987241953611374, -0.1661321371793747, 1.30523681640625, -1.7976785898208618, -0.8013173341751099, 0.09653587639331818, -0.38362327218055725, -0.8284299969673157, -2.106724739074707, 0.1543947160243988, 0.9352371096611023, -0.022504892200231552, -1.3214049339294434, -0.26565906405448914, 1.1094027757644653, 0.3700750172138214, 0.5088443160057068, 0.7125608325004578, 0.2853335738182068, -1.1597108840942383, 1.2660831212997437, 0.14878231287002563, 0.2525630295276642, -0.46364736557006836, -2.743851661682129, 0.4654735028743744, 0.3546237051486969, 0.3487085998058319, 0.6214255690574646, 1.582499384880066, 0.005744138266891241, 0.5556614398956299, -1.5244020223617554], [0.1805991530418396, -0.8149423003196716, 1.0049357414245605, -0.04502468556165695, 0.9992277026176453, 1.16806960105896, -0.7114948034286499, -1.584756851196289, 1.4016002416610718, -0.7134919762611389, 0.2999863028526306, -0.7738378643989563, 0.47910386323928833, -1.3747243881225586, 0.11343264579772949, -0.9763340353965759, -0.10338646918535233, -0.7049481272697449, -0.43205270171165466, 0.3428793251514435, 1.1703799962997437, -0.4148500859737396, -0.4308404326438904, 0.19797003269195557, 0.43113040924072266, 0.6194371581077576, 0.4164009094238281, -1.5673162937164307, -0.01576930843293667, -1.0994888544082642, 1.2398321628570557, -0.40482449531555176, -0.37139207124710083, 0.7927281856536865, -1.225845456123352, 0.7249953746795654, 0.10068363696336746, -0.7909207344055176, -0.3586684763431549, -0.2912629246711731, -1.9605190753936768, -0.4223252534866333, -1.6473889350891113, 0.14749059081077576, 1.075021505355835, 1.1751117706298828, 0.6586413979530334, 3.526860475540161, -0.2734348177909851, 0.2749246656894684, 0.9823035597801208, 0.21411582827568054, -0.42414864897727966, 0.48965921998023987, 0.31206023693084717, -2.6725993156433105, 1.1900514364242554, 0.8874177932739258, 0.8373843431472778, 0.2840937674045563, 0.3159615695476532, 0.4281265437602997, 1.0578041076660156, -1.0543071031570435, 0.14594019949436188, 0.5259252786636353, -0.21384365856647491, 0.6854055523872375, -1.5681666135787964, -1.4291363954544067, -2.243225574493408, -1.433893084526062, -0.30021220445632935, -0.2620719075202942, 0.17727380990982056, -0.5328524112701416, -1.4642250537872314, 0.02874905988574028, 1.324684977531433, 0.4524163007736206, -1.4662336111068726, 1.512787938117981, 2.8724863529205322, 0.638555109500885, 1.6052109003067017, 0.3748854696750641, 1.0251537561416626, -0.8061088919639587, 0.4899884760379791, 1.6552671194076538, -0.10520277172327042, -0.22466489672660828, -0.37957900762557983, -0.005332229658961296, 0.5162854194641113, 1.1395384073257446, -1.2932549715042114, -0.48849114775657654, -1.542635440826416, 0.096250519156456, -0.16007332503795624, 0.31792235374450684, -0.7781190872192383, 0.41104456782341003, -0.8777158856391907, 0.17882327735424042, 0.24435663223266602, 0.8485129475593567, -1.866206169128418, -0.03968771547079086, -0.8267722129821777, -1.1041134595870972, 0.3960682451725006, -0.16460059583187103, -0.6322322487831116, -0.2962441146373749, -0.4320933222770691, -0.3913514316082001, -0.26098063588142395, 0.48339104652404785, 0.36483246088027954, -1.544442892074585, 0.6594083905220032, -0.9438731074333191, 0.5791110992431641, 1.303495168685913, 0.5434317588806152, 0.08552621304988861], [-0.9631761908531189, 0.08386354148387909, 0.7806074023246765, -0.5316099524497986, 0.44490063190460205, -0.9467031359672546, 0.754753053188324, -2.3715007305145264, 0.08709106594324112, -1.111370325088501, -1.9245105981826782, 0.06578237563371658, 1.0551578998565674, -0.44247308373451233, -0.5449503660202026, -0.3671843707561493, 0.8528934121131897, -1.1094331741333008, 0.4125667214393616, 0.31333619356155396, 1.757384181022644, -1.6552478075027466, 1.1177978515625, -0.08093857765197754, 0.1282871961593628, -0.9881163239479065, 0.8521293997764587, 1.346542477607727, 1.0489078760147095, -0.46461358666419983, -1.3530067205429077, 0.3671172857284546, 0.48944365978240967, 0.6906251907348633, 0.5079471468925476, 0.6250132918357849, 1.2667101621627808, 0.543376624584198, -0.6434288620948792, 0.824084460735321, -1.1014330387115479, -0.9598997831344604, 1.6787443161010742, 0.2533263862133026, 0.5937789082527161, -0.9539797902107239, -1.0536994934082031, -2.067347288131714, 0.21027804911136627, -0.6416088342666626, 0.907752513885498, -0.7093164920806885, -1.0931272506713867, 0.7127752900123596, 0.9172449111938477, 0.9903482794761658, -0.19678497314453125, 0.6866031885147095, -1.4803370237350464, -1.2620506286621094, -1.5039340257644653, 0.8366879224777222, 2.298309326171875, -1.0596708059310913, -2.1817314624786377, 1.0673279762268066, -0.7233244180679321, -1.2438018321990967, 0.7765849828720093, -0.8864914774894714, 0.9143152236938477, -0.09011181443929672, -1.0561012029647827, 1.609707236289978, 1.5523182153701782, -0.07671166211366653, 1.2033556699752808, 0.6838546395301819, 0.8757522702217102, -0.35062411427497864, 0.0644431933760643, -0.707743227481842, 1.8232287168502808, -1.488600254058838, -0.39719343185424805, -0.554069459438324, 1.5113060474395752, -1.250612497329712, 0.024647507816553116, 0.8164135813713074, 0.828656017780304, 0.5633889436721802, -0.8399843573570251, -2.066348075866699, -2.223724842071533, 0.9392911791801453, 1.3519443273544312, -1.68426513671875, 0.05590550974011421, 0.04394833371043205, -0.46506187319755554, -0.4525964558124542, -0.8807416558265686, 1.2823596000671387, -0.7859328985214233, 1.1620688438415527, -0.2229638397693634, -0.3850545883178711, -0.21989843249320984, -1.404145359992981, 0.8423001766204834, -0.895893931388855, 0.018290257081389427, 0.31651201844215393, 1.1706265211105347, -1.2401728630065918, -0.6163752675056458, -0.8986840844154358, 1.2608195543289185, 0.7508415579795837, -1.9807379245758057, -0.6231194138526917, -0.1368471086025238, 1.6870877742767334, -0.2176327407360077, 0.5531127452850342, -1.9995594024658203, 0.7623956203460693], [0.7876014113426208, 0.4098210632801056, 1.827574372291565, 0.8441373109817505, 0.44861146807670593, 2.0359785556793213, 0.5148507952690125, 0.7551246881484985, 0.9435137510299683, -0.36052003502845764, 0.2732522785663605, 0.4705187976360321, -0.9784670472145081, -0.14804622530937195, 0.21010765433311462, 0.01766970194876194, 0.631838858127594, -1.3182376623153687, 0.2614581882953644, -0.4882684648036957, 2.2814598083496094, -0.12250874936580658, 0.9331497550010681, -1.6273010969161987, -1.0089550018310547, 0.7726176381111145, 1.7904382944107056, 0.0680558830499649, 1.2754240036010742, -1.2921955585479736, -0.5595566630363464, -1.4734070301055908, -0.09881425648927689, -1.4879480600357056, 0.5918495655059814, -1.8903108835220337, -0.6463379859924316, 1.4523905515670776, -1.685312032699585, 0.9615461230278015, 1.8867161273956299, -1.1176613569259644, -0.0072331661358475685, 1.269345760345459, 0.49218177795410156, -0.6920385360717773, 0.39622625708580017, -2.547936201095581, 0.046943120658397675, -0.5985973477363586, 0.32870614528656006, -0.17957356572151184, -0.9508886933326721, -1.2061835527420044, 0.35109519958496094, -1.87636399269104, -0.33064767718315125, -0.11658228188753128, 0.131005197763443, -1.3424270153045654, 1.9344828128814697, -0.5102603435516357, -1.069591760635376, 0.8900083303451538, 0.05457259342074394, -0.13303767144680023, 0.04282072186470032, -0.7044004201889038, -0.7818335890769958, -1.3040704727172852, -0.8080421090126038, 0.13228332996368408, 0.5213266611099243, 1.4057559967041016, -0.38830798864364624, 0.5068698525428772, -0.5350320339202881, 0.8232542276382446, -0.20498760044574738, 0.6424475312232971, 1.0084213018417358, -0.5093315243721008, 0.412630170583725, 1.2265021800994873, 1.839425802230835, 1.1325128078460693, -1.1613577604293823, -0.8007517457008362, 0.05502729490399361, 0.7202135324478149, 1.077092170715332, 0.885096549987793, -0.02478465624153614, 0.1585812121629715, -1.0143380165100098, 0.8312819600105286, -0.31292012333869934, -0.9409881830215454, 0.2677406072616577, 0.3612228035926819, -1.8325111865997314, -0.1504751443862915, -1.3678719997406006, -0.1895570158958435, -0.37270355224609375, 1.1026265621185303, -2.2995243072509766, 0.3839074969291687, 1.9461171627044678, -0.19756881892681122, 1.9025604724884033, 0.6774458289146423, -0.604780375957489, -2.2311813831329346, 1.5755434036254883, -1.2494133710861206, 0.2014576941728592, 0.06921765953302383, 0.5692740082740784, 0.6049584746360779, -0.3409581482410431, -0.5028918981552124, -0.6119521260261536, -0.6067598462104797, -1.5738322734832764, 0.9407591223716736, 0.831997275352478, 1.0682578086853027], [-1.8130489587783813, -0.7532697319984436, -0.21511562168598175, -0.11008195579051971, -0.11370845139026642, 0.15523944795131683, 0.4971175193786621, -0.7151369452476501, -0.7242702841758728, 0.13731233775615692, -1.860802173614502, -1.4061514139175415, 1.2674994468688965, -0.8662703633308411, -0.4383070468902588, 1.0935038328170776, 0.010931217111647129, 0.6385025978088379, 0.930118978023529, -0.11314540356397629, -0.42198997735977173, 1.7188633680343628, -0.29014572501182556, 1.364240050315857, -1.230212926864624, -1.3900829553604126, -0.5597432255744934, -0.2840706706047058, 0.42361652851104736, -0.602273166179657, 0.6483479142189026, -0.7548866868019104, -0.9863840341567993, 0.35147908329963684, -0.5070585608482361, -0.38110587000846863, 0.26808786392211914, -0.2083304226398468, 2.5759012699127197, -0.9814900755882263, -0.48672428727149963, 1.2980128526687622, -0.7247599363327026, 0.025782456621527672, 1.0676864385604858, -0.8304023146629333, -0.11482904851436615, 0.9303866028785706, -0.6879487633705139, -0.657431423664093, 0.07004263997077942, 0.4944739043712616, -0.12328511476516724, -0.023384980857372284, 0.032976023852825165, -1.167457938194275, 0.5796491503715515, 0.6077963709831238, -1.3726742267608643, 0.05925622954964638, 1.0800968408584595, 0.6713741421699524, 0.06195162236690521, -0.4979918599128723, 0.8162909746170044, 0.19908766448497772, -0.5558722615242004, -1.561557412147522, -0.9016515612602234, -0.6465153098106384, 0.5807545185089111, -0.26208972930908203, 0.1753215789794922, -0.8566918969154358, -1.6255946159362793, -0.43880775570869446, 1.3637250661849976, -0.04126450791954994, 0.3405710756778717, 1.3459224700927734, -0.08162490278482437, -3.2900853157043457, 0.6361278891563416, 0.9739153981208801, -0.017481151968240738, 0.9742453098297119, 0.2251734733581543, -1.3177423477172852, 0.6341832876205444, -0.8595354557037354, -0.4565245807170868, 0.9535468220710754, -0.885907769203186, 0.6748508810997009, 1.4691020250320435, -0.732846200466156, -0.17519409954547882, -2.0666372776031494, -0.6553443074226379, 0.37865230441093445, 0.8002825379371643, -1.5898488759994507, -1.0084645748138428, -0.8107156157493591, -1.4471964836120605, 0.6473281979560852, -0.6504616737365723, -0.0852067843079567, 1.3340128660202026, 0.5384382605552673, 0.0654616504907608, -0.6634412407875061, -0.2703835666179657, -0.37372392416000366, 0.17818687856197357, -0.032572079449892044, 0.06486538052558899, 1.8429256677627563, 0.47710639238357544, -0.48076510429382324, -1.0076267719268799, 1.0840611457824707, 1.9560240507125854, -0.838348388671875, -0.7555561065673828, 1.9004019498825073, 1.588626742362976, -1.7507784366607666], [1.0446152687072754, 1.1987433433532715, -0.4306342899799347, 1.2760565280914307, 0.07866129279136658, 0.102293461561203, -0.5317283868789673, 0.054063525050878525, -0.203574076294899, 1.3382959365844727, -0.4848160743713379, 1.6673164367675781, 2.280555486679077, 0.47437769174575806, 1.5680930614471436, 0.41467222571372986, 1.0072866678237915, 0.5836345553398132, -2.2747864723205566, -1.5276333093643188, 0.30190566182136536, -1.0143080949783325, -1.4191235303878784, -0.6684384942054749, 0.4094897508621216, 0.7728801965713501, -0.6425119042396545, 0.8931166529655457, -0.46168968081474304, -0.36000877618789673, 2.063556432723999, -1.9167189598083496, 0.5910840034484863, -1.2692945003509521, 0.07974506169557571, -1.977162480354309, 0.4094085395336151, 1.442637324333191, -2.698450803756714, -0.3974297344684601, 0.9371092915534973, 1.4043797254562378, 0.5783794522285461, -0.9891344308853149, -1.1159276962280273, 0.29386305809020996, 1.50831139087677, 0.49030256271362305, 0.823084831237793, -0.24388501048088074, 1.7960429191589355, -1.3754549026489258, 1.7035541534423828, 1.6115435361862183, -1.8949968814849854, -0.39422041177749634, -0.8474051356315613, 1.3864537477493286, -0.9473704099655151, 0.4425317049026489, 1.7190485000610352, 0.3385635018348694, 1.1838276386260986, 1.2091797590255737, -0.2826399803161621, -1.3224856853485107, 0.3854495584964752, -0.05653541162610054, -0.01798444427549839, -0.14246322214603424, -0.2707922160625458, -0.13977330923080444, 0.9142537713050842, -0.5596529841423035, 0.5407794117927551, -0.8906012177467346, -0.1352119892835617, 0.23921945691108704, 0.881168007850647, -1.816694736480713, 0.5582927465438843, -1.9784718751907349, -1.6067118644714355, 0.7178635001182556, 2.2555463314056396, 2.218332052230835, -0.3780832886695862, -0.939447283744812, 1.4051334857940674, -1.7604572772979736, 0.5528884530067444, -0.8132126331329346, 0.1693354994058609, -0.9193041920661926, 0.45930221676826477, 0.39076218008995056, -0.3015381395816803, -0.809612512588501, 0.043456945568323135, 0.7389119863510132, 0.04834522306919098, 1.8646866083145142, 2.4908041954040527, 1.2177023887634277, -1.3491896390914917, 0.4795784652233124, 1.3276567459106445, 1.4211993217468262, -0.06547534465789795, -1.1929354667663574, 1.5319666862487793, 1.7302483320236206, -0.01640857197344303, 0.6637023091316223, -0.15931788086891174, -0.27636590600013733, 0.7863408327102661, 0.5746524930000305, 0.42117127776145935, -0.32417258620262146, 0.8538606762886047, -0.2958614230155945, -0.6982446312904358, 0.19456450641155243, 0.3426881432533264, -0.05301786586642265, -1.0709242820739746, -0.3645651936531067], [-0.44975319504737854, -2.087319850921631, -1.4696805477142334, -0.036994386464357376, 0.12575753033161163, -0.8828648924827576, -0.049987878650426865, 0.4651399254798889, 2.090240240097046, 0.20803596079349518, -1.1453049182891846, 0.0023596002720296383, 0.43204525113105774, 1.2530267238616943, -0.6865115165710449, 0.9161192774772644, -0.34299933910369873, -1.1807003021240234, -0.5740821361541748, 1.4145407676696777, 1.208114743232727, 1.1664952039718628, 0.5393140316009521, 0.26784998178482056, -0.05313286930322647, -0.31352731585502625, 0.1885557323694229, -0.09961703419685364, 0.6263260841369629, 0.22558358311653137, 0.7111029624938965, -0.2851516306400299, 0.2142127901315689, 0.7524172067642212, 0.8027958273887634, -0.35940882563591003, -0.13920021057128906, -0.3444863557815552, -1.0679593086242676, -0.4500674605369568, -1.3813248872756958, 0.9859441518783569, 1.4119004011154175, 0.339552104473114, 1.3631658554077148, -1.3900822401046753, -1.9094758033752441, 1.3008017539978027, 1.6382368803024292, 1.6216256618499756, 0.24058084189891815, -1.808547019958496, -0.6474630832672119, 2.0961525440216064, 0.4936557114124298, -0.4130082130432129, 1.1467092037200928, -0.27721041440963745, -1.129460334777832, -0.4047236442565918, -1.206182599067688, 0.49513307213783264, 0.25328922271728516, -0.19080296158790588, 1.8701027631759644, -0.8752073645591736, 0.8485209941864014, 0.4862962067127228, -0.46267858147621155, 1.7087361812591553, -0.3176760673522949, 0.2002212405204773, -1.2306153774261475, 0.2786504626274109, -0.8799369931221008, -0.06162628158926964, 0.13966579735279083, -2.442593812942505, 2.1060705184936523, 0.631836473941803, 0.5600054264068604, 0.5475159883499146, 0.3092030882835388, -1.2232341766357422, 0.8495033979415894, 1.177944302558899, -0.02936091646552086, 0.06720343232154846, -1.5311472415924072, -0.7041718363761902, 0.5003190040588379, -0.3533058762550354, -0.17242081463336945, 0.15470139682292938, 0.002047755755484104, -0.9930675029754639, 0.6501007676124573, 0.6692903637886047, -0.0643077865242958, -0.729266881942749, 0.773637056350708, -0.2919090688228607, -0.6416095495223999, -0.4070096015930176, -0.7200239896774292, -0.9456514120101929, 2.0131404399871826, -0.5781165957450867, 0.12537354230880737, 1.1164686679840088, 1.4815946817398071, -0.48171770572662354, -1.956799030303955, -0.6960263252258301, -0.9057596921920776, 0.9835545420646667, -2.2393527030944824, 1.1202725172042847, -1.5211552381515503, 1.2233006954193115, -0.8283947706222534, 1.1649367809295654, 0.6485283374786377, -1.9583638906478882, 0.5640899538993835, 0.6995895504951477, 0.37328824400901794, -0.24470236897468567], [0.7349538803100586, -1.3633804321289062, 0.3970055878162384, 0.009052041918039322, 0.689110279083252, 0.4125012457370758, 0.3136538565158844, 0.18474438786506653, -1.6955219507217407, -1.1910991668701172, -1.024391531944275, 1.0916316509246826, -0.17944923043251038, -0.9408652186393738, 0.543321967124939, 0.5114859342575073, -0.29728227853775024, -0.5778466463088989, -0.0667450800538063, -0.6320835947990417, 0.17630691826343536, -0.5587319135665894, -0.7390833497047424, 1.2685328722000122, -0.1481812745332718, -0.18187573552131653, -0.2468760758638382, 1.9308420419692993, -0.26873326301574707, 1.1274127960205078, 0.6023752093315125, 0.48222002387046814, 0.20825402438640594, 0.8769439458847046, -0.2795673608779907, 1.6414155960083008, 1.9367233514785767, 1.3690392971038818, 0.10823118686676025, -0.7829055786132812, -2.159816265106201, 1.9587785005569458, -0.34782305359840393, -0.9387139678001404, -1.013771414756775, 0.1641925722360611, -4.382387638092041, 0.5573262572288513, 0.1732538938522339, 0.11900615692138672, -1.4802457094192505, 1.5484404563903809, 0.02168204076588154, 0.14916737377643585, 0.5194555521011353, 1.9311000108718872, 0.8781182765960693, -1.5377806425094604, 0.2516620457172394, -0.4669591188430786, 0.8296661376953125, 0.6997907161712646, 0.7210355401039124, -1.418851613998413, 0.23403091728687286, 1.130807638168335, -1.114972472190857, -0.7332242131233215, 1.9326657056808472, -0.05330643802881241, -0.31435590982437134, -0.5856471657752991, 0.7977575659751892, 0.6980481147766113, -1.3471782207489014, -0.37078016996383667, 0.029560578987002373, 2.9507551193237305, -0.3815613090991974, 1.0931540727615356, 0.9122863411903381, 2.2172927856445312, 0.019466815516352654, 0.8697165250778198, -0.4415578246116638, -1.4072803258895874, -0.3775152564048767, -2.0277347564697266, 0.1465265154838562, 0.5915288329124451, -0.09729540348052979, -1.0487452745437622, 0.4345555007457733, -0.48810431361198425, 1.515658974647522, -0.6183377504348755, -0.07237055897712708, -0.4910281300544739, 0.1345936357975006, 1.4879610538482666, 0.7952346205711365, 0.005136968567967415, -0.6851027011871338, -0.8004783391952515, -0.4697955548763275, 0.07157615572214127, -0.760430097579956, 0.36930224299430847, -0.2601935565471649, -0.7450746893882751, 2.309535503387451, 1.8763858079910278, 0.3552324175834656, -0.8008949756622314, -0.7120702266693115, -0.8525498509407043, -0.2899819314479828, -0.4020546078681946, -1.0161898136138916, 1.4321370124816895, 1.5066088438034058, 0.4333072900772095, -1.4718122482299805, 0.333555668592453, -0.3521045744419098, -1.3589707612991333, -1.3952600955963135, -0.24540828168392181], [0.9991796612739563, -0.06903046369552612, 0.9813942909240723, 0.3689226508140564, -0.06870271265506744, -0.7745459675788879, 0.7141112089157104, 1.4632861614227295, 2.785261392593384, 0.35919833183288574, -0.326849102973938, -0.4330519437789917, 1.7202410697937012, 0.4117293655872345, 0.4639982581138611, -0.04830217733979225, -0.027423765510320663, 1.457888126373291, -1.8730323314666748, 0.17210175096988678, 0.00836389884352684, -0.4510977268218994, 1.3078538179397583, 1.8660513162612915, 0.5605189800262451, -0.6563670039176941, -1.2508848905563354, 1.061715841293335, -0.18853625655174255, -0.9872311353683472, -2.0258901119232178, -0.3089141845703125, -0.9563030004501343, -0.30119559168815613, -0.37964650988578796, 0.3179834187030792, -0.18648646771907806, 1.9508471488952637, -1.2498250007629395, -1.4651418924331665, 0.9727944731712341, -0.37928882241249084, -1.1101943254470825, -0.6674862504005432, 1.0483481884002686, 2.7888214588165283, -0.007652444764971733, 0.036007169634103775, -1.9413080215454102, 0.6247364282608032, -0.2377735823392868, -2.141397476196289, 0.6268752813339233, -0.3406834006309509, -1.1321992874145508, -0.184251606464386, 1.066544771194458, 1.5304323434829712, -0.9459478855133057, 0.1637207418680191, 0.8892076015472412, -0.28963160514831543, 0.7210166454315186, 0.1791122853755951, 0.030004462227225304, 0.5248413681983948, -1.2893896102905273, -0.1597513109445572, -1.0047022104263306, -0.09230861812829971, 1.6307727098464966, -1.2380456924438477, 0.6428968906402588, -0.3863022029399872, -0.6021042466163635, 0.015179705806076527, 0.27925947308540344, -0.6628772616386414, -0.4102543890476227, -0.7914603352546692, -0.08537863940000534, -0.579018235206604, -0.5236241817474365, 0.24176988005638123, -1.4391118288040161, -1.2862299680709839, -1.7411390542984009, -1.07935631275177, 0.9447683095932007, -0.6845271587371826, 1.7846425771713257, -0.8325803279876709, -0.19848230481147766, 0.2639593780040741, 0.9196956157684326, -0.7944180369377136, 0.9371254444122314, 0.24278827011585236, 0.3659883141517639, 0.7758619785308838, 1.0477190017700195, -0.9015384316444397, -1.300222396850586, -0.6693583130836487, 1.2157530784606934, -2.111964464187622, 1.0117697715759277, -1.0671581029891968, -0.24533389508724213, -0.8781354427337646, -0.698695719242096, -0.7565325498580933, -1.0426852703094482, -0.514737606048584, 0.6442345380783081, 0.9106840491294861, -0.8036614656448364, -0.03859950602054596, -0.0771193578839302, 0.8240886926651001, 0.7065809369087219, 1.8493168354034424, 0.15706340968608856, -0.3465271592140198, -0.34726476669311523, 0.13216626644134521, -0.31737732887268066, 1.685070276260376], [-0.21478012204170227, 0.5896304249763489, 1.3553670644760132, 0.7982346415519714, 0.37822362780570984, -0.1336037665605545, 0.7294740080833435, -1.2612923383712769, -3.323054313659668, -0.9598535299301147, -0.8487690091133118, -0.12854880094528198, -0.43047431111335754, 1.4169870615005493, 0.6184359192848206, -0.9661453366279602, -1.0760990381240845, -0.8590399622917175, 1.415751576423645, 0.31366389989852905, -0.516751229763031, -0.7251625657081604, 0.6968727707862854, 0.35166651010513306, 1.4820754528045654, 0.7265753746032715, 2.0033090114593506, 1.611768126487732, -0.5956417918205261, 0.3263351619243622, 0.19838757812976837, -0.31169062852859497, 0.019006118178367615, -1.1672966480255127, 1.6983643770217896, 0.7334259748458862, -0.49611833691596985, -1.631351113319397, 1.0538827180862427, -0.09134738147258759, 1.7434494495391846, 1.5455543994903564, -0.17032726109027863, -0.853819727897644, -0.6614672541618347, 0.258914053440094, -0.4576897919178009, 0.7292094826698303, -0.20250442624092102, 0.6811067461967468, -1.077890396118164, -0.12477359175682068, -0.5956704020500183, -1.495535135269165, 1.3539634943008423, -0.4102095067501068, -1.3017768859863281, -0.21899281442165375, -0.7479321360588074, 0.7522959113121033, 0.5406613945960999, -0.6588290333747864, 1.3430988788604736, 0.44890928268432617, 1.8072688579559326, -0.06967703253030777, 0.8137949705123901, 1.1953257322311401, 0.46439802646636963, 1.4119021892547607, -1.1788060665130615, 1.5116711854934692, -0.7670356631278992, 0.502155065536499, -0.7566239833831787, -0.43261396884918213, 0.2349621057510376, 0.679611325263977, 1.8521267175674438, 0.8310896158218384, -0.5095139145851135, 0.1817086786031723, -0.1831567883491516, -0.2593778371810913, -1.5433950424194336, 1.6090896129608154, 0.10931280255317688, 2.609766721725464, 0.7359176874160767, 0.3654105067253113, -0.5005168318748474, 0.5385298728942871, 0.9588127136230469, -1.20775306224823, -0.1273561418056488, 0.670247495174408, -0.24351298809051514, -0.7034509181976318, 0.21518461406230927, 0.9627794623374939, -1.383854627609253, 0.619185745716095, 2.6106834411621094, 1.6205893754959106, 0.08013160526752472, -1.7474305629730225, -0.17459633946418762, -0.34572476148605347, 0.11425083875656128, 0.7014874815940857, -0.6939305663108826, -0.0069562154822051525, -1.306900978088379, -1.426390528678894, -0.8509894609451294, 2.1591923236846924, 0.459403395652771, 1.1579123735427856, -0.8030810356140137, 0.1373043954372406, -2.4452385902404785, 0.07192962616682053, 1.0883597135543823, -2.0091195106506348, 0.8838411569595337, -0.16174842417240143, -0.3438524007797241, 0.6818291544914246], [1.2530165910720825, 0.7297548651695251, -1.31944739818573, -0.08632657676935196, -2.228541374206543, 0.4020591974258423, -0.24284586310386658, -0.2733806371688843, -0.7281299829483032, -0.8238123059272766, -0.07490862160921097, 0.7206299901008606, 0.43615782260894775, -0.20809629559516907, -0.25124990940093994, -0.49690765142440796, 1.4496296644210815, 1.3672573566436768, -0.5135579109191895, 1.9895713329315186, 1.2133251428604126, 0.2981802821159363, -0.3301399052143097, 0.2562907934188843, 0.385624498128891, -0.7824780941009521, -0.8159643411636353, 1.5164388418197632, -0.6379663944244385, 0.2719319760799408, 0.3557199537754059, 0.788632869720459, -0.5055037140846252, 0.3249605596065521, 0.10076744854450226, -0.5376793742179871, -1.3767951726913452, 0.26343274116516113, -2.296665906906128, 0.3948051333427429, 0.16453193128108978, -0.39388588070869446, 2.153393507003784, 0.6086313128471375, 0.8671054840087891, 0.9417647123336792, 0.5012568831443787, -0.6564873456954956, -0.4613085985183716, 0.46870964765548706, 0.44876939058303833, -0.6992343068122864, 0.8656768202781677, 1.1761058568954468, -1.1274592876434326, -0.4847016930580139, 2.272697925567627, -0.7454976439476013, 0.16774308681488037, 0.10521438717842102, 0.027992187067866325, 1.7128983736038208, -0.2211577445268631, 0.8241661787033081, -1.5606955289840698, 0.6130976676940918, 0.5035315752029419, -1.6902672052383423, 1.3810924291610718, -0.8578317761421204, -0.8030008673667908, 0.41628822684288025, 0.08886358886957169, 0.0475354939699173, -0.5354928374290466, -0.04829986393451691, 0.731130063533783, -0.2505607306957245, -0.02220093086361885, 1.0817441940307617, -1.294183373451233, -0.7334455251693726, 0.04286579415202141, -0.30840831995010376, -0.5763421058654785, -0.5799010396003723, -0.7324506044387817, 0.004847686272114515, -0.8615010380744934, -2.2533397674560547, -0.2606772184371948, 0.09632875770330429, 1.225852131843567, 1.7361105680465698, 0.0461171418428421, -0.09669627249240875, 1.4513285160064697, 0.3942663073539734, 1.2235125303268433, -1.342736005783081, 0.5992283225059509, -0.5933912992477417, 0.2107168734073639, 0.1891425997018814, 0.7349480390548706, -0.8141602277755737, 0.1454663872718811, -1.7628962993621826, 1.746970534324646, 0.12883085012435913, 0.12276049703359604, -0.1759594976902008, -0.9004366397857666, 0.7425017952919006, 1.4663382768630981, -0.66950923204422, -2.4018120765686035, 0.14120277762413025, 1.1223629713058472, 3.7508575916290283, 0.3156338930130005, 1.2651057243347168, 2.271759271621704, -0.1838434636592865, 0.6778661012649536, -1.1069204807281494, -0.5534077882766724, -0.2979142665863037], [-0.2713527977466583, -1.1794557571411133, 2.8996505737304688, 0.2493610978126526, 0.846952497959137, 0.7200072407722473, -0.38444480299949646, -0.22828082740306854, 0.19346804916858673, 0.06962442398071289, -0.9141241312026978, -2.2298049926757812, -0.8633300065994263, -0.9217420816421509, 1.4248664379119873, 2.062892436981201, 0.8607137799263, -1.104779839515686, 0.5811346173286438, -0.42649397253990173, -0.8594467639923096, -0.19291524589061737, 2.199845790863037, 0.41950416564941406, 0.3325979709625244, 0.5197526216506958, 0.4723578095436096, -1.1446102857589722, -1.3863359689712524, 1.2878143787384033, -0.5639610886573792, 2.740786075592041, 1.0994893312454224, -0.15126244723796844, -0.08291146159172058, 1.2786407470703125, -0.14851857721805573, -2.420602321624756, -0.5354164838790894, -0.5285741090774536, -0.012111084535717964, -0.12379293888807297, -0.48666462302207947, 0.05554676800966263, 0.07532770931720734, -0.4902442693710327, 1.0479865074157715, -1.6103672981262207, -0.06994170695543289, -0.997214674949646, -0.7603632807731628, 0.6441243886947632, 0.3758850693702698, -0.47760918736457825, 1.2422841787338257, 1.7036911249160767, 1.06596839427948, -0.518952488899231, -0.9869217872619629, -0.7362849712371826, -1.1489481925964355, -0.43136465549468994, -0.7364645004272461, 0.04049078747630119, 1.1797325611114502, -0.9224774837493896, -0.25415781140327454, 1.0331965684890747, -0.42117998003959656, 0.12713946402072906, -1.4870556592941284, -0.8828758001327515, 0.2586139738559723, -0.13458074629306793, -1.5761219263076782, 0.7441103458404541, -0.4194144904613495, 1.4432213306427002, 0.6181281208992004, 0.7131283283233643, 2.1970572471618652, -0.8369495868682861, -0.940693736076355, -0.01877065747976303, -0.44246163964271545, 0.6307921409606934, -0.026352299377322197, 1.270564079284668, 0.4683425724506378, 0.4662937521934509, 0.2551098167896271, 2.3152248859405518, -0.304375022649765, 2.2839598655700684, -0.7653570771217346, -0.838309645652771, -1.2150509357452393, 2.2769923210144043, 0.09957344084978104, 0.954594612121582, 0.7955763339996338, -0.525572657585144, -0.4729667603969574, -0.7108611464500427, 0.03737397864460945, -0.5163099765777588, 1.451669454574585, -0.2247937023639679, -1.497383713722229, 0.38793620467185974, 0.6413658857345581, -1.1597851514816284, 0.09181046485900879, 0.4353775680065155, 0.1385030597448349, 0.5284630060195923, -0.30165401101112366, -0.45657339692115784, 0.6626756191253662, 2.6402595043182373, 0.1083415076136589, -1.1992437839508057, 0.010354406200349331, 0.07165706902742386, -0.17496861517429352, 0.5081847906112671, -0.8277058601379395, 0.3804490566253662], [0.0903020054101944, -0.7745579481124878, 0.9237047433853149, 1.088895320892334, 0.21290482580661774, 1.517892599105835, -0.15573446452617645, -0.009513490833342075, -1.0893241167068481, 0.5195685625076294, -0.16674800217151642, 0.5594092607498169, -0.11508803814649582, -0.19511447846889496, 0.24799087643623352, 0.21280904114246368, 1.4859328269958496, 0.2253817617893219, 0.598173201084137, -0.009238161146640778, -0.10709074884653091, 0.20506559312343597, 0.31000858545303345, -0.41473910212516785, 1.3420263528823853, -0.25213515758514404, 0.8088359236717224, -0.5415826439857483, -0.5440188050270081, 0.5739822387695312, -0.07627162337303162, -0.8663800954818726, -0.5487571358680725, -1.4497960805892944, -0.2557539641857147, 1.2319018840789795, 3.215435266494751, -0.498248815536499, -0.3660905361175537, 0.7403830885887146, -1.0171658992767334, 0.31787553429603577, -1.0489767789840698, 1.4511569738388062, -0.20069068670272827, 1.0086013078689575, -0.04623350873589516, -1.6803388595581055, -0.8760236501693726, -1.068858027458191, -1.061911702156067, -1.338597297668457, 1.1791887283325195, 0.2610420882701874, -0.034317295998334885, 0.9705420136451721, 1.2318884134292603, -0.559536337852478, 0.9766148924827576, 0.7126208543777466, -0.007647916208952665, -1.1699039936065674, 0.05418603494763374, 0.694403886795044, 0.29229775071144104, 1.5493422746658325, -0.9763754606246948, -0.29665857553482056, 0.08946521580219269, 0.29279279708862305, -0.09891591966152191, -0.9484776258468628, -0.9505126476287842, -0.5401000380516052, 0.42658036947250366, 1.4393880367279053, -1.4864270687103271, -1.3082515001296997, 0.06773564964532852, 2.359117031097412, -0.6516420245170593, 1.9332773685455322, 0.8111924529075623, 0.4300588369369507, -0.44349244236946106, -0.9123741388320923, 0.15831221640110016, -1.1545337438583374, 0.5763441920280457, -1.623558759689331, 1.3304111957550049, -1.9277746677398682, 0.5885696411132812, -0.6469408869743347, -0.6432281732559204, -0.681158185005188, 1.2921191453933716, 3.1206700801849365, -0.5821714401245117, -1.973427653312683, -1.1101727485656738, -0.2270786613225937, -0.4507387578487396, -1.2314841747283936, -1.5014164447784424, -0.729236364364624, -2.0454795360565186, -0.03437713906168938, -0.35734787583351135, 0.9640845060348511, 0.20551621913909912, 0.17397576570510864, -0.30947649478912354, -0.6557357907295227, -0.9503382444381714, 0.9142186045646667, 0.9919440746307373, -0.35370469093322754, -0.6279417276382446, 0.49512678384780884, -0.9273764491081238, 1.4463512897491455, 0.861723780632019, -0.15562400221824646, 1.2210137844085693, -1.3977411985397339, 0.4458158314228058, 0.5797070264816284], [1.4837769269943237, 0.2499249279499054, -1.2301805019378662, 1.2275680303573608, -1.1087658405303955, 0.3557143807411194, -0.9888405203819275, -0.3650883734226227, -2.61430287361145, 0.6472769975662231, 1.0981518030166626, -0.22708451747894287, -0.17651918530464172, -0.05899844318628311, -2.0106630325317383, 0.8960748314857483, -0.27010712027549744, 0.20796489715576172, 0.2939833104610443, 0.8087818622589111, 1.263837218284607, 0.15845251083374023, 0.4468599855899811, -0.6853539943695068, 0.5869438648223877, -1.4250376224517822, -1.1238313913345337, -0.7084149718284607, -0.4027126431465149, 0.1311168223619461, 1.3674211502075195, -0.8993816375732422, 1.6444988250732422, -1.5815701484680176, -1.6304774284362793, 1.3840781450271606, 2.473686695098877, -0.10707510262727737, 0.7548389434814453, 2.10113263130188, 1.2929681539535522, -0.12573546171188354, -1.3569469451904297, 0.9446715116500854, -0.5845617055892944, 2.1866331100463867, 0.49771761894226074, 0.010913213714957237, -0.40476828813552856, 0.9665819406509399, -0.6529648900032043, 0.370069682598114, -1.9486383199691772, -1.26583731174469, 0.14359498023986816, -0.5230740904808044, 0.5795847773551941, -2.4156734943389893, 0.9891582727432251, 1.3184940814971924, -1.1508119106292725, -1.2010743618011475, -1.4849507808685303, 0.32806670665740967, -0.3488410711288452, 0.17405001819133759, -0.27833324670791626, -0.07430259138345718, -0.945231556892395, -1.2697029113769531, -1.1259733438491821, -0.9546549916267395, 0.6819477081298828, -0.9584105610847473, 0.4615933895111084, -0.21361671388149261, -0.8210596442222595, -1.3059066534042358, 0.9371469020843506, -0.1518275886774063, 1.4274932146072388, 0.35409674048423767, 0.3554631769657135, -1.111609935760498, 1.3364535570144653, 0.028086282312870026, 0.92745441198349, -1.870558500289917, 0.5973371267318726, 0.189982607960701, -1.5118943452835083, -1.4394549131393433, -0.8942546844482422, 0.7949581146240234, 0.002134020207449794, 1.0902687311172485, 1.5958287715911865, 0.303305447101593, -2.208061695098877, 0.4101911783218384, -1.2774454355239868, -0.8362212777137756, -0.49406731128692627, 0.32270151376724243, 1.736135482788086, -0.5817633867263794, -0.21993836760520935, 0.9747670292854309, -0.2772938907146454, -0.5176421999931335, 0.06727635115385056, 0.6982216238975525, -1.510913610458374, 0.2718985974788666, 0.16138476133346558, 1.5445425510406494, 0.30190160870552063, 0.5101211667060852, -1.2251518964767456, -2.3743600845336914, 0.16222815215587616, 0.04248581454157829, -1.2243770360946655, -1.8912228345870972, 0.513338029384613, -0.009123210795223713, 0.56770920753479, 0.09096469730138779], [-1.0364960432052612, 0.52358478307724, 0.5492187142372131, -0.7298382520675659, -0.1027473732829094, 0.8169311881065369, 1.1321545839309692, -1.4161165952682495, -0.1606171429157257, -1.9965872764587402, -1.4992146492004395, -1.0114946365356445, -0.6810156106948853, -1.5721869468688965, -1.9550294876098633, 0.45706015825271606, -0.6242448687553406, 0.9971113204956055, 0.9094748497009277, -0.80164635181427, 1.111625075340271, 1.5617891550064087, 0.6773713827133179, 0.9720927476882935, -0.09888322651386261, 1.0938694477081299, -1.300485372543335, -0.7926932573318481, 0.9665700793266296, 0.3571647107601166, -1.3564627170562744, -0.12507300078868866, -0.46996065974235535, -1.2453358173370361, 0.9507103562355042, -0.5842574834823608, 0.5838879942893982, 1.2726914882659912, 0.8385705947875977, -2.808600425720215, -0.5308098196983337, 0.3004109561443329, 0.34637555480003357, -0.5074244141578674, -0.5917381644248962, -0.5593882203102112, 0.21665433049201965, 0.021809564903378487, 0.48408451676368713, -0.45251843333244324, 1.6092337369918823, -0.26787617802619934, -0.6613832116127014, -1.8340225219726562, 2.5920517444610596, -0.8749484419822693, -0.7065663933753967, 1.776267409324646, -0.39159122109413147, -0.581503689289093, -0.04756337031722069, -1.8243613243103027, -1.610580325126648, 1.7842882871627808, 0.5627720952033997, 0.5497815608978271, 1.6515626907348633, -1.3690334558486938, 0.17603003978729248, -0.051611755043268204, -0.33815503120422363, 0.20840726792812347, 1.517895221710205, 0.034618619829416275, 1.034735083580017, -0.9862583875656128, 1.7628707885742188, -0.2992229759693146, -1.5196688175201416, -1.0946165323257446, -1.0510172843933105, 0.025596998631954193, 1.2378703355789185, 0.6416824460029602, 0.6166762709617615, 0.1061728298664093, -2.4237890243530273, 0.6811916828155518, 0.24451351165771484, -0.912448525428772, 1.1011017560958862, 0.23509691655635834, 0.3180679380893707, 0.7324916124343872, 2.0166962146759033, 0.15499018132686615, 0.23013712465763092, -0.5783370137214661, 0.8785852789878845, -0.22501321136951447, -1.098861575126648, 1.883553147315979, 0.45064178109169006, 1.7129945755004883, -1.5604126453399658, 0.6132831573486328, 0.21828243136405945, -0.9578239321708679, 1.2207870483398438, 2.2069802284240723, -1.064028263092041, -0.0655733123421669, -0.9411952495574951, -0.3013077676296234, 0.40516987442970276, 0.49111589789390564, 0.24762946367263794, -1.1942687034606934, -0.7339929938316345, -0.4344700276851654, 0.05012919753789902, -0.7255603671073914, 0.18490561842918396, -0.19604238867759705, 1.500788688659668, -0.038661856204271317, 0.8706231117248535, 0.5942040085792542], [-0.7137963771820068, 0.6240535974502563, 1.1390265226364136, -0.1846192181110382, 0.40231069922447205, -0.1142972856760025, -2.2975664138793945, 2.0977752208709717, 0.9393872618675232, 1.3513506650924683, 0.3011378347873688, -1.2066922187805176, 1.4149495363235474, 0.5217241048812866, 0.23962172865867615, -0.6551946401596069, 0.6339879631996155, 0.08286204934120178, -0.6380648612976074, 1.33041512966156, -0.9326826333999634, -0.21336457133293152, 0.3928847014904022, -0.33787256479263306, -0.9536442160606384, -0.3238879442214966, 0.4929760992527008, 0.022409847006201744, 1.1913573741912842, -0.8893808722496033, -1.4036195278167725, -0.7701592445373535, 0.2579396665096283, -0.4901960790157318, -0.9332776069641113, 0.7371659278869629, 0.6709371209144592, 0.7211383581161499, 0.13381211459636688, 0.2137538492679596, 0.1993933767080307, -0.5763421058654785, -0.5483883619308472, -1.3356733322143555, -1.5456620454788208, 1.0409258604049683, 0.5551472306251526, -1.5456287860870361, 0.08324377983808517, 0.5797606706619263, -1.0278258323669434, -1.53426194190979, 0.34951087832450867, -0.20205025374889374, 0.36798909306526184, -1.4866427183151245, -0.9858075380325317, -0.3885856866836548, -0.7599990963935852, -0.09766508638858795, 0.5315786600112915, 0.043699294328689575, -2.977297306060791, 1.3418657779693604, -0.2843725085258484, -1.257507562637329, 1.9604731798171997, 1.7935221195220947, -2.023224115371704, 0.6006753444671631, 0.7797456979751587, 0.27610576152801514, -0.5546584725379944, 1.0973832607269287, -1.5942227840423584, 1.1447434425354004, 1.1740236282348633, -0.6672118306159973, 3.921876907348633, -1.0279803276062012, -1.3622478246688843, -0.1960763931274414, -0.44067832827568054, 0.5974191427230835, 1.0066912174224854, 2.6885669231414795, -0.5813180208206177, -1.9116345643997192, 1.2358016967773438, 2.074759006500244, -0.16110868752002716, -1.584932804107666, -0.5722953677177429, 0.6255776882171631, -0.050071269273757935, 0.2356780469417572, 0.7159756422042847, -0.2900885343551636, -0.6714972257614136, 1.5441434383392334, 0.7775202989578247, -0.43714648485183716, -0.170018270611763, -0.011236256919801235, 0.763455867767334, 1.6353790760040283, -1.2471846342086792, 0.10173254460096359, -1.2681599855422974, 0.6658365726470947, 0.048160020262002945, -0.4359339475631714, -0.2054765671491623, 1.28629732131958, -0.2929866313934326, -1.8725920915603638, 0.16372908651828766, 0.40440213680267334, -0.5142337679862976, 0.2812986671924591, 0.29869332909584045, -1.2885581254959106, 0.7420706152915955, 0.5486494898796082, 0.6301495432853699, -0.4599475562572479, -1.2059861421585083, -0.05591996759176254], [-0.6073417067527771, 0.715324878692627, -1.96518874168396, -0.1733497977256775, 1.694392442703247, -0.7606102228164673, 0.038696009665727615, 0.5858436226844788, 0.3726484775543213, 0.5923415422439575, 0.8520359396934509, 1.6937659978866577, -0.1874067634344101, 0.627558708190918, -1.00265634059906, -0.7242002487182617, 1.2103310823440552, -0.17591534554958344, 1.0589730739593506, -1.6009835004806519, -0.015241624787449837, 0.8123657703399658, 0.8305124640464783, -0.35625240206718445, 1.3186726570129395, 0.38482949137687683, -0.44423583149909973, -0.6931627988815308, -0.5343884825706482, -1.5494579076766968, -0.6384563446044922, -1.3379358053207397, -0.5927478075027466, 0.850222110748291, -0.08739922940731049, 0.3983617126941681, 0.37283122539520264, -0.9813453555107117, -1.6953147649765015, -0.13881652057170868, 0.27375930547714233, 2.0522050857543945, 3.233677864074707, 0.036306802183389664, 1.4786983728408813, -0.03723270818591118, -2.3042776584625244, 2.0468454360961914, 0.5376561284065247, -1.5859363079071045, 0.10946048051118851, 0.7027504444122314, 0.36144331097602844, -1.9391098022460938, 1.0316847562789917, -0.32917243242263794, -0.0007758027641102672, 1.0232272148132324, 1.3510839939117432, -0.9105021953582764, 0.5351513028144836, 0.5826292037963867, 0.7999575734138489, -1.2734923362731934, -0.6744710206985474, 0.6399408578872681, -0.5764740109443665, 0.23444893956184387, 1.034557580947876, 0.27883630990982056, 2.0819990634918213, 1.0137920379638672, 0.25978124141693115, -0.12828406691551208, -0.9413191676139832, -1.076436161994934, -0.9132140278816223, -0.014703563414514065, 0.27215754985809326, 0.9422681331634521, -0.8981719613075256, 0.017285162582993507, -0.7311748266220093, -0.08828526735305786, 0.45768076181411743, -1.545657992362976, 0.04512082785367966, -0.2961164712905884, -0.5228654146194458, -0.7812454104423523, -0.2686491310596466, -2.0968077182769775, 0.8191205859184265, -0.19604498147964478, 0.6864717602729797, 0.7079153656959534, 0.40943753719329834, 1.9947850704193115, 0.9225395917892456, -0.5043929815292358, -1.4866995811462402, 0.014937832951545715, -1.113773226737976, -1.567147970199585, 0.665155827999115, 0.4936182200908661, -1.3583117723464966, -1.4527255296707153, -1.544012427330017, -1.387152910232544, -2.0937912464141846, -1.029326319694519, -0.8543629050254822, 0.836335301399231, -0.6750411987304688, -1.5198125839233398, -0.08131694793701172, -1.3184891939163208, -0.0779990628361702, 0.809775710105896, 0.9986532330513, 1.0500398874282837, 0.14817848801612854, 0.6158517599105835, -1.0318195819854736, -1.6567178964614868, 0.8881301879882812, -0.6452752947807312], [-0.42925897240638733, 1.350711464881897, 0.8425392508506775, -1.2360577583312988, -0.21212878823280334, 0.656927764415741, -1.678101897239685, 0.06583813577890396, -0.4251435399055481, -0.6829357743263245, 0.2849924862384796, -0.6176222562789917, 1.0187327861785889, 0.3711659908294678, -0.9270361065864563, -1.122045636177063, -1.6368112564086914, 0.9904547929763794, 0.2300473302602768, 0.020408036187291145, -0.6502978205680847, 0.4948585033416748, 1.407111644744873, -0.2903232276439667, -1.3253158330917358, -0.14106309413909912, 0.6682369709014893, 0.5845721364021301, -1.714699625968933, 0.2302052229642868, -1.2434086799621582, -0.7396031618118286, 0.9964135885238647, -1.4964021444320679, -0.4199239909648895, -1.3091295957565308, 0.8483973145484924, -1.2157193422317505, 0.6038553714752197, -1.2324270009994507, -0.7627761960029602, -0.44930508732795715, -0.5066472887992859, 0.44443193078041077, 1.4641045331954956, 0.8054574728012085, -1.2544529438018799, 1.5014586448669434, 0.9582397937774658, 1.6038284301757812, -1.7662731409072876, 0.9937960505485535, 2.5267813205718994, -0.7951005697250366, -0.07728415727615356, 0.44620606303215027, -1.7766584157943726, 0.15119510889053345, 0.21403047442436218, -0.6892127990722656, -0.3590462803840637, 1.325053095817566, 0.20372819900512695, 1.5248820781707764, 0.27811333537101746, 0.28317537903785706, -1.8783460855484009, 2.574613332748413, -1.703911304473877, -1.01185142993927, -0.359250545501709, -0.49927470088005066, -0.8932503461837769, -1.2992141246795654, 2.016780376434326, -0.3171551823616028, -1.0845590829849243, 1.8630777597427368, 0.4609907269477844, -0.36812832951545715, -0.3361195921897888, -1.3698253631591797, -0.4529010057449341, 0.24240237474441528, 0.37722480297088623, 1.6402909755706787, -0.24609722197055817, 1.1429569721221924, 1.1477524042129517, -0.29243525862693787, -1.0062438249588013, -1.5406807661056519, -0.055213604122400284, -0.7075706720352173, -0.6308181881904602, -1.3259944915771484, 0.45749592781066895, 0.061899881809949875, 0.4443018436431885, -1.1346502304077148, -1.1675044298171997, -0.4928935170173645, -1.8407323360443115, -0.24265839159488678, 0.4879976809024811, -1.517281174659729, -0.9969269037246704, 1.397237777709961, -0.8744505643844604, -0.13044455647468567, -0.7312482595443726, 2.0948100090026855, -0.284680038690567, -1.1149396896362305, -0.7007074952125549, -0.11689706146717072, 0.5699794888496399, 1.2577154636383057, 0.41618913412094116, 0.006415301933884621, 1.744848370552063, 1.0615121126174927, 2.5117084980010986, 1.7796425819396973, 2.3862032890319824, 0.1801571547985077, 0.2781553864479065, -0.11964309215545654], [0.9459315538406372, -1.4814825057983398, 0.3611210584640503, -0.39669227600097656, 1.1909399032592773, -0.4145660698413849, 1.045027494430542, -0.7489169836044312, 0.5911754369735718, -0.06814667582511902, -0.133200541138649, 2.2206153869628906, -0.45235198736190796, 1.5213301181793213, -0.16264501214027405, -2.154438018798828, -0.2712762653827667, 1.0139209032058716, -1.0951036214828491, -0.6448658108711243, -0.4875733554363251, -1.4016133546829224, 1.8533453941345215, 1.3439522981643677, -2.015687942504883, -0.6293838620185852, -0.3300435543060303, -0.18145237863063812, -0.7651617527008057, -0.28496837615966797, 0.15273737907409668, -1.237752079963684, -0.3006354570388794, 0.40583619475364685, -0.7541241645812988, 0.9779804348945618, 0.9767709374427795, 1.060477375984192, 0.22250118851661682, 1.2554004192352295, 2.006542682647705, -1.7457960844039917, -0.229575052857399, -0.3711394965648651, -0.017710881307721138, -1.5177741050720215, -1.3909779787063599, -1.3767178058624268, -2.9622702598571777, -0.766573965549469, -0.3371082842350006, -0.05735066533088684, 0.9288415312767029, 1.3667874336242676, -0.5786752700805664, 0.5993700623512268, -1.2666866779327393, -0.7163198590278625, 0.6062964200973511, 1.8174077272415161, 1.0692787170410156, -1.108128309249878, 1.215822458267212, 0.8229497671127319, 1.9033253192901611, 1.4184067249298096, -0.6114270687103271, -1.1229981184005737, -0.03911655396223068, -1.2174710035324097, -0.38623279333114624, 0.3214700222015381, 0.3859404921531677, 0.8367681503295898, -0.5205668210983276, 1.255658745765686, 0.947597861289978, -0.3561728596687317, -0.763602614402771, 0.6806750297546387, -0.7695128917694092, 0.7553917765617371, 0.1264653354883194, -1.262840986251831, 0.9873745441436768, 1.9085297584533691, -0.4817259609699249, 1.6024202108383179, -1.327700138092041, -1.3640072345733643, 0.1790304332971573, -1.0861713886260986, -1.4938594102859497, 0.4287669062614441, 0.989185094833374, -1.7162513732910156, -0.2931857407093048, -2.525987386703491, 1.128921389579773, 2.2041335105895996, 0.28161218762397766, -0.3729446828365326, -0.5287735462188721, -3.4843356609344482, 0.14715155959129333, -0.4491308629512787, 1.4648751020431519, 1.0659325122833252, 1.2584316730499268, 0.9179895520210266, 0.4275067448616028, 1.5761274099349976, 0.6476832032203674, 0.060252733528614044, 0.3273888826370239, -0.1910117268562317, -0.2351950705051422, -1.3568036556243896, -0.05533023923635483, 1.4895799160003662, 0.11985990405082703, -0.645237922668457, -0.16922560334205627, -1.1297935247421265, -0.5322529673576355, 2.0047268867492676, 0.9072077870368958, -0.06656602770090103], [1.1760506629943848, 0.08523296564817429, -0.06360368430614471, -0.10734744369983673, -3.089843511581421, 1.4675723314285278, -1.1524229049682617, 0.5199482440948486, -0.2667538523674011, 0.0008738380274735391, 0.30407676100730896, -0.9518132209777832, 0.07484331727027893, 0.3432486057281494, -0.08221999555826187, -1.4377553462982178, -0.05467141047120094, 0.07230851799249649, 2.1421494483947754, -1.4144445657730103, -0.9515750408172607, -0.735923171043396, 0.48389461636543274, -0.6614649295806885, -0.07079502195119858, -0.44598865509033203, -0.18006344139575958, -0.2274015098810196, 0.24130085110664368, 0.4819374680519104, 0.15975767374038696, -0.7764260172843933, 2.3122498989105225, -0.009377910755574703, -1.1642591953277588, 0.8454559445381165, -0.6090335845947266, 0.3443411886692047, 1.11044442653656, -1.1712286472320557, 0.7434118390083313, 1.4574556350708008, 0.7650328278541565, -0.5876054167747498, -1.6111607551574707, 0.8484520316123962, 0.36353036761283875, -0.8477944731712341, -0.47340941429138184, -0.1501445323228836, -2.1253347396850586, 0.6580616235733032, -0.566614031791687, 0.48068737983703613, -0.34770119190216064, 0.05979655310511589, 0.7205435037612915, 0.6003321409225464, -1.306830883026123, -0.836862325668335, 0.4526415765285492, 0.8214948177337646, -1.666181206703186, 0.37187156081199646, 0.37053588032722473, 0.7681783437728882, -0.37705129384994507, -0.2837282419204712, 0.40777671337127686, -0.6658517718315125, -1.633309245109558, 0.09282603859901428, 1.6510717868804932, 0.22115840017795563, 0.7139649987220764, -0.46772125363349915, -1.4586535692214966, -0.025964051485061646, 0.9988943934440613, 2.0292110443115234, 1.6097251176834106, -0.13136039674282074, 0.5766556262969971, 0.45032933354377747, 1.7295641899108887, -0.2059403657913208, 0.7080113887786865, -0.7447535991668701, 0.8397557139396667, -0.06686799228191376, -0.2819705009460449, 0.4460492730140686, 0.6064682602882385, 1.977424144744873, 0.9961812496185303, -1.213646411895752, -1.7445040941238403, 0.8014857172966003, 1.8486131429672241, 0.09789160639047623, 1.6830838918685913, 0.48748651146888733, -0.5264514684677124, 0.9241936206817627, -1.1973305940628052, -0.7357286810874939, 0.7342759370803833, -0.8831776976585388, -2.1252665519714355, 1.3247206211090088, 0.9921937584877014, 0.8136600255966187, -0.7688493132591248, 0.5692254900932312, 1.105656385421753, -0.7628198266029358, -0.056970518082380295, 0.5128364562988281, -0.6637957692146301, -0.16341948509216309, -0.4268386960029602, 0.8416000604629517, -0.02862258441746235, 1.6727646589279175, 0.13292574882507324, 0.2147781401872635, 1.7906709909439087, 0.5480947494506836], [0.20132337510585785, -0.7271623015403748, 0.05642851069569588, -0.938263475894928, -0.49880555272102356, -0.3679916560649872, -0.12046214938163757, -0.35305193066596985, 1.6740493774414062, -0.8128640651702881, 1.0093790292739868, 0.3254949450492859, -0.4827359616756439, -0.5466570854187012, 1.1614102125167847, -0.5361940860748291, -0.800208568572998, 1.2991348505020142, -0.4325386881828308, 1.100088119506836, -0.16657929122447968, 0.7163146138191223, -0.2853311002254486, -0.11467020958662033, 0.3562687039375305, 0.09499318152666092, -0.009427832439541817, -0.6443984508514404, -1.2817838191986084, -0.13158200681209564, 0.7505026459693909, 1.5513300895690918, 1.2744413614273071, -0.3688759207725525, 1.5956090688705444, 0.9760947823524475, -0.6374647617340088, -0.3087766468524933, 0.4964786469936371, 0.2522140443325043, -0.7911044359207153, -0.9800117611885071, 0.23481860756874084, -0.05669146776199341, 1.0230331420898438, 0.7160046696662903, 0.0013876709854230285, 0.20352156460285187, 0.4117608368396759, 0.35302045941352844, 0.03023824840784073, 0.8006412386894226, 0.4554528594017029, 0.5278847813606262, 1.7857612371444702, 0.9738712906837463, -0.005393819883465767, 0.07532558590173721, 0.47635361552238464, 1.553066372871399, -1.3658912181854248, -0.7040998339653015, -1.3215632438659668, 1.3099647760391235, -1.780947208404541, 0.3474237620830536, -0.5303136706352234, 0.12833824753761292, 0.12400486320257187, 1.0178720951080322, -1.1635369062423706, -1.8375658988952637, -1.018796682357788, 0.34659937024116516, 1.0936932563781738, -1.6418426036834717, 0.17569012939929962, -1.6369448900222778, -0.14349494874477386, -0.4285648465156555, -0.7988197207450867, -0.8426796793937683, 0.14738202095031738, -0.5690450668334961, -0.8098708391189575, 1.0356014966964722, -0.3761899173259735, -0.2022686004638672, -1.581063151359558, -0.6635468006134033, -0.27657628059387207, -0.3523165285587311, 0.49558165669441223, 0.33188849687576294, 0.3195018768310547, -0.39191219210624695, -1.2119367122650146, 0.792496919631958, -0.052174217998981476, -1.753472924232483, -0.13562177121639252, -0.5251463055610657, 0.25921809673309326, -0.9749635457992554, 0.7731924653053284, -1.1268117427825928, 2.9049489498138428, 0.5706400871276855, 0.6985705494880676, 1.6101832389831543, -0.31110671162605286, 0.6530537605285645, 0.1948351413011551, 0.20036153495311737, 0.9301453232765198, 0.6333446502685547, -0.5714458227157593, 1.0176969766616821, 0.6095972061157227, 0.5472838878631592, 0.5042749643325806, 0.47054392099380493, 1.1965056657791138, -1.3214178085327148, -0.18368087708950043, -1.3172621726989746, 0.3110991418361664, -1.1921087503433228], [0.4108850061893463, 0.8755598664283752, 0.5298058986663818, -1.33009672164917, 0.8578712940216064, -0.9348858594894409, -0.5949655175209045, -1.0773334503173828, 1.1452771425247192, 0.8771899938583374, 0.43969693779945374, -1.0925649404525757, 0.049081236124038696, 2.3477585315704346, 2.419440984725952, 0.1997756063938141, -0.5497023463249207, 0.047863442450761795, -1.1903780698776245, -0.5906874537467957, -0.4707213044166565, 0.6040259003639221, 0.4688703417778015, -0.5285370349884033, -0.04299642890691757, 0.895257830619812, 0.5497535467147827, 0.09479779750108719, -0.38524848222732544, 0.8085905313491821, -0.11790579557418823, 0.57316654920578, 0.6758608222007751, 0.3621233403682709, 2.2900631427764893, 0.7540165185928345, -0.4205559194087982, -1.752762794494629, 0.09661313146352768, -0.5667917132377625, 0.8259682655334473, -0.04946041852235794, -0.08270194381475449, -2.9689781665802, -1.617796540260315, -0.5610092878341675, 2.118474006652832, -0.2148444950580597, -0.2215128093957901, 0.7641497254371643, -0.46092724800109863, -0.3751997649669647, 0.5068181753158569, -1.3385812044143677, -0.6266672611236572, -1.6780545711517334, -1.3235018253326416, -0.047372158616781235, 4.241214275360107, -0.6502453684806824, -0.6681503057479858, 1.0065431594848633, -0.7957695126533508, 0.9491212964057922, 0.6862686276435852, -1.1512292623519897, 0.9929004907608032, 1.3888075351715088, 0.6426990032196045, 0.3968881368637085, 0.6409083008766174, 1.1109540462493896, -1.2971376180648804, 0.390482097864151, 2.4175429344177246, 0.7193760871887207, -2.4251325130462646, 0.21366183459758759, -0.3443259000778198, -0.46935397386550903, 1.1463721990585327, -2.102628469467163, -0.6472599506378174, -0.8000525832176208, -1.4515643119812012, -0.609041154384613, 1.3353208303451538, 0.7531793117523193, 0.6877306699752808, -0.052531227469444275, -0.20097967982292175, -1.528678059577942, 1.0442620515823364, -0.4285324513912201, -0.22003698348999023, -0.8880614042282104, 0.2905820310115814, -0.0031945579685270786, -0.4947323799133301, -1.1125068664550781, -1.9675191640853882, -1.57778000831604, -0.07605832815170288, 2.0062918663024902, 0.3983733654022217, -0.3835467994213104, -2.2746355533599854, -0.8699488639831543, -1.077762246131897, -0.27207550406455994, 0.18220283091068268, -1.75176203250885, -1.9224402904510498, -0.701600193977356, -0.42336833477020264, -1.0780683755874634, 0.40526726841926575, 1.0007855892181396, 0.35434776544570923, -0.8523192405700684, -0.9385517835617065, -0.3960702419281006, -1.4598605632781982, 0.36776065826416016, -0.3139464259147644, -0.30757445096969604, -0.11765418946743011, 0.5880032777786255], [-0.638331413269043, 0.1355012208223343, 0.619240939617157, -1.018544316291809, -1.456470251083374, -1.2511203289031982, -0.4412466883659363, -0.538024365901947, -0.3119668662548065, 0.04544089734554291, 2.4324660301208496, 1.5096848011016846, 0.8187823295593262, -0.16752193868160248, -0.2944198250770569, 2.4848475456237793, 0.7186804413795471, -2.1699936389923096, 0.29932770133018494, 0.19128340482711792, 0.4620092213153839, 0.6988661289215088, 1.1372333765029907, 0.1673198789358139, -0.14521683752536774, 0.7391597628593445, 0.4448310434818268, 0.15241342782974243, -1.7026816606521606, 2.6985185146331787, 0.5028271675109863, -0.23546481132507324, -0.4692234992980957, 0.026845932006835938, -0.38262447714805603, 1.130120873451233, -1.5650815963745117, -0.9901665449142456, 0.23716507852077484, 0.1260862797498703, 1.146206021308899, -0.5684670209884644, -0.43453046679496765, -0.886784017086029, -0.08495113998651505, 0.39988937973976135, -0.308732271194458, 0.6330843567848206, -0.007856558077037334, 0.6447945237159729, 0.473483145236969, -1.9452732801437378, 0.7583450078964233, -0.5363526940345764, 0.6835777163505554, 0.4243922531604767, -0.1121433675289154, 0.7842845916748047, -1.6559351682662964, -0.15773296356201172, 0.43443992733955383, 0.18360066413879395, 0.9648165702819824, 0.6680917739868164, 0.7462512254714966, 0.28758522868156433, 0.218858540058136, 1.1080036163330078, -0.4102650284767151, 1.3578102588653564, 0.10436802357435226, -0.13556674122810364, 0.3819067180156708, -0.22109298408031464, -1.2979803085327148, 0.26039037108421326, 1.0387256145477295, 0.005461495369672775, 0.7536124587059021, -1.6228448152542114, -2.072310447692871, -0.9509515166282654, 0.775083065032959, 1.4084419012069702, 0.9233278632164001, -0.2106277048587799, 1.6966947317123413, 0.08091229945421219, -0.5717231631278992, -1.412128210067749, -0.7423869371414185, 0.3462531566619873, -1.0624524354934692, 0.5390222072601318, 0.6117066740989685, -2.4070487022399902, -0.5019392967224121, 0.7001580595970154, -1.0982997417449951, -0.7634445428848267, -0.641262948513031, 2.026157855987549, 0.9650152325630188, -0.7468101978302002, 0.1364506334066391, -0.04038281366229057, 0.566763699054718, 0.5327801704406738, 0.35138118267059326, 0.5182957053184509, 0.18890412151813507, -0.9708215594291687, 0.8367193341255188, 1.033706784248352, -0.32946667075157166, 1.25155508518219, -0.5544800162315369, 1.6837198734283447, -0.4684949815273285, -1.3538142442703247, -1.3767249584197998, 1.355107069015503, 0.18968211114406586, -0.3258480727672577, 0.5876644849777222, 0.7768446207046509, 0.18816891312599182, -0.3656143546104431]]]], "dtype": "float32"}, {"__nd__": [[[[0.5368158221244812, 0.017400164157152176, 1.1949751377105713, -0.20110706984996796, -1.3800745010375977, 0.400943398475647, -1.0118361711502075, -0.05473710596561432, 0.06862156838178635, -0.43199172616004944, -0.417115181684494, -0.0631939023733139, -0.7628386616706848, 0.92723548412323, -0.6686881184577942, -0.5349510908126831, -0.47628915309906006, -0.026795655488967896, 0.551253616809845, -0.3932415246963501, -0.11949573457241058, -0.4536638855934143, -1.8522911071777344, 0.016763480380177498, -1.7530547380447388, -1.7213430404663086, 1.4023301601409912, -1.4207547903060913, 0.35673949122428894, 1.1914528608322144, -0.21311421692371368, 0.5729233622550964, -0.23020873963832855, -0.4996718466281891, 0.8948315978050232, -0.20894977450370789, -0.5284685492515564, 0.306109219789505, 0.6321607232093811, -0.12033744156360626, -1.2826859951019287, -0.3322564363479614, -0.0812881663441658, -0.7915240526199341, -0.840208888053894, -0.5370051860809326, 0.4835725724697113, -1.1581951379776, 1.2830586433410645, 1.4508861303329468, -2.135451555252075, -0.5516238808631897, -0.8649051189422607, -0.6903594732284546, -0.6305643320083618, -0.47897422313690186, -0.11917991936206818, -1.1306116580963135, -1.1236211061477661, -0.6542324423789978, 1.3505157232284546, -0.97711580991745, -0.1586431860923767, 1.3388310670852661, 1.6547048091888428, 0.538483738899231, 0.05727827548980713, 0.06795436888933182, 1.9388165473937988, 0.579855740070343, 0.2205633521080017, -0.009285973384976387, 0.43343228101730347, 0.3067690432071686, 1.1113094091415405, 0.220603808760643, -2.5573794841766357, -0.9248543381690979, -2.071884870529175, -0.1681174486875534, -0.3303106427192688, -0.9019407033920288, -0.18799428641796112, -0.0245341919362545, 0.31358176469802856, 0.45756256580352783, -0.7529453039169312, 0.49241209030151367, -2.1165311336517334, 0.8472545146942139, 0.059856612235307693, -1.1782366037368774, -1.126229166984558, -1.3968030214309692, -0.5567728281021118, 0.06830932199954987, -1.0832775831222534, 0.3837514817714691, -1.5954797267913818, 0.5858510136604309, 2.729919195175171, 0.033704206347465515, 1.160508394241333, -0.5342431664466858, -0.03609530255198479, 1.5401949882507324, 0.8444561958312988, 0.6775966286659241, -0.33197838068008423, -0.7051905989646912, -0.25800421833992004, 0.4003200829029083, -1.3833030462265015, -0.4155385196208954, 0.2328791618347168, 0.3212478458881378, -0.14884203672409058, -0.2446255087852478, 1.9925055503845215, -0.39383113384246826, 0.02131551504135132, -0.46400779485702515, 1.040266752243042, 1.0748202800750732, -0.829272449016571, -0.443354994058609, 0.47999367117881775, 0.9569171667098999], [1.8014895915985107, 0.8984771370887756, 2.1619887351989746, 1.266956090927124, 0.811139702796936, 1.0273858308792114, -0.3727159798145294, -0.7305028438568115, -2.0733370780944824, 0.11752304434776306, -1.6374139785766602, -0.6789928078651428, 1.078574299812317, 0.4845871329307556, 0.15192589163780212, -0.21124430000782013, 1.573773980140686, 0.12216396629810333, 0.8983485102653503, -0.5677343606948853, 0.6153537631034851, 0.19141769409179688, 1.1450752019882202, 1.15092933177948, -0.14486275613307953, 0.631232738494873, -0.838866651058197, -1.7243361473083496, 1.699590802192688, -0.6749004125595093, -0.19780464470386505, 1.6883885860443115, 0.011639991775155067, -0.8129627704620361, 2.9426965713500977, -0.93264240026474, -0.15306924283504486, 0.4261174499988556, -0.3075977861881256, -1.3791015148162842, 0.6203441619873047, 0.5422742962837219, -0.7109851837158203, 0.1673075258731842, -0.7519710659980774, 1.084520936012268, 0.1996101438999176, -2.1833081245422363, -0.7742457985877991, -0.6936057209968567, 0.27343785762786865, 0.2923717498779297, -0.018753642216324806, -0.13450969755649567, -0.0733054056763649, -0.5414842367172241, 1.2396585941314697, -1.1002140045166016, 0.10764359682798386, -1.0734643936157227, -0.011117640882730484, -0.5132078528404236, -0.29412809014320374, 0.9982079863548279, -0.9888743758201599, -0.2948371469974518, -1.897809624671936, -0.5330404043197632, 0.6399369835853577, -0.3551885187625885, 1.4028102159500122, 0.6833454370498657, -1.387239694595337, 2.2347404956817627, 0.8027409911155701, -0.2686130702495575, -1.3135979175567627, 1.1052273511886597, 0.4103839695453644, -0.6554260849952698, -1.2663804292678833, 0.17988157272338867, -1.1317265033721924, 1.6275030374526978, -2.18485164642334, -0.6346295475959778, -0.23120179772377014, 0.0989132821559906, 0.5750064849853516, 1.668648362159729, 2.131103038787842, -0.35182690620422363, -0.20666497945785522, -0.44415849447250366, 0.2432549148797989, -1.8586764335632324, -0.9179351329803467, 0.031441543251276016, 0.41170018911361694, 0.10566863417625427, 0.6621536016464233, 1.8798060417175293, -0.28532764315605164, 0.8057782649993896, -0.7691777348518372, 1.4765292406082153, 0.46411198377609253, 0.3474316895008087, -0.5126126408576965, -1.1291935443878174, -0.42085304856300354, 0.5997634530067444, 0.6566535234451294, -0.26113325357437134, -0.6920136213302612, -0.6062897443771362, 0.7510854005813599, 0.5456804633140564, 0.37596410512924194, -0.7033575773239136, 1.9657806158065796, -1.7370326519012451, -1.6835260391235352, 1.801596760749817, 1.0560743808746338, 0.114568792283535, 1.6073002815246582, 0.5734452605247498], [-0.8758911490440369, -0.5234928727149963, -0.6552647948265076, -0.6111379265785217, 0.2332548350095749, 1.5031636953353882, 0.3944416642189026, 1.0301475524902344, -1.5885895490646362, -0.9375389218330383, -0.7828952670097351, -0.6375077366828918, 0.34847432374954224, -0.7906064987182617, 2.2679853439331055, -0.6928210854530334, -1.1850019693374634, -0.5387393236160278, 0.4381082057952881, -0.02480441704392433, 0.5717008709907532, 0.6509735584259033, 0.9917028546333313, 2.0615975856781006, 0.8188779950141907, 1.8698663711547852, 2.240557909011841, -1.167313814163208, 0.8297750949859619, 0.055769968777894974, 0.6702504754066467, 1.2281218767166138, -0.5982961058616638, 1.2674458026885986, -0.39423471689224243, 0.962780773639679, -1.2801826000213623, 0.8785181641578674, 0.6653000116348267, -0.9954953193664551, -0.5267863869667053, 0.300520122051239, -0.34683462977409363, -0.6263895034790039, -1.1006253957748413, -2.205841064453125, -1.2006889581680298, 0.05810932070016861, 0.5528948307037354, 1.0272623300552368, -0.7922982573509216, -1.1157989501953125, -0.5909491181373596, 0.33267515897750854, -2.5209689140319824, -0.9317444562911987, 0.026000214740633965, -0.5513144731521606, 0.4025748372077942, 0.04938599839806557, -1.373757004737854, 0.5857062339782715, 0.7853514552116394, 0.2153932899236679, -1.3561972379684448, 0.6011730432510376, 1.1400340795516968, 0.7153142094612122, -1.1915044784545898, 1.635393500328064, -1.4954067468643188, 1.7498195171356201, 0.1666061133146286, -0.8659096360206604, 0.4044359624385834, 1.4635521173477173, -1.5200996398925781, -0.6034727692604065, -1.034595012664795, -0.5134424567222595, -0.44350025057792664, 0.5970656871795654, -0.028120527043938637, 1.3315171003341675, 0.0728277862071991, 0.4080889821052551, -0.3880634903907776, -0.17210209369659424, 1.6690160036087036, -0.7474591135978699, 0.6700291633605957, -1.607887864112854, 0.9314542412757874, -1.6655925512313843, -2.1173899173736572, -0.42593151330947876, -1.5776675939559937, -0.5163537263870239, -0.27590247988700867, -0.21315650641918182, 1.0294703245162964, 0.38327810168266296, 1.777993083000183, 0.25196996331214905, 0.6226068139076233, -0.27077844738960266, -0.8666415214538574, 1.471728801727295, -1.3166288137435913, 1.1130733489990234, 2.321141242980957, -0.5074915289878845, -0.7639350891113281, -0.3385631740093231, 0.35726308822631836, -0.8596016764640808, -1.3681154251098633, 1.4090365171432495, -1.220049262046814, 0.4944453537464142, 0.3292560279369354, -0.18019762635231018, -0.5951625108718872, -0.9720396399497986, -0.822668731212616, -0.36517763137817383, -0.3111065626144409, -0.3278350830078125], [1.472047209739685, -0.427308589220047, -0.19568972289562225, 0.4257834255695343, -0.8621846437454224, 1.2729251384735107, -0.20051512122154236, 1.5652315616607666, -2.729325532913208, 0.44603532552719116, -0.012227533385157585, 0.27768105268478394, 1.0214606523513794, -0.3166544735431671, 0.8763495683670044, 1.448882818222046, -0.5520051717758179, -0.6322644352912903, 0.35344457626342773, -1.7235389947891235, 1.4833112955093384, 1.356718897819519, -0.8894299864768982, -3.287152051925659, -0.8355063199996948, -1.762876272201538, 0.5071732401847839, 0.938889741897583, 0.888730525970459, -0.6721625328063965, 0.15153171122074127, 1.2138895988464355, -1.2374531030654907, 0.539772093296051, 0.33723774552345276, -1.6439684629440308, 0.16279840469360352, -0.5608586072921753, -0.7173736095428467, -0.5045642852783203, 1.213778018951416, -0.4986526072025299, 1.963431715965271, -0.05256425961852074, 0.2677488327026367, -0.5884141325950623, -0.7382019758224487, -0.25491583347320557, -0.11883365362882614, -0.192685067653656, 1.567077398300171, -1.3235888481140137, 0.9130211472511292, 0.19119791686534882, 2.8078818321228027, 0.8392814993858337, -1.621073603630066, 0.024753648787736893, -0.22427138686180115, -0.2435491681098938, -0.5026379227638245, -0.43719685077667236, 0.21503090858459473, -1.5814478397369385, 0.691647469997406, -0.9547410607337952, -0.5731542110443115, -1.1411542892456055, 0.015973912551999092, -0.3439839482307434, 0.6186464428901672, -0.26686686277389526, 0.19613619148731232, -0.22458335757255554, 0.7322002649307251, 1.1042691469192505, 1.7737256288528442, -1.372308373451233, 0.11906714737415314, -1.0655317306518555, 0.855896532535553, -0.009288793429732323, 2.148266077041626, -1.0258512496948242, 0.3915567100048065, -0.14632096886634827, 2.747640371322632, -0.16913706064224243, -0.6370219588279724, 0.10053690522909164, -0.2609602212905884, -1.7092055082321167, 0.9275481104850769, 0.1167772188782692, -0.9423406720161438, -1.0321056842803955, 0.6663272380828857, 0.9260188937187195, -1.6603648662567139, 0.7037687301635742, -0.8186649680137634, 0.03654469549655914, -0.6512214541435242, -0.5692063570022583, 0.9059736132621765, 0.048980943858623505, 1.8330605030059814, 1.0072375535964966, -0.6814755797386169, -0.00044875298044644296, -0.43759095668792725, 0.2289956957101822, 0.27089935541152954, 1.0783849954605103, 0.6252867579460144, -0.6486555337905884, -1.813012957572937, 0.10888936370611191, 2.0237722396850586, 0.6389346718788147, -1.3485578298568726, -0.47816163301467896, 1.4704450368881226, -0.19021844863891602, -0.3316501975059509, -0.852569043636322, 2.146953582763672, -0.8883844614028931], [-0.3123481869697571, 1.1281232833862305, -0.7313296794891357, 0.4064358174800873, -0.46912717819213867, 1.3405529260635376, 2.350200891494751, -0.29088228940963745, -0.49525701999664307, 0.8331283330917358, -0.7630026340484619, -0.5299777984619141, 0.8674607276916504, 0.07979430258274078, 0.2143866866827011, 0.18991385400295258, 1.4984229803085327, 1.207565426826477, -1.0145072937011719, -0.9171162843704224, 1.384706735610962, -1.169238567352295, -0.554840087890625, 0.17840811610221863, 0.7616071701049805, -0.5581950545310974, 1.248831868171692, 0.9157195091247559, -0.07948027551174164, 1.4743990898132324, 0.6606506109237671, 0.34085360169410706, -1.0458358526229858, -1.4994440078735352, -1.0139358043670654, 0.5911911129951477, -0.9817991852760315, -1.4131619930267334, -0.5357678532600403, -1.652106523513794, -0.39413848519325256, -0.8647427558898926, -0.20031295716762543, -0.47709643840789795, -0.49628564715385437, 0.6839599609375, 1.4202258586883545, 1.628991723060608, -0.14263124763965607, 0.7679255604743958, 1.5366637706756592, 0.4758911430835724, -0.7149755954742432, 0.8050458431243896, -0.8955910801887512, 0.8787998557090759, -1.939607858657837, -0.37798038125038147, -0.20979724824428558, 0.14615429937839508, 3.0802884101867676, -0.2470797300338745, -1.1448646783828735, 0.789366602897644, -1.5213617086410522, -0.0484621562063694, -0.1085469052195549, -0.7695095539093018, -0.19900216162204742, 1.5377415418624878, 1.359700083732605, -0.6190752983093262, 1.9908641576766968, 0.24874721467494965, -0.24948368966579437, 1.6017833948135376, 1.565977931022644, -1.2924613952636719, -0.5626232028007507, 1.7604016065597534, -0.22267204523086548, -1.315241813659668, -0.8193427324295044, -1.3219224214553833, 0.2661561369895935, -1.380832314491272, -0.35907799005508423, 0.25642046332359314, -1.6024574041366577, 1.6627376079559326, -1.363034725189209, 0.33307507634162903, -0.5361300706863403, -1.4032034873962402, 0.9376595616340637, -0.8606804013252258, -0.5317961573600769, 0.5049076676368713, -0.2041098177433014, -2.237955331802368, 0.44619712233543396, -0.011666566133499146, -1.6917468309402466, -1.603097915649414, 0.03561517223715782, 0.22158193588256836, 0.5022101998329163, 0.2626190483570099, -2.788832426071167, 1.3259837627410889, -1.237579584121704, 0.02567705698311329, -1.1342484951019287, 0.40065324306488037, -0.6695393919944763, 0.5264970064163208, -1.3363620042800903, -1.5555801391601562, 0.16320666670799255, -0.5412778854370117, -0.09256335347890854, -1.2659307718276978, -0.0429210402071476, -0.26489999890327454, -0.23868252336978912, -0.2743722200393677, 2.8390045166015625, 1.610830545425415], [-0.1863134801387787, -0.46991783380508423, 1.302363634109497, 0.5524309277534485, 1.860939860343933, 0.638103723526001, -0.8422749042510986, 0.6500913500785828, -0.8990122675895691, 0.8187171220779419, -2.013343334197998, -0.14627766609191895, -0.9512719511985779, 0.10824848711490631, 0.02463272586464882, 2.3748555183410645, -0.1957506239414215, -0.6764720678329468, 0.45054182410240173, 0.9089306592941284, -0.3419398069381714, 1.4957094192504883, -1.2571147680282593, 1.3812044858932495, -0.11729265749454498, 1.111161231994629, -1.2306116819381714, -2.024770736694336, -0.8954148888587952, -0.7972789406776428, 1.06571364402771, -0.1402883380651474, 0.07694670557975769, 0.9303086400032043, -0.5456685423851013, -2.051555633544922, -0.7022165060043335, -0.1792767494916916, -0.10070131719112396, 0.07286888360977173, -0.008779678493738174, 1.0437098741531372, 0.9866663217544556, -0.18441565334796906, -2.1034817695617676, 1.180718183517456, -1.8031116724014282, -1.7271274328231812, -0.014794074930250645, 0.3645652234554291, 0.8746567964553833, 1.5351463556289673, 1.6490665674209595, 0.010552664287388325, -0.9831928610801697, -0.15862347185611725, 0.27802059054374695, 0.9420045018196106, 0.7335190176963806, -0.12918759882450104, -0.8339707255363464, -0.638556957244873, 0.022018227726221085, -0.6323885917663574, -0.903583288192749, -0.30363279581069946, -2.361734628677368, 0.14047320187091827, 0.15651661157608032, 0.027585675939917564, -0.24471309781074524, 1.2154420614242554, 1.52760910987854, 0.13538168370723724, 0.3303191363811493, -0.4362338185310364, 0.48564067482948303, 0.0774400383234024, 1.2537477016448975, -1.9776537418365479, 0.3482576906681061, -1.3495858907699585, -1.3907113075256348, -0.06880448758602142, 1.1192080974578857, -2.4950029850006104, 0.8881480097770691, -0.2147577404975891, 0.16898757219314575, 0.10985004156827927, -0.29161375761032104, -1.575249433517456, 0.20521511137485504, 0.37039563059806824, 0.17045284807682037, -1.8461852073669434, -0.8227080702781677, 0.042937714606523514, 0.18872876465320587, -0.01280092354863882, 1.0247312784194946, 0.6550725102424622, 2.703561544418335, -1.1601197719573975, -0.025615127757191658, -0.03155270591378212, 0.6044774651527405, 1.5168966054916382, 0.17076879739761353, 0.22091172635555267, -0.7454089522361755, 1.4603420495986938, 0.4812532067298889, -0.8810521960258484, 0.48450714349746704, -1.067063331604004, -0.07784955948591232, -0.46105527877807617, 1.0688140392303467, 0.2771206200122833, -0.2363147735595703, 0.6645918488502502, -1.5681661367416382, -1.3205580711364746, -0.14647559821605682, -1.7138855457305908, -1.3420557975769043, 1.0792912244796753], [-0.568223774433136, 2.4101321697235107, 1.032137393951416, 0.5456179976463318, -1.4397011995315552, 1.7110761404037476, 0.6092399954795837, 0.9014051556587219, 1.8400956392288208, 1.609431505203247, -0.5497449040412903, -1.2024292945861816, -0.28087496757507324, 0.7644311189651489, 2.1874351501464844, 1.9127819538116455, -1.2767564058303833, -0.18324878811836243, -0.2808273434638977, 2.1448631286621094, 0.7430150508880615, 2.8146584033966064, -1.5664947032928467, -0.5914834141731262, -0.16469886898994446, -1.8506006002426147, -0.9470599889755249, 1.2222602367401123, 0.8561381697654724, 1.9919707775115967, -0.7911340594291687, -1.0266623497009277, -1.6671968698501587, 0.023455973714590073, 0.9100407361984253, 0.3648688495159149, 0.5956366062164307, -0.19479960203170776, -1.4331848621368408, -1.626241683959961, -0.617988646030426, -1.04293954372406, 1.011666178703308, -0.7229908108711243, -0.13882018625736237, 0.104397252202034, 0.47947174310684204, -1.4308291673660278, -0.8083516359329224, -0.31598734855651855, 0.515734076499939, -0.8644903898239136, 0.4452956020832062, -1.056546926498413, 0.4384270906448364, -2.401200771331787, 0.5985812544822693, 1.7881584167480469, 1.4726142883300781, -0.6317392587661743, 2.206975221633911, -0.28224948048591614, -0.6261340975761414, 0.40962207317352295, 0.6480312943458557, 0.806420087814331, 0.6725264191627502, 1.4878915548324585, -0.239496111869812, -0.8683184385299683, -1.9169795513153076, -0.01943100243806839, 0.7017636299133301, 1.3994762897491455, 0.5945018529891968, -1.4137424230575562, 0.8676640391349792, -0.6884518265724182, 0.42274320125579834, -0.7090911269187927, 0.4741019904613495, -0.43378058075904846, -2.4897754192352295, 2.5058326721191406, 0.39639750123023987, 0.6804618239402771, 0.15169687569141388, 0.6676197052001953, -1.5225183963775635, -1.386281132698059, -0.5483179092407227, 0.7397450804710388, -0.8608710765838623, 1.608378291130066, -0.9331023097038269, 0.8893269300460815, -0.14165157079696655, 1.1301000118255615, 0.8497282266616821, -0.01318956259638071, 1.6442183256149292, -2.0153353214263916, 0.0062677632085978985, 2.3695530891418457, 0.7867471575737, 1.552266001701355, -0.36285075545310974, -0.07239232212305069, -0.6938270926475525, -0.8175718188285828, -1.4468637704849243, 1.9701255559921265, -0.594860851764679, 0.8960858583450317, -0.667569637298584, 0.7803464531898499, 1.0079407691955566, 1.1873517036437988, 0.4104098081588745, -1.594433069229126, 1.8568353652954102, -0.037636447697877884, -0.7140305042266846, -0.12577350437641144, 1.0250825881958008, 1.951338529586792, 0.2217535525560379, 0.9031888246536255], [-1.4882867336273193, -0.009005128405988216, 0.3487968146800995, 0.990906298160553, 0.12585656344890594, -1.2621937990188599, -0.9406265616416931, -1.2643392086029053, -0.011706502176821232, -2.5475192070007324, 1.4133280515670776, 0.305128276348114, 0.3608209490776062, -1.097654104232788, -0.6679834723472595, 1.567677617073059, -0.31046968698501587, 0.35168901085853577, -2.1144590377807617, 0.6721532940864563, 0.34337860345840454, 1.4164139032363892, 0.6189881563186646, -0.002264807466417551, -1.5454318523406982, 0.5470484495162964, -1.6930091381072998, -0.8287070393562317, -1.1256414651870728, -0.2338506281375885, 1.31621515750885, 0.01619771681725979, -0.021627800539135933, -1.4980839490890503, -1.083984613418579, -1.122803807258606, -0.26923856139183044, -1.0795544385910034, 1.3675562143325806, -0.7593164443969727, -2.530616044998169, 0.14293432235717773, -0.6160132884979248, 2.3577864170074463, 0.6718742847442627, -0.15229398012161255, 0.20299147069454193, -0.02313896082341671, 0.6765668988227844, -0.3350970447063446, 0.8900789618492126, 1.987207055091858, -0.557245135307312, 0.4389112591743469, 0.6006157994270325, 0.5736732482910156, 0.5249831080436707, -0.04084233567118645, 0.9578880071640015, 0.3377189040184021, -1.7968134880065918, 0.3369530141353607, 0.6156513094902039, -0.43691492080688477, 0.2614656686782837, -0.023965250700712204, -0.4170483648777008, -1.3446379899978638, 0.46284401416778564, 1.3750184774398804, -0.4496939480304718, 1.6684272289276123, 0.0011723204515874386, 2.103336811065674, -1.1291027069091797, -0.5343546867370605, 0.1593019813299179, -1.3888810873031616, -1.1582729816436768, -0.20946386456489563, -0.534902811050415, -0.361297070980072, -0.0572386234998703, -1.0220402479171753, -0.15097816288471222, -0.0800916776061058, -0.3910726010799408, -0.5418884754180908, 0.568605363368988, 0.04536835849285126, -1.432010531425476, 1.1793724298477173, 0.47693875432014465, -1.0288347005844116, -1.4572666883468628, 2.4373016357421875, -0.1736941784620285, 0.8644205331802368, 0.6128370761871338, -1.1864550113677979, 1.0547513961791992, -1.491275668144226, 1.8202511072158813, -0.6809278130531311, 0.47590959072113037, -0.6221792101860046, -1.795210599899292, 0.33352136611938477, 1.247892141342163, -0.034564059227705, -0.39089435338974, -2.094791889190674, 0.4950109124183655, 0.836614191532135, 0.8224915862083435, -1.2233494520187378, -0.3972005248069763, 1.880025863647461, -1.144038438796997, -1.2657777070999146, 1.6718134880065918, -0.9370592832565308, 0.07313207536935806, 0.06276392191648483, 1.5028096437454224, -1.268284559249878, -1.7554658651351929, 0.5230245590209961], [0.19492407143115997, 0.4749293327331543, -0.1135808527469635, -0.13364146649837494, 0.14652585983276367, 1.3167057037353516, 0.09779142588376999, -1.8288452625274658, 0.8865215182304382, -1.0710723400115967, 0.48180484771728516, 0.13849720358848572, 0.13585713505744934, -0.4775784909725189, -0.20709514617919922, -1.7215923070907593, -0.5482048988342285, 0.3543149530887604, 0.11479818820953369, -0.8164299726486206, -0.0031777408439666033, 0.01709907315671444, -0.7304301857948303, -0.43443042039871216, -0.8764010071754456, -0.347575306892395, -0.5125715136528015, 0.5394070148468018, 0.34733253717422485, -0.3431466519832611, -0.6513870358467102, -0.9401265978813171, 1.4359030723571777, -0.23463264107704163, -0.21928325295448303, 2.2561306953430176, -0.0864100232720375, 0.890595018863678, -0.25004860758781433, -0.2876260578632355, 1.558569073677063, 1.4188979864120483, -0.3980391323566437, 1.110421061515808, -1.235140085220337, 0.1536746323108673, -1.0357067584991455, 1.991597294807434, 0.6620555520057678, 0.5875470042228699, 0.6416986584663391, -0.779770016670227, 0.23993992805480957, 1.7857025861740112, 0.38782447576522827, -0.6849282383918762, -0.4015924334526062, 1.3797491788864136, 0.010111622512340546, -1.1581900119781494, 1.1412814855575562, -0.33503949642181396, -0.20414862036705017, -0.1607270985841751, 0.590628981590271, -0.13500291109085083, -1.3892171382904053, 1.498863697052002, -0.3770653307437897, 0.09513810276985168, 0.21204854547977448, 1.6639763116836548, 0.4754190146923065, 0.4602862000465393, 2.3220877647399902, -1.1068462133407593, -1.00282883644104, 2.5091967582702637, 0.5567962527275085, 0.4369409382343292, -1.1502258777618408, 0.5653576850891113, 1.8680702447891235, -0.7579100131988525, -1.6024456024169922, -1.7183810472488403, -1.0384610891342163, -0.3340728282928467, -1.5373528003692627, -0.154231995344162, -0.745593786239624, 0.20140840113162994, -0.45106396079063416, 0.15838642418384552, -0.13658951222896576, 0.01982497237622738, -0.715020477771759, -1.6419098377227783, -1.5073281526565552, -0.005523839499801397, -0.2343040108680725, 0.023205876350402832, -0.744585394859314, -1.3990881443023682, 0.01785110868513584, -0.13381940126419067, 0.6818129420280457, 1.6012117862701416, -0.6971069574356079, 0.5176330208778381, 0.38756853342056274, -0.17956975102424622, -1.011223316192627, 0.06308680027723312, 1.8971436023712158, -1.1983767747879028, 0.2694348692893982, 1.4042599201202393, -0.7838257551193237, 0.6338623762130737, 0.36783361434936523, -1.2013332843780518, 0.5253501534461975, -1.8332297801971436, 0.015113363973796368, 1.434035301208496, 1.3174731731414795, -0.3174515962600708], [-0.280278742313385, 1.4181814193725586, 0.553074836730957, -0.337760329246521, -0.21524403989315033, -0.05457804724574089, 0.006845393683761358, 0.09677186608314514, -1.3624686002731323, 2.1414291858673096, -2.265526294708252, 1.2703529596328735, -1.8244162797927856, 0.6273747682571411, -2.9077248573303223, -1.0390751361846924, 1.2865931987762451, -0.24582457542419434, -0.18457110226154327, 1.4291939735412598, 2.20906400680542, -1.4929733276367188, 2.854631185531616, -0.1555793136358261, 0.7506193518638611, 1.811134934425354, 0.7531416416168213, -0.2950066328048706, -0.08011770248413086, 0.5235303044319153, 0.4333442151546478, 1.0866657495498657, 1.1765090227127075, 1.2914777994155884, -0.44286367297172546, 0.15429575741291046, -0.8036713600158691, 0.4614294469356537, -1.6394425630569458, 0.09964919090270996, 0.37101513147354126, 0.717206597328186, 0.13485431671142578, 0.27876508235931396, -0.9898895025253296, 0.19685699045658112, -0.007848389446735382, 0.8493924736976624, -1.0289232730865479, -0.057311754673719406, 0.10364943742752075, 2.822106122970581, -0.3202288746833801, -0.19903449714183807, 0.7788196802139282, -2.007032632827759, 0.054428521543741226, -0.8899638056755066, 0.5967931747436523, 0.003696602303534746, 0.9597073197364807, -0.3410187065601349, 0.6904222369194031, -1.0123999118804932, 0.5970801115036011, -0.22677397727966309, -1.7115219831466675, 1.1769800186157227, 0.45293542742729187, -0.48149073123931885, -1.16067636013031, -0.4828251004219055, -0.36046719551086426, -0.4167191684246063, -0.8470567464828491, -0.15438497066497803, 0.04699571058154106, 1.7339718341827393, 1.4056624174118042, 1.5676257610321045, 0.3267582058906555, 0.12553155422210693, 0.5617830753326416, -0.7740615606307983, -1.2370699644088745, 2.731264114379883, 0.35023701190948486, 0.6863725781440735, -0.8692349791526794, -0.6705254912376404, 0.021688859909772873, 1.503808856010437, -0.8523605465888977, 1.6270177364349365, 0.7098421454429626, -1.7561086416244507, 0.3089805245399475, -0.7517444491386414, -0.3791223466396332, -1.1077384948730469, -2.86883282661438, -0.8436217904090881, -0.26616573333740234, -0.6507405638694763, -1.3440595865249634, -0.6110255122184753, -1.0913782119750977, -0.04220953956246376, 0.4711490869522095, -0.4263499081134796, 1.364475965499878, 0.2546393573284149, -0.6069545745849609, 0.3730439841747284, -0.8502646088600159, 0.40737083554267883, 1.8276914358139038, -0.38053855299949646, 0.016419146209955215, -0.4649880826473236, 0.6941439509391785, -0.7551228404045105, 1.6348038911819458, -0.3870977461338043, -0.05086928606033325, -0.09061981737613678, -0.5627607703208923, 0.9763091802597046], [-1.008291482925415, -0.452269047498703, -1.2299463748931885, 1.7502657175064087, -1.218950867652893, -2.326047658920288, -0.607535719871521, -0.3603646755218506, -1.022282600402832, 0.6635264158248901, -0.2687382996082306, 0.5525631308555603, -0.7904410362243652, 0.31949177384376526, 0.690979540348053, 0.922814667224884, -0.735448956489563, 1.5942025184631348, 0.24488197267055511, 1.0794520378112793, -0.09776957333087921, 0.5462605953216553, -0.07892017811536789, -1.1968367099761963, 2.0208423137664795, 1.8421109914779663, -0.9712513089179993, -2.4117538928985596, 0.2143482267856598, -0.8138126134872437, 2.3357090950012207, -0.47631874680519104, 0.8160786032676697, -2.0894339084625244, 1.2931357622146606, 1.4426193237304688, 1.5058567523956299, 0.04319620132446289, -0.43900322914123535, -0.028101496398448944, 0.8373923301696777, -0.2660089433193207, -0.5157502293586731, 2.1796138286590576, 0.6094139814376831, 1.6664916276931763, 0.1442584991455078, -0.9237440824508667, 0.8664920330047607, 0.9427074193954468, -0.6378358006477356, 0.6261070370674133, 0.4576669931411743, -1.608896017074585, -0.17518101632595062, 0.35551393032073975, 0.5009080767631531, 0.40844735503196716, -0.9668251872062683, 1.2429994344711304, -0.8027772307395935, 0.039458148181438446, 1.0939993858337402, -0.3329204320907593, 1.218769907951355, 0.6303650736808777, -0.4531557559967041, -0.5811835527420044, 2.5269906520843506, 1.0527094602584839, 1.3502932786941528, 0.6089913249015808, 1.4769859313964844, -0.11095300316810608, -0.12835176289081573, 1.0761317014694214, -0.16329701244831085, 1.370437741279602, -0.8275322914123535, -1.962775707244873, -0.542542576789856, -1.4745488166809082, -0.16328832507133484, -0.5277456641197205, 0.6946157217025757, 1.7377164363861084, -0.7832353115081787, -1.327001929283142, -1.8350590467453003, -0.4417720437049866, -0.9437364935874939, -0.12071393430233002, 0.09066605567932129, -0.8599240183830261, 0.22551818192005157, 1.3655389547348022, 0.5669003129005432, 0.6790515780448914, -1.4869626760482788, -1.7056361436843872, 0.31613433361053467, 0.5728935599327087, -0.38999414443969727, -0.3870203197002411, 0.684634268283844, -0.9460073113441467, -1.9371612071990967, -1.778329610824585, 1.568339467048645, -0.40703555941581726, -1.1786043643951416, -2.245091199874878, -0.7612466812133789, -1.6704494953155518, 0.8816890716552734, -1.2820104360580444, 2.5710036754608154, -1.7445039749145508, -0.9411260485649109, -1.7755613327026367, 1.3129998445510864, 0.14395377039909363, -0.2824030816555023, -0.8404339551925659, 0.21018865704536438, 0.23077137768268585, 0.019009128212928772, 0.09602419286966324], [0.388517826795578, 1.4075706005096436, 0.23086421191692352, -0.13203632831573486, 0.7863192558288574, -0.46945223212242126, -0.08307003974914551, 2.6486682891845703, -0.07300131022930145, -0.5822718143463135, -1.1196643114089966, 1.4904727935791016, -0.4691406190395355, 0.630318820476532, -0.4867769181728363, 0.20680126547813416, 1.663637638092041, -0.4912574291229248, 1.8896207809448242, 0.9074490070343018, -0.8490904569625854, -0.21979007124900818, 0.2771419584751129, 0.7600645422935486, -0.1498090624809265, 0.4206083118915558, 0.5439375638961792, -1.3486497402191162, 2.0628724098205566, -0.5777602195739746, -1.074711799621582, 0.035253334790468216, 0.0022750901989638805, -1.149627447128296, 0.5277760624885559, -0.7222553491592407, 0.22084097564220428, -0.3802008330821991, -0.6671008467674255, -0.6815140247344971, 0.9442863464355469, 1.3921325206756592, -0.2641994059085846, 1.1898610591888428, 1.5898220539093018, -0.04742661863565445, -0.11969726532697678, 0.6561510562896729, -0.4507095515727997, -0.8199189901351929, 0.370278924703598, -0.5711394548416138, -0.42523613572120667, -1.6979701519012451, -0.06173371151089668, 1.479871153831482, -1.5147182941436768, -1.1265870332717896, 0.719566285610199, 1.1226338148117065, -0.38006478548049927, 1.1784415245056152, 0.6551491022109985, 1.1518887281417847, 0.11068569123744965, -0.29865339398384094, -0.27694010734558105, -0.01530519500374794, 1.9091228246688843, 0.24577456712722778, -1.324962854385376, 1.1555712223052979, -0.9561665654182434, 0.36274248361587524, -0.7088911533355713, -0.2575962543487549, -0.34085848927497864, -0.28242287039756775, 1.2727183103561401, -0.5902232527732849, -0.21799412369728088, 0.18375979363918304, 0.23584085702896118, -0.603023111820221, 0.27899909019470215, 2.029679298400879, -0.8144429326057434, -0.3572344183921814, 0.34897735714912415, 0.4310496151447296, 0.0056328182108700275, 0.9610227346420288, -2.027050495147705, 0.026031624525785446, -0.8167117238044739, -0.05120163410902023, -0.5014893412590027, 0.8099048733711243, 0.03772038221359253, -1.0759717226028442, 0.431082546710968, 1.1158183813095093, 0.25359243154525757, -1.0273743867874146, -0.5229146480560303, 1.1357017755508423, -0.7447065711021423, 0.3114802837371826, -0.4809655547142029, 0.14436985552310944, -0.4850425720214844, 0.24904415011405945, -0.02000284567475319, -0.19046460092067719, 1.5487360954284668, 0.9604208469390869, -1.0016815662384033, -0.9545254111289978, -0.44562265276908875, -0.08794379234313965, 1.222804069519043, -1.2739428281784058, -2.0725810527801514, -0.847383975982666, 1.0149599313735962, 1.208930253982544, 0.6710101962089539, 0.44555163383483887], [-0.3339356482028961, 0.05174853280186653, 0.3828153908252716, 1.1776700019836426, -0.12372022122144699, -1.399592638015747, 2.0020878314971924, 0.6795803904533386, -1.107030987739563, -1.3671759366989136, 1.2716585397720337, 0.34743189811706543, 0.14239715039730072, 1.1109017133712769, -1.3062242269515991, 0.42373794317245483, 0.9165028929710388, 0.5452788472175598, 0.21071214973926544, 0.09452494233846664, -0.05878623574972153, 0.8221970200538635, 0.004706131760030985, -0.5773299336433411, 0.23414917290210724, -1.8230254650115967, -1.0804427862167358, 0.6661469340324402, 1.4641081094741821, -0.7119266986846924, 0.6225830912590027, 0.35854464769363403, -0.5363466143608093, 0.021008845418691635, -0.1661689132452011, -0.15900550782680511, 0.8286726474761963, 0.9638702273368835, -0.6763132810592651, -0.7675787806510925, 1.2362059354782104, -0.1395522654056549, 1.984683632850647, 2.6037638187408447, -0.21656186878681183, -0.5460304021835327, 0.15373998880386353, -0.2173824906349182, -0.08486788719892502, 1.1508736610412598, 0.645743191242218, 1.0522099733352661, 1.2450605630874634, -0.920735239982605, -0.9594312310218811, 0.7253172397613525, -0.016070803627371788, -0.2131127417087555, 0.18191440403461456, -0.36337947845458984, 1.228388786315918, 0.3871726095676422, 0.5048183798789978, -0.2773977518081665, 0.5000874400138855, -0.9279802441596985, 1.749257206916809, 2.9732635021209717, 2.0139904022216797, 0.8278279900550842, -0.7533509135246277, -1.529244065284729, -0.07265780866146088, 0.28017881512641907, 0.12041288614273071, 0.05106968805193901, 0.7975912690162659, -0.2710587680339813, -1.26042902469635, -0.5498237013816833, -1.0825496912002563, 0.9398345351219177, 0.4628385007381439, -0.41000664234161377, -0.9917135834693909, 1.0582265853881836, -1.710986852645874, -0.909640908241272, -1.015798568725586, 2.066270589828491, -0.23847244679927826, -0.8875304460525513, -1.5026015043258667, -0.15790347754955292, -0.7543395757675171, -1.7969175577163696, 0.7871437668800354, 0.33544594049453735, -3.354722023010254, -1.0726312398910522, -0.7118281126022339, -0.4558960795402527, -1.589382529258728, -1.291982889175415, -0.2483268827199936, -0.2580495774745941, 0.4809165596961975, 0.1429218351840973, -0.16591481864452362, 1.2247695922851562, -0.07174813002347946, 1.0648592710494995, 1.0536620616912842, 0.9459879994392395, 0.8076099157333374, 0.7468958497047424, 0.19309325516223907, 0.5897903442382812, -0.3556837737560272, -0.3494495749473572, -1.3785111904144287, 0.9303238987922668, 0.4298751652240753, 0.9799334406852722, -0.332830548286438, -0.7235985398292542, 0.31879380345344543, -1.4862390756607056], [-0.3642050623893738, 0.8918607234954834, 1.1014025211334229, 0.10492092370986938, -1.6957083940505981, 0.23122373223304749, 0.035632211714982986, 0.4835328161716461, 0.14770229160785675, 0.45489615201950073, 0.7863138318061829, 0.8495399951934814, -1.5280518531799316, 0.5457873344421387, -0.8198163509368896, -0.7161357998847961, 0.652144730091095, -0.3816497027873993, -0.5535739064216614, -0.34370213747024536, -1.7599046230316162, 2.497823476791382, 1.567496418952942, -0.2979956865310669, 0.1774071753025055, -0.5019416809082031, -0.04291978478431702, -0.2806733548641205, 0.6699947714805603, 1.406975269317627, -0.14711952209472656, 1.315799593925476, 1.269278645515442, 0.5418538451194763, -0.24597808718681335, 0.0423581600189209, -0.12281659990549088, -0.30752354860305786, 0.10154154151678085, 0.9462315440177917, -0.3834758698940277, 0.020361505448818207, 0.47719210386276245, -0.7820863723754883, -0.5977550745010376, 0.05171281099319458, -1.7869510650634766, 0.04346829652786255, -1.7655577659606934, 1.8650786876678467, 1.0277801752090454, 0.4164730906486511, -0.5968509316444397, 0.5933004021644592, 0.0258916225284338, 0.38704052567481995, -0.8293403387069702, 1.0076606273651123, -0.945378303527832, 1.0138403177261353, -1.273323655128479, -0.49111077189445496, -0.548160970211029, -0.3206290900707245, 0.540431797504425, 0.45998337864875793, -0.5485316514968872, -0.7126602530479431, 0.9834320545196533, -1.6663539409637451, 0.6147223711013794, -0.3491942584514618, 1.0109230279922485, 0.5453396439552307, 2.408939838409424, 0.8762232661247253, 1.8548377752304077, -0.3876637816429138, -0.7201331853866577, 0.5086321830749512, 1.01084566116333, -3.359968900680542, -1.1412510871887207, -0.557487964630127, 0.8979785442352295, -0.23814398050308228, 0.19552341103553772, 0.699687659740448, 0.886317253112793, -1.3950222730636597, 0.030175792053341866, -0.1814134567975998, 0.5978013277053833, 0.07338031381368637, -0.44843989610671997, 0.13899266719818115, -0.7136801481246948, 0.9794888496398926, 1.0871517658233643, -1.4254087209701538, 0.2911298871040344, -2.2645373344421387, -0.12628357112407684, 1.0329686403274536, -0.5941689610481262, -0.7776635885238647, -0.8893194198608398, -0.604411780834198, -0.9484230875968933, 0.21238088607788086, -1.6877081394195557, -1.2517962455749512, -1.2827401161193848, 1.0211817026138306, 0.05567455664277077, 0.7052598595619202, 0.39927616715431213, 1.3269661664962769, -1.0394240617752075, -1.59891676902771, 0.6533134579658508, -1.259236216545105, -1.4687855243682861, -0.34569820761680603, -0.43850693106651306, 0.5286256074905396, -0.4690081775188446, 0.26702892780303955], [-0.18075837194919586, -1.1552152633666992, 1.2476154565811157, -0.2393883913755417, -1.3081387281417847, 0.32082992792129517, 1.1577575206756592, 1.3366146087646484, -0.6749230027198792, 0.08456166833639145, 0.34638458490371704, -1.3064292669296265, -1.3157520294189453, -0.22275587916374207, 1.5294443368911743, -0.16647064685821533, -0.7307062149047852, -0.7181161046028137, -0.14082768559455872, -0.28308066725730896, -0.12136346101760864, 0.825066864490509, -0.7401139736175537, 0.9697120189666748, 0.6235195398330688, 0.15120971202850342, 0.8792194724082947, 0.6355419754981995, -0.2334539145231247, -1.4639137983322144, -0.7656736373901367, -0.8474270105361938, -1.8856613636016846, 0.007725880015641451, 0.5989628434181213, -0.7869800925254822, 0.2728240489959717, 0.673316240310669, 2.4244282245635986, 0.49208950996398926, 0.06742563098669052, 0.27256396412849426, -0.7057034373283386, -0.5995675921440125, 1.612841248512268, 1.9378973245620728, 0.0006111098336987197, -1.8737527132034302, 1.7916122674942017, 0.5946391224861145, 0.18545334041118622, 0.17608042061328888, -0.2049129456281662, 0.48355498909950256, -0.09067229926586151, -0.249428853392601, -0.6079823970794678, 0.3234964907169342, -0.4886132776737213, -0.4003585875034332, 0.7652459144592285, -0.298620343208313, -0.3105047345161438, 1.5645490884780884, 0.8766108751296997, 0.21585264801979065, 0.32017549872398376, -0.5855024456977844, -1.3303524255752563, 0.3740740716457367, -0.3216126263141632, 0.2826957106590271, -1.1066092252731323, -1.5986605882644653, -0.5163409113883972, -0.7022867798805237, -0.3307170867919922, 1.3489736318588257, -0.5803616046905518, -0.1210077702999115, -0.9748380780220032, -1.4213422536849976, -0.5469187498092651, 0.44098415970802307, -0.6803067326545715, -0.9728977084159851, -0.030980052426457405, 0.04107256233692169, -1.7661712169647217, 0.4734082520008087, 0.2976859509944916, -0.3388669490814209, 0.2677479684352875, 0.24836456775665283, 2.651690721511841, -1.297838807106018, 0.59771329164505, -0.12956281006336212, -1.1346068382263184, -0.7946326732635498, -0.5609453916549683, -0.050965938717126846, -0.2293090671300888, 0.7867798209190369, 0.1015087366104126, 0.7365343570709229, -0.018040666356682777, -1.3006538152694702, -1.3951561450958252, -1.291175127029419, -1.7341129779815674, -2.090724468231201, 0.7786068320274353, -0.5966225862503052, -0.9022120237350464, -0.5644835233688354, -0.23176483809947968, -1.6665332317352295, -0.039234716445207596, -0.2929723858833313, -0.27323487401008606, -1.3170316219329834, -0.4781893789768219, -0.4263822138309479, 1.3750733137130737, -0.16627472639083862, 0.37075158953666687, 0.07337221503257751], [-0.7228338718414307, 0.4290429949760437, -0.24990123510360718, 0.056407324969768524, -0.684170663356781, 2.0762107372283936, -1.2224323749542236, 1.212095022201538, 0.04470253735780716, 0.9426518082618713, -0.726190447807312, -0.7562248110771179, -0.9522578120231628, -0.5763408541679382, 0.5743377208709717, 0.060134049504995346, -0.815482497215271, -0.16339907050132751, -1.6089035272598267, -1.0530962944030762, 0.80397629737854, 1.294310212135315, 0.04145164042711258, 0.05540020018815994, 0.43434447050094604, 1.2200541496276855, 0.8187593221664429, 0.7165545225143433, 0.8594781160354614, -0.18071220815181732, 0.8410778045654297, 1.4547511339187622, -0.07367227971553802, -0.7528087496757507, -0.1767112761735916, -2.379077434539795, -0.024828750640153885, -0.09652683138847351, -0.44458749890327454, 0.20712615549564362, 0.24730664491653442, -1.2288570404052734, -0.7233604788780212, 0.5738692283630371, 0.3480352759361267, -1.1940757036209106, 0.00333898956887424, 0.1733754426240921, -1.7423659563064575, 0.09930140525102615, -0.9274579882621765, 1.3310035467147827, -1.4020963907241821, -1.207546353340149, 1.7421045303344727, -0.5239306092262268, 0.38113072514533997, -0.14030490815639496, -1.9813159704208374, 0.7376054525375366, 1.293673038482666, 0.4207283854484558, -0.9056480526924133, 0.5089423060417175, -0.3942617177963257, 0.12087056040763855, -0.267652690410614, 0.35473623871803284, -0.5036453008651733, 1.3542243242263794, -1.5252877473831177, -1.2670698165893555, 1.3253843784332275, 1.379814624786377, 0.8918910622596741, 0.6159023642539978, 2.0332682132720947, -0.050009585916996, 0.7958296537399292, -1.8842494487762451, 1.1452585458755493, 1.0799744129180908, -0.8076341152191162, 1.6270347833633423, 0.6158432960510254, -0.8485251069068909, 1.0042123794555664, -0.8423710465431213, 0.36483249068260193, 0.11177323013544083, -0.09783174842596054, -0.46833693981170654, -0.49844974279403687, -0.4269031286239624, 0.07384753227233887, 1.2680362462997437, 0.3062341511249542, -0.5263457894325256, 0.05242112651467323, -1.6276259422302246, -0.7736378312110901, -2.7192184925079346, -0.18524549901485443, -0.2057746797800064, 0.4529813528060913, 2.3190596103668213, -1.4135764837265015, 1.148343563079834, -0.7612844109535217, 0.9080901741981506, 1.0691348314285278, -0.7796432375907898, 0.6718202233314514, 0.7167635560035706, 0.6719757318496704, 0.8155499696731567, -0.0736442431807518, -1.1028419733047485, -0.5692880749702454, -0.6527979373931885, -0.9298956990242004, -0.9288349747657776, 0.2061293125152588, 0.38358575105667114, -0.2792827785015106, -0.618334174156189, 1.1822359561920166, 0.532854437828064], [0.25007864832878113, -0.5206233263015747, 1.0800420045852661, -1.2127422094345093, 0.48673903942108154, -1.5227309465408325, -0.6748208403587341, -1.110503911972046, -0.5680015683174133, 0.756248950958252, 0.08959007263183594, -1.0417453050613403, -0.7291980981826782, 1.0078152418136597, -0.4218031167984009, -0.6333810687065125, 0.9652929902076721, 0.8242243528366089, 0.5266627073287964, 1.3006278276443481, 2.8741016387939453, -0.23304730653762817, 1.4691046476364136, 0.625775158405304, 1.5468668937683105, -0.617637038230896, 0.2374972254037857, 0.13423697650432587, -0.14303752779960632, -0.2275102287530899, 0.9406168460845947, 0.49017274379730225, -0.6878593564033508, -0.5220850706100464, 0.6534568071365356, 1.5729856491088867, 0.6859768629074097, -0.4107462763786316, -0.27314841747283936, -0.15342235565185547, -0.36164993047714233, 0.8813027739524841, -1.466370940208435, -0.9707542061805725, -1.1147513389587402, -1.2144768238067627, 2.8266289234161377, 0.24953755736351013, -0.3179033696651459, 1.0491544008255005, -0.4253138601779938, 0.4908059537410736, -0.5241073369979858, -0.5280212759971619, -1.5339961051940918, 0.5700079798698425, -1.6502187252044678, -1.5476741790771484, -0.6417062878608704, 1.0741095542907715, 0.3727647364139557, 1.5391772985458374, -0.03447066247463226, -0.42556068301200867, -1.19495689868927, -1.3159842491149902, -0.08756152540445328, -1.2384161949157715, 1.3719773292541504, 0.08616327494382858, -0.5210302472114563, -0.009809902869164944, -0.096157006919384, -0.3606466054916382, 0.36632537841796875, -1.328033685684204, 0.3486107885837555, 0.038320932537317276, 0.577238917350769, -0.7241696119308472, -0.46735718846321106, 0.9986860752105713, -0.7978129982948303, 0.9306080341339111, 0.663975715637207, 0.9299127459526062, -0.14552609622478485, 0.3177701532840729, -0.24527743458747864, -1.103980302810669, 1.04805588722229, -0.6206135749816895, -0.029794711619615555, 0.5135818123817444, 1.6177399158477783, -0.11624076217412949, -0.06294867396354675, 0.5883307456970215, -0.483058363199234, -0.3355151116847992, 0.258923202753067, -1.0414177179336548, -1.1509078741073608, 0.21107855439186096, 1.6630440950393677, 1.2392605543136597, -0.5789041519165039, 0.665418803691864, 0.7873919606208801, -0.11110841482877731, -0.17423675954341888, -0.0940956100821495, -1.8788989782333374, 0.6860837340354919, 0.3462821841239929, -0.10822582989931107, -0.6822536587715149, -0.19721719622612, -1.1474074125289917, -0.3016039729118347, -1.0784915685653687, 0.6532345414161682, 0.009383110329508781, 0.6293533444404602, 0.36611172556877136, -1.3951431512832642, -0.22132690250873566, 0.6637972593307495], [1.0345160961151123, -0.7029151916503906, -1.449015736579895, -2.0100693702697754, 0.7705094814300537, -1.1796619892120361, -0.07492417842149734, -0.11050130426883698, 0.3316018581390381, -0.5750051140785217, -0.8052307963371277, -0.5418902039527893, -0.1060701459646225, 1.1625720262527466, -2.510850667953491, -1.0579644441604614, -0.5037018656730652, 0.9145151376724243, -0.5765073299407959, -1.4585837125778198, 1.3079253435134888, 0.29626330733299255, 0.27556803822517395, -0.0346316322684288, 2.968885660171509, -0.4474763870239258, -0.7442400455474854, -0.15839549899101257, 0.36900973320007324, -1.7807183265686035, 0.8907408714294434, 0.8282129764556885, 1.2401278018951416, 0.42903679609298706, 0.7326138019561768, 0.2224249243736267, 0.3014664649963379, 0.7697083950042725, 2.1490557193756104, -1.9319146871566772, -2.628915786743164, -0.9196813106536865, 1.0340238809585571, 2.661298990249634, 1.7585890293121338, 0.16859062016010284, -0.7754116058349609, -0.1795833557844162, 0.8949164748191833, -1.7152701616287231, -0.08680885285139084, -1.1828826665878296, 1.424387812614441, 1.7174818515777588, -0.3533131778240204, -1.4326273202896118, -1.0778330564498901, 0.02062753587961197, 1.1560678482055664, -1.1990506649017334, 0.9537563920021057, 0.024908073246479034, 0.07109005749225616, -1.4999034404754639, -0.8302112221717834, -1.1677545309066772, -0.5964097380638123, -0.3380226790904999, -1.2859768867492676, 1.3518152236938477, -0.3328901529312134, 0.7369962334632874, 0.12072016298770905, 0.24182714521884918, 1.742351770401001, 0.6146605014801025, 0.2951231002807617, -0.3971552550792694, 0.4407522678375244, 1.3239012956619263, 0.42666035890579224, -0.7136914730072021, -0.32747945189476013, 1.1422713994979858, 1.3771579265594482, -2.4375858306884766, 0.8727623820304871, -0.9424575567245483, -0.7496774792671204, -1.043174386024475, 0.15456177294254303, 0.14411988854408264, 0.3091101050376892, 0.6790308356285095, 1.6784281730651855, 1.2040694952011108, 0.47092747688293457, 1.2357053756713867, 0.37785542011260986, 0.36044076085090637, -0.9312777519226074, -0.22871142625808716, 3.0281050205230713, -0.027650773525238037, -0.9159150719642639, 1.1630812883377075, 1.8005521297454834, -0.18026696145534515, -0.2488214075565338, 0.4777429699897766, -0.6636210680007935, 2.2209911346435547, -0.33081671595573425, 1.8580683469772339, 0.7743390798568726, -0.26435306668281555, 0.5467543005943298, 0.14427512884140015, 0.02613675221800804, -0.46602702140808105, 0.5159659385681152, -1.8127301931381226, 0.15065312385559082, 0.42695197463035583, 0.34916406869888306, -1.0999810695648193, -0.06491435319185257, 0.4301350712776184], [-0.7138373255729675, 0.8985629081726074, 0.36619648337364197, -0.40464386343955994, -0.7835190892219543, -0.5881392955780029, -0.7775647640228271, -0.04367884248495102, -1.4308079481124878, 0.7039965987205505, -1.0171197652816772, 1.9066784381866455, -0.6954113245010376, 0.7718522548675537, -0.2972131669521332, 0.5514988899230957, 1.745591163635254, -2.0780351161956787, 0.2655455768108368, 0.6082578897476196, -0.40475380420684814, 1.448634386062622, 0.9991030693054199, 0.7565971612930298, -0.014954835176467896, -0.7287941575050354, -0.5705578923225403, -0.43260079622268677, 0.4114123284816742, -1.0174837112426758, -1.4815841913223267, -0.4620262086391449, 0.5583934187889099, -0.8860387206077576, -0.729969322681427, 1.2778156995773315, 0.1538163721561432, -0.3754500150680542, 0.3438607156276703, 0.905634880065918, -1.8845844268798828, 1.2533842325210571, 1.5027378797531128, 0.39976510405540466, -0.6025615930557251, -1.566540241241455, -0.22975586354732513, 1.6385842561721802, 1.601532220840454, 0.4435330629348755, 0.7384047508239746, -2.1453776359558105, 1.4365154504776, 0.13205036520957947, -0.22051860392093658, -0.8057955503463745, 0.5543657541275024, -0.06326046586036682, 1.457468032836914, -0.6115310788154602, 1.3398845195770264, -0.6837214231491089, 0.6659744381904602, -0.15738330781459808, -1.9930264949798584, -1.140009880065918, 0.646162211894989, 0.7636879682540894, 0.20585328340530396, 0.37749597430229187, -0.2082623541355133, -0.8237892389297485, -0.05765915289521217, 1.4784669876098633, 1.2122377157211304, -1.265486478805542, 0.7125622630119324, 0.6003449559211731, -0.10936595499515533, -0.20802897214889526, -0.7231214642524719, 0.17525193095207214, 0.27069810032844543, 0.03655868023633957, -0.4468819200992584, 0.6785078644752502, 0.3645686209201813, 0.366775244474411, -0.40216368436813354, 1.8612799644470215, 0.38173550367355347, -0.3605223000049591, -0.548015832901001, 1.941380500793457, 0.059813134372234344, 0.42010563611984253, 0.8814806938171387, -2.2509329319000244, 0.48492464423179626, 2.2512965202331543, 0.401838093996048, 0.6187499165534973, -2.385713577270508, 0.1590552031993866, 0.887933075428009, -0.4859393835067749, -0.007158899214118719, 1.610772967338562, -0.4004013240337372, -0.6562118530273438, 0.8250229358673096, -0.38141757249832153, -0.21841251850128174, -0.34906819462776184, -0.8546259999275208, 0.25674405694007874, -0.15673144161701202, 1.5557470321655273, 0.2356998473405838, 1.405366063117981, -0.7054730653762817, -2.8812267780303955, 0.7635289430618286, -0.5683350563049316, -0.23634356260299683, 0.7515131235122681, -0.10123033076524734, -0.6368334293365479], [-0.09688937664031982, -1.264306664466858, 0.24498139321804047, 0.14091931283473969, -1.026837706565857, 0.7007052898406982, -0.800110936164856, -0.1755833923816681, -1.4386042356491089, -1.5325075387954712, -1.6710213422775269, -0.5945645570755005, 0.04289952665567398, -0.1672159880399704, 0.7180605530738831, 0.6755856275558472, -0.2861538529396057, 1.5772780179977417, 0.48071131110191345, 1.3750650882720947, 0.5802878737449646, -0.13920745253562927, 0.29769396781921387, 0.06354415416717529, 0.28152018785476685, 0.022061236202716827, -1.2640304565429688, 0.20989663898944855, -1.771133303642273, -0.5344191789627075, -0.8474499583244324, 1.1161566972732544, -0.37592580914497375, 1.337342619895935, 1.0375972986221313, 1.0206965208053589, 0.4463522434234619, 0.5394230484962463, -0.2063807249069214, -1.8182893991470337, -0.01664457656443119, -0.6753355860710144, -0.554753839969635, 0.27896416187286377, -0.49199503660202026, 2.188987970352173, -1.192238211631775, -0.07650479674339294, 0.9595559239387512, -0.8532639145851135, -0.06707105785608292, -0.5998110771179199, -0.9102367162704468, -0.8798350691795349, 0.36818358302116394, -0.6664880514144897, 0.9380871057510376, -0.6765561103820801, 0.9542500972747803, -0.9451412558555603, -0.28756698966026306, -0.36175137758255005, -0.09219199419021606, 0.6292813420295715, 0.8518858551979065, 3.009636878967285, 0.6435562968254089, 0.47414547204971313, -1.7234177589416504, -0.8813554644584656, -0.34218764305114746, 0.6186651587486267, -0.04341837391257286, -1.553114891052246, -0.33804136514663696, -1.291905164718628, 1.5069630146026611, 0.14736251533031464, -0.38451334834098816, -0.3020496070384979, -0.0625334233045578, 0.2742874026298523, 1.2307612895965576, -0.43513715267181396, -1.0350896120071411, 1.4601246118545532, -0.4014342129230499, 1.1025621891021729, 0.9038759469985962, 0.4540729224681854, -0.3194378614425659, 1.0285735130310059, -1.0166774988174438, 2.5738131999969482, 0.1781255006790161, 0.40634647011756897, -0.7357560992240906, 0.20624135434627533, 1.7552803754806519, 0.05183342099189758, -1.9809612035751343, -1.148607611656189, -0.5036087036132812, -0.17625205218791962, 0.9961512088775635, 1.2422292232513428, -0.4475887715816498, 0.9563677310943604, -1.3483177423477173, 1.0806515216827393, 1.475245475769043, -0.9185391664505005, 0.015623752027750015, 0.6592830419540405, -1.4543567895889282, -0.41187191009521484, 0.19332142174243927, -0.04811776056885719, -0.2043643593788147, 0.4948544502258301, 0.26756104826927185, -0.3260943293571472, 0.809264600276947, 0.14271773397922516, 0.6133323311805725, -1.574177861213684, -0.5146463513374329, 0.3887926936149597], [-1.7185343503952026, 0.5844621062278748, -0.8775491118431091, -0.9060021042823792, 0.9355549812316895, -0.8287932276725769, 2.0018880367279053, 1.4860100746154785, -0.10850018262863159, -1.2836382389068604, 1.1523852348327637, -0.34221595525741577, -0.5140712261199951, 0.39305219054222107, -0.15973889827728271, -0.15592598915100098, -0.27891725301742554, 1.516549825668335, -1.3101201057434082, 1.2866438627243042, 1.1570771932601929, 0.3679156005382538, -1.466030478477478, 2.9171266555786133, -2.0120997428894043, 1.3334178924560547, 1.7600600719451904, 1.093946099281311, -0.933985710144043, -1.1789671182632446, -0.6179296970367432, 0.18441557884216309, -0.9923132658004761, 0.25537267327308655, 0.8299576640129089, 1.1909537315368652, 0.07695471495389938, -0.41346997022628784, -0.9293057918548584, -0.7688480019569397, -0.5010169148445129, -1.2254951000213623, -1.067701816558838, -1.0524705648422241, -0.29948553442955017, -1.1683564186096191, 0.10198213905096054, -1.2860554456710815, -0.8552472591400146, -1.2478023767471313, 0.4822794198989868, -0.2991820275783539, 0.001140450476668775, 0.3256991505622864, -0.7244772911071777, -0.089279405772686, -0.24408994615077972, -1.2022548913955688, -0.8876194953918457, -2.581556558609009, -0.9789288640022278, -1.921307921409607, -0.27311041951179504, 0.06657418608665466, 1.5563528537750244, -0.24937748908996582, 2.0386626720428467, 1.5623770952224731, 2.1133744716644287, -0.17572590708732605, -0.3400711715221405, -0.5340999364852905, -1.02240788936615, -0.16679827868938446, -1.484224796295166, -0.5412599444389343, 0.12359382957220078, -2.0279219150543213, 0.050866883248090744, 1.1141324043273926, -2.4410979747772217, -1.3774001598358154, 1.6040160655975342, 0.747119128704071, 1.3956992626190186, 0.4517519772052765, -1.1935267448425293, -0.4628300070762634, -0.07814965397119522, 0.4725605845451355, -1.0870897769927979, -0.7274201512336731, 1.1353826522827148, -0.04318058490753174, -0.64493328332901, 1.663928508758545, 1.8202629089355469, 1.1922061443328857, 0.7770865559577942, 1.0971083641052246, -1.3853225708007812, -0.6857210397720337, -0.7403590083122253, -1.3379288911819458, 0.5647596716880798, 0.19613999128341675, 0.14265644550323486, -0.9768009185791016, 0.7845420241355896, 1.684813141822815, 0.44733887910842896, 1.7887730598449707, -1.2149112224578857, -0.21580037474632263, -1.5134403705596924, 0.9587295651435852, -0.8458865880966187, -0.011496414430439472, -0.6939015984535217, -1.7875211238861084, -0.12409254163503647, -1.1655582189559937, 0.6274858713150024, 0.03899960592389107, -0.288043737411499, 0.8395183086395264, -1.8537253141403198, 0.24295099079608917], [1.4845466613769531, -1.4530746936798096, -0.9426484704017639, 0.23694391548633575, 0.9656118154525757, 0.176759272813797, 0.3898359537124634, -0.6353397369384766, 0.5915883779525757, 0.443417489528656, -0.38048025965690613, -0.8610967993736267, -1.0824774503707886, 0.839651882648468, -0.16093182563781738, -0.7829784750938416, -0.5278117060661316, 1.799862265586853, 1.0645958185195923, 0.6112946271896362, 0.26282158493995667, 2.0345444679260254, 0.6807430386543274, -0.7526203989982605, 2.2179481983184814, 1.7970948219299316, -0.353913813829422, -0.481202632188797, 0.8935627341270447, -1.3539061546325684, 0.39793872833251953, -0.4200199246406555, -0.7088749408721924, 0.9843339920043945, 0.4536096155643463, -0.6725768446922302, -1.0898109674453735, -0.13826435804367065, -0.10730469226837158, 1.3112492561340332, -0.3618951439857483, -1.4719947576522827, -1.4599004983901978, 0.7920001745223999, 0.579193651676178, 0.2896445095539093, 0.5178086757659912, 1.842644214630127, -2.3152244091033936, 1.359492540359497, 0.02990475296974182, -0.3232317864894867, 0.4369351267814636, 0.04546542838215828, 2.217810869216919, -0.4666697084903717, -3.1018831729888916, -0.4286099076271057, -0.9137938022613525, 0.843999445438385, -0.07645364850759506, 0.501216471195221, -0.5558485984802246, -0.2595585286617279, -1.4968712329864502, 1.2517229318618774, 0.26343873143196106, -0.7258853912353516, -1.0129332542419434, 1.4682480096817017, -0.6915336847305298, 0.5616964101791382, 1.0386688709259033, -0.47512301802635193, 1.2938302755355835, 0.14850430190563202, 1.3071538209915161, 0.6170454025268555, -1.171161413192749, -0.5649974942207336, -0.15704253315925598, 0.891512393951416, -0.9881414175033569, 1.0628433227539062, 0.5757375359535217, 0.512172520160675, -0.4478548467159271, 0.7802630066871643, 0.5746827721595764, -0.8751661777496338, 0.8525634407997131, -0.5121864080429077, -0.18642285466194153, -0.6444066762924194, 0.11146131902933121, 1.204604983329773, -1.2900722026824951, 0.09841455519199371, 0.24788881838321686, 0.8802076578140259, 1.0108563899993896, -0.24667778611183167, -0.27633872628211975, -0.6938880681991577, 0.5357847809791565, 0.5455586910247803, 2.2959558963775635, -1.5047568082809448, -0.03527827188372612, 1.1185301542282104, -0.6973056197166443, -0.8170482516288757, -0.3614473044872284, -0.5785354375839233, -0.5154195427894592, -0.31820547580718994, -0.12279139459133148, -1.5817084312438965, 0.4919084310531616, -0.15543842315673828, -1.0762524604797363, 0.5970404148101807, 0.6565452814102173, -0.23714391887187958, 0.3407619595527649, -0.35659173130989075, -0.4738559126853943, 0.11374816298484802], [0.17376987636089325, 0.1580846607685089, 0.5369531512260437, 0.17086957395076752, 1.917181134223938, -0.3635421395301819, 0.4010249972343445, -1.1129746437072754, 0.4617047905921936, -1.5525188446044922, 0.47542086243629456, -1.3314348459243774, 0.004756197798997164, -1.6629509925842285, 0.5506753325462341, 0.7711108326911926, 1.2861182689666748, -0.36761078238487244, 0.8689521551132202, 0.910839319229126, 0.7091551423072815, 0.576958954334259, 0.2937423586845398, -0.1728540062904358, 1.2313263416290283, 1.3757281303405762, -1.194831132888794, -0.38441574573516846, 0.19398513436317444, 0.3847164809703827, -0.8599063754081726, 0.6681845188140869, -0.3738495707511902, 0.18332621455192566, -0.709144115447998, 0.38466310501098633, -0.7188466191291809, 1.2730659246444702, -1.4528820514678955, -1.6321865320205688, 0.603732705116272, -0.7228405475616455, 1.7400864362716675, -1.4799222946166992, -1.2969746589660645, 1.2559244632720947, 0.10140058398246765, 0.4239143133163452, -0.8862611055374146, -0.4245227873325348, 1.1845709085464478, -0.27009883522987366, 1.4610819816589355, 0.04990030825138092, 0.37511807680130005, 0.2323097586631775, 2.293651580810547, 1.26498281955719, 1.0580592155456543, -0.101500004529953, 0.17996907234191895, 0.8118654489517212, 0.15577174723148346, -0.4946209192276001, 1.2887860536575317, -0.4628288447856903, -1.529442310333252, 0.6175712943077087, 1.050909399986267, -1.5379388332366943, 0.024374326691031456, -0.36196985840797424, -1.4803224802017212, -1.0869327783584595, 1.73381769657135, -2.0353424549102783, 0.18663741648197174, 1.1139402389526367, 0.3079702854156494, 0.5084275007247925, 0.4789709150791168, 0.7044499516487122, 0.28905561566352844, 2.045149087905884, 0.036744460463523865, -1.2606717348098755, 0.566913902759552, -0.5359453558921814, 1.0197241306304932, 1.433053731918335, 1.4687778949737549, 1.0596227645874023, 0.9774709343910217, 0.4355272352695465, 0.5172345042228699, 0.8329154253005981, 0.8603315949440002, 0.9293287396430969, 0.3063989579677582, -0.4934452474117279, -1.3796535730361938, 0.8659892678260803, -0.5762650370597839, 0.6901881098747253, -1.0803841352462769, 1.421047568321228, -2.121659278869629, 1.2109570503234863, 0.8720601201057434, -0.1506071537733078, -0.11866500228643417, 2.421459913253784, 0.49739518761634827, 0.5933446884155273, 1.242518663406372, 0.3932533860206604, -0.5094192028045654, -0.011867410503327847, 1.4377708435058594, 0.21416926383972168, -1.6539820432662964, -1.742102026939392, -0.43879246711730957, 0.1008719801902771, 1.299628734588623, -1.1905733346939087, 1.603270411491394, 0.30460163950920105], [-0.6845543384552002, 0.49784258008003235, 0.22129559516906738, -0.1487741321325302, 0.7587487697601318, 1.2098267078399658, -1.8922629356384277, -1.1132500171661377, -0.9334916472434998, -0.22374282777309418, -0.7649292945861816, -0.9155425429344177, -0.6826375722885132, 0.3786104619503021, -1.8495533466339111, 0.19579480588436127, 0.0893591120839119, 1.2777765989303589, -1.6450018882751465, -1.005244493484497, -1.609052062034607, 0.17998048663139343, -0.34902188181877136, -0.23516187071800232, 0.4440125524997711, -0.6350281238555908, -0.10114266723394394, 0.44855430722236633, -0.222991943359375, -0.6534389853477478, 0.1146698072552681, -1.0179693698883057, -0.6506587862968445, -1.4808205366134644, -0.7273284196853638, -0.5303381085395813, 1.097245454788208, -0.740712583065033, -1.411265254020691, 0.18068476021289825, -0.8847800493240356, -0.9521703720092773, -0.4409381151199341, -1.060434341430664, 1.6519312858581543, 0.618534505367279, -1.998105525970459, -1.0281435251235962, -0.7912125587463379, -0.06229032948613167, 1.7959293127059937, 0.3852284550666809, 1.0819623470306396, 0.17312434315681458, 0.7439590692520142, -1.8985117673873901, 1.7053428888320923, 0.5221513509750366, 0.8378278613090515, 1.718919038772583, 0.7585578560829163, 1.3296281099319458, 1.6906591653823853, 1.6011286973953247, -0.648047924041748, 2.493595600128174, -2.0702245235443115, 0.9974349737167358, 2.2171456813812256, -1.0631729364395142, 0.8296155333518982, -1.5520992279052734, -0.28263622522354126, -1.6114857196807861, -1.020833134651184, 0.7756467461585999, -1.0292874574661255, 2.5151567459106445, 0.32551711797714233, 1.394582748413086, -0.31862446665763855, 0.32343119382858276, -0.8986349701881409, -0.6190289855003357, -0.5393344759941101, 0.36554527282714844, 1.2339025735855103, -1.3300849199295044, 0.051398638635873795, -0.5221713781356812, 0.6351902484893799, 1.5711917877197266, 0.2680946886539459, 1.2330189943313599, -0.622198760509491, 0.49186456203460693, 1.2018132209777832, -0.9054999351501465, -1.117488980293274, 0.6146106123924255, 0.27383366227149963, 2.314361095428467, 0.5087080597877502, 0.22472068667411804, -2.067610263824463, -0.3100586533546448, 1.2246747016906738, -0.3442760705947876, -0.8765071630477905, -1.7080018520355225, -0.2692205011844635, -0.5088642239570618, 0.7473351955413818, 0.02137712575495243, -0.5727624297142029, 0.08251345157623291, -1.7070305347442627, -1.478298544883728, -0.5466018319129944, -1.135042428970337, 0.6274139881134033, 0.6206240653991699, -0.482145220041275, -0.9838913083076477, -0.1773221790790558, 0.10462582856416702, 1.3260674476623535, 1.2081544399261475], [1.3773268461227417, 2.1752240657806396, -1.046521782875061, -0.3415161669254303, -1.0063663721084595, 1.2561379671096802, -1.6497786045074463, 1.2990227937698364, -1.9099498987197876, -0.19120053946971893, 0.008939802646636963, -2.015219211578369, -0.02930486388504505, 0.622725784778595, -0.4436260759830475, -0.47351452708244324, -1.1779649257659912, 1.2443203926086426, -0.7036229372024536, 0.18461941182613373, 1.5133318901062012, 1.2194904088974, 0.6256777048110962, -0.03711920976638794, -1.8381761312484741, -2.326237678527832, -2.524754762649536, -1.7768480777740479, -0.5841582417488098, -0.9992542862892151, 2.0314106941223145, 1.3791248798370361, 0.7448453307151794, 0.2475033849477768, -0.3611724078655243, -0.04773669317364693, -0.38637420535087585, -0.4039194583892822, -2.4197397232055664, -0.3782009780406952, -0.18960747122764587, -1.4309933185577393, -0.0932963415980339, 0.4116743206977844, -0.13791970908641815, -0.8976038098335266, -1.1172966957092285, 0.023387176916003227, -0.3896160125732422, -1.9413474798202515, -0.9941412806510925, -0.3068968653678894, 1.135347604751587, -0.5886215567588806, -1.2465243339538574, -0.5215175151824951, -0.10095544159412384, 0.386417418718338, 1.6972429752349854, -2.1533100605010986, -0.2042846977710724, -1.3724452257156372, -1.5113399028778076, 1.0889568328857422, -1.2705743312835693, 1.1060484647750854, 0.13704434037208557, 0.361403226852417, 0.1954011470079422, 0.18845081329345703, 1.0622386932373047, -1.7630128860473633, 0.01083091739565134, 1.3738547563552856, 1.347135066986084, -0.4411231279373169, 0.06259598582983017, -1.3079397678375244, -1.6701401472091675, 0.9635195732116699, 0.0178348608314991, 0.15693651139736176, 1.2086155414581299, -1.4792628288269043, -0.048429038375616074, 0.3728458285331726, 0.43081629276275635, -0.5736127495765686, -0.31761524081230164, 1.1038638353347778, 0.7034063935279846, -1.6030007600784302, -1.3630590438842773, 1.3119580745697021, 0.060519035905599594, 0.7696927189826965, 0.775582492351532, 0.06783227622509003, 0.8869290351867676, 0.05154141038656235, 0.12972339987754822, -0.08477922528982162, -0.2672257721424103, 0.5143044590950012, -2.198030710220337, -0.11683865636587143, 0.3699733316898346, -1.6345982551574707, -0.4941008388996124, 0.3729454278945923, -0.09071226418018341, -0.7022407650947571, -0.14657072722911835, -1.1729644536972046, 1.0998764038085938, -1.7045258283615112, -0.6581389904022217, 1.5795241594314575, -0.949862539768219, -0.3694508969783783, 0.3135393261909485, -0.22154587507247925, 0.11275588721036911, -0.9261012673377991, -0.38855239748954773, 0.8780120015144348, 1.4631476402282715, 0.28366759419441223], [-0.9834604859352112, -2.3336567878723145, 0.4193167984485626, 0.24569308757781982, 0.13349762558937073, -0.3843420743942261, 0.07300878316164017, 0.8990660905838013, -0.6774271130561829, -0.6502034664154053, -0.2373550534248352, 1.361519694328308, 2.08439040184021, 1.4296659231185913, -0.7816455960273743, 1.0718681812286377, -2.3401143550872803, 0.37882179021835327, 0.8000146746635437, -0.4827513098716736, 0.7549057602882385, -0.6342306137084961, -0.27640172839164734, 1.1488677263259888, 0.8749376535415649, 0.21094933152198792, 0.606374979019165, 1.3787471055984497, 0.2132641077041626, -2.0014679431915283, -0.17041924595832825, 0.5674716234207153, 1.275861144065857, -1.7902277708053589, 0.695401668548584, -0.20972543954849243, -0.44397035241127014, -1.6710647344589233, 0.27692726254463196, 0.9893659949302673, -0.8787100315093994, 0.1558234542608261, 1.2221882343292236, -0.6944878101348877, -0.1690809279680252, 0.5651835799217224, -0.0012261237716302276, 0.5206515789031982, 2.4400687217712402, -0.6768484711647034, -1.4354472160339355, 0.8449659943580627, 2.722135066986084, 0.4241791069507599, 0.84124356508255, -0.8227185606956482, 1.5462783575057983, 1.4197418689727783, 0.2769899070262909, 0.47100257873535156, -0.05282631516456604, -1.6709059476852417, 0.4411802589893341, 1.3528560400009155, 0.02749902568757534, 0.6499047875404358, -0.4607563316822052, 1.3092573881149292, -1.3825801610946655, 0.8462154865264893, -0.3200109302997589, 0.15624700486660004, -1.061739206314087, 0.9415487051010132, -0.770645022392273, -0.35085567831993103, 0.001559785334393382, -1.5955995321273804, -1.2538174390792847, 0.7487775683403015, -0.8240882158279419, -0.08713414520025253, 0.18285688757896423, 1.9659730195999146, -0.19404913485050201, 1.7424898147583008, 0.8117782473564148, -0.12178255617618561, 0.2645409107208252, 0.009816095232963562, 1.5998953580856323, -1.4677910804748535, -0.1664552539587021, 0.7303770780563354, 0.22212092578411102, 0.558464527130127, 0.8870834708213806, -0.3580911457538605, 1.6907120943069458, -1.1479277610778809, 1.2774040699005127, -2.0088517665863037, -0.6553175449371338, 2.3460423946380615, 1.5107331275939941, 1.4942657947540283, -0.563246488571167, 0.8437086343765259, 0.06597395241260529, -0.5744007229804993, 0.23757004737854004, 0.861303985118866, 0.6270812153816223, 1.2544721364974976, -1.5321685075759888, -0.14801329374313354, 0.7680618762969971, 0.43849918246269226, 0.5170113444328308, -0.40001505613327026, 0.8487398028373718, -1.4836056232452393, 0.37993523478507996, -0.2011246234178543, 2.215423345565796, 0.10455437004566193, -1.2064741849899292, -1.1203950643539429], [-0.07198012620210648, -0.11586503684520721, -0.17835181951522827, -0.14569588005542755, 0.3835071623325348, -2.4300506114959717, 0.21968267858028412, -0.04186583682894707, -0.9707732200622559, 1.216091275215149, 0.5643820762634277, -0.09939154982566833, -0.8263343572616577, 0.09913605451583862, 1.1992714405059814, -0.05678873881697655, -0.24979643523693085, -0.2199866622686386, 0.5691598057746887, 0.8371328711509705, 0.5877779126167297, -0.513678252696991, -0.4921039044857025, 0.49139392375946045, -1.0287630558013916, -1.8668051958084106, 0.8744378685951233, -0.333961546421051, 0.44632527232170105, -0.837960422039032, 0.3945518732070923, 0.6819336414337158, -2.2722439765930176, 0.678037703037262, 2.8997693061828613, -0.14720243215560913, -2.264146566390991, -0.20033790171146393, 0.27321580052375793, -0.24112479388713837, 0.9582080841064453, -1.0982897281646729, 0.5028430819511414, -0.47791212797164917, 0.3183114230632782, -0.5631192326545715, 1.5141524076461792, -0.4536619782447815, -1.3608969449996948, -0.24545344710350037, -0.5233398079872131, -1.512007474899292, 0.9301827549934387, 1.6098934412002563, 1.3684805631637573, -0.4411427974700928, 1.3567701578140259, 0.169564887881279, -1.2595109939575195, 0.7571480870246887, -1.3917840719223022, -0.22425654530525208, 1.0064656734466553, 0.7725038528442383, 0.4659383296966553, -1.0869358777999878, -1.4884997606277466, 1.0201724767684937, -0.008903112262487411, -0.48632460832595825, 1.7358312606811523, -0.4603421688079834, 1.8572722673416138, 0.8712555766105652, 0.9593076705932617, -1.201663613319397, 1.2475271224975586, 0.13113415241241455, 0.41865575313568115, -0.37075406312942505, -0.10278232395648956, 0.34839192032814026, -0.32294541597366333, -0.34368905425071716, -0.36852625012397766, -1.2405906915664673, -0.1577458679676056, -0.2521861493587494, -0.389801561832428, 1.2084959745407104, 0.6914556622505188, -0.07027438282966614, 0.23623962700366974, 0.6057632565498352, -0.5747313499450684, 1.5970816612243652, 1.0524792671203613, -0.26888760924339294, -1.7856218814849854, 0.955148458480835, -2.7033374309539795, -1.4956785440444946, -1.4592466354370117, -0.5496740937232971, -0.7547287344932556, 0.43884676694869995, 0.910227358341217, 0.14168724417686462, -0.25888800621032715, 0.4109335243701935, 1.2121528387069702, -2.1053812503814697, -0.9513780474662781, -0.45435473322868347, -0.3897199034690857, -0.23108187317848206, 0.6846364736557007, 0.7644644975662231, 0.7729949951171875, -0.6120282411575317, -1.602647304534912, -1.185451626777649, 2.179189443588257, -0.17947377264499664, -0.47014352679252625, 1.0794330835342407, -0.5909483432769775, 1.0655122995376587], [-0.8445492386817932, 0.523982584476471, -0.34520599246025085, -0.867226779460907, 0.3449365198612213, -0.8675128817558289, 2.086944103240967, -0.9193361401557922, -0.7727876901626587, -1.745056390762329, -1.6924610137939453, -0.7836194634437561, 0.04568206146359444, 0.3394640386104584, 1.2826778888702393, -0.49273595213890076, 0.09884659200906754, 0.026374151930212975, -0.18046553432941437, -0.20120441913604736, -1.0751066207885742, -2.0886969566345215, -0.7536940574645996, 0.16488642990589142, -0.8945974111557007, -0.3863219916820526, -1.653747797012329, 0.10657721012830734, 0.3643631935119629, 0.09221521764993668, -1.2790945768356323, 0.08035951852798462, -1.0385841131210327, 1.0491424798965454, -1.7654494047164917, -0.735100507736206, 1.3648642301559448, -0.36376017332077026, -0.6750050783157349, -1.9923419952392578, 0.38287168741226196, -1.9330984354019165, -0.05264710634946823, 0.6290063858032227, -0.08928800374269485, -2.8509130477905273, -0.49880000948905945, 0.160391166806221, 0.31951165199279785, 0.23933246731758118, 0.20893175899982452, 0.14243711531162262, 0.26324746012687683, 0.3875759541988373, 0.28731608390808105, 0.14068445563316345, 0.07016322016716003, 0.9760490655899048, 0.13746456801891327, -0.7590054273605347, 0.40128839015960693, -0.37848061323165894, -0.5430054068565369, -0.12814395129680634, 0.22240711748600006, 0.06179218739271164, -0.970258355140686, 0.6752035617828369, -1.4280214309692383, 0.3968120813369751, -0.39237964153289795, -2.016838312149048, 0.04181678965687752, -0.8829590082168579, -0.36561673879623413, -0.6502624154090881, -1.1713452339172363, -1.4633435010910034, -1.2400760650634766, -0.2815627157688141, 0.30245667695999146, -2.071195602416992, 0.16295138001441956, -0.9571903347969055, -1.671005368232727, 0.47664839029312134, -0.6307140588760376, -0.9442316293716431, -0.8666202425956726, 1.0843528509140015, 1.2380895614624023, 1.1368377208709717, -0.8397323489189148, 1.2004388570785522, -0.438653826713562, -1.8544056415557861, 0.6314064264297485, -0.854424238204956, -1.2472645044326782, -0.9339354634284973, -1.2562137842178345, -0.04579378291964531, 1.708073616027832, -0.8457837700843811, -0.4207944869995117, 1.511649250984192, -0.7073748111724854, -1.5433876514434814, -0.9460551738739014, 0.433366984128952, -0.5400964617729187, 0.7090063095092773, 0.2596133053302765, -0.5044562816619873, 0.9210072755813599, -0.2775764465332031, -0.15905487537384033, 0.2756122946739197, -0.3113723397254944, 0.6797493100166321, -1.4336388111114502, -0.5964478850364685, -1.1502920389175415, 0.8132113218307495, -1.5772151947021484, 0.1932189166545868, 1.2097692489624023, 1.0926603078842163], [-1.0644367933273315, 0.16874699294567108, -1.037724256515503, -0.8036165833473206, -0.7959085702896118, -0.17604325711727142, 0.2011864334344864, -0.16828931868076324, -1.4842108488082886, 0.4075338542461395, 0.4046163558959961, 0.05055401474237442, 0.7988041639328003, -0.3427291214466095, -1.6235313415527344, 0.3755394220352173, 0.3681562840938568, 0.9330556392669678, 0.8080666661262512, -0.9083654284477234, 0.6424902081489563, -0.6869274377822876, 1.694922924041748, -0.43084779381752014, 1.1735104322433472, -1.7126595973968506, 0.4425244927406311, 1.6914551258087158, 0.7970492243766785, -1.5654693841934204, 2.904951572418213, 0.6150455474853516, 0.2432301938533783, -1.2167996168136597, 1.6121301651000977, 1.3275529146194458, 0.8806323409080505, -1.2284270524978638, -0.0004896115860901773, -0.37083667516708374, -1.7234426736831665, 0.22402949631214142, -0.7696893811225891, 0.8993138074874878, 0.4542047381401062, 0.7646125555038452, -1.5426321029663086, -1.533588171005249, -1.20792555809021, -0.45589613914489746, 1.8663878440856934, 1.8452445268630981, -1.2099485397338867, -1.0200252532958984, -1.4572031497955322, -1.0789660215377808, -0.16770777106285095, -0.03453248739242554, 0.25526341795921326, 0.3193279504776001, 1.7948431968688965, 0.5020915269851685, -1.2943899631500244, -1.4900952577590942, 0.7521477937698364, 0.7947655320167542, -1.2470608949661255, -1.6620656251907349, -0.0409933365881443, -0.8867301940917969, -0.5094950199127197, -0.8222892880439758, 1.4408352375030518, -1.4087039232254028, -1.1807632446289062, -0.7299975156784058, 1.1312543153762817, -0.3673991858959198, 0.8208549618721008, -0.5269708037376404, -0.5316130518913269, -0.22730404138565063, -0.6011272072792053, 2.3621909618377686, 0.5799289345741272, -1.4370450973510742, 0.5678030252456665, -1.8462505340576172, -1.796079158782959, -0.19030369818210602, -0.5667824149131775, -0.9725388884544373, -0.1726573258638382, 1.3396728038787842, -2.065002202987671, 0.854415237903595, 1.3038119077682495, 0.8209731578826904, -0.6803712248802185, -0.6925195455551147, 0.3043293058872223, 2.694936990737915, 0.45565107464790344, -1.2950667142868042, 0.5276229977607727, -0.5571892261505127, 0.2762758731842041, -0.3925302028656006, 1.1063429117202759, -1.5729612112045288, 0.9672338962554932, 0.9190536737442017, -1.211325764656067, 1.4115009307861328, 1.3741174936294556, 0.19459286332130432, -0.037738583981990814, -0.7491486668586731, 0.12016112357378006, 0.13977272808551788, 0.7553443908691406, -0.8822531700134277, 3.491431951522827, -0.012111193500459194, -2.0725209712982178, 0.8786983489990234, 0.24915429949760437, -0.8274610042572021], [0.7103219032287598, -0.157759428024292, 0.6890414953231812, -0.07602379471063614, -1.3033932447433472, -0.30205395817756653, 1.7328376770019531, -0.9112242460250854, 0.950854480266571, -0.5930396318435669, 0.10169734060764313, -0.913486123085022, -0.7591954469680786, 0.3969641625881195, -1.1433972120285034, 1.105819582939148, -0.18131747841835022, -0.7155658602714539, 0.8566440343856812, 2.2256038188934326, -0.2583107352256775, 0.6220922470092773, -0.1350538432598114, 1.4201645851135254, 1.4482754468917847, -0.29138484597206116, -0.33504951000213623, 0.6435990333557129, 0.0013337009586393833, 1.0845533609390259, -0.14881643652915955, -0.90360426902771, 2.606153726577759, 0.5779941082000732, -0.5544302463531494, 1.8479195833206177, -1.368610143661499, 0.035927966237068176, 0.48931819200515747, 1.189255714416504, 0.6488276124000549, 0.2884197533130646, -0.1451469510793686, -1.6467698812484741, -1.9659417867660522, -0.4675889313220978, -0.7267632484436035, 0.0684162899851799, 0.014160756021738052, 1.198352575302124, -0.06739351153373718, -0.1388894021511078, 0.44837337732315063, -1.7586807012557983, 1.9263086318969727, 0.05446416884660721, -0.5353861451148987, 1.238385796546936, 0.6384013295173645, -0.4037693440914154, -0.37677305936813354, -1.4428352117538452, -0.41264402866363525, 2.3897464275360107, -0.6322770118713379, -0.11072990298271179, 0.988383412361145, -0.22580581903457642, 0.36159268021583557, -0.5653302669525146, 1.2971744537353516, 0.03256687521934509, -0.32226356863975525, 0.4633808434009552, -0.5153371691703796, -0.6928022503852844, 0.29272380471229553, -0.4271729290485382, 0.8357064127922058, -0.7908331751823425, -0.8057664036750793, -1.4977443218231201, -1.0405714511871338, -0.4478082060813904, -0.8167814612388611, -0.19317619502544403, 0.2938477098941803, -0.7607828378677368, -0.15052270889282227, 0.017488429322838783, 0.13841985166072845, -3.0766372680664062, -0.28013092279434204, -0.46165838837623596, -2.0949902534484863, -0.7310066223144531, 0.40404483675956726, -0.7849501371383667, 0.6647784113883972, -1.9020860195159912, 0.45866695046424866, -0.9372609853744507, -0.158090278506279, 0.615820050239563, 0.7008856534957886, -0.35208648443222046, -0.5309911370277405, 0.5194383859634399, 0.7819715738296509, -0.06821949034929276, -0.06524384766817093, 0.5546058416366577, 0.3184063136577606, 0.9394866824150085, 1.076135277748108, -0.31116819381713867, -0.46105629205703735, 1.0154025554656982, 0.4927959144115448, -1.6787121295928955, 2.5459067821502686, 0.7214220762252808, -0.21527472138404846, -1.10198175907135, -1.3385934829711914, 1.2152690887451172, -0.36547979712486267, 0.7058997750282288], [-0.11795974522829056, 0.040603965520858765, -2.275398015975952, -0.45059773325920105, -1.5292699337005615, 1.0775365829467773, -1.9524424076080322, 0.37799420952796936, 0.7206186056137085, -0.7754952907562256, 1.101784110069275, 0.11746305972337723, -0.8965616226196289, 1.3679238557815552, 0.024676818400621414, -2.0741586685180664, 0.36045292019844055, 0.4195259213447571, 0.6813971400260925, 1.2576833963394165, -0.3020270764827728, 1.0342497825622559, 0.22568443417549133, 0.5315616726875305, 0.8790395259857178, -1.0270028114318848, 0.061056505888700485, -1.112534999847412, 0.5033658146858215, 1.2479623556137085, -1.163116455078125, -1.2867060899734497, 1.0471042394638062, 0.3998680114746094, -0.7658713459968567, 0.820436954498291, 0.37953388690948486, 0.054514702409505844, -0.7985320091247559, -0.04582345485687256, 1.4327973127365112, 1.8143925666809082, 0.32425951957702637, 0.6338897943496704, 1.6479955911636353, 2.0897209644317627, 1.1518712043762207, 0.38487157225608826, 1.3630796670913696, 0.09061279147863388, 0.6394331455230713, 0.0757412537932396, 0.1686287373304367, 1.0423541069030762, -0.6153050661087036, 1.2641117572784424, -1.4024507999420166, -2.4755303859710693, -2.157365322113037, 0.21108877658843994, 0.37616485357284546, -0.19573283195495605, 0.3715958893299103, 0.17579764127731323, 0.5779634118080139, -0.1539950966835022, -0.46489182114601135, -0.035445552319288254, 0.8342950940132141, 0.5026935935020447, 0.02889348939061165, 0.4685863256454468, -2.1939213275909424, 0.5256017446517944, 0.34456685185432434, -0.5331105589866638, 2.134758710861206, 0.4719327688217163, 0.8046357035636902, -0.05104396492242813, -0.9113500714302063, -1.468673825263977, -0.9618932008743286, -0.4450979232788086, 0.3622839152812958, 0.22638440132141113, 0.7463685274124146, -1.8018914461135864, -2.245976448059082, -0.9532520174980164, 1.406014084815979, -0.9123337268829346, -0.4403274953365326, 0.5803377628326416, 0.9780154824256897, 0.010258863680064678, 0.8129311800003052, -0.36146441102027893, 0.08924073725938797, -0.32616448402404785, -0.053830843418836594, -1.616368055343628, 0.432121604681015, 1.115925669670105, 0.7077178955078125, -0.908534824848175, 0.07388029992580414, -0.8759299516677856, -0.3771875500679016, 0.04965149983763695, 0.26223865151405334, 0.8464440703392029, -1.1257448196411133, -0.2045956552028656, -0.2432226836681366, -0.3174108862876892, -0.38362646102905273, -2.9020378589630127, -1.1196047067642212, -1.0239022970199585, 0.502689003944397, -0.1598685085773468, -1.4165912866592407, 1.1284384727478027, 0.3237462639808655, 0.8834415674209595, 0.6090177893638611, -0.3807346224784851], [0.18452347815036774, -1.2948737144470215, -2.709782600402832, -0.7971478700637817, 0.4109034538269043, 0.4805262088775635, 0.007627987768501043, 0.6986969709396362, -0.13176408410072327, 1.0013790130615234, 0.8064802885055542, -2.126131534576416, 2.0330185890197754, 0.6852513551712036, 0.9035237431526184, 0.5014878511428833, 2.3129889965057373, -0.7127532362937927, 0.07323115319013596, 0.9435606598854065, -1.2512049674987793, -0.39732030034065247, -0.8574550151824951, 1.5780531167984009, -1.343035101890564, 1.461709976196289, 1.5368995666503906, -1.5009838342666626, -2.572504997253418, 0.42106151580810547, -0.08416448533535004, -0.2204272300004959, 1.1710516214370728, -1.1308810710906982, 0.05814114585518837, -1.2824360132217407, 1.5757931470870972, 0.24490520358085632, 1.9150522947311401, -0.17083780467510223, -0.9068835377693176, 1.219719648361206, 0.42437514662742615, 0.23563392460346222, -1.8854719400405884, 1.5294737815856934, 0.41676321625709534, -0.1832636594772339, -0.05311069265007973, 0.3884250819683075, 0.2751558721065521, -0.2090965360403061, 0.14709530770778656, -0.19262753427028656, -2.2379326820373535, -0.6365894079208374, -0.30107226967811584, -0.4414339065551758, 0.03696627542376518, -2.4262218475341797, 2.734912157058716, -1.405276894569397, 0.5911991596221924, 0.5434941649436951, 2.45839786529541, 0.8057692050933838, -1.5438348054885864, 0.8273604512214661, -0.3581047058105469, -0.3780194818973541, 0.509796142578125, -0.14234521985054016, -0.2256501019001007, -0.42501991987228394, 0.9097827076911926, 1.522832989692688, 1.1974507570266724, -0.5617519617080688, -1.54074227809906, 0.08571361750364304, 0.539345383644104, -0.32141610980033875, 0.3257800340652466, 0.989176332950592, 1.5315933227539062, 2.537471294403076, -1.635766863822937, 0.20047610998153687, -1.7985117435455322, -0.30776166915893555, 0.9120646119117737, -0.3206985592842102, -0.43768003582954407, -0.2134036123752594, 1.587519884109497, -0.8261560797691345, 0.5352194309234619, -0.5269564390182495, 0.9073746204376221, 0.9187785983085632, 0.43446406722068787, 1.9552106857299805, -0.45813214778900146, 0.7061251997947693, 0.5929637551307678, 0.5677840709686279, 1.8385567665100098, -1.0363410711288452, 1.1368753910064697, -1.5832487344741821, 1.200189471244812, -1.2240068912506104, -0.644117534160614, -0.4875083267688751, -1.8728995323181152, 0.28115400671958923, 0.21808549761772156, -1.240675926208496, -1.6033356189727783, -1.176861047744751, 0.6183575987815857, 0.9035018086433411, -1.6119318008422852, -2.4616284370422363, 0.777903139591217, -0.6277235150337219, 0.5305060148239136, -0.2689114212989807], [-1.2850699424743652, -0.2837212085723877, -0.8592202663421631, 1.2128313779830933, 0.44223275780677795, -0.11809047311544418, -0.6631138920783997, -0.8527535200119019, -0.1674792617559433, -1.2035634517669678, -1.7485260963439941, -0.8257246613502502, -1.064101219177246, 0.2656809389591217, -0.2621838450431824, 1.0035122632980347, 0.39163118600845337, 0.09552162140607834, -1.5233665704727173, -1.2881784439086914, -0.11930505186319351, 0.6052052974700928, 0.8075092434883118, 0.22842994332313538, -0.3054649829864502, -0.29845455288887024, 0.6321150660514832, 1.0598483085632324, 0.4804542064666748, 2.233076333999634, 0.6191790103912354, -0.3269891142845154, -0.44954678416252136, 0.1913747936487198, 1.885140299797058, -0.41952526569366455, -0.529725968837738, 2.125122547149658, -0.13551843166351318, 1.922666072845459, -0.36846935749053955, -2.5894436836242676, -0.18820525705814362, -2.2643511295318604, 0.5988726615905762, -1.5350556373596191, -2.2932803630828857, 0.9740200042724609, -1.07050359249115, -0.7084567546844482, 0.5690698623657227, 0.1969289630651474, -0.341627836227417, 0.0340990349650383, 0.30918335914611816, 0.061620648950338364, 1.4953720569610596, 1.1612950563430786, 2.5512022972106934, -0.053188662976026535, -0.3713362514972687, -1.4781420230865479, -0.30832403898239136, -0.23780158162117004, 0.9889786243438721, -0.40615060925483704, -0.12289241701364517, 0.5779420137405396, 0.1661786437034607, -0.25262340903282166, -0.487102746963501, -0.5341364145278931, 0.9093722701072693, -0.21614065766334534, -0.16262111067771912, 0.790874719619751, -1.3769035339355469, -0.5911641716957092, -0.5469487309455872, -1.0485023260116577, 1.572277545928955, -0.2646383047103882, 1.1115504503250122, 0.637254536151886, 1.0059330463409424, -1.5383949279785156, -2.009399890899658, 1.5648168325424194, 1.9520416259765625, -1.854871392250061, -0.13891403377056122, -1.9447017908096313, -0.8156247138977051, 0.9739569425582886, -1.2166094779968262, -0.004809491336345673, -0.514009952545166, 0.20748499035835266, -0.5684792399406433, -1.3302134275436401, -1.0906071662902832, -0.128744438290596, 0.37902405858039856, 1.274849534034729, -1.4052766561508179, -0.8791601061820984, 0.8648802042007446, 1.3426494598388672, -0.9235819578170776, -0.51717609167099, -0.29664748907089233, -1.3936797380447388, -0.02663770504295826, -1.9344773292541504, -0.5287437438964844, -2.4302175045013428, -1.7463955879211426, -0.8480915427207947, -1.3599450588226318, 0.5904796123504639, -0.13469097018241882, -1.3109791278839111, 0.477875679731369, 0.9944291114807129, -2.0007688999176025, -0.9049683809280396, 1.311734676361084, 1.0130680799484253], [-1.8377487659454346, 0.292710542678833, 1.0513745546340942, 0.6684594750404358, 2.1052184104919434, -1.6785218715667725, 1.5390874147415161, -0.008075736463069916, 0.27956312894821167, 0.7831598520278931, -0.9786888360977173, -0.2873113751411438, -0.30077528953552246, -0.8191893100738525, -0.47500768303871155, 0.18891216814517975, -0.22433732450008392, -0.39402705430984497, -0.04159710556268692, -1.1241862773895264, -0.04468000680208206, 1.2646636962890625, 0.4783063232898712, 0.30426937341690063, 1.2663699388504028, -0.6832696199417114, 0.07964687794446945, 0.087233766913414, -0.2694110870361328, 0.029417559504508972, 0.4091421365737915, -1.1836514472961426, -0.7782700657844543, 0.292100191116333, -0.612784206867218, -1.6611384153366089, -0.6649827361106873, -1.1166905164718628, 0.24093855917453766, -2.4018361568450928, 1.4939301013946533, -0.055668361485004425, 0.17535530030727386, 0.7392535209655762, -0.4850621819496155, -0.5938119888305664, 3.24161958694458, 1.5252221822738647, -0.09871526062488556, -1.2700859308242798, 0.04898351430892944, 1.0072699785232544, 1.0813430547714233, -1.2337579727172852, 1.0057084560394287, -0.1309116631746292, -1.1067893505096436, 0.396316260099411, -0.7878347635269165, -0.0866631492972374, 1.2529370784759521, 0.3511156439781189, -0.8873412013053894, -0.675288736820221, 1.6075578927993774, -1.1466466188430786, -0.08054322749376297, -0.4932636320590973, -0.637539803981781, 0.6262279748916626, -0.40289998054504395, -0.852040708065033, 0.40405336022377014, 0.12439921498298645, 1.7229132652282715, -0.9207313060760498, -0.6260778307914734, -0.3033808171749115, -1.1056468486785889, -0.543159544467926, -0.6389986872673035, -1.3847711086273193, -0.2860238254070282, 0.8763782978057861, 0.47700268030166626, 0.3386857211589813, -1.1806963682174683, -0.15661270916461945, -0.21816813945770264, 0.6981492042541504, -1.932819128036499, 0.48920920491218567, -0.320208340883255, 0.9619318246841431, 0.3178335130214691, 0.22009459137916565, 0.42564818263053894, 2.3630735874176025, -0.2007209062576294, -0.10800277441740036, -0.6549263596534729, -0.027777986600995064, -0.18137101829051971, -1.1169861555099487, -0.8235462307929993, -1.7528977394104004, 0.003456207923591137, 0.5228543877601624, -1.6878584623336792, 0.11133862286806107, 1.7888407707214355, -0.41713428497314453, 0.14230994880199432, -0.8064126968383789, -0.4486471712589264, 1.0080444812774658, 0.4741514027118683, 0.6694178581237793, -0.6199427247047424, 0.47276273369789124, 1.282277226448059, -1.4532580375671387, -0.2773049473762512, 0.886039137840271, -1.0442276000976562, -0.49435967206954956, 0.40545204281806946, -1.7622313499450684], [2.1150550842285156, -0.06244899705052376, -0.9798406958580017, -0.4133186638355255, -0.03686145320534706, 0.8190032839775085, -1.4765207767486572, -0.12624339759349823, 1.9486395120620728, -1.1161214113235474, -0.12162616103887558, -0.059179700911045074, -0.11673286557197571, 1.2932604551315308, 0.24591287970542908, 0.124483622610569, 0.309841513633728, -0.7195194959640503, 0.9557701945304871, 0.19032606482505798, 0.47397932410240173, 1.5524591207504272, -0.5654220581054688, 0.1851656138896942, -0.19892185926437378, -0.7282542586326599, 0.358104944229126, -1.7593165636062622, 0.9387956261634827, 1.3555443286895752, 0.8400138020515442, -1.1045657396316528, -0.1862672120332718, -1.0166302919387817, -0.32698002457618713, 1.576321005821228, -1.3816403150558472, -0.5133637189865112, 0.3964463174343109, -0.17019006609916687, -0.6082544922828674, 1.239253282546997, -1.278692603111267, 0.19370076060295105, -0.3440762460231781, -0.6767024397850037, -1.9265899658203125, 0.1977076530456543, -0.23348326981067657, -0.45048531889915466, 0.20867511630058289, 0.69902503490448, 1.3414585590362549, -1.360960841178894, 0.22123457491397858, -1.0903327465057373, 0.07586241513490677, -1.8570435047149658, -1.0063159465789795, -0.3663656413555145, 0.30606386065483093, 0.8484284281730652, -0.03018045425415039, 0.6102296710014343, 1.738634467124939, -0.2897525131702423, 1.627145528793335, 0.37794846296310425, 0.8392622470855713, -0.49431145191192627, -0.237489715218544, 1.5912001132965088, 2.6211209297180176, 0.5855433940887451, -0.12172305583953857, 0.04962458834052086, -1.600575566291809, 2.148772716522217, 1.4606492519378662, 0.2302009016275406, 0.04315422102808952, 0.34655800461769104, -2.1130573749542236, -2.8229482173919678, -1.0298550128936768, 0.29633963108062744, 1.1683094501495361, 0.011401066556572914, 0.5854065418243408, 0.6072731614112854, -0.44419339299201965, 0.2031727284193039, -0.9824261665344238, 1.4176076650619507, 0.00515463761985302, -1.6194363832473755, 1.6174378395080566, 0.4409846365451813, -0.36552831530570984, -1.5947951078414917, -0.35676199197769165, 1.3585925102233887, -0.6725028157234192, -0.7103462815284729, -0.29229477047920227, 0.6530802249908447, -0.41638559103012085, -1.5463391542434692, -0.18026675283908844, 0.11413378268480301, 1.2799967527389526, -0.9157272577285767, 0.24701279401779175, 0.6114689111709595, -0.6708455085754395, -1.8872044086456299, -3.059748649597168, -1.0145444869995117, -1.3144371509552002, 0.28345951437950134, -0.8654153347015381, -0.02896689809858799, -0.019287314265966415, -0.5339698195457458, -0.7645854949951172, 0.29074612259864807, 0.1267508566379547, -0.6360254287719727], [-1.1703612804412842, 0.6463920474052429, 1.394001841545105, -1.3038065433502197, -0.7294603586196899, 0.9969279766082764, -0.20784074068069458, -0.5557499527931213, 2.673069715499878, 0.594588041305542, -0.11908790469169617, 0.5501518249511719, 0.2768436670303345, -0.05891431123018265, -0.5213053822517395, 0.4682263433933258, -0.8837743401527405, 0.28686344623565674, 0.16438737511634827, 0.932801365852356, -0.34634342789649963, 0.21987546980381012, -0.16840100288391113, -0.5570856332778931, 1.3358718156814575, 0.7213472723960876, 0.37591761350631714, 0.10715600848197937, -2.146343946456909, 1.1087976694107056, 0.646530270576477, 0.5191330313682556, 0.9332978129386902, 0.16611959040164948, -0.8083900809288025, -0.09328092634677887, -0.04949413612484932, -1.2008743286132812, -0.4926585257053375, 0.18870194256305695, 0.5669971704483032, 1.5926865339279175, 0.1987149715423584, 1.783722996711731, -0.35047778487205505, 1.0770944356918335, 1.0207253694534302, -0.6093838214874268, -1.2172569036483765, -0.13041089475154877, -0.9187542200088501, 0.4741981029510498, -1.0039715766906738, 0.8310981392860413, -1.835207223892212, -0.20651593804359436, -1.1165746450424194, -2.3233649730682373, -0.41691309213638306, 0.49246135354042053, -1.827287197113037, -0.3776581883430481, -1.0068693161010742, 0.8695411682128906, 0.32329261302948, -1.7456399202346802, 0.5283035635948181, 0.4514530301094055, -0.4839031398296356, -0.8152837753295898, 0.11285095661878586, 0.23301325738430023, -1.184983253479004, -0.7542911767959595, 0.9371098279953003, 0.41756513714790344, 0.5271623730659485, 0.7653596997261047, -0.7985213398933411, 0.22827719151973724, -0.06833599507808685, 1.1756311655044556, 1.1280109882354736, 1.147499680519104, 1.5079777240753174, -0.3906329274177551, 0.6470046639442444, -0.07675182074308395, -1.0253173112869263, -0.7549947500228882, 0.2588950991630554, -1.525508999824524, -0.6567940711975098, 1.45186448097229, -0.7627173066139221, 1.5017547607421875, -0.7747279405593872, -0.3109496533870697, 1.147506833076477, 0.5511366128921509, -0.2448277473449707, 0.6292498111724854, 0.8042236566543579, 0.43546369671821594, 0.7797446250915527, -0.2562348544597626, -1.9355863332748413, -0.6564077734947205, 0.8244401216506958, 1.2234870195388794, -1.3439085483551025, 1.249650239944458, 0.3344477713108063, -0.18108518421649933, 0.43443387746810913, -1.0859569311141968, -2.4432125091552734, -0.42046672105789185, -0.07260094583034515, 1.208707332611084, -0.8148388266563416, -0.04024932160973549, 0.5876018404960632, 0.03595077246427536, 0.21555495262145996, -0.5998379588127136, 0.5359464287757874, -1.0985609292984009], [-2.312222719192505, 2.0362112522125244, 0.608553946018219, -0.3814612627029419, -1.5381252765655518, -0.7907854914665222, 0.7630885243415833, -0.8353654742240906, 0.7874574065208435, 0.7244221568107605, -0.6176889538764954, -1.9448148012161255, -1.7693692445755005, 0.017672419548034668, 0.048444658517837524, -1.7921141386032104, 0.3989093005657196, 1.8001258373260498, 1.2421634197235107, -1.3220258951187134, 2.0303847789764404, -0.5925384163856506, 1.0075280666351318, 0.18988890945911407, -0.12409237027168274, 1.838563323020935, 0.8257033824920654, 0.33701425790786743, -0.33939966559410095, -0.25688713788986206, -2.119391918182373, 0.059375084936618805, 0.06109478697180748, -1.3397973775863647, -0.7649480104446411, -0.8822196125984192, 1.4189200401306152, 1.2013949155807495, 0.42254823446273804, 1.5437318086624146, -0.2554857134819031, 0.9003597497940063, 1.065533995628357, -0.5468212962150574, 2.3266735076904297, -0.7778978943824768, 0.16293838620185852, -0.3120104968547821, -0.497158944606781, -0.5409621000289917, 0.827733039855957, -1.3745183944702148, 0.6250457167625427, -0.8491692543029785, 0.8293036222457886, 0.40998074412345886, 1.4842240810394287, -0.5496082305908203, -2.5033652782440186, 0.19619762897491455, 0.7203214764595032, 0.9490160346031189, 1.0643699169158936, -0.4245189428329468, 0.9018571972846985, -2.6021056175231934, 0.1683712899684906, 1.5502394437789917, 0.8927854895591736, 0.49348708987236023, 0.6435174345970154, -0.3354557752609253, -1.99643874168396, 0.8524473905563354, 1.2830063104629517, 0.5134819149971008, 1.979651689529419, 0.6252214312553406, 0.724524199962616, 0.00559201929718256, -0.9710010886192322, 1.553381085395813, 0.16453351080417633, -1.2269715070724487, 0.17233146727085114, -0.6036054491996765, -0.4076315760612488, 0.01993527263402939, 1.0289374589920044, -1.0044009685516357, -0.9281111359596252, 0.9627405405044556, 0.734655499458313, -0.5403357148170471, 0.3854936957359314, -0.46524739265441895, -0.1491316854953766, -0.1775071620941162, 0.4057357907295227, 1.3409374952316284, -0.9246581792831421, 1.1276589632034302, 0.5641264319419861, -0.46007660031318665, -0.7805416584014893, -0.16393540799617767, 0.8580485582351685, 0.9205703735351562, -0.8498530983924866, -0.39421674609184265, 0.4496128559112549, 2.3701701164245605, 0.0024906250182539225, 0.589695155620575, 0.41982343792915344, -0.6239952445030212, -0.5929815173149109, -1.113471508026123, -0.6642359495162964, -2.0123648643493652, -0.16016608476638794, -0.4700867831707001, 0.32740703225135803, 0.8000689744949341, 0.0380048006772995, -0.2873845100402832, 0.9473317265510559, -2.284860849380493], [0.41314491629600525, -0.4578384459018707, 0.48242780566215515, 0.3469196856021881, -0.22108979523181915, -0.03605937585234642, -0.015639571473002434, 1.1072930097579956, 1.6511762142181396, 0.321760356426239, -0.7183125019073486, -1.088197946548462, -0.7869359254837036, -0.16862350702285767, 1.0054678916931152, 0.5353026986122131, -1.4853343963623047, 1.7303181886672974, -1.7402794361114502, -1.6353700160980225, -0.03492395207285881, 0.9015170931816101, 0.7379442453384399, -0.37309861183166504, -0.5965679883956909, -0.5546799898147583, -0.7238768935203552, -1.8242887258529663, -0.17313691973686218, -0.5690616369247437, 0.17385709285736084, 0.4221796691417694, -1.5569318532943726, -1.0109877586364746, 0.2725764811038971, -0.438137412071228, 3.087918519973755, 0.2194017469882965, -0.10568229854106903, -0.992052435874939, -0.283149391412735, 0.7840816378593445, -0.931840717792511, -0.21489953994750977, 1.4220627546310425, 0.9349819421768188, -1.219454050064087, 0.15430043637752533, -0.1731271594762802, -2.0803935527801514, 1.0969692468643188, -0.8101131916046143, 0.17093250155448914, -1.2541402578353882, -1.0209224224090576, 0.1150549054145813, 1.654495120048523, -1.7554519176483154, -0.6077611446380615, 0.43897026777267456, 0.8862146735191345, -0.8067681193351746, -1.1292558908462524, 0.4742101728916168, -0.9333103895187378, 1.107800006866455, 0.30362817645072937, -0.5794764757156372, -1.687314510345459, 1.3691102266311646, 0.9612482190132141, 1.9179937839508057, -1.2423516511917114, 0.5497958660125732, 1.4855551719665527, 0.6622785329818726, 1.4927136898040771, -0.4724522829055786, 0.8106483817100525, -0.0480235256254673, 0.31500244140625, 0.10911782085895538, -0.03914329409599304, 0.21649564802646637, 1.8699748516082764, -1.2908319234848022, -1.3901022672653198, 0.16587373614311218, 0.0876648798584938, -0.8333489298820496, -0.5308243632316589, 0.1572251319885254, -0.6168481111526489, 0.876624584197998, -1.3977712392807007, -0.8021825551986694, -0.29264217615127563, -0.29732370376586914, -0.06690751016139984, 3.084604263305664, -0.4195244610309601, 0.8085845112800598, 0.6568796038627625, 0.9928302764892578, 0.9785898327827454, -0.8056426048278809, -0.46412965655326843, 1.4091273546218872, 1.1298556327819824, -0.5281152129173279, -0.7803735733032227, -1.5737009048461914, 1.178893804550171, 1.3635482788085938, -0.32140886783599854, -0.641156792640686, 0.8122513890266418, -0.3820757567882538, 0.9580053091049194, 0.7333027720451355, -1.3843274116516113, 0.6133813261985779, 1.037719964981079, -0.15599213540554047, -0.4332343637943268, -2.2959299087524414, 1.2418889999389648, -0.5656570196151733], [-1.1793818473815918, -0.9691523909568787, -1.2042547464370728, -0.1731712967157364, 0.9335501790046692, 2.1121444702148438, 0.2092958688735962, -2.441035032272339, 0.36254140734672546, -0.9029020071029663, -0.24650996923446655, -0.14751476049423218, -0.13683299720287323, 1.266451120376587, -0.03339202329516411, 0.8323701024055481, -0.35684216022491455, 0.8198513984680176, 1.1873888969421387, -0.5624055862426758, -1.008657693862915, -0.6512978076934814, 2.196319103240967, 1.6080816984176636, -0.6132509112358093, -0.7666359543800354, -0.12582355737686157, -0.2993949055671692, -0.7352665662765503, 1.381048560142517, -1.46052086353302, 0.5036657452583313, -0.2673104405403137, 0.12868404388427734, -2.106815814971924, 0.7224384546279907, -0.9292327165603638, 0.13870707154273987, 1.4815435409545898, -0.8677628636360168, -0.6998814940452576, -0.38645049929618835, 1.4066578149795532, 0.5667375922203064, 0.6532531380653381, -1.2261959314346313, -0.05377061292529106, 1.1535171270370483, -0.276127427816391, -0.3071327209472656, 2.1950042247772217, 0.04246869683265686, 0.6802001595497131, -0.11970117688179016, 0.29180270433425903, 1.7233086824417114, -1.4535117149353027, 1.2205232381820679, 1.7733153104782104, 2.409130096435547, 1.01979398727417, 1.052949070930481, -0.6158210635185242, 0.7227067351341248, 0.517386794090271, -1.4094845056533813, -0.4996410012245178, 0.7610018849372864, 0.6281679272651672, 0.3202788233757019, 1.992536187171936, -1.231419563293457, 1.3679347038269043, -1.7898290157318115, 0.9161989092826843, -0.3639044165611267, 0.7371453642845154, 1.7025554180145264, 0.8061277270317078, -0.7156916856765747, 0.2548220157623291, 0.980528712272644, 1.482065200805664, -1.2704635858535767, 1.0945854187011719, 0.1517239809036255, -0.36944180727005005, 0.5725399255752563, 0.16303129494190216, 2.9754648208618164, 0.47688543796539307, -0.9247902035713196, -0.31180015206336975, -1.7605704069137573, -0.33273014426231384, -1.6137133836746216, 0.5847814083099365, 0.44958946108818054, -0.24993479251861572, -0.5403432846069336, 0.19624802470207214, -0.792509138584137, -0.6916514039039612, 2.2486536502838135, -0.2260470688343048, 0.9318564534187317, -0.2332368940114975, -0.001868538442067802, 0.575504720211029, 1.583804726600647, 1.0244189500808716, 1.367233395576477, 0.2868192791938782, -0.017270319163799286, 0.09296084940433502, -1.2311350107192993, 0.7554808259010315, -0.12728756666183472, 2.307204246520996, 1.5172245502471924, -1.0255763530731201, -1.39614999294281, 0.4222574234008789, -0.7946445941925049, 1.2222414016723633, -0.5309463143348694, -1.4784036874771118, -2.187476873397827], [0.3151944577693939, 0.0353274941444397, -0.10128474235534668, 0.5580521821975708, -0.213577538728714, 0.7798823118209839, -1.3442989587783813, -1.4362387657165527, 0.06742919236421585, 0.5518338680267334, -0.5409186482429504, 0.5693228840827942, 0.6337783932685852, -1.3413760662078857, 1.3593595027923584, -1.2562754154205322, 0.21580754220485687, 1.489850401878357, -0.8530662655830383, -1.7828809022903442, 0.5464775562286377, 0.18598724901676178, 1.9822217226028442, 1.2679178714752197, -0.8343557119369507, 0.3169077932834625, 0.40516597032546997, 0.08699293434619904, -0.4225459396839142, 1.8347580432891846, -1.103108286857605, -1.1051050424575806, -0.7576999664306641, -1.7188483476638794, -0.7977892756462097, 0.9038504362106323, -0.47859281301498413, 0.30173438787460327, -1.772251844406128, -0.5025641918182373, -1.2060556411743164, -0.9133169054985046, -0.0035952364560216665, -0.28381747007369995, 0.16032113134860992, 1.5119208097457886, -0.32720938324928284, 0.5367597937583923, -1.1578893661499023, 0.6655886173248291, -0.1941806674003601, 0.32646191120147705, -2.488396406173706, -2.190297842025757, -0.7901114225387573, 1.1731153726577759, 0.1615573912858963, -2.5653297901153564, -1.034717082977295, 0.7784799337387085, 0.5001821517944336, 0.9931417107582092, 0.639677107334137, 0.10536538809537888, -0.2860841453075409, -1.617249608039856, -0.2589458227157593, 0.5508815050125122, 0.15987347066402435, 1.4255956411361694, -0.39849579334259033, 2.1134612560272217, -1.1904197931289673, -0.11765540391206741, -0.9599023461341858, -0.10725998133420944, -0.8416301012039185, -1.5820519924163818, -1.102604866027832, -0.44630929827690125, -0.09488572925329208, 2.0104050636291504, 0.43277406692504883, 0.8062460422515869, -0.10104651749134064, -1.0078506469726562, -1.3301795721054077, 0.8722902536392212, 1.9622470140457153, 0.41999244689941406, 0.22362276911735535, -0.6404998898506165, -0.477745920419693, -0.5730760097503662, 0.29047173261642456, 0.37721505761146545, -2.0394210815429688, -0.4552609622478485, -1.8755772113800049, -0.5757124423980713, 0.7516320943832397, -2.1103084087371826, 0.519500195980072, 0.8242090940475464, 0.31064367294311523, -0.3193725049495697, 0.48096731305122375, -1.1946995258331299, -1.424623966217041, -1.0714612007141113, -0.014808538369834423, -0.0667017325758934, 1.0413323640823364, 0.7291344404220581, 0.6867502331733704, -1.6081737279891968, -1.5157109498977661, -0.9843993782997131, 0.5862807631492615, 1.5480209589004517, -0.17564840614795685, -1.2256181240081787, 0.4341263771057129, 1.2578963041305542, 0.07550371438264847, -0.831194281578064, -1.0372661352157593, -0.2917104661464691], [1.9924044609069824, -0.2963757812976837, -0.2730730473995209, -0.6008786559104919, -0.2701203227043152, -2.3111932277679443, -1.3760669231414795, -0.28725361824035645, 1.9302020072937012, 0.9389249086380005, 0.9533659219741821, 0.9746780395507812, 0.8194661736488342, 0.44337326288223267, 0.5085577368736267, 0.016875512897968292, -0.2510046660900116, -0.37500086426734924, 0.23274508118629456, 0.43867436051368713, 2.5939462184906006, 0.8306047320365906, -0.32297012209892273, 0.6224552392959595, 1.3227664232254028, -0.10468319803476334, -0.44588199257850647, 0.5992541909217834, -0.17078541219234467, 1.5511690378189087, -0.6857785582542419, 0.7661744952201843, 0.5462507009506226, -2.0731418132781982, 0.19056746363639832, -1.7689300775527954, 1.083280324935913, 1.9845932722091675, 1.0539053678512573, 0.3487316071987152, -0.9160438179969788, 0.2825961112976074, -1.4533811807632446, -0.45558375120162964, -0.08189194649457932, -0.5587628483772278, 0.5870912671089172, 1.5796189308166504, 0.0997047945857048, 1.052164077758789, -0.6394441723823547, 0.4128682315349579, -0.17045506834983826, 0.442950040102005, 0.7914907932281494, 0.21447964012622833, 0.7773144841194153, 0.07167592644691467, 0.012383315712213516, 0.20956894755363464, 2.1271276473999023, 1.6193900108337402, -0.2035004198551178, -1.0710184574127197, -1.0481009483337402, -0.002498236019164324, -1.1178609132766724, 1.1720547676086426, -0.3709101378917694, -1.545809268951416, -0.6861756443977356, -0.021581226959824562, 1.6518148183822632, -0.9217554330825806, -0.7206245064735413, 0.09103074669837952, 0.504912793636322, 0.4885253310203552, -0.8405448198318481, 0.5619248747825623, -0.2948397397994995, 0.1068606898188591, -0.6923274397850037, 1.752565622329712, -1.7820885181427002, -1.113143801689148, -0.9314524531364441, -0.3827410638332367, -0.2905086576938629, -0.9433107376098633, -1.4959807395935059, -0.7503088116645813, -0.19225996732711792, -0.30433154106140137, -1.2634985446929932, -0.06548969447612762, -1.8279950618743896, 0.42596468329429626, 1.297785758972168, 1.169886589050293, -0.08368496596813202, 2.369023084640503, -3.046663761138916, 0.2645598351955414, -1.010962724685669, -2.1642906665802, -0.7412445545196533, 1.101391077041626, 1.610006332397461, 0.5359302759170532, -0.770751416683197, -1.0071412324905396, -0.004379851743578911, 0.13251729309558868, 1.0337529182434082, 0.09969673305749893, -0.23237697780132294, 0.608547568321228, 2.249008893966675, -1.8730649948120117, -0.3380458652973175, 1.571729302406311, -0.39341816306114197, -1.1730149984359741, 0.5032935738563538, -1.049843668937683, 0.7118709087371826, 0.26439741253852844], [-0.1700112372636795, -0.24972623586654663, 0.004550222307443619, -1.0298914909362793, 0.17436696588993073, 1.564527153968811, -0.02754979208111763, 0.6100925803184509, 0.8733229637145996, 1.0050938129425049, 1.0378694534301758, -0.2530405521392822, 0.5060774683952332, -0.29774704575538635, -0.004022953100502491, -0.19735057651996613, -0.17354093492031097, 1.0412644147872925, 1.6399961709976196, 0.3919662833213806, -0.059076663106679916, -1.200275182723999, 1.4535194635391235, 0.1860046237707138, -0.033929627388715744, -1.0238016843795776, -0.04861856624484062, 0.20527364313602448, 0.634780764579773, 0.21972015500068665, -0.4535001814365387, 1.499816656112671, 1.622542142868042, -0.39479219913482666, 0.26034873723983765, 2.4644150733947754, -0.6646388173103333, -0.2662515938282013, -0.37053918838500977, -0.7759472727775574, -0.8715546727180481, -0.03253161907196045, 0.41560202836990356, 0.4889586567878723, 0.9297463893890381, -0.1931571513414383, 0.7162110805511475, 0.1682429164648056, -0.6150180101394653, 0.1517646610736847, -1.1761788129806519, 1.031816840171814, 0.6125857830047607, -2.1057350635528564, -0.653329074382782, 2.7315492630004883, 0.4270800054073334, 0.952265739440918, 0.7249360680580139, 0.168839693069458, -1.2695584297180176, 1.584567666053772, -0.1089368611574173, -1.1124695539474487, -1.079330325126648, 0.38800281286239624, 1.6663137674331665, 1.1486676931381226, -0.12936866283416748, 0.501194179058075, 0.13877525925636292, -1.2248494625091553, 0.2780435085296631, 1.827455997467041, 0.7687318921089172, -0.5239502191543579, 0.9459444880485535, 1.4508559703826904, 0.050437744706869125, -0.5284714102745056, -1.3553951978683472, -0.066207155585289, -0.6821417212486267, -1.1018812656402588, 1.4699039459228516, 0.4607103765010834, 1.3634250164031982, -0.8250292539596558, -1.2729125022888184, 0.439718097448349, 0.4177030622959137, 0.21187381446361542, -0.11961684376001358, 0.7828705906867981, 0.4324112832546234, -1.017736554145813, -1.5319916009902954, 0.0534382238984108, -0.31296470761299133, -0.005327577702701092, -0.6819450259208679, -1.6917738914489746, -0.7467686533927917, -1.5722428560256958, -0.868241012096405, -0.09810608625411987, 0.9838906526565552, -0.30440428853034973, -0.6436026692390442, -0.35344064235687256, -0.31887340545654297, 1.2086405754089355, -1.5480488538742065, -0.6996582746505737, -0.47950977087020874, 0.23028253018856049, -0.42694300413131714, 1.3688349723815918, -0.3527846038341522, -0.005487870424985886, -1.4790695905685425, 0.7405405044555664, -0.29284486174583435, 1.1928681135177612, 0.23474392294883728, 1.1080015897750854, -0.5444324016571045, 1.1259437799453735], [-0.8231655359268188, -1.4482131004333496, 0.26607605814933777, -0.9243489503860474, -0.8515746593475342, -1.8822734355926514, -0.6421617865562439, -1.2666325569152832, -2.1342365741729736, 0.638405978679657, 0.14219655096530914, 1.256454586982727, 1.175175428390503, 1.7387094497680664, -1.523596167564392, -1.187185525894165, -0.6128743886947632, 0.4512249231338501, 2.1714587211608887, 0.42216697335243225, -1.3983287811279297, -1.0505422353744507, -1.87167227268219, -0.3934653699398041, -1.7606174945831299, -0.40559378266334534, -0.09390965104103088, -0.6242135167121887, 0.011038663797080517, -0.5580646395683289, -0.7429541349411011, 1.1299866437911987, -0.9346215128898621, -1.6734850406646729, 0.5591429471969604, -2.420015573501587, 0.33912035822868347, 0.37398046255111694, -0.7990489602088928, -0.7869206666946411, 1.0664907693862915, -0.6694494485855103, -0.22936730086803436, -1.0926271677017212, -0.8671879768371582, -0.7197368741035461, -0.0005770261632278562, 1.2372074127197266, 0.3432866334915161, -0.18750950694084167, 0.700994074344635, 0.549431324005127, -1.6377465724945068, -0.3619835376739502, -1.1219532489776611, 0.6494469046592712, -2.0300447940826416, -2.461639881134033, 0.37253546714782715, 1.433536171913147, 1.8727364540100098, -0.011539667844772339, 1.5152833461761475, -0.07696178555488586, -0.286026269197464, -0.07378003746271133, 0.5274993777275085, 0.22597916424274445, 0.03730863332748413, -0.43478071689605713, -0.6828286051750183, -1.252172827720642, 0.5777444839477539, -2.1051244735717773, -1.9734166860580444, -0.056566931307315826, -0.8238738179206848, 1.5249375104904175, 0.4444628655910492, -0.9840953946113586, -0.8804362416267395, -0.33256858587265015, -1.6126327514648438, -0.3568260371685028, 1.0925195217132568, -2.940444231033325, 1.4639469385147095, 0.5410438179969788, -0.026712743565440178, 0.46497777104377747, -0.9449261426925659, 2.037294626235962, -0.7075104713439941, -0.07091283053159714, 0.32114049792289734, -0.19989781081676483, -0.6627373099327087, -0.5767390131950378, -0.5110672116279602, -0.049163300544023514, 2.204533576965332, -0.1725170612335205, 1.1662676334381104, -0.9525913000106812, -0.8590327501296997, 0.053006116300821304, 0.6402266621589661, -1.2356922626495361, -0.03117128647863865, -1.1713051795959473, 0.15329411625862122, -0.610207736492157, 1.1655521392822266, -0.06865257769823074, -1.022825002670288, -2.145936965942383, 1.5822720527648926, 1.9516651630401611, 1.1366474628448486, -1.167132019996643, 1.3650429248809814, 0.8863165974617004, -0.4574109613895416, -0.9445978403091431, 2.365140676498413, -0.12425693869590759, 0.5839931964874268, -1.7744311094284058], [-0.7514384984970093, 1.7876650094985962, -0.18452541530132294, 0.6475257277488708, 0.23419947922229767, -0.8179528713226318, -0.7600873112678528, -0.43887436389923096, -0.5720087885856628, 0.01293894276022911, 0.23318150639533997, -0.08004387468099594, -1.3417876958847046, 0.3661905825138092, -0.3877660036087036, -1.157180666923523, -0.30090388655662537, 0.35501518845558167, -0.7128844857215881, 0.037236735224723816, -0.2710738778114319, -0.7107152938842773, 0.4319082498550415, 0.13130466639995575, 0.24044229090213776, 1.1082046031951904, 0.20766304433345795, 0.27016326785087585, -1.3760349750518799, 1.3540539741516113, 0.2077939659357071, -0.41563910245895386, -0.1271757334470749, -0.41209352016448975, -0.3660365045070648, -1.943165898323059, -1.2822986841201782, -0.31963813304901123, 1.1035486459732056, -0.04028981178998947, -1.5943976640701294, -0.09434493631124496, 1.2918813228607178, -0.20943494141101837, 0.020209910348057747, -0.7525678873062134, -0.9611560702323914, 0.5979342460632324, -1.1214022636413574, 0.973377525806427, 0.515099287033081, 1.0860687494277954, -0.8421705365180969, -0.36363711953163147, -0.3088856637477875, -0.31683430075645447, 0.10962929576635361, -0.9911530017852783, -0.13186152279376984, -1.4239472150802612, -0.5679847598075867, -2.12945294380188, -1.2521812915802002, 0.003766619134694338, -0.6453173160552979, -1.2170679569244385, -1.49192476272583, 0.3088270127773285, -1.453066110610962, 0.44200584292411804, -0.8532068729400635, 0.26489749550819397, -0.3555615246295929, 0.946552574634552, 0.5885602235794067, 0.39788249135017395, -0.11475393921136856, 0.40897536277770996, -1.577483057975769, -0.13007183372974396, -0.5087064504623413, -3.913365364074707, 1.4066742658615112, -0.6377365589141846, -3.2009618282318115, -0.29072415828704834, 0.290374755859375, 0.19685906171798706, 0.900833010673523, -0.13753262162208557, 0.552893877029419, 0.3799780607223511, -0.9220564961433411, 0.39004334807395935, -1.7777799367904663, -0.13337938487529755, -1.8783254623413086, 0.9997181296348572, 0.037277594208717346, -0.6767539381980896, 0.5881136059761047, 2.310365915298462, -1.0148513317108154, 0.23900486528873444, 2.0985007286071777, -0.4620666205883026, 0.8985950946807861, 0.24022400379180908, 0.9832249283790588, -0.7169920802116394, -2.2697224617004395, -1.0079530477523804, 0.8987348079681396, 0.40115416049957275, 1.1785571575164795, 0.9624180197715759, 0.8935242295265198, 0.09563165158033371, -1.4137405157089233, 0.9687098264694214, -2.047250270843506, 2.273914098739624, 0.4595748484134674, 0.7756199836730957, 1.3425238132476807, 0.8558202981948853, 0.9567402601242065, 0.6818370819091797], [0.3150216042995453, 1.3229025602340698, -0.31711238622665405, 0.26112136244773865, 0.7201184630393982, -0.4440552294254303, -0.6509445309638977, 1.997796893119812, 0.10349638015031815, 1.1339757442474365, 2.3751728534698486, 0.3663339614868164, -0.537024199962616, 0.478665292263031, -0.7252113819122314, -0.926256000995636, 0.46449121832847595, -0.7860791087150574, 0.0024327216669917107, 0.591278612613678, -0.5869317650794983, 1.027775526046753, 0.06456936150789261, 0.08221190422773361, 0.5268551707267761, 0.5648001432418823, -1.66691255569458, -0.9216638207435608, 0.20627273619174957, -2.1031200885772705, 2.2926084995269775, -0.5246593952178955, 0.24914279580116272, 1.1210355758666992, -0.7506995797157288, -0.007669087499380112, -0.19733279943466187, 0.4640458822250366, 0.1252342015504837, 0.07808367908000946, 1.015509843826294, -0.3138625919818878, -1.9087835550308228, -0.27513936161994934, -1.5747689008712769, -2.091921091079712, -0.3192911446094513, -1.114411473274231, 0.06762823462486267, -0.5338783264160156, 0.044481903314590454, 0.5520485043525696, 0.29870471358299255, -0.6472238898277283, 0.4152451455593109, -0.07711531966924667, 0.36121901869773865, 1.0204023122787476, 1.012010097503662, 0.34976181387901306, 0.4286235272884369, 0.19503962993621826, 0.013563546352088451, 1.5261633396148682, -0.07784608751535416, 0.755659282207489, 0.10464134812355042, -0.22843188047409058, 0.26017308235168457, 0.2467806339263916, 2.5849716663360596, 1.5503381490707397, -0.8591768145561218, -0.1846776306629181, 0.5031746625900269, 0.3348053991794586, -0.7611238956451416, -1.5115046501159668, -2.6685287952423096, -0.23719972372055054, -1.356979489326477, -0.5213393568992615, 0.05102420970797539, 0.1880679428577423, -1.5980435609817505, -0.6698369383811951, -1.1313759088516235, 1.4433468580245972, -1.6610205173492432, 0.8315054774284363, -0.4109721779823303, -0.3686865568161011, 0.05919913575053215, 0.7048037052154541, -0.3171514868736267, 0.21671441197395325, 0.688790500164032, -0.019278433173894882, 0.3164944350719452, -0.7017083168029785, 0.8748452663421631, -0.5644981265068054, 0.08185382932424545, 0.6857990026473999, 0.31364718079566956, -0.4548443555831909, -0.27330049872398376, -1.9684375524520874, -1.5387990474700928, 1.9958500862121582, 1.4802312850952148, -1.5972620248794556, 1.5942003726959229, 1.19791579246521, 1.1998331546783447, 1.1672942638397217, 1.4372611045837402, -0.8439496159553528, -0.3137733042240143, 0.676242470741272, 0.16033287346363068, -2.1012210845947266, -0.6210862398147583, 0.46735885739326477, -0.010065740905702114, 0.43341752886772156, 0.9665812849998474, -0.6221611499786377], [-0.6931732892990112, 0.7154157757759094, -1.901537537574768, -0.4387417435646057, 0.6343092322349548, -1.067177414894104, -0.23270565271377563, 1.6134932041168213, -0.19359353184700012, 1.3946120738983154, -0.9189944267272949, 0.28201425075531006, -0.2610403895378113, -0.4294814169406891, 0.8243165016174316, 0.30259087681770325, 1.2228909730911255, 0.020763183012604713, 0.700625479221344, 0.09287505596876144, -1.0676980018615723, 1.592734694480896, -0.28220030665397644, 0.6677469611167908, -0.6129546165466309, 0.2680814862251282, -0.7336916923522949, 1.6713584661483765, -0.7587386965751648, -0.3393820524215698, -0.3171863853931427, 0.22269748151302338, 0.015388301573693752, -1.203568935394287, -1.0957022905349731, 0.38114669919013977, -1.6025290489196777, 0.06897246092557907, -0.801550030708313, -0.18306945264339447, 1.0450044870376587, -0.36792680621147156, -0.2719951570034027, 0.6492713689804077, -1.5071598291397095, -0.25702595710754395, -0.40221714973449707, -1.3800771236419678, 0.20119760930538177, 1.4122084379196167, 0.29496318101882935, -0.04760747402906418, -0.7459059357643127, -1.8604949712753296, -0.9141982793807983, -0.7472121715545654, -1.460134744644165, 0.3363099992275238, -0.4671074450016022, -0.02315266989171505, -0.39616528153419495, 1.2976624965667725, 0.6607766151428223, -1.2156034708023071, -0.292361855506897, -0.7143233418464661, 0.1818704903125763, 0.05400446057319641, -1.5899895429611206, -0.7025303840637207, -2.4627761840820312, 1.9680012464523315, -1.2511509656906128, -2.4622209072113037, 0.7970895171165466, 0.7784944176673889, -0.19158542156219482, 0.7759881615638733, -0.44622722268104553, 0.7866335511207581, -1.6497105360031128, 1.6118144989013672, -0.5906370282173157, -2.3443763256073, -2.1217899322509766, -0.4733923077583313, 0.32342225313186646, 2.4315998554229736, -0.04512638598680496, -1.17646062374115, -0.9728392362594604, -0.3749024569988251, -0.030264588072896004, 0.19537125527858734, 0.19854429364204407, 0.827774703502655, 0.10649026930332184, -0.5171186923980713, -2.020796775817871, -0.5946732759475708, 1.304234266281128, 0.013786002062261105, 1.5354548692703247, -1.4574663639068604, 0.20423556864261627, 0.5129984021186829, 0.28540295362472534, -0.6733002066612244, 0.2594938576221466, 0.8641485571861267, -0.47908732295036316, -2.555220603942871, -0.20659124851226807, 0.9301505088806152, 1.7403823137283325, 0.5169568061828613, -1.8400354385375977, -2.133295774459839, -0.4308835566043854, -0.290025919675827, -0.19563035666942596, -0.08169642835855484, 0.6117023229598999, 1.0527788400650024, -2.206892967224121, -1.8692126274108887, -0.030343158170580864, -1.1440263986587524], [-0.3195064961910248, -0.9000806212425232, -0.00779650267213583, 0.7199313044548035, 0.5644962787628174, 1.0741291046142578, 0.027251699939370155, 0.5247096419334412, -1.3038179874420166, 1.9010379314422607, 0.18315789103507996, -0.6039009690284729, -0.6130905747413635, -0.6189845204353333, -0.6804516315460205, 1.2528315782546997, 0.5105147957801819, 0.5539053678512573, 0.6804497241973877, 0.8651159405708313, -0.1787077635526657, -0.4169580638408661, -1.2352349758148193, -0.7734435200691223, 0.07215676456689835, -0.4305342137813568, -0.18121913075447083, -0.768796980381012, 0.5107847452163696, -0.6236332654953003, 1.5532763004302979, -0.04762621223926544, 1.4310603141784668, -1.1307238340377808, -2.211419105529785, -0.45308253169059753, -1.739364504814148, -0.2823926508426666, 0.02524845115840435, 0.859851062297821, 1.3595528602600098, 0.6072261333465576, -2.163447856903076, 0.29735103249549866, -1.801838755607605, -0.6950037479400635, 1.3322279453277588, -0.0367448516190052, 0.2479642778635025, 0.155410498380661, -1.6472798585891724, 1.2215839624404907, -0.5659969449043274, -0.35583287477493286, -0.8068262934684753, -0.549446165561676, 0.04144257679581642, 1.010787010192871, 0.3174806833267212, -0.07907886058092117, -1.3869508504867554, -1.0833075046539307, 0.6424002051353455, -0.1726813167333603, -1.3014602661132812, -0.7526232004165649, -3.006631851196289, 1.071537971496582, 1.414533257484436, -1.2037943601608276, -1.1643904447555542, 1.0742318630218506, -0.4941527545452118, 0.8604716062545776, -0.007927370257675648, -0.9514182209968567, -0.9837591052055359, -0.10963242501020432, -0.21532638370990753, 0.2505328357219696, -0.8009370565414429, 0.38100457191467285, -0.9692860245704651, 0.28430405259132385, 1.4161603450775146, -2.1623356342315674, 0.6798945069313049, -0.726822018623352, -0.29614612460136414, 0.19749197363853455, -0.6735495328903198, -1.100111484527588, 0.49253228306770325, 1.4299159049987793, -1.0762479305267334, -1.7458728551864624, 1.2126712799072266, -0.19584079086780548, 1.3591301441192627, 0.7080671787261963, -2.448418617248535, 0.9685931205749512, -0.7396999001502991, 1.6527959108352661, -0.6338944435119629, -1.1352708339691162, -0.03798169270157814, -1.1150939464569092, 1.6436283588409424, 1.2335416078567505, 1.1124833822250366, 1.1472028493881226, 0.867579996585846, 1.0531922578811646, -0.3282488286495209, 1.1512600183486938, -1.0920132398605347, 0.5757165551185608, 0.7590432167053223, -1.074188232421875, -0.12230171263217926, 0.664619505405426, -1.2525742053985596, -0.8793455958366394, 0.26966825127601624, 0.309781014919281, 2.716860294342041, 0.034697212278842926], [0.28385236859321594, 0.49548816680908203, 0.04879119619727135, 0.5064529776573181, 1.4170691967010498, 1.0730364322662354, -0.46378040313720703, 1.6777184009552002, 0.12633875012397766, -1.2523033618927002, 0.8958192467689514, 0.4394444227218628, 1.217315435409546, 0.8802675008773804, -1.3150700330734253, -1.9515516757965088, 0.8209222555160522, -1.6511293649673462, 1.8091983795166016, 2.0255696773529053, -0.9111452698707581, -1.883543610572815, 0.39216870069503784, 1.2232545614242554, 0.9509831666946411, 1.7522504329681396, 0.8413135409355164, -0.24601148068904877, -1.809975504875183, 0.6158125400543213, -0.6836929321289062, -0.34943363070487976, -0.8067507147789001, -1.1844494342803955, -1.7155160903930664, 2.223180055618286, -0.4352821409702301, 1.998738408088684, -2.1112301349639893, -0.3229517638683319, 0.9640598893165588, 0.8925865888595581, 0.2560376822948456, 0.5964938402175903, -0.6451162099838257, 1.107755184173584, 1.0412248373031616, 0.33926698565483093, 1.4470694065093994, 0.03008842281997204, 2.20666241645813, -0.9068015217781067, 0.4150276482105255, -1.6109281778335571, 0.7510877847671509, -1.6377432346343994, 0.29733455181121826, 0.6705491542816162, -0.11130266636610031, -0.20759892463684082, -0.2756447196006775, -1.5328259468078613, 1.3221029043197632, 1.4539324045181274, 0.511053204536438, 1.5277509689331055, -0.6159142851829529, 0.7234610319137573, -0.5322519540786743, -0.21035397052764893, 0.974124550819397, 0.41451483964920044, 1.4086958169937134, 0.1247294545173645, -1.0264474153518677, -0.877417802810669, -0.7838707566261292, -0.44248613715171814, -0.28273484110832214, -0.05172128602862358, 0.8978837728500366, 0.43531176447868347, 0.4466964304447174, -1.1939254999160767, -1.4052326679229736, 1.4292129278182983, 0.7858210206031799, 2.05922269821167, 0.27404606342315674, 0.5325826406478882, -0.7732357382774353, 0.2539973855018616, 0.418539434671402, 1.3139609098434448, 0.9613282084465027, 0.012195025570690632, 0.3042098879814148, -1.3484379053115845, -0.4061959683895111, -1.208538293838501, -0.2856697738170624, -0.2910955846309662, 0.3208426237106323, -1.6631972789764404, -0.2980119287967682, -0.004277638625353575, -1.7446597814559937, 0.6497793197631836, -0.38138967752456665, 0.76155686378479, -0.5107035040855408, -1.9886980056762695, 0.5128013491630554, 1.6572805643081665, -0.8079398274421692, 0.7318899035453796, -0.32489755749702454, -0.9007245302200317, -0.2781646251678467, -0.6518648266792297, -0.07932861149311066, 0.7225682139396667, -0.5836015939712524, -0.45289358496665955, 1.0865715742111206, -0.813671886920929, 0.6234909892082214, 1.983241081237793], [-1.6015080213546753, 1.7083481550216675, -1.4703576564788818, -1.145937204360962, -0.21898137032985687, 0.13818196952342987, -0.5132145881652832, 1.2495954036712646, -1.536609172821045, -0.36660337448120117, 1.0359783172607422, 1.4577996730804443, -0.6499732732772827, -1.6078782081604004, -0.01700221374630928, 0.5666564702987671, -0.3180152177810669, 1.361931562423706, 1.612396478652954, -0.0993112325668335, 0.7967976927757263, -0.5801258087158203, 0.13279198110103607, 0.37095844745635986, 1.1475274562835693, -0.4908497929573059, -0.06822175532579422, -0.2549569010734558, -0.8509354591369629, 1.092057228088379, 0.35205334424972534, -0.32591792941093445, 0.22407151758670807, -1.0381604433059692, -0.3380867838859558, 1.8476769924163818, -0.8611261248588562, 0.04084956645965576, 0.7623947262763977, 0.8069690465927124, -0.5086444616317749, -0.13292080163955688, 0.9064065217971802, 0.0021440000273287296, 0.9449922442436218, 0.7156099081039429, 2.2440097332000732, 0.03445990011096001, -0.6878041625022888, 1.403617024421692, 1.8410931825637817, 1.2283694744110107, -0.7788668274879456, 0.3635922968387604, 0.5510674118995667, -1.0122740268707275, -0.53537917137146, 0.23439234495162964, 0.2043444812297821, -0.3426993191242218, -0.18488861620426178, -1.1171048879623413, -0.2391592562198639, 0.12567713856697083, 1.2019238471984863, -1.1939529180526733, -1.2165533304214478, 0.9184771776199341, -0.29146328568458557, -0.3569357693195343, -0.39495226740837097, -1.2696723937988281, 0.3523561954498291, -0.6968569159507751, 0.9622945189476013, -0.01898966357111931, -0.6744415163993835, 0.34094753861427307, -0.8434404134750366, 0.49895867705345154, 0.009291648864746094, 0.6649344563484192, -0.9786757826805115, -0.9108564257621765, -0.7765651941299438, 0.3873082101345062, -1.7821046113967896, 2.564877986907959, -0.32821595668792725, 1.448074460029602, -1.067667841911316, 0.3997957706451416, -0.25432461500167847, 0.6149337291717529, -2.4626715183258057, 1.1933847665786743, 2.122621774673462, 0.820419430732727, -0.1230105385184288, 1.165751338005066, -0.31545764207839966, 0.7111151814460754, -0.23481804132461548, -0.599324107170105, 0.4920911490917206, 0.2452482283115387, 0.13566239178180695, 0.5056699514389038, -0.5840802788734436, -0.021965691819787025, -0.6278289556503296, -0.008169972337782383, 0.166852205991745, 0.6120385527610779, 0.5347495675086975, 0.34363266825675964, -0.18081043660640717, -0.9908987879753113, 0.03145194798707962, 1.5938833951950073, 0.98526531457901, -1.072269082069397, -0.9163285493850708, 0.23356525599956512, -0.2777491807937622, 0.6669449210166931, -1.1551296710968018, 1.3329144716262817], [-0.24307700991630554, -0.6575393080711365, -0.028431963175535202, -0.2844676673412323, -0.7686806321144104, 1.3680588006973267, -1.1652811765670776, -0.9586372971534729, -1.3945363759994507, 1.5506510734558105, 0.32563403248786926, -0.49174681305885315, 0.8466713428497314, 1.1342740058898926, -0.8449051380157471, 1.3019144535064697, 0.7646569609642029, 1.9962643384933472, 1.267159104347229, 0.6210744976997375, 1.8025845289230347, 0.21052183210849762, -0.4142565131187439, -0.4056083858013153, 0.43620219826698303, -1.270909070968628, -1.9171925783157349, 0.9921433329582214, 0.19735880196094513, 0.28780725598335266, -0.7311092615127563, -0.058764632791280746, -1.4042203426361084, -1.65461003780365, 0.891538143157959, 0.34594765305519104, -0.35043013095855713, -1.787903904914856, 0.9766891598701477, 0.7261414527893066, -0.4831160306930542, -0.8419263958930969, 0.5477914810180664, 0.4114735424518585, -0.31442877650260925, -0.24206961691379547, 0.30155789852142334, 1.3484495878219604, -1.3625980615615845, 0.6755140423774719, 1.0962735414505005, -0.5654622912406921, 0.013305941596627235, -0.6002437472343445, 0.3762199580669403, 0.7636286020278931, 0.3162877559661865, -0.42179712653160095, 0.2621725797653198, 0.8981115221977234, 0.9774752855300903, -0.9327704310417175, -0.054509758949279785, 0.5048276782035828, -0.6135635375976562, 0.27532321214675903, -1.2257838249206543, -0.45319926738739014, -0.02017834037542343, -1.1471717357635498, 0.6840828657150269, -0.2976306974887848, -0.3840833604335785, -0.17570123076438904, -0.7161846160888672, -0.3972632586956024, -0.16957156360149384, -1.126919150352478, -0.8105515837669373, 1.9705536365509033, 1.8709839582443237, 0.2068680077791214, -0.4647487699985504, 0.9319584965705872, -1.2049800157546997, 0.33826979994773865, 0.39467373490333557, 1.1263536214828491, -0.8222817182540894, 0.9165626168251038, 1.0693249702453613, 1.709841251373291, 1.4336576461791992, 0.4154238700866699, -0.9007406234741211, 1.6371514797210693, -0.16222308576107025, -2.3548567295074463, -0.7074117660522461, -0.4578084945678711, 0.07705754041671753, -0.18839284777641296, 0.28721916675567627, 1.6624113321304321, -0.8410131931304932, 1.0630537271499634, 1.3300862312316895, -1.100724697113037, 1.1261159181594849, 0.1899295151233673, -0.2998758852481842, 1.1618186235427856, 0.4371272921562195, -1.2078076601028442, 1.0257925987243652, -0.40347352623939514, -0.049634288996458054, 0.26158860325813293, 0.5182268619537354, -0.9901565909385681, -1.2337161302566528, -1.0926090478897095, 0.007325420156121254, 0.2625093162059784, -0.014050556346774101, 0.755939245223999, 0.3166183829307556, 0.8081046938896179], [0.8629504442214966, -0.5650230050086975, -0.580757737159729, -0.2434488832950592, 0.8578222393989563, -1.5243431329727173, -1.3847047090530396, 1.4466992616653442, -0.9926925897598267, 0.7520524859428406, -0.43534955382347107, -0.5436500906944275, 1.2775858640670776, -0.7264047265052795, 0.19483378529548645, 0.4886927008628845, -1.289710283279419, 0.3635764718055725, -1.2638392448425293, 0.025075865909457207, 0.46659258008003235, -1.7382234334945679, -0.7863590121269226, 0.12540249526500702, 0.5956776738166809, -0.12827517092227936, -0.20187613368034363, 0.8511190414428711, 1.9601013660430908, -0.5792720913887024, 1.5119948387145996, -0.34256646037101746, 1.491677165031433, 0.7252174019813538, -1.244367003440857, 0.815463125705719, 0.6480402946472168, 0.5326828360557556, 1.1585131883621216, -1.0117824077606201, 0.42268747091293335, -0.8999729752540588, 2.0597927570343018, 0.22672139108181, -1.395398736000061, 1.5949712991714478, -0.3877086341381073, 0.043865613639354706, 0.5829578638076782, -0.07831504195928574, 1.3940240144729614, -0.7715445160865784, -0.14676614105701447, -1.216050386428833, -0.7406434416770935, 2.0236151218414307, 0.26647740602493286, 2.6204988956451416, -0.11192755401134491, 0.9613311886787415, -0.42774930596351624, -0.7607734799385071, -0.43491417169570923, -2.7046139240264893, -1.4253599643707275, -1.0671945810317993, -0.46159180998802185, -0.6689049005508423, -1.2926607131958008, 0.5539661049842834, 1.6431055068969727, 0.23872779309749603, 0.6577457189559937, -0.39218980073928833, -0.6271362900733948, 0.9260771870613098, 0.5770087242126465, 0.7372595071792603, 1.5467050075531006, -0.07351985573768616, -0.09923961758613586, -0.6806169152259827, 1.4563671350479126, 0.055722810328006744, 1.4788475036621094, -0.2927452623844147, -1.2264528274536133, -1.3624011278152466, -1.5668847560882568, 0.8697587251663208, -0.4409201741218567, -0.781838595867157, -0.29733213782310486, 0.2603284418582916, 0.14549089968204498, 0.4909093677997589, -0.35980501770973206, 0.031475380063056946, 0.564946174621582, -1.4179271459579468, -1.4081416130065918, 0.16844479739665985, 0.47075119614601135, -0.6337170600891113, 0.7888874411582947, -0.8699222207069397, -0.1909901350736618, -0.6497540473937988, -1.3408125638961792, 0.6013335585594177, 0.857905387878418, -0.7643713355064392, 2.8952620029449463, -0.33617836236953735, 1.0953766107559204, -1.5452654361724854, -0.7293042540550232, -0.6566927433013916, 1.4240226745605469, 1.2975794076919556, 0.0590878501534462, 0.8940995931625366, 0.775481104850769, 0.8154813051223755, 0.23501892387866974, 2.048168182373047, 2.041517734527588, -0.40937742590904236], [0.630527675151825, 1.1167099475860596, 0.5773797631263733, -1.1666936874389648, -0.20423364639282227, -0.12943792343139648, 0.6823202967643738, -0.7278050780296326, 0.015735529363155365, 0.1432938128709793, -0.972726047039032, -0.01861373893916607, -0.041184037923812866, -0.23966507613658905, -0.758276104927063, -1.8246865272521973, -0.11242256313562393, 0.4706744849681854, 0.12887616455554962, -1.265640139579773, -1.6993961334228516, -0.20133936405181885, 1.356943130493164, 0.13425327837467194, 0.39895662665367126, 1.3514577150344849, -0.39720383286476135, 0.3343423008918762, -0.9134420156478882, 0.16450142860412598, -0.5566185712814331, 0.2995785176753998, 2.1499602794647217, 1.7186295986175537, 2.0550241470336914, 0.5855568051338196, 1.0309473276138306, 1.1137715578079224, 0.10259900987148285, -0.2864040732383728, 1.8035387992858887, -0.5922171473503113, 0.6839326024055481, 0.482038676738739, -0.842116117477417, -0.40987691283226013, -0.5694090127944946, -0.31337970495224, -0.5629380941390991, -0.6067894697189331, 2.1821722984313965, -0.8622442483901978, -1.123761773109436, -0.07975298166275024, -0.6762461066246033, 1.4096815586090088, -0.5392870306968689, -1.0944571495056152, -0.6445599794387817, -0.9504204392433167, 0.27734747529029846, 0.19591064751148224, -1.5645588636398315, -2.9669318199157715, 0.2546710968017578, 0.3377751410007477, -0.29828235507011414, -0.6683251857757568, 0.6860413551330566, -0.30332839488983154, 1.7910627126693726, -0.23409464955329895, 0.38356930017471313, -1.2916680574417114, 1.8813543319702148, 1.337127685546875, -1.8674613237380981, -0.693773090839386, -0.6265044212341309, 0.21039454638957977, 0.08585646003484726, 0.29562920331954956, -1.1175354719161987, -2.4023566246032715, -0.6957089304924011, -0.8327818512916565, 1.3417859077453613, -0.26304903626441956, 2.496986150741577, 0.6002850532531738, -0.8550732731819153, 1.2521932125091553, 0.475785493850708, 1.131496548652649, -0.7505886554718018, 1.3283166885375977, 1.1062885522842407, 0.0022400631569325924, -0.031035592779517174, -0.6575331091880798, 1.7037057876586914, 0.5026508569717407, 0.20810562372207642, 0.9779855012893677, 0.4996776580810547, 1.1279948949813843, 1.467631220817566, 1.383221983909607, -0.24271847307682037, -0.9065336585044861, -0.7636026740074158, -0.5286493897438049, -1.2365928888320923, -0.44164833426475525, -0.9055131077766418, 1.101684808731079, 1.585908055305481, -0.015644796192646027, -0.6838741898536682, 0.8387162685394287, 0.7277409434318542, -1.4548486471176147, 1.3566612005233765, -0.8309329748153687, -1.5478214025497437, -0.17256532609462738, 1.6359578371047974, 0.10690879821777344], [-0.2685145437717438, -0.9642971754074097, -0.5644928812980652, 0.46569642424583435, -0.3193870186805725, 0.47576072812080383, -0.25377193093299866, 0.38016608357429504, -0.01114364992827177, -0.4989801049232483, -0.03933035954833031, 1.5830808877944946, -0.822580099105835, -1.7152605056762695, -2.236215591430664, -0.8627592325210571, 0.7203444242477417, 1.6152859926223755, 0.41652509570121765, -0.7832414507865906, 0.6361833810806274, 0.1553652286529541, 0.5575217008590698, -1.5987178087234497, -0.9343347549438477, -0.5055623650550842, -0.9188657999038696, -0.09587831795215607, -0.5143592357635498, -1.2424731254577637, 0.7041764259338379, 1.0114177465438843, -0.1446721851825714, 1.0993491411209106, 1.5041804313659668, 0.7451491355895996, -0.8323767781257629, 0.12117790430784225, -0.7879189848899841, -1.9487292766571045, 0.16052183508872986, 0.8206382989883423, 0.05640243738889694, -1.714057445526123, -0.06303856521844864, -1.3210875988006592, 0.18434575200080872, 0.6568261384963989, 0.9582630395889282, 0.012115748599171638, 0.7162569165229797, 0.6982558965682983, -0.9188044667243958, -0.42488279938697815, 0.4383692741394043, 0.9529396295547485, -0.9091134071350098, 1.9721026420593262, 0.725783109664917, 2.29943585395813, -1.4189162254333496, 1.4467893838882446, 0.22642701864242554, 1.5685375928878784, -1.93328857421875, -0.8836246132850647, -0.34811413288116455, -0.3250971734523773, 0.042594410479068756, 1.537543535232544, 0.6823397278785706, 1.1337072849273682, -1.1047720909118652, -0.5727764964103699, -1.3705744743347168, 1.0398393869400024, -0.027151010930538177, -1.8561980724334717, 1.501543641090393, 1.2287523746490479, -2.0139079093933105, -0.6782679557800293, -2.2812235355377197, -1.355806827545166, 1.2347108125686646, -0.21415387094020844, 0.46874603629112244, 0.29436194896698, 0.16568239033222198, 1.0987685918807983, -3.0642292499542236, -0.47993454337120056, 1.3555022478103638, -0.7411549687385559, -1.057732343673706, -1.3292804956436157, -0.6226372122764587, -0.1799962967634201, -0.781035840511322, -0.14607307314872742, 0.8561006188392639, 0.5803797245025635, -0.6415521502494812, -1.1558936834335327, 0.8163569569587708, -0.44014644622802734, -0.6532779335975647, -1.1468528509140015, 0.17423979938030243, -1.68118417263031, -2.469431161880493, -0.4332216680049896, 1.7497304677963257, 1.7885024547576904, -1.742850661277771, -0.8216127753257751, -1.4286928176879883, -2.064990520477295, 1.2963117361068726, 0.057854581624269485, -2.0280983448028564, -2.3527567386627197, -0.419355571269989, 0.3195107579231262, 0.15128657221794128, 0.6848876476287842, 0.07043538987636566, 0.9290240406990051], [0.6875327229499817, -0.6990479230880737, 0.6448693871498108, 0.6691114902496338, -1.131746530532837, 1.494178295135498, 0.21731184422969818, 1.5045970678329468, 0.20214006304740906, -0.8930855989456177, 0.2328585833311081, -0.6629664897918701, 0.3978551924228668, 0.0813651978969574, -0.5691048502922058, -0.2278931736946106, -0.9274947643280029, -0.9155572056770325, 0.399801641702652, -1.327746868133545, 0.5870356559753418, 0.14242589473724365, 0.07706678658723831, -1.4447263479232788, -0.1717938780784607, -1.7729291915893555, 1.3687691688537598, 0.7770280241966248, 0.01659277081489563, -1.580962061882019, 0.5072579383850098, 0.3190677762031555, -1.3249030113220215, 0.4157037138938904, -0.5437216758728027, -0.5593048930168152, -0.039992619305849075, 0.03701487556099892, -0.49494510889053345, 1.0991867780685425, 0.35093918442726135, 1.2658286094665527, -1.2334855794906616, 2.64396595954895, 0.8275249600410461, 0.9325824975967407, -0.18936939537525177, -0.28773054480552673, -2.6372644901275635, -0.4971829950809479, -0.00457165390253067, 0.7196036577224731, 1.8674731254577637, 0.46397462487220764, 0.11306324601173401, -1.2426280975341797, -0.2974499464035034, -0.20842786133289337, -0.5088187456130981, -0.057402923703193665, 0.5067251920700073, 0.10521191358566284, 0.923630952835083, 2.0561788082122803, -0.9168256521224976, 0.2407151311635971, 0.5676459074020386, 1.2225497961044312, 0.33604612946510315, -0.2021128237247467, 0.41567766666412354, -0.2110777646303177, -1.7519464492797852, -0.5867804288864136, -1.404999017715454, 0.7087294459342957, 1.2667806148529053, -0.4983779489994049, -0.9826879501342773, 0.07782056927680969, -0.8947595357894897, -0.5841363668441772, 0.26572293043136597, 0.7366328835487366, 1.5843828916549683, -0.5900309085845947, 0.015032588504254818, -1.011080026626587, -0.3119608163833618, -1.1045541763305664, 0.42826318740844727, -0.5358708500862122, -0.26983463764190674, 1.1327372789382935, 0.5007541179656982, 0.4470658302307129, -0.24896198511123657, 0.3561106324195862, 0.5753005743026733, 1.5129488706588745, -0.48287472128868103, -0.9412567019462585, 0.027134345844388008, 0.1096217930316925, -0.28176406025886536, 0.6281770467758179, -1.2866811752319336, 0.10307804495096207, -0.4298889636993408, 0.6153203845024109, 0.25000327825546265, -0.8326479196548462, 1.188071370124817, 0.47691410779953003, 0.3699812591075897, -0.640632688999176, -1.3457218408584595, -1.69032883644104, -0.05684350058436394, 1.0542906522750854, 0.16384337842464447, -0.4335925281047821, -1.2021539211273193, 1.8494077920913696, 0.4242740869522095, 1.062804102897644, -0.5166356563568115, -0.14378152787685394], [1.0828850269317627, -1.043185830116272, 0.484809547662735, 1.1125526428222656, 1.0900354385375977, -0.07080663740634918, -0.7842194437980652, 1.167866587638855, -0.5476298928260803, -1.4223833084106445, -1.0458030700683594, -0.6364363431930542, -3.139719009399414, -0.6486035585403442, 0.9872317910194397, -0.8564687371253967, 0.0259651280939579, 2.094815969467163, -0.9202114343643188, -1.1309672594070435, -1.0615113973617554, -0.758682370185852, -0.45646023750305176, -0.6611680388450623, 1.1209604740142822, -1.915219783782959, 1.0693538188934326, -0.2520296573638916, 1.2085998058319092, -1.0126006603240967, -0.37916308641433716, -0.7076258659362793, -0.6094344854354858, -0.13174033164978027, -0.4122907519340515, 0.4949747920036316, 1.1070410013198853, 0.4791516661643982, 0.1676960289478302, -0.3713599145412445, 0.8807001709938049, 1.650209903717041, -0.506750226020813, -0.27678000926971436, -0.37401604652404785, 0.3503193259239197, 0.5517043471336365, 0.3082509934902191, 1.477726936340332, -0.19534313678741455, -0.33032935857772827, -0.37469810247421265, 0.05847936123609543, -1.8053863048553467, 0.023339882493019104, 0.03186599165201187, -0.41195163130760193, 0.9324195981025696, -0.3116166293621063, -0.11749826371669769, 0.06207869574427605, 1.203042984008789, -0.07412776350975037, 0.7262385487556458, -0.7108239531517029, 1.6788922548294067, 2.3770289421081543, 0.5974072217941284, -0.7545536160469055, 0.8259047269821167, -0.9399678707122803, 0.22376854717731476, 0.24190360307693481, -0.7111555933952332, 0.6179551482200623, -0.05234575271606445, 2.240302324295044, -0.39869236946105957, 0.7571436166763306, 1.645331859588623, 0.4356859028339386, 0.3906651735305786, -0.43107709288597107, 0.19221146404743195, 0.6774627566337585, 1.825186848640442, 0.6112116575241089, 1.0795536041259766, -0.18200930953025818, -1.5147966146469116, -0.4928475320339203, 0.9191567301750183, 0.1878088414669037, -0.34636861085891724, 1.2117137908935547, -0.6392501592636108, -1.234081745147705, 0.40529218316078186, -0.6436198949813843, 0.2645454704761505, 0.8810487985610962, -1.4005236625671387, 1.8794684410095215, 0.5192140936851501, -0.5022281408309937, -0.9740509390830994, -1.9286574125289917, 0.15813902020454407, 1.0548186302185059, -1.3131036758422852, -0.18892914056777954, -1.1716549396514893, -0.37257882952690125, -1.5908246040344238, -0.27181580662727356, -1.1744316816329956, 0.5508080720901489, 1.3048064708709717, 0.8652123808860779, -0.8373660445213318, -1.0424776077270508, 0.19148938357830048, -0.5980005264282227, -0.0003982710768468678, 1.1411436796188354, -0.06532531976699829, -1.1431280374526978, 1.3738137483596802], [0.9875635504722595, 0.7106571197509766, -0.3703140914440155, -1.8033064603805542, 0.8368107676506042, 1.2111626863479614, 1.7724820375442505, 0.7357441186904907, 0.46409568190574646, 0.09399328380823135, -0.392843633890152, -1.412471055984497, 2.628887176513672, 0.4315565228462219, -0.6473241448402405, 0.6787309646606445, -1.4508552551269531, -0.773688554763794, 1.3448947668075562, -1.0610265731811523, 0.38108524680137634, 0.13200899958610535, -0.734281063079834, -0.93482506275177, 1.1404211521148682, 1.5744000673294067, 0.16653305292129517, -1.2212133407592773, 1.7503395080566406, -0.5763489007949829, 0.8640809655189514, 0.24464240670204163, 0.2977394461631775, -1.0539588928222656, -1.040305495262146, 2.094228982925415, -2.0327823162078857, 0.9785243272781372, 0.8243180513381958, 1.3804811239242554, -0.4746348261833191, -0.9078764915466309, -0.902886688709259, 0.013767242431640625, 1.3938014507293701, 0.020918190479278564, 0.9412503838539124, 1.3440444469451904, 0.22754411399364471, -0.3265666961669922, 0.02788688614964485, 0.9835500121116638, 2.0209391117095947, 0.5560780167579651, 0.3722682595252991, 0.9114773869514465, -0.8587914705276489, 0.03630368784070015, -1.1592680215835571, 1.1669045686721802, -0.9220910668373108, -1.5923572778701782, 0.6068839430809021, -1.6514190435409546, -0.6597515344619751, 0.14963527023792267, -0.2829686105251312, -1.4861692190170288, 0.9881991744041443, 1.1165293455123901, 0.44544389843940735, 0.015762783586978912, -1.338854432106018, 0.4954797029495239, -0.7669620513916016, -0.7125805616378784, 0.8292396068572998, -0.13989147543907166, -0.5088364481925964, -0.8488661050796509, -1.1012250185012817, 0.4480779767036438, -1.5390444993972778, -0.6134766936302185, 0.5985565185546875, 1.4980674982070923, 1.8100931644439697, -0.32414302229881287, -0.12186142802238464, -0.11086864024400711, 0.3031907379627228, 0.7633602023124695, -0.5228243470191956, 0.18210144340991974, 0.08778555691242218, 0.1567901223897934, -0.8203081488609314, -1.0010861158370972, -0.9563438892364502, -1.0894194841384888, -0.24535703659057617, -0.1769663691520691, -1.1999499797821045, -0.8262958526611328, -0.8563947677612305, -0.5290654897689819, -0.6792460083961487, -0.8646905422210693, -1.2336472272872925, 0.6184349060058594, -0.7635157108306885, 1.061342716217041, 0.2429436892271042, 0.25305619835853577, 1.0076320171356201, 0.49316534399986267, 0.1187102347612381, 0.7952638864517212, 0.06198597699403763, 0.5009201169013977, -1.7474297285079956, 2.363365650177002, 1.1529146432876587, -0.5069460868835449, -0.175327330827713, -0.9075943827629089, 1.347048282623291, -0.12480194866657257], [1.1058719158172607, 0.24985001981258392, -0.4958631992340088, 0.3049474060535431, -0.5250764489173889, 0.8755057454109192, 1.7571520805358887, -1.189368486404419, 0.695101261138916, 1.436816692352295, 0.7664560079574585, -1.3701980113983154, 2.529409408569336, -0.07542110979557037, 1.0302101373672485, -1.2845687866210938, -2.065551519393921, 0.5011000037193298, 0.9954838752746582, 0.9143640995025635, 0.3858250379562378, 0.549329400062561, -0.5821005702018738, -0.001905615907162428, 0.3401605188846588, 0.47000154852867126, -0.5089500546455383, 0.7681246399879456, -0.5059454441070557, -0.7003998756408691, -0.6350627541542053, -1.1389449834823608, -0.10030554234981537, -1.760684609413147, -0.5021637082099915, 0.037290602922439575, -0.39088383316993713, 0.04167487844824791, -0.34777650237083435, 0.1978990137577057, -1.1438814401626587, 1.5881963968276978, 0.44954946637153625, -1.6332663297653198, 0.32933327555656433, -0.9195049405097961, -1.1443463563919067, -0.49253666400909424, -1.1081669330596924, 0.430682510137558, 0.3319668769836426, 0.033427126705646515, -0.5207065343856812, -0.21600472927093506, 0.8283418416976929, 0.23753543198108673, -0.9776282906532288, 0.35169050097465515, -0.6272318363189697, -0.826063334941864, 1.4813852310180664, 0.8791214823722839, -1.2824442386627197, 0.12023629248142242, -1.587345838546753, 0.7171316146850586, 0.2788964807987213, 2.3098058700561523, -0.19837942719459534, -0.6433173418045044, 0.057161808013916016, 0.8806589841842651, 0.03750273957848549, -1.0430784225463867, -1.3209342956542969, -0.5680311918258667, 1.0458241701126099, 0.6361413598060608, 0.041411515325307846, 0.7785686254501343, 0.46592846512794495, -0.5575410723686218, 0.04015839844942093, -0.06581961363554001, 1.2385566234588623, -0.6227011680603027, -1.496935248374939, -0.5851626396179199, -1.2281980514526367, -1.083221435546875, -1.7359305620193481, -0.043577924370765686, -0.3817480504512787, -0.050982605665922165, -0.904679536819458, -1.447031855583191, -1.288280725479126, -0.1465943604707718, 1.8257360458374023, -2.1957406997680664, 2.2096195220947266, 0.7784435749053955, -0.15994305908679962, 0.882845401763916, -0.08181723952293396, -0.4702579081058502, 1.333179235458374, -1.6168240308761597, -0.14458346366882324, -1.879597544670105, 0.7644445300102234, 0.07202822715044022, -0.45346975326538086, 0.18218815326690674, 1.1837276220321655, 0.8511608839035034, 0.6425119042396545, 1.4685752391815186, -0.8867225050926208, -0.5443267822265625, -1.1812248229980469, 0.49785372614860535, -1.0070611238479614, -1.1111022233963013, -0.7238017916679382, 0.2185298204421997, -1.4509156942367554, 0.1510181874036789], [-2.0657687187194824, 0.4442603290081024, -1.2835516929626465, -1.4786286354064941, 0.6030573844909668, -1.5815837383270264, 1.1533154249191284, -1.5539571046829224, 2.1405999660491943, -0.12644647061824799, -0.30485212802886963, -1.8806077241897583, 0.6710077524185181, -1.5341408252716064, 0.41913995146751404, 0.011328546330332756, 0.6309918165206909, 0.12390344589948654, -0.4108574688434601, 1.6303412914276123, -0.9793126583099365, 1.7359448671340942, 1.4341908693313599, -0.45974552631378174, -0.4027661979198456, -1.8892557621002197, -0.8229811787605286, 0.18028272688388824, -0.25759705901145935, 0.7616479992866516, -0.7082675099372864, -1.3085427284240723, 1.25983726978302, 1.3869059085845947, -0.7556940317153931, -0.8067017793655396, 0.8029425144195557, 0.3160765767097473, -2.022771120071411, 0.5598899126052856, -0.37245962023735046, 1.743863821029663, -1.3947945833206177, 0.06123363599181175, -0.2797735035419464, -0.7546589970588684, -0.5700068473815918, -0.42963892221450806, 1.2816641330718994, 1.9835224151611328, 1.5718433856964111, 1.109383225440979, -0.36284956336021423, -0.9440135955810547, -0.8251965045928955, -1.084912657737732, 0.8119699954986572, 0.1202327162027359, 0.7676766514778137, 0.8992988467216492, 0.5921011567115784, 0.39065423607826233, 1.2571237087249756, -0.015935607254505157, -0.8181871175765991, 0.13504569232463837, 0.4048058092594147, 0.27592164278030396, -0.5176492929458618, -0.12331607192754745, -0.38686656951904297, 1.8202227354049683, -0.7468834519386292, 0.798725962638855, 1.019317865371704, -0.5756940245628357, 0.8121436834335327, -0.3500230312347412, 0.44246774911880493, -0.6215612888336182, -1.010300636291504, -1.0349929332733154, -1.2104662656784058, -0.38864895701408386, 0.8442561030387878, -0.18556439876556396, -1.6382521390914917, -0.6644630432128906, -0.8028655648231506, -2.17606520652771, 0.45164060592651367, 0.5476129651069641, -0.7796578407287598, 0.9296051263809204, -0.5204445123672485, -1.0702334642410278, -0.8349510431289673, -0.3657158613204956, -0.8605725765228271, -1.7848652601242065, -0.9774037599563599, -0.7129999995231628, 0.030429868027567863, -0.4602736830711365, 0.9945288896560669, 0.4464389383792877, -0.5467206239700317, -0.37711846828460693, 1.9853421449661255, 0.35976484417915344, -0.14460539817810059, -2.533923864364624, -1.0684194564819336, 0.2659176290035248, 0.9743749499320984, 0.0186961367726326, 0.7542034983634949, -0.6939195990562439, 1.6482101678848267, -0.055512964725494385, 1.4430510997772217, -0.12165561318397522, -0.9113843441009521, -0.02937544696033001, -0.08458665013313293, 0.2718493938446045, -1.0723488330841064, 0.25663772225379944], [0.2525128126144409, 1.3654505014419556, 1.5733754634857178, -0.8662809133529663, 1.1669361591339111, 0.01460704393684864, 0.43369919061660767, 0.2423851639032364, -0.6593736410140991, -0.2841282784938812, -0.5616608262062073, 0.06987373530864716, -0.6901443004608154, -1.523279070854187, -0.05164384841918945, 1.0755023956298828, 0.42868733406066895, -0.07464397698640823, -0.4638644754886627, -0.1644780933856964, -0.677381157875061, 0.10136822611093521, 0.7423426508903503, 0.6885100603103638, 0.05307850241661072, -0.779905378818512, 0.33198317885398865, 1.5853500366210938, -0.07129368931055069, -0.9089173078536987, 1.5434684753417969, -0.2882164418697357, 0.6350427269935608, 0.002355640521273017, -1.2058310508728027, -2.1492395401000977, -0.0007924233213998377, -1.1077338457107544, -1.245634913444519, 1.2516423463821411, 1.0141785144805908, 0.3743586838245392, -0.840750515460968, -1.7724480628967285, 0.5510990619659424, -0.33233514428138733, 1.5263597965240479, 1.2176350355148315, -0.9603660106658936, 0.9754586219787598, 0.1651579737663269, 1.8947269916534424, -1.1579651832580566, -0.17000991106033325, -0.36877110600471497, -0.8381294012069702, -0.8441891670227051, 0.638219952583313, 1.1812478303909302, -0.3223109543323517, 0.6995672583580017, 0.7338795065879822, -0.3174026310443878, -1.8152966499328613, -0.7065332531929016, -0.17244389653205872, -0.10400860756635666, 1.0793639421463013, 1.222424030303955, 0.794625461101532, -0.44659626483917236, -0.3915954530239105, -0.5264186263084412, 0.6737996935844421, 1.0742660760879517, 0.6597336530685425, 0.897725522518158, -0.5971322655677795, -1.601253628730774, 0.5692500472068787, -1.0955039262771606, 1.4052832126617432, -0.7778180837631226, -1.1342524290084839, -1.0126111507415771, 1.1255213022232056, -0.5930168628692627, 0.9290914535522461, 1.1052117347717285, 1.7187258005142212, -0.47232770919799805, 0.977793276309967, -0.7133373618125916, 0.7424631714820862, 0.04344656690955162, 1.2522666454315186, -0.16163696348667145, -0.8788609504699707, -1.5412615537643433, 2.6106956005096436, 0.9028280377388, -0.26934728026390076, -0.017409687861800194, -1.550903081893921, -1.4492077827453613, 0.26980915665626526, 0.9236029386520386, 0.009009521454572678, -0.18405047059059143, -1.5370631217956543, 1.3132628202438354, -1.7288624048233032, -0.7492702603340149, -0.044876549392938614, 1.3805336952209473, -2.091261148452759, -0.080775186419487, -0.4825155735015869, -1.4446821212768555, 0.2105221450328827, -0.49816784262657166, 2.109931707382202, 0.14831481873989105, 1.5527011156082153, -0.6466114521026611, -0.48232728242874146, 0.10203985869884491, -0.9458127617835999], [-0.8253836631774902, -0.32939064502716064, -2.2816925048828125, -0.05642727017402649, 0.5303189158439636, -1.0805230140686035, 0.6962999701499939, 0.4447301924228668, -0.030507130548357964, 0.09843431413173676, 0.5106981992721558, -0.2976751923561096, -1.5580713748931885, -0.3844568729400635, 1.0363935232162476, 0.6988388299942017, -0.44476789236068726, -2.0233633518218994, 0.4978199005126953, 0.640159547328949, 0.8144403100013733, -0.7204949259757996, -0.0005679088644683361, 0.028908083215355873, 1.1770132780075073, 0.3608643710613251, 0.8381146788597107, 0.5801121592521667, -0.14993101358413696, 1.264432430267334, -0.19520308077335358, -2.554110288619995, -0.682018518447876, 0.5356997847557068, -0.7435308694839478, -0.8812665939331055, -0.3516343832015991, 0.9418008923530579, 1.091576337814331, -1.008359670639038, -0.3490547239780426, 1.115903377532959, -2.66265869140625, -0.15041399002075195, 0.5118740797042847, 0.8248732686042786, -0.6060695052146912, 1.8004686832427979, -1.352373719215393, -1.2247987985610962, -0.3140398859977722, -0.47712036967277527, -0.5777231454849243, -2.2792232036590576, 2.065816640853882, -0.2597368657588959, 0.05625029280781746, 0.8779167532920837, -0.5218720436096191, -0.3975057899951935, 0.6492283344268799, -0.04963495582342148, -1.614949107170105, -1.9759776592254639, -0.24999724328517914, -1.4247431755065918, -0.5943536758422852, 0.3452064096927643, -1.375575065612793, -0.9535964727401733, 0.28972792625427246, -0.047952208667993546, 0.04207994416356087, 0.11433924734592438, -0.8778679370880127, -0.9528608918190002, 0.3084927797317505, 0.5771422386169434, -0.13819175958633423, 1.1050381660461426, 0.6249117255210876, -0.41432854533195496, -0.822729229927063, -1.1842514276504517, 0.252033531665802, 0.6279094815254211, 0.08098123967647552, 0.35491275787353516, 0.07293321192264557, 0.6977837681770325, 0.11777829378843307, -0.6846734881401062, -0.25164690613746643, 0.1850193589925766, 0.543069064617157, -0.1721583753824234, 0.574238121509552, -0.2066454142332077, 0.8161927461624146, -1.9877198934555054, -0.5773515701293945, 0.34055817127227783, -0.8573221564292908, -1.5782907009124756, -0.9517345428466797, -0.013751848600804806, -0.7442995309829712, 0.7273837924003601, -0.3075796961784363, -1.556952714920044, -1.1710338592529297, -1.179721713066101, 0.11778243631124496, -0.33627185225486755, -0.5562222599983215, 0.5618762373924255, 0.23209315538406372, -0.7180267572402954, 0.7488852143287659, -1.998500943183899, -0.35633453726768494, -1.854569673538208, -2.1960461139678955, -0.3515932559967041, -0.6970126032829285, 1.4277842044830322, -1.2047622203826904, 0.7321934700012207], [2.020022392272949, -1.1063518524169922, 0.4483128488063812, 0.5948765277862549, 1.435850739479065, 1.2381612062454224, -0.5499488115310669, 1.2505937814712524, -0.4288925528526306, 0.494060754776001, -0.5328083038330078, 0.7856963872909546, 2.975400447845459, -0.15642258524894714, -2.3240597248077393, -0.6377583146095276, 0.42289167642593384, 2.2367045879364014, -0.23509691655635834, 0.03207008168101311, 0.4380224347114563, -2.48311710357666, -0.7631372213363647, -0.3825649917125702, -1.4545401334762573, -0.9449548721313477, 1.3930994272232056, 0.0019886938389390707, 1.820888638496399, 0.6690206527709961, 0.21711696684360504, 0.3411419093608856, 0.5230120420455933, -0.21878807246685028, -0.06631974130868912, 0.14297573268413544, 0.6321848034858704, -0.3976614773273468, 0.6944437623023987, 3.2143778800964355, -1.6541038751602173, -1.1827841997146606, 0.02138817310333252, -0.4321402311325073, 0.326837956905365, -1.5266940593719482, 0.49674201011657715, 2.0972352027893066, -0.13900776207447052, 1.6784738302230835, -0.08650533854961395, 0.2584240734577179, 1.0813086032867432, -0.3460486829280853, 1.5953401327133179, 0.07575302571058273, 0.07090575248003006, 1.0371097326278687, 0.6286666989326477, -1.2669503688812256, 0.640791654586792, 1.032059669494629, 0.76800936460495, -0.06983403116464615, -2.1922073364257812, -1.366174340248108, -1.2116848230361938, -0.0038708304055035114, 0.4594471752643585, -1.53272545337677, -0.8314085006713867, -1.203393578529358, -1.094663143157959, 1.4216727018356323, 0.5025566816329956, -0.9645790457725525, -0.8372818231582642, -0.8372560143470764, 1.3012714385986328, -0.8301416635513306, 0.5085532069206238, -0.014780985191464424, 0.4857886731624603, 0.8816977739334106, 1.202946424484253, -0.30017632246017456, 0.5162080526351929, -0.6935550570487976, -1.478600025177002, 1.0320416688919067, -0.24106258153915405, 0.740500807762146, -0.04865249618887901, -0.28854653239250183, 0.05497930571436882, 0.946502685546875, 0.8622372150421143, 0.13384570181369781, -1.2288199663162231, 1.0128824710845947, -0.12018422782421112, 0.8268786072731018, -1.084030270576477, 1.1804859638214111, 1.8793721199035645, 1.9601510763168335, 0.7195363640785217, -0.7090837359428406, -0.7818976044654846, 0.5268459916114807, 1.1095402240753174, -0.30556321144104004, 0.5990506410598755, -0.647344708442688, 0.08914624899625778, 0.8789525032043457, 0.8767639398574829, -0.044508595019578934, 1.0054538249969482, -1.0819376707077026, -0.7255539298057556, -0.02367640845477581, -1.058111548423767, -2.3656005859375, -0.7450903654098511, 0.7749686241149902, -0.436616450548172, 0.09564271569252014], [0.10346030443906784, 0.5027302503585815, 0.787590742111206, 0.10549139231443405, 1.353097915649414, 0.9050573110580444, 0.38246938586235046, 2.334566593170166, -1.3164235353469849, -0.19210417568683624, -0.2240784615278244, 0.9300000667572021, -1.0937596559524536, 0.5284508466720581, 0.3067714273929596, 0.0807800143957138, -0.39595329761505127, 2.6608049869537354, -0.23890170454978943, 0.6243833899497986, -0.5801904797554016, 0.31532037258148193, -3.7310028076171875, -1.106113314628601, -0.21443970501422882, -0.34480002522468567, 0.2846609055995941, -0.3125007152557373, 0.9268493056297302, 2.8775250911712646, 1.7670642137527466, 0.4225143492221832, -0.1728711873292923, -0.6398742198944092, 1.4546337127685547, -0.08797252923250198, -0.6217410564422607, 0.4850068688392639, 0.13774915039539337, -1.0853471755981445, 0.6907548904418945, -1.0913408994674683, 1.4734331369400024, 1.4068549871444702, -0.82436603307724, 0.197829470038414, 1.531307339668274, -0.28544479608535767, -0.2628948986530304, 0.08991360664367676, 0.7701998353004456, 2.3191893100738525, -1.4091486930847168, 0.2999225854873657, -0.5794910788536072, 0.7099626660346985, -0.2738576829433441, 2.1926000118255615, -0.17255529761314392, 0.48754262924194336, 1.500862956047058, -0.5101545453071594, -0.11753931641578674, 2.6323351860046387, 0.9140709042549133, 0.686635434627533, -0.7575664520263672, 1.3916409015655518, -1.5494047403335571, -1.5978666543960571, 1.1842316389083862, -0.7336153388023376, -1.3163189888000488, 0.21338452398777008, 0.7425477504730225, 0.15262556076049805, -0.24690686166286469, 0.562578558921814, -0.5723855495452881, 1.4124079942703247, 1.1370984315872192, 0.20557789504528046, 1.093289852142334, -0.6356152296066284, -0.3539772629737854, -0.5701287388801575, 2.1980814933776855, 0.7703547477722168, 0.8172037601470947, -1.5594228506088257, 0.12687604129314423, -1.149518609046936, -1.3210575580596924, 0.6257348656654358, 1.6459932327270508, 0.696608304977417, 1.8604398965835571, 0.41995736956596375, -0.10460945218801498, -0.9716426134109497, 0.7159726619720459, 0.5869568586349487, 2.457019567489624, -0.170450359582901, 0.5720545053482056, 0.6485550999641418, 0.27571308612823486, -0.3271101415157318, 0.1433717906475067, 0.42769908905029297, 0.1480536013841629, -0.9862471222877502, -0.9327905774116516, -0.5616992712020874, 0.49126318097114563, -0.9232273101806641, 0.3147880733013153, 0.18830162286758423, -0.4026700556278229, -0.5639081001281738, 1.3756893873214722, 0.8043032884597778, 1.3286607265472412, -0.6237722039222717, 0.49654558300971985, -0.115397147834301, 0.0760161355137825, 1.8820364475250244], [0.4789736568927765, 1.630270004272461, 1.1157499551773071, 1.518370270729065, -0.9846839904785156, -0.2836090326309204, -1.952473759651184, 0.5314865708351135, 0.4189787805080414, -2.6341638565063477, 0.11441300064325333, 0.6286715269088745, -1.8390496969223022, -0.13167260587215424, -1.0596363544464111, -1.4247698783874512, -1.2626562118530273, -0.4321384131908417, 0.12195966392755508, -0.2807944715023041, 2.153769016265869, 1.0085835456848145, 0.8296831846237183, -0.6697829961776733, -0.7459443807601929, -1.0726557970046997, 0.14069686830043793, -2.876694679260254, 0.8111623525619507, 1.6099443435668945, 0.13859926164150238, 0.7050315737724304, 0.6877648234367371, 0.7090487480163574, -0.3168920576572418, 1.036395788192749, -0.26923856139183044, -0.629125714302063, 1.5574697256088257, -0.3430454134941101, 1.5617505311965942, -1.153456687927246, 0.9550564289093018, -0.892389714717865, -0.36005982756614685, -1.0015126466751099, 1.740899920463562, 0.060416873544454575, 1.4177420139312744, -0.6249960660934448, 0.2990417778491974, -0.29031774401664734, -1.6604291200637817, 0.9100052714347839, 0.2991275489330292, -0.37203508615493774, -0.7554753422737122, -1.031917929649353, -0.07896026223897934, -0.027322908863425255, 1.0878535509109497, 1.2170549631118774, -0.0595819316804409, 1.5215449333190918, 0.14190776646137238, 0.20745256543159485, -0.05727715045213699, 0.45033687353134155, 1.0184873342514038, 0.36189305782318115, 0.3711569309234619, 0.20929042994976044, -1.298750638961792, 0.23794293403625488, -0.803007960319519, -1.1868096590042114, 0.9484495520591736, 0.4135209619998932, -0.481856107711792, -1.1493234634399414, -1.0706719160079956, -1.2310134172439575, 0.7686243653297424, 1.5589118003845215, -1.6992089748382568, 0.30977863073349, 0.4388161599636078, 0.08888761699199677, -1.1416791677474976, 0.38353797793388367, 0.5207920670509338, -0.9896220564842224, 0.34411516785621643, -1.0312377214431763, 1.4106082916259766, -0.11584263294935226, -0.15732841193675995, -0.574113667011261, 0.9228411316871643, -0.3779764473438263, 0.23891447484493256, 0.06914569437503815, -1.6454746723175049, -0.579361617565155, 0.6369186639785767, -0.17354503273963928, -0.89406418800354, -0.6100804209709167, -0.39441967010498047, 0.7385411262512207, -0.2306058257818222, 0.18204571306705475, -0.6085112690925598, 0.1879727691411972, 1.5801976919174194, 0.21452836692333221, 0.7924319505691528, 0.7050980925559998, 0.5202728509902954, -1.6465429067611694, 0.7517677545547485, -1.5124858617782593, -0.019925124943256378, 0.2418486326932907, -0.4324641823768616, -0.16799017786979675, -0.3554763197898865, 0.39775580167770386], [0.9836179614067078, 1.954807996749878, 0.8048021197319031, 0.7749669551849365, 0.5528250932693481, 0.7290352582931519, 0.7060976624488831, 0.09282837808132172, 0.2594432830810547, -0.3831571340560913, 1.6454676389694214, 0.1452692449092865, 0.8873177766799927, -2.257225513458252, -0.4454389214515686, -1.326246976852417, 0.0005061337142251432, -0.7910308837890625, -1.2086305618286133, 0.5139352083206177, 1.5339531898498535, -0.7167582511901855, -1.8821545839309692, 0.4804443418979645, 1.1636241674423218, 0.8759632706642151, -1.9470659494400024, 0.45559343695640564, -0.8412266969680786, 0.09367825835943222, 0.7663674354553223, 0.19851447641849518, 0.25305214524269104, 0.022564295679330826, -0.9884417653083801, 2.281557321548462, -0.004248074721544981, 0.05596637725830078, 0.38269323110580444, -0.20610742270946503, 0.8624358773231506, 0.2509535849094391, -0.48707130551338196, -1.1536668539047241, -1.5754847526550293, 0.500505268573761, 1.316752552986145, -0.6480655074119568, 0.08664651960134506, 1.8988269567489624, 0.6820027828216553, 2.6780829429626465, -0.7320425510406494, 0.8236650228500366, -0.4350736737251282, -0.22696928679943085, -1.3199915885925293, -0.6462429165840149, -0.6953455805778503, -0.15036174654960632, -1.544627070426941, -1.8505417108535767, 0.8201656341552734, 1.6377490758895874, 0.5081837773323059, 0.24263039231300354, -0.73225998878479, 0.7861945033073425, 1.7070255279541016, -0.14610625803470612, -0.7764768600463867, -1.4649404287338257, -0.06874315440654755, 1.4001307487487793, -0.9912800788879395, -0.39343103766441345, 0.31903526186943054, -1.340308666229248, -0.34851551055908203, 1.269597053527832, 1.1082097291946411, -1.514883041381836, -0.4263908565044403, 1.4389989376068115, -1.2443161010742188, -0.11496171355247498, -0.0780920460820198, -1.5452613830566406, -0.15904855728149414, 1.2015022039413452, -0.9184226393699646, -0.1478891223669052, -0.8211660385131836, 0.15067081153392792, -0.9101524949073792, 0.3881470263004303, 0.7036682367324829, -0.13087517023086548, 1.022112250328064, 0.22616514563560486, -0.4191749393939972, -0.03124815784394741, 0.5977073907852173, 0.4731887876987457, 0.0010826861253008246, -0.7676045298576355, -0.2425500750541687, -0.07222963869571686, -0.4281708002090454, 0.28271135687828064, 0.376444548368454, -0.34183651208877563, 0.665925920009613, -0.029150469228625298, 1.0156757831573486, -0.03820095956325531, -1.1130927801132202, -0.9176698327064514, 0.726189136505127, -0.08560872822999954, -1.594160556793213, -1.0994545221328735, -0.31082066893577576, -0.46236005425453186, 0.17871931195259094, -0.19700023531913757, 0.49583667516708374, 0.9288219213485718], [-0.23820748925209045, -0.7160402536392212, 0.632004976272583, 0.9106009006500244, -0.588966965675354, 0.25743308663368225, 0.799089252948761, -1.2145195007324219, 1.9060431718826294, -1.238734483718872, 0.5200387835502625, -0.7273283004760742, -0.17432211339473724, -1.2030843496322632, -1.288825273513794, -0.9264672994613647, -0.8188288807868958, 0.7578743100166321, 0.1058567613363266, 1.1151111125946045, 0.6715306043624878, 0.3055153787136078, 0.7282056212425232, 0.9282946586608887, -0.11991474777460098, 0.5213010907173157, 0.8925080895423889, -0.47808218002319336, -1.1619540452957153, 0.07166186720132828, 0.0831683874130249, 0.3448250889778137, -0.8108181357383728, 1.5053374767303467, 0.38559260964393616, -1.8711836338043213, 0.1842050701379776, 1.965672254562378, -0.5439854860305786, -0.2604592442512512, -0.5621957778930664, -0.3164907395839691, 2.0581862926483154, 0.7324358224868774, -1.8087193965911865, 1.2742804288864136, -0.2826195955276489, -1.0789589881896973, -1.575555682182312, -0.5597314834594727, 0.25985559821128845, -0.5890852808952332, 1.1622530221939087, -0.8635218143463135, 0.31094890832901, -1.5707778930664062, -0.7423379421234131, -1.179118275642395, -1.6972509622573853, 0.6837417483329773, -0.24342936277389526, -0.09632518887519836, 0.13641943037509918, 1.1310135126113892, 0.596762478351593, -1.6657099723815918, -0.39499524235725403, -0.9949217438697815, -1.6550028324127197, 1.3061983585357666, 1.485191822052002, 0.9907213449478149, 0.9436014890670776, 0.985805869102478, -0.5695549845695496, -0.7631189227104187, -0.1436585783958435, -0.19262027740478516, 0.9317420721054077, -1.7588839530944824, -3.0269412994384766, -0.7911104559898376, -0.9776365756988525, -1.3641300201416016, -0.7739136815071106, -0.39077165722846985, 0.4674573838710785, -0.4146292805671692, -1.0508780479431152, -0.2802102565765381, 1.2477519512176514, 1.0020668506622314, -1.8861806392669678, -0.7655889391899109, 0.04966936260461807, 0.04152410849928856, -0.23658333718776703, 1.0293651819229126, -1.068355917930603, 1.5066624879837036, -1.3500044345855713, -1.1646859645843506, -0.08748861402273178, 0.49819186329841614, 0.2905169129371643, 0.11737454682588577, 0.4848717153072357, -1.7215170860290527, 0.0408477783203125, -0.2633719742298126, -0.46739140152931213, 0.564140796661377, -1.2184709310531616, -0.33867931365966797, -0.25248652696609497, 0.5411511063575745, 0.15066470205783844, 0.17901945114135742, 0.028717998415231705, -0.0847262591123581, -0.7939825654029846, 2.1324174404144287, -0.7283585667610168, 0.9290100336074829, 0.6582456827163696, 0.15837879478931427, 0.5619005560874939, 0.20623502135276794]]]], "dtype": "float32"}, true], "e": {"__tup__": [{"__nd__": [[[[0.5368158221244812, 0.017400164157152176, 1.1949751377105713, -0.20110706984996796, -1.3800745010375977, 0.400943398475647, -1.0118361711502075, -0.05473710596561432, 0.06862156838178635, -0.43199172616004944, -0.417115181684494, -0.0631939023733139, -0.7628386616706848, 0.92723548412323, -0.6686881184577942, -0.5349510908126831, -0.47628915309906006, -0.026795655488967896, 0.551253616809845, -0.3932415246963501, -0.11949573457241058, -0.4536638855934143, -1.8522911071777344, 0.016763480380177498, -1.7530547380447388, -1.7213430404663086, 1.4023301601409912, -1.4207547903060913, 0.35673949122428894, 1.1914528608322144, -0.21311421692371368, 0.5729233622550964, -0.23020873963832855, -0.4996718466281891, 0.8948315978050232, -0.20894977450370789, -0.5284685492515564, 0.306109219789505, 0.6321607232093811, -0.12033744156360626, -1.2826859951019287, -0.3322564363479614, -0.0812881663441658, -0.7915240526199341, -0.840208888053894, -0.5370051860809326, 0.4835725724697113, -1.1581951379776, 1.2830586433410645, 1.4508861303329468, -2.135451555252075, -0.5516238808631897, -0.8649051189422607, -0.6903594732284546, -0.6305643320083618, -0.47897422313690186, -0.11917991936206818, -1.1306116580963135, -1.1236211061477661, -0.6542324423789978, 1.3505157232284546, -0.97711580991745, -0.1586431860923767, 1.3388310670852661, 1.6547048091888428, 0.538483738899231, 0.05727827548980713, 0.06795436888933182, 1.9388165473937988, 0.579855740070343, 0.2205633521080017, -0.009285973384976387, 0.43343228101730347, 0.3067690432071686, 1.1113094091415405, 0.220603808760643, -2.5573794841766357, -0.9248543381690979, -2.071884870529175, -0.1681174486875534, -0.3303106427192688, -0.9019407033920288, -0.18799428641796112, -0.0245341919362545, 0.31358176469802856, 0.45756256580352783, -0.7529453039169312, 0.49241209030151367, -2.1165311336517334, 0.8472545146942139, 0.059856612235307693, -1.1782366037368774, -1.126229166984558, -1.3968030214309692, -0.5567728281021118, 0.06830932199954987, -1.0832775831222534, 0.3837514817714691, -1.5954797267913818, 0.5858510136604309, 2.729919195175171, 0.033704206347465515, 1.160508394241333, -0.5342431664466858, -0.03609530255198479, 1.5401949882507324, 0.8444561958312988, 0.6775966286659241, -0.33197838068008423, -0.7051905989646912, -0.25800421833992004, 0.4003200829029083, -1.3833030462265015, -0.4155385196208954, 0.2328791618347168, 0.3212478458881378, -0.14884203672409058, -0.2446255087852478, 1.9925055503845215, -0.39383113384246826, 0.02131551504135132, -0.46400779485702515, 1.040266752243042, 1.0748202800750732, -0.829272449016571, -0.443354994058609, 0.47999367117881775, 0.9569171667098999], [1.59258234500885, 0.752934992313385, 2.002250909805298, 1.024451732635498, 0.4491804242134094, 0.9239059090614319, -0.4782900810241699, -0.6188753843307495, -1.7195141315460205, 0.026750555261969566, -1.435836911201477, -0.577271044254303, 0.7743974924087524, 0.5577067136764526, 0.016371436417102814, -0.26471632719039917, 1.235131025314331, 0.09755782783031464, 0.8410130739212036, -0.5389105081558228, 0.4939664602279663, 0.08485884964466095, 0.649950385093689, 0.963580310344696, -0.41051456332206726, 0.24261869490146637, -0.4686509072780609, -1.6741886138916016, 1.4777697324752808, -0.366603821516037, -0.2003335803747177, 1.5041286945343018, -0.028310183435678482, -0.7612112760543823, 2.604416608810425, -0.8130980730056763, -0.21508018672466278, 0.40629369020462036, -0.15236225724220276, -1.1711704730987549, 0.30598902702331543, 0.3978135287761688, -0.6069676876068115, 0.008921376429498196, -0.7665467858314514, 0.8166665434837341, 0.24651694297790527, -2.0139732360839844, -0.4344066381454468, -0.33936434984207153, -0.12447844445705414, 0.15295496582984924, -0.15852653980255127, -0.22632864117622375, -0.16535712778568268, -0.531158447265625, 1.015196681022644, -1.1052353382110596, -0.09574486315250397, -1.0042128562927246, 0.21380597352981567, -0.5898392200469971, -0.27174779772758484, 1.0544743537902832, -0.5521904826164246, -0.1571836918592453, -1.5748552083969116, -0.43376410007476807, 0.8544945120811462, -0.20073170959949493, 1.2075188159942627, 0.5689319968223572, -1.0864890813827515, 1.9162653684616089, 0.8537123799324036, -0.18780098855495453, -1.5190539360046387, 0.7698850035667419, 0.00034636599593795836, -0.5749292373657227, -1.1117541790008545, 0.0011790057178586721, -0.9758345484733582, 1.3546085357666016, -1.772143840789795, -0.45421403646469116, -0.31738683581352234, 0.1639140248298645, 0.13040047883987427, 1.532965064048767, 1.7889608144760132, -0.48833873867988586, -0.35856467485427856, -0.6015226244926453, 0.11110103130340576, -1.540364146232605, -0.94524747133255, 0.08963843435049057, 0.08014091849327087, 0.184988334774971, 1.003720760345459, 1.5748547315597534, -0.046494871377944946, 0.5844246745109558, -0.6480823159217834, 1.4870460033416748, 0.5269397497177124, 0.4019705057144165, -0.4827742874622345, -1.0591539144515991, -0.3939526081085205, 0.5668180584907532, 0.31967997550964355, -0.28663894534111023, -0.5392336845397949, -0.4530729353427887, 0.6024294495582581, 0.4151324927806854, 0.6429951190948486, -0.6522279381752014, 1.6445809602737427, -1.5267460346221924, -1.233591914176941, 1.681542992591858, 0.7446402907371521, 0.022407246753573418, 1.4210842847824097, 0.6367896795272827], [1.239558458328247, 0.5638400316238403, 1.633567214012146, 0.7723721265792847, 0.35830679535865784, 0.9801266193389893, -0.3847998082637787, -0.3879189193248749, -1.6409612894058228, -0.1127551943063736, -1.3169164657592773, -0.5672393441200256, 0.6666802167892456, 0.39756259322166443, 0.28145632147789, -0.32895055413246155, 0.8656596541404724, 0.01166379451751709, 0.7793360948562622, -0.4679499864578247, 0.48274585604667664, 0.13885661959648132, 0.6073481440544128, 1.0716866254806519, -0.29922911524772644, 0.3834609091281891, -0.05659715086221695, -1.6004332304000854, 1.355949878692627, -0.25862255692481995, -0.08913125097751617, 1.4365692138671875, -0.10837890207767487, -0.49202242493629456, 2.1608195304870605, -0.5644916296005249, -0.3624936044216156, 0.46339139342308044, -0.02040737122297287, -1.1123456954956055, 0.14432024955749512, 0.36011964082717896, -0.555450975894928, -0.10019871592521667, -0.8119333982467651, 0.382303386926651, 0.06911513209342957, -1.7186901569366455, -0.2484729290008545, -0.1022719219326973, -0.27957969903945923, -0.034086644649505615, -0.23837910592556, -0.1706697940826416, -0.48350948095321655, -0.5807271599769592, 0.8491600751876831, -1.0350769758224487, -0.06734362244606018, -0.8570107817649841, 0.04947923868894577, -0.45246389508247375, -0.13227860629558563, 0.9566923379898071, -0.5790691375732422, -0.035903263837099075, -1.1703228950500488, -0.2690766155719757, 0.6295683979988098, 0.06169501692056656, 0.8268055319786072, 0.7003971934318542, -0.8732932209968567, 1.5038862228393555, 0.8049946427345276, 0.03807518631219864, -1.5550522804260254, 0.5352283716201782, -0.20395046472549438, -0.5529927015304565, -0.9990658164024353, 0.04640138894319534, -0.8270872235298157, 1.3040111064910889, -1.4635025262832642, -0.31213900446891785, -0.3414948284626007, 0.13216976821422577, 0.25010260939598083, 1.2168384790420532, 1.585744857788086, -0.6557391881942749, -0.21964803338050842, -0.7654483914375305, -0.1977495402097702, -1.3418848514556885, -1.0311167240142822, 0.022084742784500122, -0.02339484915137291, 0.14777618646621704, 1.066646695137024, 1.3688147068023682, 0.229168102145195, 0.5031512975692749, -0.463990718126297, 1.2634581327438354, 0.35919392108917236, 0.5486766695976257, -0.5844992995262146, -0.7683612108230591, -0.04107298702001572, 0.42329758405685425, 0.12189524620771408, -0.29774996638298035, -0.3975977301597595, -0.47846081852912903, 0.32377728819847107, 0.5198025107383728, 0.4506908655166626, -0.4962533116340637, 1.4198352098464966, -1.317357063293457, -1.0731794834136963, 1.3202908039093018, 0.4892847537994385, -0.04338432103395462, 1.1664422750473022, 0.5241434574127197], [0.2203042209148407, -0.1603132039308548, 0.5225344896316528, -0.22046104073524475, -0.7425713539123535, 0.8501177430152893, -0.4714559018611908, 0.4069475531578064, -0.7833934426307678, -0.4934023320674896, -0.5421791672706604, -0.23738329112529755, -0.1845882683992386, 0.25725555419921875, 0.4273982644081116, -0.396652489900589, -0.6308115124702454, -0.2368616908788681, 0.5108285546302795, -0.4013896584510803, 0.26867228746414185, 0.0802987739443779, -0.7562440037727356, 0.4119836986064911, -0.7987034916877747, -0.5022522211074829, 1.5044727325439453, -1.142386794090271, 0.6034637093544006, 0.5975275039672852, 0.09909262508153915, 0.8784546852111816, -0.4270211160182953, 0.13961578905582428, 0.5139700770378113, 0.007120770402252674, -0.6907595992088318, 0.41453951597213745, 0.48784565925598145, -0.47793492674827576, -0.7512592077255249, -0.11446874588727951, -0.007081156596541405, -0.6380393505096436, -0.8208643794059753, -1.008829116821289, -0.16792593896389008, -0.7316005825996399, 0.8512290120124817, 1.0912585258483887, -1.2924636602401733, -0.7671371698379517, -0.5889483094215393, -0.26799312233924866, -0.9021115303039551, -0.5073700547218323, -0.1561165452003479, -0.8437512516975403, -0.5154914259910583, -0.4109216630458832, 0.27409273386001587, -0.41786473989486694, 0.16792389750480652, 0.7116885185241699, 0.5190339088439941, 0.3947090208530426, 0.27087175846099854, 0.14288093149662018, 0.7301756739616394, 0.796727180480957, -0.24258321523666382, 0.5497036576271057, 0.26029491424560547, -0.039427824318408966, 0.842705249786377, 0.6738104224205017, -1.7986010313034058, -0.7874886989593506, -1.4570895433425903, -0.3750995993614197, -0.2955707013607025, -0.30855920910835266, 0.03490964695811272, 0.3766760230064392, 0.15147581696510315, 0.347676545381546, -0.30733564496040344, 0.20895923674106598, -0.6883279085159302, 0.30736029148101807, 0.3012937605381012, -1.3304765224456787, -0.2591595947742462, -1.3119369745254517, -1.0546122789382935, -0.2573201060295105, -1.0780521631240845, 0.13425372540950775, -1.1094354391098022, 0.32587459683418274, 1.800337553024292, 0.21312634646892548, 1.1410560607910156, -0.23896604776382446, 0.2285311222076416, 0.8329059481620789, 0.37717336416244507, 0.9456372857093811, -0.681007981300354, -0.08357364684343338, 0.5353323221206665, 0.10553907603025436, -0.9647389054298401, -0.2527998685836792, 0.27264827489852905, -0.1726771891117096, -0.6485990285873413, 0.33892905712127686, 0.9195808172225952, -0.03282300755381584, 0.06933604925870895, -0.4228902757167816, 0.4601590931415558, 0.34244340658187866, -0.7128761410713196, -0.43422940373420715, 0.41985884308815, 0.37264400720596313], [0.6516945362091064, 0.17099842429161072, 0.7515513896942139, 0.17311984300613403, -0.639010488986969, 0.8627226948738098, -0.2595279812812805, 0.15480798482894897, -0.9641488790512085, -0.12322356551885605, -0.6526209712028503, -0.23648785054683685, 0.12474828213453293, 0.3865291476249695, 0.12035933881998062, -0.1254841387271881, 0.015019053593277931, -0.010310251265764236, 0.4011126458644867, -0.635671854019165, 0.4850609600543976, -0.03138386458158493, -0.7261615991592407, -0.02367987297475338, -0.7777795195579529, -0.789804220199585, 0.9475921392440796, -0.8473796248435974, 0.6775848269462585, 0.4936131238937378, 0.034661781042814255, 0.9114053249359131, -0.46359121799468994, -0.3192082345485687, 0.8287510275840759, -0.332367479801178, -0.4921115040779114, 0.07588332891464233, 0.14398057758808136, -0.6629628539085388, -0.3996237814426422, -0.1899002641439438, 0.06378020346164703, -0.4605790972709656, -0.6534446477890015, -0.3054022789001465, 0.17237462103366852, -0.7718677520751953, 0.4769384562969208, 0.7083227634429932, -0.6185662150382996, -0.4640965759754181, -0.40807414054870605, -0.18783558905124664, -0.2639533281326294, -0.19882626831531525, -0.2796041667461395, -0.8109211325645447, -0.5067325830459595, -0.5046564340591431, 0.7280400991439819, -0.5653219223022461, -0.13239099085330963, 0.6684939861297607, 0.3685682415962219, 0.1147475317120552, -0.28247007727622986, -0.23630774021148682, 0.8515706658363342, 0.49809810519218445, 0.42570313811302185, 0.20202401280403137, 0.2153797447681427, 0.4375547170639038, 0.7754744291305542, 0.549117922782898, -1.140072226524353, -0.6341118812561035, -1.0320961475372314, -0.21368157863616943, -0.3239672780036926, -0.4601762592792511, -0.06548017263412476, 0.1297057420015335, -0.1509224772453308, -0.030746622011065483, -0.0653495267033577, 0.22733409702777863, -0.9517064094543457, 0.7987497448921204, 0.29302629828453064, -0.9911804795265198, -0.3731343150138855, -1.0360283851623535, -0.478558748960495, -0.5906404256820679, -0.7919007539749146, 0.31524237990379333, -0.9521462917327881, 0.12272268533706665, 1.4084558486938477, 0.3957684636116028, 0.39504504203796387, -0.3315308094024658, 0.05107599496841431, 0.9692198634147644, 0.6971405744552612, 0.7086395025253296, -0.7904918193817139, -0.2575661242008209, -0.1381494402885437, 0.2707291543483734, -0.6836656928062439, -0.07223256677389145, 0.041021183133125305, -0.09227380901575089, -0.4947691559791565, -0.014473489485681057, 1.1591925621032715, -0.21684202551841736, 0.18808774650096893, -0.7488892078399658, 0.3219509720802307, 0.6494603753089905, -0.356046587228775, -0.3770081400871277, 1.0949115753173828, 0.5496679544448853], [0.7986953258514404, -0.1042865440249443, 0.5270543694496155, 0.30182603001594543, -0.4480246305465698, 0.9182606339454651, -0.33266618847846985, 0.6680924892425537, -1.414008378982544, 0.24612832069396973, -0.6021502614021301, -0.017566116526722908, 0.18891942501068115, 0.17982211709022522, 0.22196312248706818, 0.7970608472824097, -0.19270555675029755, -0.272633820772171, 0.349948912858963, -0.7835153341293335, 0.6718830466270447, 0.5793055295944214, -1.052241563796997, -1.0167485475540161, -0.8008860945701599, -1.0274879932403564, 0.4958336651325226, -0.38028639554977417, 0.43234455585479736, 0.022925416007637978, 0.21307431161403656, 0.7806457281112671, -0.6486376523971558, 0.06674857437610626, 0.4013325870037079, -1.0436807870864868, -0.2955781817436218, -0.24511809647083282, -0.17982056736946106, -0.454720139503479, 0.12503144145011902, -0.17120473086833954, 0.8669356107711792, -0.3124116063117981, -0.5528692603111267, -0.11930225044488907, -0.3287394940853119, -0.7025182843208313, 0.2665797173976898, 0.4257892072200775, 0.28984174132347107, -0.4200807511806488, 0.3127862215042114, -0.05875759571790695, 0.7081395387649536, 0.20707610249519348, -0.7327043414115906, -0.27204379439353943, -0.3069402873516083, -0.35910049080848694, 0.2828088402748108, -0.6006752252578735, -0.05389951914548874, -0.23295634984970093, 0.4224529266357422, -0.29600197076797485, -0.6893291473388672, -0.5091524124145508, 0.5958981513977051, 0.15066109597682953, 0.4526194930076599, 0.08746891468763351, 0.5167924761772156, 0.16054761409759521, 0.7015753388404846, 0.5752766132354736, 0.09207946807146072, -0.8573817610740662, -0.38599416613578796, -0.6923069953918457, 0.209269180893898, -0.5572426319122314, 0.47543713450431824, -0.4284154176712036, 0.3133288323879242, -0.45496872067451477, 0.9944008588790894, 0.06368439644575119, -0.9153209328651428, 0.524505078792572, -0.10728687793016434, -1.2913317680358887, 0.04243411123752594, -0.4542239308357239, -0.4538176655769348, -0.8699408769607544, -0.28143495321273804, 0.5306122899055481, -1.1014540195465088, 0.27923014760017395, 0.694751501083374, 0.24127085506916046, 0.35257241129875183, -0.640457034111023, 0.3194536566734314, 0.5602051019668579, 1.1408923864364624, 0.8977066874504089, -0.6149974465370178, -0.1177506372332573, -0.4636833965778351, 0.4649330675601959, -0.272659569978714, 0.19742095470428467, 0.30605536699295044, -0.3409075140953064, -0.871806800365448, -0.1713458001613617, 1.5834569931030273, 0.11086835712194443, -0.46639391779899597, -0.43279093503952026, 0.5464951992034912, 0.10326070338487625, -0.361481249332428, -0.7579832673072815, 1.1256526708602905, 0.2369292974472046], [0.8109833598136902, 1.0759693384170532, 1.4022730588912964, 0.8123130798339844, -0.01904417760670185, 1.1821997165679932, 0.030247023329138756, -0.05596047267317772, -0.8197802305221558, 0.5084418654441833, -1.152917504310608, -0.6941234469413757, 0.49875614047050476, 0.4701077938079834, 0.6834001541137695, 0.47353148460388184, 0.49592721462249756, 0.033990733325481415, 0.39519599080085754, 0.11446438729763031, 0.6454316973686218, 0.8045139312744141, -0.027217835187911987, 0.4400687515735626, -0.19111652672290802, -0.2590380609035492, -0.43775704503059387, -0.6844517588615417, 1.1099001169204712, 0.2753049433231354, -0.19119077920913696, 0.743732213973999, -0.5556468963623047, -0.45145273208618164, 1.6788227558135986, -0.4754311442375183, -0.09791320562362671, 0.08289378881454468, -0.5153125524520874, -1.2688456773757935, 0.07933546602725983, -0.03329004347324371, -0.024712812155485153, -0.2080312818288803, -0.6155403852462769, 0.5494966506958008, 0.21541108191013336, -1.4958735704421997, -0.5024963617324829, -0.23763443529605865, 0.305437296628952, -0.08713949471712112, 0.07619118690490723, -0.29783013463020325, -0.04089035093784332, -0.8315889239311218, 0.5749696493148804, -0.19893565773963928, 0.35701513290405273, -0.7451089024543762, 0.7542166709899902, -0.43929898738861084, -0.36339429020881653, 0.6713012456893921, -0.38920485973358154, 0.04863821715116501, -0.8869476318359375, 0.02544037438929081, 0.3547210991382599, -0.20721644163131714, 0.3245481550693512, 0.3862770199775696, -0.2818606495857239, 1.4841300249099731, 0.6629675030708313, -0.2841624617576599, -0.4671727418899536, 0.1672687828540802, 0.1570226103067398, -0.5279788970947266, -0.5283160209655762, -0.20987269282341003, -1.2284454107284546, 1.335740327835083, -0.9047427177429199, -0.3203190267086029, -0.023892968893051147, 0.2448214292526245, -0.30999910831451416, 0.6693002581596375, 0.8473230004310608, -0.23108063638210297, -0.35851627588272095, -0.0550771988928318, -0.16633878648281097, -0.9094626903533936, -0.6645181179046631, 0.37791961431503296, 0.22868923842906952, -0.05068249627947807, 0.9946213364601135, 0.4991709589958191, -0.04860794171690941, 0.7732167840003967, -0.14427122473716736, 1.240244746208191, 0.3109564483165741, 0.3675234913825989, -0.7104742527008057, -0.6824474334716797, -0.6546416878700256, 0.8693834543228149, 0.019770685583353043, 0.08386017382144928, -0.4914001226425171, -0.15047357976436615, 0.40684568881988525, 0.4640621542930603, 0.5252645015716553, -0.7612395882606506, 1.3913230895996094, -1.0074838399887085, -0.9842082858085632, 0.844553530216217, 0.6693306565284729, 0.37114110589027405, 1.124367594718933, 0.6926443576812744], [-0.4288252890110016, 0.2388417273759842, 0.5982487201690674, 0.4096202552318573, -0.12252669036388397, 0.34517085552215576, -0.3520214855670929, -0.04701453819870949, -0.29284369945526123, -0.5502792596817017, -0.2349836528301239, -0.20790833234786987, -0.050950344651937485, -0.11322693526744843, 0.2462814301252365, 0.9038500189781189, -0.36650437116622925, -0.02968951128423214, -0.44064557552337646, 0.36588239669799805, 0.33731570839881897, 0.9304977059364319, -0.46062102913856506, 0.32135993242263794, -0.716956377029419, -0.07976636290550232, -0.18862327933311462, -0.7863354086875916, -0.11002237349748611, 0.3282136619091034, 0.5259833931922913, 0.24086976051330566, -0.4103440046310425, -0.3476978838443756, -0.10939975827932358, -0.5725388526916504, -0.4271656572818756, -0.2942359745502472, 0.34606412053108215, -0.7081782221794128, -1.0799977779388428, -0.004784195683896542, 0.05962763726711273, 0.3203847408294678, -0.47708627581596375, -0.15308566391468048, -0.12990893423557281, -0.6356677412986755, 0.37288957834243774, 0.33874770998954773, 0.10235439985990524, 0.46575915813446045, -0.11358723789453506, -0.03740793094038963, -0.3009975552558899, -0.2739024758338928, 0.11134009063243866, -0.008097009733319283, 0.37121880054473877, -0.15389497578144073, -0.11053333431482315, -0.20167742669582367, 0.09418193250894547, 0.14289233088493347, 0.07243531942367554, 0.1698438674211502, -0.3483116328716278, -0.2059255689382553, 0.37962546944618225, 0.6769409775733948, -0.36459487676620483, 0.8424426317214966, 0.4972282946109772, 0.8413456082344055, 0.09183962643146515, -0.044257231056690216, -0.3417004942893982, -0.809788703918457, -0.638154149055481, -0.4816453456878662, -0.19377636909484863, -0.5407647490501404, -0.5746163725852966, 0.07495620101690292, 0.19161084294319153, -0.2969288229942322, -0.09161718189716339, -0.0221394170075655, -0.3064272701740265, 0.11383369565010071, -0.4523215591907501, -0.2731064260005951, -0.051534734666347504, -0.6206036806106567, -0.8243696093559265, 0.34225380420684814, -0.627819836139679, 0.468036949634552, -0.09120393544435501, -0.37183794379234314, 1.3258408308029175, -0.4505707323551178, 1.2285770177841187, -0.26173076033592224, 0.2965889573097229, 0.33189213275909424, -0.29882729053497314, 0.6817637085914612, -0.12683026492595673, -0.03717837855219841, -0.28260743618011475, -0.1095837950706482, -0.24939730763435364, 0.14729487895965576, 0.27485793828964233, -0.4724445343017578, -0.31575724482536316, 0.6563173532485962, 0.21425187587738037, -0.5684248805046082, 0.7274842262268066, -0.4457205533981323, -0.1774803102016449, -0.057180311530828476, 0.2887294888496399, -0.5689371228218079, -0.3195177912712097, 0.6588709354400635], [0.022733263671398163, 0.18212465941905975, 0.7327281832695007, 0.44501742720603943, 0.5498752593994141, 0.9643452167510986, -0.11990420520305634, 0.41918662190437317, -0.8990408778190613, 0.32582637667655945, -1.1346079111099243, -0.366094708442688, -0.031568583101034164, 0.00788801908493042, 0.6380019187927246, 1.1331720352172852, -0.13375058770179749, -0.2682569921016693, 0.18249094486236572, 0.272585928440094, 0.3689606487751007, 1.0493131875991821, -0.4867018461227417, 0.6215800046920776, -0.08344574272632599, 0.3593340218067169, -0.2511056363582611, -0.9111584424972534, 0.12662868201732635, -0.09596884250640869, 0.4974304139614105, 0.3615573048591614, -0.40798190236091614, 0.27416151762008667, 0.12006261944770813, -0.8154156804084778, -0.49483993649482727, -0.09871112555265427, -0.1717344969511032, -0.6861168742179871, -0.10032004117965698, 0.3014272451400757, 0.45323455333709717, -0.1515296995639801, -1.0803678035736084, 0.26241835951805115, -0.717596173286438, -0.9408687949180603, -0.06803561002016068, 0.23167011141777039, 0.5766428709030151, 0.3459749221801758, 0.558994710445404, 0.04399953782558441, -0.5052200555801392, -0.3659657835960388, 0.04073702543973923, 0.29238152503967285, 0.4866478443145752, -0.2700026333332062, -0.10189839452505112, -0.31560131907463074, -0.03736407682299614, -0.10410314053297043, -0.6076648831367493, -0.05933193117380142, -0.9870675206184387, 0.04996369034051895, -0.02492722123861313, 0.2806415259838104, -0.17721915245056152, 0.8189334869384766, 0.6904327273368835, 0.463760644197464, 0.39954209327697754, 0.05708879604935646, 0.11388588696718216, -0.2520337402820587, 0.3006100654602051, -0.9225296378135681, -0.03841033950448036, -0.5728814601898193, -0.7866897583007812, 0.382665753364563, 0.2201385796070099, -1.0336488485336304, 0.422839492559433, -0.035967372357845306, -0.009453610517084599, 0.160480335354805, -0.007935382425785065, -0.8753303289413452, 0.123740553855896, -0.1638510525226593, -0.3907235860824585, -0.947801411151886, -0.693457841873169, 0.20382092893123627, -0.015524880029261112, -0.190138041973114, 0.8468396663665771, 0.26530155539512634, 1.1224364042282104, -0.2894580364227295, 0.16785486042499542, 0.32172292470932007, 0.27002835273742676, 0.9796850085258484, -0.5202123522758484, 0.11867514252662659, -0.3041307330131531, 0.6909897923469543, -0.001007719081826508, -0.15974061191082, 0.14467249810695648, -0.598598837852478, -0.30954158306121826, 0.2085149735212326, 0.4751654267311096, -0.12386887520551682, 0.36084508895874023, -0.21952521800994873, -0.8074459433555603, -0.46542710065841675, 0.07119908928871155, -0.6085963845252991, 0.05860048532485962, 0.6154035925865173], [-0.5277857184410095, -0.054689180105924606, -0.10333998501300812, -0.1805403232574463, 0.11429021507501602, 1.2102112770080566, 0.21555085480213165, 0.5417734384536743, -1.0091307163238525, -0.5246189832687378, -0.6931042671203613, -0.4852259159088135, 0.1971758008003235, -0.4265592098236084, 1.430243730545044, -0.14705848693847656, -0.738691508769989, -0.30176860094070435, 0.19599945843219757, 0.08837704360485077, 0.5223159790039062, 0.7018796801567078, 0.35276904702186584, 1.254863977432251, 0.2866850197315216, 0.9888176918029785, 1.174911379814148, -0.8452317714691162, 0.5581626892089844, 0.1679093837738037, 0.4666505753993988, 0.7511885166168213, -0.45364806056022644, 0.637262761592865, -0.1521366536617279, 0.4764333963394165, -0.867981493473053, 0.4599449634552002, 0.3052310645580292, -0.9003698825836182, -0.42056453227996826, 0.23322372138500214, -0.12839505076408386, -0.3029138743877411, -0.9400798678398132, -1.186773657798767, -0.7745538949966431, -0.14266549050807953, 0.346139520406723, 0.694469690322876, -0.2851865887641907, -0.5471855998039246, -0.28307145833969116, 0.24532662332057953, -1.4982876777648926, -0.7577733397483826, 0.015735652297735214, -0.19291938841342926, 0.3898056745529175, -0.14252997934818268, -0.558555543422699, 0.19799090921878815, 0.39082983136177063, 0.1781184822320938, -0.7953521013259888, 0.37457361817359924, 0.35663145780563354, 0.4940192699432373, -0.6046314239501953, 1.0574688911437988, -0.927882969379425, 1.2925366163253784, 0.3209247887134552, -0.16871081292629242, 0.4511367678642273, 0.7453007102012634, -0.9311398863792419, -0.3891769051551819, -0.6113395094871521, -0.4363496005535126, -0.35741186141967773, 0.16139237582683563, -0.22914563119411469, 0.8553630113601685, -0.025608912110328674, -0.010501595214009285, -0.24148011207580566, -0.0743604525923729, 0.6878685355186462, -0.36691680550575256, 0.2611643970012665, -1.0111565589904785, 0.43525874614715576, -1.0212383270263672, -1.3819849491119385, -0.3474663197994232, -1.1496739387512207, -0.22535660862922668, -0.2416643500328064, -0.2553362250328064, 0.9565094113349915, 0.1318569928407669, 1.2467334270477295, 0.0497332401573658, 0.4169853925704956, 0.03960373252630234, -0.45603638887405396, 1.154923439025879, -0.9616837501525879, 0.6216390132904053, 1.1637508869171143, -0.1321144849061966, -0.5794035792350769, -0.14163479208946228, 0.29788580536842346, -0.6548683643341064, -0.801116406917572, 1.008007287979126, -0.5801600813865662, 0.10958033055067062, 0.47902917861938477, -0.34627702832221985, -0.4592995345592499, -0.6696459054946899, -0.3618454039096832, -0.22729557752609253, -0.0320403017103672, 0.07984788715839386], [-0.18536582589149475, -0.01879606395959854, 0.199000284075737, -0.036092452704906464, 0.053287435322999954, 0.9495329856872559, -0.044325537979602814, 0.13616743683815002, -0.6914545297622681, -0.3023272752761841, -0.7006059885025024, -0.24077798426151276, -0.08921152353286743, -0.1191663146018982, 0.5981656908988953, -0.10850776731967926, -0.45714423060417175, -0.1259804219007492, 0.24749824404716492, -0.0806545689702034, 0.3221529424190521, 0.3673434555530548, -0.2917819917201996, 0.7414216995239258, -0.13992060720920563, 0.41141295433044434, 0.7337540984153748, -0.8810763359069824, 0.3040197491645813, 0.1442941427230835, 0.36255985498428345, 0.4382196068763733, -0.09662289917469025, 0.2689054310321808, -0.048893023282289505, 0.2819322645664215, -0.6761171817779541, 0.32964882254600525, 0.17181414365768433, -0.5756179690361023, -0.18353204429149628, 0.34506842494010925, -0.023743964731693268, -0.15205274522304535, -1.0548375844955444, -0.501462459564209, -0.629033625125885, -0.14176486432552338, 0.46554675698280334, 0.7699289321899414, -0.2456153929233551, -0.23014122247695923, -0.08571774512529373, 0.25559723377227783, -0.983649492263794, -0.5059863328933716, -0.13637404143810272, -0.07460810989141464, 0.057362087070941925, -0.29280686378479004, 0.04039469361305237, -0.1798275113105774, 0.1573401242494583, 0.20155905187129974, -0.2540663182735443, 0.20958495140075684, -0.30690649151802063, 0.3975851535797119, 0.006956145167350769, 0.7619844675064087, -0.33373814821243286, 0.9962863922119141, 0.5728183388710022, -0.012215672060847282, 0.7101350426673889, 0.4010254144668579, -0.8930895328521729, -0.041939377784729004, -0.45869362354278564, -0.38198545575141907, -0.3548471927642822, -0.22173471748828888, -0.04824964329600334, 0.22363196313381195, -0.02531139738857746, -0.4311484396457672, -0.2543645203113556, -0.046987179666757584, -0.22112278640270233, 0.024896904826164246, -0.022450625896453857, -0.9073306918144226, 0.023455282673239708, -0.7923538088798523, -0.7492967247962952, -0.49085474014282227, -0.9660227298736572, -0.2411089539527893, -0.6260198950767517, -0.18376679718494415, 0.9630122780799866, 0.22234511375427246, 0.9662831425666809, -0.4980745017528534, 0.20270691812038422, 0.2003895491361618, 0.051976192742586136, 1.0807417631149292, -0.7254831790924072, 0.37034159898757935, 0.5113170146942139, 0.0768577829003334, -0.6507202982902527, -0.2619396150112152, 0.4648348391056061, -0.5856006145477295, -0.4522351622581482, 0.4632660448551178, 0.048930224031209946, 0.08601661026477814, 0.2466103583574295, -0.39430418610572815, -0.16352595388889313, -0.5908325910568237, -0.3816922605037689, -0.27787116169929504, 0.22476178407669067, 0.3365991413593292], [-0.0647774413228035, 0.24028372764587402, 0.1258460432291031, 0.21195852756500244, -0.14226946234703064, 0.8313109874725342, 0.02065093442797661, 0.6316964030265808, -0.9233039617538452, 0.042858850210905075, -0.6446478366851807, -0.14549562335014343, 0.08360623568296432, -0.09436006844043732, 0.8491327166557312, 0.49753960967063904, -0.4040808081626892, -0.19257798790931702, 0.22852301597595215, 0.10477698594331741, 0.5789395570755005, 0.8419008255004883, -0.1340247541666031, 0.022902769967913628, 0.01520942710340023, 0.2016870230436325, 0.4041128158569336, -0.5049894452095032, 0.5604628324508667, 0.04359946772456169, 0.32580626010894775, 0.469952255487442, -0.4418390989303589, 0.18571943044662476, 0.16950827836990356, -0.09493441134691238, -0.3075057566165924, 0.0645391047000885, -0.2156870812177658, -0.737352728843689, 0.048520129173994064, 0.10767649114131927, 0.33192890882492065, 0.04257618263363838, -0.4816242754459381, -0.39152243733406067, -0.5066907405853271, -0.32222622632980347, 0.08351701498031616, 0.30290669202804565, 0.25627240538597107, -0.31090372800827026, 0.1293279230594635, -0.08579761534929276, -0.2013375163078308, -0.38386455178260803, -0.2484954297542572, 0.0511256568133831, 0.3057689070701599, -0.09683936834335327, -0.1391291320323944, -0.03669295087456703, 0.2428211271762848, -0.12379074841737747, -0.13720418512821198, 0.06470172852277756, -0.22447995841503143, 0.15108926594257355, 0.04077916964888573, 0.44411683082580566, -0.44572004675865173, 0.7539042234420776, 0.37285158038139343, 0.14837099611759186, 0.3929353952407837, 0.38011693954467773, -0.10376933962106705, -0.3052200376987457, -0.10989759117364883, -0.6222838759422302, -0.06817822903394699, -0.15552681684494019, -0.028604287654161453, 0.30114445090293884, 0.08035663515329361, 0.08193662017583847, 0.27750343084335327, -0.09144091606140137, -0.13119812309741974, -0.18871821463108063, -0.0692666694521904, -0.6539161205291748, 0.10795523971319199, -0.3345372974872589, -0.8481981754302979, -0.35184356570243835, -0.5117327570915222, 0.20668071508407593, -0.437197208404541, -0.23968558013439178, 0.5289373397827148, 0.013122286647558212, 0.5816885828971863, -0.10865788906812668, 0.35431405901908875, 0.20254835486412048, -0.0521380752325058, 0.744020938873291, -0.564365565776825, 0.17661353945732117, 0.16617296636104584, 0.11450687795877457, -0.33732303977012634, 0.05865025520324707, 0.35152173042297363, -0.45435380935668945, -0.4197331368923187, 0.4209955036640167, 0.11652097851037979, -0.13558126986026764, 0.3885766267776489, -0.40475043654441833, -0.2017694115638733, -0.5044187903404236, -0.025414006784558296, -0.0991072952747345, 0.40044817328453064, 0.1621527075767517], [-0.2925461232662201, 0.508844256401062, -0.4100933074951172, 0.763258159160614, -0.5011566281318665, 0.0707683190703392, 0.9821441173553467, -0.0571775808930397, -0.7369199395179749, 0.40710002183914185, -0.4555833339691162, -0.03090164251625538, 0.1949157565832138, 0.24841324985027313, 0.1255134493112564, 0.4313681721687317, 0.5973147749900818, 0.8941160440444946, -0.2723727524280548, -0.14249081909656525, 0.6821664571762085, -0.18279224634170532, -0.3206370770931244, -0.31570491194725037, 0.690860390663147, -0.028258642181754112, 0.21312092244625092, -0.24384690821170807, 0.29438167810440063, 0.3666425347328186, 0.8767714500427246, 0.17220844328403473, -0.3401342034339905, -1.1678996086120605, -0.03611502796411514, 0.4542974829673767, -0.045743633061647415, -0.5242106318473816, -0.4777218699455261, -0.9304568767547607, 0.19276520609855652, -0.35768938064575195, 0.041151225566864014, 0.5965260863304138, -0.18341954052448273, 0.6597822904586792, 0.5818858742713928, 0.3828369677066803, 0.1249910220503807, 0.660580575466156, 0.6637909412384033, 0.44574999809265137, -0.055994611233472824, -0.13816604018211365, -0.4093416929244995, 0.4860886037349701, -0.7968959808349609, -0.11777292191982269, -0.24760295450687408, 0.25374147295951843, 1.2983217239379883, -0.12153199315071106, -0.17366141080856323, 0.24092602729797363, -0.27084752917289734, 0.0005677714361809194, -0.21398398280143738, -0.20464354753494263, 0.8165833950042725, 0.9948253035545349, 0.8123646974563599, -0.11524488776922226, 1.2129852771759033, 0.24792906641960144, -0.010341138578951359, 0.9227687120437622, 0.6611562371253967, -0.2614542841911316, -0.4933598041534424, 0.14321182668209076, -0.35519513487815857, -0.8794513940811157, -0.312193363904953, -0.754013180732727, 0.12550020217895508, -0.10302061587572098, -0.42660796642303467, -0.2780686020851135, -1.3401058912277222, 0.8578647375106812, -0.8336780071258545, -0.03490531072020531, -0.4310723841190338, -0.8276646137237549, 0.28785350918769836, -0.3454892337322235, -0.1246321052312851, 0.45842447876930237, -0.854631781578064, -1.4898271560668945, 0.30002105236053467, 0.1325257420539856, -0.8816879391670227, -0.9994378089904785, 0.14973381161689758, -0.018480969592928886, -0.11828777939081192, -0.11177612096071243, -0.9378969073295593, 0.5401467084884644, -0.8800537586212158, -0.3360697627067566, -0.6209906339645386, -0.10026901960372925, 0.09603575617074966, -0.0752270445227623, -0.01226112712174654, -0.9757987856864929, -0.04319547116756439, -0.7177573442459106, 0.26209375262260437, -0.6110482215881348, -0.07854653149843216, -0.268854558467865, -0.02593175321817398, -0.13439996540546417, 1.414948582649231, 0.6757929921150208], [-0.37456372380256653, 0.48243674635887146, 0.26735442876815796, 0.5340306162834167, -0.1012360006570816, -0.1585818976163864, -0.09249939024448395, 0.5768164396286011, -0.3762255609035492, 0.10603936016559601, -0.5284591913223267, 0.35180848836898804, -0.49960681796073914, 0.22804927825927734, 0.0667053833603859, 0.725096583366394, 0.20950791239738464, 0.1028522402048111, 0.16478465497493744, 0.5945343375205994, -0.19356273114681244, 0.8825988173484802, 0.0014951343182474375, 0.16215401887893677, 0.23920316994190216, 0.43898752331733704, -0.298000693321228, -1.0254356861114502, 0.41234999895095825, 0.021578432992100716, 0.49837931990623474, 0.13129502534866333, 0.10482970625162125, -0.5150542259216309, 0.11618471145629883, -0.2143385112285614, 0.04995133727788925, -0.2597021758556366, -0.23586441576480865, -0.37920576333999634, -0.002579854568466544, 0.30264151096343994, 0.11487504839897156, 0.608214795589447, -0.1126706674695015, 0.4153241515159607, -0.43070611357688904, -0.31537818908691406, -0.15144212543964386, 0.3882722556591034, 0.4287033677101135, 0.4461061358451843, 0.11716939508914948, -0.48749232292175293, -0.2803375720977783, 0.22302299737930298, -0.3046562969684601, 0.18189018964767456, 0.15971502661705017, 0.4815053641796112, -0.2892160713672638, 0.050861719995737076, 0.22113977372646332, 0.0613696314394474, 0.08790192008018494, 0.11232743412256241, -0.5730607509613037, -0.12852632999420166, 0.9637240171432495, 0.2662489116191864, -0.13138633966445923, 0.6422317028045654, 0.6386451125144958, 0.4329914450645447, 0.19351541996002197, 0.20166242122650146, 0.3092750012874603, -0.10225701332092285, 0.04001785069704056, -0.6719081401824951, -0.034705694764852524, -0.9777337312698364, -0.560053288936615, -0.24462224543094635, 0.4302274286746979, 0.23692387342453003, -0.20450472831726074, -0.2302928864955902, -0.2769280970096588, -0.10810544341802597, -0.41871699690818787, -0.04243956133723259, -0.3218182623386383, -0.21713055670261383, -0.3797000050544739, 0.03777061030268669, -0.36536189913749695, 0.5534241795539856, -0.10209368169307709, -0.9663770794868469, 0.5997686982154846, -0.12782658636569977, 0.4984034597873688, -0.36052483320236206, 0.037771742790937424, 0.02787231095135212, -0.6371557712554932, 0.07856790721416473, -0.12850463390350342, 0.10006986558437347, -0.7317615747451782, -0.2807317078113556, -0.33897846937179565, -0.16280676424503326, 0.5373175144195557, -0.18092654645442963, 0.200003981590271, -0.1565285325050354, -0.2784882187843323, -0.7298346757888794, 0.7837963104248047, -0.48309093713760376, -0.9062814116477966, -0.6010487675666809, 0.28640055656433105, 0.02777811698615551, -0.05199550837278366, 0.5180231332778931], [-0.21597866714000702, 0.3115392327308655, 0.4895675778388977, 0.41567152738571167, -0.36494505405426025, 0.022068826481699944, -0.2382744401693344, 0.14355483651161194, -0.8178213834762573, 0.04296507313847542, -0.3631618618965149, 0.28553634881973267, -0.42798951268196106, 0.07750937342643738, -0.37378084659576416, 0.4097549319267273, 0.07990013808012009, -0.03626852482557297, -0.24876132607460022, 0.2632812261581421, 0.4169659912586212, 0.7076395153999329, 0.4235631823539734, -0.2188817411661148, -0.15800221264362335, 0.25376683473587036, -0.12291119247674942, -0.6606667637825012, 0.15590347349643707, 0.09010115265846252, 0.5349248647689819, 0.5630248785018921, 0.09519062936306, -0.075063556432724, 0.03674611076712608, -0.4007192850112915, -0.19714108109474182, -0.1125822588801384, -0.08412092179059982, -0.2922990918159485, -0.29945939779281616, 0.14844754338264465, 0.21130245923995972, 0.46810460090637207, -0.33425387740135193, 0.10181821137666702, -0.35809290409088135, -0.3469325006008148, -0.0905199721455574, 0.39364397525787354, 0.34258851408958435, 0.7706094980239868, -0.026489470154047012, -0.07678088545799255, 0.21967510879039764, -0.2183067500591278, -0.09387743473052979, -0.05890839919447899, 0.09797102957963943, 0.09588032960891724, -0.28501129150390625, -0.2127242237329483, 0.25207558274269104, -0.2790733575820923, 0.3316109776496887, -0.005939063150435686, -0.6558058261871338, -0.15849444270133972, 0.6230677366256714, 0.16916300356388092, -0.11092251539230347, 0.39829808473587036, 0.29853424429893494, 0.4604063928127289, 0.19925034046173096, 0.18737201392650604, 0.15587936341762543, -0.10439135134220123, -0.23380149900913239, -0.24351567029953003, -0.015856897458434105, -0.6994860172271729, -0.09312545508146286, -0.3148801028728485, -0.053864605724811554, 0.35269081592559814, 0.15649758279323578, -0.013653941452503204, -0.3194209933280945, -0.1012280061841011, -0.2521652281284332, -0.08493154495954514, -0.006958651356399059, -0.1341276317834854, -0.3962528705596924, -0.16526131331920624, -0.22227951884269714, 0.34685060381889343, -0.32351258397102356, -0.6456175446510315, 0.12187432497739792, -0.4827350378036499, 0.4732625484466553, -0.26159000396728516, -0.04297091066837311, -0.10116050392389297, -0.43706953525543213, 0.21915209293365479, 0.010234528221189976, -0.1122826412320137, -0.2582955062389374, -0.4114634394645691, -0.25858330726623535, 0.21455511450767517, 0.18943187594413757, -0.32906726002693176, 0.21086733043193817, 0.332724004983902, 0.022760186344385147, -0.6263368129730225, 0.5625724792480469, -0.5936035513877869, 0.03475094959139824, -0.1615740954875946, 0.17784331738948822, -0.37535062432289124, -0.1309758424758911, 0.37799596786499023], [-0.271286278963089, 0.4169315695762634, 0.2514443099498749, 0.49209722876548767, -0.39675959944725037, 0.08947010338306427, -0.06997669488191605, 0.1658862680196762, -0.5160196423530579, 0.2145872414112091, -0.42032590508461, 0.17665529251098633, -0.48565927147865295, 0.21797162294387817, -0.24421612918376923, 0.240213081240654, 0.06982112675905228, 0.20960009098052979, -0.14503692090511322, 0.21126118302345276, 0.31487083435058594, 0.5673394203186035, 0.10952521115541458, -0.18468473851680756, 0.1614016890525818, 0.24931228160858154, -0.12609398365020752, -0.5329970121383667, 0.4123438000679016, 0.042547669261693954, 0.5578083992004395, 0.36875948309898376, 0.1737564504146576, -0.37796494364738464, 0.1669880598783493, -0.12987636029720306, 0.03633813560009003, 0.03898227587342262, -0.3485713005065918, -0.3179325461387634, 0.16133826971054077, 0.0433465912938118, 0.12107495218515396, 0.6851089000701904, -0.31193381547927856, 0.1787092089653015, -0.19072245061397552, -0.1675574779510498, -0.1697051227092743, 0.5017982125282288, 0.12734559178352356, 0.6748188138008118, -0.01575966365635395, -0.35160428285598755, 0.09418414533138275, -0.2078993171453476, -0.0641070008277893, 0.049290671944618225, -0.19490781426429749, 0.13268011808395386, 0.34278568625450134, -0.11942142993211746, 0.10710466653108597, -0.03372451290488243, 0.30742979049682617, 0.031115110963582993, -0.5003399848937988, 0.29819920659065247, 0.7939574122428894, 0.3741389214992523, -0.1267862766981125, 0.1329209804534912, 0.5796459317207336, 0.4679371416568756, 0.4010362923145294, 0.192987859249115, 0.23090343177318573, 0.24363629519939423, -0.11339196562767029, -0.4651840627193451, -0.10861259698867798, -0.45923343300819397, -0.18778377771377563, -0.1157265231013298, -0.05536209046840668, 0.26596084237098694, -0.17393805086612701, -0.2329188734292984, -0.6736653447151184, 0.11432086676359177, -0.3177359700202942, -0.1173740103840828, -0.37857937812805176, -0.17110100388526917, -0.1790960282087326, -0.12176456302404404, -0.13551779091358185, 0.15632688999176025, -0.6477801203727722, -0.8678339719772339, 0.07912860065698624, -0.46234071254730225, 0.011523894034326077, -0.43177536129951477, -0.009273149073123932, 0.18571683764457703, -0.4566231369972229, 0.200481578707695, -0.13374871015548706, 0.10597123950719833, -0.25335997343063354, -0.2591303586959839, -0.2806790769100189, 0.05039657652378082, 0.3904534876346588, -0.13104821741580963, 0.458312064409256, -0.10172874480485916, -0.12602849304676056, -0.6457246541976929, 0.37917137145996094, -0.49209097027778625, -0.049618374556303024, -0.21159233152866364, 0.05491451174020767, -0.10013692080974579, 0.28634461760520935, 0.34277912974357605], [-0.2637462019920349, 0.22862300276756287, 0.20199979841709137, 0.3375365436077118, -0.5690250992774963, -0.5635401010513306, -0.29020705819129944, 0.046475399285554886, -0.4891047477722168, 0.6506024599075317, -0.36633074283599854, -0.04810105636715889, -0.6665033102035522, 0.45893508195877075, 0.1331564038991928, 0.30102774500846863, -0.00024144661438185722, 0.4940311014652252, 0.14011144638061523, 0.7427287101745605, 0.7561516761779785, 0.5726566314697266, 0.2539981007575989, -0.31786197423934937, 0.8125014305114746, 0.24590139091014862, -0.1464737355709076, -0.6453317403793335, 0.34856775403022766, -0.02565917745232582, 0.7839449644088745, 0.15042714774608612, -0.046944282948970795, -0.688419759273529, 0.619945228099823, 0.501628577709198, 0.5163227915763855, -0.09845127165317535, -0.42291364073753357, -0.25690966844558716, 0.16099023818969727, 0.06997466087341309, -0.275795578956604, 0.4041111469268799, -0.13765862584114075, 0.23411712050437927, 0.509781539440155, -0.3726281225681305, -0.06675317138433456, 0.600212037563324, -0.11743813008069992, 0.4163073003292084, -0.04822279512882233, -0.7436143755912781, -0.13604584336280823, -0.08775464445352554, -0.324201375246048, -0.10629324615001678, -0.3529498875141144, 0.5540111064910889, 0.20356711745262146, 0.226422980427742, 0.22368483245372772, -0.10318643599748611, 0.29534000158309937, -0.03553541377186775, -0.3461868166923523, -0.24381564557552338, 1.1722902059555054, 0.2472391128540039, -0.048811376094818115, 0.17629963159561157, 0.554225504398346, 0.15565159916877747, 0.2818205952644348, -0.021256187930703163, 0.2491554468870163, 0.3516077697277069, -0.016891825944185257, -0.8166099190711975, -0.13590508699417114, -0.4798242151737213, -0.46465516090393066, 0.19970346987247467, 0.3390292525291443, 0.8524237275123596, -0.1393016278743744, -0.2121962308883667, -0.886629581451416, -0.45398402214050293, -0.1351870596408844, -0.12458550930023193, -0.23370566964149475, 0.03038734570145607, 0.27130207419395447, 0.24418877065181732, 0.044377584010362625, 0.4842645227909088, -0.5929861664772034, -0.8491301536560059, 0.22750917077064514, -0.4759039878845215, -0.3148031532764435, 0.021349018439650536, 0.5058892369270325, 0.31633299589157104, -0.7817232608795166, -0.22147086262702942, 0.3071659207344055, -0.21755345165729523, -0.5898213386535645, -0.5277438759803772, -0.7575516700744629, -0.11777113378047943, 0.3451380133628845, -0.23239576816558838, 0.6655284762382507, -0.4720730781555176, -0.3874989449977875, -0.8847856521606445, 0.4160041809082031, -0.19201794266700745, -0.1651611179113388, -0.19100089371204376, 0.2535206079483032, -0.06092134490609169, 0.1741478443145752, 0.427883118391037], [0.005040188319981098, 0.5084340572357178, 0.5651834011077881, 0.0056939395144581795, -0.09865714609622955, 0.06708478182554245, -0.06002093851566315, 0.5239682793617249, -0.32921114563941956, 0.28179019689559937, -0.5228078365325928, 0.24579425156116486, -0.6805683970451355, 0.46695369482040405, -0.47974535822868347, 0.10324735194444656, 0.3561950922012329, -0.09258469939231873, 0.18503138422966003, 0.3169141113758087, 0.09513627737760544, 0.8434571623802185, 0.385713666677475, 0.14290864765644073, 0.3245920240879059, 0.012410295195877552, -0.13995149731636047, -0.5103666186332703, 0.5449036955833435, 0.027220746502280235, 0.15110152959823608, 0.45015817880630493, 0.2530871033668518, 0.08017665147781372, 0.1806984841823578, -0.1845167875289917, -0.02250051312148571, 0.019065041095018387, -0.19141241908073425, -0.27679184079170227, -0.023929676041007042, 0.3463357090950012, 0.28486090898513794, 0.30570170283317566, -0.2814342677593231, 0.10725314915180206, -0.48481327295303345, -0.200872004032135, -0.4289841949939728, 0.3605940043926239, 0.407540887594223, 0.3493039011955261, 0.1362503319978714, -0.12356138974428177, -0.15479233860969543, -0.14590942859649658, -0.4808361530303955, 0.09653054922819138, 0.13192778825759888, 0.19227008521556854, 0.0110738230869174, 0.007683733943849802, 0.04060050845146179, -0.14460024237632751, 0.04126618057489395, -0.15716363489627838, -0.6301484107971191, 0.11084649711847305, 0.6351633071899414, -0.13268183171749115, -0.35507526993751526, 0.32309871912002563, 0.3138205111026764, 0.3245714604854584, 0.6871204376220703, -0.030684819445014, 0.41404151916503906, 0.07114865630865097, 0.2802636921405792, -0.24093689024448395, 0.1073947548866272, -0.7783594727516174, -0.45567864179611206, 0.03376166895031929, 0.24866659939289093, 0.05456570163369179, 0.030873145908117294, 0.014827691949903965, -0.19280287623405457, -0.32424959540367126, -0.000531670346390456, -0.10690221190452576, -0.3134106993675232, 0.26000192761421204, -0.04069928452372551, -0.3521865904331207, -0.3046717047691345, 0.41551879048347473, -0.08948768675327301, -0.640006959438324, 0.09823460131883621, -0.4815632402896881, 0.3855123817920685, -0.11180036514997482, -0.17392794787883759, 0.27759313583374023, -0.20012624561786652, 0.2593611180782318, -0.3051356077194214, 0.06783106923103333, -0.5341737270355225, 0.202871635556221, -0.42663711309432983, 0.34107983112335205, 0.3911416828632355, 0.08089106529951096, 0.15296299755573273, 0.17623111605644226, -0.18493907153606415, -0.5441678762435913, 0.4263944625854492, -0.6210158467292786, -0.6273623108863831, -0.3242736756801605, 0.15880726277828217, -0.025438524782657623, -0.015883969143033028, 0.4086615741252899], [0.16585342586040497, 0.2921609580516815, 0.433271586894989, 0.128445103764534, -0.1236855760216713, -0.04885666444897652, 0.13636191189289093, 0.8352939486503601, -0.6647660732269287, -0.1978626698255539, -0.2797427177429199, 0.2126057893037796, -0.2679762542247772, 0.39410412311553955, -0.1732010394334793, 0.27566996216773987, 0.3849031925201416, -0.16861584782600403, 0.37295931577682495, 0.04419762268662453, 0.18484769761562347, 0.6483510136604309, -0.026056237518787384, -0.030037490651011467, 0.07056836783885956, -0.3663698434829712, -0.018773822113871574, -0.2876008152961731, 0.831403911113739, -0.46339061856269836, -0.06521224975585938, 0.30172795057296753, -0.2862498164176941, -0.3000738024711609, 0.3324580192565918, -0.4096161425113678, 0.17061953246593475, 0.06898327171802521, 0.0034293439239263535, -0.4556744396686554, 0.16624458134174347, 0.30550986528396606, 0.3860169053077698, 0.7029948830604553, 0.2661670446395874, -0.04595411196351051, -0.12363255769014359, -0.22401532530784607, 0.0800020769238472, 0.14316102862358093, 0.35069355368614197, -0.06784578412771225, 0.21466095745563507, -0.37388500571250916, 0.037805430591106415, 0.20973598957061768, -0.5481796860694885, -0.24674545228481293, 0.16308575868606567, 0.020206056535243988, 0.2932637631893158, 0.15149688720703125, 0.18026916682720184, 0.15213724970817566, 0.10629910975694656, -0.3643021583557129, -0.10356173664331436, 0.2242456078529358, 0.6968773603439331, 0.3147623836994171, -0.42736953496932983, 0.18867647647857666, -0.12140558660030365, 0.3048062026500702, 0.30082130432128906, -0.09478527307510376, 0.19531285762786865, -0.08766165375709534, 0.03342912346124649, -0.4718768894672394, -0.25822609663009644, -0.1384139060974121, 0.03079644963145256, -0.016359776258468628, -0.05815466120839119, 0.2752242386341095, -0.09089463204145432, -0.22815722227096558, -0.4706123173236847, 0.416066974401474, 0.023641124367713928, -0.2576833665370941, -0.5719563961029053, 0.06080953776836395, -0.10022080689668655, -0.4126141667366028, -0.0072327591478824615, 0.331368625164032, -0.7316179275512695, -0.4218383729457855, 0.10203288495540619, -0.023987866938114166, -0.05211595445871353, -0.36648955941200256, 0.027442792430520058, 0.5439082384109497, 0.07094809412956238, 0.33872485160827637, -0.40747499465942383, 0.030858706682920456, -0.41056954860687256, 0.11029301583766937, 0.03039136901497841, 0.27692341804504395, 0.505102276802063, 0.0863848328590393, -0.30063396692276, -0.050906214863061905, 0.05358956381678581, -0.21772558987140656, 0.08647504448890686, -0.7100647687911987, -0.3432631194591522, -0.07700072228908539, 0.3252883851528168, -0.0688018798828125, 0.46819087862968445, 0.025104710832238197], [0.21118822693824768, -0.08517316728830338, -0.0062222834676504135, -0.37533825635910034, 0.04278063029050827, 0.1972597986459732, -0.07765365391969681, -0.06998113542795181, -0.39403101801872253, -0.25942301750183105, -0.5811511278152466, -0.26113319396972656, -0.2063472718000412, 0.2777096927165985, -0.4573681056499481, -0.3151237666606903, -0.15319916605949402, 0.27013012766838074, -0.10602635890245438, -0.3284972310066223, 0.6053763031959534, 0.4618305563926697, 0.14091810584068298, 0.20209607481956482, 0.7275283336639404, 0.03310829773545265, -0.1975105255842209, -0.2666940987110138, 0.25570017099380493, -0.5714860558509827, 0.26849645376205444, 0.4846375584602356, 0.314668208360672, 0.1309017837047577, 0.36052900552749634, 0.19343994557857513, -0.04523167386651039, 0.3817433714866638, 0.5174065232276917, -0.8166897892951965, -0.5195726156234741, 0.010469738394021988, 0.16849060356616974, 0.8288931846618652, -0.023262465372681618, 0.16313430666923523, -0.5094690918922424, -0.09121893346309662, 0.3327993154525757, -0.12088650465011597, 0.12448594719171524, -0.20516227185726166, 0.2830374240875244, 0.5218376517295837, -0.18922121822834015, -0.6496570110321045, -0.2935604453086853, 0.037574488669633865, 0.30329665541648865, -0.5111014246940613, 0.4389941692352295, -0.11202292144298553, -0.0030804562848061323, -0.24742725491523743, -0.21090289950370789, -0.14550939202308655, -0.5069612264633179, 0.15627500414848328, -0.3338029682636261, 0.5023561120033264, -0.2148262858390808, 0.5892133116722107, 0.21869151294231415, 0.2125006765127182, 0.9192925691604614, 0.0037411930970847607, 0.055686887353658676, 0.2643386125564575, 0.08268075436353683, 0.07751931995153427, -0.1590159833431244, -0.29517263174057007, -0.03417818620800972, 0.3587509095668793, 3.478608732621069e-06, -0.7861568927764893, 0.08761215209960938, -0.22143830358982086, -0.4240531921386719, -0.13329678773880005, 0.03232313692569733, -0.16198192536830902, -0.07898301631212234, 0.21814414858818054, 0.36791783571243286, 0.014196469448506832, -0.21764488518238068, 0.13181358575820923, -0.1749476045370102, -0.19900424778461456, -0.20764018595218658, -0.24609439074993134, 0.7525566220283508, -0.22962012887001038, -0.09962362051010132, 0.5708337426185608, 0.35152485966682434, 0.4792713522911072, -0.5098210573196411, 0.23942792415618896, -0.07032907009124756, 0.34920698404312134, -0.2990962564945221, 0.5175213813781738, 0.3555375337600708, -0.3683791160583496, 0.13710352778434753, 0.25777292251586914, -0.13527390360832214, -0.2040322721004486, 0.32072827219963074, -0.9728755354881287, -0.03576771169900894, -0.15052396059036255, 0.18555361032485962, -0.38495251536369324, 0.25280430912971497, 0.28890305757522583], [-0.1626666933298111, 0.4369065761566162, 0.2222476750612259, -0.04405028000473976, -0.11489152163267136, -0.24784497916698456, 0.07728064060211182, 0.2624572515487671, -0.8123593330383301, 0.09630326181650162, -0.6375614404678345, 0.38310500979423523, -0.5115957260131836, 0.48737531900405884, -0.9718561768531799, -0.08339483290910721, 0.5177329182624817, -0.01250709779560566, -0.11267642676830292, 0.3020804524421692, 0.7195907235145569, 0.29724109172821045, 0.7622225284576416, 0.15642547607421875, 0.4049075245857239, 0.1653381735086441, -0.099272221326828, -0.25703883171081543, 0.43470996618270874, -0.3678688108921051, 0.22379350662231445, 0.5930166244506836, 0.2587157189846039, 0.05536908656358719, 0.09159497916698456, 0.0027993484400212765, -0.029096242040395737, 0.24173060059547424, -0.18500970304012299, -0.4671080708503723, -0.32190877199172974, 0.14471998810768127, 0.42324960231781006, 0.9100102782249451, -0.16049641370773315, -0.23711039125919342, -0.13889487087726593, 0.09310197085142136, -0.04035051539540291, 0.03846309334039688, 0.1893656849861145, 0.4971318244934082, 0.24628238379955292, -0.13477887213230133, -0.03124680556356907, -0.5840749144554138, -0.039571862667798996, -0.38109204173088074, 0.36436718702316284, -0.27346986532211304, 0.5336005091667175, -0.09710246324539185, 0.298505574464798, -0.28975385427474976, -0.06445968151092529, -0.3451712131500244, -0.1889289915561676, 0.68263840675354, 0.47829359769821167, 0.38203707337379456, -0.5498435497283936, -0.12274695932865143, -0.04301058128476143, 0.4082338213920593, 0.2319488525390625, -0.09659792482852936, 0.25484082102775574, 0.21471287310123444, 0.08980327844619751, 0.10707051306962967, -0.2495080679655075, 0.016426410526037216, 0.09439985454082489, 0.06435774266719818, -0.30425825715065, 0.5949236154556274, -0.10469420999288559, -0.11990401893854141, -0.4166107773780823, 0.31675291061401367, 0.011968602426350117, 0.08218442648649216, -0.48867619037628174, 0.45880213379859924, 0.0287326630204916, -0.31695568561553955, 0.18735750019550323, -0.06284315884113312, -0.45384645462036133, -0.33802950382232666, -0.6133320927619934, -0.388619065284729, -0.07200685143470764, -0.3904130160808563, -0.2105477899312973, 0.21059486269950867, -0.18950849771499634, 0.3478739559650421, -0.10542862117290497, 0.14675195515155792, 0.29960599541664124, 0.27235299348831177, -0.10773060470819473, 0.4390276372432709, 0.021896356716752052, 0.12756739556789398, 0.3710671365261078, 0.2573432922363281, -0.136506587266922, -0.3076508939266205, 0.16570492088794708, -0.8496708869934082, 0.3425110876560211, 0.07806957513093948, 0.0925481915473938, -0.25912174582481384, -0.03642721101641655, 0.14247627556324005], [-0.0930129811167717, 0.26878437399864197, 0.26822975277900696, -0.05798658728599548, -0.5229570865631104, 0.059369899332523346, -0.04118797555565834, 0.10445501655340195, -0.43511492013931274, 0.11897704005241394, -0.2632952332496643, 0.3060694634914398, -0.643692672252655, 0.41031545400619507, -0.3245762586593628, -0.25118276476860046, 0.23349885642528534, -0.04337636008858681, 0.0013042892096564174, 0.005804318003356457, -0.054738420993089676, 0.8956615924835205, 0.3300451934337616, 0.11027160286903381, 0.2462175488471985, -0.0342060811817646, 0.05667562782764435, -0.4109981954097748, 0.3852198123931885, -0.087273009121418, -0.0318407341837883, 0.3753877878189087, 0.34612002968788147, -0.05527113005518913, 0.1566746085882187, 0.36700743436813354, -0.0775509774684906, 0.04410794749855995, 0.12403349578380585, 0.004855426028370857, -0.38254448771476746, 0.13144265115261078, 0.177350714802742, 0.1997934728860855, -0.3214617967605591, -0.08206368237733841, -0.4961856007575989, 0.15099167823791504, -0.030126504600048065, 0.6451833844184875, 0.25645577907562256, -0.23602236807346344, 0.11348046362400055, 0.2150508165359497, -0.07074781507253647, -0.2865186333656311, -0.4208719730377197, 0.07961525768041611, -0.1192750334739685, -0.09532944113016129, 0.16621476411819458, -0.3435710668563843, -0.004673594143241644, -0.0650283619761467, -0.011581014841794968, 0.00020304183999542147, -0.23832447826862335, 0.06715981662273407, 0.452260822057724, 0.06229561194777489, 0.05773509666323662, 0.14442111551761627, 0.4007655680179596, 0.3104799687862396, 1.0338338613510132, 0.07619382441043854, 0.3020228147506714, 0.17305883765220642, -0.3212752044200897, 0.004206053446978331, -0.1701570600271225, -0.8955583572387695, -0.12181662768125534, -0.026166338473558426, 0.1370447278022766, -0.0284950602799654, -0.07783062756061554, 0.1029435470700264, -0.3726394772529602, -0.03462977707386017, -0.012087322771549225, -0.30050963163375854, -0.068893663585186, 0.1254812628030777, -0.006553561892360449, 0.08190124481916428, -0.200310617685318, 0.0030140900053083897, -0.10752742737531662, -0.14941798150539398, 0.2031257003545761, -0.40959790349006653, -0.15500356256961823, -0.1066284254193306, 0.04043741524219513, 0.05819389224052429, 0.0005716505693271756, 0.18303000926971436, -0.4409762918949127, 0.09075357019901276, -0.3433885872364044, -0.2881874740123749, -0.6494907736778259, 0.14336051046848297, 0.06537268310785294, -0.032222140580415726, 0.14989826083183289, 0.3318164050579071, -0.13950826227664948, -0.3874437212944031, 0.18498530983924866, -1.0515085458755493, -0.08789821714162827, -0.29678210616111755, -0.08746337890625, 0.12015559524297714, 0.08442124724388123, 0.18843470513820648], [-0.0037642752286046743, 0.4382144808769226, -0.09658978134393692, -0.0240533035248518, -0.2817406952381134, 0.6921057105064392, 0.5336313247680664, 0.31144198775291443, -0.48154717683792114, 0.42028507590293884, -0.6616740822792053, -0.3298099637031555, -0.11021982878446579, 0.15989917516708374, -0.007238529622554779, 0.08365319669246674, 0.22304767370224, 0.2999853789806366, -0.35934725403785706, -0.262382447719574, 0.8871181607246399, 0.33817267417907715, 0.06691965460777283, -0.027844084426760674, 0.6744795441627502, 0.03126977011561394, 0.38159996271133423, 0.1706121414899826, 0.35025689005851746, 0.18722523748874664, 0.33092740178108215, 0.5688981413841248, -0.33080363273620605, -0.18353872001171112, -0.019040830433368683, -0.12580668926239014, -0.31729352474212646, -0.1813989132642746, -0.13194458186626434, -0.8460834622383118, -0.3090910315513611, -0.4394739866256714, 0.1688341200351715, 0.15545614063739777, -0.07277335971593857, 0.13637718558311462, 0.0810098871588707, 0.20987610518932343, -0.258806437253952, 0.15373867750167847, 0.49660617113113403, 0.15396220982074738, -0.08388736099004745, 0.20147812366485596, 0.053556740283966064, -0.30337658524513245, -0.6616095900535583, -0.062030352652072906, 0.024243848398327827, -0.08849848061800003, 1.038975477218628, -0.09452934563159943, -0.3048725128173828, 0.013623951002955437, -0.3742210566997528, -0.05310656875371933, -0.30462169647216797, -0.08316052705049515, -0.1748497635126114, 0.5786902904510498, -0.08810003846883774, -0.04662757366895676, 0.6339623928070068, 0.20994552969932556, 0.4359946548938751, 0.4751688838005066, 0.7580797672271729, -0.29449427127838135, -0.005154912825673819, 0.2838532626628876, 0.1015084907412529, -0.494522362947464, -0.39636290073394775, 0.1545669585466385, 0.1982414722442627, -0.45406413078308105, 0.25652918219566345, -0.01634008251130581, -0.5790818929672241, 0.19152873754501343, -0.24593903124332428, -0.030288873240351677, -0.15457205474376678, -0.14450183510780334, 0.3061966598033905, -0.19182632863521576, -0.1513630747795105, 0.35919564962387085, -0.11279302090406418, -0.8262825608253479, -0.13959205150604248, -0.4753016531467438, -0.07746274024248123, -0.30446192622184753, -0.02080707997083664, 0.5649784803390503, 0.09652810543775558, 0.27542391419410706, -1.0074243545532227, 0.4031940996646881, -0.3595072329044342, 0.22166140377521515, -0.39217665791511536, 0.49808549880981445, -0.06905423104763031, 0.11637280881404877, -0.2514030933380127, -0.36692461371421814, 0.09481997042894363, -0.44926679134368896, 0.10677509009838104, -0.9323280453681946, 0.005774447228759527, -0.11297223716974258, 0.09263411909341812, -0.26095443964004517, 0.9382030367851257, 0.6276777982711792], [-0.12565621733665466, 0.3865891695022583, 0.11957722902297974, 0.07959069311618805, 0.036231353878974915, 0.07555419951677322, -0.11753206700086594, 0.4398120939731598, -0.5891777276992798, 0.1135035827755928, -0.6533875465393066, 0.213618665933609, -0.4283978343009949, 0.2938324511051178, -0.08696114271879196, 0.21845471858978271, 0.3952355682849884, 0.038990795612335205, 0.19591480493545532, 0.40164995193481445, 0.3154533803462982, 0.6019676327705383, 0.283021479845047, 0.2716105580329895, 0.3132733404636383, 0.3830172121524811, 0.06058663874864578, -0.4667488932609558, 0.5338322520256042, -0.3791489899158478, 0.029275795444846153, 0.22281323373317719, -0.11888429522514343, -0.36267638206481934, 0.16422274708747864, -0.0405181385576725, -0.03292061388492584, -0.1324973851442337, -0.3060379922389984, -0.2811882793903351, -0.14359290897846222, 0.14998900890350342, -0.04138991981744766, 0.41659054160118103, 0.004490752704441547, -0.15304160118103027, -0.0713263526558876, 0.2041541039943695, -0.18930761516094208, 0.1530740112066269, 0.29275745153427124, -0.04904329404234886, 0.11251839250326157, -0.368135541677475, 0.09210321307182312, -0.15620669722557068, -0.36976414918899536, -0.2899019420146942, 0.1680721938610077, 0.15616773068904877, 0.25477057695388794, 0.11949633806943893, 0.20386791229248047, 0.14299552142620087, -0.31507357954978943, -0.009211721830070019, -0.21342723071575165, 0.15959464013576508, 0.5644369721412659, 0.4069196879863739, -0.40566596388816833, 0.20438872277736664, 0.15443634986877441, 0.31963032484054565, 0.22342951595783234, -0.09802345931529999, 0.28227514028549194, 0.13340260088443756, 0.12689176201820374, -0.36354756355285645, -0.2642063796520233, -0.032880477607250214, -0.13284339010715485, 0.15260863304138184, 0.05615217611193657, 0.49064499139785767, -0.06930671632289886, -0.1091371476650238, -0.16729691624641418, 0.22430196404457092, 0.024136176332831383, -0.02981993742287159, -0.40375810861587524, 0.15769045054912567, -0.17615504562854767, 0.12292147427797318, -0.07680434733629227, -0.0038136972580105066, -0.12672755122184753, -0.22777444124221802, 0.12496989220380783, -0.0041991486214101315, -0.20204265415668488, -0.35938963294029236, 0.148004949092865, 0.4046834111213684, -0.19192664325237274, 0.3347933292388916, -0.27836060523986816, 0.12982648611068726, -0.021986177191138268, -0.10339464247226715, -0.2449382096529007, -0.01883411593735218, 0.15033413469791412, 0.03754507377743721, -0.16699907183647156, -0.16635151207447052, -0.1382029503583908, -0.24875321984291077, 0.1993604600429535, -0.7990899085998535, -0.2023535519838333, -0.31890034675598145, 0.22311647236347198, 0.045175060629844666, 0.23848988115787506, 0.30185267329216003], [-0.19496949017047882, 0.33678144216537476, 0.26781386137008667, 0.11658484488725662, -0.23631218075752258, 0.04326567053794861, -0.19993624091148376, 0.3028256595134735, -0.4629177451133728, 0.04478128254413605, -0.31172579526901245, -0.13318857550621033, -0.5390883088111877, 0.26699763536453247, -0.06566265225410461, 0.1871512234210968, 0.06843382120132446, 0.29860085248947144, -0.05472593009471893, 0.329063355922699, 0.2254466563463211, 0.7135141491889954, 0.07285089790821075, 0.11476562172174454, 0.22839820384979248, 0.12012525647878647, -0.05945458635687828, -0.422698974609375, 0.2917806804180145, -0.14715944230556488, 0.2449534684419632, 0.2964051067829132, -0.09434481710195541, -0.3282637298107147, 0.3059838116168976, 0.05249698460102081, 0.17257362604141235, -0.052946265786886215, -0.33941763639450073, -0.3316778242588043, -0.07075607031583786, -0.09385767579078674, -0.07772152125835419, 0.11377730965614319, -0.047661881893873215, 0.15197524428367615, -0.2360066920518875, -0.354544073343277, -0.2329692393541336, 0.24882453680038452, 0.22128283977508545, 0.15391014516353607, 0.05872918665409088, -0.32463338971138, -0.04515892639756203, -0.19397078454494476, -0.10573253780603409, -0.012917611747980118, -0.0542428120970726, 0.1405649036169052, 0.06828754395246506, 0.008794001303613186, 0.11008283495903015, 0.1882273405790329, 0.20981848239898682, 0.23340608179569244, -0.3468981683254242, 0.08868023753166199, 0.8413337469100952, 0.0070207081735134125, -0.09724816679954529, 0.062092579901218414, 0.20158879458904266, 0.1020616888999939, 0.36568590998649597, -0.04543480649590492, 0.19861508905887604, 0.14590021967887878, -0.0635741651058197, -0.27891889214515686, -0.172442227602005, -0.5225487351417542, -0.2410857081413269, 0.1395636647939682, 0.20348478853702545, 0.2934921383857727, -0.02295609936118126, -0.1917867660522461, -0.32722198963165283, -0.08185956627130508, -0.009263901971280575, -0.07480749487876892, -0.15852320194244385, 0.14362938702106476, -0.050211332738399506, 0.2060445100069046, 0.033368710428476334, 0.34712880849838257, -0.20582978427410126, -0.47641879320144653, 0.11059066653251648, -0.2543516755104065, 0.0018942138412967324, -0.06139250844717026, -0.01603851467370987, 0.379900187253952, -0.3488543629646301, 0.013241049833595753, -0.18174174427986145, -0.029414191842079163, -0.3814042806625366, -0.17016728222370148, -0.38407349586486816, 0.05044516175985336, 0.19691602885723114, -0.04564913734793663, 0.009296899661421776, -0.15795008838176727, -0.22655530273914337, -0.6716935038566589, 0.3142774999141693, -0.5590054988861084, -0.3161681294441223, -0.24868223071098328, 0.1888357698917389, -0.019568609073758125, 0.22082263231277466, 0.41004225611686707], [-0.20837992429733276, 0.010962391272187233, 0.10553138703107834, -0.04754688963294029, 0.1619575023651123, 0.05446214973926544, -0.17762979865074158, -0.09210819005966187, -0.44683837890625, -0.36582151055336, -0.24969811737537384, -0.1668124496936798, -0.26546525955200195, 0.11825723201036453, -0.42913147807121277, 0.09005080163478851, 0.015758128836750984, 0.2362843155860901, -0.20530596375465393, 0.06295648962259293, 0.42240196466445923, 0.6212557554244995, 0.30819904804229736, 0.10802965611219406, 0.5018997192382812, 0.21560148894786835, -0.20525816082954407, -0.10440186411142349, 0.24689067900180817, -0.4269247055053711, 0.20337022840976715, 0.3684938848018646, 0.03010864369571209, -0.172188863158226, -0.06231790781021118, -0.014918629080057144, -0.05151359736919403, 0.1027202382683754, -0.1458245813846588, -0.42094749212265015, -0.42029884457588196, -0.19130036234855652, 0.24985459446907043, 0.3793396055698395, -0.07420410215854645, -0.06389086693525314, -0.3082757294178009, 0.10430688410997391, -0.08309635519981384, 0.09100668132305145, 0.28712376952171326, 0.17613133788108826, 0.42015886306762695, 0.06895213574171066, 0.14403113722801208, -0.49749860167503357, 0.1971774697303772, 0.09537331014871597, 0.2739211618900299, 0.027239374816417694, 0.2645918130874634, 0.13553674519062042, 0.2503189444541931, -0.03427271172404289, -0.09285029023885727, 0.10870324820280075, -0.3758450746536255, 0.43559029698371887, 0.3117011487483978, 0.27274197340011597, -0.3488975763320923, 0.008207947947084904, -0.0019148841965943575, 0.10724256932735443, 0.47682327032089233, -0.17324784398078918, 0.21485038101673126, 0.24761825799942017, -0.13603366911411285, 0.0371415801346302, -0.16540083289146423, 0.09763530641794205, -0.05590357258915901, 0.4139326810836792, -0.04462042823433876, 0.0209815613925457, 0.12880006432533264, -0.2946348786354065, -0.039693962782621384, 0.1014169231057167, 0.20836098492145538, 0.024574950337409973, -0.04832695424556732, 0.25174450874328613, -0.0836639478802681, 0.3246309459209442, 0.15310439467430115, 0.031136050820350647, -0.21349723637104034, -0.2688356041908264, -0.14893274009227753, -0.2919164299964905, 0.11700372397899628, -0.06922649592161179, -0.09657570719718933, 0.4392782747745514, -0.1894102841615677, 0.430529922246933, -0.15305781364440918, 0.10047131031751633, 0.1051485612988472, 0.15905791521072388, -0.03695686161518097, 0.431965172290802, 0.20406672358512878, -0.07835567742586136, -0.13335414230823517, 0.1562385857105255, -0.12149640172719955, -0.3278903067111969, -0.025529203936457634, -0.709344208240509, 0.08959090709686279, -0.14236244559288025, 0.28240257501602173, -0.3827143609523773, 0.17127425968647003, 0.17851148545742035], [0.050164029002189636, 0.3420688509941101, -0.11451240628957748, 0.062391143292188644, -0.1939539909362793, 0.4964202642440796, 0.2596195340156555, 0.4555484354496002, -0.9423474073410034, 0.0903855487704277, -0.3818899095058441, -0.3294292986392975, 0.11936581879854202, 0.2730715870857239, -0.014284437522292137, 0.21950004994869232, -0.060200970619916916, 0.36079320311546326, -0.1420499086380005, -0.10820015519857407, 0.8466864228248596, 0.32379454374313354, -0.01938566565513611, 0.11507941037416458, 0.044453151524066925, -0.4029825031757355, 0.050415750592947006, -0.08800635486841202, 0.16772806644439697, -0.23947694897651672, 0.48185020685195923, 0.5767490267753601, -0.26481249928474426, -0.20904555916786194, -0.05343249440193176, -0.048392705619335175, -0.4015708267688751, -0.3361567258834839, -0.5111746191978455, -0.5366116166114807, -0.13198792934417725, -0.3846823573112488, 0.31648412346839905, -0.0032060928642749786, -0.29229485988616943, -0.09720925241708755, -0.06859931349754333, 0.19691580533981323, 0.016476796939969063, -0.01930544339120388, 0.24450038373470306, 0.04555357247591019, 0.4545852541923523, 0.11141092330217361, -0.18438871204853058, -0.16021357476711273, -0.33974701166152954, 0.04193646460771561, 0.2640758454799652, -0.29788699746131897, 0.5566765069961548, -0.3773477077484131, -0.22199873626232147, 0.28579261898994446, -0.42735520005226135, 0.08365171402692795, -0.19333486258983612, 0.24109873175621033, 0.07133221626281738, 0.48297807574272156, 0.18678335845470428, -0.28388169407844543, 0.3910398483276367, 0.2807016372680664, 0.34652799367904663, 0.19315385818481445, 0.47332268953323364, -0.5294322967529297, -0.4363267421722412, 0.2912137508392334, -0.18340256810188293, -0.2474515438079834, 0.09342405200004578, -0.1677101105451584, 0.036666031926870346, -0.07053842395544052, 0.2032748907804489, -0.1273418813943863, -0.4453107416629791, 0.5713388919830322, 0.03746343404054642, -0.5449848175048828, -0.2761533260345459, 0.05400731787085533, 0.027096480131149292, -0.176879420876503, 0.10516431927680969, 0.1445942223072052, -0.18665671348571777, -0.5050262212753296, 0.034960150718688965, -0.26086845993995667, -0.36665475368499756, -0.22926726937294006, -0.13113097846508026, 0.31731751561164856, 0.1987745612859726, 0.20186804234981537, -0.7656869888305664, 0.3622206151485443, -0.13507190346717834, 0.06552799791097641, -0.2674061059951782, 0.13437721133232117, 0.003078881883993745, -0.23013514280319214, -0.45291805267333984, 0.04241473600268364, 0.08640623837709427, -0.2623801827430725, -0.09108105301856995, -0.6890314817428589, 0.17035847902297974, -0.31171080470085144, 0.057347483932971954, -0.12723369896411896, 0.7626141309738159, 0.3137112259864807], [-0.17063486576080322, -0.1992328017950058, 0.2051870971918106, 0.004290191922336817, -0.03368513286113739, -0.010851515457034111, -0.277325838804245, 0.1438080072402954, -0.5012468099594116, 0.23215259611606598, -0.3472951054573059, -0.23565390706062317, -0.2908899784088135, 0.45993444323539734, -0.2958161532878876, 0.1205049529671669, -0.22903235256671906, 0.40118077397346497, -0.05838819220662117, 0.14062635600566864, 0.8476970791816711, 0.46031060814857483, 0.30418264865875244, 0.14195522665977478, 0.7195987701416016, 0.17254596948623657, -0.007508575916290283, 0.05742280185222626, 0.3409472107887268, -0.5557981729507446, 0.4967115819454193, 0.4771649241447449, -0.025439495220780373, -0.47435909509658813, 0.25009647011756897, -0.20050811767578125, 0.03991606459021568, -0.2102886140346527, -0.33095595240592957, -0.015847299247980118, -0.14482921361923218, -0.23851996660232544, -0.05691833049058914, 0.16632211208343506, -0.14466813206672668, -0.15887945890426636, 0.31912583112716675, 0.08634507656097412, -0.1786639243364334, 0.24744459986686707, -0.24777813255786896, 0.5477480888366699, 0.3528017997741699, -0.32726937532424927, 0.2506188452243805, -0.35964450240135193, -0.04325549677014351, -0.02894546277821064, -0.27677375078201294, 0.3452366590499878, 0.4854550063610077, 0.08713700622320175, 0.001634752843528986, 0.20514240860939026, -0.2565303146839142, 0.02636832371354103, -0.3256314992904663, 0.2948211133480072, 0.291900098323822, 0.4900008738040924, -0.3923303782939911, -0.3058471381664276, 0.20903809368610382, 0.2981640696525574, 0.35016366839408875, -0.18722496926784515, 0.503146767616272, 0.04999249055981636, -0.11899719387292862, -0.37289488315582275, -0.11547163873910904, 0.2504573464393616, -0.3365272283554077, 0.6908413171768188, 0.10950065404176712, 0.38812047243118286, 0.19380243122577667, -0.23911090195178986, -0.24007008969783783, 0.00022879597963765264, 0.4324454963207245, -0.39583665132522583, -0.2887873351573944, 0.23910248279571533, 0.321967214345932, 0.24326935410499573, 0.23603735864162445, 0.009582172147929668, -0.20077374577522278, -0.6128695607185364, 0.011560016311705112, -0.8404801487922668, -0.3632849156856537, 0.15539062023162842, 0.3329257369041443, 0.8579584360122681, -0.31315878033638, 0.2775134742259979, -0.06099842116236687, 0.0739816278219223, 0.01196538656949997, -0.04550779610872269, -0.1374986469745636, 0.34953829646110535, 0.04265203699469566, -0.004486510530114174, 0.10576488077640533, -0.28994208574295044, -0.17070907354354858, -0.4616768956184387, -0.22804054617881775, -0.43573564291000366, 0.15161719918251038, 0.027947982773184776, 0.3648211359977722, -0.38665977120399475, 0.18026527762413025, 0.18788179755210876], [0.0373305082321167, -0.24413259327411652, 0.2755154073238373, -0.012999053113162518, 0.01610623300075531, 0.3625032603740692, 0.016757916659116745, 0.43866628408432007, -0.5874300003051758, 0.020797399803996086, -0.36589542031288147, -0.4406125247478485, -0.4202497601509094, 0.052424490451812744, 0.2664133310317993, 0.29996320605278015, -0.14860384166240692, 0.12546156346797943, 0.09655575454235077, 0.060838066041469574, 0.46285730600357056, 0.7402653694152832, -0.13801734149456024, 0.10055017471313477, 0.39718297123908997, 0.25348132848739624, 0.10642041265964508, -0.038251541554927826, 0.20288370549678802, -0.5384342074394226, 0.08488163352012634, 0.3042125701904297, -0.5116872191429138, 0.026704926043748856, 0.18801981210708618, -0.4471416771411896, -0.09711295366287231, -0.0044760918244719505, -0.04009630158543587, -0.20697368681430817, 0.0114041268825531, -0.23152466118335724, 0.012092113494873047, -0.12707316875457764, -0.03371575474739075, 0.25483110547065735, -0.18708603084087372, -0.26179638504981995, -0.1537213772535324, 0.2231036275625229, 0.2845448851585388, 0.09783416241407394, 0.22523494064807892, -0.09812850505113602, 0.34587711095809937, -0.15259207785129547, -0.33994489908218384, 0.042939841747283936, -0.15055203437805176, 0.033297449350357056, 0.20243902504444122, -0.012212500907480717, -0.09059229493141174, 0.1428976207971573, 0.04325173795223236, 0.14168380200862885, -0.3621634542942047, -0.0451391376554966, 0.03306620568037033, 0.18928839266300201, -0.28681671619415283, 0.044387638568878174, 0.10983811318874359, -0.2536316514015198, 0.3691062331199646, -0.19593916833400726, 0.481134295463562, 0.08730093389749527, -0.03997192159295082, -0.4145078957080841, -0.08753904700279236, -0.22042520344257355, -0.13935814797878265, 0.43003493547439575, 0.17877615988254547, -0.22660501301288605, 0.32658758759498596, -0.11235298216342926, -0.22853389382362366, 0.12080811709165573, 0.21528835594654083, -0.38582029938697815, 0.018106743693351746, 0.17216375470161438, 0.2854325473308563, -0.07542766630649567, 0.063548244535923, 0.2117750495672226, -0.24191446602344513, -0.2991218566894531, -0.1092263013124466, -0.2659035325050354, 0.0035014052409678698, -0.11875829100608826, 0.19791124761104584, 0.6973329186439514, 0.04539655148983002, 0.21019217371940613, -0.39737826585769653, 0.07190217822790146, -0.2959758937358856, -0.05438067391514778, -0.03334769606590271, 0.12135598063468933, 0.05447646975517273, -0.13243785500526428, -0.36727315187454224, -0.4765593111515045, 0.22080472111701965, -0.21446780860424042, -0.3539457619190216, -0.5839682817459106, -0.045053184032440186, -0.2227201610803604, 0.29612576961517334, -0.43974781036376953, 0.34818026423454285, 0.2648603320121765], [-0.6261960864067078, 0.19145745038986206, -0.24032160639762878, -0.32560718059539795, 0.17031243443489075, -0.0891556590795517, 0.6417627930641174, 0.40404239296913147, -0.502050518989563, -0.5497555136680603, 0.21646246314048767, -0.14683179557323456, -0.2592220902442932, 0.12097818404436111, 0.1423603892326355, -0.154208704829216, -0.23375734686851501, 0.5133115649223328, -0.27186349034309387, 0.3132580518722534, 0.4308600425720215, 0.32814085483551025, -0.49680256843566895, 0.930598258972168, -0.6765831112861633, 0.1513768881559372, 0.6447016596794128, 0.3067604899406433, -0.03780170902609825, -0.6294902563095093, -0.1623019278049469, 0.2086218148469925, -0.46323779225349426, 0.11664935201406479, 0.49923941493034363, 0.6471787095069885, -0.15404970943927765, -0.07291577011346817, -0.3055167496204376, -0.4925001263618469, -0.023771004751324654, -0.34620511531829834, -0.2540794312953949, -0.22369559109210968, -0.26326438784599304, -0.46810382604599, -0.32578200101852417, -0.30918875336647034, -0.21546049416065216, -0.15237505733966827, 0.36104974150657654, -0.3900325894355774, 0.15935185551643372, 0.398709237575531, -0.15041598677635193, -0.32135462760925293, -0.052334871143102646, -0.1675073355436325, -0.26780712604522705, -0.8266617059707642, -0.24154545366764069, -0.6450852751731873, 0.056107714772224426, 0.13267433643341064, 0.5451152324676514, 0.05409404635429382, 0.24396280944347382, 0.7958471179008484, 0.6139506697654724, -0.0110364044085145, -0.04664408415555954, 0.027963511645793915, -0.00017767715326044708, -0.09406013786792755, 0.01595601998269558, -0.3161138892173767, 0.06842714548110962, -0.35962703824043274, -0.07019054144620895, 0.2414519339799881, -0.9691237211227417, -0.5112453699111938, 0.6738255023956299, 0.19187454879283905, 0.1599084585905075, -0.07018650323152542, -0.40655753016471863, -0.29902228713035583, -0.3026648163795471, 0.2611364424228668, -0.27828627824783325, -0.3967875838279724, 0.2878805100917816, 0.1640758514404297, -0.5190092325210571, 0.5374624729156494, 0.5128192901611328, 0.14766061305999756, -0.2592219412326813, 0.294876366853714, -0.584158718585968, -0.26515263319015503, -0.32669875025749207, -0.6613539457321167, 0.1934823989868164, 0.18733273446559906, 0.1960909217596054, 0.06463708728551865, -0.17628808319568634, 0.60166335105896, 0.34031596779823303, 0.25129473209381104, -0.6739828586578369, -0.0012725924607366323, -0.2215527445077896, 0.02739698626101017, -0.39395222067832947, 0.27717941999435425, -0.26851654052734375, -0.5606393218040466, -0.0863257572054863, -0.8767815232276917, 0.4524134695529938, -0.3718355596065521, -0.21326619386672974, 0.4354209899902344, -0.2917834520339966, 0.16141845285892487], [-0.2482399195432663, 0.7806908488273621, 0.48792511224746704, 0.17196901142597198, -0.5848994255065918, 0.5317271947860718, 0.2310648411512375, 0.372671514749527, 0.18750371038913727, 0.4267405867576599, -0.24661782383918762, -0.44382017850875854, -0.3757557272911072, 0.25710031390190125, 0.5271478891372681, 0.6127283573150635, -0.26928046345710754, -0.011765878647565842, 0.024082660675048828, 0.7473986744880676, 0.39399784803390503, 1.1935113668441772, -0.24544772505760193, -0.1390933245420456, 0.2185305953025818, -0.5145105123519897, -0.31616175174713135, 0.31403234601020813, 0.4674055874347687, 0.354316771030426, -0.16189415752887726, -0.1019522026181221, -0.5234844088554382, -0.09016001224517822, 0.44484955072402954, 0.14555858075618744, 0.12381578981876373, -0.04359643906354904, -0.4969577193260193, -0.6296098232269287, -0.18588992953300476, -0.29716914892196655, 0.3834308385848999, -0.19824601709842682, -0.05374901369214058, 0.35939204692840576, -0.08622217178344727, -0.5624168515205383, -0.3425697982311249, 0.031745847314596176, 0.4835326671600342, -0.06006339192390442, 0.2603791654109955, -0.31592804193496704, 0.27699780464172363, -0.8695167899131775, 0.16834615170955658, 0.6565495133399963, 0.44134393334388733, -0.15016208589076996, 0.8011198043823242, -0.1385287195444107, -0.20032644271850586, 0.1861969530582428, 0.44677913188934326, 0.3184581995010376, -0.22525696456432343, 0.43900635838508606, 0.13811032474040985, -0.3805277645587921, -0.5988260507583618, -0.006309527438133955, 0.17833343148231506, 0.31771084666252136, 0.44514283537864685, -0.6788198947906494, 0.5220288634300232, 0.03370237350463867, 0.13468606770038605, -0.2390041947364807, 0.005760527681559324, -0.4751840829849243, -0.8119208216667175, 0.9542949795722961, 0.0481116846203804, 0.1265711486339569, 0.1609649360179901, 0.017694341018795967, -0.7488492131233215, -0.31028637290000916, -0.01210564374923706, 0.15181151032447815, -0.2862975299358368, 0.7263999581336975, -0.21342645585536957, 0.28165343403816223, 0.1341763585805893, 0.45743900537490845, 0.08768755197525024, -0.3761430084705353, 0.27999749779701233, -0.6860756874084473, -0.10101102292537689, 0.6987972855567932, 0.15831716358661652, 0.6763035655021667, -0.31909000873565674, 0.0010433400748297572, -0.3270616829395294, -0.39977923035621643, -0.6107356548309326, 0.5068181157112122, -0.3175707757472992, 0.4380924701690674, -0.0506436824798584, 0.16925302147865295, 0.3033451437950134, 0.2252301722764969, 0.1544676125049591, -0.7478070259094238, 0.617743968963623, -0.6478663682937622, -0.1401103138923645, -0.20283566415309906, 0.46782994270324707, 0.58671635389328, 0.29454120993614197, 0.4006396532058716], [-0.2429826557636261, -0.5107530951499939, 0.2140888124704361, -0.06764991581439972, -0.2945074737071991, -0.10805284976959229, 0.19934538006782532, 0.5962194204330444, -0.485482394695282, -0.2357473522424698, -0.17540213465690613, -0.08433615416288376, -0.017895251512527466, 0.4628269672393799, 0.12279179692268372, 0.26931801438331604, -0.4912490248680115, 0.06616444140672684, 0.2667250633239746, 0.10876065492630005, 0.2642861008644104, 0.2864043414592743, -0.30870282649993896, 0.4918087422847748, 0.34882330894470215, -0.04112071171402931, 0.2041587382555008, 0.09313572198152542, 0.1846790313720703, -0.7369304299354553, -0.16290444135665894, 0.11266494542360306, -0.1366266906261444, -0.4115026295185089, 0.4369775354862213, -0.13263310492038727, 0.029316453263163567, -0.19788126647472382, 0.34339359402656555, -0.10746067017316818, -0.2183935046195984, -0.03391087055206299, 0.2518249750137329, -0.06658206135034561, 0.23173771798610687, 0.46154704689979553, -0.05755098536610603, -0.38596639037132263, 0.7800647020339966, -0.021044403314590454, -0.14979152381420135, 0.043411169201135635, 0.653195858001709, 0.04663923755288124, 0.13532131910324097, -0.2796967327594757, 0.09273521602153778, 0.2479950338602066, -0.029905961826443672, -0.1329856812953949, 0.27375757694244385, -0.4985172152519226, 0.08413273096084595, 0.6009129881858826, 0.38137972354888916, 0.17249520123004913, -0.23141685128211975, 0.2751934230327606, -0.1421249806880951, 0.28858351707458496, -0.1564285308122635, 0.07523330301046371, -0.36053866147994995, -0.019341669976711273, 0.02856721170246601, -0.3244992792606354, 0.0957241952419281, -0.21487103402614594, -0.4093440771102905, -0.04324399307370186, -0.4376581013202667, -0.5967139601707458, -0.1101650521159172, 0.5710790753364563, -0.021149974316358566, 0.27870088815689087, 0.1271853744983673, -0.23193445801734924, -0.622917890548706, 0.19692710041999817, 0.4616871178150177, -0.47544318437576294, -0.1622847020626068, 0.3429780900478363, 0.42492929100990295, 0.0070710210129618645, 0.4043288230895996, 0.10247771441936493, -0.02414468489587307, -0.4757634401321411, 0.12012509256601334, -0.48358142375946045, -0.04924086481332779, 0.49173298478126526, 0.33337366580963135, 0.7540225982666016, -0.1334628313779831, -0.058448780328035355, -0.298994243144989, -0.3023192882537842, -0.35437262058258057, -0.014585147611796856, 0.004425589460879564, 0.19149862229824066, -0.29128867387771606, -0.14285026490688324, 0.0769551545381546, -0.31588658690452576, 0.13382570445537567, -0.42339885234832764, 0.19064661860466003, -0.9274226427078247, -0.09777722507715225, -0.16650015115737915, 0.6792644262313843, 0.012795680202543736, 0.02115372195839882, 0.014653796330094337], [0.011730601079761982, 0.45291799306869507, -0.0027265017852187157, -0.040015723556280136, 0.3611944913864136, -0.2199307233095169, -0.0023114809300750494, 1.167070984840393, -0.3043175935745239, -0.4740794599056244, -0.6561645269393921, 0.4437190592288971, -0.2196611613035202, 0.4341442286968231, -0.2820834219455719, 0.2500120997428894, 0.7440584897994995, -0.011287719011306763, 0.8180786967277527, 0.5263903737068176, -0.1275986284017563, 0.10030121356248856, 0.152299165725708, 0.5159204006195068, 0.08491457253694534, 0.2248937487602234, 0.19872452318668365, -0.5323503017425537, 0.8656368851661682, -0.40307772159576416, -0.4030705392360687, 0.18934321403503418, 0.01333532389253378, -0.5408673882484436, 0.46040305495262146, -0.2253570556640625, 0.12093978375196457, -0.03870651498436928, -0.36452600359916687, -0.47651243209838867, 0.2987721264362335, 0.40474894642829895, 0.02268192358314991, 0.5197175741195679, 0.5164137482643127, 0.09514697641134262, -0.23610854148864746, 0.2912678122520447, -0.1724306344985962, -0.40657851099967957, 0.26401570439338684, -0.20128300786018372, 0.042844731360673904, -0.7714712023735046, -0.055606942623853683, 0.4321826696395874, -0.48386940360069275, -0.4108576774597168, 0.5024524927139282, 0.24958203732967377, 0.05794521048665047, 0.2694075405597687, 0.296445369720459, 0.5438422560691833, 0.2820030450820923, 0.00404887180775404, -0.24462702870368958, 0.31817662715911865, 0.8752507567405701, 0.12753549218177795, -0.6824390292167664, 0.43856140971183777, -0.4083407521247864, 0.10771799832582474, -0.13577382266521454, -0.2601415812969208, 0.04698719456791878, -0.18934135138988495, 0.3856608271598816, -0.31647777557373047, -0.1891023814678192, 0.009004419669508934, 0.2290850132703781, 0.05012210085988045, 0.1734725534915924, 0.9660419821739197, -0.4980783462524414, -0.18823295831680298, 0.050504159182310104, 0.2471698373556137, 0.1327260434627533, 0.19841724634170532, -1.0158590078353882, 0.3128047585487366, -0.3435216248035431, 0.05465245246887207, -0.07870980352163315, 0.41362711787223816, 0.010044831782579422, -0.6337786316871643, -0.022515172138810158, 0.3207353949546814, -0.0009772363118827343, -0.38200971484184265, -0.15874643623828888, 0.7613644003868103, -0.3034181594848633, 0.31349512934684753, -0.28569990396499634, 0.10582195222377777, -0.10643308609724045, 0.20325665175914764, -0.1049141213297844, 0.01289490144699812, 0.5575240850448608, 0.28687185049057007, -0.4685359001159668, -0.47902625799179077, -0.2813654839992523, -0.1938944011926651, 0.506301999092102, -0.9190720319747925, -0.7796429991722107, -0.4189678430557251, 0.5112034678459167, 0.3312159478664398, 0.3910251259803772, 0.25502556562423706], [-0.18357053399085999, 0.22657540440559387, -0.022075524553656578, 0.11990544945001602, 0.026315761730074883, -0.13886359333992004, -0.003099092748016119, 0.2684643268585205, -0.70307856798172, 0.09532741457223892, -0.24329768121242523, -0.4246818423271179, -0.16942903399467468, 0.23928776383399963, -0.18513260781764984, 0.3013174831867218, 0.007657197304069996, 0.18629513680934906, -0.029589153826236725, 0.16941380500793457, 0.322417676448822, 0.6103959083557129, 0.11594270169734955, 0.17213334143161774, 0.13277122378349304, -0.34978169202804565, -0.32554367184638977, -0.38951581716537476, 0.04891754686832428, -0.39843475818634033, 0.5644479393959045, 0.1795731484889984, -0.10046298801898956, -0.14414775371551514, 0.22449235618114471, -0.25412940979003906, -0.0005016525974497199, -0.10745552182197571, -0.2289319485425949, -0.5084170699119568, -0.021185260266065598, -0.15792158246040344, 0.26679855585098267, 0.38690003752708435, -0.16436652839183807, 0.13010174036026, -0.03769053518772125, -0.27555134892463684, -0.09413881599903107, -0.16227750480175018, 0.22935684025287628, 0.2978801727294922, 0.554450273513794, -0.24411538243293762, -0.16098079085350037, -0.3705049753189087, 0.02889871969819069, 0.14564143121242523, 0.2268335372209549, -0.31095966696739197, 0.39549338817596436, -0.2348157912492752, -0.036979030817747116, 0.08875174820423126, 0.23813830316066742, 0.04696935787796974, -0.4540603458881378, 0.2831604778766632, 0.3165855407714844, 0.1316954642534256, 0.04426521435379982, -0.29470399022102356, 0.20302000641822815, 0.18546859920024872, 0.42871081829071045, -0.20587316155433655, 0.24704165756702423, -0.055465828627347946, -0.333777517080307, -0.20366090536117554, -0.22922015190124512, -0.35829097032546997, -0.11946172267198563, 0.19334790110588074, 0.06765546649694443, 0.11277405917644501, -0.08397821336984634, -0.37771669030189514, -0.5544946193695068, 0.30976602435112, 0.04299187660217285, -0.3674280047416687, -0.2983037531375885, 0.44153913855552673, -0.015034685842692852, -0.01414788980036974, 0.37053921818733215, 0.29089394211769104, -0.34065285325050354, -0.25974246859550476, -0.13400539755821228, 0.04622337967157364, 0.00022657078807242215, -0.11125162243843079, -0.29391294717788696, 0.03330749645829201, 0.10757084935903549, -0.0892585813999176, -0.182138130068779, -0.14907892048358917, 0.06856757402420044, -0.10977159440517426, -0.10203041136264801, -0.0813969150185585, 0.10474854707717896, -0.23021988570690155, 0.06528608500957489, 0.05646603927016258, -0.18846364319324493, -0.41199448704719543, 0.25283384323120117, -0.4825892746448517, 0.03119901940226555, -0.22651830315589905, -0.04441676661372185, -0.0733252465724945, 0.2993571162223816, 0.037475232034921646], [-0.205547034740448, -0.20099185407161713, -0.05218829959630966, 0.12656676769256592, -0.04137885197997093, 0.06923521310091019, -0.02371562458574772, 0.0962655246257782, -0.44733285903930664, -0.23944337666034698, -0.2174445241689682, -0.013232986442744732, -0.05680685490369797, 0.31840085983276367, -0.09121891111135483, 0.1171877458691597, -0.16654536128044128, 0.23034916818141937, 0.1775786429643631, 0.07236900180578232, 0.2469356805086136, 0.5795383453369141, 0.18368695676326752, 0.2824164032936096, 0.3100462555885315, 0.06016981974244118, 0.029793700203299522, -0.13908284902572632, 0.3426452875137329, -0.35966652631759644, 0.31156352162361145, 0.24756993353366852, 0.11349766701459885, -0.19823014736175537, 0.3519417643547058, 0.1403244137763977, -0.17419809103012085, -0.08016208559274673, -0.06386357545852661, -0.14844973385334015, -0.07459461688995361, 0.010247326456010342, 0.06030912697315216, 0.2462884783744812, -0.2530158460140228, 0.047277841717004776, -0.3759397566318512, 0.04724900797009468, -0.03156865015625954, 0.1664564311504364, 0.1574394702911377, 0.23765890300273895, 0.40850409865379333, -0.03130367025732994, 0.04899076744914055, -0.3456933796405792, 0.032884933054447174, 0.08143609762191772, -0.06143507361412048, -0.0456991083920002, 0.1149028092622757, -0.23543831706047058, 0.03481924533843994, 0.09383820742368698, 0.1794317215681076, 0.16515827178955078, -0.27519774436950684, 0.3656625747680664, 0.10198231786489487, 0.2176714539527893, -0.16110844910144806, 0.19293101131916046, 0.30683189630508423, 0.21048370003700256, 0.3919626772403717, -0.11548925191164017, 0.24242226779460907, 0.05526585876941681, -0.20434904098510742, -0.25617244839668274, -0.301994264125824, -0.2323657125234604, -0.125406876206398, 0.34691184759140015, -0.06292519718408585, 0.09096908569335938, -0.08195196092128754, -0.20913071930408478, -0.15625165402889252, 0.04468538984656334, 0.17672492563724518, -0.48745912313461304, -0.18691377341747284, 0.25685155391693115, -0.2711447775363922, 0.05188260227441788, 0.0995434820652008, 0.04065326228737831, -0.15573060512542725, -0.44724929332733154, 0.02213001623749733, -0.28926220536231995, -0.10164778679609299, -0.036649104207754135, 0.19535468518733978, 0.24964721500873566, -0.036058343946933746, 0.21617235243320465, -0.21263615787029266, 0.10178303718566895, 0.19888053834438324, -0.11556800454854965, -0.22054101526737213, 0.18135686218738556, 0.000466163270175457, -0.30652233958244324, -0.04739635810256004, 0.03600728511810303, -0.2468036264181137, -0.29913485050201416, 0.1509305238723755, -0.6310006976127625, 0.1223379373550415, -0.2007957547903061, 0.040828436613082886, -0.018335629254579544, 0.03270478546619415, -0.07084092497825623], [-0.4536179304122925, -0.15854813158512115, -0.05442309379577637, 0.05573243275284767, -0.24945960938930511, -0.22687648236751556, 0.13619661331176758, 0.008652303367853165, -0.23586337268352509, -0.39564627408981323, -0.1781097948551178, -0.038753051310777664, -0.18036682903766632, 0.2954254150390625, -0.0695888102054596, 0.22478076815605164, -0.16395139694213867, 0.15460319817066193, 0.18157543241977692, 0.36088964343070984, 0.011013686656951904, 0.29262176156044006, 0.01717492938041687, 0.22140493988990784, 0.5560089349746704, 0.06264688819646835, -0.18712633848190308, -0.1830918788909912, 0.13678212463855743, -0.16167964041233063, 0.1357569545507431, -0.06051601096987724, 0.19351673126220703, -0.23969422280788422, -0.09174473583698273, 0.07137241959571838, 0.08721031993627548, -0.17630869150161743, -0.09643761068582535, -0.2010299265384674, 0.13959605991840363, 0.06187191233038902, 0.2004690319299698, 0.3494972586631775, 0.04758254811167717, 0.2361719012260437, -0.04386477544903755, -0.10780809074640274, 0.33049309253692627, 0.17722834646701813, 0.12875939905643463, 0.37002819776535034, 0.42326754331588745, -0.10324990004301071, 0.03620399907231331, -0.0074598416686058044, -0.01099055353552103, 0.013460351154208183, -0.08646593987941742, 0.1151248887181282, 0.020580707117915154, -0.2531905472278595, 0.0812431201338768, 0.2977982759475708, 0.30999985337257385, -0.03903885930776596, -0.2980521321296692, 0.2148568481206894, 0.2136402577161789, 0.2431638389825821, -0.02739180065691471, -0.05564656853675842, -0.21234163641929626, -0.03387151286005974, 0.045473210513591766, -0.21313117444515228, 0.20243023335933685, -0.012190325185656548, -0.2898423373699188, -0.29561373591423035, -0.3400518596172333, -0.5926787257194519, -0.1394900679588318, 0.19458122551441193, -0.048467736691236496, 0.218205064535141, 0.019573800265789032, -0.5618720054626465, -0.6032700538635254, 0.10299666970968246, 0.308310329914093, -0.48346251249313354, -0.24496985971927643, 0.3180766999721527, -0.14287982881069183, -0.000327871908666566, 0.3489914536476135, -0.0044832993298769, -0.2552645802497864, -0.6922292709350586, -0.06054377555847168, -0.25149238109588623, 0.1723950356245041, 0.12576048076152802, 0.21102984249591827, 0.14088720083236694, -0.4896693527698517, -0.14821358025074005, -0.02168208919465542, -0.04629397392272949, -0.20347650349140167, 0.053352680057287216, -0.022110043093562126, 0.013403893448412418, 0.20484638214111328, -0.33703142404556274, -0.10247614979743958, -0.3723212778568268, -0.16941529512405396, -0.35775503516197205, 0.1927330642938614, -0.5343437790870667, -0.1554691344499588, -0.04117463529109955, 0.07303500175476074, -0.01751788891851902, 0.20384952425956726, -0.002462849486619234], [-0.1914682537317276, 0.29289114475250244, 0.23395046591758728, -0.16215409338474274, -0.44117921590805054, -0.09339312463998795, 0.014497586525976658, -0.09021397680044174, -0.31940022110939026, 0.011882394552230835, -0.43321287631988525, 0.16623947024345398, -0.4749351143836975, 0.47565364837646484, -0.5160651803016663, 0.14299462735652924, 0.35020554065704346, -0.328523725271225, 0.08694496750831604, 0.23849759995937347, 0.0177595317363739, 0.6992089748382568, 0.35197940468788147, 0.3142361640930176, 0.37018632888793945, -0.28938794136047363, -0.18618933856487274, -0.025373894721269608, 0.24822618067264557, -0.11837748438119888, -0.2341991811990738, 0.06946936994791031, 0.548719048500061, -0.1424211710691452, -0.08817078918218613, 0.33851656317710876, -0.09168363362550735, 0.0010681336279958487, 0.061510395258665085, 0.21980266273021698, -0.3642464578151703, 0.16967208683490753, 0.46379023790359497, 0.10579843074083328, -0.3213714361190796, -0.43488502502441406, -0.37710902094841003, 0.3526756465435028, 0.1185092180967331, 0.2701113224029541, 0.26369306445121765, -0.35374003648757935, 0.5096659064292908, -0.10447347164154053, 0.2495705932378769, -0.36510995030403137, -0.017248066142201424, 0.14042425155639648, 0.3448447287082672, -0.2232147753238678, 0.3914960026741028, -0.4506533741950989, 0.050681039690971375, 0.2646413743495941, -0.30955296754837036, -0.3524230420589447, -0.01905990205705166, 0.4095367193222046, 0.22401168942451477, 0.0416354238986969, 0.04666578769683838, -0.2871963381767273, 0.013051637448370457, 0.4797194004058838, 0.6765061616897583, -0.28228890895843506, 0.4040534794330597, 0.11429040879011154, -0.06504707038402557, -0.14657144248485565, -0.23177309334278107, -0.3452766537666321, -0.05451682582497597, 0.06767479330301285, -0.11115394532680511, 0.034618817269802094, 0.07549183070659637, -0.093305304646492, -0.25632455945014954, 0.27099767327308655, 0.20375892519950867, -0.6327730417251587, -0.34596097469329834, 0.621265172958374, -0.25589028000831604, -0.0013519739732146263, 0.2511315941810608, -0.46506616473197937, 0.03447239473462105, -0.007917902432382107, -0.06426206231117249, -0.236146941781044, -0.3930652439594269, 0.11854405701160431, 0.09426721930503845, 0.06589962542057037, 0.010378703474998474, 0.47702664136886597, -0.25472769141197205, -0.07061364501714706, 0.043567731976509094, 0.07527836412191391, -0.12688937783241272, 0.2155560404062271, 0.05340020731091499, -0.126547172665596, -0.20757366716861725, 0.4186079502105713, -0.0693112164735794, -0.15139906108379364, 0.16324517130851746, -0.9682111144065857, 0.12825460731983185, -0.21842478215694427, -0.2605971097946167, 0.1817907989025116, 0.211035817861557, -0.011952769011259079], [-0.07571837306022644, 0.31478163599967957, 0.07800715416669846, -0.07063964009284973, -0.10634029656648636, -0.02705339342355728, -0.05245538800954819, 0.2526175081729889, -0.12505021691322327, -0.11547450721263885, -0.24930715560913086, -0.2291652411222458, -0.4025792181491852, 0.270772784948349, -0.12161878496408463, -0.021232454106211662, 0.2068963348865509, 0.23496882617473602, 0.19838538765907288, 0.16620413959026337, 0.20127876102924347, 0.5414109826087952, 0.1812964528799057, 0.13751426339149475, 0.29292169213294983, 0.0017445675330236554, -0.07830741256475449, -0.35752055048942566, 0.3839326500892639, -0.14138688147068024, 0.0008268622914329171, 0.2226746380329132, 0.04609858617186546, -0.3021668791770935, 0.2195369303226471, 0.05621090531349182, 0.2099824696779251, 0.08565038442611694, -0.22453641891479492, -0.3346928060054779, 0.033690016716718674, 0.09168224036693573, 0.08282268047332764, 0.20626439154148102, 0.08756382018327713, 0.09479493647813797, -0.18352201581001282, -0.010900859721004963, -0.23955786228179932, 0.002796838292852044, 0.2840428948402405, -0.09020859003067017, 0.19297409057617188, -0.2742537558078766, -0.054588403552770615, -0.06202428415417671, -0.1302141696214676, -0.10262618213891983, -0.03733116760849953, 0.02317063882946968, 0.27487054467201233, 0.023879922926425934, 0.009199146181344986, 0.16293933987617493, 0.23683956265449524, -0.061574142426252365, -0.3353332281112671, 0.19426952302455902, 0.5287366509437561, 0.0466022863984108, -0.11430424451828003, 0.10673170536756516, -0.09171737730503082, 0.1481572836637497, 0.6137624382972717, -0.1691112369298935, 0.31901833415031433, 0.12092140316963196, 0.05187366157770157, -0.1930551677942276, -0.1398763358592987, -0.25469404458999634, -0.10393573343753815, 0.22835978865623474, 0.21065226197242737, 0.08630756288766861, -0.145198255777359, -0.24509288370609283, -0.2843766212463379, -0.03803310915827751, 0.14595845341682434, -0.11923204362392426, -0.3188406229019165, 0.268252968788147, -0.07119112461805344, 0.11113595962524414, 0.07362908869981766, 0.23818428814411163, -0.1506464034318924, -0.2888873517513275, -0.06275366246700287, -0.04433019086718559, 0.012293441221117973, -0.019993770867586136, -0.06771783530712128, 0.40345752239227295, -0.18621329963207245, 0.15095899999141693, -0.1510257124900818, -0.022655533626675606, -0.21994605660438538, 0.12461204826831818, -0.269633024930954, 0.18805021047592163, 0.36869892477989197, -0.06967131048440933, -0.14319749176502228, -0.18187673389911652, -0.19698981940746307, -0.42407137155532837, 0.13044752180576324, -0.6556248664855957, -0.25243309140205383, -0.1850057691335678, 0.1295391321182251, 0.00176687550265342, 0.46317681670188904, 0.1192517802119255], [-0.22524526715278625, 0.35550016164779663, 0.023741627112030983, 0.13329960405826569, -0.2618037760257721, 0.33942675590515137, 0.3790813386440277, -0.008432012982666492, -0.0029832161962985992, 0.07927785068750381, -0.2912352383136749, -0.47893744707107544, -0.1672208607196808, 0.2168184220790863, 0.2855479121208191, 0.44894126057624817, -0.066475510597229, 0.4532930254936218, -0.19588418304920197, 0.11290992796421051, 0.4716171622276306, 0.5293074250221252, 0.09682321548461914, 0.01824687048792839, 0.2059701383113861, -0.42619097232818604, -0.14831025898456573, -0.015482619404792786, 0.16110950708389282, 0.2626538574695587, 0.11261308938264847, 0.1514880210161209, -0.5450138449668884, -0.3406856656074524, 0.0540495291352272, 0.2200571894645691, 0.14802205562591553, -0.11743292957544327, -0.1725289523601532, -0.8193185329437256, -0.2640567719936371, -0.26203152537345886, 0.17307893931865692, 0.07995931059122086, 0.08785593509674072, 0.16255894303321838, 0.0015459003625437617, -0.01918722130358219, -0.19709983468055725, -0.11574970185756683, 0.6933949589729309, -0.023176394402980804, 0.17681941390037537, -0.19973286986351013, -0.05090196430683136, -0.20914103090763092, -0.09006761014461517, 0.11531458050012589, 0.28897225856781006, 0.09500358253717422, 0.8134396076202393, -0.03376542031764984, -0.2566213607788086, 0.20615796744823456, 0.05664140731096268, 0.07305888086557388, -0.14835728704929352, 0.14271657168865204, 0.07666801661252975, 0.22996194660663605, 0.10507631301879883, 0.04508120194077492, 0.36195266246795654, 0.11838839948177338, 0.40512824058532715, -0.11039772629737854, 0.6131350994110107, -0.07350695878267288, 0.05440504848957062, -0.03047577291727066, -0.06008429452776909, -0.2808789014816284, -0.3225520849227905, 0.16638796031475067, 0.25249776244163513, -0.1782495379447937, -0.1389799565076828, -0.04664837196469307, -0.5509212017059326, 0.21124836802482605, -0.200489804148674, 0.013431351631879807, -0.2759483754634857, 0.1794673502445221, -0.20487985014915466, -0.06335381418466568, 0.10991425067186356, 0.3745054006576538, -0.18170121312141418, -0.3273901343345642, 0.0034251678735017776, -0.290859192609787, -0.1789606511592865, 0.22510023415088654, 0.13558584451675415, 0.3632824420928955, -0.17025654017925262, 0.17514091730117798, -0.3240760862827301, 0.0840739905834198, -0.33990678191185, 0.11751049011945724, -0.20942364633083344, 0.4338303804397583, 0.0011930465698242188, -0.15120507776737213, -0.0028642734978348017, -0.06820999085903168, 0.18607424199581146, -0.361663818359375, 0.09955230355262756, -0.5897238850593567, 0.13097944855690002, -0.1860877126455307, 0.19269971549510956, -0.12656548619270325, 0.525037944316864, 0.20493081212043762], [-0.34289807081222534, 0.12541204690933228, -0.3698508143424988, -0.14229457080364227, -0.08790814876556396, 0.38522839546203613, -0.32391393184661865, 0.09995954483747482, -0.2063351273536682, -0.001384517177939415, -0.2532687187194824, -0.5237790942192078, -0.39287278056144714, 0.35025647282600403, 0.14966033399105072, 0.012082641012966633, 0.054507989436388016, 0.17184074223041534, 0.17403090000152588, 0.30231717228889465, 0.03563068062067032, 0.5673046708106995, 0.07940814644098282, 0.674367368221283, 0.03445765748620033, 0.04245319217443466, 0.10292886942625046, -0.5949657559394836, -0.0858994796872139, 0.09578985720872879, -0.02136661671102047, -0.01738021709024906, 0.07440804690122604, -0.00523817352950573, -0.11123687028884888, -0.12769103050231934, -0.06063684821128845, 0.08279085159301758, -0.03620632737874985, -0.11569201946258545, -0.06907396763563156, 0.2573215961456299, 0.27828818559646606, -0.050741322338581085, -0.026410311460494995, 0.1717437207698822, -0.3245849609375, -0.24297824501991272, -0.09540018439292908, 0.04129771515727043, 0.4322188198566437, -0.040727294981479645, 0.39908909797668457, 0.03446603938937187, -0.3422214090824127, -0.12132871150970459, 0.0179348923265934, -0.18926452100276947, -0.15456493198871613, -0.051685333251953125, 0.18278588354587555, -0.17660972476005554, 0.17710310220718384, 0.2419295758008957, 0.11722695827484131, -0.09181378036737442, -0.5534928441047668, 0.40873467922210693, 0.28685930371284485, 0.14711126685142517, 0.13452266156673431, 0.11472824215888977, -0.02535131573677063, 0.11476241797208786, 0.46743136644363403, -0.009475785307586193, 0.57859206199646, 0.0915333479642868, 0.1309252232313156, -0.3155196011066437, -0.18911653757095337, -0.3337899446487427, -0.33646610379219055, 0.03984574228525162, 0.35521459579467773, -0.22961945831775665, 0.03052378073334694, -0.3881029486656189, -0.3058355450630188, -0.08877710998058319, 0.2934911847114563, -0.5850521326065063, -0.0852198675274849, 0.1437128782272339, -0.15220099687576294, -0.17752324044704437, 0.12207009643316269, -0.10122188180685043, -0.033548545092344284, -0.014675166457891464, 0.04935775324702263, -0.030278194695711136, 0.4467489719390869, 0.17806476354599, -0.06029577553272247, 0.018418053165078163, 0.22890572249889374, 0.1455278992652893, -0.19792985916137695, -0.021496402099728584, 0.22763386368751526, 0.24107033014297485, -0.26280274987220764, -0.09519348293542862, 0.02023320645093918, -0.4330785870552063, -0.304532915353775, -0.41957369446754456, -0.22321093082427979, -0.4715981185436249, 0.14490815997123718, -0.2917643189430237, -0.34798747301101685, -0.33264628052711487, -0.0765315443277359, -0.05214717239141464, 0.10695954412221909, 0.025479864329099655], [-0.05005289986729622, -0.002852256176993251, -0.04695933684706688, -0.2014763057231903, -0.12653310596942902, 0.2013213187456131, -0.3314228951931, -0.11008095741271973, 0.19982460141181946, -0.3075161278247833, -0.5260010361671448, -0.007283948361873627, -0.1658312976360321, 0.5447013974189758, -0.08164334297180176, 0.1788204312324524, 0.029297608882188797, 0.026868311688303947, 0.2317529022693634, 0.09945789724588394, 0.19577796757221222, 0.49805891513824463, 0.19296984374523163, 0.297370582818985, 0.2435396909713745, -0.13151441514492035, 0.013375510461628437, -0.2892397940158844, 0.0728670135140419, 0.27886632084846497, 0.04864473640918732, 0.05037948116660118, 0.07484442740678787, -0.24878738820552826, -0.010519582778215408, 0.20204702019691467, -0.17915819585323334, -0.1046874150633812, -0.027446206659078598, -0.015127314254641533, -0.2957049012184143, 0.14894647896289825, -0.013721791096031666, 0.11627763509750366, -0.08837219327688217, -0.24288485944271088, -0.46406811475753784, 0.19819338619709015, -0.06822513788938522, -0.10435707867145538, 0.10862614214420319, 0.10081809759140015, 0.3777497410774231, -0.16917982697486877, -0.13190153241157532, -0.27432382106781006, -0.016762152314186096, -0.44071072340011597, 0.11433884501457214, -0.038997188210487366, 0.09989912062883377, -0.16205541789531708, -0.1290528029203415, 0.2656283378601074, 0.24998310208320618, -0.26400917768478394, 0.08723372220993042, 0.3717058598995209, 0.05484713986515999, -0.018007896840572357, -0.13413141667842865, 0.11838212609291077, 0.2921023964881897, 0.07091494649648666, 0.3199354410171509, -0.09900233894586563, -0.045234810560941696, 0.24068672955036163, -0.07693564146757126, -0.12017954885959625, -0.046541742980480194, -0.03382612764835358, -0.13350050151348114, -0.09207018464803696, 0.025815537199378014, 0.07400839030742645, 0.05425707623362541, 0.05673163756728172, 0.0131596140563488, 0.07525133341550827, 0.21878047287464142, -0.504743754863739, -0.5037788152694702, 0.7553720474243164, -0.18098466098308563, -0.199034184217453, 0.1786205917596817, -0.03947334364056587, 0.10367982089519501, -0.3188141882419586, -0.13737823069095612, 0.09670468419790268, -0.08274971693754196, 0.17949801683425903, 0.05142071470618248, 0.2713056802749634, -0.2104269415140152, -0.030046626925468445, -0.2172439694404602, 0.12463784217834473, 0.08574721962213516, 0.00962946005165577, -0.032896384596824646, 0.02736268751323223, -0.1552705317735672, -0.6993480920791626, -0.8718332648277283, -0.24027487635612488, -0.2614184617996216, 0.16095377504825592, -0.2557111978530884, -0.6503878235816956, 0.05302120000123978, -0.03312069922685623, -0.15079736709594727, -0.1804986298084259, 0.23571835458278656, -0.16906006634235382], [0.04279213771224022, 0.5250465869903564, -0.05655818060040474, -0.04150499776005745, -0.44636303186416626, 0.23297980427742004, 0.07223914563655853, 0.07838203012943268, -0.12729142606258392, -0.004803600255399942, -0.11896703392267227, -0.6772112250328064, -0.3081744313240051, 0.2733865976333618, -0.040482476353645325, 0.21683280169963837, -0.21836423873901367, 0.40664371848106384, -0.08989672362804413, 0.38005995750427246, 0.4321017563343048, 0.767119824886322, 0.198876291513443, 0.36594074964523315, -0.12688763439655304, -0.5727320909500122, -0.3535396158695221, -0.029109468683600426, -0.003617321839556098, 0.16327616572380066, 0.19394923746585846, 0.12634174525737762, 0.13338284194469452, -0.04948868602514267, 0.16553740203380585, 0.34662964940071106, -0.15218739211559296, -0.02494841441512108, -0.5176750421524048, -0.13694046437740326, -0.1684972047805786, -0.44668376445770264, 0.021213805302977562, -0.33613330125808716, -0.23695681989192963, -0.17364422976970673, -0.4328753650188446, -0.08787739276885986, -0.49009278416633606, -0.19060322642326355, 0.22876304388046265, -0.1460215151309967, 0.2582579255104065, -0.503528356552124, 0.12077982723712921, -0.2826519310474396, 0.016500715166330338, 0.20925848186016083, 0.3485368490219116, -0.4333266615867615, 0.22144004702568054, -0.5030955672264099, -0.3606922924518585, 0.4779186546802521, -0.011584688909351826, 0.13482804596424103, 0.04509541019797325, 0.3188704252243042, 0.23083576560020447, -0.14381499588489532, 0.24648411571979523, -0.25043025612831116, 0.08739720284938812, 0.3309336006641388, 0.39239829778671265, -0.33222007751464844, 0.43037939071655273, -0.39000874757766724, -0.16163618862628937, -0.07536772638559341, -0.2041832059621811, -0.3181963860988617, -0.08610956370830536, 0.10023410618305206, 0.11358961462974548, -0.10318072885274887, -0.10381686687469482, -0.13576896488666534, -0.14764516055583954, 0.040365684777498245, 0.04598530754446983, -0.788000226020813, -0.33071357011795044, 0.46620795130729675, -0.5491527915000916, 0.19971252977848053, 0.19727854430675507, 0.1290281116962433, 0.16036880016326904, -0.19923682510852814, -0.03405803069472313, -0.37564611434936523, -0.1930067241191864, 0.3183022737503052, -0.19071544706821442, 0.046931564807891846, 0.099441297352314, -0.12915083765983582, -0.14711536467075348, -0.018751442432403564, -0.10001712292432785, 0.08547231554985046, -0.19555741548538208, 0.14119206368923187, 0.19281719624996185, -0.40947574377059937, -0.303498238325119, 0.34713810682296753, -0.0713421180844307, -0.5693675875663757, 0.4150516986846924, -0.3773687779903412, 0.17362231016159058, -0.2914532423019409, -0.2941383719444275, 0.33074623346328735, 0.24948528409004211, 0.18995867669582367], [-0.15670259296894073, 0.20077627897262573, 0.2098551243543625, -0.21398144960403442, -0.18865624070167542, 0.29379943013191223, -0.24834896624088287, -0.049905817955732346, -0.12036978453397751, 0.3147302567958832, -0.6658278107643127, -0.004303735680878162, -0.3792293071746826, 0.3034546971321106, -0.46921417117118835, -0.04493802785873413, -0.052913907915353775, 0.3493695855140686, -0.04501534253358841, 0.10002991557121277, 0.2801646292209625, 0.11634741723537445, 0.3652230501174927, 0.1265164464712143, 0.17315614223480225, 0.16235852241516113, 0.10507051646709442, -0.3382455110549927, -0.11403621733188629, 0.14499109983444214, 0.1518879383802414, 0.25427117943763733, -0.007161313202232122, -0.2179909646511078, -0.15745101869106293, -0.0848783403635025, 0.04242856055498123, -0.07375731319189072, -0.47743406891822815, -0.29251256585121155, -0.052057355642318726, 0.13452081382274628, -0.08581104874610901, 0.08217635005712509, -0.07629682123661041, 0.20682388544082642, -0.3974921703338623, 0.011555389501154423, -0.3665011525154114, -0.12398013472557068, 0.2747098207473755, 0.5461633205413818, -0.0691782757639885, -0.35379213094711304, -0.14892667531967163, -0.4012035131454468, -0.05291726440191269, -0.6852200627326965, 0.010593552142381668, 0.21717560291290283, 0.3625195622444153, -0.044188812375068665, 0.07206812500953674, 0.21207579970359802, 0.07916829735040665, -0.00678223604336381, -0.3199962377548218, 0.37056517601013184, 0.15506868064403534, 0.02667556144297123, -0.10478166490793228, 0.06298424303531647, -0.0006487735663540661, -0.2511616051197052, -0.0009245761320926249, 0.0716017484664917, 0.09201350063085556, 0.5090494751930237, 0.23348169028759003, 0.08371113985776901, -0.10161136090755463, -0.0008523731958121061, -0.14824043214321136, -0.12939344346523285, 0.12187827378511429, -0.001663049217313528, 0.20273913443088531, -0.050906527787446976, -0.261227011680603, -0.05935802310705185, -0.018761709332466125, 0.020109446719288826, -0.4165022373199463, 0.5919292569160461, -0.12839028239250183, -0.34202322363853455, -0.0022758347913622856, -0.1609613597393036, -0.23366865515708923, -0.20101740956306458, -0.30868732929229736, -0.004697748925536871, 0.25365307927131653, -0.09462108463048935, -0.22595928609371185, 0.033327214419841766, -0.2291708141565323, -0.09725348651409149, -0.11708487570285797, -0.1286783069372177, 0.038813408464193344, 0.0073228743858635426, 0.10452677309513092, 0.22150211036205292, -0.11009207367897034, -0.4757252335548401, -0.25823843479156494, -0.2841581404209137, 0.06516247242689133, -0.1231122761964798, 0.04031330719590187, -0.29135096073150635, 0.242826446890831, -0.2379952073097229, 0.01841854490339756, -0.3031008243560791, 0.26142993569374084, 0.05757681652903557], [-0.2589004933834076, 0.20799314975738525, -0.017089148983359337, -0.05583180859684944, -0.03309174254536629, -0.022740479558706284, 0.4353633522987366, -0.10911703109741211, -0.5776673555374146, 0.06907348334789276, -0.8171811103820801, -0.10021228343248367, -0.2534002959728241, 0.34518539905548096, -0.2340504378080368, 0.003052807180210948, 0.20897802710533142, 0.3229319751262665, -0.10407575964927673, -0.037152715027332306, 0.5806602835655212, -0.2591244876384735, 0.3658043146133423, -0.026286596432328224, 0.16854222118854523, -0.03253454342484474, 0.05850588530302048, 0.059548571705818176, 0.13584458827972412, 0.16931971907615662, 0.04533887282013893, 0.3289528787136078, -0.2851044237613678, -0.0014997057151049376, -0.2257343977689743, -0.18908554315567017, -0.03417167067527771, -0.2266213744878769, -0.3773987889289856, -0.5109742283821106, -0.06361915171146393, -0.4457106590270996, 0.021178733557462692, 0.05351869389414787, -0.10558998584747314, -0.40632322430610657, 0.11116330325603485, 0.44012346863746643, -0.18462343513965607, 0.09762731939554214, 0.3393802344799042, 0.4726276993751526, 0.0778113305568695, -0.04841243103146553, 0.3067728281021118, -0.10992171615362167, -0.43530717492103577, -0.07018107920885086, 0.11122464388608932, 0.0629400983452797, 0.5867023468017578, -0.18088066577911377, -0.142242431640625, -0.06124736741185188, 0.08819323778152466, -0.08331955969333649, -0.5155383944511414, 0.10190939903259277, -0.04754151031374931, 0.313520610332489, -0.10498663783073425, -0.3844465911388397, 0.26774850487709045, -0.14861169457435608, 0.04096147045493126, 0.02991877868771553, 0.18269570171833038, -0.29608041048049927, -0.2546268105506897, 0.12278223782777786, 0.0453248955309391, -0.4957365393638611, 0.10895661264657974, -0.24440522491931915, -0.14671292901039124, 0.4101390242576599, -0.11194761842489243, 0.053506795316934586, -0.44808435440063477, 0.16140860319137573, 0.06383261829614639, 0.06165311112999916, -0.4166778028011322, 0.35106438398361206, 0.015075433067977428, -0.5100081562995911, 0.05565932020545006, 0.10443246364593506, -0.36866527795791626, -0.6217989325523376, -0.44467422366142273, -0.3419961929321289, -0.029739681631326675, -0.2742500901222229, -0.09985671192407608, 0.30868175625801086, -0.05851646140217781, -0.04295144975185394, -0.48203280568122864, 0.1330188363790512, -0.10374197363853455, -0.004726864397525787, -0.2807984948158264, 0.09565666317939758, -0.05935961380600929, -0.16755330562591553, -0.13119098544120789, -0.2810458540916443, 0.05763455480337143, -0.1078639104962349, -0.16009525954723358, -0.6583023071289062, 0.20654161274433136, 0.05443926155567169, -0.18764349818229675, -0.2768673300743103, 0.520888090133667, 0.4036106467247009], [-0.07385312020778656, -0.08279764652252197, -0.40414097905158997, -0.32316237688064575, -0.2385120838880539, -0.09121722728013992, -0.20297594368457794, -0.04373041167855263, -0.20306815207004547, 0.14526250958442688, 0.011957322247326374, -0.11725564301013947, -0.05380577966570854, 0.500311017036438, -0.2340964376926422, -0.1383742094039917, 0.21369247138500214, 0.16788224875926971, 0.341316819190979, 0.32236531376838684, 0.21241919696331024, 0.17414960265159607, 0.12175548821687698, 0.273013174533844, 0.11381077021360397, -0.23456016182899475, 0.1828690618276596, -0.2730017304420471, -0.024282682687044144, 0.004050601273775101, 0.1365726888179779, 0.12873631715774536, 0.052358053624629974, -0.33700740337371826, 0.1743549257516861, 0.07140334695577621, 0.0219827052205801, -0.0752471387386322, -0.0013772472739219666, -0.27206340432167053, -0.1543140411376953, 0.1969590187072754, 0.0028537160251289606, 0.1873973160982132, -0.13034431636333466, 0.13425832986831665, 0.032696593552827835, 0.0694274827837944, -0.12753821909427643, 0.053027112036943436, 0.2841125726699829, 0.23349568247795105, 0.09384326636791229, -0.15191225707530975, -0.24621526896953583, -0.17926228046417236, -0.2809167206287384, -0.574133038520813, -0.19097743928432465, -0.09661723673343658, 0.6505548357963562, -0.12513378262519836, 0.045317526906728745, 0.06917352974414825, 0.36296042799949646, -0.08496014028787613, -0.28846225142478943, 0.25989365577697754, 0.19309023022651672, -0.019833099097013474, 0.005500515922904015, -0.12956318259239197, 0.18629302084445953, -0.11291723698377609, 0.1714705526828766, -0.15430402755737305, 0.42095160484313965, 0.2599741816520691, -0.020914509892463684, -0.10983188450336456, -0.30702897906303406, -0.23927544057369232, -0.241679847240448, 0.13014116883277893, 0.13161839544773102, 0.03253690153360367, 0.06387805193662643, -0.2846248745918274, -0.7004909515380859, 0.14929653704166412, 0.148906409740448, -0.25083792209625244, -0.349728524684906, 0.438211590051651, -0.03118753246963024, -0.07715991139411926, 0.2923472821712494, -0.01973138563334942, -0.21383759379386902, -0.09206186234951019, -0.11329953372478485, 0.10889755189418793, -0.20855633914470673, -0.1010952740907669, 0.08518698066473007, 0.09218738228082657, 0.15011820197105408, -0.24270795285701752, 0.09350071102380753, -0.1769823282957077, 0.26296764612197876, -0.06371158361434937, -0.28476089239120483, 0.1544472724199295, -0.1355130970478058, -0.2746240496635437, -0.11249208450317383, -0.2968601882457733, -0.05294680967926979, -0.32434532046318054, -0.040086403489112854, -0.3212328851222992, 0.26984044909477234, -0.2078186571598053, 0.03599727526307106, 0.024116259068250656, 0.2258828580379486, -0.13251331448554993], [-0.546919047832489, 0.05277300253510475, 0.1600610464811325, 0.131990447640419, 0.614422082901001, -0.658582866191864, 0.39536309242248535, -0.012884348630905151, -0.008260936476290226, 0.19574610888957977, -0.4831903576850891, -0.22887417674064636, -0.1160048246383667, -0.017387017607688904, -0.1256328523159027, 0.15464800596237183, -0.009955248795449734, 0.06706339120864868, 0.15705576539039612, -0.1828593909740448, 0.23945656418800354, 0.5520392656326294, 0.22369061410427094, 0.26958540081977844, 0.5421035289764404, -0.35772424936294556, -0.022796891629695892, -0.05648433789610863, -0.13286982476711273, 0.0622374527156353, 0.19657449424266815, -0.1887909322977066, -0.30938029289245605, -0.07769228518009186, 0.045940596610307693, -0.4830169081687927, -0.28565311431884766, -0.2663787305355072, 0.10848985612392426, -1.052925705909729, 0.3464239835739136, -0.1054745763540268, 0.06477499008178711, 0.36361032724380493, -0.22936798632144928, -0.1294848471879959, 0.876819372177124, 0.5278087854385376, -0.14030520617961884, -0.43336495757102966, 0.11679326742887497, 0.37169310450553894, 0.47928744554519653, -0.5477845668792725, 0.2919808328151703, -0.19965723156929016, -0.27430400252342224, -0.028699444606900215, -0.16584473848342896, -0.1603512167930603, 0.6388711929321289, 0.0652766227722168, -0.3011069595813751, -0.13859860599040985, 0.705380380153656, -0.3309878408908844, -0.17089536786079407, 0.1595696061849594, -0.2529335021972656, 0.11905914545059204, -0.15576833486557007, -0.24807068705558777, 0.3835665285587311, -0.04036659747362137, 0.7076181173324585, -0.3934858441352844, 0.012505772523581982, 0.03699620068073273, -0.4768250286579132, -0.22746916115283966, -0.33162596821784973, -0.5298685431480408, -0.1193215548992157, 0.29077696800231934, -0.00871356576681137, 0.13348044455051422, -0.47415876388549805, -0.07241763919591904, -0.2302435040473938, 0.34234926104545593, -0.5912831425666809, 0.01565743423998356, -0.395452082157135, 0.6925063133239746, -0.036932606250047684, 0.024750474840402603, 0.15494906902313232, 0.8583766222000122, -0.13155019283294678, -0.17293713986873627, -0.4804627299308777, 0.0914047434926033, -0.3001030683517456, -0.4079018235206604, -0.2762852907180786, -0.41377660632133484, 0.048549383878707886, 0.17418678104877472, -0.5967500805854797, 0.10862430185079575, 0.6525254845619202, -0.2921433448791504, -0.04182707890868187, -0.14746133983135223, -0.17221973836421967, 0.10274636745452881, -0.022670362144708633, 0.11978577822446823, -0.22486336529254913, 0.007344007026404142, 0.2787450850009918, -0.6623753905296326, 0.07649927586317062, 0.1964711993932724, -0.32386165857315063, -0.290060818195343, 0.3054773509502411, -0.5161570310592651], [-0.22174911201000214, -0.02259470336139202, -0.07530365139245987, -0.07049775868654251, -0.16378797590732574, 0.10479003936052322, -0.20175780355930328, 0.270668089389801, -0.22622762620449066, -0.02671746537089348, -0.183799147605896, -0.18905718624591827, -0.24212618172168732, 0.1933925300836563, 0.10383357107639313, 0.050203945487737656, -0.21455791592597961, 0.11275067180395126, 0.09229996055364609, -0.004716940224170685, 0.21379126608371735, 0.39143994450569153, 0.02165129780769348, 0.4114729166030884, 0.0028854706324636936, -0.048466723412275314, 0.1546339988708496, -0.09331861883401871, 0.06990790367126465, -0.10694489628076553, 0.1501377522945404, 0.24999234080314636, -0.040158916264772415, -0.38422828912734985, 0.001399608445353806, -0.11461599916219711, -0.5047587752342224, -0.07286238670349121, 0.20205122232437134, -0.1198522299528122, -0.26556870341300964, -0.09933467209339142, -0.036036569625139236, 0.08600111305713654, -0.09406526386737823, -0.26825109124183655, -0.2320968359708786, -0.08640313148498535, 0.04711534455418587, 0.11455909162759781, -0.1682964563369751, 0.1908172369003296, 0.11933545023202896, -0.3280050754547119, 0.0025918439496308565, -0.24388183653354645, 0.095188669860363, -0.10937962681055069, -0.15528199076652527, -0.07650946825742722, 0.015180890448391438, -0.15691065788269043, 0.022603418678045273, 0.327987939119339, 0.05354224517941475, -0.15080823004245758, -0.10056271404027939, 0.3909340500831604, -0.16270747780799866, 0.2739083170890808, -0.27209845185279846, 0.24420589208602905, 0.14026771485805511, 0.2348538488149643, 0.3189527988433838, -0.04764534533023834, 0.04700548201799393, 0.036871496587991714, -0.3115183413028717, -0.14337840676307678, -0.33476823568344116, -0.14984145760536194, -0.2053290605545044, -0.11963338404893875, -0.22881081700325012, -0.17987659573554993, 0.12723183631896973, 0.004085831809788942, -0.060210101306438446, 0.17354129254817963, 0.0655045136809349, -0.3922843933105469, -0.2580135464668274, 0.26381227374076843, -0.18784698843955994, 0.00976879894733429, 0.050096865743398666, -0.007129687815904617, -0.17909660935401917, -0.42788103222846985, -0.05172624811530113, -0.3373202085494995, 0.012502451427280903, 0.03757666423916817, 0.11486902087926865, 0.3912850022315979, -0.05498118698596954, 0.03111610747873783, -0.27049174904823303, 0.11603940278291702, 0.1316971629858017, -0.39922913908958435, 0.04733656719326973, 0.27288973331451416, 0.06984420865774155, -0.24513046443462372, -0.41988810896873474, -0.06244601309299469, -0.18999172747135162, -0.21254201233386993, -0.1833045780658722, -0.4874313473701477, 0.13139905035495758, 0.013472909107804298, 0.046763792634010315, -0.12841802835464478, 0.18588289618492126, -0.01184275932610035], [-0.14475691318511963, 0.14699916541576385, -0.6373981833457947, -0.20377284288406372, -0.09297618269920349, -0.06655867397785187, -0.2212839126586914, 0.37283992767333984, -0.22739559412002563, 0.32318705320358276, -0.15479901432991028, -0.16261664032936096, -0.17934787273406982, 0.29562821984291077, 0.09451527893543243, -0.060906197875738144, 0.20438283681869507, 0.2003171443939209, 0.38558512926101685, 0.32849425077438354, 0.0017765832599252462, 0.4906391203403473, 0.07125478237867355, 0.4280206859111786, 0.05367168039083481, -0.21503713726997375, -0.09013421833515167, -0.0802115797996521, -0.01283541601151228, -0.016727885231375694, 0.07053542137145996, 0.1669154018163681, 0.17441681027412415, -0.3197914659976959, -0.04784687981009483, 0.19504399597644806, -0.5071595907211304, -0.038287270814180374, -0.2649887502193451, -0.1653476506471634, 0.16139563918113708, -0.04521091654896736, 0.06390617787837982, 0.16741067171096802, -0.24755486845970154, 0.030798286199569702, -0.01925700530409813, -0.10634978115558624, -0.052161797881126404, 0.24688035249710083, 0.09995570778846741, 0.016001632437109947, 0.14187975227832794, -0.3635801076889038, -0.2557609975337982, -0.04206132888793945, -0.3830280303955078, -0.13310526311397552, -0.15068338811397552, -0.05552222579717636, 0.06954480707645416, -0.007977196015417576, 0.13506434857845306, 0.027578014880418777, -0.10286592692136765, -0.17509831488132477, -0.21775871515274048, 0.27661076188087463, -0.12533387541770935, 0.05855134502053261, -0.20410266518592834, 0.24128979444503784, -0.07198892533779144, -0.08670694380998611, 0.43849819898605347, 0.004185600206255913, 0.42628782987594604, 0.10528240352869034, -0.1663270890712738, -0.040838077664375305, -0.49284306168556213, -0.07338156551122665, -0.3374272882938385, -0.381055623292923, -0.140931636095047, -0.1970696896314621, 0.1517253965139389, 0.08298706263303757, -0.435295432806015, -0.0803968608379364, 0.10046868026256561, -0.5273579955101013, -0.2504262924194336, 0.2984130084514618, -0.03443878889083862, 0.1263432651758194, 0.0865309089422226, -0.12907768785953522, -0.36739400029182434, -0.2928783893585205, 0.10288336127996445, -0.2963894009590149, 0.1527249962091446, -0.18826428055763245, -0.022955430671572685, 0.10353049635887146, 0.22652992606163025, -0.22448673844337463, -0.15247884392738342, 0.17208784818649292, 0.005713979247957468, -0.41419050097465515, -0.3116278648376465, 0.10677788406610489, 0.25506919622421265, -0.2086738646030426, -0.440038800239563, -0.4666520357131958, -0.191757470369339, -0.3941478133201599, -0.007905030623078346, -0.31595245003700256, 0.0255562923848629, 0.044219981878995895, -0.3225804269313812, -0.10119970142841339, 0.23284383118152618, -0.0240220595151186], [-0.7487906813621521, 0.052661169320344925, 0.07858043909072876, 0.5365999937057495, -0.1486731618642807, -0.558072030544281, -0.29217639565467834, -0.4542474150657654, -0.18875832855701447, -0.6368942260742188, 0.34058064222335815, 0.0659155398607254, -0.13399840891361237, -0.25113561749458313, -0.22057925164699554, 0.655099093914032, -0.07891882210969925, 0.455023854970932, -0.7154516577720642, 0.30014756321907043, 0.19428947567939758, 0.8255175352096558, 0.33741050958633423, -0.06660985946655273, -0.1702498495578766, 0.2826748192310333, -0.5875264406204224, -0.6028455495834351, -0.32130056619644165, -0.016836553812026978, 0.7723493576049805, 0.08098964393138885, -0.02378058061003685, -0.914971649646759, -0.3341608941555023, -0.30172672867774963, -0.07280749827623367, -0.5165492296218872, 0.4127078354358673, -0.4380830228328705, -0.9111018180847168, -0.06340233981609344, -0.30984291434288025, 1.0154130458831787, 0.1773185282945633, 0.10903795808553696, 0.24984388053417206, 0.07080640643835068, 0.11336566507816315, 0.1934177428483963, 0.4208389222621918, 0.8793516755104065, -0.2132907509803772, -0.12105949223041534, 0.18355408310890198, 0.22456255555152893, -0.08735272288322449, -0.028435207903385162, 0.13143986463546753, 0.3695406913757324, -0.41528555750846863, 0.08613309264183044, 0.1896103173494339, -0.09825602173805237, 0.19523008167743683, 0.03525613620877266, -0.3635156750679016, -0.5657860636711121, 0.5002241730690002, 0.695512592792511, -0.00714871846139431, 0.6127379536628723, 0.36350899934768677, 0.7661628723144531, -0.04601086676120758, -0.030852964147925377, 0.26925593614578247, -0.41771554946899414, -0.7207626104354858, -0.26916787028312683, -0.2928151488304138, -0.5423750281333923, -0.24620378017425537, -0.36801761388778687, 0.18926911056041718, 0.08176752924919128, -0.3069988191127777, -0.21097305417060852, -0.16011092066764832, 0.04389883950352669, -0.6837277412414551, 0.30960148572921753, 0.049420807510614395, -0.41667065024375916, -0.5196821689605713, 0.9768397808074951, -0.10671163350343704, 0.55523282289505, 0.028811894357204437, -0.7589608430862427, 0.5047106146812439, -0.6418447494506836, 0.47101765871047974, -0.3076377511024475, 0.2597399652004242, -0.2381664663553238, -0.7899109721183777, -0.04166486859321594, 0.3109498918056488, 0.03201722353696823, -0.3912815451622009, -1.0897682905197144, -0.0852317214012146, 0.25770676136016846, 0.3380769193172455, -0.5003280639648438, -0.00331072136759758, 0.39523568749427795, -0.47375234961509705, -0.7725270390510559, 0.7154033184051514, -0.6384304761886597, -0.046020738780498505, -0.0949660986661911, 0.5431088209152222, -0.5341238975524902, -0.3580033481121063, 0.2906464636325836], [-0.2841605544090271, 0.25090429186820984, -0.21192559599876404, -0.2824866771697998, -0.057601198554039, 0.01063076313585043, -0.15088200569152832, 0.4002610445022583, -0.44900405406951904, -0.007571364287286997, -0.35919293761253357, -0.19121019542217255, -0.22448357939720154, 0.20533518493175507, -0.14899742603302002, 0.014502277597784996, -0.043282121419906616, 0.47786745429039, 0.10112867504358292, 0.483068585395813, 0.5324159264564514, 0.34574010968208313, 0.1810528188943863, 0.44092294573783875, -0.06489354372024536, 0.0742359608411789, -0.10033650696277618, -0.012554068118333817, -0.40986642241477966, -0.252279669046402, 0.24640056490898132, 0.4152129292488098, -0.10391142964363098, -0.039569273591041565, 0.051187340170145035, 0.29254385828971863, -0.1952114999294281, -0.059906747192144394, -0.46822983026504517, -0.502732515335083, -0.03009127266705036, -0.15143896639347076, -0.2969888150691986, 0.12248270958662033, -0.41420426964759827, -0.05098880082368851, -0.1961432248353958, -0.16885942220687866, -0.14839695394039154, -0.2186766117811203, 0.13055430352687836, 0.22811169922351837, -0.09984716773033142, -0.47177717089653015, -0.305673211812973, -0.46940553188323975, -0.1965520828962326, -0.43053117394447327, 0.11135075241327286, -0.4129440188407898, 0.052182503044605255, -0.21291343867778778, -0.10612290352582932, 0.0694979801774025, 0.22273685038089752, 0.23703329265117645, 0.1022736057639122, 0.347612589597702, -0.07712872326374054, -0.13037653267383575, -0.3623124957084656, 0.1296008825302124, -0.04612482339143753, -0.39220404624938965, 0.07276638597249985, -0.23708969354629517, 0.1868920773267746, -0.022887494415044785, -0.19464340806007385, 0.1428365260362625, -0.48085758090019226, 0.024368280544877052, 0.20663949847221375, -0.16738112270832062, -0.1807025820016861, 0.27916213870048523, -0.04417763650417328, 0.2702617943286896, -0.1571383774280548, 0.037721797823905945, -0.23304177820682526, -0.18378785252571106, -0.3038429319858551, 0.735392153263092, -0.18206341564655304, 0.2454863041639328, 0.19141216576099396, 0.14360490441322327, 0.0713295042514801, -0.12951864302158356, -0.321627676486969, -0.14721259474754333, 0.006557483226060867, -0.345721036195755, 0.07470093667507172, 0.21151845157146454, -0.19793692231178284, -0.11643730103969574, -0.09757158160209656, 0.393100768327713, 0.320108026266098, -0.26632681488990784, -0.21906150877475739, 0.22620642185211182, -0.04010936617851257, -0.1904940903186798, -0.3594827651977539, -0.1337151676416397, -0.26119011640548706, -0.21012797951698303, 0.04314810410141945, -0.3877800703048706, 0.3987768888473511, -0.07685624808073044, -0.16255244612693787, -0.3149665296077728, -0.006876933854073286, -0.025235310196876526], [-0.15777792036533356, -0.12906281650066376, -0.014765603467822075, -0.06817033886909485, -0.20094957947731018, -0.055256351828575134, -0.17264652252197266, 0.0644090473651886, -0.576683759689331, 0.042779650539159775, -0.2609471380710602, -0.10422886162996292, -0.26533082127571106, 0.16270799934864044, 0.028949983417987823, 0.18030479550361633, -0.027449997141957283, 0.3133147656917572, 0.2573726773262024, 0.162220761179924, 0.34901002049446106, 0.19526903331279755, 0.15316236019134521, 0.09474208950996399, 0.26248547434806824, -0.03419527783989906, -0.14992187917232513, -0.12148045003414154, -0.1413603574037552, -0.11220472306013107, -0.03065420314669609, 0.31483331322669983, -0.23219119012355804, -0.21577420830726624, 0.08657146990299225, -0.0490758903324604, -0.0034904186613857746, 0.18856126070022583, 0.18095426261425018, -0.3683135211467743, -0.056509602814912796, -0.12441759556531906, 0.11669488251209259, 0.06981799751520157, -0.0545971617102623, 0.22072060406208038, -0.07248101383447647, 0.06341084092855453, 0.05492231249809265, 0.08212806284427643, 0.42323678731918335, 0.10143423825502396, -0.0310013759881258, -0.24511073529720306, 0.09969409555196762, -0.060480229556560516, -0.1190115436911583, -0.1190919280052185, 0.1775442212820053, 0.061862070113420486, 0.3584597408771515, -0.13661253452301025, -0.00453078281134367, 0.07536321878433228, 0.1733003407716751, -0.023932170122861862, -0.3403254747390747, 0.1771080642938614, -0.11913444101810455, -0.027418863028287888, 0.014143063686788082, 0.0026759447064250708, 0.11681892722845078, -0.400913268327713, 0.25111129879951477, -0.1867045760154724, 0.34314486384391785, 0.3401468098163605, -0.13092055916786194, -0.18993987143039703, -0.15818658471107483, -0.20817862451076508, 0.10278649628162384, -0.05060703307390213, -0.13876433670520782, -0.10252828150987625, -0.04786115884780884, 0.21222087740898132, -0.29118266701698303, 0.34971165657043457, -0.07906918227672577, -0.03230799362063408, -0.23770716786384583, 0.4155220687389374, -0.028245067223906517, -0.1236954852938652, 0.06533177942037582, 0.1256527602672577, -0.14191636443138123, -0.1519966423511505, -0.34709954261779785, -0.042976438999176025, -0.2618772089481354, 0.05289158225059509, 0.15052103996276855, 0.2658136487007141, -0.08310028910636902, 0.12093058973550797, -0.16674531996250153, 0.07763520628213882, -0.05315447598695755, -0.3376818895339966, 0.12544815242290497, 0.21790644526481628, 0.013551046140491962, -0.2938178777694702, 0.05668003857135773, -0.18754597008228302, 0.24908952414989471, -0.022805411368608475, -0.16655050218105316, -0.47025054693222046, 0.15149402618408203, -0.1422901153564453, 0.3425777554512024, -0.3506071865558624, 0.29732728004455566, -0.0008781148353591561], [-0.1505420207977295, 0.10386495292186737, -0.28783756494522095, -0.20446336269378662, -0.2095106542110443, -0.0018328119767829776, -0.41584575176239014, 0.09609784930944443, -0.7400889992713928, 0.10301432013511658, -0.08098874986171722, -0.14400044083595276, -0.06844981759786606, 0.31041476130485535, -0.20713289082050323, -0.15959088504314423, -0.20666813850402832, 0.45660239458084106, 0.4398846924304962, 0.24063712358474731, 0.10832982510328293, 0.04036005958914757, 0.026847083121538162, 0.16481202840805054, -0.10268574208021164, -0.23607830703258514, -0.2785643935203552, -0.4448483884334564, -0.10783068835735321, -0.18129028379917145, 0.3241736590862274, 0.3541184365749359, 0.0006680716178379953, -0.3298966586589813, -0.027425149455666542, -0.06982994079589844, -0.2348340004682541, 0.02433260902762413, -0.2463391274213791, -0.3597806394100189, 0.08800347149372101, -0.22321797907352448, -0.04498621076345444, 0.14558954536914825, -0.18911822140216827, -0.13962936401367188, -0.1382705718278885, 0.09065724164247513, -0.10762334614992142, -0.12844622135162354, 0.22787481546401978, 0.15291541814804077, -0.14976418018341064, -0.38777586817741394, -0.30022528767585754, -0.02960411086678505, -0.2959247827529907, -0.33716484904289246, 0.1710835099220276, 0.018165379762649536, 0.139345183968544, -0.2518499195575714, 0.042256854474544525, 0.16543443500995636, -0.10379933565855026, -0.012017294764518738, -0.3237534165382385, 0.24367308616638184, 0.03575383126735687, 0.04288766533136368, 0.0863577276468277, -0.10354804247617722, 0.2357722818851471, -0.15763214230537415, 0.116397924721241, -0.13714854419231415, 0.04628613591194153, 0.10654328763484955, -0.21869058907032013, -0.22195225954055786, -0.2624661922454834, -0.16006909310817719, -0.13031478226184845, -0.30177628993988037, 0.0325617752969265, -0.4244927763938904, 0.19545193016529083, -0.0017478617373853922, -0.16793248057365417, 0.3739696145057678, -0.07834338396787643, -0.11084632575511932, -0.34330448508262634, 0.4395785331726074, -0.2844209671020508, 0.07115566730499268, 0.11055956780910492, 0.004946098197251558, -0.0828547477722168, -0.2368459850549698, 0.009068668819963932, -0.04459255933761597, 0.1284458041191101, -0.04869692772626877, -0.13279253244400024, 0.12232259660959244, 0.07252144813537598, -0.32897087931632996, -0.1393938660621643, 0.09518527239561081, 0.13273584842681885, -0.2835218012332916, 0.14118711650371552, 0.11442670226097107, 0.1673714965581894, -0.5193451642990112, -0.0017515529179945588, 0.147861048579216, -0.013959039002656937, -0.17014901340007782, 0.10269957780838013, -0.20289967954158783, 0.015525260008871555, -0.2652164101600647, 0.2474721521139145, 0.05638550966978073, 0.41558313369750977, -0.06868800520896912], [-0.16848625242710114, 0.4475801885128021, -0.004911019466817379, -0.13644073903560638, 0.1382993459701538, -0.004243162460625172, -0.036554206162691116, 0.3483336269855499, -0.15335559844970703, 0.05185159295797348, -0.35539504885673523, -0.10297472029924393, -0.1855771541595459, 0.26243582367897034, -0.12963835895061493, 0.17662681639194489, 0.4894222617149353, 0.04245505854487419, 0.40289878845214844, 0.45724043250083923, 0.22877730429172516, 0.1638137251138687, 0.07729218155145645, 0.32912594079971313, 0.19520656764507294, 0.12704883515834808, -0.10773797333240509, -0.18634355068206787, 0.21141280233860016, 0.035518307238817215, -0.20952005684375763, 0.019354630261659622, -0.01579129882156849, -0.33465775847435, -0.06788253784179688, -0.03160944581031799, 0.11732278019189835, 0.062466733157634735, -0.29145804047584534, -0.31725239753723145, 0.08672228455543518, 0.16578811407089233, 0.2292058765888214, 0.06695297360420227, -0.09892460703849792, 0.022978609427809715, -0.05254092067480087, 0.11362439393997192, -0.20483067631721497, -0.04873083531856537, 0.5389221906661987, -0.003386635798960924, 0.20586884021759033, -0.3944298326969147, -0.07191503047943115, 0.03201720491051674, -0.056570809334516525, -0.01973198540508747, 0.27607646584510803, 0.02560833841562271, 0.4335312247276306, -0.003450091229751706, 0.10579393059015274, 0.13108597695827484, 0.1546957790851593, -0.18748268485069275, -0.5111513733863831, 0.3548140227794647, 0.3952208161354065, -0.11385051161050797, -0.1642673760652542, 0.04548792168498039, -0.18014363944530487, -0.033079974353313446, 0.137503981590271, -0.2563604414463043, 0.20477451384067535, -0.0056131621822714806, 0.12849873304367065, -0.06918377429246902, -0.0942014679312706, -0.13186542689800262, -0.08039502799510956, 0.14405788481235504, -0.007911951281130314, 0.2249140590429306, -0.1552358716726303, -0.0703052431344986, -0.3361063003540039, 0.15673765540122986, 0.0870525911450386, 0.048151612281799316, -0.36158379912376404, 0.46415576338768005, -0.16279266774654388, -0.08421671390533447, 0.09647860378026962, 0.009016194380819798, -0.0460936464369297, -0.31816160678863525, -0.11393977701663971, 0.2524573504924774, -0.001839111209847033, -0.06856020539999008, -0.11829506605863571, 0.32553717494010925, -0.15959472954273224, 0.1400844305753708, -0.13372261822223663, -0.02620645798742771, -0.12045243382453918, 0.3335670828819275, -0.027826370671391487, 0.07338301092386246, 0.31882384419441223, -0.00246653170324862, -0.32226133346557617, -0.3442677855491638, -0.022272709757089615, -0.252524733543396, 0.07309215515851974, -0.5847779512405396, -0.3580728769302368, -0.2410050332546234, 0.16689281165599823, 0.10360387712717056, 0.5190426111221313, 0.23028163611888885], [0.013006256893277168, -0.08334247022867203, 0.0337240993976593, 0.010694896802306175, -0.12598779797554016, 0.20105215907096863, -0.21819588541984558, 0.17578594386577606, -0.20409180223941803, 0.15498311817646027, -0.22873646020889282, -0.2546190321445465, -0.11071930825710297, 0.08588762581348419, 0.12746725976467133, 0.0753776878118515, 0.02625993639230728, 0.4629930555820465, 0.22514180839061737, 0.30968910455703735, 0.2430790513753891, 0.27436763048171997, 0.16325098276138306, 0.1728300303220749, 0.03039591945707798, -0.21126924455165863, -0.22729229927062988, -0.10734168440103531, -0.1313859522342682, 0.04194790869951248, 0.02871912159025669, 0.20886455476284027, -0.16209375858306885, -0.2894448935985565, -0.07346515357494354, 0.35012340545654297, -0.09176892787218094, 0.11311285942792892, -0.343077689409256, -0.44088953733444214, 0.07070865482091904, -0.04150450974702835, -0.3524440824985504, 0.0017969453474506736, -0.3842243254184723, 0.1422443836927414, -0.07767928391695023, -0.03613905981183052, -0.12186107784509659, -0.000527574447914958, 0.13431741297245026, 0.05026937648653984, -0.23931854963302612, -0.69139564037323, -0.18136544525623322, -0.14602252840995789, 0.023759448900818825, -0.28131452202796936, 0.025534262880682945, -0.011310189962387085, 0.05948285758495331, -0.09643620252609253, 0.04849047213792801, 0.1827429234981537, -0.10487663745880127, 0.1703212410211563, -0.18696939945220947, 0.3019002676010132, -0.13011938333511353, -0.10898023098707199, -0.15763932466506958, 0.5032905340194702, -0.08755363523960114, -0.0997101441025734, 0.14127984642982483, -0.30502450466156006, 0.21665462851524353, -0.10249689221382141, -0.14521989226341248, -0.1291673630475998, -0.28994449973106384, 0.09266135841608047, -0.19027668237686157, -0.03611859679222107, -0.02939131297171116, -0.17590051889419556, -0.08023997396230698, 0.288397878408432, -0.019114771857857704, 0.012048718519508839, -0.1344471424818039, -0.1870490461587906, -0.24999132752418518, 0.7027484774589539, -0.25986963510513306, -0.15698620676994324, -0.1278417706489563, -0.028233712539076805, -0.021456247195601463, -0.03316834941506386, -0.3082202076911926, -0.32919564843177795, -0.07534626871347427, 0.11953511089086533, 0.20266573131084442, 0.14789247512817383, -0.11359627544879913, -0.056789327412843704, -0.12886770069599152, 0.20411023497581482, 0.10916925221681595, -0.29294225573539734, 0.10660857707262039, 0.5517139434814453, -0.04918621852993965, -0.18460485339164734, -0.3554753363132477, -0.0803154855966568, 0.06343735754489899, -0.13671375811100006, -0.1132122203707695, -0.1707736998796463, 0.04555997997522354, 0.06262687593698502, 0.020995229482650757, -0.34015506505966187, 0.3429129123687744, 0.1007968932390213], [-0.08510828763246536, 0.07983344793319702, 0.10520613938570023, -0.1975204348564148, 0.017173629254102707, -0.24106350541114807, -0.05534611642360687, -0.2935444712638855, -0.16550949215888977, 0.2173510193824768, -0.14988145232200623, -0.3341396450996399, -0.273759663105011, 0.16829171776771545, -0.19616198539733887, -0.32696518301963806, 0.2833873927593231, 0.581950843334198, 0.2677367031574249, 0.053910888731479645, 0.741566002368927, 0.24523086845874786, 0.5880743265151978, 0.304238498210907, 0.4273213744163513, 0.06569138914346695, -0.025960398837924004, 0.11505427956581116, -0.03480798751115799, -0.04281731694936752, -0.007373065687716007, 0.07031909376382828, -0.06038729473948479, -0.39418143033981323, -0.04050792008638382, 0.3748781085014343, -0.02423299290239811, 0.06780700385570526, -0.23838451504707336, -0.131183460354805, -0.07810047268867493, 0.03449034318327904, -0.15714585781097412, -0.019414281472563744, -0.2602354884147644, -0.13566860556602478, 0.39168673753738403, 0.19641144573688507, -0.3673439621925354, 0.2970121204853058, 0.2638458013534546, 0.1190849095582962, -0.10468948632478714, -0.5362407565116882, -0.1291067898273468, 0.1288495659828186, -0.39497873187065125, -0.462726354598999, -0.3063422739505768, 0.18877708911895752, 0.36581143736839294, 0.3273574709892273, 0.07392411679029465, -0.07946785539388657, -0.18383751809597015, -0.5293745994567871, -0.16380427777767181, 0.11442588269710541, 0.3108808994293213, 0.20310474932193756, -0.0740518867969513, 0.2512994110584259, -0.28234705328941345, -0.13987144827842712, 0.3217792809009552, -0.2129967212677002, 0.2311408370733261, -0.07398689538240433, -0.15071764588356018, -0.08272311091423035, -0.40887442231178284, 0.3465072214603424, -0.1865547001361847, 0.09713408350944519, -0.03565976768732071, -0.055752597749233246, -0.18410706520080566, 0.1760817915201187, 0.055519990622997284, -0.0843065083026886, 0.060362350195646286, -0.23529823124408722, -0.08327381312847137, 0.19823691248893738, 0.13728612661361694, 0.059589773416519165, -0.10665939003229141, 0.06406987458467484, -0.3468632400035858, -0.2150714248418808, 0.059370383620262146, -0.34272921085357666, -0.1556418091058731, -0.010144446045160294, 0.25435078144073486, 0.28035077452659607, 0.09191077202558517, 0.05855005234479904, -0.11178567260503769, 0.022717459127306938, -0.026430295780301094, 0.057667799293994904, -0.19560594856739044, 0.4193980395793915, 0.2793404459953308, -0.20687681436538696, -0.37606120109558105, -0.31733396649360657, -0.14546756446361542, -0.19034647941589355, -0.1609298288822174, -0.39090192317962646, 0.13468660414218903, 0.2273886352777481, 0.0017140532145276666, -0.3813204765319824, 0.15688692033290863, -0.07970698177814484], [-0.12127480655908585, 0.1114996075630188, -0.007083416450768709, -0.011604107916355133, -0.018215011805295944, 0.2576870024204254, -0.17168447375297546, 0.10616029798984528, -0.38831326365470886, 0.29200175404548645, -0.46953022480010986, -0.14218218624591827, -0.40529683232307434, 0.1476682722568512, -0.36851078271865845, 0.09188399463891983, 0.10563453286886215, 0.40491315722465515, 0.07448834180831909, 0.23220138251781464, 0.45194026827812195, 0.15477150678634644, 0.43495699763298035, 0.10367437452077866, 0.10476889461278915, 0.0028442961629480124, -0.11328839510679245, -0.3154410719871521, -0.03731859475374222, 0.002003739820793271, 0.35649001598358154, 0.38646605610847473, 0.15835480391979218, -0.032771240919828415, -0.1116049736738205, 0.13095960021018982, -0.34055760502815247, -0.01348877977579832, -0.24872034788131714, -0.30437538027763367, -0.11584857106208801, -0.021381519734859467, -0.07840904593467712, 0.11908192932605743, -0.29341650009155273, -0.0030796886421740055, -0.10069111734628677, 0.08807070553302765, -0.21088851988315582, 0.0220169797539711, 0.15402434766292572, 0.5615640878677368, 0.06770512461662292, -0.3706932067871094, -0.05206678807735443, -0.10176640748977661, -0.15135252475738525, 0.035962652415037155, 0.20709151029586792, 0.07753656804561615, 0.06281647086143494, -0.04662536084651947, -0.007411361206322908, -0.04773067310452461, -0.09908794611692429, 0.0035264596808701754, -0.3946380019187927, 0.24713772535324097, 0.1377880871295929, 0.12762697041034698, -0.04489327594637871, 0.08299127966165543, 0.24206824600696564, 0.07577501237392426, 0.1960136592388153, -0.06239505112171173, 0.2928152084350586, 0.13609258830547333, 0.020884880796074867, 0.03911050409078598, -0.19827891886234283, -0.1929413080215454, -0.1905088871717453, -0.3163563311100006, 0.16529791057109833, 0.08699171245098114, 0.10312008112668991, 0.08017363399267197, -0.2861329913139343, 0.08007066696882248, -0.17600774765014648, -0.14080439507961273, -0.2249579131603241, 0.41353821754455566, -0.02903660759329796, -0.34544211626052856, -0.0931696742773056, 0.07285867631435394, -0.07893939316272736, -0.37631693482398987, -0.3496829867362976, -0.3218618929386139, 0.03947720676660538, -0.18387570977210999, -0.20712856948375702, 0.004217558074742556, -0.09598862379789352, -0.08139541745185852, -0.14788752794265747, 0.09609728306531906, 0.04650704935193062, -0.10348524153232574, -0.14640654623508453, 0.12001247704029083, -0.09297631680965424, -0.31297439336776733, -0.13514970242977142, -0.10101007670164108, 0.024691380560398102, -0.2474767565727234, -0.0688396468758583, -0.3948144018650055, 0.10869850218296051, -0.10922390222549438, 0.00930073019117117, -0.1933397650718689, 0.14759179949760437, 0.31253668665885925], [-0.13872350752353668, 0.161971777677536, -0.09952545166015625, -0.07133239507675171, -0.11346115916967392, -0.0624583475291729, 0.10622743517160416, 0.023478511720895767, -0.2221618890762329, -0.04465574398636818, 0.03522631898522377, -0.2485518753528595, -0.03545505553483963, 0.24831002950668335, -0.15744774043560028, 0.00976038258522749, 0.034138452261686325, 0.3426307737827301, 0.43800508975982666, 0.3114994168281555, 0.12042055279016495, 0.26731595396995544, 0.03096695989370346, 0.17984086275100708, 0.21130168437957764, -0.11824696511030197, -0.14078287780284882, -0.1314207762479782, -0.0048398044891655445, -0.028874484822154045, 0.042376548051834106, 0.04123586416244507, -0.06083398312330246, -0.36010831594467163, 0.08080723881721497, 0.13441646099090576, 0.00025394067051820457, 0.10526051372289658, -0.07349055260419846, -0.3043651580810547, -0.08106968551874161, -0.0032787113450467587, 0.15246444940567017, 0.058962464332580566, -0.15512417256832123, 0.034948211163282394, 0.06366342306137085, -0.00035017725895158947, -0.2272638976573944, 0.14953865110874176, 0.46869733929634094, 0.11774041503667831, 0.05936252325773239, -0.4390842318534851, -0.061507489532232285, -0.11247299611568451, -0.2226521223783493, -0.1197722777724266, 0.05394755303859711, -0.09033878892660141, 0.5055562257766724, -0.15153345465660095, -0.06665521115064621, 0.19231820106506348, 0.173799067735672, -0.02533060312271118, -0.15354225039482117, 0.37686118483543396, 0.22720320522785187, 0.0019287096802145243, 0.009373167529702187, -0.09116925299167633, 0.018346639350056648, -0.15113642811775208, 0.21431632339954376, -0.20768781006336212, 0.22333404421806335, 0.059498924762010574, -0.24108433723449707, -0.16671334207057953, -0.32015085220336914, -0.18492642045021057, -0.2516164481639862, 0.1205376386642456, 0.006774581968784332, 0.04011451452970505, -0.23443326354026794, 0.01872308738529682, -0.36117881536483765, 0.27888044714927673, -0.10237768292427063, -0.1367446929216385, -0.3301662802696228, 0.2911781668663025, -0.369534969329834, -0.0026318582240492105, 0.10775632411241531, 0.15704870223999023, -0.13568615913391113, -0.20162874460220337, 0.2209637463092804, 0.1501498818397522, -0.15764711797237396, -0.01880510523915291, 0.13358572125434875, 0.21473480761051178, 0.07742881029844284, -0.0930132046341896, -0.03407843038439751, -0.1287134289741516, -0.08276966959238052, 0.04709434509277344, 0.022283080965280533, 0.2667141556739807, 0.1259891837835312, -0.07154220342636108, 0.0018073630053550005, -0.07744939625263214, 0.022582942619919777, -0.23539327085018158, 0.04714002087712288, -0.24148908257484436, -0.01528162881731987, -0.11217678338289261, 0.13235251605510712, 0.029436977580189705, 0.23850880563259125, -0.019192449748516083], [-0.2813947796821594, 0.19192607700824738, -0.20975719392299652, -0.1374022364616394, 0.12763461470603943, -0.1508590281009674, -0.018398120999336243, -0.00286010280251503, -0.011950287036597729, -0.017257025465369225, -0.2609788477420807, -0.0977209210395813, -0.17207884788513184, 0.015865005552768707, -0.2201053351163864, -0.022009195759892464, 0.19925718009471893, 0.2176189422607422, 0.16953112185001373, 0.19007067382335663, -0.034429896622896194, 0.4698623716831207, 0.32293999195098877, 0.15606078505516052, 0.12625379860401154, -0.03488609194755554, -0.19028694927692413, -0.01403668150305748, -0.0271158404648304, -0.02250000275671482, 0.0008699747850187123, -0.02134384959936142, 0.05346114560961723, -0.2505977153778076, -0.14163094758987427, 0.2320467233657837, -0.22435910999774933, 0.09685686230659485, -0.25835493206977844, -0.16035699844360352, -0.08847639709711075, 0.03797281160950661, -0.07982936501502991, 0.19759394228458405, -0.19049546122550964, -0.1014331579208374, -0.12089414149522781, 0.1552213579416275, -0.07480339705944061, 0.2348930537700653, 0.5268072485923767, 0.1348242163658142, -0.0008973993244580925, -0.442939430475235, -0.023869425058364868, -0.16593524813652039, -0.21480457484722137, -0.05263602361083031, 0.10269889235496521, 0.12691737711429596, 0.1812564879655838, 0.01939072087407112, 0.09825150668621063, -0.0655064731836319, -0.07345248013734818, -0.06638504564762115, -0.1621292233467102, 0.1573866903781891, 0.002259068889543414, 0.10331866890192032, -0.17832116782665253, 0.3630916178226471, -0.0427476167678833, -0.13762077689170837, 0.33294618129730225, -0.13922154903411865, 0.11309400945901871, 0.08449608832597733, -0.1470119059085846, -0.017570024356245995, -0.3639850914478302, -0.02197367325425148, -0.2059583216905594, -0.2662321925163269, -0.08317264914512634, -0.0636606514453888, -0.1580948829650879, 0.2180681824684143, -0.13956037163734436, -0.04039115831255913, -0.2004748284816742, -0.05037400871515274, -0.24111022055149078, 0.39435869455337524, -0.28113600611686707, 0.16519694030284882, -0.03909990191459656, -0.09059039503335953, -0.3518521785736084, -0.31225937604904175, 0.07817544043064117, -0.08871088922023773, 0.07981757819652557, -0.27558615803718567, 0.07140602171421051, 0.0948660746216774, -0.0545966774225235, 0.0370321162045002, -0.014942994341254234, 0.0696892961859703, -0.09405235946178436, -0.5041189193725586, -0.09836365282535553, 0.23913416266441345, 0.2527819871902466, -0.13667066395282745, -0.27565357089042664, -0.2551952600479126, 0.02200767584145069, -0.060808565467596054, -0.008844602853059769, -0.5088797807693481, 0.18020860850811005, -0.003889873158186674, -0.18841944634914398, -0.22029170393943787, 0.1475592404603958, 0.056951623409986496], [-0.0390499122440815, 0.028380904346704483, -0.1178964152932167, 0.02724648267030716, 0.009022423066198826, 0.1883915215730667, -0.1428091824054718, 0.20061258971691132, -0.26382753252983093, 0.025802627205848694, -0.23116694390773773, -0.27701497077941895, -0.24267378449440002, 0.042120762169361115, 0.08237893134355545, 0.010027504526078701, -0.06816136837005615, 0.32742616534233093, 0.21291619539260864, 0.1525830328464508, 0.08940651267766953, 0.2084110975265503, -0.06478993594646454, 0.13113081455230713, 0.13443908095359802, -0.05782102048397064, -0.06181599944829941, -0.22770239412784576, 0.05563119053840637, -0.11486274003982544, 0.09217589348554611, -0.0008254458662122488, 0.011239849962294102, -0.3144814670085907, -0.15588247776031494, -0.08972042053937912, -0.11583515256643295, 0.15738177299499512, -0.10267277806997299, -0.10206926614046097, 0.13774900138378143, 0.1570623517036438, -0.22263203561306, 0.056111447513103485, -0.08772727847099304, 0.05485527217388153, -0.12438706308603287, 0.03403889760375023, -0.05624978989362717, 0.1053927019238472, 0.17397309839725494, 0.18925994634628296, -0.02590259723365307, -0.27156785130500793, -0.046600811183452606, -0.15612909197807312, -0.05391981825232506, -0.019781121984124184, -0.028914455324411392, -0.08319617807865143, 0.12603448331356049, -0.18327254056930542, 0.027499990537762642, 0.12947924435138702, -0.06435936689376831, 0.045590370893478394, -0.5145623087882996, 0.4512499272823334, 0.08361072838306427, 0.016259996220469475, -0.1588696539402008, 0.33296117186546326, -0.04438965395092964, -0.03970837593078613, 0.21367274224758148, -0.15507377684116364, 0.18377403914928436, 0.08442205935716629, -0.14977626502513885, -0.09602144360542297, -0.2445959895849228, -0.19623902440071106, -0.12723323702812195, -0.05817612633109093, -0.0016044502845034003, -0.20609234273433685, -0.14830391108989716, -0.03692575916647911, -0.22694548964500427, 0.004612424410879612, -0.10833652317523956, -0.1851377934217453, -0.2632054388523102, 0.44461315870285034, -0.27038708329200745, -0.22532832622528076, -0.057490259408950806, -0.040388524532318115, -0.07796125113964081, -0.12165963649749756, -0.2145826816558838, 0.08376102149486542, 0.005382079631090164, 0.06721305102109909, -0.02100486308336258, 0.05583822354674339, -0.011014300398528576, -0.05246186628937721, -0.03353488817811012, 0.03465275093913078, 0.026964833959937096, -0.23619914054870605, 0.05413808673620224, 0.1384512484073639, 0.21401278674602509, -0.214561328291893, -0.20560204982757568, -0.2231132537126541, -0.01641751267015934, -0.18824392557144165, -0.15327738225460052, -0.16093283891677856, -0.21814028918743134, -0.146469384431839, 0.0719657689332962, -0.07705642282962799, 0.43009230494499207, 0.16607451438903809], [-0.1636897772550583, -0.0003493220137897879, -0.061621006578207016, 0.013694994151592255, -0.13157209753990173, 0.284988135099411, -0.48746782541275024, 0.0638558566570282, -0.5402998924255371, 0.3010253608226776, -0.08427846431732178, -0.11606119573116302, -0.08442452549934387, 0.10919977724552155, -0.2812345325946808, 0.3361848294734955, 0.08540478348731995, 0.7225276231765747, 0.09401653707027435, 0.09605754166841507, 0.4794512987136841, 0.2817320227622986, 0.07038184255361557, -0.07098620384931564, 0.22850625216960907, -0.31972450017929077, -0.5009698271751404, 0.10462215542793274, 0.11017409712076187, -0.039371732622385025, 0.22964215278625488, 0.08909095078706741, -0.20063872635364532, -0.4194258451461792, 0.11474539339542389, 0.0947648137807846, -0.14929042756557465, -0.3468475043773651, 0.10206460952758789, -0.07013681530952454, -0.15326948463916779, -0.27100300788879395, 0.10574674606323242, 0.09476163238286972, -0.1957979053258896, -0.04472026228904724, -0.013828122057020664, 0.33046776056289673, -0.3848985433578491, 0.18951471149921417, 0.4838014543056488, 0.17542487382888794, -0.06721388548612595, -0.45606425404548645, 0.1443275511264801, 0.1367274522781372, 0.034506503492593765, 0.1511027216911316, 0.13257606327533722, 0.3079901337623596, 0.17970888316631317, -0.2522016167640686, -0.07437048852443695, 0.08140174299478531, -0.1985105723142624, 0.058296993374824524, -0.39698028564453125, 0.05331737548112869, 0.025812562555074692, -0.020027952268719673, 0.2446063756942749, 0.047178003937006, 0.077494777739048, 0.034280817955732346, 0.0333033911883831, 0.033450622111558914, 0.2303663194179535, -0.36511877179145813, -0.33982643485069275, 0.39016762375831604, 0.20282365381717682, -0.16402235627174377, -0.23317715525627136, 0.06897930800914764, -0.1371898502111435, 0.12521471083164215, -0.06350003182888031, 0.344718873500824, -0.2902994155883789, 0.3213091492652893, -0.07298824191093445, 0.2331111580133438, 0.1509988009929657, 0.3453921973705292, -0.3849005401134491, 0.4489954113960266, -0.08181565999984741, -0.35784828662872314, -0.20663705468177795, -0.36504364013671875, -0.03635687381029129, -0.2906445562839508, 0.09427763521671295, 0.2171640694141388, -0.1454215794801712, 0.1704300045967102, 0.07802808284759521, -0.33571574091911316, -0.03402413800358772, 0.029331272467970848, -0.10999671369791031, -0.183636873960495, 0.27573564648628235, -0.11050646752119064, 0.2676985561847687, -0.3953617513179779, -0.1566685140132904, -0.10310877859592438, 0.07600310444831848, -0.25391656160354614, -0.3016659617424011, -0.6067430377006531, -0.015460082329809666, 0.13062256574630737, 0.027486585080623627, 0.1693722903728485, 0.23343142867088318, 0.34273406863212585], [-0.09266118705272675, -0.1765005886554718, -0.07829457521438599, -0.15073072910308838, 0.05990628898143768, 0.13236624002456665, -0.05030838027596474, 0.017485611140727997, -0.24488404393196106, -0.04493992030620575, -0.07809164375066757, -0.03066437691450119, 0.26725858449935913, 0.17916032671928406, -0.44635695219039917, -0.06654250621795654, -0.056182052940130234, 0.4684833884239197, 0.29153168201446533, 0.00941537320613861, 0.09520908445119858, -0.07995246350765228, -0.07706243544816971, -0.12663042545318604, 0.02002812922000885, -0.252655565738678, 0.02595832198858261, -0.0017916469369083643, 0.17316904664039612, -0.0376819409430027, 0.08547227829694748, 0.20290224254131317, -0.001392102101817727, -0.2848256826400757, -0.13599584996700287, -0.11996959149837494, -0.01233602687716484, -0.015304368920624256, -0.1794959306716919, -0.025217873975634575, 0.0279588233679533, -0.11610332131385803, -0.010109326802194118, 0.21233414113521576, -0.04098424315452576, -0.14812901616096497, 0.06113371253013611, 0.35035425424575806, -0.24953263998031616, 0.12254329770803452, 0.33171921968460083, 0.30007532238960266, 0.0686667263507843, -0.18064771592617035, -0.011047976091504097, 0.10170257091522217, -0.3302542567253113, -0.21312862634658813, 0.06611562520265579, 0.09724976867437363, 0.27973830699920654, -0.02253872901201248, 0.10553532093763351, -0.003250076901167631, -0.16311942040920258, -0.3211192786693573, -0.3113659918308258, 0.303329199552536, 0.17478346824645996, -0.06880407780408859, -0.10504277795553207, -0.16161778569221497, -0.20216649770736694, -0.13295798003673553, 0.0961964800953865, -0.1368662714958191, 0.11684495955705643, 0.008057212457060814, -0.08628752827644348, -0.15391306579113007, -0.23565849661827087, -0.08004384487867355, -0.07448232918977737, 0.09524710476398468, 0.36325857043266296, -0.38529759645462036, 0.14840403199195862, -0.17127127945423126, -0.3553031086921692, 0.211586594581604, -0.014555131085216999, -0.047659315168857574, -0.23572790622711182, 0.3488835394382477, -0.18785755336284637, 0.0560208261013031, 0.1020013839006424, 0.01533921342343092, -0.2859272360801697, -0.15824665129184723, -0.07567940652370453, -0.06089458614587784, -0.02566588670015335, 0.016675293445587158, 0.10592997074127197, 0.32066842913627625, 0.008155499584972858, -0.19000910222530365, -0.09053748100996017, 0.12213373929262161, -0.04249994084239006, 0.07293327897787094, 0.2005753219127655, 0.1674441546201706, 0.14572133123874664, -0.2662715017795563, -0.20616461336612701, -0.13500748574733734, 0.21470268070697784, -0.11742403358221054, -0.2109953612089157, -0.31562238931655884, -0.11929666996002197, -0.11645084619522095, 0.15396438539028168, -0.028779542073607445, 0.3834591507911682, -0.1207042783498764], [-0.13139216601848602, -0.07099051773548126, -0.07260611653327942, -0.37697258591651917, 0.004677463788539171, -0.3323618769645691, -0.03557595610618591, 0.06725777685642242, -0.5229089856147766, 0.23536472022533417, -0.36578693985939026, 0.12594610452651978, 0.06792117655277252, 0.3496442437171936, -0.6021110415458679, -0.137144535779953, -0.08121779561042786, 0.254940927028656, 0.45668908953666687, 0.11476149410009384, 0.17384596168994904, 0.03201533854007721, 0.08232807368040085, 0.07054946571588516, 0.3146068751811981, 0.012517415918409824, -0.12033951282501221, 0.0544622540473938, 0.12871499359607697, -0.35713639855384827, 0.09416606277227402, 0.3925585150718689, 0.20140895247459412, -0.343650758266449, 0.08244343847036362, -0.27391812205314636, -0.13402222096920013, 0.03533286973834038, -0.18247200548648834, -0.3583168387413025, -0.03073071502149105, -0.15524911880493164, 0.13895604014396667, 0.07982587814331055, -0.1690162718296051, -0.07865924388170242, -0.0481853261590004, 0.29775089025497437, -0.02329489402472973, -0.04148600623011589, 0.18837352097034454, 0.3056676685810089, 0.0276801660656929, -0.2967716455459595, -0.03130427375435829, -0.2662399709224701, -0.37400925159454346, -0.12993071973323822, 0.2372896671295166, 0.07851805537939072, 0.5306320190429688, -0.03512270376086235, 0.187118798494339, -0.28391143679618835, -0.11256717145442963, -0.10432601720094681, -0.27609536051750183, 0.308483749628067, -0.09870002418756485, 0.05382755026221275, -0.3117174506187439, -0.0725061371922493, -0.07658453285694122, -0.27805009484291077, 0.09548982977867126, -0.041895415633916855, 0.10865616798400879, 0.15536357462406158, -0.10424195975065231, 0.12581342458724976, -0.2837502956390381, -0.12243945151567459, -0.36067935824394226, 0.054701387882232666, 0.03183142468333244, -0.1855190098285675, 0.33199113607406616, 0.08647835999727249, -0.21561363339424133, 0.07561211287975311, -0.0815315991640091, 0.2500704526901245, -0.24456405639648438, 0.42801108956336975, 0.07750983536243439, 0.15821252763271332, 0.0023945653811097145, -0.058916185051202774, -0.1730976551771164, -0.27377647161483765, 0.17031991481781006, -0.22306253015995026, 0.32243436574935913, -0.18455085158348083, -0.25417548418045044, 0.3218444585800171, 0.09532641619443893, -0.1635582149028778, -0.07151786237955093, -0.24062079191207886, 0.0514632947742939, -0.016213631257414818, 0.02899685688316822, 0.32564905285835266, 0.10771891474723816, -0.3204384446144104, 0.29388508200645447, 0.13301853835582733, 0.12635833024978638, -0.39511051774024963, 0.2113787829875946, -0.3784061074256897, 0.06775377690792084, -0.06075714901089668, 0.2743503451347351, -0.13743935525417328, 0.1807219684123993, -0.1689545065164566], [-0.19340109825134277, 0.1862180531024933, 0.013255147263407707, -0.14574462175369263, -0.12627778947353363, 0.037056636065244675, -0.18188942968845367, -0.20452164113521576, -0.03549707308411598, -0.09455995261669159, 0.022849582135677338, -0.19376860558986664, -0.1653965562582016, -0.1398097723722458, -0.08816792070865631, -0.24931217730045319, -0.09799151122570038, 0.4969976246356964, 0.16422216594219208, -0.08560004830360413, 0.38113880157470703, 0.23243308067321777, 0.3478050231933594, 0.12843424081802368, 0.06784132868051529, -0.07797626405954361, -0.013911422342061996, -0.1741226613521576, -0.02750530280172825, 0.09095702320337296, 0.0726160928606987, -0.024033555760979652, 0.04779445379972458, -0.4042781889438629, -0.09599840641021729, 0.4023886024951935, -0.07524536550045013, 0.12033657729625702, -0.0882466733455658, -0.12118352949619293, -0.13709001243114471, 0.1313566118478775, -0.08569937944412231, 0.08236654102802277, 0.017443018034100533, 0.0258717630058527, 0.3298667371273041, 0.21433141827583313, -0.15003515779972076, 0.2687891125679016, 0.34767597913742065, 0.17859043180942535, -0.17942620813846588, -0.30762240290641785, -0.07306421548128128, 0.12382764369249344, -0.20861174166202545, -0.21397775411605835, -0.27831363677978516, 0.2168031632900238, 0.18114805221557617, 0.10466741770505905, 0.08581769466400146, -0.026506660506129265, -0.03642672300338745, -0.3380793333053589, -0.28410235047340393, 0.15292327105998993, 0.30856853723526, 0.3668324649333954, 0.10950583964586258, 0.33525320887565613, -0.2042902261018753, 0.11711902916431427, 0.346312016248703, -0.14076927304267883, 0.13171793520450592, -0.0750952810049057, -0.2280953973531723, -0.13874997198581696, -0.4276772737503052, 0.152617946267128, -0.25236791372299194, -0.1360204964876175, 0.11894389986991882, -0.10408112406730652, -0.2488664984703064, 0.058928001672029495, -0.10486474633216858, 0.06213918700814247, -0.1976545751094818, -0.11637567728757858, -0.11693336814641953, 0.06323635578155518, -0.2286776602268219, 0.25703203678131104, -0.1462291181087494, -0.019869577139616013, -0.3801362216472626, -0.07004328817129135, 0.23094868659973145, -0.3316975235939026, 0.013142256997525692, -0.10697507858276367, 0.19788485765457153, 0.10456351935863495, -0.051405176520347595, 0.11113937944173813, -0.15320724248886108, -0.1525694578886032, -0.0980360135436058, -0.20338305830955505, -0.12467862665653229, 0.31241774559020996, 0.322152316570282, -0.38111671805381775, -0.2174111157655716, -0.1712602823972702, -0.05043358728289604, -0.07589883357286453, -0.007741844281554222, -0.37722402811050415, 0.09337560087442398, 0.10834184288978577, 0.04680599272251129, -0.11189878731966019, 0.10685475915670395, 0.05490538105368614], [0.3506794273853302, 0.5045953392982483, -0.3000844419002533, -0.14738765358924866, -0.24320724606513977, 0.3589850664138794, -0.4935999810695648, 0.492946594953537, -0.5754081010818481, 0.08729492872953415, -0.04662618413567543, -0.47796985507011414, -0.0062664276920259, 0.3088149428367615, -0.3135519027709961, -0.05446616932749748, -0.24106721580028534, 0.6513293385505676, 0.019369518384337425, 0.19528767466545105, 0.6346251964569092, 0.4498125910758972, -0.10249127447605133, -0.08979522436857224, -0.3104695975780487, -0.7532810568809509, -0.7373539805412292, -0.36533576250076294, 0.003704879665747285, -0.10038402676582336, 0.6085395812988281, 0.39033982157707214, 0.08777152001857758, -0.1981654018163681, 0.1010681688785553, 0.07104956358671188, -0.18180295825004578, -0.09612300992012024, -0.4201350808143616, -0.34735074639320374, -0.0927022248506546, -0.4213482737541199, 0.10259439051151276, 0.2793486714363098, -0.21405775845050812, -0.26979178190231323, -0.14022496342658997, 0.11076946556568146, -0.2106797844171524, -0.28972098231315613, 0.06296760588884354, 0.12409931421279907, 0.22585897147655487, -0.3794146180152893, -0.28323209285736084, -0.16712301969528198, -0.1463988870382309, 0.23954986035823822, 0.41766121983528137, -0.41532567143440247, 0.3282650113105774, -0.3493419885635376, -0.3111967146396637, 0.3661639988422394, -0.32329481840133667, 0.14915332198143005, -0.21790136396884918, 0.3582354187965393, 0.007207956165075302, -0.04079286754131317, 0.256500244140625, -0.32283103466033936, -0.036535054445266724, 0.30036047101020813, 0.3453887403011322, -0.11095957458019257, 0.07286370545625687, -0.3483472168445587, -0.4825405180454254, 0.4268537759780884, -0.060508761554956436, -0.11503253132104874, -0.0008970624767243862, -0.30850207805633545, -0.1086161658167839, 0.04999542236328125, 0.20369839668273926, -0.035014405846595764, -0.39966750144958496, 0.34322604537010193, 0.018113581463694572, -0.3307730257511139, -0.38707906007766724, 0.5485050678253174, -0.033651456236839294, 0.2830835282802582, 0.24599261581897736, -0.08373933285474777, -0.026372846215963364, -0.23937025666236877, 0.08604368567466736, -0.02408369816839695, -0.0016447360394522548, 0.11130189150571823, -0.4530547261238098, 0.2369874268770218, 0.16033650934696198, -0.607680082321167, -0.09827408194541931, 0.05179457738995552, -0.177150696516037, -0.2663169503211975, 0.06330585479736328, -0.13195982575416565, 0.3765150308609009, -0.5699148774147034, -0.22643636167049408, 0.20759907364845276, -0.11504746973514557, -0.3883189558982849, -0.05079605430364609, -0.33550336956977844, 0.015575644560158253, -0.22501201927661896, -0.02265450730919838, 0.3006669580936432, 0.5152472853660583, 0.27562958002090454], [-0.33420073986053467, 0.40015658736228943, -0.2390594482421875, -0.1137208417057991, -0.055877309292554855, 0.010244798846542835, -0.20998860895633698, 0.1369425356388092, -0.48141688108444214, -0.017519477754831314, 0.04978957399725914, -0.008182222954928875, -0.37303829193115234, -0.14753076434135437, -0.07031507045030594, -0.05361293628811836, -0.1988670527935028, 0.5927957892417908, 0.23267433047294617, -0.07274728268384933, 0.04247556999325752, 0.021581314504146576, 0.08671983331441879, 0.028526997193694115, 0.2552075684070587, -0.163517564535141, -0.007248944137245417, -0.11675411462783813, -0.23222999274730682, 0.3002868592739105, 0.10539453476667404, -0.07500540465116501, -0.08750949054956436, -0.3802586495876312, -0.02628822810947895, 0.0032393066212534904, -0.14633582532405853, 0.04842458292841911, 0.17337319254875183, 0.01593911089003086, -0.17891453206539154, -0.11947433650493622, 0.27424493432044983, 0.017320221289992332, 0.15078558027744293, 0.08361915498971939, 0.1445089876651764, 0.04085160046815872, -0.3840329647064209, 0.36711758375167847, 0.7405015230178833, 0.3572130799293518, -0.18105252087116241, -0.22834914922714233, 0.11383459717035294, -0.18084295094013214, -0.15092003345489502, -0.05663343146443367, -0.029249409213662148, -0.0042305500246584415, 0.17897704243659973, -0.34002822637557983, -0.11042741686105728, 0.16072748601436615, 0.07217929512262344, -0.22435641288757324, -0.6114444136619568, 0.317933052778244, -0.09856481105089188, -0.050914958119392395, 0.10250431299209595, -0.18527352809906006, 0.014318455941975117, -0.20278459787368774, 0.330059289932251, 0.04043131321668625, 0.058937203139066696, 0.16278761625289917, -0.38191384077072144, 0.004377252422273159, -0.1715582013130188, -0.4262738823890686, -0.1779605895280838, -0.2911611497402191, -0.3444550335407257, -0.12254427373409271, -0.10014937818050385, 0.31042733788490295, -0.08664483577013016, 0.21351595222949982, -0.10739541798830032, 0.05172397941350937, -0.2196473330259323, 0.32989147305488586, -0.6548007130622864, 0.16338865458965302, 0.21464116871356964, 0.1988171935081482, -0.2610510289669037, 0.023158837109804153, 0.06267543137073517, 0.3646898567676544, -0.10549046844244003, 0.08099757879972458, 0.25753796100616455, 0.06935471296310425, 0.18679724633693695, -0.025822818279266357, 0.00334634305909276, -0.24274533987045288, -0.45769354701042175, -0.4029834270477295, 0.12136393785476685, 0.14084871113300323, 0.2590681314468384, -0.0765773355960846, 0.054066501557826996, -0.26173681020736694, 0.05622145161032677, 0.1270928531885147, -0.07530343532562256, -0.12698593735694885, -0.06790310889482498, -0.08271026611328125, 0.13517314195632935, 0.15875431895256042, 0.1530112624168396, 0.3622991740703583]]]], "dtype": "float32"}, {"__nd__": [[[-0.21019108593463898, 0.6994386911392212, 0.6902567744255066, 2.1562659740448, 2.4085512161254883, 2.4700779914855957, 2.6417248249053955, 2.082996368408203, 2.45430850982666, 2.7222630977630615, 3.21394944190979, 2.8072593212127686, 3.4088425636291504, 2.5120348930358887, 2.993457555770874, 3.1439285278320312, 3.762765645980835, 3.503880023956299, 2.965334892272949, 3.2145376205444336, 3.9100189208984375, 3.9595699310302734, 3.3473713397979736, 3.038750410079956, 3.325629472732544, 3.4944143295288086, 3.359886646270752, 3.8355674743652344, 3.707515239715576, 4.205406665802002, 4.16417932510376, 3.8356518745422363, 4.1839776039123535, 3.592161178588867, 3.803316354751587, 4.023355960845947, 4.450526237487793, 4.196099281311035, 4.494104862213135, 4.320095539093018, 4.282341480255127, 4.0003342628479, 4.070116996765137, 4.792689323425293, 3.9330554008483887, 4.591315269470215, 4.312838077545166, 4.263938903808594, 4.711732387542725, 4.6462178230285645, 4.250148296356201, 4.397007465362549, 4.241947174072266, 4.393022537231445, 4.443912506103516, 4.592034816741943, 4.510146617889404, 4.440439701080322, 4.381387710571289, 4.5112738609313965, 4.435217380523682, 4.856910228729248, 4.654081344604492, 4.51605749130249, 4.798791885375977]]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(flash_attn_fwd, json.loads(_TESTS), 'triton', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")